# Stage 2: Predictor Construction

## Literature basis

Predictor constructs were defined before model fitting and without using the age-18 outcome to select variables. Previous work using Next Steps and related longitudinal education data, particularly Dickerson, McDool and Morris (2023), informed the substantive coverage. Comparable predictive studies were used to check whether broad behavioural, social, educational and contextual information had been omitted.

The review considers the following 11 conceptual domains. Ten are represented in the final 69-predictor matrix because no suitable direct prior-attainment measure was retained.

| Predictor domain | Constructs considered |
| --- | --- |
| Demographic background | Sex, ethnicity, relative age and pre-transition language background |
| Family socioeconomic background | Parental education, occupational position, household structure, financial circumstances, housing and home learning resources |
| Prior attainment | KS2, KS3 and KS4 attainment, including English and mathematics |
| SEN, disability and health | SEN provision, disability, general health and related limitations |
| Educational aspirations and post-16 plans | Own expected route and higher-education application expectations |
| School experiences and engagement | School attitudes, homework, truancy, exclusion, school climate, school mobility and vocational-course experience |
| Psychosocial characteristics | Psychological distress and academic self-concept |
| Experiences and behaviours | Bullying, paid work, caring responsibilities, substance use, antisocial behaviour and police contact |
| Parental attitudes, support and engagement | Parental aspirations, educational support, monitoring, communication and school engagement |
| Post-16 social influences and guidance | Peer expectations, family and peer discussion, parental training discussion, teacher, careers, Connexions and mentor guidance |
| School and local context | School sector, region and urban–rural setting |

The representation of each construct is determined through source-file inspection, documentation review, measurement timing, routing, coverage, coding and comparison with credible alternatives. Domain sizes are not forced to be equal because the later domain analysis concerns the predictive contribution of substantive information bundles rather than an equal number of variables per domain.

## Data and documentation review

The source directory is inventoried before files are selected for detailed predictor review. Candidate files are identified from the observed file names and compared with the complete Stata-file inventory. Files outside the eligible predictor period and post-16 activity sources are recorded separately with their exclusion basis.

Candidate files are then reviewed using variable labels, value labels, questionnaires, data dictionaries and derived-variable documentation. The review covers:

- respondent and source file;
- measurement wave, timing and reference period;
- questionnaire routing and structural non-applicability;
- variable coding, negative values and derived-variable definitions;
- participant coverage and missingness;
- repeated, alternative and overlapping measures.

Wave 1 and Wave 2 are treated as pre-transition sources. Wave 3 requires interview-timing and item-reference-period checks. Wave 4 is restricted to stable characteristics or retrospective information that refers to the pre-transition period. Current Wave 4 measures are not used as fallbacks for time-varying pre-transition characteristics.

## Predictor review sequence

1. Definition of substantive predictor constructs from previous research.
2. Inventory of available source files.
3. Identification of candidate source files and documentation of exclusions.
4. Inspection of candidate file structure, identifiers and participant coverage.
5. Identification of candidate variables within each domain.
6. Assessment of timing, routing, coding, coverage and overlap.
7. Comparison of alternative representations.
8. Final cross-domain specification and provenance checks.
9. Construction of the participant-level predictor matrix and decision register.

Imputation, encoding, scaling, resampling, model-based feature selection and model fitting are outside this notebook. Stage 2 transformations are deterministic; no random procedure is used.

## Data source

University College London, UCL Institute of Education, Centre for Longitudinal Studies. *Next Steps* [data collection]. UK Data Service. Study Number 5545.

# Part 1: Project setup, participant coverage and source file review


In [1]:
# 1: Project paths and package imports

import os
from pathlib import Path
import re

import pandas as pd
from IPython.display import display

TABLE_ROW_LIMIT = 5

def display_limited(*objects, **kwargs):
    for obj in objects:
        if isinstance(obj, (pd.DataFrame, pd.Series)):
            display(obj.head(TABLE_ROW_LIMIT), **kwargs)
        else:
            display(obj, **kwargs)

def display_full(obj):
    with pd.option_context(
        'display.max_rows', None,
        'display.max_columns', None,
        'display.width', 220,
    ):
        display(obj)

def summarise_nsid(data):
    """Summarise NSID completeness and uniqueness."""
    nsid_present = 'NSID' in data.columns
    if not nsid_present:
        return pd.DataFrame({
            'Check': ['NSID variable present'],
            'Result': [False],
        })

    return pd.DataFrame({
        'Check': [
            'NSID variable present',
            'Number of rows',
            'Unique NSID values',
            'Missing NSID values',
            'Duplicate NSID rows',
        ],
        'Result': [
            True,
            len(data),
            data['NSID'].nunique(dropna=True),
            int(data['NSID'].isna().sum()),
            int(data['NSID'].duplicated().sum()),
        ],
    })

pd.set_option('display.max_rows', TABLE_ROW_LIMIT)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 180)

# Locate the same project root whether the notebook is run from the root or notebooks/.
current_path = Path.cwd().resolve()
resolved_project_root = next(
    (
        path
        for path in [current_path, *current_path.parents]
        if (path / 'data_working').is_dir()
        and (path / 'data_derived').is_dir()
    ),
    None,
)

if resolved_project_root is None:
    raise FileNotFoundError(
        'A project folder containing data_working and data_derived was not found.'
    )

# Work from the project root so paths stored in outputs remain portable and non-identifying.
os.chdir(resolved_project_root)
project_root = Path('.')

stage_1_outcome_file = (
    project_root
    / 'data_derived'
    / 'stage_1_outcome_construction'
    / 'age18_activity_outcome.csv'
)
source_directory = (
    project_root
    / 'data_working'
    / 'nextsteps_5545'
    / 'UKDA-5545-stata'
    / 'stata'
    / 'stata13'
    / 'safeguarded_eul'
)
stage_2_output_directory = (
    project_root
    / 'data_derived'
    / 'stage_2_predictor_construction'
)

# Only the participant identifier is taken from Stage 1; outcome values are not loaded here.
input_path_checks = pd.DataFrame({
    'Item': [
        'Project root',
        'Stage 1 outcome file used to define the participant roster',
        'Next Steps source directory',
    ],
    'Path': [
        str(project_root),
        str(stage_1_outcome_file),
        str(source_directory),
    ],
    'Exists': [
        project_root.is_dir(),
        stage_1_outcome_file.is_file(),
        source_directory.is_dir(),
    ],
})
display(input_path_checks)

if not input_path_checks['Exists'].all():
    missing_inputs = input_path_checks.loc[
        ~input_path_checks['Exists'],
        ['Item', 'Path'],
    ]
    raise FileNotFoundError(
        'Required project inputs were not found:\n'
        + missing_inputs.to_string(index=False)
    )

stage_2_output_directory.mkdir(parents=True, exist_ok=True)

output_path_check = pd.DataFrame({
    'Item': ['Stage 2 output directory'],
    'Path': [str(stage_2_output_directory)],
    'Exists': [stage_2_output_directory.is_dir()],
})
display(output_path_check)
assert output_path_check['Exists'].all()


,Item,Path,Exists
0,Project root,.,True
1,Stage 1 outcome file used to define the partic...,data_derived\stage_1_outcome_construction\age1...,True
2,Next Steps source directory,data_working\nextsteps_5545\UKDA-5545-stata\st...,True


,Item,Path,Exists
0,Stage 2 output directory,data_derived\stage_2_predictor_construction,True


In [2]:
# 2: Participant identifiers from the Stage 1 outcome file

participant_roster = pd.read_csv(stage_1_outcome_file, usecols=['NSID'])
# The Stage 1 file supplies the eligible outcome roster, but the outcome itself is deliberately excluded from predictor construction.
assert participant_roster.columns.tolist() == ['NSID']
print(f'Rows: {participant_roster.shape[0]:,}')
print(f'Columns: {participant_roster.shape[1]}')
print(f'Column names: {participant_roster.columns.tolist()}')

Rows: 9,767
Columns: 1
Column names: ['NSID']


In [3]:
# 3: Participant identifier checks

participant_identifier_checks = summarise_nsid(participant_roster)
display(participant_identifier_checks)
assert participant_roster['NSID'].notna().all()
assert not participant_roster['NSID'].duplicated().any()
participant_roster_nsids = set(participant_roster['NSID'])

,Check,Result
0,NSID variable present,True
1,Number of rows,9767
2,Unique NSID values,9767
3,Missing NSID values,0
4,Duplicate NSID rows,0


In [4]:
# 4: Source file inventory

stata_files = sorted(source_directory.rglob('*.dta'), key=lambda path: str(path).lower())
source_file_inventory = pd.DataFrame({'File name': [file_path.name for file_path in stata_files],
    'Relative path': [str(file_path.relative_to(source_directory)) for file_path in stata_files]})
duplicate_source_file_names = source_file_inventory['File name'].duplicated(keep=False)
print(f'Stata files in source directory: {len(source_file_inventory):,}')
print(f'Duplicated file names: {int(duplicate_source_file_names.sum()):,}')
assert not source_file_inventory.empty
assert not duplicate_source_file_names.any()
display_limited(source_file_inventory)

Stata files in source directory: 38
Duplicated file names: 0


,File name,Relative path
0,lsype_history_file_wave_one_and_wave_two_june_...,lsype_history_file_wave_one_and_wave_two_june_...
1,lsype_main_activity_w4-7_nov2011_suppressed.dta,lsype_main_activity_w4-7_nov2011_suppressed.dta
2,next_steps_activities_longitudinal.dta,next_steps_activities_longitudinal.dta
3,ns8_2015_benefits.dta,ns8_2015_benefits.dta
4,ns8_2015_benefits_unfolding_brackets.dta,ns8_2015_benefits_unfolding_brackets.dta


## Candidate source file naming rules

The source-file inventory was reviewed before the candidate-file rule was defined. The Wave 1–4 young-person, family-background and parental-attitudes files shared the same wave-based naming structure. The Wave 4 history file followed the same structure. The combined Waves 1–2 history file used a different file name and was therefore identified separately by its complete name.


In [5]:
# 5: Candidate source files for detailed review

wave_1_to_4_pattern = '^wave_(?:one|two|three|four)_lsype_(?:young_person|family_background|parental_attitudes|history)'
wave_1_to_4_mask = source_file_inventory['File name'].str.match(wave_1_to_4_pattern, case=False, na=False)
wave_1_2_history_mask = source_file_inventory['File name'].eq('lsype_history_file_wave_one_and_wave_two_june_2008.dta')
candidate_source_files = source_file_inventory.loc[wave_1_to_4_mask | wave_1_2_history_mask].copy().reset_index(drop=True)
candidate_source_files['Candidate basis'] = candidate_source_files['File name'].map(lambda file_name: 'Waves 1–2 LSYPE history file identified from the file name' if file_name == 'lsype_history_file_wave_one_and_wave_two_june_2008.dta' else 'Wave 1–4 LSYPE file identified from the file name')
assert not candidate_source_files.empty
assert not candidate_source_files['File name'].duplicated().any()
print(f'Candidate files for detailed review: {len(candidate_source_files):,}')
display_limited(candidate_source_files)

Candidate files for detailed review: 14


,File name,Relative path,Candidate basis
0,lsype_history_file_wave_one_and_wave_two_june_...,lsype_history_file_wave_one_and_wave_two_june_...,Waves 1–2 LSYPE history file identified from t...
1,wave_four_lsype_family_background_2020.dta,wave_four_lsype_family_background_2020.dta,Wave 1–4 LSYPE file identified from the file name
2,wave_four_lsype_history_2020.dta,wave_four_lsype_history_2020.dta,Wave 1–4 LSYPE file identified from the file name
3,wave_four_lsype_parental_attitudes_june_2009.dta,wave_four_lsype_parental_attitudes_june_2009.dta,Wave 1–4 LSYPE file identified from the file name
4,wave_four_lsype_young_person_2020.dta,wave_four_lsype_young_person_2020.dta,Wave 1–4 LSYPE file identified from the file name


In [6]:
# 6: Files outside the detailed predictor review

candidate_source_file_names = set(candidate_source_files['File name'])
files_outside_detailed_review = source_file_inventory.loc[~source_file_inventory['File name'].isin(candidate_source_file_names)].copy().reset_index(drop=True)
source_file_partition_checks = pd.DataFrame({'Check': ['Stata files in source directory',
    'Candidate files for detailed review', 'Files outside detailed review', 'Files assigned to both groups', 'Files assigned to neither group'], 'Result': [len(source_file_inventory),
    len(candidate_source_files), len(files_outside_detailed_review), len(candidate_source_file_names & set(files_outside_detailed_review['File name'])), len(source_file_inventory) - len(candidate_source_files) - len(files_outside_detailed_review)]})
display(source_file_partition_checks)
assert len(candidate_source_files) + len(files_outside_detailed_review) == len(source_file_inventory)

,Check,Result
0,Stata files in source directory,38
1,Candidate files for detailed review,14
2,Files outside detailed review,24
3,Files assigned to both groups,0
4,Files assigned to neither group,0


## Source file exclusion criteria

Files outside detailed predictor review were classified using the wave, follow-up year and activity-related information recorded in their file names.

Post-16 activity files were separated from predictor sources because they relate to outcome construction or supplementary activity-history review. Wave 5–7 files and the 2015, 2019 and 2022 follow-up files were excluded because they fall outside the eligible pre-transition predictor period.

The following audit records the classification and exclusion basis for each file outside detailed predictor review.

In [7]:
# 7: Source file exclusion check

post16_activity_files = {'lsype_main_activity_w4-7_nov2011_suppressed.dta', 'next_steps_activities_longitudinal.dta'}

def classify_file_outside_review(file_name):
    if file_name in post16_activity_files:
        return {'File group': 'Post-16 activity source',
            'Exclusion basis': 'The file name identifies post-16 activity information. This information belongs to outcome construction or supplementary activity-history review rather than predictor construction.'}
    if re.match('^wave_(?:five|six|seven)_lsype_', file_name, flags=re.IGNORECASE):
        return {'File group': 'Wave 5–7 source',
            'Exclusion basis': 'The file name identifies a wave after the eligible pre-transition predictor period.'}
    if file_name.startswith('ns8_2015_'):
        return {'File group': '2015 follow-up source',
            'Exclusion basis': 'The file name identifies the 2015 follow-up, which is after the eligible predictor and age-18 outcome periods.'}
    if file_name == 'ns_2019_web_survey.dta':
        return {'File group': '2019 follow-up source',
            'Exclusion basis': 'The file name identifies the 2019 follow-up, which is after the eligible predictor and age-18 outcome periods.'}
    if file_name.startswith('ns9_2022_'):
        return {'File group': '2022 follow-up source',
            'Exclusion basis': 'The file name identifies the 2022 follow-up, which is after the eligible predictor and age-18 outcome periods.'}
    return {'File group': 'Requires review', 'Exclusion basis': 'No file-name rule assigned.'}
source_file_exclusion_audit = pd.DataFrame([{'File name': row['File name'], 'Relative path': row['Relative path'],
    **classify_file_outside_review(row['File name'])} for _, row in files_outside_detailed_review.iterrows()])
unclassified_source_files = source_file_exclusion_audit.loc[source_file_exclusion_audit['File group'].eq('Requires review')]
if not unclassified_source_files.empty:
    raise ValueError('Files requiring a separate source-level review were found:\n' + unclassified_source_files.to_string(index=False))
source_file_exclusion_summary = source_file_exclusion_audit['File group'].value_counts().rename_axis('File group').reset_index(name='Files')
display(source_file_exclusion_summary)
display_limited(source_file_exclusion_audit)

,File group,Files
0,2022 follow-up source,9
1,2015 follow-up source,8
2,Wave 5–7 source,4
3,Post-16 activity source,2
4,2019 follow-up source,1


,File name,Relative path,File group,Exclusion basis
0,lsype_main_activity_w4-7_nov2011_suppressed.dta,lsype_main_activity_w4-7_nov2011_suppressed.dta,Post-16 activity source,The file name identifies post-16 activity info...
1,next_steps_activities_longitudinal.dta,next_steps_activities_longitudinal.dta,Post-16 activity source,The file name identifies post-16 activity info...
2,ns8_2015_benefits.dta,ns8_2015_benefits.dta,2015 follow-up source,"The file name identifies the 2015 follow-up, w..."
3,ns8_2015_benefits_unfolding_brackets.dta,ns8_2015_benefits_unfolding_brackets.dta,2015 follow-up source,"The file name identifies the 2015 follow-up, w..."
4,ns8_2015_children.dta,ns8_2015_children.dta,2015 follow-up source,"The file name identifies the 2015 follow-up, w..."


In [8]:
# 8: Candidate source file resolver

def resolve_candidate_source_file(file_name):
    matches = candidate_source_files.loc[candidate_source_files['File name'].eq(file_name), 'Relative path']
    if len(matches) != 1:
        raise ValueError(f'Expected one candidate source file named {file_name!r}; found {len(matches)}.')
    file_path = source_directory / matches.iloc[0]
    if not file_path.is_file():
        raise FileNotFoundError(file_path)
    return file_path

In [9]:
# 9: Wave 1 young person file structure

wave_1_young_person_file = resolve_candidate_source_file('wave_one_lsype_young_person_2020.dta')
wave_1_young_person_data = pd.read_stata(wave_1_young_person_file, convert_categoricals=False)
print(f'File name: {wave_1_young_person_file.name}')
print(f'Rows: {wave_1_young_person_data.shape[0]:,}')
print(f'Variables: {wave_1_young_person_data.shape[1]:,}')

File name: wave_one_lsype_young_person_2020.dta
Rows: 15,760
Variables: 350


In [10]:
# 10: Wave 1 young person identifier checks

wave_1_identifier_checks = summarise_nsid(wave_1_young_person_data)
display(wave_1_identifier_checks)

,Check,Result
0,NSID variable present,True
1,Number of rows,15760
2,Unique NSID values,15760
3,Missing NSID values,0
4,Duplicate NSID rows,0


In [11]:
# 11: Participant identifier data types

participant_roster_nsid_type = participant_roster['NSID'].dtype
wave_1_nsid_type = wave_1_young_person_data['NSID'].dtype
print(f'Participant roster NSID type: {participant_roster_nsid_type}')
print(f'Wave 1 young person NSID type: {wave_1_nsid_type}')
assert participant_roster_nsid_type == wave_1_nsid_type, 'NSID data types do not match.'
assert participant_roster['NSID'].map(lambda value: isinstance(value,
    str)).all(), 'Participant roster NSID contains non-string values.'
assert wave_1_young_person_data['NSID'].map(lambda value: isinstance(value,
    str)).all(), 'Wave 1 NSID contains non-string values.'

Participant roster NSID type: str
Wave 1 young person NSID type: str


In [12]:
# 12: Wave 1 young person participant coverage

wave_1_nsids = set(wave_1_young_person_data['NSID'])
wave_1_absent_nsids = participant_roster_nsids - wave_1_nsids
wave_1_young_person_coverage = pd.DataFrame({'Check': ['Stage 2 roster participants',
    'Present in Wave 1 young person file', 'Absent from Wave 1 young person file', 'Coverage percentage'], 'Result': [len(participant_roster_nsids),
    len(participant_roster_nsids & wave_1_nsids), len(wave_1_absent_nsids), round(len(participant_roster_nsids & wave_1_nsids) / len(participant_roster_nsids) * 100,
    2)]})
display(wave_1_young_person_coverage)

,Check,Result
0,Stage 2 roster participants,9767.00
1,Present in Wave 1 young person file,9524.00
2,Absent from Wave 1 young person file,243.00
3,Coverage percentage,97.51


In [13]:
# 13: Wave 1 family background file structure

wave_1_family_background_file = resolve_candidate_source_file('wave_one_lsype_family_background_2020.dta')
wave_1_family_background_data = pd.read_stata(wave_1_family_background_file, convert_categoricals=False)
print(f'File name: {wave_1_family_background_file.name}')
print(f'Rows: {wave_1_family_background_data.shape[0]:,}')
print(f'Variables: {wave_1_family_background_data.shape[1]:,}')

File name: wave_one_lsype_family_background_2020.dta
Rows: 15,760
Variables: 381


In [14]:
# 14: Wave 1 family background identifier checks

wave_1_family_identifier_checks = summarise_nsid(wave_1_family_background_data)
display(wave_1_family_identifier_checks)

,Check,Result
0,NSID variable present,True
1,Number of rows,15760
2,Unique NSID values,15760
3,Missing NSID values,0
4,Duplicate NSID rows,0


In [15]:
# 15: Wave 1 family background participant coverage

wave_1_family_nsids = set(wave_1_family_background_data['NSID'])
wave_1_family_absent_nsids = participant_roster_nsids - wave_1_family_nsids
wave_1_family_coverage = pd.DataFrame({'Check': ['Stage 2 roster participants',
    'Present in Wave 1 family background file', 'Absent from Wave 1 family background file', 'Coverage percentage'], 'Result': [len(participant_roster_nsids),
    len(participant_roster_nsids & wave_1_family_nsids), len(wave_1_family_absent_nsids), round(len(participant_roster_nsids & wave_1_family_nsids) / len(participant_roster_nsids) * 100,
    2)]})
display(wave_1_family_coverage)

,Check,Result
0,Stage 2 roster participants,9767.00
1,Present in Wave 1 family background file,9524.00
2,Absent from Wave 1 family background file,243.00
3,Coverage percentage,97.51


In [16]:
# 16: Wave 1 parental attitudes file structure

wave_1_parental_attitudes_file = resolve_candidate_source_file('wave_one_lsype_parental_attitudes_file_16_05_08.dta')
wave_1_parental_attitudes_data = pd.read_stata(wave_1_parental_attitudes_file, convert_categoricals=False)
print(f'File name: {wave_1_parental_attitudes_file.name}')
print(f'Rows: {wave_1_parental_attitudes_data.shape[0]:,}')
print(f'Variables: {wave_1_parental_attitudes_data.shape[1]:,}')

File name: wave_one_lsype_parental_attitudes_file_16_05_08.dta
Rows: 15,760
Variables: 303


In [17]:
# 17: Wave 1 parental attitudes identifier checks

wave_1_parental_identifier_checks = summarise_nsid(wave_1_parental_attitudes_data)
display(wave_1_parental_identifier_checks)

,Check,Result
0,NSID variable present,True
1,Number of rows,15760
2,Unique NSID values,15760
3,Missing NSID values,0
4,Duplicate NSID rows,0


In [18]:
# 18: Wave 1 parental attitudes participant coverage

wave_1_parental_nsids = set(wave_1_parental_attitudes_data['NSID'])
wave_1_parental_absent_nsids = participant_roster_nsids - wave_1_parental_nsids
wave_1_parental_coverage = pd.DataFrame({'Check': ['Stage 2 roster participants',
    'Present in Wave 1 parental attitudes file', 'Absent from Wave 1 parental attitudes file', 'Coverage percentage'], 'Result': [len(participant_roster_nsids),
    len(participant_roster_nsids & wave_1_parental_nsids), len(wave_1_parental_absent_nsids), round(len(participant_roster_nsids & wave_1_parental_nsids) / len(participant_roster_nsids) * 100,
    2)]})
display(wave_1_parental_coverage)

,Check,Result
0,Stage 2 roster participants,9767.00
1,Present in Wave 1 parental attitudes file,9524.00
2,Absent from Wave 1 parental attitudes file,243.00
3,Coverage percentage,97.51


In [19]:
# 19: Wave 1 participant coverage consistency

wave_1_coverage_consistency = pd.DataFrame({'Check': ['Absent from young person file',
    'Absent from family background file', 'Absent from parental attitudes file', 'Young person and family background absence sets match', 'Young person and parental attitudes absence sets match', 'All three Wave 1 absence sets match'], 'Result': [len(wave_1_absent_nsids),
    len(wave_1_family_absent_nsids), len(wave_1_parental_absent_nsids), wave_1_absent_nsids == wave_1_family_absent_nsids, wave_1_absent_nsids == wave_1_parental_absent_nsids, wave_1_absent_nsids == wave_1_family_absent_nsids == wave_1_parental_absent_nsids]})
display(wave_1_coverage_consistency)

,Check,Result
0,Absent from young person file,243
1,Absent from family background file,243
...,...,...
4,Young person and parental attitudes absence se...,True
5,All three Wave 1 absence sets match,True


In [20]:
# 20: Wave 4 coverage among participants absent from Wave 1

wave_4_young_person_file = resolve_candidate_source_file('wave_four_lsype_young_person_2020.dta')
wave_4_identifiers = pd.read_stata(wave_4_young_person_file, columns=['NSID'], convert_categoricals=False)
wave_4_identifier_nsids = set(wave_4_identifiers['NSID'])
wave_1_absence_wave_4_coverage = pd.DataFrame({'Check': ['Participants absent from Wave 1 young person file',
    'Present in Wave 4 young person file', 'Absent from Wave 4 young person file'], 'Result': [len(wave_1_absent_nsids),
    len(wave_1_absent_nsids & wave_4_identifier_nsids), len(wave_1_absent_nsids - wave_4_identifier_nsids)]})
display(wave_1_absence_wave_4_coverage)

,Check,Result
0,Participants absent from Wave 1 young person file,243
1,Present in Wave 4 young person file,243
2,Absent from Wave 4 young person file,0


In [21]:
# 21: Wave 4 sample-status metadata

with pd.io.stata.StataReader(wave_4_young_person_file, convert_categoricals=False) as reader:
    wave_4_variable_labels = reader.variable_labels()
    wave_4_value_labels = reader.value_labels()
sample_status_search_terms = ['boost', 'sample', 'cohort']
wave_4_sample_status_metadata = pd.DataFrame([{'Variable': variable, 'Variable label': label,
    'Value labels available': variable in wave_4_value_labels} for variable, label in wave_4_variable_labels.items() if any((term in f'{variable} {label}'.lower() for term in sample_status_search_terms))])
if wave_4_sample_status_metadata.empty:
    raise ValueError('No Wave 4 sample-status metadata matched the review terms.')
display_limited(wave_4_sample_status_metadata)

,Variable,Variable label,Value labels available
0,NSID,NSID - cohort member identifier,False
1,W4Weight_MAIN_BOOST,Weight: Cross-sectional weight for main and bo...,True
2,W4Boost,Admin: Whether this is a boost respondent,True
3,W4WhoPreMP0c,Admin: Who else present during MP section: Sam...,True
4,w4ethnic2YP,DV: Young person's ethnic group (detailed) Sou...,True


In [22]:
# 22: Wave 4 boost-sample status

wave_4_boost_variable = 'W4Boost'
assert wave_4_boost_variable in wave_4_variable_labels
wave_4_boost_codes = pd.read_stata(wave_4_young_person_file, columns=['NSID', wave_4_boost_variable],
    convert_categoricals=False)
wave_4_boost_labels = pd.read_stata(wave_4_young_person_file, columns=['NSID', wave_4_boost_variable],
    convert_categoricals=True)
assert wave_4_boost_codes['NSID'].equals(wave_4_boost_labels['NSID'])
wave_4_boost_review = pd.DataFrame({'NSID': wave_4_boost_codes['NSID'],
    'Value code': wave_4_boost_codes[wave_4_boost_variable], 'Value label': wave_4_boost_labels[wave_4_boost_variable].astype('string')})
wave_1_absent_boost_review = wave_4_boost_review.loc[wave_4_boost_review['NSID'].isin(wave_1_absent_nsids)].copy()
assert len(wave_1_absent_boost_review) == len(wave_1_absent_nsids & set(wave_4_boost_review['NSID']))
print('Variable label:', wave_4_variable_labels[wave_4_boost_variable])
display(wave_1_absent_boost_review.groupby(['Value code', 'Value label'],
    dropna=False).size().rename('Participants').reset_index())

Variable label: Admin: Whether this is a boost respondent


,Value code,Value label,Participants
0,1,Yes,243


### Wave 4 boost-sample confirmation

The `W4Boost` variable is labelled `Admin: Whether this is a boost respondent`. All 243 participants who were absent from the Wave 1 young person file and present in the Wave 4 young person file had the value `1`, labelled `Yes`.

These participants were therefore identified as Wave 4 boost respondents. They remain in the Stage 2 participant roster, although predictors drawn from earlier waves are structurally unavailable for this group.

In [23]:
# 23: Wave 2 young person file structure

wave_2_young_person_file = resolve_candidate_source_file('wave_two_lsype_young_person_2020.dta')
wave_2_young_person_data = pd.read_stata(wave_2_young_person_file, convert_categoricals=False)
print(f'File name: {wave_2_young_person_file.name}')
print(f'Rows: {wave_2_young_person_data.shape[0]:,}')
print(f'Variables: {wave_2_young_person_data.shape[1]:,}')

File name: wave_two_lsype_young_person_2020.dta
Rows: 13,530
Variables: 564


In [24]:
# 24: Wave 2 young person identifier checks

wave_2_identifier_checks = summarise_nsid(wave_2_young_person_data)
display(wave_2_identifier_checks)

,Check,Result
0,NSID variable present,True
1,Number of rows,13530
2,Unique NSID values,13530
3,Missing NSID values,0
4,Duplicate NSID rows,0


In [25]:
# 25: Wave 2 young person participant coverage

wave_2_nsids = set(wave_2_young_person_data['NSID'])
wave_2_absent_nsids = participant_roster_nsids - wave_2_nsids
wave_2_young_person_coverage = pd.DataFrame({'Check': ['Stage 2 roster participants',
    'Present in Wave 2 young person file', 'Absent from Wave 2 young person file', 'Coverage percentage'], 'Result': [len(participant_roster_nsids),
    len(participant_roster_nsids & wave_2_nsids), len(wave_2_absent_nsids), round(len(participant_roster_nsids & wave_2_nsids) / len(participant_roster_nsids) * 100,
    2)]})
display(wave_2_young_person_coverage)

,Check,Result
0,Stage 2 roster participants,9767.00
1,Present in Wave 2 young person file,9521.00
2,Absent from Wave 2 young person file,246.00
3,Coverage percentage,97.48


In [26]:
# 26: Wave 2 family background file structure

wave_2_family_background_file = resolve_candidate_source_file('wave_two_lsype_family_background_2020.dta')
wave_2_family_background_data = pd.read_stata(wave_2_family_background_file, convert_categoricals=False)
print(f'File name: {wave_2_family_background_file.name}')
print(f'Rows: {wave_2_family_background_data.shape[0]:,}')
print(f'Variables: {wave_2_family_background_data.shape[1]:,}')

File name: wave_two_lsype_family_background_2020.dta
Rows: 13,530
Variables: 1,030


In [27]:
# 27: Wave 2 family background identifier checks

wave_2_family_identifier_checks = summarise_nsid(wave_2_family_background_data)
display(wave_2_family_identifier_checks)

,Check,Result
0,NSID variable present,True
1,Number of rows,13530
2,Unique NSID values,13530
3,Missing NSID values,0
4,Duplicate NSID rows,0


In [28]:
# 28: Wave 2 family background participant coverage

wave_2_family_nsids = set(wave_2_family_background_data['NSID'])
wave_2_family_absent_nsids = participant_roster_nsids - wave_2_family_nsids
wave_2_family_coverage = pd.DataFrame({'Check': ['Stage 2 roster participants',
    'Present in Wave 2 family background file', 'Absent from Wave 2 family background file', 'Coverage percentage'], 'Result': [len(participant_roster_nsids),
    len(participant_roster_nsids & wave_2_family_nsids), len(wave_2_family_absent_nsids), round(len(participant_roster_nsids & wave_2_family_nsids) / len(participant_roster_nsids) * 100,
    2)]})
display(wave_2_family_coverage)

,Check,Result
0,Stage 2 roster participants,9767.00
1,Present in Wave 2 family background file,9521.00
2,Absent from Wave 2 family background file,246.00
3,Coverage percentage,97.48


In [29]:
# 29: Wave 2 parental attitudes file structure

wave_2_parental_attitudes_file = resolve_candidate_source_file('wave_two_lsype_parental_attitudes_file_16_06_08.dta')
wave_2_parental_attitudes_data = pd.read_stata(wave_2_parental_attitudes_file, convert_categoricals=False)
print(f'File name: {wave_2_parental_attitudes_file.name}')
print(f'Rows: {wave_2_parental_attitudes_data.shape[0]:,}')
print(f'Variables: {wave_2_parental_attitudes_data.shape[1]:,}')

File name: wave_two_lsype_parental_attitudes_file_16_06_08.dta
Rows: 13,530
Variables: 78


In [30]:
# 30: Wave 2 parental attitudes identifier checks

wave_2_parental_identifier_checks = summarise_nsid(wave_2_parental_attitudes_data)
display(wave_2_parental_identifier_checks)

,Check,Result
0,NSID variable present,True
1,Number of rows,13530
2,Unique NSID values,13530
3,Missing NSID values,0
4,Duplicate NSID rows,0


In [31]:
# 31: Wave 2 parental attitudes participant coverage

wave_2_parental_nsids = set(wave_2_parental_attitudes_data['NSID'])
wave_2_parental_absent_nsids = participant_roster_nsids - wave_2_parental_nsids
wave_2_parental_coverage = pd.DataFrame({'Check': ['Stage 2 roster participants',
    'Present in Wave 2 parental attitudes file', 'Absent from Wave 2 parental attitudes file', 'Coverage percentage'], 'Result': [len(participant_roster_nsids),
    len(participant_roster_nsids & wave_2_parental_nsids), len(wave_2_parental_absent_nsids), round(len(participant_roster_nsids & wave_2_parental_nsids) / len(participant_roster_nsids) * 100,
    2)]})
display(wave_2_parental_coverage)

,Check,Result
0,Stage 2 roster participants,9767.00
1,Present in Wave 2 parental attitudes file,9521.00
2,Absent from Wave 2 parental attitudes file,246.00
3,Coverage percentage,97.48


In [32]:
# 32: Wave 2 participant coverage consistency

wave_2_coverage_consistency = pd.DataFrame({'Check': ['Absent from young person file',
    'Absent from family background file', 'Absent from parental attitudes file', 'Young person and family background absence sets match', 'Young person and parental attitudes absence sets match', 'All three Wave 2 absence sets match'], 'Result': [len(wave_2_absent_nsids),
    len(wave_2_family_absent_nsids), len(wave_2_parental_absent_nsids), wave_2_absent_nsids == wave_2_family_absent_nsids, wave_2_absent_nsids == wave_2_parental_absent_nsids, wave_2_absent_nsids == wave_2_family_absent_nsids == wave_2_parental_absent_nsids]})
display(wave_2_coverage_consistency)

,Check,Result
0,Absent from young person file,246
1,Absent from family background file,246
...,...,...
4,Young person and parental attitudes absence se...,True
5,All three Wave 2 absence sets match,True


In [33]:
# 33: Wave 1 and Wave 2 coverage comparison

wave_1_wave_2_coverage_comparison = pd.DataFrame({'Check': ['Absent from Wave 1 young person file',
    'Absent from Wave 2 young person file', 'Absent from both waves', 'Present in Wave 1 but absent from Wave 2', 'Absent from Wave 1 but present in Wave 2'], 'Result': [len(wave_1_absent_nsids),
    len(wave_2_absent_nsids), len(wave_1_absent_nsids & wave_2_absent_nsids), len(wave_2_absent_nsids - wave_1_absent_nsids), len(wave_1_absent_nsids - wave_2_absent_nsids)]})
display(wave_1_wave_2_coverage_comparison)

,Check,Result
0,Absent from Wave 1 young person file,243
1,Absent from Wave 2 young person file,246
2,Absent from both waves,243
3,Present in Wave 1 but absent from Wave 2,3
4,Absent from Wave 1 but present in Wave 2,0


In [34]:
# 34: Waves 1 and 2 history file structure

wave_1_2_history_file = resolve_candidate_source_file('lsype_history_file_wave_one_and_wave_two_june_2008.dta')
wave_1_2_history_data = pd.read_stata(wave_1_2_history_file, convert_categoricals=False)
print(f'File name: {wave_1_2_history_file.name}')
print(f'Rows: {wave_1_2_history_data.shape[0]:,}')
print(f'Variables: {wave_1_2_history_data.shape[1]:,}')

File name: lsype_history_file_wave_one_and_wave_two_june_2008.dta
Rows: 15,760
Variables: 76


In [35]:
# 35: Waves 1 and 2 history identifier checks

wave_1_2_history_identifier_checks = summarise_nsid(wave_1_2_history_data)
display(wave_1_2_history_identifier_checks)

,Check,Result
0,NSID variable present,True
1,Number of rows,15760
2,Unique NSID values,15760
3,Missing NSID values,0
4,Duplicate NSID rows,0


In [36]:
# 36: Waves 1 and 2 history participant coverage

wave_1_2_history_nsids = set(wave_1_2_history_data['NSID'])
wave_1_2_history_absent_nsids = participant_roster_nsids - wave_1_2_history_nsids
wave_1_2_history_coverage = pd.DataFrame({'Check': ['Stage 2 roster participants',
    'Present in Waves 1 and 2 history file', 'Absent from Waves 1 and 2 history file', 'Coverage percentage', 'Coverage set matches Wave 1 young person file', 'Coverage set matches Wave 2 young person file'], 'Result': [len(participant_roster_nsids),
    len(participant_roster_nsids & wave_1_2_history_nsids), len(wave_1_2_history_absent_nsids), round(len(participant_roster_nsids & wave_1_2_history_nsids) / len(participant_roster_nsids) * 100,
    2), participant_roster_nsids & wave_1_2_history_nsids == participant_roster_nsids & wave_1_nsids, participant_roster_nsids & wave_1_2_history_nsids == participant_roster_nsids & wave_2_nsids]})
display(wave_1_2_history_coverage)

,Check,Result
0,Stage 2 roster participants,9767
1,Present in Waves 1 and 2 history file,9524
...,...,...
4,Coverage set matches Wave 1 young person file,True
5,Coverage set matches Wave 2 young person file,False


In [37]:
# 37: Wave 3 young person file structure

wave_3_young_person_file = resolve_candidate_source_file('wave_three_lsype_young_person_2020.dta')
wave_3_young_person_data = pd.read_stata(wave_3_young_person_file, convert_categoricals=False)
print(f'File name: {wave_3_young_person_file.name}')
print(f'Rows: {wave_3_young_person_data.shape[0]:,}')
print(f'Variables: {wave_3_young_person_data.shape[1]:,}')

File name: wave_three_lsype_young_person_2020.dta
Rows: 12,430
Variables: 652


In [38]:
# 38: Wave 3 young person identifier checks

wave_3_young_person_identifier_checks = summarise_nsid(wave_3_young_person_data)
display(wave_3_young_person_identifier_checks)

,Check,Result
0,NSID variable present,True
1,Number of rows,12430
2,Unique NSID values,12430
3,Missing NSID values,0
4,Duplicate NSID rows,0


In [39]:
# 39: Wave 3 young person participant coverage

wave_3_nsids = set(wave_3_young_person_data['NSID'])
wave_3_absent_nsids = participant_roster_nsids - wave_3_nsids
wave_3_young_person_coverage = pd.DataFrame({'Check': ['Stage 2 roster participants',
    'Present in Wave 3 young person file', 'Absent from Wave 3 young person file', 'Coverage percentage'], 'Result': [len(participant_roster_nsids),
    len(participant_roster_nsids & wave_3_nsids), len(wave_3_absent_nsids), round(len(participant_roster_nsids & wave_3_nsids) / len(participant_roster_nsids) * 100,
    2)]})
display(wave_3_young_person_coverage)

,Check,Result
0,Stage 2 roster participants,9767.00
1,Present in Wave 3 young person file,9509.00
2,Absent from Wave 3 young person file,258.00
3,Coverage percentage,97.36


In [40]:
# 40: Wave 3 family background file structure

wave_3_family_background_file = resolve_candidate_source_file('wave_three_lsype_family_background_2020.dta')
wave_3_family_background_data = pd.read_stata(wave_3_family_background_file, convert_categoricals=False)
print(f'File name: {wave_3_family_background_file.name}')
print(f'Rows: {wave_3_family_background_data.shape[0]:,}')
print(f'Variables: {wave_3_family_background_data.shape[1]:,}')

File name: wave_three_lsype_family_background_2020.dta
Rows: 12,430
Variables: 161


In [41]:
# 41: Wave 3 family background identifier checks

wave_3_family_identifier_checks = summarise_nsid(wave_3_family_background_data)
display(wave_3_family_identifier_checks)

,Check,Result
0,NSID variable present,True
1,Number of rows,12430
2,Unique NSID values,12430
3,Missing NSID values,0
4,Duplicate NSID rows,0


In [42]:
# 42: Wave 3 family background participant coverage

wave_3_family_nsids = set(wave_3_family_background_data['NSID'])
wave_3_family_absent_nsids = participant_roster_nsids - wave_3_family_nsids
wave_3_family_coverage = pd.DataFrame({'Check': ['Stage 2 roster participants',
    'Present in Wave 3 family background file', 'Absent from Wave 3 family background file', 'Coverage percentage'], 'Result': [len(participant_roster_nsids),
    len(participant_roster_nsids & wave_3_family_nsids), len(wave_3_family_absent_nsids), round(len(participant_roster_nsids & wave_3_family_nsids) / len(participant_roster_nsids) * 100,
    2)]})
display(wave_3_family_coverage)

,Check,Result
0,Stage 2 roster participants,9767.00
1,Present in Wave 3 family background file,9509.00
2,Absent from Wave 3 family background file,258.00
3,Coverage percentage,97.36


In [43]:
# 43: Wave 3 parental attitudes file structure

wave_3_parental_attitudes_file = resolve_candidate_source_file('wave_three_lsype_parental_attitudes_file_16_06_08.dta')
wave_3_parental_attitudes_data = pd.read_stata(wave_3_parental_attitudes_file, convert_categoricals=False)
print(f'File name: {wave_3_parental_attitudes_file.name}')
print(f'Rows: {wave_3_parental_attitudes_data.shape[0]:,}')
print(f'Variables: {wave_3_parental_attitudes_data.shape[1]:,}')

File name: wave_three_lsype_parental_attitudes_file_16_06_08.dta
Rows: 12,430
Variables: 76


In [44]:
# 44: Wave 3 parental attitudes identifier checks

wave_3_parental_identifier_checks = summarise_nsid(wave_3_parental_attitudes_data)
display(wave_3_parental_identifier_checks)

,Check,Result
0,NSID variable present,True
1,Number of rows,12430
2,Unique NSID values,12430
3,Missing NSID values,0
4,Duplicate NSID rows,0


In [45]:
# 45: Wave 3 parental attitudes participant coverage

wave_3_parental_nsids = set(wave_3_parental_attitudes_data['NSID'])
wave_3_parental_absent_nsids = participant_roster_nsids - wave_3_parental_nsids
wave_3_parental_coverage = pd.DataFrame({'Check': ['Stage 2 roster participants',
    'Present in Wave 3 parental attitudes file', 'Absent from Wave 3 parental attitudes file', 'Coverage percentage'], 'Result': [len(participant_roster_nsids),
    len(participant_roster_nsids & wave_3_parental_nsids), len(wave_3_parental_absent_nsids), round(len(participant_roster_nsids & wave_3_parental_nsids) / len(participant_roster_nsids) * 100,
    2)]})
display(wave_3_parental_coverage)

,Check,Result
0,Stage 2 roster participants,9767.00
1,Present in Wave 3 parental attitudes file,9509.00
2,Absent from Wave 3 parental attitudes file,258.00
3,Coverage percentage,97.36


In [46]:
# 46: Wave 3 participant coverage consistency

wave_3_coverage_consistency = pd.DataFrame({'Check': ['Absent from young person file',
    'Absent from family background file', 'Absent from parental attitudes file', 'Young person and family background absence sets match', 'Young person and parental attitudes absence sets match', 'All three Wave 3 absence sets match'], 'Result': [len(wave_3_absent_nsids),
    len(wave_3_family_absent_nsids), len(wave_3_parental_absent_nsids), wave_3_absent_nsids == wave_3_family_absent_nsids, wave_3_absent_nsids == wave_3_parental_absent_nsids, wave_3_absent_nsids == wave_3_family_absent_nsids == wave_3_parental_absent_nsids]})
display(wave_3_coverage_consistency)

,Check,Result
0,Absent from young person file,258
1,Absent from family background file,258
...,...,...
4,Young person and parental attitudes absence se...,True
5,All three Wave 3 absence sets match,True


In [47]:
# 47: Wave 2 and Wave 3 coverage comparison

wave_2_wave_3_coverage_comparison = pd.DataFrame({'Check': ['Absent from Wave 2 young person file',
    'Absent from Wave 3 young person file', 'Absent from both waves', 'Present in Wave 2 but absent from Wave 3', 'Absent from Wave 2 but present in Wave 3'], 'Result': [len(wave_2_absent_nsids),
    len(wave_3_absent_nsids), len(wave_2_absent_nsids & wave_3_absent_nsids), len(wave_3_absent_nsids - wave_2_absent_nsids), len(wave_2_absent_nsids - wave_3_absent_nsids)]})
display(wave_2_wave_3_coverage_comparison)

,Check,Result
0,Absent from Wave 2 young person file,246
1,Absent from Wave 3 young person file,258
2,Absent from both waves,244
3,Present in Wave 2 but absent from Wave 3,14
4,Absent from Wave 2 but present in Wave 3,2


In [48]:
# 48: Wave 4 young person file structure

wave_4_young_person_data = pd.read_stata(wave_4_young_person_file, convert_categoricals=False)
print(f'File name: {wave_4_young_person_file.name}')
print(f'Rows: {wave_4_young_person_data.shape[0]:,}')
print(f'Variables: {wave_4_young_person_data.shape[1]:,}')

File name: wave_four_lsype_young_person_2020.dta
Rows: 11,791
Variables: 1,049


In [49]:
# 49: Wave 4 young person identifier checks

wave_4_young_person_identifier_checks = summarise_nsid(wave_4_young_person_data)
display(wave_4_young_person_identifier_checks)

,Check,Result
0,NSID variable present,True
1,Number of rows,11791
2,Unique NSID values,11791
3,Missing NSID values,0
4,Duplicate NSID rows,0


In [50]:
# 50: Wave 4 young person participant coverage

wave_4_nsids = set(wave_4_young_person_data['NSID'])
wave_4_absent_nsids = participant_roster_nsids - wave_4_nsids
wave_4_young_person_coverage = pd.DataFrame({'Check': ['Stage 2 roster participants',
    'Present in Wave 4 young person file', 'Absent from Wave 4 young person file', 'Coverage percentage'], 'Result': [len(participant_roster_nsids),
    len(participant_roster_nsids & wave_4_nsids), len(wave_4_absent_nsids), round(len(participant_roster_nsids & wave_4_nsids) / len(participant_roster_nsids) * 100,
    2)]})
display(wave_4_young_person_coverage)

,Check,Result
0,Stage 2 roster participants,9767.00
1,Present in Wave 4 young person file,9756.00
2,Absent from Wave 4 young person file,11.00
3,Coverage percentage,99.89


In [51]:
# 51: Wave 4 family background file structure

wave_4_family_background_file = resolve_candidate_source_file('wave_four_lsype_family_background_2020.dta')
wave_4_family_background_data = pd.read_stata(wave_4_family_background_file, convert_categoricals=False)
print(f'File name: {wave_4_family_background_file.name}')
print(f'Rows: {wave_4_family_background_data.shape[0]:,}')
print(f'Variables: {wave_4_family_background_data.shape[1]:,}')

File name: wave_four_lsype_family_background_2020.dta
Rows: 11,791
Variables: 422


In [52]:
# 52: Wave 4 family background identifier checks

wave_4_family_identifier_checks = summarise_nsid(wave_4_family_background_data)
display(wave_4_family_identifier_checks)

,Check,Result
0,NSID variable present,True
1,Number of rows,11791
2,Unique NSID values,11791
3,Missing NSID values,0
4,Duplicate NSID rows,0


In [53]:
# 53: Wave 4 family background participant coverage

wave_4_family_nsids = set(wave_4_family_background_data['NSID'])
wave_4_family_absent_nsids = participant_roster_nsids - wave_4_family_nsids
wave_4_family_coverage = pd.DataFrame({'Check': ['Stage 2 roster participants',
    'Present in Wave 4 family background file', 'Absent from Wave 4 family background file', 'Coverage percentage'], 'Result': [len(participant_roster_nsids),
    len(participant_roster_nsids & wave_4_family_nsids), len(wave_4_family_absent_nsids), round(len(participant_roster_nsids & wave_4_family_nsids) / len(participant_roster_nsids) * 100,
    2)]})
display(wave_4_family_coverage)

,Check,Result
0,Stage 2 roster participants,9767.00
1,Present in Wave 4 family background file,9756.00
2,Absent from Wave 4 family background file,11.00
3,Coverage percentage,99.89


In [54]:
# 54: Wave 4 parental attitudes file structure

wave_4_parental_attitudes_file = resolve_candidate_source_file('wave_four_lsype_parental_attitudes_june_2009.dta')
wave_4_parental_attitudes_data = pd.read_stata(wave_4_parental_attitudes_file, convert_categoricals=False)
print(f'File name: {wave_4_parental_attitudes_file.name}')
print(f'Rows: {wave_4_parental_attitudes_data.shape[0]:,}')
print(f'Variables: {wave_4_parental_attitudes_data.shape[1]:,}')

File name: wave_four_lsype_parental_attitudes_june_2009.dta
Rows: 11,791
Variables: 77


In [55]:
# 55: Wave 4 parental attitudes identifier checks

wave_4_parental_identifier_checks = summarise_nsid(wave_4_parental_attitudes_data)
display(wave_4_parental_identifier_checks)

,Check,Result
0,NSID variable present,True
1,Number of rows,11791
2,Unique NSID values,11791
3,Missing NSID values,0
4,Duplicate NSID rows,0


In [56]:
# 56: Wave 4 parental attitudes participant coverage

wave_4_parental_nsids = set(wave_4_parental_attitudes_data['NSID'])
wave_4_parental_absent_nsids = participant_roster_nsids - wave_4_parental_nsids
wave_4_parental_coverage = pd.DataFrame({'Check': ['Stage 2 roster participants',
    'Present in Wave 4 parental attitudes file', 'Absent from Wave 4 parental attitudes file', 'Coverage percentage'], 'Result': [len(participant_roster_nsids),
    len(participant_roster_nsids & wave_4_parental_nsids), len(wave_4_parental_absent_nsids), round(len(participant_roster_nsids & wave_4_parental_nsids) / len(participant_roster_nsids) * 100,
    2)]})
display(wave_4_parental_coverage)

,Check,Result
0,Stage 2 roster participants,9767.00
1,Present in Wave 4 parental attitudes file,9756.00
2,Absent from Wave 4 parental attitudes file,11.00
3,Coverage percentage,99.89


In [57]:
# 57: Wave 4 history file structure

wave_4_history_file = resolve_candidate_source_file('wave_four_lsype_history_2020.dta')
wave_4_history_data = pd.read_stata(wave_4_history_file, convert_categoricals=False)
print(f'File name: {wave_4_history_file.name}')
print(f'Rows: {wave_4_history_data.shape[0]:,}')
print(f'Variables: {wave_4_history_data.shape[1]:,}')

File name: wave_four_lsype_history_2020.dta
Rows: 11,791
Variables: 42


In [58]:
# 58: Wave 4 history identifier checks

wave_4_history_identifier_checks = summarise_nsid(wave_4_history_data)
display(wave_4_history_identifier_checks)

,Check,Result
0,NSID variable present,True
1,Number of rows,11791
2,Unique NSID values,11791
3,Missing NSID values,0
4,Duplicate NSID rows,0


In [59]:
# 59: Wave 4 history participant coverage

wave_4_history_nsids = set(wave_4_history_data['NSID'])
wave_4_history_absent_nsids = participant_roster_nsids - wave_4_history_nsids
wave_4_history_coverage = pd.DataFrame({'Check': ['Stage 2 roster participants', 'Present in Wave 4 history file',
    'Absent from Wave 4 history file', 'Coverage percentage'], 'Result': [len(participant_roster_nsids),
    len(participant_roster_nsids & wave_4_history_nsids), len(wave_4_history_absent_nsids), round(len(participant_roster_nsids & wave_4_history_nsids) / len(participant_roster_nsids) * 100,
    2)]})
display(wave_4_history_coverage)

,Check,Result
0,Stage 2 roster participants,9767.00
1,Present in Wave 4 history file,9756.00
2,Absent from Wave 4 history file,11.00
3,Coverage percentage,99.89


In [60]:
# 60: Wave 4 participant coverage consistency

wave_4_coverage_consistency = pd.DataFrame({'Check': ['Absent from young person file',
    'Absent from family background file', 'Absent from parental attitudes file', 'Absent from history file', 'Young person and family background absence sets match', 'Young person and parental attitudes absence sets match', 'Young person and history absence sets match', 'All four Wave 4 absence sets match', 'Present in Wave 3 but absent from Wave 4', 'Absent from Wave 3 but present in Wave 4'], 'Result': [len(wave_4_absent_nsids),
    len(wave_4_family_absent_nsids), len(wave_4_parental_absent_nsids), len(wave_4_history_absent_nsids), wave_4_absent_nsids == wave_4_family_absent_nsids, wave_4_absent_nsids == wave_4_parental_absent_nsids, wave_4_absent_nsids == wave_4_history_absent_nsids, wave_4_absent_nsids == wave_4_family_absent_nsids == wave_4_parental_absent_nsids == wave_4_history_absent_nsids, len(wave_4_absent_nsids - wave_3_absent_nsids), len(wave_3_absent_nsids - wave_4_absent_nsids)]})
display(wave_4_coverage_consistency)

,Check,Result
0,Absent from young person file,11
1,Absent from family background file,11
...,...,...
8,Present in Wave 3 but absent from Wave 4,11
9,Absent from Wave 3 but present in Wave 4,258


In [61]:
# 61: Reviewed source file register

def classify_reviewed_source_file(file_name):
    if file_name == 'lsype_history_file_wave_one_and_wave_two_june_2008.dta':
        return {'Wave': 'Waves 1–2', 'Source type': 'History', 'Classification basis': 'File name'}
    match = re.match('^wave_(one|two|three|four)_lsype_(young_person|family_background|parental_attitudes|history)',
        file_name, flags=re.IGNORECASE)
    if match is None:
        raise ValueError(f'Source classification failed for {file_name!r}.')
    wave_names = {'one': 'Wave 1', 'two': 'Wave 2', 'three': 'Wave 3', 'four': 'Wave 4'}
    source_types = {'young_person': 'Young person', 'family_background': 'Family background',
        'parental_attitudes': 'Parental attitudes', 'history': 'History'}
    return {'Wave': wave_names[match.group(1).lower()], 'Source type': source_types[match.group(2).lower()],
        'Classification basis': 'File name'}

def source_timing_rule(wave):
    if wave in {'Wave 1', 'Wave 2', 'Waves 1–2'}:
        return {'Timing status': 'Pre-transition source',
            'Review status': 'Variable-level coding, routing and reference-period review required.'}
    if wave == 'Wave 3':
        return {'Timing status': 'Near-transition source',
            'Review status': 'Use requires interview timing confirming January–August 2006 and item-level reference-period review.'}
    if wave == 'Wave 4':
        return {'Timing status': 'At or after transition',
            'Review status': 'Stable characteristics or retrospective pre-transition information only.'}
    raise ValueError(f'No timing rule was defined for {wave!r}.')
loaded_source_files = [(wave_1_young_person_file, wave_1_young_person_data), (wave_1_family_background_file,
    wave_1_family_background_data), (wave_1_parental_attitudes_file,
    wave_1_parental_attitudes_data), (wave_2_young_person_file,
    wave_2_young_person_data), (wave_2_family_background_file,
    wave_2_family_background_data), (wave_2_parental_attitudes_file,
    wave_2_parental_attitudes_data), (wave_1_2_history_file, wave_1_2_history_data), (wave_3_young_person_file,
    wave_3_young_person_data), (wave_3_family_background_file,
    wave_3_family_background_data), (wave_3_parental_attitudes_file,
    wave_3_parental_attitudes_data), (wave_4_young_person_file,
    wave_4_young_person_data), (wave_4_family_background_file,
    wave_4_family_background_data), (wave_4_parental_attitudes_file,
    wave_4_parental_attitudes_data), (wave_4_history_file, wave_4_history_data)]
loaded_source_file_names = {file_path.name for file_path, _ in loaded_source_files}
assert loaded_source_file_names == candidate_source_file_names

def create_reviewed_source_row(file_path, data):
    classification = classify_reviewed_source_file(file_path.name)
    timing_rule = source_timing_rule(classification['Wave'])
    nsid_present = 'NSID' in data.columns
    if nsid_present:
        missing_nsid = int(data['NSID'].isna().sum())
        duplicate_nsid = int(data['NSID'].duplicated().sum())
        unique_nsid = int(data['NSID'].nunique(dropna=True))
        source_nsids = set(data['NSID'].dropna())
        roster_present = len(participant_roster_nsids & source_nsids)
        roster_absent = len(participant_roster_nsids) - roster_present
        coverage_percentage = round(roster_present / len(participant_roster_nsids) * 100, 2)
    else:
        missing_nsid = pd.NA
        duplicate_nsid = pd.NA
        unique_nsid = pd.NA
        roster_present = pd.NA
        roster_absent = pd.NA
        coverage_percentage = pd.NA
    one_row_per_nsid = nsid_present and missing_nsid == 0 and (duplicate_nsid == 0)
    return {**classification, 'File name': file_path.name,
        'Relative path': str(file_path.relative_to(source_directory)), 'Rows': len(data), 'Variables': data.shape[1], 'NSID present': nsid_present, 'Unique NSID values': unique_nsid, 'Missing NSID values': missing_nsid, 'Duplicate NSID rows': duplicate_nsid, 'Data structure': 'One row per NSID' if one_row_per_nsid else 'Not one row per NSID', 'Roster participants present': roster_present, 'Roster participants absent': roster_absent, 'Coverage percentage': coverage_percentage, **timing_rule}
reviewed_source_register = pd.DataFrame([create_reviewed_source_row(file_path, data) for file_path,
    data in loaded_source_files])
expected_source_register_columns = ['Wave', 'Source type', 'Classification basis', 'File name', 'Relative path',
    'Rows', 'Variables', 'NSID present', 'Unique NSID values', 'Missing NSID values', 'Duplicate NSID rows', 'Data structure', 'Roster participants present', 'Roster participants absent', 'Coverage percentage', 'Timing status', 'Review status']
assert reviewed_source_register.columns.tolist() == expected_source_register_columns
assert not reviewed_source_register['File name'].duplicated().any()
display_limited(reviewed_source_register)

,Wave,Source type,Classification basis,File name,Relative path,Rows,Variables,NSID present,Unique NSID values,Missing NSID values,Duplicate NSID rows,Data structure,Roster participants present,Roster participants absent,Coverage percentage,Timing status,Review status
0,Wave 1,Young person,File name,wave_one_lsype_young_person_2020.dta,wave_one_lsype_young_person_2020.dta,15760,350,True,15760,0,0,One row per NSID,9524,243,97.51,Pre-transition source,"Variable-level coding, routing and reference-p..."
1,Wave 1,Family background,File name,wave_one_lsype_family_background_2020.dta,wave_one_lsype_family_background_2020.dta,15760,381,True,15760,0,0,One row per NSID,9524,243,97.51,Pre-transition source,"Variable-level coding, routing and reference-p..."
2,Wave 1,Parental attitudes,File name,wave_one_lsype_parental_attitudes_file_16_05_0...,wave_one_lsype_parental_attitudes_file_16_05_0...,15760,303,True,15760,0,0,One row per NSID,9524,243,97.51,Pre-transition source,"Variable-level coding, routing and reference-p..."
3,Wave 2,Young person,File name,wave_two_lsype_young_person_2020.dta,wave_two_lsype_young_person_2020.dta,13530,564,True,13530,0,0,One row per NSID,9521,246,97.48,Pre-transition source,"Variable-level coding, routing and reference-p..."
4,Wave 2,Family background,File name,wave_two_lsype_family_background_2020.dta,wave_two_lsype_family_background_2020.dta,13530,1030,True,13530,0,0,One row per NSID,9521,246,97.48,Pre-transition source,"Variable-level coding, routing and reference-p..."


In [62]:
# 62: Source file register export

source_register_file = stage_2_output_directory / 'stage_2_source_file_register.csv'
reviewed_source_register.to_csv(source_register_file, index=False)
print(f'Saved file: {source_register_file}')
print(f'File exists: {source_register_file.exists()}')
print(f'Rows saved: {len(reviewed_source_register):,}')

Saved file: data_derived\stage_2_predictor_construction\stage_2_source_file_register.csv
File exists: True
Rows saved: 14


## Participant roster and source file review

The Stage 2 participant roster was defined from the `NSID` column of the Stage 1 outcome file. It contained 9,767 unique participant identifiers, with no missing or duplicated values. Outcome codes and outcome labels were not loaded.

The source directory contained 38 Stata files. File-name rules identified 14 Wave 1–4 participant-level files for detailed predictor review. The remaining files were recorded separately as post-16 activity sources or later follow-up sources outside the eligible predictor period.

All 14 reviewed files contained `NSID`, had no missing or duplicated identifiers, and contained one row per participant. Coverage of the Stage 2 roster was 9,524 participants in Wave 1, 9,521 in Wave 2, 9,509 in Wave 3 and 9,756 in Wave 4. Coverage was consistent across the young-person, family-background and parental-attitudes files within each wave. The Waves 1–2 history file covered the same 9,524 roster participants as the Wave 1 files.

The 243 roster participants absent from the Wave 1 files were present in the Wave 4 young-person file and all had `W4Boost = 1`, labelled `Yes`. They were therefore identified as Wave 4 boost respondents. Eleven roster participants were absent from the Wave 4 source files.

Wave 1 and Wave 2 were classified as pre-transition sources. Wave 3 variables require confirmed interview timing within January–August 2006 and item-level reference-period review. Wave 4 variables are restricted to stable characteristics or retrospective information referring to the pre-transition period. These source-level rules define the scope of subsequent variable review and do not establish the eligibility of individual predictors.

# Part 2: Initial metadata and coding review

Wave 1 young-person metadata were used to establish the coding checks applied throughout Stage 2. The review covered variable labels, data types, value labels, negative codes, administrative fields and variables marked as derived or edited.


In [63]:
# 1: Wave 1 young person variable labels

from pandas.io.stata import StataReader
with StataReader(wave_1_young_person_file, convert_categoricals=False) as reader:
    wave_1_young_person_variable_labels = reader.variable_labels()
wave_1_young_person_metadata = pd.DataFrame({'Variable': wave_1_young_person_data.columns,
    'Variable label': [wave_1_young_person_variable_labels.get(variable,
    '') for variable in wave_1_young_person_data.columns], 'Data type': [str(wave_1_young_person_data[variable].dtype) for variable in wave_1_young_person_data.columns]})
print(f'Variables: {len(wave_1_young_person_metadata):,}')
print(f"Variables without labels: {wave_1_young_person_metadata['Variable label'].eq('').sum():,}")
wave_1_young_person_metadata.head(25)

Variables: 350
Variables without labels: 0


,Variable,Variable label,Data type
0,NSID,NSID - cohort member identifier,str
1,Designweight,Weight: Design weight,float64
...,...,...,...
23,W1sen1MP0k,MP: Nature of YP's special needs: Don't know,int16
24,W1statedMP,MP: Whether YP has ever been given statement o...,int8


In [64]:
# 2: Wave 1 young person variable-label prefix counts

wave_1_young_person_metadata['Label prefix'] = wave_1_young_person_metadata['Variable label'].str.split(':',
    n=1).str[0].str.strip()
wave_1_young_person_prefix_counts = wave_1_young_person_metadata['Label prefix'].value_counts(dropna=False).rename_axis('Label prefix').reset_index(name='Number of variables')
wave_1_young_person_prefix_counts

,Label prefix,Number of variables
0,YP,231
1,MP,75
...,...,...
7,Edited,2
8,NSID - cohort member identifier,1


In [65]:
# 3: Administrative, design and identifier variable review

wave_1_young_person_non_substantive_review = wave_1_young_person_metadata.loc[wave_1_young_person_metadata['Label prefix'].isin(['Admin',
    'Weight', 'Sampling', 'NSID - cohort member identifier']), ['Variable', 'Variable label', 'Label prefix',
    'Data type']].reset_index(drop=True)
print(f'Variables requiring administrative or design review: {len(wave_1_young_person_non_substantive_review):,}')
wave_1_young_person_non_substantive_review

Variables requiring administrative or design review: 24


,Variable,Variable label,Label prefix,Data type
0,NSID,NSID - cohort member identifier,NSID - cohort member identifier,str
1,Designweight,Weight: Design weight,Weight,float64
...,...,...,...,...
22,W1chpreYP0g,Admin: Who else present during YP interview: O...,Admin,int8
23,W1chpreYP0h,Admin: Who else present during YP interview: O...,Admin,int8


In [66]:
# 4: Administrative, design and identifier decisions

wave_1_young_person_initial_decisions = wave_1_young_person_non_substantive_review.copy()

def assign_initial_decision(variable):
    if variable == 'NSID':
        return pd.Series({'Decision': 'Retain as identifier only', 'Analytical role': 'Record linkage',
            'Decision reason': 'Cohort member identifier; not a substantive predictor', 'Leakage concern': 'No'})
    if variable in ['Designweight', 'W1FinWt']:
        return pd.Series({'Decision': 'Exclude from predictor set', 'Analytical role': 'Survey weighting support',
            'Decision reason': 'Survey weight rather than a participant characteristic', 'Leakage concern': 'No'})
    if variable in ['SampPSU', 'SampStratum']:
        return pd.Series({'Decision': 'Exclude from predictor set', 'Analytical role': 'Sample-design support',
            'Decision reason': 'Sampling structure variable rather than a participant characteristic', 'Leakage concern': 'No'})
    if variable == 'W1sexYP':
        return pd.Series({'Decision': 'Retain as predictor candidate', 'Analytical role': 'Demographic background',
            'Decision reason': 'Interviewer-coded sex of the young person; substantive pre-transition information', 'Leakage concern': 'No'})
    if variable in ['W1scomadiMP', 'W1scompinYP']:
        return pd.Series({'Decision': 'Exclude from predictor set', 'Analytical role': 'Survey administration',
            'Decision reason': 'Records acceptance of a self-completion section rather than a substantive characteristic', 'Leakage concern': 'No'})
    return pd.Series({'Decision': 'Exclude from predictor set', 'Analytical role': 'Interview context',
        'Decision reason': 'Records who was present during the interview or self-completion section', 'Leakage concern': 'No'})
wave_1_young_person_initial_decisions = pd.concat([wave_1_young_person_initial_decisions,
    wave_1_young_person_initial_decisions['Variable'].apply(assign_initial_decision)], axis=1)
wave_1_young_person_initial_decisions[['Variable', 'Variable label', 'Decision', 'Analytical role', 'Decision reason',
    'Leakage concern']]

,Variable,Variable label,Decision,Analytical role,Decision reason,Leakage concern
0,NSID,NSID - cohort member identifier,Retain as identifier only,Record linkage,Cohort member identifier; not a substantive pr...,No
1,Designweight,Weight: Design weight,Exclude from predictor set,Survey weighting support,Survey weight rather than a participant charac...,No
...,...,...,...,...,...,...
22,W1chpreYP0g,Admin: Who else present during YP interview: O...,Exclude from predictor set,Interview context,Records who was present during the interview o...,No
23,W1chpreYP0h,Admin: Who else present during YP interview: O...,Exclude from predictor set,Interview context,Records who was present during the interview o...,No


In [67]:
# 5: Review of derived (DV) and edited variables

wave_1_young_person_derived_edited_review = wave_1_young_person_metadata.loc[wave_1_young_person_metadata['Label prefix'].isin(['DV',
    'Edited']), ['Variable', 'Variable label', 'Label prefix', 'Data type']].reset_index(drop=True)
print(f'Derived and edited variables requiring review: {len(wave_1_young_person_derived_edited_review):,}')
wave_1_young_person_derived_edited_review

Derived and edited variables requiring review: 14


,Variable,Variable label,Label prefix,Data type
0,IndSchool,DV: Whether YP was at an independent or mainta...,DV,int8
1,DobyearYP,Edited: Year of birth of YP,Edited,int16
...,...,...,...,...
12,W1ethgrpYP,DV: Young person's ethnic group (grouped),DV,int16
13,W1ethnic2YP,DV: Young person's ethnic group (detailed),DV,int16


In [68]:
# 6: Full labels for variables marked DV or Edited

with pd.option_context('display.max_colwidth', None, 'display.width', 240):
    print(wave_1_young_person_derived_edited_review[['Variable', 'Variable label', 'Label prefix',
        'Data type']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))

   Variable                                                              Variable label Label prefix Data type
  IndSchool DV: Whether YP was at an independent or maintained school at sampling stage           DV      int8
  DobyearYP                                                 Edited: Year of birth of YP       Edited     int16
        ...                                                                         ...          ...       ...
 W1ethgrpYP                                   DV: Young person's ethnic group (grouped)           DV     int16
W1ethnic2YP                                  DV: Young person's ethnic group (detailed)           DV     int16


In [69]:
# 7: Code information for variables marked DV or Edited

from numbers import Real
variables_marked_dv_or_edited = wave_1_young_person_derived_edited_review['Variable'].tolist()
with StataReader(wave_1_young_person_file, convert_categoricals=False) as reader:
    wave_1_value_labels = reader.value_labels()
code_information_records = []
for variable in variables_marked_dv_or_edited:
    value_labels = wave_1_value_labels.get(variable, {})
    observed_values = wave_1_young_person_data[variable].dropna().unique().tolist()
    observed_negative_codes = sorted((value for value in observed_values if isinstance(value, Real) and value < 0))
    labelled_negative_codes = {code: label for code, label in value_labels.items() if isinstance(code,
        Real) and code < 0}
    code_information_records.append({'Variable': variable, 'Number of labelled codes': len(value_labels),
        'Number of observed values': len(observed_values), 'Observed negative codes': observed_negative_codes, 'Labelled negative codes': labelled_negative_codes})
wave_1_dv_edited_code_information = pd.DataFrame(code_information_records)
with pd.option_context('display.max_colwidth', None, 'display.width', 240):
    print(wave_1_dv_edited_code_information.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

   Variable  Number of labelled codes  Number of observed values Observed negative codes                                                                                                           Labelled negative codes
  IndSchool                         0                          2                      []                                                                                                                                {}
  DobyearYP                         1                          8                   [-94]                                                                                                 {-94: 'Insufficient information'}
        ...                       ...                        ...                     ...                                                                                                                               ...
 W1ethgrpYP                        13                         10             [-999, -92] {-999: 'Missing - household data lo

In [70]:
# 8: Observed ranges and negative-code frequencies

observed_code_summary_records = []
for variable in variables_marked_dv_or_edited:
    variable_values = wave_1_young_person_data[variable]
    negative_code_counts = variable_values.loc[variable_values < 0].value_counts().sort_index()
    non_negative_values = variable_values.loc[variable_values >= 0]
    negative_code_summary = ', '.join((f'{int(code)}: {int(count):,}' for code,
        count in negative_code_counts.items()))
    observed_code_summary_records.append({'Variable': variable, 'Source rows': len(variable_values),
        'Negative-coded rows': int((variable_values < 0).sum()), 'Negative-coded percentage': round((variable_values < 0).mean() * 100,
        2), 'Negative-code counts': negative_code_summary if negative_code_summary else 'None', 'Non-negative unique values': int(non_negative_values.nunique()), 'Non-negative minimum': non_negative_values.min() if not non_negative_values.empty else None, 'Non-negative maximum': non_negative_values.max() if not non_negative_values.empty else None})
wave_1_documented_variable_code_summary = pd.DataFrame(observed_code_summary_records)
with pd.option_context('display.max_colwidth', None, 'display.width', 260):
    print(wave_1_documented_variable_code_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

   Variable  Source rows  Negative-coded rows  Negative-coded percentage Negative-code counts  Non-negative unique values  Non-negative minimum  Non-negative maximum
  IndSchool        15760                    0                       0.00                 None                           2                     0                     1
  DobyearYP        15760                    4                       0.03               -94: 4                           7                  1985                  1993
        ...          ...                  ...                        ...                  ...                         ...                   ...                   ...
 W1ethgrpYP        15760                   26                       0.16    -999: 10, -92: 16                           8                     1                     8
W1ethnic2YP        15760                   26                       0.16    -999: 10, -92: 16                          16                     1                    16


In [71]:
# 9: Valid response labels for selected variables

selected_categorical_variables = ['IndSchool', 'W1stschHS', 'W1bulrc', 'W1pbulrc', 'W1disabYP', 'W1ethgrpYP',
    'W1ethnic2YP']
valid_response_label_records = []
for variable in selected_categorical_variables:
    value_labels = wave_1_value_labels.get(variable, {})
    non_negative_labels = {code: label for code, label in value_labels.items() if isinstance(code,
        Real) and code >= 0}
    observed_non_negative_values = sorted(wave_1_young_person_data.loc[wave_1_young_person_data[variable] >= 0,
        variable].dropna().unique().tolist())
    valid_response_label_records.append({'Variable': variable, 'Observed valid values': observed_non_negative_values,
        'Valid value labels': non_negative_labels if non_negative_labels else 'No value labels recorded'})
wave_1_selected_valid_response_labels = pd.DataFrame(valid_response_label_records)
with pd.option_context('display.max_colwidth', None, 'display.width', 260):
    print(wave_1_selected_valid_response_labels.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

   Variable                                   Observed valid values                                                                                                                                                                                                                                                                                                                                                                                                                         Valid value labels
  IndSchool                                                  [0, 1]                                                                                                                                                                                                                                                                                                                                                                                                                   No value labels recorded
  W1stschH

In [72]:
# 10: Category frequencies in the Stage 2 participant sample

wave_1_selected_category_data = participant_roster[['NSID']].merge(wave_1_young_person_data[['NSID'] + selected_categorical_variables],
    on='NSID', how='left', validate='one_to_one')
category_frequency_records = []
for variable in selected_categorical_variables:
    value_labels = wave_1_value_labels.get(variable, {})
    counts = wave_1_selected_category_data[variable].value_counts(dropna=False).sort_index()
    for value, count in counts.items():
        if pd.isna(value):
            response_label = 'Not present in Wave 1'
            displayed_value = 'Not present in Wave 1'
        else:
            displayed_value = int(value)
            response_label = value_labels.get(value, 'No embedded value label')
        category_frequency_records.append({'Variable': variable, 'Value': displayed_value,
            'Response label': response_label, 'Count': int(count), 'Percentage of Stage 2 sample': round(count / len(wave_1_selected_category_data) * 100,
            2)})
wave_1_selected_category_frequencies = pd.DataFrame(category_frequency_records)
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None, 'display.width', 260):
    print(wave_1_selected_category_frequencies.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

   Variable                 Value              Response label  Count  Percentage of Stage 2 sample
  IndSchool                     0     No embedded value label   9157                         93.75
  IndSchool                     1     No embedded value label    367                          3.76
        ...                   ...                         ...    ...                           ...
W1ethnic2YP                    16 Any other ethnic background     72                          0.74
W1ethnic2YP Not present in Wave 1       Not present in Wave 1    243                          2.49


## Metadata and coding findings

Variable names and observed values were not treated as sufficient documentation. Identifiers, weights and survey-operation fields were excluded from substantive consideration. Negative codes were interpreted from variable-specific labels because their meanings differed across measures. Derived and edited variables were compared with source items before any representation was retained.

These checks defined the fields and coding rules used in the master variable register.


# Part 3: Variable register and pre-domain review

Variable metadata from the reviewed source files were combined before domain-level decisions. The register records source file, wave, respondent, variable name, label, data type, timing and review status.


In [73]:
# 1: Source-variable inventory

from pathlib import Path
from pandas.io.stata import StataReader
import pandas as pd
source_register_file = stage_2_output_directory / 'stage_2_source_file_register.csv'
stage_2_source_file_register = pd.read_csv(source_register_file)
assert len(stage_2_source_file_register) == 14
assert stage_2_source_file_register.columns.tolist() == expected_source_register_columns

def resolve_registered_source_path(source_row):
    source_path = source_directory / str(source_row['File name'])
    if not source_path.is_file():
        raise FileNotFoundError(f'Registered source file not found: {source_path}')
    return source_path
master_variable_records = []
for source_order, (_, source_row) in enumerate(stage_2_source_file_register.iterrows(), start=1):
    source_path = resolve_registered_source_path(source_row)
    with StataReader(source_path, convert_categoricals=False) as reader:
        variable_labels = reader.variable_labels()
        first_row = reader.read(nrows=1)
    for variable_position, variable in enumerate(first_row.columns, start=1):
        master_variable_records.append({'Source order': source_order, 'Source file': source_path.stem,
            'Source path': str(source_path), 'Variable position': variable_position, 'Variable': variable, 'Variable label': variable_labels.get(variable,
            ''), 'Data type': str(first_row[variable].dtype)})
stage_2_master_variable_register = pd.DataFrame(master_variable_records)
master_register_output_path = stage_2_output_directory / 'stage_2_master_variable_register.csv'
stage_2_master_variable_register.to_csv(master_register_output_path, index=False)
master_register_file_summary = stage_2_master_variable_register.groupby(['Source order', 'Source file'],
    as_index=False).agg(Variables=('Variable', 'size'), Variables_without_labels=('Variable label',
    lambda values: values.eq('').sum())).sort_values('Source order').reset_index(drop=True)
print(f"Registered source files: {stage_2_master_variable_register['Source file'].nunique():,}")
print(f'Registered variables: {len(stage_2_master_variable_register):,}')
print(f"Duplicate source-variable pairs: {stage_2_master_variable_register.duplicated(['Source file', 'Variable']).sum():,}")
print(f"Variables without labels: {stage_2_master_variable_register['Variable label'].eq('').sum():,}")
print(f'Register columns: {stage_2_master_variable_register.columns.tolist()}')
print(f'Saved to: {master_register_output_path}')
master_register_file_summary

Registered source files: 14
Registered variables: 5,261
Duplicate source-variable pairs: 0
Variables without labels: 0
Register columns: ['Source order', 'Source file', 'Source path', 'Variable position', 'Variable', 'Variable label', 'Data type']
Saved to: data_derived\stage_2_predictor_construction\stage_2_master_variable_register.csv


,Source order,Source file,Variables,Variables_without_labels
0,1,wave_one_lsype_young_person_2020,350,0
1,2,wave_one_lsype_family_background_2020,381,0
...,...,...,...,...
12,13,wave_four_lsype_parental_attitudes_june_2009,77,0
13,14,wave_four_lsype_history_2020,42,0


In [74]:
# 2: Source-register content

print('Source-register columns:')
print(stage_2_source_file_register.columns.tolist())
print(f'\nSource-register rows: {len(stage_2_source_file_register):,}')
with pd.option_context('display.max_columns', None, 'display.max_colwidth', 120, 'display.width', 320):
    print(stage_2_source_file_register.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Source-register columns:
['Wave', 'Source type', 'Classification basis', 'File name', 'Relative path', 'Rows', 'Variables', 'NSID present', 'Unique NSID values', 'Missing NSID values', 'Duplicate NSID rows', 'Data structure', 'Roster participants present', 'Roster participants absent', 'Coverage percentage', 'Timing status', 'Review status']

Source-register rows: 14
  Wave        Source type Classification basis                                        File name                                    Relative path  Rows  Variables  NSID present  Unique NSID values  Missing NSID values  Duplicate NSID rows   Data structure  Roster participants present  Roster participants absent  Coverage percentage          Timing status                                                            Review status
Wave 1       Young person            File name             wave_one_lsype_young_person_2020.dta             wave_one_lsype_young_person_2020.dta 15760        350          True               15760      

In [75]:
# 3: Source-level metadata linkage

source_metadata_for_linkage = stage_2_source_file_register.copy()
source_metadata_for_linkage['Source file'] = source_metadata_for_linkage['File name'].str.replace('\\.dta$', '',
    regex=True)
required_source_metadata_columns = ['Source file', 'Wave', 'Source type', 'Data structure', 'Rows', 'Variables',
    'Roster participants present', 'Roster participants absent', 'Coverage percentage', 'Timing status', 'Review status']
missing_source_metadata_columns = [column for column in required_source_metadata_columns if column not in source_metadata_for_linkage.columns]
if missing_source_metadata_columns:
    raise KeyError(f'Required source-register columns were not found: {missing_source_metadata_columns}')
source_metadata_for_linkage = source_metadata_for_linkage[required_source_metadata_columns].copy()
stage_2_master_variable_register = stage_2_master_variable_register.drop(columns=['Wave', 'Source type',
    'Data structure', 'Rows', 'Variables', 'Roster participants present', 'Roster participants absent', 'Coverage percentage', 'Timing status', 'Review status'], errors='ignore').merge(source_metadata_for_linkage,
    on='Source file', how='left', validate='many_to_one')
linked_metadata_columns = ['Wave', 'Source type', 'Data structure', 'Timing status', 'Review status']
unmatched_source_rows = stage_2_master_variable_register[linked_metadata_columns].isna().all(axis=1).sum()
stage_2_master_variable_register.to_csv(master_register_output_path, index=False)
print(f'Registered variables: {len(stage_2_master_variable_register):,}')
print(f'Variables without linked source metadata: {unmatched_source_rows:,}')
print(f"Source files represented: {stage_2_master_variable_register['Source file'].nunique():,}")
print('Register columns:')
print(stage_2_master_variable_register.columns.tolist())
source_linkage_summary = stage_2_master_variable_register.groupby(['Wave', 'Source type', 'Timing status',
    'Review status'], dropna=False, as_index=False).agg(Source_files=('Source file', 'nunique'),
    Registered_variables=('Variable', 'size'))
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 100, 'display.width', 280):
    print('\nSource metadata summary:')
    print(source_linkage_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nSaved to: {master_register_output_path}')

Registered variables: 5,261
Variables without linked source metadata: 0
Source files represented: 14
Register columns:
['Source order', 'Source file', 'Source path', 'Variable position', 'Variable', 'Variable label', 'Data type', 'Wave', 'Source type', 'Data structure', 'Rows', 'Variables', 'Roster participants present', 'Roster participants absent', 'Coverage percentage', 'Timing status', 'Review status']

Source metadata summary:
     Wave        Source type          Timing status                                                            Review status  Source_files  Registered_variables
   Wave 1  Family background  Pre-transition source     Variable-level coding, routing and reference-period review required.             1                   381
   Wave 1 Parental attitudes  Pre-transition source     Variable-level coding, routing and reference-period review required.             1                   303
      ...                ...                    ...                              

In [76]:
# 4: Variable decision register

decision_register_columns = ['Source order', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable',
    'Variable label', 'Data type', 'Timing status', 'Review status']
stage_2_variable_decision_register = stage_2_master_variable_register[decision_register_columns].copy()
stage_2_variable_decision_register['Review outcome'] = 'Pending review'
stage_2_variable_decision_register['Substantive domain'] = pd.NA
stage_2_variable_decision_register['Decision reason'] = pd.NA
stage_2_variable_decision_register['Leakage assessment'] = pd.NA
stage_2_variable_decision_register['Reference-period assessment'] = pd.NA
stage_2_variable_decision_register['Documentation source'] = pd.NA
stage_2_variable_decision_register['Review notes'] = pd.NA
decision_register_output_path = stage_2_output_directory / 'stage_2_variable_decision_register.csv'
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
review_outcome_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Variables in decision register: {len(stage_2_variable_decision_register):,}')
print(f"Duplicate source-variable pairs: {stage_2_variable_decision_register.duplicated(['Source file', 'Variable']).sum():,}")
print(f"Variables with missing labels: {stage_2_variable_decision_register['Variable label'].isna().sum():,}")
print('Decision-register columns:')
print(stage_2_variable_decision_register.columns.tolist())
print(f'Saved to: {decision_register_output_path}')
review_outcome_summary

Variables in decision register: 5,261
Duplicate source-variable pairs: 0
Variables with missing labels: 0
Decision-register columns:
['Source order', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Data type', 'Timing status', 'Review status', 'Review outcome', 'Substantive domain', 'Decision reason', 'Leakage assessment', 'Reference-period assessment', 'Documentation source', 'Review notes']
Saved to: data_derived\stage_2_predictor_construction\stage_2_variable_decision_register.csv


,Review outcome,Variables
0,Pending review,5261


## Pre-domain variable review

The following lists are used to locate variables for manual review. A match does not determine inclusion or exclusion. Decisions are recorded in the subsequent cells after the variable name, label, source and timing have been checked.


In [77]:
# 5: Identifier and survey-design review list

variable_names = stage_2_variable_decision_register['Variable'].astype('string').str.strip()
variable_labels = stage_2_variable_decision_register['Variable label'].fillna('').astype('string').str.strip().str.lower()
technical_role = pd.Series(pd.NA, index=stage_2_variable_decision_register.index, dtype='string')
technical_role.loc[variable_names.str.upper().eq('NSID') | variable_labels.eq('nsid - cohort member identifier')] = 'Cohort member identifier'
technical_role.loc[variable_names.str.lower().eq('samppsu') | variable_labels.str.contains('primary sampling unit',
    regex=False)] = 'Primary sampling unit'
technical_role.loc[variable_names.str.lower().eq('sampstratum') | variable_labels.isin(['sampling: stratum',
    'stratum'])] = 'Sampling stratum'
technical_role.loc[variable_labels.str.contains('\\b(?:design|final|survey) weight\\b|\\bnon[- ]?response weight\\b',
    regex=True)] = 'Survey weight'
technical_review_mask = technical_role.notna()
stage_2_technical_variable_review = stage_2_variable_decision_register.loc[technical_review_mask, ['Source order',
    'Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label']].copy()
stage_2_technical_variable_review['Technical role'] = technical_role.loc[technical_review_mask].values
stage_2_technical_variable_review = stage_2_technical_variable_review.sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
print(f'Variables identified for technical review: {len(stage_2_technical_variable_review):,}')
display_limited(stage_2_technical_variable_review)

Variables identified for technical review: 55


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Technical role
0,1,Wave 1,Young person,wave_one_lsype_young_person_2020,1,NSID,NSID - cohort member identifier,Cohort member identifier
1,1,Wave 1,Young person,wave_one_lsype_young_person_2020,2,Designweight,Weight: Design weight,Survey weight
2,1,Wave 1,Young person,wave_one_lsype_young_person_2020,3,SampPSU,Sampling: School (primary sampling unit),Primary sampling unit
3,1,Wave 1,Young person,wave_one_lsype_young_person_2020,4,SampStratum,Sampling: Stratum,Sampling stratum
4,1,Wave 1,Young person,wave_one_lsype_young_person_2020,6,W1FinWt,Weight: Design weight * NR weights (trimmed fo...,Survey weight


In [78]:
# 6: Decisions for identifiers and survey-design variables

technical_decisions = stage_2_technical_variable_review[['Source file', 'Variable', 'Technical role']].copy()
review_outcome_by_role = {'Cohort member identifier': 'Retain as identifier only',
    'Survey weight': 'Exclude from predictor set', 'Primary sampling unit': 'Exclude from predictor set', 'Sampling stratum': 'Exclude from predictor set'}
domain_by_role = {'Cohort member identifier': 'Identifier', 'Survey weight': 'Survey design and weighting',
    'Primary sampling unit': 'Survey design and weighting', 'Sampling stratum': 'Survey design and weighting'}
decision_reason_by_role = {'Cohort member identifier': 'Unique cohort-member key required for file linkage; not a substantive predictor.',
    'Survey weight': 'Technical survey-design or non-response adjustment variable; not a pre-transition participant characteristic.', 'Primary sampling unit': 'Raw sampling-cluster or sampled-school code; not a substantive participant-level predictor.', 'Sampling stratum': 'Technical sampling-design category; not a substantive participant-level predictor.'}
leakage_assessment_by_role = {'Cohort member identifier': 'Not outcome-derived; identifier memorisation risk if entered as a predictor.',
    'Survey weight': 'No direct outcome leakage identified.', 'Primary sampling unit': 'Not outcome-derived; cluster memorisation risk if entered as a predictor.', 'Sampling stratum': 'No direct outcome leakage identified.'}
review_notes_by_role = {'Cohort member identifier': 'Retain for linkage and quality checks only.',
    'Survey weight': 'Retain outside the predictor matrix for possible survey-weighted descriptive or sensitivity analysis.', 'Primary sampling unit': 'Retain outside the predictor matrix as survey-design information.', 'Sampling stratum': 'Retain outside the predictor matrix as survey-design information.'}
technical_decisions['Review outcome'] = technical_decisions['Technical role'].map(review_outcome_by_role)
technical_decisions['Substantive domain'] = technical_decisions['Technical role'].map(domain_by_role)
technical_decisions['Decision reason'] = technical_decisions['Technical role'].map(decision_reason_by_role)
technical_decisions['Leakage assessment'] = technical_decisions['Technical role'].map(leakage_assessment_by_role)
technical_decisions['Reference-period assessment'] = 'Not applicable'
technical_decisions['Documentation source'] = 'Stata variable label and source-file metadata'
technical_decisions['Review notes'] = technical_decisions['Technical role'].map(review_notes_by_role)
decision_fields = ['Review outcome', 'Substantive domain', 'Decision reason', 'Leakage assessment',
    'Reference-period assessment', 'Documentation source', 'Review notes']
for _, decision_row in technical_decisions.iterrows():
    matching_rows = stage_2_variable_decision_register['Source file'].eq(decision_row['Source file']) & stage_2_variable_decision_register['Variable'].eq(decision_row['Variable'])
    for field in decision_fields:
        stage_2_variable_decision_register.loc[matching_rows, field] = decision_row[field]
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
technical_decision_output = stage_2_variable_decision_register.merge(technical_decisions[['Source file', 'Variable',
    'Technical role']], on=['Source file',
    'Variable'], how='inner', validate='one_to_one').sort_values(['Source order', 'Variable position'])
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Technical entries reviewed: {len(technical_decision_output):,}')
print(f"Retained as identifier only: {technical_decision_output['Review outcome'].eq('Retain as identifier only').sum():,}")
print(f"Excluded from predictor set: {technical_decision_output['Review outcome'].eq('Exclude from predictor set').sum():,}")
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nTechnical decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 100, 'display.width', 320):
    print(technical_decision_output[['Wave', 'Source type', 'Variable', 'Variable label', 'Technical role',
        'Review outcome', 'Decision reason', 'Leakage assessment']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nSaved to: {decision_register_output_path}')

Technical entries reviewed: 55
Retained as identifier only: 14
Excluded from predictor set: 41

Overall decision-register status:
            Review outcome  Variables
            Pending review       5206
Exclude from predictor set         41
 Retain as identifier only         14

Technical decisions:
  Wave  Source type     Variable                           Variable label           Technical role             Review outcome                                                                                               Decision reason                                                           Leakage assessment
Wave 1 Young person         NSID         NSID - cohort member identifier  Cohort member identifier  Retain as identifier only                              Unique cohort-member key required for file linkage; not a substantive predictor. Not outcome-derived; identifier memorisation risk if entered as a predictor.
Wave 1 Young person Designweight                    Weight: Design wei

In [79]:
# 7: First fieldwork-review group

first_fieldwork_variables = {'W1scomadiMP', 'W1scompinYP', 'W2scomadiMP', 'W2scompinYP', 'W3scomadiMP', 'W3scompinYP',
    'W4ScomAdiMP', 'W4ScompinYP', 'W1sexYP', 'W2SexYP', 'W3sexYP', 'W4SexYP', 'W4AccTypeYP', 'W4AccTypeMP', 'W4sourceSP'}
stage_2_fieldwork_review = stage_2_variable_decision_register.loc[stage_2_variable_decision_register['Variable'].isin(first_fieldwork_variables),
    ['Source order', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Review outcome']].copy().sort_values(['Source order', 'Variable position']).reset_index(drop=True)
identified_fieldwork_variables = set(stage_2_fieldwork_review['Variable'])
missing_fieldwork_variables = first_fieldwork_variables - identified_fieldwork_variables
if missing_fieldwork_variables:
    raise ValueError(f'Fieldwork-review variables not found: {sorted(missing_fieldwork_variables)}')
print(f'Variables identified for fieldwork review: {len(stage_2_fieldwork_review):,}')
display_limited(stage_2_fieldwork_review)

Variables identified for fieldwork review: 16


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Review outcome
0,1,Wave 1,Young person,wave_one_lsype_young_person_2020,29,W1scomadiMP,Admin: Interviewer code whether MP accepted se...,Pending review
1,1,Wave 1,Young person,wave_one_lsype_young_person_2020,92,W1sexYP,Admin: Interviewer code sex of YP,Pending review
2,1,Wave 1,Young person,wave_one_lsype_young_person_2020,219,W1scompinYP,Admin: Interviewer code whether YP accepted se...,Pending review
3,4,Wave 2,Young person,wave_two_lsype_young_person_2020,37,W2scomadiMP,Admin: Interviewer code whether MP accepted se...,Pending review
4,4,Wave 2,Young person,wave_two_lsype_young_person_2020,84,W2SexYP,Admin: Interviewer code sex of YP,Pending review


In [80]:
# 8: Decisions for the first fieldwork-review group

self_completion_variables = {'W1scomadiMP', 'W1scompinYP', 'W2scomadiMP', 'W2scompinYP', 'W3scomadiMP', 'W3scompinYP',
    'W4ScomAdiMP', 'W4ScompinYP'}
sex_variables = {'W1sexYP', 'W2SexYP', 'W3sexYP', 'W4SexYP'}
accommodation_variables = {'W4AccTypeYP', 'W4AccTypeMP'}
response_source_variables = {'W4sourceSP'}

def assign_fieldwork_decision(variable):
    if variable in self_completion_variables:
        return {'Review outcome': 'Exclude from predictor set', 'Substantive domain': 'Survey administration',
            'Decision reason': 'Records acceptance of a self-completion section and therefore reflects the survey process rather than a substantive pre-transition characteristic.', 'Leakage assessment': 'No direct outcome leakage identified; survey-process information only.', 'Reference-period assessment': 'Interview administration at the relevant wave.', 'Documentation source': 'Wave-specific questionnaire and Stata variable label', 'Review notes': 'Excluded from the predictor matrix; not used as a proxy for questionnaire completion or item availability.'}
    if variable in sex_variables:
        return {'Review outcome': 'Candidate measure - cross-wave consolidation required',
            'Substantive domain': 'Demographic background', 'Decision reason': "Records the young person's sex and is therefore substantive demographic information, despite being recorded by the interviewer.", 'Leakage assessment': 'No direct outcome leakage identified.', 'Reference-period assessment': "Young person's sex recorded at the relevant wave.", 'Documentation source': 'Wave-specific data dictionary and Stata variable label', 'Review notes': 'Compare values and availability across waves before constructing one participant-level measure.'}
    if variable in accommodation_variables:
        return {'Review outcome': 'Detailed timing and source review required',
            'Substantive domain': 'Housing and material circumstances', 'Decision reason': 'Records substantive accommodation type, but was measured at Wave 4 and is available from both young-person and main-parent interview contexts.', 'Leakage assessment': 'Not outcome-derived, but Wave 4 timing may reflect circumstances after the post-16 transition began.', 'Reference-period assessment': 'Accommodation at the Wave 4 interview; eligibility as a pre-transition measure has not been established.', 'Documentation source': 'Wave 4 data dictionary and Stata variable label', 'Review notes': 'Compare with earlier-wave accommodation measures and examine agreement between young-person and main-parent records.'}
    if variable in response_source_variables:
        return {'Review outcome': 'Retain as review support only', 'Substantive domain': 'Data provenance',
            'Decision reason': 'Identifies whether partner information was reported directly, by the main adult, or with consultation; it is not a participant characteristic.', 'Leakage assessment': 'No direct outcome leakage identified; survey response-source metadata only.', 'Reference-period assessment': 'Wave 4 interview response source.', 'Documentation source': 'LSYPE User Guide and Wave 4 family-background data dictionary', 'Review notes': 'Retain outside the predictor matrix to support review of partner-reported variables.'}
    return None
fieldwork_decision_records = []
for _, review_row in stage_2_fieldwork_review.iterrows():
    decision = assign_fieldwork_decision(review_row['Variable'])
    if decision is None:
        raise ValueError(f"No decision was assigned to {review_row['Variable']}.")
    fieldwork_decision_records.append({'Source file': review_row['Source file'], 'Variable': review_row['Variable'],
        **decision})
fieldwork_decisions = pd.DataFrame(fieldwork_decision_records)
decision_fields = ['Review outcome', 'Substantive domain', 'Decision reason', 'Leakage assessment',
    'Reference-period assessment', 'Documentation source', 'Review notes']
for _, decision_row in fieldwork_decisions.iterrows():
    matching_rows = stage_2_variable_decision_register['Source file'].eq(decision_row['Source file']) & stage_2_variable_decision_register['Variable'].eq(decision_row['Variable'])
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one matching source-variable entry for {decision_row['Variable']}, found {matching_rows.sum()}.")
    for field in decision_fields:
        stage_2_variable_decision_register.loc[matching_rows, field] = decision_row[field]
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
fieldwork_decision_output = stage_2_variable_decision_register.merge(fieldwork_decisions[['Source file', 'Variable']],
    on=['Source file', 'Variable'], how='inner', validate='one_to_one').sort_values(['Source order',
    'Variable position'])
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Fieldwork entries reviewed: {len(fieldwork_decision_output):,}')
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nFieldwork decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 110, 'display.width', 320):
    print(fieldwork_decision_output[['Wave', 'Source type', 'Variable', 'Variable label', 'Review outcome',
        'Decision reason', 'Leakage assessment']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nSaved to: {decision_register_output_path}')

Fieldwork entries reviewed: 16

Overall decision-register status:
                            Review outcome  Variables
                            Pending review       5190
                Exclude from predictor set         50
                                       ...        ...
Detailed timing and source review required          2
             Retain as review support only          1

Fieldwork decisions:
  Wave       Source type    Variable                                                      Variable label                                        Review outcome                                                                                                                                    Decision reason                                                                                   Leakage assessment
Wave 1      Young person W1scomadiMP Admin: Interviewer code whether MP accepted self-completion section                            Exclude from predictor set Records acceptance of 

In [81]:
# 9: Interview-presence variable review list

pending_variable_register = stage_2_variable_decision_register.loc[stage_2_variable_decision_register['Review outcome'].eq('Pending review')].copy()
pending_labels = pending_variable_register['Variable label'].fillna('').astype('string')
interview_presence_mask = pending_labels.str.contains('\\bpresent\\b', case=False,
    regex=True) & pending_labels.str.contains('\\binterview\\b', case=False,
    regex=True) | pending_labels.str.contains('\\bothers? present\\b', case=False, regex=True)
stage_2_survey_operation_review = pending_variable_register.loc[interview_presence_mask, ['Source order', 'Wave',
    'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Review outcome']].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
print(f'Interview-presence variables identified: {len(stage_2_survey_operation_review):,}')
display_limited(stage_2_survey_operation_review)

Interview-presence variables identified: 98


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Review outcome
0,1,Wave 1,Young person,wave_one_lsype_young_person_2020,333,W1chpreYP0a,Admin: Who else present during YP interview : ...,Pending review
1,1,Wave 1,Young person,wave_one_lsype_young_person_2020,334,W1chpreYP0b,Admin: Who else present during YP interview: M...,Pending review
2,1,Wave 1,Young person,wave_one_lsype_young_person_2020,335,W1chpreYP0c,Admin: Who else present during YP interview: F...,Pending review
3,1,Wave 1,Young person,wave_one_lsype_young_person_2020,336,W1chpreYP0d,Admin: Who else present during YP interview: (...,Pending review
4,1,Wave 1,Young person,wave_one_lsype_young_person_2020,337,W1chpreYP0e,Admin: Who else present during YP interview: O...,Pending review


In [82]:
# 10: Decisions for interview-presence variables

interview_presence_decisions = stage_2_survey_operation_review[['Source file', 'Variable']].copy()
interview_presence_decisions['Review outcome'] = 'Exclude from predictor set'
interview_presence_decisions['Substantive domain'] = 'Survey administration'
interview_presence_decisions['Decision reason'] = 'Records who was present during an interview and therefore describes the survey context rather than a substantive pre-transition characteristic.'
interview_presence_decisions['Leakage assessment'] = 'No direct outcome leakage identified; survey-process information only.'
interview_presence_decisions['Reference-period assessment'] = 'Interview administration at the relevant wave.'
interview_presence_decisions['Documentation source'] = 'Stata variable label and wave-specific data dictionary'
interview_presence_decisions['Review notes'] = 'Excluded from the predictor matrix. Household composition will be assessed using substantive household and family variables rather than interview-presence records.'
decision_fields = ['Review outcome', 'Substantive domain', 'Decision reason', 'Leakage assessment',
    'Reference-period assessment', 'Documentation source', 'Review notes']
for _, decision_row in interview_presence_decisions.iterrows():
    matching_rows = stage_2_variable_decision_register['Source file'].eq(decision_row['Source file']) & stage_2_variable_decision_register['Variable'].eq(decision_row['Variable'])
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one matching source-variable entry for {decision_row['Variable']}, found {matching_rows.sum()}.")
    for field in decision_fields:
        stage_2_variable_decision_register.loc[matching_rows, field] = decision_row[field]
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
interview_presence_decision_output = stage_2_variable_decision_register.merge(interview_presence_decisions[['Source file',
    'Variable']], on=['Source file', 'Variable'], how='inner', validate='one_to_one').sort_values(['Source order',
    'Variable position'])
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Interview-presence entries reviewed: {len(interview_presence_decision_output):,}')
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nExcluded interview-presence variables:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 110, 'display.width', 320):
    print(interview_presence_decision_output[['Wave', 'Source type', 'Variable', 'Variable label', 'Review outcome',
        'Decision reason', 'Leakage assessment']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nSaved to: {decision_register_output_path}')

Interview-presence entries reviewed: 98

Overall decision-register status:
                            Review outcome  Variables
                            Pending review       5092
                Exclude from predictor set        148
                                       ...        ...
Detailed timing and source review required          2
             Retain as review support only          1

Excluded interview-presence variables:
  Wave  Source type       Variable                                                                   Variable label             Review outcome                                                                                                                                 Decision reason                                                     Leakage assessment
Wave 1 Young person    W1chpreYP0a                Admin: Who else present during YP interview : No-one else in room Exclude from predictor set Records who was present during an interview and therefore de

In [83]:
# 11: Record-management and data-editing review list

pending_variable_register = stage_2_variable_decision_register.loc[stage_2_variable_decision_register['Review outcome'].eq('Pending review')].copy()
pending_names = pending_variable_register['Variable'].fillna('').astype('string')
pending_labels = pending_variable_register['Variable label'].fillna('').astype('string')
record_identifier_mask = pending_labels.str.contains('\\b(?:record|case|serial|respondent|person) number\\b|\\brecord type\\b',
    case=False, regex=True) | pending_names.str.contains('(?:record|case|serial|respondent|person)(?:no|num|number)',
    case=False, regex=True)
data_editing_mask = pending_labels.str.contains('\\bedit(?:ed|ing)?\\b|\\bimput(?:ed|ation)\\b', case=False,
    regex=True)
review_area = pd.Series(pd.NA, index=pending_variable_register.index, dtype='string')
review_area.loc[record_identifier_mask] = 'Record-management identifier'
review_area.loc[data_editing_mask & review_area.isna()] = 'Data editing or imputation'
technical_control_mask = review_area.notna()
stage_2_technical_control_review = pending_variable_register.loc[technical_control_mask, ['Source order', 'Wave',
    'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Review outcome']].copy()
stage_2_technical_control_review['Review area'] = review_area.loc[technical_control_mask].values
stage_2_technical_control_review = stage_2_technical_control_review.sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
print(f'Variables identified for technical-control review: {len(stage_2_technical_control_review):,}')
display_limited(stage_2_technical_control_review)

Variables identified for technical-control review: 49


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Review outcome,Review area
0,1,Wave 1,Young person,wave_one_lsype_young_person_2020,7,DobyearYP,Edited: Year of birth of YP,Pending review,Data editing or imputation
1,1,Wave 1,Young person,wave_one_lsype_young_person_2020,8,DobmonthYP,Edited: Month of birth of YP,Pending review,Data editing or imputation
2,2,Wave 1,Family background,wave_one_lsype_family_background_2020,20,W1mainres,Admin: Person number of main parent,Pending review,Record-management identifier
3,2,Wave 1,Family background,wave_one_lsype_family_background_2020,21,W1secores,Admin: Person number of second parent,Pending review,Record-management identifier
4,2,Wave 1,Family background,wave_one_lsype_family_background_2020,22,W1InfoN,Admin: Person number of household reference pe...,Pending review,Record-management identifier


In [84]:
# 12: Decisions for record identifiers and data-editing flags

technical_control_decisions = stage_2_technical_control_review[['Source file', 'Variable', 'Review area']].copy()

def assign_technical_control_decision(review_area):
    """Assign a decision according to the technical function."""
    if 'Record-management identifier' in review_area:
        return {'Review outcome': 'Retain as review support only', 'Substantive domain': 'Data provenance',
            'Decision reason': 'Identifies a respondent or household member within the survey record and is not a substantive participant characteristic.', 'Leakage assessment': 'Not outcome-derived; identifier or record memorisation risk if entered as a predictor.', 'Reference-period assessment': 'Not applicable.', 'Documentation source': 'Stata variable label and source-file metadata', 'Review notes': 'Retain outside the predictor matrix for respondent and record-source checks only.'}
    return {'Review outcome': 'Retain as review support only', 'Substantive domain': 'Data quality',
        'Decision reason': "Indicates whether an income value was edited during data processing and does not measure the household's income or circumstances.", 'Leakage assessment': 'No direct outcome leakage identified; data-processing information only.', 'Reference-period assessment': 'Applies to income data recorded at the relevant wave.', 'Documentation source': 'Stata variable label and wave-specific data dictionary', 'Review notes': 'Retain outside the predictor matrix to support the later review of income variables.'}
technical_decision_details = technical_control_decisions['Review area'].apply(assign_technical_control_decision).apply(pd.Series)
technical_control_decisions = pd.concat([technical_control_decisions, technical_decision_details], axis=1)
decision_fields = ['Review outcome', 'Substantive domain', 'Decision reason', 'Leakage assessment',
    'Reference-period assessment', 'Documentation source', 'Review notes']
for _, decision_row in technical_control_decisions.iterrows():
    matching_rows = stage_2_variable_decision_register['Source file'].eq(decision_row['Source file']) & stage_2_variable_decision_register['Variable'].eq(decision_row['Variable'])
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one matching source-variable entry for {decision_row['Variable']}, found {matching_rows.sum()}.")
    for field in decision_fields:
        stage_2_variable_decision_register.loc[matching_rows, field] = decision_row[field]
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
technical_control_decision_output = stage_2_variable_decision_register.merge(technical_control_decisions[['Source file',
    'Variable', 'Review area']], on=['Source file',
    'Variable'], how='inner', validate='one_to_one').sort_values(['Source order', 'Variable position'])
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Technical-control entries reviewed: {len(technical_control_decision_output):,}')
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nTechnical-control decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 110, 'display.width', 320):
    print(technical_control_decision_output[['Wave', 'Source type', 'Variable', 'Variable label', 'Review area',
        'Review outcome', 'Decision reason', 'Leakage assessment']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nSaved to: {decision_register_output_path}')

Technical-control entries reviewed: 49

Overall decision-register status:
                                       Review outcome  Variables
                                       Pending review       5043
                           Exclude from predictor set        148
                                                  ...        ...
Candidate measure - cross-wave consolidation required          4
           Detailed timing and source review required          2

Technical-control decisions:
  Wave  Source type    Variable                                                                   Variable label                Review area                Review outcome                                                                                                                   Decision reason                                                      Leakage assessment
Wave 1 Young person   DobyearYP                                                      Edited: Year of birth of YP Data editing or imput

In [85]:
# 13: Core demographic background review list

demographic_review_statuses = ['Pending review', 'Candidate measure - cross-wave consolidation required']
demographic_review_base = stage_2_variable_decision_register.loc[stage_2_variable_decision_register['Review outcome'].isin(demographic_review_statuses)].copy()
demographic_label_pattern = "\\bsex of (?:the )?(?:yp|young person)\\b|\\b(?:yp|young person)(?:'s)? sex\\b|\\bdate of birth\\b|\\byear of birth\\b|\\bmonth of birth\\b|\\bethnic group\\b|\\bethnicity\\b|\\bfirst language\\b|\\blanguage spoken at home\\b|\\breligion\\b|\\bcountry of birth\\b|\\bnationality\\b|\\bcitizenship\\b"
demographic_review_mask = demographic_review_base['Variable label'].fillna('').astype('string').str.contains(demographic_label_pattern,
    case=False, regex=True)
stage_2_core_demographic_review = demographic_review_base.loc[demographic_review_mask, ['Source order', 'Wave',
    'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Review outcome']].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
print(f'Variables identified for core demographic review: {len(stage_2_core_demographic_review):,}')
display_limited(stage_2_core_demographic_review)

Variables identified for core demographic review: 76


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Review outcome
0,1,Wave 1,Young person,wave_one_lsype_young_person_2020,16,W1sen1MP0c,MP: Nature of YP's special needs: English not ...,Pending review
1,1,Wave 1,Young person,wave_one_lsype_young_person_2020,92,W1sexYP,Admin: Interviewer code sex of YP,Candidate measure - cross-wave consolidation r...
2,1,Wave 1,Young person,wave_one_lsype_young_person_2020,95,W1relig1YP,YP: YP's religion,Pending review
3,1,Wave 1,Young person,wave_one_lsype_young_person_2020,97,W1relig3YP,YP: Importance of religion to YP's way of life,Pending review
4,1,Wave 1,Young person,wave_one_lsype_young_person_2020,349,W1ethgrpYP,DV: Young person's ethnic group (grouped),Pending review


In [86]:
# 14: Birth-information and sex value review

basic_demographic_variables = ['DobyearYP', 'DobmonthYP', 'W1sexYP', 'W2SexYP', 'W3sexYP', 'W4SexYP']
basic_demographic_metadata = stage_2_master_variable_register.loc[stage_2_master_variable_register['Variable'].isin(basic_demographic_variables),
    ['Source order', 'Wave', 'Source type', 'Source file', 'Source path', 'Variable position', 'Variable',
    'Variable label']].copy().sort_values(['Source order', 'Variable position']).reset_index(drop=True)
matched_variables = set(basic_demographic_metadata['Variable'])
missing_variables = set(basic_demographic_variables) - matched_variables
duplicate_variables = basic_demographic_metadata['Variable'].value_counts().loc[lambda counts: counts.ne(1)]
if missing_variables:
    raise ValueError(f'Variables not found in the master register: {sorted(missing_variables)}')
if not duplicate_variables.empty:
    raise ValueError(f'Variables found more than once:\n{duplicate_variables}')
stage_2_sample_ids = participant_roster[['NSID']].drop_duplicates().copy()
basic_demographic_summary_records = []
basic_demographic_frequency_tables = {}
for source_path, source_group in basic_demographic_metadata.groupby('Source path', sort=False):
    source_variables = source_group['Variable'].tolist()
    raw_source_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_source_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    raw_sample_data = stage_2_sample_ids.merge(raw_source_data, on='NSID', how='left', indicator=True)
    labelled_sample_data = stage_2_sample_ids.merge(labelled_source_data, on='NSID', how='left')
    source_present = raw_sample_data['_merge'].eq('both')
    for variable in source_variables:
        metadata_row = source_group.loc[source_group['Variable'].eq(variable)].iloc[0]
        raw_values = raw_sample_data.loc[source_present, variable]
        negative_code_count = pd.to_numeric(raw_values, errors='coerce').lt(0).sum()
        basic_demographic_summary_records.append({'Wave': metadata_row['Wave'], 'Variable': variable,
            'Variable label': metadata_row['Variable label'], 'Participants present in source': int(source_present.sum()), 'Participants absent from source': int((~source_present).sum()), 'Negative-coded records': int(negative_code_count), 'Distinct raw values': int(raw_values.nunique(dropna=True))})
        raw_frequency = raw_values.value_counts(dropna=False).rename_axis('Raw value').reset_index(name='Count')
        labelled_frequency = labelled_sample_data.loc[source_present,
            variable].value_counts(dropna=False).rename_axis('Labelled value').reset_index(name='Count')
        basic_demographic_frequency_tables[variable] = {'raw': raw_frequency, 'labelled': labelled_frequency}
basic_demographic_summary = pd.DataFrame(basic_demographic_summary_records)
print('Basic demographic summary:')
with pd.option_context('display.max_colwidth', 100, 'display.width', 260):
    print(basic_demographic_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
for variable in basic_demographic_variables:
    variable_label = basic_demographic_metadata.loc[basic_demographic_metadata['Variable'].eq(variable),
        'Variable label'].iloc[0]
    print(f'\n{variable}: {variable_label}')
    print('Raw values:')
    print(basic_demographic_frequency_tables[variable]['raw'].to_string(max_rows=TABLE_ROW_LIMIT, index=False))
    print('Labelled values:')
    print(basic_demographic_frequency_tables[variable]['labelled'].to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Basic demographic summary:
  Wave   Variable                    Variable label  Participants present in source  Participants absent from source  Negative-coded records  Distinct raw values
Wave 1  DobyearYP       Edited: Year of birth of YP                            9524                              243                       0                    4
Wave 1 DobmonthYP      Edited: Month of birth of YP                            9524                              243                       0                   12
   ...        ...                               ...                             ...                              ...                     ...                  ...
Wave 3    W3sexYP Admin: Interviewer code sex of YP                            9509                              258                      54                    3
Wave 4    W4SexYP Admin: Interviewer code sex of YP                            9756                               11                      93                    3



In [87]:
# 15: Cross-wave sex consistency and coverage

import numpy as np
from itertools import combinations
sex_variables = ['W1sexYP', 'W2SexYP', 'W3sexYP', 'W4SexYP']
sex_metadata = basic_demographic_metadata.loc[basic_demographic_metadata['Variable'].isin(sex_variables), ['Wave',
    'Source path', 'Variable']].copy()
sex_review_data = stage_2_sample_ids.copy()
for _, metadata_row in sex_metadata.iterrows():
    variable = metadata_row['Variable']
    source_data = pd.read_stata(metadata_row['Source path'], columns=['NSID', variable], convert_categoricals=False)
    sex_review_data = sex_review_data.merge(source_data, on='NSID', how='left', validate='one_to_one')
valid_sex_columns = []
for variable in sex_variables:
    valid_column = f'{variable}_valid'
    sex_review_data[valid_column] = pd.to_numeric(sex_review_data[variable],
        errors='coerce').where(pd.to_numeric(sex_review_data[variable], errors='coerce').isin([1, 2]))
    valid_sex_columns.append(valid_column)
sex_review_data['Valid sex records'] = sex_review_data[valid_sex_columns].notna().sum(axis=1)
sex_review_data['Distinct valid sex codes'] = sex_review_data[valid_sex_columns].nunique(axis=1, dropna=True)
sex_review_data['Sex agreement status'] = np.select([sex_review_data['Valid sex records'].eq(0),
    sex_review_data['Valid sex records'].eq(1), sex_review_data['Distinct valid sex codes'].eq(1), sex_review_data['Distinct valid sex codes'].gt(1)], ['No valid record',
    'One valid record', 'Multiple records agreeing', 'Multiple records disagreeing'], default='Check required')
sex_review_data['First available sex code'] = sex_review_data[valid_sex_columns].bfill(axis=1).iloc[:, 0]
first_available_source = sex_review_data[valid_sex_columns].notna().idxmax(axis=1).str.replace('_valid', '',
    regex=False)
sex_review_data['First available source'] = first_available_source.where(sex_review_data['Valid sex records'].gt(0),
    pd.NA)
agreement_summary = sex_review_data['Sex agreement status'].value_counts().rename_axis('Agreement status').reset_index(name='Participants')
valid_record_summary = sex_review_data['Valid sex records'].value_counts().sort_index().rename_axis('Valid wave-specific records').reset_index(name='Participants')
pairwise_records = []
for first_variable, second_variable in combinations(sex_variables, 2):
    first_column = f'{first_variable}_valid'
    second_column = f'{second_variable}_valid'
    both_available = sex_review_data[first_column].notna() & sex_review_data[second_column].notna()
    agreements = sex_review_data.loc[both_available, first_column].eq(sex_review_data.loc[both_available,
        second_column]).sum()
    pairwise_records.append({'First variable': first_variable, 'Second variable': second_variable,
        'Both available': int(both_available.sum()), 'Agree': int(agreements), 'Disagree': int(both_available.sum() - agreements), 'Agreement percentage': round(100 * agreements / both_available.sum(),
        2) if both_available.sum() > 0 else np.nan})
sex_pairwise_agreement = pd.DataFrame(pairwise_records)
wave_1_valid = sex_review_data['W1sexYP_valid'].notna()
later_wave_valid = sex_review_data[['W2SexYP_valid', 'W3sexYP_valid', 'W4SexYP_valid']].notna().any(axis=1)
later_wave_recovery = ~wave_1_valid & later_wave_valid
consolidated_distribution = sex_review_data['First available sex code'].map({1.0: 'Male',
    2.0: 'Female'}).fillna('No valid record').value_counts().rename_axis('Provisional value').reset_index(name='Participants')
print(f'Stage 2 participants: {len(sex_review_data):,}')
print(f"Participants with at least one valid sex record: {sex_review_data['Valid sex records'].gt(0).sum():,}")
print(f"Participants with no valid sex record: {sex_review_data['Valid sex records'].eq(0).sum():,}")
print(f'Participants without a valid Wave 1 record but recovered from a later wave: {later_wave_recovery.sum():,}')
print('\nAgreement status:')
print(agreement_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nNumber of valid wave-specific records:')
print(valid_record_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nPairwise agreement:')
print(sex_pairwise_agreement.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nProvisional first-available distribution:')
print(consolidated_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Stage 2 participants: 9,767
Participants with at least one valid sex record: 9,758
Participants with no valid sex record: 9
Participants without a valid Wave 1 record but recovered from a later wave: 323

Agreement status:
            Agreement status  Participants
   Multiple records agreeing          9443
            One valid record           244
Multiple records disagreeing            71
             No valid record             9

Number of valid wave-specific records:
 Valid wave-specific records  Participants
                           0             9
                           1           244
                           2            24
                           3           254
                           4          9236

Pairwise agreement:
First variable Second variable  Both available  Agree  Disagree  Agreement percentage
       W1sexYP         W2SexYP            9364   9342        22                 99.77
       W1sexYP         W3sexYP            9371   9337        34        

In [88]:
# 16: Cross-wave sex disagreement review

sex_code_labels = {1.0: 'Male', 2.0: 'Female'}
sex_disagreement_data = sex_review_data.loc[sex_review_data['Sex agreement status'].eq('Multiple records disagreeing')].copy()
sex_disagreement_data['Male records'] = sex_disagreement_data[valid_sex_columns].eq(1).sum(axis=1)
sex_disagreement_data['Female records'] = sex_disagreement_data[valid_sex_columns].eq(2).sum(axis=1)
sex_disagreement_data['Majority status'] = np.select([sex_disagreement_data['Male records'].gt(sex_disagreement_data['Female records']),
    sex_disagreement_data['Female records'].gt(sex_disagreement_data['Male records']), sex_disagreement_data['Male records'].eq(sex_disagreement_data['Female records'])], ['Male majority',
    'Female majority', 'Tie'], default='Check required')
sex_disagreement_data['Majority sex code'] = np.select([sex_disagreement_data['Majority status'].eq('Male majority'),
    sex_disagreement_data['Majority status'].eq('Female majority')], [1.0, 2.0], default=np.nan)

def describe_sex_pattern(row):
    pattern_parts = []
    for variable in sex_variables:
        value = row[f'{variable}_valid']
        if pd.isna(value):
            readable_value = 'Unavailable'
        else:
            readable_value = sex_code_labels[float(value)]
        pattern_parts.append(f'{variable}={readable_value}')
    return ' | '.join(pattern_parts)
sex_disagreement_data['Cross-wave pattern'] = sex_disagreement_data.apply(describe_sex_pattern, axis=1)
sex_disagreement_pattern_summary = sex_disagreement_data['Cross-wave pattern'].value_counts().rename_axis('Cross-wave pattern').reset_index(name='Participants')
sex_majority_summary = sex_disagreement_data[['Valid sex records', 'Male records', 'Female records',
    'Majority status']].value_counts().rename('Participants').reset_index().sort_values(['Valid sex records',
    'Majority status', 'Male records'])
wave_disagreement_records = []
unique_majority_mask = sex_disagreement_data['Majority sex code'].notna()
for variable in sex_variables:
    valid_column = f'{variable}_valid'
    variable_available = sex_disagreement_data[valid_column].notna()
    differs_from_majority = unique_majority_mask & variable_available & sex_disagreement_data[valid_column].ne(sex_disagreement_data['Majority sex code'])
    wave_disagreement_records.append({'Variable': variable,
        'Available in unique-majority cases': int((unique_majority_mask & variable_available).sum()), 'Different from cross-wave majority': int(differs_from_majority.sum())})
wave_disagreement_summary = pd.DataFrame(wave_disagreement_records)
print(f'Participants with disagreeing sex records: {len(sex_disagreement_data):,}')
print('\nMajority and tie structure:')
print(sex_majority_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave records differing from a unique majority:')
print(wave_disagreement_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nExact cross-wave disagreement patterns:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 180, 'display.width', 300):
    print(sex_disagreement_pattern_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Participants with disagreeing sex records: 71

Majority and tie structure:
 Valid sex records  Male records  Female records Majority status  Participants
                 3             1               2 Female majority             2
                 4             1               3 Female majority            38
                 4             3               1   Male majority            30
                 4             2               2             Tie             1

Wave records differing from a unique majority:
Variable  Available in unique-majority cases  Different from cross-wave majority
 W1sexYP                                  70                                  10
 W2SexYP                                  70                                  12
 W3sexYP                                  68                                  23
 W4SexYP                                  70                                  25

Exact cross-wave disagreement patterns:
                                    

In [89]:
# 17: Review of the consolidated sex measure

sex_review_data['Male records'] = sex_review_data[valid_sex_columns].eq(1).sum(axis=1)
sex_review_data['Female records'] = sex_review_data[valid_sex_columns].eq(2).sum(axis=1)
sex_review_data['Consolidated sex code'] = np.select([sex_review_data['Male records'].gt(sex_review_data['Female records']),
    sex_review_data['Female records'].gt(sex_review_data['Male records'])], [1.0, 2.0], default=np.nan)
sex_review_data['Consolidation basis'] = np.select([sex_review_data['Valid sex records'].eq(0),
    sex_review_data['Valid sex records'].gt(0) & sex_review_data['Male records'].eq(sex_review_data['Female records']), sex_review_data['Valid sex records'].eq(1), sex_review_data['Valid sex records'].gt(1) & sex_review_data['Distinct valid sex codes'].eq(1), sex_review_data['Valid sex records'].gt(1) & sex_review_data['Distinct valid sex codes'].gt(1)], ['No valid record',
    'Tie unresolved', 'One valid record', 'Multiple records agreeing', 'Cross-wave majority'], default='Check required')
sex_review_data['Consolidated sex'] = sex_review_data['Consolidated sex code'].map({1.0: 'Male',
    2.0: 'Female'}).fillna('Missing')
consolidation_basis_summary = sex_review_data['Consolidation basis'].value_counts().rename_axis('Consolidation basis').reset_index(name='Participants')
consolidated_sex_summary = sex_review_data['Consolidated sex'].value_counts().rename_axis('Consolidated sex').reset_index(name='Participants')
unexpected_basis = sex_review_data['Consolidation basis'].eq('Check required').sum()
print(f'Stage 2 participants: {len(sex_review_data):,}')
print(f"Participants with a consolidated value: {sex_review_data['Consolidated sex code'].notna().sum():,}")
print(f"Participants remaining missing: {sex_review_data['Consolidated sex code'].isna().sum():,}")
print(f'Unexpected consolidation cases: {unexpected_basis:,}')
print('\nConsolidation basis:')
print(consolidation_basis_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nConsolidated distribution:')
print(consolidated_sex_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nUnresolved-record summary:')
print(
    sex_review_data.loc[sex_review_data['Consolidated sex code'].isna(), 'Consolidation basis']
    .value_counts(dropna=False)
    .rename_axis('Consolidation basis')
    .reset_index(name='Participants')
    .to_string(index=False)
)
# Individual participant records remain available in memory for checking but are not displayed in the public notebook.

Stage 2 participants: 9,767
Participants with a consolidated value: 9,757
Participants remaining missing: 10
Unexpected consolidation cases: 0

Consolidation basis:
      Consolidation basis  Participants
Multiple records agreeing          9443
         One valid record           244
      Cross-wave majority            70
          No valid record             9
           Tie unresolved             1

Consolidated distribution:
Consolidated sex  Participants
          Female          4976
            Male          4781
         Missing            10

Unresolved-record summary:
Consolidation basis  Participants
    No valid record             9
     Tie unresolved             1


In [90]:
# 18: Earliest-available sex measure

sex_review_data['Sex code'] = sex_review_data[valid_sex_columns].bfill(axis=1).iloc[:, 0]
first_valid_column = sex_review_data[valid_sex_columns].notna().idxmax(axis=1)
sex_source_map = {'W1sexYP_valid': 'Wave 1', 'W2SexYP_valid': 'Wave 2', 'W3sexYP_valid': 'Wave 3',
    'W4SexYP_valid': 'Wave 4'}
sex_review_data['Sex source'] = first_valid_column.map(sex_source_map).where(sex_review_data['Valid sex records'].gt(0),
    pd.NA)
sex_review_data['Sex'] = sex_review_data['Sex code'].map({1.0: 'Male', 2.0: 'Female'})
sex_review_data['Sex cross-wave disagreement'] = sex_review_data['Distinct valid sex codes'].gt(1)
stage_2_sex_measure = sex_review_data[['NSID', 'Sex code', 'Sex', 'Sex source', 'Valid sex records',
    'Sex cross-wave disagreement']].copy()
print(f'Rows: {len(stage_2_sex_measure):,}')
print(f"Unique NSID: {stage_2_sex_measure['NSID'].nunique():,}")
print(f"Duplicate NSID: {stage_2_sex_measure['NSID'].duplicated().sum():,}")
print(f"Participants with a sex value: {stage_2_sex_measure['Sex code'].notna().sum():,}")
print(f"Participants with a missing sex value: {stage_2_sex_measure['Sex code'].isna().sum():,}")
print(f"Participants with cross-wave disagreement: {stage_2_sex_measure['Sex cross-wave disagreement'].sum():,}")
print('\nSex distribution:')
print(stage_2_sex_measure['Sex'].fillna('Missing').value_counts().rename_axis('Sex').reset_index(name='Participants').to_string(max_rows=TABLE_ROW_LIMIT,
    index=False))
print('\nSource of retained value:')
print(stage_2_sex_measure['Sex source'].fillna('No valid source').value_counts().rename_axis('Source').reset_index(name='Participants').to_string(max_rows=TABLE_ROW_LIMIT,
    index=False))
print('\nParticipants without a valid sex record:', int(stage_2_sex_measure['Sex code'].isna().sum()))
# Participant identifiers are not displayed; the missing records remain in the in-memory review table.

Rows: 9,767
Unique NSID: 9,767
Duplicate NSID: 0
Participants with a sex value: 9,758
Participants with a missing sex value: 9
Participants with cross-wave disagreement: 71

Sex distribution:
    Sex  Participants
 Female          4979
   Male          4779
Missing             9

Source of retained value:
         Source  Participants
         Wave 1          9435
         Wave 4           237
         Wave 2            81
No valid source             9
         Wave 3             5

Participants without a valid sex record: 9


In [91]:
# 19: Sex-measure decision and documentation

sex_source_roles = {'W1sexYP': 'Primary source for the earliest available sex value.',
    'W2SexYP': 'Used only when no valid Wave 1 value is available.', 'W3sexYP': 'Used only when no valid Wave 1 or Wave 2 value is available.', 'W4SexYP': 'Used only when no valid earlier-wave value is available; also provides coverage for Wave 4 boost participants.'}
for variable, source_role in sex_source_roles.items():
    matching_rows = stage_2_variable_decision_register['Variable'].eq(variable)
    if matching_rows.sum() != 1:
        raise ValueError(f'Expected one decision-register entry for {variable}, found {matching_rows.sum()}.')
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = 'Retain as construction source'
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'Demographic background'
    stage_2_variable_decision_register.loc[matching_rows,
        'Decision reason'] = "Provides a measure of the young person's sex. Wave-specific records are consolidated into one participant-level measure using the earliest valid value."
    stage_2_variable_decision_register.loc[matching_rows,
        'Leakage assessment'] = 'No direct outcome leakage identified; stable demographic information.'
    stage_2_variable_decision_register.loc[matching_rows,
        'Reference-period assessment'] = 'Recorded at the relevant survey wave; treated as a stable background characteristic.'
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave-specific data dictionaries, Stata value labels and cross-wave consistency review'
    stage_2_variable_decision_register.loc[matching_rows,
        'Review notes'] = f'{source_role} Cross-wave disagreement was observed for 71 participants and is retained for quality control only. The disagreement indicator will not be used as a predictor.'
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
sex_measure_review_output_path = stage_2_output_directory / 'stage_2_sex_measure_review.csv'
stage_2_sex_measure.to_csv(sex_measure_review_output_path, index=False)
sex_decision_output = stage_2_variable_decision_register.loc[stage_2_variable_decision_register['Variable'].isin(sex_variables),
    ['Wave', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment',
    'Review notes']].sort_values('Wave')
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Participants in sex-measure file: {len(stage_2_sex_measure):,}')
print(f"Participants with a valid sex value: {stage_2_sex_measure['Sex code'].notna().sum():,}")
print(f"Participants with a missing sex value: {stage_2_sex_measure['Sex code'].isna().sum():,}")
print(f"Cross-wave disagreements retained for quality control: {stage_2_sex_measure['Sex cross-wave disagreement'].sum():,}")
print('\nSex-variable decisions:')
with pd.option_context('display.max_colwidth', 120, 'display.width', 300):
    print(sex_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'Sex-measure review file saved to: {sex_measure_review_output_path}')

Participants in sex-measure file: 9,767
Participants with a valid sex value: 9,758
Participants with a missing sex value: 9
Cross-wave disagreements retained for quality control: 71

Sex-variable decisions:
  Wave Variable                    Variable label                Review outcome                                                                                                                                         Decision reason                                                    Leakage assessment                                                                                                                                                                                                                                                                  Review notes
Wave 1  W1sexYP Admin: Interviewer code sex of YP Retain as construction source Provides a measure of the young person's sex. Wave-specific records are consolidated into one participant-level measure using the earliest va

In [92]:
# 20: Young-person birth-date variable list

import re
young_person_birth_pattern = re.compile("\\b(?:year|month|day|date) of birth of (?:the )?(?:yp|young person)\\b|\\b(?:yp|young person)(?:'s)? (?:year|month|day|date) of birth\\b",
    flags=re.IGNORECASE)
birth_name_mask = stage_2_master_variable_register['Variable'].astype(str).str.contains('dob', case=False, regex=False,
    na=False)
birth_label_mask = stage_2_master_variable_register['Variable label'].fillna('').apply(lambda label: bool(young_person_birth_pattern.search(str(label))))
stage_2_young_person_birth_review = stage_2_master_variable_register.loc[birth_name_mask | birth_label_mask,
    ['Source order', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable',
    'Variable label']].copy().sort_values(['Source order', 'Variable position']).reset_index(drop=True)
stage_2_young_person_birth_review = stage_2_young_person_birth_review.merge(stage_2_variable_decision_register[['Source file',
    'Variable', 'Review outcome']], on=['Source file', 'Variable'], how='left', validate='one_to_one')
print(f'Young-person birth-date variables identified: {len(stage_2_young_person_birth_review):,}')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 140, 'display.width', 300):
    print(stage_2_young_person_birth_review[['Wave', 'Source type', 'Variable', 'Variable label',
        'Review outcome']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Young-person birth-date variables identified: 2
  Wave  Source type   Variable               Variable label                Review outcome
Wave 1 Young person  DobyearYP  Edited: Year of birth of YP Retain as review support only
Wave 1 Young person DobmonthYP Edited: Month of birth of YP Retain as review support only


In [93]:
# 21: Longitudinal index birth-variable location

from pandas.io.stata import StataReader
from pathlib import Path
import pandas as pd
stata13_directory = source_directory.parent
target_birth_variables = {'hdobm', 'hdoby'}
possible_index_identifiers = {'nsid', 'surveyid', 'hhid', 'reltoyp', 'reltoyp2', 'age', 'sex'}
index_birth_file_records = []
metadata_read_errors = []
for source_path in sorted(stata13_directory.rglob('*.dta')):
    try:
        with StataReader(source_path, convert_categoricals=False) as reader:
            variable_labels = reader.variable_labels()
            first_row = reader.read(nrows=1)
        variables = list(first_row.columns)
        variable_lookup = {variable.lower(): variable for variable in variables}
        matched_birth_variables = [variable_lookup[target] for target in target_birth_variables if target in variable_lookup]
        if matched_birth_variables:
            matched_identifiers = [variable_lookup[target] for target in possible_index_identifiers if target in variable_lookup]
            for variable in matched_identifiers + matched_birth_variables:
                index_birth_file_records.append({'Source file': source_path.name, 'Source path': str(source_path),
                    'Variables in file': len(variables), 'Variable': variable, 'Variable label': variable_labels.get(variable,
                    ''), 'Variable role': 'Birth information' if variable.lower() in target_birth_variables else 'Possible identifier or record descriptor', 'Both birth variables present': target_birth_variables.issubset(variable_lookup.keys())})
    except Exception as error:
        metadata_read_errors.append({'Source file': source_path.name, 'Error': str(error)})
stage_2_index_birth_file_review = pd.DataFrame(index_birth_file_records)
if not stage_2_index_birth_file_review.empty:
    stage_2_index_birth_file_review = stage_2_index_birth_file_review.sort_values(['Source file', 'Variable role',
        'Variable']).reset_index(drop=True)
index_named_files = [path for path in sorted(stata13_directory.rglob('*.dta')) if 'index' in path.name.lower()]
print(f"Stata files searched: {len(list(stata13_directory.rglob('*.dta'))):,}")
print(f"Files containing Hdobm or Hdoby: {(stage_2_index_birth_file_review['Source file'].nunique() if not stage_2_index_birth_file_review.empty else 0):,}")
print(f'Metadata read errors: {len(metadata_read_errors):,}')
print('\nMatched files and variables:')
if stage_2_index_birth_file_review.empty:
    print('No exact Hdobm or Hdoby variables were found.')
else:
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 320):
        print(stage_2_index_birth_file_review.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print("\nStata files with 'index' in the filename:")
if index_named_files:
    for path in index_named_files:
        print(path)
else:
    print("No Stata filenames containing 'index' were found.")

Stata files searched: 53
Files containing Hdobm or Hdoby: 0
Metadata read errors: 0

Matched files and variables:
No exact Hdobm or Hdoby variables were found.

Stata files with 'index' in the filename:
No Stata filenames containing 'index' were found.


In [94]:
# 22: Birth-date variables in the Stata delivery

from pandas.io.stata import StataReader
import re
import pandas as pd
birth_variable_name_pattern = re.compile('dob', flags=re.IGNORECASE)
birth_variable_label_pattern = re.compile('\\bdate of birth\\b|\\bmonth of birth\\b|\\byear of birth\\b',
    flags=re.IGNORECASE)
birth_variable_records = []
metadata_read_errors = []
all_stata_files = sorted(stata13_directory.rglob('*.dta'))
for source_path in all_stata_files:
    try:
        with StataReader(source_path, convert_categoricals=False) as reader:
            variable_labels = reader.variable_labels()
            first_row = reader.read(nrows=1)
        for variable in first_row.columns:
            variable_label = variable_labels.get(variable, '')
            name_match = bool(birth_variable_name_pattern.search(str(variable)))
            label_match = bool(birth_variable_label_pattern.search(str(variable_label)))
            if name_match or label_match:
                birth_variable_records.append({'Source file': source_path.name,
                    'Source directory': source_path.parent.name, 'Source path': str(source_path), 'Variables in file': len(first_row.columns), 'Variable': variable, 'Variable label': variable_label, 'Matched by name': name_match, 'Matched by label': label_match, 'Data type': str(first_row[variable].dtype)})
    except Exception as error:
        metadata_read_errors.append({'Source file': source_path.name, 'Error': str(error)})
stage_2_delivery_birth_review = pd.DataFrame(birth_variable_records)
if not stage_2_delivery_birth_review.empty:
    stage_2_delivery_birth_review = stage_2_delivery_birth_review.sort_values(['Source directory', 'Source file',
        'Variable']).reset_index(drop=True)
print(f'Stata files searched: {len(all_stata_files):,}')
print(f"Files containing possible birth-date variables: {(stage_2_delivery_birth_review['Source path'].nunique() if not stage_2_delivery_birth_review.empty else 0):,}")
print(f'Possible birth-date variables identified: {len(stage_2_delivery_birth_review):,}')
print(f'Metadata read errors: {len(metadata_read_errors):,}')
print('\nVariables identified:')
if stage_2_delivery_birth_review.empty:
    print('No birth-date variables were identified.')
else:
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 340):
        print(stage_2_delivery_birth_review[['Source directory', 'Source file', 'Variable', 'Variable label',
            'Matched by name', 'Matched by label']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Stata files searched: 53
Files containing possible birth-date variables: 9
Possible birth-date variables identified: 29
Metadata read errors: 0

Variables identified:
Source directory                          Source file   Variable                   Variable label  Matched by name  Matched by label
 household_grids       ns8_2015_household_members.dta  W8HHMDOBM Other HH member month of birth               True              True
 household_grids       ns8_2015_household_members.dta  W8HHMDOBY Other HH member year of birth                True              True
             ...                                  ...        ...                              ...              ...               ...
 safeguarded_eul wave_one_lsype_young_person_2020.dta DobmonthYP     Edited: Month of birth of YP             True              True
 safeguarded_eul wave_one_lsype_young_person_2020.dta  DobyearYP      Edited: Year of birth of YP             True              True


In [95]:
# 23: Joint review of year and month of birth

birth_variables = ['DobyearYP', 'DobmonthYP']
birth_source_matches = stage_2_master_variable_register.loc[stage_2_master_variable_register['Variable'].isin(birth_variables),
    ['Source file', 'Source path', 'Variable']].drop_duplicates()
variables_found = set(birth_source_matches['Variable'])
if variables_found != set(birth_variables):
    raise ValueError(f'The expected birth variables were not both found. Variables found: {sorted(variables_found)}')
birth_source_paths = birth_source_matches['Source path'].drop_duplicates().tolist()
if len(birth_source_paths) != 1:
    raise ValueError(f'Expected one source file for DobyearYP and DobmonthYP, but found {len(birth_source_paths)}.')
birth_source_path = Path(birth_source_paths[0])
wave_1_birth_data = pd.read_stata(birth_source_path, columns=['NSID', 'DobyearYP', 'DobmonthYP'],
    convert_categoricals=False)
stage_2_birth_review_data = stage_2_sex_measure[['NSID']].merge(wave_1_birth_data, on='NSID', how='left',
    validate='one_to_one', indicator=True)
stage_2_birth_review_data['Wave 1 birth record'] = np.where(stage_2_birth_review_data['_merge'].eq('both'), 'Present',
    'Absent')
stage_2_birth_review_data = stage_2_birth_review_data.drop(columns='_merge')
stage_2_birth_review_data['Valid birth year'] = stage_2_birth_review_data['DobyearYP'].gt(0)
stage_2_birth_review_data['Valid birth month'] = stage_2_birth_review_data['DobmonthYP'].between(1, 12,
    inclusive='both')
stage_2_birth_review_data['Birth-information status'] = np.select([stage_2_birth_review_data['Wave 1 birth record'].eq('Absent'),
    stage_2_birth_review_data['Valid birth year'] & stage_2_birth_review_data['Valid birth month'], stage_2_birth_review_data['DobyearYP'].lt(0) | stage_2_birth_review_data['DobmonthYP'].lt(0), stage_2_birth_review_data['DobyearYP'].isna() | stage_2_birth_review_data['DobmonthYP'].isna()], ['No Wave 1 record',
    'Valid year and month', 'Negative survey code', 'Incomplete birth information'], default='Other value requiring review')
birth_status_summary = stage_2_birth_review_data['Birth-information status'].value_counts().rename_axis('Birth-information status').reset_index(name='Participants')
birth_year_summary = stage_2_birth_review_data['DobyearYP'].value_counts(dropna=False).sort_index().rename_axis('Year of birth').reset_index(name='Participants')
birth_month_summary = stage_2_birth_review_data['DobmonthYP'].value_counts(dropna=False).sort_index().rename_axis('Month of birth').reset_index(name='Participants')
valid_birth_records = stage_2_birth_review_data.loc[stage_2_birth_review_data['Birth-information status'].eq('Valid year and month')]
birth_year_month_table = pd.crosstab(valid_birth_records['DobyearYP'].astype(int),
    valid_birth_records['DobmonthYP'].astype(int), margins=True, margins_name='Total')
print(f'Birth-information source file: {birth_source_path.name}')
print(f'Stage 2 participants: {len(stage_2_birth_review_data):,}')
print(f"Wave 1 birth records present: {stage_2_birth_review_data['Wave 1 birth record'].eq('Present').sum():,}")
print(f"Wave 1 birth records absent: {stage_2_birth_review_data['Wave 1 birth record'].eq('Absent').sum():,}")
print('\nBirth-information status:')
print(birth_status_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nYear-of-birth distribution:')
print(birth_year_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMonth-of-birth distribution:')
print(birth_month_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nYear-by-month distribution:')
print(birth_year_month_table.to_string(max_rows=TABLE_ROW_LIMIT))

Birth-information source file: wave_one_lsype_young_person_2020.dta
Stage 2 participants: 9,767
Wave 1 birth records present: 9,524
Wave 1 birth records absent: 243

Birth-information status:
Birth-information status  Participants
    Valid year and month          9524
        No Wave 1 record           243

Year-of-birth distribution:
 Year of birth  Participants
        1988.0             2
        1989.0          3065
        1990.0          6452
        1991.0             5
           NaN           243

Month-of-birth distribution:
 Month of birth  Participants
            1.0           778
            2.0           765
            ...           ...
           12.0           759
            NaN           243

Year-by-month distribution:
DobmonthYP    1    2    3    4    5    6    7    8    9   10   11   12  Total
DobyearYP                                                                    
1988          0    0    0    0    0    0    0    0    0    1    0    1      2
1989          1

In [96]:
# 24: Review of the documented cohort birth window

birth_year_numeric = pd.to_numeric(stage_2_birth_review_data['DobyearYP'], errors='coerce')
birth_month_numeric = pd.to_numeric(stage_2_birth_review_data['DobmonthYP'], errors='coerce')
birth_calendar_month = birth_year_numeric * 12 + birth_month_numeric
cohort_start_month = 1989 * 12 + 9
cohort_end_month = 1990 * 12 + 8
has_valid_birth_information = stage_2_birth_review_data['Birth-information status'].eq('Valid year and month')
within_documented_window = has_valid_birth_information & birth_calendar_month.between(cohort_start_month,
    cohort_end_month, inclusive='both')
before_documented_window = has_valid_birth_information & birth_calendar_month.lt(cohort_start_month)
after_documented_window = has_valid_birth_information & birth_calendar_month.gt(cohort_end_month)
stage_2_birth_review_data['Documented birth-window status'] = np.select([within_documented_window,
    before_documented_window, after_documented_window, ~has_valid_birth_information], ['Within documented window',
    'Before documented window', 'After documented window', 'Birth information unavailable'], default='Check required')
stage_2_birth_review_data['Months outside documented window'] = np.select([before_documented_window,
    after_documented_window, within_documented_window], [cohort_start_month - birth_calendar_month,
    birth_calendar_month - cohort_end_month, 0], default=np.nan)
birth_window_summary = stage_2_birth_review_data['Documented birth-window status'].value_counts().rename_axis('Documented birth-window status').reset_index(name='Participants')
outside_birth_window_records = stage_2_birth_review_data.loc[before_documented_window | after_documented_window,
    ['NSID', 'DobyearYP', 'DobmonthYP', 'Documented birth-window status',
    'Months outside documented window']].copy().sort_values(['DobyearYP', 'DobmonthYP',
    'NSID']).reset_index(drop=True)
outside_birth_month_summary = outside_birth_window_records.groupby(['DobyearYP', 'DobmonthYP',
    'Documented birth-window status', 'Months outside documented window'], dropna=False).size().reset_index(name='Participants').sort_values(['DobyearYP',
    'DobmonthYP'])
print(f'Participants with valid birth information: {has_valid_birth_information.sum():,}')
print(f'Participants within the documented birth window: {within_documented_window.sum():,}')
print(f'Participants outside the documented birth window: {len(outside_birth_window_records):,}')
print('\nBirth-window status:')
print(birth_window_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nExact birth-month combinations outside the window:')
print(outside_birth_month_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nParticipant-level records outside the window are retained in memory but are not displayed.')

Participants with valid birth information: 9,524
Participants within the documented birth window: 9,476
Participants outside the documented birth window: 48

Birth-window status:
Documented birth-window status  Participants
      Within documented window          9476
 Birth information unavailable           243
      Before documented window            28
       After documented window            20

Exact birth-month combinations outside the window:
 DobyearYP  DobmonthYP Documented birth-window status  Months outside documented window  Participants
    1988.0        10.0       Before documented window                              11.0             1
    1988.0        12.0       Before documented window                               9.0             1
       ...         ...                            ...                               ...           ...
    1991.0         5.0        After documented window                               9.0             2
    1991.0         7.0        Afte

In [97]:
# 25: Age and interview-date review list

import re
young_person_wave_mask = stage_2_master_variable_register['Source type'].eq('Young person') & stage_2_master_variable_register['Wave'].isin(['Wave 1',
    'Wave 2', 'Wave 3', 'Wave 4'])
age_interview_label_pattern = re.compile("\\bage (?:of )?(?:the )?(?:yp|young person) (?:at|on) (?:the )?interview\\b|\\b(?:yp|young person)(?:'s)? age (?:at|on) (?:the )?interview\\b|\\bage at (?:date of )?interview\\b|\\bdate of interview\\b|\\binterview date\\b|\\bmonth of interview\\b|\\byear of interview\\b",
    flags=re.IGNORECASE)
age_interview_name_pattern = re.compile('ageyp|ypage|intdate|intdat|intmonth|intyear', flags=re.IGNORECASE)
candidate_register = stage_2_master_variable_register.loc[young_person_wave_mask].copy()
label_match = candidate_register['Variable label'].fillna('').apply(lambda label: bool(age_interview_label_pattern.search(str(label))))
name_match = candidate_register['Variable'].fillna('').apply(lambda variable: bool(age_interview_name_pattern.search(str(variable))))
stage_2_age_interview_review = candidate_register.loc[label_match | name_match, ['Source order', 'Wave', 'Source type',
    'Source file', 'Source path', 'Variable position', 'Variable', 'Variable label', 'Data type']].copy()
stage_2_age_interview_review['Matched by name'] = name_match.loc[label_match | name_match].values
stage_2_age_interview_review['Matched by label'] = label_match.loc[label_match | name_match].values
stage_2_age_interview_review = stage_2_age_interview_review.sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
print(f'Possible age or interview-date variables identified: {len(stage_2_age_interview_review):,}')
print('\nVariables identified:')
if stage_2_age_interview_review.empty:
    print('No possible age or interview-date variables were identified.')
else:
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 340):
        print(stage_2_age_interview_review[['Wave', 'Variable', 'Variable label', 'Matched by name',
            'Matched by label']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Possible age or interview-date variables identified: 7

Variables identified:
  Wave     Variable                                                                Variable label  Matched by name  Matched by label
Wave 1  W1quawageYP YP: How much this matters in deciding on a job: To have a job which pays well             True             False
Wave 2 w2intmonthMP                                                  Admin: MP month of interview             True              True
   ...          ...                                                                           ...              ...               ...
Wave 4   W4intmonth                                                  Admin: Month of W4 interview             True             False
Wave 4    W4intyear                                                   Admin: Year of W4 interview             True             False


In [98]:
# 26: Birth-information decision and construction

reference_birth_month = 1989 * 12 + 9
valid_birth_mask = stage_2_birth_review_data['Birth-information status'].eq('Valid year and month')
birth_month_position = pd.to_numeric(stage_2_birth_review_data['DobyearYP'],
    errors='coerce') * 12 + pd.to_numeric(stage_2_birth_review_data['DobmonthYP'],
    errors='coerce') - reference_birth_month
stage_2_birth_review_data['birth_month_position'] = birth_month_position.where(valid_birth_mask).astype('Int64')
stage_2_birth_measure = stage_2_birth_review_data[['NSID', 'DobyearYP', 'DobmonthYP', 'birth_month_position',
    'Documented birth-window status', 'Months outside documented window']].copy()
birth_source_variables = {'DobyearYP': "Used jointly with DobmonthYP to construct the participant's ordered birth-month position.",
    'DobmonthYP': "Used jointly with DobyearYP to construct the participant's ordered birth-month position."}
for variable, source_role in birth_source_variables.items():
    matching_rows = stage_2_variable_decision_register['Variable'].eq(variable)
    if matching_rows.sum() != 1:
        raise ValueError(f'Expected one decision-register entry for {variable}, found {matching_rows.sum()}.')
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = 'Retain as construction source'
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'Demographic background'
    stage_2_variable_decision_register.loc[matching_rows,
        'Decision reason'] = 'Provides birth-timing information. Birth year and birth month are combined into one ordered month measure rather than retained as two separate predictors.'
    stage_2_variable_decision_register.loc[matching_rows,
        'Leakage assessment'] = 'No direct outcome leakage identified; birth predates all survey waves and the outcome.'
    stage_2_variable_decision_register.loc[matching_rows,
        'Reference-period assessment'] = "Stable historical characteristic referring to the participant's birth."
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave 1 young-person data dictionary, edited variable labels and cohort-window review'
    stage_2_variable_decision_register.loc[matching_rows,
        'Review notes'] = f'{source_role} Valid information is available for 9,524 participants and unavailable for 243 participants. Forty-eight records fall outside the documented cohort window but are retained as recorded because no independent information supports correction. The window-status field is retained for quality control only.'
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
birth_measure_review_output_path = stage_2_output_directory / 'stage_2_birth_measure_review.csv'
stage_2_birth_measure.to_csv(birth_measure_review_output_path, index=False)
birth_position_summary = stage_2_birth_measure['birth_month_position'].describe()
birth_variable_decisions = stage_2_variable_decision_register.loc[stage_2_variable_decision_register['Variable'].isin(birth_source_variables),
    ['Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment', 'Review notes']]
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Participants in birth-measure file: {len(stage_2_birth_measure):,}')
print(f"Participants with a birth-month position: {stage_2_birth_measure['birth_month_position'].notna().sum():,}")
print(f"Participants with missing birth information: {stage_2_birth_measure['birth_month_position'].isna().sum():,}")
print(f"Records outside the documented window: {stage_2_birth_measure['Documented birth-window status'].isin(['Before documented window', 'After documented window']).sum():,}")
print('\nBirth-month-position summary:')
print(birth_position_summary.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nBirth-variable decisions:')
with pd.option_context('display.max_colwidth', 130, 'display.width', 320):
    print(birth_variable_decisions.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'Birth-measure review file saved to: {birth_measure_review_output_path}')

Participants in birth-measure file: 9,767
Participants with a birth-month position: 9,524
Participants with missing birth information: 243
Records outside the documented window: 48

Birth-month-position summary:
count      9524.0
mean     5.599118
           ...   
75%           9.0
max          22.0

Birth-variable decisions:
  Variable               Variable label                Review outcome                                                                                                                                            Decision reason                                                                     Leakage assessment                                                                                                                                                                                                                                                                                                                                                                        

In [99]:
# 27: Young-person ethnicity variable list

import re
ethnicity_name_pattern = re.compile('ethgrp|ethnic|ethnicity', flags=re.IGNORECASE)
ethnicity_label_pattern = re.compile('\\bethnic group\\b|\\bethnicity\\b|\\bethnic background\\b',
    flags=re.IGNORECASE)
ethnicity_register = stage_2_master_variable_register.copy()
name_match = ethnicity_register['Variable'].fillna('').apply(lambda value: bool(ethnicity_name_pattern.search(str(value))))
label_match = ethnicity_register['Variable label'].fillna('').apply(lambda value: bool(ethnicity_label_pattern.search(str(value))))
stage_2_ethnicity_review = ethnicity_register.loc[name_match | label_match, ['Source order', 'Wave', 'Source type',
    'Source file', 'Source path', 'Variable position', 'Variable', 'Variable label', 'Data type']].copy()
direct_yp_label_pattern = re.compile('(?:ethnic group|ethnicity|ethnic background).*(?:\\byp\\b|young person)|(?:\\byp\\b|young person).*(?:ethnic group|ethnicity|ethnic background)',
    flags=re.IGNORECASE)
direct_yp_name_pattern = re.compile('(?:ethgrp|ethnic|ethnicity).*yp$', flags=re.IGNORECASE)
stage_2_ethnicity_review['Review category'] = np.where(stage_2_ethnicity_review['Variable label'].fillna('').apply(lambda value: bool(direct_yp_label_pattern.search(str(value)))) | stage_2_ethnicity_review['Variable'].fillna('').apply(lambda value: bool(direct_yp_name_pattern.search(str(value)))),
    'Possible direct young-person ethnicity measure', 'Other ethnicity-related variable')
stage_2_ethnicity_review = stage_2_ethnicity_review.merge(stage_2_variable_decision_register[['Source file',
    'Variable', 'Review outcome']], on=['Source file',
    'Variable'], how='left', validate='one_to_one').sort_values(['Review category', 'Source order',
    'Variable position']).reset_index(drop=True)
ethnicity_review_summary = stage_2_ethnicity_review['Review category'].value_counts().rename_axis('Review category').reset_index(name='Variables')
print(f'Ethnicity-related variables identified: {len(stage_2_ethnicity_review):,}')
print('\nReview-category summary:')
print(ethnicity_review_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables identified:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 340):
    print(stage_2_ethnicity_review[['Wave', 'Source type', 'Variable', 'Variable label', 'Review category',
        'Review outcome']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Ethnicity-related variables identified: 43

Review-category summary:
                               Review category  Variables
              Other ethnicity-related variable         32
Possible direct young-person ethnicity measure         11

Variables identified:
  Wave       Source type    Variable                                                                  Variable label                                Review category Review outcome
Wave 1 Family background W1pethniCMP                                     MP: Ethnic origin of main parent respondent               Other ethnicity-related variable Pending review
Wave 1 Family background W1pethnicSP                                   SP: Ethnic origin of second parent respondent               Other ethnicity-related variable Pending review
   ...               ...         ...                                                                             ...                                            ...            ...
Wave 4      Young 

In [100]:
# 28: Direct young-person ethnicity measure review

direct_ethnicity_variable_names = {'w1ethnicyp', 'w1ethgrpyp', 'w1ethnic2yp', 'w2ethnicyp', 'w2ethgrpyp',
    'w4ethnic2yp', 'w4ethgrpyp'}
direct_ethnicity_entries = stage_2_master_variable_register.loc[stage_2_master_variable_register['Variable'].str.lower().isin(direct_ethnicity_variable_names),
    ['Source order', 'Wave', 'Source type', 'Source file', 'Source path', 'Variable position', 'Variable',
    'Variable label']].copy().sort_values(['Source order', 'Variable position']).reset_index(drop=True)
direct_ethnicity_entries['Source entries for variable'] = direct_ethnicity_entries.groupby(direct_ethnicity_entries['Variable'].str.lower())['Source file'].transform('count')
ethnicity_value_records = []
ethnicity_coverage_records = []
stage_2_participant_ids = stage_2_sex_measure[['NSID']].copy()
for _, entry in direct_ethnicity_entries.iterrows():
    source_path = Path(entry['Source path'])
    variable = entry['Variable']
    numeric_data = pd.read_stata(source_path, columns=['NSID', variable], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', variable], convert_categoricals=True)
    source_value_data = numeric_data.copy()
    source_value_data['Value label'] = labelled_data[variable].astype('string')
    sample_value_data = stage_2_participant_ids.merge(source_value_data, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = sample_value_data['_merge'].eq('both')
    numeric_values = pd.to_numeric(sample_value_data[variable], errors='coerce')
    ethnicity_coverage_records.append({'Wave': entry['Wave'], 'Source type': entry['Source type'],
        'Source file': entry['Source file'], 'Variable': variable, 'Variable label': entry['Variable label'], 'Stage 2 participants': len(sample_value_data), 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Non-missing stored value': int(sample_value_data[variable].notna().sum()), 'Negative survey code': int(numeric_values.lt(0).sum()), 'Non-negative stored value': int(numeric_values.ge(0).sum()), 'Distinct stored codes': int(numeric_values.nunique(dropna=True))})
    value_summary = sample_value_data.loc[source_record_present, [variable,
        'Value label']].value_counts(dropna=False).rename('Participants').reset_index().rename(columns={variable: 'Value code'})
    value_summary.insert(0, 'Variable', variable)
    value_summary.insert(0, 'Source file', entry['Source file'])
    value_summary.insert(0, 'Wave', entry['Wave'])
    ethnicity_value_records.append(value_summary)
stage_2_direct_ethnicity_coverage = pd.DataFrame(ethnicity_coverage_records)
stage_2_direct_ethnicity_values = pd.concat(ethnicity_value_records,
    ignore_index=True) if ethnicity_value_records else pd.DataFrame()
print(f'Direct ethnicity source entries identified: {len(direct_ethnicity_entries):,}')
print(f"Distinct variable names: {direct_ethnicity_entries['Variable'].str.lower().nunique():,}")
print('\nSource entries:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 140, 'display.width', 340):
    print(direct_ethnicity_entries[['Wave', 'Source type', 'Source file', 'Variable', 'Variable label',
        'Source entries for variable']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCoverage and stored-code summary:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 120, 'display.width', 340):
    print(stage_2_direct_ethnicity_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nValue codes and labels:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 130, 'display.width', 300):
    print(stage_2_direct_ethnicity_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Direct ethnicity source entries identified: 9
Distinct variable names: 7

Source entries:
  Wave  Source type                       Source file    Variable                                                                  Variable label  Source entries for variable
Wave 1 Young person  wave_one_lsype_young_person_2020  W1ethnicYP                                            YP: Ethnic origin (self-designation)                            1
Wave 1 Young person  wave_one_lsype_young_person_2020  W1ethgrpYP                                       DV: Young person's ethnic group (grouped)                            2
   ...          ...                               ...         ...                                                                             ...                          ...
Wave 4 Young person wave_four_lsype_young_person_2020 w4ethnic2YP DV: Young person's ethnic group (detailed) Source: Main W1 YP, Boost W4 YP & HH                            1
Wave 4 Young person wave_four_lsype

In [101]:
# 29: Grouped ethnicity consistency review

grouped_ethnicity_sources = [{'Wave': 'Wave 1', 'Source type': 'Young person', 'Variable': 'W1ethgrpYP',
    'Output name': 'W1 grouped ethnicity - young person file'}, {'Wave': 'Wave 1', 'Source type': 'Family background',
    'Variable': 'W1ethgrpYP', 'Output name': 'W1 grouped ethnicity - family file'}, {'Wave': 'Wave 2',
    'Source type': 'Young person', 'Variable': 'W2ethgrpYP', 'Output name': 'W2 grouped ethnicity - young person file'}, {'Wave': 'Wave 2',
    'Source type': 'Family background', 'Variable': 'W2ethgrpYP', 'Output name': 'W2 grouped ethnicity - family file'}, {'Wave': 'Wave 4',
    'Source type': 'Young person', 'Variable': 'W4ethgrpYP', 'Output name': 'W4 grouped ethnicity'}]
grouped_ethnicity_data = stage_2_sex_measure[['NSID']].copy()
for source_specification in grouped_ethnicity_sources:
    matching_entry = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq(source_specification['Wave']) & stage_2_master_variable_register['Source type'].eq(source_specification['Source type']) & stage_2_master_variable_register['Variable'].str.lower().eq(source_specification['Variable'].lower())]
    if len(matching_entry) != 1:
        raise ValueError(f'Expected one source entry for {source_specification}, but found {len(matching_entry)}.')
    source_path = Path(matching_entry.iloc[0]['Source path'])
    source_variable = matching_entry.iloc[0]['Variable']
    source_data = pd.read_stata(source_path, columns=['NSID', source_variable],
        convert_categoricals=False).rename(columns={source_variable: source_specification['Output name']})
    grouped_ethnicity_data = grouped_ethnicity_data.merge(source_data, on='NSID', how='left', validate='one_to_one')
w1_yp = 'W1 grouped ethnicity - young person file'
w1_family = 'W1 grouped ethnicity - family file'
w2_yp = 'W2 grouped ethnicity - young person file'
w2_family = 'W2 grouped ethnicity - family file'
w4_grouped = 'W4 grouped ethnicity'
valid_grouped_ethnicity = {}
for column in [w1_yp, w1_family, w2_yp, w2_family, w4_grouped]:
    valid_grouped_ethnicity[column] = pd.to_numeric(grouped_ethnicity_data[column], errors='coerce').between(1, 8,
        inclusive='both')
duplicate_comparison_records = []
for wave, first_column, second_column in [('Wave 1', w1_yp, w1_family), ('Wave 2', w2_yp, w2_family)]:
    both_stored = grouped_ethnicity_data[first_column].notna() & grouped_ethnicity_data[second_column].notna()
    stored_values_equal = grouped_ethnicity_data[first_column].eq(grouped_ethnicity_data[second_column])
    duplicate_comparison_records.append({'Wave': wave, 'Both source values stored': int(both_stored.sum()),
        'Identical stored values': int((both_stored & stored_values_equal).sum()), 'Different stored values': int((both_stored & ~stored_values_equal).sum()), 'Value stored in one file only': int(grouped_ethnicity_data[first_column].notna().ne(grouped_ethnicity_data[second_column].notna()).sum())})
duplicate_ethnicity_summary = pd.DataFrame(duplicate_comparison_records)
cross_wave_comparison_records = []
for comparison, first_column, second_column in [('Wave 1 versus Wave 2', w1_yp, w2_yp), ('Wave 1 versus Wave 4', w1_yp,
    w4_grouped), ('Wave 2 versus Wave 4', w2_yp, w4_grouped)]:
    both_valid = valid_grouped_ethnicity[first_column] & valid_grouped_ethnicity[second_column]
    agreement = grouped_ethnicity_data[first_column].eq(grouped_ethnicity_data[second_column])
    valid_comparisons = int(both_valid.sum())
    agreeing_values = int((both_valid & agreement).sum())
    cross_wave_comparison_records.append({'Comparison': comparison, 'Both values valid': valid_comparisons,
        'Values agreeing': agreeing_values, 'Values disagreeing': int(valid_comparisons - agreeing_values), 'Agreement percentage': agreeing_values / valid_comparisons * 100 if valid_comparisons else np.nan})
cross_wave_ethnicity_summary = pd.DataFrame(cross_wave_comparison_records)
w1_valid = valid_grouped_ethnicity[w1_yp]
w2_valid = valid_grouped_ethnicity[w2_yp]
w4_valid = valid_grouped_ethnicity[w4_grouped]
ethnicity_coverage_summary = pd.DataFrame([{'Coverage measure': 'Valid Wave 1 grouped value',
    'Participants': int(w1_valid.sum())}, {'Coverage measure': 'Valid Wave 2 grouped value',
    'Participants': int(w2_valid.sum())}, {'Coverage measure': 'Valid Wave 4 grouped value',
    'Participants': int(w4_valid.sum())}, {'Coverage measure': 'Wave 4 valid where Wave 1 is unavailable',
    'Participants': int((w4_valid & ~w1_valid).sum())}, {'Coverage measure': 'Wave 1 or Wave 2 valid where Wave 4 is unavailable',
    'Participants': int((~w4_valid & (w1_valid | w2_valid)).sum())}, {'Coverage measure': 'No valid grouped value in Waves 1, 2 or 4',
    'Participants': int((~w1_valid & ~w2_valid & ~w4_valid).sum())}])
w1_w4_disagreement_records = grouped_ethnicity_data.loc[w1_valid & w4_valid & grouped_ethnicity_data[w1_yp].ne(grouped_ethnicity_data[w4_grouped]),
    ['NSID', w1_yp, w2_yp, w4_grouped]].copy().sort_values('NSID').reset_index(drop=True)
print(f'Stage 2 participants: {len(grouped_ethnicity_data):,}')
print('\nDuplicate source-file comparison:')
print(duplicate_ethnicity_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCross-wave consistency:')
print(cross_wave_ethnicity_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Agreement percentage': lambda value: f'{value:.2f}%'}))
print('\nCoverage comparison:')
print(ethnicity_coverage_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nParticipants with differing valid Wave 1 and Wave 4 grouped values: {len(w1_w4_disagreement_records):,}')
print('\nWave 1–Wave 4 disagreement records:')
if w1_w4_disagreement_records.empty:
    print('No valid disagreements identified.')
else:
    print(w1_w4_disagreement_records.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Stage 2 participants: 9,767

Duplicate source-file comparison:
  Wave  Both source values stored  Identical stored values  Different stored values  Value stored in one file only
Wave 1                       9524                     9524                        0                              0
Wave 2                       9521                     9521                        0                              0

Cross-wave consistency:
          Comparison  Both values valid  Values agreeing  Values disagreeing Agreement percentage
Wave 1 versus Wave 2               9420             8910                 510               94.59%
Wave 1 versus Wave 4               9502             9502                   0              100.00%
Wave 2 versus Wave 4               9409             8901                 508               94.60%

Coverage comparison:
                                  Coverage measure  Participants
                        Valid Wave 1 grouped value          9513
                       

In [102]:
# 30: Consolidated grouped ethnicity review

w1_grouped_numeric = pd.to_numeric(grouped_ethnicity_data[w1_yp], errors='coerce')
w2_grouped_numeric = pd.to_numeric(grouped_ethnicity_data[w2_yp], errors='coerce')
w4_grouped_numeric = pd.to_numeric(grouped_ethnicity_data[w4_grouped], errors='coerce')
w1_grouped_valid = w1_grouped_numeric.where(w1_grouped_numeric.between(1, 8, inclusive='both'))
w2_grouped_valid = w2_grouped_numeric.where(w2_grouped_numeric.between(1, 8, inclusive='both'))
w4_grouped_valid = w4_grouped_numeric.where(w4_grouped_numeric.between(1, 8, inclusive='both'))
grouped_ethnicity_data['Ethnicity code'] = w4_grouped_valid.combine_first(w1_grouped_valid).combine_first(w2_grouped_valid)
grouped_ethnicity_data['Ethnicity source'] = np.select([w4_grouped_valid.notna(),
    w4_grouped_valid.isna() & w1_grouped_valid.notna(), w4_grouped_valid.isna() & w1_grouped_valid.isna() & w2_grouped_valid.notna()], ['Wave 4 harmonised grouped measure',
    'Wave 1 grouped measure', 'Wave 2 grouped measure'], default='No valid source')
ethnicity_label_map = {1.0: 'White', 2.0: 'Mixed', 3.0: 'Indian', 4.0: 'Pakistani', 5.0: 'Bangladeshi',
    6.0: 'Black Caribbean', 7.0: 'Black African', 8.0: 'Other'}
grouped_ethnicity_data['Ethnicity'] = grouped_ethnicity_data['Ethnicity code'].map(ethnicity_label_map)
stage_2_ethnicity_measure = grouped_ethnicity_data[['NSID', 'Ethnicity code', 'Ethnicity', 'Ethnicity source']].copy()
ethnicity_source_summary = stage_2_ethnicity_measure['Ethnicity source'].value_counts().rename_axis('Ethnicity source').reset_index(name='Participants')
ethnicity_distribution_summary = stage_2_ethnicity_measure['Ethnicity'].fillna('Missing').value_counts().rename_axis('Ethnicity').reset_index(name='Participants')
fallback_ethnicity_records = grouped_ethnicity_data.loc[grouped_ethnicity_data['Ethnicity source'].isin(['Wave 1 grouped measure',
    'Wave 2 grouped measure']), ['NSID', w1_yp, w2_yp, w4_grouped, 'Ethnicity code', 'Ethnicity',
    'Ethnicity source']].copy().sort_values(['Ethnicity source', 'NSID']).reset_index(drop=True)
missing_ethnicity_records = grouped_ethnicity_data.loc[grouped_ethnicity_data['Ethnicity code'].isna(), ['NSID', w1_yp,
    w2_yp, w4_grouped, 'Ethnicity source']].copy()
print(f'Stage 2 participants: {len(stage_2_ethnicity_measure):,}')
print(f"Participants with a grouped ethnicity value: {stage_2_ethnicity_measure['Ethnicity code'].notna().sum():,}")
print(f"Participants with missing grouped ethnicity: {stage_2_ethnicity_measure['Ethnicity code'].isna().sum():,}")
print('\nSource of retained ethnicity value:')
print(ethnicity_source_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nGrouped ethnicity distribution:')
print(ethnicity_distribution_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nFallback-record summary:')
print(
    fallback_ethnicity_records['Ethnicity source']
    .value_counts(dropna=False)
    .rename_axis('Ethnicity source')
    .reset_index(name='Participants')
    .to_string(index=False)
)
print(
    '\nParticipants without a valid grouped ethnicity value:',
    len(missing_ethnicity_records),
)
# Participant-level fallback records are retained for checking but are not displayed.

Stage 2 participants: 9,767
Participants with a grouped ethnicity value: 9,765
Participants with missing grouped ethnicity: 2

Source of retained ethnicity value:
                 Ethnicity source  Participants
Wave 4 harmonised grouped measure          9743
           Wave 1 grouped measure            11
           Wave 2 grouped measure            11
                  No valid source             2

Grouped ethnicity distribution:
Ethnicity  Participants
    White          6637
   Indian           689
      ...           ...
    Other           242
  Missing             2

Fallback-record summary:
      Ethnicity source  Participants
Wave 1 grouped measure            11
Wave 2 grouped measure            11

Participants without a valid grouped ethnicity value: 2


In [103]:
# 31: Review of unresolved and conflicting ethnicity records

fallback_disagreement_mask = w4_grouped_valid.isna() & w1_grouped_valid.notna() & w2_grouped_valid.notna() & w1_grouped_valid.ne(w2_grouped_valid)
no_valid_grouped_mask = grouped_ethnicity_data['Ethnicity code'].isna()
ethnicity_case_ids = grouped_ethnicity_data.loc[fallback_disagreement_mask | no_valid_grouped_mask,
    'NSID'].drop_duplicates().tolist()
ethnicity_review_sources = [{'Wave': 'Wave 1', 'Source type': 'Young person', 'Variables': ['W1ethnicYP', 'W1ethgrpYP',
    'W1ethnic2YP']}, {'Wave': 'Wave 2', 'Source type': 'Young person', 'Variables': ['W2ethnicYP',
    'W2ethgrpYP']}, {'Wave': 'Wave 4', 'Source type': 'Young person', 'Variables': ['w4ethnic2YP', 'W4ethgrpYP']}]
ethnicity_case_review = pd.DataFrame({'NSID': ethnicity_case_ids})
for source_specification in ethnicity_review_sources:
    source_entries = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq(source_specification['Wave']) & stage_2_master_variable_register['Source type'].eq(source_specification['Source type']) & stage_2_master_variable_register['Variable'].str.lower().isin([variable.lower() for variable in source_specification['Variables']])]
    if len(source_entries) != len(source_specification['Variables']):
        raise ValueError(f"Could not locate every requested ethnicity variable for {source_specification['Wave']}.")
    source_paths = source_entries['Source path'].drop_duplicates().tolist()
    if len(source_paths) != 1:
        raise ValueError(f"Expected one source file for {source_specification['Wave']}, but found {len(source_paths)}.")
    source_path = Path(source_paths[0])
    source_variables = source_entries['Variable'].tolist()
    numeric_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    numeric_data = numeric_data.loc[numeric_data['NSID'].isin(ethnicity_case_ids)].copy()
    labelled_data = labelled_data.loc[labelled_data['NSID'].isin(ethnicity_case_ids)].copy()
    source_review = numeric_data[['NSID']].copy()
    for variable in source_variables:
        source_review[f'{variable} code'] = numeric_data[variable].values
        source_review[f'{variable} label'] = labelled_data[variable].astype('string').values
    ethnicity_case_review = ethnicity_case_review.merge(source_review, on='NSID', how='left', validate='one_to_one')
ethnicity_case_review = ethnicity_case_review.merge(stage_2_ethnicity_measure[['NSID', 'Ethnicity code', 'Ethnicity',
    'Ethnicity source']], on='NSID', how='left', validate='one_to_one').sort_values('NSID').reset_index(drop=True)
print(f'Participants requiring direct ethnicity review: {len(ethnicity_case_review):,}')
# The detailed participant-level comparison is retained in memory and is not displayed in the public notebook.

Participants requiring direct ethnicity review: 4


In [104]:
# 32: Resolution of conflicting ethnicity records

conflicting_ethnicity_ids = grouped_ethnicity_data.loc[fallback_disagreement_mask, 'NSID'].drop_duplicates().tolist()
stage_2_ethnicity_measure_review = stage_2_ethnicity_measure.copy()
conflict_mask = stage_2_ethnicity_measure_review['NSID'].isin(conflicting_ethnicity_ids)
stage_2_ethnicity_measure_review.loc[conflict_mask, 'Ethnicity code'] = np.nan
stage_2_ethnicity_measure_review.loc[conflict_mask, 'Ethnicity'] = pd.NA
stage_2_ethnicity_measure_review.loc[conflict_mask, 'Ethnicity source'] = 'Unresolved Wave 1–Wave 2 conflict'
ethnicity_source_summary = stage_2_ethnicity_measure_review['Ethnicity source'].value_counts().rename_axis('Ethnicity source').reset_index(name='Participants')
ethnicity_distribution = stage_2_ethnicity_measure_review['Ethnicity'].fillna('Missing').value_counts().rename_axis('Ethnicity').reset_index(name='Participants')
missing_ethnicity_records = stage_2_ethnicity_measure_review.loc[stage_2_ethnicity_measure_review['Ethnicity code'].isna()].merge(ethnicity_case_review,
    on='NSID', how='left', suffixes=(' retained',
    ' review'), validate='one_to_one').sort_values('NSID').reset_index(drop=True)
print(f"Participants with a grouped ethnicity value: {stage_2_ethnicity_measure_review['Ethnicity code'].notna().sum():,}")
print(f"Participants with missing grouped ethnicity: {stage_2_ethnicity_measure_review['Ethnicity code'].isna().sum():,}")
print(f'Records set to missing because of unresolved conflict: {len(conflicting_ethnicity_ids):,}')
print('\nSource of retained ethnicity value:')
print(ethnicity_source_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nGrouped ethnicity distribution:')
print(ethnicity_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(
    '\nRecords without a retained grouped ethnicity value:',
    len(missing_ethnicity_records),
)
# Detailed participant-level records remain in memory for the construction audit but are not displayed.

Participants with a grouped ethnicity value: 9,763
Participants with missing grouped ethnicity: 4
Records set to missing because of unresolved conflict: 2

Source of retained ethnicity value:
                 Ethnicity source  Participants
Wave 4 harmonised grouped measure          9743
           Wave 2 grouped measure            11
           Wave 1 grouped measure             9
Unresolved Wave 1–Wave 2 conflict             2
                  No valid source             2

Grouped ethnicity distribution:
Ethnicity  Participants
    White          6636
   Indian           689
      ...           ...
    Other           242
  Missing             4

Records without a retained grouped ethnicity value: 4


In [105]:
# 33: Ethnicity-measure decision and documentation

ethnicity_decision_rules = [{'Wave': 'Wave 1', 'Source type': 'Young person', 'Variable': 'W1ethgrpYP',
    'Review outcome': 'Retain as construction source', 'Decision reason': "Provides a grouped measure of the young person's ethnicity and supplies a fallback value when the Wave 4 harmonised measure is unavailable.", 'Review notes': 'Used after W4ethgrpYP and before W2ethgrpYP. Nine participants receive their retained value from this source. Two Wave 1–Wave 2 conflicts remain unresolved and are coded as missing.'}, {'Wave': 'Wave 2',
    'Source type': 'Young person', 'Variable': 'W2ethgrpYP', 'Review outcome': 'Retain as construction source', 'Decision reason': "Provides a grouped measure of the young person's ethnicity and supplies a fallback value when valid Wave 4 and Wave 1 grouped values are unavailable.", 'Review notes': 'Eleven participants receive their retained value from this source. It is not used to overwrite a valid Wave 4 or Wave 1 grouped value.'}, {'Wave': 'Wave 4',
    'Source type': 'Young person', 'Variable': 'W4ethgrpYP', 'Review outcome': 'Retain as construction source', 'Decision reason': 'Provides the harmonised grouped ethnicity measure. It draws on Wave 1 information for the main sample and Wave 4 or household information for boost participants.', 'Review notes': 'Used as the primary construction source. It provides 9,743 valid values and agrees with every valid overlapping Wave 1 grouped record.'}, {'Wave': 'Wave 1',
    'Source type': 'Family background', 'Variable': 'W1ethgrpYP', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Exact duplicate of W1ethgrpYP in the Wave 1 young-person file. Retaining both entries would duplicate the same information.', 'Review notes': 'All 9,524 stored values were identical across the two source files.'}, {'Wave': 'Wave 2',
    'Source type': 'Family background', 'Variable': 'W2ethgrpYP', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Exact duplicate of W2ethgrpYP in the Wave 2 young-person file. Retaining both entries would duplicate the same information.', 'Review notes': 'All 9,521 stored values were identical across the two source files.'}, {'Wave': 'Wave 1',
    'Source type': 'Young person', 'Variable': 'W1ethnicYP', 'Review outcome': 'Retain as review support only', 'Decision reason': 'Self-designated detailed ethnicity measure. It is not retained alongside the grouped measure because this would provide overlapping representations of the same characteristic.', 'Review notes': 'Used only to inspect records with conflicting grouped ethnicity values.'}, {'Wave': 'Wave 1',
    'Source type': 'Young person', 'Variable': 'W1ethnic2YP', 'Review outcome': 'Retain as review support only', 'Decision reason': 'Detailed derived ethnicity measure. It is not retained alongside the grouped measure because this would provide overlapping representations of the same characteristic.', 'Review notes': 'Used only to inspect records with conflicting grouped ethnicity values.'}, {'Wave': 'Wave 2',
    'Source type': 'Young person', 'Variable': 'W2ethnicYP', 'Review outcome': 'Retain as review support only', 'Decision reason': 'Self-designated detailed ethnicity measure. It is not retained alongside the grouped measure because this would provide overlapping representations of the same characteristic.', 'Review notes': 'Used only to inspect records with conflicting grouped ethnicity values.'}, {'Wave': 'Wave 4',
    'Source type': 'Young person', 'Variable': 'w4ethnic2YP', 'Review outcome': 'Retain as review support only', 'Decision reason': 'Detailed harmonised ethnicity measure. It is not retained alongside the grouped measure because this would provide overlapping representations of the same characteristic.', 'Review notes': 'Used only to inspect records without a clear grouped ethnicity value.'}]
for rule in ethnicity_decision_rules:
    matching_rows = stage_2_variable_decision_register['Wave'].eq(rule['Wave']) & stage_2_variable_decision_register['Source type'].eq(rule['Source type']) & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {rule['Wave']}, {rule['Source type']}, {rule['Variable']}; found {matching_rows.sum()}.")
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = rule['Review outcome']
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'Demographic background'
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = rule['Decision reason']
    stage_2_variable_decision_register.loc[matching_rows,
        'Leakage assessment'] = 'No direct outcome leakage identified; ethnicity is treated as a stable background characteristic.'
    stage_2_variable_decision_register.loc[matching_rows,
        'Reference-period assessment'] = 'Stable background characteristic recorded or harmonised across the survey waves.'
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave 1, Wave 2 and Wave 4 young-person data dictionaries and cross-wave consistency review'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = rule['Review notes']
stage_2_ethnicity_measure_review['Ethnicity conflict unresolved'] = stage_2_ethnicity_measure_review['Ethnicity source'].eq('Unresolved Wave 1–Wave 2 conflict')
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
ethnicity_measure_review_output_path = stage_2_output_directory / 'stage_2_ethnicity_measure_review.csv'
stage_2_ethnicity_measure_review.to_csv(ethnicity_measure_review_output_path, index=False)
ethnicity_decision_output = stage_2_variable_decision_register.loc[stage_2_variable_decision_register['Variable'].str.lower().isin(direct_ethnicity_variable_names),
    ['Wave', 'Source type', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment',
    'Review notes']].sort_values(['Review outcome', 'Wave', 'Source type', 'Variable'])
ethnicity_exclusions = ethnicity_decision_output.loc[ethnicity_decision_output['Review outcome'].eq('Exclude from predictor set'),
    ['Wave', 'Source type', 'Variable', 'Decision reason', 'Leakage assessment']]
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Participants in ethnicity-measure file: {len(stage_2_ethnicity_measure_review):,}')
print(f"Participants with a grouped ethnicity value: {stage_2_ethnicity_measure_review['Ethnicity code'].notna().sum():,}")
print(f"Participants with missing grouped ethnicity: {stage_2_ethnicity_measure_review['Ethnicity code'].isna().sum():,}")
print(f"Unresolved cross-wave conflicts: {stage_2_ethnicity_measure_review['Ethnicity conflict unresolved'].sum():,}")
print('\nDirect ethnicity-variable decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 130, 'display.width', 360):
    print(ethnicity_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables excluded in this step:')
print(ethnicity_exclusions.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'Ethnicity-measure review file saved to: {ethnicity_measure_review_output_path}')

Participants in ethnicity-measure file: 9,767
Participants with a grouped ethnicity value: 9,763
Participants with missing grouped ethnicity: 4
Unresolved cross-wave conflicts: 2

Direct ethnicity-variable decisions:
  Wave       Source type    Variable                                                                  Variable label                Review outcome                                                                                                                                                                 Decision reason                                                                                Leakage assessment                                                            Review notes
Wave 1 Family background  W1ethgrpYP                                       DV: Young person's ethnic group (grouped)    Exclude from predictor set                                                     Exact duplicate of W1ethgrpYP in the Wave 1 young-person file. Retaining both entries would

In [106]:
# 34: Young-person language-background variable list

import re
language_name_pattern = re.compile('langhom|langeng|englang|homelang|firstlang|mainlang|mothertongue',
    flags=re.IGNORECASE)
language_label_pattern = re.compile('\\blanguage spoken at home\\b|\\blanguage(?:s)? spoken in the home\\b|\\bhome language\\b|\\bfirst language\\b|\\bmain language\\b|\\bmother tongue\\b|\\benglish as (?:a |the )?first language\\b|\\benglish as an additional language\\b|\\blanguage other than english\\b|\\b(?:can|could) speak english\\b|\\b(?:can|could) understand english\\b|\\benglish language background\\b',
    flags=re.IGNORECASE)
non_background_pattern = re.compile('\\bqualification\\b|\\bexam\\b|\\bgcse\\b|\\bgrade\\b|\\bsubject\\b|\\blesson\\b|\\bteacher\\b|\\battainment\\b|\\btest score\\b',
    flags=re.IGNORECASE)
language_register = stage_2_master_variable_register.copy()
name_match = language_register['Variable'].fillna('').apply(lambda value: bool(language_name_pattern.search(str(value))))
label_match = language_register['Variable label'].fillna('').apply(lambda value: bool(language_label_pattern.search(str(value))))
stage_2_language_review = language_register.loc[name_match | label_match, ['Source order', 'Wave', 'Source type',
    'Source file', 'Source path', 'Variable position', 'Variable', 'Variable label', 'Data type']].copy()
direct_yp_language_pattern = re.compile('(?:\\byp\\b|young person).*(?:language|english)|(?:language|english).*(?:\\byp\\b|young person)',
    flags=re.IGNORECASE)
parent_language_pattern = re.compile('\\bmain parent\\b|\\bsecond parent\\b|\\bmother\\b|\\bfather\\b|\\bparent\\b|\\bmp\\b|\\bsp\\b',
    flags=re.IGNORECASE)

def classify_language_review(variable, variable_label, source_type):
    """Classify a language-related variable for manual review."""
    variable_text = str(variable)
    label_text = str(variable_label)
    if non_background_pattern.search(label_text):
        return 'Other language-related variable'
    if direct_yp_language_pattern.search(label_text) or (source_type == 'Young person' and language_name_pattern.search(variable_text)):
        return 'Possible direct young-person language-background measure'
    if parent_language_pattern.search(label_text):
        return 'Parent or household language measure'
    return 'Other language-related variable'
stage_2_language_review['Review category'] = stage_2_language_review.apply(lambda row: classify_language_review(row['Variable'],
    row['Variable label'], row['Source type']), axis=1)
stage_2_language_review = stage_2_language_review.merge(stage_2_variable_decision_register[['Source file', 'Variable',
    'Review outcome']], on=['Source file',
    'Variable'], how='left', validate='one_to_one').sort_values(['Review category', 'Source order',
    'Variable position']).reset_index(drop=True)
language_review_summary = stage_2_language_review['Review category'].value_counts().rename_axis('Review category').reset_index(name='Variables')
print(f'Language-background variables identified: {len(stage_2_language_review):,}')
print('\nReview-category summary:')
print(language_review_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables identified:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 350):
    print(stage_2_language_review[['Wave', 'Source type', 'Variable', 'Variable label', 'Review category',
        'Review outcome']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Language-background variables identified: 19

Review-category summary:
                                         Review category  Variables
                         Other language-related variable         10
Possible direct young-person language-background measure          9

Variables identified:
  Wave       Source type    Variable                                               Variable label                                          Review category Review outcome
Wave 1 Family background W1englangHH            HH: Whether English is main language of household                          Other language-related variable Pending review
Wave 1 Family background  W1LangHom1           DV: 1st language other than English spoken at home                          Other language-related variable Pending review
   ...               ...         ...                                                          ...                                                      ...            ...
Wave 3      Young pers

In [107]:
# 35: English-language indicator review

english_language_sources = [{'Measure type': 'Young-person language', 'Wave': 'Wave 1', 'Source type': 'Young person',
    'Variable': 'W1englangYP', 'Output name': 'W1 young-person English language'}, {'Measure type': 'Young-person language',
    'Wave': 'Wave 2', 'Source type': 'Young person', 'Variable': 'W2EnglangYP', 'Output name': 'W2 young-person English language'}, {'Measure type': 'Young-person language',
    'Wave': 'Wave 3', 'Source type': 'Young person', 'Variable': 'W3englangYP', 'Output name': 'W3 young-person English language'}, {'Measure type': 'Household language',
    'Wave': 'Wave 1', 'Source type': 'Family background', 'Variable': 'W1englangHH', 'Output name': 'W1 household English language'}, {'Measure type': 'Household language',
    'Wave': 'Wave 2', 'Source type': 'Family background', 'Variable': 'W2EnglangHH', 'Output name': 'W2 household English language'}, {'Measure type': 'Household language',
    'Wave': 'Wave 3', 'Source type': 'Family background', 'Variable': 'W3englangHH', 'Output name': 'W3 household English language'}, {'Measure type': 'Household language',
    'Wave': 'Wave 4', 'Source type': 'Family background', 'Variable': 'W4EngLangHH', 'Output name': 'W4 household English language'}]
english_language_review_data = stage_2_sex_measure[['NSID']].copy()
language_source_records = []
language_value_records = []
for source_specification in english_language_sources:
    matching_entry = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq(source_specification['Wave']) & stage_2_master_variable_register['Source type'].eq(source_specification['Source type']) & stage_2_master_variable_register['Variable'].str.lower().eq(source_specification['Variable'].lower())]
    if len(matching_entry) != 1:
        raise ValueError(f'Expected one source entry for {source_specification}, but found {len(matching_entry)}.')
    source_path = Path(matching_entry.iloc[0]['Source path'])
    source_variable = matching_entry.iloc[0]['Variable']
    numeric_data = pd.read_stata(source_path, columns=['NSID', source_variable], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', source_variable], convert_categoricals=True)
    source_value_data = numeric_data.copy()
    source_value_data['Value label'] = labelled_data[source_variable].astype('string')
    sample_value_data = stage_2_sex_measure[['NSID']].merge(source_value_data, on='NSID', how='left',
        validate='one_to_one', indicator=True)
    numeric_values = pd.to_numeric(sample_value_data[source_variable], errors='coerce')
    source_record_present = sample_value_data['_merge'].eq('both')
    language_source_records.append({'Measure type': source_specification['Measure type'],
        'Wave': source_specification['Wave'], 'Source type': source_specification['Source type'], 'Variable': source_variable, 'Variable label': matching_entry.iloc[0]['Variable label'], 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Negative survey code': int(numeric_values.lt(0).sum()), 'Non-negative stored value': int(numeric_values.ge(0).sum()), 'Distinct stored codes': int(numeric_values.nunique(dropna=True))})
    value_summary = sample_value_data.loc[source_record_present, [source_variable,
        'Value label']].value_counts(dropna=False).rename('Participants').reset_index().rename(columns={source_variable: 'Value code'})
    value_summary.insert(0, 'Variable', source_variable)
    value_summary.insert(0, 'Wave', source_specification['Wave'])
    value_summary.insert(0, 'Measure type', source_specification['Measure type'])
    language_value_records.append(value_summary)
    source_column = source_specification['Output name']
    english_language_review_data = english_language_review_data.merge(numeric_data.rename(columns={source_variable: source_column}),
        on='NSID', how='left', validate='one_to_one')
stage_2_english_language_coverage = pd.DataFrame(language_source_records)
stage_2_english_language_values = pd.concat(language_value_records, ignore_index=True)
pairwise_language_comparisons = [('Young person: Wave 1 versus Wave 2', 'W1 young-person English language',
    'W2 young-person English language'), ('Young person: Wave 1 versus Wave 3', 'W1 young-person English language',
    'W3 young-person English language'), ('Young person: Wave 2 versus Wave 3', 'W2 young-person English language',
    'W3 young-person English language'), ('Household: Wave 1 versus Wave 2', 'W1 household English language',
    'W2 household English language'), ('Household: Wave 1 versus Wave 3', 'W1 household English language',
    'W3 household English language'), ('Household: Wave 1 versus Wave 4', 'W1 household English language',
    'W4 household English language'), ('Household: Wave 2 versus Wave 3', 'W2 household English language',
    'W3 household English language'), ('Household: Wave 2 versus Wave 4', 'W2 household English language',
    'W4 household English language'), ('Household: Wave 3 versus Wave 4', 'W3 household English language',
    'W4 household English language')]
language_consistency_records = []
for comparison, first_column, second_column in pairwise_language_comparisons:
    first_values = pd.to_numeric(english_language_review_data[first_column], errors='coerce')
    second_values = pd.to_numeric(english_language_review_data[second_column], errors='coerce')
    both_valid = first_values.ge(0) & second_values.ge(0)
    values_agree = first_values.eq(second_values)
    valid_comparisons = int(both_valid.sum())
    agreeing_values = int((both_valid & values_agree).sum())
    language_consistency_records.append({'Comparison': comparison, 'Both values valid': valid_comparisons,
        'Values agreeing': agreeing_values, 'Values disagreeing': valid_comparisons - agreeing_values, 'Agreement percentage': agreeing_values / valid_comparisons * 100 if valid_comparisons else np.nan})
stage_2_english_language_consistency = pd.DataFrame(language_consistency_records)
print(f'English-language indicator entries reviewed: {len(stage_2_english_language_coverage):,}')
print('\nCoverage and stored-code summary:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 140, 'display.width', 360):
    print(stage_2_english_language_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nValue codes and labels:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 130, 'display.width', 320):
    print(stage_2_english_language_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCross-wave consistency:')
print(stage_2_english_language_consistency.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Agreement percentage': lambda value: f'{value:.2f}%'}))

English-language indicator entries reviewed: 7

Coverage and stored-code summary:
         Measure type   Wave       Source type    Variable                                    Variable label  Source record present  Source record absent  Negative survey code  Non-negative stored value  Distinct stored codes
Young-person language Wave 1      Young person W1englangYP     YP: Whether English is first or main language                   9524                   243                    89                       9435                      5
Young-person language Wave 2      Young person W2EnglangYP     YP: Whether English is first or main language                   9521                   246                  9440                         81                      6
                  ...    ...               ...         ...                                               ...                    ...                   ...                   ...                        ...                    ...
   Household l

In [108]:
# 36: Language-variable routing review

w3_language_frequency_entry = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq('Wave 3') & stage_2_master_variable_register['Source type'].eq('Young person') & stage_2_master_variable_register['Variable'].str.lower().eq('w3langfreqyp')]
if len(w3_language_frequency_entry) != 1:
    raise ValueError(f'Expected one W3langfreqYP source entry, but found {len(w3_language_frequency_entry)}.')
w3_language_frequency_path = Path(w3_language_frequency_entry.iloc[0]['Source path'])
w3_language_frequency_variable = w3_language_frequency_entry.iloc[0]['Variable']
w3_language_frequency_data = pd.read_stata(w3_language_frequency_path, columns=['NSID',
    w3_language_frequency_variable], convert_categoricals=False)
language_routing_review = english_language_review_data.merge(w3_language_frequency_data, on='NSID', how='left',
    validate='one_to_one')
w1_yp_language = pd.to_numeric(language_routing_review['W1 young-person English language'], errors='coerce')
w2_yp_language = pd.to_numeric(language_routing_review['W2 young-person English language'], errors='coerce')
w3_yp_language = pd.to_numeric(language_routing_review['W3 young-person English language'], errors='coerce')
w3_language_frequency = pd.to_numeric(language_routing_review[w3_language_frequency_variable], errors='coerce')
w1_status = np.select([w1_yp_language.isin([1, 2, 3, 4]), w1_yp_language.eq(-99), w1_yp_language.lt(0),
    w1_yp_language.isna()], ['Valid Wave 1 category', 'Wave 1 YP not interviewed', 'Other Wave 1 negative code',
    'No Wave 1 source record'], default='Other Wave 1 value')
w2_status = np.select([w2_yp_language.isin([1, 2, 3, 4]), w2_yp_language.eq(-91), w2_yp_language.eq(-99),
    w2_yp_language.lt(0), w2_yp_language.isna()], ['Valid Wave 2 category', 'Wave 2 not applicable',
    'Wave 2 YP not interviewed', 'Other Wave 2 negative code', 'No Wave 2 source record'], default='Other Wave 2 value')
w2_routing_table = pd.crosstab(pd.Series(w1_status, name='Wave 1 status'), pd.Series(w2_status, name='Wave 2 status'),
    margins=True, margins_name='Total')
w1_category_labels = w1_yp_language.map({1.0: 'English only', 2.0: 'English first/main and speaks other languages',
    3.0: 'Another language is first/main', 4.0: 'Bilingual'})
w3_category_labels = w3_yp_language.map({1.0: 'Yes', 2.0: 'No'})
w1_w3_category_table = pd.crosstab(w1_category_labels, w3_category_labels, margins=True, margins_name='Total')
w3_frequency_labels = w3_language_frequency.map({1.0: 'All or most of the time', 2.0: 'About half of the time',
    3.0: 'Just now and then', -1.0: "Don't know", -91.0: 'Not applicable', -92.0: 'Refused', -99.0: 'YP not interviewed'}).fillna('No source record or other value')
w3_language_labels_complete = w3_yp_language.map({1.0: 'Yes', 2.0: 'No', -1.0: "Don't know", -91.0: 'Not applicable',
    -92.0: 'Refused', -99.0: 'YP not interviewed'}).fillna('No source record or other value')
w3_routing_table = pd.crosstab(w3_language_labels_complete, w3_frequency_labels, margins=True, margins_name='Total')
w3_valid_frequency = w3_language_frequency.isin([1, 2, 3])
w3_frequency_by_language = pd.DataFrame({'W3 language response': w3_language_labels_complete,
    'Valid frequency response': w3_valid_frequency}).groupby('W3 language response',
    dropna=False).agg(Participants=('Valid frequency response', 'size'),
    Valid_frequency_responses=('Valid frequency response', 'sum')).reset_index()
print('Wave 1–Wave 2 routing structure:')
print(w2_routing_table.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nWave 1 category by Wave 3 Yes/No response:')
print(w1_w3_category_table.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nWave 3 language response by language-frequency response:')
print(w3_routing_table.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nValid Wave 3 frequency responses by language response:')
print(w3_frequency_by_language.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Wave 1–Wave 2 routing structure:
Wave 2 status              No Wave 2 source record  Valid Wave 2 category  Wave 2 YP not interviewed  Wave 2 not applicable  Total
Wave 1 status                                                                                                                     
No Wave 1 source record                        243                      0                          0                      0    243
Valid Wave 1 category                            3                      0                         68                   9364   9435
Wave 1 YP not interviewed                        0                     81                          8                      0     89
Total                                          246                     81                         76                   9364   9767

Wave 1 category by Wave 3 Yes/No response:
W3 young-person English language                 No   Yes  Total
W1 young-person English language                                
Another

In [109]:
# 37: Harmonised home-language variable review

home_language_sources = [{'Wave': 'Wave 1', 'Source type': 'Young person', 'Variables': ['W1LangHom1', 'W1LangHom2',
    'W1LangHom3'], 'Source label': 'Wave 1 young-person file'}, {'Wave': 'Wave 1', 'Source type': 'Family background',
    'Variables': ['W1LangHom1', 'W1LangHom2',
    'W1LangHom3'], 'Source label': 'Wave 1 family-background file'}, {'Wave': 'Wave 4',
    'Source type': 'Family background', 'Variables': ['W4langhom1', 'W4langhom2',
    'W4langhom3'], 'Source label': 'Wave 4 harmonised family file'}]
home_language_source_data = {}
home_language_coverage_records = []
home_language_value_records = []
stage_2_participant_ids = stage_2_sex_measure[['NSID']].copy()
for source_specification in home_language_sources:
    requested_variables_lower = {variable.lower() for variable in source_specification['Variables']}
    matching_entries = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq(source_specification['Wave']) & stage_2_master_variable_register['Source type'].eq(source_specification['Source type']) & stage_2_master_variable_register['Variable'].str.lower().isin(requested_variables_lower)].sort_values('Variable position')
    if len(matching_entries) != 3:
        raise ValueError(f"Expected three home-language variables for {source_specification['Source label']}, but found {len(matching_entries)}.")
    source_paths = matching_entries['Source path'].drop_duplicates().tolist()
    if len(source_paths) != 1:
        raise ValueError(f"Expected one source file for {source_specification['Source label']}, but found {len(source_paths)}.")
    source_path = Path(source_paths[0])
    source_variables = matching_entries['Variable'].tolist()
    numeric_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    sample_numeric_data = stage_2_participant_ids.merge(numeric_data, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = sample_numeric_data['_merge'].eq('both')
    sample_numeric_data = sample_numeric_data.drop(columns='_merge')
    sample_labelled_data = stage_2_participant_ids.merge(labelled_data, on='NSID', how='left', validate='one_to_one')
    home_language_source_data[source_specification['Source label']] = {'Numeric': sample_numeric_data,
        'Labelled': sample_labelled_data, 'Variables': source_variables}
    for variable in source_variables:
        numeric_values = pd.to_numeric(sample_numeric_data[variable], errors='coerce')
        home_language_coverage_records.append({'Wave': source_specification['Wave'],
            'Source': source_specification['Source label'], 'Variable': variable, 'Variable label': matching_entries.loc[matching_entries['Variable'].eq(variable),
            'Variable label'].iloc[0], 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Stored value': int(numeric_values.notna().sum()), 'Negative code': int(numeric_values.lt(0).sum()), 'Zero code': int(numeric_values.eq(0).sum()), 'Positive code': int(numeric_values.gt(0).sum()), 'Distinct stored codes': int(numeric_values.nunique(dropna=True))})
        value_summary = pd.DataFrame({'Value code': sample_numeric_data[variable],
            'Value label': sample_labelled_data[variable].astype('string')}).loc[source_record_present].value_counts(dropna=False).rename('Participants').reset_index()
        value_summary.insert(0, 'Variable', variable)
        value_summary.insert(0, 'Source', source_specification['Source label'])
        home_language_value_records.append(value_summary)
stage_2_home_language_coverage = pd.DataFrame(home_language_coverage_records)
stage_2_home_language_values = pd.concat(home_language_value_records, ignore_index=True)
w1_yp_data = home_language_source_data['Wave 1 young-person file']['Numeric']
w1_family_data = home_language_source_data['Wave 1 family-background file']['Numeric']
w1_yp_variables = home_language_source_data['Wave 1 young-person file']['Variables']
w1_family_variables = home_language_source_data['Wave 1 family-background file']['Variables']
wave_1_duplicate_comparison = stage_2_participant_ids.copy()
for position in range(3):
    wave_1_duplicate_comparison[f'YP value {position + 1}'] = w1_yp_data[w1_yp_variables[position]]
    wave_1_duplicate_comparison[f'Family value {position + 1}'] = w1_family_data[w1_family_variables[position]]
comparison_columns = []
for position in range(3):
    yp_column = f'YP value {position + 1}'
    family_column = f'Family value {position + 1}'
    comparison_column = f'Variable {position + 1} identical'
    wave_1_duplicate_comparison[comparison_column] = wave_1_duplicate_comparison[yp_column].fillna('__MISSING__').eq(wave_1_duplicate_comparison[family_column].fillna('__MISSING__'))
    comparison_columns.append(comparison_column)
wave_1_duplicate_comparison['All three variables identical'] = wave_1_duplicate_comparison[comparison_columns].all(axis=1)
print(f'Home-language source-variable entries reviewed: {len(stage_2_home_language_coverage):,}')
print('\nCoverage and code-type summary:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 140, 'display.width', 360):
    print(stage_2_home_language_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f"\nWave 1 participants with all three duplicate variables identical: {wave_1_duplicate_comparison['All three variables identical'].sum():,}")
print(f"Wave 1 participants with at least one different duplicate value: {(~wave_1_duplicate_comparison['All three variables identical']).sum():,}")
print('\nValue codes and labels:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 140, 'display.width', 330):
    print(stage_2_home_language_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Home-language source-variable entries reviewed: 9

Coverage and code-type summary:
  Wave                        Source   Variable                                                                   Variable label  Source record present  Source record absent  Stored value  Negative code  Zero code  Positive code  Distinct stored codes
Wave 1      Wave 1 young-person file W1LangHom1                               DV: 1st language other than English spoken at home                   9524                   243          9524           7651          0           1873                     58
Wave 1      Wave 1 young-person file W1LangHom2                               DV: 2nd language other than English spoken at home                   9524                   243          9524           7651       1650            223                     26
   ...                           ...        ...                                                                              ...                    ...          

In [110]:
# 38: Wave 1–Wave 4 home-language consistency review

w1_home_numeric = home_language_source_data['Wave 1 young-person file']['Numeric'].copy()
w1_home_labelled = home_language_source_data['Wave 1 young-person file']['Labelled'].copy()
w1_home_variables = home_language_source_data['Wave 1 young-person file']['Variables']
w4_home_numeric = home_language_source_data['Wave 4 harmonised family file']['Numeric'].copy()
w4_home_labelled = home_language_source_data['Wave 4 harmonised family file']['Labelled'].copy()
w4_home_variables = home_language_source_data['Wave 4 harmonised family file']['Variables']
home_language_consistency_data = stage_2_participant_ids.copy()
for position in range(3):
    w1_variable = w1_home_variables[position]
    w4_variable = w4_home_variables[position]
    home_language_consistency_data[f'W1 code {position + 1}'] = pd.to_numeric(w1_home_numeric[w1_variable],
        errors='coerce')
    home_language_consistency_data[f'W1 label {position + 1}'] = w1_home_labelled[w1_variable].astype('string')
    home_language_consistency_data[f'W4 code {position + 1}'] = pd.to_numeric(w4_home_numeric[w4_variable],
        errors='coerce')
    home_language_consistency_data[f'W4 label {position + 1}'] = w4_home_labelled[w4_variable].astype('string')

def classify_home_language_status(first_code):
    """Classify source availability from the first derived value."""
    if pd.isna(first_code):
        return 'No source record'
    if first_code == -91:
        return 'English only'
    if first_code < 0:
        return 'Information unavailable'
    return 'Language list available'
home_language_consistency_data['W1 home-language status'] = home_language_consistency_data['W1 code 1'].apply(classify_home_language_status)
home_language_consistency_data['W4 home-language status'] = home_language_consistency_data['W4 code 1'].apply(classify_home_language_status)

def extract_language_set(row, prefix):
    """Return the distinct positive-code language labels."""
    language_labels = []
    for position in range(1, 4):
        code = row[f'{prefix} code {position}']
        label = row[f'{prefix} label {position}']
        if pd.isna(code) or code <= 0:
            continue
        if pd.isna(label):
            continue
        label_text = str(label).strip()
        if label_text == 'No further languages':
            continue
        language_labels.append(label_text)
    return tuple(sorted(set(language_labels)))
home_language_consistency_data['W1 language set'] = home_language_consistency_data.apply(lambda row: extract_language_set(row,
    'W1'), axis=1)
home_language_consistency_data['W4 language set'] = home_language_consistency_data.apply(lambda row: extract_language_set(row,
    'W4'), axis=1)

def classify_non_english_language(status, language_set):
    """Identify whether a non-English home language is recorded."""
    if status == 'English only':
        return 'No'
    if status != 'Language list available':
        return 'Missing'
    non_substantive_labels = {'English', "Don't know", 'Refused', 'Other answers'}
    substantive_non_english_languages = [language for language in language_set if language not in non_substantive_labels]
    if substantive_non_english_languages:
        return 'Yes'
    if language_set == ('English',):
        return 'No'
    return 'Unclear'
home_language_consistency_data['W1 non-English language recorded'] = home_language_consistency_data.apply(lambda row: classify_non_english_language(row['W1 home-language status'],
    row['W1 language set']), axis=1)
home_language_consistency_data['W4 non-English language recorded'] = home_language_consistency_data.apply(lambda row: classify_non_english_language(row['W4 home-language status'],
    row['W4 language set']), axis=1)
home_language_status_table = pd.crosstab(home_language_consistency_data['W1 home-language status'],
    home_language_consistency_data['W4 home-language status'], margins=True, margins_name='Total')
non_english_language_table = pd.crosstab(home_language_consistency_data['W1 non-English language recorded'],
    home_language_consistency_data['W4 non-English language recorded'], margins=True, margins_name='Total')
w1_broad_valid = home_language_consistency_data['W1 non-English language recorded'].isin(['Yes', 'No'])
w4_broad_valid = home_language_consistency_data['W4 non-English language recorded'].isin(['Yes', 'No'])
both_broad_indicators_valid = w1_broad_valid & w4_broad_valid
broad_indicators_agree = home_language_consistency_data['W1 non-English language recorded'].eq(home_language_consistency_data['W4 non-English language recorded'])
broad_valid_comparisons = int(both_broad_indicators_valid.sum())
broad_agreements = int((both_broad_indicators_valid & broad_indicators_agree).sum())
both_language_lists_available = home_language_consistency_data['W1 home-language status'].eq('Language list available') & home_language_consistency_data['W4 home-language status'].eq('Language list available')
exact_language_sets_agree = home_language_consistency_data['W1 language set'].eq(home_language_consistency_data['W4 language set'])
exact_set_comparisons = int(both_language_lists_available.sum())
exact_set_agreements = int((both_language_lists_available & exact_language_sets_agree).sum())
language_set_disagreement_patterns = home_language_consistency_data.loc[both_language_lists_available & ~exact_language_sets_agree,
    ['W1 language set', 'W4 language set']].value_counts().rename('Participants').reset_index().head(25)
coverage_summary = pd.DataFrame([{'Coverage measure': 'Valid Wave 1 broad indicator',
    'Participants': int(w1_broad_valid.sum())}, {'Coverage measure': 'Valid Wave 4 broad indicator',
    'Participants': int(w4_broad_valid.sum())}, {'Coverage measure': 'Wave 4 valid where Wave 1 is unavailable',
    'Participants': int((w4_broad_valid & ~w1_broad_valid).sum())}, {'Coverage measure': 'Wave 1 valid where Wave 4 is unavailable',
    'Participants': int((w1_broad_valid & ~w4_broad_valid).sum())}, {'Coverage measure': 'No valid broad indicator in either source',
    'Participants': int((~w1_broad_valid & ~w4_broad_valid).sum())}])
print(f'Stage 2 participants: {len(home_language_consistency_data):,}')
print('\nWave 1–Wave 4 information-status comparison:')
print(home_language_status_table.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nWave 1–Wave 4 non-English-language comparison:')
print(non_english_language_table.to_string(max_rows=TABLE_ROW_LIMIT))
print(f'\nValid broad-indicator comparisons: {broad_valid_comparisons:,}')
print(f'Broad indicators agreeing: {broad_agreements:,}')
print(f'Broad-indicator agreement: {(broad_agreements / broad_valid_comparisons * 100 if broad_valid_comparisons else np.nan):.2f}%')
print(f'\nParticipants with language lists at both sources: {exact_set_comparisons:,}')
print(f'Exact language sets agreeing: {exact_set_agreements:,}')
print(f'Exact-set agreement: {(exact_set_agreements / exact_set_comparisons * 100 if exact_set_comparisons else np.nan):.2f}%')
print('\nCoverage comparison:')
print(coverage_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMost common differing language-set patterns:')
if language_set_disagreement_patterns.empty:
    print('No differing language sets were identified.')
else:
    print(language_set_disagreement_patterns.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Stage 2 participants: 9,767

Wave 1–Wave 4 information-status comparison:
W4 home-language status  English only  Information unavailable  Language list available  No source record  Total
W1 home-language status                                                                                         
English only                     7434                       53                       51                 9   7547
Information unavailable            61                       21                       22                 0    104
Language list available            31                       23                     1817                 2   1873
No source record                  144                        9                       90                 0    243
Total                            7670                      106                     1980                11   9767

Wave 1–Wave 4 non-English-language comparison:
W4 non-English language recorded  Missing    No  Unclear   Yes  Total
W1 non-English la

In [111]:
# 39: Personal and household language-measure comparison

w1_personal_language_code = pd.to_numeric(english_language_review_data['W1 young-person English language'],
    errors='coerce')
w2_personal_language_code = pd.to_numeric(english_language_review_data['W2 young-person English language'],
    errors='coerce')
personal_language_map = {1.0: 'Yes', 2.0: 'Yes', 3.0: 'No', 4.0: 'Yes'}
w1_personal_language = w1_personal_language_code.map(personal_language_map)
w2_personal_language = w2_personal_language_code.map(personal_language_map)
personal_language_measure = w1_personal_language.combine_first(w2_personal_language)
personal_language_source = np.select([w1_personal_language.notna(),
    w1_personal_language.isna() & w2_personal_language.notna()], ['Wave 1 young-person response',
    'Wave 2 young-person fallback'], default='No valid source')
w1_household_language = home_language_consistency_data['W1 non-English language recorded'].where(home_language_consistency_data['W1 non-English language recorded'].isin(['Yes',
    'No']))
w4_household_language = home_language_consistency_data['W4 non-English language recorded'].where(home_language_consistency_data['W4 non-English language recorded'].isin(['Yes',
    'No']))
household_language_measure = w1_household_language.combine_first(w4_household_language)
household_language_source = np.select([w1_household_language.notna(),
    w1_household_language.isna() & w4_household_language.notna()], ['Wave 1 home-language measure',
    'Wave 4 harmonised fallback'], default='No valid source')
household_language_disagreement = w1_household_language.notna() & w4_household_language.notna() & w1_household_language.ne(w4_household_language)
stage_2_language_measure_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'English first or main language': personal_language_measure, 'Personal language source': personal_language_source, 'Non-English home language recorded': household_language_measure, 'Household language source': household_language_source, 'Household language cross-wave disagreement': household_language_disagreement})
personal_language_distribution = stage_2_language_measure_review['English first or main language'].fillna('Missing').value_counts().rename_axis('English first or main language').reset_index(name='Participants')
personal_language_source_summary = stage_2_language_measure_review['Personal language source'].value_counts().rename_axis('Personal language source').reset_index(name='Participants')
household_language_distribution = stage_2_language_measure_review['Non-English home language recorded'].fillna('Missing').value_counts().rename_axis('Non-English home language recorded').reset_index(name='Participants')
household_language_source_summary = stage_2_language_measure_review['Household language source'].value_counts().rename_axis('Household language source').reset_index(name='Participants')
personal_household_language_table = pd.crosstab(stage_2_language_measure_review['English first or main language'],
    stage_2_language_measure_review['Non-English home language recorded'], margins=True, margins_name='Total')
personal_available = stage_2_language_measure_review['English first or main language'].notna()
household_available = stage_2_language_measure_review['Non-English home language recorded'].notna()
joint_availability_summary = pd.DataFrame([{'Availability status': 'Both measures available',
    'Participants': int((personal_available & household_available).sum())}, {'Availability status': 'Personal measure only',
    'Participants': int((personal_available & ~household_available).sum())}, {'Availability status': 'Household measure only',
    'Participants': int((~personal_available & household_available).sum())}, {'Availability status': 'Neither measure available',
    'Participants': int((~personal_available & ~household_available).sum())}])
print(f'Stage 2 participants: {len(stage_2_language_measure_review):,}')
print('\nPersonal language distribution:')
print(personal_language_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nPersonal language source:')
print(personal_language_source_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nHousehold language distribution:')
print(household_language_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nHousehold language source:')
print(household_language_source_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nHousehold cross-wave disagreements retained for quality control: {household_language_disagreement.sum():,}')
print('\nPersonal by household language measure:')
print(personal_household_language_table.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nJoint availability:')
print(joint_availability_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Stage 2 participants: 9,767

Personal language distribution:
English first or main language  Participants
                           Yes          8959
                            No           557
                       Missing           251

Personal language source:
    Personal language source  Participants
Wave 1 young-person response          9435
             No valid source           251
Wave 2 young-person fallback            81

Household language distribution:
Non-English home language recorded  Participants
                                No          8393
                               Yes          1277
                           Missing            97

Household language source:
   Household language source  Participants
Wave 1 home-language measure          9349
  Wave 4 harmonised fallback           321
             No valid source            97

Household cross-wave disagreements retained for quality control: 142

Personal by household language measure:
Non-English home la

In [112]:
# 40: Language-measure decision and documentation

stage_2_language_measure_review['english_first_or_main_language'] = stage_2_language_measure_review['English first or main language'].map({'Yes': 1,
    'No': 0}).astype('Int64')
stage_2_language_measure_review['non_english_home_language'] = stage_2_language_measure_review['Non-English home language recorded'].map({'Yes': 1,
    'No': 0}).astype('Int64')
language_decision_rules = [{'Wave': 'Wave 1', 'Source type': 'Young person', 'Variable': 'W1englangYP',
    'Review outcome': 'Retain as construction source', 'Decision reason': "Primary source for a binary measure of whether English is the young person's first or main language.", 'Reference-period assessment': 'Wave 1 pre-transition measure.', 'Review notes': 'Supplies 9,435 valid participant-level values. The four response categories are consolidated into Yes or No without changing the original source variable.'}, {'Wave': 'Wave 2',
    'Source type': 'Young person', 'Variable': 'W2EnglangYP', 'Review outcome': 'Retain as construction source', 'Decision reason': 'Fallback source for the personal language measure when Wave 1 does not contain a valid response.', 'Reference-period assessment': 'Wave 2 pre-transition measure used only to supplement missing Wave 1 information.', 'Review notes': 'Supplies 81 additional valid values. The remaining Wave 2 records are structurally not applicable or contain survey non-response codes.'}, {'Wave': 'Wave 3',
    'Source type': 'Young person', 'Variable': 'W3englangYP', 'Review outcome': 'Retain as review support only', 'Decision reason': 'Later repeated language item with a different two-category response structure. It is not required after construction from the earlier Wave 1 and Wave 2 measures.', 'Reference-period assessment': 'Wave 3 measure close to the post-16 transition.', 'Review notes': 'Used to inspect routing and cross-wave response structure only. It will not be used as a predictor or as a fallback construction source.'}, {'Wave': 'Wave 3',
    'Source type': 'Young person', 'Variable': 'W3langfreqYP', 'Review outcome': 'Retain as review support only', 'Decision reason': 'Conditional follow-up on the frequency of language use at home. Its structural routing and overlap with the retained language-background measures make it unsuitable as a separate predictor.', 'Reference-period assessment': 'Wave 3 measure close to the post-16 transition.', 'Review notes': 'Valid responses are available only for participants routed from W3englangYP. Used only to confirm the Wave 3 routing structure.'}]
for variable in ['W1LangHom1', 'W1LangHom2', 'W1LangHom3']:
    language_decision_rules.append({'Wave': 'Wave 1', 'Source type': 'Young person', 'Variable': variable,
        'Review outcome': 'Retain as construction source', 'Decision reason': "Source for the Wave 1 binary measure of whether a non-English language is recorded as spoken in the young person's home.", 'Reference-period assessment': 'Wave 1 pre-transition household-language measure.', 'Review notes': 'Used jointly with the other Wave 1 LangHom variables. Individual language names are not retained as separate predictors.'})
for variable in ['W4langhom1', 'W4langhom2', 'W4langhom3']:
    language_decision_rules.append({'Wave': 'Wave 4', 'Source type': 'Family background', 'Variable': variable,
        'Review outcome': 'Retain as construction source', 'Decision reason': 'Harmonised fallback source for the binary non-English home-language measure when Wave 1 does not provide a valid classification.', 'Reference-period assessment': 'Later household-language information used only when the Wave 1 measure is unavailable.', 'Review notes': 'Provides fallback information for 321 participants. It does not overwrite a valid Wave 1 measure. Source and cross-wave disagreement are retained for quality control.'})
for variable in ['W1LangHom1', 'W1LangHom2', 'W1LangHom3']:
    language_decision_rules.append({'Wave': 'Wave 1', 'Source type': 'Family background', 'Variable': variable,
        'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Exact duplicate of the corresponding variable in the Wave 1 young-person file. Retaining both would duplicate identical information.', 'Reference-period assessment': 'Wave 1 pre-transition household-language measure.', 'Review notes': 'All participant-level values were identical between the two Wave 1 source files.'})
for wave, variable in [('Wave 1', 'W1englangHH'), ('Wave 2', 'W2EnglangHH'), ('Wave 3', 'W3englangHH'), ('Wave 4',
    'W4EngLangHH')]:
    language_decision_rules.append({'Wave': wave, 'Source type': 'Family background', 'Variable': variable,
        'Review outcome': 'Retain as review support only', 'Decision reason': 'Household main-language indicator used to inspect routing and the derivation of home-language variables. It is not retained separately because it overlaps with the constructed non-English home-language measure.', 'Reference-period assessment': f'{wave} household-language measure.', 'Review notes': 'Retained for documentation and quality review only; not included in the predictor matrix.'})
language_decision_indices = []
for rule in language_decision_rules:
    matching_rows = stage_2_variable_decision_register['Wave'].eq(rule['Wave']) & stage_2_variable_decision_register['Source type'].eq(rule['Source type']) & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {rule['Wave']}, {rule['Source type']}, {rule['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    language_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = rule['Review outcome']
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'Demographic background'
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = rule['Decision reason']
    stage_2_variable_decision_register.loc[matching_rows,
        'Leakage assessment'] = 'No direct outcome leakage identified. The measure describes personal or household language background rather than post-16 activity.'
    stage_2_variable_decision_register.loc[matching_rows,
        'Reference-period assessment'] = rule['Reference-period assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave-specific data dictionaries, Wave 1 and Wave 4 derivation documentation, and Stage 2 routing review'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = rule['Review notes']
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
language_measure_review_output_path = stage_2_output_directory / 'stage_2_language_measure_review.csv'
stage_2_language_measure_review.to_csv(language_measure_review_output_path, index=False)
language_decision_output = stage_2_variable_decision_register.loc[language_decision_indices, ['Wave', 'Source type',
    'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment', 'Review notes']].sort_values(['Review outcome',
    'Wave', 'Source type', 'Variable'])
language_exclusions = language_decision_output.loc[language_decision_output['Review outcome'].eq('Exclude from predictor set'),
    ['Wave', 'Source type', 'Variable', 'Decision reason', 'Leakage assessment']]
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Participants in language-measure file: {len(stage_2_language_measure_review):,}')
print(f"Participants with personal language information: {stage_2_language_measure_review['english_first_or_main_language'].notna().sum():,}")
print(f"Participants with household language information: {stage_2_language_measure_review['non_english_home_language'].notna().sum():,}")
print(f"Household cross-wave disagreements retained for quality control: {stage_2_language_measure_review['Household language cross-wave disagreement'].sum():,}")
print('\nLanguage-variable decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 130, 'display.width', 370):
    print(language_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables excluded in this step:')
print(language_exclusions.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'Language-measure review file saved to: {language_measure_review_output_path}')

Participants in language-measure file: 9,767
Participants with personal language information: 9,516
Participants with household language information: 9,670
Household cross-wave disagreements retained for quality control: 142

Language-variable decisions:
  Wave       Source type     Variable                                     Variable label                Review outcome                                                                                                                                                                                                    Decision reason                                                                                                                  Leakage assessment                                                                                                                    Review notes
Wave 1 Family background   W1LangHom1 DV: 1st language other than English spoken at home    Exclude from predictor set                                     

In [113]:
# 41: School-sector and school-type variable list

import re
school_type_name_pattern = re.compile('indschool|schtype|schooltype|schsect|schoolsect|schtyp', flags=re.IGNORECASE)
school_type_label_pattern = re.compile('\\btype of (?:the )?school\\b|\\bschool type\\b|\\btype of school attended\\b|\\bschool sector\\b|\\bmaintained school\\b|\\bindependent school\\b|\\bprivate school\\b|\\bcomprehensive school\\b|\\bgrammar school\\b|\\bselective school\\b|\\bspecial school\\b|\\bpupil referral unit\\b',
    flags=re.IGNORECASE)
school_type_register = stage_2_master_variable_register.copy()
name_match = school_type_register['Variable'].fillna('').apply(lambda value: bool(school_type_name_pattern.search(str(value))))
label_match = school_type_register['Variable label'].fillna('').apply(lambda value: bool(school_type_label_pattern.search(str(value))))
stage_2_school_type_review = school_type_register.loc[name_match | label_match, ['Source order', 'Wave', 'Source type',
    'Source file', 'Source path', 'Variable position', 'Variable', 'Variable label', 'Data type']].copy()
actual_school_pattern = re.compile("\\bcurrent school\\b|\\bschool attended\\b|\\btype of (?:the )?school attended\\b|\\byoung person'?s school\\b|\\byp'?s school\\b|\\bschool sector\\b|\\bmaintained or independent\\b",
    flags=re.IGNORECASE)
preference_pattern = re.compile('\\bprefer\\b|\\bpreference\\b|\\bchoice\\b|\\bwould like\\b|\\bwanted\\b|\\breason\\b',
    flags=re.IGNORECASE)

def classify_school_type_variable(variable, variable_label):
    """Classify each school-type match for manual review."""
    variable_text = str(variable)
    label_text = str(variable_label)
    if preference_pattern.search(label_text):
        return 'School-type preference or choice variable'
    if actual_school_pattern.search(label_text) or variable_text.lower() == 'indschool':
        return 'Possible actual school-sector or school-type measure'
    return 'Other school-type-related variable'
stage_2_school_type_review['Review category'] = stage_2_school_type_review.apply(lambda row: classify_school_type_variable(row['Variable'],
    row['Variable label']), axis=1)
stage_2_school_type_review = stage_2_school_type_review.merge(stage_2_variable_decision_register[['Source file',
    'Variable', 'Review outcome']], on=['Source file',
    'Variable'], how='left', validate='one_to_one').sort_values(['Review category', 'Source order',
    'Variable position']).reset_index(drop=True)
school_type_review_summary = stage_2_school_type_review['Review category'].value_counts().rename_axis('Review category').reset_index(name='Variables')
print(f'School-sector or school-type variables identified: {len(stage_2_school_type_review):,}')
print('\nReview-category summary:')
print(school_type_review_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables identified:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 360):
    print(stage_2_school_type_review[['Wave', 'Source type', 'Variable', 'Variable label', 'Review category',
        'Review outcome']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))

School-sector or school-type variables identified: 62

Review-category summary:
                                     Review category  Variables
                  Other school-type-related variable         51
           School-type preference or choice variable          8
Possible actual school-sector or school-type measure          3

Variables identified:
  Wave        Source type      Variable                                                                Variable label                           Review category Review outcome
Wave 1       Young person W1expwhatMP0b     MP: Result of YP's most recent exclusion - Went to special school or unit        Other school-type-related variable Pending review
Wave 1 Parental attitudes   W1YNtApHS0a HR: Chose independent or private school because YP's friends were going there        Other school-type-related variable Pending review
   ...                ...           ...                                                                           ..

In [114]:
# 42: Independent-school indicator source review

from itertools import combinations
indschool_source_specifications = [{'Wave': 'Wave 1', 'Source type': 'Young person',
    'Source label': 'Wave 1 young-person file'}, {'Wave': 'Wave 1', 'Source type': 'Family background',
    'Source label': 'Wave 1 family-background file'}, {'Wave': 'Wave 1', 'Source type': 'Parental attitudes',
    'Source label': 'Wave 1 parental-attitudes file'}]
indschool_source_data = {}
indschool_coverage_records = []
indschool_value_records = []
indschool_comparison_data = stage_2_participant_ids.copy()
for source_specification in indschool_source_specifications:
    matching_entry = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq(source_specification['Wave']) & stage_2_master_variable_register['Source type'].eq(source_specification['Source type']) & stage_2_master_variable_register['Variable'].str.lower().eq('indschool')]
    if len(matching_entry) != 1:
        raise ValueError(f"Expected one IndSchool entry for {source_specification['Source label']}, but found {len(matching_entry)}.")
    source_path = Path(matching_entry.iloc[0]['Source path'])
    source_variable = matching_entry.iloc[0]['Variable']
    numeric_data = pd.read_stata(source_path, columns=['NSID', source_variable], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', source_variable], convert_categoricals=True)
    sample_numeric_data = stage_2_participant_ids.merge(numeric_data, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = sample_numeric_data['_merge'].eq('both')
    sample_numeric_data = sample_numeric_data.drop(columns='_merge')
    sample_labelled_data = stage_2_participant_ids.merge(labelled_data, on='NSID', how='left', validate='one_to_one')
    numeric_values = pd.to_numeric(sample_numeric_data[source_variable], errors='coerce')
    source_column = source_specification['Source label']
    indschool_comparison_data[source_column] = numeric_values
    indschool_comparison_data[f'{source_column} present'] = source_record_present
    indschool_source_data[source_column] = sample_numeric_data
    indschool_coverage_records.append({'Wave': source_specification['Wave'],
        'Source type': source_specification['Source type'], 'Source': source_column, 'Variable': source_variable, 'Variable label': matching_entry.iloc[0]['Variable label'], 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Stored value': int(numeric_values.notna().sum()), 'Negative code': int(numeric_values.lt(0).sum()), 'Non-negative code': int(numeric_values.ge(0).sum()), 'Distinct stored codes': int(numeric_values.nunique(dropna=True))})
    value_summary = pd.DataFrame({'Value code': sample_numeric_data[source_variable],
        'Value label': sample_labelled_data[source_variable].astype('string'), 'Source record present': source_record_present}).loc[lambda frame: frame['Source record present']].drop(columns='Source record present').value_counts(dropna=False).rename('Participants').reset_index()
    value_summary.insert(0, 'Source', source_column)
    indschool_value_records.append(value_summary)
stage_2_indschool_coverage = pd.DataFrame(indschool_coverage_records)
stage_2_indschool_values = pd.concat(indschool_value_records, ignore_index=True)
indschool_pairwise_records = []
source_columns = [specification['Source label'] for specification in indschool_source_specifications]
for first_source, second_source in combinations(source_columns, 2):
    first_present = indschool_comparison_data[f'{first_source} present']
    second_present = indschool_comparison_data[f'{second_source} present']
    both_present = first_present & second_present
    first_values = indschool_comparison_data[first_source]
    second_values = indschool_comparison_data[second_source]
    values_identical = first_values.eq(second_values) | first_values.isna() & second_values.isna()
    overlapping_records = int(both_present.sum())
    matching_records = int((both_present & values_identical).sum())
    indschool_pairwise_records.append({'First source': first_source, 'Second source': second_source,
        'Both source records present': overlapping_records, 'Values identical': matching_records, 'Values different': overlapping_records - matching_records, 'Agreement percentage': matching_records / overlapping_records * 100 if overlapping_records else np.nan})
stage_2_indschool_pairwise = pd.DataFrame(indschool_pairwise_records)
all_three_present = indschool_comparison_data[[f'{source} present' for source in source_columns]].all(axis=1)
all_three_identical = indschool_comparison_data[source_columns].nunique(axis=1, dropna=False).eq(1)
indschool_disagreements = indschool_comparison_data.loc[all_three_present & ~all_three_identical, ['NSID',
    *source_columns]].copy()
print(f'IndSchool source-variable entries reviewed: {len(stage_2_indschool_coverage):,}')
print('\nCoverage and stored-code summary:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 140, 'display.width', 360):
    print(stage_2_indschool_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nValue codes and labels:')
print(stage_2_indschool_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nPairwise source agreement:')
print(stage_2_indschool_pairwise.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Agreement percentage': lambda value: f'{value:.2f}%'}))
print(f'\nParticipants with all three source records: {all_three_present.sum():,}')
print(f'Participants with identical values across all three sources: {(all_three_present & all_three_identical).sum():,}')
print(f'Participants with a disagreement across the three sources: {len(indschool_disagreements):,}')
if not indschool_disagreements.empty:
    print('\nRecords with differing values:')
    print(indschool_disagreements.head(20).to_string(max_rows=TABLE_ROW_LIMIT, index=False))

IndSchool source-variable entries reviewed: 3

Coverage and stored-code summary:
  Wave        Source type                         Source  Variable                                                              Variable label  Source record present  Source record absent  Stored value  Negative code  Non-negative code  Distinct stored codes
Wave 1       Young person       Wave 1 young-person file IndSchool DV: Whether YP was at an independent or maintained school at sampling stage                   9524                   243          9524              0               9524                      2
Wave 1  Family background  Wave 1 family-background file IndSchool DV: Whether YP was at an independent or maintained school at sampling stage                   9524                   243          9524              0               9524                      2
Wave 1 Parental attitudes Wave 1 parental-attitudes file IndSchool DV: Whether YP was at an independent or maintained school at sampling stage

In [115]:
# 43: School-sector measure decision and documentation

school_sector_code = pd.to_numeric(indschool_comparison_data['Wave 1 young-person file'], errors='coerce')
stage_2_school_sector_measure_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'independent_school_at_sampling_stage': school_sector_code.astype('Int64'), 'School sector': school_sector_code.map({0.0: 'Maintained',
    1.0: 'Independent'}), 'School-sector source': np.where(school_sector_code.notna(), 'Wave 1 IndSchool',
    'No Wave 1 source record')})
stage_2_school_sector_measure_review['School-sector value missing'] = school_sector_code.isna()
school_sector_decision_rules = [{'Wave': 'Wave 1', 'Source type': 'Young person', 'Variable': 'IndSchool',
    'Review outcome': 'Retain as construction source', 'Decision reason': 'Provides a binary measure of whether the young person was at a maintained or independent school at the sampling stage.', 'Review notes': 'Retained as the single source copy. It provides values for 9,524 participants. The 243 participants without a Wave 1 source record remain missing; no later school variable is substituted.'}, {'Wave': 'Wave 1',
    'Source type': 'Family background', 'Variable': 'IndSchool', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Exact duplicate of IndSchool in the Wave 1 young-person file. Retaining both entries would duplicate identical school-sector information.', 'Review notes': 'All 9,524 overlapping participant-level values were identical across the two source files.'}, {'Wave': 'Wave 1',
    'Source type': 'Parental attitudes', 'Variable': 'IndSchool', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Exact duplicate of IndSchool in the Wave 1 young-person file. Retaining both entries would duplicate identical school-sector information.', 'Review notes': 'All 9,524 overlapping participant-level values were identical across the two source files.'}]
school_sector_decision_indices = []
for rule in school_sector_decision_rules:
    matching_rows = stage_2_variable_decision_register['Wave'].eq(rule['Wave']) & stage_2_variable_decision_register['Source type'].eq(rule['Source type']) & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {rule['Wave']}, {rule['Source type']}, {rule['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    school_sector_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = rule['Review outcome']
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'School context'
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = rule['Decision reason']
    stage_2_variable_decision_register.loc[matching_rows,
        'Leakage assessment'] = 'No direct outcome leakage identified. School sector was recorded at the sampling stage before the post-16 outcome.'
    stage_2_variable_decision_register.loc[matching_rows,
        'Reference-period assessment'] = 'School sector at the Wave 1 sampling stage; pre-transition.'
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave 1 data dictionary and Stage 2 duplicate-source review'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = rule['Review notes']
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
school_sector_review_output_path = stage_2_output_directory / 'stage_2_school_sector_measure_review.csv'
stage_2_school_sector_measure_review.to_csv(school_sector_review_output_path, index=False)
school_sector_decision_output = stage_2_variable_decision_register.loc[school_sector_decision_indices, ['Wave',
    'Source type', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment', 'Review notes']].sort_values(['Review outcome',
    'Source type'])
school_sector_exclusions = school_sector_decision_output.loc[school_sector_decision_output['Review outcome'].eq('Exclude from predictor set'),
    ['Wave', 'Source type', 'Variable', 'Decision reason', 'Leakage assessment']]
school_sector_distribution = stage_2_school_sector_measure_review['School sector'].fillna('Missing').value_counts().rename_axis('School sector').reset_index(name='Participants')
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Participants in school-sector review file: {len(stage_2_school_sector_measure_review):,}')
print('\nSchool-sector distribution:')
print(school_sector_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSchool-sector variable decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 130, 'display.width', 370):
    print(school_sector_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables excluded in this step:')
print(school_sector_exclusions.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'School-sector review file saved to: {school_sector_review_output_path}')

Participants in school-sector review file: 9,767

School-sector distribution:
School sector  Participants
   Maintained          9157
  Independent           367
      Missing           243

School-sector variable decisions:
  Wave        Source type  Variable                                                              Variable label                Review outcome                                                                                                                           Decision reason                                                                                                 Leakage assessment                                                                                                                                                                                Review notes
Wave 1  Family background IndSchool DV: Whether YP was at an independent or maintained school at sampling stage    Exclude from predictor set Exact duplicate of IndSchool in the Wave 1 young-p

In [116]:
# 44: Current-school start and duration variable list

import re
school_start_name_pattern = re.compile('stsch|schstart|startsch|schbeg|begsch|joinsch|schdur', flags=re.IGNORECASE)
school_start_label_pattern = re.compile('\\bstarted (?:at |the )?(?:current )?school\\b|\\bstart(?:ed)? current school\\b|\\bmonth (?:young person|yp) started\\b|\\byear (?:young person|yp) started\\b|\\bwhen (?:young person|yp) started\\b|\\bjoined (?:the |this |current )?school\\b|\\bfirst attended (?:the |this |current )?school\\b|\\bhow long .* (?:at|in) .*school\\b|\\blength of time .*school\\b|\\bduration .*school\\b',
    flags=re.IGNORECASE)
school_start_register = stage_2_master_variable_register.copy()
name_match = school_start_register['Variable'].fillna('').apply(lambda value: bool(school_start_name_pattern.search(str(value))))
label_match = school_start_register['Variable label'].fillna('').apply(lambda value: bool(school_start_label_pattern.search(str(value))))
stage_2_school_start_review = school_start_register.loc[name_match | label_match, ['Source order', 'Wave',
    'Source type', 'Source file', 'Source path', 'Variable position', 'Variable', 'Variable label', 'Data type', 'Timing status']].copy()
school_leaving_pattern = re.compile('\\bleft school\\b|\\bleaving school\\b|\\bschool leaving\\b|\\bpost[- ]?16\\b|\\bsixth form\\b|\\bcollege\\b|\\bcurrent course\\b|\\bfinished school\\b',
    flags=re.IGNORECASE)
current_school_pattern = re.compile('\\bcurrent school\\b|\\bthis school\\b|\\bsampling school\\b|\\bschool at sampling\\b',
    flags=re.IGNORECASE)

def classify_school_start_match(variable, variable_label, wave):
    """Classify school-start matches for manual review."""
    variable_text = str(variable)
    label_text = str(variable_label)
    if school_leaving_pattern.search(label_text):
        return 'School-leaving or post-16 timing variable'
    if current_school_pattern.search(label_text) or variable_text.lower() == 'w1stschhs':
        return 'Possible current-school start measure'
    if wave == 'Wave 4':
        return 'Wave 4 school-timing variable requiring leakage review'
    return 'Other school-start-related variable'
stage_2_school_start_review['Review category'] = stage_2_school_start_review.apply(lambda row: classify_school_start_match(row['Variable'],
    row['Variable label'], row['Wave']), axis=1)
stage_2_school_start_review = stage_2_school_start_review.merge(stage_2_variable_decision_register[['Source file',
    'Variable', 'Review outcome']], on=['Source file',
    'Variable'], how='left', validate='one_to_one').sort_values(['Review category', 'Source order',
    'Variable position']).reset_index(drop=True)
school_start_review_summary = stage_2_school_start_review['Review category'].value_counts().rename_axis('Review category').reset_index(name='Variables')
print(f'School-start or duration variables identified: {len(stage_2_school_start_review):,}')
print('\nReview-category summary:')
print(school_start_review_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables identified:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 160, 'display.width', 380):
    print(stage_2_school_start_review[['Wave', 'Source type', 'Variable', 'Variable label', 'Timing status',
        'Review category', 'Review outcome']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))

School-start or duration variables identified: 10

Review-category summary:
                      Review category  Variables
  Other school-start-related variable          8
Possible current-school start measure          2

Variables identified:
  Wave  Source type   Variable                                                         Variable label         Timing status                       Review category Review outcome
Wave 1 Young person W1chstyrMP  MP: Year YP started living with guardian (no natural parents present) Pre-transition source   Other school-start-related variable Pending review
Wave 1 Young person  W1chstmMP MP: Month YP started living with guardian (no natural parents present) Pre-transition source   Other school-start-related variable Pending review
   ...          ...        ...                                                                    ...                   ...                                   ...            ...
Wave 1 Young person W1schstyHS                

In [117]:
# 45: School-start timing and cross-wave update review

school_start_source_specifications = [{'Wave': 'Wave 1', 'Source type': 'Young person', 'Year variable': 'W1schstyHS',
    'Month variable': 'W1stschHS', 'Output prefix': 'W1'}, {'Wave': 'Wave 2', 'Source type': 'Young person',
    'Year variable': 'W2SchstyHS', 'Month variable': 'W2stschHS', 'Output prefix': 'W2'}, {'Wave': 'Wave 3',
    'Source type': 'Young person', 'Year variable': 'W3SchstyHS', 'Month variable': 'W3stschHS', 'Output prefix': 'W3'}]
stage_2_school_start_data = stage_2_participant_ids.copy()
school_start_coverage_records = []
school_start_year_records = []
school_start_month_records = []
for source_specification in school_start_source_specifications:
    requested_variables = {source_specification['Year variable'].lower(),
        source_specification['Month variable'].lower()}
    matching_entries = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq(source_specification['Wave']) & stage_2_master_variable_register['Source type'].eq(source_specification['Source type']) & stage_2_master_variable_register['Variable'].str.lower().isin(requested_variables)].copy()
    if len(matching_entries) != 2:
        raise ValueError(f"Expected two school-start entries for {source_specification['Wave']}, but found {len(matching_entries)}.")
    source_paths = matching_entries['Source path'].drop_duplicates().tolist()
    if len(source_paths) != 1:
        raise ValueError(f"Expected one source file for {source_specification['Wave']}.")
    source_path = Path(source_paths[0])
    year_variable = matching_entries.loc[matching_entries['Variable'].str.lower().eq(source_specification['Year variable'].lower()),
        'Variable'].iloc[0]
    month_variable = matching_entries.loc[matching_entries['Variable'].str.lower().eq(source_specification['Month variable'].lower()),
        'Variable'].iloc[0]
    numeric_data = pd.read_stata(source_path, columns=['NSID', year_variable, month_variable],
        convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', year_variable, month_variable],
        convert_categoricals=True)
    sample_numeric_data = stage_2_participant_ids.merge(numeric_data, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = sample_numeric_data['_merge'].eq('both')
    sample_numeric_data = sample_numeric_data.drop(columns='_merge')
    sample_labelled_data = stage_2_participant_ids.merge(labelled_data, on='NSID', how='left', validate='one_to_one')
    year_values = pd.to_numeric(sample_numeric_data[year_variable], errors='coerce')
    month_values = pd.to_numeric(sample_numeric_data[month_variable], errors='coerce')
    prefix = source_specification['Output prefix']
    stage_2_school_start_data[f'{prefix} school-start year'] = year_values
    stage_2_school_start_data[f'{prefix} school-start month category'] = month_values
    for variable, measure_type, values in [(year_variable, 'Year started school', year_values), (month_variable,
        'Month-category indicator', month_values)]:
        variable_label = matching_entries.loc[matching_entries['Variable'].eq(variable), 'Variable label'].iloc[0]
        school_start_coverage_records.append({'Wave': source_specification['Wave'], 'Variable': variable,
            'Measure type': measure_type, 'Variable label': variable_label, 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Stored value': int(values.notna().sum()), 'Negative code': int(values.lt(0).sum()), 'Non-negative value': int(values.ge(0).sum()), 'Distinct stored codes': int(values.nunique(dropna=True))})
    valid_year_values = year_values.where(year_values.gt(0))
    year_summary = valid_year_values.value_counts().sort_index().rename('Participants').reset_index().rename(columns={year_variable: 'Start year',
        'index': 'Start year'})
    year_summary.insert(0, 'Wave', source_specification['Wave'])
    school_start_year_records.append(year_summary)
    month_summary = pd.DataFrame({'Value code': month_values,
        'Value label': sample_labelled_data[month_variable].astype('string'), 'Source record present': source_record_present}).loc[lambda frame: frame['Source record present']].drop(columns='Source record present').value_counts(dropna=False).rename('Participants').reset_index()
    month_summary.insert(0, 'Wave', source_specification['Wave'])
    month_summary.insert(1, 'Variable', month_variable)
    school_start_month_records.append(month_summary)
stage_2_school_start_coverage = pd.DataFrame(school_start_coverage_records)
stage_2_school_start_year_values = pd.concat(school_start_year_records, ignore_index=True)
stage_2_school_start_month_values = pd.concat(school_start_month_records, ignore_index=True)
w1_year = stage_2_school_start_data['W1 school-start year'].where(stage_2_school_start_data['W1 school-start year'].gt(0))
w2_year = stage_2_school_start_data['W2 school-start year'].where(stage_2_school_start_data['W2 school-start year'].gt(0))
w3_year = stage_2_school_start_data['W3 school-start year'].where(stage_2_school_start_data['W3 school-start year'].gt(0))
w1_month = stage_2_school_start_data['W1 school-start month category'].where(stage_2_school_start_data['W1 school-start month category'].isin([0,
    1]))
w2_month = stage_2_school_start_data['W2 school-start month category'].where(stage_2_school_start_data['W2 school-start month category'].isin([0,
    1]))
w3_month = stage_2_school_start_data['W3 school-start month category'].where(stage_2_school_start_data['W3 school-start month category'].isin([0,
    1]))
year_comparison_records = []
for comparison, first_year, second_year in [('Wave 1 versus Wave 2', w1_year, w2_year), ('Wave 1 versus Wave 3',
    w1_year, w3_year), ('Wave 2 versus Wave 3', w2_year, w3_year)]:
    both_valid = first_year.notna() & second_year.notna()
    year_comparison_records.append({'Comparison': comparison, 'Both years valid': int(both_valid.sum()),
        'Same year': int((both_valid & first_year.eq(second_year)).sum()), 'Second wave later': int((both_valid & second_year.gt(first_year)).sum()), 'Second wave earlier': int((both_valid & second_year.lt(first_year)).sum())})
stage_2_school_start_year_comparisons = pd.DataFrame(year_comparison_records)
month_comparison_records = []
for comparison, first_month, second_month in [('Wave 1 versus Wave 2', w1_month, w2_month), ('Wave 1 versus Wave 3',
    w1_month, w3_month), ('Wave 2 versus Wave 3', w2_month, w3_month)]:
    both_valid = first_month.notna() & second_month.notna()
    agreeing = both_valid & first_month.eq(second_month)
    valid_comparisons = int(both_valid.sum())
    agreement_count = int(agreeing.sum())
    month_comparison_records.append({'Comparison': comparison, 'Both categories valid': valid_comparisons,
        'Categories agreeing': agreement_count, 'Categories differing': valid_comparisons - agreement_count, 'Agreement percentage': agreement_count / valid_comparisons * 100 if valid_comparisons else np.nan})
stage_2_school_start_month_comparisons = pd.DataFrame(month_comparison_records)
latest_year_before_wave_3 = w2_year.combine_first(w1_year)
school_start_update_summary = pd.DataFrame([{'Update status': 'Wave 2 valid where Wave 1 is unavailable',
    'Participants': int((w2_year.notna() & w1_year.isna()).sum())}, {'Update status': 'Wave 2 same as valid Wave 1',
    'Participants': int((w2_year.notna() & w1_year.notna() & w2_year.eq(w1_year)).sum())}, {'Update status': 'Wave 2 different from valid Wave 1',
    'Participants': int((w2_year.notna() & w1_year.notna() & w2_year.ne(w1_year)).sum())}, {'Update status': 'Wave 3 valid with no earlier valid year',
    'Participants': int((w3_year.notna() & latest_year_before_wave_3.isna()).sum())}, {'Update status': 'Wave 3 same as latest earlier valid year',
    'Participants': int((w3_year.notna() & latest_year_before_wave_3.notna() & w3_year.eq(latest_year_before_wave_3)).sum())}, {'Update status': 'Wave 3 different from latest earlier valid year',
    'Participants': int((w3_year.notna() & latest_year_before_wave_3.notna() & w3_year.ne(latest_year_before_wave_3)).sum())}])
print(f'School-start source variables reviewed: {len(stage_2_school_start_coverage):,}')
print('\nCoverage and code summary:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 370):
    print(stage_2_school_start_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nPositive school-start year distributions:')
print(stage_2_school_start_year_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMonth-category codes and labels:')
print(stage_2_school_start_month_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCross-wave year comparison:')
print(stage_2_school_start_year_comparisons.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCross-wave month-category comparison:')
print(stage_2_school_start_month_comparisons.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Agreement percentage': lambda value: f'{value:.2f}%'}))
print('\nLater-wave update structure:')
print(school_start_update_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

School-start source variables reviewed: 6

Coverage and code summary:
  Wave   Variable             Measure type                     Variable label  Source record present  Source record absent  Stored value  Negative code  Non-negative value  Distinct stored codes
Wave 1 W1schstyHS      Year started school HR: Year YP started current school                   9524                   243          9524            252                9272                     19
Wave 1  W1stschHS Month-category indicator        DV: Month YP started school                   9524                   243          9524            263                9261                      7
   ...        ...                      ...                                ...                    ...                   ...           ...            ...                 ...                    ...
Wave 3 W3SchstyHS      Year started school         YP: Year YP started school                   9509                   258          9509           936

In [118]:
# 46: School-change and routing variable list

import re
school_change_label_pattern = re.compile('\\bsame school\\b|\\bstill at (?:the )?(?:same|previous|current) school\\b|\\bchanged school\\b|\\bchange(?:d)? (?:of )?school\\b|\\bmoved (?:to )?(?:a )?(?:new|different) school\\b|\\bnew school\\b|\\bleft (?:the |their |current )?school\\b|\\bstarted at (?:a )?(?:new|different) school\\b|\\bschool attended at (?:the )?previous wave\\b|\\bschool since (?:the )?(?:last|previous) interview\\b|\\bno longer at (?:the )?(?:same|current) school\\b',
    flags=re.IGNORECASE)
school_change_name_pattern = re.compile('samesch|schsame|chgsch|schchg|chsch|movsch|schmov|newsch|schnew|leftsch|schleft',
    flags=re.IGNORECASE)
school_change_register = stage_2_master_variable_register.copy()
name_match = school_change_register['Variable'].fillna('').apply(lambda value: bool(school_change_name_pattern.search(str(value))))
label_match = school_change_register['Variable label'].fillna('').apply(lambda value: bool(school_change_label_pattern.search(str(value))))
stage_2_school_change_review = school_change_register.loc[name_match | label_match, ['Source order', 'Wave',
    'Source type', 'Source file', 'Source path', 'Variable position', 'Variable', 'Variable label', 'Data type', 'Timing status']].copy()
direct_change_pattern = re.compile('\\bsame school\\b|\\bstill at\\b|\\bchanged school\\b|\\bchange(?:d)? (?:of )?school\\b|\\bmoved .*school\\b|\\bnew school\\b|\\bno longer at\\b',
    flags=re.IGNORECASE)
reason_pattern = re.compile('\\breason\\b|\\bwhy\\b|\\bbecause\\b', flags=re.IGNORECASE)
timing_pattern = re.compile('\\bwhen\\b|\\bmonth\\b|\\byear\\b|\\bdate\\b|\\bhow long\\b', flags=re.IGNORECASE)
possible_outcome_overlap_pattern = re.compile('\\bleft school\\b|\\bleaving school\\b|\\bfinished school\\b|\\bpost[- ]?16\\b|\\bsixth form\\b|\\bcollege\\b',
    flags=re.IGNORECASE)

def classify_school_change_match(variable_label):
    """Classify each school-change match for manual review."""
    label_text = str(variable_label)
    if possible_outcome_overlap_pattern.search(label_text):
        return 'Possible school-leaving or outcome-overlap variable'
    if reason_pattern.search(label_text):
        return 'Reason for school change'
    if timing_pattern.search(label_text):
        return 'School-change timing variable'
    if direct_change_pattern.search(label_text):
        return 'Possible direct school-change or routing indicator'
    return 'Other school-change-related variable'
stage_2_school_change_review['Review category'] = stage_2_school_change_review['Variable label'].apply(classify_school_change_match)
stage_2_school_change_review = stage_2_school_change_review.merge(stage_2_variable_decision_register[['Source file',
    'Variable', 'Review outcome']], on=['Source file',
    'Variable'], how='left', validate='one_to_one').sort_values(['Review category', 'Source order',
    'Variable position']).reset_index(drop=True)
school_change_review_summary = stage_2_school_change_review['Review category'].value_counts().rename_axis('Review category').reset_index(name='Variables')
print(f'School-change or routing variables identified: {len(stage_2_school_change_review):,}')
print('\nReview-category summary:')
print(school_change_review_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables identified:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 160, 'display.width', 390):
    print(stage_2_school_change_review[['Wave', 'Source type', 'Variable', 'Variable label', 'Timing status',
        'Review category', 'Review outcome']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))

School-change or routing variables identified: 37

Review-category summary:
                                    Review category  Variables
Possible school-leaving or outcome-overlap variable         24
 Possible direct school-change or routing indicator         10
                           Reason for school change          2
                      School-change timing variable          1

Variables identified:
  Wave  Source type      Variable                                                                 Variable label          Timing status                                    Review category Review outcome
Wave 1 Young person W1expwhatMP0f MP: Result of YP's most recent exclusion - Eventually went back to same school  Pre-transition source Possible direct school-change or routing indicator Pending review
Wave 2 Young person   W2schnameMP                                            MP: Whether YP still at same school  Pre-transition source Possible direct school-change or routing indic

In [119]:
# 47: School-start routing structure review

school_routing_specifications = [{'Wave': 'Wave 2', 'Source type': 'Young person', 'Variable': 'W2schnameMP',
    'Output prefix': 'W2'}, {'Wave': 'Wave 3', 'Source type': 'Young person', 'Variable': 'W3schnameYP',
    'Output prefix': 'W3'}]
stage_2_school_routing_data = stage_2_school_start_data.copy()
school_routing_value_records = []
school_routing_coverage_records = []
school_routing_tables = {}
for specification in school_routing_specifications:
    matching_entry = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq(specification['Wave']) & stage_2_master_variable_register['Source type'].eq(specification['Source type']) & stage_2_master_variable_register['Variable'].str.lower().eq(specification['Variable'].lower())]
    if len(matching_entry) != 1:
        raise ValueError(f"Expected one source entry for {specification['Wave']}, {specification['Variable']}; found {len(matching_entry)}.")
    source_path = Path(matching_entry.iloc[0]['Source path'])
    source_variable = matching_entry.iloc[0]['Variable']
    numeric_data = pd.read_stata(source_path, columns=['NSID', source_variable], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', source_variable], convert_categoricals=True)
    sample_numeric_data = stage_2_participant_ids.merge(numeric_data, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = sample_numeric_data['_merge'].eq('both')
    sample_numeric_data = sample_numeric_data.drop(columns='_merge')
    sample_labelled_data = stage_2_participant_ids.merge(labelled_data, on='NSID', how='left', validate='one_to_one')
    routing_code = pd.to_numeric(sample_numeric_data[source_variable], errors='coerce')
    routing_label = sample_labelled_data[source_variable].astype('string')
    prefix = specification['Output prefix']
    stage_2_school_routing_data[f'{prefix} routing code'] = routing_code
    stage_2_school_routing_data[f'{prefix} routing label'] = routing_label
    stage_2_school_routing_data[f'{prefix} routing source present'] = source_record_present
    school_routing_coverage_records.append({'Wave': specification['Wave'], 'Variable': source_variable,
        'Variable label': matching_entry.iloc[0]['Variable label'], 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Stored value': int(routing_code.notna().sum()), 'Negative code': int(routing_code.lt(0).sum()), 'Non-negative value': int(routing_code.ge(0).sum()), 'Distinct stored codes': int(routing_code.nunique(dropna=True))})
    value_summary = pd.DataFrame({'Value code': routing_code, 'Value label': routing_label,
        'Source record present': source_record_present}).loc[lambda frame: frame['Source record present']].drop(columns='Source record present').value_counts(dropna=False).rename('Participants').reset_index()
    value_summary.insert(0, 'Wave', specification['Wave'])
    value_summary.insert(1, 'Variable', source_variable)
    school_routing_value_records.append(value_summary)
w1_start_year = stage_2_school_routing_data['W1 school-start year'].where(stage_2_school_routing_data['W1 school-start year'].gt(0))
w2_start_year = stage_2_school_routing_data['W2 school-start year'].where(stage_2_school_routing_data['W2 school-start year'].gt(0))
w3_start_year = stage_2_school_routing_data['W3 school-start year'].where(stage_2_school_routing_data['W3 school-start year'].gt(0))
w2_start_month = stage_2_school_routing_data['W2 school-start month category'].where(stage_2_school_routing_data['W2 school-start month category'].isin([0,
    1]))
w3_start_month = stage_2_school_routing_data['W3 school-start month category'].where(stage_2_school_routing_data['W3 school-start month category'].isin([0,
    1]))
latest_year_before_wave_3 = w2_start_year.combine_first(w1_start_year)

def create_start_update_status(current_year, current_month, previous_year):
    """Classify whether the current wave supplies new start information."""
    return pd.Series(np.select([current_year.notna() & previous_year.isna(),
        current_year.notna() & previous_year.notna() & current_year.eq(previous_year), current_year.notna() & previous_year.notna() & current_year.gt(previous_year), current_year.notna() & previous_year.notna() & current_year.lt(previous_year), current_year.isna() & current_month.notna()], ['Valid year with no earlier valid year',
        'Same as earlier valid year', 'Later than earlier valid year', 'Earlier than earlier valid year', 'Valid month category only'], default='No valid start information'), index=current_year.index)
stage_2_school_routing_data['W2 school-start update status'] = create_start_update_status(current_year=w2_start_year,
    current_month=w2_start_month, previous_year=w1_start_year)
stage_2_school_routing_data['W3 school-start update status'] = create_start_update_status(current_year=w3_start_year,
    current_month=w3_start_month, previous_year=latest_year_before_wave_3)
for prefix in ['W2', 'W3']:
    complete_label_column = f'{prefix} routing category'
    stage_2_school_routing_data[complete_label_column] = stage_2_school_routing_data[f'{prefix} routing label'].astype('string')
    no_source_record = ~stage_2_school_routing_data[f'{prefix} routing source present']
    stage_2_school_routing_data.loc[no_source_record, complete_label_column] = 'No source record'
    stored_label_missing = stage_2_school_routing_data[complete_label_column].isna() & ~no_source_record
    stage_2_school_routing_data.loc[stored_label_missing,
        complete_label_column] = 'Stored value without readable label'
for prefix in ['W2', 'W3']:
    routing_table = pd.crosstab(stage_2_school_routing_data[f'{prefix} routing category'],
        stage_2_school_routing_data[f'{prefix} school-start update status'], margins=True, margins_name='Total')
    school_routing_tables[prefix] = routing_table
routing_availability_records = []
for prefix, year_values, month_values in [('W2', w2_start_year, w2_start_month), ('W3', w3_start_year,
    w3_start_month)]:
    routing_category = stage_2_school_routing_data[f'{prefix} routing category']
    availability_data = pd.DataFrame({'Routing category': routing_category, 'Valid start year': year_values.notna(),
        'Valid month category': month_values.notna(), 'Any valid start information': year_values.notna() | month_values.notna()})
    availability_summary = availability_data.groupby('Routing category',
        dropna=False).agg(Participants=('Routing category', 'size'), Valid_start_year=('Valid start year', 'sum'),
        Valid_month_category=('Valid month category',
        'sum'), Any_valid_start_information=('Any valid start information', 'sum')).reset_index()
    availability_summary.insert(0, 'Wave', prefix)
    routing_availability_records.append(availability_summary)
stage_2_school_routing_coverage = pd.DataFrame(school_routing_coverage_records)
stage_2_school_routing_values = pd.concat(school_routing_value_records, ignore_index=True)
stage_2_school_routing_availability = pd.concat(routing_availability_records, ignore_index=True)
print(f'School-start routing variables reviewed: {len(stage_2_school_routing_coverage):,}')
print('\nCoverage and code summary:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 370):
    print(stage_2_school_routing_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nValue codes and labels:')
print(stage_2_school_routing_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nRouting response by availability of school-start information:')
print(stage_2_school_routing_availability.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 2 routing by school-start update status:')
print(school_routing_tables['W2'].to_string(max_rows=TABLE_ROW_LIMIT))
print('\nWave 3 routing by school-start update status:')
print(school_routing_tables['W3'].to_string(max_rows=TABLE_ROW_LIMIT))

School-start routing variables reviewed: 2

Coverage and code summary:
  Wave    Variable                      Variable label  Source record present  Source record absent  Stored value  Negative code  Non-negative value  Distinct stored codes
Wave 2 W2schnameMP MP: Whether YP still at same school                   9521                   246          9521             79                9442                      3
Wave 3 W3schnameYP YP: Whether YP still at same school                   9509                   258          9509             54                9455                      4

Value codes and labels:
  Wave    Variable  Value code                Value label  Participants
Wave 2 W2schnameMP         1.0                        Yes          9201
Wave 2 W2schnameMP         2.0                         No           241
   ...         ...         ...                        ...           ...
Wave 3 W3schnameYP       -99.0         YP not interviewed            54
Wave 3 W3schnameYP         3

In [120]:
# 48: Pre-transition school-change measure review

w2_same_school_code = pd.to_numeric(stage_2_school_routing_data['W2 routing code'], errors='coerce')
w3_same_school_code = pd.to_numeric(stage_2_school_routing_data['W3 routing code'], errors='coerce')
w2_school_change = w2_same_school_code.map({1.0: 0, 2.0: 1}).astype('Int64')
w3_school_change = w3_same_school_code.map({1.0: 0, 2.0: 1}).astype('Int64')
w2_routing_valid = w2_school_change.notna()
w3_routing_valid = w3_school_change.notna()
school_change_waves_observed = w2_routing_valid.astype(int) + w3_routing_valid.astype(int)
school_change_count_observed = (w2_school_change.fillna(0) + w3_school_change.fillna(0)).astype('Int64')
any_school_change_by_wave3 = pd.Series(pd.NA, index=stage_2_school_routing_data.index, dtype='Int64')
known_school_change = w2_school_change.eq(1).fillna(False) | w3_school_change.eq(1).fillna(False)
confirmed_no_school_change = w2_school_change.eq(0).fillna(False) & w3_school_change.eq(0).fillna(False)
any_school_change_by_wave3.loc[known_school_change] = 1
any_school_change_by_wave3.loc[confirmed_no_school_change] = 0
school_mobility_pattern = pd.Series('Incomplete school-change information', index=stage_2_school_routing_data.index,
    dtype='string')
school_mobility_pattern.loc[w2_school_change.eq(0).fillna(False) & w3_school_change.eq(0).fillna(False)] = 'Same school reported at both waves'
school_mobility_pattern.loc[w2_school_change.eq(1).fillna(False) & w3_school_change.eq(0).fillna(False)] = 'Change reported at Wave 2 only'
school_mobility_pattern.loc[w2_school_change.eq(0).fillna(False) & w3_school_change.eq(1).fillna(False)] = 'Change reported at Wave 3 only'
school_mobility_pattern.loc[w2_school_change.eq(1).fillna(False) & w3_school_change.eq(1).fillna(False)] = 'Change reported at both waves'
school_mobility_pattern.loc[known_school_change & (~w2_routing_valid | ~w3_routing_valid)] = 'Change reported; other wave unavailable'
school_mobility_pattern.loc[school_change_waves_observed.eq(0)] = 'No valid school-change response'
stage_2_school_change_measure_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'school_change_wave2': w2_school_change, 'school_change_wave3': w3_school_change, 'school_change_waves_observed': school_change_waves_observed, 'school_change_count_observed': school_change_count_observed, 'any_school_change_by_wave3': any_school_change_by_wave3, 'School mobility pattern': school_mobility_pattern})
school_change_cross_tabulation = pd.crosstab(w2_school_change.map({0: 'Same school', 1: 'Changed school'}),
    w3_school_change.map({0: 'Same school', 1: 'Changed school'}), margins=True, margins_name='Total')
w2_school_change_distribution = w2_school_change.map({0: 'Same school',
    1: 'Changed school'}).fillna('Missing or invalid').value_counts().rename_axis('Wave 2 school-change status').reset_index(name='Participants')
w3_school_change_distribution = w3_school_change.map({0: 'Same school',
    1: 'Changed school'}).fillna('Missing or invalid').value_counts().rename_axis('Wave 3 school-change status').reset_index(name='Participants')
combined_school_change_distribution = any_school_change_by_wave3.map({0: 'No change confirmed at both waves',
    1: 'At least one change reported'}).fillna('Incomplete information without recorded change').value_counts().rename_axis('Combined school-change status').reset_index(name='Participants')
school_mobility_pattern_distribution = school_mobility_pattern.value_counts().rename_axis('School mobility pattern').reset_index(name='Participants')
observed_wave_distribution = school_change_waves_observed.value_counts().sort_index().rename_axis('Valid routing waves').reset_index(name='Participants')
print(f'Participants in school-change review: {len(stage_2_school_change_measure_review):,}')
print('\nWave 2 school-change distribution:')
print(w2_school_change_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 3 school-change distribution:')
print(w3_school_change_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 2 by Wave 3 school-change response:')
print(school_change_cross_tabulation.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nNumber of waves with a valid response:')
print(observed_wave_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCombined indicator:')
print(combined_school_change_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSchool-mobility patterns:')
print(school_mobility_pattern_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Participants in school-change review: 9,767

Wave 2 school-change distribution:
Wave 2 school-change status  Participants
                Same school          9201
         Missing or invalid           325
             Changed school           241

Wave 3 school-change distribution:
Wave 3 school-change status  Participants
                Same school          9310
         Missing or invalid           354
             Changed school           103

Wave 2 by Wave 3 school-change response:
W3 routing code  Changed school  Same school  Total
W2 routing code                                    
Changed school               18          210    228
Same school                  83         9024   9107
Total                       101         9234   9335

Number of waves with a valid response:
 Valid routing waves  Participants
                   0           247
                   1           185
                   2          9335

Combined indicator:
                 Combined school-change statu

In [121]:
# 49: Pre-transition school-change decision and documentation

stage_2_school_change_measure_review['Any school change by Wave 3'] = stage_2_school_change_measure_review['any_school_change_by_wave3'].map({0: 'No change confirmed at both waves',
    1: 'At least one change reported'})
stage_2_school_change_measure_review['School-change information complete'] = stage_2_school_change_measure_review['school_change_waves_observed'].eq(2)
school_change_decision_rules = [{'Wave': 'Wave 2', 'Source type': 'Young person', 'Variable': 'W2schnameMP',
    'Review outcome': 'Retain as construction source', 'Decision reason': 'Provides the Wave 2 component of a cumulative pre-transition school-change indicator. A response of No identifies that the young person was no longer at the same school.', 'Reference-period assessment': 'Wave 2 pre-transition school status.', 'Review notes': 'Valid Yes or No responses are used. Main-parent non-response codes remain missing. The variable is combined with W3schnameYP rather than retained as a separate predictor.'}, {'Wave': 'Wave 3',
    'Source type': 'Young person', 'Variable': 'W3schnameYP', 'Review outcome': 'Retain as construction source', 'Decision reason': 'Provides the Wave 3 component of a cumulative pre-transition school-change indicator. A response of No identifies a further school change before the post-16 transition.', 'Reference-period assessment': 'Wave 3 school status measured near, but before, the post-16 transition.', 'Review notes': "Valid Yes or No responses are used. The value 'School name makes no sense' and survey non-response codes remain missing. The variable is combined with W2schnameMP rather than retained separately."}]
school_start_support_rules = [{'Wave': 'Wave 1', 'Variable': 'W1schstyHS'}, {'Wave': 'Wave 1',
    'Variable': 'W1stschHS'}, {'Wave': 'Wave 2', 'Variable': 'W2SchstyHS'}, {'Wave': 'Wave 2',
    'Variable': 'W2stschHS'}, {'Wave': 'Wave 3', 'Variable': 'W3SchstyHS'}, {'Wave': 'Wave 3',
    'Variable': 'W3stschHS'}]
for specification in school_start_support_rules:
    school_change_decision_rules.append({'Wave': specification['Wave'], 'Source type': 'Young person',
        'Variable': specification['Variable'], 'Review outcome': 'Retain as review support only', 'Decision reason': 'School-start timing variable used to verify the routing and update structure of the direct school-change indicators. It is not retained as a separate predictor because the variables do not represent comparable repeated measures across waves and overlap with the constructed school-change measure.', 'Reference-period assessment': f"{specification['Wave']} school-start information.", 'Review notes': 'Wave 1 mainly records the baseline current-school start date, whereas Wave 2 and Wave 3 values are largely conditional additions or updates. Retained for audit and quality review only.'})
school_change_decision_indices = []
for rule in school_change_decision_rules:
    matching_rows = stage_2_variable_decision_register['Wave'].eq(rule['Wave']) & stage_2_variable_decision_register['Source type'].eq(rule['Source type']) & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {rule['Wave']}, {rule['Source type']}, {rule['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    school_change_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = rule['Review outcome']
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'School context'
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = rule['Decision reason']
    stage_2_variable_decision_register.loc[matching_rows,
        'Leakage assessment'] = 'No direct outcome leakage identified. The information was recorded by Wave 3, before the post-16 outcome period.'
    stage_2_variable_decision_register.loc[matching_rows,
        'Reference-period assessment'] = rule['Reference-period assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave 2 and Wave 3 data dictionaries and Stage 2 school-start routing review'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = rule['Review notes']
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
school_change_review_output_path = stage_2_output_directory / 'stage_2_school_change_measure_review.csv'
stage_2_school_change_measure_review.to_csv(school_change_review_output_path, index=False)
school_change_decision_output = stage_2_variable_decision_register.loc[school_change_decision_indices, ['Wave',
    'Source type', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment', 'Review notes']].sort_values(['Review outcome',
    'Wave', 'Variable'])
school_start_review_support_output = school_change_decision_output.loc[school_change_decision_output['Review outcome'].eq('Retain as review support only'),
    ['Wave', 'Variable', 'Variable label', 'Decision reason', 'Leakage assessment']]
school_change_distribution = stage_2_school_change_measure_review['Any school change by Wave 3'].fillna('Incomplete information without recorded change').value_counts().rename_axis('School-change measure').reset_index(name='Participants')
school_change_source_coverage = stage_2_school_change_measure_review['school_change_waves_observed'].value_counts().sort_index().rename_axis('Valid school-change source waves').reset_index(name='Participants')
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Participants in school-change review file: {len(stage_2_school_change_measure_review):,}')
print('\nConstructed school-change distribution:')
print(school_change_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nValid school-change source waves:')
print(school_change_source_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSchool-change variable decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 135, 'display.width', 380):
    print(school_change_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSchool-start variables retained for review support only and not entering the predictor matrix:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 135, 'display.width', 360):
    print(school_start_review_support_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'School-change review file saved to: {school_change_review_output_path}')

Participants in school-change review file: 9,767

Constructed school-change distribution:
                         School-change measure  Participants
             No change confirmed at both waves          9024
Incomplete information without recorded change           417
                  At least one change reported           326

Valid school-change source waves:
 Valid school-change source waves  Participants
                                0           247
                                1           185
                                2          9335

School-change variable decisions:
  Wave  Source type    Variable                      Variable label                Review outcome                                                                                                                                                                                                                                                                                          Decision reason          

In [122]:
# 50: Young-person disability and SEN variable list

import re
disability_sen_name_pattern = re.compile('disab|disabl|longill|limill|healthlim|healthprob|specialneed|sen\\d|sen[a-z]|statement',
    flags=re.IGNORECASE)
disability_sen_label_pattern = re.compile('\\bdisab(?:ility|led)\\b|\\blong[- ]standing illness\\b|\\blong[- ]term illness\\b|\\bchronic illness\\b|\\bhealth (?:problem|condition).*(?:limit|affect)\\b|\\blimit(?:s|ed|ing)?.*(?:daily|normal|school) activit|\\bspecial educational needs?\\b|\\bspecial needs?\\b|\\bstatement of (?:special educational needs|sen)\\b|\\bstatemented\\b|\\bsen provision\\b|\\bsen support\\b',
    flags=re.IGNORECASE)
disability_sen_register = stage_2_master_variable_register.copy()
name_match = disability_sen_register['Variable'].fillna('').apply(lambda value: bool(disability_sen_name_pattern.search(str(value))))
label_match = disability_sen_register['Variable label'].fillna('').apply(lambda value: bool(disability_sen_label_pattern.search(str(value))))
stage_2_disability_sen_review = disability_sen_register.loc[name_match | label_match, ['Source order', 'Wave',
    'Source type', 'Source file', 'Source path', 'Variable position', 'Variable', 'Variable label', 'Data type', 'Timing status']].copy()
young_person_target_pattern = re.compile('\\byoung person\\b|\\byp\\b|\\brespondent\\b', flags=re.IGNORECASE)
parent_target_pattern = re.compile('\\bmain parent\\b|\\bsecond parent\\b|\\bmother\\b|\\bfather\\b|\\bpartner\\b|\\bmp\\b|\\bsp\\b',
    flags=re.IGNORECASE)
sen_type_pattern = re.compile('\\bnature of .*special needs\\b|\\btype of .*special needs\\b|\\bspecial needs?:\\b|\\bsen type\\b',
    flags=re.IGNORECASE)
sen_status_pattern = re.compile('\\bhas special educational needs\\b|\\bhas special needs\\b|\\bwhether .*special educational needs\\b|\\bwhether .*special needs\\b|\\bidentified as having .*sen\\b',
    flags=re.IGNORECASE)
statement_support_pattern = re.compile('\\bstatement of\\b|\\bstatemented\\b|\\bsen support\\b|\\bschool action\\b|\\bschool provision\\b|\\badditional support\\b',
    flags=re.IGNORECASE)
disability_status_pattern = re.compile('\\bwhether .*disab|\\bhas .*disab|\\blong[- ]standing illness\\b|\\blong[- ]term illness\\b|\\bhealth (?:problem|condition).*(?:limit|affect)\\b|\\blimit(?:s|ed|ing)?.*(?:daily|normal|school) activit',
    flags=re.IGNORECASE)
preference_or_reason_pattern = re.compile('\\breason\\b|\\bwhy\\b|\\bchoice\\b|\\bpreferred\\b|\\bwanted\\b|\\bsuited to\\b',
    flags=re.IGNORECASE)

def classify_disability_sen_match(variable, variable_label):
    """Classify disability and SEN matches for manual review."""
    variable_text = str(variable)
    label_text = str(variable_label)
    if preference_or_reason_pattern.search(label_text):
        return 'Preference, reason or contextual variable'
    if parent_target_pattern.search(label_text) and (not young_person_target_pattern.search(label_text)):
        return 'Parent or other-person health measure'
    if sen_type_pattern.search(label_text):
        return 'Possible young-person SEN-type item'
    if statement_support_pattern.search(label_text):
        return 'Possible SEN statement or support measure'
    if sen_status_pattern.search(label_text):
        return 'Possible young-person SEN-status measure'
    if disability_status_pattern.search(label_text):
        return 'Possible young-person disability or limiting-health measure'
    if re.search('sen', variable_text, flags=re.IGNORECASE):
        return 'Other SEN-related variable'
    return 'Other disability-related variable'
stage_2_disability_sen_review['Review category'] = stage_2_disability_sen_review.apply(lambda row: classify_disability_sen_match(row['Variable'],
    row['Variable label']), axis=1)
stage_2_disability_sen_review = stage_2_disability_sen_review.merge(stage_2_variable_decision_register[['Source file',
    'Variable', 'Review outcome']], on=['Source file',
    'Variable'], how='left', validate='one_to_one').sort_values(['Review category', 'Source order',
    'Variable position']).reset_index(drop=True)
disability_sen_review_summary = stage_2_disability_sen_review['Review category'].value_counts().rename_axis('Review category').reset_index(name='Variables')
print(f'Disability or SEN-related variables identified: {len(stage_2_disability_sen_review):,}')
print('\nReview-category summary:')
print(disability_sen_review_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables identified:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 165, 'display.width', 400):
    print(stage_2_disability_sen_review[['Wave', 'Source type', 'Variable', 'Variable label', 'Timing status',
        'Review category', 'Review outcome']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Disability or SEN-related variables identified: 137

Review-category summary:
                          Review category  Variables
    Parent or other-person health measure         50
      Possible young-person SEN-type item         34
                                      ...        ...
 Possible young-person SEN-status measure          7
Preference, reason or contextual variable          4

Variables identified:
  Wave        Source type       Variable                                                                Variable label         Timing status                           Review category Review outcome
Wave 1       Young person     W1senageMP       MP: Age of YP when first identified as having special educational needs Pre-transition source                Other SEN-related variable Pending review
Wave 1       Young person   W1sencurr2MP          MP: Satisfaction with how YP's current school deals with their needs Pre-transition source                Other SEN-related variable Pe

In [123]:
# 51: Core young-person disability-measure review

from itertools import combinations
disability_measure_specifications = [{'Wave': 'Wave 1', 'Source type': 'Young person', 'Variable': 'W1disabYP',
    'Measure form': 'Schooling impact', 'Output name': 'W1 disability and schooling'}, {'Wave': 'Wave 2',
    'Source type': 'Young person', 'Variable': 'W2disabYP', 'Measure form': 'Schooling impact', 'Output name': 'W2 disability and schooling'}, {'Wave': 'Wave 4',
    'Source type': 'Young person', 'Variable': 'W4disabYP', 'Measure form': 'Schooling impact', 'Output name': 'W4 disability and schooling'}, {'Wave': 'Wave 4',
    'Source type': 'Young person', 'Variable': 'W4disab2YP', 'Measure form': 'Daily-activity limitation', 'Output name': 'W4 disability and daily activity'}]
stage_2_disability_measure_data = stage_2_participant_ids.copy()
disability_coverage_records = []
disability_value_records = []
for specification in disability_measure_specifications:
    matching_entry = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq(specification['Wave']) & stage_2_master_variable_register['Source type'].eq(specification['Source type']) & stage_2_master_variable_register['Variable'].str.lower().eq(specification['Variable'].lower())]
    if len(matching_entry) != 1:
        raise ValueError(f"Expected one source entry for {specification['Wave']}, {specification['Variable']}; found {len(matching_entry)}.")
    source_path = Path(matching_entry.iloc[0]['Source path'])
    source_variable = matching_entry.iloc[0]['Variable']
    numeric_data = pd.read_stata(source_path, columns=['NSID', source_variable], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', source_variable], convert_categoricals=True)
    sample_numeric_data = stage_2_participant_ids.merge(numeric_data, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = sample_numeric_data['_merge'].eq('both')
    sample_numeric_data = sample_numeric_data.drop(columns='_merge')
    sample_labelled_data = stage_2_participant_ids.merge(labelled_data, on='NSID', how='left', validate='one_to_one')
    numeric_values = pd.to_numeric(sample_numeric_data[source_variable], errors='coerce')
    output_name = specification['Output name']
    stage_2_disability_measure_data[output_name] = numeric_values
    stage_2_disability_measure_data[f'{output_name} source present'] = source_record_present
    disability_coverage_records.append({'Wave': specification['Wave'], 'Variable': source_variable,
        'Measure form': specification['Measure form'], 'Variable label': matching_entry.iloc[0]['Variable label'], 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Stored value': int(numeric_values.notna().sum()), 'Valid category': int(numeric_values.isin([1,
        2, 3]).sum()), 'Negative survey code': int(numeric_values.lt(0).sum()), 'Distinct stored codes': int(numeric_values.nunique(dropna=True))})
    value_summary = pd.DataFrame({'Value code': numeric_values,
        'Value label': sample_labelled_data[source_variable].astype('string'), 'Source record present': source_record_present}).loc[lambda frame: frame['Source record present']].drop(columns='Source record present').value_counts(dropna=False).rename('Participants').reset_index()
    value_summary.insert(0, 'Wave', specification['Wave'])
    value_summary.insert(1, 'Variable', source_variable)
    value_summary.insert(2, 'Measure form', specification['Measure form'])
    disability_value_records.append(value_summary)
stage_2_disability_coverage = pd.DataFrame(disability_coverage_records)
stage_2_disability_values = pd.concat(disability_value_records, ignore_index=True)
schooling_measure_columns = ['W1 disability and schooling', 'W2 disability and schooling',
    'W4 disability and schooling']
valid_disability_measures = {}
for column in [*schooling_measure_columns, 'W4 disability and daily activity']:
    values = pd.to_numeric(stage_2_disability_measure_data[column], errors='coerce')
    valid_disability_measures[column] = values.where(values.isin([1, 2, 3]))
exact_category_comparison_records = []
for first_column, second_column in combinations(schooling_measure_columns, 2):
    first_values = valid_disability_measures[first_column]
    second_values = valid_disability_measures[second_column]
    both_valid = first_values.notna() & second_values.notna()
    agreeing = both_valid & first_values.eq(second_values)
    valid_comparisons = int(both_valid.sum())
    agreement_count = int(agreeing.sum())
    exact_category_comparison_records.append({'First measure': first_column, 'Second measure': second_column,
        'Both categories valid': valid_comparisons, 'Categories agreeing': agreement_count, 'Categories differing': valid_comparisons - agreement_count, 'Agreement percentage': agreement_count / valid_comparisons * 100 if valid_comparisons else np.nan})
stage_2_disability_exact_comparisons = pd.DataFrame(exact_category_comparison_records)
broad_disability_measures = {}
for column, values in valid_disability_measures.items():
    broad_disability_measures[column] = values.map({1.0: 'Disability reported', 2.0: 'Disability reported',
        3.0: 'No disability reported'})
broad_comparison_records = []
comparison_columns = list(broad_disability_measures.keys())
for first_column, second_column in combinations(comparison_columns, 2):
    first_values = broad_disability_measures[first_column]
    second_values = broad_disability_measures[second_column]
    both_valid = first_values.notna() & second_values.notna()
    agreeing = both_valid & first_values.eq(second_values)
    valid_comparisons = int(both_valid.sum())
    agreement_count = int(agreeing.sum())
    broad_comparison_records.append({'First measure': first_column, 'Second measure': second_column,
        'Both values valid': valid_comparisons, 'Broad values agreeing': agreement_count, 'Broad values differing': valid_comparisons - agreement_count, 'Agreement percentage': agreement_count / valid_comparisons * 100 if valid_comparisons else np.nan})
stage_2_disability_broad_comparisons = pd.DataFrame(broad_comparison_records)
w1_valid = valid_disability_measures['W1 disability and schooling'].notna()
w2_valid = valid_disability_measures['W2 disability and schooling'].notna()
w4_school_valid = valid_disability_measures['W4 disability and schooling'].notna()
w4_activity_valid = valid_disability_measures['W4 disability and daily activity'].notna()
disability_coverage_comparison = pd.DataFrame([{'Coverage measure': 'Valid Wave 1 schooling-impact measure',
    'Participants': int(w1_valid.sum())}, {'Coverage measure': 'Valid Wave 2 schooling-impact measure',
    'Participants': int(w2_valid.sum())}, {'Coverage measure': 'Wave 2 valid where Wave 1 is unavailable',
    'Participants': int((w2_valid & ~w1_valid).sum())}, {'Coverage measure': 'Valid Wave 4 schooling-impact measure',
    'Participants': int(w4_school_valid.sum())}, {'Coverage measure': 'Valid Wave 4 daily-activity measure',
    'Participants': int(w4_activity_valid.sum())}, {'Coverage measure': 'No valid Wave 1 or Wave 2 schooling measure',
    'Participants': int((~w1_valid & ~w2_valid).sum())}, {'Coverage measure': 'Wave 4 schooling measure valid where Wave 1 and Wave 2 are unavailable',
    'Participants': int((w4_school_valid & ~w1_valid & ~w2_valid).sum())}])
print(f'Core disability variables reviewed: {len(stage_2_disability_coverage):,}')
print('\nCoverage and valid-category summary:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 155, 'display.width', 390):
    print(stage_2_disability_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nValue codes and labels:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 145, 'display.width', 370):
    print(stage_2_disability_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nExact schooling-impact category agreement:')
print(stage_2_disability_exact_comparisons.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Agreement percentage': lambda value: f'{value:.2f}%'}))
print('\nBroad disability-presence agreement:')
print(stage_2_disability_broad_comparisons.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Agreement percentage': lambda value: f'{value:.2f}%'}))
print('\nCoverage comparison:')
print(disability_coverage_comparison.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Core disability variables reviewed: 4

Coverage and valid-category summary:
  Wave   Variable              Measure form                                                                   Variable label  Source record present  Source record absent  Stored value  Valid category  Negative survey code  Distinct stored codes
Wave 1  W1disabYP          Schooling impact   DV: Whether young person has a disability/long term illness or health problem                    9524                   243          9524            9288                   236                      7
Wave 2  W2disabYP          Schooling impact DV: Whether young person has a disability/long term illness or health problem -                    9521                   246          9521            9463                    58                      7
Wave 4  W4disabYP          Schooling impact DV: Whether young person has a disability/long term illness that affects schooli                   9756                    11          9756      

In [124]:
# 52: Young-person disability-measure decision and documentation

w1_disability_category = valid_disability_measures['W1 disability and schooling']
w2_disability_category = valid_disability_measures['W2 disability and schooling']
disability_schooling_category = w1_disability_category.combine_first(w2_disability_category)
disability_measure_source = np.select([w1_disability_category.notna(),
    w1_disability_category.isna() & w2_disability_category.notna()], ['Wave 1 derived measure',
    'Wave 2 fallback'], default='No valid pre-transition source')
disability_schooling_label = disability_schooling_category.map({1.0: 'Disability or long-standing illness; schooling affected',
    2.0: 'Disability or long-standing illness; schooling not affected', 3.0: 'No disability or long-standing illness'})
stage_2_disability_measure_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'disability_schooling_category': disability_schooling_category.astype('Int64'), 'Disability and schooling': disability_schooling_label, 'Disability-measure source': disability_measure_source, 'Wave 1 source valid': w1_disability_category.notna(), 'Wave 2 source valid': w2_disability_category.notna()})
both_pre_transition_sources_valid = w1_disability_category.notna() & w2_disability_category.notna()
pre_transition_sources_disagree = both_pre_transition_sources_valid & w1_disability_category.ne(w2_disability_category)
stage_2_disability_measure_review['Wave 1–Wave 2 disagreement'] = pre_transition_sources_disagree
disability_decision_rules = [{'Wave': 'Wave 1', 'Source type': 'Young person', 'Variable': 'W1disabYP',
    'Review outcome': 'Retain as construction source', 'Decision reason': 'Primary source for a three-category pre-transition measure distinguishing no disability or long-standing illness, disability without schooling impact, and disability with schooling impact.', 'Leakage assessment': 'No direct outcome leakage identified. The measure was derived from Wave 1 information collected before the post-16 transition.', 'Reference-period assessment': 'Wave 1 pre-transition disability and schooling measure.', 'Review notes': 'Provides 9,288 valid values. Wave 2 is used only where this source does not contain a valid category.'}, {'Wave': 'Wave 2',
    'Source type': 'Young person', 'Variable': 'W2disabYP', 'Review outcome': 'Retain as construction source', 'Decision reason': 'Fallback source for the three-category pre-transition disability and schooling measure when Wave 1 is unavailable.', 'Leakage assessment': 'No direct outcome leakage identified. The measure was derived from information collected before the post-16 transition.', 'Reference-period assessment': 'Wave 2 pre-transition disability and schooling measure.', 'Review notes': 'Provides 178 additional valid values where Wave 1 is unavailable. All 9,285 overlapping valid values agree exactly with Wave 1.'}, {'Wave': 'Wave 4',
    'Source type': 'Young person', 'Variable': 'W4disabYP', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Wave 4 disability measure incorporates whether the condition affected school or college attendance and coursework. It was measured at or after the transition period and may overlap with the post-16 outcome.', 'Leakage assessment': 'Potential temporal and construct leakage. The variable uses schooling or college impact measured close to the outcome period.', 'Reference-period assessment': 'Wave 4 measure collected at or after the post-16 transition boundary.', 'Review notes': 'Not used as a fallback despite providing additional coverage. It agrees with the earlier broad disability measure for approximately 88% of overlapping records.'}, {'Wave': 'Wave 4',
    'Source type': 'Young person', 'Variable': 'W4disab2YP', 'Review outcome': 'Exclude from predictor set', 'Decision reason': "Wave 4 measure describes whether disability limits the young person's current daily activities. It was measured at or after the transition period and does not represent a pre-transition predictor.", 'Leakage assessment': 'Potential temporal leakage because the measure records current activity limitation at or after the outcome transition period.', 'Reference-period assessment': 'Wave 4 current disability-limitation measure.', 'Review notes': 'Not used as a fallback or separate predictor. Pre-transition information is constructed from W1disabYP and W2disabYP only.'}]
disability_decision_indices = []
for rule in disability_decision_rules:
    matching_rows = stage_2_variable_decision_register['Wave'].eq(rule['Wave']) & stage_2_variable_decision_register['Source type'].eq(rule['Source type']) & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {rule['Wave']}, {rule['Source type']}, {rule['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    disability_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = rule['Review outcome']
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'Health and disability'
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = rule['Decision reason']
    stage_2_variable_decision_register.loc[matching_rows, 'Leakage assessment'] = rule['Leakage assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Reference-period assessment'] = rule['Reference-period assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave-specific data dictionaries and Stage 2 cross-wave disability review'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = rule['Review notes']
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
disability_measure_review_output_path = stage_2_output_directory / 'stage_2_disability_measure_review.csv'
stage_2_disability_measure_review.to_csv(disability_measure_review_output_path, index=False)
disability_decision_output = stage_2_variable_decision_register.loc[disability_decision_indices, ['Wave',
    'Source type', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment', 'Review notes']].sort_values(['Review outcome',
    'Wave', 'Variable'])
disability_exclusions = disability_decision_output.loc[disability_decision_output['Review outcome'].eq('Exclude from predictor set'),
    ['Wave', 'Source type', 'Variable', 'Decision reason', 'Leakage assessment']]
disability_distribution = stage_2_disability_measure_review['Disability and schooling'].fillna('Missing').value_counts().rename_axis('Disability and schooling').reset_index(name='Participants')
disability_source_summary = stage_2_disability_measure_review['Disability-measure source'].value_counts().rename_axis('Disability-measure source').reset_index(name='Participants')
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Participants in disability-measure file: {len(stage_2_disability_measure_review):,}')
print(f"Participants with a valid pre-transition disability measure: {stage_2_disability_measure_review['disability_schooling_category'].notna().sum():,}")
print(f'Wave 1–Wave 2 disagreements: {pre_transition_sources_disagree.sum():,}')
print('\nDisability-measure distribution:')
print(disability_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nDisability-measure source:')
print(disability_source_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nDisability-variable decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 140, 'display.width', 390):
    print(disability_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables excluded in this step:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 140, 'display.width', 370):
    print(disability_exclusions.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'Disability-measure review file saved to: {disability_measure_review_output_path}')

Participants in disability-measure file: 9,767
Participants with a valid pre-transition disability measure: 9,466
Wave 1–Wave 2 disagreements: 0

Disability-measure distribution:
                                   Disability and schooling  Participants
                     No disability or long-standing illness          8241
Disability or long-standing illness; schooling not affected           714
    Disability or long-standing illness; schooling affected           511
                                                    Missing           301

Disability-measure source:
     Disability-measure source  Participants
        Wave 1 derived measure          9288
No valid pre-transition source           301
               Wave 2 fallback           178

Disability-variable decisions:
  Wave  Source type   Variable                                                                   Variable label                Review outcome                                                                      

In [125]:
# 53: Core young-person SEN-status and routing review

from itertools import combinations
sen_status_specifications = [{'Wave': 'Wave 1', 'Source type': 'Young person', 'Variable': 'W1senMP',
    'Measure form': 'Ever identified', 'Output name': 'W1 SEN ever'}, {'Wave': 'Wave 2', 'Source type': 'Young person',
    'Variable': 'W2senMP', 'Measure form': 'Ever identified', 'Output name': 'W2 SEN ever'}, {'Wave': 'Wave 4',
    'Source type': 'Young person', 'Variable': 'W4SENMP', 'Measure form': 'Ever identified', 'Output name': 'W4 SEN ever'}, {'Wave': 'Wave 1',
    'Source type': 'Young person', 'Variable': 'W1sencurrMP', 'Measure form': 'Currently identified', 'Output name': 'W1 SEN current'}, {'Wave': 'Wave 2',
    'Source type': 'Young person', 'Variable': 'W2sencurrMP', 'Measure form': 'Currently identified', 'Output name': 'W2 SEN current'}, {'Wave': 'Wave 3',
    'Source type': 'Young person', 'Variable': 'W3sencurrMP', 'Measure form': 'Currently identified', 'Output name': 'W3 SEN current'}, {'Wave': 'Wave 4',
    'Source type': 'Young person', 'Variable': 'W4SENcurrMP', 'Measure form': 'Currently identified', 'Output name': 'W4 SEN current'}]
stage_2_sen_status_data = stage_2_participant_ids.copy()
sen_status_coverage_records = []
sen_status_value_records = []
for specification in sen_status_specifications:
    matching_entry = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq(specification['Wave']) & stage_2_master_variable_register['Source type'].eq(specification['Source type']) & stage_2_master_variable_register['Variable'].str.lower().eq(specification['Variable'].lower())]
    if len(matching_entry) != 1:
        raise ValueError(f"Expected one source entry for {specification['Wave']}, {specification['Variable']}; found {len(matching_entry)}.")
    source_path = Path(matching_entry.iloc[0]['Source path'])
    source_variable = matching_entry.iloc[0]['Variable']
    numeric_data = pd.read_stata(source_path, columns=['NSID', source_variable], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', source_variable], convert_categoricals=True)
    sample_numeric_data = stage_2_participant_ids.merge(numeric_data, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = sample_numeric_data['_merge'].eq('both')
    sample_numeric_data = sample_numeric_data.drop(columns='_merge')
    sample_labelled_data = stage_2_participant_ids.merge(labelled_data, on='NSID', how='left', validate='one_to_one')
    numeric_values = pd.to_numeric(sample_numeric_data[source_variable], errors='coerce')
    labelled_values = sample_labelled_data[source_variable].astype('string')
    output_name = specification['Output name']
    stage_2_sen_status_data[f'{output_name} code'] = numeric_values
    stage_2_sen_status_data[f'{output_name} label'] = labelled_values
    stage_2_sen_status_data[f'{output_name} source present'] = source_record_present
    valid_yes_no = numeric_values.isin([1, 2])
    sen_status_coverage_records.append({'Wave': specification['Wave'], 'Variable': source_variable,
        'Measure form': specification['Measure form'], 'Variable label': matching_entry.iloc[0]['Variable label'], 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Valid Yes or No': int(valid_yes_no.sum()), 'Not applicable': int(numeric_values.eq(-91).sum()), 'Other negative code': int((numeric_values.lt(0) & ~numeric_values.eq(-91)).sum())})
    value_summary = pd.DataFrame({'Value code': numeric_values, 'Value label': labelled_values,
        'Source record present': source_record_present}).loc[lambda frame: frame['Source record present']].drop(columns='Source record present').value_counts(dropna=False).rename('Participants').reset_index()
    value_summary.insert(0, 'Wave', specification['Wave'])
    value_summary.insert(1, 'Variable', source_variable)
    value_summary.insert(2, 'Measure form', specification['Measure form'])
    sen_status_value_records.append(value_summary)
stage_2_sen_status_coverage = pd.DataFrame(sen_status_coverage_records)
stage_2_sen_status_values = pd.concat(sen_status_value_records, ignore_index=True)
w4_boost_entry = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq('Wave 4') & stage_2_master_variable_register['Source type'].eq('Young person') & stage_2_master_variable_register['Variable'].str.lower().eq('w4boost')]
if len(w4_boost_entry) != 1:
    raise ValueError(f'Expected one W4Boost entry, but found {len(w4_boost_entry)}.')
w4_boost_data = pd.read_stata(Path(w4_boost_entry.iloc[0]['Source path']), columns=['NSID',
    w4_boost_entry.iloc[0]['Variable']], convert_categoricals=False)
stage_2_sen_status_data = stage_2_sen_status_data.merge(w4_boost_data.rename(columns={w4_boost_entry.iloc[0]['Variable']: 'W4 boost code'}),
    on='NSID', how='left', validate='one_to_one')

def valid_sen_response(series):
    """Retain only explicit Yes or No response codes."""
    numeric_series = pd.to_numeric(series, errors='coerce')
    return numeric_series.where(numeric_series.isin([1, 2]))

def readable_sen_status(series):
    """Create readable routing categories without merging -91 with No."""
    numeric_series = pd.to_numeric(series, errors='coerce')
    return pd.Series(np.select([numeric_series.eq(1), numeric_series.eq(2), numeric_series.eq(-91),
        numeric_series.eq(-99), numeric_series.eq(-996), numeric_series.lt(0), numeric_series.isna()], ['Yes', 'No',
        'Not applicable', 'Respondent not interviewed', 'No parent in household', 'Other negative code', 'No source record'], default='Other stored value'), index=series.index, dtype='string')
for specification in sen_status_specifications:
    output_name = specification['Output name']
    stage_2_sen_status_data[f'{output_name} status'] = readable_sen_status(stage_2_sen_status_data[f'{output_name} code'])
comparison_groups = {'Ever identified': ['W1 SEN ever', 'W2 SEN ever', 'W4 SEN ever'],
    'Currently identified': ['W1 SEN current', 'W2 SEN current', 'W3 SEN current', 'W4 SEN current']}
sen_agreement_records = []
for measure_form, measure_names in comparison_groups.items():
    for first_measure, second_measure in combinations(measure_names, 2):
        first_values = valid_sen_response(stage_2_sen_status_data[f'{first_measure} code'])
        second_values = valid_sen_response(stage_2_sen_status_data[f'{second_measure} code'])
        both_valid = first_values.notna() & second_values.notna()
        agreeing = both_valid & first_values.eq(second_values)
        valid_comparisons = int(both_valid.sum())
        agreement_count = int(agreeing.sum())
        sen_agreement_records.append({'Measure form': measure_form, 'First measure': first_measure,
            'Second measure': second_measure, 'Both values valid': valid_comparisons, 'Values agreeing': agreement_count, 'Values differing': valid_comparisons - agreement_count, 'Agreement percentage': agreement_count / valid_comparisons * 100 if valid_comparisons else np.nan})
stage_2_sen_status_agreement = pd.DataFrame(sen_agreement_records)
w1_w2_ever_routing = pd.crosstab(stage_2_sen_status_data['W1 SEN ever status'],
    stage_2_sen_status_data['W2 SEN ever status'], margins=True, margins_name='Total')
w1_ever_current_routing = pd.crosstab(stage_2_sen_status_data['W1 SEN ever status'],
    stage_2_sen_status_data['W1 SEN current status'], margins=True, margins_name='Total')
w2_ever_current_routing = pd.crosstab(stage_2_sen_status_data['W2 SEN ever status'],
    stage_2_sen_status_data['W2 SEN current status'], margins=True, margins_name='Total')
w4_ever_current_routing = pd.crosstab(stage_2_sen_status_data['W4 SEN ever status'],
    stage_2_sen_status_data['W4 SEN current status'], margins=True, margins_name='Total')
w4_boost_status = pd.to_numeric(stage_2_sen_status_data['W4 boost code'],
    errors='coerce').map({1.0: 'Boost respondent', 2.0: 'Main sample respondent'}).fillna('No Wave 4 boost status')
w4_boost_ever_routing = pd.crosstab(w4_boost_status, stage_2_sen_status_data['W4 SEN ever status'], margins=True,
    margins_name='Total')
print(f'Core SEN-status variables reviewed: {len(stage_2_sen_status_coverage):,}')
print('\nCoverage after restricting valid responses to codes 1 and 2:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 155, 'display.width', 390):
    print(stage_2_sen_status_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nValue codes and labels:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 145, 'display.width', 370):
    print(stage_2_sen_status_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCross-wave agreement using only explicit Yes and No responses:')
print(stage_2_sen_status_agreement.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Agreement percentage': lambda value: f'{value:.2f}%'}))
print('\nWave 1 ever status by Wave 2 ever status:')
print(w1_w2_ever_routing.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nWave 1 ever status by Wave 1 current status:')
print(w1_ever_current_routing.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nWave 2 ever status by Wave 2 current status:')
print(w2_ever_current_routing.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nWave 4 ever status by Wave 4 current status:')
print(w4_ever_current_routing.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nWave 4 boost status by Wave 4 ever-SEN status:')
print(w4_boost_ever_routing.to_string(max_rows=TABLE_ROW_LIMIT))

Core SEN-status variables reviewed: 7

Coverage after restricting valid responses to codes 1 and 2:
  Wave    Variable         Measure form                                                                 Variable label  Source record present  Source record absent  Valid Yes or No  Not applicable  Other negative code
Wave 1     W1senMP      Ever identified MP: Whether YP ever identified (by anyone) as having special educational needs                   9524                   243             9396               0                  128
Wave 2     W2senMP      Ever identified MP: Whether YP ever identified (by anyone) as having special educational needs                   9521                   246             7819            1608                   94
   ...         ...                  ...                                                                            ...                    ...                   ...              ...             ...                  ...
Wave 3 W3sencurrMP Currently

In [126]:
# 54: Routing-based SEN-status construction review

w1_sen_ever_code = valid_sen_response(stage_2_sen_status_data['W1 SEN ever code'])
w2_sen_ever_code = valid_sen_response(stage_2_sen_status_data['W2 SEN ever code'])
w1_sen_current_code = valid_sen_response(stage_2_sen_status_data['W1 SEN current code'])
w2_sen_current_code = valid_sen_response(stage_2_sen_status_data['W2 SEN current code'])
w3_sen_current_code = valid_sen_response(stage_2_sen_status_data['W3 SEN current code'])
sen_ever_identified_by_wave2 = pd.Series(pd.NA, index=stage_2_sen_status_data.index, dtype='Int64')
sen_ever_identified_by_wave2.loc[w1_sen_ever_code.eq(1) | w2_sen_ever_code.eq(1)] = 1
sen_ever_identified_by_wave2.loc[sen_ever_identified_by_wave2.isna() & w2_sen_ever_code.eq(2)] = 0
sen_ever_source = pd.Series('No valid cumulative status', index=stage_2_sen_status_data.index, dtype='string')
sen_ever_source.loc[w1_sen_ever_code.eq(1)] = 'Wave 1 identified as having SEN'
sen_ever_source.loc[~w1_sen_ever_code.eq(1).fillna(False) & w2_sen_ever_code.eq(1)] = 'Wave 2 newly identified as having SEN'
sen_ever_source.loc[sen_ever_identified_by_wave2.eq(0)] = 'Wave 2 explicit No'
sen_current_wave1 = pd.Series(pd.NA, index=stage_2_sen_status_data.index, dtype='Int64')
sen_current_wave1.loc[w1_sen_ever_code.eq(2)] = 0
sen_current_wave1.loc[w1_sen_ever_code.eq(1) & w1_sen_current_code.eq(1)] = 1
sen_current_wave1.loc[w1_sen_ever_code.eq(1) & w1_sen_current_code.eq(2)] = 0
sen_current_wave2 = pd.Series(pd.NA, index=stage_2_sen_status_data.index, dtype='Int64')
sen_current_wave2.loc[sen_ever_identified_by_wave2.eq(0)] = 0
sen_current_wave2.loc[sen_ever_identified_by_wave2.eq(1) & w2_sen_current_code.eq(1)] = 1
sen_current_wave2.loc[sen_ever_identified_by_wave2.eq(1) & w2_sen_current_code.eq(2)] = 0
sen_current_wave3_explicit = w3_sen_current_code.map({1.0: 1, 2.0: 0}).astype('Int64')
stage_2_sen_routing_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'sen_ever_identified_by_wave2': sen_ever_identified_by_wave2, 'SEN-ever source': sen_ever_source, 'sen_current_wave1': sen_current_wave1, 'sen_current_wave2': sen_current_wave2, 'sen_current_wave3_explicit': sen_current_wave3_explicit, 'W3 SEN current source status': stage_2_sen_status_data['W3 SEN current status']})
cumulative_sen_label = sen_ever_identified_by_wave2.map({0: 'No SEN identification by Wave 2',
    1: 'SEN identified by Wave 2'}).fillna('Cumulative status unavailable')
wave1_current_label = sen_current_wave1.map({0: 'No current SEN',
    1: 'Current SEN'}).fillna('Current status unavailable')
wave2_current_label = sen_current_wave2.map({0: 'No current SEN',
    1: 'Current SEN'}).fillna('Current status unavailable')
wave3_explicit_label = sen_current_wave3_explicit.map({0: 'No current SEN',
    1: 'Current SEN'}).fillna('No explicit Yes or No')
sen_ever_distribution = cumulative_sen_label.value_counts().rename_axis('SEN identification by Wave 2').reset_index(name='Participants')
sen_ever_source_summary = sen_ever_source.value_counts().rename_axis('Cumulative SEN source').reset_index(name='Participants')
sen_current_wave1_distribution = wave1_current_label.value_counts().rename_axis('Wave 1 current SEN status').reset_index(name='Participants')
sen_current_wave2_distribution = wave2_current_label.value_counts().rename_axis('Wave 2 current SEN status').reset_index(name='Participants')
wave2_ever_by_wave3_current_status = pd.crosstab(cumulative_sen_label,
    stage_2_sen_status_data['W3 SEN current status'], margins=True, margins_name='Total')
wave2_current_by_wave3_explicit = pd.crosstab(wave2_current_label, wave3_explicit_label, margins=True,
    margins_name='Total')
both_current_statuses_valid = sen_current_wave2.notna() & sen_current_wave3_explicit.notna()
sen_current_transition = pd.Series('One or both current statuses unavailable', index=stage_2_sen_status_data.index,
    dtype='string')
sen_current_transition.loc[both_current_statuses_valid & sen_current_wave2.eq(0) & sen_current_wave3_explicit.eq(0)] = 'No current SEN at both waves'
sen_current_transition.loc[both_current_statuses_valid & sen_current_wave2.eq(0) & sen_current_wave3_explicit.eq(1)] = 'No at Wave 2; current SEN at Wave 3'
sen_current_transition.loc[both_current_statuses_valid & sen_current_wave2.eq(1) & sen_current_wave3_explicit.eq(0)] = 'Current SEN at Wave 2; no at Wave 3'
sen_current_transition.loc[both_current_statuses_valid & sen_current_wave2.eq(1) & sen_current_wave3_explicit.eq(1)] = 'Current SEN at both waves'
sen_current_transition_summary = sen_current_transition.value_counts().rename_axis('Wave 2–Wave 3 current-SEN pattern').reset_index(name='Participants')
wave3_additional_information_summary = pd.DataFrame([{'Review item': 'Wave 3 explicit Yes or No responses',
    'Participants': int(sen_current_wave3_explicit.notna().sum())}, {'Review item': 'Wave 3 explicit response where cumulative Wave 2 ever-status is unavailable',
    'Participants': int((sen_current_wave3_explicit.notna() & sen_ever_identified_by_wave2.isna()).sum())}, {'Review item': 'Wave 3 Current SEN where cumulative Wave 2 status was No',
    'Participants': int((sen_current_wave3_explicit.eq(1) & sen_ever_identified_by_wave2.eq(0)).sum())}, {'Review item': 'Wave 3 No current SEN where cumulative Wave 2 status was No',
    'Participants': int((sen_current_wave3_explicit.eq(0) & sen_ever_identified_by_wave2.eq(0)).sum())}])
print(f'Participants in SEN routing review: {len(stage_2_sen_routing_review):,}')
print('\nCumulative SEN identification by Wave 2:')
print(sen_ever_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCumulative SEN-identification source:')
print(sen_ever_source_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nRouting-derived Wave 1 current-SEN status:')
print(sen_current_wave1_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nRouting-derived Wave 2 current-SEN status:')
print(sen_current_wave2_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCumulative Wave 2 SEN identification by Wave 3 current-SEN source status:')
print(wave2_ever_by_wave3_current_status.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nRouting-derived Wave 2 current status by explicit Wave 3 response:')
print(wave2_current_by_wave3_explicit.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nWave 2–Wave 3 current-SEN patterns:')
print(sen_current_transition_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 3 additional-information checks:')
print(wave3_additional_information_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Participants in SEN routing review: 9,767

Cumulative SEN identification by Wave 2:
   SEN identification by Wave 2  Participants
No SEN identification by Wave 2          7624
       SEN identified by Wave 2          1815
  Cumulative status unavailable           328

Cumulative SEN-identification source:
                Cumulative SEN source  Participants
                   Wave 2 explicit No          7624
      Wave 1 identified as having SEN          1620
           No valid cumulative status           328
Wave 2 newly identified as having SEN           195

Routing-derived Wave 1 current-SEN status:
 Wave 1 current SEN status  Participants
            No current SEN          8447
               Current SEN           908
Current status unavailable           412

Routing-derived Wave 2 current-SEN status:
 Wave 2 current SEN status  Participants
            No current SEN          8564
               Current SEN           828
Current status unavailable           375

Cumulative Wave 

In [127]:
# 55: Combined pre-transition SEN-status measure review

latest_recorded_current_sen = pd.Series(pd.NA, index=stage_2_sen_routing_review.index, dtype='Int64')
latest_current_sen_source = pd.Series('No valid current-SEN source', index=stage_2_sen_routing_review.index,
    dtype='string')
no_sen_identification = sen_ever_identified_by_wave2.eq(0)
latest_recorded_current_sen.loc[no_sen_identification] = 0
latest_current_sen_source.loc[no_sen_identification] = 'No SEN identification by Wave 2'
identified_by_wave2 = sen_ever_identified_by_wave2.eq(1)
wave3_current_available = identified_by_wave2 & sen_current_wave3_explicit.notna()
latest_recorded_current_sen.loc[wave3_current_available] = sen_current_wave3_explicit.loc[wave3_current_available]
latest_current_sen_source.loc[wave3_current_available] = 'Wave 3 explicit current-SEN status'
wave2_current_fallback = identified_by_wave2 & latest_recorded_current_sen.isna() & sen_current_wave2.notna()
latest_recorded_current_sen.loc[wave2_current_fallback] = sen_current_wave2.loc[wave2_current_fallback]
latest_current_sen_source.loc[wave2_current_fallback] = 'Wave 2 current-SEN fallback'
wave1_current_fallback = identified_by_wave2 & latest_recorded_current_sen.isna() & sen_current_wave1.notna()
latest_recorded_current_sen.loc[wave1_current_fallback] = sen_current_wave1.loc[wave1_current_fallback]
latest_current_sen_source.loc[wave1_current_fallback] = 'Wave 1 current-SEN fallback'
combined_sen_status = pd.Series(pd.NA, index=stage_2_sen_routing_review.index, dtype='Int64')
combined_sen_status.loc[sen_ever_identified_by_wave2.eq(0)] = 0
combined_sen_status.loc[sen_ever_identified_by_wave2.eq(1) & latest_recorded_current_sen.eq(0)] = 1
combined_sen_status.loc[sen_ever_identified_by_wave2.eq(1) & latest_recorded_current_sen.eq(1)] = 2
combined_sen_label = combined_sen_status.map({0: 'No SEN identification by Wave 2',
    1: 'SEN identified by Wave 2; not current at latest record', 2: 'SEN identified by Wave 2; current at latest record'})
stage_2_combined_sen_measure_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'sen_identified_by_wave2': sen_ever_identified_by_wave2, 'latest_recorded_current_sen': latest_recorded_current_sen, 'Latest current-SEN source': latest_current_sen_source, 'combined_sen_status': combined_sen_status, 'Combined pre-transition SEN status': combined_sen_label, 'sen_current_wave1': sen_current_wave1, 'sen_current_wave2': sen_current_wave2, 'sen_current_wave3_explicit': sen_current_wave3_explicit})
identified_current_status_unavailable = identified_by_wave2 & latest_recorded_current_sen.isna()
cumulative_status_unavailable = sen_ever_identified_by_wave2.isna()
current_yes_with_cumulative_unavailable = cumulative_status_unavailable & (sen_current_wave1.eq(1).fillna(False) | sen_current_wave2.eq(1).fillna(False) | sen_current_wave3_explicit.eq(1).fillna(False))
current_no_with_cumulative_unavailable = cumulative_status_unavailable & (sen_current_wave1.eq(0).fillna(False) | sen_current_wave2.eq(0).fillna(False) | sen_current_wave3_explicit.eq(0).fillna(False))
combined_sen_distribution = combined_sen_label.fillna('Combined SEN status unavailable').value_counts().rename_axis('Combined pre-transition SEN status').reset_index(name='Participants')
current_sen_source_distribution = latest_current_sen_source.value_counts().rename_axis('Latest current-SEN source').reset_index(name='Participants')
identified_source_distribution = latest_current_sen_source.loc[identified_by_wave2].value_counts().rename_axis('Current-SEN source among identified participants').reset_index(name='Participants')
combined_sen_quality_checks = pd.DataFrame([{'Quality check': 'SEN identified by Wave 2',
    'Participants': int(identified_by_wave2.sum())}, {'Quality check': 'Identified participants with latest current status available',
    'Participants': int((identified_by_wave2 & latest_recorded_current_sen.notna()).sum())}, {'Quality check': 'Identified participants with no valid current status after fallback',
    'Participants': int(identified_current_status_unavailable.sum())}, {'Quality check': 'Cumulative identification unavailable',
    'Participants': int(cumulative_status_unavailable.sum())}, {'Quality check': 'Current SEN Yes despite cumulative identification unavailable',
    'Participants': int(current_yes_with_cumulative_unavailable.sum())}, {'Quality check': 'Current SEN No despite cumulative identification unavailable',
    'Participants': int(current_no_with_cumulative_unavailable.sum())}])
cumulative_by_combined_status = pd.crosstab(sen_ever_identified_by_wave2.map({0: 'No SEN identification by Wave 2',
    1: 'SEN identified by Wave 2'}).fillna('Cumulative status unavailable'), combined_sen_label.fillna('Combined SEN status unavailable'), margins=True, margins_name='Total')
print(f'Participants in combined SEN review: {len(stage_2_combined_sen_measure_review):,}')
print('\nCombined pre-transition SEN-status distribution:')
print(combined_sen_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nLatest current-SEN source for the full sample:')
print(current_sen_source_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCurrent-SEN source among participants identified as having SEN:')
print(identified_source_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(combined_sen_quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCumulative SEN identification by combined SEN status:')
print(cumulative_by_combined_status.to_string(max_rows=TABLE_ROW_LIMIT))

Participants in combined SEN review: 9,767

Combined pre-transition SEN-status distribution:
                    Combined pre-transition SEN status  Participants
                       No SEN identification by Wave 2          7624
SEN identified by Wave 2; not current at latest record          1035
    SEN identified by Wave 2; current at latest record           779
                       Combined SEN status unavailable           329

Latest current-SEN source for the full sample:
         Latest current-SEN source  Participants
   No SEN identification by Wave 2          7624
Wave 3 explicit current-SEN status          1733
       No valid current-SEN source           329
       Wave 2 current-SEN fallback            63
       Wave 1 current-SEN fallback            18

Current-SEN source among participants identified as having SEN:
Current-SEN source among identified participants  Participants
              Wave 3 explicit current-SEN status          1733
                     Wave 2 c

In [128]:
# 56: Unresolved combined SEN-status review

combined_sen_unresolved = combined_sen_status.isna()
identified_without_current_status = sen_ever_identified_by_wave2.eq(1) & latest_recorded_current_sen.isna()
unresolved_with_current_no = sen_ever_identified_by_wave2.isna() & (sen_current_wave1.eq(0).fillna(False) | sen_current_wave2.eq(0).fillna(False) | sen_current_wave3_explicit.eq(0).fillna(False))
stage_2_unresolved_sen_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'Combined SEN unresolved': combined_sen_unresolved, 'Cumulative SEN by Wave 2': sen_ever_identified_by_wave2, 'Latest current SEN': latest_recorded_current_sen, 'Latest current-SEN source': latest_current_sen_source, 'W1 SEN ever code': stage_2_sen_status_data['W1 SEN ever code'], 'W1 SEN ever status': stage_2_sen_status_data['W1 SEN ever status'], 'W2 SEN ever code': stage_2_sen_status_data['W2 SEN ever code'], 'W2 SEN ever status': stage_2_sen_status_data['W2 SEN ever status'], 'W1 SEN current code': stage_2_sen_status_data['W1 SEN current code'], 'W1 SEN current status': stage_2_sen_status_data['W1 SEN current status'], 'W2 SEN current code': stage_2_sen_status_data['W2 SEN current code'], 'W2 SEN current status': stage_2_sen_status_data['W2 SEN current status'], 'W3 SEN current code': stage_2_sen_status_data['W3 SEN current code'], 'W3 SEN current status': stage_2_sen_status_data['W3 SEN current status'], 'W1 SEN source present': stage_2_sen_status_data['W1 SEN ever source present'], 'W2 SEN source present': stage_2_sen_status_data['W2 SEN ever source present'], 'W3 SEN source present': stage_2_sen_status_data['W3 SEN current source present']})
unresolved_sen_cases = stage_2_unresolved_sen_review.loc[combined_sen_unresolved].copy()
unresolved_sen_pattern_summary = unresolved_sen_cases.groupby(['W1 SEN ever status', 'W2 SEN ever status',
    'W1 SEN current status', 'W2 SEN current status', 'W3 SEN current status'], dropna=False).size().reset_index(name='Participants').sort_values('Participants',
    ascending=False).reset_index(drop=True)
unresolved_source_pattern_summary = unresolved_sen_cases.groupby(['W1 SEN source present', 'W2 SEN source present',
    'W3 SEN source present'], dropna=False).size().reset_index(name='Participants').sort_values('Participants',
    ascending=False).reset_index(drop=True)
wave1_no_cumulative_unavailable = unresolved_sen_cases.loc[unresolved_sen_cases['W1 SEN ever status'].eq('No')].copy()
wave1_no_follow_up_summary = wave1_no_cumulative_unavailable.groupby(['W2 SEN ever status', 'W2 SEN source present',
    'W3 SEN current status'], dropna=False).size().reset_index(name='Participants').sort_values('Participants',
    ascending=False).reset_index(drop=True)
identified_without_current_details = stage_2_unresolved_sen_review.loc[identified_without_current_status, ['NSID',
    'W1 SEN ever code', 'W1 SEN ever status', 'W2 SEN ever code', 'W2 SEN ever status', 'W1 SEN current code', 'W1 SEN current status', 'W2 SEN current code', 'W2 SEN current status', 'W3 SEN current code', 'W3 SEN current status']]
unresolved_quality_summary = pd.DataFrame([{'Review group': 'Combined SEN status unresolved',
    'Participants': int(combined_sen_unresolved.sum())}, {'Review group': 'Unresolved cumulative status with an available current-SEN No',
    'Participants': int(unresolved_with_current_no.sum())}, {'Review group': 'Unresolved cases with Wave 1 ever-identified No',
    'Participants': int(wave1_no_cumulative_unavailable.shape[0])}, {'Review group': 'SEN identified but no valid current-status source',
    'Participants': int(identified_without_current_status.sum())}])
print(f'Participants with unresolved combined SEN status: {len(unresolved_sen_cases):,}')
print('\nUnresolved-case summary:')
print(unresolved_quality_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nRouting patterns among unresolved cases:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 120, 'display.width', 360):
    print(unresolved_sen_pattern_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSource availability among unresolved cases:')
print(unresolved_source_pattern_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 1 No cases without a cumulative Wave 2 classification:')
print(wave1_no_follow_up_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(
    '\nIdentified participants without a valid current-SEN status:',
    len(identified_without_current_details),
)
# Participant-level SEN review records are retained in memory but are not displayed.

Participants with unresolved combined SEN status: 329

Unresolved-case summary:
                                                 Review group  Participants
                               Combined SEN status unresolved           329
Unresolved cumulative status with an available current-SEN No            69
              Unresolved cases with Wave 1 ever-identified No            69
            SEN identified but no valid current-status source             1

Routing patterns among unresolved cases:
        W1 SEN ever status         W2 SEN ever status      W1 SEN current status      W2 SEN current status      W3 SEN current status  Participants
          No source record           No source record           No source record           No source record           No source record           243
                        No Respondent not interviewed             Not applicable Respondent not interviewed             Not applicable            46
                       ...                        .

In [129]:
# 57: Wave 4 boost-sample retrospective SEN-timing review

w4_sen_age_entry = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq('Wave 4') & stage_2_master_variable_register['Source type'].eq('Young person') & stage_2_master_variable_register['Variable'].str.lower().eq('w4senagemp')]
if len(w4_sen_age_entry) != 1:
    raise ValueError(f'Expected one W4SENageMP entry, but found {len(w4_sen_age_entry)}.')
w4_sen_age_variable = w4_sen_age_entry.iloc[0]['Variable']
w4_sen_age_path = Path(w4_sen_age_entry.iloc[0]['Source path'])
w4_sen_age_numeric_data = pd.read_stata(w4_sen_age_path, columns=['NSID', w4_sen_age_variable],
    convert_categoricals=False)
w4_sen_age_labelled_data = pd.read_stata(w4_sen_age_path, columns=['NSID', w4_sen_age_variable],
    convert_categoricals=True)
w4_sen_age_numeric_sample = stage_2_participant_ids.merge(w4_sen_age_numeric_data, on='NSID', how='left',
    validate='one_to_one', indicator=True)
w4_sen_age_source_present = w4_sen_age_numeric_sample['_merge'].eq('both')
w4_sen_age_numeric_sample = w4_sen_age_numeric_sample.drop(columns='_merge')
w4_sen_age_labelled_sample = stage_2_participant_ids.merge(w4_sen_age_labelled_data, on='NSID', how='left',
    validate='one_to_one')
w4_sen_age_code = pd.to_numeric(w4_sen_age_numeric_sample[w4_sen_age_variable], errors='coerce')
w4_sen_age_label = w4_sen_age_labelled_sample[w4_sen_age_variable].astype('string')
w4_boost_code = pd.to_numeric(stage_2_sen_status_data['W4 boost code'], errors='coerce')
w4_boost_participant = w4_boost_code.eq(1)
no_early_sen_source_record = ~stage_2_sen_status_data['W1 SEN ever source present'] & ~stage_2_sen_status_data['W2 SEN ever source present'] & ~stage_2_sen_status_data['W3 SEN current source present']
w4_sen_ever_code = valid_sen_response(stage_2_sen_status_data['W4 SEN ever code'])
w4_sen_age_valid = w4_sen_age_code.where(w4_sen_age_code.ge(0))
w4_sen_identification_timing = pd.Series('No valid identification age', index=stage_2_participant_ids.index,
    dtype='string')
w4_sen_identification_timing.loc[w4_sen_age_valid.le(15)] = 'Age 15 or younger'
w4_sen_identification_timing.loc[w4_sen_age_valid.eq(16)] = 'Age 16: timing unresolved'
w4_sen_identification_timing.loc[w4_sen_age_valid.ge(17)] = 'Age 17 or older'
stage_2_w4_sen_timing_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'W4 boost participant': w4_boost_participant, 'No Wave 1–3 SEN source record': no_early_sen_source_record, 'W4 SEN ever code': stage_2_sen_status_data['W4 SEN ever code'], 'W4 SEN ever status': stage_2_sen_status_data['W4 SEN ever status'], 'W4 SEN age code': w4_sen_age_code, 'W4 SEN age label': w4_sen_age_label, 'W4 SEN age source present': w4_sen_age_source_present, 'Valid age at first SEN identification': w4_sen_age_valid, 'Identification timing category': w4_sen_identification_timing})
w4_boost_sen_age_values = stage_2_w4_sen_timing_review.loc[w4_boost_participant, ['W4 SEN age code',
    'W4 SEN age label']].value_counts(dropna=False).rename('Participants').reset_index().sort_values(['W4 SEN age code',
    'Participants'], na_position='last').reset_index(drop=True)
w4_ever_status_for_early_nonparticipants = stage_2_w4_sen_timing_review.loc[no_early_sen_source_record,
    'W4 SEN ever status'].value_counts(dropna=False).rename_axis('W4 retrospective ever-SEN status').reset_index(name='Participants')
w4_yes_identification_age_distribution = stage_2_w4_sen_timing_review.loc[w4_boost_participant & w4_sen_ever_code.eq(1),
    'Valid age at first SEN identification'].value_counts(dropna=False).sort_index().rename_axis('Age at first SEN identification').reset_index(name='Participants')
w4_yes_timing_distribution = stage_2_w4_sen_timing_review.loc[w4_boost_participant & w4_sen_ever_code.eq(1),
    'Identification timing category'].value_counts().rename_axis('Identification timing category').reset_index(name='Participants')
w4_ever_by_age_timing = pd.crosstab(stage_2_w4_sen_timing_review.loc[w4_boost_participant, 'W4 SEN ever status'],
    stage_2_w4_sen_timing_review.loc[w4_boost_participant,
    'Identification timing category'], margins=True, margins_name='Total')
boost_alignment_summary = pd.DataFrame([{'Review group': 'Wave 4 boost participants',
    'Participants': int(w4_boost_participant.sum())}, {'Review group': 'No Wave 1–3 SEN source record',
    'Participants': int(no_early_sen_source_record.sum())}, {'Review group': 'Both boost and no Wave 1–3 SEN source',
    'Participants': int((w4_boost_participant & no_early_sen_source_record).sum())}, {'Review group': 'No early SEN source but not a boost participant',
    'Participants': int((no_early_sen_source_record & ~w4_boost_participant).sum())}])
print(f'Participants in Wave 4 retrospective SEN review: {len(stage_2_w4_sen_timing_review):,}')
print('\nBoost-sample alignment:')
print(boost_alignment_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 4 ever-SEN status among participants without Wave 1–3 SEN records:')
print(w4_ever_status_for_early_nonparticipants.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nStored Wave 4 SEN-age values among boost participants:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 120, 'display.width', 300):
    print(w4_boost_sen_age_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nAge at first identification among Wave 4 boost participants reporting SEN:')
print(w4_yes_identification_age_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nIdentification-timing categories among Wave 4 SEN Yes cases:')
print(w4_yes_timing_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 4 ever-SEN status by identification timing:')
print(w4_ever_by_age_timing.to_string(max_rows=TABLE_ROW_LIMIT))

Participants in Wave 4 retrospective SEN review: 9,767

Boost-sample alignment:
                                   Review group  Participants
                      Wave 4 boost participants           243
                  No Wave 1–3 SEN source record           243
          Both boost and no Wave 1–3 SEN source           243
No early SEN source but not a boost participant             0

Wave 4 ever-SEN status among participants without Wave 1–3 SEN records:
W4 retrospective ever-SEN status  Participants
                              No           204
                             Yes            21
      Respondent not interviewed            15
          No parent in household             2
             Other negative code             1

Stored Wave 4 SEN-age values among boost participants:
 W4 SEN age code       W4 SEN age label  Participants
          -996.0 No Parent in household             2
           -99.0     MP not interviewed            15
             ...                    .

In [130]:
# 58: Pre-transition SEN-identification measure review

sen_identified_pretransition = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
sen_identification_source = pd.Series('No valid pre-transition SEN source', index=stage_2_participant_ids.index,
    dtype='string')
early_sen_status_available = sen_ever_identified_by_wave2.notna()
sen_identified_pretransition.loc[early_sen_status_available] = sen_ever_identified_by_wave2.loc[early_sen_status_available]
sen_identification_source.loc[sen_ever_identified_by_wave2.eq(0)] = 'Wave 1–Wave 2 cumulative No'
sen_identification_source.loc[sen_ever_identified_by_wave2.eq(1)] = 'Wave 1–Wave 2 cumulative Yes'
w4_retrospective_no = no_early_sen_source_record & w4_boost_participant & w4_sen_ever_code.eq(2)
w4_retrospective_pretransition_yes = no_early_sen_source_record & w4_boost_participant & w4_sen_ever_code.eq(1) & w4_sen_age_valid.le(15)
w4_retrospective_timing_unresolved = no_early_sen_source_record & w4_boost_participant & w4_sen_ever_code.eq(1) & (w4_sen_age_valid.isna() | w4_sen_age_valid.ge(16))
sen_identified_pretransition.loc[w4_retrospective_no] = 0
sen_identification_source.loc[w4_retrospective_no] = 'Wave 4 boost retrospective No'
sen_identified_pretransition.loc[w4_retrospective_pretransition_yes] = 1
sen_identification_source.loc[w4_retrospective_pretransition_yes] = 'Wave 4 boost retrospective Yes; identified by age 15'
sen_identified_pretransition_label = sen_identified_pretransition.map({0: 'No pre-transition SEN identification',
    1: 'Pre-transition SEN identification'})
sen_identification_missing_reason = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
final_sen_missing = sen_identified_pretransition.isna()
sen_identification_missing_reason.loc[final_sen_missing & no_early_sen_source_record & stage_2_sen_status_data['W4 SEN ever code'].eq(-99)] = 'Wave 4 main parent not interviewed'
sen_identification_missing_reason.loc[final_sen_missing & no_early_sen_source_record & stage_2_sen_status_data['W4 SEN ever code'].eq(-996)] = 'No parent in Wave 4 household'
sen_identification_missing_reason.loc[final_sen_missing & no_early_sen_source_record & ~stage_2_sen_status_data['W4 SEN ever code'].isin([-99,
    -996])] = 'Other unusable Wave 4 retrospective response'
sen_identification_missing_reason.loc[final_sen_missing & ~no_early_sen_source_record & stage_2_sen_status_data['W1 SEN ever code'].eq(2)] = 'Wave 1 No but Wave 2 cumulative update unavailable'
sen_identification_missing_reason.loc[final_sen_missing & ~no_early_sen_source_record & ~stage_2_sen_status_data['W1 SEN ever code'].eq(2)] = 'No valid cumulative SEN-identification response'
sen_identification_missing_reason.loc[w4_retrospective_timing_unresolved] = 'Wave 4 SEN identification timing not demonstrably pre-transition'
stage_2_sen_identification_measure_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'sen_identified_pretransition': sen_identified_pretransition, 'Pre-transition SEN identification': sen_identified_pretransition_label, 'SEN-identification source': sen_identification_source, 'Missing reason': sen_identification_missing_reason, 'Wave 1–Wave 2 cumulative status': sen_ever_identified_by_wave2, 'W4 boost participant': w4_boost_participant, 'W4 retrospective SEN status': stage_2_sen_status_data['W4 SEN ever status'], 'W4 age at first SEN identification': w4_sen_age_valid})
sen_identification_distribution = sen_identified_pretransition_label.fillna('Missing').value_counts().rename_axis('Pre-transition SEN identification').reset_index(name='Participants')
sen_identification_source_distribution = sen_identification_source.value_counts().rename_axis('SEN-identification source').reset_index(name='Participants')
sen_identification_missing_distribution = sen_identification_missing_reason.loc[final_sen_missing].fillna('Unclassified missing reason').value_counts().rename_axis('Missing reason').reset_index(name='Participants')
w4_retrospective_quality_checks = pd.DataFrame([{'Quality check': 'Wave 4 retrospective No classifications',
    'Participants': int(w4_retrospective_no.sum())}, {'Quality check': 'Wave 4 retrospective Yes identified by age 15',
    'Participants': int(w4_retrospective_pretransition_yes.sum())}, {'Quality check': 'Wave 4 Yes with unresolved pre-transition timing',
    'Participants': int(w4_retrospective_timing_unresolved.sum())}, {'Quality check': 'Final valid pre-transition SEN measure',
    'Participants': int(sen_identified_pretransition.notna().sum())}, {'Quality check': 'Final missing pre-transition SEN measure',
    'Participants': int(sen_identified_pretransition.isna().sum())}])
print(f'Participants in SEN-identification review: {len(stage_2_sen_identification_measure_review):,}')
print('\nProposed pre-transition SEN-identification measure:')
print(sen_identification_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMeasure source:')
print(sen_identification_source_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 4 retrospective-use checks:')
print(w4_retrospective_quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nReasons for remaining missing values:')
print(sen_identification_missing_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Participants in SEN-identification review: 9,767

Proposed pre-transition SEN-identification measure:
   Pre-transition SEN identification  Participants
No pre-transition SEN identification          7828
   Pre-transition SEN identification          1836
                             Missing           103

Measure source:
                           SEN-identification source  Participants
                         Wave 1–Wave 2 cumulative No          7624
                        Wave 1–Wave 2 cumulative Yes          1815
                       Wave 4 boost retrospective No           204
                  No valid pre-transition SEN source           103
Wave 4 boost retrospective Yes; identified by age 15            21

Wave 4 retrospective-use checks:
                                   Quality check  Participants
         Wave 4 retrospective No classifications           204
   Wave 4 retrospective Yes identified by age 15            21
Wave 4 Yes with unresolved pre-transition timing    

In [131]:
# 59: Comparison of consolidated SEN-measure options

binary_sen_measure = sen_identified_pretransition.copy()
binary_sen_label = binary_sen_measure.map({0: 'No pre-transition SEN identification',
    1: 'Pre-transition SEN identification'})
three_category_sen_measure = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
three_category_sen_measure.loc[sen_identified_pretransition.eq(0)] = 0
three_category_sen_measure.loc[sen_identified_pretransition.eq(1) & latest_recorded_current_sen.eq(0)] = 1
three_category_sen_measure.loc[sen_identified_pretransition.eq(1) & latest_recorded_current_sen.eq(1)] = 2
three_category_sen_label = three_category_sen_measure.map({0: 'No pre-transition SEN identification',
    1: 'SEN identified; not current at latest pre-transition record', 2: 'SEN identified; current at latest pre-transition record'})
three_category_missing_reason = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
three_category_missing = three_category_sen_measure.isna()
three_category_missing_reason.loc[three_category_missing & sen_identified_pretransition.isna()] = 'Pre-transition SEN identification unavailable'
three_category_missing_reason.loc[three_category_missing & sen_identified_pretransition.eq(1) & latest_recorded_current_sen.isna() & w4_retrospective_pretransition_yes] = 'Wave 4 retrospective identification available, but pre-transition current status unavailable'
three_category_missing_reason.loc[three_category_missing & sen_identified_pretransition.eq(1) & latest_recorded_current_sen.isna() & ~w4_retrospective_pretransition_yes] = 'SEN identified, but no valid current-status record before transition'
stage_2_sen_measure_option_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'binary_sen_identification': binary_sen_measure, 'Binary SEN identification': binary_sen_label, 'three_category_sen_status': three_category_sen_measure, 'Three-category SEN status': three_category_sen_label, 'Three-category missing reason': three_category_missing_reason, 'Latest current-SEN source': latest_current_sen_source})
binary_sen_distribution = binary_sen_label.fillna('Missing').value_counts().rename_axis('Binary SEN-identification measure').reset_index(name='Participants')
three_category_sen_distribution = three_category_sen_label.fillna('Missing').value_counts().rename_axis('Three-category SEN measure').reset_index(name='Participants')
sen_option_coverage_comparison = pd.DataFrame([{'Measure option': 'Binary SEN identification',
    'Valid participants': int(binary_sen_measure.notna().sum()), 'Missing participants': int(binary_sen_measure.isna().sum()), 'Coverage percentage': binary_sen_measure.notna().mean() * 100}, {'Measure option': 'Three-category SEN status',
    'Valid participants': int(three_category_sen_measure.notna().sum()), 'Missing participants': int(three_category_sen_measure.isna().sum()), 'Coverage percentage': three_category_sen_measure.notna().mean() * 100}])
three_category_missing_summary = three_category_missing_reason.loc[three_category_missing].fillna('Unclassified missing reason').value_counts().rename_axis('Three-category missing reason').reset_index(name='Participants')
identified_group_split = three_category_sen_label.loc[binary_sen_measure.eq(1)].fillna('SEN identified; current status unavailable').value_counts().rename_axis('Status among participants with SEN identification').reset_index(name='Participants')
logical_inconsistency = binary_sen_measure.eq(0) & three_category_sen_measure.isin([1, 2])
quality_checks = pd.DataFrame([{'Quality check': 'Participants with pre-transition SEN identification',
    'Participants': int(binary_sen_measure.eq(1).sum())}, {'Quality check': 'Identified participants with current status available',
    'Participants': int((binary_sen_measure.eq(1) & latest_recorded_current_sen.notna()).sum())}, {'Quality check': 'Identified participants with current status unavailable',
    'Participants': int((binary_sen_measure.eq(1) & latest_recorded_current_sen.isna()).sum())}, {'Quality check': 'No-SEN cases assigned an identified-current category',
    'Participants': int(logical_inconsistency.sum())}])
print(f'Participants in SEN-measure option review: {len(stage_2_sen_measure_option_review):,}')
print('\nBinary SEN-identification distribution:')
print(binary_sen_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nThree-category SEN-status distribution:')
print(three_category_sen_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCoverage comparison:')
print(sen_option_coverage_comparison.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Coverage percentage': lambda value: f'{value:.2f}%'}))
print('\nStatus among participants with SEN identification:')
print(identified_group_split.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nThree-category missing-value reasons:')
print(three_category_missing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Participants in SEN-measure option review: 9,767

Binary SEN-identification distribution:
   Binary SEN-identification measure  Participants
No pre-transition SEN identification          7828
   Pre-transition SEN identification          1836
                             Missing           103

Three-category SEN-status distribution:
                                 Three-category SEN measure  Participants
                       No pre-transition SEN identification          7828
SEN identified; not current at latest pre-transition record          1035
    SEN identified; current at latest pre-transition record           779
                                                    Missing           125

Coverage comparison:
           Measure option  Valid participants  Missing participants Coverage percentage
Binary SEN identification                9664                   103              98.95%
Three-category SEN status                9642                   125              98.72%

Status a

In [132]:
# 60: Consolidated pre-transition SEN-measure decision

selected_sen_measure = three_category_sen_measure.copy()
selected_sen_label = three_category_sen_label.copy()
selected_sen_measure_source = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
selected_sen_measure_source.loc[selected_sen_measure.eq(0) & no_early_sen_source_record] = 'Wave 4 boost retrospective No'
selected_sen_measure_source.loc[selected_sen_measure.eq(0) & ~no_early_sen_source_record] = 'Wave 1–Wave 2 cumulative identification'
selected_sen_measure_source.loc[selected_sen_measure.isin([1,
    2]) & latest_current_sen_source.eq('Wave 3 explicit current-SEN status')] = 'Wave 1–Wave 2 identification and Wave 3 current status'
selected_sen_measure_source.loc[selected_sen_measure.isin([1,
    2]) & latest_current_sen_source.eq('Wave 2 current-SEN fallback')] = 'Wave 1–Wave 2 identification and Wave 2 current-status fallback'
selected_sen_measure_source.loc[selected_sen_measure.isin([1,
    2]) & latest_current_sen_source.eq('Wave 1 current-SEN fallback')] = 'Wave 1–Wave 2 identification and Wave 1 current-status fallback'
selected_sen_missing_reason = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
selected_sen_missing = selected_sen_measure.isna()
selected_sen_missing_reason.loc[selected_sen_missing & sen_identified_pretransition.isna()] = 'Pre-transition SEN identification unavailable'
selected_sen_missing_reason.loc[selected_sen_missing & sen_identified_pretransition.eq(1) & w4_retrospective_pretransition_yes] = 'Retrospective pre-transition identification available, but current status unavailable'
selected_sen_missing_reason.loc[selected_sen_missing & sen_identified_pretransition.eq(1) & ~w4_retrospective_pretransition_yes] = 'SEN identified, but no valid current-status record before transition'
stage_2_sen_measure_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'sen_status_pretransition': selected_sen_measure, 'Pre-transition SEN status': selected_sen_label, 'SEN-measure source': selected_sen_measure_source, 'SEN-measure missing reason': selected_sen_missing_reason, 'sen_identified_pretransition': binary_sen_measure, 'Pre-transition SEN identification': binary_sen_label, 'latest_recorded_current_sen': latest_recorded_current_sen, 'Latest current-SEN source': latest_current_sen_source})
sen_decision_rules = [{'Wave': 'Wave 1', 'Source type': 'Young person', 'Variable': 'W1senMP',
    'Review outcome': 'Retain as construction source', 'Decision reason': 'Baseline source for cumulative pre-transition SEN identification.', 'Leakage assessment': 'No direct outcome leakage identified. The information was collected before transition.', 'Reference-period assessment': 'Whether the young person had ever been identified as having SEN by Wave 1.', 'Review notes': 'Used with W2senMP under its documented routing. It is not retained as a separate predictor.'}, {'Wave': 'Wave 2',
    'Source type': 'Young person', 'Variable': 'W2senMP', 'Review outcome': 'Retain as construction source', 'Decision reason': 'Updates cumulative SEN identification for participants not already identified at Wave 1.', 'Leakage assessment': 'No direct outcome leakage identified. The information was collected before transition.', 'Reference-period assessment': 'New SEN identification recorded by Wave 2.', 'Review notes': 'The structural Not applicable code preserves Wave 1 identification rather than representing a No response.'}, {'Wave': 'Wave 1',
    'Source type': 'Young person', 'Variable': 'W1sencurrMP', 'Review outcome': 'Retain as construction source', 'Decision reason': 'Fallback source for current SEN status among participants previously identified as having SEN.', 'Leakage assessment': 'No direct outcome leakage identified. The information was collected before transition.', 'Reference-period assessment': 'Current SEN status at Wave 1.', 'Review notes': 'Used only when later valid current-status records are unavailable.'}, {'Wave': 'Wave 2',
    'Source type': 'Young person', 'Variable': 'W2sencurrMP', 'Review outcome': 'Retain as construction source', 'Decision reason': 'Fallback source for the latest available pre-transition current SEN status.', 'Leakage assessment': 'No direct outcome leakage identified. The information was collected before transition.', 'Reference-period assessment': 'Current SEN status at Wave 2.', 'Review notes': 'Used when Wave 3 current status is unavailable. Structural Not applicable values are interpreted through the SEN-identification routing.'}, {'Wave': 'Wave 3',
    'Source type': 'Young person', 'Variable': 'W3sencurrMP', 'Review outcome': 'Retain as construction source', 'Decision reason': 'Preferred source for the latest current SEN status recorded before the post-16 transition.', 'Leakage assessment': 'No direct outcome leakage identified. The measure records SEN status before transition.', 'Reference-period assessment': 'Current SEN status at Wave 3, near but before transition.', 'Review notes': 'Used only for participants routed to the SEN section. It does not identify new SEN cases independently.'}, {'Wave': 'Wave 4',
    'Source type': 'Young person', 'Variable': 'W4SENMP', 'Review outcome': 'Retain as construction source', 'Decision reason': 'Restricted retrospective source for Wave 4 boost participants without Wave 1–Wave 3 SEN records.', 'Leakage assessment': 'Restricted retrospective use only. A No response implies no earlier identification, while a Yes response is used only when W4SENageMP confirms identification by age 15.', 'Reference-period assessment': 'Retrospective ever-identified SEN history collected at Wave 4.', 'Review notes': 'Not used for main-sample participants and not used as a current SEN measure.'}, {'Wave': 'Wave 4',
    'Source type': 'Young person', 'Variable': 'W4SENageMP', 'Review outcome': 'Retain as construction source', 'Decision reason': 'Establishes whether retrospective SEN identification among Wave 4 boost participants occurred before transition.', 'Leakage assessment': 'No direct outcome leakage when restricted to reported identification at age 15 or younger.', 'Reference-period assessment': 'Reported age at first SEN identification.', 'Review notes': 'Used only with W4SENMP for boost participants. All 21 valid Yes cases were identified by age 14.'}, {'Wave': 'Wave 4',
    'Source type': 'Young person', 'Variable': 'W4SENcurrMP', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Records current SEN status at Wave 4 rather than a clearly pre-transition status.', 'Leakage assessment': 'Potential temporal leakage because current status was measured at or after the transition boundary.', 'Reference-period assessment': 'Wave 4 current SEN status.', 'Review notes': 'Not used in the consolidated predictor. Only 19 explicit Yes or No responses were present.'}]
sen_decision_indices = []
for rule in sen_decision_rules:
    matching_rows = stage_2_variable_decision_register['Wave'].eq(rule['Wave']) & stage_2_variable_decision_register['Source type'].eq(rule['Source type']) & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {rule['Wave']}, {rule['Source type']}, {rule['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    sen_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = rule['Review outcome']
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'Special educational needs'
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = rule['Decision reason']
    stage_2_variable_decision_register.loc[matching_rows, 'Leakage assessment'] = rule['Leakage assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Reference-period assessment'] = rule['Reference-period assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave-specific data dictionaries and Stage 2 SEN routing and timing review'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = rule['Review notes']
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
sen_measure_review_output_path = stage_2_output_directory / 'stage_2_sen_measure_review.csv'
stage_2_sen_measure_review.to_csv(sen_measure_review_output_path, index=False)
sen_decision_output = stage_2_variable_decision_register.loc[sen_decision_indices, ['Wave', 'Source type', 'Variable',
    'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment', 'Review notes']].sort_values(['Review outcome',
    'Wave', 'Variable'])
sen_exclusion_output = sen_decision_output.loc[sen_decision_output['Review outcome'].eq('Exclude from predictor set'),
    ['Wave', 'Variable', 'Variable label', 'Decision reason', 'Leakage assessment']]
selected_sen_distribution = selected_sen_label.fillna('Missing').value_counts().rename_axis('Selected pre-transition SEN status').reset_index(name='Participants')
selected_sen_source_distribution = selected_sen_measure_source.fillna('No valid selected-measure source').value_counts().rename_axis('Selected SEN-measure source').reset_index(name='Participants')
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Participants in selected SEN-measure file: {len(stage_2_sen_measure_review):,}')
print('\nSelected three-category SEN distribution:')
print(selected_sen_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSelected SEN-measure source:')
print(selected_sen_source_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSEN source-variable decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 140, 'display.width', 390):
    print(sen_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables excluded in this step:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 140, 'display.width', 360):
    print(sen_exclusion_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'SEN-measure review file saved to: {sen_measure_review_output_path}')

Participants in selected SEN-measure file: 9,767

Selected three-category SEN distribution:
                         Selected pre-transition SEN status  Participants
                       No pre-transition SEN identification          7828
SEN identified; not current at latest pre-transition record          1035
    SEN identified; current at latest pre-transition record           779
                                                    Missing           125

Selected SEN-measure source:
                                    Selected SEN-measure source  Participants
                        Wave 1–Wave 2 cumulative identification          7624
         Wave 1–Wave 2 identification and Wave 3 current status          1733
                                                            ...           ...
Wave 1–Wave 2 identification and Wave 2 current-status fallback            63
Wave 1–Wave 2 identification and Wave 1 current-status fallback            18

SEN source-variable decisions:
  Wave  

In [133]:
# 61: SEN-type item coding and routing review

import re
sen_type_entries = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].isin(['Wave 1',
    'Wave 2', 'Wave 3']) & stage_2_master_variable_register['Source type'].eq('Young person') & stage_2_master_variable_register['Variable'].str.match('^W[123]sen1MP0[a-l]$',
    case=False, na=False), ['Source order', 'Wave', 'Source type', 'Source file', 'Source path', 'Variable position',
    'Variable', 'Variable label', 'Timing status']].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
if len(sen_type_entries) != 34:
    raise ValueError(f'Expected 34 Wave 1–Wave 3 SEN-type entries, but found {len(sen_type_entries)}.')
sen_type_entries['Option code'] = sen_type_entries['Variable'].str.extract('0([a-l])$', flags=re.IGNORECASE,
    expand=False).str.lower()
sen_type_entries['Option label'] = sen_type_entries['Variable label'].apply(lambda value: str(value).rsplit(':',
    1)[-1].strip())

def classify_sen_type_option(option_code):
    """Organise SEN-type options for review."""
    if option_code in list('abcdefgh'):
        return 'Named SEN-type option'
    if option_code == 'i':
        return 'Gifted or talented response'
    if option_code == 'j':
        return 'Other SEN response'
    if option_code == 'k':
        return "Don't-know response"
    if option_code == 'l':
        return 'Refused response'
    return 'Unclassified response option'
sen_type_entries['Review category'] = sen_type_entries['Option code'].apply(classify_sen_type_option)
sen_type_entries = sen_type_entries.merge(stage_2_variable_decision_register[['Source file', 'Variable',
    'Review outcome']], on=['Source file', 'Variable'], how='left', validate='one_to_one')
stage_2_sen_type_data = stage_2_participant_ids.copy()
sen_type_long_records = []
sen_type_source_presence = {}
for wave in ['Wave 1', 'Wave 2', 'Wave 3']:
    wave_entries = sen_type_entries.loc[sen_type_entries['Wave'].eq(wave)].copy()
    source_paths = wave_entries['Source path'].drop_duplicates().tolist()
    if len(source_paths) != 1:
        raise ValueError(f'Expected one source path for {wave}, but found {len(source_paths)}.')
    source_path = Path(source_paths[0])
    wave_variables = wave_entries['Variable'].tolist()
    numeric_data = pd.read_stata(source_path, columns=['NSID', *wave_variables], convert_categoricals=False)
    numeric_sample = stage_2_participant_ids.merge(numeric_data, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = numeric_sample['_merge'].eq('both')
    numeric_sample = numeric_sample.drop(columns='_merge')
    sen_type_source_presence[wave] = source_record_present
    for variable in wave_variables:
        numeric_values = pd.to_numeric(numeric_sample[variable], errors='coerce')
        stage_2_sen_type_data[f'{variable} code'] = numeric_values
        variable_entry = wave_entries.loc[wave_entries['Variable'].eq(variable)].iloc[0]
        sen_type_long_records.append(pd.DataFrame({'NSID': stage_2_participant_ids['NSID'], 'Wave': wave,
            'Variable': variable, 'Option code': variable_entry['Option code'], 'Option label': variable_entry['Option label'], 'Review category': variable_entry['Review category'], 'Value code': numeric_values, 'Source record present': source_record_present}))
stage_2_sen_type_long = pd.concat(sen_type_long_records, ignore_index=True)
stage_2_sen_type_long['Mentioned'] = stage_2_sen_type_long['Value code'].eq(1)
stage_2_sen_type_long['Not mentioned'] = stage_2_sen_type_long['Value code'].eq(0)
stage_2_sen_type_long['Valid multiple-response code'] = stage_2_sen_type_long['Value code'].isin([0, 1])

def sen_type_code_label(value):
    """Provide readable labels for stored SEN-type codes."""
    if pd.isna(value):
        return 'No source record'
    if value == 1:
        return 'Mentioned'
    if value == 0:
        return 'Not mentioned'
    if value == -91:
        return 'Not applicable'
    if value == -99:
        return 'MP not interviewed'
    if value == -998:
        return 'Interviewer missed question'
    return 'Other stored code'
stage_2_sen_type_long['Stored-code category'] = stage_2_sen_type_long['Value code'].apply(sen_type_code_label)
sen_type_code_structure = stage_2_sen_type_long.loc[stage_2_sen_type_long['Source record present']].groupby(['Wave',
    'Value code', 'Stored-code category'], dropna=False).agg(Variables=('Variable', 'nunique'),
    Participant_values=('NSID', 'size')).reset_index().sort_values(['Wave', 'Value code'],
    na_position='last').reset_index(drop=True)
sen_type_option_summary = stage_2_sen_type_long.groupby(['Wave', 'Variable', 'Option code', 'Option label',
    'Review category'], dropna=False).agg(Mentioned=('Mentioned', 'sum'), Not_mentioned=('Not mentioned', 'sum'),
    Not_applicable=('Value code', lambda values: int(values.eq(-91).sum())), Respondent_not_interviewed=('Value code',
    lambda values: int(values.eq(-99).sum())), Interviewer_missed=('Value code',
    lambda values: int(values.eq(-998).sum()))).reset_index().sort_values(['Wave',
    'Option code']).reset_index(drop=True)
current_sen_by_wave = {'Wave 1': sen_current_wave1, 'Wave 2': sen_current_wave2, 'Wave 3': sen_current_wave3_explicit}
substantive_categories = ['Named SEN-type option', 'Gifted or talented response', 'Other SEN response']
uncertainty_categories = ["Don't-know response", 'Refused response']
routing_records = []
type_count_records = []
for wave in ['Wave 1', 'Wave 2', 'Wave 3']:
    wave_entries = sen_type_entries.loc[sen_type_entries['Wave'].eq(wave)]
    all_variables = wave_entries['Variable'].tolist()
    substantive_variables = wave_entries.loc[wave_entries['Review category'].isin(substantive_categories),
        'Variable'].tolist()
    uncertainty_variables = wave_entries.loc[wave_entries['Review category'].isin(uncertainty_categories),
        'Variable'].tolist()
    all_code_matrix = pd.concat([stage_2_sen_type_data[f'{variable} code'].rename(variable) for variable in all_variables],
        axis=1)
    substantive_code_matrix = all_code_matrix[substantive_variables]
    if uncertainty_variables:
        uncertainty_code_matrix = all_code_matrix[uncertainty_variables]
    else:
        uncertainty_code_matrix = pd.DataFrame(index=all_code_matrix.index)
    any_type_section_response = all_code_matrix.isin([0, 1, -998]).any(axis=1)
    complete_substantive_response = substantive_code_matrix.isin([0, 1]).all(axis=1)
    selected_type_count = substantive_code_matrix.eq(1).sum(axis=1)
    any_substantive_type_selected = selected_type_count.gt(0)
    multiple_substantive_types = selected_type_count.gt(1)
    if uncertainty_variables:
        uncertainty_option_selected = uncertainty_code_matrix.eq(1).any(axis=1)
    else:
        uncertainty_option_selected = pd.Series(False, index=all_code_matrix.index)
    interviewer_missed_any_item = all_code_matrix.eq(-998).any(axis=1)
    current_sen_status = current_sen_by_wave[wave]
    current_sen_yes = current_sen_status.eq(1)
    current_sen_no = current_sen_status.eq(0)
    current_sen_unavailable = current_sen_status.isna()
    routing_records.append({'Wave': wave, 'Source record present': int(sen_type_source_presence[wave].sum()),
        'Current SEN Yes': int(current_sen_yes.sum()), 'Current SEN No': int(current_sen_no.sum()), 'Current SEN unavailable': int(current_sen_unavailable.sum()), 'Any SEN-type section response': int(any_type_section_response.sum()), 'Complete substantive option response': int(complete_substantive_response.sum()), 'At least one substantive type mentioned': int(any_substantive_type_selected.sum()), 'Multiple substantive types mentioned': int(multiple_substantive_types.sum()), 'Current SEN Yes with no type section response': int((current_sen_yes & ~any_type_section_response).sum()), 'Type section response outside Current SEN Yes': int((any_type_section_response & ~current_sen_yes).sum()), "Don't-know or refused option mentioned": int(uncertainty_option_selected.sum()), 'Interviewer missed at least one item': int(interviewer_missed_any_item.sum())})
    type_count_summary = selected_type_count.loc[current_sen_yes].value_counts().sort_index().rename_axis('Substantive types mentioned').reset_index(name='Participants')
    type_count_summary.insert(0, 'Wave', wave)
    type_count_records.append(type_count_summary)
sen_type_routing_summary = pd.DataFrame(routing_records)
sen_type_count_summary = pd.concat(type_count_records, ignore_index=True)
sen_type_variable_list = sen_type_entries[['Wave', 'Variable', 'Option code', 'Option label', 'Review category',
    'Timing status', 'Review outcome']].copy()
print(f'SEN-type variables reviewed: {len(sen_type_variable_list):,}')
print('\nStored code structure:')
print(sen_type_code_structure.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nRouting and response summary:')
with pd.option_context('display.max_columns', None, 'display.width', 390):
    print(sen_type_routing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nNumber of substantive SEN types mentioned among participants with current SEN:')
print(sen_type_count_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMention frequency by SEN-type option:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 125, 'display.width', 370):
    print(sen_type_option_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

SEN-type variables reviewed: 34

Stored code structure:
  Wave  Value code Stored-code category  Variables  Participant_values
Wave 1       -99.0   MP not interviewed         11                1133
Wave 1       -91.0       Not applicable         11               93643
   ...         ...                  ...        ...                 ...
Wave 3         0.0        Not mentioned         11                7178
Wave 3         1.0            Mentioned         11                 907

Routing and response summary:
  Wave  Source record present  Current SEN Yes  Current SEN No  Current SEN unavailable  Any SEN-type section response  Complete substantive option response  At least one substantive type mentioned  Multiple substantive types mentioned  Current SEN Yes with no type section response  Type section response outside Current SEN Yes  Don't-know or refused option mentioned  Interviewer missed at least one item
Wave 1                   9524              908            8447                 

In [134]:
# 62: SEN-type item exclusion and documentation

stage_2_sen_type_item_review = sen_type_entries.merge(sen_type_option_summary, on=['Wave', 'Variable', 'Option code',
    'Option label', 'Review category'], how='left', validate='one_to_one')

def sen_type_exclusion_reason(review_category):
    """Provide a reason suited to each SEN-type option."""
    if review_category == 'Named SEN-type option':
        return 'Conditional multiple-response SEN-type item asked only among participants routed through current SEN status. The item is sparse, non-mutually exclusive, repeated across waves and overlaps with the consolidated pre-transition SEN-status measure.'
    if review_category == 'Gifted or talented response':
        return 'Conditional multiple-response item that is conceptually different from disability or educational difficulty. It is sparse and cannot be combined defensibly with the other SEN-type responses.'
    if review_category == 'Other SEN response':
        return 'Non-specific conditional SEN response that cannot be assigned to a consistent substantive category. It also overlaps with the consolidated SEN-status measure.'
    if review_category == "Don't-know response":
        return 'Survey-response category rather than a substantive young-person characteristic.'
    if review_category == 'Refused response':
        return 'Survey non-response category rather than a substantive young-person characteristic.'
    return 'Conditional SEN-type response not retained as a distinct pre-transition predictor.'
stage_2_sen_type_item_review['Decision reason'] = stage_2_sen_type_item_review['Review category'].apply(sen_type_exclusion_reason)
stage_2_sen_type_item_review['Leakage assessment'] = 'No direct outcome leakage identified. Exclusion is based on conditional routing, overlap, sparsity and construct definition.'
stage_2_sen_type_item_review['Reference-period assessment'] = np.where(stage_2_sen_type_item_review['Wave'].eq('Wave 3'),
    'Near-transition SEN-type information collected before the post-16 transition.', 'Pre-transition SEN-type information.')
stage_2_sen_type_item_review['Review notes'] = 'Mentioned by ' + stage_2_sen_type_item_review['Mentioned'].fillna(0).astype(int).astype(str) + ' participants at ' + stage_2_sen_type_item_review['Wave'] + '. Retained in the audit file only; no consolidated SEN-type predictor was constructed.'
sen_type_decision_indices = []
for _, item in stage_2_sen_type_item_review.iterrows():
    matching_rows = stage_2_variable_decision_register['Source file'].eq(item['Source file']) & stage_2_variable_decision_register['Variable'].str.lower().eq(item['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {item['Wave']}, {item['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    sen_type_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = 'Exclude from predictor set'
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'Special educational needs'
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = item['Decision reason']
    stage_2_variable_decision_register.loc[matching_rows, 'Leakage assessment'] = item['Leakage assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Reference-period assessment'] = item['Reference-period assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave-specific data dictionaries and Stage 2 SEN-type coding and routing review'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = item['Review notes']
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
stage_2_sen_type_item_review = stage_2_sen_type_item_review.drop(columns='Review outcome').merge(stage_2_variable_decision_register[['Source file',
    'Variable', 'Review outcome']], on=['Source file', 'Variable'], how='left', validate='one_to_one')
sen_type_item_review_output_path = stage_2_output_directory / 'stage_2_sen_type_item_review.csv'
sen_type_routing_output_path = stage_2_output_directory / 'stage_2_sen_type_routing_summary.csv'
stage_2_sen_type_item_review.to_csv(sen_type_item_review_output_path, index=False)
sen_type_routing_summary.to_csv(sen_type_routing_output_path, index=False)
sen_type_exclusion_output = stage_2_variable_decision_register.loc[sen_type_decision_indices, ['Wave', 'Variable',
    'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment', 'Review notes']].sort_values(['Wave',
    'Variable'])
sen_type_exclusion_category_summary = stage_2_sen_type_item_review.groupby('Review category',
    dropna=False).agg(Variables=('Variable', 'size'), Total_mentions=('Mentioned',
    'sum')).reset_index().sort_values('Variables', ascending=False)
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'SEN-type variables excluded in this step: {len(sen_type_exclusion_output):,}')
print('\nExclusion summary by option category:')
print(sen_type_exclusion_category_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nEvery variable excluded in this step:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 145, 'display.width', 390):
    print(sen_type_exclusion_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'SEN-type item review saved to: {sen_type_item_review_output_path}')
print(f'SEN-type routing summary saved to: {sen_type_routing_output_path}')

SEN-type variables excluded in this step: 34

Exclusion summary by option category:
            Review category  Variables  Total_mentions
      Named SEN-type option         24            2891
        Don't-know response          3              26
Gifted or talented response          3             106
         Other SEN response          3             122
           Refused response          1               2

Every variable excluded in this step:
  Wave   Variable                                                                   Variable label             Review outcome                                                                                                                                                                                                                                      Decision reason                                                                                                           Leakage assessment                                                    

In [135]:
# 63: Age at first SEN-identification review

sen_age_specifications = [{'Wave': 'Wave 1', 'Source type': 'Young person', 'Variable': 'W1senageMP',
    'Output name': 'W1 SEN identification age'}, {'Wave': 'Wave 2', 'Source type': 'Young person',
    'Variable': 'W2SenAgeMP', 'Output name': 'W2 SEN identification age'}, {'Wave': 'Wave 4',
    'Source type': 'Young person', 'Variable': 'W4SENageMP', 'Output name': 'W4 SEN identification age'}]
stage_2_sen_age_data = stage_2_participant_ids.copy()
sen_age_coverage_records = []
sen_age_value_records = []
for specification in sen_age_specifications:
    matching_entry = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq(specification['Wave']) & stage_2_master_variable_register['Source type'].eq(specification['Source type']) & stage_2_master_variable_register['Variable'].str.lower().eq(specification['Variable'].lower())]
    if len(matching_entry) != 1:
        raise ValueError(f"Expected one source entry for {specification['Wave']}, {specification['Variable']}; found {len(matching_entry)}.")
    source_path = Path(matching_entry.iloc[0]['Source path'])
    source_variable = matching_entry.iloc[0]['Variable']
    numeric_data = pd.read_stata(source_path, columns=['NSID', source_variable], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', source_variable], convert_categoricals=True)
    numeric_sample = stage_2_participant_ids.merge(numeric_data, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = numeric_sample['_merge'].eq('both')
    numeric_sample = numeric_sample.drop(columns='_merge')
    labelled_sample = stage_2_participant_ids.merge(labelled_data, on='NSID', how='left', validate='one_to_one')
    numeric_values = pd.to_numeric(numeric_sample[source_variable], errors='coerce')
    labelled_values = labelled_sample[source_variable].astype('string')
    output_name = specification['Output name']
    stage_2_sen_age_data[f'{output_name} code'] = numeric_values
    stage_2_sen_age_data[f'{output_name} label'] = labelled_values
    stage_2_sen_age_data[f'{output_name} source present'] = source_record_present
    possible_age = numeric_values.where(numeric_values.ge(0))
    stage_2_sen_age_data[f'{output_name} valid age'] = possible_age
    sen_age_coverage_records.append({'Wave': specification['Wave'], 'Variable': source_variable,
        'Variable label': matching_entry.iloc[0]['Variable label'], 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Reported non-negative age': int(possible_age.notna().sum()), 'Not applicable': int(numeric_values.eq(-91).sum()), 'Respondent not interviewed': int(numeric_values.eq(-99).sum()), 'Other negative code': int((numeric_values.lt(0) & ~numeric_values.isin([-91,
        -99])).sum()), 'Minimum reported age': possible_age.min(), 'Maximum reported age': possible_age.max()})
    value_summary = pd.DataFrame({'Value code': numeric_values, 'Value label': labelled_values,
        'Source record present': source_record_present}).loc[lambda frame: frame['Source record present']].drop(columns='Source record present').value_counts(dropna=False).rename('Participants').reset_index()
    value_summary.insert(0, 'Wave', specification['Wave'])
    value_summary.insert(1, 'Variable', source_variable)
    sen_age_value_records.append(value_summary)
stage_2_sen_age_coverage = pd.DataFrame(sen_age_coverage_records)
stage_2_sen_age_values = pd.concat(sen_age_value_records, ignore_index=True)
w1_sen_age = stage_2_sen_age_data['W1 SEN identification age valid age']
w2_sen_age = stage_2_sen_age_data['W2 SEN identification age valid age']
w4_sen_age = stage_2_sen_age_data['W4 SEN identification age valid age']
w1_w2_age_both_valid = w1_sen_age.notna() & w2_sen_age.notna()
w1_w2_age_agree = w1_w2_age_both_valid & w1_sen_age.eq(w2_sen_age)
w1_w2_age_comparison = pd.DataFrame([{'Comparison': 'Wave 1 versus Wave 2',
    'Both ages valid': int(w1_w2_age_both_valid.sum()), 'Same reported age': int(w1_w2_age_agree.sum()), 'Wave 2 older': int((w1_w2_age_both_valid & w2_sen_age.gt(w1_sen_age)).sum()), 'Wave 2 younger': int((w1_w2_age_both_valid & w2_sen_age.lt(w1_sen_age)).sum())}])
w1_age_by_ever_status = pd.crosstab(stage_2_sen_status_data['W1 SEN ever status'],
    w1_sen_age.notna().map({True: 'Reported age', False: 'No reported age'}), margins=True, margins_name='Total')
w2_age_by_ever_status = pd.crosstab(stage_2_sen_status_data['W2 SEN ever status'],
    w2_sen_age.notna().map({True: 'Reported age', False: 'No reported age'}), margins=True, margins_name='Total')
w2_newly_identified = stage_2_sen_status_data['W2 SEN ever code'].eq(1)
w1_previously_identified = stage_2_sen_status_data['W1 SEN ever code'].eq(1)
w2_age_routing_summary = pd.DataFrame([{'Review group': 'Wave 2 reported age',
    'Participants': int(w2_sen_age.notna().sum())}, {'Review group': 'Wave 2 age among newly identified Wave 2 SEN cases',
    'Participants': int((w2_sen_age.notna() & w2_newly_identified).sum())}, {'Review group': 'Wave 2 age among participants already identified at Wave 1',
    'Participants': int((w2_sen_age.notna() & w1_previously_identified).sum())}, {'Review group': 'Wave 2 age where Wave 1 age unavailable',
    'Participants': int((w2_sen_age.notna() & w1_sen_age.isna()).sum())}, {'Review group': 'Wave 4 retrospective age for boost participants',
    'Participants': int((w4_sen_age.notna() & w4_boost_participant).sum())}])

def group_sen_identification_age(age_values):
    """Group reported ages without changing source values."""
    return pd.Series(np.select([age_values.between(0, 5, inclusive='both'), age_values.between(6, 10,
        inclusive='both'), age_values.between(11, 15, inclusive='both'), age_values.ge(16)], ['Age 0–5', 'Age 6–10',
        'Age 11–15', 'Age 16 or older'], default='No valid age'), index=age_values.index, dtype='string')
sen_age_group_records = []
for wave, ages in [('Wave 1', w1_sen_age), ('Wave 2', w2_sen_age), ('Wave 4', w4_sen_age)]:
    age_group_summary = group_sen_identification_age(ages).value_counts().rename_axis('Age group').reset_index(name='Participants')
    age_group_summary.insert(0, 'Wave', wave)
    sen_age_group_records.append(age_group_summary)
stage_2_sen_age_group_summary = pd.concat(sen_age_group_records, ignore_index=True)
print(f'Age-at-first-SEN-identification variables reviewed: {len(stage_2_sen_age_coverage):,}')
print('\nCoverage and code summary:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 380):
    print(stage_2_sen_age_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nStored values and labels:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 110, 'display.width', 300):
    print(stage_2_sen_age_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 1–Wave 2 reported-age comparison:')
print(w1_w2_age_comparison.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 1 ever-SEN status by age availability:')
print(w1_age_by_ever_status.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nWave 2 ever-SEN status by age availability:')
print(w2_age_by_ever_status.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nWave 2 and Wave 4 routing summary:')
print(w2_age_routing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nReported age-group distribution:')
print(stage_2_sen_age_group_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Age-at-first-SEN-identification variables reviewed: 3

Coverage and code summary:
  Wave   Variable                                                          Variable label  Source record present  Source record absent  Reported non-negative age  Not applicable  Respondent not interviewed  Other negative code  Minimum reported age  Maximum reported age
Wave 1 W1senageMP MP: Age of YP when first identified as having special educational needs                   9524                   243                       1590            7801                         103                   30                   0.0                  14.0
Wave 2 W2SenAgeMP MP: Age of YP when first identified as having special educational needs                   9521                   246                        192            9247                          79                    3                   0.0                  15.0
Wave 4 W4SENageMP MP: Age of YP when first identified as having special educational needs                

In [136]:
# 64: Consolidated SEN-identification timing review

sen_identification_age_pretransition = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Float64')
sen_identification_age_source = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
wave1_age_source = sen_identified_pretransition.eq(1) & w1_sen_age.notna()
sen_identification_age_pretransition.loc[wave1_age_source] = w1_sen_age.loc[wave1_age_source]
sen_identification_age_source.loc[wave1_age_source] = 'Wave 1 age at first identification'
wave2_age_source = sen_identified_pretransition.eq(1) & sen_identification_age_pretransition.isna() & w2_sen_age.notna()
sen_identification_age_pretransition.loc[wave2_age_source] = w2_sen_age.loc[wave2_age_source]
sen_identification_age_source.loc[wave2_age_source] = 'Wave 2 newly identified SEN age'
wave4_age_source = w4_retrospective_pretransition_yes & sen_identification_age_pretransition.isna() & w4_sen_age.notna() & w4_sen_age.le(15)
sen_identification_age_pretransition.loc[wave4_age_source] = w4_sen_age.loc[wave4_age_source]
sen_identification_age_source.loc[wave4_age_source] = 'Wave 4 boost retrospective SEN age'
sen_identification_age_group = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
sen_identification_age_group.loc[sen_identification_age_pretransition.between(0, 5, inclusive='both')] = 'Age 0–5'
sen_identification_age_group.loc[sen_identification_age_pretransition.between(6, 10, inclusive='both')] = 'Age 6–10'
sen_identification_age_group.loc[sen_identification_age_pretransition.between(11, 15, inclusive='both')] = 'Age 11–15'
sen_identification_age_group.loc[sen_identified_pretransition.eq(1) & sen_identification_age_pretransition.isna()] = 'SEN identified; age unavailable'
sen_identification_age_group.loc[sen_identified_pretransition.eq(0)] = 'No pre-transition SEN identification'
stage_2_sen_identification_timing_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'sen_identified_pretransition': sen_identified_pretransition, 'sen_identification_age_pretransition': sen_identification_age_pretransition, 'SEN-identification age group': sen_identification_age_group, 'SEN-identification age source': sen_identification_age_source, 'sen_status_pretransition': selected_sen_measure, 'Pre-transition SEN status': selected_sen_label})
sen_age_source_distribution = sen_identification_age_source.fillna('No age source used').value_counts().rename_axis('SEN-identification age source').reset_index(name='Participants')
identified_age_group_distribution = sen_identification_age_group.loc[sen_identified_pretransition.eq(1)].fillna('Age group unavailable').value_counts().rename_axis('Age at first SEN identification').reset_index(name='Participants')
sen_age_by_selected_status = pd.crosstab(selected_sen_label.fillna('Selected SEN status unavailable'),
    sen_identification_age_group.fillna('Identification timing unavailable'), margins=True, margins_name='Total')
sen_age_quality_checks = pd.DataFrame([{'Quality check': 'Participants identified as having pre-transition SEN',
    'Participants': int(sen_identified_pretransition.eq(1).sum())}, {'Quality check': 'Identified participants with a valid identification age',
    'Participants': int((sen_identified_pretransition.eq(1) & sen_identification_age_pretransition.notna()).sum())}, {'Quality check': 'Identified participants without a valid identification age',
    'Participants': int((sen_identified_pretransition.eq(1) & sen_identification_age_pretransition.isna()).sum())}, {'Quality check': 'Age recorded for participants classified as having no SEN',
    'Participants': int((sen_identified_pretransition.eq(0) & sen_identification_age_pretransition.notna()).sum())}, {'Quality check': 'Reported identification age above 15',
    'Participants': int(sen_identification_age_pretransition.gt(15).sum())}, {'Quality check': 'Participants assigned more than one age source',
    'Participants': int((wave1_age_source.astype(int) + wave2_age_source.astype(int) + wave4_age_source.astype(int)).gt(1).sum())}])
sen_age_by_source = stage_2_sen_identification_timing_review.loc[sen_identification_age_pretransition.notna()].groupby(['SEN-identification age source',
    'SEN-identification age group'], dropna=False).size().reset_index(name='Participants').sort_values(['SEN-identification age source',
    'SEN-identification age group']).reset_index(drop=True)
print(f'Participants in SEN-identification timing review: {len(stage_2_sen_identification_timing_review):,}')
print('\nIdentification-age source:')
print(sen_age_source_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nAge at first identification among participants with pre-transition SEN identification:')
print(identified_age_group_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nIdentification-age groups by source:')
print(sen_age_by_source.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSelected SEN status by age at first identification:')
print(sen_age_by_selected_status.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nQuality checks:')
print(sen_age_quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in SEN-identification timing review: 9,767

Identification-age source:
     SEN-identification age source  Participants
                No age source used          7964
Wave 1 age at first identification          1590
   Wave 2 newly identified SEN age           192
Wave 4 boost retrospective SEN age            21

Age at first identification among participants with pre-transition SEN identification:
Age at first SEN identification  Participants
                       Age 6–10           995
                      Age 11–15           491
                        Age 0–5           317
SEN identified; age unavailable            33

Identification-age groups by source:
     SEN-identification age source SEN-identification age group  Participants
Wave 1 age at first identification                      Age 0–5           282
Wave 1 age at first identification                    Age 11–15           396
                               ...                          ...           ...
Wav

In [137]:
# 65: SEN-identification timing decision and documentation

sen_age_decision_rules = [{'Wave': 'Wave 1', 'Source type': 'Young person', 'Variable': 'W1senageMP',
    'Review outcome': 'Retain as review support only', 'Decision reason': 'Provides age at first SEN identification for participants already identified by Wave 1. It is not retained as a separate predictor because it is structurally defined only among SEN-identified participants and overlaps with the selected three-category SEN-status measure.', 'Leakage assessment': 'No direct outcome leakage identified. The information was collected before transition.', 'Reference-period assessment': 'Age at first SEN identification reported at Wave 1.', 'Review notes': 'Valid age was available for 1,590 participants. Retained for timing checks and audit only.'}, {'Wave': 'Wave 2',
    'Source type': 'Young person', 'Variable': 'W2SenAgeMP', 'Review outcome': 'Retain as review support only', 'Decision reason': 'Provides age at first SEN identification for cases newly identified at Wave 2. It is not retained as a separate predictor because it is conditionally routed and overlaps with the selected three-category SEN-status measure.', 'Leakage assessment': 'No direct outcome leakage identified. The information was collected before transition.', 'Reference-period assessment': 'Age at first SEN identification for newly identified Wave 2 cases.', 'Review notes': 'Valid age was available for 192 newly identified participants. No Wave 1 and Wave 2 valid ages overlapped.'}, {'Wave': 'Wave 4',
    'Source type': 'Young person', 'Variable': 'W4SENageMP', 'Review outcome': 'Retain as construction source', 'Decision reason': 'Used only to confirm that retrospective SEN identification among Wave 4 boost participants occurred before the post-16 transition.', 'Leakage assessment': 'Restricted retrospective use only. A Wave 4 SEN Yes response is used only where reported first identification occurred by age 15.', 'Reference-period assessment': 'Retrospective age at first SEN identification.', 'Review notes': 'All 21 boost participants with a valid retrospective SEN Yes response reported first identification between ages 1 and 14.'}]
sen_age_decision_indices = []
for rule in sen_age_decision_rules:
    matching_rows = stage_2_variable_decision_register['Wave'].eq(rule['Wave']) & stage_2_variable_decision_register['Source type'].eq(rule['Source type']) & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {rule['Wave']}, {rule['Source type']}, {rule['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    sen_age_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = rule['Review outcome']
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'Special educational needs'
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = rule['Decision reason']
    stage_2_variable_decision_register.loc[matching_rows, 'Leakage assessment'] = rule['Leakage assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Reference-period assessment'] = rule['Reference-period assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave-specific data dictionaries and Stage 2 SEN-identification timing review'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = rule['Review notes']
sen_identification_timing_output_path = stage_2_output_directory / 'stage_2_sen_identification_timing_review.csv'
stage_2_sen_identification_timing_review.to_csv(sen_identification_timing_output_path, index=False)
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
sen_age_decision_output = stage_2_variable_decision_register.loc[sen_age_decision_indices, ['Wave', 'Source type',
    'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment', 'Review notes']].sort_values(['Review outcome',
    'Wave', 'Variable'])
sen_age_review_support_output = sen_age_decision_output.loc[sen_age_decision_output['Review outcome'].eq('Retain as review support only'),
    ['Wave', 'Variable', 'Variable label', 'Decision reason', 'Leakage assessment']]
sen_age_review_summary = pd.DataFrame([{'Review item': 'Pre-transition SEN identification',
    'Participants': int(sen_identified_pretransition.eq(1).sum())}, {'Review item': 'Valid age at first identification',
    'Participants': int(sen_identification_age_pretransition.notna().sum())}, {'Review item': 'SEN identified but age unavailable',
    'Participants': int((sen_identified_pretransition.eq(1) & sen_identification_age_pretransition.isna()).sum())}, {'Review item': 'Age above 15 retained',
    'Participants': int(sen_identification_age_pretransition.gt(15).sum())}])
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'SEN age-at-identification variables reviewed: {len(sen_age_decision_output):,}')
print('\nTiming-review summary:')
print(sen_age_review_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSEN age-variable decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 145, 'display.width', 390):
    print(sen_age_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables retained for review support only and not entering the predictor matrix:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 145, 'display.width', 360):
    print(sen_age_review_support_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'SEN-identification timing review saved to: {sen_identification_timing_output_path}')

SEN age-at-identification variables reviewed: 3

Timing-review summary:
                       Review item  Participants
 Pre-transition SEN identification          1836
 Valid age at first identification          1803
SEN identified but age unavailable            33
             Age above 15 retained             0

SEN age-variable decisions:
  Wave  Source type   Variable                                                          Variable label                Review outcome                                                                                                                                                                                                                                                                Decision reason                                                                                                                Leakage assessment                                                                                                               Review not

In [138]:
# 66: Disability derivation-input decisions

disability_input_rules = [{'Wave': 'Wave 1', 'Variable': 'W1chea1HS',
    'Input role': 'Long-standing illness, disability or infirmity status'}, {'Wave': 'Wave 1', 'Variable': 'W1chea7HS',
    'Input role': 'Effect on regular school attendance'}, {'Wave': 'Wave 1', 'Variable': 'W1chea8HS',
    'Input role': 'Effect on ability to do schoolwork'}, {'Wave': 'Wave 2', 'Variable': 'W2chea1HS',
    'Input role': 'Long-standing illness, disability or infirmity status'}, {'Wave': 'Wave 2', 'Variable': 'W2chea7HS',
    'Input role': 'Effect on regular school attendance'}, {'Wave': 'Wave 2', 'Variable': 'W2chea8HS',
    'Input role': 'Effect on ability to do schoolwork'}]
disability_input_decision_indices = []
for rule in disability_input_rules:
    matching_rows = stage_2_variable_decision_register['Wave'].eq(rule['Wave']) & stage_2_variable_decision_register['Source type'].eq('Young person') & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {rule['Wave']}, {rule['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    disability_input_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = 'Retain as review support only'
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'Health and disability'
    stage_2_variable_decision_register.loc[matching_rows,
        'Decision reason'] = f"Documented derivation input to the selected {rule['Wave']} disability and schooling measure. It is not retained as a separate predictor because its information is already represented in the three-category derived disability measure."
    stage_2_variable_decision_register.loc[matching_rows,
        'Leakage assessment'] = 'No direct outcome leakage identified. The information was collected before transition.'
    stage_2_variable_decision_register.loc[matching_rows,
        'Reference-period assessment'] = f"{rule['Wave']} pre-transition health and schooling information."
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Official Wave 1 and Wave 2 derived-variable documentation for W1disabYP and W2disabYP'
    stage_2_variable_decision_register.loc[matching_rows,
        'Review notes'] = f"Derivation role: {rule['Input role']}. Retained for audit and derivation verification only."
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
disability_input_decision_output = stage_2_variable_decision_register.loc[disability_input_decision_indices, ['Wave',
    'Source type', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment', 'Review notes']].sort_values(['Wave',
    'Variable']).reset_index(drop=True)
disability_input_review_output_path = stage_2_output_directory / 'stage_2_disability_derivation_input_review.csv'
disability_input_decision_output.to_csv(disability_input_review_output_path, index=False)
care_burden_check = stage_2_variable_decision_register.loc[stage_2_variable_decision_register['Variable'].str.lower().isin(['w1chea4bhs',
    'w2chea4bhs']), ['Wave', 'Variable', 'Variable label', 'Review outcome']].sort_values(['Wave', 'Variable'])
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Disability derivation inputs reviewed: {len(disability_input_decision_output):,}')
print('\nDerivation-input decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 145, 'display.width', 390):
    print(disability_input_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nRelated care-burden variables not decided in this step:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 145, 'display.width', 350):
    print(care_burden_check.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'Disability derivation-input review saved to: {disability_input_review_output_path}')

Disability derivation inputs reviewed: 6

Derivation-input decisions:
  Wave  Source type  Variable                                                            Variable label                Review outcome                                                                                                                                                                                                                  Decision reason                                                                     Leakage assessment                                                                                                                 Review notes
Wave 1 Young person W1chea1HS     HR: Whether YP has any long-standing illness, disability or infirmity Retain as review support only Documented derivation input to the selected Wave 1 disability and schooling measure. It is not retained as a separate predictor because its information is already represented in the three-category derived disability measure. 

In [139]:
# 67: Additional parental care-burden review

care_burden_specifications = [{'Wave': 'Wave 1', 'Source type': 'Young person', 'Variable': 'W1chea4bHS',
    'Output name': 'W1 additional care burden'}, {'Wave': 'Wave 2', 'Source type': 'Young person',
    'Variable': 'W2chea4bHS', 'Output name': 'W2 additional care burden'}]
stage_2_care_burden_data = stage_2_participant_ids.copy()
care_burden_coverage_records = []
care_burden_value_records = []
for specification in care_burden_specifications:
    matching_entry = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq(specification['Wave']) & stage_2_master_variable_register['Source type'].eq(specification['Source type']) & stage_2_master_variable_register['Variable'].str.lower().eq(specification['Variable'].lower())]
    if len(matching_entry) != 1:
        raise ValueError(f"Expected one source entry for {specification['Wave']}, {specification['Variable']}; found {len(matching_entry)}.")
    source_path = Path(matching_entry.iloc[0]['Source path'])
    source_variable = matching_entry.iloc[0]['Variable']
    numeric_data = pd.read_stata(source_path, columns=['NSID', source_variable], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', source_variable], convert_categoricals=True)
    numeric_sample = stage_2_participant_ids.merge(numeric_data, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = numeric_sample['_merge'].eq('both')
    numeric_sample = numeric_sample.drop(columns='_merge')
    labelled_sample = stage_2_participant_ids.merge(labelled_data, on='NSID', how='left', validate='one_to_one')
    numeric_values = pd.to_numeric(numeric_sample[source_variable], errors='coerce')
    labelled_values = labelled_sample[source_variable].astype('string')
    output_name = specification['Output name']
    stage_2_care_burden_data[f'{output_name} code'] = numeric_values
    stage_2_care_burden_data[f'{output_name} label'] = labelled_values
    stage_2_care_burden_data[f'{output_name} source present'] = source_record_present
    valid_yes_no = numeric_values.where(numeric_values.isin([1, 2]))
    stage_2_care_burden_data[f'{output_name} valid'] = valid_yes_no
    care_burden_coverage_records.append({'Wave': specification['Wave'], 'Variable': source_variable,
        'Variable label': matching_entry.iloc[0]['Variable label'], 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Valid Yes or No': int(valid_yes_no.notna().sum()), 'Yes: additional care required': int(numeric_values.eq(1).sum()), 'No: no additional care': int(numeric_values.eq(2).sum()), 'Not applicable': int(numeric_values.eq(-91).sum()), 'Respondent not interviewed': int(numeric_values.eq(-99).sum()), 'Other negative code': int((numeric_values.lt(0) & ~numeric_values.isin([-91,
        -99])).sum())})
    value_summary = pd.DataFrame({'Value code': numeric_values, 'Value label': labelled_values,
        'Source record present': source_record_present}).loc[lambda frame: frame['Source record present']].drop(columns='Source record present').value_counts(dropna=False).rename('Participants').reset_index()
    value_summary.insert(0, 'Wave', specification['Wave'])
    value_summary.insert(1, 'Variable', source_variable)
    care_burden_value_records.append(value_summary)
stage_2_care_burden_coverage = pd.DataFrame(care_burden_coverage_records)
stage_2_care_burden_values = pd.concat(care_burden_value_records, ignore_index=True)
w1_care_burden = stage_2_care_burden_data['W1 additional care burden valid']
w2_care_burden = stage_2_care_burden_data['W2 additional care burden valid']
disability_category_labels = {1.0: 'Disability or long-standing illness; schooling affected',
    2.0: 'Disability or long-standing illness; schooling not affected', 3.0: 'No disability or long-standing illness'}
w1_disability_label = valid_disability_measures['W1 disability and schooling'].map(disability_category_labels).fillna('Disability category unavailable')
w2_disability_label = valid_disability_measures['W2 disability and schooling'].map(disability_category_labels).fillna('Disability category unavailable')
care_burden_labels = {1.0: 'Additional care required', 2.0: 'No additional care required'}
w1_care_burden_label = w1_care_burden.map(care_burden_labels).fillna('No valid care-burden response')
w2_care_burden_label = w2_care_burden.map(care_burden_labels).fillna('No valid care-burden response')
w1_disability_by_care_burden = pd.crosstab(w1_disability_label, w1_care_burden_label, margins=True,
    margins_name='Total')
w2_disability_by_care_burden = pd.crosstab(w2_disability_label, w2_care_burden_label, margins=True,
    margins_name='Total')
both_care_burden_valid = w1_care_burden.notna() & w2_care_burden.notna()
care_burden_agree = both_care_burden_valid & w1_care_burden.eq(w2_care_burden)
care_burden_cross_wave_summary = pd.DataFrame([{'Comparison': 'Wave 1 versus Wave 2',
    'Both responses valid': int(both_care_burden_valid.sum()), 'Responses agreeing': int(care_burden_agree.sum()), 'Responses differing': int((both_care_burden_valid & w1_care_burden.ne(w2_care_burden)).sum()), 'Agreement percentage': care_burden_agree.sum() / both_care_burden_valid.sum() * 100 if both_care_burden_valid.sum() else np.nan}])
care_burden_cross_tabulation = pd.crosstab(w1_care_burden.map(care_burden_labels),
    w2_care_burden.map(care_burden_labels), margins=True, margins_name='Total')
care_burden_coverage_comparison = pd.DataFrame([{'Coverage item': 'Valid Wave 1 care-burden response',
    'Participants': int(w1_care_burden.notna().sum())}, {'Coverage item': 'Valid Wave 2 care-burden response',
    'Participants': int(w2_care_burden.notna().sum())}, {'Coverage item': 'Wave 2 valid where Wave 1 is unavailable',
    'Participants': int((w2_care_burden.notna() & w1_care_burden.isna()).sum())}, {'Coverage item': 'Wave 1 valid where Wave 2 is unavailable',
    'Participants': int((w1_care_burden.notna() & w2_care_burden.isna()).sum())}, {'Coverage item': 'At least one valid care-burden response',
    'Participants': int((w1_care_burden.notna() | w2_care_burden.notna()).sum())}])
care_burden_routing_checks = pd.DataFrame([{'Routing check': 'Wave 1 valid care response with no Wave 1 disability',
    'Participants': int((w1_care_burden.notna() & valid_disability_measures['W1 disability and schooling'].eq(3)).sum())}, {'Routing check': 'Wave 2 valid care response with no Wave 2 disability',
    'Participants': int((w2_care_burden.notna() & valid_disability_measures['W2 disability and schooling'].eq(3)).sum())}, {'Routing check': 'Wave 1 disability reported but care response unavailable',
    'Participants': int((valid_disability_measures['W1 disability and schooling'].isin([1,
    2]) & w1_care_burden.isna()).sum())}, {'Routing check': 'Wave 2 disability reported but care response unavailable',
    'Participants': int((valid_disability_measures['W2 disability and schooling'].isin([1,
    2]) & w2_care_burden.isna()).sum())}])
print(f'Additional parental care-burden variables reviewed: {len(stage_2_care_burden_coverage):,}')
print('\nCoverage and code summary:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 380):
    print(stage_2_care_burden_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nStored values and labels:')
print(stage_2_care_burden_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 1 disability category by care burden:')
print(w1_disability_by_care_burden.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nWave 2 disability category by care burden:')
print(w2_disability_by_care_burden.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nCross-wave care-burden comparison:')
print(care_burden_cross_wave_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Agreement percentage': lambda value: f'{value:.2f}%'}))
print('\nWave 1 by Wave 2 care-burden response:')
print(care_burden_cross_tabulation.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nCoverage comparison:')
print(care_burden_coverage_comparison.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nRouting checks:')
print(care_burden_routing_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Additional parental care-burden variables reviewed: 2

Coverage and code summary:
  Wave   Variable                                                                   Variable label  Source record present  Source record absent  Valid Yes or No  Yes: additional care required  No: no additional care  Not applicable  Respondent not interviewed  Other negative code
Wave 1 W1chea4bHS HR: Whether have to spend longer looking after YP because of illness, disability                   9524                   243             1208                            345                     863            8084                         176                   56
Wave 2 W2chea4bHS HR: Whether have to spend longer looking after YP because of illness, disability                   9521                   246               16                              4                      12            9283                         143                   79

Stored values and labels:
  Wave   Variable  Value code                   

In [140]:
# 68: Consolidated additional-care indicator review

w1_disability_for_care_review = valid_disability_measures['W1 disability and schooling'].copy()
w2_disability_for_care_review = valid_disability_measures['W2 disability and schooling'].copy()
consolidated_disability_for_care_review = w1_disability_for_care_review.copy()
wave2_disability_fallback = consolidated_disability_for_care_review.isna() & w2_disability_for_care_review.notna()
consolidated_disability_for_care_review.loc[wave2_disability_fallback] = w2_disability_for_care_review.loc[wave2_disability_fallback]
consolidated_care_burden_response = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
consolidated_care_burden_source = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
wave1_care_source = w1_care_burden.notna()
consolidated_care_burden_response.loc[wave1_care_source] = w1_care_burden.loc[wave1_care_source].astype('Int64')
consolidated_care_burden_source.loc[wave1_care_source] = 'Wave 1 care-burden response'
wave2_care_source = consolidated_care_burden_response.isna() & w2_care_burden.notna()
consolidated_care_burden_response.loc[wave2_care_source] = w2_care_burden.loc[wave2_care_source].astype('Int64')
consolidated_care_burden_source.loc[wave2_care_source] = 'Wave 2 care-burden response'
additional_parental_care_indicator = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
additional_parental_care_indicator.loc[consolidated_care_burden_response.eq(1)] = 1
additional_parental_care_indicator.loc[consolidated_care_burden_response.eq(2)] = 0
no_disability_status = consolidated_disability_for_care_review.eq(3)
no_disability_without_care_response = no_disability_status & consolidated_care_burden_response.isna()
additional_parental_care_indicator.loc[no_disability_without_care_response] = 0
consolidated_care_burden_source.loc[no_disability_without_care_response] = 'No disability or long-standing illness'
additional_parental_care_label = additional_parental_care_indicator.map({0: 'No additional care due to disability',
    1: 'Additional care due to disability'})
additional_parental_care_missing_reason = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
additional_parental_care_missing = additional_parental_care_indicator.isna()
additional_parental_care_missing_reason.loc[additional_parental_care_missing & consolidated_disability_for_care_review.isna()] = 'Consolidated disability status unavailable'
additional_parental_care_missing_reason.loc[additional_parental_care_missing & consolidated_disability_for_care_review.isin([1,
    2]) & consolidated_care_burden_response.isna()] = 'Disability reported, but valid care-burden response unavailable'
stage_2_additional_care_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'consolidated_disability_status': consolidated_disability_for_care_review, 'additional_parental_care_due_to_disability': additional_parental_care_indicator, 'Additional parental care': additional_parental_care_label, 'Additional-care source': consolidated_care_burden_source, 'Additional-care missing reason': additional_parental_care_missing_reason, 'W1 care-burden response': w1_care_burden, 'W2 care-burden response': w2_care_burden})
disability_category_labels = {1.0: 'Disability or long-standing illness; schooling affected',
    2.0: 'Disability or long-standing illness; schooling not affected', 3.0: 'No disability or long-standing illness'}
consolidated_disability_label = consolidated_disability_for_care_review.map(disability_category_labels).fillna('Disability category unavailable')
additional_care_distribution = additional_parental_care_label.fillna('Missing').value_counts().rename_axis('Additional parental care indicator').reset_index(name='Participants')
additional_care_source_distribution = consolidated_care_burden_source.fillna('No valid construction source').value_counts().rename_axis('Additional-care construction source').reset_index(name='Participants')
additional_care_missing_summary = additional_parental_care_missing_reason.loc[additional_parental_care_missing].fillna('Unclassified missing reason').value_counts().rename_axis('Missing reason').reset_index(name='Participants')
disability_by_additional_care = pd.crosstab(consolidated_disability_label,
    additional_parental_care_label.fillna('Additional-care indicator unavailable'), margins=True, margins_name='Total')
care_prevalence_by_disability = pd.DataFrame({'Disability category': consolidated_disability_label,
    'Additional care': additional_parental_care_indicator}).loc[lambda frame: frame['Disability category'].isin(['Disability or long-standing illness; schooling affected',
    'Disability or long-standing illness; schooling not affected'])].groupby('Disability category',
    dropna=False).agg(Disability_cases=('Additional care', 'size'), Valid_care_indicator=('Additional care', 'count'),
    Additional_care_cases=('Additional care', lambda values: int(values.eq(1).sum()))).reset_index()
care_prevalence_by_disability['Additional care percentage'] = care_prevalence_by_disability['Additional_care_cases'] / care_prevalence_by_disability['Valid_care_indicator'] * 100
logical_care_conflict = no_disability_status & consolidated_care_burden_response.eq(1)
additional_care_quality_checks = pd.DataFrame([{'Quality check': 'Participants with at least one valid care-burden response',
    'Participants': int(consolidated_care_burden_response.notna().sum())}, {'Quality check': 'Additional-care Yes',
    'Participants': int(additional_parental_care_indicator.eq(1).sum())}, {'Quality check': 'Additional-care No',
    'Participants': int(additional_parental_care_indicator.eq(0).sum())}, {'Quality check': 'Additional-care indicator missing',
    'Participants': int(additional_parental_care_indicator.isna().sum())}, {'Quality check': 'Disability cases with care indicator missing',
    'Participants': int((consolidated_disability_for_care_review.isin([1,
    2]) & additional_parental_care_indicator.isna()).sum())}, {'Quality check': 'No-disability cases reporting additional care',
    'Participants': int(logical_care_conflict.sum())}, {'Quality check': 'Participants assigned both Wave 1 and Wave 2 care sources',
    'Participants': int((w1_care_burden.notna() & w2_care_burden.notna()).sum())}])
print(f'Participants in additional-care indicator review: {len(stage_2_additional_care_review):,}')
print('\nProposed additional-care indicator:')
print(additional_care_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nConstruction source:')
print(additional_care_source_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nDisability category by additional-care indicator:')
print(disability_by_additional_care.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nAdditional-care prevalence within disability categories:')
print(care_prevalence_by_disability.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Additional care percentage': lambda value: f'{value:.2f}%'}))
print('\nMissing-value reasons:')
print(additional_care_missing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(additional_care_quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in additional-care indicator review: 9,767

Proposed additional-care indicator:
  Additional parental care indicator  Participants
No additional care due to disability          9116
   Additional care due to disability           349
                             Missing           302

Construction source:
   Additional-care construction source  Participants
No disability or long-standing illness          8241
           Wave 1 care-burden response          1208
          No valid construction source           302
           Wave 2 care-burden response            16

Disability category by additional-care indicator:
col_0                                                        Additional care due to disability  Additional-care indicator unavailable  No additional care due to disability  Total
W1 disability and schooling                                                                                                                                                       
Disabil

In [141]:
# 69: Additional parental-care measure decision

selected_additional_parental_care_measure = additional_parental_care_indicator.copy()
selected_additional_parental_care_label = additional_parental_care_label.copy()
selected_additional_parental_care_source = consolidated_care_burden_source.copy()
stage_2_additional_care_review['selected_additional_parental_care_measure'] = selected_additional_parental_care_measure
stage_2_additional_care_review['Selected additional parental-care measure'] = selected_additional_parental_care_label
stage_2_additional_care_review['Selected measure source'] = selected_additional_parental_care_source
stage_2_additional_care_review['Predictor decision'] = 'Retain as constructed predictor candidate'
additional_care_decision_rules = [{'Wave': 'Wave 1', 'Source type': 'Young person', 'Variable': 'W1chea4bHS',
    'Review outcome': 'Retain as construction source', 'Decision reason': "Primary source for the constructed indicator of additional parental care required because of the young person's illness or disability. The response adds information about family care burden beyond the selected disability and schooling category.", 'Leakage assessment': 'No direct outcome leakage identified. The information was collected before transition.', 'Reference-period assessment': "Additional parental care associated with the young person's illness or disability at Wave 1.", 'Review notes': 'Provided 1,208 valid responses: 345 Yes and 863 No. Used as the primary source and not retained as a separate raw predictor.'}, {'Wave': 'Wave 2',
    'Source type': 'Young person', 'Variable': 'W2chea4bHS', 'Review outcome': 'Retain as construction source', 'Decision reason': "Fallback source for the constructed indicator of additional parental care required because of the young person's illness or disability.", 'Leakage assessment': 'No direct outcome leakage identified. The information was collected before transition.', 'Reference-period assessment': "Additional parental care associated with the young person's illness or disability at Wave 2.", 'Review notes': 'Provided 16 valid responses where the Wave 1 response was unavailable: 4 Yes and 12 No. It is not retained as a separate raw predictor.'}]
additional_care_decision_indices = []
for rule in additional_care_decision_rules:
    matching_rows = stage_2_variable_decision_register['Wave'].eq(rule['Wave']) & stage_2_variable_decision_register['Source type'].eq(rule['Source type']) & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {rule['Wave']}, {rule['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    additional_care_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = rule['Review outcome']
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'Health and family care burden'
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = rule['Decision reason']
    stage_2_variable_decision_register.loc[matching_rows, 'Leakage assessment'] = rule['Leakage assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Reference-period assessment'] = rule['Reference-period assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave 1 and Wave 2 data dictionaries and Stage 2 care-burden coding and routing review'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = rule['Review notes']
additional_care_decision_checks = pd.DataFrame([{'Quality check': 'Participants in constructed measure',
    'Participants': int(len(selected_additional_parental_care_measure))}, {'Quality check': 'Valid constructed values',
    'Participants': int(selected_additional_parental_care_measure.notna().sum())}, {'Quality check': 'Missing constructed values',
    'Participants': int(selected_additional_parental_care_measure.isna().sum())}, {'Quality check': 'Additional-care Yes',
    'Participants': int(selected_additional_parental_care_measure.eq(1).sum())}, {'Quality check': 'Additional-care No',
    'Participants': int(selected_additional_parental_care_measure.eq(0).sum())}, {'Quality check': 'No-disability cases classified as additional-care Yes',
    'Participants': int((consolidated_disability_for_care_review.eq(3) & selected_additional_parental_care_measure.eq(1)).sum())}, {'Quality check': 'Participants with both Wave 1 and Wave 2 care sources',
    'Participants': int((w1_care_burden.notna() & w2_care_burden.notna()).sum())}])
additional_care_review_output_path = stage_2_output_directory / 'stage_2_additional_parental_care_review.csv'
stage_2_additional_care_review.to_csv(additional_care_review_output_path, index=False)
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
additional_care_decision_output = stage_2_variable_decision_register.loc[additional_care_decision_indices, ['Wave',
    'Source type', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment', 'Review notes']].sort_values(['Wave',
    'Variable']).reset_index(drop=True)
selected_additional_care_distribution = selected_additional_parental_care_label.fillna('Missing').value_counts().rename_axis('Selected additional parental-care measure').reset_index(name='Participants')
selected_additional_care_source_distribution = selected_additional_parental_care_source.fillna('No valid construction source').value_counts().rename_axis('Selected measure source').reset_index(name='Participants')
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Additional parental-care source variables reviewed: {len(additional_care_decision_output):,}')
print('\nSelected constructed-measure distribution:')
print(selected_additional_care_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSelected measure source:')
print(selected_additional_care_source_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSource-variable decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 145, 'display.width', 390):
    print(additional_care_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(additional_care_decision_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'Additional parental-care review saved to: {additional_care_review_output_path}')

Additional parental-care source variables reviewed: 2

Selected constructed-measure distribution:
Selected additional parental-care measure  Participants
     No additional care due to disability          9116
        Additional care due to disability           349
                                  Missing           302

Selected measure source:
               Selected measure source  Participants
No disability or long-standing illness          8241
           Wave 1 care-burden response          1208
          No valid construction source           302
           Wave 2 care-burden response            16

Source-variable decisions:
  Wave  Source type   Variable                                                                   Variable label                Review outcome                                                                                                                                                                                                                          

In [142]:
# 70: Remaining SEN and disability-related variable screen

pending_variable_register = stage_2_variable_decision_register.loc[stage_2_variable_decision_register['Review outcome'].eq('Pending review')].copy()
pending_variable_names = pending_variable_register['Variable'].fillna('').str.lower()
pending_variable_labels = pending_variable_register['Variable label'].fillna('').str.lower()
sen_disability_name_match = pending_variable_names.str.contains('sen|disab|chea', regex=True, na=False)
sen_disability_label_match = pending_variable_labels.str.contains('special educational need|long-standing illness|longstanding illness|disability|disabled|infirmity',
    regex=True, na=False)
remaining_sen_disability_entries = pending_variable_register.loc[sen_disability_name_match | sen_disability_label_match].copy()

def organise_sen_disability_entry(row):
    """Organise remaining entries for manual review."""
    variable = str(row['Variable']).lower()
    label = str(row['Variable label']).lower()
    combined_text = variable + ' ' + label
    if 'special educational need' in combined_text or 'sen' in variable:
        return 'SEN-related item'
    if 'attend school' in combined_text or 'schoolwork' in combined_text or 'schooling' in combined_text:
        return 'Schooling impact of health condition'
    if 'look after' in combined_text or 'care' in combined_text or 'help' in combined_text or ('support' in combined_text):
        return 'Care or support related to health'
    if 'long-standing illness' in combined_text or 'longstanding illness' in combined_text or 'disability' in combined_text or ('infirmity' in combined_text) or ('disab' in variable) or ('chea' in variable):
        return 'Illness or disability item'
    return 'Other related item'
remaining_sen_disability_entries['Review group'] = remaining_sen_disability_entries.apply(organise_sen_disability_entry,
    axis=1)
remaining_sen_disability_summary = remaining_sen_disability_entries.groupby(['Review group', 'Wave', 'Source type'],
    dropna=False).agg(Variables=('Variable', 'size')).reset_index().sort_values(['Review group', 'Wave',
    'Source type']).reset_index(drop=True)
remaining_sen_disability_list = remaining_sen_disability_entries[['Wave', 'Source type', 'Source file', 'Variable',
    'Variable label', 'Timing status', 'Review group', 'Review outcome']].sort_values(['Review group', 'Wave',
    'Source type', 'Variable']).reset_index(drop=True)
print(f'Remaining pending SEN or disability-related entries: {len(remaining_sen_disability_list):,}')
print('\nSummary by review group, wave and source:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 120, 'display.width', 320):
    print(remaining_sen_disability_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nFull remaining review list:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 145, 'display.width', 390):
    print(remaining_sen_disability_list.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Remaining pending SEN or disability-related entries: 84

Summary by review group, wave and source:
              Review group   Wave       Source type  Variables
Illness or disability item Wave 1 Family background          7
Illness or disability item Wave 1      Young person          1
                       ...    ...               ...        ...
          SEN-related item Wave 3      Young person          1
          SEN-related item Wave 4      Young person          3

Full remaining review list:
  Wave       Source type                           Source file     Variable                                                                   Variable label          Timing status               Review group Review outcome
Wave 1 Family background wave_one_lsype_family_background_2020   W1ben1MP0e MP: State benefits currently received by MP (or partner): Disability Living Allo  Pre-transition source Illness or disability item Pending review
Wave 1 Family background wave_one_lsype_family_bac

In [143]:
# 71: Wave 1 parental health and disability review

w1_parental_health_variables = ['W1hea2MP', 'W1hea2SP', 'W1disabMP', 'W1disabSP', 'W1disabmum', 'W1disabdad']
w1_parental_health_entries = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq('Wave 1') & stage_2_master_variable_register['Source type'].eq('Family background') & stage_2_master_variable_register['Variable'].str.lower().isin([variable.lower() for variable in w1_parental_health_variables])].copy().sort_values('Variable position').reset_index(drop=True)
if len(w1_parental_health_entries) != 6:
    raise ValueError(f'Expected six Wave 1 parental health variables, but found {len(w1_parental_health_entries)}.')
source_paths = w1_parental_health_entries['Source path'].drop_duplicates().tolist()
if len(source_paths) != 1:
    raise ValueError(f'Expected one Wave 1 family-background source path, but found {len(source_paths)}.')
w1_family_source_path = Path(source_paths[0])
source_variable_names = w1_parental_health_entries['Variable'].tolist()
w1_parental_health_numeric = pd.read_stata(w1_family_source_path, columns=['NSID', *source_variable_names],
    convert_categoricals=False)
w1_parental_health_labelled = pd.read_stata(w1_family_source_path, columns=['NSID', *source_variable_names],
    convert_categoricals=True)
w1_parental_health_numeric_sample = stage_2_participant_ids.merge(w1_parental_health_numeric, on='NSID', how='left',
    validate='one_to_one', indicator=True)
w1_source_record_present = w1_parental_health_numeric_sample['_merge'].eq('both')
w1_parental_health_numeric_sample = w1_parental_health_numeric_sample.drop(columns='_merge')
w1_parental_health_labelled_sample = stage_2_participant_ids.merge(w1_parental_health_labelled, on='NSID', how='left',
    validate='one_to_one')
stage_2_w1_parental_health_review = stage_2_participant_ids.copy()
for variable in source_variable_names:
    stage_2_w1_parental_health_review[f'{variable} code'] = pd.to_numeric(w1_parental_health_numeric_sample[variable],
        errors='coerce')
    stage_2_w1_parental_health_review[f'{variable} label'] = w1_parental_health_labelled_sample[variable].astype('string')
coverage_records = []
for _, entry in w1_parental_health_entries.iterrows():
    variable = entry['Variable']
    numeric_values = stage_2_w1_parental_health_review[f'{variable} code']
    coverage_records.append({'Variable': variable, 'Variable label': entry['Variable label'],
        'Current register status': stage_2_variable_decision_register.loc[stage_2_variable_decision_register['Source file'].eq(entry['Source file']) & stage_2_variable_decision_register['Variable'].str.lower().eq(variable.lower()),
        'Review outcome'].iloc[0], 'Source record present': int(w1_source_record_present.sum()), 'Non-negative stored value': int(numeric_values.ge(0).sum()), 'Negative stored value': int(numeric_values.lt(0).sum()), 'No source record': int((~w1_source_record_present).sum())})
w1_parental_health_coverage = pd.DataFrame(coverage_records)
value_summary_records = []
for variable in source_variable_names:
    value_summary = pd.DataFrame({'Value code': stage_2_w1_parental_health_review[f'{variable} code'],
        'Value label': stage_2_w1_parental_health_review[f'{variable} label'], 'Source record present': w1_source_record_present}).loc[lambda frame: frame['Source record present']].drop(columns='Source record present').value_counts(dropna=False).rename('Participants').reset_index()
    value_summary.insert(0, 'Variable', variable)
    value_summary_records.append(value_summary)
w1_parental_health_value_summary = pd.concat(value_summary_records, ignore_index=True)
readable_labels = {}
for variable in source_variable_names:
    readable_labels[variable] = stage_2_w1_parental_health_review[f'{variable} label'].fillna('No source record')
w1_mp_health_by_disability = pd.crosstab(readable_labels['W1hea2MP'], readable_labels['W1disabMP'], margins=True,
    margins_name='Total')
w1_sp_health_by_disability = pd.crosstab(readable_labels['W1hea2SP'], readable_labels['W1disabSP'], margins=True,
    margins_name='Total')
w1_mp_disability_by_mother = pd.crosstab(readable_labels['W1disabMP'], readable_labels['W1disabmum'], margins=True,
    margins_name='Total')
w1_mp_disability_by_father = pd.crosstab(readable_labels['W1disabMP'], readable_labels['W1disabdad'], margins=True,
    margins_name='Total')
w1_sp_disability_by_mother = pd.crosstab(readable_labels['W1disabSP'], readable_labels['W1disabmum'], margins=True,
    margins_name='Total')
w1_sp_disability_by_father = pd.crosstab(readable_labels['W1disabSP'], readable_labels['W1disabdad'], margins=True,
    margins_name='Total')
derived_parental_disability_variables = ['W1disabMP', 'W1disabSP', 'W1disabmum', 'W1disabdad']
agreement_records = []
for first_position, first_variable in enumerate(derived_parental_disability_variables):
    for second_variable in derived_parental_disability_variables[first_position + 1:]:
        first_values = stage_2_w1_parental_health_review[f'{first_variable} code']
        second_values = stage_2_w1_parental_health_review[f'{second_variable} code']
        both_non_negative = first_values.ge(0) & second_values.ge(0)
        same_code = both_non_negative & first_values.eq(second_values)
        agreement_records.append({'First variable': first_variable, 'Second variable': second_variable,
            'Both non-negative': int(both_non_negative.sum()), 'Same stored code': int(same_code.sum()), 'Different stored code': int((both_non_negative & first_values.ne(second_values)).sum()), 'Code agreement percentage': same_code.sum() / both_non_negative.sum() * 100 if both_non_negative.sum() else np.nan})
w1_parental_disability_code_agreement = pd.DataFrame(agreement_records)
print(f'Wave 1 parental health and disability variables reviewed: {len(w1_parental_health_entries):,}')
print('\nVariable coverage:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 145, 'display.width', 370):
    print(w1_parental_health_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nStored values and labels:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 110, 'display.width', 300):
    print(w1_parental_health_value_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMain-parent illness status by limiting disability:')
print(w1_mp_health_by_disability.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nSecond-parent illness status by limiting disability:')
print(w1_sp_health_by_disability.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nMain-parent disability by mother disability:')
print(w1_mp_disability_by_mother.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nMain-parent disability by father disability:')
print(w1_mp_disability_by_father.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nSecond-parent disability by mother disability:')
print(w1_sp_disability_by_mother.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nSecond-parent disability by father disability:')
print(w1_sp_disability_by_father.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nCode agreement among derived disability variables:')
print(w1_parental_disability_code_agreement.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Code agreement percentage': lambda value: f'{value:.2f}%' if pd.notna(value) else 'Not available'}))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Wave 1 parental health and disability variables reviewed: 6

Variable coverage:
  Variable                                                  Variable label Current register status  Source record present  Non-negative stored value  Negative stored value  No source record
  W1hea2MP MP: Whether have long-standing illness, disability or infirmity          Pending review                   9524                       9396                    128               243
  W1hea2SP SP: Whether have long-standing illness, disability or infirmity          Pending review                   9524                       6615                   2909               243
       ...                                                             ...                     ...                    ...                        ...                    ...               ...
W1disabmum                       DV: Disability limiting mother's activity          Pending review                   9524                       8975            

In [144]:
# 72: Comparison of parental-disability construction options

w1_disability_mp_code = pd.to_numeric(stage_2_w1_parental_health_review['W1disabMP code'], errors='coerce')
w1_disability_sp_code = pd.to_numeric(stage_2_w1_parental_health_review['W1disabSP code'], errors='coerce')
w1_disability_mother_code = pd.to_numeric(stage_2_w1_parental_health_review['W1disabmum code'], errors='coerce')
w1_disability_father_code = pd.to_numeric(stage_2_w1_parental_health_review['W1disabdad code'], errors='coerce')

def recode_parental_disability(source_values):
    """Recode parental disability by increasing severity."""
    recoded_values = pd.Series(pd.NA, index=source_values.index, dtype='Int64')
    recoded_values.loc[source_values.eq(3)] = 0
    recoded_values.loc[source_values.eq(2)] = 1
    recoded_values.loc[source_values.eq(1)] = 2
    return recoded_values
w1_disability_mp_severity = recode_parental_disability(w1_disability_mp_code)
w1_disability_sp_severity = recode_parental_disability(w1_disability_sp_code)
w1_disability_mother_severity = recode_parental_disability(w1_disability_mother_code)
w1_disability_father_severity = recode_parental_disability(w1_disability_father_code)

def construct_household_parental_disability(first_severity, second_severity, first_absent, second_absent):
    """Construct the highest known severity among resident parents."""
    result = pd.Series(pd.NA, index=first_severity.index, dtype='Int64')
    first_valid = first_severity.notna()
    second_valid = second_severity.notna()
    first_resolved = first_valid | first_absent
    second_resolved = second_valid | second_absent
    all_resident_statuses_resolved = first_resolved & second_resolved
    at_least_one_valid_parent = first_valid | second_valid
    any_limiting_disability = first_severity.eq(2) | second_severity.eq(2)
    result.loc[any_limiting_disability] = 2
    any_non_limiting_disability = first_severity.eq(1) | second_severity.eq(1)
    result.loc[result.isna() & all_resident_statuses_resolved & any_non_limiting_disability] = 1
    result.loc[result.isna() & all_resident_statuses_resolved & at_least_one_valid_parent & ~any_non_limiting_disability] = 0
    return result
mother_absent = w1_disability_mother_code.eq(-98)
father_absent = w1_disability_father_code.eq(-98)
parent_role_disability_measure = construct_household_parental_disability(first_severity=w1_disability_mother_severity,
    second_severity=w1_disability_father_severity, first_absent=mother_absent, second_absent=father_absent)
main_parent_absent = pd.Series(False, index=stage_2_participant_ids.index)
second_parent_absent = w1_disability_sp_code.eq(-98)
respondent_role_disability_measure = construct_household_parental_disability(first_severity=w1_disability_mp_severity,
    second_severity=w1_disability_sp_severity, first_absent=main_parent_absent, second_absent=second_parent_absent)
parental_disability_labels = {0: 'No disability among resident parents',
    1: 'Non-limiting disability among resident parents', 2: 'Activity-limiting disability among resident parents'}
parent_role_disability_label = parent_role_disability_measure.map(parental_disability_labels)
respondent_role_disability_label = respondent_role_disability_measure.map(parental_disability_labels)
stage_2_parental_disability_option_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'mother_disability_severity': w1_disability_mother_severity, 'father_disability_severity': w1_disability_father_severity, 'main_parent_disability_severity': w1_disability_mp_severity, 'second_parent_disability_severity': w1_disability_sp_severity, 'parent_role_parental_disability': parent_role_disability_measure, 'Parent-role parental disability': parent_role_disability_label, 'respondent_role_parental_disability': respondent_role_disability_measure, 'Respondent-role parental disability': respondent_role_disability_label})
parental_disability_option_distribution = pd.concat([parent_role_disability_label.fillna('Missing').value_counts().rename_axis('Parental-disability category').reset_index(name='Participants').assign(Construction_option='Mother and father roles'),
    respondent_role_disability_label.fillna('Missing').value_counts().rename_axis('Parental-disability category').reset_index(name='Participants').assign(Construction_option='Main and second parent roles')], ignore_index=True)
parental_disability_option_distribution = parental_disability_option_distribution[['Construction_option',
    'Parental-disability category', 'Participants']]
parental_disability_coverage_comparison = pd.DataFrame([{'Construction option': 'Mother and father roles',
    'Valid participants': int(parent_role_disability_measure.notna().sum()), 'Missing participants': int(parent_role_disability_measure.isna().sum()), 'Coverage percentage': parent_role_disability_measure.notna().mean() * 100}, {'Construction option': 'Main and second parent roles',
    'Valid participants': int(respondent_role_disability_measure.notna().sum()), 'Missing participants': int(respondent_role_disability_measure.isna().sum()), 'Coverage percentage': respondent_role_disability_measure.notna().mean() * 100}])
both_parental_options_valid = parent_role_disability_measure.notna() & respondent_role_disability_measure.notna()
parental_options_agree = both_parental_options_valid & parent_role_disability_measure.eq(respondent_role_disability_measure)
parental_disability_option_comparison = pd.DataFrame([{'Comparison': 'Parent-role versus respondent-role construction',
    'Both options valid': int(both_parental_options_valid.sum()), 'Same category': int(parental_options_agree.sum()), 'Different category': int((both_parental_options_valid & parent_role_disability_measure.ne(respondent_role_disability_measure)).sum()), 'Agreement percentage': parental_options_agree.sum() / both_parental_options_valid.sum() * 100 if both_parental_options_valid.sum() else np.nan, 'Parent-role valid only': int((parent_role_disability_measure.notna() & respondent_role_disability_measure.isna()).sum()), 'Respondent-role valid only': int((parent_role_disability_measure.isna() & respondent_role_disability_measure.notna()).sum()), 'Both options missing': int((parent_role_disability_measure.isna() & respondent_role_disability_measure.isna()).sum())}])
parental_disability_cross_tabulation = pd.crosstab(parent_role_disability_label.fillna('Parent-role measure missing'),
    respondent_role_disability_label.fillna('Respondent-role measure missing'), margins=True, margins_name='Total')
individual_parent_coverage = pd.DataFrame([{'Parent role': 'Mother',
    'Valid disability status': int(w1_disability_mother_severity.notna().sum()), 'Parent not present': int(mother_absent.sum()), 'Present but status unavailable': int((~mother_absent & w1_disability_mother_severity.isna() & w1_source_record_present).sum()), 'No source record': int((~w1_source_record_present).sum())}, {'Parent role': 'Father',
    'Valid disability status': int(w1_disability_father_severity.notna().sum()), 'Parent not present': int(father_absent.sum()), 'Present but status unavailable': int((~father_absent & w1_disability_father_severity.isna() & w1_source_record_present).sum()), 'No source record': int((~w1_source_record_present).sum())}, {'Parent role': 'Main parent',
    'Valid disability status': int(w1_disability_mp_severity.notna().sum()), 'Parent not present': 0, 'Present but status unavailable': int((w1_disability_mp_severity.isna() & w1_source_record_present).sum()), 'No source record': int((~w1_source_record_present).sum())}, {'Parent role': 'Second parent',
    'Valid disability status': int(w1_disability_sp_severity.notna().sum()), 'Parent not present': int(second_parent_absent.sum()), 'Present but status unavailable': int((~second_parent_absent & w1_disability_sp_severity.isna() & w1_source_record_present).sum()), 'No source record': int((~w1_source_record_present).sum())}])
parental_disability_quality_checks = pd.DataFrame([{'Quality check': 'Parent-role limiting disability without any parent-level limiting value',
    'Participants': int((parent_role_disability_measure.eq(2) & ~(w1_disability_mother_severity.eq(2) | w1_disability_father_severity.eq(2))).sum())}, {'Quality check': 'Respondent-role limiting disability without any respondent-level limiting value',
    'Participants': int((respondent_role_disability_measure.eq(2) & ~(w1_disability_mp_severity.eq(2) | w1_disability_sp_severity.eq(2))).sum())}, {'Quality check': 'Parent-role no-disability category with an observed parental disability',
    'Participants': int((parent_role_disability_measure.eq(0) & (w1_disability_mother_severity.isin([1,
    2]) | w1_disability_father_severity.isin([1,
    2]))).sum())}, {'Quality check': 'Respondent-role no-disability category with an observed parental disability',
    'Participants': int((respondent_role_disability_measure.eq(0) & (w1_disability_mp_severity.isin([1,
    2]) | w1_disability_sp_severity.isin([1, 2]))).sum())}])
print(f'Participants in parental-disability option review: {len(stage_2_parental_disability_option_review):,}')
print('\nConstruction-option distributions:')
print(parental_disability_option_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCoverage comparison:')
print(parental_disability_coverage_comparison.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Coverage percentage': lambda value: f'{value:.2f}%'}))
print('\nConstruction-option comparison:')
print(parental_disability_option_comparison.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Agreement percentage': lambda value: f'{value:.2f}%'}))
print('\nParent-role by respondent-role category:')
print(parental_disability_cross_tabulation.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nIndividual-parent information coverage:')
print(individual_parent_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(parental_disability_quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in parental-disability option review: 9,767

Construction-option distributions:
         Construction_option                        Parental-disability category  Participants
     Mother and father roles                No disability among resident parents          4262
     Mother and father roles                                             Missing          2615
                         ...                                                 ...           ...
Main and second parent roles Activity-limiting disability among resident parents          1908
Main and second parent roles      Non-limiting disability among resident parents          1042

Coverage comparison:
         Construction option  Valid participants  Missing participants Coverage percentage
     Mother and father roles                7152                  2615              73.23%
Main and second parent roles                7245                  2522              74.18%

Construction-option comparison:
         

In [145]:
# 73: Main-parent and household parental-disability comparison

main_parent_disability_measure = w1_disability_mp_severity.copy()
main_parent_disability_label = main_parent_disability_measure.map({0: 'No main-parent disability',
    1: 'Main-parent non-limiting disability', 2: 'Main-parent activity-limiting disability'})
household_parental_disability_measure = respondent_role_disability_measure.copy()
household_parental_disability_label = household_parental_disability_measure.map(parental_disability_labels)
stage_2_parental_disability_measure_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'main_parent_disability': main_parent_disability_measure, 'Main-parent disability': main_parent_disability_label, 'second_parent_disability': w1_disability_sp_severity, 'household_parental_disability': household_parental_disability_measure, 'Household parental disability': household_parental_disability_label, 'second_parent_not_present': second_parent_absent})
parental_disability_measure_distribution = pd.concat([main_parent_disability_label.fillna('Missing').value_counts().rename_axis('Disability category').reset_index(name='Participants').assign(Measure_option='Main parent only'),
    household_parental_disability_label.fillna('Missing').value_counts().rename_axis('Disability category').reset_index(name='Participants').assign(Measure_option='Main and second resident parents')], ignore_index=True)
parental_disability_measure_distribution = parental_disability_measure_distribution[['Measure_option',
    'Disability category', 'Participants']]
parental_disability_measure_coverage = pd.DataFrame([{'Measure option': 'Main parent only',
    'Valid participants': int(main_parent_disability_measure.notna().sum()), 'Missing participants': int(main_parent_disability_measure.isna().sum()), 'Coverage percentage': main_parent_disability_measure.notna().mean() * 100}, {'Measure option': 'Main and second resident parents',
    'Valid participants': int(household_parental_disability_measure.notna().sum()), 'Missing participants': int(household_parental_disability_measure.isna().sum()), 'Coverage percentage': household_parental_disability_measure.notna().mean() * 100}])
both_options_valid = main_parent_disability_measure.notna() & household_parental_disability_measure.notna()
same_severity = both_options_valid & main_parent_disability_measure.eq(household_parental_disability_measure)
household_higher_severity = both_options_valid & household_parental_disability_measure.gt(main_parent_disability_measure)
household_lower_severity = both_options_valid & household_parental_disability_measure.lt(main_parent_disability_measure)
parental_disability_measure_comparison = pd.DataFrame([{'Comparison': 'Main-parent versus household measure',
    'Both measures valid': int(both_options_valid.sum()), 'Same category': int(same_severity.sum()), 'Household category more severe': int(household_higher_severity.sum()), 'Household category less severe': int(household_lower_severity.sum()), 'Main-parent measure valid only': int((main_parent_disability_measure.notna() & household_parental_disability_measure.isna()).sum()), 'Household measure valid only': int((main_parent_disability_measure.isna() & household_parental_disability_measure.notna()).sum()), 'Both measures missing': int((main_parent_disability_measure.isna() & household_parental_disability_measure.isna()).sum())}])
main_parent_by_household_disability = pd.crosstab(main_parent_disability_label.fillna('Main-parent measure missing'),
    household_parental_disability_label.fillna('Household measure missing'), margins=True, margins_name='Total')
second_parent_increment_summary = pd.DataFrame([{'Incremental information': 'Main parent has no disability; second parent has non-limiting disability',
    'Participants': int((main_parent_disability_measure.eq(0) & w1_disability_sp_severity.eq(1)).sum())}, {'Incremental information': 'Main parent has no disability; second parent has limiting disability',
    'Participants': int((main_parent_disability_measure.eq(0) & w1_disability_sp_severity.eq(2)).sum())}, {'Incremental information': 'Main parent has non-limiting disability; second parent has limiting disability',
    'Participants': int((main_parent_disability_measure.eq(1) & w1_disability_sp_severity.eq(2)).sum())}, {'Incremental information': 'Main-parent status valid, but household measure missing',
    'Participants': int((main_parent_disability_measure.notna() & household_parental_disability_measure.isna()).sum())}, {'Incremental information': 'Household measure resolved by an observed second-parent limiting disability while main-parent status is unavailable',
    'Participants': int((main_parent_disability_measure.isna() & household_parental_disability_measure.eq(2) & w1_disability_sp_severity.eq(2)).sum())}])
household_measure_missing_reason = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
household_measure_missing = household_parental_disability_measure.isna()
household_measure_missing_reason.loc[household_measure_missing & ~w1_source_record_present] = 'No Wave 1 family-background source record'
household_measure_missing_reason.loc[household_measure_missing & w1_source_record_present & main_parent_disability_measure.isna()] = 'Main-parent disability status unavailable'
household_measure_missing_reason.loc[household_measure_missing & w1_source_record_present & main_parent_disability_measure.notna() & ~second_parent_absent & w1_disability_sp_severity.isna()] = 'Second parent present, but disability status unavailable'
household_measure_missing_reason.loc[household_measure_missing & household_measure_missing_reason.isna()] = 'Other unresolved resident-parent information'
household_missing_summary = household_measure_missing_reason.loc[household_measure_missing].value_counts().rename_axis('Household-measure missing reason').reset_index(name='Participants')
parental_disability_comparison_checks = pd.DataFrame([{'Quality check': 'Household category lower than main-parent category',
    'Participants': int(household_lower_severity.sum())}, {'Quality check': 'Household limiting category without a limiting parent value',
    'Participants': int((household_parental_disability_measure.eq(2) & ~(main_parent_disability_measure.eq(2) | w1_disability_sp_severity.eq(2))).sum())}, {'Quality check': 'Household no-disability category with an observed parental disability',
    'Participants': int((household_parental_disability_measure.eq(0) & (main_parent_disability_measure.isin([1,
    2]) | w1_disability_sp_severity.isin([1, 2]))).sum())}])
print(f'Participants in parental-disability measure comparison: {len(stage_2_parental_disability_measure_review):,}')
print('\nMeasure distributions:')
print(parental_disability_measure_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCoverage comparison:')
print(parental_disability_measure_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Coverage percentage': lambda value: f'{value:.2f}%'}))
print('\nMeasure comparison:')
print(parental_disability_measure_comparison.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMain-parent category by household category:')
print(main_parent_by_household_disability.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nIncremental information from the second parent:')
with pd.option_context('display.max_colwidth', 140, 'display.width', 270):
    print(second_parent_increment_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nHousehold-measure missing reasons:')
print(household_missing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(parental_disability_comparison_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in parental-disability measure comparison: 9,767

Measure distributions:
                  Measure_option                                 Disability category  Participants
                Main parent only                           No main-parent disability          7357
                Main parent only            Main-parent activity-limiting disability          1306
                             ...                                                 ...           ...
Main and second resident parents Activity-limiting disability among resident parents          1908
Main and second resident parents      Non-limiting disability among resident parents          1042

Coverage comparison:
                  Measure option  Valid participants  Missing participants Coverage percentage
                Main parent only                9389                   378              96.13%
Main and second resident parents                7245                  2522              74.18%

Measure comp

In [146]:
# 74: Main-parent disability measure decision

selected_main_parent_disability_measure = main_parent_disability_measure.copy()
selected_main_parent_disability_label = main_parent_disability_label.copy()
main_parent_disability_missing_reason = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
main_parent_disability_missing = selected_main_parent_disability_measure.isna()
main_parent_disability_missing_reason.loc[main_parent_disability_missing & ~w1_source_record_present] = 'No Wave 1 family-background source record'
main_parent_disability_missing_reason.loc[main_parent_disability_missing & w1_disability_mp_code.eq(-99)] = 'Main parent not interviewed'
main_parent_disability_missing_reason.loc[main_parent_disability_missing & w1_disability_mp_code.eq(-1)] = "Main parent answered Don't know"
main_parent_disability_missing_reason.loc[main_parent_disability_missing & w1_disability_mp_code.eq(-92)] = 'Main parent refused'
main_parent_disability_missing_reason.loc[main_parent_disability_missing & w1_disability_mp_code.eq(-91)] = 'Not applicable'
main_parent_disability_missing_reason.loc[main_parent_disability_missing & main_parent_disability_missing_reason.isna()] = 'Other unavailable main-parent disability value'
stage_2_parental_disability_measure_review['selected_main_parent_disability'] = selected_main_parent_disability_measure
stage_2_parental_disability_measure_review['Selected main-parent disability'] = selected_main_parent_disability_label
stage_2_parental_disability_measure_review['Main-parent disability missing reason'] = main_parent_disability_missing_reason
stage_2_parental_disability_measure_review['Predictor decision'] = 'Retain as constructed predictor candidate'
parental_disability_decision_rules = [{'Variable': 'W1disabMP', 'Review outcome': 'Retain as construction source',
    'Decision reason': 'Source for the selected three-category main-parent disability measure. It distinguishes no disability, non-limiting disability and activity-limiting disability with substantially higher coverage than the household-level alternative.', 'Reference-period assessment': "Main parent's illness or disability and its effect on activity at Wave 1.", 'Review notes': 'Valid for 9,389 participants: 7,357 no disability, 726 non-limiting disability and 1,306 activity-limiting disability.'}, {'Variable': 'W1hea2MP',
    'Review outcome': 'Retain as review support only', 'Decision reason': 'Raw long-standing illness or disability input underlying W1disabMP. It is not retained separately because the selected derived measure also records whether the condition limits activity.', 'Reference-period assessment': "Main parent's long-standing illness, disability or infirmity status at Wave 1.", 'Review notes': 'Retained to verify the derivation and coding of W1disabMP.'}, {'Variable': 'W1disabSP',
    'Review outcome': 'Retain as review support only', 'Decision reason': 'Alternative second-parent disability measure considered for a household-level construction. It is not retained because the household measure reduced coverage from 96.13% to 74.18% and would mix disability information with second-parent presence and response availability.', 'Reference-period assessment': "Second parent's illness or disability and its effect on activity at Wave 1.", 'Review notes': 'Retained for comparison and audit only. Family structure will be reviewed separately.'}, {'Variable': 'W1hea2SP',
    'Review outcome': 'Retain as review support only', 'Decision reason': 'Raw long-standing illness or disability input underlying W1disabSP. The second-parent measure was not selected for the predictor matrix.', 'Reference-period assessment': "Second parent's long-standing illness, disability or infirmity status at Wave 1.", 'Review notes': 'Retained to verify the derivation and coding of W1disabSP.'}, {'Variable': 'W1disabmum',
    'Review outcome': 'Retain as review support only', 'Decision reason': 'Mother-role reformulation of the Wave 1 parental disability information. It produced the same household classification as the respondent-role construction where both were available, but with lower coverage.', 'Reference-period assessment': "Mother's illness or disability and its effect on activity at Wave 1.", 'Review notes': 'Retained for comparison with the main-parent and second-parent variables only.'}, {'Variable': 'W1disabdad',
    'Review outcome': 'Retain as review support only', 'Decision reason': 'Father-role reformulation of the Wave 1 parental disability information. It produced the same household classification as the respondent-role construction where both were available, but with lower coverage.', 'Reference-period assessment': "Father's illness or disability and its effect on activity at Wave 1.", 'Review notes': 'Retained for comparison with the main-parent and second-parent variables only.'}]
parental_disability_decision_indices = []
for rule in parental_disability_decision_rules:
    matching_rows = stage_2_variable_decision_register['Wave'].eq('Wave 1') & stage_2_variable_decision_register['Source type'].eq('Family background') & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for Wave 1, {rule['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    parental_disability_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = rule['Review outcome']
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'Parental health and disability'
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = rule['Decision reason']
    stage_2_variable_decision_register.loc[matching_rows,
        'Leakage assessment'] = 'No direct outcome leakage identified. The information was collected before transition.'
    stage_2_variable_decision_register.loc[matching_rows,
        'Reference-period assessment'] = rule['Reference-period assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave 1 family-background data dictionary and Stage 2 parental-disability construction review'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = rule['Review notes']
parental_disability_review_output_path = stage_2_output_directory / 'stage_2_main_parent_disability_review.csv'
stage_2_parental_disability_measure_review.to_csv(parental_disability_review_output_path, index=False)
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
parental_disability_decision_output = stage_2_variable_decision_register.loc[parental_disability_decision_indices,
    ['Wave', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment',
    'Review notes']].sort_values(['Review outcome', 'Variable']).reset_index(drop=True)
parental_disability_support_output = parental_disability_decision_output.loc[parental_disability_decision_output['Review outcome'].eq('Retain as review support only')]
selected_main_parent_disability_distribution = selected_main_parent_disability_label.fillna('Missing').value_counts().rename_axis('Selected main-parent disability measure').reset_index(name='Participants')
main_parent_disability_missing_summary = main_parent_disability_missing_reason.loc[main_parent_disability_missing].value_counts().rename_axis('Missing reason').reset_index(name='Participants')
main_parent_disability_quality_checks = pd.DataFrame([{'Quality check': 'Participants in selected measure',
    'Participants': int(len(selected_main_parent_disability_measure))}, {'Quality check': 'Valid selected values',
    'Participants': int(selected_main_parent_disability_measure.notna().sum())}, {'Quality check': 'Missing selected values',
    'Participants': int(selected_main_parent_disability_measure.isna().sum())}, {'Quality check': 'Values outside categories 0, 1 and 2',
    'Participants': int((selected_main_parent_disability_measure.notna() & ~selected_main_parent_disability_measure.isin([0,
    1, 2])).sum())}])
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Wave 1 parental health and disability variables decided: {len(parental_disability_decision_output):,}')
print('\nSelected main-parent disability distribution:')
print(selected_main_parent_disability_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMissing-value reasons:')
print(main_parent_disability_missing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSource-variable decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 145, 'display.width', 390):
    print(parental_disability_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables retained for review support only and not entering the predictor matrix:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 145, 'display.width', 390):
    print(parental_disability_support_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(main_parent_disability_quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'Main-parent disability review saved to: {parental_disability_review_output_path}')

Wave 1 parental health and disability variables decided: 6

Selected main-parent disability distribution:
 Selected main-parent disability measure  Participants
               No main-parent disability          7357
Main-parent activity-limiting disability          1306
     Main-parent non-limiting disability           726
                                 Missing           378

Missing-value reasons:
                           Missing reason  Participants
No Wave 1 family-background source record           243
              Main parent not interviewed           103
          Main parent answered Don't know            24
                      Main parent refused             7
                           Not applicable             1

Source-variable decisions:
  Wave  Variable                                                  Variable label                Review outcome                                                                                                                         

In [147]:
# 75: Expected persistence of young-person health problems

health_persistence_specifications = [{'Wave': 'Wave 1', 'Source type': 'Young person', 'Variable': 'W1chea5HS',
    'Output name': 'W1 expected health-problem persistence'}, {'Wave': 'Wave 2', 'Source type': 'Young person',
    'Variable': 'W2chea5HS', 'Output name': 'W2 expected health-problem persistence'}]
stage_2_health_persistence_review = stage_2_participant_ids.copy()
health_persistence_coverage_records = []
health_persistence_value_records = []
for specification in health_persistence_specifications:
    matching_entry = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq(specification['Wave']) & stage_2_master_variable_register['Source type'].eq(specification['Source type']) & stage_2_master_variable_register['Variable'].str.lower().eq(specification['Variable'].lower())]
    if len(matching_entry) != 1:
        raise ValueError(f"Expected one source entry for {specification['Wave']}, {specification['Variable']}; found {len(matching_entry)}.")
    source_path = Path(matching_entry.iloc[0]['Source path'])
    source_variable = matching_entry.iloc[0]['Variable']
    numeric_data = pd.read_stata(source_path, columns=['NSID', source_variable], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', source_variable], convert_categoricals=True)
    numeric_sample = stage_2_participant_ids.merge(numeric_data, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = numeric_sample['_merge'].eq('both')
    numeric_sample = numeric_sample.drop(columns='_merge')
    labelled_sample = stage_2_participant_ids.merge(labelled_data, on='NSID', how='left', validate='one_to_one')
    numeric_values = pd.to_numeric(numeric_sample[source_variable], errors='coerce')
    labelled_values = labelled_sample[source_variable].astype('string')
    valid_yes_no = numeric_values.where(numeric_values.isin([1, 2]))
    output_name = specification['Output name']
    stage_2_health_persistence_review[f'{output_name} code'] = numeric_values
    stage_2_health_persistence_review[f'{output_name} label'] = labelled_values
    stage_2_health_persistence_review[f'{output_name} valid'] = valid_yes_no
    stage_2_health_persistence_review[f'{output_name} source present'] = source_record_present
    health_persistence_coverage_records.append({'Wave': specification['Wave'], 'Variable': source_variable,
        'Variable label': matching_entry.iloc[0]['Variable label'], 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Valid Yes or No': int(valid_yes_no.notna().sum()), 'Yes: expected to continue': int(numeric_values.eq(1).sum()), 'No: not expected to continue': int(numeric_values.eq(2).sum()), 'Not applicable': int(numeric_values.eq(-91).sum()), 'Respondent not interviewed': int(numeric_values.eq(-99).sum()), 'Other negative code': int((numeric_values.lt(0) & ~numeric_values.isin([-91,
        -99])).sum())})
    value_summary = pd.DataFrame({'Value code': numeric_values, 'Value label': labelled_values,
        'Source record present': source_record_present}).loc[lambda frame: frame['Source record present']].drop(columns='Source record present').value_counts(dropna=False).rename('Participants').reset_index()
    value_summary.insert(0, 'Wave', specification['Wave'])
    value_summary.insert(1, 'Variable', source_variable)
    health_persistence_value_records.append(value_summary)
stage_2_health_persistence_coverage = pd.DataFrame(health_persistence_coverage_records)
stage_2_health_persistence_values = pd.concat(health_persistence_value_records, ignore_index=True)
w1_health_persistence = stage_2_health_persistence_review['W1 expected health-problem persistence valid']
w2_health_persistence = stage_2_health_persistence_review['W2 expected health-problem persistence valid']
w1_young_person_disability = valid_disability_measures['W1 disability and schooling']
w2_young_person_disability = valid_disability_measures['W2 disability and schooling']
young_person_disability_labels = {1.0: 'Disability or long-standing illness; schooling affected',
    2.0: 'Disability or long-standing illness; schooling not affected', 3.0: 'No disability or long-standing illness'}
w1_young_person_disability_label = w1_young_person_disability.map(young_person_disability_labels).fillna('Disability category unavailable')
w2_young_person_disability_label = w2_young_person_disability.map(young_person_disability_labels).fillna('Disability category unavailable')
health_persistence_labels = {1.0: 'Expected to continue until at least age 16',
    2.0: 'Not expected to continue until age 16'}
w1_health_persistence_label = w1_health_persistence.map(health_persistence_labels).fillna('No valid persistence response')
w2_health_persistence_label = w2_health_persistence.map(health_persistence_labels).fillna('No valid persistence response')
w1_disability_by_persistence = pd.crosstab(w1_young_person_disability_label, w1_health_persistence_label, margins=True,
    margins_name='Total')
w2_disability_by_persistence = pd.crosstab(w2_young_person_disability_label, w2_health_persistence_label, margins=True,
    margins_name='Total')
both_persistence_responses_valid = w1_health_persistence.notna() & w2_health_persistence.notna()
persistence_responses_agree = both_persistence_responses_valid & w1_health_persistence.eq(w2_health_persistence)
health_persistence_cross_wave_comparison = pd.DataFrame([{'Comparison': 'Wave 1 versus Wave 2',
    'Both responses valid': int(both_persistence_responses_valid.sum()), 'Responses agreeing': int(persistence_responses_agree.sum()), 'Responses differing': int((both_persistence_responses_valid & w1_health_persistence.ne(w2_health_persistence)).sum()), 'Agreement percentage': persistence_responses_agree.sum() / both_persistence_responses_valid.sum() * 100 if both_persistence_responses_valid.sum() else np.nan}])
health_persistence_cross_tabulation = pd.crosstab(w1_health_persistence.map(health_persistence_labels),
    w2_health_persistence.map(health_persistence_labels), margins=True, margins_name='Total')
health_persistence_coverage_comparison = pd.DataFrame([{'Coverage item': 'Valid Wave 1 persistence response',
    'Participants': int(w1_health_persistence.notna().sum())}, {'Coverage item': 'Valid Wave 2 persistence response',
    'Participants': int(w2_health_persistence.notna().sum())}, {'Coverage item': 'Wave 2 valid where Wave 1 is unavailable',
    'Participants': int((w2_health_persistence.notna() & w1_health_persistence.isna()).sum())}, {'Coverage item': 'Wave 1 valid where Wave 2 is unavailable',
    'Participants': int((w1_health_persistence.notna() & w2_health_persistence.isna()).sum())}, {'Coverage item': 'At least one valid persistence response',
    'Participants': int((w1_health_persistence.notna() | w2_health_persistence.notna()).sum())}])
health_persistence_routing_checks = pd.DataFrame([{'Routing check': 'Wave 1 valid persistence response with no Wave 1 disability',
    'Participants': int((w1_health_persistence.notna() & w1_young_person_disability.eq(3)).sum())}, {'Routing check': 'Wave 2 valid persistence response with no Wave 2 disability',
    'Participants': int((w2_health_persistence.notna() & w2_young_person_disability.eq(3)).sum())}, {'Routing check': 'Wave 1 disability reported but persistence response unavailable',
    'Participants': int((w1_young_person_disability.isin([1,
    2]) & w1_health_persistence.isna()).sum())}, {'Routing check': 'Wave 2 disability reported but persistence response unavailable',
    'Participants': int((w2_young_person_disability.isin([1, 2]) & w2_health_persistence.isna()).sum())}])
print(f'Health-problem persistence variables reviewed: {len(stage_2_health_persistence_coverage):,}')
print('\nCoverage and code summary:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 155, 'display.width', 390):
    print(stage_2_health_persistence_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nStored values and labels:')
print(stage_2_health_persistence_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 1 disability category by persistence expectation:')
print(w1_disability_by_persistence.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nWave 2 disability category by persistence expectation:')
print(w2_disability_by_persistence.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nCross-wave comparison:')
print(health_persistence_cross_wave_comparison.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Agreement percentage': lambda value: f'{value:.2f}%' if pd.notna(value) else 'Not available'}))
print('\nWave 1 by Wave 2 persistence response:')
print(health_persistence_cross_tabulation.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nCoverage comparison:')
print(health_persistence_coverage_comparison.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nRouting checks:')
print(health_persistence_routing_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Health-problem persistence variables reviewed: 2

Coverage and code summary:
  Wave  Variable                                                                Variable label  Source record present  Source record absent  Valid Yes or No  Yes: expected to continue  No: not expected to continue  Not applicable  Respondent not interviewed  Other negative code
Wave 1 W1chea5HS HR: Whether MP expects these problems to continue until YP is at least age 16                   9524                   243             1141                        996                           145            8084                         176                  123
Wave 2 W2chea5HS HR: Whether MP expects these problems to continue until YP is at least age 16                   9521                   246               16                         13                             3            9283                         143                   79

Stored values and labels:
  Wave  Variable  Value code                     Value labe

In [148]:
# 76: Consolidated health-problem persistence options

consolidated_young_person_disability = w1_young_person_disability.copy()
wave2_young_person_disability_fallback = consolidated_young_person_disability.isna() & w2_young_person_disability.notna()
consolidated_young_person_disability.loc[wave2_young_person_disability_fallback] = w2_young_person_disability.loc[wave2_young_person_disability_fallback]
consolidated_health_persistence_response = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
health_persistence_source = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
wave1_persistence_source = w1_health_persistence.notna()
consolidated_health_persistence_response.loc[wave1_persistence_source] = w1_health_persistence.loc[wave1_persistence_source].astype('Int64')
health_persistence_source.loc[wave1_persistence_source] = 'Wave 1 persistence expectation'
wave2_persistence_source = consolidated_health_persistence_response.isna() & w2_health_persistence.notna()
consolidated_health_persistence_response.loc[wave2_persistence_source] = w2_health_persistence.loc[wave2_persistence_source].astype('Int64')
health_persistence_source.loc[wave2_persistence_source] = 'Wave 2 persistence expectation'
binary_persistent_health_problem = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
binary_persistent_health_problem.loc[consolidated_health_persistence_response.eq(1)] = 1
binary_persistent_health_problem.loc[consolidated_health_persistence_response.eq(2)] = 0
binary_persistent_health_problem.loc[consolidated_young_person_disability.eq(3)] = 0
binary_persistent_health_problem_label = binary_persistent_health_problem.map({0: 'No persistent health problem expected',
    1: 'Health problem expected to persist until age 16'})
three_category_health_persistence = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
three_category_health_persistence.loc[consolidated_young_person_disability.eq(3)] = 0
three_category_health_persistence.loc[consolidated_young_person_disability.isin([1,
    2]) & consolidated_health_persistence_response.eq(2)] = 1
three_category_health_persistence.loc[consolidated_young_person_disability.isin([1,
    2]) & consolidated_health_persistence_response.eq(1)] = 2
three_category_health_persistence_label = three_category_health_persistence.map({0: 'No disability or long-standing illness',
    1: 'Health problem not expected to persist until age 16', 2: 'Health problem expected to persist until age 16'})
health_persistence_missing_reason = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
health_persistence_measure_missing = three_category_health_persistence.isna()
health_persistence_missing_reason.loc[health_persistence_measure_missing & consolidated_young_person_disability.isna()] = 'Consolidated disability status unavailable'
health_persistence_missing_reason.loc[health_persistence_measure_missing & consolidated_young_person_disability.isin([1,
    2]) & consolidated_health_persistence_response.isna()] = 'Disability reported, but persistence expectation unavailable'
health_persistence_missing_reason.loc[health_persistence_measure_missing & health_persistence_missing_reason.isna()] = 'Other unresolved persistence status'
health_persistence_source.loc[consolidated_young_person_disability.eq(3) & consolidated_health_persistence_response.isna()] = 'No disability or long-standing illness'
stage_2_health_persistence_option_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'consolidated_young_person_disability': consolidated_young_person_disability, 'consolidated_health_persistence_response': consolidated_health_persistence_response, 'Health-persistence source': health_persistence_source, 'binary_persistent_health_problem': binary_persistent_health_problem, 'Binary persistent-health label': binary_persistent_health_problem_label, 'three_category_health_persistence': three_category_health_persistence, 'Three-category health-persistence label': three_category_health_persistence_label, 'Health-persistence missing reason': health_persistence_missing_reason})
binary_health_persistence_distribution = binary_persistent_health_problem_label.fillna('Missing').value_counts().rename_axis('Binary health-persistence option').reset_index(name='Participants')
three_category_health_persistence_distribution = three_category_health_persistence_label.fillna('Missing').value_counts().rename_axis('Three-category health-persistence option').reset_index(name='Participants')
health_persistence_option_coverage = pd.DataFrame([{'Measure option': 'Binary persistent-health indicator',
    'Valid participants': int(binary_persistent_health_problem.notna().sum()), 'Missing participants': int(binary_persistent_health_problem.isna().sum()), 'Coverage percentage': binary_persistent_health_problem.notna().mean() * 100}, {'Measure option': 'Three-category persistence status',
    'Valid participants': int(three_category_health_persistence.notna().sum()), 'Missing participants': int(three_category_health_persistence.isna().sum()), 'Coverage percentage': three_category_health_persistence.notna().mean() * 100}])
consolidated_young_person_disability_label = consolidated_young_person_disability.map(young_person_disability_labels).fillna('Disability category unavailable')
disability_by_health_persistence = pd.crosstab(consolidated_young_person_disability_label,
    three_category_health_persistence_label.fillna('Health-persistence status unavailable'), margins=True, margins_name='Total')
persistence_prevalence_by_disability = pd.DataFrame({'Disability category': consolidated_young_person_disability_label,
    'Persistence response': consolidated_health_persistence_response}).loc[lambda frame: frame['Disability category'].isin(['Disability or long-standing illness; schooling affected',
    'Disability or long-standing illness; schooling not affected'])].groupby('Disability category',
    dropna=False).agg(Disability_cases=('Persistence response', 'size'),
    Valid_persistence_response=('Persistence response', 'count'), Expected_to_persist=('Persistence response',
    lambda values: int(values.eq(1).sum())), Not_expected_to_persist=('Persistence response',
    lambda values: int(values.eq(2).sum()))).reset_index()
persistence_prevalence_by_disability['Expected to persist percentage'] = persistence_prevalence_by_disability['Expected_to_persist'] / persistence_prevalence_by_disability['Valid_persistence_response'] * 100
health_persistence_source_distribution = health_persistence_source.fillna('No valid construction source').value_counts().rename_axis('Health-persistence construction source').reset_index(name='Participants')
health_persistence_missing_summary = health_persistence_missing_reason.loc[health_persistence_measure_missing].value_counts().rename_axis('Missing reason').reset_index(name='Participants')
health_persistence_quality_checks = pd.DataFrame([{'Quality check': 'Participants with a valid persistence response',
    'Participants': int(consolidated_health_persistence_response.notna().sum())}, {'Quality check': 'Wave 1 persistence sources',
    'Participants': int(wave1_persistence_source.sum())}, {'Quality check': 'Wave 2 fallback persistence sources',
    'Participants': int(wave2_persistence_source.sum())}, {'Quality check': 'Participants with valid responses at both waves',
    'Participants': int((w1_health_persistence.notna() & w2_health_persistence.notna()).sum())}, {'Quality check': 'No-disability cases classified as persistent',
    'Participants': int((consolidated_young_person_disability.eq(3) & binary_persistent_health_problem.eq(1)).sum())}, {'Quality check': 'Disability cases with persistence unavailable',
    'Participants': int((consolidated_young_person_disability.isin([1,
    2]) & consolidated_health_persistence_response.isna()).sum())}, {'Quality check': 'Three-category values outside 0, 1 and 2',
    'Participants': int((three_category_health_persistence.notna() & ~three_category_health_persistence.isin([0, 1,
    2])).sum())}])
print(f'Participants in health-persistence option review: {len(stage_2_health_persistence_option_review):,}')
print('\nBinary option distribution:')
print(binary_health_persistence_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nThree-category option distribution:')
print(three_category_health_persistence_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCoverage comparison:')
print(health_persistence_option_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Coverage percentage': lambda value: f'{value:.2f}%'}))
print('\nDisability category by persistence status:')
print(disability_by_health_persistence.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nPersistence expectation within disability categories:')
print(persistence_prevalence_by_disability.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Expected to persist percentage': lambda value: f'{value:.2f}%'}))
print('\nConstruction source:')
print(health_persistence_source_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMissing-value reasons:')
print(health_persistence_missing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(health_persistence_quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in health-persistence option review: 9,767

Binary option distribution:
               Binary health-persistence option  Participants
          No persistent health problem expected          8389
Health problem expected to persist until age 16          1009
                                        Missing           369

Three-category option distribution:
           Three-category health-persistence option  Participants
             No disability or long-standing illness          8241
    Health problem expected to persist until age 16          1009
                                            Missing           369
Health problem not expected to persist until age 16           148

Coverage comparison:
                    Measure option  Valid participants  Missing participants Coverage percentage
Binary persistent-health indicator                9398                   369              96.22%
 Three-category persistence status                9398                   369        

In [149]:
# 77: Health-problem persistence decision

stage_2_health_persistence_option_review['Predictor decision'] = 'Retain for review support only; do not include in the predictor matrix'
stage_2_health_persistence_option_review['Decision reason'] = 'The proposed measure largely repeats the selected young-person disability measure. The no-disability category is inherited directly from that measure, while most disability cases were expected to persist.'
health_persistence_decision_rules = [{'Wave': 'Wave 1', 'Variable': 'W1chea5HS',
    'Review outcome': 'Retain as review support only', 'Decision reason': "Conditional parental expectation about whether the young person's health problem would continue until at least age 16. It is not retained as a separate predictor because the proposed construction largely repeats the selected disability measure and would add another closely related health indicator.", 'Reference-period assessment': "Wave 1 parental expectation about persistence of the young person's existing health problem until at least age 16.", 'Review notes': 'Provided 1,141 valid responses: 996 expected to continue and 145 not expected to continue. Retained for audit and construct review only.'}, {'Wave': 'Wave 2',
    'Variable': 'W2chea5HS', 'Review outcome': 'Retain as review support only', 'Decision reason': 'Fallback conditional parental expectation for participants without a valid Wave 1 response. It is not retained as a separate predictor because the proposed construction overlaps with the selected disability measure.', 'Reference-period assessment': "Wave 2 parental expectation about persistence of the young person's existing health problem until at least age 16.", 'Review notes': 'Provided 16 non-overlapping valid responses: 13 expected to continue and 3 not expected to continue. Retained for audit only.'}]
health_persistence_decision_indices = []
for rule in health_persistence_decision_rules:
    matching_rows = stage_2_variable_decision_register['Wave'].eq(rule['Wave']) & stage_2_variable_decision_register['Source type'].eq('Young person') & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {rule['Wave']}, {rule['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    health_persistence_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = rule['Review outcome']
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'Health and disability'
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = rule['Decision reason']
    stage_2_variable_decision_register.loc[matching_rows,
        'Leakage assessment'] = 'No direct outcome leakage identified. The information was collected before transition.'
    stage_2_variable_decision_register.loc[matching_rows,
        'Reference-period assessment'] = rule['Reference-period assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave 1 and Wave 2 data dictionaries and Stage 2 health-problem persistence review'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = rule['Review notes']
health_persistence_review_output_path = stage_2_output_directory / 'stage_2_health_persistence_review.csv'
stage_2_health_persistence_option_review.to_csv(health_persistence_review_output_path, index=False)
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
health_persistence_decision_output = stage_2_variable_decision_register.loc[health_persistence_decision_indices,
    ['Wave', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment',
    'Review notes']].sort_values(['Wave', 'Variable']).reset_index(drop=True)
health_persistence_decision_summary = pd.DataFrame([{'Review item': 'Valid consolidated persistence status',
    'Participants': int(three_category_health_persistence.notna().sum())}, {'Review item': 'Health problem expected to persist',
    'Participants': int(three_category_health_persistence.eq(2).sum())}, {'Review item': 'Health problem not expected to persist',
    'Participants': int(three_category_health_persistence.eq(1).sum())}, {'Review item': 'No disability or long-standing illness',
    'Participants': int(three_category_health_persistence.eq(0).sum())}, {'Review item': 'Persistence status unavailable',
    'Participants': int(three_category_health_persistence.isna().sum())}])
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Health-problem persistence variables decided: {len(health_persistence_decision_output):,}')
print('\nPersistence-review summary:')
print(health_persistence_decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSource-variable decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 145, 'display.width', 390):
    print(health_persistence_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables retained for review support only and not entering the predictor matrix: 2')
print('Variables explicitly excluded in this step: 0')
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'Health-persistence review saved to: {health_persistence_review_output_path}')

Health-problem persistence variables decided: 2

Persistence-review summary:
                           Review item  Participants
 Valid consolidated persistence status          9398
    Health problem expected to persist          1009
Health problem not expected to persist           148
No disability or long-standing illness          8241
        Persistence status unavailable           369

Source-variable decisions:
  Wave  Variable                                                                Variable label                Review outcome                                                                                                                                                                                                                                                                                             Decision reason                                                                     Leakage assessment                                                                  

In [150]:
# 78: Pre-transition disability-related benefit structure

disability_benefit_variable_names = ['W1ben1MP0e', 'W1bendisMP', 'W2Ben1AmtMP0g', 'W2Ben1AmtSP0g', 'W2Ben1PdMPag',
    'W2Ben1PdSP0g', 'W2Ben1amtDKMP0g', 'W2Ben1amtDKSP0g', 'W2Ben2QMP0a', 'W2Ben2QSP0a', 'W2BenDLAWeTot', 'W2BenDLAWeWho', 'W2BenDisBand', 'W2BenDisMoAm', 'W2BenDisWeAm', 'W2BenDisYrAm', 'W2BenSickDis']
disability_benefit_entries = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].isin(['Wave 1',
    'Wave 2']) & stage_2_master_variable_register['Source type'].eq('Family background') & stage_2_master_variable_register['Variable'].str.lower().isin([variable.lower() for variable in disability_benefit_variable_names]), ['Source order',
    'Wave', 'Source type', 'Source file', 'Source path', 'Variable position', 'Variable', 'Variable label', 'Data type', 'Timing status']].copy()
disability_benefit_entries = disability_benefit_entries.merge(stage_2_variable_decision_register[['Source file',
    'Variable', 'Review outcome']], on=['Source file',
    'Variable'], how='left', validate='one_to_one').sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
if len(disability_benefit_entries) != 17:
    raise ValueError(f'Expected 17 pre-transition disability-related benefit entries, but found {len(disability_benefit_entries)}.')
if disability_benefit_entries['Source path'].isna().any():
    raise ValueError('At least one disability-benefit entry has no source path.')
if disability_benefit_entries['Review outcome'].isna().any():
    raise ValueError('At least one disability-benefit entry did not match the decision register.')
stage_2_disability_benefit_review = stage_2_participant_ids.copy()
variable_summary_records = []
negative_code_records = []
categorical_value_records = []
for source_path_text, source_entries in disability_benefit_entries.groupby('Source path', sort=False):
    source_path = Path(source_path_text)
    source_variables = source_entries['Variable'].tolist()
    numeric_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    numeric_sample = stage_2_participant_ids.merge(numeric_source, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = numeric_sample['_merge'].eq('both')
    numeric_sample = numeric_sample.drop(columns='_merge')
    labelled_sample = stage_2_participant_ids.merge(labelled_source, on='NSID', how='left', validate='one_to_one')
    for _, entry in source_entries.iterrows():
        variable = entry['Variable']
        numeric_values = pd.to_numeric(numeric_sample[variable], errors='coerce')
        labelled_values = labelled_sample[variable].astype('string')
        stage_2_disability_benefit_review[f'{variable} code'] = numeric_values
        stage_2_disability_benefit_review[f'{variable} label'] = labelled_values
        stage_2_disability_benefit_review[f'{variable} source present'] = source_record_present
        non_negative_values = numeric_values.where(numeric_values.ge(0))
        positive_values = numeric_values.where(numeric_values.gt(0))
        unique_non_negative = int(non_negative_values.dropna().nunique())
        variable_summary_records.append({'Wave': entry['Wave'], 'Variable': variable,
            'Variable label': entry['Variable label'], 'Data type': entry['Data type'], 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Non-negative values': int(non_negative_values.notna().sum()), 'Positive values': int(positive_values.notna().sum()), 'Zero values': int(numeric_values.eq(0).sum()), 'Negative values': int(numeric_values.lt(0).sum()), 'Unique non-negative values': unique_non_negative, 'Minimum non-negative value': non_negative_values.min(), 'Maximum non-negative value': non_negative_values.max(), 'Current register status': entry['Review outcome']})
        negative_summary = pd.DataFrame({'Value code': numeric_values, 'Value label': labelled_values,
            'Source record present': source_record_present}).loc[lambda frame: frame['Source record present'] & frame['Value code'].lt(0)].drop(columns='Source record present').value_counts(dropna=False).rename('Participants').reset_index()
        if not negative_summary.empty:
            negative_summary.insert(0, 'Wave', entry['Wave'])
            negative_summary.insert(1, 'Variable', variable)
            negative_code_records.append(negative_summary)
        if unique_non_negative <= 20:
            categorical_summary = pd.DataFrame({'Value code': numeric_values, 'Value label': labelled_values,
                'Source record present': source_record_present}).loc[lambda frame: frame['Source record present'] & frame['Value code'].ge(0)].drop(columns='Source record present').value_counts(dropna=False).rename('Participants').reset_index()
            if not categorical_summary.empty:
                categorical_summary.insert(0, 'Wave', entry['Wave'])
                categorical_summary.insert(1, 'Variable', variable)
                categorical_value_records.append(categorical_summary)
disability_benefit_variable_summary = pd.DataFrame(variable_summary_records)
if negative_code_records:
    disability_benefit_negative_codes = pd.concat(negative_code_records, ignore_index=True)
else:
    disability_benefit_negative_codes = pd.DataFrame(columns=['Wave', 'Variable', 'Value code', 'Value label',
        'Participants'])
if categorical_value_records:
    disability_benefit_categorical_values = pd.concat(categorical_value_records, ignore_index=True)
else:
    disability_benefit_categorical_values = pd.DataFrame(columns=['Wave', 'Variable', 'Value code', 'Value label',
        'Participants'])

def organise_disability_benefit_variable(row):
    """Organise disability-benefit variables for review."""
    variable = str(row['Variable']).lower()
    label = str(row['Variable label']).lower()
    if 'provided figures' in label or 'whether dk' in label or 'who' in variable:
        return 'Derivation or response support'
    if 'band' in variable or 'banded' in label:
        return 'Derived amount band'
    if 'how long' in label or 'last payment covered' in label or 'pd' in variable:
        return 'Payment-period information'
    if 'amount' in label or 'amt' in variable or 'weam' in variable or ('moam' in variable) or ('yram' in variable) or ('wetot' in variable):
        return 'Benefit amount'
    if 'whether' in label and ('receiv' in label or 'benefit' in label):
        return 'Receipt indicator'
    return 'Other benefit-related item'
disability_benefit_variable_summary['Review group'] = disability_benefit_variable_summary.apply(organise_disability_benefit_variable,
    axis=1)
disability_benefit_group_summary = disability_benefit_variable_summary.groupby(['Review group', 'Wave'],
    dropna=False).agg(Variables=('Variable', 'size'), Variables_with_non_negative_values=('Non-negative values',
    lambda values: int(values.gt(0).sum()))).reset_index().sort_values(['Review group',
    'Wave']).reset_index(drop=True)
print(f'Pre-transition disability-related benefit variables reviewed: {len(disability_benefit_variable_summary):,}')
print('\nSummary by review group and wave:')
print(disability_benefit_group_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariable structure and coverage:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 430):
    print(disability_benefit_variable_summary[['Wave', 'Variable', 'Variable label', 'Review group',
        'Source record present', 'Source record absent', 'Non-negative values', 'Positive values', 'Zero values', 'Negative values', 'Unique non-negative values', 'Minimum non-negative value', 'Maximum non-negative value', 'Current register status']].to_string(max_rows=TABLE_ROW_LIMIT,
        index=False))
print('\nNon-negative categorical value frequencies:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 120, 'display.width', 330):
    print(disability_benefit_categorical_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nNegative-code frequencies:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 120, 'display.width', 330):
    print(disability_benefit_negative_codes.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Pre-transition disability-related benefit variables reviewed: 17

Summary by review group and wave:
                  Review group   Wave  Variables  Variables_with_non_negative_values
                Benefit amount Wave 2          6                                   6
Derivation or response support Wave 2          3                                   3
                           ...    ...        ...                                 ...
             Receipt indicator Wave 1          1                                   1
             Receipt indicator Wave 2          3                                   3

Variable structure and coverage:
  Wave     Variable                                                                   Variable label               Review group  Source record present  Source record absent  Non-negative values  Positive values  Zero values  Negative values  Unique non-negative values  Minimum non-negative value  Maximum non-negative value Current register status
Wave 1 

In [151]:
# 79: Disability-related benefit receipt options

w1_disability_benefit_receipt = stage_2_disability_benefit_review['W1bendisMP code'].where(stage_2_disability_benefit_review['W1bendisMP code'].isin([0,
    1])).astype('Int64')
w2_disability_benefit_receipt = stage_2_disability_benefit_review['W2BenSickDis code'].where(stage_2_disability_benefit_review['W2BenSickDis code'].isin([0,
    1])).astype('Int64')
w1_dla_receipt = stage_2_disability_benefit_review['W1ben1MP0e code'].where(stage_2_disability_benefit_review['W1ben1MP0e code'].isin([0,
    1])).astype('Int64')
w2_dla_mp_receipt = stage_2_disability_benefit_review['W2Ben2QMP0a code'].where(stage_2_disability_benefit_review['W2Ben2QMP0a code'].isin([0,
    1])).astype('Int64')
w2_dla_sp_receipt = stage_2_disability_benefit_review['W2Ben2QSP0a code'].where(stage_2_disability_benefit_review['W2Ben2QSP0a code'].isin([0,
    1])).astype('Int64')
benefit_receipt_labels = {0: 'No disability-related benefit receipt', 1: 'Disability-related benefit receipt'}
both_broad_receipt_valid = w1_disability_benefit_receipt.notna() & w2_disability_benefit_receipt.notna()
broad_receipt_agreement = both_broad_receipt_valid & w1_disability_benefit_receipt.eq(w2_disability_benefit_receipt)
broad_receipt_transition_table = pd.crosstab(w1_disability_benefit_receipt.map(benefit_receipt_labels),
    w2_disability_benefit_receipt.map(benefit_receipt_labels), margins=True, margins_name='Total')
broad_receipt_comparison = pd.DataFrame([{'Comparison': 'Wave 1 versus Wave 2 broad receipt indicator',
    'Both responses valid': int(both_broad_receipt_valid.sum()), 'Responses agreeing': int(broad_receipt_agreement.sum()), 'Responses differing': int((both_broad_receipt_valid & w1_disability_benefit_receipt.ne(w2_disability_benefit_receipt)).sum()), 'Agreement percentage': broad_receipt_agreement.sum() / both_broad_receipt_valid.sum() * 100 if both_broad_receipt_valid.sum() else np.nan, 'Wave 1 valid only': int((w1_disability_benefit_receipt.notna() & w2_disability_benefit_receipt.isna()).sum()), 'Wave 2 valid only': int((w1_disability_benefit_receipt.isna() & w2_disability_benefit_receipt.notna()).sum()), 'Both unavailable': int((w1_disability_benefit_receipt.isna() & w2_disability_benefit_receipt.isna()).sum())}])
benefit_nesting_checks = pd.DataFrame([{'Nesting check': 'Wave 1 DLA receipt but broad receipt = No',
    'Participants': int((w1_dla_receipt.eq(1) & w1_disability_benefit_receipt.eq(0)).sum())}, {'Nesting check': 'Wave 2 MP DLA receipt but broad receipt = No',
    'Participants': int((w2_dla_mp_receipt.eq(1) & w2_disability_benefit_receipt.eq(0)).sum())}, {'Nesting check': 'Wave 2 SP DLA receipt but broad receipt = No',
    'Participants': int((w2_dla_sp_receipt.eq(1) & w2_disability_benefit_receipt.eq(0)).sum())}])
both_w2_dla_reports_valid = w2_dla_mp_receipt.notna() & w2_dla_sp_receipt.notna()
w2_dla_report_agreement = both_w2_dla_reports_valid & w2_dla_mp_receipt.eq(w2_dla_sp_receipt)
w2_dla_report_comparison = pd.DataFrame([{'Comparison': 'Wave 2 MP and SP DLA reports',
    'Both reports valid': int(both_w2_dla_reports_valid.sum()), 'Reports agreeing': int(w2_dla_report_agreement.sum()), 'Reports differing': int((both_w2_dla_reports_valid & w2_dla_mp_receipt.ne(w2_dla_sp_receipt)).sum()), 'Agreement percentage': w2_dla_report_agreement.sum() / both_w2_dla_reports_valid.sum() * 100 if both_w2_dla_reports_valid.sum() else np.nan}])
latest_available_benefit_receipt = w2_disability_benefit_receipt.copy()
latest_available_benefit_source = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
latest_available_benefit_source.loc[w2_disability_benefit_receipt.notna()] = 'Wave 2 broad receipt indicator'
wave1_benefit_fallback = latest_available_benefit_receipt.isna() & w1_disability_benefit_receipt.notna()
latest_available_benefit_receipt.loc[wave1_benefit_fallback] = w1_disability_benefit_receipt.loc[wave1_benefit_fallback]
latest_available_benefit_source.loc[wave1_benefit_fallback] = 'Wave 1 broad receipt fallback'
receipt_at_either_wave = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
receipt_at_either_wave.loc[w1_disability_benefit_receipt.eq(1) | w2_disability_benefit_receipt.eq(1)] = 1
receipt_at_either_wave.loc[w1_disability_benefit_receipt.eq(0) & w2_disability_benefit_receipt.eq(0)] = 0
receipt_at_either_wave_source = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
receipt_at_either_wave_source.loc[w1_disability_benefit_receipt.eq(1) & w2_disability_benefit_receipt.eq(1)] = 'Receipt reported at both waves'
receipt_at_either_wave_source.loc[w1_disability_benefit_receipt.eq(1) & ~w2_disability_benefit_receipt.eq(1)] = 'Receipt reported at Wave 1'
receipt_at_either_wave_source.loc[w2_disability_benefit_receipt.eq(1) & ~w1_disability_benefit_receipt.eq(1)] = 'Receipt reported at Wave 2'
receipt_at_either_wave_source.loc[w1_disability_benefit_receipt.eq(0) & w2_disability_benefit_receipt.eq(0)] = 'No receipt reported at either wave'
latest_available_benefit_label = latest_available_benefit_receipt.map(benefit_receipt_labels)
receipt_at_either_wave_label = receipt_at_either_wave.map(benefit_receipt_labels)
benefit_receipt_option_coverage = pd.DataFrame([{'Measure option': 'Latest available receipt status',
    'Valid participants': int(latest_available_benefit_receipt.notna().sum()), 'Missing participants': int(latest_available_benefit_receipt.isna().sum()), 'Receipt Yes': int(latest_available_benefit_receipt.eq(1).sum()), 'Receipt No': int(latest_available_benefit_receipt.eq(0).sum()), 'Coverage percentage': latest_available_benefit_receipt.notna().mean() * 100}, {'Measure option': 'Receipt at either Wave 1 or Wave 2',
    'Valid participants': int(receipt_at_either_wave.notna().sum()), 'Missing participants': int(receipt_at_either_wave.isna().sum()), 'Receipt Yes': int(receipt_at_either_wave.eq(1).sum()), 'Receipt No': int(receipt_at_either_wave.eq(0).sum()), 'Coverage percentage': receipt_at_either_wave.notna().mean() * 100}])
both_options_valid = latest_available_benefit_receipt.notna() & receipt_at_either_wave.notna()
benefit_option_comparison = pd.DataFrame([{'Comparison': 'Latest available versus receipt at either wave',
    'Both options valid': int(both_options_valid.sum()), 'Same classification': int((both_options_valid & latest_available_benefit_receipt.eq(receipt_at_either_wave)).sum()), 'Different classification': int((both_options_valid & latest_available_benefit_receipt.ne(receipt_at_either_wave)).sum()), 'Latest available valid only': int((latest_available_benefit_receipt.notna() & receipt_at_either_wave.isna()).sum()), 'Either-wave option valid only': int((latest_available_benefit_receipt.isna() & receipt_at_either_wave.notna()).sum())}])
stage_2_disability_benefit_receipt_option_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'W1 broad disability-benefit receipt': w1_disability_benefit_receipt, 'W2 broad disability-benefit receipt': w2_disability_benefit_receipt, 'latest_available_benefit_receipt': latest_available_benefit_receipt, 'Latest available benefit receipt': latest_available_benefit_label, 'Latest available source': latest_available_benefit_source, 'receipt_at_either_pretransition_wave': receipt_at_either_wave, 'Receipt at either pre-transition wave': receipt_at_either_wave_label, 'Either-wave source': receipt_at_either_wave_source})
print(f'Participants in disability-benefit receipt-option review: {len(stage_2_disability_benefit_receipt_option_review):,}')
print('\nWave 1 by Wave 2 broad receipt status:')
print(broad_receipt_transition_table.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nCross-wave broad-indicator comparison:')
print(broad_receipt_comparison.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Agreement percentage': lambda value: f'{value:.2f}%' if pd.notna(value) else 'Not available'}))
print('\nDLA nesting checks:')
print(benefit_nesting_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 2 MP and SP DLA-report comparison:')
print(w2_dla_report_comparison.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Agreement percentage': lambda value: f'{value:.2f}%' if pd.notna(value) else 'Not available'}))
print('\nReceipt-option coverage:')
print(benefit_receipt_option_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Coverage percentage': lambda value: f'{value:.2f}%'}))
print('\nReceipt-option comparison:')
print(benefit_option_comparison.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nLatest-available source:')
print(latest_available_benefit_source.fillna('No valid source').value_counts().rename_axis('Source').reset_index(name='Participants').to_string(max_rows=TABLE_ROW_LIMIT,
    index=False))
print('\nEither-wave source:')
print(receipt_at_either_wave_source.fillna('No resolved source').value_counts().rename_axis('Source').reset_index(name='Participants').to_string(max_rows=TABLE_ROW_LIMIT,
    index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in disability-benefit receipt-option review: 9,767

Wave 1 by Wave 2 broad receipt status:
W2BenSickDis code                      Disability-related benefit receipt  No disability-related benefit receipt  Total
W1bendisMP code                                                                                                        
Disability-related benefit receipt                                    784                                    250   1034
No disability-related benefit receipt                                 311                                   8023   8334
Total                                                                1095                                   8273   9368

Cross-wave broad-indicator comparison:
                                  Comparison  Both responses valid  Responses agreeing  Responses differing Agreement percentage  Wave 1 valid only  Wave 2 valid only  Both unavailable
Wave 1 versus Wave 2 broad receipt indicator                  9368     

In [152]:
# 80: Disability-benefit receipt overlap review

benefit_receipt_candidate = latest_available_benefit_receipt.copy()
benefit_receipt_candidate_label = benefit_receipt_candidate.map(benefit_receipt_labels)
main_parent_disability_review_label = selected_main_parent_disability_measure.map({0: 'No main-parent disability',
    1: 'Main-parent non-limiting disability', 2: 'Main-parent activity-limiting disability'}).fillna('Main-parent disability unavailable')
young_person_disability_review_label = consolidated_young_person_disability.map({1.0: 'Young-person disability or long-standing illness; schooling affected',
    2.0: 'Young-person disability or long-standing illness; schooling not affected', 3.0: 'No young-person disability or long-standing illness'}).fillna('Young-person disability unavailable')
additional_care_review_label = selected_additional_parental_care_measure.map({0: 'No additional care due to disability',
    1: 'Additional care due to disability'}).fillna('Additional-care status unavailable')
benefit_receipt_review_label = benefit_receipt_candidate_label.fillna('Benefit-receipt status unavailable')
stage_2_disability_benefit_overlap_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'disability_benefit_receipt': benefit_receipt_candidate, 'Disability-benefit receipt': benefit_receipt_review_label, 'Benefit-receipt source': latest_available_benefit_source, 'main_parent_disability': selected_main_parent_disability_measure, 'Main-parent disability': main_parent_disability_review_label, 'young_person_disability': consolidated_young_person_disability, 'Young-person disability': young_person_disability_review_label, 'additional_parental_care': selected_additional_parental_care_measure, 'Additional parental care': additional_care_review_label})
main_parent_disability_by_benefit = pd.crosstab(main_parent_disability_review_label, benefit_receipt_review_label,
    margins=True, margins_name='Total')
young_person_disability_by_benefit = pd.crosstab(young_person_disability_review_label, benefit_receipt_review_label,
    margins=True, margins_name='Total')
additional_care_by_benefit = pd.crosstab(additional_care_review_label, benefit_receipt_review_label, margins=True,
    margins_name='Total')

def calculate_receipt_prevalence(group_label, group_name):
    """Calculate receipt prevalence within a reviewed group."""
    review_frame = pd.DataFrame({'Review group': group_label, 'Benefit receipt': benefit_receipt_candidate})
    prevalence = review_frame.groupby('Review group', dropna=False).agg(Participants=('Benefit receipt', 'size'),
        Valid_benefit_status=('Benefit receipt', 'count'), Benefit_receipt_cases=('Benefit receipt',
        lambda values: int(values.eq(1).sum()))).reset_index()
    prevalence['Benefit receipt percentage'] = prevalence['Benefit_receipt_cases'] / prevalence['Valid_benefit_status'] * 100
    prevalence.insert(0, 'Comparison measure', group_name)
    return prevalence
main_parent_benefit_prevalence = calculate_receipt_prevalence(main_parent_disability_review_label,
    'Main-parent disability')
young_person_benefit_prevalence = calculate_receipt_prevalence(young_person_disability_review_label,
    'Young-person disability')
additional_care_benefit_prevalence = calculate_receipt_prevalence(additional_care_review_label,
    'Additional parental care')
benefit_prevalence_summary = pd.concat([main_parent_benefit_prevalence, young_person_benefit_prevalence,
    additional_care_benefit_prevalence], ignore_index=True)
benefit_increment_checks = pd.DataFrame([{'Incremental-information check': 'Benefit receipt with no main-parent disability',
    'Participants': int((benefit_receipt_candidate.eq(1) & selected_main_parent_disability_measure.eq(0)).sum())}, {'Incremental-information check': 'Benefit receipt with no young-person disability',
    'Participants': int((benefit_receipt_candidate.eq(1) & consolidated_young_person_disability.eq(3)).sum())}, {'Incremental-information check': 'Benefit receipt with no additional parental care',
    'Participants': int((benefit_receipt_candidate.eq(1) & selected_additional_parental_care_measure.eq(0)).sum())}, {'Incremental-information check': 'Benefit receipt with neither main-parent nor young-person disability recorded',
    'Participants': int((benefit_receipt_candidate.eq(1) & selected_main_parent_disability_measure.eq(0) & consolidated_young_person_disability.eq(3)).sum())}, {'Incremental-information check': 'No benefit receipt despite main-parent activity-limiting disability',
    'Participants': int((benefit_receipt_candidate.eq(0) & selected_main_parent_disability_measure.eq(2)).sum())}, {'Incremental-information check': 'No benefit receipt despite young-person disability',
    'Participants': int((benefit_receipt_candidate.eq(0) & consolidated_young_person_disability.isin([1,
    2])).sum())}])
benefit_receipt_missing_reason = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
benefit_receipt_missing = benefit_receipt_candidate.isna()
w1_benefit_source_present = stage_2_disability_benefit_review['W1bendisMP source present']
w2_benefit_source_present = stage_2_disability_benefit_review['W2BenSickDis source present']
benefit_receipt_missing_reason.loc[benefit_receipt_missing & ~w1_benefit_source_present & ~w2_benefit_source_present] = 'No Wave 1 or Wave 2 family-background source record'
benefit_receipt_missing_reason.loc[benefit_receipt_missing & w2_benefit_source_present & w2_disability_benefit_receipt.isna() & w1_disability_benefit_receipt.isna()] = 'Wave 2 broad indicator unresolved and no valid Wave 1 fallback'
benefit_receipt_missing_reason.loc[benefit_receipt_missing & ~w2_benefit_source_present & w1_benefit_source_present & w1_disability_benefit_receipt.isna()] = 'Wave 2 source absent and Wave 1 broad indicator unresolved'
benefit_receipt_missing_reason.loc[benefit_receipt_missing & benefit_receipt_missing_reason.isna()] = 'Other unresolved benefit-receipt status'
benefit_receipt_missing_summary = benefit_receipt_missing_reason.loc[benefit_receipt_missing].value_counts().rename_axis('Benefit-receipt missing reason').reset_index(name='Participants')
benefit_receipt_overlap_quality_checks = pd.DataFrame([{'Quality check': 'Participants in candidate measure',
    'Participants': int(len(benefit_receipt_candidate))}, {'Quality check': 'Valid benefit-receipt status',
    'Participants': int(benefit_receipt_candidate.notna().sum())}, {'Quality check': 'Benefit-receipt status unavailable',
    'Participants': int(benefit_receipt_candidate.isna().sum())}, {'Quality check': 'Benefit-receipt Yes',
    'Participants': int(benefit_receipt_candidate.eq(1).sum())}, {'Quality check': 'Benefit-receipt No',
    'Participants': int(benefit_receipt_candidate.eq(0).sum())}, {'Quality check': 'Values outside categories 0 and 1',
    'Participants': int((benefit_receipt_candidate.notna() & ~benefit_receipt_candidate.isin([0, 1])).sum())}])
print(f'Participants in disability-benefit overlap review: {len(stage_2_disability_benefit_overlap_review):,}')
print('\nMain-parent disability by benefit receipt:')
print(main_parent_disability_by_benefit.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nYoung-person disability by benefit receipt:')
print(young_person_disability_by_benefit.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nAdditional parental care by benefit receipt:')
print(additional_care_by_benefit.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nBenefit-receipt prevalence within reviewed groups:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 145, 'display.width', 360):
    print(benefit_prevalence_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
        formatters={'Benefit receipt percentage': lambda value: f'{value:.2f}%' if pd.notna(value) else 'Not available'}))
print('\nIncremental-information checks:')
with pd.option_context('display.max_colwidth', 150, 'display.width', 300):
    print(benefit_increment_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMissing-value reasons:')
print(benefit_receipt_missing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(benefit_receipt_overlap_quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in disability-benefit overlap review: 9,767

Main-parent disability by benefit receipt:
W2BenSickDis code                         Benefit-receipt status unavailable  Disability-related benefit receipt  No disability-related benefit receipt  Total
row_0                                                                                                                                                         
Main-parent activity-limiting disability                                   0                                 465                                    841   1306
Main-parent disability unavailable                                       255                                  17                                    106    378
Main-parent non-limiting disability                                        0                                  79                                    647    726
No main-parent disability                                                  0                            

In [153]:
# 81: Disability-related benefit receipt decision

selected_disability_benefit_receipt_measure = benefit_receipt_candidate.copy()
selected_disability_benefit_receipt_label = selected_disability_benefit_receipt_measure.map({0: 'No disability-related benefit receipt',
    1: 'Disability-related benefit receipt'})
stage_2_disability_benefit_receipt_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'W1 disability-related benefit receipt': w1_disability_benefit_receipt, 'W2 disability-related benefit receipt': w2_disability_benefit_receipt, 'selected_disability_benefit_receipt': selected_disability_benefit_receipt_measure, 'Selected disability-related benefit receipt': selected_disability_benefit_receipt_label, 'Construction source': latest_available_benefit_source, 'Missing reason': benefit_receipt_missing_reason})
stage_2_disability_benefit_receipt_review['Predictor decision'] = 'Retain as constructed predictor candidate'
disability_benefit_decision_rules = [{'Wave': 'Wave 1', 'Variable': 'W1bendisMP',
    'Review outcome': 'Retain as construction source', 'Decision reason': 'Broad household indicator of receipt of benefits for sick or disabled people. It is used as a fallback when the corresponding Wave 2 indicator is unavailable.', 'Reference-period assessment': 'Current household receipt of sickness- or disability-related benefits at Wave 1.', 'Review notes': 'Provided 52 fallback values after Wave 2 was prioritised. The combined measure was valid for 9,512 participants.'}, {'Wave': 'Wave 2',
    'Variable': 'W2BenSickDis', 'Review outcome': 'Retain as construction source', 'Decision reason': 'Broad derived indicator of whether at least one parent received a benefit for sick or disabled people. It is the primary source because Wave 2 is the latest eligible pre-transition measurement.', 'Reference-period assessment': 'Current parental receipt of sickness- or disability-related benefits at Wave 2.', 'Review notes': 'Primary source for 9,460 participants. The final measure identified 1,114 recipients and 8,398 non-recipients.'}, {'Wave': 'Wave 1',
    'Variable': 'W1ben1MP0e', 'Review outcome': 'Retain as review support only', 'Decision reason': 'Narrow Disability Living Allowance receipt item. It is not retained separately because it is nested within the broader selected disability-related benefit receipt measure.', 'Reference-period assessment': 'Current Disability Living Allowance receipt at Wave 1.', 'Review notes': 'Used to confirm that reported DLA receipt was contained within the broader Wave 1 indicator.'}, {'Wave': 'Wave 2',
    'Variable': 'W2Ben2QMP0a', 'Review outcome': 'Retain as review support only', 'Decision reason': 'Main-parent report of Disability Living Allowance receipt. It is not retained separately because DLA is a narrower component of the selected broad benefit receipt measure.', 'Reference-period assessment': 'Current household Disability Living Allowance receipt reported by the main parent at Wave 2.', 'Review notes': 'Used to verify nesting within W2BenSickDis.'}, {'Wave': 'Wave 2',
    'Variable': 'W2Ben2QSP0a', 'Review outcome': 'Retain as review support only', 'Decision reason': 'Second-parent report of Disability Living Allowance receipt. It is not retained separately because DLA is a narrower component of the selected broad benefit receipt measure.', 'Reference-period assessment': 'Current household Disability Living Allowance receipt reported by the second parent at Wave 2.', 'Review notes': 'Used to assess consistency between main- and second-parent reports and nesting within W2BenSickDis.'}, {'Wave': 'Wave 2',
    'Variable': 'W2BenDLAWeWho', 'Review outcome': 'Retain as review support only', 'Decision reason': 'Derivation-support variable identifying whose figures contributed to the total DLA amount. It does not represent a substantive participant characteristic.', 'Reference-period assessment': 'Source of parental figures used in the Wave 2 DLA amount derivation.', 'Review notes': 'Retained only to audit the DLA derivation.'}, {'Wave': 'Wave 2',
    'Variable': 'W2Ben1AmtMP0g', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Conditional main-parent DLA payment amount with very limited coverage and substantial structural non-applicability. It overlaps with the selected broad receipt indicator.', 'Reference-period assessment': "Amount of the main parent's most recent DLA payment at Wave 2.", 'Review notes': 'Only 457 non-negative values; not suitable as a general participant-level predictor.'}, {'Wave': 'Wave 2',
    'Variable': 'W2Ben1AmtSP0g', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Conditional second-parent DLA payment amount with very limited coverage and substantial structural non-applicability. It overlaps with the selected broad receipt indicator.', 'Reference-period assessment': "Amount of the second parent's most recent DLA payment at Wave 2.", 'Review notes': 'Only 134 non-negative values; not suitable as a general participant-level predictor.'}, {'Wave': 'Wave 2',
    'Variable': 'W2Ben1PdMPag', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Payment-period information used to interpret the conditional main-parent DLA amount. It is not an independent substantive predictor.', 'Reference-period assessment': "Period covered by the main parent's most recent DLA payment at Wave 2.", 'Review notes': 'Valid only for a small routed subgroup.'}, {'Wave': 'Wave 2',
    'Variable': 'W2Ben1PdSP0g', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Payment-period information used to interpret the conditional second-parent DLA amount. It is not an independent substantive predictor.', 'Reference-period assessment': "Period covered by the second parent's most recent DLA payment at Wave 2.", 'Review notes': 'Valid only for a small routed subgroup.'}, {'Wave': 'Wave 2',
    'Variable': 'W2Ben1amtDKMP0g', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Response-process indicator recording whether the main parent could not report an amount because payments were combined. It is not a substantive participant characteristic.', 'Reference-period assessment': 'Wave 2 response status for the main-parent DLA amount question.', 'Review notes': 'Survey-response support variable with only 67 non-negative values.'}, {'Wave': 'Wave 2',
    'Variable': 'W2Ben1amtDKSP0g', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Response-process indicator recording whether the second parent could not report an amount because payments were combined. It is not a substantive participant characteristic.', 'Reference-period assessment': 'Wave 2 response status for the second-parent DLA amount question.', 'Review notes': 'Survey-response support variable with only 13 non-negative values.'}, {'Wave': 'Wave 2',
    'Variable': 'W2BenDLAWeTot', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Conditional derived weekly DLA amount. It is available only for a routed recipient subgroup, has substantial insufficient information and overlaps with the selected receipt indicator.', 'Reference-period assessment': 'Derived total weekly DLA payment at Wave 2.', 'Review notes': 'Only 564 non-negative values and 943 insufficient-information values.'}, {'Wave': 'Wave 2',
    'Variable': 'W2BenDisWeAm', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Conditional derived weekly amount of sickness- and disability-related benefits. It combines receipt status with payment level and has substantial structural and unresolved missingness.', 'Reference-period assessment': 'Derived total weekly amount of benefits for sick and disabled people at Wave 2.', 'Review notes': 'Only 870 non-negative values; 1,005 cases had insufficient information.'}, {'Wave': 'Wave 2',
    'Variable': 'W2BenDisMoAm', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Monthly transformation of the same conditional sickness- and disability-benefit amount represented by W2BenDisWeAm. Retaining both would duplicate the same information.', 'Reference-period assessment': 'Derived monthly amount of benefits for sick and disabled people at Wave 2.', 'Review notes': 'Direct transformation of the weekly amount.'}, {'Wave': 'Wave 2',
    'Variable': 'W2BenDisYrAm', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Annual transformation of the same conditional sickness- and disability-benefit amount represented by W2BenDisWeAm. Retaining both would duplicate the same information.', 'Reference-period assessment': 'Derived annual amount of benefits for sick and disabled people at Wave 2.', 'Review notes': 'Direct transformation of the weekly amount.'}, {'Wave': 'Wave 2',
    'Variable': 'W2BenDisBand', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Banded transformation of the conditional total sickness- and disability-benefit amount. It remains structurally restricted to recipients and overlaps with both the continuous amount and selected receipt indicator.', 'Reference-period assessment': 'Banded total amount of benefits for sick and disabled people at Wave 2.', 'Review notes': 'Not retained because it does not resolve the conditional-measurement problem.'}]
disability_benefit_decision_indices = []
for rule in disability_benefit_decision_rules:
    matching_rows = stage_2_variable_decision_register['Wave'].eq(rule['Wave']) & stage_2_variable_decision_register['Source type'].eq('Family background') & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {rule['Wave']}, {rule['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    disability_benefit_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = rule['Review outcome']
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'Family material circumstances'
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = rule['Decision reason']
    stage_2_variable_decision_register.loc[matching_rows,
        'Leakage assessment'] = 'No direct outcome leakage identified. The information was measured at Wave 1 or Wave 2, before the post-16 transition.'
    stage_2_variable_decision_register.loc[matching_rows,
        'Reference-period assessment'] = rule['Reference-period assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave 1 and Wave 2 family-background data dictionaries and Stage 2 disability-related benefit review'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = rule['Review notes']
disability_benefit_review_output_path = stage_2_output_directory / 'stage_2_disability_benefit_receipt_review.csv'
stage_2_disability_benefit_receipt_review.to_csv(disability_benefit_review_output_path, index=False)
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
disability_benefit_decision_output = stage_2_variable_decision_register.loc[disability_benefit_decision_indices,
    ['Wave', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment',
    'Review notes']].sort_values(['Review outcome', 'Wave', 'Variable']).reset_index(drop=True)
explicitly_excluded_disability_benefit_variables = disability_benefit_decision_output.loc[disability_benefit_decision_output['Review outcome'].eq('Exclude from predictor set')].reset_index(drop=True)
support_only_disability_benefit_variables = disability_benefit_decision_output.loc[disability_benefit_decision_output['Review outcome'].eq('Retain as review support only')].reset_index(drop=True)
selected_disability_benefit_distribution = selected_disability_benefit_receipt_label.fillna('Missing').value_counts().rename_axis('Selected disability-related benefit receipt').reset_index(name='Participants')
selected_disability_benefit_source_summary = latest_available_benefit_source.fillna('No valid construction source').value_counts().rename_axis('Construction source').reset_index(name='Participants')
selected_disability_benefit_quality_checks = pd.DataFrame([{'Quality check': 'Participants in selected measure',
    'Participants': int(len(selected_disability_benefit_receipt_measure))}, {'Quality check': 'Valid selected values',
    'Participants': int(selected_disability_benefit_receipt_measure.notna().sum())}, {'Quality check': 'Missing selected values',
    'Participants': int(selected_disability_benefit_receipt_measure.isna().sum())}, {'Quality check': 'Values outside categories 0 and 1',
    'Participants': int((selected_disability_benefit_receipt_measure.notna() & ~selected_disability_benefit_receipt_measure.isin([0,
    1])).sum())}])
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'Disability-related benefit variables decided: {len(disability_benefit_decision_output):,}')
print('\nSelected measure distribution:')
print(selected_disability_benefit_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nConstruction source:')
print(selected_disability_benefit_source_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nConstruction-source decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 410):
    print(disability_benefit_decision_output.loc[disability_benefit_decision_output['Review outcome'].eq('Retain as construction source')].to_string(max_rows=TABLE_ROW_LIMIT,
        index=False))
print('\nVariables retained for review support only and not entering the predictor matrix:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 410):
    print(support_only_disability_benefit_variables.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded from the predictor set:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 410):
    print(explicitly_excluded_disability_benefit_variables.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(selected_disability_benefit_quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nVariables explicitly excluded in this step: {len(explicitly_excluded_disability_benefit_variables):,}')
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'Disability-benefit receipt review saved to: {disability_benefit_review_output_path}')

Disability-related benefit variables decided: 17

Selected measure distribution:
Selected disability-related benefit receipt  Participants
      No disability-related benefit receipt          8398
         Disability-related benefit receipt          1114
                                    Missing           255

Construction source:
           Construction source  Participants
Wave 2 broad receipt indicator          9460
  No valid construction source           255
 Wave 1 broad receipt fallback            52

Construction-source decisions:
  Wave     Variable                                                                   Variable label                Review outcome                                                                                                                                                                                   Decision reason                                                                                                     Leakage assessment          

In [154]:
# 82: SEN support and transition-planning variable structure

sen_support_variable_names = ['W1sencurr2MP', 'W1sentranMP', 'W2sencurr2MP', 'W2sentranMP', 'W3sentranMP']
sen_support_entries = stage_2_master_variable_register.loc[stage_2_master_variable_register['Variable'].str.lower().isin([variable.lower() for variable in sen_support_variable_names]),
    ['Source order', 'Wave', 'Source type', 'Source file', 'Source path', 'Variable position', 'Variable',
    'Variable label', 'Data type', 'Timing status']].copy()
sen_support_entries = sen_support_entries.merge(stage_2_variable_decision_register[['Source file', 'Variable',
    'Review outcome']], on=['Source file', 'Variable'], how='left', validate='one_to_one').sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
if len(sen_support_entries) != 5:
    raise ValueError(f'Expected five SEN support and transition-planning variables, but found {len(sen_support_entries)}.')
if sen_support_entries['Source path'].isna().any():
    raise ValueError('At least one SEN support variable has no source path.')
stage_2_sen_support_review = stage_2_participant_ids.copy()
sen_support_coverage_records = []
sen_support_value_records = []
for source_path_text, source_entries in sen_support_entries.groupby('Source path', sort=False):
    source_path = Path(source_path_text)
    source_variables = source_entries['Variable'].tolist()
    numeric_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    numeric_sample = stage_2_participant_ids.merge(numeric_source, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = numeric_sample['_merge'].eq('both')
    numeric_sample = numeric_sample.drop(columns='_merge')
    labelled_sample = stage_2_participant_ids.merge(labelled_source, on='NSID', how='left', validate='one_to_one')
    for _, entry in source_entries.iterrows():
        variable = entry['Variable']
        numeric_values = pd.to_numeric(numeric_sample[variable], errors='coerce')
        labelled_values = labelled_sample[variable].astype('string')
        stage_2_sen_support_review[f'{variable} code'] = numeric_values
        stage_2_sen_support_review[f'{variable} label'] = labelled_values
        stage_2_sen_support_review[f'{variable} source present'] = source_record_present
        non_negative_values = numeric_values.where(numeric_values.ge(0))
        sen_support_coverage_records.append({'Wave': entry['Wave'], 'Source type': entry['Source type'],
            'Variable': variable, 'Variable label': entry['Variable label'], 'Timing status': entry['Timing status'], 'Current register status': entry['Review outcome'], 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Non-negative stored value': int(non_negative_values.notna().sum()), 'Negative stored value': int(numeric_values.lt(0).sum()), 'Unique non-negative values': int(non_negative_values.dropna().nunique()), 'Minimum non-negative value': non_negative_values.min(), 'Maximum non-negative value': non_negative_values.max()})
        value_summary = pd.DataFrame({'Value code': numeric_values, 'Value label': labelled_values,
            'Source record present': source_record_present}).loc[lambda frame: frame['Source record present']].drop(columns='Source record present').value_counts(dropna=False).rename('Participants').reset_index()
        value_summary.insert(0, 'Wave', entry['Wave'])
        value_summary.insert(1, 'Variable', variable)
        sen_support_value_records.append(value_summary)
sen_support_coverage_summary = pd.DataFrame(sen_support_coverage_records)
sen_support_value_summary = pd.concat(sen_support_value_records, ignore_index=True)
sen_support_non_negative_values = sen_support_value_summary.loc[sen_support_value_summary['Value code'].ge(0)].reset_index(drop=True)
sen_support_negative_values = sen_support_value_summary.loc[sen_support_value_summary['Value code'].lt(0)].reset_index(drop=True)
valid_response_flags = pd.DataFrame({variable: stage_2_sen_support_review[f'{variable} code'].ge(0) for variable in sen_support_variable_names})
sen_support_response_pattern = valid_response_flags.astype(int).astype(str).agg(''.join,
    axis=1).value_counts().rename_axis('Valid-response pattern (W1curr2, W1transition, W2curr2, W2transition, W3transition)').reset_index(name='Participants')
valid_response_count_summary = valid_response_flags.sum(axis=1).value_counts().sort_index().rename_axis('Number of valid SEN support responses').reset_index(name='Participants')
print(f'SEN support and transition-planning variables reviewed: {len(sen_support_coverage_summary):,}')
print('\nSource entries:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 160, 'display.width', 410):
    print(sen_support_entries[['Wave', 'Source type', 'Source file', 'Variable', 'Variable label', 'Timing status',
        'Review outcome']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCoverage and stored-value structure:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 160, 'display.width', 430):
    print(sen_support_coverage_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nNon-negative stored values and labels:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 145, 'display.width', 350):
    print(sen_support_non_negative_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nNegative routing and missing-value codes:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 145, 'display.width', 350):
    print(sen_support_negative_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nValid-response counts per participant:')
print(valid_response_count_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMost common cross-wave response patterns:')
print(sen_support_response_pattern.head(15).to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

SEN support and transition-planning variables reviewed: 5

Source entries:
  Wave  Source type                        Source file     Variable                                                       Variable label          Timing status Review outcome
Wave 1 Young person   wave_one_lsype_young_person_2020 W1sencurr2MP MP: Satisfaction with how YP's current school deals with their needs  Pre-transition source Pending review
Wave 1 Young person   wave_one_lsype_young_person_2020  W1sentranMP                 MP: Whether transition plan has been drawn up for YP  Pre-transition source Pending review
Wave 2 Young person   wave_two_lsype_young_person_2020 W2sencurr2MP MP: Satisfaction with how YP's current school deals with their needs  Pre-transition source Pending review
Wave 2 Young person   wave_two_lsype_young_person_2020  W2sentranMP                 MP: Whether transition plan has been drawn up for YP  Pre-transition source Pending review
Wave 3 Young person wave_three_lsype_young_person_

In [155]:
# 83: SEN support and transition-plan routing review

current_sen_variable_names = ['W1sencurrMP', 'W2sencurrMP', 'W3sencurrMP']
current_sen_entries = stage_2_master_variable_register.loc[stage_2_master_variable_register['Variable'].str.lower().isin([variable.lower() for variable in current_sen_variable_names]) & stage_2_master_variable_register['Source type'].eq('Young person'),
    ['Source order', 'Wave', 'Source type', 'Source file', 'Source path', 'Variable position', 'Variable',
    'Variable label']].copy().sort_values(['Source order', 'Variable position']).reset_index(drop=True)
if len(current_sen_entries) != 3:
    raise ValueError(f'Expected three current-SEN variables, but found {len(current_sen_entries)}.')
stage_2_sen_support_routing_review = stage_2_sen_support_review.copy()
for source_path_text, source_entries in current_sen_entries.groupby('Source path', sort=False):
    source_path = Path(source_path_text)
    source_variables = source_entries['Variable'].tolist()
    numeric_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    numeric_sample = stage_2_participant_ids.merge(numeric_source, on='NSID', how='left', validate='one_to_one')
    labelled_sample = stage_2_participant_ids.merge(labelled_source, on='NSID', how='left', validate='one_to_one')
    for variable in source_variables:
        stage_2_sen_support_routing_review[f'{variable} code'] = pd.to_numeric(numeric_sample[variable],
            errors='coerce')
        stage_2_sen_support_routing_review[f'{variable} label'] = labelled_sample[variable].astype('string')
sen_routing_pairs = [{'Wave': 'Wave 1', 'Target variable': 'W1sencurr2MP', 'Current SEN variable': 'W1sencurrMP',
    'Construct': 'School response satisfaction'}, {'Wave': 'Wave 1', 'Target variable': 'W1sentranMP',
    'Current SEN variable': 'W1sencurrMP', 'Construct': 'Transition plan'}, {'Wave': 'Wave 2',
    'Target variable': 'W2sencurr2MP', 'Current SEN variable': 'W2sencurrMP', 'Construct': 'School response satisfaction'}, {'Wave': 'Wave 2',
    'Target variable': 'W2sentranMP', 'Current SEN variable': 'W2sencurrMP', 'Construct': 'Transition plan'}, {'Wave': 'Wave 3',
    'Target variable': 'W3sentranMP', 'Current SEN variable': 'W3sencurrMP', 'Construct': 'Transition plan'}]
sen_routing_summary_records = []
sen_routing_cross_tabs = {}
for pair in sen_routing_pairs:
    target_variable = pair['Target variable']
    current_sen_variable = pair['Current SEN variable']
    target_code = pd.to_numeric(stage_2_sen_support_routing_review[f'{target_variable} code'], errors='coerce')
    current_sen_code = pd.to_numeric(stage_2_sen_support_routing_review[f'{current_sen_variable} code'],
        errors='coerce')
    current_sen_yes = current_sen_code.eq(1)
    current_sen_no = current_sen_code.eq(2)
    target_routed_evidence = target_code.ge(0) | target_code.eq(-1)
    target_valid_response = target_code.ge(0)
    target_not_applicable = target_code.eq(-91)
    sen_routing_summary_records.append({'Wave': pair['Wave'], 'Construct': pair['Construct'],
        'Target variable': target_variable, 'Current SEN Yes': int(current_sen_yes.sum()), 'Question reached': int(target_routed_evidence.sum()), 'Valid response': int(target_valid_response.sum()), "Don't know": int(target_code.eq(-1).sum()), 'Current SEN Yes and question reached': int((current_sen_yes & target_routed_evidence).sum()), 'Current SEN Yes but question not reached': int((current_sen_yes & ~target_routed_evidence).sum()), 'Question reached without current SEN Yes': int((target_routed_evidence & ~current_sen_yes).sum()), 'Current SEN No but question reached': int((current_sen_no & target_routed_evidence).sum()), 'Not applicable': int(target_not_applicable.sum()), 'Percentage of current SEN cases reached': (current_sen_yes & target_routed_evidence).sum() / current_sen_yes.sum() * 100 if current_sen_yes.sum() else np.nan})
    current_sen_review_label = pd.Series('Current SEN status unavailable', index=stage_2_participant_ids.index,
        dtype='string')
    current_sen_review_label.loc[current_sen_yes] = 'Current SEN Yes'
    current_sen_review_label.loc[current_sen_no] = 'Current SEN No'
    target_routing_label = pd.Series('Other unavailable response', index=stage_2_participant_ids.index,
        dtype='string')
    target_routing_label.loc[target_not_applicable] = 'Not applicable'
    target_routing_label.loc[target_code.eq(-99)] = 'Main parent not interviewed'
    target_routing_label.loc[target_code.eq(-1)] = "Question reached: Don't know"
    target_routing_label.loc[target_valid_response] = 'Question reached: valid response'
    sen_routing_cross_tabs[target_variable] = pd.crosstab(current_sen_review_label, target_routing_label, margins=True,
        margins_name='Total')
sen_routing_summary = pd.DataFrame(sen_routing_summary_records)
nearby_variable_records = []
for _, target_entry in sen_support_entries.iterrows():
    same_source_variables = stage_2_master_variable_register.loc[stage_2_master_variable_register['Source file'].eq(target_entry['Source file']) & stage_2_master_variable_register['Variable position'].between(target_entry['Variable position'] - 6,
        target_entry['Variable position'] + 6), ['Wave', 'Source file', 'Variable position', 'Variable',
        'Variable label']].copy()
    same_source_variables.insert(0, 'Reviewed target', target_entry['Variable'])
    same_source_variables['Relative position'] = same_source_variables['Variable position'] - target_entry['Variable position']
    nearby_variable_records.append(same_source_variables)
sen_support_nearby_variables = pd.concat(nearby_variable_records,
    ignore_index=True).drop_duplicates(subset=['Reviewed target', 'Source file', 'Variable']).sort_values(['Wave',
    'Reviewed target', 'Variable position']).reset_index(drop=True)
routing_keyword_pattern = 'statement|transition|plan|current school|special educational|school action|needs'
possible_sen_routing_variables = sen_support_nearby_variables.loc[sen_support_nearby_variables['Variable label'].str.contains(routing_keyword_pattern,
    case=False, na=False, regex=True)].reset_index(drop=True)
print(f'SEN support routing comparisons completed: {len(sen_routing_summary):,}')
print('\nRouting summary:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 125, 'display.width', 390):
    print(sen_routing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
        formatters={'Percentage of current SEN cases reached': lambda value: f'{value:.2f}%' if pd.notna(value) else 'Not available'}))
for target_variable, cross_tab in sen_routing_cross_tabs.items():
    print(f'\nCurrent SEN status by routing: {target_variable}')
    print(cross_tab.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nNearby variables potentially relevant to routing or interpretation:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 170, 'display.width', 410):
    print(possible_sen_routing_variables[['Wave', 'Reviewed target', 'Relative position', 'Variable',
        'Variable label']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

SEN support routing comparisons completed: 5

Routing summary:
  Wave                    Construct Target variable  Current SEN Yes  Question reached  Valid response  Don't know  Current SEN Yes and question reached  Current SEN Yes but question not reached  Question reached without current SEN Yes  Current SEN No but question reached  Not applicable Percentage of current SEN cases reached
Wave 1 School response satisfaction    W1sencurr2MP              908               908             892          16                                   908                                         0                                         0                                    0            8513                                 100.00%
Wave 1              Transition plan     W1sentranMP              908               908             816          92                                   908                                         0                                         0                                    0    

In [156]:
# 84: Consolidated pre-transition SEN transition-plan status

w1_transition_plan_code = pd.to_numeric(stage_2_sen_support_routing_review['W1sentranMP code'], errors='coerce')
w2_transition_plan_code = pd.to_numeric(stage_2_sen_support_routing_review['W2sentranMP code'], errors='coerce')
w3_transition_plan_code = pd.to_numeric(stage_2_sen_support_routing_review['W3sentranMP code'], errors='coerce')
w1_current_sen_code = pd.to_numeric(stage_2_sen_support_routing_review['W1sencurrMP code'], errors='coerce')
w2_current_sen_code = pd.to_numeric(stage_2_sen_support_routing_review['W2sencurrMP code'], errors='coerce')
w3_current_sen_code = pd.to_numeric(stage_2_sen_support_routing_review['W3sencurrMP code'], errors='coerce')
w1_current_sen_yes = w1_current_sen_code.eq(1)
w2_current_sen_yes = w2_current_sen_code.eq(1)
w3_current_sen_yes = w3_current_sen_code.eq(1)
any_current_sen_pretransition = w1_current_sen_yes | w2_current_sen_yes | w3_current_sen_yes
w1_transition_plan_yes = w1_transition_plan_code.eq(1)
w2_transition_plan_yes = w2_transition_plan_code.eq(1)
w3_transition_plan_yes = w3_transition_plan_code.eq(1)
w1_transition_plan_no = w1_transition_plan_code.eq(2)
w2_transition_plan_no = w2_transition_plan_code.eq(2)
w3_transition_plan_no = w3_transition_plan_code.eq(2)
w1_transition_plan_reached = w1_transition_plan_code.isin([1, 2, -1])
w2_transition_plan_reached = w2_transition_plan_code.isin([1, 2, -1])
w3_transition_plan_reached = w3_transition_plan_code.isin([1, 2, -1])
w1_plan_yes_before_wave2 = w1_transition_plan_yes
plan_yes_before_wave3 = w1_transition_plan_yes | w2_transition_plan_yes
transition_plan_routing_validation = pd.DataFrame([{'Routing check': 'Wave 2 current SEN cases with a Wave 1 plan Yes',
    'Participants': int((w2_current_sen_yes & w1_plan_yes_before_wave2).sum())}, {'Routing check': 'Wave 2 prior plan Yes and Wave 2 Not applicable',
    'Participants': int((w2_current_sen_yes & w1_plan_yes_before_wave2 & w2_transition_plan_code.eq(-91)).sum())}, {'Routing check': 'Wave 2 prior plan Yes but question asked again',
    'Participants': int((w2_current_sen_yes & w1_plan_yes_before_wave2 & w2_transition_plan_reached).sum())}, {'Routing check': 'Wave 2 current SEN without prior plan Yes but question not reached',
    'Participants': int((w2_current_sen_yes & ~w1_plan_yes_before_wave2 & ~w2_transition_plan_reached).sum())}, {'Routing check': 'Wave 3 current SEN cases with a prior plan Yes',
    'Participants': int((w3_current_sen_yes & plan_yes_before_wave3).sum())}, {'Routing check': 'Wave 3 prior plan Yes and Wave 3 Not applicable',
    'Participants': int((w3_current_sen_yes & plan_yes_before_wave3 & w3_transition_plan_code.eq(-91)).sum())}, {'Routing check': 'Wave 3 prior plan Yes but question asked again',
    'Participants': int((w3_current_sen_yes & plan_yes_before_wave3 & w3_transition_plan_reached).sum())}, {'Routing check': 'Wave 3 current SEN without prior plan Yes but question not reached',
    'Participants': int((w3_current_sen_yes & ~plan_yes_before_wave3 & ~w3_transition_plan_reached).sum())}])
transition_plan_valid_matrix = pd.DataFrame({'Wave 1': w1_transition_plan_code.where(w1_transition_plan_code.isin([1,
    2])), 'Wave 2': w2_transition_plan_code.where(w2_transition_plan_code.isin([1,
    2])), 'Wave 3': w3_transition_plan_code.where(w3_transition_plan_code.isin([1, 2]))})
latest_valid_transition_plan_code = transition_plan_valid_matrix.ffill(axis=1).iloc[:, -1]
transition_plan_ever_yes = transition_plan_valid_matrix.eq(1).any(axis=1)
transition_plan_any_valid_response = transition_plan_valid_matrix.notna().any(axis=1)
cumulative_transition_plan_status = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
cumulative_transition_plan_status.loc[transition_plan_ever_yes] = 1
cumulative_transition_plan_status.loc[~transition_plan_ever_yes & latest_valid_transition_plan_code.eq(2)] = 0
cumulative_transition_plan_label = cumulative_transition_plan_status.map({0: 'No transition plan reported',
    1: 'Transition plan reported'})
sen_transition_plan_context = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
sen_transition_plan_context.loc[~any_current_sen_pretransition] = 0
sen_transition_plan_context.loc[any_current_sen_pretransition & cumulative_transition_plan_status.eq(0)] = 1
sen_transition_plan_context.loc[any_current_sen_pretransition & cumulative_transition_plan_status.eq(1)] = 2
sen_transition_plan_context_label = sen_transition_plan_context.map({0: 'No current SEN recorded at Waves 1–3',
    1: 'Current SEN recorded; no transition plan reported', 2: 'Current SEN recorded; transition plan reported'})
transition_plan_status_source = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
transition_plan_status_source.loc[w1_transition_plan_yes] = 'Transition plan Yes at Wave 1'
transition_plan_status_source.loc[~w1_transition_plan_yes & w2_transition_plan_yes] = 'Transition plan Yes at Wave 2'
transition_plan_status_source.loc[~w1_transition_plan_yes & ~w2_transition_plan_yes & w3_transition_plan_yes] = 'Transition plan Yes at Wave 3'
transition_plan_status_source.loc[~transition_plan_ever_yes & w3_transition_plan_no] = 'Latest No response at Wave 3'
transition_plan_status_source.loc[~transition_plan_ever_yes & ~w3_transition_plan_no & w2_transition_plan_no] = 'Latest No response at Wave 2'
transition_plan_status_source.loc[~transition_plan_ever_yes & ~w3_transition_plan_no & ~w2_transition_plan_no & w1_transition_plan_no] = 'Latest No response at Wave 1'
transition_plan_status_source.loc[~any_current_sen_pretransition] = 'No current SEN recorded at Waves 1–3'
transition_plan_unresolved_reason = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
transition_plan_unresolved = any_current_sen_pretransition & cumulative_transition_plan_status.isna()
transition_plan_unresolved_reason.loc[transition_plan_unresolved & (w1_transition_plan_code.eq(-1) | w2_transition_plan_code.eq(-1) | w3_transition_plan_code.eq(-1))] = "Only Don't know response available"
transition_plan_unresolved_reason.loc[transition_plan_unresolved & transition_plan_unresolved_reason.isna()] = 'No valid transition-plan response available'
stage_2_sen_transition_plan_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'W1 current SEN': w1_current_sen_code, 'W2 current SEN': w2_current_sen_code, 'W3 current SEN': w3_current_sen_code, 'W1 transition plan': w1_transition_plan_code, 'W2 transition plan': w2_transition_plan_code, 'W3 transition plan': w3_transition_plan_code, 'any_current_sen_pretransition': any_current_sen_pretransition.astype('Int64'), 'cumulative_transition_plan_status': cumulative_transition_plan_status, 'Cumulative transition-plan status': cumulative_transition_plan_label, 'sen_transition_plan_context': sen_transition_plan_context, 'SEN transition-plan context': sen_transition_plan_context_label, 'Construction source': transition_plan_status_source, 'Unresolved reason': transition_plan_unresolved_reason})
transition_plan_status_among_current_sen = cumulative_transition_plan_label.loc[any_current_sen_pretransition].fillna('Transition-plan status unresolved').value_counts().rename_axis('Cumulative transition-plan status').reset_index(name='Participants')
sen_transition_plan_context_distribution = sen_transition_plan_context_label.fillna('Current SEN recorded; transition-plan status unresolved').value_counts().rename_axis('SEN transition-plan context').reset_index(name='Participants')
transition_plan_source_summary = transition_plan_status_source.fillna('No resolved construction source').value_counts().rename_axis('Construction source').reset_index(name='Participants')
transition_plan_unresolved_summary = transition_plan_unresolved_reason.loc[transition_plan_unresolved].value_counts().rename_axis('Unresolved reason').reset_index(name='Participants')

def label_transition_plan_code(code):
    """Label transition-plan codes for pattern review."""
    if code == 1:
        return 'Yes'
    if code == 2:
        return 'No'
    if code == -1:
        return "Don't know"
    if code == -91:
        return 'Not applicable'
    if pd.isna(code):
        return 'No source record'
    return 'Other unavailable'
transition_plan_response_patterns = pd.DataFrame({'Wave 1': w1_transition_plan_code.map(label_transition_plan_code),
    'Wave 2': w2_transition_plan_code.map(label_transition_plan_code), 'Wave 3': w3_transition_plan_code.map(label_transition_plan_code)}).value_counts(dropna=False).rename('Participants').reset_index().sort_values('Participants',
    ascending=False).reset_index(drop=True)
transition_plan_quality_checks = pd.DataFrame([{'Quality check': 'Current SEN recorded at any reviewed wave',
    'Participants': int(any_current_sen_pretransition.sum())}, {'Quality check': 'Transition plan reported at any wave',
    'Participants': int(transition_plan_ever_yes.sum())}, {'Quality check': 'Valid No with no plan Yes at any wave',
    'Participants': int(cumulative_transition_plan_status.eq(0).sum())}, {'Quality check': 'Plan status unresolved among current-SEN cases',
    'Participants': int(transition_plan_unresolved.sum())}, {'Quality check': 'Plan response without current SEN at any reviewed wave',
    'Participants': int((transition_plan_any_valid_response & ~any_current_sen_pretransition).sum())}, {'Quality check': 'Values outside cumulative categories 0 and 1',
    'Participants': int((cumulative_transition_plan_status.notna() & ~cumulative_transition_plan_status.isin([0,
    1])).sum())}])
print(f'Participants in SEN transition-plan review: {len(stage_2_sen_transition_plan_review):,}')
print('\nRouting validation:')
print(transition_plan_routing_validation.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCumulative transition-plan status among current-SEN cases:')
print(transition_plan_status_among_current_sen.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWhole-sample contextual representation:')
with pd.option_context('display.max_colwidth', 145, 'display.width', 300):
    print(sen_transition_plan_context_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nConstruction source:')
print(transition_plan_source_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nUnresolved transition-plan status:')
print(transition_plan_unresolved_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMost common cross-wave response patterns:')
with pd.option_context('display.max_rows', 15, 'display.max_colwidth', 100, 'display.width', 300):
    print(transition_plan_response_patterns.head(15).to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(transition_plan_quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in SEN transition-plan review: 9,767

Routing validation:
                                                     Routing check  Participants
                   Wave 2 current SEN cases with a Wave 1 plan Yes           153
                   Wave 2 prior plan Yes and Wave 2 Not applicable           153
                                                               ...           ...
                    Wave 3 prior plan Yes but question asked again             0
Wave 3 current SEN without prior plan Yes but question not reached             0

Cumulative transition-plan status among current-SEN cases:
Cumulative transition-plan status  Participants
      No transition plan reported           731
         Transition plan reported           386
Transition-plan status unresolved            44

Whole-sample contextual representation:
                            SEN transition-plan context  Participants
                   No current SEN recorded at Waves 1–3          8606
      Curr

In [157]:
# 85: SEN school-response satisfaction consolidation

w1_sen_satisfaction_code = pd.to_numeric(stage_2_sen_support_routing_review['W1sencurr2MP code'], errors='coerce')
w2_sen_satisfaction_code = pd.to_numeric(stage_2_sen_support_routing_review['W2sencurr2MP code'], errors='coerce')
w1_sen_satisfaction_valid = w1_sen_satisfaction_code.where(w1_sen_satisfaction_code.isin([1, 2, 3, 4]))
w2_sen_satisfaction_valid = w2_sen_satisfaction_code.where(w2_sen_satisfaction_code.isin([1, 2, 3, 4]))
sen_satisfaction_labels = {1.0: 'Very satisfied', 2.0: 'Fairly satisfied', 3.0: 'Not very satisfied',
    4.0: 'Not at all satisfied'}
w1_sen_satisfaction_label = w1_sen_satisfaction_valid.map(sen_satisfaction_labels)
w2_sen_satisfaction_label = w2_sen_satisfaction_valid.map(sen_satisfaction_labels)
both_satisfaction_responses_valid = w1_sen_satisfaction_valid.notna() & w2_sen_satisfaction_valid.notna()
exact_satisfaction_agreement = both_satisfaction_responses_valid & w1_sen_satisfaction_valid.eq(w2_sen_satisfaction_valid)
w1_sen_satisfied_binary = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
w2_sen_satisfied_binary = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
w1_sen_satisfied_binary.loc[w1_sen_satisfaction_valid.isin([1, 2])] = 1
w1_sen_satisfied_binary.loc[w1_sen_satisfaction_valid.isin([3, 4])] = 0
w2_sen_satisfied_binary.loc[w2_sen_satisfaction_valid.isin([1, 2])] = 1
w2_sen_satisfied_binary.loc[w2_sen_satisfaction_valid.isin([3, 4])] = 0
binary_satisfaction_agreement = both_satisfaction_responses_valid & w1_sen_satisfied_binary.eq(w2_sen_satisfied_binary)
sen_satisfaction_cross_wave_comparison = pd.DataFrame([{'Comparison': 'Wave 1 versus Wave 2 four-category satisfaction',
    'Both responses valid': int(both_satisfaction_responses_valid.sum()), 'Responses agreeing': int(exact_satisfaction_agreement.sum()), 'Responses differing': int((both_satisfaction_responses_valid & w1_sen_satisfaction_valid.ne(w2_sen_satisfaction_valid)).sum()), 'Agreement percentage': exact_satisfaction_agreement.sum() / both_satisfaction_responses_valid.sum() * 100 if both_satisfaction_responses_valid.sum() else np.nan}, {'Comparison': 'Wave 1 versus Wave 2 satisfied/dissatisfied',
    'Both responses valid': int(both_satisfaction_responses_valid.sum()), 'Responses agreeing': int(binary_satisfaction_agreement.sum()), 'Responses differing': int((both_satisfaction_responses_valid & w1_sen_satisfied_binary.ne(w2_sen_satisfied_binary)).sum()), 'Agreement percentage': binary_satisfaction_agreement.sum() / both_satisfaction_responses_valid.sum() * 100 if both_satisfaction_responses_valid.sum() else np.nan}])
sen_satisfaction_cross_tabulation = pd.crosstab(w1_sen_satisfaction_label, w2_sen_satisfaction_label, margins=True,
    margins_name='Total')
latest_sen_satisfaction = w2_sen_satisfaction_valid.copy()
latest_sen_satisfaction_source = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
latest_sen_satisfaction_source.loc[w2_sen_satisfaction_valid.notna()] = 'Wave 2 satisfaction'
wave1_satisfaction_fallback = latest_sen_satisfaction.isna() & w1_sen_satisfaction_valid.notna()
latest_sen_satisfaction.loc[wave1_satisfaction_fallback] = w1_sen_satisfaction_valid.loc[wave1_satisfaction_fallback]
latest_sen_satisfaction_source.loc[wave1_satisfaction_fallback] = 'Wave 1 satisfaction fallback'
latest_sen_satisfaction_label = latest_sen_satisfaction.map(sen_satisfaction_labels)
latest_sen_satisfied_binary = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
latest_sen_satisfied_binary.loc[latest_sen_satisfaction.isin([1, 2])] = 1
latest_sen_satisfied_binary.loc[latest_sen_satisfaction.isin([3, 4])] = 0
latest_sen_satisfied_binary_label = latest_sen_satisfied_binary.map({0: 'Dissatisfied with school response',
    1: 'Satisfied with school response'})
sen_school_response_context = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
sen_school_response_context.loc[~any_current_sen_pretransition] = 0
sen_school_response_context.loc[any_current_sen_pretransition & latest_sen_satisfied_binary.eq(1)] = 1
sen_school_response_context.loc[any_current_sen_pretransition & latest_sen_satisfied_binary.eq(0)] = 2
sen_school_response_context_label = sen_school_response_context.map({0: 'No current SEN recorded at Waves 1–3',
    1: 'Current SEN recorded; parent satisfied with school response', 2: 'Current SEN recorded; parent dissatisfied with school response'})
sen_satisfaction_unresolved_reason = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
sen_satisfaction_unresolved = any_current_sen_pretransition & latest_sen_satisfaction.isna()
sen_satisfaction_unresolved_reason.loc[sen_satisfaction_unresolved & (w1_sen_satisfaction_code.eq(-1) | w2_sen_satisfaction_code.eq(-1))] = "Only Don't know response available"
sen_satisfaction_unresolved_reason.loc[sen_satisfaction_unresolved & sen_satisfaction_unresolved_reason.isna()] = 'No valid Wave 1 or Wave 2 satisfaction response'
stage_2_sen_satisfaction_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'W1 SEN satisfaction': w1_sen_satisfaction_valid, 'W2 SEN satisfaction': w2_sen_satisfaction_valid, 'latest_sen_satisfaction': latest_sen_satisfaction, 'Latest SEN satisfaction': latest_sen_satisfaction_label, 'latest_sen_satisfied_binary': latest_sen_satisfied_binary, 'Latest binary SEN satisfaction': latest_sen_satisfied_binary_label, 'sen_school_response_context': sen_school_response_context, 'SEN school-response context': sen_school_response_context_label, 'Construction source': latest_sen_satisfaction_source, 'Unresolved reason': sen_satisfaction_unresolved_reason})
latest_sen_satisfaction_distribution = latest_sen_satisfaction_label.fillna('No valid satisfaction response').value_counts().rename_axis('Latest SEN school-response satisfaction').reset_index(name='Participants')
binary_satisfaction_among_current_sen = latest_sen_satisfied_binary_label.loc[any_current_sen_pretransition].fillna('Satisfaction status unresolved').value_counts().rename_axis('Satisfaction among current-SEN cases').reset_index(name='Participants')
sen_school_response_context_distribution = sen_school_response_context_label.fillna('Current SEN recorded; satisfaction status unresolved').value_counts().rename_axis('SEN school-response context').reset_index(name='Participants')
sen_satisfaction_source_summary = latest_sen_satisfaction_source.fillna('No valid construction source').value_counts().rename_axis('Construction source').reset_index(name='Participants')
sen_satisfaction_unresolved_summary = sen_satisfaction_unresolved_reason.loc[sen_satisfaction_unresolved].value_counts().rename_axis('Unresolved reason').reset_index(name='Participants')
plan_by_satisfaction_among_current_sen = pd.crosstab(cumulative_transition_plan_label.loc[any_current_sen_pretransition].fillna('Transition-plan status unresolved'),
    latest_sen_satisfied_binary_label.loc[any_current_sen_pretransition].fillna('Satisfaction status unresolved'), margins=True, margins_name='Total')
sen_satisfaction_quality_checks = pd.DataFrame([{'Quality check': 'Valid Wave 1 satisfaction responses',
    'Participants': int(w1_sen_satisfaction_valid.notna().sum())}, {'Quality check': 'Valid Wave 2 satisfaction responses',
    'Participants': int(w2_sen_satisfaction_valid.notna().sum())}, {'Quality check': 'Wave 1 and Wave 2 both valid',
    'Participants': int(both_satisfaction_responses_valid.sum())}, {'Quality check': 'Latest available satisfaction responses',
    'Participants': int(latest_sen_satisfaction.notna().sum())}, {'Quality check': 'Current-SEN cases with satisfaction unresolved',
    'Participants': int(sen_satisfaction_unresolved.sum())}, {'Quality check': 'Satisfaction response without current SEN at any reviewed wave',
    'Participants': int((latest_sen_satisfaction.notna() & ~any_current_sen_pretransition).sum())}, {'Quality check': 'Values outside satisfaction categories 1–4',
    'Participants': int((latest_sen_satisfaction.notna() & ~latest_sen_satisfaction.isin([1, 2, 3, 4])).sum())}])
print(f'Participants in SEN satisfaction review: {len(stage_2_sen_satisfaction_review):,}')
print('\nCross-wave satisfaction comparison:')
print(sen_satisfaction_cross_wave_comparison.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Agreement percentage': lambda value: f'{value:.2f}%' if pd.notna(value) else 'Not available'}))
print('\nWave 1 by Wave 2 satisfaction response:')
print(sen_satisfaction_cross_tabulation.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nLatest four-category satisfaction distribution:')
print(latest_sen_satisfaction_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nBinary satisfaction among current-SEN cases:')
print(binary_satisfaction_among_current_sen.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWhole-sample contextual representation:')
with pd.option_context('display.max_colwidth', 145, 'display.width', 300):
    print(sen_school_response_context_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nConstruction source:')
print(sen_satisfaction_source_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nUnresolved satisfaction status:')
print(sen_satisfaction_unresolved_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nTransition-plan status by satisfaction among current-SEN cases:')
print(plan_by_satisfaction_among_current_sen.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nQuality checks:')
print(sen_satisfaction_quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in SEN satisfaction review: 9,767

Cross-wave satisfaction comparison:
                                     Comparison  Both responses valid  Responses agreeing  Responses differing Agreement percentage
Wave 1 versus Wave 2 four-category satisfaction                   619                 306                  313               49.43%
    Wave 1 versus Wave 2 satisfied/dissatisfied                   619                 479                  140               77.38%

Wave 1 by Wave 2 satisfaction response:
W2sencurr2MP code     Fairly satisfied  Not at all satisfied  Not very satisfied  Very satisfied  Total
W1sencurr2MP code                                                                                      
Fairly satisfied                    83                    16                  28              53    180
Not at all satisfied                15                    29                  21              10     75
Not very satisfied                  31                    12   

In [158]:
# 86: Selected SEN measure check

sen_measure_review_path = stage_2_output_directory / 'stage_2_sen_measure_review.csv'
if not sen_measure_review_path.exists():
    raise FileNotFoundError(f'The saved SEN measure review was not found: {sen_measure_review_path}')
saved_sen_measure_review = pd.read_csv(sen_measure_review_path)
if 'NSID' not in saved_sen_measure_review.columns:
    raise ValueError('The saved SEN review does not contain NSID.')
sen_review_identifier_checks = pd.DataFrame([{'Check': 'Rows in saved SEN review',
    'Result': len(saved_sen_measure_review)}, {'Check': 'Unique NSID values',
    'Result': saved_sen_measure_review['NSID'].nunique(dropna=True)}, {'Check': 'Missing NSID values',
    'Result': int(saved_sen_measure_review['NSID'].isna().sum())}, {'Check': 'Duplicate NSID values',
    'Result': int(saved_sen_measure_review['NSID'].duplicated().sum())}])
sen_candidate_columns = [column for column in saved_sen_measure_review.columns if column != 'NSID' and any((term in column.lower() for term in ['sen',
    'selected', 'status', 'category', 'predictor', 'current', 'identif']))]
sen_candidate_column_summary = []
for column in sen_candidate_columns:
    non_missing = saved_sen_measure_review[column].notna().sum()
    unique_non_missing = saved_sen_measure_review[column].dropna().nunique()
    sen_candidate_column_summary.append({'Column': column, 'Data type': str(saved_sen_measure_review[column].dtype),
        'Non-missing values': int(non_missing), 'Missing values': int(len(saved_sen_measure_review) - non_missing), 'Unique non-missing values': int(unique_non_missing)})
sen_candidate_column_summary = pd.DataFrame(sen_candidate_column_summary)
sen_low_cardinality_columns = sen_candidate_column_summary.loc[sen_candidate_column_summary['Unique non-missing values'].between(2,
    10, inclusive='both'), 'Column'].tolist()
sen_candidate_distribution_records = []
for column in sen_low_cardinality_columns:
    distribution = saved_sen_measure_review[column].astype('string').fillna('<Missing>').value_counts(dropna=False).rename_axis('Stored value').reset_index(name='Participants')
    distribution.insert(0, 'Column', column)
    sen_candidate_distribution_records.append(distribution)
if sen_candidate_distribution_records:
    sen_candidate_distributions = pd.concat(sen_candidate_distribution_records, ignore_index=True)
else:
    sen_candidate_distributions = pd.DataFrame(columns=['Column', 'Stored value', 'Participants'])
print(f'Saved SEN review file: {sen_measure_review_path}')
print('\nIdentifier checks:')
print(sen_review_identifier_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nAll columns in the saved SEN review:')
for position, column in enumerate(saved_sen_measure_review.columns, start=1):
    print(f'{position:>2}. {column}')
print('\nCandidate SEN-related columns:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 310):
    print(sen_candidate_column_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nDistributions for candidate columns with two to ten values:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 310):
    print(sen_candidate_distributions.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Saved SEN review file: data_derived\stage_2_predictor_construction\stage_2_sen_measure_review.csv

Identifier checks:
                   Check  Result
Rows in saved SEN review    9767
      Unique NSID values    9767
     Missing NSID values       0
   Duplicate NSID values       0

All columns in the saved SEN review:
 1. NSID
 2. sen_status_pretransition
 3. Pre-transition SEN status
 4. SEN-measure source
 5. SEN-measure missing reason
 6. sen_identified_pretransition
 7. Pre-transition SEN identification
 8. latest_recorded_current_sen
 9. Latest current-SEN source

Candidate SEN-related columns:
                     Column Data type  Non-missing values  Missing values  Unique non-missing values
   sen_status_pretransition   float64                9642             125                          3
  Pre-transition SEN status       str                9642             125                          3
                        ...       ...                 ...             ...                

In [159]:
# 87: Selected SEN status and current-SEN history comparison

selected_sen_fields = saved_sen_measure_review[['NSID', 'sen_status_pretransition', 'Pre-transition SEN status',
    'sen_identified_pretransition', 'latest_recorded_current_sen', 'Latest current-SEN source']].copy()
current_sen_history_fields = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'W1 current SEN code': w1_current_sen_code, 'W2 current SEN code': w2_current_sen_code, 'W3 current SEN code': w3_current_sen_code, 'any_current_sen_waves_1_to_3': any_current_sen_pretransition.astype('Int64'), 'latest_sen_satisfaction': latest_sen_satisfaction, 'cumulative_transition_plan_status': cumulative_transition_plan_status})
stage_2_selected_sen_history_review = selected_sen_fields.merge(current_sen_history_fields, on='NSID', how='left',
    validate='one_to_one')
if len(stage_2_selected_sen_history_review) != 9767:
    raise ValueError('The SEN history comparison does not contain the expected 9,767 participants.')
selected_sen_status_label = stage_2_selected_sen_history_review['Pre-transition SEN status'].astype('string').fillna('Selected SEN status unavailable')
any_current_sen_label = stage_2_selected_sen_history_review['any_current_sen_waves_1_to_3'].map({0: 'No current SEN recorded at Waves 1–3',
    1: 'Current SEN recorded at least once at Waves 1–3'}).fillna('Current-SEN history unavailable')
latest_current_sen_label = stage_2_selected_sen_history_review['latest_recorded_current_sen'].map({0.0: 'Not current at latest valid record',
    1.0: 'Current at latest valid record'}).fillna('Latest current-SEN status unavailable')
selected_sen_by_any_current_history = pd.crosstab(selected_sen_status_label, any_current_sen_label, margins=True,
    margins_name='Total')
selected_sen_by_latest_current_status = pd.crosstab(selected_sen_status_label, latest_current_sen_label, margins=True,
    margins_name='Total')

def label_current_sen_wave(code):
    """Label current-SEN status at one wave."""
    if code == 1:
        return 'Yes'
    if code == 2:
        return 'No'
    return 'Unavailable'
current_sen_wave_patterns = pd.DataFrame({'Selected SEN status': selected_sen_status_label,
    'Wave 1 current SEN': stage_2_selected_sen_history_review['W1 current SEN code'].map(label_current_sen_wave), 'Wave 2 current SEN': stage_2_selected_sen_history_review['W2 current SEN code'].map(label_current_sen_wave), 'Wave 3 current SEN': stage_2_selected_sen_history_review['W3 current SEN code'].map(label_current_sen_wave)}).value_counts(dropna=False).rename('Participants').reset_index().sort_values('Participants',
    ascending=False).reset_index(drop=True)
selected_sen_code = pd.to_numeric(stage_2_selected_sen_history_review['sen_status_pretransition'], errors='coerce')
any_current_sen_code = stage_2_selected_sen_history_review['any_current_sen_waves_1_to_3']
sen_definition_difference_summary = pd.DataFrame([{'Comparison group': 'Selected current-SEN category and current SEN recorded at least once',
    'Participants': int((selected_sen_code.eq(2) & any_current_sen_code.eq(1)).sum())}, {'Comparison group': 'Selected identified-but-not-current category and earlier current SEN recorded',
    'Participants': int((selected_sen_code.eq(1) & any_current_sen_code.eq(1)).sum())}, {'Comparison group': 'Selected identified-but-not-current category with no current SEN recorded at Waves 1–3',
    'Participants': int((selected_sen_code.eq(1) & any_current_sen_code.eq(0)).sum())}, {'Comparison group': 'Selected no-identification category but current SEN recorded at Waves 1–3',
    'Participants': int((selected_sen_code.eq(0) & any_current_sen_code.eq(1)).sum())}, {'Comparison group': 'Selected current-SEN category but no current SEN recorded at Waves 1–3',
    'Participants': int((selected_sen_code.eq(2) & any_current_sen_code.eq(0)).sum())}, {'Comparison group': 'Selected SEN status unavailable but current SEN recorded at Waves 1–3',
    'Participants': int((selected_sen_code.isna() & any_current_sen_code.eq(1)).sum())}])
stage_2_selected_sen_history_review['Valid school-response satisfaction'] = stage_2_selected_sen_history_review['latest_sen_satisfaction'].notna()
stage_2_selected_sen_history_review['Resolved transition-plan status'] = stage_2_selected_sen_history_review['cumulative_transition_plan_status'].notna()
sen_support_coverage_by_selected_status = stage_2_selected_sen_history_review.assign(selected_sen_status=selected_sen_status_label).groupby('selected_sen_status',
    dropna=False).agg(Participants=('NSID', 'size'),
    Current_SEN_recorded_at_least_once=('any_current_sen_waves_1_to_3',
    lambda values: int(values.eq(1).sum())), Valid_school_response_satisfaction=('Valid school-response satisfaction',
    'sum'), Resolved_transition_plan_status=('Resolved transition-plan status',
    'sum')).reset_index().rename(columns={'selected_sen_status': 'Selected SEN status'})
sen_support_coverage_by_selected_status['Satisfaction coverage percentage'] = sen_support_coverage_by_selected_status['Valid_school_response_satisfaction'] / sen_support_coverage_by_selected_status['Participants'] * 100
sen_support_coverage_by_selected_status['Transition-plan coverage percentage'] = sen_support_coverage_by_selected_status['Resolved_transition_plan_status'] / sen_support_coverage_by_selected_status['Participants'] * 100
sen_support_context_cross_tab = pd.crosstab(selected_sen_status_label,
    [latest_sen_satisfied_binary_label.fillna('Satisfaction unresolved or not applicable'),
    cumulative_transition_plan_label.fillna('Transition-plan unresolved or not applicable')], margins=True, margins_name='Total')
selected_sen_history_quality_checks = pd.DataFrame([{'Quality check': 'Participants in comparison',
    'Participants': len(stage_2_selected_sen_history_review)}, {'Quality check': 'Missing merged current-SEN history',
    'Participants': int(stage_2_selected_sen_history_review['any_current_sen_waves_1_to_3'].isna().sum())}, {'Quality check': 'Selected current SEN without any Wave 1–3 current-SEN record',
    'Participants': int((selected_sen_code.eq(2) & any_current_sen_code.eq(0)).sum())}, {'Quality check': 'Support response without any Wave 1–3 current-SEN record',
    'Participants': int(((stage_2_selected_sen_history_review['latest_sen_satisfaction'].notna() | stage_2_selected_sen_history_review['cumulative_transition_plan_status'].notna()) & any_current_sen_code.eq(0)).sum())}])
print(f'Participants in selected-SEN history comparison: {len(stage_2_selected_sen_history_review):,}')
print('\nSelected SEN status by any current-SEN record at Waves 1–3:')
print(selected_sen_by_any_current_history.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nSelected SEN status by latest current-SEN status:')
print(selected_sen_by_latest_current_status.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nMain differences between SEN definitions:')
with pd.option_context('display.max_colwidth', 150, 'display.width', 300):
    print(sen_definition_difference_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSupport-variable coverage by selected SEN status:')
with pd.option_context('display.max_colwidth', 145, 'display.width', 360):
    print(sen_support_coverage_by_selected_status.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
        formatters={'Satisfaction coverage percentage': lambda value: f'{value:.2f}%',
        'Transition-plan coverage percentage': lambda value: f'{value:.2f}%'}))
print('\nSatisfaction and transition-plan status by selected SEN category:')
print(sen_support_context_cross_tab.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nMost common current-SEN wave patterns:')
with pd.option_context('display.max_rows', 20, 'display.max_colwidth', 120, 'display.width', 360):
    print(current_sen_wave_patterns.head(20).to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(selected_sen_history_quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in selected-SEN history comparison: 9,767

Selected SEN status by any current-SEN record at Waves 1–3:
any_current_sen_waves_1_to_3                                 Current SEN recorded at least once at Waves 1–3  No current SEN recorded at Waves 1–3  Total
Pre-transition SEN status                                                                                                                                
No pre-transition SEN identification                                                                       0                                  7828   7828
SEN identified; current at latest pre-transition record                                                  779                                     0    779
SEN identified; not current at latest pre-transition record                                              382                                   653   1035
Selected SEN status unavailable                                                                            0      

In [160]:
# 88: SEN school-support variable decisions

stage_2_sen_transition_plan_review['Predictor decision'] = 'Retain as review support only; do not include in the predictor matrix'
stage_2_sen_transition_plan_review['Decision reason'] = 'Transition-plan responses are conditionally available only for participants with current SEN and largely repeat information already represented by the selected pre-transition SEN status measure.'
stage_2_sen_satisfaction_review['Predictor decision'] = 'Retain as review support only; do not include in the predictor matrix'
stage_2_sen_satisfaction_review['Decision reason'] = 'School-response satisfaction is conditionally available only for participants with current SEN and would introduce a structural not-applicable category that largely re-identifies SEN status.'
sen_school_support_decision_rules = [{'Wave': 'Wave 1', 'Variable': 'W1sencurr2MP',
    'Decision reason': "Parental satisfaction with the school's response to the young person's SEN. The item was asked only when current SEN was recorded and is therefore structurally nested within the selected SEN measure.", 'Reference-period assessment': "Parental assessment of how the current school dealt with the young person's needs at Wave 1.", 'Leakage assessment': 'No direct outcome leakage identified. The information was collected before transition.', 'Review notes': 'Retained for construct review only. Wave 1 provided 892 valid responses.'}, {'Wave': 'Wave 2',
    'Variable': 'W2sencurr2MP', 'Decision reason': 'Later parental satisfaction measure for the same conditional SEN subgroup. Although it adds school-response information, including it would largely duplicate current-SEN status through structural applicability.', 'Reference-period assessment': "Parental assessment of how the current school dealt with the young person's needs at Wave 2.", 'Leakage assessment': 'No direct outcome leakage identified. The information was collected before transition.', 'Review notes': 'Retained for construct review only. Wave 2 provided 811 valid responses; the latest-available review measure was valid for 1,084 participants.'}, {'Wave': 'Wave 1',
    'Variable': 'W1sentranMP', 'Decision reason': 'Conditional transition-planning item asked only for participants with current SEN. A separate whole-sample predictor would largely repeat the selected SEN status through its routing structure.', 'Reference-period assessment': 'Whether a transition plan had been drawn up for the young person by Wave 1.', 'Leakage assessment': 'No direct outcome leakage identified. The information was collected before transition.', 'Review notes': 'Retained for routing and cumulative-plan review. Wave 1 provided 816 valid responses.'}, {'Wave': 'Wave 2',
    'Variable': 'W2sentranMP', 'Decision reason': 'Conditional update to the SEN transition-plan item. Participants with a previous Yes response were not asked again, so the item requires cumulative routing reconstruction and remains nested within SEN status.', 'Reference-period assessment': 'Whether a transition plan had been drawn up for the young person by Wave 2.', 'Leakage assessment': 'No direct outcome leakage identified. The information was collected before transition.', 'Review notes': 'Retained for routing and cumulative-plan review. Wave 2 added 127 new Yes responses.'}, {'Wave': 'Wave 3',
    'Variable': 'W3sentranMP', 'Decision reason': 'Near-transition update to the conditional SEN transition-plan item. It provides institutional planning information but remains structurally restricted to current-SEN cases and overlaps with the selected SEN status measure.', 'Reference-period assessment': 'Whether a transition plan had been drawn up for the young person by Wave 3.', 'Leakage assessment': 'No direct outcome leakage identified. The item records pre-transition planning, but it was measured near the transition and is not retained as a predictor.', 'Review notes': 'Retained for timing and cumulative-plan review. Wave 3 added 78 new Yes responses.'}]
sen_school_support_decision_indices = []
for rule in sen_school_support_decision_rules:
    matching_rows = stage_2_variable_decision_register['Wave'].eq(rule['Wave']) & stage_2_variable_decision_register['Source type'].eq('Young person') & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {rule['Wave']}, {rule['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    sen_school_support_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = 'Retain as review support only'
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'SEN and school support'
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = rule['Decision reason']
    stage_2_variable_decision_register.loc[matching_rows, 'Leakage assessment'] = rule['Leakage assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Reference-period assessment'] = rule['Reference-period assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave 1–Wave 3 young-person data dictionaries and Stage 2 SEN school-support routing review'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = rule['Review notes']
sen_transition_plan_review_output_path = stage_2_output_directory / 'stage_2_sen_transition_plan_review.csv'
sen_satisfaction_review_output_path = stage_2_output_directory / 'stage_2_sen_school_response_satisfaction_review.csv'
stage_2_sen_transition_plan_review.to_csv(sen_transition_plan_review_output_path, index=False)
stage_2_sen_satisfaction_review.to_csv(sen_satisfaction_review_output_path, index=False)
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
sen_school_support_decision_output = stage_2_variable_decision_register.loc[sen_school_support_decision_indices,
    ['Wave', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment',
    'Review notes']].sort_values(['Wave', 'Variable']).reset_index(drop=True)
sen_school_support_review_summary = pd.DataFrame([{'Review item': 'Current SEN recorded at any Wave 1–3',
    'Participants': int(any_current_sen_pretransition.sum())}, {'Review item': 'Latest school-response satisfaction available',
    'Participants': int(latest_sen_satisfaction.notna().sum())}, {'Review item': 'Transition-plan status resolved',
    'Participants': int(cumulative_transition_plan_status.notna().sum())}, {'Review item': 'Satisfaction response without current-SEN history',
    'Participants': int((latest_sen_satisfaction.notna() & ~any_current_sen_pretransition).sum())}, {'Review item': 'Transition-plan response without current-SEN history',
    'Participants': int((cumulative_transition_plan_status.notna() & ~any_current_sen_pretransition).sum())}])
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
print(f'SEN school-support variables decided: {len(sen_school_support_decision_output):,}')
print('\nReviewed construction summary:')
print(sen_school_support_review_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables retained for review support only and not entering the predictor matrix:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 420):
    print(sen_school_support_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'SEN transition-plan review saved to: {sen_transition_plan_review_output_path}')
print(f'SEN satisfaction review saved to: {sen_satisfaction_review_output_path}')

SEN school-support variables decided: 5

Reviewed construction summary:
                                         Review item  Participants
                Current SEN recorded at any Wave 1–3          1161
       Latest school-response satisfaction available          1084
                     Transition-plan status resolved          1117
   Satisfaction response without current-SEN history             0
Transition-plan response without current-SEN history             0

Variables retained for review support only and not entering the predictor matrix:
  Wave     Variable                                                       Variable label                Review outcome                                                                                                                                                                                                                Decision reason                                                                                                      

In [161]:
# 89: Remaining health, disability and SEN review candidates

pending_variable_register = stage_2_variable_decision_register.loc[stage_2_variable_decision_register['Review outcome'].eq('Pending review')].copy()
health_disability_label_pattern = '\\bdisab\\w*\\b|\\blong[- ]standing illness\\b|\\billness\\b|\\bhealth\\b|\\bsick\\b|\\binfirm\\w*\\b|\\bspecial educational needs?\\b|\\bspecial needs?\\b|\\bstatement of needs?\\b|\\btransition plan\\b'
health_disability_variable_pattern = '(?i)^W[1-4](?:disab|hea|chea|sen|stated)'
label_match = pending_variable_register['Variable label'].astype('string').str.contains(health_disability_label_pattern,
    case=False, na=False, regex=True)
variable_name_match = pending_variable_register['Variable'].astype('string').str.contains(health_disability_variable_pattern,
    na=False, regex=True)
remaining_health_disability_candidates = pending_variable_register.loc[label_match | variable_name_match,
    ['Source order', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status', 'Review outcome']].copy()
remaining_health_disability_candidates['Candidate-detection basis'] = 'Variable label'
remaining_health_disability_candidates.loc[variable_name_match.loc[remaining_health_disability_candidates.index] & ~label_match.loc[remaining_health_disability_candidates.index],
    'Candidate-detection basis'] = 'Variable name only'
remaining_health_disability_candidates.loc[variable_name_match.loc[remaining_health_disability_candidates.index] & label_match.loc[remaining_health_disability_candidates.index],
    'Candidate-detection basis'] = 'Variable label and variable name'
candidate_label_text = remaining_health_disability_candidates['Variable label'].astype('string').str.lower()
remaining_health_disability_candidates['Review navigation group'] = 'Other health, disability or SEN item'
remaining_health_disability_candidates.loc[candidate_label_text.str.contains('benefit|allowance|payment', na=False,
    regex=True), 'Review navigation group'] = 'Benefit-related item'
remaining_health_disability_candidates.loc[candidate_label_text.str.contains('special educational|special needs|statement of needs|transition plan',
    na=False, regex=True), 'Review navigation group'] = 'SEN identification or support'
remaining_health_disability_candidates.loc[candidate_label_text.str.contains('\\bparent\\b|\\bMP\\b|\\bSP\\b|mother|father',
    case=False, na=False, regex=True), 'Review navigation group'] = 'Parental health or disability'
remaining_health_disability_candidates.loc[remaining_health_disability_candidates['Wave'].eq('Wave 4'),
    'Review navigation group'] = 'Wave 4 item requiring timing and leakage review'
remaining_health_disability_candidates = remaining_health_disability_candidates.sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
remaining_health_disability_group_summary = remaining_health_disability_candidates.groupby(['Wave', 'Source type',
    'Review navigation group'], dropna=False).agg(Variables=('Variable', 'size')).reset_index().sort_values(['Wave',
    'Source type', 'Review navigation group']).reset_index(drop=True)
remaining_health_disability_wave_summary = remaining_health_disability_candidates.groupby(['Wave', 'Source type'],
    dropna=False).agg(Variables=('Variable', 'size')).reset_index().sort_values(['Wave',
    'Source type']).reset_index(drop=True)
remaining_health_disability_checks = pd.DataFrame([{'Check': 'Remaining candidate variables',
    'Result': len(remaining_health_disability_candidates)}, {'Check': 'Candidates not marked Pending review',
    'Result': int(~remaining_health_disability_candidates['Review outcome'].eq('Pending review').sum()) if len(remaining_health_disability_candidates) == 0 else int((~remaining_health_disability_candidates['Review outcome'].eq('Pending review')).sum())}, {'Check': 'Duplicate source-variable entries',
    'Result': int(remaining_health_disability_candidates[['Source file', 'Variable']].duplicated().sum())}])
print(f'Remaining pending health, disability and SEN candidates: {len(remaining_health_disability_candidates):,}')
print('\nSummary by wave and source type:')
print(remaining_health_disability_wave_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSummary by review navigation group:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 130, 'display.width', 330):
    print(remaining_health_disability_group_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nFirst 20 remaining candidate variables:')
remaining_candidate_preview = remaining_health_disability_candidates[['Wave', 'Source type', 'Variable',
    'Variable label', 'Timing status', 'Candidate-detection basis', 'Review navigation group']].head(20)
with pd.option_context('display.max_colwidth', 120, 'display.width', 300):
    print(remaining_candidate_preview.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
remaining_not_displayed = len(remaining_health_disability_candidates) - len(remaining_candidate_preview)
print(f'\nAdditional candidate variables not displayed: {remaining_not_displayed:,}')

Remaining pending health, disability and SEN candidates: 155

Summary by wave and source type:
  Wave        Source type  Variables
Wave 1  Family background         10
Wave 1 Parental attitudes          4
   ...                ...        ...
Wave 4  Family background         52
Wave 4       Young person         36

Summary by review navigation group:
  Wave        Source type                         Review navigation group  Variables
Wave 1  Family background                   Parental health or disability         10
Wave 1 Parental attitudes                   Parental health or disability          1
   ...                ...                                             ...        ...
Wave 4  Family background Wave 4 item requiring timing and leakage review         52
Wave 4       Young person Wave 4 item requiring timing and leakage review         36

First 20 remaining candidate variables:
  Wave  Source type     Variable                                                      Variable 

In [162]:
# 90: Statement-of-needs structure and overlap review

statement_variable_names = ['W1statedMP', 'W1stated2MP', 'W2statedMP', 'W2stated2MP', 'W3statedMP', 'W3stated2MP']
statement_entries = stage_2_master_variable_register.loc[stage_2_master_variable_register['Variable'].str.lower().isin([variable.lower() for variable in statement_variable_names]) & stage_2_master_variable_register['Source type'].eq('Young person'),
    ['Source order', 'Wave', 'Source type', 'Source file', 'Source path', 'Variable position', 'Variable',
    'Variable label', 'Timing status']].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
if len(statement_entries) != 6:
    raise ValueError(f'Expected six pre-transition statement-of-needs variables, but found {len(statement_entries)}.')
statement_entries = statement_entries.merge(stage_2_variable_decision_register[['Source file', 'Variable',
    'Review outcome']], on=['Source file', 'Variable'], how='left', validate='one_to_one')
stage_2_statement_of_needs_review = stage_2_participant_ids.copy()
statement_coverage_records = []
statement_value_records = []
for source_path_text, source_entries in statement_entries.groupby('Source path', sort=False):
    source_path = Path(source_path_text)
    source_variables = source_entries['Variable'].tolist()
    numeric_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    numeric_sample = stage_2_participant_ids.merge(numeric_source, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = numeric_sample['_merge'].eq('both')
    numeric_sample = numeric_sample.drop(columns='_merge')
    labelled_sample = stage_2_participant_ids.merge(labelled_source, on='NSID', how='left', validate='one_to_one')
    for _, entry in source_entries.iterrows():
        variable = entry['Variable']
        numeric_values = pd.to_numeric(numeric_sample[variable], errors='coerce')
        labelled_values = labelled_sample[variable].astype('string')
        stage_2_statement_of_needs_review[f'{variable} code'] = numeric_values
        stage_2_statement_of_needs_review[f'{variable} label'] = labelled_values
        stage_2_statement_of_needs_review[f'{variable} source present'] = source_record_present
        valid_yes_no = numeric_values.where(numeric_values.isin([1, 2]))
        statement_coverage_records.append({'Wave': entry['Wave'], 'Variable': variable,
            'Variable label': entry['Variable label'], 'Timing status': entry['Timing status'], 'Current register status': entry['Review outcome'], 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Valid Yes or No': int(valid_yes_no.notna().sum()), 'Yes': int(numeric_values.eq(1).sum()), 'No': int(numeric_values.eq(2).sum()), 'Not applicable': int(numeric_values.eq(-91).sum()), "Don't know": int(numeric_values.eq(-1).sum()), 'Respondent not interviewed': int(numeric_values.eq(-99).sum()), 'Other negative code': int((numeric_values.lt(0) & ~numeric_values.isin([-91,
            -1, -99])).sum())})
        value_summary = pd.DataFrame({'Value code': numeric_values, 'Value label': labelled_values,
            'Source record present': source_record_present}).loc[lambda frame: frame['Source record present']].drop(columns='Source record present').value_counts(dropna=False).rename('Participants').reset_index()
        value_summary.insert(0, 'Wave', entry['Wave'])
        value_summary.insert(1, 'Variable', variable)
        statement_value_records.append(value_summary)
statement_coverage_summary = pd.DataFrame(statement_coverage_records)
statement_value_summary = pd.concat(statement_value_records, ignore_index=True)
statement_codes = {variable: pd.to_numeric(stage_2_statement_of_needs_review[f'{variable} code'],
    errors='coerce') for variable in statement_variable_names}
statement_valid = {variable: statement_codes[variable].where(statement_codes[variable].isin([1,
    2])) for variable in statement_variable_names}
statement_routing_records = []
for wave_number in [1, 2, 3]:
    ever_variable = f'W{wave_number}statedMP'
    current_variable = f'W{wave_number}stated2MP'
    ever_code = statement_codes[ever_variable]
    current_code = statement_codes[current_variable]
    ever_yes = ever_code.eq(1)
    ever_no = ever_code.eq(2)
    current_valid = current_code.isin([1, 2])
    statement_routing_records.append({'Wave': f'Wave {wave_number}', 'Ever statement Yes': int(ever_yes.sum()),
        'Ever statement No': int(ever_no.sum()), 'Current statement valid': int(current_valid.sum()), 'Current statement valid when ever = Yes': int((current_valid & ever_yes).sum()), 'Current statement valid when ever = No': int((current_valid & ever_no).sum()), 'Ever = Yes but current status unavailable': int((ever_yes & ~current_valid).sum()), 'Current statement Yes': int(current_code.eq(1).sum()), 'Current statement No': int(current_code.eq(2).sum())})
statement_routing_summary = pd.DataFrame(statement_routing_records)
ever_statement_matrix = pd.DataFrame({'Wave 1': statement_valid['W1statedMP'], 'Wave 2': statement_valid['W2statedMP'],
    'Wave 3': statement_valid['W3statedMP']})
current_statement_matrix = pd.DataFrame({'Wave 1': statement_valid['W1stated2MP'],
    'Wave 2': statement_valid['W2stated2MP'], 'Wave 3': statement_valid['W3stated2MP']})
statement_ever_yes = ever_statement_matrix.eq(1).any(axis=1)
statement_ever_valid = ever_statement_matrix.notna().any(axis=1)
latest_current_statement = current_statement_matrix.ffill(axis=1).iloc[:, -1]
latest_current_statement_label = latest_current_statement.map({1.0: 'Currently has a statement of needs',
    2.0: 'Does not currently have a statement of needs'})
statement_sen_comparison = stage_2_statement_of_needs_review[['NSID']].merge(saved_sen_measure_review[['NSID',
    'sen_status_pretransition', 'Pre-transition SEN status']], on='NSID', how='left', validate='one_to_one')
statement_sen_comparison['statement_ever_reported'] = statement_ever_yes.astype('Int64')
statement_sen_comparison['latest_current_statement'] = latest_current_statement
statement_ever_label = statement_ever_yes.map({False: 'No statement reported at Waves 1–3',
    True: 'Statement reported at least once at Waves 1–3'})
selected_sen_label = statement_sen_comparison['Pre-transition SEN status'].astype('string').fillna('Selected SEN status unavailable')
selected_sen_by_statement_ever = pd.crosstab(selected_sen_label, statement_ever_label, margins=True,
    margins_name='Total')
selected_sen_by_latest_statement = pd.crosstab(selected_sen_label,
    latest_current_statement_label.fillna('Current statement status unavailable'), margins=True, margins_name='Total')
statement_overlap_checks = pd.DataFrame([{'Overlap check': 'Statement ever reported Yes',
    'Participants': int(statement_ever_yes.sum())}, {'Overlap check': 'Latest current-statement status available',
    'Participants': int(latest_current_statement.notna().sum())}, {'Overlap check': 'Statement reported but selected SEN shows no identification',
    'Participants': int((statement_ever_yes & statement_sen_comparison['sen_status_pretransition'].eq(0)).sum())}, {'Overlap check': 'Current statement Yes but selected SEN is identified-not-current',
    'Participants': int((latest_current_statement.eq(1) & statement_sen_comparison['sen_status_pretransition'].eq(1)).sum())}, {'Overlap check': 'Selected current SEN but no statement reported at any wave',
    'Participants': int((statement_sen_comparison['sen_status_pretransition'].eq(2) & ~statement_ever_yes).sum())}, {'Overlap check': 'Selected SEN identified and statement ever status unavailable',
    'Participants': int((statement_sen_comparison['sen_status_pretransition'].isin([1,
    2]) & ~statement_ever_valid).sum())}])
statement_response_patterns = pd.DataFrame({'W1 ever': statement_codes['W1statedMP'],
    'W1 current': statement_codes['W1stated2MP'], 'W2 ever': statement_codes['W2statedMP'], 'W2 current': statement_codes['W2stated2MP'], 'W3 ever': statement_codes['W3statedMP'], 'W3 current': statement_codes['W3stated2MP']}).value_counts(dropna=False).rename('Participants').reset_index().sort_values('Participants',
    ascending=False).reset_index(drop=True)
print(f'Statement-of-needs variables reviewed: {len(statement_entries):,}')
print('\nCoverage and code summary:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 410):
    print(statement_coverage_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nStored values and labels:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 130, 'display.width', 330):
    print(statement_value_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nEver/current routing summary:')
print(statement_routing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSelected SEN status by statement ever reported:')
print(selected_sen_by_statement_ever.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nSelected SEN status by latest current-statement status:')
print(selected_sen_by_latest_statement.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nOverlap checks:')
with pd.option_context('display.max_colwidth', 150, 'display.width', 300):
    print(statement_overlap_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMost common six-variable response patterns:')
with pd.option_context('display.max_rows', 15, 'display.width', 280):
    print(statement_response_patterns.head(15).to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Statement-of-needs variables reviewed: 6

Coverage and code summary:
  Wave    Variable                                        Variable label          Timing status Current register status  Source record present  Source record absent  Valid Yes or No  Yes  No  Not applicable  Don't know  Respondent not interviewed  Other negative code
Wave 1  W1statedMP MP: Whether YP has ever been given statement of needs  Pre-transition source          Pending review                   9524                   243              430  418  12            7801          61                         103                    0
Wave 1 W1stated2MP       MP: Whether YP currently has statement of needs  Pre-transition source          Pending review                   9524                   243              402  281 121            9003          16                         103                    0
   ...         ...                                                   ...                    ...                     ...        

In [163]:
# 91: Statement-of-needs construction

ever_statement_updates = pd.DataFrame({'Wave 1': statement_codes['W1statedMP'].where(statement_codes['W1statedMP'].isin([1,
    2, 3])), 'Wave 2': statement_codes['W2statedMP'].where(statement_codes['W2statedMP'].isin([1, 2,
    3])), 'Wave 3': statement_codes['W3statedMP'].where(statement_codes['W3statedMP'].isin([1, 2, 3]))})
current_statement_updates = pd.DataFrame({'Wave 1': statement_codes['W1stated2MP'].where(statement_codes['W1stated2MP'].isin([1,
    2])), 'Wave 2': statement_codes['W2stated2MP'].where(statement_codes['W2stated2MP'].isin([1,
    2])), 'Wave 3': statement_codes['W3stated2MP'].where(statement_codes['W3stated2MP'].isin([1, 2]))})
statement_received_by_wave = pd.DataFrame({'Wave 1': ever_statement_updates[['Wave 1']].eq(1).any(axis=1),
    'Wave 2': ever_statement_updates[['Wave 1',
    'Wave 2']].eq(1).any(axis=1), 'Wave 3': ever_statement_updates[['Wave 1', 'Wave 2', 'Wave 3']].eq(1).any(axis=1)})
statement_received_ever = statement_received_by_wave['Wave 3']
latest_ever_statement_update = ever_statement_updates.ffill(axis=1).iloc[:, -1]
latest_current_statement = current_statement_updates.ffill(axis=1).iloc[:, -1]
corrected_statement_routing_records = []
for wave_number in [1, 2, 3]:
    wave_name = f'Wave {wave_number}'
    current_code = statement_codes[f'W{wave_number}stated2MP']
    statement_received_by_this_wave = statement_received_by_wave[wave_name]
    current_question_reached = current_code.isin([1, 2, -1])
    current_valid = current_code.isin([1, 2])
    corrected_statement_routing_records.append({'Wave': wave_name,
        'Statement received by this wave': int(statement_received_by_this_wave.sum()), 'Current-status question reached': int(current_question_reached.sum()), 'Valid current-status response': int(current_valid.sum()), "Don't know current status": int(current_code.eq(-1).sum()), 'Received statement and question reached': int((statement_received_by_this_wave & current_question_reached).sum()), 'Received statement but question not reached': int((statement_received_by_this_wave & ~current_question_reached).sum()), 'Question reached without a received statement': int((~statement_received_by_this_wave & current_question_reached).sum())})
corrected_statement_routing_summary = pd.DataFrame(corrected_statement_routing_records)
corrected_statement_status = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
corrected_statement_status.loc[~statement_received_ever & latest_ever_statement_update.eq(3)] = 0
corrected_statement_status.loc[~statement_received_ever & latest_ever_statement_update.eq(2)] = 1
corrected_statement_status.loc[statement_received_ever & latest_current_statement.eq(2)] = 2
corrected_statement_status.loc[statement_received_ever & latest_current_statement.eq(1)] = 3
corrected_statement_status_labels = {0: 'No statement of needs received', 1: 'Awaiting a statement of needs',
    2: 'Statement received; not current at latest valid record', 3: 'Statement received; current at latest valid record'}
corrected_statement_status_label = corrected_statement_status.map(corrected_statement_status_labels)
stage_2_corrected_statement_review = stage_2_statement_of_needs_review[['NSID']].merge(saved_sen_measure_review[['NSID',
    'sen_status_pretransition', 'Pre-transition SEN status']], on='NSID', how='left', validate='one_to_one')
stage_2_corrected_statement_review['statement_received_ever'] = statement_received_ever.astype('Int64')
stage_2_corrected_statement_review['latest_ever_statement_update'] = latest_ever_statement_update
stage_2_corrected_statement_review['latest_current_statement'] = latest_current_statement
stage_2_corrected_statement_review['corrected_statement_status'] = corrected_statement_status
stage_2_corrected_statement_review['Corrected statement status'] = corrected_statement_status_label
statement_status_unresolved_reason = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
selected_sen_code_for_statement = pd.to_numeric(stage_2_corrected_statement_review['sen_status_pretransition'],
    errors='coerce')
statement_status_unresolved = corrected_statement_status.isna()
statement_status_unresolved_reason.loc[statement_status_unresolved & selected_sen_code_for_statement.eq(0)] = 'No pre-transition SEN identification; statement questions structurally not applicable'
statement_status_unresolved_reason.loc[statement_status_unresolved & statement_received_ever & latest_current_statement.isna()] = 'Statement received, but current-statement status unavailable'
statement_status_unresolved_reason.loc[statement_status_unresolved & selected_sen_code_for_statement.isin([1,
    2]) & ~statement_received_ever & latest_ever_statement_update.isna()] = 'SEN identified, but no valid statement-history update available'
statement_status_unresolved_reason.loc[statement_status_unresolved & statement_status_unresolved_reason.isna()] = 'Other unresolved statement status'
stage_2_corrected_statement_review['Statement-status unresolved reason'] = statement_status_unresolved_reason
corrected_statement_distribution = corrected_statement_status_label.fillna('Statement status unresolved or not applicable').value_counts().rename_axis('Corrected statement-of-needs status').reset_index(name='Participants')
sen_identified_for_statement = selected_sen_code_for_statement.isin([1, 2])
corrected_statement_distribution_among_sen = corrected_statement_status_label.loc[sen_identified_for_statement].fillna('Statement status unresolved').value_counts().rename_axis('Statement status among SEN-identified participants').reset_index(name='Participants')
selected_sen_label_for_statement = stage_2_corrected_statement_review['Pre-transition SEN status'].astype('string').fillna('Selected SEN status unavailable')
selected_sen_by_corrected_statement = pd.crosstab(selected_sen_label_for_statement,
    corrected_statement_status_label.fillna('Statement status unresolved or not applicable'), margins=True, margins_name='Total')
statement_unresolved_summary = statement_status_unresolved_reason.loc[statement_status_unresolved].value_counts().rename_axis('Unresolved reason').reset_index(name='Participants')
corrected_statement_overlap_checks = pd.DataFrame([{'Overlap check': 'Statement received at any Wave 1–3',
    'Participants': int(statement_received_ever.sum())}, {'Overlap check': 'Awaiting statement at latest explicit update',
    'Participants': int(corrected_statement_status.eq(1).sum())}, {'Overlap check': 'Current statement at latest valid record',
    'Participants': int(corrected_statement_status.eq(3).sum())}, {'Overlap check': 'Statement received but not current at latest valid record',
    'Participants': int(corrected_statement_status.eq(2).sum())}, {'Overlap check': 'Substantive statement status with no SEN identification',
    'Participants': int((corrected_statement_status.notna() & selected_sen_code_for_statement.eq(0)).sum())}, {'Overlap check': 'Current statement but selected SEN identified-not-current',
    'Participants': int((corrected_statement_status.eq(3) & selected_sen_code_for_statement.eq(1)).sum())}, {'Overlap check': 'Selected current SEN without a current statement',
    'Participants': int((selected_sen_code_for_statement.eq(2) & ~corrected_statement_status.eq(3)).sum())}])
corrected_statement_quality_checks = pd.DataFrame([{'Quality check': 'Participants in corrected review',
    'Participants': len(stage_2_corrected_statement_review)}, {'Quality check': 'Statement received but classified as never received or awaiting',
    'Participants': int((statement_received_ever & corrected_statement_status.isin([0,
    1])).sum())}, {'Quality check': 'Current-status response without a received statement',
    'Participants': int((latest_current_statement.notna() & ~statement_received_ever).sum())}, {'Quality check': 'Corrected values outside categories 0–3',
    'Participants': int((corrected_statement_status.notna() & ~corrected_statement_status.isin([0, 1, 2,
    3])).sum())}])
print(f'Participants in corrected statement review: {len(stage_2_corrected_statement_review):,}')
print('\nCorrected routing summary:')
print(corrected_statement_routing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCorrected full-sample distribution:')
with pd.option_context('display.max_colwidth', 150, 'display.width', 300):
    print(corrected_statement_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCorrected distribution among SEN-identified participants:')
with pd.option_context('display.max_colwidth', 150, 'display.width', 300):
    print(corrected_statement_distribution_among_sen.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSelected SEN status by corrected statement status:')
print(selected_sen_by_corrected_statement.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nUnresolved statement status:')
with pd.option_context('display.max_colwidth', 150, 'display.width', 300):
    print(statement_unresolved_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCorrected overlap checks:')
with pd.option_context('display.max_colwidth', 150, 'display.width', 300):
    print(corrected_statement_overlap_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(corrected_statement_quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in corrected statement review: 9,767

Corrected routing summary:
  Wave  Statement received by this wave  Current-status question reached  Valid current-status response  Don't know current status  Received statement and question reached  Received statement but question not reached  Question reached without a received statement
Wave 1                              418                              418                            402                         16                                      418                                            0                                              0
Wave 2                              525                              518                            498                         20                                      518                                            7                                              0
Wave 3                              592                              581                            557                         24

In [164]:
# 92: SEN and statement-of-needs representation options

selected_sen_measure = pd.to_numeric(stage_2_corrected_statement_review['sen_status_pretransition'], errors='coerce')
selected_sen_measure_label = stage_2_corrected_statement_review['Pre-transition SEN status'].astype('string')
current_statement_indicator = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
current_statement_indicator.loc[selected_sen_measure.eq(0)] = 0
current_statement_indicator.loc[selected_sen_measure.isin([1, 2]) & corrected_statement_status.isin([0, 1, 2])] = 0
current_statement_indicator.loc[selected_sen_measure.isin([1, 2]) & corrected_statement_status.eq(3)] = 1
current_statement_indicator_label = current_statement_indicator.map({0: 'No current statement of needs',
    1: 'Current statement of needs'})
statement_sen_conflict = selected_sen_measure.eq(1) & corrected_statement_status.eq(3)
current_sen_statement_unresolved = selected_sen_measure.eq(2) & corrected_statement_status.isna()
refined_sen_statement_status = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='Int64')
refined_sen_statement_status.loc[selected_sen_measure.eq(0)] = 0
refined_sen_statement_status.loc[selected_sen_measure.eq(1) & ~statement_sen_conflict] = 1
refined_sen_statement_status.loc[selected_sen_measure.eq(2) & corrected_statement_status.isin([0, 1, 2])] = 2
refined_sen_statement_status.loc[selected_sen_measure.eq(2) & corrected_statement_status.eq(3)] = 3
refined_sen_statement_labels = {0: 'No pre-transition SEN identification',
    1: 'SEN identified; not current at latest pre-transition record', 2: 'Current SEN; no current statement of needs', 3: 'Current SEN; current statement of needs'}
refined_sen_statement_status_label = refined_sen_statement_status.map(refined_sen_statement_labels)
refined_sen_statement_missing_reason = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
refined_measure_missing = refined_sen_statement_status.isna()
refined_sen_statement_missing_reason.loc[refined_measure_missing & selected_sen_measure.isna()] = 'Selected pre-transition SEN status unavailable'
refined_sen_statement_missing_reason.loc[refined_measure_missing & statement_sen_conflict] = 'Selected SEN status is not current, but current statement is recorded'
refined_sen_statement_missing_reason.loc[refined_measure_missing & current_sen_statement_unresolved] = 'Current SEN recorded, but current-statement status is unresolved'
refined_sen_statement_missing_reason.loc[refined_measure_missing & refined_sen_statement_missing_reason.isna()] = 'Other unresolved combined status'
stage_2_sen_statement_option_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'selected_sen_status': selected_sen_measure, 'Selected SEN status': selected_sen_measure_label, 'corrected_statement_status': corrected_statement_status, 'Corrected statement status': corrected_statement_status_label, 'current_statement_indicator': current_statement_indicator, 'Current statement indicator': current_statement_indicator_label, 'refined_sen_statement_status': refined_sen_statement_status, 'Refined SEN-statement status': refined_sen_statement_status_label, 'Statement-SEN conflict': statement_sen_conflict, 'Refined-status missing reason': refined_sen_statement_missing_reason})
existing_sen_distribution = selected_sen_measure_label.fillna('Selected SEN status unavailable').value_counts().rename_axis('Existing three-category SEN measure').reset_index(name='Participants')
current_statement_indicator_distribution = current_statement_indicator_label.fillna('Current-statement status unavailable').value_counts().rename_axis('Separate current-statement indicator').reset_index(name='Participants')
refined_sen_statement_distribution = refined_sen_statement_status_label.fillna('Refined SEN-statement status unavailable').value_counts().rename_axis('Refined four-category SEN-statement measure').reset_index(name='Participants')
sen_statement_option_coverage = pd.DataFrame([{'Representation option': 'Existing three-category SEN measure',
    'Valid participants': int(selected_sen_measure.notna().sum()), 'Missing participants': int(selected_sen_measure.isna().sum()), 'Coverage percentage': selected_sen_measure.notna().mean() * 100}, {'Representation option': 'Separate current-statement indicator',
    'Valid participants': int(current_statement_indicator.notna().sum()), 'Missing participants': int(current_statement_indicator.isna().sum()), 'Coverage percentage': current_statement_indicator.notna().mean() * 100}, {'Representation option': 'Refined four-category SEN-statement measure',
    'Valid participants': int(refined_sen_statement_status.notna().sum()), 'Missing participants': int(refined_sen_statement_status.isna().sum()), 'Coverage percentage': refined_sen_statement_status.notna().mean() * 100}])
existing_sen_by_current_statement = pd.crosstab(selected_sen_measure_label.fillna('Selected SEN status unavailable'),
    current_statement_indicator_label.fillna('Current-statement status unavailable'), margins=True, margins_name='Total')
existing_sen_by_refined_status = pd.crosstab(selected_sen_measure_label.fillna('Selected SEN status unavailable'),
    refined_sen_statement_status_label.fillna('Refined SEN-statement status unavailable'), margins=True, margins_name='Total')
sen_statement_problem_summary = pd.DataFrame([{'Review issue': 'Selected SEN status unavailable',
    'Participants': int(selected_sen_measure.isna().sum())}, {'Review issue': 'Not-current SEN status with a current statement',
    'Participants': int(statement_sen_conflict.sum())}, {'Review issue': 'Current SEN with unresolved statement status',
    'Participants': int(current_sen_statement_unresolved.sum())}])
refined_missing_reason_summary = refined_sen_statement_missing_reason.loc[refined_measure_missing].value_counts().rename_axis('Refined-measure missing reason').reset_index(name='Participants')
sen_statement_option_quality_checks = pd.DataFrame([{'Quality check': 'Participants in option review',
    'Participants': len(stage_2_sen_statement_option_review)}, {'Quality check': 'No-SEN cases with current statement',
    'Participants': int((selected_sen_measure.eq(0) & current_statement_indicator.eq(1)).sum())}, {'Quality check': 'Refined current-statement category without current SEN',
    'Participants': int((refined_sen_statement_status.eq(3) & ~selected_sen_measure.eq(2)).sum())}, {'Quality check': 'Refined values outside categories 0–3',
    'Participants': int((refined_sen_statement_status.notna() & ~refined_sen_statement_status.isin([0, 1, 2,
    3])).sum())}])
print(f'Participants in SEN-statement option review: {len(stage_2_sen_statement_option_review):,}')
print('\nExisting SEN measure distribution:')
print(existing_sen_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSeparate current-statement indicator:')
print(current_statement_indicator_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nRefined SEN-statement distribution:')
with pd.option_context('display.max_colwidth', 150, 'display.width', 300):
    print(refined_sen_statement_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCoverage comparison:')
print(sen_statement_option_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Coverage percentage': lambda value: f'{value:.2f}%'}))
print('\nExisting SEN status by current-statement indicator:')
print(existing_sen_by_current_statement.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nExisting SEN status by refined status:')
print(existing_sen_by_refined_status.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nInconsistent or unresolved cases:')
print(sen_statement_problem_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nRefined-measure missing reasons:')
print(refined_missing_reason_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(sen_statement_option_quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in SEN-statement option review: 9,767

Existing SEN measure distribution:
                        Existing three-category SEN measure  Participants
                       No pre-transition SEN identification          7828
SEN identified; not current at latest pre-transition record          1035
    SEN identified; current at latest pre-transition record           779
                            Selected SEN status unavailable           125

Separate current-statement indicator:
Separate current-statement indicator  Participants
       No current statement of needs          9339
          Current statement of needs           286
Current-statement status unavailable           142

Refined SEN-statement distribution:
                Refined four-category SEN-statement measure  Participants
                       No pre-transition SEN identification          7828
SEN identified; not current at latest pre-transition record          1021
                 Current SEN; no current 

In [165]:
# 93: Statement-of-needs predictor decision

statement_predictor_name = 'current_statement_of_needs_pretransition'
statement_predictor_decision = 'Retain as a separate binary predictor alongside the selected three-category pre-transition SEN measure'
statement_predictor_reason = 'A current statement of needs represents formal educational support and is not equivalent to current SEN status. Among participants with current SEN, some had a current statement and others did not. Keeping the indicator separate preserves this distinction without replacing the existing SEN measure with a more complex combined categorisation.'
stage_2_statement_predictor_review = stage_2_sen_statement_option_review[['NSID', 'selected_sen_status',
    'Selected SEN status', 'corrected_statement_status', 'Corrected statement status', 'current_statement_indicator', 'Current statement indicator', 'Statement-SEN conflict']].copy().rename(columns={'current_statement_indicator': statement_predictor_name,
    'Current statement indicator': 'Pre-transition current statement of needs'})
stage_2_statement_predictor_review['Predictor decision'] = statement_predictor_decision
stage_2_statement_predictor_review['Decision reason'] = statement_predictor_reason
statement_predictor_missing = stage_2_statement_predictor_review[statement_predictor_name].isna()
selected_sen_for_statement = pd.to_numeric(stage_2_statement_predictor_review['selected_sen_status'], errors='coerce')
statement_predictor_missing_reason = pd.Series(pd.NA, index=stage_2_statement_predictor_review.index, dtype='string')
statement_predictor_missing_reason.loc[statement_predictor_missing & selected_sen_for_statement.isna()] = 'Pre-transition SEN status unavailable'
statement_predictor_missing_reason.loc[statement_predictor_missing & selected_sen_for_statement.eq(2)] = 'Current SEN recorded, but current-statement status unavailable'
statement_predictor_missing_reason.loc[statement_predictor_missing & selected_sen_for_statement.eq(1)] = 'SEN identified but current-statement status unavailable'
statement_predictor_missing_reason.loc[statement_predictor_missing & statement_predictor_missing_reason.isna()] = 'Other unresolved statement status'
stage_2_statement_predictor_review['Statement-predictor missing reason'] = statement_predictor_missing_reason
statement_source_decisions = [{'Wave': 'Wave 1', 'Variable': 'W1statedMP',
    'Reference period': 'Whether the young person had received or was awaiting a statement of needs by Wave 1.', 'Review note': 'Used to establish statement history and distinguish no statement from structural non-applicability.'}, {'Wave': 'Wave 1',
    'Variable': 'W1stated2MP', 'Reference period': 'Whether the young person currently had a statement of needs at Wave 1.', 'Review note': 'Used as the earliest current-statement status source.'}, {'Wave': 'Wave 2',
    'Variable': 'W2statedMP', 'Reference period': 'Whether the young person had received or was awaiting a statement of needs by Wave 2.', 'Review note': 'Used to update cumulative statement history.'}, {'Wave': 'Wave 2',
    'Variable': 'W2stated2MP', 'Reference period': 'Whether the young person currently had a statement of needs at Wave 2.', 'Review note': 'Used to update current-statement status.'}, {'Wave': 'Wave 3',
    'Variable': 'W3statedMP', 'Reference period': 'Whether the young person had received or was awaiting a statement of needs by Wave 3.', 'Review note': 'Near-transition update to cumulative statement history.'}, {'Wave': 'Wave 3',
    'Variable': 'W3stated2MP', 'Reference period': 'Whether the young person currently had a statement of needs at Wave 3.', 'Review note': 'Used as the latest available pre-transition current-statement status source.'}]
statement_decision_indices = []
for rule in statement_source_decisions:
    matching_rows = stage_2_variable_decision_register['Wave'].eq(rule['Wave']) & stage_2_variable_decision_register['Source type'].eq('Young person') & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {rule['Wave']}, {rule['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    statement_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = 'Retain as construction source'
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'SEN and formal educational support'
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = statement_predictor_reason
    stage_2_variable_decision_register.loc[matching_rows, 'Reference-period assessment'] = rule['Reference period']
    if rule['Wave'] in ['Wave 1', 'Wave 2']:
        leakage_assessment = 'No direct outcome leakage identified. The measure was collected before transition.'
    else:
        leakage_assessment = 'No direct outcome leakage identified. The measure was collected near transition but records formal SEN support before the age-18 outcome.'
    stage_2_variable_decision_register.loc[matching_rows, 'Leakage assessment'] = leakage_assessment
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave 1–Wave 3 young-person data dictionaries and Stage 2 statement-of-needs routing review'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = rule['Review note']
statement_predictor_review_output_path = stage_2_output_directory / 'stage_2_statement_of_needs_predictor_review.csv'
stage_2_statement_predictor_review.to_csv(statement_predictor_review_output_path, index=False)
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
statement_source_decision_output = stage_2_variable_decision_register.loc[statement_decision_indices, ['Source order',
    'Wave', 'Variable position', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Reference-period assessment', 'Leakage assessment']].sort_values(['Source order',
    'Variable position']).drop(columns=['Source order', 'Variable position']).reset_index(drop=True)
statement_predictor_distribution = stage_2_statement_predictor_review['Pre-transition current statement of needs'].fillna('Current-statement status unavailable').value_counts().rename_axis('Pre-transition current statement of needs').reset_index(name='Participants')
statement_predictor_missing_summary = statement_predictor_missing_reason.loc[statement_predictor_missing].value_counts().rename_axis('Missing reason').reset_index(name='Participants')
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
statement_decision_file_checks = pd.DataFrame([{'Output file': 'Variable decision register',
    'Path': str(decision_register_output_path), 'Exists': decision_register_output_path.exists()}, {'Output file': 'Statement-of-needs predictor review',
    'Path': str(statement_predictor_review_output_path), 'Exists': statement_predictor_review_output_path.exists()}])
print(f'Statement-of-needs source variables decided: {len(statement_source_decision_output):,}')
print(f'\nSelected predictor: {statement_predictor_name}')
print('\nPredictor distribution:')
print(statement_predictor_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMissing-value reasons:')
print(statement_predictor_missing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables retained as construction sources:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 160, 'display.width', 430):
    print(statement_source_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nOutput-file checks:')
with pd.option_context('display.max_colwidth', 180, 'display.width', 320):
    print(statement_decision_file_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Statement-of-needs source variables decided: 6

Selected predictor: current_statement_of_needs_pretransition

Predictor distribution:
Pre-transition current statement of needs  Participants
            No current statement of needs          9339
               Current statement of needs           286
     Current-statement status unavailable           142

Missing-value reasons:
                                                Missing reason  Participants
                         Pre-transition SEN status unavailable           125
Current SEN recorded, but current-statement status unavailable            14
       SEN identified but current-statement status unavailable             3

Variables retained as construction sources:
  Wave    Variable                                        Variable label                Review outcome                                                                                                                                                                   

In [166]:
# 94: Young-person general-health measure structure

yp_general_health_name_pattern = '(?i)^W[1-4]hea1cyp$'
yp_general_health_label_pattern = '(?i)\\bYP\\b.*(?:quality|general).*health.*(?:last 12 months|past 12 months)'
yp_general_health_entries = stage_2_master_variable_register.loc[stage_2_master_variable_register['Source type'].eq('Young person') & (stage_2_master_variable_register['Variable'].astype('string').str.contains(yp_general_health_name_pattern,
    na=False, regex=True) | stage_2_master_variable_register['Variable label'].astype('string').str.contains(yp_general_health_label_pattern,
    na=False, regex=True)), ['Source order', 'Wave', 'Source type', 'Source file', 'Source path', 'Variable position',
    'Variable', 'Variable label', 'Timing status']].copy().drop_duplicates(subset=['Source file',
    'Variable']).sort_values(['Source order', 'Variable position']).reset_index(drop=True)
if yp_general_health_entries.empty:
    raise ValueError('No comparable young-person general-health variables were found.')
yp_general_health_entries = yp_general_health_entries.merge(stage_2_variable_decision_register[['Source file',
    'Variable', 'Review outcome']], on=['Source file', 'Variable'], how='left', validate='one_to_one')
stage_2_yp_general_health_review = stage_2_participant_ids.copy()
general_health_structure_records = []
general_health_value_records = []
for source_path_text, source_entries in yp_general_health_entries.groupby('Source path', sort=False):
    source_path = Path(source_path_text)
    source_variables = source_entries['Variable'].tolist()
    numeric_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    numeric_sample = stage_2_participant_ids.merge(numeric_source, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = numeric_sample['_merge'].eq('both')
    numeric_sample = numeric_sample.drop(columns='_merge')
    labelled_sample = stage_2_participant_ids.merge(labelled_source, on='NSID', how='left', validate='one_to_one')
    for _, entry in source_entries.iterrows():
        variable = entry['Variable']
        numeric_values = pd.to_numeric(numeric_sample[variable], errors='coerce')
        labelled_values = labelled_sample[variable].astype('string')
        stage_2_yp_general_health_review[f'{variable} code'] = numeric_values
        stage_2_yp_general_health_review[f'{variable} label'] = labelled_values
        stage_2_yp_general_health_review[f'{variable} source present'] = source_record_present
        valid_values = numeric_values.ge(0)
        general_health_structure_records.append({'Wave': entry['Wave'], 'Variable': variable,
            'Variable label': entry['Variable label'], 'Timing status': entry['Timing status'], 'Current register status': entry['Review outcome'], 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Non-negative stored value': int(valid_values.sum()), 'Negative stored value': int(numeric_values.lt(0).sum()), 'Missing within source record': int((source_record_present & numeric_values.isna()).sum()), 'Unique non-negative values': int(numeric_values.loc[valid_values].nunique()), 'Minimum non-negative value': numeric_values.loc[valid_values].min() if valid_values.any() else np.nan, 'Maximum non-negative value': numeric_values.loc[valid_values].max() if valid_values.any() else np.nan})
        value_summary = pd.DataFrame({'Value code': numeric_values, 'Value label': labelled_values,
            'Source record present': source_record_present}).loc[lambda frame: frame['Source record present']].drop(columns='Source record present').value_counts(dropna=False).rename('Participants').reset_index()
        value_summary.insert(0, 'Wave', entry['Wave'])
        value_summary.insert(1, 'Variable', variable)
        general_health_value_records.append(value_summary)
yp_general_health_structure_summary = pd.DataFrame(general_health_structure_records)
yp_general_health_value_summary = pd.concat(general_health_value_records, ignore_index=True)
general_health_pairwise_records = []
general_health_variables = yp_general_health_entries['Variable'].tolist()
for first_position in range(len(general_health_variables)):
    for second_position in range(first_position + 1, len(general_health_variables)):
        first_variable = general_health_variables[first_position]
        second_variable = general_health_variables[second_position]
        first_code = pd.to_numeric(stage_2_yp_general_health_review[f'{first_variable} code'], errors='coerce')
        second_code = pd.to_numeric(stage_2_yp_general_health_review[f'{second_variable} code'], errors='coerce')
        first_label = stage_2_yp_general_health_review[f'{first_variable} label'].astype('string').str.strip().str.lower()
        second_label = stage_2_yp_general_health_review[f'{second_variable} label'].astype('string').str.strip().str.lower()
        both_valid = first_code.ge(0) & second_code.ge(0)
        code_agreement = both_valid & first_code.eq(second_code)
        label_agreement = both_valid & first_label.eq(second_label)
        first_wave = yp_general_health_entries.loc[yp_general_health_entries['Variable'].eq(first_variable),
            'Wave'].iloc[0]
        second_wave = yp_general_health_entries.loc[yp_general_health_entries['Variable'].eq(second_variable),
            'Wave'].iloc[0]
        general_health_pairwise_records.append({'First wave': first_wave, 'First variable': first_variable,
            'Second wave': second_wave, 'Second variable': second_variable, 'Both responses valid': int(both_valid.sum()), 'Exact code agreement': int(code_agreement.sum()), 'Exact code agreement percentage': code_agreement.sum() / both_valid.sum() * 100 if both_valid.sum() else np.nan, 'Exact label agreement': int(label_agreement.sum()), 'Exact label agreement percentage': label_agreement.sum() / both_valid.sum() * 100 if both_valid.sum() else np.nan})
yp_general_health_pairwise_summary = pd.DataFrame(general_health_pairwise_records)
yp_general_health_negative_values = yp_general_health_value_summary.loc[pd.to_numeric(yp_general_health_value_summary['Value code'],
    errors='coerce').lt(0) | yp_general_health_value_summary['Value code'].isna()].copy().reset_index(drop=True)
yp_general_health_substantive_values = yp_general_health_value_summary.loc[pd.to_numeric(yp_general_health_value_summary['Value code'],
    errors='coerce').ge(0)].copy().reset_index(drop=True)
yp_general_health_checks = pd.DataFrame([{'Check': 'Comparable variables found',
    'Result': len(yp_general_health_entries)}, {'Check': 'Duplicate source-variable entries',
    'Result': int(yp_general_health_entries[['Source file',
    'Variable']].duplicated().sum())}, {'Check': 'Variables without register status',
    'Result': int(yp_general_health_entries['Review outcome'].isna().sum())}])
print(f'Young-person general-health variables reviewed: {len(yp_general_health_entries):,}')
print('\nSource entries:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 160, 'display.width', 380):
    print(yp_general_health_entries[['Wave', 'Source type', 'Variable', 'Variable label', 'Timing status',
        'Review outcome']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCoverage and stored-value structure:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 420):
    print(yp_general_health_structure_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSubstantive values and labels:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 130, 'display.width', 330):
    print(yp_general_health_substantive_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nNegative and missing values:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 130, 'display.width', 330):
    print(yp_general_health_negative_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nPairwise wave comparison:')
with pd.option_context('display.max_rows', None, 'display.width', 360):
    print(yp_general_health_pairwise_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
        formatters={'Exact code agreement percentage': lambda value: f'{value:.2f}%' if pd.notna(value) else 'Not available',
        'Exact label agreement percentage': lambda value: f'{value:.2f}%' if pd.notna(value) else 'Not available'}))
print('\nReview checks:')
print(yp_general_health_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Young-person general-health variables reviewed: 3

Source entries:
  Wave  Source type  Variable                                 Variable label          Timing status Review outcome
Wave 2 Young person W2hea1cYP   YP: Quality of YP's health in last 12 months  Pre-transition source Pending review
Wave 3 Young person W3hea1cYP   YP: Quality of YP's health in last 12 months Near-transition source Pending review
Wave 4 Young person W4Hea1CYP YP: General health of YP in the last 12 months At or after transition Pending review

Coverage and stored-value structure:
  Wave  Variable                                 Variable label          Timing status Current register status  Source record present  Source record absent  Non-negative stored value  Negative stored value  Missing within source record  Unique non-negative values  Minimum non-negative value  Maximum non-negative value
Wave 2 W2hea1cYP   YP: Quality of YP's health in last 12 months  Pre-transition source          Pending review     

In [167]:
# 95: Young-person general-health harmonisation

general_health_category_map = {'very good': 1, 'fairly good': 2, 'not very good': 3, 'not good at all': 4}
general_health_category_labels = {1: 'Very good', 2: 'Fairly good', 3: 'Not very good', 4: 'Not good at all'}
w2_general_health_label = stage_2_yp_general_health_review['W2hea1cYP label'].astype('string').str.strip().str.lower()
w3_general_health_label = stage_2_yp_general_health_review['W3hea1cYP label'].astype('string').str.strip().str.lower()
w4_general_health_label = stage_2_yp_general_health_review['W4Hea1CYP label'].astype('string').str.strip().str.lower()
w2_general_health_harmonised = w2_general_health_label.map(general_health_category_map).astype('Int64')
w3_general_health_harmonised = w3_general_health_label.map(general_health_category_map).astype('Int64')
w4_general_health_harmonised = w4_general_health_label.map(general_health_category_map).astype('Int64')
pretransition_general_health = w3_general_health_harmonised.copy()
general_health_source = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
general_health_source.loc[w3_general_health_harmonised.notna()] = 'Wave 3'
wave2_general_health_fallback = pretransition_general_health.isna() & w2_general_health_harmonised.notna()
pretransition_general_health.loc[wave2_general_health_fallback] = w2_general_health_harmonised.loc[wave2_general_health_fallback]
general_health_source.loc[wave2_general_health_fallback] = 'Wave 2 fallback'
pretransition_general_health_label = pretransition_general_health.map(general_health_category_labels)
w2_w3_general_health_both_valid = w2_general_health_harmonised.notna() & w3_general_health_harmonised.notna()
w2_w3_general_health_difference = w3_general_health_harmonised - w2_general_health_harmonised
w2_w3_general_health_absolute_difference = w2_w3_general_health_difference.abs()
general_health_cross_wave_summary = pd.DataFrame([{'Comparison': 'Wave 2 and Wave 3 both valid',
    'Participants': int(w2_w3_general_health_both_valid.sum())}, {'Comparison': 'Exact category agreement',
    'Participants': int((w2_w3_general_health_both_valid & w2_general_health_harmonised.eq(w3_general_health_harmonised)).sum())}, {'Comparison': 'One-category difference',
    'Participants': int((w2_w3_general_health_both_valid & w2_w3_general_health_absolute_difference.eq(1)).sum())}, {'Comparison': 'Two-category difference',
    'Participants': int((w2_w3_general_health_both_valid & w2_w3_general_health_absolute_difference.eq(2)).sum())}, {'Comparison': 'Three-category difference',
    'Participants': int((w2_w3_general_health_both_valid & w2_w3_general_health_absolute_difference.eq(3)).sum())}, {'Comparison': 'Wave 3 health worse than Wave 2',
    'Participants': int((w2_w3_general_health_both_valid & w2_w3_general_health_difference.gt(0)).sum())}, {'Comparison': 'Wave 3 health better than Wave 2',
    'Participants': int((w2_w3_general_health_both_valid & w2_w3_general_health_difference.lt(0)).sum())}])
general_health_cross_tabulation = pd.crosstab(w2_general_health_harmonised.map(general_health_category_labels),
    w3_general_health_harmonised.map(general_health_category_labels), margins=True, margins_name='Total')
stage_2_yp_general_health_harmonised_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'W2 general health': w2_general_health_harmonised, 'W3 general health': w3_general_health_harmonised, 'W4 general health - timing review only': w4_general_health_harmonised, 'pretransition_general_health': pretransition_general_health, 'Pre-transition general health': pretransition_general_health_label, 'Construction source': general_health_source, 'W2-W3 category difference': w2_w3_general_health_difference})
pretransition_general_health_distribution = pretransition_general_health_label.fillna('General-health status unavailable').value_counts().rename_axis('Pre-transition general health').reset_index(name='Participants')
general_health_source_summary = general_health_source.fillna('No valid Wave 2 or Wave 3 response').value_counts().rename_axis('Construction source').reset_index(name='Participants')
general_health_timing_comparison = pd.DataFrame([{'Measure': 'Wave 2 valid response',
    'Participants': int(w2_general_health_harmonised.notna().sum()), 'Predictor-use status': 'Eligible pre-transition source'}, {'Measure': 'Wave 3 valid response',
    'Participants': int(w3_general_health_harmonised.notna().sum()), 'Predictor-use status': 'Eligible latest pre-transition source'}, {'Measure': 'Wave 4 valid response',
    'Participants': int(w4_general_health_harmonised.notna().sum()), 'Predictor-use status': 'Timing review only; reference period extends into or beyond transition'}])
general_health_quality_checks = pd.DataFrame([{'Quality check': 'Participants in review',
    'Participants': len(stage_2_yp_general_health_harmonised_review)}, {'Quality check': 'Constructed general-health values',
    'Participants': int(pretransition_general_health.notna().sum())}, {'Quality check': 'Wave 3 primary sources',
    'Participants': int(general_health_source.eq('Wave 3').sum())}, {'Quality check': 'Wave 2 fallback sources',
    'Participants': int(general_health_source.eq('Wave 2 fallback').sum())}, {'Quality check': 'Values outside categories 1–4',
    'Participants': int((pretransition_general_health.notna() & ~pretransition_general_health.isin([1, 2, 3,
    4])).sum())}])
print(f'Participants in general-health construction review: {len(stage_2_yp_general_health_harmonised_review):,}')
print('\nConstructed pre-transition distribution:')
print(pretransition_general_health_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nConstruction source:')
print(general_health_source_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 2–Wave 3 harmonised comparison:')
print(general_health_cross_wave_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 2 by Wave 3 harmonised category:')
print(general_health_cross_tabulation.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nTiming comparison:')
with pd.option_context('display.max_colwidth', 150, 'display.width', 320):
    print(general_health_timing_comparison.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(general_health_quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in general-health construction review: 9,767

Constructed pre-transition distribution:
    Pre-transition general health  Participants
                        Very good          5940
                      Fairly good          3286
General-health status unavailable           264
                    Not very good           241
                  Not good at all            36

Construction source:
               Construction source  Participants
                            Wave 3          9445
No valid Wave 2 or Wave 3 response           264
                   Wave 2 fallback            58

Wave 2–Wave 3 harmonised comparison:
                      Comparison  Participants
    Wave 2 and Wave 3 both valid          8842
        Exact category agreement          5117
                             ...           ...
 Wave 3 health worse than Wave 2          1016
Wave 3 health better than Wave 2          2709

Wave 2 by Wave 3 harmonised category:
W3hea1cYP label  Fairly good  Not g

In [168]:
# 96: Wave 3 general-health timing linkage

wave3_timing_variable_names = ['W3intmnthMP', 'W3intyearMP']
wave3_timing_entries = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].eq('Wave 3') & stage_2_master_variable_register['Source type'].isin(['Family background',
    'Parental attitudes']) & stage_2_master_variable_register['Variable'].str.lower().isin([variable.lower() for variable in wave3_timing_variable_names]), ['Source order',
    'Wave', 'Source type', 'Source file', 'Source path', 'Variable position', 'Variable', 'Variable label']].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
if len(wave3_timing_entries) != 4:
    raise ValueError(f'Expected four Wave 3 timing entries across the family-background and parental-attitudes files, but found {len(wave3_timing_entries)}.')
wave3_timing_sources = {}
for source_type, source_entries in wave3_timing_entries.groupby('Source type', sort=False):
    source_path = Path(source_entries['Source path'].iloc[0])
    source_data = pd.read_stata(source_path, columns=['NSID', *wave3_timing_variable_names],
        convert_categoricals=False)
    source_sample = stage_2_participant_ids.merge(source_data, on='NSID', how='left', validate='one_to_one')
    wave3_timing_sources[source_type] = source_sample
family_timing = wave3_timing_sources['Family background']
parental_timing = wave3_timing_sources['Parental attitudes']
timing_copy_comparison_records = []
for variable in wave3_timing_variable_names:
    family_values = pd.to_numeric(family_timing[variable], errors='coerce')
    parental_values = pd.to_numeric(parental_timing[variable], errors='coerce')
    both_stored = family_values.notna() & parental_values.notna()
    both_valid = family_values.ge(0) & parental_values.ge(0)
    timing_copy_comparison_records.append({'Variable': variable, 'Both copies stored': int(both_stored.sum()),
        'Stored values identical': int((both_stored & family_values.eq(parental_values)).sum()), 'Both copies valid': int(both_valid.sum()), 'Valid values identical': int((both_valid & family_values.eq(parental_values)).sum()), 'Valid-value disagreements': int((both_valid & family_values.ne(parental_values)).sum())})
timing_copy_comparison = pd.DataFrame(timing_copy_comparison_records)
w3_interview_month = pd.to_numeric(family_timing['W3intmnthMP'],
    errors='coerce').where(lambda values: values.between(1, 12, inclusive='both')).astype('Int64')
w3_interview_year = pd.to_numeric(family_timing['W3intyearMP'],
    errors='coerce').where(lambda values: values.between(1900, 2100, inclusive='both')).astype('Int64')
month_labels = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June', 7: 'July', 8: 'August',
    9: 'September', 10: 'October', 11: 'November', 12: 'December'}
w3_interview_month_label = w3_interview_month.map(month_labels)
stage_2_w3_health_timing_review = stage_2_yp_general_health_harmonised_review.merge(pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'W3 interview month': w3_interview_month, 'W3 interview month label': w3_interview_month_label, 'W3 interview year': w3_interview_year}), on='NSID', how='left', validate='one_to_one')
w3_health_valid = stage_2_w3_health_timing_review['W3 general health'].notna()
w2_health_valid = stage_2_w3_health_timing_review['W2 general health'].notna()
valid_w3_timing = (stage_2_w3_health_timing_review['W3 interview year'].eq(2006) & stage_2_w3_health_timing_review['W3 interview month'].between(1,
    12, inclusive='both')).fillna(False).astype(bool)
w3_timing_interpretation = pd.Series('Timing unavailable', index=stage_2_w3_health_timing_review.index,
    dtype='string')
w3_timing_interpretation.loc[valid_w3_timing & stage_2_w3_health_timing_review['W3 interview month'].le(8)] = 'March–August 2006'
w3_timing_interpretation.loc[valid_w3_timing & stage_2_w3_health_timing_review['W3 interview month'].eq(9)] = 'September 2006; interview day unavailable'
stage_2_w3_health_timing_review['Timing interpretation'] = w3_timing_interpretation
w3_health_timing_month_summary = stage_2_w3_health_timing_review.loc[valid_w3_timing].groupby(['W3 interview year',
    'W3 interview month', 'W3 interview month label'], dropna=False).agg(Participants=('NSID', 'size'),
    Valid_W3_general_health=('W3 general health',
    lambda values: int(values.notna().sum())), Valid_W2_general_health=('W2 general health',
    lambda values: int(values.notna().sum()))).reset_index().sort_values(['W3 interview year',
    'W3 interview month']).reset_index(drop=True)
september_cases = valid_w3_timing & stage_2_w3_health_timing_review['W3 interview month'].eq(9)
september_health_summary = pd.DataFrame([{'September timing check': 'Participants interviewed in September 2006',
    'Participants': int(september_cases.sum())}, {'September timing check': 'September participants with valid Wave 3 general health',
    'Participants': int((september_cases & w3_health_valid).sum())}, {'September timing check': 'September participants with valid Wave 2 general health',
    'Participants': int((september_cases & w2_health_valid).sum())}, {'September timing check': 'September participants with valid Wave 3 health but no Wave 2 fallback',
    'Participants': int((september_cases & w3_health_valid & ~w2_health_valid).sum())}])
w3_health_by_timing_group = w3_timing_interpretation.loc[w3_health_valid].value_counts().rename_axis('Timing interpretation among valid Wave 3 health responses').reset_index(name='Participants')
w3_health_timing_quality_checks = pd.DataFrame([{'Quality check': 'Participants in linked timing review',
    'Participants': len(stage_2_w3_health_timing_review)}, {'Quality check': 'Valid 2006 interview month and year',
    'Participants': int(valid_w3_timing.sum())}, {'Quality check': 'Valid Wave 3 health with timing unavailable',
    'Participants': int((w3_health_valid & ~valid_w3_timing).sum())}, {'Quality check': 'Interview years other than 2006',
    'Participants': int((w3_interview_year.notna() & ~w3_interview_year.eq(2006)).sum())}, {'Quality check': 'Duplicated NSID after timing linkage',
    'Participants': int(stage_2_w3_health_timing_review['NSID'].duplicated().sum())}])
print(f'Participants in Wave 3 health-timing review: {len(stage_2_w3_health_timing_review):,}')
print('\nDuplicate timing-variable comparison:')
print(timing_copy_comparison.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nInterview timing and health coverage by month:')
print(w3_health_timing_month_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSeptember timing review:')
print(september_health_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nTiming among valid Wave 3 health responses:')
with pd.option_context('display.max_colwidth', 150, 'display.width', 280):
    print(w3_health_by_timing_group.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(w3_health_timing_quality_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in Wave 3 health-timing review: 9,767

Duplicate timing-variable comparison:
   Variable  Both copies stored  Stored values identical  Both copies valid  Valid values identical  Valid-value disagreements
W3intmnthMP                9509                     9509               9381                    9381                          0
W3intyearMP                9509                     9509               9381                    9381                          0

Interview timing and health coverage by month:
 W3 interview year  W3 interview month W3 interview month label  Participants  Valid_W3_general_health  Valid_W2_general_health
              2006                   3                    March           435                      435                      397
              2006                   4                    April          4267                     4245                     4011
               ...                 ...                      ...           ...                    

In [169]:
# 97: Wave 3 general-health timing comparison

health_timing_review = stage_2_w3_health_timing_review.copy().reset_index(drop=True)
raw_family_timing = family_timing[['NSID', 'W3intmnthMP',
    'W3intyearMP']].copy().rename(columns={'W3intmnthMP': 'Raw W3 interview month',
    'W3intyearMP': 'Raw W3 interview year'})
health_timing_review = health_timing_review.merge(raw_family_timing, on='NSID', how='left', validate='one_to_one')
w2_health_review = pd.to_numeric(health_timing_review['W2 general health'], errors='coerce').astype('Int64')
w3_health_review = pd.to_numeric(health_timing_review['W3 general health'], errors='coerce').astype('Int64')
standard_general_health_review = pd.to_numeric(health_timing_review['pretransition_general_health'],
    errors='coerce').astype('Int64')
w3_month_review = pd.to_numeric(health_timing_review['W3 interview month'], errors='coerce').astype('Int64')
w3_year_review = pd.to_numeric(health_timing_review['W3 interview year'], errors='coerce').astype('Int64')
w2_health_available = w2_health_review.notna()
w3_health_available = w3_health_review.notna()
w3_year_is_2006 = w3_year_review.eq(2006).fillna(False).astype(bool)
w3_month_march_to_august = w3_month_review.between(3, 8, inclusive='both').fillna(False).astype(bool)
w3_month_september = w3_month_review.eq(9).fillna(False).astype(bool)
clear_pretransition_w3_timing = w3_year_is_2006 & w3_month_march_to_august
september_w3_timing = w3_year_is_2006 & w3_month_september
unavailable_w3_timing = w3_health_available & ~(clear_pretransition_w3_timing | september_w3_timing)
recalculated_w3_timing_summary = pd.DataFrame([{'Wave 3 health-timing group': 'March–August 2006',
    'Valid Wave 3 health': int((w3_health_available & clear_pretransition_w3_timing).sum()), 'Wave 2 health also available': int((w3_health_available & clear_pretransition_w3_timing & w2_health_available).sum()), 'Wave 2 health unavailable': int((w3_health_available & clear_pretransition_w3_timing & ~w2_health_available).sum())}, {'Wave 3 health-timing group': 'September 2006',
    'Valid Wave 3 health': int((w3_health_available & september_w3_timing).sum()), 'Wave 2 health also available': int((w3_health_available & september_w3_timing & w2_health_available).sum()), 'Wave 2 health unavailable': int((w3_health_available & september_w3_timing & ~w2_health_available).sum())}, {'Wave 3 health-timing group': 'Timing unavailable',
    'Valid Wave 3 health': int(unavailable_w3_timing.sum()), 'Wave 2 health also available': int((unavailable_w3_timing & w2_health_available).sum()), 'Wave 2 health unavailable': int((unavailable_w3_timing & ~w2_health_available).sum())}])
unavailable_timing_code_summary = health_timing_review.loc[unavailable_w3_timing, ['Raw W3 interview month',
    'Raw W3 interview year']].value_counts(dropna=False).rename('Participants').reset_index()
timing_restricted_general_health = w2_health_review.copy()
timing_restricted_general_health_source = pd.Series(pd.NA, index=health_timing_review.index, dtype='string')
timing_restricted_general_health_source.loc[w2_health_available] = 'Wave 2'
use_clear_pretransition_w3 = w3_health_available & clear_pretransition_w3_timing
timing_restricted_general_health.loc[use_clear_pretransition_w3] = w3_health_review.loc[use_clear_pretransition_w3]
timing_restricted_general_health_source.loc[use_clear_pretransition_w3] = 'Wave 3, March–August 2006'
timing_restricted_general_health_source.loc[w3_health_available & september_w3_timing & w2_health_available] = 'Wave 2 fallback: Wave 3 interview in September 2006'
timing_restricted_general_health_source.loc[unavailable_w3_timing & w2_health_available] = 'Wave 2 fallback: Wave 3 interview timing unavailable'
timing_restricted_general_health_source.loc[~w3_health_available & w2_health_available] = 'Wave 2 fallback: Wave 3 health unavailable'
timing_restricted_general_health_label = timing_restricted_general_health.map(general_health_category_labels)
both_options_valid = standard_general_health_review.notna() & timing_restricted_general_health.notna()
standard_timing_restricted_comparison = pd.DataFrame([{'Comparison': 'Standard construction valid',
    'Participants': int(standard_general_health_review.notna().sum())}, {'Comparison': 'Timing-restricted construction valid',
    'Participants': int(timing_restricted_general_health.notna().sum())}, {'Comparison': 'Both constructions valid',
    'Participants': int(both_options_valid.sum())}, {'Comparison': 'Same category in both constructions',
    'Participants': int((both_options_valid & standard_general_health_review.eq(timing_restricted_general_health)).sum())}, {'Comparison': 'Different category between constructions',
    'Participants': int((both_options_valid & standard_general_health_review.ne(timing_restricted_general_health)).sum())}, {'Comparison': 'Standard valid but timing-restricted missing',
    'Participants': int((standard_general_health_review.notna() & timing_restricted_general_health.isna()).sum())}, {'Comparison': 'Timing-restricted valid but standard missing',
    'Participants': int((timing_restricted_general_health.notna() & standard_general_health_review.isna()).sum())}])
construction_difference_group = pd.Series(pd.NA, index=health_timing_review.index, dtype='string')
different_category = both_options_valid & standard_general_health_review.ne(timing_restricted_general_health)
construction_difference_group.loc[different_category & september_w3_timing] = 'September 2006'
construction_difference_group.loc[different_category & unavailable_w3_timing] = 'Wave 3 timing unavailable'
construction_difference_summary = construction_difference_group.dropna().value_counts().rename_axis('Timing group producing a category difference').reset_index(name='Participants')
timing_restricted_general_health_distribution = timing_restricted_general_health_label.fillna('General-health status unavailable').value_counts().rename_axis('Timing-restricted pre-transition general health').reset_index(name='Participants')
timing_restricted_general_health_source_summary = timing_restricted_general_health_source.fillna('No eligible Wave 2 or clearly timed Wave 3 response').value_counts().rename_axis('Timing-restricted construction source').reset_index(name='Participants')
stage_2_yp_general_health_timing_comparison = health_timing_review[['NSID', 'W2 general health', 'W3 general health',
    'W3 interview month', 'W3 interview year', 'pretransition_general_health', 'Pre-transition general health']].copy()
stage_2_yp_general_health_timing_comparison['timing_restricted_pretransition_general_health'] = timing_restricted_general_health
stage_2_yp_general_health_timing_comparison['Timing-restricted pre-transition general health'] = timing_restricted_general_health_label
stage_2_yp_general_health_timing_comparison['Timing-restricted construction source'] = timing_restricted_general_health_source
corrected_timing_group_total = (w3_health_available & clear_pretransition_w3_timing).sum() + (w3_health_available & september_w3_timing).sum() + unavailable_w3_timing.sum()
general_health_timing_recalculation_checks = pd.DataFrame([{'Quality check': 'Valid Wave 3 health responses',
    'Participants': int(w3_health_available.sum())}, {'Quality check': 'Corrected timing-group total',
    'Participants': int(corrected_timing_group_total)}, {'Quality check': 'Difference between valid health and timing groups',
    'Participants': int(w3_health_available.sum() - corrected_timing_group_total)}, {'Quality check': 'Valid Wave 3 health with timing unavailable',
    'Participants': int(unavailable_w3_timing.sum())}, {'Quality check': 'Timing-restricted values outside categories 1–4',
    'Participants': int((timing_restricted_general_health.notna() & ~timing_restricted_general_health.isin([1, 2, 3,
    4])).sum())}, {'Quality check': 'Duplicate NSID values',
    'Participants': int(stage_2_yp_general_health_timing_comparison['NSID'].duplicated().sum())}])
print(f'Participants in timing-restricted health-timing review: {len(stage_2_yp_general_health_timing_comparison):,}')
print('\nCorrected Wave 3 timing groups:')
print(recalculated_w3_timing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nOriginal timing codes among valid Wave 3 health responses with unavailable timing:')
print(unavailable_timing_code_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nInitial and timing-restricted comparison:')
print(standard_timing_restricted_comparison.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCategory differences by timing group:')
print(construction_difference_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nTiming-restricted distribution:')
print(timing_restricted_general_health_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nTiming-restricted construction source:')
with pd.option_context('display.max_colwidth', 150, 'display.width', 320):
    print(timing_restricted_general_health_source_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(general_health_timing_recalculation_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in timing-restricted health-timing review: 9,767

Corrected Wave 3 timing groups:
Wave 3 health-timing group  Valid Wave 3 health  Wave 2 health also available  Wave 2 health unavailable
         March–August 2006                 9303                          8716                        587
            September 2006                   14                            14                          0
        Timing unavailable                  128                           112                         16

Original timing codes among valid Wave 3 health responses with unavailable timing:
 Raw W3 interview month  Raw W3 interview year  Participants
                  -99.0                  -99.0           128

Initial and timing-restricted comparison:
                                  Comparison  Participants
                 Standard construction valid          9503
        Timing-restricted construction valid          9487
                                         ...           ...


### General-health timing restriction

The young-person general-health predictor was constructed using responses that met the pre-transition timing requirement. Wave 3 general health was retained where interview timing confirmed measurement between March and August 2006. Where a valid Wave 3 response was recorded in September 2006, or where Wave 3 interview timing was unavailable, Wave 2 general health was used where available. The predictor remained missing where neither an eligible Wave 3 response nor a valid Wave 2 response was available.

Among the 9,445 participants with a valid Wave 3 general-health response, 9,303 had confirmed interview timing between March and August 2006, 14 had interview timing in September 2006, and 128 had unavailable interview timing. Wave 2 general health was available for all 14 September cases and for 112 of the 128 cases with unavailable timing. The remaining 16 participants had no eligible value under the timing rule.

The resulting predictor contained 9,487 valid values and 280 missing values across the 9,767-participant roster. Of the valid values, 9,303 were taken from Wave 3 and 184 from Wave 2. No outcome variables were used in this construction decision.

In [170]:
# 98: Young-person general-health predictor decision

general_health_predictor_name = 'young_person_general_health'
general_health_predictor_reason = 'Young-person self-rated general health provides a broad pre-transition health measure distinct from the selected disability and SEN indicators. Wave 3 is used only where the interview was dated before September 2006. Wave 2 is used when Wave 3 is unavailable or does not meet this timing rule.'
general_health_decision_review = stage_2_yp_general_health_timing_comparison.copy().reset_index(drop=True)
w2_health_for_decision = pd.to_numeric(general_health_decision_review['W2 general health'],
    errors='coerce').astype('Int64')
w3_health_for_decision = pd.to_numeric(general_health_decision_review['W3 general health'],
    errors='coerce').astype('Int64')
w3_month_for_decision = pd.to_numeric(general_health_decision_review['W3 interview month'],
    errors='coerce').astype('Int64')
w3_year_for_decision = pd.to_numeric(general_health_decision_review['W3 interview year'],
    errors='coerce').astype('Int64')
selected_general_health = pd.to_numeric(general_health_decision_review['timing_restricted_pretransition_general_health'],
    errors='coerce').astype('Int64')
selected_general_health_source = general_health_decision_review['Timing-restricted construction source'].astype('string')
w2_health_valid_for_decision = w2_health_for_decision.notna()
w3_health_valid_for_decision = w3_health_for_decision.notna()
w3_pretransition_eligible_for_decision = (w3_year_for_decision.eq(2006) & w3_month_for_decision.between(1, 8,
    inclusive='both')).fillna(False)
w3_interview_in_september = (w3_year_for_decision.eq(2006) & w3_month_for_decision.eq(9)).fillna(False)
w3_timing_unavailable_for_decision = w3_health_valid_for_decision & (w3_year_for_decision.isna() | w3_month_for_decision.isna())
w3_timing_not_eligible_for_decision = w3_health_valid_for_decision & ~w3_pretransition_eligible_for_decision
use_wave2_for_unavailable_timing = w3_timing_unavailable_for_decision & w2_health_valid_for_decision
use_wave2_for_september = w3_interview_in_september & w2_health_valid_for_decision
use_wave2_when_wave3_unavailable = ~w3_health_valid_for_decision & w2_health_valid_for_decision
selected_general_health_label = selected_general_health.map(general_health_category_labels)
general_health_missing = selected_general_health.isna()
general_health_missing_reason = pd.Series(pd.NA, index=general_health_decision_review.index, dtype='string')
general_health_missing_reason.loc[general_health_missing & ~w2_health_valid_for_decision & ~w3_health_valid_for_decision] = 'No valid Wave 2 or Wave 3 general-health response'
general_health_missing_reason.loc[general_health_missing & w3_timing_unavailable_for_decision & ~w2_health_valid_for_decision] = 'Wave 3 interview timing unavailable and no valid Wave 2 fallback'
general_health_missing_reason.loc[general_health_missing & w3_timing_not_eligible_for_decision & ~w3_timing_unavailable_for_decision & ~w2_health_valid_for_decision] = 'Wave 3 interview not dated before September 2006 and no valid Wave 2 fallback'
general_health_missing_reason.loc[general_health_missing & general_health_missing_reason.isna()] = 'Other unresolved general-health status'
stage_2_yp_general_health_predictor_review = pd.DataFrame({'NSID': general_health_decision_review['NSID'],
    'W2 general health': w2_health_for_decision, 'W3 general health': w3_health_for_decision, 'W3 interview month': w3_month_for_decision, 'W3 interview year': w3_year_for_decision, general_health_predictor_name: selected_general_health, 'Pre-transition general health': selected_general_health_label, 'Construction source': selected_general_health_source, 'Missing reason': general_health_missing_reason})
general_health_source_decisions = [{'Wave': 'Wave 2', 'Source type': 'Young person', 'Variable': 'W2hea1cYP',
    'Review outcome': 'Retain as construction source', 'Domain': 'Young-person health', 'Decision reason': 'Harmonised pre-transition general-health source. Used when no valid Wave 3 response has an interview date before September 2006.', 'Reference period': "Young person's self-rated health during the 12 months preceding the Wave 2 interview.", 'Leakage assessment': 'No direct outcome leakage identified. The measure was collected before transition.', 'Review note': 'Numeric codes 3–6 were harmonised to the same four-category ordering used at Wave 3.'}, {'Wave': 'Wave 3',
    'Source type': 'Young person', 'Variable': 'W3hea1cYP', 'Review outcome': 'Retain as construction source', 'Domain': 'Young-person health', 'Decision reason': 'Primary source where the linked interview date is recorded before September 2006. Responses with later or unavailable timing are not used directly.', 'Reference period': "Young person's self-rated health during the 12 months preceding the Wave 3 interview.", 'Leakage assessment': 'Near-transition timing reviewed. Wave 3 is used only for interviews dated before September 2006; otherwise Wave 2 is used when available.', 'Review note': 'No separate young-person interview-date field was identified. The linked main-parent interview month and year provide the available timing check.'}, {'Wave': 'Wave 4',
    'Source type': 'Young person', 'Variable': 'W4Hea1CYP', 'Review outcome': 'Exclude from predictor set', 'Domain': 'Young-person health', 'Decision reason': 'Measured at or after the post-16 transition and therefore not eligible for the pre-transition predictor set.', 'Reference period': "Young person's general health during the 12 months preceding the Wave 4 interview.", 'Leakage assessment': 'Temporal leakage risk. The Wave 4 reference period extends across post-transition experience.', 'Review note': 'Retained in source data only; not used in predictor construction or fallback.'}, {'Wave': 'Wave 3',
    'Source type': 'Family background', 'Variable': 'W3intmnthMP', 'Review outcome': 'Retain as construction source', 'Domain': 'Survey timing support', 'Decision reason': 'Used with interview year to identify Wave 3 interviews dated before September 2006.', 'Reference period': 'Recorded month of the Wave 3 main-parent interview.', 'Leakage assessment': 'No direct outcome leakage. Survey timing metadata is not included as a predictor.', 'Review note': 'Construction support only; the value is not entered in the predictor matrix.'}, {'Wave': 'Wave 3',
    'Source type': 'Family background', 'Variable': 'W3intyearMP', 'Review outcome': 'Retain as construction source', 'Domain': 'Survey timing support', 'Decision reason': 'Used with interview month to identify Wave 3 interviews dated before September 2006.', 'Reference period': 'Recorded year of the Wave 3 main-parent interview.', 'Leakage assessment': 'No direct outcome leakage. Survey timing metadata is not included as a predictor.', 'Review note': 'Construction support only; the value is not entered in the predictor matrix.'}, {'Wave': 'Wave 3',
    'Source type': 'Parental attitudes', 'Variable': 'W3intmnthMP', 'Review outcome': 'Exclude from predictor set', 'Domain': 'Survey timing support', 'Decision reason': 'Exact duplicate of the Wave 3 family-background interview-month field selected as the construction source.', 'Reference period': 'Recorded month of the Wave 3 main-parent interview.', 'Leakage assessment': 'No direct outcome leakage identified. Excluded as duplicate survey metadata.', 'Review note': 'The two copies agreed for all 9,381 valid values.'}, {'Wave': 'Wave 3',
    'Source type': 'Parental attitudes', 'Variable': 'W3intyearMP', 'Review outcome': 'Exclude from predictor set', 'Domain': 'Survey timing support', 'Decision reason': 'Exact duplicate of the Wave 3 family-background interview-year field selected as the construction source.', 'Reference period': 'Recorded year of the Wave 3 main-parent interview.', 'Leakage assessment': 'No direct outcome leakage identified. Excluded as duplicate survey metadata.', 'Review note': 'The two copies agreed for all 9,381 valid values.'}]
general_health_decision_indices = []
for rule in general_health_source_decisions:
    matching_rows = stage_2_variable_decision_register['Wave'].eq(rule['Wave']) & stage_2_variable_decision_register['Source type'].eq(rule['Source type']) & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {rule['Wave']}, {rule['Source type']}, {rule['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    general_health_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = rule['Review outcome']
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = rule['Domain']
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = rule['Decision reason']
    stage_2_variable_decision_register.loc[matching_rows, 'Reference-period assessment'] = rule['Reference period']
    stage_2_variable_decision_register.loc[matching_rows, 'Leakage assessment'] = rule['Leakage assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave 2–Wave 4 young-person data dictionaries and Stage 2 general-health timing review'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = rule['Review note']
general_health_review_output_path = stage_2_output_directory / 'stage_2_young_person_general_health_review.csv'
stage_2_yp_general_health_predictor_review.to_csv(general_health_review_output_path, index=False)
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
general_health_distribution = selected_general_health_label.fillna('General-health status unavailable').value_counts().rename_axis('Pre-transition general health').reset_index(name='Participants')
general_health_source_summary = selected_general_health_source.fillna('No valid construction source').value_counts().rename_axis('Construction source').reset_index(name='Participants')
general_health_missing_summary = general_health_missing_reason.loc[general_health_missing].value_counts().rename_axis('Missing reason').reset_index(name='Participants')
general_health_decision_output = stage_2_variable_decision_register.loc[general_health_decision_indices,
    ['Source order', 'Wave', 'Source type', 'Variable position', 'Variable', 'Variable label', 'Review outcome',
    'Decision reason', 'Leakage assessment']].sort_values(['Source order',
    'Variable position']).drop(columns=['Source order', 'Variable position']).reset_index(drop=True)
explicit_general_health_exclusions = general_health_decision_output.loc[general_health_decision_output['Review outcome'].eq('Exclude from predictor set')].reset_index(drop=True)
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
ineligible_wave3_used_directly = w3_timing_not_eligible_for_decision & selected_general_health_source.fillna('').str.startswith('Wave 3')
general_health_decision_checks = pd.DataFrame([{'Quality check': 'Participants in predictor review',
    'Participants': len(stage_2_yp_general_health_predictor_review)}, {'Quality check': 'Valid constructed predictor values',
    'Participants': int(selected_general_health.notna().sum())}, {'Quality check': 'Eligible Wave 3 responses used directly',
    'Participants': int((w3_health_valid_for_decision & w3_pretransition_eligible_for_decision).sum())}, {'Quality check': 'Explicit September Wave 3 cases',
    'Participants': int(w3_interview_in_september.sum())}, {'Quality check': 'Wave 3 health with timing unavailable',
    'Participants': int(w3_timing_unavailable_for_decision.sum())}, {'Quality check': 'Timing-unavailable cases replaced by Wave 2',
    'Participants': int(use_wave2_for_unavailable_timing.sum())}, {'Quality check': 'Timing-unavailable cases without Wave 2',
    'Participants': int((w3_timing_unavailable_for_decision & ~w2_health_valid_for_decision).sum())}, {'Quality check': 'Ineligible Wave 3 responses used directly',
    'Participants': int(ineligible_wave3_used_directly.sum())}, {'Quality check': 'Values outside categories 1–4',
    'Participants': int((selected_general_health.notna() & ~selected_general_health.isin([1, 2, 3,
    4])).sum())}, {'Quality check': 'Duplicate NSID values',
    'Participants': int(stage_2_yp_general_health_predictor_review['NSID'].duplicated().sum())}])
assert int(ineligible_wave3_used_directly.sum()) == 0
print(f'Young-person general-health predictor decided: {general_health_predictor_name}')
print('\nPredictor distribution:')
print(general_health_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nConstruction source:')
with pd.option_context('display.max_colwidth', 160, 'display.width', 320):
    print(general_health_source_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMissing-value reasons:')
print(general_health_missing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSource-variable decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 170, 'display.width', 450):
    print(general_health_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nVariables explicitly excluded in this step: {len(explicit_general_health_exclusions):,}')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 180, 'display.width', 430):
    print(explicit_general_health_exclusions[['Wave', 'Source type', 'Variable', 'Variable label', 'Decision reason',
        'Leakage assessment']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(general_health_decision_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'General-health review saved to: {general_health_review_output_path}')

Young-person general-health predictor decided: young_person_general_health

Predictor distribution:
    Pre-transition general health  Participants
                        Very good          5904
                      Fairly good          3305
General-health status unavailable           280
                    Not very good           241
                  Not good at all            37

Construction source:
                                 Construction source  Participants
                           Wave 3, March–August 2006          9303
                        No valid construction source           280
Wave 2 fallback: Wave 3 interview timing unavailable           112
          Wave 2 fallback: Wave 3 health unavailable            58
 Wave 2 fallback: Wave 3 interview in September 2006            14

Missing-value reasons:
                                                  Missing reason  Participants
               No valid Wave 2 or Wave 3 general-health response           264
Wave 3

In [171]:
# 99: Parental general-health measure structure

parent_general_health_label_pattern = '(?i)^(?:MP|SP|SP/MP):.*(?:general health|quality of .*health).*last 12 months'
parent_general_health_entries = stage_2_master_variable_register.loc[stage_2_master_variable_register['Source type'].eq('Family background') & stage_2_master_variable_register['Variable label'].astype('string').str.contains(parent_general_health_label_pattern,
    na=False, regex=True), ['Source order', 'Wave', 'Source type', 'Source file', 'Source path', 'Variable position',
    'Variable', 'Variable label', 'Timing status']].copy().drop_duplicates(subset=['Source file',
    'Variable']).sort_values(['Source order', 'Variable position']).reset_index(drop=True)
if parent_general_health_entries.empty:
    raise ValueError('No parental general-health variables were found.')
parent_general_health_entries = parent_general_health_entries.merge(stage_2_variable_decision_register[['Source file',
    'Variable', 'Review outcome']], on=['Source file', 'Variable'], how='left', validate='one_to_one')
parent_general_health_entries['Parent role'] = 'Unresolved parent role'
parent_general_health_entries.loc[parent_general_health_entries['Variable label'].str.match('(?i)^MP:', na=False),
    'Parent role'] = 'Main parent'
parent_general_health_entries.loc[parent_general_health_entries['Variable label'].str.match('(?i)^(?:SP|SP/MP):',
    na=False), 'Parent role'] = 'Second parent'
stage_2_parent_general_health_review = stage_2_participant_ids.copy()
parent_health_structure_records = []
parent_health_value_records = []
for source_path_text, source_entries in parent_general_health_entries.groupby('Source path', sort=False):
    source_path = Path(source_path_text)
    source_variables = source_entries['Variable'].tolist()
    numeric_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    numeric_sample = stage_2_participant_ids.merge(numeric_source, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = numeric_sample['_merge'].eq('both')
    numeric_sample = numeric_sample.drop(columns='_merge')
    labelled_sample = stage_2_participant_ids.merge(labelled_source, on='NSID', how='left', validate='one_to_one')
    for _, entry in source_entries.iterrows():
        variable = entry['Variable']
        numeric_values = pd.to_numeric(numeric_sample[variable], errors='coerce')
        labelled_values = labelled_sample[variable].astype('string')
        stage_2_parent_general_health_review[f'{variable} code'] = numeric_values
        stage_2_parent_general_health_review[f'{variable} label'] = labelled_values
        stage_2_parent_general_health_review[f'{variable} source present'] = source_record_present
        non_negative_values = numeric_values.ge(0)
        parent_health_structure_records.append({'Wave': entry['Wave'], 'Parent role': entry['Parent role'],
            'Variable': variable, 'Variable label': entry['Variable label'], 'Timing status': entry['Timing status'], 'Current register status': entry['Review outcome'], 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Non-negative stored value': int(non_negative_values.sum()), 'Negative stored value': int(numeric_values.lt(0).sum()), 'Unique non-negative values': int(numeric_values.loc[non_negative_values].nunique()), 'Minimum non-negative value': numeric_values.loc[non_negative_values].min() if non_negative_values.any() else np.nan, 'Maximum non-negative value': numeric_values.loc[non_negative_values].max() if non_negative_values.any() else np.nan})
        value_summary = pd.DataFrame({'Value code': numeric_values, 'Value label': labelled_values,
            'Source record present': source_record_present}).loc[lambda frame: frame['Source record present']].drop(columns='Source record present').value_counts(dropna=False).rename('Participants').reset_index()
        value_summary.insert(0, 'Wave', entry['Wave'])
        value_summary.insert(1, 'Parent role', entry['Parent role'])
        value_summary.insert(2, 'Variable', variable)
        parent_health_value_records.append(value_summary)
parent_general_health_structure_summary = pd.DataFrame(parent_health_structure_records)
parent_general_health_value_summary = pd.concat(parent_health_value_records, ignore_index=True)
parent_general_health_substantive_values = parent_general_health_value_summary.loc[pd.to_numeric(parent_general_health_value_summary['Value code'],
    errors='coerce').ge(0)].copy().reset_index(drop=True)
parent_general_health_negative_values = parent_general_health_value_summary.loc[pd.to_numeric(parent_general_health_value_summary['Value code'],
    errors='coerce').lt(0) | parent_general_health_value_summary['Value code'].isna()].copy().reset_index(drop=True)
parent_health_pairwise_records = []
for parent_role in ['Main parent', 'Second parent']:
    role_variables = parent_general_health_entries.loc[parent_general_health_entries['Parent role'].eq(parent_role),
        ['Wave', 'Variable']].reset_index(drop=True)
    for first_position in range(len(role_variables)):
        for second_position in range(first_position + 1, len(role_variables)):
            first_wave = role_variables.loc[first_position, 'Wave']
            first_variable = role_variables.loc[first_position, 'Variable']
            second_wave = role_variables.loc[second_position, 'Wave']
            second_variable = role_variables.loc[second_position, 'Variable']
            first_code = pd.to_numeric(stage_2_parent_general_health_review[f'{first_variable} code'],
                errors='coerce')
            second_code = pd.to_numeric(stage_2_parent_general_health_review[f'{second_variable} code'],
                errors='coerce')
            first_label = stage_2_parent_general_health_review[f'{first_variable} label'].astype('string').str.strip().str.lower()
            second_label = stage_2_parent_general_health_review[f'{second_variable} label'].astype('string').str.strip().str.lower()
            both_valid = first_code.ge(0) & second_code.ge(0)
            exact_label_agreement = both_valid & first_label.eq(second_label)
            parent_health_pairwise_records.append({'Parent role': parent_role, 'First wave': first_wave,
                'First variable': first_variable, 'Second wave': second_wave, 'Second variable': second_variable, 'Both responses valid': int(both_valid.sum()), 'Exact label agreement': int(exact_label_agreement.sum()), 'Exact label agreement percentage': exact_label_agreement.sum() / both_valid.sum() * 100 if both_valid.sum() else np.nan})
parent_general_health_pairwise_summary = pd.DataFrame(parent_health_pairwise_records)
parent_general_health_coverage_summary = parent_general_health_structure_summary.groupby(['Wave', 'Parent role',
    'Timing status'], dropna=False).agg(Variables=('Variable', 'size'), Valid_responses=('Non-negative stored value',
    'sum')).reset_index().sort_values(['Parent role', 'Wave']).reset_index(drop=True)
parent_general_health_checks = pd.DataFrame([{'Check': 'Parental general-health variables found',
    'Result': len(parent_general_health_entries)}, {'Check': 'Main-parent variables',
    'Result': int(parent_general_health_entries['Parent role'].eq('Main parent').sum())}, {'Check': 'Second-parent variables',
    'Result': int(parent_general_health_entries['Parent role'].eq('Second parent').sum())}, {'Check': 'Unresolved parent-role variables',
    'Result': int(parent_general_health_entries['Parent role'].eq('Unresolved parent role').sum())}, {'Check': 'Duplicate source-variable entries',
    'Result': int(parent_general_health_entries[['Source file',
    'Variable']].duplicated().sum())}, {'Check': 'Variables without register status',
    'Result': int(parent_general_health_entries['Review outcome'].isna().sum())}])
print(f'Parental general-health variables reviewed: {len(parent_general_health_entries):,}')
print('\nSource entries:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 160, 'display.width', 400):
    print(parent_general_health_entries[['Wave', 'Parent role', 'Variable', 'Variable label', 'Timing status',
        'Review outcome']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCoverage and stored-value structure:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 420):
    print(parent_general_health_structure_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSubstantive values and labels:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 130, 'display.width', 340):
    print(parent_general_health_substantive_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nNegative and missing values:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 130, 'display.width', 340):
    print(parent_general_health_negative_values.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCoverage by wave and parent role:')
print(parent_general_health_coverage_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nPairwise label agreement:')
with pd.option_context('display.max_rows', None, 'display.width', 340):
    print(parent_general_health_pairwise_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
        formatters={'Exact label agreement percentage': lambda value: f'{value:.2f}%' if pd.notna(value) else 'Not available'}))
print('\nReview checks:')
print(parent_general_health_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Parental general-health variables reviewed: 3

Source entries:
  Wave   Parent role  Variable                               Variable label          Timing status Review outcome
Wave 1   Main parent  W1hea1MP         MP: General health in last 12 months  Pre-transition source Pending review
Wave 1 Second parent  W1hea1SP         SP: General health in last 12 months  Pre-transition source Pending review
Wave 4   Main parent W4Hea1CMP MP: Quality of MP's health in last 12 months At or after transition Pending review

Coverage and stored-value structure:
  Wave   Parent role  Variable                               Variable label          Timing status Current register status  Source record present  Source record absent  Non-negative stored value  Negative stored value  Unique non-negative values  Minimum non-negative value  Maximum non-negative value
Wave 1   Main parent  W1hea1MP         MP: General health in last 12 months  Pre-transition source          Pending review                   

In [172]:
# 100: Main-parent and household general-health options

w1_main_parent_health = pd.to_numeric(stage_2_parent_general_health_review['W1hea1MP code'],
    errors='coerce').where(lambda values: values.isin([1, 2, 3, 4])).astype('Float64')
w1_second_parent_health = pd.to_numeric(stage_2_parent_general_health_review['W1hea1SP code'],
    errors='coerce').where(lambda values: values.isin([1, 2, 3, 4])).astype('Float64')
parent_health_category_labels = {1.0: 'Very good', 2.0: 'Fairly good', 3.0: 'Not very good', 4.0: 'Not good at all'}
main_parent_health_valid = w1_main_parent_health.notna()
second_parent_health_valid = w1_second_parent_health.notna()
both_parent_health_valid = main_parent_health_valid & second_parent_health_valid
main_parent_only = main_parent_health_valid & ~second_parent_health_valid
second_parent_only = ~main_parent_health_valid & second_parent_health_valid
neither_parent_health_valid = ~main_parent_health_valid & ~second_parent_health_valid
second_parent_health_worse = both_parent_health_valid & w1_second_parent_health.gt(w1_main_parent_health)
main_parent_health_worse = both_parent_health_valid & w1_main_parent_health.gt(w1_second_parent_health)
parent_health_equal = both_parent_health_valid & w1_main_parent_health.eq(w1_second_parent_health)
parent_health_absolute_difference = (w1_main_parent_health - w1_second_parent_health).abs()
parent_health_comparison_summary = pd.DataFrame([{'Comparison': 'Both parent responses valid',
    'Participants': int(both_parent_health_valid.sum())}, {'Comparison': 'Parent responses identical',
    'Participants': int(parent_health_equal.sum())}, {'Comparison': 'Second parent reports worse health',
    'Participants': int(second_parent_health_worse.sum())}, {'Comparison': 'Main parent reports worse health',
    'Participants': int(main_parent_health_worse.sum())}, {'Comparison': 'One-category difference',
    'Participants': int((both_parent_health_valid & parent_health_absolute_difference.eq(1)).sum())}, {'Comparison': 'Two-category difference',
    'Participants': int((both_parent_health_valid & parent_health_absolute_difference.eq(2)).sum())}, {'Comparison': 'Three-category difference',
    'Participants': int((both_parent_health_valid & parent_health_absolute_difference.eq(3)).sum())}])
household_worst_parental_health = pd.concat([w1_main_parent_health, w1_second_parent_health], axis=1).max(axis=1,
    skipna=True).astype('Float64')
household_worst_parental_health.loc[neither_parent_health_valid] = pd.NA
household_worst_parental_health_label = household_worst_parental_health.map(parent_health_category_labels)
household_parent_health_source = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
household_parent_health_source.loc[main_parent_only] = 'Main parent only'
household_parent_health_source.loc[second_parent_only] = 'Second parent only'
household_parent_health_source.loc[parent_health_equal] = 'Both parents; same category'
household_parent_health_source.loc[second_parent_health_worse] = 'Second parent has poorer health'
household_parent_health_source.loc[main_parent_health_worse] = 'Main parent has poorer health'
stage_2_parent_general_health_option_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    'main_parent_general_health': w1_main_parent_health, 'Main-parent general health': w1_main_parent_health.map(parent_health_category_labels), 'second_parent_general_health': w1_second_parent_health, 'Second-parent general health': w1_second_parent_health.map(parent_health_category_labels), 'household_worst_parental_health': household_worst_parental_health, 'Household worst parental health': household_worst_parental_health_label, 'Household-summary source': household_parent_health_source})
parent_health_option_coverage = pd.DataFrame([{'Representation option': 'Main-parent general health',
    'Valid participants': int(w1_main_parent_health.notna().sum()), 'Missing participants': int(w1_main_parent_health.isna().sum()), 'Coverage percentage': w1_main_parent_health.notna().mean() * 100}, {'Representation option': 'Household worst parental health',
    'Valid participants': int(household_worst_parental_health.notna().sum()), 'Missing participants': int(household_worst_parental_health.isna().sum()), 'Coverage percentage': household_worst_parental_health.notna().mean() * 100}])
main_parent_health_distribution = w1_main_parent_health.map(parent_health_category_labels).fillna('Main-parent health unavailable').value_counts().rename_axis('Main-parent general health').reset_index(name='Participants')
household_parent_health_distribution = household_worst_parental_health_label.fillna('Household parental health unavailable').value_counts().rename_axis('Household worst parental health').reset_index(name='Participants')
household_parent_health_source_summary = household_parent_health_source.fillna('No valid parental-health response').value_counts().rename_axis('Household-summary source').reset_index(name='Participants')
second_parent_increment_summary = pd.DataFrame([{'Second-parent contribution': 'Second-parent response adds coverage when main parent is unavailable',
    'Participants': int(second_parent_only.sum())}, {'Second-parent contribution': 'Second-parent poorer health changes the household category',
    'Participants': int(second_parent_health_worse.sum())}, {'Second-parent contribution': 'Second-parent response does not change the main-parent category',
    'Participants': int((both_parent_health_valid & ~second_parent_health_worse).sum())}])
parent_health_cross_tabulation = pd.crosstab(w1_main_parent_health.map(parent_health_category_labels),
    w1_second_parent_health.map(parent_health_category_labels), margins=True, margins_name='Total')
parent_health_option_checks = pd.DataFrame([{'Quality check': 'Participants in option review',
    'Participants': len(stage_2_parent_general_health_option_review)}, {'Quality check': 'Both parent responses valid',
    'Participants': int(both_parent_health_valid.sum())}, {'Quality check': 'Main-parent response only',
    'Participants': int(main_parent_only.sum())}, {'Quality check': 'Second-parent response only',
    'Participants': int(second_parent_only.sum())}, {'Quality check': 'Neither parent response available',
    'Participants': int(neither_parent_health_valid.sum())}, {'Quality check': 'Response-availability groups total',
    'Participants': int(both_parent_health_valid.sum() + main_parent_only.sum() + second_parent_only.sum() + neither_parent_health_valid.sum())}, {'Quality check': 'Household values outside categories 1–4',
    'Participants': int((household_worst_parental_health.notna() & ~household_worst_parental_health.isin([1, 2, 3,
    4])).sum())}, {'Quality check': 'Duplicate NSID values',
    'Participants': int(stage_2_parent_general_health_option_review['NSID'].duplicated().sum())}])
print(f'Participants in parental general-health option review: {len(stage_2_parent_general_health_option_review):,}')
print('\nCoverage comparison:')
print(parent_health_option_coverage.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Coverage percentage': lambda value: f'{value:.2f}%'}))
print('\nMain-parent distribution:')
print(main_parent_health_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nHousehold worst-health distribution:')
print(household_parent_health_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nHousehold-summary source:')
print(household_parent_health_source_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMain-parent by second-parent health:')
print(parent_health_cross_tabulation.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nParent-response comparison:')
print(parent_health_comparison_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSecond-parent incremental contribution:')
with pd.option_context('display.max_colwidth', 150, 'display.width', 300):
    print(second_parent_increment_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(parent_health_option_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in parental general-health option review: 9,767

Coverage comparison:
          Representation option  Valid participants  Missing participants Coverage percentage
     Main-parent general health                9415                   352              96.40%
Household worst parental health                9458                   309              96.84%

Main-parent distribution:
    Main-parent general health  Participants
                     Very good          4833
                   Fairly good          3304
                 Not very good           951
Main-parent health unavailable           352
               Not good at all           327

Household worst-health distribution:
      Household worst parental health  Participants
                          Fairly good          4120
                            Very good          3555
                        Not very good          1290
                      Not good at all           493
Household parental health unavailable   

In [173]:
# 101: Main-parent disability review-file check

main_parent_disability_review_path = stage_2_output_directory / 'stage_2_main_parent_disability_review.csv'
if not main_parent_disability_review_path.exists():
    raise FileNotFoundError(f'The main-parent disability review file was not found: {main_parent_disability_review_path}')
main_parent_disability_review = pd.read_csv(main_parent_disability_review_path)
if 'NSID' not in main_parent_disability_review.columns:
    raise ValueError('The main-parent disability review file does not contain NSID.')
if main_parent_disability_review['NSID'].duplicated().any():
    raise ValueError('Duplicate NSID values were found in the main-parent disability review file.')
disability_column_candidates = [column for column in main_parent_disability_review.columns if any((term in column.lower() for term in ['disab',
    'disability', 'long illness', 'limiting', 'health condition']))]
if not disability_column_candidates:
    raise ValueError('No likely main-parent disability columns were identified.')
disability_column_structure_records = []
disability_value_records = []
for column in disability_column_candidates:
    values = main_parent_disability_review[column]
    non_missing_values = values.dropna()
    disability_column_structure_records.append({'Column': column, 'Data type': str(values.dtype),
        'Non-missing values': int(values.notna().sum()), 'Missing values': int(values.isna().sum()), 'Unique non-missing values': int(non_missing_values.nunique())})
    if not non_missing_values.empty and non_missing_values.nunique() <= 20:
        value_summary = values.value_counts(dropna=False).rename_axis('Stored value').reset_index(name='Participants')
        value_summary.insert(0, 'Column', column)
        disability_value_records.append(value_summary)
main_parent_disability_column_structure = pd.DataFrame(disability_column_structure_records)
if disability_value_records:
    main_parent_disability_value_summary = pd.concat(disability_value_records, ignore_index=True)
else:
    main_parent_disability_value_summary = pd.DataFrame(columns=['Column', 'Stored value', 'Participants'])
disability_health_linkage_check = stage_2_parent_general_health_option_review[['NSID']].merge(main_parent_disability_review[['NSID']],
    on='NSID', how='outer', indicator=True, validate='one_to_one')
disability_review_checks = pd.DataFrame([{'Check': 'Rows in disability review file',
    'Result': len(main_parent_disability_review)}, {'Check': 'Columns in disability review file',
    'Result': len(main_parent_disability_review.columns)}, {'Check': 'Likely disability-related columns',
    'Result': len(disability_column_candidates)}, {'Check': 'Participants present in both reviews',
    'Result': int(disability_health_linkage_check['_merge'].eq('both').sum())}, {'Check': 'Participants present only in health review',
    'Result': int(disability_health_linkage_check['_merge'].eq('left_only').sum())}, {'Check': 'Participants present only in disability review',
    'Result': int(disability_health_linkage_check['_merge'].eq('right_only').sum())}])
print(f'Main-parent disability review file: {main_parent_disability_review_path}')
print('\nAll columns:')
for column in main_parent_disability_review.columns:
    print(f'- {column}')
print('\nLikely disability-related column structure:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 160, 'display.width', 330):
    print(main_parent_disability_column_structure.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nLow-cardinality disability-related values:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 160, 'display.width', 330):
    print(main_parent_disability_value_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nReview checks:')
print(disability_review_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Main-parent disability review file: data_derived\stage_2_predictor_construction\stage_2_main_parent_disability_review.csv

All columns:
- NSID
- main_parent_disability
- Main-parent disability
- second_parent_disability
- household_parental_disability
- Household parental disability
- second_parent_not_present
- selected_main_parent_disability
- Selected main-parent disability
- Main-parent disability missing reason
- Predictor decision

Likely disability-related column structure:
                               Column Data type  Non-missing values  Missing values  Unique non-missing values
               main_parent_disability   float64                9389             378                          3
               Main-parent disability       str                9389             378                          3
                                  ...       ...                 ...             ...                        ...
      Selected main-parent disability       str                9389   

In [174]:
# 102: Main-parent general-health and disability comparison

from scipy.stats import chi2_contingency
main_parent_health_disability_review = stage_2_parent_general_health_option_review[['NSID',
    'main_parent_general_health', 'Main-parent general health']].merge(main_parent_disability_review[['NSID',
    'selected_main_parent_disability', 'Selected main-parent disability']], on='NSID', how='left', validate='one_to_one')
main_parent_health_code = pd.to_numeric(main_parent_health_disability_review['main_parent_general_health'],
    errors='coerce').where(lambda values: values.isin([1, 2, 3, 4])).astype('Int64')
main_parent_disability_code = pd.to_numeric(main_parent_health_disability_review['selected_main_parent_disability'],
    errors='coerce').where(lambda values: values.isin([0, 1, 2])).astype('Int64')
health_label_map = {1: 'Very good', 2: 'Fairly good', 3: 'Not very good', 4: 'Not good at all'}
disability_label_map = {0: 'No main-parent disability', 1: 'Non-limiting disability',
    2: 'Activity-limiting disability'}
main_parent_health_label = main_parent_health_code.map(health_label_map)
main_parent_disability_label = main_parent_disability_code.map(disability_label_map)
health_valid = main_parent_health_code.notna()
disability_valid = main_parent_disability_code.notna()
both_measures_valid = health_valid & disability_valid
health_disability_availability = pd.DataFrame([{'Measure availability': 'Both measures valid',
    'Participants': int(both_measures_valid.sum())}, {'Measure availability': 'General health valid; disability unavailable',
    'Participants': int((health_valid & ~disability_valid).sum())}, {'Measure availability': 'Disability valid; general health unavailable',
    'Participants': int((~health_valid & disability_valid).sum())}, {'Measure availability': 'Both measures unavailable',
    'Participants': int((~health_valid & ~disability_valid).sum())}])
valid_health_categories = pd.Categorical(main_parent_health_label.loc[both_measures_valid], categories=['Very good',
    'Fairly good', 'Not very good', 'Not good at all'], ordered=True)
valid_disability_categories = pd.Categorical(main_parent_disability_label.loc[both_measures_valid],
    categories=['No main-parent disability', 'Non-limiting disability', 'Activity-limiting disability'], ordered=True)
health_by_disability_counts = pd.crosstab(valid_disability_categories, valid_health_categories, dropna=False)
health_by_disability_counts.index.name = 'Selected main-parent disability'
health_by_disability_counts.columns.name = 'Main-parent general health'
health_by_disability_row_percentages = health_by_disability_counts.div(health_by_disability_counts.sum(axis=1),
    axis=0).mul(100).round(2)
chi_square_value, _, _, _ = chi2_contingency(health_by_disability_counts)
association_sample_size = int(health_by_disability_counts.to_numpy().sum())
association_dimension = min(health_by_disability_counts.shape[0] - 1, health_by_disability_counts.shape[1] - 1)
if association_sample_size > 0 and association_dimension > 0:
    cramers_v = np.sqrt(chi_square_value / (association_sample_size * association_dimension))
else:
    cramers_v = np.nan
health_disability_association_summary = pd.DataFrame([{'Association measure': 'Participants with both measures',
    'Value': float(association_sample_size)}, {'Association measure': "Cramer's V", 'Value': float(cramers_v)}])
better_general_health = main_parent_health_code.isin([1, 2])
poorer_general_health = main_parent_health_code.isin([3, 4])
no_main_parent_disability = main_parent_disability_code.eq(0).fillna(False)
any_main_parent_disability = main_parent_disability_code.isin([1, 2])
limiting_main_parent_disability = main_parent_disability_code.eq(2).fillna(False)
health_disability_distinctness = pd.DataFrame([{'Descriptive comparison': 'No disability but not very good or not good health',
    'Participants': int((both_measures_valid & no_main_parent_disability & poorer_general_health).sum())}, {'Descriptive comparison': 'Any disability but very good or fairly good health',
    'Participants': int((both_measures_valid & any_main_parent_disability & better_general_health).sum())}, {'Descriptive comparison': 'Activity-limiting disability but very good or fairly good health',
    'Participants': int((both_measures_valid & limiting_main_parent_disability & better_general_health).sum())}, {'Descriptive comparison': 'Any disability and not very good or not good health',
    'Participants': int((both_measures_valid & any_main_parent_disability & poorer_general_health).sum())}])
health_disability_distinctness['Percentage of participants with both measures'] = health_disability_distinctness['Participants'].div(association_sample_size).mul(100)
stage_2_main_parent_health_disability_comparison = main_parent_health_disability_review.copy()
stage_2_main_parent_health_disability_comparison['main_parent_general_health'] = main_parent_health_code
stage_2_main_parent_health_disability_comparison['Main-parent general health'] = main_parent_health_label
stage_2_main_parent_health_disability_comparison['selected_main_parent_disability'] = main_parent_disability_code
stage_2_main_parent_health_disability_comparison['Selected main-parent disability'] = main_parent_disability_label
main_parent_health_disability_checks = pd.DataFrame([{'Quality check': 'Participants in comparison',
    'Participants': len(stage_2_main_parent_health_disability_comparison)}, {'Quality check': 'Availability groups total',
    'Participants': int(health_disability_availability['Participants'].sum())}, {'Quality check': 'Cross-tabulation total',
    'Participants': association_sample_size}, {'Quality check': 'Expected both-valid total',
    'Participants': int(both_measures_valid.sum())}, {'Quality check': 'Health values outside categories 1–4',
    'Participants': int((health_valid & ~main_parent_health_code.isin([1, 2, 3,
    4])).sum())}, {'Quality check': 'Disability values outside categories 0–2',
    'Participants': int((disability_valid & ~main_parent_disability_code.isin([0, 1,
    2])).sum())}, {'Quality check': 'Duplicate NSID values',
    'Participants': int(stage_2_main_parent_health_disability_comparison['NSID'].duplicated().sum())}])
print(f'Participants in main-parent health–disability comparison: {len(stage_2_main_parent_health_disability_comparison):,}')
print('\nMeasure availability:')
print(health_disability_availability.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nGeneral health by disability: counts')
print(health_by_disability_counts.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nGeneral health by disability: row percentages')
print(health_by_disability_row_percentages.to_string(max_rows=TABLE_ROW_LIMIT))
print('\nCategorical association:')
print(health_disability_association_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
    formatters={'Value': lambda value: f'{value:.4f}' if value < 1 else f'{value:,.0f}'}))
print('\nDescriptive evidence of distinct information:')
with pd.option_context('display.max_colwidth', 160, 'display.width', 330):
    print(health_disability_distinctness.to_string(max_rows=TABLE_ROW_LIMIT, index=False,
        formatters={'Percentage of participants with both measures': lambda value: f'{value:.2f}%'}))
print('\nQuality checks:')
print(main_parent_health_disability_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Participants in main-parent health–disability comparison: 9,767

Measure availability:
                        Measure availability  Participants
                         Both measures valid          9386
General health valid; disability unavailable            29
Disability valid; general health unavailable             3
                   Both measures unavailable           349

General health by disability: counts
Main-parent general health       Very good  Fairly good  Not very good  Not good at all
Selected main-parent disability                                                        
No main-parent disability             4587         2452            283               32
Non-limiting disability                168          435            111               12
Activity-limiting disability            75          406            545              280

General health by disability: row percentages
Main-parent general health       Very good  Fairly good  Not very good  Not good at all
Selec

In [175]:
# 103: Main-parent general-health predictor decision

main_parent_health_predictor_name = 'main_parent_general_health'
selected_main_parent_health = pd.to_numeric(stage_2_parent_general_health_option_review['main_parent_general_health'],
    errors='coerce').where(lambda values: values.isin([1, 2, 3, 4])).astype('Int64')
selected_main_parent_health_label = selected_main_parent_health.map({1: 'Very good', 2: 'Fairly good',
    3: 'Not very good', 4: 'Not good at all'})
w1_main_parent_health_raw = pd.to_numeric(stage_2_parent_general_health_review['W1hea1MP code'], errors='coerce')
w1_main_parent_health_source_present = stage_2_parent_general_health_review['W1hea1MP source present'].fillna(False).astype(bool)
main_parent_health_missing = selected_main_parent_health.isna()
main_parent_health_missing_reason = pd.Series(pd.NA, index=stage_2_participant_ids.index, dtype='string')
main_parent_health_missing_reason.loc[main_parent_health_missing & ~w1_main_parent_health_source_present] = 'No Wave 1 family-background source record'
main_parent_health_missing_reason.loc[main_parent_health_missing & w1_main_parent_health_raw.eq(-99)] = 'Main parent not interviewed'
main_parent_health_missing_reason.loc[main_parent_health_missing & w1_main_parent_health_raw.eq(-92)] = 'Main parent refused'
main_parent_health_missing_reason.loc[main_parent_health_missing & w1_main_parent_health_raw.eq(-91)] = 'Not applicable'
main_parent_health_missing_reason.loc[main_parent_health_missing & main_parent_health_missing_reason.isna()] = 'Other unresolved main-parent health status'
stage_2_main_parent_general_health_predictor_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    main_parent_health_predictor_name: selected_main_parent_health, 'Main-parent general health': selected_main_parent_health_label, 'Main-parent general-health missing reason': main_parent_health_missing_reason, 'Predictor decision': 'Retain Wave 1 main-parent general health as a four-category predictor'})
main_parent_health_source_decisions = [{'Wave': 'Wave 1', 'Source type': 'Family background', 'Variable': 'W1hea1MP',
    'Review outcome': 'Retain as construction source', 'Domain': 'Parental health', 'Decision reason': 'Selected as a four-category main-parent general-health predictor. It has high coverage and is related to, but not interchangeable with, the selected main-parent disability measure.', 'Reference period': "Main parent's self-rated general health during the 12 months preceding the Wave 1 interview.", 'Leakage assessment': 'No direct outcome leakage identified. The measure was collected before transition.', 'Review note': f"The measure was retained separately from disability. Cramer's V with the selected main-parent disability measure was {cramers_v:.4f}. The original four categories were preserved."}, {'Wave': 'Wave 1',
    'Source type': 'Family background', 'Variable': 'W1hea1SP', 'Review outcome': 'Retain as review support only', 'Domain': 'Parental health', 'Decision reason': 'Used to examine a household-level parental-health alternative but not selected for predictor construction. Its availability depends strongly on second-parent presence and interview participation.', 'Reference period': "Second parent's self-rated general health during the 12 months preceding the Wave 1 interview.", 'Leakage assessment': 'No direct outcome leakage identified. The measure was collected before transition.', 'Review note': 'Adding the second-parent response increased coverage by only 43 participants. A household worst-health summary was not selected because it would combine parental health with household and response structure.'}, {'Wave': 'Wave 4',
    'Source type': 'Family background', 'Variable': 'W4Hea1CMP', 'Review outcome': 'Exclude from predictor set', 'Domain': 'Parental health', 'Decision reason': 'Measured at or after the post-16 transition and not eligible for the pre-transition predictor set.', 'Reference period': "Main parent's health during the 12 months preceding the Wave 4 interview.", 'Leakage assessment': 'Temporal leakage risk. The reference period extends across post-transition experience.', 'Review note': 'Only 211 valid responses were observed in the analysis sample. The measure is not used as a source or fallback.'}]
main_parent_health_decision_indices = []
for rule in main_parent_health_source_decisions:
    matching_rows = stage_2_variable_decision_register['Wave'].eq(rule['Wave']) & stage_2_variable_decision_register['Source type'].eq(rule['Source type']) & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {rule['Wave']}, {rule['Source type']}, {rule['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    main_parent_health_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = rule['Review outcome']
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = rule['Domain']
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = rule['Decision reason']
    stage_2_variable_decision_register.loc[matching_rows, 'Reference-period assessment'] = rule['Reference period']
    stage_2_variable_decision_register.loc[matching_rows, 'Leakage assessment'] = rule['Leakage assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave 1 and Wave 4 family-background data dictionaries and Stage 2 parental-health comparison'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = rule['Review note']
main_parent_health_review_output_path = stage_2_output_directory / 'stage_2_main_parent_general_health_review.csv'
stage_2_main_parent_general_health_predictor_review.to_csv(main_parent_health_review_output_path, index=False)
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
main_parent_health_distribution = selected_main_parent_health_label.fillna('Main-parent general health unavailable').value_counts().rename_axis('Main-parent general health').reset_index(name='Participants')
main_parent_health_missing_summary = main_parent_health_missing_reason.loc[main_parent_health_missing].value_counts().rename_axis('Missing reason').reset_index(name='Participants')
main_parent_health_decision_output = stage_2_variable_decision_register.loc[main_parent_health_decision_indices,
    ['Source order', 'Wave', 'Source type', 'Variable position', 'Variable', 'Variable label', 'Review outcome',
    'Decision reason', 'Leakage assessment']].sort_values(['Source order',
    'Variable position']).drop(columns=['Source order', 'Variable position']).reset_index(drop=True)
explicit_main_parent_health_exclusions = main_parent_health_decision_output.loc[main_parent_health_decision_output['Review outcome'].eq('Exclude from predictor set')].reset_index(drop=True)
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
main_parent_health_decision_checks = pd.DataFrame([{'Quality check': 'Participants in predictor review',
    'Participants': len(stage_2_main_parent_general_health_predictor_review)}, {'Quality check': 'Valid predictor values',
    'Participants': int(selected_main_parent_health.notna().sum())}, {'Quality check': 'Missing predictor values',
    'Participants': int(selected_main_parent_health.isna().sum())}, {'Quality check': 'Values outside categories 1–4',
    'Participants': int((selected_main_parent_health.notna() & ~selected_main_parent_health.isin([1, 2, 3,
    4])).sum())}, {'Quality check': 'Duplicate NSID values',
    'Participants': int(stage_2_main_parent_general_health_predictor_review['NSID'].duplicated().sum())}])
print(f'Main-parent general-health predictor decided: {main_parent_health_predictor_name}')
print('\nPredictor distribution:')
print(main_parent_health_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMissing-value reasons:')
print(main_parent_health_missing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSource-variable decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 180, 'display.width', 450):
    print(main_parent_health_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nVariables explicitly excluded in this step: {len(explicit_main_parent_health_exclusions):,}')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 180, 'display.width', 430):
    print(explicit_main_parent_health_exclusions[['Wave', 'Source type', 'Variable', 'Variable label',
        'Decision reason', 'Leakage assessment']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(main_parent_health_decision_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'Main-parent general-health review saved to: {main_parent_health_review_output_path}')

Main-parent general-health predictor decided: main_parent_general_health

Predictor distribution:
            Main-parent general health  Participants
                             Very good          4833
                           Fairly good          3304
                         Not very good           951
Main-parent general health unavailable           352
                       Not good at all           327

Missing-value reasons:
                           Missing reason  Participants
No Wave 1 family-background source record           243
              Main parent not interviewed           103
                      Main parent refused             5
                           Not applicable             1

Source-variable decisions:
  Wave       Source type  Variable                               Variable label                Review outcome                                                                                                                                               

In [176]:
# 104: Remaining health-related pending-variable refinement

pending_variable_review = stage_2_variable_decision_register.loc[stage_2_variable_decision_register['Review outcome'].eq('Pending review')].copy().reset_index(drop=True)
pending_variable_review['Search text'] = pending_variable_review['Variable'].astype('string').fillna('').str.lower() + ' ' + pending_variable_review['Variable label'].astype('string').fillna('').str.lower()
health_candidate_patterns = {'Emotional and psychological health': '(?:mental health|emotional|emotion|psychological|wellbeing|well-being|depress|anxi|stress|worried|worry|unhappy|sad|happy|self-esteem|life satisfaction|satisfied with life|temper|mood)',
    'Physical symptoms and illness': '(?:physical health|general health|illness|long-standing illness|longstanding illness|health problem|medical condition|pain|headache|stomach|asthma|epilep|diabet|hospital|doctor|general practitioner|\\bgp\\b|medicine|medication|treatment)', 'Vision, hearing and communication': '(?:eyesight|vision|sight|blind|hearing|deaf|speech|communication)', 'Health-related behaviour': '(?:smok|cigarette|alcohol|drink|drug use|exercise|physical activity|sport|diet|food|sleep)', 'Disability and SEN wording': '(?:disab|special educational need|\\bsen\\b|statement of needs|statemented|learning difficult|additional support)', 'Other health wording': '(?:\\bhealth\\b|\\bhealthy\\b|healthcare|health care)'}
remaining_health_candidate_frames = []
already_selected = pd.Series(False, index=pending_variable_review.index, dtype=bool)
for theme, pattern in health_candidate_patterns.items():
    theme_match = pending_variable_review['Search text'].str.contains(pattern, regex=True,
        na=False) & ~already_selected
    if theme_match.any():
        theme_frame = pending_variable_review.loc[theme_match, ['Source order', 'Wave', 'Source type', 'Source file',
            'Variable position', 'Variable', 'Variable label', 'Timing status', 'Review outcome']].copy()
        theme_frame.insert(0, 'Candidate-detection theme', theme)
        remaining_health_candidate_frames.append(theme_frame)
        already_selected.loc[theme_match] = True
if remaining_health_candidate_frames:
    remaining_health_pending_candidates = pd.concat(remaining_health_candidate_frames,
        ignore_index=True).sort_values(['Candidate-detection theme', 'Source order',
        'Variable position']).reset_index(drop=True)
else:
    remaining_health_pending_candidates = pd.DataFrame(columns=['Candidate-detection theme', 'Source order', 'Wave',
        'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Timing status', 'Review outcome'])
likely_false_positive_pattern = '(?:qualification|qualifications|course|subject|career|job|occupation|industry|training course|school choice)'
remaining_health_pending_candidates['Possible wording false positive'] = remaining_health_pending_candidates['Variable label'].astype('string').fillna('').str.lower().str.contains(likely_false_positive_pattern,
    regex=True, na=False)
if not remaining_health_pending_candidates.empty:
    remaining_health_theme_summary = remaining_health_pending_candidates.groupby('Candidate-detection theme',
        dropna=False).agg(Variables=('Variable', 'size'), Source_files=('Source file',
        'nunique')).reset_index().sort_values('Variables', ascending=False).reset_index(drop=True)
    remaining_health_timing_summary = remaining_health_pending_candidates.groupby(['Candidate-detection theme', 'Wave',
        'Source type', 'Timing status'], dropna=False).size().rename('Variables').reset_index().sort_values(['Candidate-detection theme',
        'Wave', 'Source type']).reset_index(drop=True)
else:
    remaining_health_theme_summary = pd.DataFrame(columns=['Candidate-detection theme', 'Variables', 'Source_files'])
    remaining_health_timing_summary = pd.DataFrame(columns=['Candidate-detection theme', 'Wave', 'Source type',
        'Timing status', 'Variables'])
remaining_health_false_positive_summary = remaining_health_pending_candidates.loc[remaining_health_pending_candidates['Possible wording false positive'],
    ['Candidate-detection theme', 'Wave', 'Source type', 'Variable', 'Variable label']].copy().reset_index(drop=True)
non_pending_count = int((~remaining_health_pending_candidates['Review outcome'].eq('Pending review')).sum())
duplicate_candidate_count = int(remaining_health_pending_candidates[['Source file', 'Variable']].duplicated().sum())
false_positive_count = int(remaining_health_pending_candidates['Possible wording false positive'].sum())
remaining_health_screen_checks = pd.DataFrame([{'Check': 'Pending health-related candidates found',
    'Result': len(remaining_health_pending_candidates)}, {'Check': 'Distinct candidate-detection themes',
    'Result': remaining_health_pending_candidates['Candidate-detection theme'].nunique()}, {'Check': 'Possible wording false positives',
    'Result': false_positive_count}, {'Check': 'Non-pending variables included',
    'Result': non_pending_count}, {'Check': 'Duplicate source-variable entries',
    'Result': duplicate_candidate_count}])
print(f'Remaining pending health-related candidates: {len(remaining_health_pending_candidates):,}')
print('\nCandidate counts by review theme:')
print(remaining_health_theme_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nTiming and source composition:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 140, 'display.width', 400):
    print(remaining_health_timing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nFull pending candidate list:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 180, 'display.width', 460):
    print(remaining_health_pending_candidates[['Candidate-detection theme', 'Wave', 'Source type', 'Variable',
        'Variable label', 'Timing status', 'Possible wording false positive']].to_string(max_rows=TABLE_ROW_LIMIT,
        index=False))
print('\nPossible wording false positives:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 180, 'display.width', 430):
    print(remaining_health_false_positive_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nReview checks:')
print(remaining_health_screen_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Remaining pending health-related candidates: 244

Candidate counts by review theme:
        Candidate-detection theme  Variables  Source_files
    Physical symptoms and illness         72             4
         Health-related behaviour         66            11
                              ...        ...           ...
       Disability and SEN wording         28             5
Vision, hearing and communication         11             5

Timing and source composition:
        Candidate-detection theme   Wave        Source type          Timing status  Variables
       Disability and SEN wording Wave 1  Family background  Pre-transition source          1
       Disability and SEN wording Wave 1 Parental attitudes  Pre-transition source          1
                              ...    ...                ...                    ...        ...
Vision, hearing and communication Wave 4 Parental attitudes At or after transition          1
Vision, hearing and communication Wave 4       Young person 

In [177]:
# 105: Psychological-health item-set boundary review

psychological_health_anchor_variables = ['W2nosleepYP', 'W2depressYP', 'W2happyYP', 'W4NoSleepYP', 'W4DepressYP',
    'W4HappyYP']
psychological_health_anchor_entries = stage_2_master_variable_register.loc[stage_2_master_variable_register['Variable'].str.lower().isin([variable.lower() for variable in psychological_health_anchor_variables]),
    ['Source order', 'Wave', 'Source type', 'Source file', 'Source path', 'Variable position', 'Variable',
    'Variable label', 'Timing status']].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
if len(psychological_health_anchor_entries) != 6:
    raise ValueError(f'Expected six psychological-health anchor variables, but found {len(psychological_health_anchor_entries)}.')
psychological_health_anchor_entries = psychological_health_anchor_entries.merge(stage_2_variable_decision_register[['Source file',
    'Variable', 'Review outcome']], on=['Source file', 'Variable'], how='left', validate='one_to_one')
psychological_health_neighbourhood_frames = []
for _, anchor in psychological_health_anchor_entries.iterrows():
    lower_position = max(1, int(anchor['Variable position']) - 10)
    upper_position = int(anchor['Variable position']) + 10
    neighbourhood = stage_2_master_variable_register.loc[stage_2_master_variable_register['Source file'].eq(anchor['Source file']) & stage_2_master_variable_register['Variable position'].between(lower_position,
        upper_position, inclusive='both'), ['Source order', 'Wave', 'Source type', 'Source file', 'Variable position',
        'Variable', 'Variable label', 'Timing status']].copy()
    neighbourhood['Anchor variable'] = anchor['Variable']
    neighbourhood['Position relative to anchor'] = neighbourhood['Variable position'] - int(anchor['Variable position'])
    neighbourhood['Is anchor'] = neighbourhood['Variable'].str.lower().eq(anchor['Variable'].lower())
    psychological_health_neighbourhood_frames.append(neighbourhood)
psychological_health_neighbourhood = pd.concat(psychological_health_neighbourhood_frames,
    ignore_index=True).drop_duplicates(subset=['Source file', 'Variable']).sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
psychological_health_neighbourhood = psychological_health_neighbourhood.merge(stage_2_variable_decision_register[['Source file',
    'Variable', 'Review outcome']], on=['Source file', 'Variable'], how='left', validate='one_to_one')
psychological_item_wording_pattern = '(?:recently|lost much sleep|under strain|could not overcome|enjoy normal activities|face up to problems|unhappy|depressed|losing confidence|worthless|reasonably happy|concentrat|useful part|make decisions)'
psychological_health_neighbourhood['Possible psychological-health item'] = psychological_health_neighbourhood['Variable label'].astype('string').fillna('').str.lower().str.contains(psychological_item_wording_pattern,
    regex=True, na=False)
psychological_health_possible_items = psychological_health_neighbourhood.loc[psychological_health_neighbourhood['Possible psychological-health item'],
    ['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Timing status',
    'Review outcome']].copy().sort_values(['Wave', 'Variable position']).reset_index(drop=True)
psychological_health_neighbourhood_summary = psychological_health_neighbourhood.groupby(['Wave', 'Source file'],
    dropna=False).agg(Neighbourhood_variables=('Variable', 'size'),
    Possible_psychological_items=('Possible psychological-health item', 'sum'), First_position=('Variable position',
    'min'), Last_position=('Variable position', 'max')).reset_index()
psychological_health_boundary_checks = pd.DataFrame([{'Check': 'Anchor variables found',
    'Result': len(psychological_health_anchor_entries)}, {'Check': 'Anchor variables without register status',
    'Result': int(psychological_health_anchor_entries['Review outcome'].isna().sum())}, {'Check': 'Unique neighbourhood variables',
    'Result': len(psychological_health_neighbourhood)}, {'Check': 'Possible psychological-health items',
    'Result': len(psychological_health_possible_items)}, {'Check': 'Duplicate source-variable entries',
    'Result': int(psychological_health_neighbourhood[['Source file', 'Variable']].duplicated().sum())}])
print(f'Psychological-health anchor variables found: {len(psychological_health_anchor_entries):,}')
print('\nAnchor variables:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 170, 'display.width', 420):
    print(psychological_health_anchor_entries[['Wave', 'Variable position', 'Variable', 'Variable label',
        'Timing status', 'Review outcome']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nNeighbourhood summary:')
print(psychological_health_neighbourhood_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nFull variable neighbourhood:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 180, 'display.width', 470):
    print(psychological_health_neighbourhood[['Wave', 'Variable position', 'Variable', 'Variable label',
        'Possible psychological-health item', 'Review outcome']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nPossible psychological-health item set:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 180, 'display.width', 460):
    print(psychological_health_possible_items.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nReview checks:')
print(psychological_health_boundary_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Psychological-health anchor variables found: 6

Anchor variables:
  Wave  Variable position    Variable                                                       Variable label          Timing status Review outcome
Wave 2                461 W2nosleepYP               YP: Whether YP has recently lost much sleep over worry  Pre-transition source Pending review
Wave 2                468 W2depressYP      YP: How much YP has been feeling unhappy and depressed recently  Pre-transition source Pending review
   ...                ...         ...                                                                  ...                    ...            ...
Wave 4                813 W4DepressYP                    YP: Whether YP recently felt unhappy or depressed At or after transition Pending review
Wave 4                816   W4HappyYP YP: Whether YP recently felt reasonably happy, all things considered At or after transition Pending review

Neighbourhood summary:
  Wave                       Source file

In [178]:
# 106: Official GHQ-derived variable structure

ghq_derived_entries = stage_2_master_variable_register.loc[stage_2_master_variable_register['Wave'].isin(['Wave 2',
    'Wave 4']) & stage_2_master_variable_register['Source type'].eq('Young person') & (stage_2_master_variable_register['Variable'].astype('string').str.contains('(?i)ghq',
    regex=True, na=False) | stage_2_master_variable_register['Variable label'].astype('string').str.contains('(?i)ghq',
    regex=True, na=False)), ['Source order', 'Wave', 'Source type', 'Source file', 'Source path', 'Variable position',
    'Variable', 'Variable label', 'Timing status']].copy().drop_duplicates(subset=['Source file',
    'Variable']).sort_values(['Source order', 'Variable position']).reset_index(drop=True)
if ghq_derived_entries.empty:
    raise ValueError('No Wave 2 or Wave 4 GHQ-derived variables were found.')
ghq_derived_entries = ghq_derived_entries.merge(stage_2_variable_decision_register[['Source file', 'Variable',
    'Review outcome']], on=['Source file', 'Variable'], how='left', validate='one_to_one')
ghq_derived_entries['Derived measure form'] = 'Other GHQ-derived measure'
ghq_derived_entries.loc[ghq_derived_entries['Variable label'].astype('string').str.contains('(?i)12[\\s-]*point|12 point scale',
    regex=True, na=False), 'Derived measure form'] = 'GHQ-12 score: 0–12'
ghq_derived_entries.loc[ghq_derived_entries['Variable label'].astype('string').str.contains('(?i)grouped', regex=True,
    na=False), 'Derived measure form'] = 'Grouped GHQ score'
stage_2_ghq_derived_review = stage_2_participant_ids.copy()
ghq_structure_records = []
ghq_value_records = []
for source_path_text, source_entries in ghq_derived_entries.groupby('Source path', sort=False):
    source_path = Path(source_path_text)
    source_variables = source_entries['Variable'].tolist()
    numeric_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    numeric_sample = stage_2_participant_ids.merge(numeric_source, on='NSID', how='left', validate='one_to_one',
        indicator=True)
    source_record_present = numeric_sample['_merge'].eq('both')
    numeric_sample = numeric_sample.drop(columns='_merge')
    labelled_sample = stage_2_participant_ids.merge(labelled_source, on='NSID', how='left', validate='one_to_one')
    for _, entry in source_entries.iterrows():
        variable = entry['Variable']
        measure_form = entry['Derived measure form']
        numeric_values = pd.to_numeric(numeric_sample[variable], errors='coerce')
        labelled_values = labelled_sample[variable].astype('string')
        stage_2_ghq_derived_review[variable] = numeric_values
        stage_2_ghq_derived_review[f'{variable} label'] = labelled_values
        if measure_form == 'GHQ-12 score: 0–12':
            valid_values = numeric_values.between(0, 12, inclusive='both')
        elif measure_form == 'Grouped GHQ score':
            valid_values = numeric_values.isin([1, 2, 3])
        else:
            valid_values = numeric_values.ge(0)
        negative_values = numeric_values.lt(0)
        ghq_structure_records.append({'Wave': entry['Wave'], 'Variable': variable,
            'Variable label': entry['Variable label'], 'Derived measure form': measure_form, 'Timing status': entry['Timing status'], 'Current register status': entry['Review outcome'], 'Source record present': int(source_record_present.sum()), 'Source record absent': int((~source_record_present).sum()), 'Valid derived values': int(valid_values.sum()), 'Negative stored values': int(negative_values.sum()), 'Missing after source linkage': int(numeric_values.isna().sum()), 'Unique valid values': int(numeric_values.loc[valid_values].nunique()), 'Minimum valid value': numeric_values.loc[valid_values].min() if valid_values.any() else np.nan, 'Maximum valid value': numeric_values.loc[valid_values].max() if valid_values.any() else np.nan})
        value_summary = pd.DataFrame({'Stored value': numeric_values, 'Value label': labelled_values,
            'Source record present': source_record_present}).loc[lambda frame: frame['Source record present']].drop(columns='Source record present').value_counts(dropna=False).rename('Participants').reset_index()
        value_summary.insert(0, 'Wave', entry['Wave'])
        value_summary.insert(1, 'Variable', variable)
        value_summary.insert(2, 'Derived measure form', measure_form)
        ghq_value_records.append(value_summary)
ghq_derived_structure_summary = pd.DataFrame(ghq_structure_records)
ghq_derived_value_summary = pd.concat(ghq_value_records, ignore_index=True)
ghq_substantive_value_summary = ghq_derived_value_summary.loc[pd.to_numeric(ghq_derived_value_summary['Stored value'],
    errors='coerce').ge(0)].copy().reset_index(drop=True)
ghq_negative_value_summary = ghq_derived_value_summary.loc[pd.to_numeric(ghq_derived_value_summary['Stored value'],
    errors='coerce').lt(0) | ghq_derived_value_summary['Stored value'].isna()].copy().reset_index(drop=True)
ghq_group_consistency_records = []
ghq_score_group_crosstab_frames = []
for wave in ['Wave 2', 'Wave 4']:
    wave_entries = ghq_derived_entries.loc[ghq_derived_entries['Wave'].eq(wave)].copy()
    score_variables = wave_entries.loc[wave_entries['Derived measure form'].eq('GHQ-12 score: 0–12'),
        'Variable'].tolist()
    grouped_variables = wave_entries.loc[wave_entries['Derived measure form'].eq('Grouped GHQ score'),
        'Variable'].tolist()
    if len(score_variables) != 1 or len(grouped_variables) != 1:
        ghq_group_consistency_records.append({'Wave': wave,
            'Score variable': ', '.join(score_variables) if score_variables else 'Not found', 'Grouped variable': ', '.join(grouped_variables) if grouped_variables else 'Not found', 'Both values valid': 0, 'Expected grouped category agreement': 0, 'Grouped-category disagreements': 0, 'Score valid but grouped unavailable': 0, 'Grouped valid but score unavailable': 0})
        continue
    score_variable = score_variables[0]
    grouped_variable = grouped_variables[0]
    score_values = pd.to_numeric(stage_2_ghq_derived_review[score_variable], errors='coerce')
    grouped_values = pd.to_numeric(stage_2_ghq_derived_review[grouped_variable], errors='coerce')
    score_valid = score_values.between(0, 12, inclusive='both')
    grouped_valid = grouped_values.isin([1, 2, 3])
    both_valid = score_valid & grouped_valid
    expected_group = pd.Series(pd.NA, index=score_values.index, dtype='Int64')
    expected_group.loc[score_values.eq(0)] = 1
    expected_group.loc[score_values.between(1, 3, inclusive='both')] = 2
    expected_group.loc[score_values.between(4, 12, inclusive='both')] = 3
    grouped_values_int = grouped_values.astype('Int64')
    group_agreement = both_valid & grouped_values_int.eq(expected_group)
    group_disagreement = both_valid & grouped_values_int.ne(expected_group)
    ghq_group_consistency_records.append({'Wave': wave, 'Score variable': score_variable,
        'Grouped variable': grouped_variable, 'Both values valid': int(both_valid.sum()), 'Expected grouped category agreement': int(group_agreement.sum()), 'Grouped-category disagreements': int(group_disagreement.sum()), 'Score valid but grouped unavailable': int((score_valid & ~grouped_valid).sum()), 'Grouped valid but score unavailable': int((grouped_valid & ~score_valid).sum())})
    grouped_label_map = {1: 'Score 0', 2: 'Score 1–3', 3: 'Score 4+'}
    crosstab = pd.crosstab(score_values.loc[both_valid].astype('Int64'),
        grouped_values_int.loc[both_valid].map(grouped_label_map), margins=True, margins_name='Total').reset_index()
    crosstab.insert(0, 'Wave', wave)
    crosstab = crosstab.rename(columns={score_variable: 'GHQ-12 score'})
    ghq_score_group_crosstab_frames.append(crosstab)
ghq_group_consistency_summary = pd.DataFrame(ghq_group_consistency_records)
if ghq_score_group_crosstab_frames:
    ghq_score_group_crosstab = pd.concat(ghq_score_group_crosstab_frames, ignore_index=True)
else:
    ghq_score_group_crosstab = pd.DataFrame()
ghq_derived_checks = pd.DataFrame([{'Check': 'GHQ-derived variables found', 'Result': len(ghq_derived_entries)},
    {'Check': 'Wave 2 GHQ-derived variables',
    'Result': int(ghq_derived_entries['Wave'].eq('Wave 2').sum())}, {'Check': 'Wave 4 GHQ-derived variables',
    'Result': int(ghq_derived_entries['Wave'].eq('Wave 4').sum())}, {'Check': 'Variables without register status',
    'Result': int(ghq_derived_entries['Review outcome'].isna().sum())}, {'Check': 'Duplicate source-variable entries',
    'Result': int(ghq_derived_entries[['Source file',
    'Variable']].duplicated().sum())}, {'Check': 'Duplicate NSID values in participant review',
    'Result': int(stage_2_ghq_derived_review['NSID'].duplicated().sum())}])
print(f'Official GHQ-derived variables reviewed: {len(ghq_derived_entries):,}')
print('\nGHQ-derived source entries:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 180, 'display.width', 450):
    print(ghq_derived_entries[['Wave', 'Variable position', 'Variable', 'Variable label', 'Derived measure form',
        'Timing status', 'Review outcome']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nCoverage and stored-value structure:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 160, 'display.width', 450):
    print(ghq_derived_structure_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSubstantive values:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 140, 'display.width', 380):
    print(ghq_substantive_value_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nNegative and unavailable values:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 390):
    print(ghq_negative_value_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nOfficial score and grouped-variable consistency:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 150, 'display.width', 390):
    print(ghq_group_consistency_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nGHQ-12 score by grouped category:')
with pd.option_context('display.max_rows', None, 'display.width', 300):
    print(ghq_score_group_crosstab.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nReview checks:')
print(ghq_derived_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nVariables explicitly excluded in this step: 0')
print('Decision register changed in this step: No')

Official GHQ-derived variables reviewed: 4

GHQ-derived source entries:
  Wave  Variable position   Variable                                  Variable label Derived measure form          Timing status Review outcome
Wave 2                556 W2ghq12scr   DV: Young person GHQ12 score - 12 point scale   GHQ-12 score: 0–12  Pre-transition source Pending review
Wave 2                557     W2ghqg DV: Young person GHQ Score - grouped (0,1-3,4+)    Grouped GHQ score  Pre-transition source Pending review
Wave 4                999 W4ghq12scr   DV: Young person GHQ12 score - 12 point scale   GHQ-12 score: 0–12 At or after transition Pending review
Wave 4               1000     W4ghqg DV: Young person GHQ Score - grouped (0,1-3,4+)    Grouped GHQ score At or after transition Pending review

Coverage and stored-value structure:
  Wave   Variable                                  Variable label Derived measure form          Timing status Current register status  Source record present  Source recor

In [179]:
# 107: Young-person GHQ-12 predictor decision

ghq12_predictor_name = 'young_person_ghq12_score'
wave2_ghq_items = ['W2concenYP', 'W2nosleepYP', 'W2usefulYP', 'W2decideYP', 'W2strainYP', 'W2difficYP', 'W2activYP',
    'W2probsYP', 'W2depressYP', 'W2noconfYP', 'W2wthlessYP', 'W2happyYP']
wave4_ghq_items = ['W4ConcenYP', 'W4NoSleepYP', 'W4UsefulYP', 'W4DecideYP', 'W4StrainYP', 'W4DifficYP', 'W4ActivYP',
    'W4ProbsYP', 'W4DepressYP', 'W4NoConfYP', 'W4WthlessYP', 'W4HappyYP']
w2_ghq12_raw = pd.to_numeric(stage_2_ghq_derived_review['W2ghq12scr'], errors='coerce')
selected_ghq12_score = w2_ghq12_raw.where(w2_ghq12_raw.between(0, 12, inclusive='both')).astype('Int64')
selected_ghq_group_label = pd.Series(pd.NA, index=selected_ghq12_score.index, dtype='string')
selected_ghq_group_label.loc[selected_ghq12_score.eq(0).fillna(False)] = 'Score 0'
selected_ghq_group_label.loc[selected_ghq12_score.between(1, 3, inclusive='both').fillna(False)] = 'Score 1–3'
selected_ghq_group_label.loc[selected_ghq12_score.between(4, 12, inclusive='both').fillna(False)] = 'Score 4+'
ghq12_missing = selected_ghq12_score.isna()
ghq12_missing_reason = pd.Series(pd.NA, index=selected_ghq12_score.index, dtype='string')
ghq12_missing_reason.loc[ghq12_missing & w2_ghq12_raw.isna()] = 'No Wave 2 young-person source record'
ghq12_missing_reason.loc[ghq12_missing & w2_ghq12_raw.eq(-99)] = 'Young person not interviewed'
ghq12_missing_reason.loc[ghq12_missing & w2_ghq12_raw.eq(-97)] = 'Young person refused self-completion'
ghq12_missing_reason.loc[ghq12_missing & w2_ghq12_raw.eq(-96)] = 'Young person used an interpreter'
ghq12_missing_reason.loc[ghq12_missing & w2_ghq12_raw.eq(-92)] = 'Young person refused the item block'
ghq12_missing_reason.loc[ghq12_missing & ghq12_missing_reason.isna()] = 'Other unresolved GHQ-12 status'
stage_2_young_person_ghq12_predictor_review = pd.DataFrame({'NSID': stage_2_participant_ids['NSID'],
    ghq12_predictor_name: selected_ghq12_score, 'Young-person GHQ-12 grouped description': selected_ghq_group_label, 'GHQ-12 missing reason': ghq12_missing_reason, 'Predictor decision': 'Retain the official Wave 2 GHQ-12 0–12 score as one predictor'})
ghq_source_decisions = [{'Wave': 'Wave 2', 'Variable': 'W2ghq12scr', 'Review outcome': 'Retain as construction source',
    'Decision reason': 'Selected as the official pre-transition young-person GHQ-12 score. The 0–12 scale preserves more information than the grouped version.', 'Reference period': 'Recent psychological functioning reported during the Wave 2 young-person interview.', 'Leakage assessment': 'No direct outcome leakage identified. The measure was collected before transition.', 'Review note': 'The official derived score was available for 9,089 participants. Its modelling representation will be determined during preprocessing.'}, {'Wave': 'Wave 2',
    'Variable': 'W2ghqg', 'Review outcome': 'Retain as review support only', 'Decision reason': 'Official grouped version of the selected GHQ-12 score. It is not retained as a separate predictor because it is a deterministic reduction of the 0–12 score.', 'Reference period': 'Recent psychological functioning reported during the Wave 2 young-person interview.', 'Leakage assessment': 'No direct outcome leakage identified. The measure was collected before transition.', 'Review note': 'All 9,089 valid grouped values agreed with the documented grouping of the official 0–12 score.'}, {'Wave': 'Wave 4',
    'Variable': 'W4ghq12scr', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Measured at or after the post-16 transition and therefore not eligible for the pre-transition predictor set.', 'Reference period': 'Recent psychological functioning reported during the Wave 4 young-person interview.', 'Leakage assessment': 'Temporal leakage risk because psychological functioning may reflect post-transition experience.', 'Review note': 'Not used as a source, fallback or supplementary version of the Wave 2 predictor.'}, {'Wave': 'Wave 4',
    'Variable': 'W4ghqg', 'Review outcome': 'Exclude from predictor set', 'Decision reason': 'Grouped version of a psychological-health measure collected at or after the post-16 transition.', 'Reference period': 'Recent psychological functioning reported during the Wave 4 young-person interview.', 'Leakage assessment': 'Temporal leakage risk because the measure may reflect post-transition experience.', 'Review note': 'It is also a deterministic grouping of the excluded Wave 4 GHQ-12 score.'}]
for variable in wave2_ghq_items:
    ghq_source_decisions.append({'Wave': 'Wave 2', 'Variable': variable,
        'Review outcome': 'Retain as review support only', 'Decision reason': 'One of the 12 constituent items used in the official Wave 2 GHQ-12 derived score. The item is not retained separately because the official score has been selected as the predictor.', 'Reference period': 'Recent psychological functioning reported during the Wave 2 young-person interview.', 'Leakage assessment': 'No direct outcome leakage identified. The item was collected before transition.', 'Review note': 'Retained for documentation and quality review only to avoid duplicate representation of the same psychological-health construct.'})
for variable in wave4_ghq_items:
    ghq_source_decisions.append({'Wave': 'Wave 4', 'Variable': variable,
        'Review outcome': 'Exclude from predictor set', 'Decision reason': 'One of the 12 constituent Wave 4 GHQ items. It was measured at or after the post-16 transition and is not eligible as a predictor.', 'Reference period': 'Recent psychological functioning reported during the Wave 4 young-person interview.', 'Leakage assessment': 'Temporal leakage risk because the response may reflect post-transition experience.', 'Review note': 'Not used individually or as an input to predictor construction.'})
ghq_decision_indices = []
for rule in ghq_source_decisions:
    matching_rows = stage_2_variable_decision_register['Wave'].eq(rule['Wave']) & stage_2_variable_decision_register['Source type'].eq('Young person') & stage_2_variable_decision_register['Variable'].str.lower().eq(rule['Variable'].lower())
    if matching_rows.sum() != 1:
        raise ValueError(f"Expected one decision-register entry for {rule['Wave']}, Young person, {rule['Variable']}; found {matching_rows.sum()}.")
    matching_index = stage_2_variable_decision_register.loc[matching_rows].index[0]
    ghq_decision_indices.append(matching_index)
    stage_2_variable_decision_register.loc[matching_rows, 'Review outcome'] = rule['Review outcome']
    stage_2_variable_decision_register.loc[matching_rows, 'Substantive domain'] = 'Young-person psychological health'
    stage_2_variable_decision_register.loc[matching_rows, 'Decision reason'] = rule['Decision reason']
    stage_2_variable_decision_register.loc[matching_rows, 'Reference-period assessment'] = rule['Reference period']
    stage_2_variable_decision_register.loc[matching_rows, 'Leakage assessment'] = rule['Leakage assessment']
    stage_2_variable_decision_register.loc[matching_rows,
        'Documentation source'] = 'Wave 2 and Wave 4 young-person data dictionaries, official GHQ-derived variables and Stage 2 GHQ consistency review'
    stage_2_variable_decision_register.loc[matching_rows, 'Review notes'] = rule['Review note']
ghq12_review_output_path = stage_2_output_directory / 'stage_2_young_person_ghq12_review.csv'
stage_2_young_person_ghq12_predictor_review.to_csv(ghq12_review_output_path, index=False)
stage_2_variable_decision_register.to_csv(decision_register_output_path, index=False)
ghq12_score_order = [str(score) for score in range(13)] + ['Unavailable']
ghq12_score_display = selected_ghq12_score.astype('string').fillna('Unavailable')
ghq12_score_distribution = ghq12_score_display.value_counts(dropna=False).reindex(ghq12_score_order,
    fill_value=0).rename_axis('Young-person GHQ-12 score').reset_index(name='Participants')
ghq12_group_distribution = selected_ghq_group_label.fillna('GHQ-12 status unavailable').value_counts().rename_axis('Grouped description').reset_index(name='Participants')
ghq12_missing_summary = ghq12_missing_reason.loc[ghq12_missing].value_counts().rename_axis('Missing reason').reset_index(name='Participants')
ghq_decision_output = stage_2_variable_decision_register.loc[ghq_decision_indices, ['Source order', 'Wave',
    'Source type', 'Variable position', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment']].sort_values(['Source order',
    'Variable position']).drop(columns=['Source order', 'Variable position']).reset_index(drop=True)
explicit_ghq_exclusions = ghq_decision_output.loc[ghq_decision_output['Review outcome'].eq('Exclude from predictor set')].reset_index(drop=True)
decision_summary = stage_2_variable_decision_register['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
ghq12_decision_checks = pd.DataFrame([{'Quality check': 'Participants in predictor review',
    'Participants': len(stage_2_young_person_ghq12_predictor_review)}, {'Quality check': 'Valid GHQ-12 scores',
    'Participants': int(selected_ghq12_score.notna().sum())}, {'Quality check': 'Missing GHQ-12 scores',
    'Participants': int(selected_ghq12_score.isna().sum())}, {'Quality check': 'Scores outside range 0–12',
    'Participants': int((selected_ghq12_score.notna() & ~selected_ghq12_score.between(0, 12,
    inclusive='both')).sum())}, {'Quality check': 'Source variables assigned decisions',
    'Participants': len(ghq_decision_indices)}, {'Quality check': 'Duplicate source-variable decisions',
    'Participants': int(ghq_decision_output[['Wave', 'Source type',
    'Variable']].duplicated().sum())}, {'Quality check': 'Duplicate NSID values',
    'Participants': int(stage_2_young_person_ghq12_predictor_review['NSID'].duplicated().sum())}])
print(f'Young-person GHQ-12 predictor decided: {ghq12_predictor_name}')
print('\nGHQ-12 score distribution:')
print(ghq12_score_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nGrouped description:')
print(ghq12_group_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nMissing-value reasons:')
print(ghq12_missing_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nSource-variable decisions:')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 180, 'display.width', 470):
    print(ghq_decision_output.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nVariables explicitly excluded in this step: {len(explicit_ghq_exclusions):,}')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 180, 'display.width', 450):
    print(explicit_ghq_exclusions[['Wave', 'Variable', 'Variable label', 'Decision reason',
        'Leakage assessment']].to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nQuality checks:')
print(ghq12_decision_checks.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nOverall decision-register status:')
print(decision_summary.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print(f'\nDecision register saved to: {decision_register_output_path}')
print(f'GHQ-12 review saved to: {ghq12_review_output_path}')

Young-person GHQ-12 predictor decided: young_person_ghq12_score

GHQ-12 score distribution:
Young-person GHQ-12 score  Participants
                        0          4444
                        1          1505
                      ...           ...
                       12            30
              Unavailable           678

Grouped description:
      Grouped description  Participants
                  Score 0          4444
                Score 1–3          2963
                 Score 4+          1682
GHQ-12 status unavailable           678

Missing-value reasons:
                      Missing reason  Participants
 Young person refused the item block           256
No Wave 2 young-person source record           246
        Young person not interviewed            76
Young person refused self-completion            54
    Young person used an interpreter            46

Source-variable decisions:
  Wave  Source type    Variable                                         Variable label  

## Master variable register

The register contains 5,261 variables from 14 source files. No duplicate source–variable pairs or missing variable labels were found. Administrative fields, survey-design variables, timing restrictions, routing and possible overlap were recorded before domain review.

The register was saved as `stage_2_master_variable_register.csv`. Domain decisions were added to the separate variable decision register.


# Part 4: Demographic characteristics

Sex, birth timing, ethnicity and language measures were compared across available sources. Selection considered timing, coding, coverage and whether alternative measures represented the same construct.


In [180]:
# 1: Demographic-background review files

demographic_review_files = {'Sex': 'stage_2_sex_measure_review.csv',
    'Birth timing': 'stage_2_birth_measure_review.csv', 'Ethnicity': 'stage_2_ethnicity_measure_review.csv', 'Language background': 'stage_2_language_measure_review.csv'}
demographic_review_data = {}
demographic_file_checks = []
for construct, file_name in demographic_review_files.items():
    file_path = stage_2_output_directory / file_name
    if not file_path.exists():
        raise FileNotFoundError(f'Review file not found: {file_path}')
    review_data = pd.read_csv(file_path)
    demographic_review_data[construct] = review_data
    demographic_file_checks.append({'Construct': construct, 'Rows': len(review_data), 'Columns': review_data.shape[1],
        'Unique NSID': review_data['NSID'].nunique(), 'Duplicate NSID': review_data['NSID'].duplicated().sum(), 'Missing NSID': review_data['NSID'].isna().sum()})
demographic_file_checks = pd.DataFrame(demographic_file_checks)
expected_participants = 9767
demographic_file_checks['Expected rows'] = demographic_file_checks['Rows'].eq(expected_participants)
demographic_file_checks['Identifier check'] = demographic_file_checks['Unique NSID'].eq(expected_participants) & demographic_file_checks['Duplicate NSID'].eq(0) & demographic_file_checks['Missing NSID'].eq(0)
print(f'Demographic review files loaded: {len(demographic_review_data):,}')
print(f"All files retain 9,767 participants: {demographic_file_checks['Expected rows'].all()}")
print(f"All identifier checks passed: {demographic_file_checks['Identifier check'].all()}")
display_limited(demographic_file_checks)

Demographic review files loaded: 4
All files retain 9,767 participants: True
All identifier checks passed: True


,Construct,Rows,Columns,Unique NSID,Duplicate NSID,Missing NSID,Expected rows,Identifier check
0,Sex,9767,6,9767,0,0,True,True
1,Birth timing,9767,6,9767,0,0,True,True
2,Ethnicity,9767,5,9767,0,0,True,True
3,Language background,9767,8,9767,0,0,True,True


In [181]:
# 2: Demographic review-file contents

demographic_column_rows = []
for construct, review_data in demographic_review_data.items():
    for column_name in review_data.columns:
        demographic_column_rows.append({'Construct': construct, 'Column': column_name,
            'Data type': str(review_data[column_name].dtype), 'Non-missing': int(review_data[column_name].notna().sum()), 'Missing': int(review_data[column_name].isna().sum()), 'Unique values': int(review_data[column_name].nunique(dropna=True))})
demographic_column_review = pd.DataFrame(demographic_column_rows)
print(f'Columns reviewed across four files: {len(demographic_column_review):,}')
display_limited(demographic_column_review)

Columns reviewed across four files: 25


,Construct,Column,Data type,Non-missing,Missing,Unique values
0,Sex,NSID,str,9767,0,9767
1,Sex,Sex code,float64,9758,9,2
2,Sex,Sex,str,9758,9,2
3,Sex,Sex source,str,9758,9,4
4,Sex,Valid sex records,int64,9767,0,5


In [182]:
# 3: Demographic predictor candidates

sex_candidate = demographic_review_data['Sex'][['NSID', 'Sex']].rename(columns={'Sex': 'sex'})
birth_candidate = demographic_review_data['Birth timing'][['NSID', 'birth_month_position']]
ethnicity_candidate = demographic_review_data['Ethnicity'][['NSID',
    'Ethnicity']].rename(columns={'Ethnicity': 'ethnicity'})
language_candidates = demographic_review_data['Language background'][['NSID', 'English first or main language',
    'Non-English home language recorded']].rename(columns={'English first or main language': 'english_first_or_main_language',
    'Non-English home language recorded': 'non_english_home_language'})
demographic_candidates = sex_candidate.merge(birth_candidate, on='NSID', how='outer',
    validate='one_to_one').merge(ethnicity_candidate, on='NSID', how='outer',
    validate='one_to_one').merge(language_candidates, on='NSID', how='outer', validate='one_to_one')
demographic_candidate_summary = pd.DataFrame({'Predictor': [column for column in demographic_candidates.columns if column != 'NSID']})
demographic_candidate_summary['Non-missing'] = demographic_candidate_summary['Predictor'].map(demographic_candidates.notna().sum())
demographic_candidate_summary['Missing'] = demographic_candidate_summary['Predictor'].map(demographic_candidates.isna().sum())
demographic_candidate_summary['Missing percentage'] = demographic_candidate_summary['Missing'].div(len(demographic_candidates)).mul(100).round(2)
demographic_candidate_summary['Unique values'] = demographic_candidate_summary['Predictor'].map(lambda column: demographic_candidates[column].nunique(dropna=True))
print(f'Participants: {len(demographic_candidates):,}')
print(f"Unique NSID: {demographic_candidates['NSID'].nunique():,}")
print(f"Duplicate NSID: {demographic_candidates['NSID'].duplicated().sum():,}")
print(f"Missing NSID: {demographic_candidates['NSID'].isna().sum():,}")
display_limited(demographic_candidate_summary)

Participants: 9,767
Unique NSID: 9,767
Duplicate NSID: 0
Missing NSID: 0


,Predictor,Non-missing,Missing,Missing percentage,Unique values
0,sex,9758,9,0.09,2
1,birth_month_position,9524,243,2.49,28
2,ethnicity,9763,4,0.04,8
3,english_first_or_main_language,9516,251,2.57,2
4,non_english_home_language,9670,97,0.99,2


In [183]:
# 4: Language-background overlap

language_comparison = demographic_candidates[['NSID', 'english_first_or_main_language',
    'non_english_home_language']].copy()
language_comparison['english_first_or_main_language'] = language_comparison['english_first_or_main_language'].fillna('Unavailable')
language_comparison['non_english_home_language'] = language_comparison['non_english_home_language'].fillna('Unavailable')
personal_language_counts = language_comparison['english_first_or_main_language'].value_counts(dropna=False).rename_axis('English first or main language').reset_index(name='Participants')
home_language_counts = language_comparison['non_english_home_language'].value_counts(dropna=False).rename_axis('Non-English home language recorded').reset_index(name='Participants')
language_cross_tabulation = pd.crosstab(language_comparison['english_first_or_main_language'],
    language_comparison['non_english_home_language'], margins=True)
print('English first or main language:')
display_limited(personal_language_counts)
print('Non-English home language recorded:')
display_limited(home_language_counts)
print('Comparison of the two language measures:')
display_limited(language_cross_tabulation)

English first or main language:


,English first or main language,Participants
0,Yes,8959
1,No,557
2,Unavailable,251


Non-English home language recorded:


,Non-English home language recorded,Participants
0,No,8393
1,Yes,1277
2,Unavailable,97


Comparison of the two language measures:


non_english_home_language,No,Unavailable,Yes,All
english_first_or_main_language,,,,
No,57,21,479,557
Unavailable,144,29,78,251
Yes,8192,47,720,8959
All,8393,97,1277,9767


In [184]:
# 5: Language-background combinations

language_pattern_data = demographic_candidates[['NSID', 'english_first_or_main_language',
    'non_english_home_language']].copy()
language_pattern_data['Both measures available'] = language_pattern_data['english_first_or_main_language'].notna() & language_pattern_data['non_english_home_language'].notna()
language_pattern_data['Language profile'] = 'Incomplete language information'
complete_language = language_pattern_data['Both measures available']
language_pattern_data.loc[complete_language & language_pattern_data['english_first_or_main_language'].eq('Yes') & language_pattern_data['non_english_home_language'].eq('No'),
    'Language profile'] = 'English first or main language; no non-English home language'
language_pattern_data.loc[complete_language & language_pattern_data['english_first_or_main_language'].eq('Yes') & language_pattern_data['non_english_home_language'].eq('Yes'),
    'Language profile'] = 'English first or main language; non-English home language'
language_pattern_data.loc[complete_language & language_pattern_data['english_first_or_main_language'].eq('No') & language_pattern_data['non_english_home_language'].eq('Yes'),
    'Language profile'] = 'English not first or main language; non-English home language'
language_pattern_data.loc[complete_language & language_pattern_data['english_first_or_main_language'].eq('No') & language_pattern_data['non_english_home_language'].eq('No'),
    'Language profile'] = 'English not first or main language; no non-English home language'
language_profile_summary = language_pattern_data['Language profile'].value_counts(dropna=False).rename_axis('Language profile').reset_index(name='Participants')
language_profile_summary['Percentage'] = language_profile_summary['Participants'].div(len(language_pattern_data)).mul(100).round(2)
language_missing_summary = pd.DataFrame({'Missing pattern': ['Neither measure missing',
    'Personal language only missing', 'Home language only missing', 'Both measures missing'], 'Participants': [(language_pattern_data['english_first_or_main_language'].notna() & language_pattern_data['non_english_home_language'].notna()).sum(),
    (language_pattern_data['english_first_or_main_language'].isna() & language_pattern_data['non_english_home_language'].notna()).sum(), (language_pattern_data['english_first_or_main_language'].notna() & language_pattern_data['non_english_home_language'].isna()).sum(), (language_pattern_data['english_first_or_main_language'].isna() & language_pattern_data['non_english_home_language'].isna()).sum()]})
print('Language-background combinations:')
display_limited(language_profile_summary)
print('Language-measure availability:')
display_limited(language_missing_summary)

Language-background combinations:


,Language profile,Participants,Percentage
0,English first or main language; no non-English...,8192,83.87
1,English first or main language; non-English ho...,720,7.37
2,English not first or main language; non-Englis...,479,4.90
3,Incomplete language information,319,3.27
4,English not first or main language; no non-Eng...,57,0.58


Language-measure availability:


,Missing pattern,Participants
0,Neither measure missing,9448
1,Personal language only missing,222
2,Home language only missing,68
3,Both measures missing,29


In [185]:
# 6: Demographic predictor overlap

from scipy.stats import chi2_contingency
import numpy as np

def cramers_v(first_measure, second_measure):
    complete_data = pd.DataFrame({'first_measure': first_measure, 'second_measure': second_measure}).dropna()
    contingency_table = pd.crosstab(complete_data['first_measure'], complete_data['second_measure'])
    chi_square = chi2_contingency(contingency_table, correction=False)[0]
    participants = contingency_table.to_numpy().sum()
    smaller_dimension = min(contingency_table.shape) - 1
    if smaller_dimension == 0:
        return (np.nan, participants)
    association = np.sqrt(chi_square / (participants * smaller_dimension))
    return (association, participants)
demographic_overlap_pairs = [('English first or main language', 'Non-English home language',
    'english_first_or_main_language', 'non_english_home_language'), ('Ethnicity', 'English first or main language',
    'ethnicity', 'english_first_or_main_language'), ('Ethnicity', 'Non-English home language', 'ethnicity',
    'non_english_home_language')]
demographic_overlap_rows = []
for first_label, second_label, first_column, second_column in demographic_overlap_pairs:
    association, participants = cramers_v(demographic_candidates[first_column], demographic_candidates[second_column])
    demographic_overlap_rows.append({'First measure': first_label, 'Second measure': second_label,
        'Complete cases': participants, "Cramér's V": round(association, 4)})
demographic_overlap_review = pd.DataFrame(demographic_overlap_rows)
display_limited(demographic_overlap_review)

,First measure,Second measure,Complete cases,Cramér's V
0,English first or main language,Non-English home language,9448,0.5649
1,Ethnicity,English first or main language,9514,0.4676
2,Ethnicity,Non-English home language,9666,0.7126


In [186]:
# 7: Demographic-domain decisions

demographic_predictors = demographic_candidates[['NSID', 'sex', 'birth_month_position', 'ethnicity',
    'english_first_or_main_language', 'non_english_home_language']].copy()
demographic_predictor_decisions = pd.DataFrame([{'Predictor': 'sex', 'Construct': 'Sex',
    'Decision': 'Retain as predictor', 'Timing assessment': 'Pre-transition', 'Overlap assessment': 'Distinct demographic characteristic', 'Decision reason': 'Core demographic characteristic with a clear participant-level measure'}, {'Predictor': 'birth_month_position',
    'Construct': 'Relative age within the cohort', 'Decision': 'Retain as predictor', 'Timing assessment': 'Fixed before transition', 'Overlap assessment': 'Not represented by another retained predictor', 'Decision reason': 'Represents birth timing within the cohort rather than chronological age at interview'}, {'Predictor': 'ethnicity',
    'Construct': 'Ethnic group', 'Decision': 'Retain as predictor', 'Timing assessment': 'Stable background characteristic', 'Overlap assessment': 'Related to language background but not an equivalent measure', 'Decision reason': 'Provides demographic information not captured directly by language measures'}, {'Predictor': 'english_first_or_main_language',
    'Construct': 'Personal language background', 'Decision': 'Retain as predictor', 'Timing assessment': 'Pre-transition', 'Overlap assessment': 'Moderately related to ethnicity and home language but not duplicated', 'Decision reason': "Identifies whether English is the young person's first or main language"}, {'Predictor': 'non_english_home_language',
    'Construct': 'Household language environment', 'Decision': 'Retain as predictor', 'Timing assessment': 'Pre-transition or stable household background', 'Overlap assessment': 'Strongly related to ethnicity but distinguishes bilingual home contexts', 'Decision reason': 'Captures household language use separately from personal first or main language'}])
assert len(demographic_predictors) == 9767
assert demographic_predictors['NSID'].nunique() == 9767
assert demographic_predictors['NSID'].duplicated().sum() == 0
assert demographic_predictors['NSID'].isna().sum() == 0
demographic_predictor_path = stage_2_output_directory / 'stage_2_demographic_domain_predictors.csv'
demographic_decision_path = stage_2_output_directory / 'stage_2_demographic_domain_decisions.csv'
demographic_predictors.to_csv(demographic_predictor_path, index=False)
demographic_predictor_decisions.to_csv(demographic_decision_path, index=False)
print(f'Participants saved: {len(demographic_predictors):,}')
print(f'Predictors retained: {demographic_predictors.shape[1] - 1:,}')
print(f'Predictor file: {demographic_predictor_path}')
print(f'Decision file: {demographic_decision_path}')
display_limited(demographic_predictor_decisions)

Participants saved: 9,767
Predictors retained: 5
Predictor file: data_derived\stage_2_predictor_construction\stage_2_demographic_domain_predictors.csv
Decision file: data_derived\stage_2_predictor_construction\stage_2_demographic_domain_decisions.csv


,Predictor,Construct,Decision,Timing assessment,Overlap assessment,Decision reason
0,sex,Sex,Retain as predictor,Pre-transition,Distinct demographic characteristic,Core demographic characteristic with a clear p...
1,birth_month_position,Relative age within the cohort,Retain as predictor,Fixed before transition,Not represented by another retained predictor,Represents birth timing within the cohort rath...
2,ethnicity,Ethnic group,Retain as predictor,Stable background characteristic,Related to language background but not an equi...,Provides demographic information not captured ...
3,english_first_or_main_language,Personal language background,Retain as predictor,Pre-transition,Moderately related to ethnicity and home langu...,Identifies whether English is the young person...
4,non_english_home_language,Household language environment,Retain as predictor,Pre-transition or stable household background,Strongly related to ethnicity but distinguishe...,Captures household language use separately fro...


## Demographic predictors

Five predictors were retained: `sex`, `birth_month_position`, `ethnicity`, `english_first_or_main_language` and `non_english_home_language`. The two language predictors were kept separately because one concerns the young person’s first or main language and the other concerns language use at home.

The domain file contains all 9,767 participants with unique identifiers. Predictor and source-decision files were saved for consolidation.


# Part 5: Family socioeconomic circumstances

Parental education, occupational position, household income, housing tenure, family composition and parental employment were reviewed across the family and parental files. Alternative measures were compared for timing, coding, coverage and overlap.


In [187]:
# 1: Parental-education candidate variables

decision_register_path = decision_register_output_path
variable_decision_register = stage_2_variable_decision_register.copy()
family_parent_source = variable_decision_register['Source type'].astype(str).str.contains('family|parent', case=False,
    na=False)
parental_education_wording = variable_decision_register['Variable'].astype(str).str.contains('hiqual', case=False,
    na=False) | variable_decision_register['Variable label'].astype(str).str.contains('highest qualification|highest educational qualification|qualification level',
    case=False, na=False)
parental_education_candidates = variable_decision_register.loc[family_parent_source & parental_education_wording,
    ['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Review outcome',
    'Decision reason']].sort_values(['Wave', 'Source type', 'Source file',
    'Variable position']).reset_index(drop=True)
print(f'Parental-education candidate variables found: {len(parental_education_candidates):,}')
display_limited(parental_education_candidates)

Parental-education candidate variables found: 28


,Wave,Source type,Source file,Variable position,Variable,Variable label,Review outcome,Decision reason
0,Wave 1,Family background,wave_one_lsype_family_background_2020,341,W1hiqualgMP,DV: Highest qualification held by main parent ...,Pending review,<NA>
1,Wave 1,Family background,wave_one_lsype_family_background_2020,342,W1hiqualgSP,DV: Highest qualification held by second paren...,Pending review,<NA>
2,Wave 1,Family background,wave_one_lsype_family_background_2020,343,W1hiqualgmum,DV: Mother's highest qualification (grouped),Pending review,<NA>
3,Wave 1,Family background,wave_one_lsype_family_background_2020,344,W1hiqualgdad,DV: Father's highest qualification (grouped),Pending review,<NA>
4,Wave 1,Family background,wave_one_lsype_family_background_2020,345,W1hiqualMP,DV: Main parent's highest qualification held (...,Pending review,<NA>


In [188]:
# 2: Source-register structure check

source_register_path = source_register_file
source_register = stage_2_source_file_register.copy()
source_register_columns = pd.DataFrame({'Column position': range(1, len(source_register.columns) + 1),
    'Column name': source_register.columns, 'Data type': [str(source_register[column].dtype) for column in source_register.columns]})
print(f'Source-register rows: {len(source_register):,}')
print(f'Source-register columns: {source_register.shape[1]:,}')
display_limited(source_register_columns)
display_limited(source_register.head())

Source-register rows: 14
Source-register columns: 17


,Column position,Column name,Data type
0,1,Wave,str
1,2,Source type,str
2,3,Classification basis,str
3,4,File name,str
4,5,Relative path,str


,Wave,Source type,Classification basis,File name,Relative path,Rows,Variables,NSID present,Unique NSID values,Missing NSID values,Duplicate NSID rows,Data structure,Roster participants present,Roster participants absent,Coverage percentage,Timing status,Review status
0,Wave 1,Young person,File name,wave_one_lsype_young_person_2020.dta,wave_one_lsype_young_person_2020.dta,15760,350,True,15760,0,0,One row per NSID,9524,243,97.51,Pre-transition source,"Variable-level coding, routing and reference-p..."
1,Wave 1,Family background,File name,wave_one_lsype_family_background_2020.dta,wave_one_lsype_family_background_2020.dta,15760,381,True,15760,0,0,One row per NSID,9524,243,97.51,Pre-transition source,"Variable-level coding, routing and reference-p..."
2,Wave 1,Parental attitudes,File name,wave_one_lsype_parental_attitudes_file_16_05_0...,wave_one_lsype_parental_attitudes_file_16_05_0...,15760,303,True,15760,0,0,One row per NSID,9524,243,97.51,Pre-transition source,"Variable-level coding, routing and reference-p..."
3,Wave 2,Young person,File name,wave_two_lsype_young_person_2020.dta,wave_two_lsype_young_person_2020.dta,13530,564,True,13530,0,0,One row per NSID,9521,246,97.48,Pre-transition source,"Variable-level coding, routing and reference-p..."
4,Wave 2,Family background,File name,wave_two_lsype_family_background_2020.dta,wave_two_lsype_family_background_2020.dta,13530,1030,True,13530,0,0,One row per NSID,9521,246,97.48,Pre-transition source,"Variable-level coding, routing and reference-p..."


In [189]:
# 3: Grouped parental-qualification coverage

source_file_lookup = {Path(str(file_name)).stem: source_directory / str(file_name) for file_name in source_register['File name']}
grouped_parental_education = parental_education_candidates.loc[parental_education_candidates['Variable'].str.contains('hiqualg',
    case=False, na=False)].copy().reset_index(drop=True)
stage_2_id_data = pd.read_csv(stage_1_outcome_file, usecols=['NSID'], dtype={'NSID': 'string'})
assert stage_2_id_data.columns.tolist() == ['NSID']
stage_2_ids = stage_2_id_data['NSID'].str.strip().str.replace('\\.0$', '', regex=True)
parental_education_rows = []
for source_file, candidate_group in grouped_parental_education.groupby('Source file', sort=False):
    source_key = Path(str(source_file)).stem
    if source_key not in source_file_lookup:
        raise KeyError(f'Source file not found in the register: {source_file}')
    source_path = source_file_lookup[source_key]
    if not source_path.exists():
        raise FileNotFoundError(f'Stata file not found: {source_path}')
    variable_names = candidate_group['Variable'].tolist()
    source_data = pd.read_stata(source_path, columns=['NSID'] + variable_names, convert_categoricals=False)
    source_data['NSID'] = source_data['NSID'].astype('string').str.strip().str.replace('\\.0$', '', regex=True)
    source_data = source_data.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)
    for _, candidate in candidate_group.iterrows():
        variable = candidate['Variable']
        values = pd.to_numeric(source_data[variable], errors='coerce')
        valid_values = values[values.ge(0)]
        negative_values = values[values.lt(0)]
        parental_education_rows.append({'Wave': candidate['Wave'], 'Variable': variable,
            'Variable label': candidate['Variable label'], 'Valid responses': int(valid_values.notna().sum()), 'Negative codes': int(negative_values.notna().sum()), 'No source record': int(values.isna().sum()), 'Valid categories': int(valid_values.nunique()), 'Observed valid codes': ', '.join((str(value) for value in sorted(valid_values.dropna().unique())))})
grouped_parental_education_review = pd.DataFrame(parental_education_rows)
print(f'Grouped parental-qualification measures reviewed: {len(grouped_parental_education_review):,}')
display_limited(grouped_parental_education_review)

Grouped parental-qualification measures reviewed: 14


,Wave,Variable,Variable label,Valid responses,Negative codes,No source record,Valid categories,Observed valid codes
0,Wave 1,W1hiqualgMP,DV: Highest qualification held by main parent ...,9153,371,243,7,"1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0"
1,Wave 1,W1hiqualgSP,DV: Highest qualification held by second paren...,6396,3128,243,7,"1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0"
2,Wave 1,W1hiqualgmum,DV: Mother's highest qualification (grouped),8767,757,243,7,"1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0"
3,Wave 1,W1hiqualgdad,DV: Father's highest qualification (grouped),6611,2913,243,7,"1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0"
4,Wave 2,W2hiqualgMP,DV: Highest qualification held by main parent ...,9484,37,246,7,"1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0"


In [190]:
# 4: Wave 2 parental-qualification missingness

w2_family_file_name = source_register.loc[source_register['Wave'].eq('Wave 2') & source_register['Source type'].eq('Family background'),
    'File name'].iloc[0]
w2_family_path = source_directory / w2_family_file_name
w2_parental_qualification_variables = ['W2hiqualgMP', 'W2hiqualgSP', 'W2hiqualgmum', 'W2hiqualgdad', 'W2hiqualgfam']
w2_parental_qualification_data = pd.read_stata(w2_family_path, columns=['NSID'] + w2_parental_qualification_variables,
    convert_categoricals=False)
w2_parental_qualification_data['NSID'] = w2_parental_qualification_data['NSID'].astype('string').str.strip().str.replace('\\.0$',
    '', regex=True)
w2_parental_qualification_data = w2_parental_qualification_data.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)
negative_codes = [-999, -99, -98, -94, -92, -91, -1]
negative_code_rows = []
for variable in w2_parental_qualification_variables:
    values = pd.to_numeric(w2_parental_qualification_data[variable], errors='coerce')
    row = {'Variable': variable, 'Valid codes 1–7': int(values.between(1, 7).sum()),
        'No source record': int(values.isna().sum())}
    for code in negative_codes:
        row[f'Code {code}'] = int(values.eq(code).sum())
    negative_code_rows.append(row)
w2_parental_qualification_missingness = pd.DataFrame(negative_code_rows)
main_parent_valid = w2_parental_qualification_data['W2hiqualgMP'].between(1, 7)
second_parent_valid = w2_parental_qualification_data['W2hiqualgSP'].between(1, 7)
family_measure_valid = w2_parental_qualification_data['W2hiqualgfam'].between(1, 7)
family_measure_availability = pd.DataFrame({'Condition': ['Official family measure valid',
    'Family measure unavailable; main parent valid', 'Family measure unavailable; second parent valid', 'Family measure unavailable; either parent valid', 'Family measure unavailable; neither parent valid'], 'Participants': [int(family_measure_valid.sum()),
    int((~family_measure_valid & main_parent_valid).sum()), int((~family_measure_valid & second_parent_valid).sum()), int((~family_measure_valid & (main_parent_valid | second_parent_valid)).sum()), int((~family_measure_valid & ~main_parent_valid & ~second_parent_valid).sum())]})
print('Negative-code structure:')
display_limited(w2_parental_qualification_missingness)
print('Official family-measure availability:')
display_limited(family_measure_availability)

Negative-code structure:


,Variable,Valid codes 1–7,No source record,Code -999,Code -99,Code -98,Code -94,Code -92,Code -91,Code -1
0,W2hiqualgMP,9484,246,0,23,0,14,0,0,0
1,W2hiqualgSP,6961,246,0,409,2143,8,0,0,0
2,W2hiqualgmum,9156,246,0,156,204,5,0,0,0
3,W2hiqualgdad,7142,246,0,600,1772,7,0,0,0
4,W2hiqualgfam,9080,246,0,0,0,441,0,0,0


Official family-measure availability:


,Condition,Participants
0,Official family measure valid,9080
1,Family measure unavailable; main parent valid,404
2,Family measure unavailable; second parent valid,10
3,Family measure unavailable; either parent valid,414
4,Family measure unavailable; neither parent valid,273


In [191]:
# 5: Available-parent qualification measure

w2_parental_qualification_candidate = w2_parental_qualification_data[['W2hiqualgMP', 'W2hiqualgSP',
    'W2hiqualgfam']].copy()
main_parent_qualification = pd.to_numeric(w2_parental_qualification_candidate['W2hiqualgMP'],
    errors='coerce').where(lambda values: values.between(1, 7))
second_parent_qualification = pd.to_numeric(w2_parental_qualification_candidate['W2hiqualgSP'],
    errors='coerce').where(lambda values: values.between(1, 7))
official_family_qualification = pd.to_numeric(w2_parental_qualification_candidate['W2hiqualgfam'],
    errors='coerce').where(lambda values: values.between(1, 7))
available_parent_qualification = pd.concat([main_parent_qualification, second_parent_qualification],
    axis=1).min(axis=1, skipna=True)
qualification_source = pd.Series('No valid parental qualification', index=w2_parental_qualification_candidate.index,
    dtype='string')
qualification_source.loc[main_parent_qualification.notna() & second_parent_qualification.isna()] = 'Main parent only'
qualification_source.loc[main_parent_qualification.isna() & second_parent_qualification.notna()] = 'Second parent only'
qualification_source.loc[main_parent_qualification.notna() & second_parent_qualification.notna()] = 'Both parents'
official_available = official_family_qualification.notna()
agreement_count = available_parent_qualification.loc[official_available].eq(official_family_qualification.loc[official_available]).sum()
comparison_summary = pd.DataFrame({'Measure': ['Official Wave 2 family qualification',
    'Available-parent qualification'], 'Valid participants': [int(official_family_qualification.notna().sum()),
    int(available_parent_qualification.notna().sum())], 'Missing participants': [int(official_family_qualification.isna().sum()),
    int(available_parent_qualification.isna().sum())]})
source_summary = qualification_source.value_counts(dropna=False).rename_axis('Information source').reset_index(name='Participants')
print(f'Agreement where the official family measure is available: {agreement_count:,} of {official_available.sum():,}')
print(f'Exact agreement percentage: {agreement_count / official_available.sum() * 100:.2f}%')
display_limited(comparison_summary)
display_limited(source_summary)

Agreement where the official family measure is available: 9,080 of 9,080
Exact agreement percentage: 100.00%


,Measure,Valid participants,Missing participants
0,Official Wave 2 family qualification,9080,687
1,Available-parent qualification,9494,273


,Information source,Participants
0,Both parents,6951
1,Main parent only,2533
2,No valid parental qualification,273
3,Second parent only,10


In [192]:
# 6: Cross-wave parental-qualification comparison

w1_family_file_name = source_register.loc[source_register['Wave'].eq('Wave 1') & source_register['Source type'].eq('Family background'),
    'File name'].iloc[0]
w1_family_path = source_directory / w1_family_file_name
w1_parental_qualification_data = pd.read_stata(w1_family_path, columns=['NSID', 'W1hiqualgMP', 'W1hiqualgSP'],
    convert_categoricals=False)
w1_parental_qualification_data['NSID'] = w1_parental_qualification_data['NSID'].astype('string').str.strip().str.replace('\\.0$',
    '', regex=True)
w1_parental_qualification_data = w1_parental_qualification_data.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)
w1_main_parent_qualification = pd.to_numeric(w1_parental_qualification_data['W1hiqualgMP'],
    errors='coerce').where(lambda values: values.between(1, 7))
w1_second_parent_qualification = pd.to_numeric(w1_parental_qualification_data['W1hiqualgSP'],
    errors='coerce').where(lambda values: values.between(1, 7))
w1_available_parent_qualification = pd.concat([w1_main_parent_qualification, w1_second_parent_qualification],
    axis=1).min(axis=1, skipna=True)
both_waves_available = w1_available_parent_qualification.notna() & available_parent_qualification.notna()
wave_agreement = w1_available_parent_qualification.loc[both_waves_available].eq(available_parent_qualification.loc[both_waves_available])
wave_direction = pd.Series(pd.NA, index=stage_2_ids, dtype='string')
wave_direction.loc[both_waves_available & available_parent_qualification.eq(w1_available_parent_qualification)] = 'Same category'
wave_direction.loc[both_waves_available & available_parent_qualification.lt(w1_available_parent_qualification)] = 'Higher qualification category at Wave 2'
wave_direction.loc[both_waves_available & available_parent_qualification.gt(w1_available_parent_qualification)] = 'Lower qualification category at Wave 2'
cross_wave_parental_qualification = available_parent_qualification.combine_first(w1_available_parent_qualification)
cross_wave_source = pd.Series('Unavailable in Waves 1 and 2', index=stage_2_ids, dtype='string')
cross_wave_source.loc[available_parent_qualification.notna()] = 'Wave 2'
cross_wave_source.loc[available_parent_qualification.isna() & w1_available_parent_qualification.notna()] = 'Wave 1 fallback'
cross_wave_summary = pd.DataFrame({'Measure or condition': ['Wave 1 available-parent measure',
    'Wave 2 available-parent measure', 'Available in both waves', 'Exact agreement across waves', 'Wave 2 unavailable; Wave 1 available', 'Unavailable in both waves', 'Combined Wave 2 with Wave 1 fallback'], 'Participants': [int(w1_available_parent_qualification.notna().sum()),
    int(available_parent_qualification.notna().sum()), int(both_waves_available.sum()), int(wave_agreement.sum()), int((available_parent_qualification.isna() & w1_available_parent_qualification.notna()).sum()), int((available_parent_qualification.isna() & w1_available_parent_qualification.isna()).sum()), int(cross_wave_parental_qualification.notna().sum())]})
wave_direction_summary = wave_direction.value_counts(dropna=True).rename_axis('Cross-wave comparison').reset_index(name='Participants')
source_summary = cross_wave_source.value_counts(dropna=False).rename_axis('Selected source').reset_index(name='Participants')
print(f'Exact agreement among participants observed in both waves: {wave_agreement.sum():,} of {both_waves_available.sum():,}')
print(f'Agreement percentage: {wave_agreement.mean() * 100:.2f}%')
display_limited(cross_wave_summary)
display_limited(wave_direction_summary)
display_limited(source_summary)

Exact agreement among participants observed in both waves: 8,682 of 9,337
Agreement percentage: 92.98%


,Measure or condition,Participants
0,Wave 1 available-parent measure,9353
1,Wave 2 available-parent measure,9494
2,Available in both waves,9337
3,Exact agreement across waves,8682
4,Wave 2 unavailable; Wave 1 available,16


,Cross-wave comparison,Participants
0,Same category,8682
1,Higher qualification category at Wave 2,588
2,Lower qualification category at Wave 2,67


,Selected source,Participants
0,Wave 2,9494
1,Unavailable in Waves 1 and 2,257
2,Wave 1 fallback,16


In [193]:
# 7: Highest parental-qualification construction

from pathlib import Path
import pandas as pd
parental_qualification_labels = {1: 'Degree or equivalent', 2: 'Higher education below degree level',
    3: 'GCE A Level or equivalent', 4: 'GCSE grades A-C or equivalent', 5: 'Qualifications at level 1 and below', 6: 'Other qualifications', 7: 'No qualification'}
parental_qualification_review = pd.DataFrame({'NSID': stage_2_ids.to_numpy(),
    'highest_parental_qualification_code': cross_wave_parental_qualification.astype('Int64').to_numpy(), 'highest_parental_qualification': cross_wave_parental_qualification.astype('Int64').map(parental_qualification_labels).to_numpy(), 'Parental qualification source': cross_wave_source.to_numpy()})
assert len(parental_qualification_review) == 9767
assert parental_qualification_review['NSID'].nunique() == 9767
assert parental_qualification_review['NSID'].duplicated().sum() == 0
assert parental_qualification_review['NSID'].isna().sum() == 0
primary_construction_variables = {'W2hiqualgMP', 'W2hiqualgSP'}
fallback_construction_variables = {'W1hiqualgMP', 'W1hiqualgSP'}
review_support_variables = {'W2hiqualgfam'}
wave_4_variables = set(parental_education_candidates.loc[parental_education_candidates['Wave'].eq('Wave 4'),
    'Variable'])
all_parental_education_variables = set(parental_education_candidates['Variable'])
excluded_overlap_variables = all_parental_education_variables - primary_construction_variables - fallback_construction_variables - review_support_variables - wave_4_variables
working_variable_decision_register = variable_decision_register.copy()
parental_education_mask = working_variable_decision_register['Variable'].isin(all_parental_education_variables)
working_variable_decision_register.loc[parental_education_mask, 'Review status'] = 'Reviewed'
working_variable_decision_register.loc[parental_education_mask,
    'Substantive domain'] = 'Family socioeconomic background'
working_variable_decision_register.loc[parental_education_mask,
    'Documentation source'] = 'Wave 1, Wave 2 and Wave 4 derived-variable documentation'
primary_mask = working_variable_decision_register['Variable'].isin(primary_construction_variables)
working_variable_decision_register.loc[primary_mask, 'Review outcome'] = 'Retain as construction source'
working_variable_decision_register.loc[primary_mask,
    'Decision reason'] = 'Primary source for the highest available parental qualification at Wave 2'
working_variable_decision_register.loc[primary_mask,
    'Leakage assessment'] = 'No leakage: measured before the post-16 transition'
working_variable_decision_register.loc[primary_mask, 'Reference-period assessment'] = 'Wave 2 parental qualification'
working_variable_decision_register.loc[primary_mask,
    'Review notes'] = 'The lower valid code across main and second parent represents the higher qualification'
fallback_mask = working_variable_decision_register['Variable'].isin(fallback_construction_variables)
working_variable_decision_register.loc[fallback_mask, 'Review outcome'] = 'Retain as construction source'
working_variable_decision_register.loc[fallback_mask,
    'Decision reason'] = 'Fallback source where no valid Wave 2 parental qualification is available'
working_variable_decision_register.loc[fallback_mask,
    'Leakage assessment'] = 'No leakage: measured before the post-16 transition'
working_variable_decision_register.loc[fallback_mask, 'Reference-period assessment'] = 'Wave 1 parental qualification'
working_variable_decision_register.loc[fallback_mask,
    'Review notes'] = 'Used for 16 participants with no valid Wave 2 measure'
support_mask = working_variable_decision_register['Variable'].isin(review_support_variables)
working_variable_decision_register.loc[support_mask, 'Review outcome'] = 'Retain as review support only'
working_variable_decision_register.loc[support_mask,
    'Decision reason'] = 'Used to validate the available-parent construction; the official derivation omits some cases with one valid parent'
working_variable_decision_register.loc[support_mask,
    'Leakage assessment'] = 'No leakage: measured before the post-16 transition'
working_variable_decision_register.loc[support_mask, 'Reference-period assessment'] = 'Wave 2 parental qualification'
working_variable_decision_register.loc[support_mask,
    'Review notes'] = 'Exact agreement with the constructed measure in all 9,080 cases where the official measure is valid'
overlap_mask = working_variable_decision_register['Variable'].isin(excluded_overlap_variables)
working_variable_decision_register.loc[overlap_mask, 'Review outcome'] = 'Exclude from predictor set'
working_variable_decision_register.loc[overlap_mask,
    'Decision reason'] = 'Overlapping detailed or parent-role-specific version of the selected family-level grouped qualification construct'
working_variable_decision_register.loc[overlap_mask,
    'Leakage assessment'] = 'No leakage; excluded because of construct overlap'
working_variable_decision_register.loc[overlap_mask, 'Reference-period assessment'] = 'Pre-transition measure'
working_variable_decision_register.loc[overlap_mask,
    'Review notes'] = 'Not required once one grouped family-level representation has been selected'
wave_4_mask = working_variable_decision_register['Variable'].isin(wave_4_variables)
working_variable_decision_register.loc[wave_4_mask, 'Review outcome'] = 'Exclude from predictor set'
working_variable_decision_register.loc[wave_4_mask,
    'Decision reason'] = 'Wave 1 and Wave 2 provide an adequate pre-transition representation; the mixed Wave 4 source is unnecessary'
working_variable_decision_register.loc[wave_4_mask,
    'Leakage assessment'] = 'Timing concern: some Wave 4 values were collected at or after the outcome boundary'
working_variable_decision_register.loc[wave_4_mask,
    'Reference-period assessment'] = 'Mixed earlier-wave and Wave 4 information'
working_variable_decision_register.loc[wave_4_mask, 'Review notes'] = 'Not used in the construction of the predictor'
parental_decision_rows = working_variable_decision_register.loc[parental_education_mask, ['Wave', 'Variable',
    'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment']].sort_values(['Wave',
    'Variable']).reset_index(drop=True)
assert len(parental_decision_rows) == 28
assert parental_decision_rows['Review outcome'].notna().all()
parental_qualification_review_path = stage_2_output_directory / 'stage_2_parental_qualification_review.csv'
parental_qualification_review.to_csv(parental_qualification_review_path, index=False)
working_variable_decision_register.to_csv(decision_register_path, index=False)
variable_decision_register = working_variable_decision_register.copy()
parental_qualification_distribution = parental_qualification_review['highest_parental_qualification'].fillna('Unavailable').value_counts(dropna=False).rename_axis('Highest parental qualification').reset_index(name='Participants')
parental_qualification_distribution['Percentage'] = parental_qualification_distribution['Participants'].div(len(parental_qualification_review)).mul(100).round(2)
decision_count_summary = parental_decision_rows['Review outcome'].value_counts().rename_axis('Review outcome').reset_index(name='Variables')
print(f"Valid parental-qualification predictors: {parental_qualification_review['highest_parental_qualification'].notna().sum():,}")
print(f"Unavailable parental-qualification predictors: {parental_qualification_review['highest_parental_qualification'].isna().sum():,}")
print(f'Review file: {parental_qualification_review_path}')
display_limited(parental_qualification_distribution)
display_limited(decision_count_summary)
print('Source-variable decisions:')
display_limited(parental_decision_rows)

Valid parental-qualification predictors: 9,510
Unavailable parental-qualification predictors: 257
Review file: data_derived\stage_2_predictor_construction\stage_2_parental_qualification_review.csv


,Highest parental qualification,Participants,Percentage
0,GCSE grades A-C or equivalent,2259,23.13
1,Degree or equivalent,1853,18.97
2,GCE A Level or equivalent,1672,17.12
3,Higher education below degree level,1530,15.66
4,No qualification,1479,15.14


,Review outcome,Variables
0,Exclude from predictor set,23
1,Retain as construction source,4
2,Retain as review support only,1


Source-variable decisions:


,Wave,Variable,Variable label,Review outcome,Decision reason,Leakage assessment
0,Wave 1,W1hiqualMP,DV: Main parent's highest qualification held (...,Exclude from predictor set,Overlapping detailed or parent-role-specific v...,No leakage; excluded because of construct overlap
1,Wave 1,W1hiqualSP,DV: Second parent's highest qualification held...,Exclude from predictor set,Overlapping detailed or parent-role-specific v...,No leakage; excluded because of construct overlap
2,Wave 1,W1hiqualdad,DV: Father's highest qualification (detailed),Exclude from predictor set,Overlapping detailed or parent-role-specific v...,No leakage; excluded because of construct overlap
3,Wave 1,W1hiqualgMP,DV: Highest qualification held by main parent ...,Retain as construction source,Fallback source where no valid Wave 2 parental...,No leakage: measured before the post-16 transi...
4,Wave 1,W1hiqualgSP,DV: Highest qualification held by second paren...,Retain as construction source,Fallback source where no valid Wave 2 parental...,No leakage: measured before the post-16 transi...


In [194]:
# 8: Parental occupational-class candidates

family_background_source = variable_decision_register['Source type'].astype(str).eq('Family background')
occupational_class_wording = variable_decision_register['Variable'].astype(str).str.contains('nssec', case=False,
    na=False) | variable_decision_register['Variable label'].astype(str).str.contains('NS-SEC|National Statistics Socio-economic Classification|socio-economic classification|occupational class',
    case=False, na=False)
parental_occupational_class_candidates = variable_decision_register.loc[family_background_source & occupational_class_wording,
    ['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Review outcome',
    'Decision reason']].sort_values(['Wave', 'Source file', 'Variable position']).reset_index(drop=True)
print(f'Parental occupational-class candidates found: {len(parental_occupational_class_candidates):,}')
display_limited(parental_occupational_class_candidates)

Parental occupational-class candidates found: 44


,Wave,Source type,Source file,Variable position,Variable,Variable label,Review outcome,Decision reason
0,Wave 1,Family background,wave_one_lsype_family_background_2020,321,W1nssecMP,DV: MP's NS-SEC class,Pending review,<NA>
1,Wave 1,Family background,wave_one_lsype_family_background_2020,322,W1nssecSP,DV: SP's NS-SEC class,Pending review,<NA>
2,Wave 1,Family background,wave_one_lsype_family_background_2020,323,W1nsseccatSP,DV: SP's NS-SEC operational category,Pending review,<NA>
3,Wave 1,Family background,wave_one_lsype_family_background_2020,324,W1nsseccatMP,DV: MP's NS-SEC operational category,Pending review,<NA>
4,Wave 1,Family background,wave_one_lsype_family_background_2020,325,W1nsseccatmum,DV: Mother's NS-SEC operational category,Pending review,<NA>


In [195]:
# 9: Family-level NS-SEC measures

family_nssec_variables = ['W1nsseccatfam', 'W1nssecfam', 'W2nsseccatfam', 'W2nssecfam', 'W3cnsseccatfam',
    'W3cnssecfam', 'w4cnsseccatfam', 'w4cnssecfam']
family_nssec_candidates = parental_occupational_class_candidates.loc[parental_occupational_class_candidates['Variable'].isin(family_nssec_variables)].copy().reset_index(drop=True)
family_nssec_rows = []
family_nssec_negative_rows = []
for source_file, candidate_group in family_nssec_candidates.groupby('Source file', sort=False):
    source_key = Path(str(source_file)).stem
    if source_key not in source_file_lookup:
        raise KeyError(f'Source file not found in the register: {source_file}')
    source_path = source_file_lookup[source_key]
    variable_names = candidate_group['Variable'].tolist()
    source_data = pd.read_stata(source_path, columns=['NSID'] + variable_names, convert_categoricals=False)
    source_data['NSID'] = source_data['NSID'].astype('string').str.strip().str.replace('\\.0$', '', regex=True)
    source_data = source_data.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)
    for _, candidate in candidate_group.iterrows():
        variable = candidate['Variable']
        values = pd.to_numeric(source_data[variable], errors='coerce')
        non_negative_values = values[values.ge(0)]
        negative_values = values[values.lt(0)]
        family_nssec_rows.append({'Wave': candidate['Wave'], 'Variable': variable,
            'Measure form': 'Operational category' if 'catfam' in variable.lower() else 'NS-SEC class', 'Variable label': candidate['Variable label'], 'Non-negative responses': int(non_negative_values.notna().sum()), 'Negative codes': int(negative_values.notna().sum()), 'No source record': int(values.isna().sum()), 'Observed non-negative categories': int(non_negative_values.nunique()), 'Observed non-negative codes': ', '.join((str(value) for value in sorted(non_negative_values.dropna().unique())))})
        for code, count in negative_values.value_counts().sort_index().items():
            family_nssec_negative_rows.append({'Wave': candidate['Wave'], 'Variable': variable, 'Negative code': code,
                'Participants': int(count)})
family_nssec_structure_review = pd.DataFrame(family_nssec_rows)
family_nssec_negative_review = pd.DataFrame(family_nssec_negative_rows)
print(f'Family-level NS-SEC measures reviewed: {len(family_nssec_structure_review):,}')
display_limited(family_nssec_structure_review)
print('Observed negative codes:')
display_limited(family_nssec_negative_review)

Family-level NS-SEC measures reviewed: 8


,Wave,Variable,Measure form,Variable label,Non-negative responses,Negative codes,No source record,Observed non-negative categories,Observed non-negative codes
0,Wave 1,W1nsseccatfam,Operational category,DV: Family's NS-SEC operational category (from...,8769,755,243,40,"1.0, 2.0, 3.1, 3.2, 3.3, 3.4, 4.1, 4.2, 4.3, 4..."
1,Wave 1,W1nssecfam,NS-SEC class,DV: Family's NS-SEC class (from household refe...,8588,936,243,8,"1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0"
2,Wave 2,W2nsseccatfam,Operational category,DV: Family's NS-SEC operational category (from...,8697,824,246,40,"1.0, 2.0, 3.1, 3.2, 3.3, 3.4, 4.1, 4.2, 4.3, 4..."
3,Wave 2,W2nssecfam,NS-SEC class,DV: Family's NS-SEC class (from household refe...,8462,1059,246,8,"1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0"
4,Wave 3,W3cnsseccatfam,Operational category,DV: Family's current NS-SEC operational catego...,9194,315,258,37,"1.0, 2.0, 3.1, 3.2, 3.3, 3.4, 4.1, 4.2, 4.3, 4..."


Observed negative codes:


,Wave,Variable,Negative code,Participants
0,Wave 1,W1nsseccatfam,-999.0,37
1,Wave 1,W1nsseccatfam,-99.0,705
2,Wave 1,W1nsseccatfam,-94.0,13
3,Wave 1,W1nssecfam,-999.0,37
4,Wave 1,W1nssecfam,-99.0,705


In [196]:
# 10: Cross-wave family NS-SEC comparison

from scipy.stats import chi2_contingency
import numpy as np
import pandas as pd
pretransition_family_nssec_variables = {'Wave 1': 'W1nssecfam', 'Wave 2': 'W2nssecfam', 'Wave 3': 'W3cnssecfam'}
family_nssec_class_data = pd.DataFrame({'NSID': stage_2_ids.to_numpy()}).set_index('NSID')
for wave, variable in pretransition_family_nssec_variables.items():
    candidate_row = parental_occupational_class_candidates.loc[parental_occupational_class_candidates['Variable'].eq(variable)].iloc[0]
    source_key = Path(str(candidate_row['Source file'])).stem
    source_path = source_file_lookup[source_key]
    source_data = pd.read_stata(source_path, columns=['NSID', variable], convert_categoricals=False)
    source_data['NSID'] = source_data['NSID'].astype('string').str.strip().str.replace('\\.0$', '', regex=True)
    source_data = source_data.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)
    family_nssec_class_data[wave] = pd.to_numeric(source_data[variable],
        errors='coerce').where(lambda values: values.between(1, 8)).to_numpy()

def categorical_association(first, second):
    paired = pd.DataFrame({'first': first, 'second': second}).dropna()
    table = pd.crosstab(paired['first'], paired['second'])
    chi_square = chi2_contingency(table, correction=False)[0]
    participants = int(table.to_numpy().sum())
    denominator = min(table.shape) - 1
    association = np.sqrt(chi_square / (participants * denominator)) if denominator > 0 else np.nan
    agreement = paired['first'].eq(paired['second'])
    return {'Complete cases': participants, 'Exact agreement': int(agreement.sum()),
        'Agreement percentage': round(agreement.mean() * 100, 2), "Cramér's V": round(association, 4)}
wave_pairs = [('Wave 1', 'Wave 2'), ('Wave 1', 'Wave 3'), ('Wave 2', 'Wave 3')]
comparison_rows = []
for first_wave, second_wave in wave_pairs:
    statistics = categorical_association(family_nssec_class_data[first_wave], family_nssec_class_data[second_wave])
    comparison_rows.append({'First measure': first_wave, 'Second measure': second_wave, **statistics})
family_nssec_wave_comparison = pd.DataFrame(comparison_rows)
coverage_summary = pd.DataFrame({'Measure': ['Wave 1 family NS-SEC class', 'Wave 2 family NS-SEC class',
    'Wave 3 family NS-SEC class', 'At least one pre-transition measure', 'All three measures unavailable'], 'Participants': [int(family_nssec_class_data['Wave 1'].notna().sum()),
    int(family_nssec_class_data['Wave 2'].notna().sum()), int(family_nssec_class_data['Wave 3'].notna().sum()), int(family_nssec_class_data.notna().any(axis=1).sum()), int(family_nssec_class_data.isna().all(axis=1).sum())]})
availability_pattern = family_nssec_class_data.notna().rename(columns={'Wave 1': 'W1', 'Wave 2': 'W2',
    'Wave 3': 'W3'}).astype(int).astype(str).agg('-'.join, axis=1).map({'1-1-1': 'Available in Waves 1, 2 and 3',
    '1-1-0': 'Available in Waves 1 and 2 only', '1-0-1': 'Available in Waves 1 and 3 only', '0-1-1': 'Available in Waves 2 and 3 only', '1-0-0': 'Wave 1 only', '0-1-0': 'Wave 2 only', '0-0-1': 'Wave 3 only', '0-0-0': 'Unavailable in all three waves'}).value_counts().rename_axis('Availability pattern').reset_index(name='Participants')
print('Family NS-SEC class coverage:')
display_limited(coverage_summary)
print('Cross-wave comparison:')
display_limited(family_nssec_wave_comparison)
print('Availability patterns:')
display_limited(availability_pattern)

Family NS-SEC class coverage:


,Measure,Participants
0,Wave 1 family NS-SEC class,8588
1,Wave 2 family NS-SEC class,8462
2,Wave 3 family NS-SEC class,9048
3,At least one pre-transition measure,9436
4,All three measures unavailable,331


Cross-wave comparison:


,First measure,Second measure,Complete cases,Exact agreement,Agreement percentage,Cramér's V
0,Wave 1,Wave 2,8010,6306,78.73,0.7501
1,Wave 1,Wave 3,8227,4281,52.04,0.4657
2,Wave 2,Wave 3,8143,4599,56.48,0.5053


Availability patterns:


,Availability pattern,Participants
0,"Available in Waves 1, 2 and 3",7718
1,Available in Waves 1 and 3 only,509
2,Available in Waves 2 and 3 only,425
3,Wave 3 only,396
4,Unavailable in all three waves,331


In [197]:
# 11: Wave 2–Wave 3 family NS-SEC transitions

w2_w3_complete = family_nssec_class_data[['Wave 2', 'Wave 3']].dropna()
w2_w3_transition_counts = pd.crosstab(w2_w3_complete['Wave 2'].astype('Int64'),
    w2_w3_complete['Wave 3'].astype('Int64'))
w2_w3_transition_counts.index.name = 'Wave 2 NS-SEC class'
w2_w3_transition_counts.columns.name = 'Wave 3 NS-SEC class'
w2_w3_transition_percentages = pd.crosstab(w2_w3_complete['Wave 2'].astype('Int64'),
    w2_w3_complete['Wave 3'].astype('Int64'), normalize='index').mul(100).round(1)
w2_w3_transition_percentages.index.name = 'Wave 2 NS-SEC class'
w2_w3_transition_percentages.columns.name = 'Wave 3 NS-SEC class'
wave_3_only_mask = family_nssec_class_data['Wave 1'].isna() & family_nssec_class_data['Wave 2'].isna() & family_nssec_class_data['Wave 3'].notna()
wave_3_only_distribution = family_nssec_class_data.loc[wave_3_only_mask,
    'Wave 3'].astype('Int64').value_counts().sort_index().rename_axis('Wave 3 NS-SEC class').reset_index(name='Participants')
wave_3_only_distribution['Percentage'] = wave_3_only_distribution['Participants'].div(wave_3_only_mask.sum()).mul(100).round(2)
print(f'Participants observed in both Waves 2 and 3: {len(w2_w3_complete):,}')
print('Wave 2 to Wave 3 transition counts:')
display_limited(w2_w3_transition_counts)
print('Wave 2 to Wave 3 row percentages:')
display_limited(w2_w3_transition_percentages)
print(f'Participants observed only at Wave 3: {wave_3_only_mask.sum():,}')
display_limited(wave_3_only_distribution)

Participants observed in both Waves 2 and 3: 8,143
Wave 2 to Wave 3 transition counts:


Wave 3 NS-SEC class,1,2,3,4,5,6,7,8
Wave 2 NS-SEC class,,,,,,,,
1,731,260,28,17,18,10,2,44
2,237,1497,123,66,126,56,40,129
3,24,102,281,11,28,31,9,107
4,24,118,14,311,71,38,84,73
5,29,110,23,38,469,88,89,122


Wave 2 to Wave 3 row percentages:


Wave 3 NS-SEC class,1,2,3,4,5,6,7,8
Wave 2 NS-SEC class,,,,,,,,
1,65.9,23.4,2.5,1.5,1.6,0.9,0.2,4.0
2,10.4,65.8,5.4,2.9,5.5,2.5,1.8,5.7
3,4.0,17.2,47.4,1.9,4.7,5.2,1.5,18.0
4,3.3,16.1,1.9,42.4,9.7,5.2,11.5,10.0
5,3.0,11.4,2.4,3.9,48.5,9.1,9.2,12.6


Participants observed only at Wave 3: 396


,Wave 3 NS-SEC class,Participants,Percentage
0,1,44,11.11
1,2,89,22.47
2,3,14,3.54
3,4,22,5.56
4,5,45,11.36


In [198]:
# 12: Family NS-SEC candidate

import pandas as pd
family_nssec_labels = {1: 'Higher managerial and professional occupations',
    2: 'Lower managerial and professional occupations', 3: 'Intermediate occupations', 4: 'Small employers and own account workers', 5: 'Lower supervisory and technical occupations', 6: 'Semi-routine occupations', 7: 'Routine occupations', 8: 'Never worked or long-term unemployed'}
w1_family_nssec = family_nssec_class_data['Wave 1'].where(lambda values: values.between(1, 8))
w2_family_nssec = family_nssec_class_data['Wave 2'].where(lambda values: values.between(1, 8))
selected_family_nssec = w2_family_nssec.combine_first(w1_family_nssec).astype('Int64')
family_nssec_source = pd.Series('Unavailable in Waves 1 and 2', index=family_nssec_class_data.index, dtype='string')
family_nssec_source.loc[w2_family_nssec.notna()] = 'Wave 2'
family_nssec_source.loc[w2_family_nssec.isna() & w1_family_nssec.notna()] = 'Wave 1 fallback'
family_nssec_candidate_review = pd.DataFrame({'NSID': family_nssec_class_data.index.astype('string'),
    'family_nssec_code': selected_family_nssec.to_numpy(), 'family_nssec': selected_family_nssec.map(family_nssec_labels).to_numpy(), 'Family NS-SEC source': family_nssec_source.to_numpy()})
assert len(family_nssec_candidate_review) == 9767
assert family_nssec_candidate_review['NSID'].nunique() == 9767
assert family_nssec_candidate_review['NSID'].duplicated().sum() == 0
assert family_nssec_candidate_review['NSID'].isna().sum() == 0
family_nssec_distribution = family_nssec_candidate_review['family_nssec'].fillna('Unavailable').value_counts(dropna=False).rename_axis('Family NS-SEC class').reset_index(name='Participants')
family_nssec_distribution['Percentage'] = family_nssec_distribution['Participants'].div(len(family_nssec_candidate_review)).mul(100).round(2)
family_nssec_source_summary = family_nssec_candidate_review['Family NS-SEC source'].value_counts(dropna=False).rename_axis('Selected source').reset_index(name='Participants')
print(f"Valid family NS-SEC candidates: {family_nssec_candidate_review['family_nssec'].notna().sum():,}")
print(f"Unavailable family NS-SEC candidates: {family_nssec_candidate_review['family_nssec'].isna().sum():,}")
display_limited(family_nssec_distribution)
display_limited(family_nssec_source_summary)

Valid family NS-SEC candidates: 9,040
Unavailable family NS-SEC candidates: 727


,Family NS-SEC class,Participants,Percentage
0,Lower managerial and professional occupations,2480,25.39
1,Higher managerial and professional occupations,1207,12.36
2,Semi-routine occupations,1124,11.51
3,Routine occupations,1100,11.26
4,Lower supervisory and technical occupations,1067,10.92


,Selected source,Participants
0,Wave 2,8462
1,Unavailable in Waves 1 and 2,727
2,Wave 1 fallback,578


In [199]:
# 13: Available-parent family NS-SEC comparison

import pandas as pd
nssec_wave_specifications = {'Wave 1': {'main': 'W1nssecMP', 'second': 'W1nssecSP', 'family': 'W1nssecfam'},
    'Wave 2': {'main': 'W2nssecMP', 'second': 'W2nssecSP', 'family': 'W2nssecfam'}}
available_parent_nssec_by_wave = {}
nssec_comparison_rows = []
nssec_source_rows = []
for wave, variables in nssec_wave_specifications.items():
    source_file_name = source_register.loc[source_register['Wave'].eq(wave) & source_register['Source type'].eq('Family background'),
        'File name'].iloc[0]
    source_path = source_directory / source_file_name
    source_data = pd.read_stata(source_path, columns=['NSID', variables['main'], variables['second'],
        variables['family']], convert_categoricals=False)
    source_data['NSID'] = source_data['NSID'].astype('string').str.strip().str.replace('\\.0$', '', regex=True)
    source_data = source_data.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)
    main_parent = pd.to_numeric(source_data[variables['main']], errors='coerce').where(lambda values: values.between(1,
        8))
    second_parent = pd.to_numeric(source_data[variables['second']],
        errors='coerce').where(lambda values: values.between(1, 8))
    official_family = pd.to_numeric(source_data[variables['family']],
        errors='coerce').where(lambda values: values.between(1, 8))
    available_parent = pd.concat([main_parent, second_parent], axis=1).min(axis=1, skipna=True)
    available_parent_nssec_by_wave[wave] = available_parent
    official_valid = official_family.notna()
    exact_agreement = available_parent.loc[official_valid].eq(official_family.loc[official_valid])
    nssec_comparison_rows.append({'Wave': wave, 'Official family measure valid': int(official_family.notna().sum()),
        'Available-parent measure valid': int(available_parent.notna().sum()), 'Additional valid participants': int((official_family.isna() & available_parent.notna()).sum()), 'Official comparison cases': int(official_valid.sum()), 'Exact agreement': int(exact_agreement.sum()), 'Agreement percentage': round(exact_agreement.mean() * 100,
        2)})
    nssec_source_rows.extend([{'Wave': wave, 'Information source': 'Both parents',
        'Participants': int((main_parent.notna() & second_parent.notna()).sum())}, {'Wave': wave,
        'Information source': 'Main parent only', 'Participants': int((main_parent.notna() & second_parent.isna()).sum())}, {'Wave': wave,
        'Information source': 'Second parent only', 'Participants': int((main_parent.isna() & second_parent.notna()).sum())}, {'Wave': wave,
        'Information source': 'No valid parent measure', 'Participants': int((main_parent.isna() & second_parent.isna()).sum())}])
available_parent_nssec_comparison = pd.DataFrame(nssec_comparison_rows)
available_parent_nssec_sources = pd.DataFrame(nssec_source_rows)
print('Official and available-parent NS-SEC comparison:')
display_limited(available_parent_nssec_comparison)
print('Parental information sources:')
display_limited(available_parent_nssec_sources)

Official and available-parent NS-SEC comparison:


,Wave,Official family measure valid,Available-parent measure valid,Additional valid participants,Official comparison cases,Exact agreement,Agreement percentage
0,Wave 1,8588,9378,790,8588,7083,82.48
1,Wave 2,8462,9340,878,8462,6984,82.53


Parental information sources:


,Wave,Information source,Participants
0,Wave 1,Both parents,6333
1,Wave 1,Main parent only,2869
2,Wave 1,Second parent only,176
3,Wave 1,No valid parent measure,389
4,Wave 2,Both parents,6221


In [200]:
# 14: Family NS-SEC construction

from pathlib import Path
import pandas as pd
family_nssec_review = family_nssec_candidate_review.copy()
assert len(family_nssec_review) == 9767
assert family_nssec_review['NSID'].nunique() == 9767
assert family_nssec_review['NSID'].duplicated().sum() == 0
assert family_nssec_review['NSID'].isna().sum() == 0
family_nssec_source_variables = set(parental_occupational_class_candidates['Variable'])
primary_construction_variables = {'W2nssecfam'}
fallback_construction_variables = {'W1nssecfam'}
family_operational_support_variables = {'W1nsseccatfam', 'W2nsseccatfam'}
parent_comparison_support_variables = {'W1nssecMP', 'W1nssecSP', 'W2nssecMP', 'W2nssecSP'}
review_support_variables = family_operational_support_variables | parent_comparison_support_variables
wave_3_variables = set(parental_occupational_class_candidates.loc[parental_occupational_class_candidates['Wave'].eq('Wave 3'),
    'Variable'])
wave_4_variables = set(parental_occupational_class_candidates.loc[parental_occupational_class_candidates['Wave'].eq('Wave 4'),
    'Variable'])
wave_1_2_overlap_variables = family_nssec_source_variables - primary_construction_variables - fallback_construction_variables - review_support_variables - wave_3_variables - wave_4_variables
working_variable_decision_register = variable_decision_register.copy()
family_nssec_mask = working_variable_decision_register['Variable'].isin(family_nssec_source_variables)
working_variable_decision_register.loc[family_nssec_mask, 'Review status'] = 'Reviewed'
working_variable_decision_register.loc[family_nssec_mask, 'Substantive domain'] = 'Family socioeconomic background'
working_variable_decision_register.loc[family_nssec_mask,
    'Documentation source'] = 'Wave 1 to Wave 4 derived-variable documentation'
primary_mask = working_variable_decision_register['Variable'].isin(primary_construction_variables)
working_variable_decision_register.loc[primary_mask, 'Review outcome'] = 'Retain as construction source'
working_variable_decision_register.loc[primary_mask,
    'Decision reason'] = 'Primary source for family NS-SEC based on the household reference person'
working_variable_decision_register.loc[primary_mask,
    'Leakage assessment'] = 'No leakage: measured before the post-16 outcome'
working_variable_decision_register.loc[primary_mask,
    'Reference-period assessment'] = 'Wave 2 household-reference-person occupational class'
working_variable_decision_register.loc[primary_mask,
    'Review notes'] = 'Provides the predictor value for 8,462 participants'
fallback_mask = working_variable_decision_register['Variable'].isin(fallback_construction_variables)
working_variable_decision_register.loc[fallback_mask, 'Review outcome'] = 'Retain as construction source'
working_variable_decision_register.loc[fallback_mask,
    'Decision reason'] = 'Fallback source where the Wave 2 family NS-SEC measure is unavailable'
working_variable_decision_register.loc[fallback_mask,
    'Leakage assessment'] = 'No leakage: measured before the post-16 outcome'
working_variable_decision_register.loc[fallback_mask,
    'Reference-period assessment'] = 'Wave 1 household-reference-person occupational class'
working_variable_decision_register.loc[fallback_mask,
    'Review notes'] = 'Provides the predictor value for 578 participants'
operational_support_mask = working_variable_decision_register['Variable'].isin(family_operational_support_variables)
working_variable_decision_register.loc[operational_support_mask, 'Review outcome'] = 'Retain as review support only'
working_variable_decision_register.loc[operational_support_mask,
    'Decision reason'] = 'Used to inspect the detailed operational categories underlying the selected eight-class measure'
working_variable_decision_register.loc[operational_support_mask,
    'Leakage assessment'] = 'No leakage: measured before the post-16 outcome'
working_variable_decision_register.loc[operational_support_mask,
    'Reference-period assessment'] = 'Wave 1 or Wave 2 household-reference-person occupation'
working_variable_decision_register.loc[operational_support_mask,
    'Review notes'] = 'Not retained as a predictor because the detailed form contains approximately 40 categories'
parent_support_mask = working_variable_decision_register['Variable'].isin(parent_comparison_support_variables)
working_variable_decision_register.loc[parent_support_mask, 'Review outcome'] = 'Retain as review support only'
working_variable_decision_register.loc[parent_support_mask,
    'Decision reason'] = 'Used to test whether a parent-specific construction could reproduce the official family measure'
working_variable_decision_register.loc[parent_support_mask,
    'Leakage assessment'] = 'No leakage: measured before the post-16 outcome'
working_variable_decision_register.loc[parent_support_mask,
    'Reference-period assessment'] = 'Wave 1 or Wave 2 parent-specific occupational class'
working_variable_decision_register.loc[parent_support_mask,
    'Review notes'] = 'The parent-specific construction was rejected because the official measure represents the household reference person'
overlap_mask = working_variable_decision_register['Variable'].isin(wave_1_2_overlap_variables)
working_variable_decision_register.loc[overlap_mask, 'Review outcome'] = 'Exclude from predictor set'
working_variable_decision_register.loc[overlap_mask,
    'Decision reason'] = 'More detailed, parent-role-specific or SOC version overlapping with the selected family NS-SEC construct'
working_variable_decision_register.loc[overlap_mask,
    'Leakage assessment'] = 'No leakage; excluded because of construct overlap'
working_variable_decision_register.loc[overlap_mask,
    'Reference-period assessment'] = 'Wave 1 or Wave 2 occupational information'
working_variable_decision_register.loc[overlap_mask,
    'Review notes'] = 'Not required after selecting one family-level household-reference-person measure'
wave_3_mask = working_variable_decision_register['Variable'].isin(wave_3_variables)
working_variable_decision_register.loc[wave_3_mask, 'Review outcome'] = 'Exclude from predictor set'
working_variable_decision_register.loc[wave_3_mask,
    'Decision reason'] = 'Current occupational measure not directly comparable with the selected Wave 1 and Wave 2 construct'
working_variable_decision_register.loc[wave_3_mask,
    'Leakage assessment'] = 'No direct outcome leakage; excluded for timing and construct consistency'
working_variable_decision_register.loc[wave_3_mask,
    'Reference-period assessment'] = 'Wave 3 current occupational position'
working_variable_decision_register.loc[wave_3_mask,
    'Review notes'] = 'Class 8 denotes not currently working rather than never worked or long-term unemployed'
wave_4_mask = working_variable_decision_register['Variable'].isin(wave_4_variables)
working_variable_decision_register.loc[wave_4_mask, 'Review outcome'] = 'Exclude from predictor set'
working_variable_decision_register.loc[wave_4_mask,
    'Decision reason'] = 'Earlier waves provide the selected construct and Wave 4 records current occupational position'
working_variable_decision_register.loc[wave_4_mask,
    'Leakage assessment'] = 'Timing concern: Wave 4 may include information measured at or after the May 2009 outcome boundary'
working_variable_decision_register.loc[wave_4_mask,
    'Reference-period assessment'] = 'Wave 4 current occupational position'
working_variable_decision_register.loc[wave_4_mask, 'Review notes'] = 'Not used in predictor construction'
family_nssec_decision_rows = working_variable_decision_register.loc[family_nssec_mask, ['Wave', 'Variable position',
    'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment']].sort_values(['Wave',
    'Variable position']).reset_index(drop=True)
assert len(family_nssec_decision_rows) == 44
assert family_nssec_decision_rows['Review outcome'].notna().all()
assert family_nssec_decision_rows['Decision reason'].notna().all()
assert family_nssec_decision_rows['Leakage assessment'].notna().all()
family_nssec_review_path = stage_2_output_directory / 'stage_2_family_nssec_review.csv'
family_nssec_review.to_csv(family_nssec_review_path, index=False)
working_variable_decision_register.to_csv(decision_register_path, index=False)
variable_decision_register = working_variable_decision_register.copy()
family_nssec_decision_summary = family_nssec_decision_rows['Review outcome'].value_counts().rename_axis('Review outcome').reset_index(name='Variables')
print(f"Valid family NS-SEC predictors: {family_nssec_review['family_nssec'].notna().sum():,}")
print(f"Unavailable family NS-SEC predictors: {family_nssec_review['family_nssec'].isna().sum():,}")
print(f'Review file: {family_nssec_review_path}')
display_limited(family_nssec_decision_summary)
print('Source-variable decisions:')
display_limited(family_nssec_decision_rows)

Valid family NS-SEC predictors: 9,040
Unavailable family NS-SEC predictors: 727
Review file: data_derived\stage_2_predictor_construction\stage_2_family_nssec_review.csv


,Review outcome,Variables
0,Exclude from predictor set,36
1,Retain as review support only,6
2,Retain as construction source,2


Source-variable decisions:


,Wave,Variable position,Variable,Variable label,Review outcome,Decision reason,Leakage assessment
0,Wave 1,321,W1nssecMP,DV: MP's NS-SEC class,Retain as review support only,Used to test whether a parent-specific constru...,No leakage: measured before the post-16 outcome
1,Wave 1,322,W1nssecSP,DV: SP's NS-SEC class,Retain as review support only,Used to test whether a parent-specific constru...,No leakage: measured before the post-16 outcome
2,Wave 1,323,W1nsseccatSP,DV: SP's NS-SEC operational category,Exclude from predictor set,"More detailed, parent-role-specific or SOC ver...",No leakage; excluded because of construct overlap
3,Wave 1,324,W1nsseccatMP,DV: MP's NS-SEC operational category,Exclude from predictor set,"More detailed, parent-role-specific or SOC ver...",No leakage; excluded because of construct overlap
4,Wave 1,325,W1nsseccatmum,DV: Mother's NS-SEC operational category,Exclude from predictor set,"More detailed, parent-role-specific or SOC ver...",No leakage; excluded because of construct overlap


In [201]:
# 15: Household-income candidate variables

family_background_source = variable_decision_register['Source type'].astype(str).eq('Family background')
household_income_variable_names = variable_decision_register['Variable'].astype(str).str.contains('inc1est|hhinc|hhincome|faminc|totinc',
    case=False, na=False)
household_income_labels = variable_decision_register['Variable label'].astype(str).str.contains('household income|family income|total income|estimated income|gross household income|net household income|income band',
    case=False, na=False)
household_income_candidates = variable_decision_register.loc[family_background_source & (household_income_variable_names | household_income_labels),
    ['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Timing status',
    'Review outcome', 'Decision reason']].sort_values(['Wave', 'Source file',
    'Variable position']).reset_index(drop=True)
print(f'Household-income candidate variables found: {len(household_income_candidates):,}')
display_limited(household_income_candidates)

Household-income candidate variables found: 18


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review outcome,Decision reason
0,Wave 1,Family background,wave_one_lsype_family_background_2020,74,W1inc1estMP,"MP: Total income from work, benefits, and anyt...",Pre-transition source,Pending review,<NA>
1,Wave 1,Family background,wave_one_lsype_family_background_2020,75,W1inc2estMP,"MP: Total income from work, benefits, and anyt...",Pre-transition source,Pending review,<NA>
2,Wave 1,Family background,wave_one_lsype_family_background_2020,162,W1SeiInc2MP,MP: MP's estimated income from self-employment...,Pre-transition source,Pending review,<NA>
3,Wave 1,Family background,wave_one_lsype_family_background_2020,381,W1inc1est,DV: Estimate of gross household income (edited),Pre-transition source,Retain as review support only,Indicates whether an income value was edited d...
4,Wave 2,Family background,wave_two_lsype_family_background_2020,68,W2Inc1estMP,"MP: Total income from work, benefits, and anyt...",Pre-transition source,Pending review,<NA>


In [202]:
# 16: Household-income measure structure

import pandas as pd
from pathlib import Path
income_structure_variables = ['W1inc1est', 'W1inc1estMP', 'W1inc2estMP', 'W2Inc1estMP', 'W2Inc2estMP', 'W3incestMP',
    'W3incestm', 'W3incestw', 'W4Inc1EstMP']
selected_income_candidates = household_income_candidates.loc[household_income_candidates['Variable'].isin(income_structure_variables)].copy().reset_index(drop=True)
income_structure_rows = []
income_negative_code_rows = []
for source_file, candidate_group in selected_income_candidates.groupby('Source file', sort=False):
    source_key = Path(str(source_file)).stem
    if source_key not in source_file_lookup:
        raise KeyError(f'Source file not found in the register: {source_file}')
    source_path = source_file_lookup[source_key]
    variable_names = candidate_group['Variable'].tolist()
    source_data = pd.read_stata(source_path, columns=['NSID'] + variable_names, convert_categoricals=False)
    source_data['NSID'] = source_data['NSID'].astype('string').str.strip().str.replace('\\.0$', '', regex=True)
    source_data = source_data.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)
    for _, candidate in candidate_group.iterrows():
        variable = candidate['Variable']
        values = pd.to_numeric(source_data[variable], errors='coerce')
        non_negative_values = values[values.ge(0)]
        negative_values = values[values.lt(0)]
        observed_values = sorted(non_negative_values.dropna().unique())
        income_structure_rows.append({'Wave': candidate['Wave'], 'Variable': variable,
            'Variable label': candidate['Variable label'], 'Non-negative responses': int(non_negative_values.notna().sum()), 'Negative codes': int(negative_values.notna().sum()), 'No source record': int(values.isna().sum()), 'Distinct non-negative values': int(non_negative_values.nunique()), 'Minimum': non_negative_values.min() if non_negative_values.notna().any() else pd.NA, 'Maximum': non_negative_values.max() if non_negative_values.notna().any() else pd.NA, 'First observed values': ', '.join((str(value) for value in observed_values[:12]))})
        for code, count in negative_values.value_counts().sort_index().items():
            income_negative_code_rows.append({'Wave': candidate['Wave'], 'Variable': variable, 'Negative code': code,
                'Participants': int(count)})
income_measure_structure = pd.DataFrame(income_structure_rows)
income_negative_code_review = pd.DataFrame(income_negative_code_rows)
print(f'Household-income measures reviewed: {len(income_measure_structure):,}')
display_limited(income_measure_structure)
print('Observed negative codes:')
display_limited(income_negative_code_review)

Household-income measures reviewed: 9


,Wave,Variable,Variable label,Non-negative responses,Negative codes,No source record,Distinct non-negative values,Minimum,Maximum,First observed values
0,Wave 1,W1inc1estMP,"MP: Total income from work, benefits, and anyt...",7309,2215,243,33,0.0,32.0,"0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9..."
1,Wave 1,W1inc2estMP,"MP: Total income from work, benefits, and anyt...",1619,7905,243,55,1.0,60.0,"1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 1..."
2,Wave 1,W1inc1est,DV: Estimate of gross household income (edited),7309,2215,243,33,0.0,32.0,"0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9..."
3,Wave 2,W2Inc1estMP,"MP: Total income from work, benefits, and anyt...",140,9381,246,31,0.0,32.0,"0.0, 1.0, 3.0, 4.0, 6.0, 7.0, 8.0, 9.0, 10.0, ..."
4,Wave 2,W2Inc2estMP,"MP: Total income from work, benefits, and anyt...",5,9516,246,4,12.0,32.0,"12.0, 17.0, 20.0, 32.0"


Observed negative codes:


,Wave,Variable,Negative code,Participants
0,Wave 1,W1inc1estMP,-99.0,103
1,Wave 1,W1inc1estMP,-92.0,814
2,Wave 1,W1inc1estMP,-91.0,1
3,Wave 1,W1inc1estMP,-1.0,1297
4,Wave 1,W1inc2estMP,-997.0,6


In [203]:
# 17: Household-income duplicate measures

import pandas as pd
w1_income_file_name = source_register.loc[source_register['Wave'].eq('Wave 1') & source_register['Source type'].eq('Family background'),
    'File name'].iloc[0]
w3_income_file_name = source_register.loc[source_register['Wave'].eq('Wave 3') & source_register['Source type'].eq('Family background'),
    'File name'].iloc[0]
w1_income_path = source_directory / w1_income_file_name
w3_income_path = source_directory / w3_income_file_name
w1_income_data = pd.read_stata(w1_income_path, columns=['NSID', 'W1inc1estMP', 'W1inc1est'],
    convert_categoricals=False)
w3_income_data = pd.read_stata(w3_income_path, columns=['NSID', 'W3incestMP', 'W3incestm', 'W3incestw'],
    convert_categoricals=False)
for data in [w1_income_data, w3_income_data]:
    data['NSID'] = data['NSID'].astype('string').str.strip().str.replace('\\.0$', '', regex=True)
w1_income_data = w1_income_data.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)
w3_income_data = w3_income_data.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)

def exact_value_comparison(first, second):
    comparable = first.notna() | second.notna()
    exact_match = first.eq(second) | first.isna() & second.isna()
    return {'Comparable participants': int(comparable.sum()), 'Exact matches': int(exact_match.sum()),
        'Different values': int((~exact_match).sum()), 'Agreement percentage': round(exact_match.mean() * 100, 2)}
income_duplicate_pairs = [('W1inc1estMP', 'W1inc1est', w1_income_data['W1inc1estMP'], w1_income_data['W1inc1est']),
    ('W3incestMP', 'W3incestm', w3_income_data['W3incestMP'], w3_income_data['W3incestm']), ('W3incestMP', 'W3incestw',
    w3_income_data['W3incestMP'], w3_income_data['W3incestw']), ('W3incestm', 'W3incestw', w3_income_data['W3incestm'],
    w3_income_data['W3incestw'])]
income_duplicate_rows = []
for first_name, second_name, first_values, second_values in income_duplicate_pairs:
    comparison = exact_value_comparison(first_values, second_values)
    income_duplicate_rows.append({'First variable': first_name, 'Second variable': second_name, **comparison})
income_duplicate_review = pd.DataFrame(income_duplicate_rows)
display_limited(income_duplicate_review)

,First variable,Second variable,Comparable participants,Exact matches,Different values,Agreement percentage
0,W1inc1estMP,W1inc1est,9524,9374,393,95.98
1,W3incestMP,W3incestm,9509,9767,0,100.00
2,W3incestMP,W3incestw,9509,9767,0,100.00
3,W3incestm,W3incestw,9509,9767,0,100.00


In [204]:
# 18: Wave 1 edited household-income differences

import pandas as pd
w1_income_raw = pd.to_numeric(w1_income_data['W1inc1estMP'], errors='coerce')
w1_income_edited = pd.to_numeric(w1_income_data['W1inc1est'], errors='coerce')
w1_income_source_record = w1_income_raw.notna() | w1_income_edited.notna()
w1_income_exact_match = w1_income_raw.eq(w1_income_edited) | w1_income_raw.isna() & w1_income_edited.isna()
w1_income_difference = w1_income_source_record & ~w1_income_exact_match
w1_income_comparison_type = pd.Series('Other pattern', index=w1_income_data.index, dtype='string')
w1_income_comparison_type.loc[~w1_income_source_record] = 'No Wave 1 source record'
w1_income_comparison_type.loc[w1_income_raw.ge(0) & w1_income_edited.ge(0) & w1_income_raw.eq(w1_income_edited)] = 'Same valid value'
w1_income_comparison_type.loc[w1_income_raw.ge(0) & w1_income_edited.ge(0) & ~w1_income_raw.eq(w1_income_edited)] = 'Different valid values'
w1_income_comparison_type.loc[w1_income_raw.lt(0) & w1_income_edited.lt(0) & w1_income_raw.eq(w1_income_edited)] = 'Same negative code'
w1_income_comparison_type.loc[w1_income_raw.lt(0) & w1_income_edited.lt(0) & ~w1_income_raw.eq(w1_income_edited)] = 'Different negative codes'
w1_income_comparison_type.loc[w1_income_raw.lt(0) & w1_income_edited.ge(0)] = 'Raw negative; edited valid'
w1_income_comparison_type.loc[w1_income_raw.ge(0) & w1_income_edited.lt(0)] = 'Raw valid; edited negative'
w1_income_comparison_type.loc[w1_income_raw.isna() & w1_income_edited.notna()] = 'Raw missing; edited stored'
w1_income_comparison_type.loc[w1_income_raw.notna() & w1_income_edited.isna()] = 'Raw stored; edited missing'
comparison_type_summary = w1_income_comparison_type.value_counts(dropna=False).rename_axis('Comparison type').reset_index(name='Participants')
w1_income_difference_pairs = pd.DataFrame({'Raw value': w1_income_raw.loc[w1_income_difference],
    'Edited value': w1_income_edited.loc[w1_income_difference]}).value_counts(dropna=False).rename('Participants').reset_index().sort_values('Participants',
    ascending=False).reset_index(drop=True)
source_record_matches = int((w1_income_source_record & w1_income_exact_match).sum())
source_record_total = int(w1_income_source_record.sum())
print(f'Exact agreement among participants with a Wave 1 source record: {source_record_matches:,} of {source_record_total:,}')
print(f'Agreement percentage among source records: {source_record_matches / source_record_total * 100:.2f}%')
print(f'Participants with different stored values: {w1_income_difference.sum():,}')
display_limited(comparison_type_summary)
print('Differing value pairs:')
display_limited(w1_income_difference_pairs.head(30))

Exact agreement among participants with a Wave 1 source record: 9,131 of 9,524
Agreement percentage among source records: 95.87%
Participants with different stored values: 393


,Comparison type,Participants
0,Same valid value,6916
1,Same negative code,2215
2,Different valid values,393
3,No Wave 1 source record,243


Differing value pairs:


,Raw value,Edited value,Participants
0,10.0,32.0,19
1,9.0,32.0,13
2,7.0,32.0,12
3,8.0,32.0,10
4,3.0,32.0,8


In [205]:
# 19: Cross-wave household-income comparison

import pandas as pd
from scipy.stats import spearmanr
w1_edited_income = pd.to_numeric(w1_income_data['W1inc1est'], errors='coerce').where(lambda values: values.between(0,
    32))
w1_to_12_band_map = {**{code: 1 for code in range(0, 6)}, **{code: 2 for code in range(6, 11)},
    **{code: 3 for code in range(11, 16)}, **{code: 4 for code in range(16, 21)}, **{code: 5 for code in range(21,
    26)}, 26: 6, 27: 7, 28: 8, 29: 9, 30: 10, 31: 11, 32: 12}
w1_income_12_band = w1_edited_income.map(w1_to_12_band_map).astype('Float64')
w3_income_12_band = pd.to_numeric(w3_income_data['W3incestMP'], errors='coerce').where(lambda values: values.between(1,
    12)).astype('Float64')
both_income_waves = w1_income_12_band.notna() & w3_income_12_band.notna()
exact_income_agreement = w1_income_12_band.loc[both_income_waves].eq(w3_income_12_band.loc[both_income_waves])
spearman_result = spearmanr(w1_income_12_band.loc[both_income_waves], w3_income_12_band.loc[both_income_waves])
income_coverage_summary = pd.DataFrame({'Measure or condition': ['Wave 1 edited income', 'Wave 3 annual income',
    'Available in both waves', 'Exact agreement across waves', 'Wave 3 unavailable; Wave 1 available', 'Wave 1 unavailable; Wave 3 available', 'Unavailable in both waves', 'Wave 3 with Wave 1 fallback'], 'Participants': [int(w1_income_12_band.notna().sum()),
    int(w3_income_12_band.notna().sum()), int(both_income_waves.sum()), int(exact_income_agreement.sum()), int((w3_income_12_band.isna() & w1_income_12_band.notna()).sum()), int((w1_income_12_band.isna() & w3_income_12_band.notna()).sum()), int((w1_income_12_band.isna() & w3_income_12_band.isna()).sum()), int(w3_income_12_band.combine_first(w1_income_12_band).notna().sum())]})
income_change_direction = pd.Series(pd.NA, index=w1_income_12_band.index, dtype='string')
income_change_direction.loc[both_income_waves & w3_income_12_band.eq(w1_income_12_band)] = 'Same income band'
income_change_direction.loc[both_income_waves & w3_income_12_band.gt(w1_income_12_band)] = 'Higher income band at Wave 3'
income_change_direction.loc[both_income_waves & w3_income_12_band.lt(w1_income_12_band)] = 'Lower income band at Wave 3'
income_change_summary = income_change_direction.value_counts(dropna=True).rename_axis('Cross-wave comparison').reset_index(name='Participants')
print(f'Exact agreement among participants observed in both waves: {exact_income_agreement.sum():,} of {both_income_waves.sum():,}')
print(f'Agreement percentage: {exact_income_agreement.mean() * 100:.2f}%')
print(f'Spearman correlation: {spearman_result.statistic:.4f}')
display_limited(income_coverage_summary)
display_limited(income_change_summary)

Exact agreement among participants observed in both waves: 2,102 of 6,465
Agreement percentage: 32.51%
Spearman correlation: 0.6891


,Measure or condition,Participants
0,Wave 1 edited income,7309
1,Wave 3 annual income,7935
2,Available in both waves,6465
3,Exact agreement across waves,2102
4,Wave 3 unavailable; Wave 1 available,844


,Cross-wave comparison,Participants
0,Lower income band at Wave 3,2493
1,Same income band,2102
2,Higher income band at Wave 3,1870


In [206]:
# 20: Wave 3 household-income timing

import pandas as pd
w3_income_timing_data = pd.read_stata(w3_income_path, columns=['NSID', 'W3incestMP', 'W3intmnthMP', 'W3intyearMP'],
    convert_categoricals=False)
w3_income_timing_data['NSID'] = w3_income_timing_data['NSID'].astype('string').str.strip().str.replace('\\.0$', '',
    regex=True)
w3_income_timing_data = w3_income_timing_data.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)
interview_month = pd.to_numeric(w3_income_timing_data['W3intmnthMP'],
    errors='coerce').where(lambda values: values.between(1, 12))
interview_year = pd.to_numeric(w3_income_timing_data['W3intyearMP'],
    errors='coerce').where(lambda values: values.between(2000, 2010))
valid_interview_date = interview_month.notna() & interview_year.notna()
w3_interview_date = pd.Series(pd.NaT, index=w3_income_timing_data.index, dtype='datetime64[ns]')
w3_interview_date.loc[valid_interview_date] = pd.to_datetime({'year': interview_year.loc[valid_interview_date].astype(int),
    'month': interview_month.loc[valid_interview_date].astype(int), 'day': 1}).to_numpy()
transition_month = pd.Timestamp('2006-09-01')
w3_income_timing_status = pd.Series('Interview date unavailable', index=w3_income_timing_data.index, dtype='string')
w3_income_timing_status.loc[w3_interview_date.lt(transition_month)] = 'Before September 2006'
w3_income_timing_status.loc[w3_interview_date.eq(transition_month)] = 'September 2006'
w3_income_timing_status.loc[w3_interview_date.gt(transition_month)] = 'After September 2006'
w3_income_valid = pd.to_numeric(w3_income_timing_data['W3incestMP'], errors='coerce').between(1, 12)
interview_month_summary = w3_interview_date.dropna().dt.to_period('M').astype(str).value_counts().sort_index().rename_axis('Interview month').reset_index(name='Participants')
timing_summary = pd.DataFrame({'Timing status': ['Before September 2006', 'September 2006', 'After September 2006',
    'Interview date unavailable'], 'All participants': [int(w3_income_timing_status.eq('Before September 2006').sum()),
    int(w3_income_timing_status.eq('September 2006').sum()), int(w3_income_timing_status.eq('After September 2006').sum()), int(w3_income_timing_status.eq('Interview date unavailable').sum())], 'Valid income responses': [int((w3_income_valid & w3_income_timing_status.eq('Before September 2006')).sum()),
    int((w3_income_valid & w3_income_timing_status.eq('September 2006')).sum()), int((w3_income_valid & w3_income_timing_status.eq('After September 2006')).sum()), int((w3_income_valid & w3_income_timing_status.eq('Interview date unavailable')).sum())]})
print('Wave 3 interview-month distribution:')
display_limited(interview_month_summary)
print('Household-income timing assessment:')
display_limited(timing_summary)

Wave 3 interview-month distribution:


,Interview month,Participants
0,2006-03,435
1,2006-04,4267
2,2006-05,3206
3,2006-06,996
4,2006-07,368


Household-income timing assessment:


,Timing status,All participants,Valid income responses
0,Before September 2006,9367,7888
1,September 2006,14,13
2,After September 2006,0,0
3,Interview date unavailable,386,34


In [207]:
# 21: Pre-transition household-income candidate

import pandas as pd
wave_3_income_before_transition = w3_income_12_band.where(w3_income_timing_status.eq('Before September 2006'))
pretransition_household_income = wave_3_income_before_transition.combine_first(w1_income_12_band).astype('Int64')
household_income_source = pd.Series('Unavailable', index=w3_income_12_band.index, dtype='string')
household_income_source.loc[wave_3_income_before_transition.notna()] = 'Wave 3 before September 2006'
household_income_source.loc[wave_3_income_before_transition.isna() & w1_income_12_band.notna()] = 'Wave 1 fallback'
household_income_candidate_review = pd.DataFrame({'NSID': stage_2_ids.to_numpy(),
    'household_income_band': pretransition_household_income.to_numpy(), 'Household income source': household_income_source.to_numpy()})
assert len(household_income_candidate_review) == 9767
assert household_income_candidate_review['NSID'].nunique() == 9767
assert household_income_candidate_review['NSID'].duplicated().sum() == 0
assert household_income_candidate_review['NSID'].isna().sum() == 0
timing_restriction_summary = pd.DataFrame({'Condition': ['Wave 3 income before September 2006',
    'Wave 3 income in September 2006', 'Wave 3 income with interview date unavailable', 'September or undated Wave 3 income recovered from Wave 1', 'September or undated Wave 3 income not recovered from Wave 1'], 'Participants': [int(wave_3_income_before_transition.notna().sum()),
    int((w3_income_12_band.notna() & w3_income_timing_status.eq('September 2006')).sum()), int((w3_income_12_band.notna() & w3_income_timing_status.eq('Interview date unavailable')).sum()), int((w3_income_12_band.notna() & ~w3_income_timing_status.eq('Before September 2006') & w1_income_12_band.notna()).sum()), int((w3_income_12_band.notna() & ~w3_income_timing_status.eq('Before September 2006') & w1_income_12_band.isna()).sum())]})
household_income_source_summary = household_income_candidate_review['Household income source'].value_counts(dropna=False).rename_axis('Selected source').reset_index(name='Participants')
household_income_distribution = household_income_candidate_review['household_income_band'].value_counts(dropna=False).sort_index().rename_axis('Household income band').reset_index(name='Participants')
print(f'Valid pre-transition household-income candidates: {pretransition_household_income.notna().sum():,}')
print(f'Unavailable household-income candidates: {pretransition_household_income.isna().sum():,}')
display_limited(timing_restriction_summary)
display_limited(household_income_source_summary)
display_limited(household_income_distribution)

Valid pre-transition household-income candidates: 8,771
Unavailable household-income candidates: 996


,Condition,Participants
0,Wave 3 income before September 2006,7888
1,Wave 3 income in September 2006,13
2,Wave 3 income with interview date unavailable,34
3,September or undated Wave 3 income recovered f...,39
4,September or undated Wave 3 income not recover...,8


,Selected source,Participants
0,Wave 3 before September 2006,7888
1,Unavailable,996
2,Wave 1 fallback,883


,Household income band,Participants
0,1.0,47
1,2.0,231
2,3.0,913
3,4.0,1177
4,5.0,1005


In [208]:
# 22: Household-income band labels

import pandas as pd

def extract_code_label_map(file_path, variable):
    raw_data = pd.read_stata(file_path, columns=[variable], convert_categoricals=False)
    labelled_data = pd.read_stata(file_path, columns=[variable], convert_categoricals=True)
    code_label_data = pd.DataFrame({'Code': pd.to_numeric(raw_data[variable], errors='coerce'),
        'Label': labelled_data[variable].astype('string')})
    code_label_map = code_label_data.dropna(subset=['Code']).drop_duplicates().sort_values('Code').reset_index(drop=True)
    code_label_map['Variable'] = variable
    return code_label_map[['Variable', 'Code', 'Label']]
w1_income_code_labels = extract_code_label_map(w1_income_path, 'W1inc1est')
w3_income_code_labels = extract_code_label_map(w3_income_path, 'W3incestMP')
print('Wave 1 edited household-income bands:')
display_limited(w1_income_code_labels)
print('Wave 3 annual household-income bands:')
display_limited(w3_income_code_labels)

Wave 1 edited household-income bands:


,Variable,Code,Label
0,W1inc1est,-99,MP not interviewed
1,W1inc1est,-92,Refused
2,W1inc1est,-91,Not applicable
3,W1inc1est,-1,Don't know
4,W1inc1est,0,No income


Wave 3 annual household-income bands:


,Variable,Code,Label
0,W3incestMP,-99,MP not interviewed
1,W3incestMP,-92,Refused
2,W3incestMP,-1,Don't know
3,W3incestMP,1,"Up to £2,599"
4,W3incestMP,2,"£2,600 up to £5,199"


In [209]:
# 23: Household-income harmonisation

import pandas as pd
from scipy.stats import spearmanr
w1_to_9_band_map = {**{code: 1 for code in range(0, 6)}, **{code: 2 for code in range(6, 11)},
    **{code: 3 for code in range(11, 16)}, **{code: 4 for code in range(16, 21)}, **{code: 5 for code in range(21,
    26)}, **{code: 6 for code in range(26, 28)}, **{code: 7 for code in range(28,
    30)}, **{code: 8 for code in range(30, 32)}, 32: 9}
w1_income_9_band = w1_edited_income.map(w1_to_9_band_map).astype('Float64')
w3_to_9_band_map = {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 9, 11: 9, 12: 9}
w3_income_9_band = w3_income_12_band.map(w3_to_9_band_map).astype('Float64')
both_income_waves_9_band = w1_income_9_band.notna() & w3_income_9_band.notna()
income_9_band_agreement = w1_income_9_band.loc[both_income_waves_9_band].eq(w3_income_9_band.loc[both_income_waves_9_band])
income_9_band_spearman = spearmanr(w1_income_9_band.loc[both_income_waves_9_band],
    w3_income_9_band.loc[both_income_waves_9_band])
corrected_income_comparison = pd.DataFrame({'Measure or condition': ['Wave 1 harmonised income',
    'Wave 3 harmonised income', 'Available in both waves', 'Exact agreement across waves', 'Wave 3 unavailable; Wave 1 available', 'Wave 1 unavailable; Wave 3 available', 'Unavailable in both waves'], 'Participants': [int(w1_income_9_band.notna().sum()),
    int(w3_income_9_band.notna().sum()), int(both_income_waves_9_band.sum()), int(income_9_band_agreement.sum()), int((w3_income_9_band.isna() & w1_income_9_band.notna()).sum()), int((w1_income_9_band.isna() & w3_income_9_band.notna()).sum()), int((w1_income_9_band.isna() & w3_income_9_band.isna()).sum())]})
income_9_band_labels = pd.DataFrame({'Income band': range(1, 10), 'Annual household income': ['Up to £2,599',
    '£2,600–£5,199', '£5,200–£10,399', '£10,400–£15,599', '£15,600–£20,799', '£20,800–£25,999', '£26,000–£31,199', '£31,200–£36,399', '£36,400 or more']})
print(f'Exact agreement among participants observed in both waves: {income_9_band_agreement.sum():,} of {both_income_waves_9_band.sum():,}')
print(f'Agreement percentage: {income_9_band_agreement.mean() * 100:.2f}%')
print(f'Spearman correlation: {income_9_band_spearman.statistic:.4f}')
display_limited(income_9_band_labels)
display_limited(corrected_income_comparison)

Exact agreement among participants observed in both waves: 2,823 of 6,465
Agreement percentage: 43.67%
Spearman correlation: 0.6893


,Income band,Annual household income
0,1,"Up to £2,599"
1,2,"£2,600–£5,199"
2,3,"£5,200–£10,399"
3,4,"£10,400–£15,599"
4,5,"£15,600–£20,799"


,Measure or condition,Participants
0,Wave 1 harmonised income,7309
1,Wave 3 harmonised income,7935
2,Available in both waves,6465
3,Exact agreement across waves,2823
4,Wave 3 unavailable; Wave 1 available,844


In [210]:
# 24: Wave 2 household-income contribution

import pandas as pd
from scipy.stats import spearmanr
w2_income_file_name = source_register.loc[source_register['Wave'].eq('Wave 2') & source_register['Source type'].eq('Family background'),
    'File name'].iloc[0]
w2_income_path = source_directory / w2_income_file_name
w2_income_data = pd.read_stata(w2_income_path, columns=['NSID', 'W2Inc1estMP'], convert_categoricals=False)
w2_income_data['NSID'] = w2_income_data['NSID'].astype('string').str.strip().str.replace('\\.0$', '', regex=True)
w2_income_data = w2_income_data.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)
w2_income_original_band = pd.to_numeric(w2_income_data['W2Inc1estMP'],
    errors='coerce').where(lambda values: values.between(0, 32))
w2_income_9_band = w2_income_original_band.map(w1_to_9_band_map).astype('Float64')
wave_3_income_9_band_before_transition = w3_income_9_band.where(w3_income_timing_status.eq('Before September 2006'))
w3_w1_income_candidate = wave_3_income_9_band_before_transition.combine_first(w1_income_9_band)
wave_2_incremental_mask = w3_w1_income_candidate.isna() & w2_income_9_band.notna()
comparison_rows = []
for comparison_name, comparison_measure in [('Wave 1', w1_income_9_band), ('Wave 3 before September 2006',
    wave_3_income_9_band_before_transition)]:
    both_available = w2_income_9_band.notna() & comparison_measure.notna()
    if both_available.sum() > 1:
        exact_agreement = w2_income_9_band.loc[both_available].eq(comparison_measure.loc[both_available])
        correlation = spearmanr(w2_income_9_band.loc[both_available],
            comparison_measure.loc[both_available]).statistic
    else:
        exact_agreement = pd.Series(dtype=bool)
        correlation = float('nan')
    comparison_rows.append({'Comparison': f'Wave 2 with {comparison_name}',
        'Complete cases': int(both_available.sum()), 'Exact agreement': int(exact_agreement.sum()), 'Agreement percentage': round(exact_agreement.mean() * 100,
        2) if len(exact_agreement) > 0 else pd.NA, 'Spearman correlation': round(correlation,
        4) if pd.notna(correlation) else pd.NA})
wave_2_income_comparison = pd.DataFrame(comparison_rows)
wave_2_income_contribution = pd.DataFrame({'Condition': ['Valid Wave 2 income', 'Wave 2 also observed at Wave 1',
    'Wave 2 also observed at eligible Wave 3', 'Wave 2 recovers current missing cases', 'Still unavailable after Wave 2 fallback', 'Valid after Wave 3, Wave 2 and Wave 1 combination'], 'Participants': [int(w2_income_9_band.notna().sum()),
    int((w2_income_9_band.notna() & w1_income_9_band.notna()).sum()), int((w2_income_9_band.notna() & wave_3_income_9_band_before_transition.notna()).sum()), int(wave_2_incremental_mask.sum()), int((w3_w1_income_candidate.isna() & w2_income_9_band.isna()).sum()), int(wave_3_income_9_band_before_transition.combine_first(w2_income_9_band).combine_first(w1_income_9_band).notna().sum())]})
print('Wave 2 comparison with other pre-transition income measures:')
display_limited(wave_2_income_comparison)
print('Wave 2 incremental contribution:')
display_limited(wave_2_income_contribution)
if wave_2_incremental_mask.any():
    print('Income-band distribution among cases recovered only by Wave 2:')
    display_limited(w2_income_9_band.loc[wave_2_incremental_mask].astype('Int64').value_counts().sort_index().rename_axis('Income band').reset_index(name='Participants'))

Wave 2 comparison with other pre-transition income measures:


,Comparison,Complete cases,Exact agreement,Agreement percentage,Spearman correlation
0,Wave 2 with Wave 1,95,15,15.79,0.3353
1,Wave 2 with Wave 3 before September 2006,103,16,15.53,0.3519


Wave 2 incremental contribution:


,Condition,Participants
0,Valid Wave 2 income,140
1,Wave 2 also observed at Wave 1,95
2,Wave 2 also observed at eligible Wave 3,103
3,Wave 2 recovers current missing cases,17
4,Still unavailable after Wave 2 fallback,979


Income-band distribution among cases recovered only by Wave 2:


,Income band,Participants
0,1,1
1,2,2
2,3,5
3,4,4
4,5,2


In [211]:
# 25: Household-income decision-variable check

import pandas as pd
expected_income_variables = set(household_income_candidates['Variable']) | {'W3intmnthMP', 'W3intyearMP'}
register_variable_names = set(variable_decision_register['Variable'].dropna().astype(str))
missing_income_variables = sorted(expected_income_variables - register_variable_names)
matched_income_rows = variable_decision_register.loc[variable_decision_register['Variable'].isin(expected_income_variables),
    ['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Review status',
    'Review outcome']].sort_values(['Wave', 'Source file', 'Variable position']).reset_index(drop=True)
matched_variable_duplicates = matched_income_rows.loc[matched_income_rows['Variable'].duplicated(keep=False)].sort_values('Variable').reset_index(drop=True)
possible_name_matches = []
for missing_variable in missing_income_variables:
    possible_rows = variable_decision_register.loc[variable_decision_register['Variable'].astype(str).str.contains(missing_variable.replace('W3',
        ''), case=False, na=False, regex=False), ['Wave', 'Source file', 'Variable position', 'Variable',
        'Variable label']].copy()
    if not possible_rows.empty:
        possible_rows.insert(0, 'Expected variable', missing_variable)
        possible_name_matches.append(possible_rows)
if possible_name_matches:
    possible_name_matches = pd.concat(possible_name_matches, ignore_index=True)
else:
    possible_name_matches = pd.DataFrame(columns=['Expected variable', 'Wave', 'Source file', 'Variable position',
        'Variable', 'Variable label'])
print(f'Expected household-income review variables: {len(expected_income_variables):,}')
print(f'Matched decision-register rows: {len(matched_income_rows):,}')
print(f"Matched unique variables: {matched_income_rows['Variable'].nunique():,}")
print('Expected variables absent from the decision register:')
display_limited(pd.DataFrame({'Missing variable': missing_income_variables}))
print('Possible alternative names:')
display_limited(possible_name_matches)
print('Duplicate matched register rows:')
display_limited(matched_variable_duplicates)

Expected household-income review variables: 20
Matched decision-register rows: 22
Matched unique variables: 20
Expected variables absent from the decision register:


,Missing variable


Possible alternative names:


,Expected variable,Wave,Source file,Variable position,Variable,Variable label


Duplicate matched register rows:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Review status,Review outcome
0,Wave 3,Family background,wave_three_lsype_family_background_2020,10,W3intmnthMP,MP- month of interview,Use requires interview timing confirming Janua...,Retain as construction source
1,Wave 3,Parental attitudes,wave_three_lsype_parental_attitudes_file_16_06_08,8,W3intmnthMP,MP- month of interview,Use requires interview timing confirming Janua...,Exclude from predictor set
2,Wave 3,Family background,wave_three_lsype_family_background_2020,11,W3intyearMP,MP- year of interview,Use requires interview timing confirming Janua...,Retain as construction source
3,Wave 3,Parental attitudes,wave_three_lsype_parental_attitudes_file_16_06_08,9,W3intyearMP,MP- year of interview,Use requires interview timing confirming Janua...,Exclude from predictor set


In [212]:
# 26: Household-income construction and source-specific decisions

import pandas as pd
wave_3_income_before_transition = w3_income_9_band.where(w3_income_timing_status.eq('Before September 2006'))
household_income_band = wave_3_income_before_transition.combine_first(w1_income_9_band).astype('Int64')
household_income_source = pd.Series('Unavailable', index=household_income_band.index, dtype='string')
household_income_source.loc[wave_3_income_before_transition.notna()] = 'Wave 3 before September 2006'
household_income_source.loc[wave_3_income_before_transition.isna() & w1_income_9_band.notna()] = 'Wave 1 edited-income fallback'
household_income_label_map = {1: 'Up to £2,599', 2: '£2,600–£5,199', 3: '£5,200–£10,399', 4: '£10,400–£15,599',
    5: '£15,600–£20,799', 6: '£20,800–£25,999', 7: '£26,000–£31,199', 8: '£31,200–£36,399', 9: '£36,400 or more'}
household_income_label = household_income_band.map(household_income_label_map).astype('string')
household_income_review = pd.DataFrame({'NSID': stage_2_ids.to_numpy(),
    'household_income_band': household_income_band.to_numpy(), 'household_income_label': household_income_label.to_numpy(), 'household_income_source': household_income_source.to_numpy()})
assert len(household_income_review) == 9767
assert household_income_review['NSID'].nunique() == 9767
assert household_income_review['NSID'].duplicated().sum() == 0
assert household_income_review['NSID'].isna().sum() == 0
assert household_income_review['household_income_band'].dropna().between(1, 9).all()
wave_1_family_source = household_income_candidates.loc[household_income_candidates['Wave'].eq('Wave 1'),
    'Source file'].iloc[0]
wave_2_family_source = household_income_candidates.loc[household_income_candidates['Wave'].eq('Wave 2'),
    'Source file'].iloc[0]
wave_3_family_source = household_income_candidates.loc[household_income_candidates['Wave'].eq('Wave 3'),
    'Source file'].iloc[0]
wave_4_family_source = household_income_candidates.loc[household_income_candidates['Wave'].eq('Wave 4'),
    'Source file'].iloc[0]
income_candidate_keys = set(household_income_candidates[['Source file', 'Variable']].itertuples(index=False,
    name=None))
primary_income_keys = {(wave_3_family_source, 'W3incestMP')}
fallback_income_keys = {(wave_1_family_source, 'W1inc1est')}
income_derivation_keys = {(wave_3_family_source, 'W3intmnthMP'), (wave_3_family_source, 'W3intyearMP')}
income_review_support_keys = {(wave_1_family_source, 'W1inc1estMP'), (wave_2_family_source, 'W2Inc1estMP'),
    (wave_3_family_source, 'W3incestm'), (wave_3_family_source, 'W3incestw')}
wave_4_income_keys = {(wave_4_family_source, 'W4Inc1EstMP')}
excluded_income_component_keys = income_candidate_keys - primary_income_keys - fallback_income_keys - income_review_support_keys - wave_4_income_keys
all_income_review_keys = income_candidate_keys | income_derivation_keys
working_variable_decision_register = variable_decision_register.copy()

def register_key_mask(register, selected_keys):
    register_keys = zip(register['Source file'].astype(str), register['Variable'].astype(str))
    return pd.Series([key in selected_keys for key in register_keys], index=register.index)
income_review_mask = register_key_mask(working_variable_decision_register, all_income_review_keys)
primary_income_mask = register_key_mask(working_variable_decision_register, primary_income_keys)
fallback_income_mask = register_key_mask(working_variable_decision_register, fallback_income_keys)
income_derivation_mask = register_key_mask(working_variable_decision_register, income_derivation_keys)
income_support_mask = register_key_mask(working_variable_decision_register, income_review_support_keys)
income_component_mask = register_key_mask(working_variable_decision_register, excluded_income_component_keys)
wave_4_income_mask = register_key_mask(working_variable_decision_register, wave_4_income_keys)
other_timing_rows = variable_decision_register['Variable'].isin(['W3intmnthMP',
    'W3intyearMP']) & ~variable_decision_register['Source file'].eq(wave_3_family_source)
other_timing_decisions_before = variable_decision_register.loc[other_timing_rows, ['Source file', 'Variable',
    'Review status', 'Review outcome', 'Decision reason']].copy().reset_index(drop=True)
working_variable_decision_register.loc[income_review_mask, 'Review status'] = 'Reviewed'
working_variable_decision_register.loc[income_review_mask, 'Substantive domain'] = 'Family socioeconomic background'
working_variable_decision_register.loc[income_review_mask,
    'Documentation source'] = 'Wave 1 to Wave 4 variable labels and value labels'
working_variable_decision_register.loc[primary_income_mask, 'Review outcome'] = 'Retain as construction source'
working_variable_decision_register.loc[primary_income_mask,
    'Decision reason'] = 'Primary annual household-income measure when the interview occurred before September 2006'
working_variable_decision_register.loc[primary_income_mask,
    'Leakage assessment'] = 'No leakage after restricting the measure to interviews before the post-16 transition'
working_variable_decision_register.loc[primary_income_mask,
    'Reference-period assessment'] = 'Wave 3 interview before September 2006'
working_variable_decision_register.loc[primary_income_mask,
    'Review notes'] = 'Harmonised to nine annual income bands; 7,888 participants contributed values'
working_variable_decision_register.loc[fallback_income_mask, 'Review outcome'] = 'Retain as construction source'
working_variable_decision_register.loc[fallback_income_mask,
    'Decision reason'] = 'Edited gross household-income measure used where an eligible Wave 3 value is unavailable'
working_variable_decision_register.loc[fallback_income_mask,
    'Leakage assessment'] = 'No leakage: measured before the post-16 transition'
working_variable_decision_register.loc[fallback_income_mask,
    'Reference-period assessment'] = 'Wave 1 gross annual household income'
working_variable_decision_register.loc[fallback_income_mask,
    'Review notes'] = 'Harmonised to nine annual income bands; 883 participants contributed fallback values'
working_variable_decision_register.loc[income_derivation_mask, 'Review outcome'] = 'Retain as derivation input'
working_variable_decision_register.loc[income_derivation_mask,
    'Decision reason'] = 'Required to determine whether the Wave 3 income measurement preceded September 2006'
working_variable_decision_register.loc[income_derivation_mask,
    'Leakage assessment'] = 'No leakage: used only to apply the pre-transition measurement restriction'
working_variable_decision_register.loc[income_derivation_mask,
    'Reference-period assessment'] = 'Wave 3 main-parent interview date'
working_variable_decision_register.loc[income_derivation_mask,
    'Review notes'] = 'Construction input only; not retained as a predictor'
working_variable_decision_register.loc[income_support_mask, 'Review outcome'] = 'Retain as review support only'
working_variable_decision_register.loc[income_support_mask,
    'Decision reason'] = 'Used to assess editing, sparse coverage, duplicate representations or cross-wave consistency'
working_variable_decision_register.loc[income_support_mask,
    'Leakage assessment'] = 'No leakage; not retained in the predictor set'
working_variable_decision_register.loc[income_support_mask,
    'Reference-period assessment'] = 'Wave-specific household-income measurement'
working_variable_decision_register.loc[income_support_mask,
    'Review notes'] = 'Includes the Wave 1 raw measure, the sparse Wave 2 measure and duplicate Wave 3 representations'
working_variable_decision_register.loc[income_component_mask, 'Review outcome'] = 'Exclude from predictor set'
working_variable_decision_register.loc[income_component_mask,
    'Decision reason'] = 'Component, follow-up, respondent-specific, benefit-specific or provenance variable rather than the selected total household-income construct'
working_variable_decision_register.loc[income_component_mask,
    'Leakage assessment'] = 'No leakage; excluded because of construct mismatch or overlap'
working_variable_decision_register.loc[income_component_mask,
    'Reference-period assessment'] = 'Wave 1 or Wave 2 income information'
working_variable_decision_register.loc[income_component_mask,
    'Review notes'] = 'Not required after selecting one harmonised household-income predictor'
working_variable_decision_register.loc[wave_4_income_mask, 'Review outcome'] = 'Exclude from predictor set'
working_variable_decision_register.loc[wave_4_income_mask,
    'Decision reason'] = 'Earlier waves provide the selected household-income construct'
working_variable_decision_register.loc[wave_4_income_mask,
    'Leakage assessment'] = 'Timing concern: Wave 4 may contain information measured at or after the outcome boundary'
working_variable_decision_register.loc[wave_4_income_mask,
    'Reference-period assessment'] = 'Wave 4 current household income'
working_variable_decision_register.loc[wave_4_income_mask, 'Review notes'] = 'Not used in predictor construction'
household_income_decision_rows = working_variable_decision_register.loc[income_review_mask, ['Wave', 'Source type',
    'Source file', 'Variable position', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment']].sort_values(['Wave',
    'Source file', 'Variable position']).reset_index(drop=True)
assert len(household_income_decision_rows) == 20
assert household_income_decision_rows[['Source file', 'Variable']].drop_duplicates().shape[0] == 20
assert household_income_decision_rows['Review outcome'].notna().all()
assert household_income_decision_rows['Decision reason'].notna().all()
assert household_income_decision_rows['Leakage assessment'].notna().all()
other_timing_rows_after = working_variable_decision_register['Variable'].isin(['W3intmnthMP',
    'W3intyearMP']) & ~working_variable_decision_register['Source file'].eq(wave_3_family_source)
other_timing_decisions_after = working_variable_decision_register.loc[other_timing_rows_after, ['Source file',
    'Variable', 'Review status', 'Review outcome', 'Decision reason']].copy().reset_index(drop=True)
assert other_timing_decisions_before.equals(other_timing_decisions_after)
household_income_review_path = stage_2_output_directory / 'stage_2_household_income_review.csv'
household_income_review.to_csv(household_income_review_path, index=False)
working_variable_decision_register.to_csv(decision_register_path, index=False)
variable_decision_register = working_variable_decision_register.copy()
household_income_source_summary = household_income_review['household_income_source'].value_counts(dropna=False).rename_axis('Selected source').reset_index(name='Participants')
household_income_decision_summary = household_income_decision_rows['Review outcome'].value_counts().rename_axis('Review outcome').reset_index(name='Variables')
print(f'Valid household-income predictors: {household_income_band.notna().sum():,}')
print(f'Unavailable household-income predictors: {household_income_band.isna().sum():,}')
print(f'Review file: {household_income_review_path}')
display_limited(household_income_source_summary)
display_limited(household_income_decision_summary)
print('Source-variable decisions:')
display_limited(household_income_decision_rows)

Valid household-income predictors: 8,771
Unavailable household-income predictors: 996
Review file: data_derived\stage_2_predictor_construction\stage_2_household_income_review.csv


,Selected source,Participants
0,Wave 3 before September 2006,7888
1,Unavailable,996
2,Wave 1 edited-income fallback,883


,Review outcome,Variables
0,Exclude from predictor set,12
1,Retain as review support only,4
2,Retain as construction source,2
3,Retain as derivation input,2


Source-variable decisions:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Review outcome,Decision reason,Leakage assessment
0,Wave 1,Family background,wave_one_lsype_family_background_2020,74,W1inc1estMP,"MP: Total income from work, benefits, and anyt...",Retain as review support only,"Used to assess editing, sparse coverage, dupli...",No leakage; not retained in the predictor set
1,Wave 1,Family background,wave_one_lsype_family_background_2020,75,W1inc2estMP,"MP: Total income from work, benefits, and anyt...",Exclude from predictor set,"Component, follow-up, respondent-specific, ben...",No leakage; excluded because of construct mism...
2,Wave 1,Family background,wave_one_lsype_family_background_2020,162,W1SeiInc2MP,MP: MP's estimated income from self-employment...,Exclude from predictor set,"Component, follow-up, respondent-specific, ben...",No leakage; excluded because of construct mism...
3,Wave 1,Family background,wave_one_lsype_family_background_2020,381,W1inc1est,DV: Estimate of gross household income (edited),Retain as construction source,Edited gross household-income measure used whe...,No leakage: measured before the post-16 transi...
4,Wave 2,Family background,wave_two_lsype_family_background_2020,68,W2Inc1estMP,"MP: Total income from work, benefits, and anyt...",Retain as review support only,"Used to assess editing, sparse coverage, dupli...",No leakage; not retained in the predictor set


In [213]:
# 27: Housing-tenure candidate variables

family_background_mask = variable_decision_register['Source type'].astype(str).eq('Family background')
housing_tenure_variable_names = variable_decision_register['Variable'].astype(str).str.contains('tenur|tenure|homeown|owner|mortg|rent|landlord|council|housing|accom',
    case=False, na=False)
housing_tenure_labels = variable_decision_register['Variable label'].astype(str).str.contains('housing tenure|tenure of|own(?:s|ed)? home|owner.?occup|mortgage|rent(?:ed|ing)?|private landlord|council tenant|housing association|accommodation tenure',
    case=False, na=False)
housing_tenure_candidates = variable_decision_register.loc[family_background_mask & (housing_tenure_variable_names | housing_tenure_labels),
    ['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Timing status',
    'Review status', 'Review outcome', 'Decision reason']].sort_values(['Wave', 'Source file',
    'Variable position']).reset_index(drop=True)
print(f'Housing-tenure candidate variables found: {len(housing_tenure_candidates):,}')
display_limited(housing_tenure_candidates)

Housing-tenure candidate variables found: 533


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Decision reason
0,Wave 1,Family background,wave_one_lsype_family_background_2020,20,W1mainres,Admin: Person number of main parent,Pre-transition source,"Variable-level coding, routing and reference-p...",Retain as review support only,Identifies a respondent or household member wi...
1,Wave 1,Family background,wave_one_lsype_family_background_2020,21,W1secores,Admin: Person number of second parent,Pre-transition source,"Variable-level coding, routing and reference-p...",Retain as review support only,Identifies a respondent or household member wi...
2,Wave 1,Family background,wave_one_lsype_family_background_2020,37,W1ben1MP0a,MP: State benefits currently received by MP (o...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>
3,Wave 1,Family background,wave_one_lsype_family_background_2020,38,W1ben1MP0b,MP: State benefits currently received by MP (o...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>
4,Wave 1,Family background,wave_one_lsype_family_background_2020,39,W1ben1MP0c,MP: State benefits currently received by MP (o...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>


In [214]:
# 28: Refined housing-tenure candidate search

import pandas as pd
family_background_rows = variable_decision_register['Source type'].astype(str).eq('Family background')
housing_tenure_name_match = variable_decision_register['Variable'].astype(str).str.contains('tenur|homeown|ownhome|houseten|accomten|renthome|mortg',
    case=False, na=False, regex=True)
housing_tenure_label_match = variable_decision_register['Variable label'].astype(str).str.contains('\\bhousing tenure\\b|\\btenure of (?:the )?(?:home|property|accommodation)\\b|\\bown(?:s|ed|ing)? (?:the )?(?:home|property|accommodation)\\b|\\bowner[- ]?occup(?:ied|ier)?\\b|\\bmortgage\\b|\\brent(?:s|ed|ing)? (?:the )?(?:home|property|accommodation)\\b|\\brent[- ]free\\b|\\bprivate landlord\\b|\\blocal authority\\b|\\bcouncil (?:tenant|housing|accommodation)\\b|\\bhousing association\\b|\\baccommodation (?:owned|rented)\\b|\\bhome (?:owned|rented)\\b',
    case=False, na=False, regex=True)
refined_housing_tenure_candidates = variable_decision_register.loc[family_background_rows & (housing_tenure_name_match | housing_tenure_label_match),
    ['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Timing status',
    'Review status', 'Review outcome', 'Decision reason']].copy()
refined_housing_tenure_candidates['Name match'] = housing_tenure_name_match.loc[refined_housing_tenure_candidates.index].to_numpy()
refined_housing_tenure_candidates['Label match'] = housing_tenure_label_match.loc[refined_housing_tenure_candidates.index].to_numpy()
refined_housing_tenure_candidates = refined_housing_tenure_candidates.sort_values(['Wave', 'Source file',
    'Variable position']).reset_index(drop=True)
print(f'Refined housing-tenure candidate variables found: {len(refined_housing_tenure_candidates):,}')
display_limited(refined_housing_tenure_candidates)

Refined housing-tenure candidate variables found: 25


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Decision reason,Name match,Label match
0,Wave 2,Family background,wave_two_lsype_family_background_2020,351,W2Mhelp1MP0a,MP: Whether anyone in MP's household gets anyt...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,True
1,Wave 2,Family background,wave_two_lsype_family_background_2020,352,W2MhelpMP0b,MP: Whether anyone in MP's household gets anyt...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,True
2,Wave 2,Family background,wave_two_lsype_family_background_2020,353,W2MhelpMP0c,MP: Whether anyone in MP's household gets anyt...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,True
3,Wave 2,Family background,wave_two_lsype_family_background_2020,354,W2MhelpMP0d,MP: Whether anyone in MP's household gets anyt...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,True
4,Wave 2,Family background,wave_two_lsype_family_background_2020,355,W2MhelpMP0e,MP: Whether anyone in MP's household gets anyt...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,True


In [215]:
# 29: Documented housing-tenure measure

import pandas as pd
housing_tenure_variable = 'W1hous12HH'
housing_tenure_register_rows = variable_decision_register.loc[variable_decision_register['Variable'].astype(str).str.casefold().eq(housing_tenure_variable.casefold()),
    ['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Timing status',
    'Review status', 'Review outcome', 'Decision reason']].copy().reset_index(drop=True)
print(f'Register rows found for W1hous12HH: {len(housing_tenure_register_rows):,}')
display_limited(housing_tenure_register_rows)
assert len(housing_tenure_register_rows) == 1
housing_tenure_source_key = housing_tenure_register_rows.loc[0, 'Source file']
housing_tenure_source_path = source_file_lookup[housing_tenure_source_key]
housing_tenure_raw = pd.read_stata(housing_tenure_source_path, columns=['NSID', housing_tenure_variable],
    convert_categoricals=False)
housing_tenure_labelled = pd.read_stata(housing_tenure_source_path, columns=[housing_tenure_variable],
    convert_categoricals=True)
housing_tenure_code_labels = pd.DataFrame({'Code': pd.to_numeric(housing_tenure_raw[housing_tenure_variable],
    errors='coerce'), 'Label': housing_tenure_labelled[housing_tenure_variable].astype('string')}).dropna(subset=['Code']).drop_duplicates().sort_values('Code').reset_index(drop=True)
housing_tenure_raw['NSID'] = housing_tenure_raw['NSID'].astype('string').str.strip().str.replace('\\.0$', '',
    regex=True)
housing_tenure_values = housing_tenure_raw.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)[housing_tenure_variable]
housing_tenure_values = pd.to_numeric(housing_tenure_values, errors='coerce')
housing_tenure_label_map = dict(zip(housing_tenure_code_labels['Code'], housing_tenure_code_labels['Label']))
housing_tenure_distribution = pd.DataFrame({'Code': housing_tenure_values,
    'Label': housing_tenure_values.map(housing_tenure_label_map)}).value_counts(['Code', 'Label'],
    dropna=False).rename('Participants').reset_index().sort_values('Code', na_position='last').reset_index(drop=True)
housing_tenure_structure = pd.DataFrame({'Condition': ['Non-negative responses', 'Negative codes', 'No source record',
    'Distinct non-negative values'], 'Participants or values': [int(housing_tenure_values.ge(0).sum()),
    int(housing_tenure_values.lt(0).sum()), int(housing_tenure_values.isna().sum()), int(housing_tenure_values.loc[housing_tenure_values.ge(0)].nunique())]})
print('Stored housing-tenure codes and labels:')
display_limited(housing_tenure_code_labels)
print('Housing-tenure coverage:')
display_limited(housing_tenure_structure)
print('Participant distribution:')
display_limited(housing_tenure_distribution)

Register rows found for W1hous12HH: 1


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Decision reason
0,Wave 1,Family background,wave_one_lsype_family_background_2020,13,W1hous12HH,HH: Tenure,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>


Stored housing-tenure codes and labels:


,Code,Label
0,-999,Household grid missing
1,-92,Refused
2,-91,Not applicable
3,-1,Don't know
4,1,Owned outright


Housing-tenure coverage:


,Condition,Participants or values
0,Non-negative responses,9454
1,Negative codes,70
2,No source record,243
3,Distinct non-negative values,8


Participant distribution:


,Code,Label,Participants
0,-999.0,Household grid missing,37
1,-92.0,Refused,16
2,-91.0,Not applicable,9
3,-1.0,Don't know,8
4,1.0,Owned outright,1281


In [216]:
# 30: Cross-wave housing-tenure measures

import pandas as pd
direct_tenure_name_match = variable_decision_register['Variable'].astype(str).str.contains('hous12|tenur', case=False,
    na=False, regex=True)
direct_tenure_label_match = variable_decision_register['Variable label'].astype(str).str.contains('\\btenure\\b',
    case=False, na=False, regex=True)
cross_wave_housing_tenure_candidates = variable_decision_register.loc[direct_tenure_name_match | direct_tenure_label_match,
    ['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Timing status',
    'Review status', 'Review outcome', 'Decision reason']].copy()
cross_wave_housing_tenure_candidates['Name match'] = direct_tenure_name_match.loc[cross_wave_housing_tenure_candidates.index].to_numpy()
cross_wave_housing_tenure_candidates['Label match'] = direct_tenure_label_match.loc[cross_wave_housing_tenure_candidates.index].to_numpy()
cross_wave_housing_tenure_candidates = cross_wave_housing_tenure_candidates.sort_values(['Wave', 'Source type',
    'Source file', 'Variable position']).reset_index(drop=True)
print(f'Direct housing-tenure variables found: {len(cross_wave_housing_tenure_candidates):,}')
display_limited(cross_wave_housing_tenure_candidates)

Direct housing-tenure variables found: 4


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Decision reason,Name match,Label match
0,Wave 1,Family background,wave_one_lsype_family_background_2020,13,W1hous12HH,HH: Tenure,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,True,True
1,Wave 2,Family background,wave_two_lsype_family_background_2020,12,W2Hous12HH,HH: Tenure,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,True,True
2,Wave 3,Family background,wave_three_lsype_family_background_2020,18,W3hous12HH,HH: Tenure,Near-transition source,Use requires interview timing confirming Janua...,Pending review,<NA>,True,True
3,Wave 4,Family background,wave_four_lsype_family_background_2020,19,W4Hous12HH,HH: Tenure,At or after transition,Stable characteristics or retrospective pre-tr...,Pending review,<NA>,True,True


In [217]:
# 31: Cross-wave housing-tenure structure

import pandas as pd
housing_tenure_wave_specifications = [{'Wave': 'Wave 1',
    'Source file': cross_wave_housing_tenure_candidates.loc[cross_wave_housing_tenure_candidates['Wave'].eq('Wave 1'),
    'Source file'].iloc[0], 'Variable': 'W1hous12HH'}, {'Wave': 'Wave 2',
    'Source file': cross_wave_housing_tenure_candidates.loc[cross_wave_housing_tenure_candidates['Wave'].eq('Wave 2'),
    'Source file'].iloc[0], 'Variable': 'W2Hous12HH'}, {'Wave': 'Wave 3',
    'Source file': cross_wave_housing_tenure_candidates.loc[cross_wave_housing_tenure_candidates['Wave'].eq('Wave 3'),
    'Source file'].iloc[0], 'Variable': 'W3hous12HH'}, {'Wave': 'Wave 4',
    'Source file': cross_wave_housing_tenure_candidates.loc[cross_wave_housing_tenure_candidates['Wave'].eq('Wave 4'),
    'Source file'].iloc[0], 'Variable': 'W4Hous12HH'}]
housing_tenure_by_wave = {}
housing_tenure_code_label_rows = []
housing_tenure_coverage_rows = []
housing_tenure_negative_code_rows = []
for specification in housing_tenure_wave_specifications:
    wave = specification['Wave']
    source_file = specification['Source file']
    variable = specification['Variable']
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', variable], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=[variable], convert_categoricals=True)
    code_label_map = pd.DataFrame({'Code': pd.to_numeric(raw_data[variable], errors='coerce'),
        'Label': labelled_data[variable].astype('string')}).dropna(subset=['Code']).drop_duplicates().sort_values('Code').reset_index(drop=True)
    code_label_map.insert(0, 'Wave', wave)
    code_label_map.insert(1, 'Variable', variable)
    housing_tenure_code_label_rows.append(code_label_map)
    raw_data['NSID'] = raw_data['NSID'].astype('string').str.strip().str.replace('\\.0$', '', regex=True)
    aligned_values = raw_data.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)[variable]
    aligned_values = pd.to_numeric(aligned_values, errors='coerce')
    housing_tenure_by_wave[wave] = aligned_values
    non_negative_values = aligned_values.loc[aligned_values.ge(0)]
    negative_values = aligned_values.loc[aligned_values.lt(0)]
    housing_tenure_coverage_rows.append({'Wave': wave, 'Variable': variable,
        'Valid responses': int(non_negative_values.notna().sum()), 'Negative codes': int(negative_values.notna().sum()), 'No source record': int(aligned_values.isna().sum()), 'Distinct valid codes': int(non_negative_values.nunique()), 'Minimum valid code': non_negative_values.min() if non_negative_values.notna().any() else pd.NA, 'Maximum valid code': non_negative_values.max() if non_negative_values.notna().any() else pd.NA})
    for code, count in negative_values.value_counts().sort_index().items():
        housing_tenure_negative_code_rows.append({'Wave': wave, 'Variable': variable, 'Negative code': code,
            'Participants': int(count)})
housing_tenure_code_labels_all = pd.concat(housing_tenure_code_label_rows, ignore_index=True)
housing_tenure_coverage = pd.DataFrame(housing_tenure_coverage_rows)
housing_tenure_negative_codes = pd.DataFrame(housing_tenure_negative_code_rows)
print('Housing-tenure codes and labels by wave:')
display_limited(housing_tenure_code_labels_all)
print('Housing-tenure coverage by wave:')
display_limited(housing_tenure_coverage)
print('Observed negative codes:')
display_limited(housing_tenure_negative_codes)

Housing-tenure codes and labels by wave:


,Wave,Variable,Code,Label
0,Wave 1,W1hous12HH,-999,Household grid missing
1,Wave 1,W1hous12HH,-92,Refused
2,Wave 1,W1hous12HH,-91,Not applicable
3,Wave 1,W1hous12HH,-1,Don't know
4,Wave 1,W1hous12HH,1,Owned outright


Housing-tenure coverage by wave:


,Wave,Variable,Valid responses,Negative codes,No source record,Distinct valid codes,Minimum valid code,Maximum valid code
0,Wave 1,W1hous12HH,9454,70,243,8,1.0,8.0
1,Wave 2,W2Hous12HH,9482,39,246,8,1.0,8.0
2,Wave 3,W3hous12HH,9461,48,258,8,1.0,8.0
3,Wave 4,W4Hous12HH,9632,124,11,8,1.0,8.0


Observed negative codes:


,Wave,Variable,Negative code,Participants
0,Wave 1,W1hous12HH,-999.0,37
1,Wave 1,W1hous12HH,-92.0,16
2,Wave 1,W1hous12HH,-91.0,9
3,Wave 1,W1hous12HH,-1.0,8
4,Wave 2,W2Hous12HH,-998.0,7


In [218]:
# 32: Cross-wave housing-tenure comparison

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
w1_housing_tenure = housing_tenure_by_wave['Wave 1'].where(lambda values: values.between(1, 8)).astype('Float64')
w2_housing_tenure = housing_tenure_by_wave['Wave 2'].where(lambda values: values.between(1, 8)).astype('Float64')
w3_housing_tenure = housing_tenure_by_wave['Wave 3'].where(lambda values: values.between(1, 8)).astype('Float64')
w3_housing_tenure_before_transition = w3_housing_tenure.where(w3_income_timing_status.eq('Before September 2006'))

def cramers_v(first, second):
    complete = first.notna() & second.notna()
    contingency_table = pd.crosstab(first.loc[complete], second.loc[complete])
    chi_square = chi2_contingency(contingency_table, correction=False)[0]
    sample_size = contingency_table.to_numpy().sum()
    rows, columns = contingency_table.shape
    denominator = min(rows - 1, columns - 1)
    if sample_size == 0 or denominator == 0:
        return np.nan
    return np.sqrt(chi_square / (sample_size * denominator))
housing_tenure_comparison_rows = []
for first_name, first_measure, second_name, second_measure in [('Wave 1', w1_housing_tenure, 'Wave 2',
    w2_housing_tenure), ('Wave 1', w1_housing_tenure, 'Wave 3', w3_housing_tenure), ('Wave 2', w2_housing_tenure,
    'Wave 3', w3_housing_tenure)]:
    complete_cases = first_measure.notna() & second_measure.notna()
    exact_agreement = first_measure.loc[complete_cases].eq(second_measure.loc[complete_cases])
    housing_tenure_comparison_rows.append({'Comparison': f'{first_name} with {second_name}',
        'Complete cases': int(complete_cases.sum()), 'Exact agreement': int(exact_agreement.sum()), 'Agreement percentage': round(exact_agreement.mean() * 100,
        2), "Cramér's V": round(cramers_v(first_measure, second_measure), 4)})
housing_tenure_cross_wave_comparison = pd.DataFrame(housing_tenure_comparison_rows)
w2_w1_housing_tenure_candidate = w2_housing_tenure.combine_first(w1_housing_tenure)
w3_w2_w1_housing_tenure_candidate = w3_housing_tenure_before_transition.combine_first(w2_housing_tenure).combine_first(w1_housing_tenure)
housing_tenure_candidate_source = pd.Series('Unavailable', index=stage_2_ids, dtype='string')
housing_tenure_candidate_source.loc[w3_housing_tenure_before_transition.notna()] = 'Wave 3 before September 2006'
housing_tenure_candidate_source.loc[w3_housing_tenure_before_transition.isna() & w2_housing_tenure.notna()] = 'Wave 2 fallback'
housing_tenure_candidate_source.loc[w3_housing_tenure_before_transition.isna() & w2_housing_tenure.isna() & w1_housing_tenure.notna()] = 'Wave 1 fallback'
housing_tenure_coverage_comparison = pd.DataFrame({'Measure or condition': ['Wave 1 valid tenure',
    'Wave 2 valid tenure', 'Wave 3 valid tenure', 'Wave 3 valid tenure before September 2006', 'Wave 3 valid tenure in September 2006', 'Wave 3 valid tenure with interview date unavailable', 'Wave 2 with Wave 1 fallback', 'Eligible Wave 3 with Wave 2 and Wave 1 fallback', 'Unavailable after all eligible sources'], 'Participants': [int(w1_housing_tenure.notna().sum()),
    int(w2_housing_tenure.notna().sum()), int(w3_housing_tenure.notna().sum()), int(w3_housing_tenure_before_transition.notna().sum()), int((w3_housing_tenure.notna() & w3_income_timing_status.eq('September 2006')).sum()), int((w3_housing_tenure.notna() & w3_income_timing_status.eq('Interview date unavailable')).sum()), int(w2_w1_housing_tenure_candidate.notna().sum()), int(w3_w2_w1_housing_tenure_candidate.notna().sum()), int(w3_w2_w1_housing_tenure_candidate.isna().sum())]})
housing_tenure_source_summary = housing_tenure_candidate_source.value_counts(dropna=False).rename_axis('Candidate source').reset_index(name='Participants')
print('Cross-wave housing-tenure agreement:')
display_limited(housing_tenure_cross_wave_comparison)
print('Housing-tenure timing and coverage:')
display_limited(housing_tenure_coverage_comparison)
print('Candidate source contribution:')
display_limited(housing_tenure_source_summary)

Cross-wave housing-tenure agreement:


,Comparison,Complete cases,Exact agreement,Agreement percentage,Cramér's V
0,Wave 1 with Wave 2,9418,8130,86.32,0.6478
1,Wave 1 with Wave 3,9397,7867,83.72,0.6029
2,Wave 2 with Wave 3,9429,8237,87.36,0.6579


Housing-tenure timing and coverage:


,Measure or condition,Participants
0,Wave 1 valid tenure,9454
1,Wave 2 valid tenure,9482
2,Wave 3 valid tenure,9461
3,Wave 3 valid tenure before September 2006,9327
4,Wave 3 valid tenure in September 2006,14


Candidate source contribution:


,Candidate source,Participants
0,Wave 3 before September 2006,9327
1,Unavailable,246
2,Wave 2 fallback,184
3,Wave 1 fallback,10


In [219]:
# 33: Housing-tenure construction and decisions

import pandas as pd
wave_3_housing_tenure_before_transition = w3_housing_tenure.where(w3_income_timing_status.eq('Before September 2006'))
housing_tenure = wave_3_housing_tenure_before_transition.combine_first(w2_housing_tenure).combine_first(w1_housing_tenure).astype('Int64')
housing_tenure_source = pd.Series('Unavailable', index=housing_tenure.index, dtype='string')
housing_tenure_source.loc[wave_3_housing_tenure_before_transition.notna()] = 'Wave 3 before September 2006'
housing_tenure_source.loc[wave_3_housing_tenure_before_transition.isna() & w2_housing_tenure.notna()] = 'Wave 2 fallback'
housing_tenure_source.loc[wave_3_housing_tenure_before_transition.isna() & w2_housing_tenure.isna() & w1_housing_tenure.notna()] = 'Wave 1 fallback'
housing_tenure_label_map = {1: 'Owned outright', 2: 'Being bought on a mortgage or bank loan', 3: 'Shared ownership',
    4: 'Rented from a Council or New Town', 5: 'Rented from a Housing Association', 6: 'Rented privately', 7: 'Rent free', 8: 'Some other arrangement'}
housing_tenure_label = housing_tenure.map(housing_tenure_label_map).astype('string')
housing_tenure_review = pd.DataFrame({'NSID': stage_2_ids.to_numpy(), 'housing_tenure': housing_tenure.to_numpy(),
    'housing_tenure_label': housing_tenure_label.to_numpy(), 'housing_tenure_source': housing_tenure_source.to_numpy()})
assert len(housing_tenure_review) == 9767
assert housing_tenure_review['NSID'].nunique() == 9767
assert housing_tenure_review['NSID'].duplicated().sum() == 0
assert housing_tenure_review['NSID'].isna().sum() == 0
assert housing_tenure_review['housing_tenure'].dropna().between(1, 8).all()
housing_tenure_keys = set(cross_wave_housing_tenure_candidates[['Source file', 'Variable']].itertuples(index=False,
    name=None))
wave_1_housing_key = {(cross_wave_housing_tenure_candidates.loc[cross_wave_housing_tenure_candidates['Wave'].eq('Wave 1'),
    'Source file'].iloc[0], 'W1hous12HH')}
wave_2_housing_key = {(cross_wave_housing_tenure_candidates.loc[cross_wave_housing_tenure_candidates['Wave'].eq('Wave 2'),
    'Source file'].iloc[0], 'W2Hous12HH')}
wave_3_housing_key = {(cross_wave_housing_tenure_candidates.loc[cross_wave_housing_tenure_candidates['Wave'].eq('Wave 3'),
    'Source file'].iloc[0], 'W3hous12HH')}
wave_4_housing_key = {(cross_wave_housing_tenure_candidates.loc[cross_wave_housing_tenure_candidates['Wave'].eq('Wave 4'),
    'Source file'].iloc[0], 'W4Hous12HH')}

def source_variable_mask(register, selected_keys):
    row_keys = zip(register['Source file'].astype(str), register['Variable'].astype(str))
    return pd.Series([key in selected_keys for key in row_keys], index=register.index)
working_variable_decision_register = variable_decision_register.copy()
housing_review_mask = source_variable_mask(working_variable_decision_register, housing_tenure_keys)
wave_1_housing_mask = source_variable_mask(working_variable_decision_register, wave_1_housing_key)
wave_2_housing_mask = source_variable_mask(working_variable_decision_register, wave_2_housing_key)
wave_3_housing_mask = source_variable_mask(working_variable_decision_register, wave_3_housing_key)
wave_4_housing_mask = source_variable_mask(working_variable_decision_register, wave_4_housing_key)
working_variable_decision_register.loc[housing_review_mask, 'Review status'] = 'Reviewed'
working_variable_decision_register.loc[housing_review_mask, 'Substantive domain'] = 'Family socioeconomic background'
working_variable_decision_register.loc[housing_review_mask,
    'Documentation source'] = 'Wave 1 to Wave 4 value labels and Wave 3 interview timing'
working_variable_decision_register.loc[wave_3_housing_mask, 'Review outcome'] = 'Retain as construction source'
working_variable_decision_register.loc[wave_3_housing_mask,
    'Decision reason'] = 'Most recent housing-tenure measure recorded before the post-16 transition'
working_variable_decision_register.loc[wave_3_housing_mask,
    'Leakage assessment'] = 'No leakage after restricting the measure to interviews before September 2006'
working_variable_decision_register.loc[wave_3_housing_mask,
    'Reference-period assessment'] = 'Wave 3 interview before September 2006'
working_variable_decision_register.loc[wave_3_housing_mask,
    'Review notes'] = 'Primary source; 9,327 participants contributed values'
working_variable_decision_register.loc[wave_2_housing_mask, 'Review outcome'] = 'Retain as construction source'
working_variable_decision_register.loc[wave_2_housing_mask,
    'Decision reason'] = 'Fallback source where eligible Wave 3 housing tenure is unavailable'
working_variable_decision_register.loc[wave_2_housing_mask,
    'Leakage assessment'] = 'No leakage: measured before the post-16 transition'
working_variable_decision_register.loc[wave_2_housing_mask, 'Reference-period assessment'] = 'Wave 2 household tenure'
working_variable_decision_register.loc[wave_2_housing_mask,
    'Review notes'] = 'First fallback source; 184 participants contributed values'
working_variable_decision_register.loc[wave_1_housing_mask, 'Review outcome'] = 'Retain as construction source'
working_variable_decision_register.loc[wave_1_housing_mask,
    'Decision reason'] = 'Second fallback source where eligible Wave 3 and Wave 2 housing tenure are unavailable'
working_variable_decision_register.loc[wave_1_housing_mask,
    'Leakage assessment'] = 'No leakage: measured before the post-16 transition'
working_variable_decision_register.loc[wave_1_housing_mask, 'Reference-period assessment'] = 'Wave 1 household tenure'
working_variable_decision_register.loc[wave_1_housing_mask,
    'Review notes'] = 'Second fallback source; 10 participants contributed values'
working_variable_decision_register.loc[wave_4_housing_mask, 'Review outcome'] = 'Exclude from predictor set'
working_variable_decision_register.loc[wave_4_housing_mask,
    'Decision reason'] = 'Earlier waves provide the selected housing-tenure construct'
working_variable_decision_register.loc[wave_4_housing_mask,
    'Leakage assessment'] = 'Timing concern: Wave 4 may contain information measured at or after the outcome boundary'
working_variable_decision_register.loc[wave_4_housing_mask, 'Reference-period assessment'] = 'Wave 4 household tenure'
working_variable_decision_register.loc[wave_4_housing_mask, 'Review notes'] = 'Not used in predictor construction'
housing_tenure_decision_rows = working_variable_decision_register.loc[housing_review_mask, ['Wave', 'Source type',
    'Source file', 'Variable position', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment']].sort_values(['Wave',
    'Source file', 'Variable position']).reset_index(drop=True)
assert len(housing_tenure_decision_rows) == 4
assert housing_tenure_decision_rows[['Source file', 'Variable']].drop_duplicates().shape[0] == 4
assert housing_tenure_decision_rows['Review outcome'].notna().all()
assert housing_tenure_decision_rows['Decision reason'].notna().all()
assert housing_tenure_decision_rows['Leakage assessment'].notna().all()
housing_tenure_review_path = stage_2_output_directory / 'stage_2_housing_tenure_review.csv'
housing_tenure_review.to_csv(housing_tenure_review_path, index=False)
working_variable_decision_register.to_csv(decision_register_path, index=False)
variable_decision_register = working_variable_decision_register.copy()
housing_tenure_source_summary = housing_tenure_review['housing_tenure_source'].value_counts(dropna=False).rename_axis('Selected source').reset_index(name='Participants')
housing_tenure_decision_summary = housing_tenure_decision_rows['Review outcome'].value_counts().rename_axis('Review outcome').reset_index(name='Variables')
housing_tenure_distribution = housing_tenure_review[['housing_tenure',
    'housing_tenure_label']].value_counts(dropna=False).rename('Participants').reset_index().sort_values('housing_tenure',
    na_position='last').reset_index(drop=True)
print(f'Valid housing-tenure predictors: {housing_tenure.notna().sum():,}')
print(f'Unavailable housing-tenure predictors: {housing_tenure.isna().sum():,}')
print(f'Review file: {housing_tenure_review_path}')
display_limited(housing_tenure_source_summary)
display_limited(housing_tenure_distribution)
display_limited(housing_tenure_decision_summary)
print('Source-variable decisions:')
display_limited(housing_tenure_decision_rows)

Valid housing-tenure predictors: 9,521
Unavailable housing-tenure predictors: 246
Review file: data_derived\stage_2_predictor_construction\stage_2_housing_tenure_review.csv


,Selected source,Participants
0,Wave 3 before September 2006,9327
1,Unavailable,246
2,Wave 2 fallback,184
3,Wave 1 fallback,10


,housing_tenure,housing_tenure_label,Participants
0,1.0,Owned outright,1401
1,2.0,Being bought on a mortgage or bank loan,5748
2,3.0,Shared ownership,31
3,4.0,Rented from a Council or New Town,1169
4,5.0,Rented from a Housing Association,703


,Review outcome,Variables
0,Retain as construction source,3
1,Exclude from predictor set,1


Source-variable decisions:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Review outcome,Decision reason,Leakage assessment
0,Wave 1,Family background,wave_one_lsype_family_background_2020,13,W1hous12HH,HH: Tenure,Retain as construction source,Second fallback source where eligible Wave 3 a...,No leakage: measured before the post-16 transi...
1,Wave 2,Family background,wave_two_lsype_family_background_2020,12,W2Hous12HH,HH: Tenure,Retain as construction source,Fallback source where eligible Wave 3 housing ...,No leakage: measured before the post-16 transi...
2,Wave 3,Family background,wave_three_lsype_family_background_2020,18,W3hous12HH,HH: Tenure,Retain as construction source,Most recent housing-tenure measure recorded be...,No leakage after restricting the measure to in...
3,Wave 4,Family background,wave_four_lsype_family_background_2020,19,W4Hous12HH,HH: Tenure,Exclude from predictor set,Earlier waves provide the selected housing-ten...,Timing concern: Wave 4 may contain information...


In [220]:
# 34: Family-structure candidate variables

import pandas as pd
family_background_rows = variable_decision_register['Source type'].astype(str).eq('Family background')
family_structure_name_match = variable_decision_register['Variable'].astype(str).str.contains('famtyp|familytyp|hhcomp|housecomp|lonepar|singlepar|twopar|numpar|parpres|parentspres|secpar|secondpar',
    case=False, na=False, regex=True)
family_structure_label_match = variable_decision_register['Variable label'].astype(str).str.contains('\\bfamily type\\b|\\bhousehold composition\\b|\\bfamily structure\\b|\\blone[- ]parent\\b|\\bsingle[- ]parent\\b|\\btwo[- ]parent\\b|\\bone[- ]parent\\b|\\bnumber of parents\\b|\\bparents? (?:living|present) in (?:the )?household\\b|\\bsecond parent (?:living|present) in (?:the )?household\\b|\\bmain parent (?:has|with) (?:a )?partner\\b|\\bparental composition\\b',
    case=False, na=False, regex=True)
family_structure_candidates = variable_decision_register.loc[family_background_rows & (family_structure_name_match | family_structure_label_match),
    ['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Timing status',
    'Review status', 'Review outcome', 'Decision reason']].copy()
family_structure_candidates['Name match'] = family_structure_name_match.loc[family_structure_candidates.index].to_numpy()
family_structure_candidates['Label match'] = family_structure_label_match.loc[family_structure_candidates.index].to_numpy()
family_structure_candidates = family_structure_candidates.sort_values(['Wave', 'Source file',
    'Variable position']).reset_index(drop=True)
print(f'Family-structure candidate variables found: {len(family_structure_candidates):,}')
display_limited(family_structure_candidates)

Family-structure candidate variables found: 27


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Decision reason,Name match,Label match
0,Wave 1,Family background,wave_one_lsype_family_background_2020,310,W1famtyp,DV: Family composition,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,True,False
1,Wave 1,Family background,wave_one_lsype_family_background_2020,311,W1famtyp2,DV: Whether single parent household (using hou...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,True,True
2,Wave 1,Family background,wave_one_lsype_family_background_2020,312,W1singlepar,DV: Whether single parent household (using cur...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,True,True
3,Wave 2,Family background,wave_two_lsype_family_background_2020,238,W2Ben5QMP0h,MP: Whether MP received this in last 6 months ...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,True
4,Wave 2,Family background,wave_two_lsype_family_background_2020,267,W2Ben1AmtMP0y,MP: How much MP received on last occasion - Lo...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,True


In [221]:
# 35: Direct family-structure variable check

import pandas as pd
direct_family_structure_variables = ['W1famtyp', 'W1famtyp2', 'W1singlepar', 'W2famtyp', 'W3parpresHH', 'W3famtyp',
    'W3lnpar', 'W4ParPresHH', 'w4famtyp', 'w4lnpar']
direct_family_structure_candidates = family_structure_candidates.loc[family_structure_candidates['Variable'].isin(direct_family_structure_variables)].copy().sort_values(['Wave',
    'Source file', 'Variable position']).reset_index(drop=True)
expected_variables = set(direct_family_structure_variables)
found_variables = set(direct_family_structure_candidates['Variable'])
missing_variables = sorted(expected_variables - found_variables)
unexpected_variables = sorted(found_variables - expected_variables)
duplicate_variables = direct_family_structure_candidates.loc[direct_family_structure_candidates['Variable'].duplicated(keep=False)].sort_values('Variable').reset_index(drop=True)
print(f'Expected direct family-structure variables: {len(expected_variables):,}')
print(f'Matched candidate rows: {len(direct_family_structure_candidates):,}')
print(f"Matched unique variables: {direct_family_structure_candidates['Variable'].nunique():,}")
print('Missing variables:')
display_limited(pd.DataFrame({'Variable': missing_variables}))
print('Unexpected variables:')
display_limited(pd.DataFrame({'Variable': unexpected_variables}))
print('Duplicate matched variables:')
display_limited(duplicate_variables)
print('Confirmed direct family-structure candidates:')
display_limited(direct_family_structure_candidates)
assert len(direct_family_structure_candidates) == 10
assert direct_family_structure_candidates['Variable'].nunique() == 10
assert not missing_variables
assert not unexpected_variables

Expected direct family-structure variables: 10
Matched candidate rows: 10
Matched unique variables: 10
Missing variables:


,Variable


Unexpected variables:


,Variable


Duplicate matched variables:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Decision reason,Name match,Label match


Confirmed direct family-structure candidates:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Decision reason,Name match,Label match
0,Wave 1,Family background,wave_one_lsype_family_background_2020,310,W1famtyp,DV: Family composition,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,True,False
1,Wave 1,Family background,wave_one_lsype_family_background_2020,311,W1famtyp2,DV: Whether single parent household (using hou...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,True,True
2,Wave 1,Family background,wave_one_lsype_family_background_2020,312,W1singlepar,DV: Whether single parent household (using cur...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,True,True
3,Wave 2,Family background,wave_two_lsype_family_background_2020,791,W2famtyp,DV: Family composition,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,True,False
4,Wave 3,Family background,wave_three_lsype_family_background_2020,13,W3parpresHH,HH: Whether parent or guardian living in house...,Near-transition source,Use requires interview timing confirming Janua...,Pending review,<NA>,True,False


In [222]:
# 36: Family-structure measure structure

import pandas as pd
family_structure_by_variable = {}
family_structure_code_label_rows = []
family_structure_coverage_rows = []
family_structure_negative_code_rows = []
family_structure_distribution_rows = []
for source_file, candidate_group in direct_family_structure_candidates.groupby('Source file', sort=False):
    source_path = source_file_lookup[source_file]
    variable_names = candidate_group['Variable'].tolist()
    raw_data = pd.read_stata(source_path, columns=['NSID'] + variable_names, convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=variable_names, convert_categoricals=True)
    raw_data['NSID'] = raw_data['NSID'].astype('string').str.strip().str.replace('\\.0$', '', regex=True)
    aligned_data = raw_data.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)
    for _, candidate in candidate_group.iterrows():
        wave = candidate['Wave']
        variable = candidate['Variable']
        variable_label = candidate['Variable label']
        values = pd.to_numeric(aligned_data[variable], errors='coerce')
        family_structure_by_variable[variable] = values
        code_label_map = pd.DataFrame({'Code': pd.to_numeric(raw_data[variable], errors='coerce'),
            'Label': labelled_data[variable].astype('string')}).dropna(subset=['Code']).drop_duplicates().sort_values('Code').reset_index(drop=True)
        code_label_map.insert(0, 'Wave', wave)
        code_label_map.insert(1, 'Variable', variable)
        family_structure_code_label_rows.append(code_label_map)
        valid_values = values.loc[values.ge(0)]
        negative_values = values.loc[values.lt(0)]
        family_structure_coverage_rows.append({'Wave': wave, 'Variable': variable, 'Variable label': variable_label,
            'Valid responses': int(valid_values.notna().sum()), 'Negative codes': int(negative_values.notna().sum()), 'No source record': int(values.isna().sum()), 'Distinct valid codes': int(valid_values.nunique()), 'Minimum valid code': valid_values.min() if valid_values.notna().any() else pd.NA, 'Maximum valid code': valid_values.max() if valid_values.notna().any() else pd.NA})
        for code, count in negative_values.value_counts().sort_index().items():
            family_structure_negative_code_rows.append({'Wave': wave, 'Variable': variable, 'Negative code': code,
                'Participants': int(count)})
        variable_label_map = dict(zip(code_label_map['Code'], code_label_map['Label']))
        variable_distribution = pd.DataFrame({'Code': values,
            'Label': values.map(variable_label_map)}).value_counts(['Code', 'Label'],
            dropna=False).rename('Participants').reset_index().sort_values('Code',
            na_position='last').reset_index(drop=True)
        variable_distribution.insert(0, 'Wave', wave)
        variable_distribution.insert(1, 'Variable', variable)
        family_structure_distribution_rows.append(variable_distribution)
family_structure_code_labels = pd.concat(family_structure_code_label_rows, ignore_index=True)
family_structure_coverage = pd.DataFrame(family_structure_coverage_rows)
family_structure_negative_codes = pd.DataFrame(family_structure_negative_code_rows)
family_structure_distributions = pd.concat(family_structure_distribution_rows, ignore_index=True)
assert family_structure_coverage['Variable'].nunique() == 10
print('Stored family-structure codes and labels:')
display_limited(family_structure_code_labels)
print('Coverage by family-structure variable:')
display_limited(family_structure_coverage)
print('Observed negative codes:')
display_limited(family_structure_negative_codes)
print('Participant distributions:')
display_limited(family_structure_distributions)

Stored family-structure codes and labels:


,Wave,Variable,Code,Label
0,Wave 1,W1famtyp,-999,Missing - household data lost
1,Wave 1,W1famtyp,-94,Insufficient information
2,Wave 1,W1famtyp,1,Married couple
3,Wave 1,W1famtyp,2,Cohabiting couple
4,Wave 1,W1famtyp,3,Lone father


Coverage by family-structure variable:


,Wave,Variable,Variable label,Valid responses,Negative codes,No source record,Distinct valid codes,Minimum valid code,Maximum valid code
0,Wave 1,W1famtyp,DV: Family composition,9473,51,243,5,1.0,5.0
1,Wave 1,W1famtyp2,DV: Whether single parent household (using hou...,9486,38,243,2,0.0,1.0
2,Wave 1,W1singlepar,DV: Whether single parent household (using cur...,9481,43,243,2,0.0,1.0
3,Wave 2,W2famtyp,DV: Family composition,9359,162,246,5,1.0,5.0
4,Wave 3,W3parpresHH,HH: Whether parent or guardian living in house...,9503,6,258,2,1.0,2.0


Observed negative codes:


,Wave,Variable,Negative code,Participants
0,Wave 1,W1famtyp,-999.0,37
1,Wave 1,W1famtyp,-94.0,14
2,Wave 1,W1famtyp2,-999.0,37
3,Wave 1,W1famtyp2,-94.0,1
4,Wave 1,W1singlepar,-999.0,37


Participant distributions:


,Wave,Variable,Code,Label,Participants
0,Wave 1,W1famtyp,-999.0,Missing - household data lost,37
1,Wave 1,W1famtyp,-94.0,Insufficient information,14
2,Wave 1,W1famtyp,1.0,Married couple,6771
3,Wave 1,W1famtyp,2.0,Cohabiting couple,654
4,Wave 1,W1famtyp,3.0,Lone father,155


In [223]:
# 37: Family-structure derivation consistency

import pandas as pd
w1_family_type = family_structure_by_variable['W1famtyp']
w1_family_type_2 = family_structure_by_variable['W1famtyp2']
w1_single_parent = family_structure_by_variable['W1singlepar']
w3_parent_present = family_structure_by_variable['W3parpresHH']
w3_family_type = family_structure_by_variable['W3famtyp']
w3_lone_parent = family_structure_by_variable['W3lnpar']
w4_parent_present = family_structure_by_variable['W4ParPresHH']
w4_family_type = family_structure_by_variable['w4famtyp']
w4_lone_parent = family_structure_by_variable['w4lnpar']
w1_expected_single_parent = pd.Series(pd.NA, index=w1_family_type.index, dtype='Float64')
w1_expected_single_parent.loc[w1_family_type.isin([1, 2, 5])] = 0
w1_expected_single_parent.loc[w1_family_type.isin([3, 4])] = 1
w3_expected_lone_parent = pd.Series(pd.NA, index=w3_family_type.index, dtype='Float64')
w3_expected_lone_parent.loc[w3_family_type.isin([3, 4])] = 1
w3_expected_lone_parent.loc[w3_family_type.isin([1, 2, 5])] = 2
w4_expected_lone_parent = pd.Series(pd.NA, index=w4_family_type.index, dtype='Float64')
w4_expected_lone_parent.loc[w4_family_type.isin([3, 4])] = 1
w4_expected_lone_parent.loc[w4_family_type.isin([1, 2, 5])] = 2
w3_expected_parent_present = pd.Series(pd.NA, index=w3_family_type.index, dtype='Float64')
w3_expected_parent_present.loc[w3_family_type.isin([1, 2, 3, 4])] = 1
w3_expected_parent_present.loc[w3_family_type.eq(5)] = 2
w4_expected_parent_present = pd.Series(pd.NA, index=w4_family_type.index, dtype='Float64')
w4_expected_parent_present.loc[w4_family_type.isin([1, 2, 3, 4])] = 1
w4_expected_parent_present.loc[w4_family_type.eq(5)] = 2

def compare_family_structure_measures(comparison_name, expected_measure, observed_measure):
    complete_cases = expected_measure.notna() & observed_measure.notna()
    exact_matches = expected_measure.loc[complete_cases].eq(observed_measure.loc[complete_cases])
    summary = {'Comparison': comparison_name, 'Complete cases': int(complete_cases.sum()),
        'Exact agreement': int(exact_matches.sum()), 'Different values': int((~exact_matches).sum()), 'Agreement percentage': round(exact_matches.mean() * 100,
        2)}
    difference_pairs = pd.DataFrame({'Expected value': expected_measure.loc[complete_cases & ~expected_measure.eq(observed_measure)],
        'Observed value': observed_measure.loc[complete_cases & ~expected_measure.eq(observed_measure)]}).value_counts(dropna=False).rename('Participants').reset_index()
    if not difference_pairs.empty:
        difference_pairs.insert(0, 'Comparison', comparison_name)
    return (summary, difference_pairs)
comparison_specifications = [('Wave 1 family type with W1famtyp2', w1_expected_single_parent,
    w1_family_type_2.where(w1_family_type_2.isin([0, 1]))), ('Wave 1 family type with W1singlepar',
    w1_expected_single_parent, w1_single_parent.where(w1_single_parent.isin([0,
    1]))), ('Wave 1 W1famtyp2 with W1singlepar', w1_family_type_2.where(w1_family_type_2.isin([0, 1])),
    w1_single_parent.where(w1_single_parent.isin([0, 1]))), ('Wave 3 family type with lone-parent indicator',
    w3_expected_lone_parent, w3_lone_parent.where(w3_lone_parent.isin([1,
    2]))), ('Wave 3 family type with parent-presence indicator', w3_expected_parent_present,
    w3_parent_present.where(w3_parent_present.isin([1, 2]))), ('Wave 4 family type with lone-parent indicator',
    w4_expected_lone_parent, w4_lone_parent.where(w4_lone_parent.isin([1,
    2]))), ('Wave 4 family type with parent-presence indicator', w4_expected_parent_present,
    w4_parent_present.where(w4_parent_present.isin([1, 2])))]
family_structure_consistency_rows = []
family_structure_difference_tables = []
for comparison_name, expected_measure, observed_measure in comparison_specifications:
    summary, difference_pairs = compare_family_structure_measures(comparison_name, expected_measure, observed_measure)
    family_structure_consistency_rows.append(summary)
    if not difference_pairs.empty:
        family_structure_difference_tables.append(difference_pairs)
family_structure_consistency = pd.DataFrame(family_structure_consistency_rows)
if family_structure_difference_tables:
    family_structure_differences = pd.concat(family_structure_difference_tables, ignore_index=True)
else:
    family_structure_differences = pd.DataFrame(columns=['Comparison', 'Expected value', 'Observed value',
        'Participants'])
print('Family-structure derivation consistency:')
display_limited(family_structure_consistency)
print('Observed differences:')
display_limited(family_structure_differences)

Family-structure derivation consistency:


,Comparison,Complete cases,Exact agreement,Different values,Agreement percentage
0,Wave 1 family type with W1famtyp2,9473,9473,0,100.00
1,Wave 1 family type with W1singlepar,9467,9376,91,99.04
2,Wave 1 W1famtyp2 with W1singlepar,9480,9389,91,99.04
3,Wave 3 family type with lone-parent indicator,9338,9338,0,100.00
4,Wave 3 family type with parent-presence indicator,9338,9256,82,99.12


Observed differences:


,Comparison,Expected value,Observed value,Participants
0,Wave 1 family type with W1singlepar,1.0,0.0,51
1,Wave 1 family type with W1singlepar,0.0,1.0,40
2,Wave 1 W1famtyp2 with W1singlepar,1.0,0.0,51
3,Wave 1 W1famtyp2 with W1singlepar,0.0,1.0,40
4,Wave 3 family type with parent-presence indicator,2.0,1.0,76


In [224]:
# 38: Cross-wave family-composition comparison

import pandas as pd
w1_family_composition = family_structure_by_variable['W1famtyp'].where(lambda values: values.between(1,
    5)).astype('Float64')
w2_family_composition = family_structure_by_variable['W2famtyp'].where(lambda values: values.between(1,
    5)).astype('Float64')
w3_family_composition = family_structure_by_variable['W3famtyp'].where(lambda values: values.between(1,
    5)).astype('Float64')
w3_family_composition_before_transition = w3_family_composition.where(w3_income_timing_status.eq('Before September 2006'))
family_composition_comparison_rows = []
for first_name, first_measure, second_name, second_measure in [('Wave 1', w1_family_composition, 'Wave 2',
    w2_family_composition), ('Wave 1', w1_family_composition, 'Wave 3', w3_family_composition), ('Wave 2',
    w2_family_composition, 'Wave 3', w3_family_composition)]:
    complete_cases = first_measure.notna() & second_measure.notna()
    exact_agreement = first_measure.loc[complete_cases].eq(second_measure.loc[complete_cases])
    family_composition_comparison_rows.append({'Comparison': f'{first_name} with {second_name}',
        'Complete cases': int(complete_cases.sum()), 'Exact agreement': int(exact_agreement.sum()), 'Agreement percentage': round(exact_agreement.mean() * 100,
        2), "Cramér's V": round(cramers_v(first_measure, second_measure), 4)})
family_composition_cross_wave_comparison = pd.DataFrame(family_composition_comparison_rows)
family_composition_candidate = w3_family_composition_before_transition.combine_first(w2_family_composition).combine_first(w1_family_composition)
family_composition_candidate_source = pd.Series('Unavailable', index=stage_2_ids, dtype='string')
family_composition_candidate_source.loc[w3_family_composition_before_transition.notna()] = 'Wave 3 before September 2006'
family_composition_candidate_source.loc[w3_family_composition_before_transition.isna() & w2_family_composition.notna()] = 'Wave 2 fallback'
family_composition_candidate_source.loc[w3_family_composition_before_transition.isna() & w2_family_composition.isna() & w1_family_composition.notna()] = 'Wave 1 fallback'
family_composition_coverage_summary = pd.DataFrame({'Measure or condition': ['Wave 1 valid family composition',
    'Wave 2 valid family composition', 'Wave 3 valid family composition', 'Wave 3 valid before September 2006', 'Wave 3 valid in September 2006', 'Wave 3 valid with interview date unavailable', 'Eligible Wave 3 with Wave 2 and Wave 1 fallback', 'Unavailable after all eligible sources'], 'Participants': [int(w1_family_composition.notna().sum()),
    int(w2_family_composition.notna().sum()), int(w3_family_composition.notna().sum()), int(w3_family_composition_before_transition.notna().sum()), int((w3_family_composition.notna() & w3_income_timing_status.eq('September 2006')).sum()), int((w3_family_composition.notna() & w3_income_timing_status.eq('Interview date unavailable')).sum()), int(family_composition_candidate.notna().sum()), int(family_composition_candidate.isna().sum())]})
family_composition_source_summary = family_composition_candidate_source.value_counts(dropna=False).rename_axis('Candidate source').reset_index(name='Participants')
wave_2_wave_3_transition = pd.crosstab(w2_family_composition, w3_family_composition,
    rownames=['Wave 2 family composition'], colnames=['Wave 3 family composition'], margins=True)
print('Cross-wave family-composition agreement:')
display_limited(family_composition_cross_wave_comparison)
print('Family-composition timing and coverage:')
display_limited(family_composition_coverage_summary)
print('Candidate source contribution:')
display_limited(family_composition_source_summary)
print('Wave 2 to Wave 3 family-composition table:')
display_limited(wave_2_wave_3_transition)

Cross-wave family-composition agreement:


,Comparison,Complete cases,Exact agreement,Agreement percentage,Cramér's V
0,Wave 1 with Wave 2,9320,9178,98.48,0.9391
1,Wave 1 with Wave 3,9297,8956,96.33,0.8801
2,Wave 2 with Wave 3,9320,9093,97.56,0.9234


Family-composition timing and coverage:


,Measure or condition,Participants
0,Wave 1 valid family composition,9473
1,Wave 2 valid family composition,9359
2,Wave 3 valid family composition,9338
3,Wave 3 valid before September 2006,9200
4,Wave 3 valid in September 2006,14


Candidate source contribution:


,Candidate source,Participants
0,Wave 3 before September 2006,9200
1,Unavailable,252
2,Wave 2 fallback,177
3,Wave 1 fallback,138


Wave 2 to Wave 3 family-composition table:


Wave 3 family composition,1.0,2.0,3.0,4.0,5.0,All
Wave 2 family composition,,,,,,
1.0,6629,6,21,99,0,6755
2.0,13,584,3,30,0,630
3.0,4,2,128,1,2,137
4.0,20,16,0,1679,2,1717
5.0,5,1,1,1,73,81


In [225]:
# 39: Family-composition construction and decisions

import pandas as pd
wave_3_family_composition_before_transition = w3_family_composition.where(w3_income_timing_status.eq('Before September 2006'))
family_composition = wave_3_family_composition_before_transition.combine_first(w2_family_composition).combine_first(w1_family_composition).astype('Int64')
family_composition_source = pd.Series('Unavailable', index=family_composition.index, dtype='string')
family_composition_source.loc[wave_3_family_composition_before_transition.notna()] = 'Wave 3 before September 2006'
family_composition_source.loc[wave_3_family_composition_before_transition.isna() & w2_family_composition.notna()] = 'Wave 2 fallback'
family_composition_source.loc[wave_3_family_composition_before_transition.isna() & w2_family_composition.isna() & w1_family_composition.notna()] = 'Wave 1 fallback'
family_composition_label_map = {1: 'Married couple', 2: 'Cohabiting couple', 3: 'Lone father', 4: 'Lone mother',
    5: 'No parents in the household'}
family_composition_label = family_composition.map(family_composition_label_map).astype('string')
family_composition_review = pd.DataFrame({'NSID': stage_2_ids.to_numpy(),
    'family_composition': family_composition.to_numpy(), 'family_composition_label': family_composition_label.to_numpy(), 'family_composition_source': family_composition_source.to_numpy()})
assert len(family_composition_review) == 9767
assert family_composition_review['NSID'].nunique() == 9767
assert family_composition_review['NSID'].duplicated().sum() == 0
assert family_composition_review['NSID'].isna().sum() == 0
assert family_composition_review['family_composition'].dropna().between(1, 5).all()
family_structure_keys = set(direct_family_structure_candidates[['Source file', 'Variable']].itertuples(index=False,
    name=None))
wave_1_source = direct_family_structure_candidates.loc[direct_family_structure_candidates['Wave'].eq('Wave 1'),
    'Source file'].iloc[0]
wave_2_source = direct_family_structure_candidates.loc[direct_family_structure_candidates['Wave'].eq('Wave 2'),
    'Source file'].iloc[0]
wave_3_source = direct_family_structure_candidates.loc[direct_family_structure_candidates['Wave'].eq('Wave 3'),
    'Source file'].iloc[0]
wave_4_source = direct_family_structure_candidates.loc[direct_family_structure_candidates['Wave'].eq('Wave 4'),
    'Source file'].iloc[0]
primary_family_structure_keys = {(wave_3_source, 'W3famtyp')}
fallback_family_structure_keys = {(wave_2_source, 'W2famtyp'), (wave_1_source, 'W1famtyp')}
family_structure_support_keys = {(wave_1_source, 'W1famtyp2'), (wave_1_source, 'W1singlepar'), (wave_3_source,
    'W3lnpar'), (wave_3_source, 'W3parpresHH')}
wave_4_family_structure_keys = {(wave_4_source, 'W4ParPresHH'), (wave_4_source, 'w4famtyp'), (wave_4_source,
    'w4lnpar')}

def family_structure_key_mask(register, selected_keys):
    row_keys = zip(register['Source file'].astype(str), register['Variable'].astype(str))
    return pd.Series([key in selected_keys for key in row_keys], index=register.index)
working_variable_decision_register = variable_decision_register.copy()
family_structure_review_mask = family_structure_key_mask(working_variable_decision_register, family_structure_keys)
primary_family_structure_mask = family_structure_key_mask(working_variable_decision_register,
    primary_family_structure_keys)
fallback_family_structure_mask = family_structure_key_mask(working_variable_decision_register,
    fallback_family_structure_keys)
family_structure_support_mask = family_structure_key_mask(working_variable_decision_register,
    family_structure_support_keys)
wave_4_family_structure_mask = family_structure_key_mask(working_variable_decision_register,
    wave_4_family_structure_keys)
working_variable_decision_register.loc[family_structure_review_mask, 'Review status'] = 'Reviewed'
working_variable_decision_register.loc[family_structure_review_mask,
    'Substantive domain'] = 'Family socioeconomic background'
working_variable_decision_register.loc[family_structure_review_mask,
    'Documentation source'] = 'Wave 1 to Wave 4 value labels and Wave 3 interview timing'
working_variable_decision_register.loc[primary_family_structure_mask,
    'Review outcome'] = 'Retain as construction source'
working_variable_decision_register.loc[primary_family_structure_mask,
    'Decision reason'] = 'Most recent five-category family-composition measure recorded before the post-16 transition'
working_variable_decision_register.loc[primary_family_structure_mask,
    'Leakage assessment'] = 'No leakage after restricting the measure to interviews before September 2006'
working_variable_decision_register.loc[primary_family_structure_mask,
    'Reference-period assessment'] = 'Wave 3 interview before September 2006'
working_variable_decision_register.loc[primary_family_structure_mask,
    'Review notes'] = 'Primary source; 9,200 participants contributed values'
working_variable_decision_register.loc[fallback_family_structure_mask,
    'Review outcome'] = 'Retain as construction source'
working_variable_decision_register.loc[fallback_family_structure_mask,
    'Decision reason'] = 'Fallback source where a later eligible family-composition value is unavailable'
working_variable_decision_register.loc[fallback_family_structure_mask,
    'Leakage assessment'] = 'No leakage: measured before the post-16 transition'
working_variable_decision_register.loc[fallback_family_structure_mask,
    'Reference-period assessment'] = 'Wave 1 or Wave 2 family composition'
working_variable_decision_register.loc[working_variable_decision_register['Variable'].eq('W2famtyp') & fallback_family_structure_mask,
    'Review notes'] = 'First fallback source; 177 participants contributed values'
working_variable_decision_register.loc[working_variable_decision_register['Variable'].eq('W1famtyp') & fallback_family_structure_mask,
    'Review notes'] = 'Second fallback source; 138 participants contributed values'
working_variable_decision_register.loc[family_structure_support_mask,
    'Review outcome'] = 'Retain as review support only'
working_variable_decision_register.loc[family_structure_support_mask,
    'Decision reason'] = 'Simplified, alternative or overlapping representation of the selected five-category family-composition construct'
working_variable_decision_register.loc[family_structure_support_mask,
    'Leakage assessment'] = 'No leakage; not retained in the predictor set'
working_variable_decision_register.loc[family_structure_support_mask,
    'Reference-period assessment'] = 'Wave 1 or eligible Wave 3 household structure'
working_variable_decision_register.loc[family_structure_support_mask,
    'Review notes'] = 'Used to assess binary lone-parent derivations, alternative definitions and parent-or-guardian presence'
working_variable_decision_register.loc[wave_4_family_structure_mask, 'Review outcome'] = 'Exclude from predictor set'
working_variable_decision_register.loc[wave_4_family_structure_mask,
    'Decision reason'] = 'Earlier waves provide the selected family-composition construct'
working_variable_decision_register.loc[wave_4_family_structure_mask,
    'Leakage assessment'] = 'Timing concern: Wave 4 may contain information measured at or after the outcome boundary'
working_variable_decision_register.loc[wave_4_family_structure_mask,
    'Reference-period assessment'] = 'Wave 4 household structure'
working_variable_decision_register.loc[wave_4_family_structure_mask,
    'Review notes'] = 'Not used in predictor construction'
family_structure_decision_rows = working_variable_decision_register.loc[family_structure_review_mask, ['Wave',
    'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment']].sort_values(['Wave',
    'Source file', 'Variable position']).reset_index(drop=True)
assert len(family_structure_decision_rows) == 10
assert family_structure_decision_rows[['Source file', 'Variable']].drop_duplicates().shape[0] == 10
assert family_structure_decision_rows['Review outcome'].notna().all()
assert family_structure_decision_rows['Decision reason'].notna().all()
assert family_structure_decision_rows['Leakage assessment'].notna().all()
family_composition_review_path = stage_2_output_directory / 'stage_2_family_composition_review.csv'
family_composition_review.to_csv(family_composition_review_path, index=False)
working_variable_decision_register.to_csv(decision_register_path, index=False)
variable_decision_register = working_variable_decision_register.copy()
family_composition_source_summary = family_composition_review['family_composition_source'].value_counts(dropna=False).rename_axis('Selected source').reset_index(name='Participants')
family_composition_distribution = family_composition_review[['family_composition',
    'family_composition_label']].value_counts(dropna=False).rename('Participants').reset_index().sort_values('family_composition',
    na_position='last').reset_index(drop=True)
family_structure_decision_summary = family_structure_decision_rows['Review outcome'].value_counts().rename_axis('Review outcome').reset_index(name='Variables')
print(f'Valid family-composition predictors: {family_composition.notna().sum():,}')
print(f'Unavailable family-composition predictors: {family_composition.isna().sum():,}')
print(f'Review file: {family_composition_review_path}')
display_limited(family_composition_source_summary)
display_limited(family_composition_distribution)
display_limited(family_structure_decision_summary)
print('Source-variable decisions:')
display_limited(family_structure_decision_rows)

Valid family-composition predictors: 9,515
Unavailable family-composition predictors: 252
Review file: data_derived\stage_2_predictor_construction\stage_2_family_composition_review.csv


,Selected source,Participants
0,Wave 3 before September 2006,9200
1,Unavailable,252
2,Wave 2 fallback,177
3,Wave 1 fallback,138


,family_composition,family_composition_label,Participants
0,1.0,Married couple,6762
1,2.0,Cohabiting couple,629
2,3.0,Lone father,165
3,4.0,Lone mother,1874
4,5.0,No parents in the household,85


,Review outcome,Variables
0,Retain as review support only,4
1,Retain as construction source,3
2,Exclude from predictor set,3


Source-variable decisions:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Review outcome,Decision reason,Leakage assessment
0,Wave 1,Family background,wave_one_lsype_family_background_2020,310,W1famtyp,DV: Family composition,Retain as construction source,Fallback source where a later eligible family-...,No leakage: measured before the post-16 transi...
1,Wave 1,Family background,wave_one_lsype_family_background_2020,311,W1famtyp2,DV: Whether single parent household (using hou...,Retain as review support only,"Simplified, alternative or overlapping represe...",No leakage; not retained in the predictor set
2,Wave 1,Family background,wave_one_lsype_family_background_2020,312,W1singlepar,DV: Whether single parent household (using cur...,Retain as review support only,"Simplified, alternative or overlapping represe...",No leakage; not retained in the predictor set
3,Wave 2,Family background,wave_two_lsype_family_background_2020,791,W2famtyp,DV: Family composition,Retain as construction source,Fallback source where a later eligible family-...,No leakage: measured before the post-16 transi...
4,Wave 3,Family background,wave_three_lsype_family_background_2020,13,W3parpresHH,HH: Whether parent or guardian living in house...,Retain as review support only,"Simplified, alternative or overlapping represe...",No leakage; not retained in the predictor set


In [226]:
# 40: Parental-employment candidate variables

import pandas as pd
family_background_rows = variable_decision_register['Source type'].astype(str).eq('Family background')
parental_employment_name_match = variable_decision_register['Variable'].astype(str).str.contains('workless|wrkless|famwork|workfam|parwork|paremp|empfam|econact|ecact|workstat',
    case=False, na=False, regex=True)
parental_employment_label_match = variable_decision_register['Variable label'].astype(str).str.contains("\\bworkless household\\b|\\bhousehold worklessness\\b|\\bfamily employment status\\b|\\bparental employment status\\b|\\bat least one parent (?:is )?(?:working|in work|employed)\\b|\\bno parents? (?:is|are)? ?(?:working|in work|employed)\\b|\\bnumber of parents? (?:working|in work|employed)\\b|\\bmain parent(?:'s)? (?:current )?(?:employment status|economic activity)\\b|\\bsecond parent(?:'s)? (?:current )?(?:employment status|economic activity)\\b|\\bMP(?:'s)? (?:current )?(?:employment status|economic activity)\\b|\\bSP(?:'s)? (?:current )?(?:employment status|economic activity)\\b|\\bwhether (?:MP|SP|main parent|second parent) (?:is )?(?:working|in work|employed)\\b",
    case=False, na=False, regex=True)
parental_employment_candidates = variable_decision_register.loc[family_background_rows & (parental_employment_name_match | parental_employment_label_match),
    ['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Timing status',
    'Review status', 'Review outcome', 'Decision reason']].copy()
parental_employment_candidates['Name match'] = parental_employment_name_match.loc[parental_employment_candidates.index].to_numpy()
parental_employment_candidates['Label match'] = parental_employment_label_match.loc[parental_employment_candidates.index].to_numpy()
parental_employment_candidates = parental_employment_candidates.sort_values(['Wave', 'Source file',
    'Variable position']).reset_index(drop=True)
print(f'Parental-employment candidate variables found: {len(parental_employment_candidates):,}')
display_limited(parental_employment_candidates)

Parental-employment candidate variables found: 27


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Decision reason,Name match,Label match
0,Wave 1,Family background,wave_one_lsype_family_background_2020,353,W1wrkcurMP,DV: Whether main parent is working or not,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,True
1,Wave 1,Family background,wave_one_lsype_family_background_2020,354,W1wrkcurSP,DV: Whether second parent is working or not,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,True
2,Wave 2,Family background,wave_two_lsype_family_background_2020,400,W2wrkstatusSP,SP: SP's current employment status,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,True
3,Wave 2,Family background,wave_two_lsype_family_background_2020,808,W2wrkstatUpdMP,DV: Whether MP employed or self-employed in la...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,True
4,Wave 2,Family background,wave_two_lsype_family_background_2020,811,W2wrk12aUpdMP,DV: Whether MP working on own or has employees...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,True


In [227]:
# 41: Parental-employment variable-family expansion

import pandas as pd
family_background_rows = variable_decision_register['Source type'].astype(str).eq('Family background')
parental_employment_family_match = variable_decision_register['Variable'].astype(str).str.contains('wrkcur|wrkstat|wrk1a|cgemps|selfemp|wrk12a|wrky|wrkm',
    case=False, na=False, regex=True)
expanded_parental_employment_candidates = variable_decision_register.loc[family_background_rows & parental_employment_family_match,
    ['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Timing status',
    'Review status', 'Review outcome', 'Decision reason']].copy().sort_values(['Wave', 'Source file',
    'Variable position']).reset_index(drop=True)
initial_parental_employment_keys = set(parental_employment_candidates[['Source file',
    'Variable']].itertuples(index=False, name=None))
expanded_parental_employment_candidates['Found in initial search'] = [(source_file,
    variable) in initial_parental_employment_keys for source_file, variable in zip(expanded_parental_employment_candidates['Source file'],
    expanded_parental_employment_candidates['Variable'])]
new_parental_employment_candidates = expanded_parental_employment_candidates.loc[~expanded_parental_employment_candidates['Found in initial search']].copy().reset_index(drop=True)

def parental_employment_measure_family(variable_name):
    variable_name_lower = str(variable_name).lower()
    if 'wrkcur' in variable_name_lower:
        return 'Current working-status summary'
    if 'wrkstat' in variable_name_lower:
        return 'Employment-status measure'
    if 'wrk1a' in variable_name_lower:
        return 'Economic-activity measure'
    if 'cgemps' in variable_name_lower:
        return 'Employment-status change'
    if 'wrky' in variable_name_lower or 'wrkm' in variable_name_lower:
        return 'Economic-activity period date'
    if 'selfemp' in variable_name_lower or 'wrk12a' in variable_name_lower:
        return 'Employment or self-employment detail'
    return 'Other employment-related measure'
expanded_parental_employment_candidates['Measure family'] = expanded_parental_employment_candidates['Variable'].map(parental_employment_measure_family)
parental_employment_family_summary = expanded_parental_employment_candidates['Measure family'].value_counts().rename_axis('Measure family').reset_index(name='Variables')
print(f'Initial parental-employment candidates: {len(parental_employment_candidates):,}')
print(f'Expanded parental-employment candidates: {len(expanded_parental_employment_candidates):,}')
print(f'New variables identified by the expanded search: {len(new_parental_employment_candidates):,}')
print('Newly identified variables:')
display_limited(new_parental_employment_candidates)
print('Expanded candidate set:')
display_limited(expanded_parental_employment_candidates)
print('Candidate measure families:')
display_limited(parental_employment_family_summary)

Initial parental-employment candidates: 27
Expanded parental-employment candidates: 70
New variables identified by the expanded search: 43
Newly identified variables:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Decision reason,Found in initial search
0,Wave 1,Family background,wave_one_lsype_family_background_2020,140,W1wrk1aMP,MP: Current working status,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False
1,Wave 1,Family background,wave_one_lsype_family_background_2020,141,W1WrkYMP,MP: Year started current activity,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False
2,Wave 1,Family background,wave_one_lsype_family_background_2020,142,W1wrkmMP,MP: Month of starting current activity,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False
3,Wave 1,Family background,wave_one_lsype_family_background_2020,146,W1wrkStatMP,MP: Whether current-last job employed or self-...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False
4,Wave 1,Family background,wave_one_lsype_family_background_2020,159,W1wrk12aMP,MP: Whether work alone or have employees,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False


Expanded candidate set:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Decision reason,Found in initial search,Measure family
0,Wave 1,Family background,wave_one_lsype_family_background_2020,140,W1wrk1aMP,MP: Current working status,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,Economic-activity measure
1,Wave 1,Family background,wave_one_lsype_family_background_2020,141,W1WrkYMP,MP: Year started current activity,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,Economic-activity period date
2,Wave 1,Family background,wave_one_lsype_family_background_2020,142,W1wrkmMP,MP: Month of starting current activity,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,Economic-activity period date
3,Wave 1,Family background,wave_one_lsype_family_background_2020,146,W1wrkStatMP,MP: Whether current-last job employed or self-...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,Employment-status measure
4,Wave 1,Family background,wave_one_lsype_family_background_2020,159,W1wrk12aMP,MP: Whether work alone or have employees,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,Employment or self-employment detail


Candidate measure families:


,Measure family,Variables
0,Economic-activity period date,16
1,Current working-status summary,16
2,Employment or self-employment detail,12
3,Employment-status change,12
4,Employment-status measure,8


In [228]:
# 42: Current parental working-status measures

import pandas as pd
current_parental_work_candidates = expanded_parental_employment_candidates.loc[expanded_parental_employment_candidates['Measure family'].eq('Current working-status summary')].copy().sort_values(['Wave',
    'Source file', 'Variable position']).reset_index(drop=True)
assert len(current_parental_work_candidates) == 16
assert current_parental_work_candidates['Variable'].nunique() == 16

def parental_work_role(variable_name):
    variable_name_lower = str(variable_name).lower()
    if 'wrkcurmp' in variable_name_lower:
        return 'Main parent'
    if 'wrkcursp' in variable_name_lower:
        return 'Second parent'
    if 'wrkcurdad' in variable_name_lower:
        return 'Father'
    if 'wrkcurmum' in variable_name_lower:
        return 'Mother'
    return 'Unclassified'
current_parental_work_candidates['Parental role'] = current_parental_work_candidates['Variable'].map(parental_work_role)
parental_work_by_variable = {}
parental_work_code_label_rows = []
parental_work_coverage_rows = []
parental_work_negative_code_rows = []
parental_work_distribution_rows = []
for source_file, candidate_group in current_parental_work_candidates.groupby('Source file', sort=False):
    source_path = source_file_lookup[source_file]
    variable_names = candidate_group['Variable'].tolist()
    raw_data = pd.read_stata(source_path, columns=['NSID'] + variable_names, convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=variable_names, convert_categoricals=True)
    raw_data['NSID'] = raw_data['NSID'].astype('string').str.strip().str.replace('\\.0$', '', regex=True)
    aligned_data = raw_data.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)
    for _, candidate in candidate_group.iterrows():
        wave = candidate['Wave']
        variable = candidate['Variable']
        variable_label = candidate['Variable label']
        parental_role = candidate['Parental role']
        values = pd.to_numeric(aligned_data[variable], errors='coerce')
        parental_work_by_variable[variable] = values
        code_label_map = pd.DataFrame({'Code': pd.to_numeric(raw_data[variable], errors='coerce'),
            'Label': labelled_data[variable].astype('string')}).dropna(subset=['Code']).drop_duplicates().sort_values('Code').reset_index(drop=True)
        code_label_map.insert(0, 'Wave', wave)
        code_label_map.insert(1, 'Variable', variable)
        code_label_map.insert(2, 'Parental role', parental_role)
        parental_work_code_label_rows.append(code_label_map)
        valid_values = values.loc[values.ge(0)]
        negative_values = values.loc[values.lt(0)]
        parental_work_coverage_rows.append({'Wave': wave, 'Variable': variable, 'Parental role': parental_role,
            'Variable label': variable_label, 'Valid responses': int(valid_values.notna().sum()), 'Negative codes': int(negative_values.notna().sum()), 'No source record': int(values.isna().sum()), 'Distinct valid codes': int(valid_values.nunique()), 'Minimum valid code': valid_values.min() if valid_values.notna().any() else pd.NA, 'Maximum valid code': valid_values.max() if valid_values.notna().any() else pd.NA})
        for code, count in negative_values.value_counts().sort_index().items():
            parental_work_negative_code_rows.append({'Wave': wave, 'Variable': variable,
                'Parental role': parental_role, 'Negative code': code, 'Participants': int(count)})
        variable_label_map = dict(zip(code_label_map['Code'], code_label_map['Label']))
        variable_distribution = pd.DataFrame({'Code': values,
            'Label': values.map(variable_label_map)}).value_counts(['Code', 'Label'],
            dropna=False).rename('Participants').reset_index().sort_values('Code',
            na_position='last').reset_index(drop=True)
        variable_distribution.insert(0, 'Wave', wave)
        variable_distribution.insert(1, 'Variable', variable)
        variable_distribution.insert(2, 'Parental role', parental_role)
        parental_work_distribution_rows.append(variable_distribution)
parental_work_code_labels = pd.concat(parental_work_code_label_rows, ignore_index=True)
parental_work_coverage = pd.DataFrame(parental_work_coverage_rows)
parental_work_negative_codes = pd.DataFrame(parental_work_negative_code_rows)
parental_work_distributions = pd.concat(parental_work_distribution_rows, ignore_index=True)
assert parental_work_coverage['Variable'].nunique() == 16
print('Direct current parental working-status variables:')
display_limited(current_parental_work_candidates)
print('Stored codes and labels:')
display_limited(parental_work_code_labels)
print('Coverage by variable:')
display_limited(parental_work_coverage)
print('Observed negative codes:')
display_limited(parental_work_negative_codes)
print('Participant distributions:')
display_limited(parental_work_distributions)

Direct current parental working-status variables:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Decision reason,Found in initial search,Measure family,Parental role
0,Wave 1,Family background,wave_one_lsype_family_background_2020,353,W1wrkcurMP,DV: Whether main parent is working or not,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,True,Current working-status summary,Main parent
1,Wave 1,Family background,wave_one_lsype_family_background_2020,354,W1wrkcurSP,DV: Whether second parent is working or not,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,True,Current working-status summary,Second parent
2,Wave 1,Family background,wave_one_lsype_family_background_2020,355,W1wrkcurdad,DV: Whether father is working or not,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,Current working-status summary,Father
3,Wave 1,Family background,wave_one_lsype_family_background_2020,356,W1wrkcurmum,DV: Whether mother is working or not,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,Current working-status summary,Mother
4,Wave 2,Family background,wave_two_lsype_family_background_2020,856,W2wrkcurMP,DV: Whether main parent is working or not,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,True,Current working-status summary,Main parent


Stored codes and labels:


,Wave,Variable,Parental role,Code,Label
0,Wave 1,W1wrkcurMP,Main parent,-94,Insufficient information
1,Wave 1,W1wrkcurMP,Main parent,1,Currently working
2,Wave 1,W1wrkcurMP,Main parent,2,Currently not working
3,Wave 1,W1wrkcurSP,Second parent,-98,SP not present
4,Wave 1,W1wrkcurSP,Second parent,-94,Insufficient information


Coverage by variable:


,Wave,Variable,Parental role,Variable label,Valid responses,Negative codes,No source record,Distinct valid codes,Minimum valid code,Maximum valid code
0,Wave 1,W1wrkcurMP,Main parent,DV: Whether main parent is working or not,9512,12,243,2,1.0,2.0
1,Wave 1,W1wrkcurSP,Second parent,DV: Whether second parent is working or not,7362,2162,243,2,1.0,2.0
2,Wave 1,W1wrkcurdad,Father,DV: Whether father is working or not,7479,2045,243,2,1.0,2.0
3,Wave 1,W1wrkcurmum,Mother,DV: Whether mother is working or not,9214,310,243,2,1.0,2.0
4,Wave 2,W2wrkcurMP,Main parent,DV: Whether main parent is working or not,9474,47,246,2,1.0,2.0


Observed negative codes:


,Wave,Variable,Parental role,Negative code,Participants
0,Wave 1,W1wrkcurMP,Main parent,-94.0,12
1,Wave 1,W1wrkcurSP,Second parent,-98.0,2151
2,Wave 1,W1wrkcurSP,Second parent,-94.0,11
3,Wave 1,W1wrkcurdad,Father,-999.0,37
4,Wave 1,W1wrkcurdad,Father,-99.0,95


Participant distributions:


,Wave,Variable,Parental role,Code,Label,Participants
0,Wave 1,W1wrkcurMP,Main parent,-94.0,Insufficient information,12
1,Wave 1,W1wrkcurMP,Main parent,1.0,Currently working,6802
2,Wave 1,W1wrkcurMP,Main parent,2.0,Currently not working,2710
3,Wave 1,W1wrkcurMP,Main parent,NaN,NaN,243
4,Wave 1,W1wrkcurSP,Second parent,-98.0,SP not present,2151


In [229]:
# 43: Alternative household working-status constructions

import pandas as pd

def derive_working_role_count(first_measure, second_measure, first_absence_codes=None, second_absence_codes=None):
    first_absence_codes = first_absence_codes or []
    second_absence_codes = second_absence_codes or []
    first_valid = first_measure.isin([1, 2])
    second_valid = second_measure.isin([1, 2])
    first_absent = first_measure.isin(first_absence_codes)
    second_absent = second_measure.isin(second_absence_codes)
    first_resolved = first_valid | first_absent
    second_resolved = second_valid | second_absent
    resolved_pair = first_resolved & second_resolved
    working_count = pd.Series(pd.NA, index=first_measure.index, dtype='Int64')
    first_working = first_measure.eq(1).fillna(False).astype(int)
    second_working = second_measure.eq(1).fillna(False).astype(int)
    working_count.loc[resolved_pair] = (first_working.loc[resolved_pair] + second_working.loc[resolved_pair]).astype('Int64')
    return working_count
family_composition_by_wave = {'Wave 1': w1_family_composition, 'Wave 2': w2_family_composition,
    'Wave 3': w3_family_composition}
main_second_parent_variables = {'Wave 1': ('W1wrkcurMP', 'W1wrkcurSP'), 'Wave 2': ('W2wrkcurMP', 'W2wrkcurSP'),
    'Wave 3': ('W3wrkcurMP', 'W3wrkcurSP')}
father_mother_variables = {'Wave 1': ('W1wrkcurdad', 'W1wrkcurmum'), 'Wave 2': ('W2wrkcurdad', 'W2wrkcurmum'),
    'Wave 3': ('W3wrkcurdad', 'W3wrkcurmum')}
parental_work_count_by_wave = {}
parental_work_comparison_rows = []
parental_work_coverage_rows = []
parental_work_difference_rows = []
for wave in ['Wave 1', 'Wave 2', 'Wave 3']:
    mp_variable, sp_variable = main_second_parent_variables[wave]
    father_variable, mother_variable = father_mother_variables[wave]
    main_parent_measure = parental_work_by_variable[mp_variable]
    second_parent_measure = parental_work_by_variable[sp_variable]
    father_measure = parental_work_by_variable[father_variable]
    mother_measure = parental_work_by_variable[mother_variable]
    main_second_count = derive_working_role_count(first_measure=main_parent_measure,
        second_measure=second_parent_measure, first_absence_codes=[], second_absence_codes=[-98])
    father_mother_count = derive_working_role_count(first_measure=father_measure, second_measure=mother_measure,
        first_absence_codes=[-98], second_absence_codes=[-98])
    parental_work_count_by_wave[wave] = {'Main and second parent': main_second_count,
        'Father and mother': father_mother_count}
    comparison_available = main_second_count.notna() & father_mother_count.notna()
    exact_agreement = main_second_count.loc[comparison_available].eq(father_mother_count.loc[comparison_available])
    parental_work_comparison_rows.append({'Wave': wave,
        'Main/second-parent valid': int(main_second_count.notna().sum()), 'Father/mother valid': int(father_mother_count.notna().sum()), 'Complete comparisons': int(comparison_available.sum()), 'Exact agreement': int(exact_agreement.sum()), 'Different values': int((~exact_agreement).sum()), 'Agreement percentage': round(exact_agreement.mean() * 100,
        2)})
    family_composition_measure = family_composition_by_wave[wave]
    for family_code in [1, 2, 3, 4, 5]:
        family_group = family_composition_measure.eq(family_code)
        group_comparison = family_group & main_second_count.notna() & father_mother_count.notna()
        group_agreement = main_second_count.loc[group_comparison].eq(father_mother_count.loc[group_comparison])
        parental_work_coverage_rows.append({'Wave': wave, 'Family composition code': family_code,
            'Participants': int(family_group.sum()), 'Main/second-parent valid': int((family_group & main_second_count.notna()).sum()), 'Father/mother valid': int((family_group & father_mother_count.notna()).sum()), 'Complete comparisons': int(group_comparison.sum()), 'Agreement percentage': round(group_agreement.mean() * 100,
            2) if group_comparison.any() else pd.NA})
    differing_cases = comparison_available & main_second_count.ne(father_mother_count)
    difference_table = pd.DataFrame({'Main/second-parent count': main_second_count.loc[differing_cases],
        'Father/mother count': father_mother_count.loc[differing_cases], 'Family composition': family_composition_measure.loc[differing_cases]}).value_counts(dropna=False).rename('Participants').reset_index()
    if not difference_table.empty:
        difference_table.insert(0, 'Wave', wave)
        parental_work_difference_rows.append(difference_table)
parental_work_construct_comparison = pd.DataFrame(parental_work_comparison_rows)
parental_work_coverage_by_family_type = pd.DataFrame(parental_work_coverage_rows)
if parental_work_difference_rows:
    parental_work_construct_differences = pd.concat(parental_work_difference_rows, ignore_index=True)
else:
    parental_work_construct_differences = pd.DataFrame(columns=['Wave', 'Main/second-parent count',
        'Father/mother count', 'Family composition', 'Participants'])
parental_work_count_distribution_rows = []
for wave, constructions in parental_work_count_by_wave.items():
    for construction_name, measure in constructions.items():
        distribution = measure.value_counts(dropna=False).rename_axis('Working-role count').reset_index(name='Participants')
        distribution.insert(0, 'Wave', wave)
        distribution.insert(1, 'Construction', construction_name)
        parental_work_count_distribution_rows.append(distribution)
parental_work_count_distributions = pd.concat(parental_work_count_distribution_rows, ignore_index=True)
print('Alternative construction agreement:')
display_limited(parental_work_construct_comparison)
print('Coverage and agreement by family composition:')
display_limited(parental_work_coverage_by_family_type)
print('Observed differences between constructions:')
display_limited(parental_work_construct_differences)
print('Working-role count distributions:')
display_limited(parental_work_count_distributions)

Alternative construction agreement:


,Wave,Main/second-parent valid,Father/mother valid,Complete comparisons,Exact agreement,Different values,Agreement percentage
0,Wave 1,9503,9341,9340,9310,30,99.68
1,Wave 2,9299,9003,8996,8980,16,99.82
2,Wave 3,9482,9373,9362,9244,118,98.74


Coverage and agreement by family composition:


,Wave,Family composition code,Participants,Main/second-parent valid,Father/mother valid,Complete comparisons,Agreement percentage
0,Wave 1,1,6771,6760,6721,6721,100.00
1,Wave 1,2,654,654,571,571,100.00
2,Wave 1,3,155,154,152,152,100.00
3,Wave 1,4,1808,1807,1800,1800,100.00
4,Wave 1,5,85,84,83,82,64.63


Observed differences between constructions:


,Wave,Main/second-parent count,Father/mother count,Family composition,Participants
0,Wave 1,1,0,5.0,29
1,Wave 1,1,0,<NA>,1
2,Wave 2,1,0,5.0,16
3,Wave 3,2,1,4.0,63
4,Wave 3,1,0,5.0,17


Working-role count distributions:


,Wave,Construction,Working-role count,Participants
0,Wave 1,Main and second parent,2,4881
1,Wave 1,Main and second parent,1,3137
2,Wave 1,Main and second parent,0,1485
3,Wave 1,Main and second parent,<NA>,264
4,Wave 1,Father and mother,2,4860


In [230]:
# 44: Cross-wave household working-status comparison

import pandas as pd
w1_household_working_count = parental_work_count_by_wave['Wave 1']['Main and second parent'].astype('Float64')
w2_household_working_count = parental_work_count_by_wave['Wave 2']['Main and second parent'].astype('Float64')
w3_household_working_count = parental_work_count_by_wave['Wave 3']['Main and second parent'].astype('Float64')
w3_household_working_count_before_transition = w3_household_working_count.where(w3_income_timing_status.eq('Before September 2006'))
household_working_count_comparison_rows = []
for first_name, first_measure, second_name, second_measure in [('Wave 1', w1_household_working_count, 'Wave 2',
    w2_household_working_count), ('Wave 1', w1_household_working_count, 'Wave 3',
    w3_household_working_count), ('Wave 2', w2_household_working_count, 'Wave 3', w3_household_working_count)]:
    complete_cases = first_measure.notna() & second_measure.notna()
    exact_agreement = first_measure.loc[complete_cases].eq(second_measure.loc[complete_cases])
    household_working_count_comparison_rows.append({'Comparison': f'{first_name} with {second_name}',
        'Complete cases': int(complete_cases.sum()), 'Exact agreement': int(exact_agreement.sum()), 'Different values': int((~exact_agreement).sum()), 'Agreement percentage': round(exact_agreement.mean() * 100,
        2), "Cramér's V": round(cramers_v(first_measure, second_measure), 4)})
household_working_count_cross_wave_comparison = pd.DataFrame(household_working_count_comparison_rows)
household_working_count_candidate = w3_household_working_count_before_transition.combine_first(w2_household_working_count).combine_first(w1_household_working_count)
household_working_count_candidate_source = pd.Series('Unavailable', index=stage_2_ids, dtype='string')
household_working_count_candidate_source.loc[w3_household_working_count_before_transition.notna()] = 'Wave 3 before September 2006'
household_working_count_candidate_source.loc[w3_household_working_count_before_transition.isna() & w2_household_working_count.notna()] = 'Wave 2 fallback'
household_working_count_candidate_source.loc[w3_household_working_count_before_transition.isna() & w2_household_working_count.isna() & w1_household_working_count.notna()] = 'Wave 1 fallback'
household_working_count_coverage_summary = pd.DataFrame({'Measure or condition': ['Wave 1 valid working count',
    'Wave 2 valid working count', 'Wave 3 valid working count', 'Wave 3 valid before September 2006', 'Wave 3 valid in September 2006', 'Wave 3 valid with interview date unavailable', 'Eligible Wave 3 with Wave 2 and Wave 1 fallback', 'Unavailable after all eligible sources'], 'Participants': [int(w1_household_working_count.notna().sum()),
    int(w2_household_working_count.notna().sum()), int(w3_household_working_count.notna().sum()), int(w3_household_working_count_before_transition.notna().sum()), int((w3_household_working_count.notna() & w3_income_timing_status.eq('September 2006')).sum()), int((w3_household_working_count.notna() & w3_income_timing_status.eq('Interview date unavailable')).sum()), int(household_working_count_candidate.notna().sum()), int(household_working_count_candidate.isna().sum())]})
household_working_count_source_summary = household_working_count_candidate_source.value_counts(dropna=False).rename_axis('Candidate source').reset_index(name='Participants')
household_working_count_distribution = household_working_count_candidate.value_counts(dropna=False).rename_axis('Number of working parents or guardians').reset_index(name='Participants').sort_values('Number of working parents or guardians',
    na_position='last').reset_index(drop=True)
wave_2_wave_3_working_count_table = pd.crosstab(w2_household_working_count, w3_household_working_count,
    rownames=['Wave 2 working count'], colnames=['Wave 3 working count'], margins=True)
print('Cross-wave household working-status agreement:')
display_limited(household_working_count_cross_wave_comparison)
print('Household working-status timing and coverage:')
display_limited(household_working_count_coverage_summary)
print('Candidate source contribution:')
display_limited(household_working_count_source_summary)
print('Candidate working-count distribution:')
display_limited(household_working_count_distribution)
print('Wave 2 to Wave 3 working-count table:')
display_limited(wave_2_wave_3_working_count_table)

Cross-wave household working-status agreement:


,Comparison,Complete cases,Exact agreement,Different values,Agreement percentage,Cramér's V
0,Wave 1 with Wave 2,9281,8237,1044,88.75,0.8194
1,Wave 1 with Wave 3,9462,7823,1639,82.68,0.7293
2,Wave 2 with Wave 3,9270,8068,1202,87.03,0.7960


Household working-status timing and coverage:


,Measure or condition,Participants
0,Wave 1 valid working count,9503
1,Wave 2 valid working count,9299
2,Wave 3 valid working count,9482
3,Wave 3 valid before September 2006,9359
4,Wave 3 valid in September 2006,14


Candidate source contribution:


,Candidate source,Participants
0,Wave 3 before September 2006,9359
1,Unavailable,244
2,Wave 2 fallback,143
3,Wave 1 fallback,21


Candidate working-count distribution:


,Number of working parents or guardians,Participants
0,0.0,1495
1,1.0,3121
2,2.0,4907
3,<NA>,244


Wave 2 to Wave 3 working-count table:


Wave 3 working count,0.0,1.0,2.0,All
Wave 2 working count,,,,
0.0,1264,210,20,1494
1.0,160,2441,416,3017
2.0,11,385,4363,4759
All,1435,3036,4799,9270


In [231]:
# 45: Detailed current parental economic-activity measures

import pandas as pd
detailed_parental_activity_variables = ['W1wrk1aMP', 'W1wrk1aSP', 'W2wrkstatusMP', 'W2wrkstatusSP', 'W3wrk1aMP',
    'W3wrk1aSP', 'W4Wrk1aMP', 'W4Wrk1aSP']
detailed_parental_activity_candidates = expanded_parental_employment_candidates.loc[expanded_parental_employment_candidates['Variable'].isin(detailed_parental_activity_variables)].copy().sort_values(['Wave',
    'Source file', 'Variable position']).reset_index(drop=True)
expected_activity_variables = set(detailed_parental_activity_variables)
found_activity_variables = set(detailed_parental_activity_candidates['Variable'])
missing_activity_variables = sorted(expected_activity_variables - found_activity_variables)
assert len(detailed_parental_activity_candidates) == 8
assert not missing_activity_variables
detailed_parental_activity_by_variable = {}
detailed_activity_code_label_rows = []
detailed_activity_coverage_rows = []
detailed_activity_negative_code_rows = []
detailed_activity_distribution_rows = []
for source_file, candidate_group in detailed_parental_activity_candidates.groupby('Source file', sort=False):
    source_path = source_file_lookup[source_file]
    variable_names = candidate_group['Variable'].tolist()
    raw_data = pd.read_stata(source_path, columns=['NSID'] + variable_names, convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=variable_names, convert_categoricals=True)
    raw_data['NSID'] = raw_data['NSID'].astype('string').str.strip().str.replace('\\.0$', '', regex=True)
    aligned_data = raw_data.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)
    for _, candidate in candidate_group.iterrows():
        wave = candidate['Wave']
        variable = candidate['Variable']
        variable_label = candidate['Variable label']
        parental_role = 'Main parent' if variable.lower().endswith('mp') else 'Second parent'
        values = pd.to_numeric(aligned_data[variable], errors='coerce')
        detailed_parental_activity_by_variable[variable] = values
        code_label_map = pd.DataFrame({'Code': pd.to_numeric(raw_data[variable], errors='coerce'),
            'Label': labelled_data[variable].astype('string')}).dropna(subset=['Code']).drop_duplicates().sort_values('Code').reset_index(drop=True)
        code_label_map.insert(0, 'Wave', wave)
        code_label_map.insert(1, 'Variable', variable)
        code_label_map.insert(2, 'Parental role', parental_role)
        detailed_activity_code_label_rows.append(code_label_map)
        valid_values = values.loc[values.ge(0)]
        negative_values = values.loc[values.lt(0)]
        detailed_activity_coverage_rows.append({'Wave': wave, 'Variable': variable, 'Parental role': parental_role,
            'Variable label': variable_label, 'Valid responses': int(valid_values.notna().sum()), 'Negative codes': int(negative_values.notna().sum()), 'No source record': int(values.isna().sum()), 'Distinct valid codes': int(valid_values.nunique()), 'Minimum valid code': valid_values.min() if valid_values.notna().any() else pd.NA, 'Maximum valid code': valid_values.max() if valid_values.notna().any() else pd.NA})
        for code, count in negative_values.value_counts().sort_index().items():
            detailed_activity_negative_code_rows.append({'Wave': wave, 'Variable': variable,
                'Parental role': parental_role, 'Negative code': code, 'Participants': int(count)})
        variable_label_map = dict(zip(code_label_map['Code'], code_label_map['Label']))
        variable_distribution = pd.DataFrame({'Code': values,
            'Label': values.map(variable_label_map)}).value_counts(['Code', 'Label'],
            dropna=False).rename('Participants').reset_index().sort_values('Code',
            na_position='last').reset_index(drop=True)
        variable_distribution.insert(0, 'Wave', wave)
        variable_distribution.insert(1, 'Variable', variable)
        variable_distribution.insert(2, 'Parental role', parental_role)
        detailed_activity_distribution_rows.append(variable_distribution)
detailed_parental_activity_code_labels = pd.concat(detailed_activity_code_label_rows, ignore_index=True)
detailed_parental_activity_coverage = pd.DataFrame(detailed_activity_coverage_rows)
detailed_parental_activity_negative_codes = pd.DataFrame(detailed_activity_negative_code_rows)
detailed_parental_activity_distributions = pd.concat(detailed_activity_distribution_rows, ignore_index=True)
print('Detailed current parental activity variables:')
display_limited(detailed_parental_activity_candidates)
print('Stored codes and labels:')
display_limited(detailed_parental_activity_code_labels)
print('Coverage by variable:')
display_limited(detailed_parental_activity_coverage)
print('Observed negative codes:')
display_limited(detailed_parental_activity_negative_codes)
print('Participant distributions:')
display_limited(detailed_parental_activity_distributions)

Detailed current parental activity variables:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Decision reason,Found in initial search,Measure family
0,Wave 1,Family background,wave_one_lsype_family_background_2020,140,W1wrk1aMP,MP: Current working status,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,Economic-activity measure
1,Wave 1,Family background,wave_one_lsype_family_background_2020,238,W1wrk1aSP,SP: Current working status,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,Economic-activity measure
2,Wave 2,Family background,wave_two_lsype_family_background_2020,41,W2wrkstatusMP,DV: Current employment status -MP,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,Employment-status measure
3,Wave 2,Family background,wave_two_lsype_family_background_2020,400,W2wrkstatusSP,SP: SP's current employment status,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,True,Employment-status measure
4,Wave 3,Family background,wave_three_lsype_family_background_2020,44,W3wrk1aMP,MP: MP's economic activity,Near-transition source,Use requires interview timing confirming Janua...,Pending review,<NA>,True,Economic-activity measure


Stored codes and labels:


,Wave,Variable,Parental role,Code,Label
0,Wave 1,W1wrk1aMP,Main parent,-99,MP not interviewed
1,Wave 1,W1wrk1aMP,Main parent,-92,Refused
2,Wave 1,W1wrk1aMP,Main parent,-91,Not applicable
3,Wave 1,W1wrk1aMP,Main parent,1,Full-time paid employee (30 or more hours a week)
4,Wave 1,W1wrk1aMP,Main parent,2,Part-time paid employee (under 30 hours a week)


Coverage by variable:


,Wave,Variable,Parental role,Variable label,Valid responses,Negative codes,No source record,Distinct valid codes,Minimum valid code,Maximum valid code
0,Wave 1,W1wrk1aMP,Main parent,MP: Current working status,9420,104,243,12,1.0,12.0
1,Wave 1,W1wrk1aSP,Second parent,SP: Current working status,6625,2899,243,12,1.0,12.0
2,Wave 2,W2wrkstatusMP,Main parent,DV: Current employment status -MP,9337,184,246,13,1.0,44.0
3,Wave 2,W2wrkstatusSP,Second parent,SP: SP's current employment status,6524,2997,246,13,1.0,44.0
4,Wave 3,W3wrk1aMP,Main parent,MP: MP's economic activity,9425,84,258,12,1.0,12.0


Observed negative codes:


,Wave,Variable,Parental role,Negative code,Participants
0,Wave 1,W1wrk1aMP,Main parent,-99.0,103
1,Wave 1,W1wrk1aMP,Main parent,-91.0,1
2,Wave 1,W1wrk1aSP,Second parent,-99.0,748
3,Wave 1,W1wrk1aSP,Second parent,-98.0,2148
4,Wave 1,W1wrk1aSP,Second parent,-92.0,3


Participant distributions:


,Wave,Variable,Parental role,Code,Label,Participants
0,Wave 1,W1wrk1aMP,Main parent,-99.0,MP not interviewed,103
1,Wave 1,W1wrk1aMP,Main parent,-91.0,Not applicable,1
2,Wave 1,W1wrk1aMP,Main parent,1.0,Full-time paid employee (30 or more hours a week),3414
3,Wave 1,W1wrk1aMP,Main parent,2.0,Part-time paid employee (under 30 hours a week),2847
4,Wave 1,W1wrk1aMP,Main parent,3.0,Full-time self-employed,330


In [232]:
# 46: Working-status summary consistency

import pandas as pd
parental_work_consistency_specifications = {'Wave 1': {'Main parent detailed': 'W1wrk1aMP',
    'Second parent detailed': 'W1wrk1aSP', 'Main parent summary': 'W1wrkcurMP', 'Second parent summary': 'W1wrkcurSP', 'Working codes': [1,
    2, 3, 4], 'Special missing codes': []}, 'Wave 2': {'Main parent detailed': 'W2wrkstatusMP',
    'Second parent detailed': 'W2wrkstatusSP', 'Main parent summary': 'W2wrkcurMP', 'Second parent summary': 'W2wrkcurSP', 'Working codes': [1,
    2, 3, 4], 'Special missing codes': [44]}, 'Wave 3': {'Main parent detailed': 'W3wrk1aMP',
    'Second parent detailed': 'W3wrk1aSP', 'Main parent summary': 'W3wrkcurMP', 'Second parent summary': 'W3wrkcurSP', 'Working codes': [1,
    2, 3, 4], 'Special missing codes': []}}

def derive_binary_working_status(detailed_measure, working_codes, special_missing_codes=None, allow_absent=False):
    special_missing_codes = special_missing_codes or []
    derived_status = pd.Series(pd.NA, index=detailed_measure.index, dtype='Float64')
    derived_status.loc[detailed_measure.isin(working_codes)] = 1
    valid_non_working = detailed_measure.gt(0) & ~detailed_measure.isin(working_codes) & ~detailed_measure.isin(special_missing_codes)
    derived_status.loc[valid_non_working] = 2
    if allow_absent:
        derived_status.loc[detailed_measure.eq(-98)] = -98
    return derived_status
parental_status_consistency_rows = []
parental_status_difference_rows = []
household_count_consistency_rows = []
detailed_working_count_by_wave = {}
for wave, specification in parental_work_consistency_specifications.items():
    working_codes = specification['Working codes']
    special_missing_codes = specification['Special missing codes']
    main_detailed = detailed_parental_activity_by_variable[specification['Main parent detailed']]
    second_detailed = detailed_parental_activity_by_variable[specification['Second parent detailed']]
    main_summary = parental_work_by_variable[specification['Main parent summary']]
    second_summary = parental_work_by_variable[specification['Second parent summary']]
    main_derived_status = derive_binary_working_status(detailed_measure=main_detailed, working_codes=working_codes,
        special_missing_codes=special_missing_codes, allow_absent=False)
    second_derived_status = derive_binary_working_status(detailed_measure=second_detailed, working_codes=working_codes,
        special_missing_codes=special_missing_codes, allow_absent=True)
    for parental_role, derived_status, summary_status in [('Main parent', main_derived_status, main_summary),
        ('Second parent', second_derived_status, second_summary)]:
        valid_summary_codes = [1, 2] if parental_role == 'Main parent' else [1, 2, -98]
        comparison_available = derived_status.notna() & summary_status.isin(valid_summary_codes)
        exact_agreement = derived_status.loc[comparison_available].eq(summary_status.loc[comparison_available])
        parental_status_consistency_rows.append({'Wave': wave, 'Parental role': parental_role,
            'Detailed measure available': int(derived_status.notna().sum()), 'Summary measure available': int(summary_status.isin(valid_summary_codes).sum()), 'Complete comparisons': int(comparison_available.sum()), 'Exact agreement': int(exact_agreement.sum()), 'Different values': int((~exact_agreement).sum()), 'Agreement percentage': round(exact_agreement.mean() * 100,
            2)})
        differing_cases = comparison_available & derived_status.ne(summary_status)
        difference_table = pd.DataFrame({'Detailed-derived status': derived_status.loc[differing_cases],
            'Stored summary status': summary_status.loc[differing_cases]}).value_counts(dropna=False).rename('Participants').reset_index()
        if not difference_table.empty:
            difference_table.insert(0, 'Wave', wave)
            difference_table.insert(1, 'Parental role', parental_role)
            parental_status_difference_rows.append(difference_table)
    detailed_household_working_count = derive_working_role_count(first_measure=main_derived_status,
        second_measure=second_derived_status, first_absence_codes=[], second_absence_codes=[-98])
    detailed_working_count_by_wave[wave] = detailed_household_working_count
    stored_household_working_count = parental_work_count_by_wave[wave]['Main and second parent']
    count_comparison_available = detailed_household_working_count.notna() & stored_household_working_count.notna()
    count_agreement = detailed_household_working_count.loc[count_comparison_available].eq(stored_household_working_count.loc[count_comparison_available])
    household_count_consistency_rows.append({'Wave': wave,
        'Detailed-derived count available': int(detailed_household_working_count.notna().sum()), 'Stored-summary count available': int(stored_household_working_count.notna().sum()), 'Complete comparisons': int(count_comparison_available.sum()), 'Exact agreement': int(count_agreement.sum()), 'Different values': int((~count_agreement).sum()), 'Agreement percentage': round(count_agreement.mean() * 100,
        2)})
parental_status_consistency = pd.DataFrame(parental_status_consistency_rows)
if parental_status_difference_rows:
    parental_status_differences = pd.concat(parental_status_difference_rows, ignore_index=True)
else:
    parental_status_differences = pd.DataFrame(columns=['Wave', 'Parental role', 'Detailed-derived status',
        'Stored summary status', 'Participants'])
household_working_count_consistency = pd.DataFrame(household_count_consistency_rows)
wave_2_special_missing_summary = pd.DataFrame({'Variable': ['W2wrkstatusMP', 'W2wrkstatusSP'],
    'Code 44 participants': [int(detailed_parental_activity_by_variable['W2wrkstatusMP'].eq(44).sum()),
    int(detailed_parental_activity_by_variable['W2wrkstatusSP'].eq(44).sum())]})
print('Wave 2 special missing-code check:')
display_limited(wave_2_special_missing_summary)
print('Detailed and stored working-status consistency:')
display_limited(parental_status_consistency)
print('Observed status differences:')
display_limited(parental_status_differences)
print('Household working-count consistency:')
display_limited(household_working_count_consistency)

Wave 2 special missing-code check:


,Variable,Code 44 participants
0,W2wrkstatusMP,10
1,W2wrkstatusSP,3


Detailed and stored working-status consistency:


,Wave,Parental role,Detailed measure available,Summary measure available,Complete comparisons,Exact agreement,Different values,Agreement percentage
0,Wave 1,Main parent,9420,9512,9420,9420,0,100.0
1,Wave 1,Second parent,8773,9513,8773,8773,0,100.0
2,Wave 2,Main parent,9327,9474,9327,9327,0,100.0
3,Wave 2,Second parent,8664,9327,8664,8664,0,100.0
4,Wave 3,Main parent,9425,9490,9425,9425,0,100.0


Observed status differences:


,Wave,Parental role,Detailed-derived status,Stored summary status,Participants


Household working-count consistency:


,Wave,Detailed-derived count available,Stored-summary count available,Complete comparisons,Exact agreement,Different values,Agreement percentage
0,Wave 1,8712,9503,8712,8712,0,100.0
1,Wave 2,8521,9299,8521,8521,0,100.0
2,Wave 3,9374,9482,9374,9374,0,100.0


In [233]:
# 47: Household working-status construction and decisions

import pandas as pd
working_parent_or_guardian_count = household_working_count_candidate.astype('Int64')
working_parent_or_guardian_source = household_working_count_candidate_source.astype('string')
working_parent_or_guardian_label_map = {0: 'No working parent or guardian', 1: 'One working parent or guardian',
    2: 'Two working parents or guardians'}
working_parent_or_guardian_label = working_parent_or_guardian_count.map(working_parent_or_guardian_label_map).astype('string')
household_working_status_review = pd.DataFrame({'NSID': stage_2_ids.to_numpy(),
    'working_parent_or_guardian_count': working_parent_or_guardian_count.to_numpy(), 'working_parent_or_guardian_label': working_parent_or_guardian_label.to_numpy(), 'working_parent_or_guardian_source': working_parent_or_guardian_source.to_numpy()})
assert len(household_working_status_review) == 9767
assert household_working_status_review['NSID'].nunique() == 9767
assert household_working_status_review['NSID'].duplicated().sum() == 0
assert household_working_status_review['NSID'].isna().sum() == 0
assert household_working_status_review['working_parent_or_guardian_count'].dropna().isin([0, 1, 2]).all()
parental_employment_keys = set(expanded_parental_employment_candidates[['Source file',
    'Variable']].itertuples(index=False, name=None))
construction_variables = {'W1wrkcurMP', 'W1wrkcurSP', 'W2wrkcurMP', 'W2wrkcurSP', 'W3wrkcurMP', 'W3wrkcurSP'}
alternative_role_variables = {'W1wrkcurdad', 'W1wrkcurmum', 'W2wrkcurdad', 'W2wrkcurmum', 'W3wrkcurdad',
    'W3wrkcurmum'}
detailed_activity_support_variables = {'W1wrk1aMP', 'W1wrk1aSP', 'W2wrkstatusMP', 'W2wrkstatusSP', 'W3wrk1aMP',
    'W3wrk1aSP'}
review_support_variables = alternative_role_variables | detailed_activity_support_variables
parental_employment_metadata = expanded_parental_employment_candidates[['Wave', 'Source file', 'Variable',
    'Measure family']].drop_duplicates(['Source file', 'Variable']).copy()
parental_employment_metadata['Key'] = list(zip(parental_employment_metadata['Source file'].astype(str),
    parental_employment_metadata['Variable'].astype(str)))
parental_employment_metadata = parental_employment_metadata.set_index('Key')
register_keys = pd.Series(list(zip(variable_decision_register['Source file'].astype(str),
    variable_decision_register['Variable'].astype(str))), index=variable_decision_register.index)
parental_employment_review_mask = register_keys.isin(parental_employment_keys)
working_variable_decision_register = variable_decision_register.copy()
working_variable_decision_register.loc[parental_employment_review_mask, 'Review status'] = 'Reviewed'
working_variable_decision_register.loc[parental_employment_review_mask,
    'Substantive domain'] = 'Family socioeconomic background'
working_variable_decision_register.loc[parental_employment_review_mask,
    'Documentation source'] = 'Wave 1 to Wave 4 value labels, detailed activity consistency and Wave 3 interview timing'
for register_index in working_variable_decision_register.index[parental_employment_review_mask]:
    source_file = str(working_variable_decision_register.at[register_index, 'Source file'])
    variable = str(working_variable_decision_register.at[register_index, 'Variable'])
    key = (source_file, variable)
    wave = parental_employment_metadata.at[key, 'Wave']
    measure_family = parental_employment_metadata.at[key, 'Measure family']
    if variable in construction_variables:
        working_variable_decision_register.at[register_index, 'Review outcome'] = 'Retain as construction source'
        if wave == 'Wave 3':
            working_variable_decision_register.at[register_index,
                'Decision reason'] = 'Primary source for the number of currently working parents or guardians'
            working_variable_decision_register.at[register_index,
                'Leakage assessment'] = 'No leakage after restricting the measure to interviews before September 2006'
            working_variable_decision_register.at[register_index,
                'Reference-period assessment'] = 'Wave 3 interview before September 2006'
            working_variable_decision_register.at[register_index,
                'Review notes'] = 'Joint primary source; 9,359 participants contributed constructed values'
        elif wave == 'Wave 2':
            working_variable_decision_register.at[register_index,
                'Decision reason'] = 'First fallback source where eligible Wave 3 working-status information is unavailable'
            working_variable_decision_register.at[register_index,
                'Leakage assessment'] = 'No leakage: measured before the post-16 transition'
            working_variable_decision_register.at[register_index,
                'Reference-period assessment'] = 'Wave 2 current working status'
            working_variable_decision_register.at[register_index,
                'Review notes'] = 'Joint first fallback source; 143 participants contributed constructed values'
        else:
            working_variable_decision_register.at[register_index,
                'Decision reason'] = 'Second fallback source where eligible Wave 3 and Wave 2 working-status information is unavailable'
            working_variable_decision_register.at[register_index,
                'Leakage assessment'] = 'No leakage: measured before the post-16 transition'
            working_variable_decision_register.at[register_index,
                'Reference-period assessment'] = 'Wave 1 current working status'
            working_variable_decision_register.at[register_index,
                'Review notes'] = 'Joint second fallback source; 21 participants contributed constructed values'
    elif variable in alternative_role_variables:
        working_variable_decision_register.at[register_index, 'Review outcome'] = 'Retain as review support only'
        working_variable_decision_register.at[register_index,
            'Decision reason'] = 'Alternative father-or-mother representation of the selected main-parent and second-parent construct'
        working_variable_decision_register.at[register_index,
            'Leakage assessment'] = 'No leakage; not retained in the predictor set'
        working_variable_decision_register.at[register_index,
            'Reference-period assessment'] = 'Current working status at the relevant survey wave'
        working_variable_decision_register.at[register_index,
            'Review notes'] = 'Used to compare parental-role definitions; main-parent and second-parent measures had higher coverage'
    elif variable in detailed_activity_support_variables:
        working_variable_decision_register.at[register_index, 'Review outcome'] = 'Retain as review support only'
        working_variable_decision_register.at[register_index,
            'Decision reason'] = 'Detailed economic-activity measure used to verify the stored binary working-status summary'
        working_variable_decision_register.at[register_index,
            'Leakage assessment'] = 'No leakage; not retained as an additional predictor'
        working_variable_decision_register.at[register_index,
            'Reference-period assessment'] = 'Current economic activity at the relevant survey wave'
        working_variable_decision_register.at[register_index,
            'Review notes'] = 'Detailed-derived and stored working statuses agreed in all complete comparisons'
        if variable in {'W2wrkstatusMP', 'W2wrkstatusSP'}:
            working_variable_decision_register.at[register_index,
                'Review notes'] = 'Detailed-derived and stored working statuses agreed in all complete comparisons; code 44 was treated as missing household data'
    else:
        working_variable_decision_register.at[register_index, 'Review outcome'] = 'Exclude from predictor set'
        if wave == 'Wave 4':
            working_variable_decision_register.at[register_index,
                'Decision reason'] = 'Earlier waves provide the selected household working-status construct'
            working_variable_decision_register.at[register_index,
                'Leakage assessment'] = 'Timing concern: Wave 4 may contain information measured at or after the outcome boundary'
            working_variable_decision_register.at[register_index,
                'Reference-period assessment'] = 'Wave 4 employment information'
            working_variable_decision_register.at[register_index,
                'Review notes'] = 'Not used in predictor construction'
        elif measure_family == 'Economic-activity period date':
            working_variable_decision_register.at[register_index,
                'Decision reason'] = 'Start year or month of an activity period is not required for the selected current working-count construct'
            working_variable_decision_register.at[register_index,
                'Leakage assessment'] = 'No leakage identified; excluded on construct and redundancy grounds'
            working_variable_decision_register.at[register_index,
                'Reference-period assessment'] = 'Date of current economic-activity period'
            working_variable_decision_register.at[register_index,
                'Review notes'] = 'Conditional timing detail not retained'
        elif measure_family == 'Employment-status change':
            working_variable_decision_register.at[register_index,
                'Decision reason'] = 'Change since an earlier wave is distinct from the selected current household working status'
            working_variable_decision_register.at[register_index,
                'Leakage assessment'] = 'No leakage identified for prospective waves; excluded on construct grounds'
            working_variable_decision_register.at[register_index,
                'Reference-period assessment'] = 'Employment-status change since an earlier wave'
            working_variable_decision_register.at[register_index, 'Review notes'] = 'Change indicator not retained'
        elif measure_family == 'Employment or self-employment detail':
            working_variable_decision_register.at[register_index,
                'Decision reason'] = 'Conditional employment or self-employment detail is narrower than the selected household working-count construct'
            working_variable_decision_register.at[register_index,
                'Leakage assessment'] = 'No leakage identified; excluded on construct and routing grounds'
            working_variable_decision_register.at[register_index,
                'Reference-period assessment'] = 'Conditional current-job information'
            working_variable_decision_register.at[register_index,
                'Review notes'] = 'Conditional employment detail not retained'
        else:
            working_variable_decision_register.at[register_index,
                'Decision reason'] = 'Alternative or conditional employment-status measure is redundant with the selected stored working-status summary'
            working_variable_decision_register.at[register_index,
                'Leakage assessment'] = 'No leakage identified; excluded on redundancy or routing grounds'
            working_variable_decision_register.at[register_index,
                'Reference-period assessment'] = 'Current or most recent parental employment status'
            working_variable_decision_register.at[register_index,
                'Review notes'] = 'Not retained as an additional predictor'
parental_employment_decision_rows = working_variable_decision_register.loc[parental_employment_review_mask, ['Wave',
    'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Review outcome', 'Decision reason', 'Leakage assessment']].sort_values(['Wave',
    'Source file', 'Variable position']).reset_index(drop=True)
assert len(parental_employment_decision_rows) == 70
assert parental_employment_decision_rows[['Source file', 'Variable']].drop_duplicates().shape[0] == 70
assert parental_employment_decision_rows['Review outcome'].notna().all()
assert parental_employment_decision_rows['Decision reason'].notna().all()
assert parental_employment_decision_rows['Leakage assessment'].notna().all()
household_working_status_review_path = stage_2_output_directory / 'stage_2_household_working_status_review.csv'
household_working_status_review.to_csv(household_working_status_review_path, index=False)
working_variable_decision_register.to_csv(decision_register_path, index=False)
variable_decision_register = working_variable_decision_register.copy()
working_status_source_summary = household_working_status_review['working_parent_or_guardian_source'].value_counts(dropna=False).rename_axis('Selected source').reset_index(name='Participants')
working_status_distribution = household_working_status_review[['working_parent_or_guardian_count',
    'working_parent_or_guardian_label']].value_counts(dropna=False).rename('Participants').reset_index().sort_values('working_parent_or_guardian_count',
    na_position='last').reset_index(drop=True)
parental_employment_decision_summary = parental_employment_decision_rows['Review outcome'].value_counts().rename_axis('Review outcome').reset_index(name='Variables')
print(f'Valid household working-status predictors: {working_parent_or_guardian_count.notna().sum():,}')
print(f'Unavailable household working-status predictors: {working_parent_or_guardian_count.isna().sum():,}')
print(f'Review file: {household_working_status_review_path}')
display_limited(working_status_source_summary)
display_limited(working_status_distribution)
display_limited(parental_employment_decision_summary)
print('Source-variable decisions:')
display_limited(parental_employment_decision_rows)

Valid household working-status predictors: 9,523
Unavailable household working-status predictors: 244
Review file: data_derived\stage_2_predictor_construction\stage_2_household_working_status_review.csv


,Selected source,Participants
0,Wave 3 before September 2006,9359
1,Unavailable,244
2,Wave 2 fallback,143
3,Wave 1 fallback,21


,working_parent_or_guardian_count,working_parent_or_guardian_label,Participants
0,0.0,No working parent or guardian,1495
1,1.0,One working parent or guardian,3121
2,2.0,Two working parents or guardians,4907
3,NaN,NaN,244


,Review outcome,Variables
0,Exclude from predictor set,52
1,Retain as review support only,12
2,Retain as construction source,6


Source-variable decisions:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Review outcome,Decision reason,Leakage assessment
0,Wave 1,Family background,wave_one_lsype_family_background_2020,140,W1wrk1aMP,MP: Current working status,Retain as review support only,Detailed economic-activity measure used to ver...,No leakage; not retained as an additional pred...
1,Wave 1,Family background,wave_one_lsype_family_background_2020,141,W1WrkYMP,MP: Year started current activity,Exclude from predictor set,Start year or month of an activity period is n...,No leakage identified; excluded on construct a...
2,Wave 1,Family background,wave_one_lsype_family_background_2020,142,W1wrkmMP,MP: Month of starting current activity,Exclude from predictor set,Start year or month of an activity period is n...,No leakage identified; excluded on construct a...
3,Wave 1,Family background,wave_one_lsype_family_background_2020,146,W1wrkStatMP,MP: Whether current-last job employed or self-...,Exclude from predictor set,Alternative or conditional employment-status m...,No leakage identified; excluded on redundancy ...
4,Wave 1,Family background,wave_one_lsype_family_background_2020,159,W1wrk12aMP,MP: Whether work alone or have employees,Exclude from predictor set,Conditional employment or self-employment deta...,No leakage identified; excluded on construct a...


In [234]:
# 48: Family socioeconomic domain checkpoint

import pandas as pd
family_socioeconomic_review_files = {'Parental qualification': stage_2_output_directory / 'stage_2_parental_qualification_review.csv',
    'Family NS-SEC': stage_2_output_directory / 'stage_2_family_nssec_review.csv', 'Household income': stage_2_output_directory / 'stage_2_household_income_review.csv', 'Housing tenure': stage_2_output_directory / 'stage_2_housing_tenure_review.csv', 'Family composition': stage_2_output_directory / 'stage_2_family_composition_review.csv', 'Household working status': stage_2_output_directory / 'stage_2_household_working_status_review.csv'}
family_socioeconomic_review_tables = {}
family_socioeconomic_file_rows = []
family_socioeconomic_column_rows = []
for construct, file_path in family_socioeconomic_review_files.items():
    assert file_path.exists(), f'Review file not found: {file_path}'
    review_table = pd.read_csv(file_path, dtype={'NSID': 'string'})
    family_socioeconomic_review_tables[construct] = review_table
    family_socioeconomic_file_rows.append({'Construct': construct, 'File': file_path.name, 'Rows': len(review_table),
        'Columns': review_table.shape[1], 'Unique NSID': review_table['NSID'].nunique() if 'NSID' in review_table.columns else pd.NA, 'Duplicate NSID': int(review_table['NSID'].duplicated().sum()) if 'NSID' in review_table.columns else pd.NA, 'Missing NSID': int(review_table['NSID'].isna().sum()) if 'NSID' in review_table.columns else pd.NA})
    for column_position, column_name in enumerate(review_table.columns, start=1):
        family_socioeconomic_column_rows.append({'Construct': construct, 'Column position': column_position,
            'Column': column_name, 'Data type': str(review_table[column_name].dtype), 'Non-missing': int(review_table[column_name].notna().sum()), 'Missing': int(review_table[column_name].isna().sum()), 'Distinct non-missing values': int(review_table[column_name].nunique(dropna=True))})
family_socioeconomic_file_check = pd.DataFrame(family_socioeconomic_file_rows)
family_socioeconomic_column_check = pd.DataFrame(family_socioeconomic_column_rows)
assert family_socioeconomic_file_check['Rows'].eq(9767).all()
assert family_socioeconomic_file_check['Unique NSID'].eq(9767).all()
assert family_socioeconomic_file_check['Duplicate NSID'].eq(0).all()
assert family_socioeconomic_file_check['Missing NSID'].eq(0).all()
print('Family socioeconomic review-file check:')
display_limited(family_socioeconomic_file_check)
print('Saved columns and missingness:')
display_limited(family_socioeconomic_column_check)

Family socioeconomic review-file check:


,Construct,File,Rows,Columns,Unique NSID,Duplicate NSID,Missing NSID
0,Parental qualification,stage_2_parental_qualification_review.csv,9767,4,9767,0,0
1,Family NS-SEC,stage_2_family_nssec_review.csv,9767,4,9767,0,0
2,Household income,stage_2_household_income_review.csv,9767,4,9767,0,0
3,Housing tenure,stage_2_housing_tenure_review.csv,9767,4,9767,0,0
4,Family composition,stage_2_family_composition_review.csv,9767,4,9767,0,0


Saved columns and missingness:


,Construct,Column position,Column,Data type,Non-missing,Missing,Distinct non-missing values
0,Parental qualification,1,NSID,string,9767,0,9767
1,Parental qualification,2,highest_parental_qualification_code,float64,9510,257,7
2,Parental qualification,3,highest_parental_qualification,str,9510,257,7
3,Parental qualification,4,Parental qualification source,str,9767,0,3
4,Family NS-SEC,1,NSID,string,9767,0,9767


In [235]:
# 49: Family socioeconomic domain assembly

import pandas as pd
family_socioeconomic_predictor_specifications = {'Parental qualification': 'highest_parental_qualification_code',
    'Family NS-SEC': 'family_nssec_code', 'Household income': 'household_income_band', 'Housing tenure': 'housing_tenure', 'Family composition': 'family_composition', 'Household working status': 'working_parent_or_guardian_count'}
family_socioeconomic_domain_predictors = pd.DataFrame({'NSID': pd.Series(stage_2_ids.to_numpy(), dtype='string')})
for construct, predictor_column in family_socioeconomic_predictor_specifications.items():
    review_table = family_socioeconomic_review_tables[construct].copy()
    review_table['NSID'] = review_table['NSID'].astype('string').str.strip()
    predictor_table = review_table[['NSID', predictor_column]].copy()
    assert predictor_table['NSID'].nunique() == 9767
    assert predictor_table['NSID'].duplicated().sum() == 0
    family_socioeconomic_domain_predictors = family_socioeconomic_domain_predictors.merge(predictor_table, on='NSID',
        how='left', validate='one_to_one')
family_socioeconomic_predictor_columns = list(family_socioeconomic_predictor_specifications.values())
for predictor_column in family_socioeconomic_predictor_columns:
    family_socioeconomic_domain_predictors[predictor_column] = pd.to_numeric(family_socioeconomic_domain_predictors[predictor_column],
        errors='coerce').astype('Int64')
assert len(family_socioeconomic_domain_predictors) == 9767
assert family_socioeconomic_domain_predictors['NSID'].nunique() == 9767
assert family_socioeconomic_domain_predictors['NSID'].duplicated().sum() == 0
assert family_socioeconomic_domain_predictors['NSID'].isna().sum() == 0
assert family_socioeconomic_domain_predictors.columns.tolist() == ['NSID', 'highest_parental_qualification_code',
    'family_nssec_code', 'household_income_band', 'housing_tenure', 'family_composition', 'working_parent_or_guardian_count']
family_socioeconomic_predictor_qc = pd.DataFrame({'Predictor': family_socioeconomic_predictor_columns,
    'Non-missing': [int(family_socioeconomic_domain_predictors[column].notna().sum()) for column in family_socioeconomic_predictor_columns], 'Missing': [int(family_socioeconomic_domain_predictors[column].isna().sum()) for column in family_socioeconomic_predictor_columns], 'Missing percentage': [round(family_socioeconomic_domain_predictors[column].isna().mean() * 100,
    2) for column in family_socioeconomic_predictor_columns], 'Distinct non-missing values': [int(family_socioeconomic_domain_predictors[column].nunique(dropna=True)) for column in family_socioeconomic_predictor_columns]})
predictors_available_per_participant = family_socioeconomic_domain_predictors[family_socioeconomic_predictor_columns].notna().sum(axis=1)
family_socioeconomic_joint_availability = predictors_available_per_participant.value_counts().sort_index().rename_axis('Family socioeconomic predictors available').reset_index(name='Participants')
family_socioeconomic_joint_availability['Percentage'] = family_socioeconomic_joint_availability['Participants'].div(9767).mul(100).round(2)
family_socioeconomic_domain_decisions = variable_decision_register.loc[variable_decision_register['Substantive domain'].eq('Family socioeconomic background') & variable_decision_register['Review status'].eq('Reviewed')].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert family_socioeconomic_domain_decisions['Review outcome'].notna().all()
assert family_socioeconomic_domain_decisions['Decision reason'].notna().all()
assert family_socioeconomic_domain_decisions['Leakage assessment'].notna().all()
family_socioeconomic_predictor_path = stage_2_output_directory / 'stage_2_family_socioeconomic_domain_predictors.csv'
family_socioeconomic_decision_path = stage_2_output_directory / 'stage_2_family_socioeconomic_domain_decisions.csv'
family_socioeconomic_domain_predictors.to_csv(family_socioeconomic_predictor_path, index=False)
family_socioeconomic_domain_decisions.to_csv(family_socioeconomic_decision_path, index=False)
family_socioeconomic_decision_summary = family_socioeconomic_domain_decisions['Review outcome'].value_counts().rename_axis('Review outcome').reset_index(name='Variables')
print(f'Family socioeconomic predictors: {len(family_socioeconomic_predictor_columns):,}')
print(f'Participants retained: {len(family_socioeconomic_domain_predictors):,}')
print(f'Predictor file: {family_socioeconomic_predictor_path}')
print(f'Decision file: {family_socioeconomic_decision_path}')
print('Predictor quality control:')
display_limited(family_socioeconomic_predictor_qc)
print('Joint predictor availability:')
display_limited(family_socioeconomic_joint_availability)
print('Domain decision summary:')
display_limited(family_socioeconomic_decision_summary)

Family socioeconomic predictors: 6
Participants retained: 9,767
Predictor file: data_derived\stage_2_predictor_construction\stage_2_family_socioeconomic_domain_predictors.csv
Decision file: data_derived\stage_2_predictor_construction\stage_2_family_socioeconomic_domain_decisions.csv
Predictor quality control:


,Predictor,Non-missing,Missing,Missing percentage,Distinct non-missing values
0,highest_parental_qualification_code,9510,257,2.63,7
1,family_nssec_code,9040,727,7.44,8
2,household_income_band,8771,996,10.20,9
3,housing_tenure,9521,246,2.52,8
4,family_composition,9515,252,2.58,5


Joint predictor availability:


,Family socioeconomic predictors available,Participants,Percentage
0,0,243,2.49
1,2,2,0.02
2,3,10,0.10
3,4,73,0.75
4,5,1080,11.06


Domain decision summary:


,Review outcome,Variables
0,Exclude from predictor set,127
1,Retain as review support only,27
2,Retain as construction source,20
3,Retain as derivation input,2


## Family socioeconomic predictors

Six predictors were retained: `highest_parental_qualification_code`, `family_nssec_code`, `household_income_band`, `housing_tenure`, `family_composition` and `working_parent_or_guardian_count`.

All 9,767 participants remain in the domain file. Complete information on the six predictors was available for 8,359 participants (85.58%); 1,165 had between two and five available measures, and 243 had none. Missing values were left unchanged.


# Part 6: Prior attainment

The registered files and the wider local Stata delivery were searched for attainment measured before the post-16 transition. Planned qualifications, school experiences and poorly timed self-reports were not accepted as substitutes for achieved attainment.


In [236]:
# 1: Prior-attainment candidate variables

import pandas as pd
prior_attainment_source_rows = ~variable_decision_register['Source type'].astype(str).isin(['Family background',
    'Parental attitudes'])
prior_attainment_name_match = variable_decision_register['Variable'].astype(str).str.contains('ks2|ks3|ks4|gcse|gcses|attain|examres|examgrade|qualach|pointscore|point_score|cappedscore|best8|best_8|level2|level_2|fiveac|5ac',
    case=False, na=False, regex=True)
prior_attainment_label_match = variable_decision_register['Variable label'].astype(str).str.contains('\\bKey Stage [234]\\b|\\bKS[234]\\b|\\bGCSE results?\\b|\\bGCSE grades?\\b|\\bnumber of GCSEs?\\b|\\bGCSE point score\\b|\\battainment score\\b|\\bprior attainment\\b|\\bqualification(?:s)? achieved\\b|\\bexam(?:ination)? results?\\b|\\bachieved Level 2\\b|\\bfive or more A\\*?-C\\b|\\bbest eight\\b|\\bcapped point score\\b',
    case=False, na=False, regex=True)
non_attainment_label_match = variable_decision_register['Variable label'].astype(str).str.contains('\\bexpect(?:ed|ation)?\\b|\\bpredicted\\b|\\bthink(?:s)? (?:will|they will)\\b|\\bhope(?:s)? to\\b|\\bplan(?:s)? to\\b|\\bwould like\\b|\\bparent(?:al|s?) qualification\\b|\\bmain parent\\b.*\\bqualification\\b|\\bsecond parent\\b.*\\bqualification\\b',
    case=False, na=False, regex=True)
prior_attainment_candidates = variable_decision_register.loc[prior_attainment_source_rows# Search both variable names and labels so the absence of a retained attainment measure is documented rather than assumed.
 & (prior_attainment_name_match | prior_attainment_label_match) & ~non_attainment_label_match,
    ['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Timing status',
    'Review status', 'Review outcome', 'Decision reason']].copy()
prior_attainment_candidates['Name match'] = prior_attainment_name_match.loc[prior_attainment_candidates.index].to_numpy()
prior_attainment_candidates['Label match'] = prior_attainment_label_match.loc[prior_attainment_candidates.index].to_numpy()
prior_attainment_candidates = prior_attainment_candidates.sort_values(['Wave', 'Source type', 'Source file',
    'Variable position']).reset_index(drop=True)
print(f'Prior-attainment candidate variables found: {len(prior_attainment_candidates):,}')
display_limited(prior_attainment_candidates)

Prior-attainment candidate variables found: 8


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Decision reason,Name match,Label match
0,Wave 2,Young person,wave_two_lsype_young_person_2020,530,W2ResPSfinMP0b,MP: Why YP left school - School did not have g...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,False,True
1,Wave 3,Young person,wave_three_lsype_young_person_2020,200,W3resparYP0a,YP: Why parents wanted YP to move school - Exa...,Near-transition source,Use requires interview timing confirming Janua...,Pending review,<NA>,False,True
2,Wave 4,Young person,wave_four_lsype_young_person_2020,143,W4ResParYP0a,YP: Parents' reasons for wanting YP to change ...,At or after transition,Stable characteristics or retrospective pre-tr...,Pending review,<NA>,False,True
3,Wave 4,Young person,wave_four_lsype_young_person_2020,217,W4GCSENoYP,YP: Number of GSCEs YP studied for since Septe...,At or after transition,Stable characteristics or retrospective pre-tr...,Pending review,<NA>,True,False
4,Wave 4,Young person,wave_four_lsype_young_person_2020,650,W4KS4check1YP,YP: Whether YP has any GCSEs (asked if attainm...,At or after transition,Stable characteristics or retrospective pre-tr...,Pending review,<NA>,True,False


In [237]:
# 2: Linked prior-attainment source check

stage_2_source_register = stage_2_source_file_register.copy()
linked_attainment_pattern = 'npd|national pupil|pupil database|key stage|ks2|ks3|ks4|attainment|linked|administrative'
source_register_search_text = stage_2_source_register.astype('string').fillna('').agg(' '.join, axis=1)
registered_linked_attainment_sources = stage_2_source_register.loc[source_register_search_text.str.contains(linked_attainment_pattern,
    case=False, regex=True, na=False)].copy().reset_index(drop=True)
master_register_search_text = variable_decision_register[['Source type', 'Source file', 'Variable',
    'Variable label']].astype('string').fillna('').agg(' '.join, axis=1)
registered_attainment_variable_columns = ['Source order', 'Wave', 'Source type', 'Source file', 'Variable position',
    'Variable', 'Variable label', 'Timing status', 'Review status', 'Review outcome']
registered_attainment_variables = variable_decision_register.loc[master_register_search_text.str.contains(linked_attainment_pattern,
    case=False, regex=True, na=False), registered_attainment_variable_columns].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
stata_search_directory = source_directory.parent
supported_file_extensions = {'.dta', '.sav', '.csv', '.parquet'}
file_name_pattern = 'npd|national[_ -]?pupil|key[_ -]?stage|ks2|ks3|ks4|attain|linked|admin|reduced'
filesystem_linked_attainment_rows = []
for file_path in stata_search_directory.rglob('*'):
    if not file_path.is_file():
        continue
    if file_path.suffix.lower() not in supported_file_extensions:
        continue
    if not pd.Series([file_path.name], dtype='string').str.contains(file_name_pattern, case=False, regex=True,
        na=False).iloc[0]:
        continue
    filesystem_linked_attainment_rows.append({'File name': file_path.name, 'Extension': file_path.suffix.lower(),
        'Parent directory': str(file_path.parent), 'Full path': str(file_path)})
filesystem_linked_attainment_files = pd.DataFrame(filesystem_linked_attainment_rows, columns=['File name', 'Extension',
    'Parent directory', 'Full path'])
if not filesystem_linked_attainment_files.empty:
    filesystem_linked_attainment_files = filesystem_linked_attainment_files.sort_values(['Parent directory',
        'File name']).reset_index(drop=True)
master_register_source_summary = variable_decision_register['Source type'].value_counts(dropna=False).rename_axis('Source type').reset_index(name='Variables')
print(f'Source register: {source_register_path}')
print(f'Directory searched: {stata_search_directory}')
print(f'Registered sources matching linked-attainment terms: {len(registered_linked_attainment_sources):,}')
display_limited(registered_linked_attainment_sources)
print(f'Master-register variables matching linked-attainment terms: {len(registered_attainment_variables):,}')
display_limited(registered_attainment_variables)
print(f'Files matching linked-attainment terms: {len(filesystem_linked_attainment_files):,}')
display_limited(filesystem_linked_attainment_files)
print('Source types currently represented:')
display_limited(master_register_source_summary)

Source register: data_derived\stage_2_predictor_construction\stage_2_source_file_register.csv
Directory searched: data_working\nextsteps_5545\UKDA-5545-stata\stata\stata13
Registered sources matching linked-attainment terms: 0


,Wave,Source type,Classification basis,File name,Relative path,Rows,Variables,NSID present,Unique NSID values,Missing NSID values,Duplicate NSID rows,Data structure,Roster participants present,Roster participants absent,Coverage percentage,Timing status,Review status


Master-register variables matching linked-attainment terms: 6


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome
0,8,Wave 3,Young person,wave_three_lsype_young_person_2020,628,W3hesubYP13,DV: Subject area of degree would like to study...,Near-transition source,Use requires interview timing confirming Janua...,Pending review
1,11,Wave 4,Young person,wave_four_lsype_young_person_2020,650,W4KS4check1YP,YP: Whether YP has any GCSEs (asked if attainm...,At or after transition,Stable characteristics or retrospective pre-tr...,Pending review
2,11,Wave 4,Young person,wave_four_lsype_young_person_2020,651,W4Goodno1YP,YP: Number of GCSEs grade C or higher YP has (...,At or after transition,Stable characteristics or retrospective pre-tr...,Pending review
3,11,Wave 4,Young person,wave_four_lsype_young_person_2020,652,W4KS4check2YP,YP: Whether YP has any intermediate GNVQs (ask...,At or after transition,Stable characteristics or retrospective pre-tr...,Pending review
4,11,Wave 4,Young person,wave_four_lsype_young_person_2020,653,W4Goodno2YP,YP: Number of intermediate GNVQs YP has (asked...,At or after transition,Stable characteristics or retrospective pre-tr...,Pending review


Files matching linked-attainment terms: 0


,File name,Extension,Parent directory,Full path


Source types currently represented:


,Source type,Variables
0,Young person,2615
1,Family background,1994
2,Parental attitudes,534
3,History,118


In [238]:
# 3: Prior-attainment measure refinement

import pandas as pd
attainment_measure_text = variable_decision_register[['Variable',
    'Variable label']].astype('string').fillna('').agg(' '.join, axis=1)
direct_attainment_match = attainment_measure_text.str.contains('\\bKS ?2\\b|\\bKS ?3\\b|\\bKey Stage 2\\b|\\bKey Stage 3\\b|\\bSATs?\\b|\\btest score\\b|\\btest result\\b|\\bexam(?:ination)? result\\b|\\bexam(?:ination)? grade\\b|\\bgrade obtained\\b|\\bgrade achieved\\b|\\bactual grade\\b|\\bmark obtained\\b|\\bEnglish (?:grade|level|result|score)\\b|\\bMaths? (?:grade|level|result|score)\\b|\\bScience (?:grade|level|result|score)\\b|\\bteacher assessment\\b|\\battainment level\\b|\\bacademic attainment\\b|\\bprior attainment\\b',
    case=False, na=False, regex=True)
non_observed_attainment_match = attainment_measure_text.str.contains('\\bexpected\\b|\\bexpectation\\b|\\bpredicted\\b|\\bhope\\b|\\bwould like\\b|\\bplans? to\\b|\\bthinks? (?:he|she|they|YP) will\\b|\\breason\\b.*\\b(?:school|move|leave|left)\\b|\\bwhy\\b.*\\b(?:school|move|leave|left)\\b|\\bsubject area of degree\\b|\\bstudied for since September 2006\\b',
    case=False, na=False, regex=True)
refined_prior_attainment_candidates = variable_decision_register.loc[direct_attainment_match & ~non_observed_attainment_match,
    ['Source order', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status', 'Review status', 'Review outcome', 'Decision reason']].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
print(f'Refined prior-attainment candidates found: {len(refined_prior_attainment_candidates):,}')
display_limited(refined_prior_attainment_candidates)

Refined prior-attainment candidates found: 0


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Decision reason


In [239]:
# 4: Wave 4 fallback attainment coverage

import pandas as pd
wave_4_attainment_source_file = variable_decision_register.loc[variable_decision_register['Variable'].eq('W4Goodno1YP'),
    'Source file'].iloc[0]
wave_4_attainment_source_path = source_file_lookup[wave_4_attainment_source_file]
wave_4_fallback_attainment_variables = ['W4KS4check1YP', 'W4Goodno1YP', 'W4KS4check2YP', 'W4Goodno2YP']
wave_4_attainment_raw = pd.read_stata(wave_4_attainment_source_path, columns=['NSID', 'W4intmonth',
    'W4intyear'] + wave_4_fallback_attainment_variables, convert_categoricals=False)
wave_4_attainment_raw['NSID'] = wave_4_attainment_raw['NSID'].astype('string').str.strip().str.replace('\\.0$', '',
    regex=True)
wave_4_attainment_aligned = wave_4_attainment_raw.drop_duplicates('NSID').set_index('NSID').reindex(stage_2_ids)
for variable in ['W4intmonth', 'W4intyear'] + wave_4_fallback_attainment_variables:
    wave_4_attainment_aligned[variable] = pd.to_numeric(wave_4_attainment_aligned[variable], errors='coerce')
wave_4_attainment_code_rows = []
for variable in wave_4_fallback_attainment_variables:
    values = wave_4_attainment_aligned[variable]
    for code, participants in values.value_counts(dropna=False).sort_index(na_position='last').items():
        if pd.isna(code):
            response_status = 'No source record'
        elif code == -99:
            response_status = 'Young person not interviewed'
        elif code == -92:
            response_status = 'Refused'
        elif code == -91:
            response_status = 'Not applicable or routed away'
        elif code == -1:
            response_status = "Don't know"
        elif code >= 0:
            response_status = 'Observed response'
        else:
            response_status = 'Other unavailable response'
        wave_4_attainment_code_rows.append({'Variable': variable, 'Code': code, 'Response status': response_status,
            'Participants': int(participants)})
wave_4_attainment_code_distribution = pd.DataFrame(wave_4_attainment_code_rows)
gcse_check = wave_4_attainment_aligned['W4KS4check1YP']
gcse_count = wave_4_attainment_aligned['W4Goodno1YP']
gnvq_check = wave_4_attainment_aligned['W4KS4check2YP']
gnvq_count = wave_4_attainment_aligned['W4Goodno2YP']
wave_4_self_reported_gcse_count = pd.Series(pd.NA, index=stage_2_ids, dtype='Float64')
wave_4_self_reported_gcse_count.loc[gcse_check.eq(2)] = 0
wave_4_self_reported_gcse_count.loc[gcse_check.eq(1) & gcse_count.ge(0)] = gcse_count.loc[gcse_check.eq(1) & gcse_count.ge(0)]
wave_4_self_reported_gnvq_count = pd.Series(pd.NA, index=stage_2_ids, dtype='Float64')
wave_4_self_reported_gnvq_count.loc[gnvq_check.eq(2)] = 0
wave_4_self_reported_gnvq_count.loc[gnvq_check.eq(1) & gnvq_count.ge(0)] = gnvq_count.loc[gnvq_check.eq(1) & gnvq_count.ge(0)]
wave_4_attainment_coverage_summary = pd.DataFrame({'Measure': ['GCSE presence check', 'GCSE grade C or higher count',
    'Constructed self-reported GCSE count', 'Intermediate GNVQ presence check', 'Intermediate GNVQ count', 'Constructed self-reported GNVQ count'], 'Observed or constructible': [int(gcse_check.isin([1,
    2]).sum()), int(gcse_count.ge(0).sum()), int(wave_4_self_reported_gcse_count.notna().sum()), int(gnvq_check.isin([1,
    2]).sum()), int(gnvq_count.ge(0).sum()), int(wave_4_self_reported_gnvq_count.notna().sum())], 'Unavailable': [int((~gcse_check.isin([1,
    2])).sum()), int((~gcse_count.ge(0)).sum()), int(wave_4_self_reported_gcse_count.isna().sum()), int((~gnvq_check.isin([1,
    2])).sum()), int((~gnvq_count.ge(0)).sum()), int(wave_4_self_reported_gnvq_count.isna().sum())]})
wave_4_attainment_coverage_summary['Coverage percentage'] = wave_4_attainment_coverage_summary['Observed or constructible'].div(9767).mul(100).round(2)
gcse_measure_available = wave_4_self_reported_gcse_count.notna()
wave_4_gcse_interview_timing = wave_4_attainment_aligned.loc[gcse_measure_available, ['W4intyear',
    'W4intmonth']].value_counts(dropna=False).rename('Participants').reset_index().sort_values(['W4intyear',
    'W4intmonth'], na_position='last').reset_index(drop=True)
wave_4_attainment_range_summary = pd.DataFrame({'Constructed measure': ['Self-reported GCSE grade C or higher count',
    'Self-reported intermediate GNVQ count'], 'Minimum': [wave_4_self_reported_gcse_count.min(),
    wave_4_self_reported_gnvq_count.min()], 'Maximum': [wave_4_self_reported_gcse_count.max(),
    wave_4_self_reported_gnvq_count.max()], 'Distinct values': [int(wave_4_self_reported_gcse_count.nunique(dropna=True)),
    int(wave_4_self_reported_gnvq_count.nunique(dropna=True))]})
print('Wave 4 fallback-attainment code distributions:')
display_limited(wave_4_attainment_code_distribution)
print('Wave 4 fallback-attainment coverage:')
display_limited(wave_4_attainment_coverage_summary)
print('Observed count ranges:')
display_limited(wave_4_attainment_range_summary)
print('Interview timing where a GCSE count can be constructed:')
display_limited(wave_4_gcse_interview_timing)

Wave 4 fallback-attainment code distributions:


,Variable,Code,Response status,Participants
0,W4KS4check1YP,-99.0,Young person not interviewed,93
1,W4KS4check1YP,-91.0,Not applicable or routed away,9125
2,W4KS4check1YP,-1.0,Don't know,2
3,W4KS4check1YP,1.0,Observed response,400
4,W4KS4check1YP,2.0,Observed response,136


Wave 4 fallback-attainment coverage:


,Measure,Observed or constructible,Unavailable,Coverage percentage
0,GCSE presence check,536,9231,5.49
1,GCSE grade C or higher count,394,9373,4.03
2,Constructed self-reported GCSE count,530,9237,5.43
3,Intermediate GNVQ presence check,524,9243,5.37
4,Intermediate GNVQ count,80,9687,0.82


Observed count ranges:


,Constructed measure,Minimum,Maximum,Distinct values
0,Self-reported GCSE grade C or higher count,0.0,17.0,17
1,Self-reported intermediate GNVQ count,0.0,9.0,7


Interview timing where a GCSE count can be constructed:


,W4intyear,W4intmonth,Participants
0,-999.0,-999.0,14
1,2007.0,6.0,179
2,2007.0,7.0,158
3,2007.0,8.0,127
4,2007.0,9.0,49


In [240]:
# 5: Prior-attainment domain decision

import pandas as pd
prior_attainment_review_variables = ['W4GCSENoYP', 'W4KS4check1YP', 'W4Goodno1YP', 'W4KS4check2YP', 'W4Goodno2YP',
    'w4gcse']
prior_attainment_review_rows = variable_decision_register.loc[variable_decision_register['Variable'].isin(prior_attainment_review_variables)].copy()
assert prior_attainment_review_rows['Variable'].nunique() == 6
assert set(prior_attainment_review_rows['Variable']) == set(prior_attainment_review_variables)
prior_attainment_keys = set(prior_attainment_review_rows[['Source file', 'Variable']].itertuples(index=False,
    name=None))
register_keys = pd.Series(list(zip(variable_decision_register['Source file'].astype(str),
    variable_decision_register['Variable'].astype(str))), index=variable_decision_register.index)
prior_attainment_review_mask = register_keys.isin(prior_attainment_keys)
working_variable_decision_register = variable_decision_register.copy()
# Keep the reviewed attainment variables in the decision register even though none are retained as predictors.
working_variable_decision_register.loc[prior_attainment_review_mask, 'Review status'] = 'Reviewed'
working_variable_decision_register.loc[prior_attainment_review_mask, 'Substantive domain'] = 'Prior attainment'
working_variable_decision_register.loc[prior_attainment_review_mask, 'Review outcome'] = 'Exclude from predictor set'
working_variable_decision_register.loc[prior_attainment_review_mask,
    'Documentation source'] = 'Variable labels and Wave 4 fallback-attainment routing and interview-timing review'
gcse_study_count_variables = {'W4GCSENoYP', 'w4gcse'}
gcse_study_count_mask = working_variable_decision_register['Variable'].isin(gcse_study_count_variables) & prior_attainment_review_mask
working_variable_decision_register.loc[gcse_study_count_mask,
    'Decision reason'] = 'Number of GCSEs studied does not measure achieved prior attainment and refers to study since September 2006'
working_variable_decision_register.loc[gcse_study_count_mask,
    'Leakage assessment'] = 'Timing concern: collected after the pre-transition predictor cut-off and may reflect post-transition study'
working_variable_decision_register.loc[gcse_study_count_mask,
    'Reference-period assessment'] = 'GCSE study since September 2006, reported at Wave 4'
working_variable_decision_register.loc[gcse_study_count_mask,
    'Review notes'] = 'Not an achieved-attainment measure; not retained'
gcse_fallback_variables = {'W4KS4check1YP', 'W4Goodno1YP'}
gcse_fallback_mask = working_variable_decision_register['Variable'].isin(gcse_fallback_variables) & prior_attainment_review_mask
working_variable_decision_register.loc[gcse_fallback_mask,
    'Decision reason'] = 'Fallback self-reported GCSE information was available only for a small routed subsample and cannot provide a comparable full-sample prior-attainment measure'
working_variable_decision_register.loc[gcse_fallback_mask,
    'Leakage assessment'] = 'Timing concern: collected in 2007 after the pre-transition predictor cut-off'
working_variable_decision_register.loc[gcse_fallback_mask,
    'Reference-period assessment'] = 'Self-reported GCSE attainment at Wave 4'
working_variable_decision_register.loc[working_variable_decision_register['Variable'].eq('W4KS4check1YP') & prior_attainment_review_mask,
    'Review notes'] = '536 observed responses; 9,125 participants were routed away'
working_variable_decision_register.loc[working_variable_decision_register['Variable'].eq('W4Goodno1YP') & prior_attainment_review_mask,
    'Review notes'] = '394 observed counts; a GCSE count could be constructed for 530 participants, or 5.43% of the sample'
gnvq_fallback_variables = {'W4KS4check2YP', 'W4Goodno2YP'}
gnvq_fallback_mask = working_variable_decision_register['Variable'].isin(gnvq_fallback_variables) & prior_attainment_review_mask
working_variable_decision_register.loc[gnvq_fallback_mask,
    'Decision reason'] = 'Fallback self-reported intermediate GNVQ information was available only for a small routed subsample and cannot provide a comparable full-sample prior-attainment measure'
working_variable_decision_register.loc[gnvq_fallback_mask,
    'Leakage assessment'] = 'Timing concern: collected in 2007 after the pre-transition predictor cut-off'
working_variable_decision_register.loc[gnvq_fallback_mask,
    'Reference-period assessment'] = 'Self-reported intermediate GNVQ attainment at Wave 4'
working_variable_decision_register.loc[working_variable_decision_register['Variable'].eq('W4KS4check2YP') & prior_attainment_review_mask,
    'Review notes'] = '524 observed responses; 9,125 participants were routed away'
working_variable_decision_register.loc[working_variable_decision_register['Variable'].eq('W4Goodno2YP') & prior_attainment_review_mask,
    'Review notes'] = '80 observed counts; a GNVQ count could be constructed for 523 participants, or 5.35% of the sample'
prior_attainment_domain_decisions = working_variable_decision_register.loc[prior_attainment_review_mask].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(prior_attainment_domain_decisions) == 6
assert prior_attainment_domain_decisions['Review outcome'].eq('Exclude from predictor set').all()
assert prior_attainment_domain_decisions['Decision reason'].notna().all()
assert prior_attainment_domain_decisions['Leakage assessment'].notna().all()
prior_attainment_domain_predictors = pd.DataFrame({'NSID': pd.Series(stage_2_ids.to_numpy(), dtype='string')})
assert len(prior_attainment_domain_predictors) == 9767
assert prior_attainment_domain_predictors['NSID'].nunique() == 9767
prior_attainment_predictor_path = stage_2_output_directory / 'stage_2_prior_attainment_domain_predictors.csv'
prior_attainment_decision_path = stage_2_output_directory / 'stage_2_prior_attainment_domain_decisions.csv'
prior_attainment_domain_predictors.to_csv(prior_attainment_predictor_path, index=False)
prior_attainment_domain_decisions.to_csv(prior_attainment_decision_path, index=False)
working_variable_decision_register.to_csv(decision_register_path, index=False)
variable_decision_register = working_variable_decision_register.copy()
prior_attainment_decision_summary = prior_attainment_domain_decisions['Review outcome'].value_counts().rename_axis('Review outcome').reset_index(name='Variables')
print('Prior-attainment predictors retained: 0')
print('Participants retained: 9,767')
print(f'Predictor file: {prior_attainment_predictor_path}')
print(f'Decision file: {prior_attainment_decision_path}')
display_limited(prior_attainment_decision_summary)
print('Reviewed prior-attainment variables:')
display_limited(prior_attainment_domain_decisions[['Wave', 'Source type', 'Source file', 'Variable', 'Variable label',
    'Review outcome', 'Decision reason', 'Leakage assessment', 'Review notes']])

Prior-attainment predictors retained: 0
Participants retained: 9,767
Predictor file: data_derived\stage_2_predictor_construction\stage_2_prior_attainment_domain_predictors.csv
Decision file: data_derived\stage_2_predictor_construction\stage_2_prior_attainment_domain_decisions.csv


,Review outcome,Variables
0,Exclude from predictor set,6


Reviewed prior-attainment variables:


,Wave,Source type,Source file,Variable,Variable label,Review outcome,Decision reason,Leakage assessment,Review notes
0,Wave 4,Young person,wave_four_lsype_young_person_2020,W4GCSENoYP,YP: Number of GSCEs YP studied for since Septe...,Exclude from predictor set,Number of GCSEs studied does not measure achie...,Timing concern: collected after the pre-transi...,Not an achieved-attainment measure; not retained
1,Wave 4,Young person,wave_four_lsype_young_person_2020,W4KS4check1YP,YP: Whether YP has any GCSEs (asked if attainm...,Exclude from predictor set,Fallback self-reported GCSE information was av...,Timing concern: collected in 2007 after the pr...,"536 observed responses; 9,125 participants wer..."
2,Wave 4,Young person,wave_four_lsype_young_person_2020,W4Goodno1YP,YP: Number of GCSEs grade C or higher YP has (...,Exclude from predictor set,Fallback self-reported GCSE information was av...,Timing concern: collected in 2007 after the pr...,394 observed counts; a GCSE count could be con...
3,Wave 4,Young person,wave_four_lsype_young_person_2020,W4KS4check2YP,YP: Whether YP has any intermediate GNVQs (ask...,Exclude from predictor set,Fallback self-reported intermediate GNVQ infor...,Timing concern: collected in 2007 after the pr...,"524 observed responses; 9,125 participants wer..."
4,Wave 4,Young person,wave_four_lsype_young_person_2020,W4Goodno2YP,YP: Number of intermediate GNVQs YP has (asked...,Exclude from predictor set,Fallback self-reported intermediate GNVQ infor...,Timing concern: collected in 2007 after the pr...,80 observed counts; a GNVQ count could be cons...


## Prior-attainment finding

No suitable linked or administrative pre-transition attainment source was identified. Six Wave 4 GCSE or GNVQ variables and fallback self-reported measures were reviewed, but none had adequate timing and coverage. No prior-attainment predictor was retained.


# Part 7: SEN, disability and health

Young-person SEN, disability and health measures were separated from parental health and benefit-related measures. Wave 4 items were subject to additional timing checks.


In [241]:
# 1: SEN, disability and health review inventory

import pandas as pd
sen_health_search_text = variable_decision_register[['Variable',
    'Variable label']].astype('string').fillna('').agg(' '.join, axis=1)
sen_health_construct_match = sen_health_search_text.str.contains('\\bspecial educational needs?\\b|\\bSEN\\b|\\bstatement of (?:special educational )?needs?\\b|\\bschool action\\b|\\bschool action plus\\b|\\bdisab(?:ility|led)?\\b|\\blong[- ]standing illness\\b|\\blimiting illness\\b|\\bhealth condition\\b|\\bgeneral health\\b|\\bself[- ]rated health\\b|\\bmental health\\b|\\bpsychological distress\\b|\\bGHQ\\b|\\badditional care\\b|\\bcare needs?\\b|\\bsick or disabled\\b|\\bdisability benefit\\b',
    case=False, na=False, regex=True)
sen_health_inventory = variable_decision_register.loc[sen_health_construct_match, ['Source order', 'Wave',
    'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Timing status', 'Review status', 'Review outcome', 'Substantive domain', 'Decision reason', 'Leakage assessment', 'Reference-period assessment', 'Review notes']].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
sen_health_decided = sen_health_inventory.loc[sen_health_inventory['Review outcome'].astype('string').ne('Pending review') & sen_health_inventory['Review outcome'].notna()].copy().reset_index(drop=True)
sen_health_pending = sen_health_inventory.loc[sen_health_inventory['Review outcome'].astype('string').eq('Pending review') | sen_health_inventory['Review outcome'].isna()].copy().reset_index(drop=True)
sen_health_decision_summary = sen_health_decided['Review outcome'].value_counts(dropna=False).rename_axis('Review outcome').reset_index(name='Variables')
sen_health_pending_source_summary = sen_health_pending[['Wave',
    'Source type']].value_counts(dropna=False).rename('Variables').reset_index().sort_values(['Wave',
    'Source type']).reset_index(drop=True)
print(f'SEN, disability and health variables identified: {len(sen_health_inventory):,}')
print(f'Variables with recorded decisions: {len(sen_health_decided):,}')
print(f'Variables still pending review: {len(sen_health_pending):,}')
print('Recorded decision summary:')
display_limited(sen_health_decision_summary)
print('Pending variables by source:')
display_limited(sen_health_pending_source_summary)
print('Variables with recorded decisions:')
display_limited(sen_health_decided)
print('Variables still pending review:')
display_limited(sen_health_pending)

SEN, disability and health variables identified: 101
Variables with recorded decisions: 52
Variables still pending review: 49
Recorded decision summary:


,Review outcome,Variables
0,Retain as construction source,21
1,Exclude from predictor set,16
2,Retain as review support only,15


Pending variables by source:


,Wave,Source type,Variables
0,Wave 1,Parental attitudes,1
1,Wave 2,Family background,2
2,Wave 4,Family background,31
3,Wave 4,Young person,15


Variables with recorded decisions:


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Substantive domain,Decision reason,Leakage assessment,Reference-period assessment,Review notes
0,1,Wave 1,Young person,wave_one_lsype_young_person_2020,11,W1senMP,MP: Whether YP ever identified (by anyone) as ...,Pre-transition source,"Variable-level coding, routing and reference-p...",Retain as construction source,Special educational needs,Baseline source for cumulative pre-transition ...,No direct outcome leakage identified. The info...,Whether the young person had ever been identif...,Used with W2senMP under its documented routing...
1,1,Wave 1,Young person,wave_one_lsype_young_person_2020,12,W1senageMP,MP: Age of YP when first identified as having ...,Pre-transition source,"Variable-level coding, routing and reference-p...",Retain as review support only,Special educational needs,Provides age at first SEN identification for p...,No direct outcome leakage identified. The info...,Age at first SEN identification reported at Wa...,"Valid age was available for 1,590 participants..."
2,1,Wave 1,Young person,wave_one_lsype_young_person_2020,13,W1sencurrMP,MP: Whether YP currently thought to have speci...,Pre-transition source,"Variable-level coding, routing and reference-p...",Retain as construction source,Special educational needs,Fallback source for current SEN status among p...,No direct outcome leakage identified. The info...,Current SEN status at Wave 1.,Used only when later valid current-status reco...
3,1,Wave 1,Young person,wave_one_lsype_young_person_2020,25,W1statedMP,MP: Whether YP has ever been given statement o...,Pre-transition source,"Variable-level coding, routing and reference-p...",Retain as construction source,SEN and formal educational support,A current statement of needs represents formal...,No direct outcome leakage identified. The meas...,Whether the young person had received or was a...,Used to establish statement history and distin...
4,1,Wave 1,Young person,wave_one_lsype_young_person_2020,26,W1stated2MP,MP: Whether YP currently has statement of needs,Pre-transition source,"Variable-level coding, routing and reference-p...",Retain as construction source,SEN and formal educational support,A current statement of needs represents formal...,No direct outcome leakage identified. The meas...,Whether the young person currently had a state...,Used as the earliest current-statement status ...


Variables still pending review:


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Substantive domain,Decision reason,Leakage assessment,Reference-period assessment,Review notes
0,3,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,109,W1henotMP0e,MP: Why think it unlikely YP will go into High...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,<NA>,<NA>
1,5,Wave 2,Family background,wave_two_lsype_family_background_2020,301,W2Ben2AmtMP0g,MP: How much MP received on last occasion (ben...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,<NA>,<NA>
2,5,Wave 2,Family background,wave_two_lsype_family_background_2020,662,W2Ben2AmtSP0g,SP: How much SP received on last occasion (ben...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,<NA>,<NA>
3,11,Wave 4,Young person,wave_four_lsype_young_person_2020,13,W4STATEDMP,MP: Whether YP has ever been given statement o...,At or after transition,Stable characteristics or retrospective pre-tr...,Pending review,<NA>,<NA>,<NA>,<NA>,<NA>
4,11,Wave 4,Young person,wave_four_lsype_young_person_2020,14,W4STATED2MP,MP: Whether YP currently has statement of needs,At or after transition,Stable characteristics or retrospective pre-tr...,Pending review,<NA>,<NA>,<NA>,<NA>,<NA>


In [242]:
# 2: Resolution of pending SEN, disability and health exclusions

import pandas as pd
deferred_aspiration_variables = {'W1henotMP0e'}
wave_2_combined_dla_amount_variables = {'W2Ben2AmtMP0g', 'W2Ben2AmtSP0g'}
wave_4_ever_statement_variables = {'W4STATEDMP'}
wave_4_current_statement_variables = {'W4STATED2MP'}
wave_4_outcome_adjacent_variables = {'W4NEETStatYP0c'}
wave_4_young_person_health_variables = {'W4Hea2YP', *{f'W4Hea3YP0{letter}' for letter in 'abcdefghijk'}}
wave_4_parent_health_variables = {'W4Hea2MP', 'W4Hea2SP', *{f'W4Hea3MP0{letter}' for letter in 'abcdefghijk'},
    *{f'W4Hea3SP0{letter}' for letter in 'abcdefghijk'}}
wave_4_parent_job_difficulty_variables = {'W4Getkeep3MP0b', 'W4Getkeep3MP0c'}
wave_4_qualification_false_positive_variables = {'W4QualMP0t'}
wave_4_parent_disability_variables = {'w4disabMP', 'w4disabSP', 'w4disabmum', 'w4disabdad'}
variables_resolved_in_this_cell = wave_2_combined_dla_amount_variables | wave_4_ever_statement_variables | wave_4_current_statement_variables | wave_4_outcome_adjacent_variables | wave_4_young_person_health_variables | wave_4_parent_health_variables | wave_4_parent_job_difficulty_variables | wave_4_qualification_false_positive_variables | wave_4_parent_disability_variables
current_pending_variables = set(sen_health_pending['Variable'].astype(str))
assert len(current_pending_variables) == 49
assert variables_resolved_in_this_cell | deferred_aspiration_variables == current_pending_variables
assert len(variables_resolved_in_this_cell) == 48
working_variable_decision_register = variable_decision_register.copy()

def update_pending_decisions(source_file, variables, substantive_domain, decision_reason, leakage_assessment,
    reference_period_assessment, documentation_source, review_notes):
    """Update a documented group of pending variables."""
    variable_mask = working_variable_decision_register['Source file'].eq(source_file) & working_variable_decision_register['Variable'].isin(variables)
    assert int(variable_mask.sum()) == len(variables)
    assert working_variable_decision_register.loc[variable_mask,
        'Review outcome'].astype('string').eq('Pending review').all()
    working_variable_decision_register.loc[variable_mask, 'Review outcome'] = 'Exclude from predictor set'
    working_variable_decision_register.loc[variable_mask, 'Substantive domain'] = substantive_domain
    working_variable_decision_register.loc[variable_mask, 'Decision reason'] = decision_reason
    working_variable_decision_register.loc[variable_mask, 'Leakage assessment'] = leakage_assessment
    working_variable_decision_register.loc[variable_mask, 'Reference-period assessment'] = reference_period_assessment
    working_variable_decision_register.loc[variable_mask, 'Documentation source'] = documentation_source
    working_variable_decision_register.loc[variable_mask, 'Review notes'] = review_notes
update_pending_decisions(source_file='wave_two_lsype_family_background_2020',
    variables=wave_2_combined_dla_amount_variables, substantive_domain='Family material circumstances', decision_reason='Conditional Disability Living Allowance amount reported only when benefits were paid in combination; the broader household sickness- or disability-benefit indicator has already been retained', leakage_assessment='No direct outcome leakage identified; exclusion is based on conditional coverage and redundancy', reference_period_assessment='Amount received on the most recent Wave 2 payment', documentation_source='Wave 2 family-background data dictionary', review_notes='Not retained as a separate monetary predictor')
update_pending_decisions(source_file='wave_four_lsype_young_person_2020', variables=wave_4_ever_statement_variables,
    substantive_domain='SEN and formal educational support', decision_reason='The Wave 4 ever-statement item does not identify whether the statement was received before the pre-transition predictor cut-off', leakage_assessment='Temporal ambiguity: an affirmative response may include a statement received after transition', reference_period_assessment='Cumulative statement history reported at Wave 4', documentation_source='Wave 4 young-person data dictionary and timing review', review_notes='Earlier Wave 1 to Wave 3 statement information provides the retained pre-transition construction')
update_pending_decisions(source_file='wave_four_lsype_young_person_2020', variables=wave_4_current_statement_variables,
    substantive_domain='SEN and formal educational support', decision_reason='Current statement status was measured after the pre-transition predictor cut-off', leakage_assessment='Temporal leakage risk because the item describes current Wave 4 status', reference_period_assessment='Current statement of needs at Wave 4', documentation_source='Wave 4 young-person data dictionary and timing review', review_notes='Not used as a fallback for the pre-transition statement predictor')
update_pending_decisions(source_file='wave_four_lsype_young_person_2020', variables=wave_4_outcome_adjacent_variables,
    substantive_domain='Outcome-adjacent post-transition status', decision_reason='Agreement that poor health or disability applies within the Wave 4 NEET-status section is not a pre-transition health measure', leakage_assessment='High temporal and construct leakage risk because the item is measured after transition and is tied to post-transition status', reference_period_assessment='Current Wave 4 post-transition circumstances', documentation_source='Variable label and Wave 4 timing classification', review_notes='Excluded from all predictor constructions')
update_pending_decisions(source_file='wave_four_lsype_young_person_2020',
    variables=wave_4_young_person_health_variables, substantive_domain='Young-person health and disability', decision_reason='Wave 4 illness and activity-impact items describe health at or after the post-16 transition and overlap with the retained pre-transition disability measure', leakage_assessment='Temporal leakage risk because health status and its effects may have changed after transition', reference_period_assessment='Current long-standing illness and affected activities reported at Wave 4', documentation_source='Wave 4 young-person variable labels and timing review', review_notes='Not used as later-wave fallback information')
update_pending_decisions(source_file='wave_four_lsype_family_background_2020',
    variables=wave_4_parent_health_variables, substantive_domain='Parental health and disability', decision_reason='Wave 4 parental illness and activity-impact items measure current post-transition circumstances; Wave 1 parental health measures have already been retained', leakage_assessment='Temporal leakage risk because parental health and functional effects may have changed after transition', reference_period_assessment='Current parental long-standing illness and affected activities at Wave 4', documentation_source='Wave 4 family-background variable labels and timing review', review_notes='Excluded rather than used to update Wave 1 parental-health predictors')
update_pending_decisions(source_file='wave_four_lsype_family_background_2020',
    variables=wave_4_parent_job_difficulty_variables, substantive_domain='Parental employment circumstances', decision_reason='These items describe difficulties taking or keeping paid work at Wave 4 rather than a direct pre-transition health construct', leakage_assessment='Temporal leakage risk because the employment difficulty is measured after transition', reference_period_assessment='Current Wave 4 difficulties taking or keeping work', documentation_source='Variable labels and Wave 4 timing classification', review_notes='Search-term match only; not retained in the health domain')
update_pending_decisions(source_file='wave_four_lsype_family_background_2020',
    variables=wave_4_qualification_false_positive_variables, substantive_domain='Parental education', decision_reason='Individual nursing-qualification item already covered by the retained highest-parental-qualification construction', leakage_assessment='No direct outcome leakage identified; exclusion is based on redundancy and incorrect health-domain match', reference_period_assessment='Parental qualification reported at Wave 4', documentation_source='Variable label and completed parental-qualification review', review_notes='The health-domain search matched wording within the qualification label')
update_pending_decisions(source_file='wave_four_lsype_family_background_2020',
    variables=wave_4_parent_disability_variables, substantive_domain='Parental health and disability', decision_reason='Wave 4 parental-disability measures describe post-transition current circumstances and duplicate the construct represented by the retained Wave 1 main-parent disability predictor', leakage_assessment='Temporal leakage risk because disability limitation is assessed at Wave 4', reference_period_assessment='Parental disability and activity limitation at Wave 4', documentation_source='Wave 4 derived-variable labels and timing review', review_notes='Not used as a later-wave fallback')
resolved_decision_mask = working_variable_decision_register['Variable'].isin(variables_resolved_in_this_cell)
assert int(resolved_decision_mask.sum()) == 48
assert working_variable_decision_register.loc[resolved_decision_mask,
    'Review outcome'].eq('Exclude from predictor set').all()
deferred_variable_mask = working_variable_decision_register['Variable'].isin(deferred_aspiration_variables)
assert int(deferred_variable_mask.sum()) == 1
assert working_variable_decision_register.loc[deferred_variable_mask, 'Review outcome'].eq('Pending review').all()
working_variable_decision_register.to_csv(decision_register_path, index=False)
variable_decision_register = working_variable_decision_register.copy()
sen_health_new_exclusions = variable_decision_register.loc[resolved_decision_mask, ['Wave', 'Source type',
    'Source file', 'Variable', 'Variable label', 'Substantive domain', 'Review outcome', 'Decision reason', 'Leakage assessment']].copy().sort_values(['Wave',
    'Source type', 'Source file', 'Variable']).reset_index(drop=True)
remaining_pending_from_inventory = variable_decision_register.loc[variable_decision_register['Variable'].isin(current_pending_variables) & variable_decision_register['Review outcome'].eq('Pending review'),
    ['Wave', 'Source type', 'Source file', 'Variable', 'Variable label']].copy().reset_index(drop=True)
print(f'Pending variables resolved in this cell: {len(sen_health_new_exclusions):,}')
print(f'Variables deferred to a later domain: {len(remaining_pending_from_inventory):,}')
print('New exclusion decisions:')
display_limited(sen_health_new_exclusions)
print('Remaining deferred variable:')
display_limited(remaining_pending_from_inventory)

Pending variables resolved in this cell: 48
Variables deferred to a later domain: 1
New exclusion decisions:


,Wave,Source type,Source file,Variable,Variable label,Substantive domain,Review outcome,Decision reason,Leakage assessment
0,Wave 2,Family background,wave_two_lsype_family_background_2020,W2Ben2AmtMP0g,MP: How much MP received on last occasion (ben...,Family material circumstances,Exclude from predictor set,Conditional Disability Living Allowance amount...,No direct outcome leakage identified; exclusio...
1,Wave 2,Family background,wave_two_lsype_family_background_2020,W2Ben2AmtSP0g,SP: How much SP received on last occasion (ben...,Family material circumstances,Exclude from predictor set,Conditional Disability Living Allowance amount...,No direct outcome leakage identified; exclusio...
2,Wave 4,Family background,wave_four_lsype_family_background_2020,W4Getkeep3MP0b,MP: Difficulties in taking/keeping a paid job ...,Parental employment circumstances,Exclude from predictor set,These items describe difficulties taking or ke...,Temporal leakage risk because the employment d...
3,Wave 4,Family background,wave_four_lsype_family_background_2020,W4Getkeep3MP0c,MP: Difficulties in taking/keeping a paid job ...,Parental employment circumstances,Exclude from predictor set,These items describe difficulties taking or ke...,Temporal leakage risk because the employment d...
4,Wave 4,Family background,wave_four_lsype_family_background_2020,W4Hea2MP,"MP: Whether MP has a long-standing illness, di...",Parental health and disability,Exclude from predictor set,Wave 4 parental illness and activity-impact it...,Temporal leakage risk because parental health ...


Remaining deferred variable:


,Wave,Source type,Source file,Variable,Variable label
0,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1henotMP0e,MP: Why think it unlikely YP will go into High...


In [243]:
# 3: Existing SEN, disability and health predictor file inventory

from pathlib import Path
import pandas as pd
health_predictor_pattern = 'sen|special_educational|statement|disab|disability|health|ghq|additional.*care|care.*parent|sickness.*benefit|disability.*benefit'
health_file_inventory_rows = []
for csv_path in sorted(stage_2_output_directory.glob('*.csv')):
    try:
        file_columns = pd.read_csv(csv_path, nrows=0).columns.tolist()
    except Exception as error:
        health_file_inventory_rows.append({'File name': csv_path.name, 'Has NSID': pd.NA, 'Total columns': pd.NA,
            'Matching columns': pd.NA, 'Read status': f'Header read failed: {error}'})
        continue
    matching_columns = pd.Index(file_columns).astype('string').to_series(index=file_columns).loc[lambda values: values.str.contains(health_predictor_pattern,
        case=False, regex=True, na=False)].tolist()
    if matching_columns:
        health_file_inventory_rows.append({'File name': csv_path.name, 'Has NSID': 'NSID' in file_columns,
            'Total columns': len(file_columns), 'Matching columns': ', '.join(matching_columns), 'Read status': 'Header read'})
health_predictor_file_inventory = pd.DataFrame(health_file_inventory_rows, columns=['File name', 'Has NSID',
    'Total columns', 'Matching columns', 'Read status']).sort_values(['File name']).reset_index(drop=True)
print(f'CSV files containing possible SEN, disability or health predictor columns: {len(health_predictor_file_inventory):,}')
display_limited(health_predictor_file_inventory)

CSV files containing possible SEN, disability or health predictor columns: 16


,File name,Has NSID,Total columns,Matching columns,Read status
0,stage_2_additional_parental_care_review.csv,True,12,"consolidated_disability_status, additional_par...",Header read
1,stage_2_disability_benefit_receipt_review.csv,True,8,"W1 disability-related benefit receipt, W2 disa...",Header read
2,stage_2_disability_measure_review.csv,True,7,"disability_schooling_category, Disability and ...",Header read
3,stage_2_health_persistence_review.csv,True,11,"consolidated_young_person_disability, consolid...",Header read
4,stage_2_main_parent_disability_review.csv,True,11,"main_parent_disability, Main-parent disability...",Header read


In [244]:
# 4: Completed health-predictor review file structure

import pandas as pd
completed_health_review_files = ['stage_2_sen_measure_review.csv', 'stage_2_statement_of_needs_predictor_review.csv',
    'stage_2_disability_measure_review.csv', 'stage_2_additional_parental_care_review.csv', 'stage_2_main_parent_disability_review.csv', 'stage_2_disability_benefit_receipt_review.csv', 'stage_2_young_person_general_health_review.csv', 'stage_2_main_parent_general_health_review.csv', 'stage_2_young_person_ghq12_review.csv']
health_review_structure_rows = []
health_review_column_summary_rows = []
for file_name in completed_health_review_files:
    file_path = stage_2_output_directory / file_name
    assert file_path.exists(), f'Missing review file: {file_path}'
    review_data = pd.read_csv(file_path)
    assert 'NSID' in review_data.columns
    assert len(review_data) == 9767
    assert review_data['NSID'].nunique() == 9767
    assert review_data['NSID'].notna().all()
    health_review_structure_rows.append({'File name': file_name, 'Rows': len(review_data),
        'Columns': len(review_data.columns), 'Column names': ' | '.join(review_data.columns.tolist())})
    for column in review_data.columns:
        if column == 'NSID':
            continue
        column_values = review_data[column]
        health_review_column_summary_rows.append({'File name': file_name, 'Column': column,
            'Data type': str(column_values.dtype), 'Non-missing': int(column_values.notna().sum()), 'Missing': int(column_values.isna().sum()), 'Distinct non-missing values': int(column_values.nunique(dropna=True)), 'Example values': ' | '.join(column_values.dropna().astype('string').drop_duplicates().head(8).tolist())})
health_review_file_structure = pd.DataFrame(health_review_structure_rows)
health_review_column_summary = pd.DataFrame(health_review_column_summary_rows)
print(f'Completed predictor review files checked: {len(health_review_file_structure):,}')
print('Review-file structure:')
display_limited(health_review_file_structure)
print('Column-level summaries:')
display_limited(health_review_column_summary)

Completed predictor review files checked: 9
Review-file structure:


,File name,Rows,Columns,Column names
0,stage_2_sen_measure_review.csv,9767,9,NSID | sen_status_pretransition | Pre-transiti...
1,stage_2_statement_of_needs_predictor_review.csv,9767,11,NSID | selected_sen_status | Selected SEN stat...
2,stage_2_disability_measure_review.csv,9767,7,NSID | disability_schooling_category | Disabil...
3,stage_2_additional_parental_care_review.csv,9767,12,NSID | consolidated_disability_status | additi...
4,stage_2_main_parent_disability_review.csv,9767,11,NSID | main_parent_disability | Main-parent di...


Column-level summaries:


,File name,Column,Data type,Non-missing,Missing,Distinct non-missing values,Example values
0,stage_2_sen_measure_review.csv,sen_status_pretransition,float64,9642,125,3,1.0 | 0.0 | 2.0
1,stage_2_sen_measure_review.csv,Pre-transition SEN status,str,9642,125,3,SEN identified; not current at latest pre-tran...
2,stage_2_sen_measure_review.csv,SEN-measure source,str,9642,125,5,Wave 1–Wave 2 identification and Wave 3 curren...
3,stage_2_sen_measure_review.csv,SEN-measure missing reason,str,125,9642,3,Retrospective pre-transition identification av...
4,stage_2_sen_measure_review.csv,sen_identified_pretransition,float64,9664,103,2,1.0 | 0.0


In [245]:
# 5: Exact columns in completed health-predictor review files

import pandas as pd
health_review_exact_column_rows = []
health_review_numeric_column_rows = []
for file_name in completed_health_review_files:
    file_path = stage_2_output_directory / file_name
    review_data = pd.read_csv(file_path)
    for column_position, column in enumerate(review_data.columns, start=1):
        health_review_exact_column_rows.append({'File name': file_name, 'Column position': column_position,
            'Column': column, 'Data type': str(review_data[column].dtype)})
    numeric_columns = review_data.select_dtypes(include=['number', 'boolean']).columns.difference(['NSID'],
        sort=False)
    for column in numeric_columns:
        values = review_data[column]
        health_review_numeric_column_rows.append({'File name': file_name, 'Column': column,
            'Data type': str(values.dtype), 'Non-missing': int(values.notna().sum()), 'Missing': int(values.isna().sum()), 'Distinct non-missing values': int(values.nunique(dropna=True)), 'Minimum': values.min() if values.notna().any() else pd.NA, 'Maximum': values.max() if values.notna().any() else pd.NA})
health_review_exact_columns = pd.DataFrame(health_review_exact_column_rows)
health_review_numeric_columns = pd.DataFrame(health_review_numeric_column_rows)
print('Exact column names by review file:')
for file_name in completed_health_review_files:
    print(f'\n{file_name}')
    file_columns = health_review_exact_columns.loc[health_review_exact_columns['File name'].eq(file_name),
        ['Column position', 'Column', 'Data type']]
    print(file_columns.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nNumeric-column summaries:')
display_limited(health_review_numeric_columns)

Exact column names by review file:

stage_2_sen_measure_review.csv
 Column position                      Column Data type
               1                        NSID       str
               2    sen_status_pretransition   float64
             ...                         ...       ...
               8 latest_recorded_current_sen   float64
               9   Latest current-SEN source       str

stage_2_statement_of_needs_predictor_review.csv
 Column position                             Column Data type
               1                               NSID       str
               2                selected_sen_status   float64
             ...                                ...       ...
              10                    Decision reason       str
              11 Statement-predictor missing reason       str

stage_2_disability_measure_review.csv
 Column position                        Column Data type
               1                          NSID       str
               2 disability_s

,File name,Column,Data type,Non-missing,Missing,Distinct non-missing values,Minimum,Maximum
0,stage_2_sen_measure_review.csv,sen_status_pretransition,float64,9642,125,3,0.0,2.0
1,stage_2_sen_measure_review.csv,sen_identified_pretransition,float64,9664,103,2,0.0,1.0
2,stage_2_sen_measure_review.csv,latest_recorded_current_sen,float64,9438,329,2,0.0,1.0
3,stage_2_statement_of_needs_predictor_review.csv,selected_sen_status,float64,9642,125,3,0.0,2.0
4,stage_2_statement_of_needs_predictor_review.csv,corrected_statement_status,float64,1798,7969,4,0.0,3.0


In [246]:
# 6: SEN, disability and health domain consolidation

import pandas as pd
health_review_file_names = {'sen': 'stage_2_sen_measure_review.csv',
    'statement': 'stage_2_statement_of_needs_predictor_review.csv', 'young_person_disability': 'stage_2_disability_measure_review.csv', 'additional_parental_care': 'stage_2_additional_parental_care_review.csv', 'main_parent_disability': 'stage_2_main_parent_disability_review.csv', 'disability_benefit': 'stage_2_disability_benefit_receipt_review.csv', 'young_person_health': 'stage_2_young_person_general_health_review.csv', 'main_parent_health': 'stage_2_main_parent_general_health_review.csv', 'ghq12': 'stage_2_young_person_ghq12_review.csv'}
health_review_tables = {}
for review_name, file_name in health_review_file_names.items():
    file_path = stage_2_output_directory / file_name
    review_table = pd.read_csv(file_path, dtype={'NSID': 'string'})
    review_table['NSID'] = review_table['NSID'].str.strip().str.replace('\\.0$', '', regex=True)
    assert len(review_table) == 9767
    assert review_table['NSID'].notna().all()
    assert review_table['NSID'].nunique() == 9767
    health_review_tables[review_name] = review_table
sen_disability_health_predictors = pd.DataFrame({'NSID': pd.Series(stage_2_ids.to_numpy(), dtype='string')})
assert len(sen_disability_health_predictors) == 9767
assert sen_disability_health_predictors['NSID'].nunique() == 9767
participant_index = pd.Index(sen_disability_health_predictors['NSID'], name='NSID')

def aligned_health_column(review_name, column_name):
    review_table = health_review_tables[review_name].set_index('NSID')
    return pd.to_numeric(review_table[column_name], errors='coerce').reindex(participant_index)

def assert_matching_health_columns(first_series, second_series, comparison_name):
    matching_values = first_series.eq(second_series) | first_series.isna() & second_series.isna()
    assert matching_values.all(), f'Inconsistent duplicate measures: {comparison_name}'
assert_matching_health_columns(aligned_health_column('sen', 'sen_status_pretransition'),
    aligned_health_column('statement', 'selected_sen_status'), 'SEN status')
assert_matching_health_columns(aligned_health_column('young_person_disability', 'disability_schooling_category'),
    aligned_health_column('additional_parental_care',
    'consolidated_disability_status'), 'Young-person disability status')
assert_matching_health_columns(aligned_health_column('additional_parental_care',
    'additional_parental_care_due_to_disability'), aligned_health_column('additional_parental_care',
    'selected_additional_parental_care_measure'), 'Additional parental care')
assert_matching_health_columns(aligned_health_column('main_parent_disability', 'main_parent_disability'),
    aligned_health_column('main_parent_disability', 'selected_main_parent_disability'), 'Main-parent disability')
health_predictor_sources = [{'Review name': 'sen', 'Source column': 'sen_status_pretransition',
    'Predictor': 'sen_status_pretransition'}, {'Review name': 'statement',
    'Source column': 'current_statement_of_needs_pretransition', 'Predictor': 'current_statement_of_needs_pretransition'}, {'Review name': 'young_person_disability',
    'Source column': 'disability_schooling_category', 'Predictor': 'disability_schooling_category'}, {'Review name': 'additional_parental_care',
    'Source column': 'selected_additional_parental_care_measure', 'Predictor': 'additional_parental_care_due_to_disability'}, {'Review name': 'main_parent_disability',
    'Source column': 'selected_main_parent_disability', 'Predictor': 'main_parent_disability'}, {'Review name': 'disability_benefit',
    'Source column': 'selected_disability_benefit_receipt', 'Predictor': 'disability_benefit_receipt'}, {'Review name': 'young_person_health',
    'Source column': 'young_person_general_health', 'Predictor': 'young_person_general_health'}, {'Review name': 'main_parent_health',
    'Source column': 'main_parent_general_health', 'Predictor': 'main_parent_general_health'}, {'Review name': 'ghq12',
    'Source column': 'young_person_ghq12_score', 'Predictor': 'young_person_ghq12_score'}]
for predictor_source in health_predictor_sources:
    predictor_values = aligned_health_column(predictor_source['Review name'], predictor_source['Source column'])
    sen_disability_health_predictors[predictor_source['Predictor']] = predictor_values.reset_index(drop=True).astype('Int64')
health_predictor_columns = [predictor_source['Predictor'] for predictor_source in health_predictor_sources]
assert len(health_predictor_columns) == 9
assert list(sen_disability_health_predictors.columns) == ['NSID', *health_predictor_columns]
sen_health_inventory_keys = set(sen_health_inventory[['Source file', 'Variable']].itertuples(index=False, name=None))
decision_register_keys = pd.Series(list(zip(variable_decision_register['Source file'].astype(str),
    variable_decision_register['Variable'].astype(str))), index=variable_decision_register.index)
sen_health_domain_decision_mask = decision_register_keys.isin(sen_health_inventory_keys) & ~variable_decision_register['Variable'].eq('W1henotMP0e')
sen_disability_health_decisions = variable_decision_register.loc[sen_health_domain_decision_mask].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(sen_disability_health_decisions) == 100
assert sen_disability_health_decisions['Review outcome'].notna().all()
assert ~sen_disability_health_decisions['Review outcome'].eq('Pending review').any()
sen_disability_health_predictor_path = stage_2_output_directory / 'stage_2_sen_disability_health_domain_predictors.csv'
sen_disability_health_decision_path = stage_2_output_directory / 'stage_2_sen_disability_health_domain_decisions.csv'
sen_disability_health_predictors.to_csv(sen_disability_health_predictor_path, index=False)
sen_disability_health_decisions.to_csv(sen_disability_health_decision_path, index=False)
sen_disability_health_quality_control = pd.DataFrame([{'Predictor': predictor,
    'Non-missing': int(sen_disability_health_predictors[predictor].notna().sum()), 'Missing': int(sen_disability_health_predictors[predictor].isna().sum()), 'Missing percentage': round(sen_disability_health_predictors[predictor].isna().mean() * 100,
    2), 'Distinct non-missing values': int(sen_disability_health_predictors[predictor].nunique(dropna=True))} for predictor in health_predictor_columns])
sen_disability_health_predictors['Health-domain predictors available'] = sen_disability_health_predictors[health_predictor_columns].notna().sum(axis=1)
sen_disability_health_joint_availability = sen_disability_health_predictors['Health-domain predictors available'].value_counts().sort_index().rename_axis('Health-domain predictors available').reset_index(name='Participants')
sen_disability_health_joint_availability['Percentage'] = sen_disability_health_joint_availability['Participants'].div(9767).mul(100).round(2)
sen_disability_health_predictors = sen_disability_health_predictors.drop(columns=['Health-domain predictors available'])
sen_disability_health_decision_summary = sen_disability_health_decisions['Review outcome'].value_counts().rename_axis('Review outcome').reset_index(name='Variables')
print(f'SEN, disability and health predictors: {len(health_predictor_columns):,}')
print('Participants retained: 9,767')
print(f'Predictor file: {sen_disability_health_predictor_path}')
print(f'Decision file: {sen_disability_health_decision_path}')
print('Predictor quality control:')
display_limited(sen_disability_health_quality_control)
print('Joint predictor availability:')
display_limited(sen_disability_health_joint_availability)
print('Domain decision summary:')
display_limited(sen_disability_health_decision_summary)

SEN, disability and health predictors: 9
Participants retained: 9,767
Predictor file: data_derived\stage_2_predictor_construction\stage_2_sen_disability_health_domain_predictors.csv
Decision file: data_derived\stage_2_predictor_construction\stage_2_sen_disability_health_domain_decisions.csv
Predictor quality control:


,Predictor,Non-missing,Missing,Missing percentage,Distinct non-missing values
0,sen_status_pretransition,9642,125,1.28,3
1,current_statement_of_needs_pretransition,9625,142,1.45,2
2,disability_schooling_category,9466,301,3.08,3
3,additional_parental_care_due_to_disability,9465,302,3.09,2
4,main_parent_disability,9389,378,3.87,3


Joint predictor availability:


,Health-domain predictors available,Participants,Percentage
0,0,39,0.40
1,1,1,0.01
2,2,215,2.20
3,3,1,0.01
4,4,2,0.02


Domain decision summary:


,Review outcome,Variables
0,Exclude from predictor set,64
1,Retain as construction source,21
2,Retain as review support only,15


## SEN, disability and health predictors

Nine predictors were retained: pre-transition SEN status, current statement-of-needs status, disability affecting schooling, additional parental care due to disability, main-parent disability, disability-related benefit receipt, young-person general health, main-parent general health and young-person psychological distress.

Participant coverage and missingness are reported in the preceding quality-control tables. Of 100 reviewed source variables, 21 were construction sources, 15 were review support and 64 were excluded. Missing values were not replaced.


# Part 8: Educational aspirations and post-16 plans

Young-person and parental reports of intended education, training and work routes were reviewed separately. Measures describing activity after the transition were excluded.


In [247]:
# 1: Educational aspirations and post-16 plans inventory

import numpy as np
import pandas as pd
aspiration_plan_search_text = variable_decision_register[['Variable',
    'Variable label']].astype('string').fillna('').agg(' '.join, axis=1)
higher_education_match = aspiration_plan_search_text.str.contains('\\bhigher education\\b|\\buniversity\\b|\\bgo(?:ing)? to HE\\b|\\bgo(?:ing)? into HE\\b|\\bdegree\\b|\\bHE possibility\\b|\\blikely.*\\bHE\\b|\\bunlikely.*\\bHE\\b|\\bexpect(?:s|ed|ation)?\\b.*\\b(?:university|higher education|HE)\\b|\\baspir(?:e|es|ation)\\b.*\\b(?:university|higher education|HE)\\b|heposs|henot',
    case=False, na=False, regex=True)
post16_plan_match = aspiration_plan_search_text.str.contains('\\bpost[- ]?16\\b|\\bafter (?:age )?16\\b|\\bafter Year 11\\b|\\bafter leaving school\\b|\\bwhen (?:YP )?leaves school\\b|\\bstay on at school\\b|\\bstay in (?:full[- ]time )?education\\b|\\bcontinue in education\\b|\\bfuture educational plans?\\b|\\bplans? for next year\\b|\\bplans? after school\\b|\\bwhat .* plans? to do\\b|\\bintend(?:s|ed)? to\\b.*\\b(?:study|education|college|sixth form|training|apprenticeship|work)\\b|\\bexpect(?:s|ed)? to\\b.*\\b(?:study|education|college|sixth form|training|apprenticeship|work)\\b|fplan16|plan16|post16',
    case=False, na=False, regex=True)
intended_route_match = aspiration_plan_search_text.str.contains('\\bplan(?:s|ned)?\\b.*\\b(?:sixth form|college|university|education|employment|job|work|training|apprenticeship)\\b|\\bwould like to\\b.*\\b(?:study|go to university|stay in education|train|work)\\b|\\bpreferred (?:future )?(?:route|activity)\\b',
    case=False, na=False, regex=True)
aspiration_plan_candidate_match = higher_education_match | post16_plan_match | intended_route_match
aspiration_plan_false_positive_match = aspiration_plan_search_text.str.contains('\\bsubject area of degree\\b|\\bqualification(?:s)? (?:already )?held\\b|\\bhighest qualification\\b|\\bcurrent activity\\b|\\bcurrent main activity\\b|\\bfinal activity\\b|\\bNEET status\\b|\\bwhy .* moved school\\b|\\bwhy .* left school\\b|\\bnumber of GCSEs\\b|\\bexam results?\\b',
    case=False, na=False, regex=True)
aspiration_plan_inventory = variable_decision_register.loc[aspiration_plan_candidate_match & ~aspiration_plan_false_positive_match,
    ['Source order', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status', 'Review status', 'Review outcome', 'Substantive domain', 'Decision reason', 'Leakage assessment']].copy()
aspiration_plan_inventory['Candidate theme'] = np.select([higher_education_match.loc[aspiration_plan_inventory.index].to_numpy(),
    post16_plan_match.loc[aspiration_plan_inventory.index].to_numpy(), intended_route_match.loc[aspiration_plan_inventory.index].to_numpy()], ['Higher-education aspiration or expectation',
    'Post-16 education or activity plan', 'Intended future route'], default='Other aspiration or plan')
aspiration_plan_inventory['Decision state'] = np.where(aspiration_plan_inventory['Review outcome'].astype('string').eq('Pending review') | aspiration_plan_inventory['Review outcome'].isna(),
    'Pending review', 'Decision already recorded')
aspiration_plan_inventory = aspiration_plan_inventory.sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
aspiration_plan_source_summary = aspiration_plan_inventory[['Wave', 'Source type',
    'Decision state']].value_counts(dropna=False).rename('Variables').reset_index().sort_values(['Wave', 'Source type',
    'Decision state']).reset_index(drop=True)
aspiration_plan_theme_summary = aspiration_plan_inventory[['Candidate theme',
    'Decision state']].value_counts(dropna=False).rename('Variables').reset_index().sort_values(['Candidate theme',
    'Decision state']).reset_index(drop=True)
print(f'Educational aspiration and post-16 plan variables identified: {len(aspiration_plan_inventory):,}')
print('Candidates by source:')
display_limited(aspiration_plan_source_summary)
print('Candidates by construct theme:')
display_limited(aspiration_plan_theme_summary)
print('Candidate variables:')
display_limited(aspiration_plan_inventory)

Educational aspiration and post-16 plan variables identified: 530
Candidates by source:


,Wave,Source type,Decision state,Variables
0,Wave 1,Family background,Pending review,18
1,Wave 1,Parental attitudes,Pending review,11
2,Wave 1,Young person,Pending review,7
3,Wave 2,Family background,Pending review,28
4,Wave 2,Young person,Pending review,8


Candidates by construct theme:


,Candidate theme,Decision state,Variables
0,Higher-education aspiration or expectation,Decision already recorded,1
1,Higher-education aspiration or expectation,Pending review,478
2,Intended future route,Pending review,4
3,Post-16 education or activity plan,Pending review,47


Candidate variables:


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Substantive domain,Decision reason,Leakage assessment,Candidate theme,Decision state
0,1,Wave 1,Young person,wave_one_lsype_young_person_2020,198,W1plann16YP,YP: YP's intentions after Year 11,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,Post-16 education or activity plan,Pending review
1,1,Wave 1,Young person,wave_one_lsype_young_person_2020,199,W1plast16YP,YP: YP's intentions for further education afte...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,Post-16 education or activity plan,Pending review
2,1,Wave 1,Young person,wave_one_lsype_young_person_2020,200,W1heposs9YP,YP: Likelihood of YP applying for university,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,Higher-education aspiration or expectation,Pending review
3,1,Wave 1,Young person,wave_one_lsype_young_person_2020,201,W1hlikeYP,YP: Likelihood of YP getting into university i...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,Higher-education aspiration or expectation,Pending review
4,1,Wave 1,Young person,wave_one_lsype_young_person_2020,203,W1pladk2YP,YP: What expect to do at age 16 other than fur...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,Post-16 education or activity plan,Pending review


In [248]:
# 2: Educational aspiration and post-16 plan candidates

import numpy as np
import pandas as pd
aspiration_variable_names = variable_decision_register['Variable'].astype('string').fillna('')
aspiration_variable_labels = variable_decision_register['Variable label'].astype('string').fillna('')
direct_measure_name_match = aspiration_variable_names.str.contains('plann16|plast16|pladk2|fplan16|plan16|heposs9|heposs|hlike',
    case=False, na=False, regex=True)
direct_measure_label_match = aspiration_variable_labels.str.contains('\\bintentions? after Year 11\\b|\\bintentions? for further education\\b|\\bwhat .* expect(?:s|ed)? to do at age 16\\b|\\bwhat .* plan(?:s|ned)? to do after Year 11\\b|\\bplan(?:s|ned)? after Year 11\\b|\\bplan(?:s|ned)? after leaving school\\b|\\blikelihood .* applying for university\\b|\\blikelihood .* getting into university\\b|\\blikely .* go into Higher Education\\b|\\blikely .* go to university\\b|\\bwhether .* apply for university\\b|\\bstay on .* full[- ]time education\\b|\\bstay in .* education\\b|\\bcontinue .* education\\b|\\bintend(?:s|ed)? to .*(?:college|sixth form|education|employment|work|training|apprenticeship)\\b',
    case=False, na=False, regex=True)
direct_aspiration_measure_match = direct_measure_name_match | direct_measure_label_match
support_name_match = aspiration_variable_names.str.contains('henot|pladk', case=False, na=False, regex=True)
support_label_match = aspiration_variable_labels.str.contains('\\bwhy .* unlikely .*(?:Higher Education|university|HE)\\b|\\breason(?:s)? .*(?:not continue|not stay|not apply|unlikely to go)\\b|\\bother than further education\\b|\\bwhy .* chose .* post[- ]?16\\b',
    case=False, na=False, regex=True)
aspiration_support_match = support_name_match | support_label_match
aspiration_false_positive_match = aspiration_variable_labels.str.contains('\\bsubject area of degree\\b|\\bdegree subject\\b|\\buniversity costs?\\b|\\btuition fees?\\b|\\bstudent loan\\b|\\bfinancial support\\b|\\bhow .* university .* paid\\b|\\bqualification(?:s)? .* has\\b|\\bhighest qualification\\b|\\bcurrently attending\\b|\\bcurrent activity\\b|\\bcurrent main activity\\b|\\bhas applied\\b|\\bapplication outcome\\b|\\baccepted .* university\\b|\\bactual destination\\b|\\bNEET status\\b|\\bwhy .* moved school\\b|\\bwhy .* left school\\b',
    case=False, na=False, regex=True)
refined_aspiration_match = (direct_aspiration_measure_match | aspiration_support_match) & ~aspiration_false_positive_match
refined_aspiration_plan_inventory = variable_decision_register.loc[refined_aspiration_match, ['Source order', 'Wave',
    'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Timing status', 'Review status', 'Review outcome', 'Substantive domain', 'Decision reason', 'Leakage assessment']].copy()
refined_aspiration_plan_inventory['Review role'] = np.where(direct_aspiration_measure_match.loc[refined_aspiration_plan_inventory.index].to_numpy(),
    'Direct construct measure', 'Routing, reason or review support')
refined_aspiration_plan_inventory['Decision state'] = np.where(refined_aspiration_plan_inventory['Review outcome'].astype('string').eq('Pending review') | refined_aspiration_plan_inventory['Review outcome'].isna(),
    'Pending review', 'Decision already recorded')
refined_aspiration_plan_inventory = refined_aspiration_plan_inventory.sort_values(['Review role', 'Source order',
    'Variable position']).reset_index(drop=True)
refined_aspiration_summary = refined_aspiration_plan_inventory[['Review role', 'Wave', 'Source type',
    'Decision state']].value_counts(dropna=False).rename('Variables').reset_index().sort_values(['Review role', 'Wave',
    'Source type']).reset_index(drop=True)
print(f'Refined educational aspiration and post-16 plan candidates: {len(refined_aspiration_plan_inventory):,}')
print('Refined candidate summary:')
display_limited(refined_aspiration_summary)
print('Direct construct measures:')
display_limited(refined_aspiration_plan_inventory.loc[refined_aspiration_plan_inventory['Review role'].eq('Direct construct measure')].reset_index(drop=True))
print('Routing, reason and review-support variables:')
display_limited(refined_aspiration_plan_inventory.loc[refined_aspiration_plan_inventory['Review role'].eq('Routing, reason or review support')].reset_index(drop=True))

Refined educational aspiration and post-16 plan candidates: 97
Refined candidate summary:


,Review role,Wave,Source type,Decision state,Variables
0,Direct construct measure,Wave 1,Parental attitudes,Pending review,1
1,Direct construct measure,Wave 1,Young person,Pending review,7
2,Direct construct measure,Wave 2,Young person,Pending review,11
3,Direct construct measure,Wave 3,Young person,Pending review,25
4,Direct construct measure,Wave 4,Parental attitudes,Pending review,1


Direct construct measures:


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Substantive domain,Decision reason,Leakage assessment,Review role,Decision state
0,1,Wave 1,Young person,wave_one_lsype_young_person_2020,198,W1plann16YP,YP: YP's intentions after Year 11,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,Direct construct measure,Pending review
1,1,Wave 1,Young person,wave_one_lsype_young_person_2020,199,W1plast16YP,YP: YP's intentions for further education afte...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,Direct construct measure,Pending review
2,1,Wave 1,Young person,wave_one_lsype_young_person_2020,200,W1heposs9YP,YP: Likelihood of YP applying for university,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,Direct construct measure,Pending review
3,1,Wave 1,Young person,wave_one_lsype_young_person_2020,201,W1hlikeYP,YP: Likelihood of YP getting into university i...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,Direct construct measure,Pending review
4,1,Wave 1,Young person,wave_one_lsype_young_person_2020,203,W1pladk2YP,YP: What expect to do at age 16 other than fur...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,Direct construct measure,Pending review


Routing, reason and review-support variables:


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Substantive domain,Decision reason,Leakage assessment,Review role,Decision state
0,1,Wave 1,Young person,wave_one_lsype_young_person_2020,202,W1pladk16YP,YP: What want to do at age 16 other than furth...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,"Routing, reason or review support",Pending review
1,3,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,105,W1henotMP0a,MP: Why think it unlikely YP will go into High...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,"Routing, reason or review support",Pending review
2,3,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,106,W1henotMP0b,MP: Why think it unlikely YP will go into High...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,"Routing, reason or review support",Pending review
3,3,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,107,W1henotMP0c,MP: Why think it unlikely YP will go into High...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,"Routing, reason or review support",Pending review
4,3,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,108,W1henotMP0d,MP: Why think it unlikely YP will go into High...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,<NA>,<NA>,"Routing, reason or review support",Pending review


In [249]:
# 3: Wave 3 young-person interview-date variable check

import pandas as pd
wave_3_young_person_source_file = 'wave_three_lsype_young_person_2020'
wave_3_young_person_variables = variable_decision_register.loc[variable_decision_register['Source file'].eq(wave_3_young_person_source_file),
    ['Variable position', 'Variable', 'Variable label', 'Data type', 'Timing status', 'Review status',
    'Review outcome']].copy().sort_values('Variable position').reset_index(drop=True)
wave_3_date_search_text = wave_3_young_person_variables[['Variable',
    'Variable label']].astype('string').fillna('').agg(' '.join, axis=1)
wave_3_interview_date_candidates = wave_3_young_person_variables.loc[wave_3_date_search_text.str.contains('interview|fieldwork|date of interview|interview date|month interviewed|year interviewed|\\bintmonth\\b|\\bintyear\\b|\\bintdate\\b|intmth|intyr|month.*interview|year.*interview',
    case=False, regex=True, na=False)].copy().reset_index(drop=True)
wave_3_name_date_candidates = wave_3_young_person_variables.loc[wave_3_young_person_variables['Variable'].astype('string').str.contains('int|date|month|mth|year|yr',
    case=False, regex=True, na=False)].copy().reset_index(drop=True)
print(f'Variables in Wave 3 young-person file: {len(wave_3_young_person_variables):,}')
print(f'Interview-date candidates from names or labels: {len(wave_3_interview_date_candidates):,}')
display_limited(wave_3_interview_date_candidates)
print(f'Broader name-based date candidates: {len(wave_3_name_date_candidates):,}')
display_limited(wave_3_name_date_candidates)

Variables in Wave 3 young-person file: 652
Interview-date candidates from names or labels: 21


,Variable position,Variable,Variable label,Data type,Timing status,Review status,Review outcome
0,24,W3scomadiMP,Admin: Interviewer code whether MP accepted se...,int8,Near-transition source,Use requires interview timing confirming Janua...,Exclude from predictor set
1,60,W3sexYP,Admin: Interviewer code sex of YP,int8,Near-transition source,Use requires interview timing confirming Janua...,Retain as construction source
2,226,W3advconYP,YP: Whether YP heard about Connexions before i...,int8,Near-transition source,Use requires interview timing confirming Janua...,Pending review
3,542,W3scompinYP,Admin: Interviewer code whether YP accepted se...,int8,Near-transition source,Use requires interview timing confirming Janua...,Exclude from predictor set
4,563,W3namesYP,"YP: Whether YP upset by name-calling, inc. tex...",int8,Near-transition source,Use requires interview timing confirming Janua...,Pending review


Broader name-based date candidates: 0


,Variable position,Variable,Variable label,Data type,Timing status,Review status,Review outcome


In [250]:
# 4: Wave 3 interview-date source identification

import pandas as pd
wave_3_other_source_variables = variable_decision_register.loc[variable_decision_register['Wave'].eq('Wave 3') & ~variable_decision_register['Source file'].eq('wave_three_lsype_young_person_2020'),
    ['Source order', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Data type',
    'Review status', 'Review outcome', 'Substantive domain']].copy()
wave_3_other_source_search_text = wave_3_other_source_variables[['Variable',
    'Variable label']].astype('string').fillna('').agg(' '.join, axis=1)
wave_3_interview_date_source_candidates = wave_3_other_source_variables.loc[wave_3_other_source_search_text.str.contains('\\binterview month\\b|\\bmonth of interview\\b|\\binterview year\\b|\\byear of interview\\b|\\binterview date\\b|\\bdate of interview\\b|\\bfieldwork month\\b|\\bfieldwork year\\b|intmonth|intyear|intmth|intyr',
    case=False, regex=True, na=False)].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
print(f'Wave 3 interview-date variables found outside the young-person file: {len(wave_3_interview_date_source_candidates):,}')
display_limited(wave_3_interview_date_source_candidates)

Wave 3 interview-date variables found outside the young-person file: 4


,Source order,Source type,Source file,Variable position,Variable,Variable label,Data type,Review status,Review outcome,Substantive domain
0,9,Family background,wave_three_lsype_family_background_2020,10,W3intmnthMP,MP- month of interview,int8,Reviewed,Retain as derivation input,Family socioeconomic background
1,9,Family background,wave_three_lsype_family_background_2020,11,W3intyearMP,MP- year of interview,int16,Reviewed,Retain as derivation input,Family socioeconomic background
2,10,Parental attitudes,wave_three_lsype_parental_attitudes_file_16_06_08,8,W3intmnthMP,MP- month of interview,int8,Use requires interview timing confirming Janua...,Exclude from predictor set,Survey timing support
3,10,Parental attitudes,wave_three_lsype_parental_attitudes_file_16_06_08,9,W3intyearMP,MP- year of interview,int16,Use requires interview timing confirming Janua...,Exclude from predictor set,Survey timing support


In [251]:
# 5: Core aspiration and post-16 plan measure review

import pandas as pd
core_aspiration_sources = {'wave_one_lsype_young_person_2020': ['W1plann16YP', 'W1plast16YP', 'W1heposs9YP',
    'W1hlikeYP', 'W1fplan16YP', 'W1plan16YP'], 'wave_two_lsype_young_person_2020': ['W2plann16YP', 'W2plast16YP',
    'W2heposs9YP', 'W2hlikeYP', 'W2fplan16YP'], 'wave_three_lsype_young_person_2020': ['W3plann16YP', 'W3plast16YP',
    'W3heposs9YP', 'W3hlikeYP', 'W3fplan16YP', 'W3plan16YP'], 'wave_one_lsype_parental_attitudes_file_16_05_08': ['W1hepossMP']}
core_aspiration_variables = [variable for variables in core_aspiration_sources.values() for variable in variables]
assert len(core_aspiration_variables) == 18
assert len(set(core_aspiration_variables)) == 18

def standardise_nsid(series):
    return series.astype('string').str.strip().str.replace('\\.0$', '', regex=True)
wave_3_timing_source_file = 'wave_three_lsype_family_background_2020'
wave_3_timing_data = pd.read_stata(source_file_lookup[wave_3_timing_source_file], columns=['NSID', 'W3intmnthMP',
    'W3intyearMP'], convert_categoricals=False)
wave_3_timing_data['NSID'] = standardise_nsid(wave_3_timing_data['NSID'])
assert wave_3_timing_data['NSID'].is_unique
wave_3_timing_data = wave_3_timing_data.set_index('NSID').reindex(stage_2_ids)
wave_3_interview_month = pd.to_numeric(wave_3_timing_data['W3intmnthMP'], errors='coerce')
wave_3_interview_year = pd.to_numeric(wave_3_timing_data['W3intyearMP'], errors='coerce')
core_aspiration_coverage_rows = []
core_aspiration_code_rows = []
core_aspiration_timing_rows = []
for source_file, variables in core_aspiration_sources.items():
    source_path = source_file_lookup[source_file]
    columns_to_read = ['NSID', *variables]
    raw_data = pd.read_stata(source_path, columns=columns_to_read, convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=columns_to_read, convert_categoricals=True)
    raw_data['NSID'] = standardise_nsid(raw_data['NSID'])
    labelled_data['NSID'] = standardise_nsid(labelled_data['NSID'])
    assert raw_data['NSID'].is_unique
    assert labelled_data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(stage_2_ids)
    labelled_data = labelled_data.set_index('NSID').reindex(stage_2_ids)
    for variable in variables:
        raw_values = pd.to_numeric(raw_data[variable], errors='coerce')
        labelled_values = labelled_data[variable].astype('string')
        observed_response = raw_values.notna() & raw_values.ge(0)
        special_code_response = raw_values.notna() & raw_values.lt(0)
        variable_label = variable_decision_register.loc[variable_decision_register['Source file'].eq(source_file) & variable_decision_register['Variable'].eq(variable),
            'Variable label'].iloc[0]
        core_aspiration_coverage_rows.append({'Source file': source_file, 'Variable': variable,
            'Variable label': variable_label, 'Observed responses': int(observed_response.sum()), 'Special-code responses': int(special_code_response.sum()), 'No source record': int(raw_values.isna().sum()), 'Coverage percentage': round(observed_response.mean() * 100,
            2), 'Distinct observed codes': int(raw_values.loc[observed_response].nunique())})
        paired_values = pd.DataFrame({'Raw code': raw_values, 'Value label': labelled_values})
        code_counts = paired_values['Raw code'].value_counts(dropna=False).sort_index(na_position='last')
        for code, participants in code_counts.items():
            if pd.isna(code):
                value_label = 'No source record'
                response_type = 'No source record'
            else:
                matching_labels = paired_values.loc[paired_values['Raw code'].eq(code),
                    'Value label'].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(code)
                response_type = 'Observed response' if code >= 0 else 'Special code'
            core_aspiration_code_rows.append({'Source file': source_file, 'Variable': variable, 'Raw code': code,
                'Value label': value_label, 'Response type': response_type, 'Participants': int(participants)})
        if source_file == 'wave_three_lsype_young_person_2020':
            timing_category = pd.Series('Undated', index=raw_data.index, dtype='string')
            dated_interview = wave_3_interview_year.ge(0) & wave_3_interview_month.between(1, 12)
            before_september_2006 = dated_interview & (wave_3_interview_year.lt(2006) | wave_3_interview_year.eq(2006) & wave_3_interview_month.le(8))
            september_2006 = dated_interview & wave_3_interview_year.eq(2006) & wave_3_interview_month.eq(9)
            after_september_2006 = dated_interview & (wave_3_interview_year.gt(2006) | wave_3_interview_year.eq(2006) & wave_3_interview_month.gt(9))
            timing_category.loc[before_september_2006] = 'Before September 2006'
            timing_category.loc[september_2006] = 'September 2006'
            timing_category.loc[after_september_2006] = 'After September 2006'
            timing_counts = timing_category.loc[observed_response].value_counts().reindex(['Before September 2006',
                'September 2006', 'After September 2006', 'Undated'], fill_value=0)
            for timing, participants in timing_counts.items():
                core_aspiration_timing_rows.append({'Variable': variable, 'Interview timing': timing,
                    'Participants': int(participants)})
core_aspiration_coverage = pd.DataFrame(core_aspiration_coverage_rows).sort_values(['Source file',
    'Variable']).reset_index(drop=True)
core_aspiration_code_distribution = pd.DataFrame(core_aspiration_code_rows).sort_values(['Source file', 'Variable',
    'Raw code'], na_position='last').reset_index(drop=True)
core_aspiration_wave_3_timing = pd.DataFrame(core_aspiration_timing_rows)
print(f'Core aspiration and post-16 plan measures reviewed: {len(core_aspiration_coverage):,}')
print('Coverage summary:')
display_limited(core_aspiration_coverage)
print('Raw-code and value-label distributions:')
display_limited(core_aspiration_code_distribution)
print('Wave 3 observed-response timing:')
display_limited(core_aspiration_wave_3_timing)

Core aspiration and post-16 plan measures reviewed: 18
Coverage summary:


,Source file,Variable,Variable label,Observed responses,Special-code responses,No source record,Coverage percentage,Distinct observed codes
0,wave_one_lsype_parental_attitudes_file_16_05_08,W1hepossMP,MP: Likelihood of YP going into Higher Education,8874,650,243,90.86,4
1,wave_one_lsype_young_person_2020,W1fplan16YP,YP: What think most of friends will do after Y...,8712,812,243,89.20,3
2,wave_one_lsype_young_person_2020,W1heposs9YP,YP: Likelihood of YP applying for university,9054,470,243,92.70,4
3,wave_one_lsype_young_person_2020,W1hlikeYP,YP: Likelihood of YP getting into university i...,7724,1800,243,79.08,4
4,wave_one_lsype_young_person_2020,W1plan16YP,YP: Agreement that having a job is better than...,9362,162,243,95.85,4


Raw-code and value-label distributions:


,Source file,Variable,Raw code,Value label,Response type,Participants
0,wave_one_lsype_parental_attitudes_file_16_05_08,W1hepossMP,-99.0,MP not interviewed,Special code,103
1,wave_one_lsype_parental_attitudes_file_16_05_08,W1hepossMP,-1.0,Don't know,Special code,547
2,wave_one_lsype_parental_attitudes_file_16_05_08,W1hepossMP,1.0,Very likely,Observed response,3691
3,wave_one_lsype_parental_attitudes_file_16_05_08,W1hepossMP,2.0,Fairly likely,Observed response,2746
4,wave_one_lsype_parental_attitudes_file_16_05_08,W1hepossMP,3.0,Not very likely,Observed response,1349


Wave 3 observed-response timing:


,Variable,Interview timing,Participants
0,W3plann16YP,Before September 2006,7496
1,W3plann16YP,September 2006,0
2,W3plann16YP,After September 2006,0
3,W3plann16YP,Undated,92
4,W3plast16YP,Before September 2006,8151


In [252]:
# 6: Core aspiration construct and coding structure

import pandas as pd
aspiration_construct_groups = {'Intention to remain in education': ['W1plann16YP', 'W2plann16YP', 'W3plann16YP'],
    'Intended further-education setting': ['W1plast16YP', 'W2plast16YP',
    'W3plast16YP'], 'Higher-education application likelihood': ['W1heposs9YP', 'W2heposs9YP',
    'W3heposs9YP'], 'Perceived likelihood of university entry': ['W1hlikeYP', 'W2hlikeYP',
    'W3hlikeYP'], 'Expected peer post-16 route': ['W1fplan16YP', 'W2fplan16YP',
    'W3fplan16YP'], 'Job-versus-qualification attitude': ['W1plan16YP'], 'Derived post-16 plan': ['W3plan16YP'], 'Parental higher-education expectation': ['W1hepossMP']}
grouped_aspiration_variables = [variable for variables in aspiration_construct_groups.values() for variable in variables]
assert len(grouped_aspiration_variables) == 18
assert len(set(grouped_aspiration_variables)) == 18
assert set(grouped_aspiration_variables) == set(core_aspiration_variables)
aspiration_construct_lookup = {variable: construct for construct,
    variables in aspiration_construct_groups.items() for variable in variables}
core_aspiration_construct_summary = core_aspiration_coverage.copy()
core_aspiration_construct_summary['Construct'] = core_aspiration_construct_summary['Variable'].map(aspiration_construct_lookup)
core_aspiration_construct_summary = core_aspiration_construct_summary[['Construct', 'Source file', 'Variable',
    'Variable label', 'Observed responses', 'Special-code responses', 'No source record', 'Coverage percentage', 'Distinct observed codes']].sort_values(['Construct',
    'Source file', 'Variable']).reset_index(drop=True)
print('Core construct summary:')
display_limited(core_aspiration_construct_summary)
for construct, variables in aspiration_construct_groups.items():
    print('\n' + '=' * 90)
    print(f'Construct: {construct}')
    print('=' * 90)
    for variable in variables:
        variable_summary = core_aspiration_construct_summary.loc[core_aspiration_construct_summary['Variable'].eq(variable)].iloc[0]
        print(f'\nVariable: {variable}')
        print(f"Label: {variable_summary['Variable label']}")
        print(f"Coverage: {int(variable_summary['Observed responses']):,} observed responses ({variable_summary['Coverage percentage']:.2f}%)")
        variable_codes = core_aspiration_code_distribution.loc[core_aspiration_code_distribution['Variable'].eq(variable),
            ['Raw code', 'Value label', 'Response type', 'Participants']].copy().reset_index(drop=True)
        print(variable_codes.to_string(max_rows=TABLE_ROW_LIMIT, index=False))

Core construct summary:


,Construct,Source file,Variable,Variable label,Observed responses,Special-code responses,No source record,Coverage percentage,Distinct observed codes
0,Derived post-16 plan,wave_three_lsype_young_person_2020,W3plan16YP,DV: YP's post 16 plans,9293,216,258,95.15,3
1,Expected peer post-16 route,wave_one_lsype_young_person_2020,W1fplan16YP,YP: What think most of friends will do after Y...,8712,812,243,89.20,3
2,Expected peer post-16 route,wave_three_lsype_young_person_2020,W3fplan16YP,YP: What YP thinks most of friends will do aft...,7586,1923,258,77.67,3
3,Expected peer post-16 route,wave_two_lsype_young_person_2020,W2fplan16YP,YP: What think most of friends will do after Y...,8819,702,246,90.29,3
4,Higher-education application likelihood,wave_one_lsype_young_person_2020,W1heposs9YP,YP: Likelihood of YP applying for university,9054,470,243,92.70,4



Construct: Intention to remain in education

Variable: W1plann16YP
Label: YP: YP's intentions after Year 11
Coverage: 8,970 observed responses (91.84%)
 Raw code                                                  Value label     Response type  Participants
    -99.0                                           YP not interviewed      Special code            89
     -1.0                                                   Don't know      Special code           465
      ...                                                          ...               ...           ...
      3.0 (DO NOT READ OUT) Leave FT education but return later, eg. G Observed response           139
      NaN                                             No source record  No source record           243

Variable: W2plann16YP
Label: YP: YP's intentions after Year 11
Coverage: 9,029 observed responses (92.44%)
 Raw code                                       Value label     Response type  Participants
    -99.0                    

In [253]:
# 7: Repeated aspiration-measure comparison

import pandas as pd

def load_aligned_aspiration_data(source_file, variables):
    source_data = pd.read_stata(source_file_lookup[source_file], columns=['NSID', *variables],
        convert_categoricals=False)
    source_data['NSID'] = standardise_nsid(source_data['NSID'])
    assert source_data['NSID'].is_unique
    source_data = source_data.set_index('NSID').reindex(stage_2_ids)
    for variable in variables:
        source_data[variable] = pd.to_numeric(source_data[variable], errors='coerce')
    return source_data
wave_1_aspiration_data = load_aligned_aspiration_data('wave_one_lsype_young_person_2020', ['W1plann16YP',
    'W1plast16YP', 'W1heposs9YP', 'W1hlikeYP', 'W1fplan16YP', 'W1plan16YP'])
wave_2_aspiration_data = load_aligned_aspiration_data('wave_two_lsype_young_person_2020', ['W2plann16YP',
    'W2plast16YP', 'W2heposs9YP', 'W2hlikeYP', 'W2fplan16YP'])
wave_3_aspiration_data = load_aligned_aspiration_data('wave_three_lsype_young_person_2020', ['W3plann16YP',
    'W3plast16YP', 'W3heposs9YP', 'W3hlikeYP', 'W3fplan16YP', 'W3plan16YP'])
wave_1_parent_aspiration_data = load_aligned_aspiration_data('wave_one_lsype_parental_attitudes_file_16_05_08',
    ['W1hepossMP'])
wave_3_pretransition_eligible = wave_3_interview_year.eq(2006) & wave_3_interview_month.between(1, 8)
print(f'Wave 3 participants with an interview dated before September 2006: {int(wave_3_pretransition_eligible.sum()):,}')
repeated_aspiration_constructs = [{'Construct': 'Post-16 education plan', 'Wave 1 variable': 'W1plann16YP',
    'Wave 2 variable': 'W2plann16YP', 'Wave 3 variable': 'W3plan16YP', 'Valid codes': [1, 2,
    3]}, {'Construct': 'Intended further-education setting', 'Wave 1 variable': 'W1plast16YP',
    'Wave 2 variable': 'W2plast16YP', 'Wave 3 variable': 'W3plast16YP', 'Valid codes': [1, 2, 3, 4,
    5]}, {'Construct': 'Higher-education application likelihood', 'Wave 1 variable': 'W1heposs9YP',
    'Wave 2 variable': 'W2heposs9YP', 'Wave 3 variable': 'W3heposs9YP', 'Valid codes': [1, 2, 3,
    4]}, {'Construct': 'Perceived likelihood of university entry', 'Wave 1 variable': 'W1hlikeYP',
    'Wave 2 variable': 'W2hlikeYP', 'Wave 3 variable': 'W3hlikeYP', 'Valid codes': [1, 2, 3,
    4]}, {'Construct': 'Expected peer post-16 route', 'Wave 1 variable': 'W1fplan16YP',
    'Wave 2 variable': 'W2fplan16YP', 'Wave 3 variable': 'W3fplan16YP', 'Valid codes': [1, 2, 3]}]

def retain_documented_codes(values, valid_codes):
    """Retain documented substantive codes only."""
    return pd.to_numeric(values, errors='coerce').where(pd.to_numeric(values,
        errors='coerce').isin(valid_codes)).astype('Float64')
aspiration_wave_agreement_rows = []
aspiration_latest_source_rows = []
aspiration_candidate_summary_rows = []
aspiration_candidate_series = {}
for construct_specification in repeated_aspiration_constructs:
    construct = construct_specification['Construct']
    valid_codes = construct_specification['Valid codes']
    wave_1_values = retain_documented_codes(wave_1_aspiration_data[construct_specification['Wave 1 variable']],
        valid_codes)
    wave_2_values = retain_documented_codes(wave_2_aspiration_data[construct_specification['Wave 2 variable']],
        valid_codes)
    wave_3_values = retain_documented_codes(wave_3_aspiration_data[construct_specification['Wave 3 variable']],
        valid_codes)
    wave_3_values = wave_3_values.where(wave_3_pretransition_eligible)
    wave_values = {'Wave 1': wave_1_values, 'Wave 2': wave_2_values, 'Wave 3 before September 2006': wave_3_values}
    comparison_pairs = [('Wave 1', 'Wave 2'), ('Wave 1', 'Wave 3 before September 2006'), ('Wave 2',
        'Wave 3 before September 2006')]
    for first_wave, second_wave in comparison_pairs:
        first_values = wave_values[first_wave]
        second_values = wave_values[second_wave]
        complete_comparison = first_values.notna() & second_values.notna()
        complete_count = int(complete_comparison.sum())
        exact_agreement = first_values.loc[complete_comparison].eq(second_values.loc[complete_comparison]).mean() * 100 if complete_count > 0 else pd.NA
        aspiration_wave_agreement_rows.append({'Construct': construct,
            'Comparison': f'{first_wave} versus {second_wave}', 'Complete comparisons': complete_count, 'Exact agreement percentage': round(exact_agreement,
            2) if pd.notna(exact_agreement) else pd.NA})
    latest_values = pd.Series(pd.NA, index=pd.Index(stage_2_ids, name='NSID'), dtype='Float64')
    latest_source = pd.Series(pd.NA, index=latest_values.index, dtype='string')
    wave_3_available = wave_3_values.notna()
    latest_values.loc[wave_3_available] = wave_3_values.loc[wave_3_available]
    latest_source.loc[wave_3_available] = 'Wave 3 before September 2006'
    wave_2_fallback = latest_values.isna() & wave_2_values.notna()
    latest_values.loc[wave_2_fallback] = wave_2_values.loc[wave_2_fallback]
    latest_source.loc[wave_2_fallback] = 'Wave 2 fallback'
    wave_1_fallback = latest_values.isna() & wave_1_values.notna()
    latest_values.loc[wave_1_fallback] = wave_1_values.loc[wave_1_fallback]
    latest_source.loc[wave_1_fallback] = 'Wave 1 fallback'
    aspiration_candidate_series[construct] = latest_values
    source_counts = latest_source.fillna('Unavailable').value_counts()
    for source, participants in source_counts.items():
        aspiration_latest_source_rows.append({'Construct': construct, 'Construction source': source,
            'Participants': int(participants), 'Percentage': round(participants / 9767 * 100, 2)})
    aspiration_candidate_summary_rows.append({'Construct': construct, 'Non-missing': int(latest_values.notna().sum()),
        'Missing': int(latest_values.isna().sum()), 'Missing percentage': round(latest_values.isna().mean() * 100,
        2), 'Distinct values': int(latest_values.nunique(dropna=True))})
wave_3_direct_plan = retain_documented_codes(wave_3_aspiration_data['W3plann16YP'], [1, 2,
    3]).where(wave_3_pretransition_eligible)
wave_3_derived_plan = retain_documented_codes(wave_3_aspiration_data['W3plan16YP'], [1, 2,
    3]).where(wave_3_pretransition_eligible)
wave_3_plan_complete_comparison = wave_3_direct_plan.notna() & wave_3_derived_plan.notna()
wave_3_plan_comparison = pd.DataFrame({'Measure': ['Direct Wave 3 intention', 'Derived Wave 3 post-16 plan',
    'Complete comparison', 'Exact agreement'], 'Participants or percentage': [int(wave_3_direct_plan.notna().sum()),
    int(wave_3_derived_plan.notna().sum()), int(wave_3_plan_complete_comparison.sum()), round(wave_3_direct_plan.loc[wave_3_plan_complete_comparison].eq(wave_3_derived_plan.loc[wave_3_plan_complete_comparison]).mean() * 100,
    2)]})
aspiration_wave_agreement = pd.DataFrame(aspiration_wave_agreement_rows)
aspiration_latest_source_summary = pd.DataFrame(aspiration_latest_source_rows).sort_values(['Construct',
    'Construction source']).reset_index(drop=True)
aspiration_candidate_coverage = pd.DataFrame(aspiration_candidate_summary_rows)
parental_he_expectation = retain_documented_codes(wave_1_parent_aspiration_data['W1hepossMP'], [1, 2, 3, 4])
parental_he_expectation_summary = pd.DataFrame({'Construct': ['Parental higher-education expectation'],
    'Non-missing': [int(parental_he_expectation.notna().sum())], 'Missing': [int(parental_he_expectation.isna().sum())], 'Missing percentage': [round(parental_he_expectation.isna().mean() * 100,
    2)], 'Distinct values': [int(parental_he_expectation.nunique(dropna=True))]})
print('Wave 3 direct and derived post-16 plan comparison:')
display_limited(wave_3_plan_comparison)
print('Repeated-measure exact agreement:')
display_limited(aspiration_wave_agreement)
print('Latest pre-transition source coverage:')
display_limited(aspiration_latest_source_summary)
print('Provisional repeated-construct coverage:')
display_limited(aspiration_candidate_coverage)
print('Separate parental expectation coverage:')
display_limited(parental_he_expectation_summary)

Wave 3 participants with an interview dated before September 2006: 9,367
Wave 3 direct and derived post-16 plan comparison:


,Measure,Participants or percentage
0,Direct Wave 3 intention,7496.0
1,Derived Wave 3 post-16 plan,9154.0
2,Complete comparison,7496.0
3,Exact agreement,100.0


Repeated-measure exact agreement:


,Construct,Comparison,Complete comparisons,Exact agreement percentage
0,Post-16 education plan,Wave 1 versus Wave 2,8583,89.61
1,Post-16 education plan,Wave 1 versus Wave 3 before September 2006,8642,88.26
2,Post-16 education plan,Wave 2 versus Wave 3 before September 2006,8715,90.84
3,Intended further-education setting,Wave 1 versus Wave 2,6824,63.14
4,Intended further-education setting,Wave 1 versus Wave 3 before September 2006,6930,58.70


Latest pre-transition source coverage:


,Construct,Construction source,Participants,Percentage
0,Expected peer post-16 route,Unavailable,308,3.15
1,Expected peer post-16 route,Wave 1 fallback,154,1.58
2,Expected peer post-16 route,Wave 2 fallback,1816,18.59
3,Expected peer post-16 route,Wave 3 before September 2006,7489,76.68
4,Higher-education application likelihood,Unavailable,262,2.68


Provisional repeated-construct coverage:


,Construct,Non-missing,Missing,Missing percentage,Distinct values
0,Post-16 education plan,9506,261,2.67,3
1,Intended further-education setting,9109,658,6.74,5
2,Higher-education application likelihood,9505,262,2.68,4
3,Perceived likelihood of university entry,8921,846,8.66,4
4,Expected peer post-16 route,9459,308,3.15,3


Separate parental expectation coverage:


,Construct,Non-missing,Missing,Missing percentage,Distinct values
0,Parental higher-education expectation,8874,893,9.14,4


In [254]:
# 8: Expected non-education route measure structure

import pandas as pd
expected_route_sources = {'wave_one_lsype_young_person_2020': ['W1pladk2YP'],
    'wave_two_lsype_young_person_2020': ['W2Pladk2YP0a', 'W2Pladk2YP0b', 'W2Pladk2YP0c', 'W2Pladk2YP0d',
    'W2Pladk2YP0e', 'W2Pladk2YP0f'], 'wave_three_lsype_young_person_2020': ['W3pladk2aYP0a', 'W3pladk2aYP0b',
    'W3pladk2aYP0c', 'W3pladk2aYP0d', 'W3pladk2aYP0e', 'W3pladk2aYP0f', 'W3pladk2aYP0g', 'W3pladk2aYP0h', 'W3pladk2bYP0a', 'W3pladk2bYP0b', 'W3pladk2bYP0c', 'W3pladk2bYP0d', 'W3pladk2bYP0e', 'W3pladk2bYP0f', 'W3pladk2bYP0g', 'W3pladk2bYP0h']}
expected_route_variables = [variable for variables in expected_route_sources.values() for variable in variables]
assert len(expected_route_variables) == 23
assert len(set(expected_route_variables)) == 23
expected_route_register_rows = variable_decision_register.loc[variable_decision_register['Variable'].isin(expected_route_variables),
    ['Source order', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status', 'Review outcome']].copy()
assert len(expected_route_register_rows) == 23
for source_file, variables in expected_route_sources.items():
    registered_variables = set(expected_route_register_rows.loc[expected_route_register_rows['Source file'].eq(source_file),
        'Variable'])
    assert registered_variables == set(variables)
expected_route_code_rows = []
expected_route_group_rows = []
expected_route_raw_tables = {}
for source_file, variables in expected_route_sources.items():
    source_path = source_file_lookup[source_file]
    columns_to_read = ['NSID', *variables]
    if source_file == 'wave_three_lsype_young_person_2020':
        columns_to_read.extend(['W3stilschHH', 'W3plan16YP'])
    raw_data = pd.read_stata(source_path, columns=columns_to_read, convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=columns_to_read, convert_categoricals=True)
    raw_data['NSID'] = standardise_nsid(raw_data['NSID'])
    labelled_data['NSID'] = standardise_nsid(labelled_data['NSID'])
    assert raw_data['NSID'].is_unique
    assert labelled_data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(stage_2_ids)
    labelled_data = labelled_data.set_index('NSID').reindex(stage_2_ids)
    for column in raw_data.columns:
        raw_data[column] = pd.to_numeric(raw_data[column], errors='coerce')
    expected_route_raw_tables[source_file] = raw_data
    for variable in variables:
        raw_values = raw_data[variable]
        labelled_values = labelled_data[variable].astype('string')
        variable_label = expected_route_register_rows.loc[expected_route_register_rows['Source file'].eq(source_file) & expected_route_register_rows['Variable'].eq(variable),
            'Variable label'].iloc[0]
        code_counts = raw_values.value_counts(dropna=False).sort_index(na_position='last')
        for code, participants in code_counts.items():
            if pd.isna(code):
                value_label = 'No source record'
                response_type = 'No source record'
            else:
                matching_labels = labelled_values.loc[raw_values.eq(code)].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(code)
                response_type = 'Observed response' if code >= 0 else 'Special code'
            expected_route_code_rows.append({'Wave': expected_route_register_rows.loc[expected_route_register_rows['Variable'].eq(variable),
                'Wave'].iloc[0], 'Source file': source_file, 'Variable': variable, 'Variable label': variable_label, 'Raw code': code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants)})
wave_1_expected_route = expected_route_raw_tables['wave_one_lsype_young_person_2020']['W1pladk2YP']
expected_route_group_rows.append({'Measure group': 'Wave 1 single-choice expected route',
    'Participants with an observed route': int(wave_1_expected_route.ge(0).sum()), 'Participants with at least one option selected': pd.NA, 'Participants routed away or unavailable': int((wave_1_expected_route.lt(0) | wave_1_expected_route.isna()).sum()), 'Confirmed before September 2006': int(wave_1_expected_route.ge(0).sum())})
wave_2_expected_route_columns = expected_route_sources['wave_two_lsype_young_person_2020']
wave_2_expected_route = expected_route_raw_tables['wave_two_lsype_young_person_2020'][wave_2_expected_route_columns]
wave_2_complete_binary_response = wave_2_expected_route.isin([0, 1]).all(axis=1)
wave_2_any_route_selected = wave_2_expected_route.eq(1).any(axis=1)
expected_route_group_rows.append({'Measure group': 'Wave 2 multiple-response expected route',
    'Participants with an observed route': int(wave_2_complete_binary_response.sum()), 'Participants with at least one option selected': int((wave_2_complete_binary_response & wave_2_any_route_selected).sum()), 'Participants routed away or unavailable': int((~wave_2_complete_binary_response).sum()), 'Confirmed before September 2006': int(wave_2_complete_binary_response.sum())})
wave_3_expected_route = expected_route_raw_tables['wave_three_lsype_young_person_2020']
wave_3_form_a_columns = [variable for variable in expected_route_sources['wave_three_lsype_young_person_2020'] if 'pladk2a' in variable.lower()]
wave_3_form_b_columns = [variable for variable in expected_route_sources['wave_three_lsype_young_person_2020'] if 'pladk2b' in variable.lower()]
assert len(wave_3_form_a_columns) == 8
assert len(wave_3_form_b_columns) == 8
wave_3_form_a_complete = wave_3_expected_route[wave_3_form_a_columns].isin([0, 1]).all(axis=1)
wave_3_form_b_complete = wave_3_expected_route[wave_3_form_b_columns].isin([0, 1]).all(axis=1)
wave_3_form_a_selected = wave_3_expected_route[wave_3_form_a_columns].eq(1).any(axis=1)
wave_3_form_b_selected = wave_3_expected_route[wave_3_form_b_columns].eq(1).any(axis=1)
for group_name, complete_response, any_selected in [('Wave 3 form A: expected route after Year 11',
    wave_3_form_a_complete, wave_3_form_a_selected), ('Wave 3 form B: expected activity in 12 months',
    wave_3_form_b_complete, wave_3_form_b_selected)]:
    expected_route_group_rows.append({'Measure group': group_name,
        'Participants with an observed route': int(complete_response.sum()), 'Participants with at least one option selected': int((complete_response & any_selected).sum()), 'Participants routed away or unavailable': int((~complete_response).sum()), 'Confirmed before September 2006': int((complete_response & wave_3_pretransition_eligible).sum())})
wave_3_form_overlap = pd.DataFrame({'Routing pattern': ['Form A only', 'Form B only', 'Both forms', 'Neither form'],
    'Participants': [int((wave_3_form_a_complete & ~wave_3_form_b_complete).sum()),
    int((wave_3_form_b_complete & ~wave_3_form_a_complete).sum()), int((wave_3_form_a_complete & wave_3_form_b_complete).sum()), int((~wave_3_form_a_complete & ~wave_3_form_b_complete).sum())]})
wave_3_year_11_status = wave_3_expected_route['W3stilschHH']
wave_3_route_routing_by_year_11 = pd.DataFrame({'Year 11 status code': wave_3_year_11_status,
    'Form A available': wave_3_form_a_complete, 'Form B available': wave_3_form_b_complete}).value_counts(dropna=False).rename('Participants').reset_index().sort_values(['Year 11 status code',
    'Form A available', 'Form B available'], na_position='last').reset_index(drop=True)
expected_route_code_distribution = pd.DataFrame(expected_route_code_rows).sort_values(['Wave', 'Source file',
    'Variable', 'Raw code'], na_position='last').reset_index(drop=True)
expected_route_group_summary = pd.DataFrame(expected_route_group_rows)
print('Expected-route variables and option labels:')
display_limited(expected_route_register_rows.sort_values(['Wave', 'Source file', 'Variable position'])[['Wave',
    'Source file', 'Variable', 'Variable label']].reset_index(drop=True))
print('Raw-code distributions:')
display_limited(expected_route_code_distribution)
print('Measure-group coverage:')
display_limited(expected_route_group_summary)
print('Wave 3 form overlap:')
display_limited(wave_3_form_overlap)
print('Wave 3 routing by Year 11 status:')
display_limited(wave_3_route_routing_by_year_11)

Expected-route variables and option labels:


,Wave,Source file,Variable,Variable label
0,Wave 1,wave_one_lsype_young_person_2020,W1pladk2YP,YP: What expect to do at age 16 other than fur...
1,Wave 2,wave_two_lsype_young_person_2020,W2Pladk2YP0a,YP: What expect to do at age 16 other than sta...
2,Wave 2,wave_two_lsype_young_person_2020,W2Pladk2YP0b,YP: What expect to do at age 16 other than sta...
3,Wave 2,wave_two_lsype_young_person_2020,W2Pladk2YP0c,YP: What expect to do at age 16 other than sta...
4,Wave 2,wave_two_lsype_young_person_2020,W2Pladk2YP0d,YP: What expect to do at age 16 other than sta...


Raw-code distributions:


,Wave,Source file,Variable,Variable label,Raw code,Value label,Response type,Participants
0,Wave 1,wave_one_lsype_young_person_2020,W1pladk2YP,YP: What expect to do at age 16 other than fur...,-99.0,YP not interviewed,Special code,89
1,Wave 1,wave_one_lsype_young_person_2020,W1pladk2YP,YP: What expect to do at age 16 other than fur...,-91.0,Not applicable,Special code,8598
2,Wave 1,wave_one_lsype_young_person_2020,W1pladk2YP,YP: What expect to do at age 16 other than fur...,-1.0,Don't know,Special code,58
3,Wave 1,wave_one_lsype_young_person_2020,W1pladk2YP,YP: What expect to do at age 16 other than fur...,1.0,Start working full time,Observed response,264
4,Wave 1,wave_one_lsype_young_person_2020,W1pladk2YP,YP: What expect to do at age 16 other than fur...,2.0,Start learning a trade/ start work-based training,Observed response,437


Measure-group coverage:


,Measure group,Participants with an observed route,Participants with at least one option selected,Participants routed away or unavailable,Confirmed before September 2006
0,Wave 1 single-choice expected route,779,<NA>,8988,779
1,Wave 2 multiple-response expected route,877,877,8890,877
2,Wave 3 form A: expected route after Year 11,674,674,9093,662
3,Wave 3 form B: expected activity in 12 months,164,164,9603,159


Wave 3 form overlap:


,Routing pattern,Participants
0,Form A only,674
1,Form B only,164
2,Both forms,0
3,Neither form,8929


Wave 3 routing by Year 11 status:


,Year 11 status code,Form A available,Form B available,Participants
0,-999.0,False,False,6
1,-997.0,False,False,2
2,1.0,False,False,1551
3,1.0,False,True,152
4,2.0,False,False,7112


In [255]:
# 9: Detailed expected-route options and routing validation

import numpy as np
import pandas as pd
wave_1_route_options = expected_route_code_distribution.loc[expected_route_code_distribution['Variable'].eq('W1pladk2YP') & expected_route_code_distribution['Raw code'].ge(0),
    ['Raw code', 'Value label', 'Participants']].copy().sort_values('Raw code').reset_index(drop=True)
wave_2_route_options = expected_route_register_rows.loc[expected_route_register_rows['Wave'].eq('Wave 2'), ['Variable',
    'Variable label']].copy().sort_values('Variable').reset_index(drop=True)
wave_3_form_a_options = expected_route_register_rows.loc[expected_route_register_rows['Variable'].isin(wave_3_form_a_columns),
    ['Variable', 'Variable label']].copy().sort_values('Variable').reset_index(drop=True)
wave_3_form_b_options = expected_route_register_rows.loc[expected_route_register_rows['Variable'].isin(wave_3_form_b_columns),
    ['Variable', 'Variable label']].copy().sort_values('Variable').reset_index(drop=True)
print('Wave 1 single-choice route options:')
print(wave_1_route_options.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 2 multiple-response route options:')
print(wave_2_route_options.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 3 form A route options:')
print(wave_3_form_a_options.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
print('\nWave 3 form B route options:')
print(wave_3_form_b_options.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
wave_3_status_raw = pd.read_stata(source_file_lookup['wave_three_lsype_young_person_2020'], columns=['NSID',
    'W3stilschHH'], convert_categoricals=False)
wave_3_status_labelled = pd.read_stata(source_file_lookup['wave_three_lsype_young_person_2020'], columns=['NSID',
    'W3stilschHH'], convert_categoricals=True)
for table in [wave_3_status_raw, wave_3_status_labelled]:
    table['NSID'] = standardise_nsid(table['NSID'])
    assert table['NSID'].is_unique
wave_3_status_raw = wave_3_status_raw.set_index('NSID').reindex(stage_2_ids)
wave_3_status_labelled = wave_3_status_labelled.set_index('NSID').reindex(stage_2_ids)
wave_3_status_codes = pd.to_numeric(wave_3_status_raw['W3stilschHH'], errors='coerce')
wave_3_status_labels = wave_3_status_labelled['W3stilschHH'].astype('string')
wave_3_status_distribution_rows = []
for code, participants in wave_3_status_codes.value_counts(dropna=False).sort_index(na_position='last').items():
    if pd.isna(code):
        value_label = 'No source record'
    else:
        matching_labels = wave_3_status_labels.loc[wave_3_status_codes.eq(code)].dropna().drop_duplicates().tolist()
        value_label = matching_labels[0] if matching_labels else str(code)
    wave_3_status_distribution_rows.append({'Raw code': code, 'Value label': value_label,
        'Participants': int(participants)})
wave_3_status_distribution = pd.DataFrame(wave_3_status_distribution_rows)
print('\nWave 3 Year 11 status codes:')
print(wave_3_status_distribution.to_string(max_rows=TABLE_ROW_LIMIT, index=False))
wave_1_plan = retain_documented_codes(wave_1_aspiration_data['W1plann16YP'], [1, 2, 3])
wave_1_route = pd.to_numeric(expected_route_raw_tables['wave_one_lsype_young_person_2020']['W1pladk2YP'],
    errors='coerce')
wave_1_route_substantive = wave_1_route.ge(1)
wave_1_route_dont_know = wave_1_route.eq(-1)
wave_1_route_asked = wave_1_route_substantive | wave_1_route_dont_know
wave_1_routing_mismatch = wave_1_plan.eq(2).ne(wave_1_route_asked)
wave_2_plan = retain_documented_codes(wave_2_aspiration_data['W2plann16YP'], [1, 2, 3])
wave_2_route_available = wave_2_complete_binary_response
wave_2_routing_mismatch = wave_2_plan.eq(2).ne(wave_2_route_available)
wave_3_plan = retain_documented_codes(wave_3_aspiration_data['W3plan16YP'], [1, 2, 3])
wave_3_route_available = wave_3_form_a_complete | wave_3_form_b_complete
wave_3_routing_mismatch = wave_3_plan.eq(2).ne(wave_3_route_available)
routing_validation_summary = pd.DataFrame([{'Wave': 'Wave 1',
    'Leaving full-time education': int(wave_1_plan.eq(2).sum()), 'Detailed route available': int(wave_1_route_substantive.sum()), 'Detailed route asked but unknown': int(wave_1_route_dont_know.sum()), 'Routing mismatches': int(wave_1_routing_mismatch.sum())}, {'Wave': 'Wave 2',
    'Leaving full-time education': int(wave_2_plan.eq(2).sum()), 'Detailed route available': int(wave_2_route_available.sum()), 'Detailed route asked but unknown': 0, 'Routing mismatches': int(wave_2_routing_mismatch.sum())}, {'Wave': 'Wave 3',
    'Leaving full-time education': int(wave_3_plan.eq(2).sum()), 'Detailed route available': int(wave_3_route_available.sum()), 'Detailed route asked but unknown': 0, 'Routing mismatches': int(wave_3_routing_mismatch.sum())}])
print('\nRouting validation:')
display_limited(routing_validation_summary)
wave_3_route_form = pd.Series('Neither form', index=pd.Index(stage_2_ids, name='NSID'), dtype='string')
wave_3_route_form.loc[wave_3_form_a_complete & ~wave_3_form_b_complete] = 'Form A'
wave_3_route_form.loc[wave_3_form_b_complete & ~wave_3_form_a_complete] = 'Form B'
wave_3_route_form.loc[wave_3_form_a_complete & wave_3_form_b_complete] = 'Both forms'
wave_3_status_and_form = pd.DataFrame({'Year 11 status code': wave_3_status_codes,
    'Year 11 status label': wave_3_status_labels, 'Route form': wave_3_route_form})
wave_3_status_form_summary = wave_3_status_and_form.value_counts(dropna=False).rename('Participants').reset_index().sort_values(['Year 11 status code',
    'Route form'], na_position='last').reset_index(drop=True)
print('Wave 3 route form by Year 11 status:')
display_limited(wave_3_status_form_summary)

Wave 1 single-choice route options:
 Raw code                                       Value label  Participants
      1.0                           Start working full time           264
      2.0 Start learning a trade/ start work-based training           437
      3.0                            Be unemployed/ sign on            15
      4.0                                    Something else            63

Wave 2 multiple-response route options:
    Variable                                                                   Variable label
W2Pladk2YP0a        YP: What expect to do at age 16 other than stay in FTE - Start working FT
W2Pladk2YP0b YP: What expect to do at age 16 other than stay in FTE - Learn a trade/ work-bas
         ...                                                                              ...
W2Pladk2YP0e          YP: What expect to do at age 16 other than stay in FTE - Something else
W2Pladk2YP0f              YP: What expect to do at age 16 other than stay in FTE - 

,Wave,Leaving full-time education,Detailed route available,Detailed route asked but unknown,Routing mismatches
0,Wave 1,837,779,58,0
1,Wave 2,877,877,0,0
2,Wave 3,838,838,0,0


Wave 3 route form by Year 11 status:


,Year 11 status code,Year 11 status label,Route form,Participants
0,-999.0,HH grid missing,Neither form,6
1,-997.0,Script error,Neither form,2
2,1.0,Yes,Form B,152
3,1.0,Yes,Neither form,1551
4,2.0,No,Form A,674


In [256]:
# 10: Expected-route selection patterns

import pandas as pd
wave_1_route_labels = {1: 'Full-time employment', 2: 'Trade or work-based training', 3: 'Unemployment', 4: 'Other',
    -1: "Don't know"}
wave_2_route_option_names = {'W2Pladk2YP0a': 'Full-time employment', 'W2Pladk2YP0b': 'Trade or work-based training',
    'W2Pladk2YP0c': 'Unemployment', 'W2Pladk2YP0d': 'Part-time education', 'W2Pladk2YP0e': 'Other', 'W2Pladk2YP0f': "Don't know"}
wave_3_form_a_option_names = {'W3pladk2aYP0a': 'Full-time employment', 'W3pladk2aYP0b': 'Trade or work-based training',
    'W3pladk2aYP0c': 'Unemployment', 'W3pladk2aYP0d': 'Part-time education', 'W3pladk2aYP0e': 'Full-time education', 'W3pladk2aYP0f': 'Family care', 'W3pladk2aYP0g': 'Other', 'W3pladk2aYP0h': "Don't know"}
wave_3_form_b_option_names = {'W3pladk2bYP0a': 'Full-time employment', 'W3pladk2bYP0b': 'Trade or work-based training',
    'W3pladk2bYP0c': 'Unemployment', 'W3pladk2bYP0d': 'Part-time education', 'W3pladk2bYP0e': 'Full-time education', 'W3pladk2bYP0f': 'Family care', 'W3pladk2bYP0g': 'Other', 'W3pladk2bYP0h': "Don't know"}
wave_1_route_values = pd.to_numeric(expected_route_raw_tables['wave_one_lsype_young_person_2020']['W1pladk2YP'],
    errors='coerce')
wave_1_route_asked = wave_1_plan.eq(2)
wave_1_route_pattern = wave_1_route_values.loc[wave_1_route_asked].map(wave_1_route_labels).fillna('Unavailable')
wave_1_route_pattern_summary = wave_1_route_pattern.value_counts(dropna=False).rename('Participants').reset_index().rename(columns={'W1pladk2YP': 'Selection pattern',
    'index': 'Selection pattern'})
wave_1_route_pattern_summary['Percentage'] = (wave_1_route_pattern_summary['Participants'] / wave_1_route_asked.sum() * 100).round(2)

def summarise_multiple_response_routes(route_data, option_name_lookup, eligible_mask, wave_label):
    """Summarise complete multiple-response route selections."""
    option_columns = list(option_name_lookup.keys())
    complete_response = route_data[option_columns].isin([0, 1]).all(axis=1) & eligible_mask
    selected_options = route_data.loc[complete_response, option_columns].eq(1)
    selected_options = selected_options.rename(columns=option_name_lookup)
    selection_count = selected_options.sum(axis=1)
    selection_patterns = selected_options.apply(lambda row: ' + '.join(row.index[row].tolist()) if row.any() else 'No option selected',
        axis=1)
    count_summary = selection_count.value_counts().sort_index().rename('Participants').reset_index().rename(columns={'index': 'Options selected'})
    count_summary['Percentage'] = (count_summary['Participants'] / complete_response.sum() * 100).round(2)
    pattern_summary = selection_patterns.value_counts().rename('Participants').reset_index().rename(columns={'index': 'Selection pattern'})
    pattern_summary['Percentage'] = (pattern_summary['Participants'] / complete_response.sum() * 100).round(2)
    option_summary = selected_options.sum().rename('Participants selecting option').reset_index().rename(columns={'index': 'Route option'})
    option_summary['Percentage of complete responses'] = (option_summary['Participants selecting option'] / complete_response.sum() * 100).round(2)
    overview = {'Wave': wave_label, 'Complete routed responses': int(complete_response.sum()),
        'One option selected': int(selection_count.eq(1).sum()), 'Multiple options selected': int(selection_count.gt(1).sum()), 'No option selected': int(selection_count.eq(0).sum()), 'Multiple-selection percentage': round(selection_count.gt(1).mean() * 100,
        2)}
    return {'overview': overview, 'count_summary': count_summary, 'pattern_summary': pattern_summary,
        'option_summary': option_summary}
wave_2_route_results = summarise_multiple_response_routes(route_data=expected_route_raw_tables['wave_two_lsype_young_person_2020'],
    option_name_lookup=wave_2_route_option_names, eligible_mask=wave_2_plan.eq(2), wave_label='Wave 2')
wave_3_route_option_order = ['Full-time employment', 'Trade or work-based training', 'Unemployment',
    'Part-time education', 'Full-time education', 'Family care', 'Other', "Don't know"]
wave_3_combined_route_data = pd.DataFrame(index=pd.Index(stage_2_ids, name='NSID'))
for form_a_variable, form_b_variable, option_name in zip(wave_3_form_a_columns, wave_3_form_b_columns,
    wave_3_route_option_order):
    form_a_values = pd.to_numeric(wave_3_expected_route[form_a_variable], errors='coerce')
    form_b_values = pd.to_numeric(wave_3_expected_route[form_b_variable], errors='coerce')
    combined_values = pd.Series(pd.NA, index=wave_3_combined_route_data.index, dtype='Float64')
    combined_values.loc[wave_3_form_a_complete] = form_a_values.loc[wave_3_form_a_complete]
    combined_values.loc[wave_3_form_b_complete] = form_b_values.loc[wave_3_form_b_complete]
    wave_3_combined_route_data[option_name] = combined_values
wave_3_combined_option_lookup = {option_name: option_name for option_name in wave_3_route_option_order}
wave_3_route_results = summarise_multiple_response_routes(route_data=wave_3_combined_route_data,
    option_name_lookup=wave_3_combined_option_lookup, eligible_mask=wave_3_plan.eq(2) & wave_3_pretransition_eligible, wave_label='Wave 3 before September 2006')
expected_route_selection_overview = pd.DataFrame([{'Wave': 'Wave 1',
    'Complete routed responses': int(wave_1_route_asked.sum()), 'One option selected': int(wave_1_route_pattern.ne("Don't know").sum()), 'Multiple options selected': 0, 'No option selected': int(wave_1_route_pattern.eq("Don't know").sum()), 'Multiple-selection percentage': 0.0}, wave_2_route_results['overview'], wave_3_route_results['overview']])
print('Expected-route selection overview:')
display_limited(expected_route_selection_overview)
print('Wave 1 route patterns:')
display_limited(wave_1_route_pattern_summary)
print('Wave 2 number of options selected:')
display_limited(wave_2_route_results['count_summary'])
print('Wave 2 route-option prevalence:')
display_limited(wave_2_route_results['option_summary'])
print('Wave 2 selection patterns:')
display_limited(wave_2_route_results['pattern_summary'])
print('Wave 3 number of options selected:')
display_limited(wave_3_route_results['count_summary'])
print('Wave 3 route-option prevalence:')
display_limited(wave_3_route_results['option_summary'])
print('Wave 3 selection patterns:')
display_limited(wave_3_route_results['pattern_summary'])

Expected-route selection overview:


,Wave,Complete routed responses,One option selected,Multiple options selected,No option selected,Multiple-selection percentage
0,Wave 1,837,779,0,58,0.00
1,Wave 2,877,830,47,0,5.36
2,Wave 3 before September 2006,821,745,76,0,9.26


Wave 1 route patterns:


,Selection pattern,Participants,Percentage
0,Trade or work-based training,437,52.21
1,Full-time employment,264,31.54
2,Other,63,7.53
3,Don't know,58,6.93
4,Unemployment,15,1.79


Wave 2 number of options selected:


...

,Options selected,Participants,Percentage
0,1,745,90.74
1,2,70,8.53
2,3,6,0.73


Wave 3 route-option prevalence:


,Route option,Participants selecting option,Percentage of complete responses
0,Full-time employment,281,34.23
1,Trade or work-based training,498,60.66
2,Unemployment,4,0.49
3,Part-time education,38,4.63
4,Full-time education,8,0.97


Wave 3 selection patterns:


,Selection pattern,Participants,Percentage
0,Trade or work-based training,430,52.38
1,Full-time employment,231,28.14
2,Full-time employment + Trade or work-based tra...,40,4.87
3,Other,33,4.02
4,Don't know,31,3.78


In [257]:
# 11: Harmonised expected post-16 route review

import numpy as np
import pandas as pd
route_indicator_names = ['Full-time education', 'Full-time employment', 'Trade or work-based training', 'Unemployment',
    'Part-time education', 'Family care', 'Other', 'Leave education and return later', "Don't know"]
substantive_route_names = [route_name for route_name in route_indicator_names if route_name != "Don't know"]
route_index = pd.Index(stage_2_ids, name='NSID')

def initialise_route_indicators():
    """Create an empty harmonised route-indicator table."""
    return pd.DataFrame(pd.NA, index=route_index, columns=route_indicator_names, dtype='Float64')
wave_1_route_indicators = initialise_route_indicators()
wave_1_detailed_route_available = wave_1_route_values.isin([-1, 1, 2, 3, 4]).fillna(False)
wave_1_route_complete = (wave_1_plan.isin([1, 3]) | wave_1_plan.eq(2) & wave_1_detailed_route_available).fillna(False)
wave_1_route_indicators.loc[wave_1_route_complete, :] = 0
wave_1_route_indicators.loc[wave_1_route_complete & wave_1_plan.eq(1).fillna(False), 'Full-time education'] = 1
wave_1_route_indicators.loc[wave_1_route_complete & wave_1_plan.eq(3).fillna(False),
    'Leave education and return later'] = 1
wave_1_route_code_mapping = {1: 'Full-time employment', 2: 'Trade or work-based training', 3: 'Unemployment',
    4: 'Other', -1: "Don't know"}
for route_code, route_name in wave_1_route_code_mapping.items():
    selected_route = wave_1_route_complete & wave_1_plan.eq(2).fillna(False) & wave_1_route_values.eq(route_code).fillna(False)
    wave_1_route_indicators.loc[selected_route, route_name] = 1
wave_2_route_indicators = initialise_route_indicators()
wave_2_route_data = expected_route_raw_tables['wave_two_lsype_young_person_2020']
wave_2_route_complete = (wave_2_plan.isin([1, 3]) | wave_2_plan.eq(2) & wave_2_complete_binary_response).fillna(False)
wave_2_route_indicators.loc[wave_2_route_complete, :] = 0
wave_2_route_indicators.loc[wave_2_route_complete & wave_2_plan.eq(1).fillna(False), 'Full-time education'] = 1
wave_2_route_indicators.loc[wave_2_route_complete & wave_2_plan.eq(3).fillna(False),
    'Leave education and return later'] = 1
for variable, route_name in wave_2_route_option_names.items():
    selected_route = wave_2_route_complete & wave_2_plan.eq(2).fillna(False) & wave_2_route_data[variable].eq(1).fillna(False)
    wave_2_route_indicators.loc[selected_route, route_name] = 1
wave_3_route_indicators = initialise_route_indicators()
wave_3_route_complete = (wave_3_pretransition_eligible & (wave_3_plan.isin([1,
    3]) | wave_3_plan.eq(2) & wave_3_route_available)).fillna(False)
wave_3_route_indicators.loc[wave_3_route_complete, :] = 0
wave_3_route_indicators.loc[wave_3_route_complete & wave_3_plan.eq(1).fillna(False), 'Full-time education'] = 1
wave_3_route_indicators.loc[wave_3_route_complete & wave_3_plan.eq(3).fillna(False),
    'Leave education and return later'] = 1
for route_name in wave_3_route_option_order:
    selected_route = wave_3_route_complete & wave_3_plan.eq(2).fillna(False) & wave_3_combined_route_data[route_name].eq(1).fillna(False)
    wave_3_route_indicators.loc[selected_route, route_name] = 1
latest_route_indicators = initialise_route_indicators()
latest_route_source = pd.Series(pd.NA, index=route_index, dtype='string', name='Construction source')
wave_3_available = wave_3_route_complete
latest_route_indicators.loc[wave_3_available, :] = wave_3_route_indicators.loc[wave_3_available, :]
latest_route_source.loc[wave_3_available] = 'Wave 3 before September 2006'
wave_2_fallback = latest_route_source.isna() & wave_2_route_complete
latest_route_indicators.loc[wave_2_fallback, :] = wave_2_route_indicators.loc[wave_2_fallback, :]
latest_route_source.loc[wave_2_fallback] = 'Wave 2 fallback'
wave_1_fallback = latest_route_source.isna() & wave_1_route_complete
latest_route_indicators.loc[wave_1_fallback, :] = wave_1_route_indicators.loc[wave_1_fallback, :]
latest_route_source.loc[wave_1_fallback] = 'Wave 1 fallback'
latest_route_available = latest_route_source.notna()
assert latest_route_indicators.loc[latest_route_available].isin([0, 1]).all().all()
assert latest_route_indicators.loc[~latest_route_available].isna().all().all()
latest_substantive_route_count = latest_route_indicators[substantive_route_names].sum(axis=1,
    min_count=1).astype('Float64')
latest_dont_know = latest_route_indicators["Don't know"]
latest_route_profile = pd.Series(pd.NA, index=route_index, dtype='string', name='Expected post-16 route profile')
single_substantive_route = latest_route_available & latest_substantive_route_count.eq(1).fillna(False)
for route_name in substantive_route_names:
    latest_route_profile.loc[single_substantive_route & latest_route_indicators[route_name].eq(1).fillna(False)] = route_name
latest_route_profile.loc[latest_route_available & latest_substantive_route_count.gt(1).fillna(False)] = 'Multiple route expectations'
latest_route_profile.loc[latest_route_available & latest_substantive_route_count.eq(0).fillna(False) & latest_dont_know.eq(1).fillna(False)] = "Don't know"
latest_route_profile.loc[latest_route_available & latest_substantive_route_count.eq(0).fillna(False) & latest_dont_know.eq(0).fillna(False)] = 'No route selected'
latest_route_pattern = pd.Series(pd.NA, index=route_index, dtype='string', name='Detailed route pattern')
for participant_id in latest_route_indicators.index[latest_route_available]:
    participant_routes = latest_route_indicators.loc[participant_id, substantive_route_names]
    selected_routes = participant_routes.index[participant_routes.eq(1)].tolist()
    if selected_routes:
        latest_route_pattern.loc[participant_id] = ' + '.join(selected_routes)
    elif latest_dont_know.loc[participant_id] == 1:
        latest_route_pattern.loc[participant_id] = "Don't know"
    else:
        latest_route_pattern.loc[participant_id] = 'No route selected'
latest_route_source_summary = latest_route_source.fillna('Unavailable').value_counts().rename('Participants').rename_axis('Construction source').reset_index()
latest_route_source_summary['Percentage'] = (latest_route_source_summary['Participants'] / len(route_index) * 100).round(2)
latest_route_profile_summary = latest_route_profile.fillna('Unavailable').value_counts().rename('Participants').rename_axis('Expected post-16 route profile').reset_index()
latest_route_profile_summary['Percentage'] = (latest_route_profile_summary['Participants'] / len(route_index) * 100).round(2)
latest_route_count_summary = latest_substantive_route_count.fillna(-1).astype('Int64').value_counts().sort_index().rename_axis('Substantive route count code').reset_index(name='Participants')
route_count_label_lookup = {-1: 'Unavailable', 0: 'None', 1: 'One', 2: 'Two', 3: 'Three'}
latest_route_count_summary['Substantive routes selected'] = latest_route_count_summary['Substantive route count code'].astype('int64').map(route_count_label_lookup)
unmapped_route_count = latest_route_count_summary['Substantive routes selected'].isna()
latest_route_count_summary.loc[unmapped_route_count,
    'Substantive routes selected'] = latest_route_count_summary.loc[unmapped_route_count,
    'Substantive route count code'].astype('string') + ' routes'
latest_route_count_summary['Percentage'] = (latest_route_count_summary['Participants'] / len(route_index) * 100).round(2)
latest_route_count_summary = latest_route_count_summary[['Substantive routes selected', 'Participants', 'Percentage']]
route_indicator_summary_rows = []
for route_name in route_indicator_names:
    route_indicator_summary_rows.append({'Route indicator': route_name,
        'Participants with representation': int(latest_route_available.sum()), 'Indicator selected': int(latest_route_indicators.loc[latest_route_available,
        route_name].eq(1).sum()), 'Percentage among represented participants': round(latest_route_indicators.loc[latest_route_available,
        route_name].eq(1).mean() * 100, 2)})
route_indicator_summary = pd.DataFrame(route_indicator_summary_rows)
dont_know_with_substantive_route = latest_route_available & latest_dont_know.eq(1).fillna(False) & latest_substantive_route_count.gt(0).fillna(False)
multiple_route_mask = latest_substantive_route_count.gt(1).fillna(False)
multiple_route_pattern_summary = latest_route_pattern.loc[multiple_route_mask].value_counts().rename('Participants').rename_axis('Detailed route pattern').reset_index()
if not multiple_route_pattern_summary.empty:
    multiple_route_pattern_summary['Percentage of multiple-route cases'] = (multiple_route_pattern_summary['Participants'] / multiple_route_pattern_summary['Participants'].sum() * 100).round(2)
else:
    multiple_route_pattern_summary['Percentage of multiple-route cases'] = pd.Series(dtype='float64')
print(f'Participants with a harmonised expected-route representation: {int(latest_route_available.sum()):,}')
print(f'Participants unavailable: {int((~latest_route_available).sum()):,}')
print(f"Don't know selected alongside a substantive route: {int(dont_know_with_substantive_route.sum()):,}")
print('Construction source:')
display_limited(latest_route_source_summary)
print('Broad expected-route profile:')
display_limited(latest_route_profile_summary)
print('Number of substantive routes selected:')
display_limited(latest_route_count_summary)
print('Route-indicator prevalence:')
display_limited(route_indicator_summary)
print('Multiple-route patterns:')
display_limited(multiple_route_pattern_summary)

Participants with a harmonised expected-route representation: 9,506
Participants unavailable: 261
Don't know selected alongside a substantive route: 0
Construction source:


,Construction source,Participants,Percentage
0,Wave 3 before September 2006,9154,93.72
1,Wave 2 fallback,314,3.21
2,Unavailable,261,2.67
3,Wave 1 fallback,38,0.39


Broad expected-route profile:


,Expected post-16 route profile,Participants,Percentage
0,Full-time education,8587,87.92
1,Trade or work-based training,461,4.72
2,Unavailable,261,2.67
3,Full-time employment,257,2.63
4,Multiple route expectations,80,0.82


Number of substantive routes selected:


,Substantive routes selected,Participants,Percentage
0,Unavailable,261,2.67
1,None,33,0.34
2,One,9393,96.17
3,Two,74,0.76
4,Three,6,0.06


Route-indicator prevalence:


,Route indicator,Participants with representation,Indicator selected,Percentage among represented participants
0,Full-time education,9506,8589,90.35
1,Full-time employment,9506,310,3.26
2,Trade or work-based training,9506,533,5.61
3,Unemployment,9506,4,0.04
4,Part-time education,9506,42,0.44


Multiple-route patterns:


,Detailed route pattern,Participants,Percentage of multiple-route cases
0,Full-time employment + Trade or work-based tra...,43,53.75
1,Trade or work-based training + Part-time educa...,20,25.0
2,Full-time employment + Trade or work-based tra...,4,5.0
3,Full-time education + Trade or work-based trai...,2,2.5
4,Full-time employment + Part-time education,2,2.5


In [258]:
# 12: Source-consistent education-setting review

import pandas as pd
education_setting_code_labels = {1: 'Sixth form at the same school', 2: 'Sixth form at a different school',
    3: 'Sixth-form college', 4: 'Further-education college', 5: 'Other college'}

def retain_education_setting(values):
    """Retain documented education-setting codes."""
    numeric_values = pd.to_numeric(values, errors='coerce')
    return numeric_values.where(numeric_values.isin(education_setting_code_labels)).astype('Float64')
wave_1_education_setting = retain_education_setting(wave_1_aspiration_data['W1plast16YP'])
wave_2_education_setting = retain_education_setting(wave_2_aspiration_data['W2plast16YP'])
wave_3_education_setting = retain_education_setting(wave_3_aspiration_data['W3plast16YP']).where(wave_3_pretransition_eligible)
education_setting_wave_rows = []
wave_setting_review_inputs = [{'Wave': 'Wave 1', 'Route complete': wave_1_route_complete,
    'Education indicator': wave_1_route_indicators['Full-time education'].eq(1).fillna(False), 'Setting': wave_1_education_setting}, {'Wave': 'Wave 2',
    'Route complete': wave_2_route_complete, 'Education indicator': wave_2_route_indicators['Full-time education'].eq(1).fillna(False), 'Setting': wave_2_education_setting}, {'Wave': 'Wave 3 before September 2006',
    'Route complete': wave_3_route_complete, 'Education indicator': wave_3_route_indicators['Full-time education'].eq(1).fillna(False), 'Setting': wave_3_education_setting}]
for review_input in wave_setting_review_inputs:
    education_plan = review_input['Route complete'] & review_input['Education indicator']
    setting_available = review_input['Setting'].notna()
    education_setting_wave_rows.append({'Wave': review_input['Wave'],
        'Complete route representations': int(review_input['Route complete'].sum()), 'Full-time education indicated': int(education_plan.sum()), 'Education setting available': int((education_plan & setting_available).sum()), 'Education setting unavailable': int((education_plan & ~setting_available).sum()), 'Setting coverage among education plans': round((education_plan & setting_available).sum() / education_plan.sum() * 100,
        2), 'Setting observed outside education plan': int((review_input['Route complete'] & ~review_input['Education indicator'] & setting_available).sum())})
education_setting_wave_summary = pd.DataFrame(education_setting_wave_rows)
latest_source_consistent_setting = pd.Series(pd.NA, index=route_index, dtype='Float64',
    name='Source-consistent education setting')
latest_source_consistent_setting.loc[latest_route_source.eq('Wave 3 before September 2006')] = wave_3_education_setting.loc[latest_route_source.eq('Wave 3 before September 2006')]
latest_source_consistent_setting.loc[latest_route_source.eq('Wave 2 fallback')] = wave_2_education_setting.loc[latest_route_source.eq('Wave 2 fallback')]
latest_source_consistent_setting.loc[latest_route_source.eq('Wave 1 fallback')] = wave_1_education_setting.loc[latest_route_source.eq('Wave 1 fallback')]
latest_full_time_education = latest_route_indicators['Full-time education'].eq(1).fillna(False)
latest_setting_available = latest_source_consistent_setting.notna()
latest_education_setting_label = latest_source_consistent_setting.map(education_setting_code_labels).astype('string')
latest_education_setting_summary = latest_education_setting_label.loc[latest_full_time_education].fillna('Full-time education, setting unavailable').value_counts().rename('Participants').rename_axis('Education setting').reset_index()
latest_education_setting_summary['Percentage of full-time education cases'] = (latest_education_setting_summary['Participants'] / latest_full_time_education.sum() * 100).round(2)
diagnostic_expected_route = pd.Series(pd.NA, index=route_index, dtype='string', name='Diagnostic expected route')
for setting_code, setting_label in education_setting_code_labels.items():
    diagnostic_expected_route.loc[latest_full_time_education & latest_source_consistent_setting.eq(setting_code).fillna(False) & latest_substantive_route_count.eq(1).fillna(False)] = setting_label
diagnostic_expected_route.loc[latest_full_time_education & ~latest_setting_available & latest_substantive_route_count.eq(1).fillna(False)] = 'Full-time education, setting unavailable'
diagnostic_expected_route.loc[latest_route_profile.eq('Full-time employment')] = 'Full-time employment'
diagnostic_expected_route.loc[latest_route_profile.eq('Trade or work-based training')] = 'Trade or work-based training'
diagnostic_expected_route.loc[latest_route_profile.eq('Leave education and return later')] = 'Leave education and return later'
diagnostic_expected_route.loc[latest_route_profile.eq("Don't know")] = "Don't know"
employment_and_training_only = latest_substantive_route_count.eq(2).fillna(False) & latest_route_indicators['Full-time employment'].eq(1).fillna(False) & latest_route_indicators['Trade or work-based training'].eq(1).fillna(False)
other_route_selected = latest_route_indicators[['Unemployment', 'Part-time education', 'Family care',
    'Other']].eq(1).any(axis=1)
employment_training_with_no_other_route = employment_and_training_only & ~other_route_selected & ~latest_full_time_education
diagnostic_expected_route.loc[employment_training_with_no_other_route] = 'Employment and work-based training'
remaining_available_route = latest_route_available & diagnostic_expected_route.isna()
diagnostic_expected_route.loc[remaining_available_route] = 'Other or mixed route expectation'
diagnostic_expected_route_summary = diagnostic_expected_route.fillna('Unavailable').value_counts().rename('Participants').rename_axis('Diagnostic expected route').reset_index()
diagnostic_expected_route_summary['Percentage'] = (diagnostic_expected_route_summary['Participants'] / len(route_index) * 100).round(2)
latest_route_setting_check = pd.DataFrame({'Broad route profile': latest_route_profile.fillna('Unavailable'),
    'Source-consistent setting available': latest_setting_available})
latest_route_setting_cross_tabulation = latest_route_setting_check.value_counts(dropna=False).rename('Participants').reset_index().sort_values(['Broad route profile',
    'Source-consistent setting available']).reset_index(drop=True)
print('Education-setting availability within each wave:')
display_limited(education_setting_wave_summary)
print('Latest source-consistent education-setting distribution:')
display_limited(latest_education_setting_summary)
print('Diagnostic unified expected-route distribution:')
display_limited(diagnostic_expected_route_summary)
print('Broad route profile and setting availability:')
display_limited(latest_route_setting_cross_tabulation)

Education-setting availability within each wave:


,Wave,Complete route representations,Full-time education indicated,Education setting available,Education setting unavailable,Setting coverage among education plans,Setting observed outside education plan
0,Wave 1,8970,7994,7722,272,96.60,0
1,Wave 2,9029,8091,7807,284,96.49,0
2,Wave 3 before September 2006,9154,8315,8151,164,98.03,0


Latest source-consistent education-setting distribution:


,Education setting,Participants,Percentage of full-time education cases
0,Sixth form at the same school,3520,40.98
1,Further-education college,2265,26.37
2,Sixth-form college,1840,21.42
3,Sixth form at a different school,527,6.14
4,Other college,265,3.09


Diagnostic unified expected-route distribution:


,Diagnostic expected route,Participants,Percentage
0,Sixth form at the same school,3520,36.04
1,Further-education college,2265,23.19
2,Sixth-form college,1840,18.84
3,Sixth form at a different school,527,5.4
4,Trade or work-based training,461,4.72


Broad route profile and setting availability:


,Broad route profile,Source-consistent setting available,Participants
0,Don't know,False,33
1,Family care,False,1
2,Full-time education,False,170
3,Full-time education,True,8417
4,Full-time employment,False,257


In [259]:
# 13: Higher-education expectation routing review

import pandas as pd
he_application_code_labels = {1: 'Very likely', 2: 'Fairly likely', 3: 'Not very likely', 4: 'Not at all likely',
    -1: "Don't know", -99: 'YP not interviewed'}
he_entry_code_labels = {1: 'Very likely', 2: 'Fairly likely', 3: 'Not very likely', 4: 'Not at all likely',
    -1: "Don't know", -91: 'Not applicable', -99: 'YP not interviewed'}
he_expectation_wave_inputs = [{'Wave': 'Wave 1', 'Application': wave_1_aspiration_data['W1heposs9YP'],
    'Entry': wave_1_aspiration_data['W1hlikeYP'], 'Eligible': pd.Series(True, index=route_index)}, {'Wave': 'Wave 2',
    'Application': wave_2_aspiration_data['W2heposs9YP'], 'Entry': wave_2_aspiration_data['W2hlikeYP'], 'Eligible': pd.Series(True,
    index=route_index)}, {'Wave': 'Wave 3 before September 2006', 'Application': wave_3_aspiration_data['W3heposs9YP'],
    'Entry': wave_3_aspiration_data['W3hlikeYP'], 'Eligible': wave_3_pretransition_eligible}]
he_expectation_coverage_rows = []
he_expectation_routing_tables = {}
for wave_input in he_expectation_wave_inputs:
    wave_name = wave_input['Wave']
    application_values = pd.to_numeric(wave_input['Application'], errors='coerce')
    entry_values = pd.to_numeric(wave_input['Entry'], errors='coerce')
    eligible = wave_input['Eligible'].fillna(False)
    application_observed = application_values.isin([1, 2, 3, 4]) & eligible
    entry_observed = entry_values.isin([1, 2, 3, 4]) & eligible
    entry_not_applicable = entry_values.eq(-91) & eligible
    entry_dont_know = entry_values.eq(-1) & eligible
    he_expectation_coverage_rows.append({'Wave': wave_name, 'Eligible participants': int(eligible.sum()),
        'Application likelihood observed': int(application_observed.sum()), 'Entry likelihood observed': int(entry_observed.sum()), 'Entry likelihood not applicable': int(entry_not_applicable.sum()), "Entry likelihood don't know": int(entry_dont_know.sum()), 'Entry observed among application responses': round((entry_observed & application_observed).sum() / application_observed.sum() * 100,
        2)})
    application_label = application_values.map(he_application_code_labels).fillna('No source record or other special code')
    entry_status = pd.Series('No source record or other special code', index=route_index, dtype='string')
    entry_status.loc[entry_observed] = 'Observed entry likelihood'
    entry_status.loc[entry_not_applicable] = 'Not applicable'
    entry_status.loc[entry_dont_know] = "Don't know"
    routing_table = pd.crosstab(application_label.loc[eligible], entry_status.loc[eligible], dropna=False)
    routing_table = routing_table.reindex(['Very likely', 'Fairly likely', 'Not very likely', 'Not at all likely',
        "Don't know", 'YP not interviewed', 'No source record or other special code'], fill_value=0)
    he_expectation_routing_tables[wave_name] = routing_table
he_expectation_coverage_summary = pd.DataFrame(he_expectation_coverage_rows)
latest_he_application = pd.Series(pd.NA, index=route_index, dtype='Float64', name='Latest HE application likelihood')
latest_he_entry = pd.Series(pd.NA, index=route_index, dtype='Float64', name='Source-consistent HE entry likelihood')
latest_he_entry_status = pd.Series(pd.NA, index=route_index, dtype='string', name='HE entry likelihood status')
latest_he_source = pd.Series(pd.NA, index=route_index, dtype='string', name='Construction source')

def add_he_expectation_source(application_values, entry_values, eligible_mask, source_label):
    """Add one source where no later application measure is available."""
    application_numeric = pd.to_numeric(application_values, errors='coerce')
    entry_numeric = pd.to_numeric(entry_values, errors='coerce')
    application_available = application_numeric.isin([1, 2, 3,
        4]) & eligible_mask.fillna(False) & latest_he_source.isna()
    latest_he_application.loc[application_available] = application_numeric.loc[application_available]
    latest_he_source.loc[application_available] = source_label
    entry_observed = application_available & entry_numeric.isin([1, 2, 3, 4])
    entry_not_applicable = application_available & entry_numeric.eq(-91)
    entry_dont_know = application_available & entry_numeric.eq(-1)
    latest_he_entry.loc[entry_observed] = entry_numeric.loc[entry_observed]
    latest_he_entry_status.loc[entry_observed] = 'Observed entry likelihood'
    latest_he_entry_status.loc[entry_not_applicable] = 'Not applicable'
    latest_he_entry_status.loc[entry_dont_know] = "Don't know"
    other_entry_status = application_available & latest_he_entry_status.isna()
    latest_he_entry_status.loc[other_entry_status] = 'Unavailable or other special code'
add_he_expectation_source(application_values=wave_3_aspiration_data['W3heposs9YP'],
    entry_values=wave_3_aspiration_data['W3hlikeYP'], eligible_mask=wave_3_pretransition_eligible, source_label='Wave 3 before September 2006')
add_he_expectation_source(application_values=wave_2_aspiration_data['W2heposs9YP'],
    entry_values=wave_2_aspiration_data['W2hlikeYP'], eligible_mask=pd.Series(True,
    index=route_index), source_label='Wave 2 fallback')
add_he_expectation_source(application_values=wave_1_aspiration_data['W1heposs9YP'],
    entry_values=wave_1_aspiration_data['W1hlikeYP'], eligible_mask=pd.Series(True,
    index=route_index), source_label='Wave 1 fallback')
latest_he_source_summary = latest_he_source.fillna('Unavailable').value_counts().rename('Participants').rename_axis('Construction source').reset_index()
latest_he_source_summary['Percentage'] = (latest_he_source_summary['Participants'] / len(route_index) * 100).round(2)
latest_he_entry_status_summary = latest_he_entry_status.fillna('Application likelihood unavailable').value_counts().rename('Participants').rename_axis('HE entry likelihood status').reset_index()
latest_he_entry_status_summary['Percentage'] = (latest_he_entry_status_summary['Participants'] / len(route_index) * 100).round(2)
latest_he_routing_review = pd.crosstab(latest_he_application.map({1: 'Very likely', 2: 'Fairly likely',
    3: 'Not very likely', 4: 'Not at all likely'}).fillna('Application likelihood unavailable'), latest_he_entry_status.fillna('Application likelihood unavailable'), dropna=False)
print('Wave-specific HE expectation coverage:')
display_limited(he_expectation_coverage_summary)
for wave_name, routing_table in he_expectation_routing_tables.items():
    print(f'{wave_name} application likelihood by entry-likelihood status:')
    display_limited(routing_table)
print('Latest application-likelihood source:')
display_limited(latest_he_source_summary)
print('Latest entry-likelihood status:')
display_limited(latest_he_entry_status_summary)
print('Latest application likelihood by source-consistent entry-likelihood status:')
display_limited(latest_he_routing_review)

Wave-specific HE expectation coverage:


,Wave,Eligible participants,Application likelihood observed,Entry likelihood observed,Entry likelihood not applicable,Entry likelihood don't know,Entry observed among application responses
0,Wave 1,9767,9054,7724,1189,522,85.31
1,Wave 2,9767,9076,7569,1377,499,83.40
2,Wave 3 before September 2006,9367,8998,7273,1609,431,80.83


Wave 1 application likelihood by entry-likelihood status:


col_0,Don't know,No source record or other special code,Not applicable,Observed entry likelihood
W1heposs9YP,,,,
Very likely,126,0,0,3388
Fairly likely,235,0,0,3119
Not very likely,161,0,0,1217
Not at all likely,0,0,808,0
Don't know,0,0,381,0


Wave 2 application likelihood by entry-likelihood status:


...

,Construction source,Participants,Percentage
0,Wave 3 before September 2006,8998,92.13
1,Wave 2 fallback,440,4.5
2,Unavailable,262,2.68
3,Wave 1 fallback,67,0.69


Latest entry-likelihood status:


,HE entry likelihood status,Participants,Percentage
0,Observed entry likelihood,7669,78.52
1,Not applicable,1355,13.87
2,Don't know,481,4.92
3,Application likelihood unavailable,262,2.68


Latest application likelihood by source-consistent entry-likelihood status:


HE entry likelihood status,Application likelihood unavailable,Don't know,Not applicable,Observed entry likelihood
Latest HE application likelihood,,,,
Application likelihood unavailable,262,0,0,0
Fairly likely,0,226,0,2446
Not at all likely,0,0,1355,0
Not very likely,0,165,0,1284
Very likely,0,90,0,3939


In [260]:
# 14: Young-person aspiration candidate-set review

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
young_person_aspiration_candidates = pd.DataFrame(index=route_index)
young_person_aspiration_candidates['expected_post16_route'] = diagnostic_expected_route
young_person_aspiration_candidates['higher_education_application_likelihood'] = latest_he_application.astype('Int64')
young_person_aspiration_candidates['expected_peer_post16_route'] = aspiration_candidate_series['Expected peer post-16 route'].astype('Int64')
he_application_label_lookup = {1: 'Very likely', 2: 'Fairly likely', 3: 'Not very likely', 4: 'Not at all likely'}
peer_route_label_lookup = {1: 'Most friends expected to remain in education',
    2: 'Most friends expected to leave education', 3: 'Most friends expected to do something else'}
young_person_aspiration_review = pd.DataFrame(index=route_index)
young_person_aspiration_review['Expected post-16 route'] = young_person_aspiration_candidates['expected_post16_route']
young_person_aspiration_review['HE application likelihood'] = young_person_aspiration_candidates['higher_education_application_likelihood'].map(he_application_label_lookup).astype('string')
young_person_aspiration_review['Expected peer post-16 route'] = young_person_aspiration_candidates['expected_peer_post16_route'].map(peer_route_label_lookup).astype('string')
wave_1_peer_route = retain_documented_codes(wave_1_aspiration_data['W1fplan16YP'], [1, 2, 3])
wave_2_peer_route = retain_documented_codes(wave_2_aspiration_data['W2fplan16YP'], [1, 2, 3])
wave_3_peer_route = retain_documented_codes(wave_3_aspiration_data['W3fplan16YP'], [1, 2,
    3]).where(wave_3_pretransition_eligible)
latest_peer_route_check = pd.Series(pd.NA, index=route_index, dtype='Float64')
latest_peer_route_source = pd.Series(pd.NA, index=route_index, dtype='string', name='Construction source')
wave_3_peer_available = wave_3_peer_route.notna()
latest_peer_route_check.loc[wave_3_peer_available] = wave_3_peer_route.loc[wave_3_peer_available]
latest_peer_route_source.loc[wave_3_peer_available] = 'Wave 3 before September 2006'
wave_2_peer_fallback = latest_peer_route_check.isna() & wave_2_peer_route.notna()
latest_peer_route_check.loc[wave_2_peer_fallback] = wave_2_peer_route.loc[wave_2_peer_fallback]
latest_peer_route_source.loc[wave_2_peer_fallback] = 'Wave 2 fallback'
wave_1_peer_fallback = latest_peer_route_check.isna() & wave_1_peer_route.notna()
latest_peer_route_check.loc[wave_1_peer_fallback] = wave_1_peer_route.loc[wave_1_peer_fallback]
latest_peer_route_source.loc[wave_1_peer_fallback] = 'Wave 1 fallback'
assert latest_peer_route_check.astype('Int64').equals(young_person_aspiration_candidates['expected_peer_post16_route'])
aspiration_candidate_quality_rows = []
for predictor in young_person_aspiration_candidates.columns:
    predictor_values = young_person_aspiration_candidates[predictor]
    aspiration_candidate_quality_rows.append({'Predictor': predictor,
        'Non-missing': int(predictor_values.notna().sum()), 'Missing': int(predictor_values.isna().sum()), 'Missing percentage': round(predictor_values.isna().mean() * 100,
        2), 'Distinct non-missing values': int(predictor_values.nunique(dropna=True))})
aspiration_candidate_quality = pd.DataFrame(aspiration_candidate_quality_rows)
aspiration_predictors_available = young_person_aspiration_candidates.notna().sum(axis=1)
aspiration_joint_availability = aspiration_predictors_available.value_counts().sort_index().rename('Participants').rename_axis('Aspiration predictors available').reset_index()
aspiration_joint_availability['Percentage'] = (aspiration_joint_availability['Participants'] / len(route_index) * 100).round(2)
peer_route_source_summary = latest_peer_route_source.fillna('Unavailable').value_counts().rename('Participants').rename_axis('Construction source').reset_index()
peer_route_source_summary['Percentage'] = (peer_route_source_summary['Participants'] / len(route_index) * 100).round(2)
aspiration_distribution_tables = {}
for predictor in young_person_aspiration_review.columns:
    distribution = young_person_aspiration_review[predictor].fillna('Unavailable').value_counts().rename('Participants').rename_axis(predictor).reset_index()
    distribution['Percentage'] = (distribution['Participants'] / len(route_index) * 100).round(2)
    aspiration_distribution_tables[predictor] = distribution

def corrected_cramers_v(first_variable, second_variable):
    """Calculate bias-corrected Cramer's V."""
    complete_cases = first_variable.notna() & second_variable.notna()
    contingency_table = pd.crosstab(first_variable.loc[complete_cases], second_variable.loc[complete_cases])
    participant_count = int(contingency_table.to_numpy().sum())
    if participant_count == 0 or min(contingency_table.shape) < 2:
        return {'Complete comparisons': participant_count, "Cramer's V": pd.NA}
    chi_square = chi2_contingency(contingency_table, correction=False)[0]
    phi_squared = chi_square / participant_count
    rows, columns = contingency_table.shape
    corrected_phi_squared = max(0, phi_squared - (columns - 1) * (rows - 1) / (participant_count - 1))
    corrected_rows = rows - (rows - 1) ** 2 / (participant_count - 1)
    corrected_columns = columns - (columns - 1) ** 2 / (participant_count - 1)
    denominator = min(corrected_rows - 1, corrected_columns - 1)
    cramers_v = np.sqrt(corrected_phi_squared / denominator) if denominator > 0 else pd.NA
    return {'Complete comparisons': participant_count, "Cramer's V": round(cramers_v,
        3) if pd.notna(cramers_v) else pd.NA}
aspiration_association_pairs = [('Expected post-16 route', 'HE application likelihood'), ('Expected post-16 route',
    'Expected peer post-16 route'), ('HE application likelihood', 'Expected peer post-16 route')]
aspiration_association_rows = []
for first_predictor, second_predictor in aspiration_association_pairs:
    association_result = corrected_cramers_v(young_person_aspiration_review[first_predictor],
        young_person_aspiration_review[second_predictor])
    aspiration_association_rows.append({'First predictor': first_predictor, 'Second predictor': second_predictor,
        **association_result})
aspiration_candidate_associations = pd.DataFrame(aspiration_association_rows)
print('Young-person aspiration candidate quality:')
display_limited(aspiration_candidate_quality)
print('Joint candidate availability:')
display_limited(aspiration_joint_availability)
print('Expected peer-route construction source:')
display_limited(peer_route_source_summary)
for predictor, distribution in aspiration_distribution_tables.items():
    print(f'{predictor} distribution:')
    display_limited(distribution)
print('Candidate associations for redundancy review:')
display_limited(aspiration_candidate_associations)

Young-person aspiration candidate quality:


,Predictor,Non-missing,Missing,Missing percentage,Distinct non-missing values
0,expected_post16_route,9506,261,2.67,12
1,higher_education_application_likelihood,9505,262,2.68,4
2,expected_peer_post16_route,9459,308,3.15,3


Joint candidate availability:


,Aspiration predictors available,Participants,Percentage
0,0,246,2.52
1,1,7,0.07
2,2,79,0.81
3,3,9435,96.60


Expected peer-route construction source:


...

,HE application likelihood,Participants,Percentage
0,Very likely,4029,41.25
1,Fairly likely,2672,27.36
2,Not very likely,1449,14.84
3,Not at all likely,1355,13.87
4,Unavailable,262,2.68


Expected peer post-16 route distribution:


,Expected peer post-16 route,Participants,Percentage
0,Most friends expected to remain in education,8383,85.83
1,Most friends expected to leave education,914,9.36
2,Unavailable,308,3.15
3,Most friends expected to do something else,162,1.66


Candidate associations for redundancy review:


,First predictor,Second predictor,Complete comparisons,Cramer's V
0,Expected post-16 route,HE application likelihood,9491,0.354
1,Expected post-16 route,Expected peer post-16 route,9448,0.256
2,HE application likelihood,Expected peer post-16 route,9445,0.208


In [261]:
# 15: Preferred non-education route measure structure

import pandas as pd
preferred_route_sources = {'wave_one_lsype_young_person_2020': ['W1pladk16YP'],
    'wave_two_lsype_young_person_2020': ['W2Pladk16YP0a', 'W2Pladk16YP0b', 'W2Pladk16YP0c', 'W2Pladk16YP0d',
    'W2Pladk16YP0e', 'W2Pladk16YP0f'], 'wave_three_lsype_young_person_2020': ['W3pladk16aYP0a', 'W3pladk16aYP0b',
    'W3pladk16aYP0c', 'W3pladk16aYP0d', 'W3pladk16aYP0e', 'W3pladk16aYP0f', 'W3pladk16aYP0g', 'W3pladk16aYP0h', 'W3pladk16bYP0a', 'W3pladk16bYP0b', 'W3pladk16bYP0c', 'W3pladk16bYP0d', 'W3pladk16bYP0e', 'W3pladk16bYP0f', 'W3pladk16bYP0g', 'W3pladk16bYP0h']}
preferred_route_variables = [variable for variables in preferred_route_sources.values() for variable in variables]
assert len(preferred_route_variables) == 23
assert len(set(preferred_route_variables)) == 23
preferred_route_register_rows = variable_decision_register.loc[variable_decision_register['Variable'].isin(preferred_route_variables),
    ['Source order', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status', 'Review outcome']].copy()
assert len(preferred_route_register_rows) == 23
for source_file, variables in preferred_route_sources.items():
    registered_variables = set(preferred_route_register_rows.loc[preferred_route_register_rows['Source file'].eq(source_file),
        'Variable'])
    assert registered_variables == set(variables)
preferred_route_raw_tables = {}
preferred_route_code_rows = []
for source_file, variables in preferred_route_sources.items():
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *variables], convert_categoricals=True)
    raw_data['NSID'] = standardise_nsid(raw_data['NSID'])
    labelled_data['NSID'] = standardise_nsid(labelled_data['NSID'])
    assert raw_data['NSID'].is_unique
    assert labelled_data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(stage_2_ids)
    labelled_data = labelled_data.set_index('NSID').reindex(stage_2_ids)
    for variable in variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    preferred_route_raw_tables[source_file] = raw_data
    for variable in variables:
        raw_values = raw_data[variable]
        labelled_values = labelled_data[variable].astype('string')
        variable_register_row = preferred_route_register_rows.loc[preferred_route_register_rows['Source file'].eq(source_file) & preferred_route_register_rows['Variable'].eq(variable)].iloc[0]
        code_counts = raw_values.value_counts(dropna=False).sort_index(na_position='last')
        for code, participants in code_counts.items():
            if pd.isna(code):
                value_label = 'No source record'
                response_type = 'No source record'
            else:
                matching_labels = labelled_values.loc[raw_values.eq(code)].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(code)
                response_type = 'Observed response' if code >= 0 else 'Special code'
            preferred_route_code_rows.append({'Wave': variable_register_row['Wave'], 'Source file': source_file,
                'Variable': variable, 'Variable label': variable_register_row['Variable label'], 'Raw code': code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants)})
preferred_route_code_distribution = pd.DataFrame(preferred_route_code_rows).sort_values(['Wave', 'Source file',
    'Variable', 'Raw code'], na_position='last').reset_index(drop=True)
wave_2_preferred_route_columns = preferred_route_sources['wave_two_lsype_young_person_2020']
wave_2_preferred_route_data = preferred_route_raw_tables['wave_two_lsype_young_person_2020'][wave_2_preferred_route_columns]
wave_2_preferred_complete = wave_2_preferred_route_data.isin([0, 1]).all(axis=1)
wave_2_preferred_selected = wave_2_preferred_route_data.eq(1).any(axis=1)
wave_3_preferred_form_a_columns = [variable for variable in preferred_route_sources['wave_three_lsype_young_person_2020'] if 'pladk16a' in variable.lower()]
wave_3_preferred_form_b_columns = [variable for variable in preferred_route_sources['wave_three_lsype_young_person_2020'] if 'pladk16b' in variable.lower()]
assert len(wave_3_preferred_form_a_columns) == 8
assert len(wave_3_preferred_form_b_columns) == 8
wave_3_preferred_route_data = preferred_route_raw_tables['wave_three_lsype_young_person_2020']
wave_3_preferred_form_a_complete = wave_3_preferred_route_data[wave_3_preferred_form_a_columns].isin([0,
    1]).all(axis=1)
wave_3_preferred_form_b_complete = wave_3_preferred_route_data[wave_3_preferred_form_b_columns].isin([0,
    1]).all(axis=1)
wave_3_preferred_form_a_selected = wave_3_preferred_route_data[wave_3_preferred_form_a_columns].eq(1).any(axis=1)
wave_3_preferred_form_b_selected = wave_3_preferred_route_data[wave_3_preferred_form_b_columns].eq(1).any(axis=1)
wave_1_preferred_route = pd.to_numeric(preferred_route_raw_tables['wave_one_lsype_young_person_2020']['W1pladk16YP'],
    errors='coerce')
wave_1_preferred_asked = wave_1_preferred_route.isin([-1, 1, 2, 3, 4])
preferred_route_group_summary = pd.DataFrame([{'Measure group': 'Wave 1 single-choice preferred route',
    'Complete routed responses': int(wave_1_preferred_asked.sum()), 'At least one substantive option selected': int(wave_1_preferred_route.isin([1,
    2, 3, 4]).sum()), "Don't know": int(wave_1_preferred_route.eq(-1).sum()), 'Confirmed before September 2006': int(wave_1_preferred_asked.sum())}, {'Measure group': 'Wave 2 multiple-response preferred route',
    'Complete routed responses': int(wave_2_preferred_complete.sum()), 'At least one substantive option selected': int((wave_2_preferred_complete & wave_2_preferred_selected).sum()), "Don't know": int((wave_2_preferred_complete & wave_2_preferred_route_data['W2Pladk16YP0f'].eq(1)).sum()), 'Confirmed before September 2006': int(wave_2_preferred_complete.sum())}, {'Measure group': 'Wave 3 form A preferred route',
    'Complete routed responses': int(wave_3_preferred_form_a_complete.sum()), 'At least one substantive option selected': int((wave_3_preferred_form_a_complete & wave_3_preferred_form_a_selected).sum()), "Don't know": int((wave_3_preferred_form_a_complete & wave_3_preferred_route_data['W3pladk16aYP0h'].eq(1)).sum()), 'Confirmed before September 2006': int((wave_3_preferred_form_a_complete & wave_3_pretransition_eligible).sum())}, {'Measure group': 'Wave 3 form B preferred route',
    'Complete routed responses': int(wave_3_preferred_form_b_complete.sum()), 'At least one substantive option selected': int((wave_3_preferred_form_b_complete & wave_3_preferred_form_b_selected).sum()), "Don't know": int((wave_3_preferred_form_b_complete & wave_3_preferred_route_data['W3pladk16bYP0h'].eq(1)).sum()), 'Confirmed before September 2006': int((wave_3_preferred_form_b_complete & wave_3_pretransition_eligible).sum())}])
wave_3_preferred_form_overlap = pd.DataFrame({'Routing pattern': ['Form A only', 'Form B only', 'Both forms',
    'Neither form'], 'Participants': [int((wave_3_preferred_form_a_complete & ~wave_3_preferred_form_b_complete).sum()),
    int((wave_3_preferred_form_b_complete & ~wave_3_preferred_form_a_complete).sum()), int((wave_3_preferred_form_a_complete & wave_3_preferred_form_b_complete).sum()), int((~wave_3_preferred_form_a_complete & ~wave_3_preferred_form_b_complete).sum())]})
preferred_route_routing_summary = pd.concat([pd.DataFrame({'Wave': 'Wave 1', 'Post-16 plan code': wave_1_plan,
    'Preferred route available': wave_1_preferred_asked}), pd.DataFrame({'Wave': 'Wave 2',
    'Post-16 plan code': wave_2_plan, 'Preferred route available': wave_2_preferred_complete}), pd.DataFrame({'Wave': 'Wave 3',
    'Post-16 plan code': wave_3_plan, 'Preferred route available': wave_3_preferred_form_a_complete | wave_3_preferred_form_b_complete})], ignore_index=True)
preferred_route_routing_summary = preferred_route_routing_summary.value_counts(dropna=False).rename('Participants').reset_index().sort_values(['Wave',
    'Post-16 plan code', 'Preferred route available'], na_position='last').reset_index(drop=True)
print('Preferred-route variables and option labels:')
display_limited(preferred_route_register_rows.sort_values(['Wave', 'Source file', 'Variable position'])[['Wave',
    'Variable', 'Variable label']].reset_index(drop=True))
print('Raw-code distributions:')
display_limited(preferred_route_code_distribution)
print('Measure-group coverage:')
display_limited(preferred_route_group_summary)
print('Wave 3 preferred-route form overlap:')
display_limited(wave_3_preferred_form_overlap)
print('Preferred-route availability by post-16 plan:')
display_limited(preferred_route_routing_summary)

Preferred-route variables and option labels:


,Wave,Variable,Variable label
0,Wave 1,W1pladk16YP,YP: What want to do at age 16 other than furth...
1,Wave 2,W2Pladk16YP0a,YP: What want to do at age 16 other than stay ...
2,Wave 2,W2Pladk16YP0b,YP: What want to do at age 16 other than stay ...
3,Wave 2,W2Pladk16YP0c,YP: What want to do at age 16 other than stay ...
4,Wave 2,W2Pladk16YP0d,YP: What want to do at age 16 other than stay ...


Raw-code distributions:


,Wave,Source file,Variable,Variable label,Raw code,Value label,Response type,Participants
0,Wave 1,wave_one_lsype_young_person_2020,W1pladk16YP,YP: What want to do at age 16 other than furth...,-99.0,YP not interviewed,Special code,89
1,Wave 1,wave_one_lsype_young_person_2020,W1pladk16YP,YP: What want to do at age 16 other than furth...,-91.0,Not applicable,Special code,8598
2,Wave 1,wave_one_lsype_young_person_2020,W1pladk16YP,YP: What want to do at age 16 other than furth...,-1.0,Don't know,Special code,27
3,Wave 1,wave_one_lsype_young_person_2020,W1pladk16YP,YP: What want to do at age 16 other than furth...,1.0,To start working full time,Observed response,298
4,Wave 1,wave_one_lsype_young_person_2020,W1pladk16YP,YP: What want to do at age 16 other than furth...,2.0,Start learning a trade/ start work-based training,Observed response,425


Measure-group coverage:


,Measure group,Complete routed responses,At least one substantive option selected,Don't know,Confirmed before September 2006
0,Wave 1 single-choice preferred route,837,810,27,837
1,Wave 2 multiple-response preferred route,877,877,14,877
2,Wave 3 form A preferred route,674,674,4,662
3,Wave 3 form B preferred route,164,164,3,159


Wave 3 preferred-route form overlap:


,Routing pattern,Participants
0,Form A only,674
1,Form B only,164
2,Both forms,0
3,Neither form,8929


Preferred-route availability by post-16 plan:


,Wave,Post-16 plan code,Preferred route available,Participants
0,Wave 1,1.0,False,7994
1,Wave 1,2.0,True,837
2,Wave 1,3.0,False,139
3,Wave 1,<NA>,False,797
4,Wave 2,1.0,False,8091


In [262]:
# 16: Expected and preferred post-16 route comparison

import pandas as pd
wave_1_preferred_route_indicators = initialise_route_indicators()
wave_1_preferred_route_complete = (wave_1_plan.isin([1,
    3]) | wave_1_plan.eq(2) & wave_1_preferred_asked).fillna(False)
wave_1_preferred_route_indicators.loc[wave_1_preferred_route_complete, :] = 0
wave_1_preferred_route_indicators.loc[wave_1_preferred_route_complete & wave_1_plan.eq(1).fillna(False),
    'Full-time education'] = 1
wave_1_preferred_route_indicators.loc[wave_1_preferred_route_complete & wave_1_plan.eq(3).fillna(False),
    'Leave education and return later'] = 1
wave_1_preferred_route_mapping = {1: 'Full-time employment', 2: 'Trade or work-based training', 3: 'Unemployment',
    4: 'Other', -1: "Don't know"}
for route_code, route_name in wave_1_preferred_route_mapping.items():
    selected_route = wave_1_preferred_route_complete & wave_1_plan.eq(2).fillna(False) & wave_1_preferred_route.eq(route_code).fillna(False)
    wave_1_preferred_route_indicators.loc[selected_route, route_name] = 1
wave_2_preferred_route_indicators = initialise_route_indicators()
wave_2_preferred_route_complete = (wave_2_plan.isin([1,
    3]) | wave_2_plan.eq(2) & wave_2_preferred_complete).fillna(False)
wave_2_preferred_route_indicators.loc[wave_2_preferred_route_complete, :] = 0
wave_2_preferred_route_indicators.loc[wave_2_preferred_route_complete & wave_2_plan.eq(1).fillna(False),
    'Full-time education'] = 1
wave_2_preferred_route_indicators.loc[wave_2_preferred_route_complete & wave_2_plan.eq(3).fillna(False),
    'Leave education and return later'] = 1
wave_2_preferred_option_names = {'W2Pladk16YP0a': 'Full-time employment',
    'W2Pladk16YP0b': 'Trade or work-based training', 'W2Pladk16YP0c': 'Unemployment', 'W2Pladk16YP0d': 'Part-time education', 'W2Pladk16YP0e': 'Other', 'W2Pladk16YP0f': "Don't know"}
for variable, route_name in wave_2_preferred_option_names.items():
    selected_route = wave_2_preferred_route_complete & wave_2_plan.eq(2).fillna(False) & wave_2_preferred_route_data[variable].eq(1).fillna(False)
    wave_2_preferred_route_indicators.loc[selected_route, route_name] = 1
wave_3_preferred_combined_data = pd.DataFrame(pd.NA, index=route_index, columns=wave_3_route_option_order,
    dtype='Float64')
for form_a_variable, form_b_variable, route_name in zip(wave_3_preferred_form_a_columns,
    wave_3_preferred_form_b_columns, wave_3_route_option_order):
    form_a_values = pd.to_numeric(wave_3_preferred_route_data[form_a_variable], errors='coerce')
    form_b_values = pd.to_numeric(wave_3_preferred_route_data[form_b_variable], errors='coerce')
    wave_3_preferred_combined_data.loc[wave_3_preferred_form_a_complete,
        route_name] = form_a_values.loc[wave_3_preferred_form_a_complete]
    wave_3_preferred_combined_data.loc[wave_3_preferred_form_b_complete,
        route_name] = form_b_values.loc[wave_3_preferred_form_b_complete]
wave_3_preferred_route_indicators = initialise_route_indicators()
wave_3_preferred_route_available = wave_3_preferred_form_a_complete | wave_3_preferred_form_b_complete
wave_3_preferred_route_complete = (wave_3_pretransition_eligible & (wave_3_plan.isin([1,
    3]) | wave_3_plan.eq(2) & wave_3_preferred_route_available)).fillna(False)
wave_3_preferred_route_indicators.loc[wave_3_preferred_route_complete, :] = 0
wave_3_preferred_route_indicators.loc[wave_3_preferred_route_complete & wave_3_plan.eq(1).fillna(False),
    'Full-time education'] = 1
wave_3_preferred_route_indicators.loc[wave_3_preferred_route_complete & wave_3_plan.eq(3).fillna(False),
    'Leave education and return later'] = 1
for route_name in wave_3_route_option_order:
    selected_route = wave_3_preferred_route_complete & wave_3_plan.eq(2).fillna(False) & wave_3_preferred_combined_data[route_name].eq(1).fillna(False)
    wave_3_preferred_route_indicators.loc[selected_route, route_name] = 1
assert wave_1_route_complete.equals(wave_1_preferred_route_complete)
assert wave_2_route_complete.equals(wave_2_preferred_route_complete)
assert wave_3_route_complete.equals(wave_3_preferred_route_complete)
latest_preferred_route_indicators = initialise_route_indicators()
wave_3_latest_source = latest_route_source.eq('Wave 3 before September 2006')
latest_preferred_route_indicators.loc[wave_3_latest_source,
    :] = wave_3_preferred_route_indicators.loc[wave_3_latest_source, :]
wave_2_latest_source = latest_route_source.eq('Wave 2 fallback')
latest_preferred_route_indicators.loc[wave_2_latest_source,
    :] = wave_2_preferred_route_indicators.loc[wave_2_latest_source, :]
wave_1_latest_source = latest_route_source.eq('Wave 1 fallback')
latest_preferred_route_indicators.loc[wave_1_latest_source,
    :] = wave_1_preferred_route_indicators.loc[wave_1_latest_source, :]
latest_preferred_route_available = latest_preferred_route_indicators.notna().all(axis=1)
assert latest_preferred_route_available.equals(latest_route_available)
assert latest_preferred_route_indicators.loc[latest_preferred_route_available].isin([0, 1]).all().all()
latest_post16_plan_code = pd.Series(pd.NA, index=route_index, dtype='Float64', name='Latest post-16 plan code')
latest_post16_plan_code.loc[wave_3_latest_source] = wave_3_plan.loc[wave_3_latest_source]
latest_post16_plan_code.loc[wave_2_latest_source] = wave_2_plan.loc[wave_2_latest_source]
latest_post16_plan_code.loc[wave_1_latest_source] = wave_1_plan.loc[wave_1_latest_source]
leaving_education_plan = latest_post16_plan_code.eq(2).fillna(False)
comparison_available = latest_route_available & latest_preferred_route_available
expected_comparison_data = latest_route_indicators.loc[comparison_available, route_indicator_names].astype('int64')
preferred_comparison_data = latest_preferred_route_indicators.loc[comparison_available,
    route_indicator_names].astype('int64')
exact_indicator_set_agreement = expected_comparison_data.eq(preferred_comparison_data).all(axis=1)
leaving_comparison_mask = comparison_available & leaving_education_plan
leaving_comparison_ids = route_index[leaving_comparison_mask]
leaving_exact_agreement = exact_indicator_set_agreement.reindex(leaving_comparison_ids)
expected_preferred_agreement_summary = pd.DataFrame([{'Comparison group': 'All represented participants',
    'Participants': int(comparison_available.sum()), 'Exact indicator-set agreement': int(exact_indicator_set_agreement.sum()), 'Exact agreement percentage': round(exact_indicator_set_agreement.mean() * 100,
    2)}, {'Comparison group': 'Participants planning to leave full-time education',
    'Participants': int(len(leaving_comparison_ids)), 'Exact indicator-set agreement': int(leaving_exact_agreement.sum()), 'Exact agreement percentage': round(leaving_exact_agreement.mean() * 100,
    2)}])
route_option_comparison_rows = []
for route_name in wave_3_route_option_order:
    expected_selected = latest_route_indicators.loc[leaving_comparison_ids, route_name].eq(1)
    preferred_selected = latest_preferred_route_indicators.loc[leaving_comparison_ids, route_name].eq(1)
    route_option_comparison_rows.append({'Route option': route_name, 'Expected selected': int(expected_selected.sum()),
        'Preferred selected': int(preferred_selected.sum()), 'Selected in both': int((expected_selected & preferred_selected).sum()), 'Expected only': int((expected_selected & ~preferred_selected).sum()), 'Preferred only': int((~expected_selected & preferred_selected).sum()), 'Neither selected': int((~expected_selected & ~preferred_selected).sum()), 'Indicator agreement percentage': round(expected_selected.eq(preferred_selected).mean() * 100,
        2)})
expected_preferred_option_comparison = pd.DataFrame(route_option_comparison_rows)

def create_route_pattern(indicator_table, available_mask):
    """Create a readable route-selection pattern."""
    route_pattern = pd.Series(pd.NA, index=route_index, dtype='string')
    for participant_id in route_index[available_mask]:
        participant_indicators = indicator_table.loc[participant_id, substantive_route_names]
        selected_names = participant_indicators.index[participant_indicators.eq(1)].tolist()
        if selected_names:
            route_pattern.loc[participant_id] = ' + '.join(selected_names)
        elif indicator_table.loc[participant_id, "Don't know"] == 1:
            route_pattern.loc[participant_id] = "Don't know"
        else:
            route_pattern.loc[participant_id] = 'No route selected'
    return route_pattern
latest_expected_route_pattern = create_route_pattern(latest_route_indicators, latest_route_available)
latest_preferred_route_pattern = create_route_pattern(latest_preferred_route_indicators,
    latest_preferred_route_available)
expected_preferred_pattern_comparison = pd.DataFrame({'Expected route pattern': latest_expected_route_pattern.loc[leaving_comparison_ids],
    'Preferred route pattern': latest_preferred_route_pattern.loc[leaving_comparison_ids]}).value_counts(dropna=False).rename('Participants').reset_index()
expected_preferred_pattern_comparison['Percentage of leaving-education plans'] = (expected_preferred_pattern_comparison['Participants'] / len(leaving_comparison_ids) * 100).round(2)
agreement_by_source = pd.DataFrame({'Construction source': latest_route_source.loc[comparison_available],
    'Exact agreement': exact_indicator_set_agreement, 'Leaving education plan': leaving_education_plan.loc[comparison_available]})
agreement_by_source_summary = agreement_by_source.groupby(['Construction source', 'Leaving education plan'],
    dropna=False).agg(Participants=('Exact agreement', 'size'), Exact_indicator_set_agreement=('Exact agreement',
    'sum'), Exact_agreement_percentage=('Exact agreement', lambda values: round(values.mean() * 100,
    2))).reset_index().rename(columns={'Leaving education plan': 'Plan to leave full-time education',
    'Exact_indicator_set_agreement': 'Exact indicator-set agreement', 'Exact_agreement_percentage': 'Exact agreement percentage'})
print('Expected-preferred route agreement:')
display_limited(expected_preferred_agreement_summary)
print('Route-option comparison among participants planning to leave full-time education:')
display_limited(expected_preferred_option_comparison)
print('Expected-preferred agreement by source:')
display_limited(agreement_by_source_summary)
print('Expected and preferred route-pattern combinations:')
display_limited(expected_preferred_pattern_comparison)

Expected-preferred route agreement:


,Comparison group,Participants,Exact indicator-set agreement,Exact agreement percentage
0,All represented participants,9506,9410,98.99
1,Participants planning to leave full-time educa...,891,795,89.23


Route-option comparison among participants planning to leave full-time education:


,Route option,Expected selected,Preferred selected,Selected in both,Expected only,Preferred only,Neither selected,Indicator agreement percentage
0,Full-time employment,310,331,292,18,39,542,93.60
1,Trade or work-based training,533,546,516,17,30,328,94.73
2,Unemployment,4,0,0,4,0,887,99.55
3,Part-time education,42,45,37,5,8,841,98.54
4,Full-time education,8,5,5,3,0,883,99.66


Expected-preferred agreement by source:


,Construction source,Plan to leave full-time education,Participants,Exact indicator-set agreement,Exact agreement percentage
0,Wave 1 fallback,False,26,26,100.00
1,Wave 1 fallback,True,12,11,91.67
2,Wave 2 fallback,False,256,256,100.00
3,Wave 2 fallback,True,58,51,87.93
4,Wave 3 before September 2006,False,8333,8333,100.00


Expected and preferred route-pattern combinations:


,Expected route pattern,Preferred route pattern,Participants,Percentage of leaving-education plans
0,Trade or work-based training,Trade or work-based training,435,48.82
1,Full-time employment,Full-time employment,239,26.82
2,Full-time employment + Trade or work-based tra...,Full-time employment + Trade or work-based tra...,39,4.38
3,Other,Other,35,3.93
4,Trade or work-based training + Part-time educa...,Trade or work-based training + Part-time educa...,19,2.13


In [263]:
# 17: Expected post-16 route category consolidation

import pandas as pd
expected_route_category_mapping = {'Sixth form at the same school': 'School sixth form',
    'Sixth form at a different school': 'School sixth form', 'Sixth-form college': 'Sixth-form college', 'Further-education college': 'Further-education college', 'Other college': 'Other or unspecified full-time education', 'Full-time education, setting unavailable': 'Other or unspecified full-time education', 'Full-time employment': 'Full-time employment', 'Trade or work-based training': 'Work-based training or employment-training', 'Employment and work-based training': 'Work-based training or employment-training', 'Other or mixed route expectation': 'Other, mixed or uncertain route', 'Leave education and return later': 'Other, mixed or uncertain route', "Don't know": 'Other, mixed or uncertain route'}
expected_post16_route = diagnostic_expected_route.map(expected_route_category_mapping).astype('string').rename('expected_post16_route')
unmapped_available_route = diagnostic_expected_route.notna() & expected_post16_route.isna()
assert int(unmapped_available_route.sum()) == 0
assert int(expected_post16_route.notna().sum()) == int(diagnostic_expected_route.notna().sum())
assert int(expected_post16_route.notna().sum()) == 9506
expected_route_mapping_review = pd.DataFrame({'Detailed expected route': diagnostic_expected_route,
    'Consolidated expected route': expected_post16_route}).dropna(subset=['Detailed expected route']).value_counts(dropna=False).rename('Participants').reset_index().sort_values(['Consolidated expected route',
    'Participants'], ascending=[True, False]).reset_index(drop=True)
expected_post16_route_summary = expected_post16_route.fillna('Unavailable').value_counts().rename('Participants').rename_axis('Expected post-16 route').reset_index()
expected_post16_route_summary['Percentage'] = (expected_post16_route_summary['Participants'] / len(route_index) * 100).round(2)
educational_aspiration_candidates = pd.DataFrame({'expected_post16_route': expected_post16_route,
    'higher_education_application_likelihood': latest_he_application.astype('Int64'), 'expected_peer_post16_route': latest_peer_route_check.astype('Int64')}, index=route_index)
educational_aspiration_quality_rows = []
for predictor in educational_aspiration_candidates.columns:
    predictor_values = educational_aspiration_candidates[predictor]
    educational_aspiration_quality_rows.append({'Predictor': predictor,
        'Non-missing': int(predictor_values.notna().sum()), 'Missing': int(predictor_values.isna().sum()), 'Missing percentage': round(predictor_values.isna().mean() * 100,
        2), 'Distinct non-missing values': int(predictor_values.nunique(dropna=True))})
educational_aspiration_quality = pd.DataFrame(educational_aspiration_quality_rows)
educational_aspiration_available_count = educational_aspiration_candidates.notna().sum(axis=1)
educational_aspiration_joint_availability = educational_aspiration_available_count.value_counts().sort_index().rename('Participants').rename_axis('Aspiration predictors available').reset_index()
educational_aspiration_joint_availability['Percentage'] = (educational_aspiration_joint_availability['Participants'] / len(route_index) * 100).round(2)
educational_aspiration_association_data = pd.DataFrame(index=route_index)
educational_aspiration_association_data['Expected post-16 route'] = expected_post16_route
educational_aspiration_association_data['HE application likelihood'] = latest_he_application.map(he_application_label_lookup).astype('string')
educational_aspiration_association_data['Expected peer post-16 route'] = latest_peer_route_check.map(peer_route_label_lookup).astype('string')
consolidated_association_pairs = [('Expected post-16 route', 'HE application likelihood'), ('Expected post-16 route',
    'Expected peer post-16 route'), ('HE application likelihood', 'Expected peer post-16 route')]
consolidated_association_rows = []
for first_predictor, second_predictor in consolidated_association_pairs:
    association_result = corrected_cramers_v(educational_aspiration_association_data[first_predictor],
        educational_aspiration_association_data[second_predictor])
    consolidated_association_rows.append({'First predictor': first_predictor, 'Second predictor': second_predictor,
        **association_result})
educational_aspiration_associations = pd.DataFrame(consolidated_association_rows)
educational_aspiration_representation_review = pd.DataFrame([{'Construct': 'Expected post-16 route',
    'Provisional representation': 'Retain as seven-category predictor', 'Reason': 'Combines education setting and non-education route without using outcome information'}, {'Construct': 'Higher-education application likelihood',
    'Provisional representation': 'Retain as four-category ordinal predictor', 'Reason': 'Direct HE expectation measure with high pre-transition coverage'}, {'Construct': 'Expected peer post-16 route',
    'Provisional representation': 'Retain as three-category predictor', 'Reason': "Measures perceived peer route rather than the young person's own plan"}, {'Construct': 'Preferred post-16 route',
    'Provisional representation': 'Retain as review support only', 'Reason': 'Conditionally observed and largely overlaps with expected route'}, {'Construct': 'University entry likelihood if applied',
    'Provisional representation': 'Retain as review support only', 'Reason': 'Conditionally routed from HE application likelihood'}])
print('Detailed-to-consolidated route mapping:')
display_limited(expected_route_mapping_review)
print('Consolidated expected-route distribution:')
display_limited(expected_post16_route_summary)
print('Provisional predictor quality:')
display_limited(educational_aspiration_quality)
print('Joint predictor availability:')
display_limited(educational_aspiration_joint_availability)
print('Predictor associations after route consolidation:')
display_limited(educational_aspiration_associations)
print('Provisional representation review:')
display_limited(educational_aspiration_representation_review)

Detailed-to-consolidated route mapping:


,Detailed expected route,Consolidated expected route,Participants
0,Full-time employment,Full-time employment,257
1,Further-education college,Further-education college,2265
2,Other college,Other or unspecified full-time education,265
3,"Full-time education, setting unavailable",Other or unspecified full-time education,170
4,Other or mixed route expectation,"Other, mixed or uncertain route",91


Consolidated expected-route distribution:


,Expected post-16 route,Participants,Percentage
0,School sixth form,4047,41.44
1,Further-education college,2265,23.19
2,Sixth-form college,1840,18.84
3,Work-based training or employment-training,504,5.16
4,Other or unspecified full-time education,435,4.45


Provisional predictor quality:


...

,Aspiration predictors available,Participants,Percentage
0,0,246,2.52
1,1,7,0.07
2,2,79,0.81
3,3,9435,96.60


Predictor associations after route consolidation:


,First predictor,Second predictor,Complete comparisons,Cramer's V
0,Expected post-16 route,HE application likelihood,9491,0.346
1,Expected post-16 route,Expected peer post-16 route,9448,0.249
2,HE application likelihood,Expected peer post-16 route,9445,0.208


Provisional representation review:


,Construct,Provisional representation,Reason
0,Expected post-16 route,Retain as seven-category predictor,Combines education setting and non-education r...
1,Higher-education application likelihood,Retain as four-category ordinal predictor,Direct HE expectation measure with high pre-tr...
2,Expected peer post-16 route,Retain as three-category predictor,Measures perceived peer route rather than the ...
3,Preferred post-16 route,Retain as review support only,Conditionally observed and largely overlaps wi...
4,University entry likelihood if applied,Retain as review support only,Conditionally routed from HE application likel...


In [264]:
# 18: Educational aspiration variable-decision check

import string
import pandas as pd
aspiration_construction_sources = ['W1plann16YP', 'W2plann16YP', 'W3plan16YP', 'W1plast16YP', 'W2plast16YP',
    'W3plast16YP', 'W1pladk2YP', *[f'W2Pladk2YP0{letter}' for letter in string.ascii_lowercase[:6]], *[f'W3pladk2aYP0{letter}' for letter in string.ascii_lowercase[:8]], *[f'W3pladk2bYP0{letter}' for letter in string.ascii_lowercase[:8]], 'W1heposs9YP', 'W2heposs9YP', 'W3heposs9YP', 'W1fplan16YP', 'W2fplan16YP', 'W3fplan16YP']
aspiration_review_support = ['W3plann16YP', 'W1hlikeYP', 'W2hlikeYP', 'W3hlikeYP', 'W1pladk16YP',
    *[f'W2Pladk16YP0{letter}' for letter in string.ascii_lowercase[:6]], *[f'W3pladk16aYP0{letter}' for letter in string.ascii_lowercase[:8]], *[f'W3pladk16bYP0{letter}' for letter in string.ascii_lowercase[:8]]]
aspiration_deferred_variables = ['W1plan16YP', 'W1hepossMP',
    *[f'W1henotMP0{letter}' for letter in string.ascii_lowercase[:10]]]
aspiration_excluded_variables = ['W3fplan162YP', 'W4AlevUniYP',
    *[f'W4EMA5YP0{letter}' for letter in string.ascii_lowercase[:19]], 'W4Heposs9YP', 'W4HepossMP']
aspiration_decision_groups = {'Retain as construction source': aspiration_construction_sources,
    'Retain as review support only': aspiration_review_support, 'Defer to later domain': aspiration_deferred_variables, 'Exclude from predictor set': aspiration_excluded_variables}
all_grouped_variables = [variable for variables in aspiration_decision_groups.values() for variable in variables]
assert len(all_grouped_variables) == 97
assert len(set(all_grouped_variables)) == 97
inventory_variables = set(refined_aspiration_plan_inventory['Variable'])
assert len(inventory_variables) == 97
assert inventory_variables == set(all_grouped_variables)
aspiration_disposition_lookup = {variable: disposition for disposition,
    variables in aspiration_decision_groups.items() for variable in variables}
aspiration_domain_lookup = {variable: 'Educational aspirations and post-16 plans' for variable in aspiration_construction_sources + aspiration_review_support + aspiration_excluded_variables}
aspiration_domain_lookup['W1plan16YP'] = 'Experiences and behaviours'
parental_aspiration_variables = ['W1hepossMP', *[f'W1henotMP0{letter}' for letter in string.ascii_lowercase[:10]]]
for variable in parental_aspiration_variables:
    aspiration_domain_lookup[variable] = 'Parental educational attitudes and support'
aspiration_reason_lookup = {}
for variable in aspiration_construction_sources:
    aspiration_reason_lookup[variable] = 'Used in a documented pre-transition construction of expected route, HE application likelihood or expected peer route'
for variable in aspiration_review_support:
    aspiration_reason_lookup[variable] = 'Retained to verify routing, coding, derived-variable consistency or construct overlap'
aspiration_reason_lookup['W1plan16YP'] = 'Measures a general employment attitude rather than a post-16 plan'
aspiration_reason_lookup['W1hepossMP'] = 'Parental expectation measure reserved for the parental educational attitudes and support domain'
for variable in [f'W1henotMP0{letter}' for letter in string.ascii_lowercase[:10]]:
    aspiration_reason_lookup[variable] = 'Conditional reason for parental HE expectation; reserved for the parental domain'
aspiration_reason_lookup['W3fplan162YP'] = 'Reports what friends were doing after leaving school rather than a prospective expectation'
aspiration_reason_lookup['W4AlevUniYP'] = 'Describes courses being taken at or after the post-16 transition'
for variable in [f'W4EMA5YP0{letter}' for letter in string.ascii_lowercase[:19]]:
    aspiration_reason_lookup[variable] = 'Describes reasons for an EMA application decision made at or after transition'
aspiration_reason_lookup['W4Heposs9YP'] = 'Young-person HE expectation measured at or after transition'
aspiration_reason_lookup['W4HepossMP'] = 'Parental HE expectation measured at or after transition'
aspiration_leakage_lookup = {}
for variable in aspiration_construction_sources + aspiration_review_support:
    aspiration_leakage_lookup[variable] = 'No outcome leakage after application of the pre-September 2006 timing rule'
for variable in aspiration_deferred_variables:
    aspiration_leakage_lookup[variable] = 'Pre-transition variable; substantive review deferred to a later domain'
aspiration_leakage_lookup['W3fplan162YP'] = 'Potential post-transition information'
for variable in ['W4AlevUniYP', *[f'W4EMA5YP0{letter}' for letter in string.ascii_lowercase[:19]], 'W4Heposs9YP',
    'W4HepossMP']:
    aspiration_leakage_lookup[variable] = 'At or after transition'
educational_aspiration_decision_audit = refined_aspiration_plan_inventory.copy()
educational_aspiration_decision_audit['Proposed disposition'] = educational_aspiration_decision_audit['Variable'].map(aspiration_disposition_lookup)
educational_aspiration_decision_audit['Proposed substantive domain'] = educational_aspiration_decision_audit['Variable'].map(aspiration_domain_lookup)
educational_aspiration_decision_audit['Proposed reason'] = educational_aspiration_decision_audit['Variable'].map(aspiration_reason_lookup)
educational_aspiration_decision_audit['Proposed leakage assessment'] = educational_aspiration_decision_audit['Variable'].map(aspiration_leakage_lookup)
assert educational_aspiration_decision_audit[['Proposed disposition', 'Proposed substantive domain', 'Proposed reason',
    'Proposed leakage assessment']].notna().all().all()
educational_aspiration_disposition_summary = educational_aspiration_decision_audit['Proposed disposition'].value_counts().reindex(['Retain as construction source',
    'Retain as review support only', 'Defer to later domain', 'Exclude from predictor set']).rename('Variables').rename_axis('Proposed disposition').reset_index()
educational_aspiration_predictor_summary = educational_aspiration_quality.copy()
educational_aspiration_predictor_summary['Proposed decision'] = 'Retain in educational aspiration domain'
educational_aspiration_decision_display = educational_aspiration_decision_audit.sort_values(['Source order',
    'Variable position'])[['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status', 'Proposed disposition', 'Proposed substantive domain', 'Proposed reason', 'Proposed leakage assessment']].reset_index(drop=True)
print(f'Aspiration-related variables accounted for: {len(educational_aspiration_decision_audit):,}')
print('Proposed variable dispositions:')
display_limited(educational_aspiration_disposition_summary)
print('Proposed domain predictors:')
display_limited(educational_aspiration_predictor_summary)
print('Variable-level decision audit:')
display_limited(educational_aspiration_decision_display)

Aspiration-related variables accounted for: 97
Proposed variable dispositions:


,Proposed disposition,Variables
0,Retain as construction source,35
1,Retain as review support only,27
2,Defer to later domain,12
3,Exclude from predictor set,23


Proposed domain predictors:


,Predictor,Non-missing,Missing,Missing percentage,Distinct non-missing values,Proposed decision
0,expected_post16_route,9506,261,2.67,7,Retain in educational aspiration domain
1,higher_education_application_likelihood,9505,262,2.68,4,Retain in educational aspiration domain
2,expected_peer_post16_route,9459,308,3.15,3,Retain in educational aspiration domain


Variable-level decision audit:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Proposed disposition,Proposed substantive domain,Proposed reason,Proposed leakage assessment
0,Wave 1,Young person,wave_one_lsype_young_person_2020,198,W1plann16YP,YP: YP's intentions after Year 11,Pre-transition source,Retain as construction source,Educational aspirations and post-16 plans,Used in a documented pre-transition constructi...,No outcome leakage after application of the pr...
1,Wave 1,Young person,wave_one_lsype_young_person_2020,199,W1plast16YP,YP: YP's intentions for further education afte...,Pre-transition source,Retain as construction source,Educational aspirations and post-16 plans,Used in a documented pre-transition constructi...,No outcome leakage after application of the pr...
2,Wave 1,Young person,wave_one_lsype_young_person_2020,200,W1heposs9YP,YP: Likelihood of YP applying for university,Pre-transition source,Retain as construction source,Educational aspirations and post-16 plans,Used in a documented pre-transition constructi...,No outcome leakage after application of the pr...
3,Wave 1,Young person,wave_one_lsype_young_person_2020,201,W1hlikeYP,YP: Likelihood of YP getting into university i...,Pre-transition source,Retain as review support only,Educational aspirations and post-16 plans,"Retained to verify routing, coding, derived-va...",No outcome leakage after application of the pr...
4,Wave 1,Young person,wave_one_lsype_young_person_2020,202,W1pladk16YP,YP: What want to do at age 16 other than furth...,Pre-transition source,Retain as review support only,Educational aspirations and post-16 plans,"Retained to verify routing, coding, derived-va...",No outcome leakage after application of the pr...


In [265]:
# 19: Educational aspiration domain consolidation

educational_aspiration_domain_predictors = educational_aspiration_candidates.copy().reset_index()
assert list(educational_aspiration_domain_predictors.columns) == ['NSID', 'expected_post16_route',
    'higher_education_application_likelihood', 'expected_peer_post16_route']
assert len(educational_aspiration_domain_predictors) == len(stage_2_ids)
assert educational_aspiration_domain_predictors['NSID'].notna().all()
assert educational_aspiration_domain_predictors['NSID'].is_unique
superseded_decision_columns = ['Substantive domain', 'Decision reason', 'Leakage assessment']
clean_decision_base_columns = [column for column in educational_aspiration_decision_audit.columns if column not in superseded_decision_columns]
educational_aspiration_domain_decisions = educational_aspiration_decision_audit[clean_decision_base_columns].copy().rename(columns={'Proposed disposition': 'Domain disposition',
    'Proposed substantive domain': 'Substantive domain', 'Proposed reason': 'Decision reason', 'Proposed leakage assessment': 'Leakage assessment'}).sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(educational_aspiration_domain_decisions) == 97
assert educational_aspiration_domain_decisions['Variable'].is_unique
assert educational_aspiration_domain_decisions.columns.is_unique
assert educational_aspiration_domain_decisions[['Domain disposition', 'Substantive domain', 'Decision reason',
    'Leakage assessment']].notna().all().all()
register_outcome_lookup = {'Retain as construction source': 'construction', 'Retain as review support only': 'support',
    'Exclude from predictor set': 'exclude'}
current_domain_decision_mask = educational_aspiration_domain_decisions['Domain disposition'].isin(register_outcome_lookup)
deferred_domain_decision_mask = educational_aspiration_domain_decisions['Domain disposition'].eq('Defer to later domain')
assert int(current_domain_decision_mask.sum()) == 85
assert int(deferred_domain_decision_mask.sum()) == 12
current_domain_decisions = educational_aspiration_domain_decisions.loc[current_domain_decision_mask].copy()
deferred_domain_decisions = educational_aspiration_domain_decisions.loc[deferred_domain_decision_mask].copy()
current_outcome_lookup = current_domain_decisions.set_index('Variable')['Domain disposition'].map(register_outcome_lookup).to_dict()
current_domain_lookup = current_domain_decisions.set_index('Variable')['Substantive domain'].to_dict()
current_reason_lookup = current_domain_decisions.set_index('Variable')['Decision reason'].to_dict()
current_leakage_lookup = current_domain_decisions.set_index('Variable')['Leakage assessment'].to_dict()
deferred_domain_lookup = deferred_domain_decisions.set_index('Variable')['Substantive domain'].to_dict()
deferred_reason_lookup = deferred_domain_decisions.set_index('Variable')['Decision reason'].to_dict()
deferred_leakage_lookup = deferred_domain_decisions.set_index('Variable')['Leakage assessment'].to_dict()
variable_decision_register = variable_decision_register.copy()
current_register_mask = variable_decision_register['Variable'].isin(current_domain_decisions['Variable'])
assert int(current_register_mask.sum()) == 85
variable_decision_register.loc[current_register_mask, 'Review status'] = 'Reviewed'
variable_decision_register.loc[current_register_mask,
    'Review outcome'] = variable_decision_register.loc[current_register_mask, 'Variable'].map(current_outcome_lookup)
variable_decision_register.loc[current_register_mask,
    'Substantive domain'] = variable_decision_register.loc[current_register_mask,
    'Variable'].map(current_domain_lookup)
variable_decision_register.loc[current_register_mask,
    'Decision reason'] = variable_decision_register.loc[current_register_mask, 'Variable'].map(current_reason_lookup)
variable_decision_register.loc[current_register_mask,
    'Leakage assessment'] = variable_decision_register.loc[current_register_mask,
    'Variable'].map(current_leakage_lookup)
variable_decision_register.loc[current_register_mask,
    'Documentation source'] = 'Next Steps variable labels, response codes and cross-wave routing review'
deferred_register_mask = variable_decision_register['Variable'].isin(deferred_domain_decisions['Variable'])
assert int(deferred_register_mask.sum()) == 12
variable_decision_register.loc[deferred_register_mask,
    'Substantive domain'] = variable_decision_register.loc[deferred_register_mask,
    'Variable'].map(deferred_domain_lookup)
variable_decision_register.loc[deferred_register_mask,
    'Decision reason'] = variable_decision_register.loc[deferred_register_mask,
    'Variable'].map(deferred_reason_lookup)
variable_decision_register.loc[deferred_register_mask,
    'Leakage assessment'] = variable_decision_register.loc[deferred_register_mask,
    'Variable'].map(deferred_leakage_lookup)
variable_decision_register.loc[deferred_register_mask,
    'Review notes'] = 'Identified during the educational aspiration review and reserved for the stated later domain'
assert variable_decision_register.loc[current_register_mask, ['Review status', 'Review outcome', 'Substantive domain',
    'Decision reason', 'Leakage assessment', 'Documentation source']].notna().all().all()
assert variable_decision_register.loc[deferred_register_mask, ['Substantive domain', 'Decision reason',
    'Leakage assessment', 'Review notes']].notna().all().all()
current_register_outcome_summary = variable_decision_register.loc[current_register_mask,
    'Review outcome'].value_counts().reindex(['construction', 'support',
    'exclude']).rename('Variables').rename_axis('Review outcome').reset_index()
assert current_register_outcome_summary['Variables'].tolist() == [35, 27, 23]
educational_aspiration_complete_count = int(educational_aspiration_domain_predictors[['expected_post16_route',
    'higher_education_application_likelihood', 'expected_peer_post16_route']].notna().all(axis=1).sum())
assert educational_aspiration_complete_count == 9435
educational_aspiration_predictor_path = stage_2_output_directory / 'stage_2_educational_aspirations_post16_plans_domain_predictors.csv'
educational_aspiration_decision_path = stage_2_output_directory / 'stage_2_educational_aspirations_post16_plans_domain_decisions.csv'
educational_aspiration_domain_predictors.to_csv(educational_aspiration_predictor_path, index=False)
educational_aspiration_domain_decisions.to_csv(educational_aspiration_decision_path, index=False)
variable_decision_register.to_csv(decision_register_path, index=False)
saved_aspiration_predictors = pd.read_csv(educational_aspiration_predictor_path)
saved_aspiration_decisions = pd.read_csv(educational_aspiration_decision_path)
saved_decision_register = pd.read_csv(decision_register_path)
assert len(saved_aspiration_predictors) == 9767
assert len(saved_aspiration_decisions) == 97
assert len(saved_decision_register) == 5261
assert saved_aspiration_decisions.columns.is_unique
assert saved_decision_register.columns.is_unique
saved_output_summary = pd.DataFrame([{'Output': 'Educational aspiration predictors',
    'Rows': len(saved_aspiration_predictors), 'Columns': len(saved_aspiration_predictors.columns), 'Unique columns': saved_aspiration_predictors.columns.is_unique}, {'Output': 'Educational aspiration decisions',
    'Rows': len(saved_aspiration_decisions), 'Columns': len(saved_aspiration_decisions.columns), 'Unique columns': saved_aspiration_decisions.columns.is_unique}, {'Output': 'Variable decision register',
    'Rows': len(saved_decision_register), 'Columns': len(saved_decision_register.columns), 'Unique columns': saved_decision_register.columns.is_unique}])
print(f'Educational aspiration domain predictors retained: {len(educational_aspiration_candidates.columns)}')
print(f'Participants with all three predictors available: {educational_aspiration_complete_count:,} ({educational_aspiration_complete_count / len(stage_2_ids) * 100:.2f}%)')
print('Current-domain register outcomes:')
display_limited(current_register_outcome_summary)
print(f'Variables deferred to later domains: {int(deferred_register_mask.sum())}')
print('Saved-output verification:')
display_limited(saved_output_summary)

Educational aspiration domain predictors retained: 3
Participants with all three predictors available: 9,435 (96.60%)
Current-domain register outcomes:


,Review outcome,Variables
0,construction,35
1,support,27
2,exclude,23


Variables deferred to later domains: 12
Saved-output verification:


,Output,Rows,Columns,Unique columns
0,Educational aspiration predictors,9767,4,True
1,Educational aspiration decisions,97,16,True
2,Variable decision register,5261,17,True


## Aspirations and plans predictors

Ninety-seven source variables were reviewed. Thirty-five were construction sources, 27 were review support, 23 were excluded and 12 were deferred to later domains.

Three predictors were retained: `expected_post16_route`, `higher_education_application_likelihood` and `expected_peer_post16_route`. Complete information on all three was available for 9,435 participants (96.60%).


# Part 9: School attitudes and engagement

School attitudes, attendance, discipline, homework and related engagement measures were reviewed. School context, aspirations, psychosocial measures and behaviour outside school were assigned to their respective domains.


In [266]:
# 1: School attitudes and engagement candidate inventory

import re
import pandas as pd
school_engagement_construct_patterns = {'School enjoyment and belonging': '\\benjoy(?:s|ed|ing)?\\b|\\blike(?:s|d)? school\\b|\\bhappy at school\\b|\\bschool belonging\\b|\\bbelong(?:s|ed|ing)?\\b|\\bfeel(?:s|ing)? part of school\\b|\\bschool is worth\\b|\\bsatisfied with school\\b',
    'Teacher relationships and support': '\\bteacher(?:s)?\\b|\\bget(?:s|ting)? on with teacher|\\bteacher support\\b|\\bteacher help\\b|\\btreated fairly\\b|\\bteacher praise\\b|\\bteacher encouragement\\b', 'Attitudes to lessons and schoolwork': '\\blesson(?:s)?\\b|\\bschoolwork\\b|\\bclasswork\\b|\\bwork hard\\b|\\btry hard\\b|\\bpay attention\\b|\\bconcentrat(?:e|es|ed|ing|ion)\\b|\\binterested in school work\\b|\\bimportance of school work\\b', 'Homework and study engagement': '\\bhomework\\b|\\bstudy(?:ing)?\\b|\\brevision\\b|\\brevis(?:e|es|ed|ing)\\b|\\bextra school work\\b|\\btime spent studying\\b', 'Attendance and disengagement': '\\btruan(?:t|ts|ted|ting|cy)\\b|\\babsen(?:t|ce|ces)\\b|\\battendance\\b|\\bskip(?:s|ped|ping)? school\\b|\\bmiss(?:es|ed|ing)? school\\b|\\bstay(?:s|ed|ing)? away from school\\b|\\bunauthorised absence\\b', 'School behaviour and discipline': '\\bbehavio(?:u)?r\\b|\\bmisbehav(?:e|es|ed|ing|iour)\\b|\\bdetention\\b|\\bdiscipline\\b|\\bschool rules\\b|\\btrouble at school\\b|\\bexclude(?:d|s|ing|ion)?\\b|\\bsuspend(?:ed|s|ing|ion)?\\b', 'School climate and peer experience': '\\bbull(?:y|ies|ied|ying)\\b|\\bsafe at school\\b|\\bschool climate\\b|\\bother pupils\\b|\\bother students\\b|\\bpupils behave\\b|\\bstudents behave\\b|\\bdisruption in class\\b'}
school_engagement_review_base = variable_decision_register.loc[variable_decision_register['Review status'].fillna('').ne('Reviewed')].copy()
school_engagement_review_base['Search text'] = school_engagement_review_base['Variable'].fillna('').astype('string') + ' ' + school_engagement_review_base['Variable label'].fillna('').astype('string')
school_engagement_match_frame = pd.DataFrame({construct: school_engagement_review_base['Search text'].str.contains(pattern,
    case=False, regex=True, na=False) for construct, pattern in school_engagement_construct_patterns.items()}, index=school_engagement_review_base.index)
school_engagement_candidate_mask = school_engagement_match_frame.any(axis=1)
school_engagement_candidate_inventory = school_engagement_review_base.loc[school_engagement_candidate_mask].copy()
school_engagement_candidate_inventory['Matched construct'] = school_engagement_match_frame.loc[school_engagement_candidate_mask].apply(lambda row: '; '.join(row.index[row].tolist()),
    axis=1)
school_engagement_candidate_inventory['Number of matched constructs'] = school_engagement_match_frame.loc[school_engagement_candidate_mask].sum(axis=1).astype('int64')
school_engagement_construct_summary = school_engagement_candidate_inventory['Matched construct'].str.get_dummies(sep='; ').sum().sort_values(ascending=False).rename('Candidate variables').rename_axis('Matched construct').reset_index()
school_engagement_wave_timing_summary = school_engagement_candidate_inventory.groupby(['Wave', 'Timing status'],
    dropna=False).size().rename('Candidate variables').reset_index().sort_values(['Wave',
    'Timing status']).reset_index(drop=True)
school_engagement_source_summary = school_engagement_candidate_inventory.groupby(['Wave', 'Source type'],
    dropna=False).size().rename('Candidate variables').reset_index().sort_values(['Wave', 'Candidate variables'],
    ascending=[True, False]).reset_index(drop=True)
school_engagement_existing_status_summary = school_engagement_candidate_inventory.groupby(['Review status',
    'Review outcome'], dropna=False).size().rename('Candidate variables').reset_index()
school_engagement_candidate_display = school_engagement_candidate_inventory.sort_values(['Source order',
    'Variable position'])[['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status', 'Review status', 'Review outcome', 'Substantive domain', 'Matched construct', 'Number of matched constructs']].reset_index(drop=True)
print(f'School attitudes and engagement candidates identified: {len(school_engagement_candidate_inventory):,}')
print('Candidates by construct:')
display_limited(school_engagement_construct_summary)
print('Candidates by wave and timing:')
display_limited(school_engagement_wave_timing_summary)
print('Candidates by wave and source type:')
display_limited(school_engagement_source_summary)
print('Existing review status:')
display_limited(school_engagement_existing_status_summary)
print('Candidate inventory:')
display_limited(school_engagement_candidate_display)

School attitudes and engagement candidates identified: 1,183
Candidates by construct:


,Matched construct,Candidate variables
0,Homework and study engagement,678
1,Teacher relationships and support,215
2,Attitudes to lessons and schoolwork,137
3,School climate and peer experience,89
4,School behaviour and discipline,52


Candidates by wave and timing:


,Wave,Timing status,Candidate variables
0,Wave 1,Pre-transition source,111
1,Wave 2,Pre-transition source,221
2,Wave 3,Near-transition source,322
3,Wave 4,At or after transition,529


Candidates by wave and source type:


,Wave,Source type,Candidate variables
0,Wave 1,Young person,75
1,Wave 1,Parental attitudes,35
2,Wave 1,Family background,1
3,Wave 2,Young person,181
4,Wave 2,Parental attitudes,28


Existing review status:


,Review status,Review outcome,Candidate variables
0,Stable characteristics or retrospective pre-tr...,Exclude from predictor set,2
1,Stable characteristics or retrospective pre-tr...,Pending review,527
2,Use requires interview timing confirming Janua...,Exclude from predictor set,1
3,Use requires interview timing confirming Janua...,Pending review,321
4,"Variable-level coding, routing and reference-p...",Exclude from predictor set,2


Candidate inventory:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Substantive domain,Matched construct,Number of matched constructs
0,Wave 1,Young person,wave_one_lsype_young_person_2020,20,W1sen1MP0g,MP: Nature of YP's special needs: Other behavi...,Pre-transition source,"Variable-level coding, routing and reference-p...",Exclude from predictor set,Special educational needs,School behaviour and discipline,1
1,Wave 1,Young person,wave_one_lsype_young_person_2020,32,W1servssMP,MP: Whether been in contact with social servic...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,School behaviour and discipline,1
2,Wave 1,Young person,wave_one_lsype_young_person_2020,34,W1servothMP,MP: Whether been in contact with any other sim...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,School behaviour and discipline,1
3,Wave 1,Young person,wave_one_lsype_young_person_2020,55,W1abs3mwMP,MP: Reason for YP's (last) extended period of ...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,Attendance and disengagement,1
4,Wave 1,Young person,wave_one_lsype_young_person_2020,56,W1abs1myMP,MP: Whether YP has been absent for 1 month or ...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,Attendance and disengagement,1


In [267]:
# 2: Refined school attitudes and engagement inventory

import re
import pandas as pd
school_engagement_timing_base = variable_decision_register.loc[variable_decision_register['Wave'].isin(['Wave 1',
    'Wave 2', 'Wave 3']) & variable_decision_register['Review status'].fillna('').ne('Reviewed')].copy()
school_engagement_timing_base['Search text'] = school_engagement_timing_base['Variable'].fillna('').astype('string') + ' ' + school_engagement_timing_base['Variable label'].fillna('').astype('string')
refined_school_engagement_patterns = {'School enjoyment, value and belonging': 'enjoy(?:s|ed|ing)? (?:being at )?school|happy (?:when|while) (?:at|in) school|like(?:s|d)? (?:being at|going to) school|school (?:is )?a waste of time|school work is worth doing|schoolwork is worth doing|feel(?:s|ing)? part of (?:the )?school|belong(?:s|ed|ing)? at school|satisfied with (?:his|her|their|the) school',
    'Effort, attention and lesson engagement': 'work(?:s|ed|ing)? as hard as|try(?:ies|ied|ing)? (?:his|her|their|my) best|pay(?:s|ing)? attention in (?:class|lessons)|concentrat(?:e|es|ed|ing) in (?:class|lessons)|bored in (?:class|lessons)|find(?:s|ing)? (?:schoolwork|school work|lessons) interesting|interested in (?:schoolwork|school work|lessons)|distract(?:s|ed|ing) in (?:class|lessons)|disrupt(?:s|ed|ing) (?:class|lessons)', 'Teacher relationships and support': 'get(?:s|ting)? on (?:well )?with (?:his|her|their|my)? ?teachers|teachers? treat(?:s|ed|ing)? (?:him|her|them|me|pupils) fairly|teachers? (?:help|support|encourage|praise)|help from teachers?|teachers? (?:are|is) interested in|teachers? listen(?:s|ed|ing)? to|relationship with teachers?', 'Homework and independent study': '\\bhomework\\b|time spent (?:on )?(?:school work|schoolwork|studying)|hours? (?:spent )?(?:studying|revising)|do(?:es|ing)? extra school work|complete(?:s|d|ing)? (?:his|her|their|my)? ?homework|hand(?:s|ed|ing)? homework in|revision for (?:school|exams?)', 'Attendance and truancy': '\\btruan(?:t|ts|ted|ting|cy)\\b|played truant|skip(?:s|ped|ping)? school|unauthorised absence|absent from school|absence from school|miss(?:es|ed|ing)? school|stay(?:s|ed|ing)? away from school|late for school|school attendance', 'Behaviour, suspension and exclusion': 'suspend(?:s|ed|ing|ion) from school|exclude(?:s|d|ing|ion) from school|expelled from school|detention at school|in trouble at school|behavio(?:u)?r at school|school behavio(?:u)?r|break(?:s|ing)? school rules|disciplin(?:e|ed|ary) at school', 'Bullying and school safety': '\\bbull(?:y|ies|ied|ying)\\b|picked on at school|feel(?:s|ing)? safe at school|afraid (?:at|in) school|threaten(?:s|ed|ing)? at school|physically hurt at school'}
school_engagement_exclusion_pattern = 'private lesson|private class|extra tuition|paid tuition|special need|nature of .* needs|social service|health service|teacher training|training to be a teacher|subject teacher|number of teachers|teacher qualification|course teacher|driving lesson|music lesson|swimming lesson'
refined_school_match_frame = pd.DataFrame({construct: school_engagement_timing_base['Search text'].str.contains(pattern,
    case=False, regex=True, na=False) for construct, pattern in refined_school_engagement_patterns.items()}, index=school_engagement_timing_base.index)
refined_school_candidate_mask = refined_school_match_frame.any(axis=1)
false_positive_context_mask = school_engagement_timing_base['Search text'].str.contains(school_engagement_exclusion_pattern,
    case=False, regex=True, na=False)
refined_school_candidate_mask = refined_school_candidate_mask & ~false_positive_context_mask
refined_school_engagement_inventory = school_engagement_timing_base.loc[refined_school_candidate_mask].copy()
refined_school_engagement_inventory['Matched construct'] = refined_school_match_frame.loc[refined_school_candidate_mask].apply(lambda row: '; '.join(row.index[row].tolist()),
    axis=1)
refined_school_engagement_inventory['Number of matched constructs'] = refined_school_match_frame.loc[refined_school_candidate_mask].sum(axis=1).astype('int64')
refined_school_construct_summary = refined_school_engagement_inventory['Matched construct'].str.get_dummies(sep='; ').sum().sort_values(ascending=False).rename('Candidate variables').rename_axis('Matched construct').reset_index()
refined_school_wave_summary = refined_school_engagement_inventory.groupby(['Wave', 'Timing status'],
    dropna=False).size().rename('Candidate variables').reset_index().sort_values(['Wave',
    'Timing status']).reset_index(drop=True)
refined_school_source_summary = refined_school_engagement_inventory.groupby(['Wave', 'Source type'],
    dropna=False).size().rename('Candidate variables').reset_index().sort_values(['Wave', 'Candidate variables'],
    ascending=[True, False]).reset_index(drop=True)
refined_school_existing_status_summary = refined_school_engagement_inventory.groupby(['Review status',
    'Review outcome'], dropna=False).size().rename('Candidate variables').reset_index()
refined_school_engagement_display = refined_school_engagement_inventory.sort_values(['Source order',
    'Variable position'])[['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status', 'Review status', 'Review outcome', 'Substantive domain', 'Matched construct', 'Number of matched constructs']].reset_index(drop=True)
print(f'Refined school attitudes and engagement candidates: {len(refined_school_engagement_inventory):,}')
print('Candidates by construct:')
display_limited(refined_school_construct_summary)
print('Candidates by wave and timing:')
display_limited(refined_school_wave_summary)
print('Candidates by wave and source type:')
display_limited(refined_school_source_summary)
print('Existing review status:')
display_limited(refined_school_existing_status_summary)
print('Refined candidate inventory:')
display_limited(refined_school_engagement_display)

Refined school attitudes and engagement candidates: 113
Candidates by construct:


,Matched construct,Candidate variables
0,Bullying and school safety,47
1,Attendance and truancy,25
2,Homework and independent study,17
3,"School enjoyment, value and belonging",9
4,"Behaviour, suspension and exclusion",8


Candidates by wave and timing:


,Wave,Timing status,Candidate variables
0,Wave 1,Pre-transition source,39
1,Wave 2,Pre-transition source,46
2,Wave 3,Near-transition source,28


Candidates by wave and source type:


,Wave,Source type,Candidate variables
0,Wave 1,Young person,37
1,Wave 1,Parental attitudes,2
2,Wave 2,Young person,46
3,Wave 3,Young person,28


Existing review status:


,Review status,Review outcome,Candidate variables
0,Use requires interview timing confirming Janua...,Pending review,28
1,"Variable-level coding, routing and reference-p...",Pending review,85


Refined candidate inventory:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Substantive domain,Matched construct,Number of matched constructs
0,Wave 1,Young person,wave_one_lsype_young_person_2020,55,W1abs3mwMP,MP: Reason for YP's (last) extended period of ...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,Attendance and truancy,1
1,Wave 1,Young person,wave_one_lsype_young_person_2020,57,W1abs1mwMP,MP: Reason for YP's period of absence from sch...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,Attendance and truancy,1
2,Wave 1,Young person,wave_one_lsype_young_person_2020,58,W1suspendMP,MP: Whether YP has ever been temporarily suspe...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,"Behaviour, suspension and exclusion",1
3,Wave 1,Young person,wave_one_lsype_young_person_2020,60,W1sutime2MP,MP: Number of times YP has been temporarily ex...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,"Behaviour, suspension and exclusion",1
4,Wave 1,Young person,wave_one_lsype_young_person_2020,61,W1expelMP,MP: Whether YP has ever been expelled or perma...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>,"Behaviour, suspension and exclusion",1


In [268]:
# 3: Core school attitude and engagement measure-family review

import re
import pandas as pd
core_school_constructs = {'School enjoyment, value and belonging', 'Effort, attention and lesson engagement',
    'Teacher relationships and support', 'Homework and independent study'}
behavioural_school_constructs = {'Attendance and truancy', 'Behaviour, suspension and exclusion'}
school_safety_constructs = {'Bullying and school safety'}

def identify_school_review_track(matched_construct_text):
    """Assign each candidate to a substantive review track."""
    matched_constructs = set(str(matched_construct_text).split('; '))
    matched_tracks = []
    if matched_constructs & core_school_constructs:
        matched_tracks.append('Core school attitudes and engagement')
    if matched_constructs & behavioural_school_constructs:
        matched_tracks.append('Attendance, behaviour and discipline')
    if matched_constructs & school_safety_constructs:
        matched_tracks.append('Bullying and school safety')
    return '; '.join(matched_tracks)
school_engagement_review_tracks = refined_school_engagement_inventory.copy()
school_engagement_review_tracks['Review track'] = school_engagement_review_tracks['Matched construct'].apply(identify_school_review_track)
assert school_engagement_review_tracks['Review track'].ne('').all()
school_review_track_summary = school_engagement_review_tracks['Review track'].str.get_dummies(sep='; ').sum().sort_values(ascending=False).rename('Candidate variables').rename_axis('Review track').reset_index()
core_school_engagement_inventory = school_engagement_review_tracks.loc[school_engagement_review_tracks['Review track'].str.contains('Core school attitudes and engagement',
    regex=False, na=False)].copy()
assert len(core_school_engagement_inventory) == 34
core_school_engagement_inventory['Variable stem'] = core_school_engagement_inventory['Variable'].astype('string').str.replace('^W[123]',
    '', regex=True).str.lower()
core_measure_family_summary = core_school_engagement_inventory.groupby('Variable stem',
    dropna=False).agg(Waves=('Wave', lambda values: '; '.join(sorted(values.dropna().astype(str).unique()))),
    Number_of_waves=('Wave', lambda values: values.dropna().nunique()), Variables=('Variable',
    lambda values: '; '.join(values.astype(str).tolist())), Matched_constructs=('Matched construct',
    lambda values: '; '.join(sorted(values.dropna().astype(str).unique()))), Variable_labels=('Variable label',
    lambda values: ' | '.join(values.dropna().astype(str).tolist()))).reset_index().sort_values(['Number_of_waves',
    'Variable stem'], ascending=[False,
    True]).reset_index(drop=True).rename(columns={'Number_of_waves': 'Number of waves',
    'Matched_constructs': 'Matched constructs', 'Variable_labels': 'Variable labels'})
repeated_core_measure_families = core_measure_family_summary.loc[core_measure_family_summary['Number of waves'].ge(2)].copy()
wave_specific_core_measure_families = core_measure_family_summary.loc[core_measure_family_summary['Number of waves'].eq(1)].copy()
core_school_construct_wave_summary = core_school_engagement_inventory.groupby(['Matched construct', 'Wave',
    'Source type'], dropna=False).size().rename('Candidate variables').reset_index().sort_values(['Matched construct',
    'Wave', 'Source type']).reset_index(drop=True)
core_school_engagement_display = core_school_engagement_inventory.sort_values(['Matched construct', 'Source order',
    'Variable position'])[['Matched construct', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable',
    'Variable stem', 'Variable label', 'Timing status', 'Review status', 'Review outcome']].reset_index(drop=True)
print('School-related candidates by review track:')
display_limited(school_review_track_summary)
print(f'Core school attitude and engagement candidates: {len(core_school_engagement_inventory):,}')
print('Core candidates by construct, wave and source:')
display_limited(core_school_construct_wave_summary)
print(f'Repeated core measure families across two or more waves: {len(repeated_core_measure_families):,}')
display_limited(repeated_core_measure_families)
print(f'Wave-specific core measure families: {len(wave_specific_core_measure_families):,}')
display_limited(wave_specific_core_measure_families)
print('Core variable-level inventory:')
display_limited(core_school_engagement_display)

School-related candidates by review track:


,Review track,Candidate variables
0,Bullying and school safety,47
1,Core school attitudes and engagement,34
2,"Attendance, behaviour and discipline",33


Core school attitude and engagement candidates: 34
Core candidates by construct, wave and source:


,Matched construct,Wave,Source type,Candidate variables
0,"Effort, attention and lesson engagement",Wave 1,Young person,2
1,"Effort, attention and lesson engagement",Wave 2,Young person,2
2,"Effort, attention and lesson engagement",Wave 3,Young person,2
3,Homework and independent study,Wave 1,Young person,8
4,Homework and independent study,Wave 2,Young person,9


Repeated core measure families across two or more waves: 13


,Variable stem,Waves,Number of waves,Variables,Matched constructs,Variable labels
0,yys2yp,Wave 1; Wave 2; Wave 3,3,W1yys2YP; W2YYS2YP; W3yys2YP,"School enjoyment, value and belonging",YP: Feelings about school: School is a waste o...
1,yys3yp,Wave 1; Wave 2; Wave 3,3,W1yys3YP; W2YYS3YP; W3yys3YP,"School enjoyment, value and belonging",YP: Feelings about school: School work is wort...
2,yys6yp,Wave 1; Wave 2; Wave 3,3,W1yys6YP; W2YYS6YP; W3yys6YP,"School enjoyment, value and belonging",YP: Feelings about school: On the whole I like...
3,yys7yp,Wave 1; Wave 2; Wave 3,3,W1yys7YP; W2YYS7YP; W3yys7YP,"Effort, attention and lesson engagement",YP: Feelings about school: I work as hard as I...
4,yys9yp,Wave 1; Wave 2; Wave 3,3,W1yys9YP; W2YYS9YP; W3yys9YP,"Effort, attention and lesson engagement",YP: Feelings about school: I am bored in lesso...


Wave-specific core measure families: 3


,Variable stem,Waves,Number of waves,Variables,Matched constructs,Variable labels
13,hwnday1yp,Wave 2,1,W2hwnday1YP,Homework and independent study,YP: How many evenings a week during term-time ...
14,hwnday2yp,Wave 2,1,W2hwnday2YP,Homework and independent study,YP: When YP does homework
15,hwndayyp,Wave 1,1,W1hwndayYP,Homework and independent study,YP: Number of evenings do homework


Core variable-level inventory:


,Matched construct,Wave,Source type,Source file,Variable position,Variable,Variable stem,Variable label,Timing status,Review status,Review outcome
0,"Effort, attention and lesson engagement",Wave 1,Young person,wave_one_lsype_young_person_2020,226,W1yys7YP,yys7yp,YP: Feelings about school: I work as hard as I...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review
1,"Effort, attention and lesson engagement",Wave 1,Young person,wave_one_lsype_young_person_2020,228,W1yys9YP,yys9yp,YP: Feelings about school: I am bored in lessons,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review
2,"Effort, attention and lesson engagement",Wave 2,Young person,wave_two_lsype_young_person_2020,357,W2YYS7YP,yys7yp,YP: Feelings about school: I work as hard as I...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review
3,"Effort, attention and lesson engagement",Wave 2,Young person,wave_two_lsype_young_person_2020,359,W2YYS9YP,yys9yp,YP: Feelings about school: I am bored in lessons,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review
4,"Effort, attention and lesson engagement",Wave 3,Young person,wave_three_lsype_young_person_2020,552,W3yys7YP,yys7yp,YP: Feelings about school: I work as hard as I...,Near-transition source,Use requires interview timing confirming Janua...,Pending review


In [269]:
# 4: Complete school-attitude battery inventory

import pandas as pd
school_attitude_battery_mask = variable_decision_register['Wave'].isin(['Wave 1', 'Wave 2',
    'Wave 3']) & variable_decision_register['Variable'].fillna('').astype('string').str.match('^W[123]YYS\\d+YP$',
    case=False, na=False)
school_attitude_battery_inventory = variable_decision_register.loc[school_attitude_battery_mask].copy()
assert not school_attitude_battery_inventory.empty
assert school_attitude_battery_inventory['Variable'].is_unique
school_attitude_battery_inventory['School-attitude item number'] = school_attitude_battery_inventory['Variable'].str.extract('(?i)^W[123]YYS(\\d+)YP$',
    expand=False).astype('int64')
school_attitude_battery_inventory['Variable stem'] = school_attitude_battery_inventory['Variable'].str.replace('(?i)^W[123]',
    '', regex=True).str.lower()
identified_core_variables = set(core_school_engagement_inventory['Variable'])
school_attitude_battery_inventory['Identified in refined keyword review'] = school_attitude_battery_inventory['Variable'].isin(identified_core_variables)
wave_item_duplicates = school_attitude_battery_inventory.duplicated(subset=['Wave', 'School-attitude item number'],
    keep=False)
assert not wave_item_duplicates.any()
school_attitude_battery_wave_summary = school_attitude_battery_inventory.groupby(['Wave', 'Timing status'],
    dropna=False).agg(Battery_items=('Variable', 'size'),
    Identified_by_keyword_review=('Identified in refined keyword review',
    'sum')).reset_index().rename(columns={'Battery_items': 'Battery items',
    'Identified_by_keyword_review': 'Identified by keyword review'})
school_attitude_battery_wave_summary['Not identified by keyword review'] = school_attitude_battery_wave_summary['Battery items'] - school_attitude_battery_wave_summary['Identified by keyword review']
school_attitude_item_family_summary = school_attitude_battery_inventory.sort_values(['School-attitude item number',
    'Source order', 'Variable position']).groupby(['School-attitude item number', 'Variable stem'],
    dropna=False).agg(Waves=('Wave', lambda values: '; '.join(values.dropna().astype(str).tolist())),
    Number_of_waves=('Wave', 'nunique'), Variables=('Variable',
    lambda values: '; '.join(values.astype(str).tolist())), Variable_labels=('Variable label',
    lambda values: ' | '.join(values.dropna().astype(str).tolist())), Identified_in_keyword_review=('Identified in refined keyword review',
    'any')).reset_index().rename(columns={'Number_of_waves': 'Number of waves', 'Variable_labels': 'Variable labels',
    'Identified_in_keyword_review': 'Identified in keyword review'}).sort_values('School-attitude item number').reset_index(drop=True)
school_attitude_unidentified_inventory = school_attitude_battery_inventory.loc[~school_attitude_battery_inventory['Identified in refined keyword review']].sort_values(['School-attitude item number',
    'Source order', 'Variable position'])[['School-attitude item number', 'Wave', 'Variable', 'Variable label',
    'Timing status', 'Review status', 'Review outcome', 'Substantive domain']].reset_index(drop=True)
school_attitude_battery_status_summary = school_attitude_battery_inventory.groupby(['Review status', 'Review outcome'],
    dropna=False).size().rename('Variables').reset_index()
school_attitude_battery_display = school_attitude_battery_inventory.sort_values(['School-attitude item number',
    'Source order', 'Variable position'])[['School-attitude item number', 'Wave', 'Source file', 'Variable position',
    'Variable', 'Variable label', 'Timing status', 'Identified in refined keyword review', 'Review status', 'Review outcome', 'Substantive domain']].reset_index(drop=True)
print(f'Numbered school-attitude battery variables identified: {len(school_attitude_battery_inventory):,}')
print(f"Distinct numbered school-attitude items: {school_attitude_battery_inventory['School-attitude item number'].nunique():,}")
print('Battery coverage by wave:')
display_limited(school_attitude_battery_wave_summary)
print('Repeated numbered item families:')
display_limited(school_attitude_item_family_summary)
print(f'Battery variables not identified by the earlier keyword review: {len(school_attitude_unidentified_inventory):,}')
display_limited(school_attitude_unidentified_inventory)
print('Current register status:')
display_limited(school_attitude_battery_status_summary)
print('Complete school-attitude battery inventory:')
display_limited(school_attitude_battery_display)

Numbered school-attitude battery variables identified: 59
Distinct numbered school-attitude items: 26
Battery coverage by wave:


,Wave,Timing status,Battery items,Identified by keyword review,Not identified by keyword review
0,Wave 1,Pre-transition source,23,7,16
1,Wave 2,Pre-transition source,24,7,17
2,Wave 3,Near-transition source,12,5,7


Repeated numbered item families:


,School-attitude item number,Variable stem,Waves,Number of waves,Variables,Variable labels,Identified in keyword review
0,1,yys1yp,Wave 1; Wave 2; Wave 3,3,W1yys1YP; W2YYS1YP; W3yys1YP,YP: Feelings about school: I am happy when I a...,False
1,2,yys2yp,Wave 1; Wave 2; Wave 3,3,W1yys2YP; W2YYS2YP; W3yys2YP,YP: Feelings about school: School is a waste o...,True
2,3,yys3yp,Wave 1; Wave 2; Wave 3,3,W1yys3YP; W2YYS3YP; W3yys3YP,YP: Feelings about school: School work is wort...,True
3,4,yys4yp,Wave 1; Wave 2; Wave 3,3,W1yys4YP; W2YYS4YP; W3yys4YP,YP: Feelings about school: Most of the time I ...,False
4,5,yys5yp,Wave 1; Wave 2; Wave 3,3,W1yys5YP; W2YYS5YP; W3yys5YP,YP: Feelings about school: People think my sch...,False


Battery variables not identified by the earlier keyword review: 40


,School-attitude item number,Wave,Variable,Variable label,Timing status,Review status,Review outcome,Substantive domain
0,1,Wave 1,W1yys1YP,YP: Feelings about school: I am happy when I a...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>
1,1,Wave 2,W2YYS1YP,YP: Feelings about school: I am happy when I a...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>
2,1,Wave 3,W3yys1YP,YP: Feelings about school: I am happy when I a...,Near-transition source,Use requires interview timing confirming Janua...,Pending review,<NA>
3,4,Wave 1,W1yys4YP,YP: Feelings about school: Most of the time I ...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>
4,4,Wave 2,W2YYS4YP,YP: Feelings about school: Most of the time I ...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,<NA>


Current register status:


,Review status,Review outcome,Variables
0,Use requires interview timing confirming Janua...,Pending review,12
1,"Variable-level coding, routing and reference-p...",Pending review,47


Complete school-attitude battery inventory:


,School-attitude item number,Wave,Source file,Variable position,Variable,Variable label,Timing status,Identified in refined keyword review,Review status,Review outcome,Substantive domain
0,1,Wave 1,wave_one_lsype_young_person_2020,220,W1yys1YP,YP: Feelings about school: I am happy when I a...,Pre-transition source,False,"Variable-level coding, routing and reference-p...",Pending review,<NA>
1,1,Wave 2,wave_two_lsype_young_person_2020,351,W2YYS1YP,YP: Feelings about school: I am happy when I a...,Pre-transition source,False,"Variable-level coding, routing and reference-p...",Pending review,<NA>
2,1,Wave 3,wave_three_lsype_young_person_2020,546,W3yys1YP,YP: Feelings about school: I am happy when I a...,Near-transition source,False,Use requires interview timing confirming Janua...,Pending review,<NA>
3,2,Wave 1,wave_one_lsype_young_person_2020,221,W1yys2YP,YP: Feelings about school: School is a waste o...,Pre-transition source,True,"Variable-level coding, routing and reference-p...",Pending review,<NA>
4,2,Wave 2,wave_two_lsype_young_person_2020,352,W2YYS2YP,YP: Feelings about school: School is a waste o...,Pre-transition source,True,"Variable-level coding, routing and reference-p...",Pending review,<NA>


In [270]:
# 5: School-attitude battery coding and coverage review

import pandas as pd
school_attitude_raw_tables = {}
school_attitude_labelled_tables = {}
school_attitude_loading_rows = []
for source_file, source_inventory in school_attitude_battery_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    raw_data['NSID'] = standardise_nsid(raw_data['NSID'])
    labelled_data['NSID'] = standardise_nsid(labelled_data['NSID'])
    assert raw_data['NSID'].is_unique
    assert labelled_data['NSID'].is_unique
    source_participants = int(raw_data['NSID'].notna().sum())
    raw_data = raw_data.set_index('NSID').reindex(stage_2_ids)
    labelled_data = labelled_data.set_index('NSID').reindex(stage_2_ids)
    for variable in source_variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    school_attitude_raw_tables[source_file] = raw_data
    school_attitude_labelled_tables[source_file] = labelled_data
    school_attitude_loading_rows.append({'Source file': source_file, 'Variables loaded': len(source_variables),
        'Source participants': source_participants, 'Stage 2 rows after alignment': len(raw_data)})
school_attitude_loading_summary = pd.DataFrame(school_attitude_loading_rows)
assert len(school_attitude_raw_tables) == school_attitude_battery_inventory['Source file'].nunique()
assert all((len(table) == len(stage_2_ids) for table in school_attitude_raw_tables.values()))
print(f'School-attitude source files loaded: {len(school_attitude_raw_tables):,}')
print(f"School-attitude variables loaded: {school_attitude_battery_inventory['Variable'].nunique():,}")
display_limited(school_attitude_loading_summary)

School-attitude source files loaded: 3
School-attitude variables loaded: 59


,Source file,Variables loaded,Source participants,Stage 2 rows after alignment
0,wave_one_lsype_young_person_2020,23,15760,9767
1,wave_two_lsype_young_person_2020,24,13530,9767
2,wave_three_lsype_young_person_2020,12,12430,9767


In [271]:
# 6: Variable coding and response-quality review

school_attitude_code_rows = []
school_attitude_quality_rows = []
for source_file, source_inventory in school_attitude_battery_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    raw_data = school_attitude_raw_tables[source_file]
    labelled_data = school_attitude_labelled_tables[source_file]
    for variable in source_variables:
        variable_row = source_inventory.loc[source_inventory['Variable'].eq(variable)].iloc[0]
        wave = variable_row['Wave']
        raw_values = raw_data[variable]
        labelled_values = labelled_data[variable].astype('string')
        if wave == 'Wave 3':
            timing_eligible = wave_3_pretransition_eligible.reindex(stage_2_ids).fillna(False)
        else:
            timing_eligible = pd.Series(True, index=raw_values.index)
        observed_response = raw_values.ge(0).fillna(False)
        special_code = raw_values.lt(0).fillna(False)
        school_attitude_quality_rows.append({'School-attitude item number': int(variable_row['School-attitude item number']),
            'Wave': wave, 'Variable': variable, 'Variable label': variable_row['Variable label'], 'Timing status': variable_row['Timing status'], 'Observed responses': int(observed_response.sum()), 'Special-code responses': int(special_code.sum()), 'No source record': int(raw_values.isna().sum()), 'Observed before September 2006': int((observed_response & timing_eligible).sum()), 'Distinct observed codes': int(raw_values.loc[observed_response].nunique()), 'Minimum observed code': raw_values.loc[observed_response].min() if observed_response.any() else pd.NA, 'Maximum observed code': raw_values.loc[observed_response].max() if observed_response.any() else pd.NA})
        code_counts = raw_values.value_counts(dropna=False)
        for raw_code, participants in code_counts.items():
            if pd.isna(raw_code):
                value_label = 'No source record'
                response_type = 'No source record'
                code_sort_value = 999999
            else:
                code_mask = raw_values.eq(raw_code)
                matching_labels = labelled_values.loc[code_mask].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(raw_code)
                response_type = 'Observed response' if raw_code >= 0 else 'Special code'
                code_sort_value = float(raw_code)
            school_attitude_code_rows.append({'School-attitude item number': int(variable_row['School-attitude item number']),
                'Wave': wave, 'Variable': variable, 'Variable label': variable_row['Variable label'], 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Code sort value': code_sort_value})
print(f'Variables reviewed: {len(school_attitude_quality_rows):,}')
print(f'Code-distribution records created: {len(school_attitude_code_rows):,}')

Variables reviewed: 59
Code-distribution records created: 544


In [272]:
# 7: Coding, wording and coverage summaries

school_attitude_variable_quality = pd.DataFrame(school_attitude_quality_rows).sort_values(['School-attitude item number',
    'Wave']).reset_index(drop=True)
school_attitude_code_distribution = pd.DataFrame(school_attitude_code_rows).sort_values(['School-attitude item number',
    'Wave', 'Code sort value']).drop(columns=['Code sort value']).reset_index(drop=True)
observed_code_labels = school_attitude_code_distribution.loc[school_attitude_code_distribution['Response type'].eq('Observed response')].copy()
observed_code_labels['Code-label pair'] = observed_code_labels['Raw code'].astype('Int64').astype('string') + ' = ' + observed_code_labels['Value label'].astype('string')
school_attitude_scale_signature = observed_code_labels.groupby(['School-attitude item number', 'Wave', 'Variable'],
    dropna=False).agg(Response_scale=('Code-label pair',
    lambda values: '; '.join(values.tolist()))).reset_index().rename(columns={'Response_scale': 'Response scale'})
school_attitude_scale_summary = school_attitude_scale_signature.groupby('Response scale',
    dropna=False).agg(Variables=('Variable', 'size'), Item_numbers=('School-attitude item number',
    lambda values: '; '.join(sorted({str(int(value)) for value in values}, key=int))), Variable_names=('Variable',
    lambda values: '; '.join(values.tolist()))).reset_index().rename(columns={'Item_numbers': 'Item numbers',
    'Variable_names': 'Variable names'}).sort_values(['Variables', 'Response scale'], ascending=[False,
    True]).reset_index(drop=True)
school_attitude_special_code_summary = school_attitude_code_distribution.loc[school_attitude_code_distribution['Response type'].eq('Special code')].groupby(['Raw code',
    'Value label'], dropna=False).agg(Variables=('Variable', 'nunique'), Total_responses=('Participants',
    'sum')).reset_index().rename(columns={'Total_responses': 'Total responses'}).sort_values('Raw code').reset_index(drop=True)
school_attitude_wording_review = school_attitude_battery_inventory.sort_values(['School-attitude item number',
    'Source order', 'Variable position']).groupby('School-attitude item number',
    dropna=False).agg(Number_of_waves=('Wave', 'nunique'), Waves=('Wave', lambda values: '; '.join(values.tolist())),
    Variables=('Variable', lambda values: '; '.join(values.tolist())), Distinct_labels=('Variable label',
    'nunique'), Full_labels=('Variable label',
    lambda values: ' | '.join(values.astype(str).tolist()))).reset_index().rename(columns={'Number_of_waves': 'Number of waves',
    'Distinct_labels': 'Distinct labels', 'Full_labels': 'Full labels'}).sort_values('School-attitude item number').reset_index(drop=True)
school_attitude_coverage_summary = school_attitude_variable_quality[['School-attitude item number', 'Wave', 'Variable',
    'Observed responses', 'Special-code responses', 'No source record', 'Observed before September 2006', 'Distinct observed codes']].copy()
school_attitude_coverage_summary['Observed-response percentage'] = (school_attitude_coverage_summary['Observed responses'] / len(stage_2_ids) * 100).round(2)
school_attitude_coverage_summary['Pre-transition observed percentage'] = (school_attitude_coverage_summary['Observed before September 2006'] / len(stage_2_ids) * 100).round(2)
school_attitude_wave_coverage_summary = school_attitude_coverage_summary.groupby('Wave',
    dropna=False).agg(Variables=('Variable', 'size'), Median_observed_percentage=('Observed-response percentage',
    'median'), Minimum_observed_percentage=('Observed-response percentage',
    'min'), Maximum_observed_percentage=('Observed-response percentage',
    'max'), Median_pretransition_percentage=('Pre-transition observed percentage',
    'median')).reset_index().rename(columns={'Median_observed_percentage': 'Median observed percentage',
    'Minimum_observed_percentage': 'Minimum observed percentage', 'Maximum_observed_percentage': 'Maximum observed percentage', 'Median_pretransition_percentage': 'Median pre-transition percentage'})
assert len(school_attitude_variable_quality) == len(school_attitude_battery_inventory)
assert school_attitude_variable_quality[['School-attitude item number', 'Wave', 'Variable']].duplicated().sum() == 0
print(f'Variable-quality rows: {len(school_attitude_variable_quality):,}')
print(f'Distinct response-scale structures: {len(school_attitude_scale_summary):,}')
print(f'Wording-review rows: {len(school_attitude_wording_review):,}')

Variable-quality rows: 59
Distinct response-scale structures: 5
Wording-review rows: 26


In [273]:
# 8: Compact review output and check-table export

school_attitude_variable_quality_file = stage_2_output_directory / 'stage_2_school_attitude_variable_quality.csv'
school_attitude_code_distribution_file = stage_2_output_directory / 'stage_2_school_attitude_code_distribution.csv'
school_attitude_scale_signature_file = stage_2_output_directory / 'stage_2_school_attitude_scale_signature.csv'
school_attitude_scale_summary_file = stage_2_output_directory / 'stage_2_school_attitude_scale_summary.csv'
school_attitude_special_code_summary_file = stage_2_output_directory / 'stage_2_school_attitude_special_code_summary.csv'
school_attitude_wording_review_file = stage_2_output_directory / 'stage_2_school_attitude_wording_review.csv'
school_attitude_coverage_summary_file = stage_2_output_directory / 'stage_2_school_attitude_coverage_summary.csv'
school_attitude_wave_coverage_summary_file = stage_2_output_directory / 'stage_2_school_attitude_wave_coverage_summary.csv'
school_attitude_variable_quality.to_csv(school_attitude_variable_quality_file, index=False)
school_attitude_code_distribution.to_csv(school_attitude_code_distribution_file, index=False)
school_attitude_scale_signature.to_csv(school_attitude_scale_signature_file, index=False)
school_attitude_scale_summary.to_csv(school_attitude_scale_summary_file, index=False)
school_attitude_special_code_summary.to_csv(school_attitude_special_code_summary_file, index=False)
school_attitude_wording_review.to_csv(school_attitude_wording_review_file, index=False)
school_attitude_coverage_summary.to_csv(school_attitude_coverage_summary_file, index=False)
school_attitude_wave_coverage_summary.to_csv(school_attitude_wave_coverage_summary_file, index=False)
school_attitude_review_checkpoints = pd.DataFrame({'Check': ['Source files reviewed', 'Variables reviewed',
    'School-attitude items', 'Response-scale structures', 'Special response-code categories', 'Raw code-distribution rows'], 'Result': [school_attitude_battery_inventory['Source file'].nunique(),
    len(school_attitude_variable_quality), school_attitude_battery_inventory['School-attitude item number'].nunique(), len(school_attitude_scale_summary), len(school_attitude_special_code_summary), len(school_attitude_code_distribution)]})
print('School-attitude coding review:')
display_limited(school_attitude_review_checkpoints)
print('\nCoverage by wave:')
display_limited(school_attitude_wave_coverage_summary)
print('\nFirst 10 variable-level coverage records:')
display_limited(school_attitude_coverage_summary[['School-attitude item number', 'Wave', 'Variable',
    'Observed-response percentage', 'Pre-transition observed percentage', 'Distinct observed codes']].head(10))
print(f'Additional variable-level coverage records not displayed: {max(len(school_attitude_coverage_summary) - 10, 0):,}')
print('\nSpecial response codes:')
display_limited(school_attitude_special_code_summary)
print(f'\nFull review tables saved to: {stage_2_output_directory}')

School-attitude coding review:


,Check,Result
0,Source files reviewed,3
1,Variables reviewed,59
2,School-attitude items,26
3,Response-scale structures,5
4,Special response-code categories,7



Coverage by wave:


,Wave,Variables,Median observed percentage,Minimum observed percentage,Maximum observed percentage,Median pre-transition percentage
0,Wave 1,23,93.540,90.01,95.22,93.540
1,Wave 2,24,93.215,89.69,95.40,93.215
2,Wave 3,12,93.895,91.52,94.73,92.470



First 10 variable-level coverage records:


,School-attitude item number,Wave,Variable,Observed-response percentage,Pre-transition observed percentage,Distinct observed codes
0,1,Wave 1,W1yys1YP,93.10,93.10,4
1,1,Wave 2,W2YYS1YP,93.11,93.11,4
2,1,Wave 3,W3yys1YP,94.19,92.76,4
3,2,Wave 1,W1yys2YP,93.44,93.44,4
4,2,Wave 2,W2YYS2YP,93.16,93.16,4


Additional variable-level coverage records not displayed: 49

Special response codes:


,Raw code,Value label,Variables,Total responses
0,-99.0,YP not interviewed,59,4519
1,-97.0,YP refused CASI section,23,1518
2,-97.0,YP refused self completion,36,1668
3,-96.0,YP unable to complete CASI section,23,1127
4,-96.0,YP using interpreter,36,1548



Full review tables saved to: data_derived\stage_2_predictor_construction


In [274]:
# 9: Twelve-item school-attitude scale diagnostic review

import numpy as np
import pandas as pd
school_attitude_core_items = list(range(1, 13))
positively_worded_items = {1, 3, 5, 6, 7, 11, 12}
negatively_worded_items = {2, 4, 8, 9, 10}
assert positively_worded_items | negatively_worded_items == set(school_attitude_core_items)
assert not positively_worded_items & negatively_worded_items
school_attitude_wave_configuration = {'Wave 1': {'Source file': 'wave_one_lsype_young_person_2020',
    'Variables': {item_number: f'W1yys{item_number}YP' for item_number in school_attitude_core_items}}, 'Wave 2': {'Source file': 'wave_two_lsype_young_person_2020',
    'Variables': {item_number: f'W2YYS{item_number}YP' for item_number in school_attitude_core_items}}, 'Wave 3': {'Source file': 'wave_three_lsype_young_person_2020',
    'Variables': {item_number: f'W3yys{item_number}YP' for item_number in school_attitude_core_items}}}
school_attitude_item_label_lookup = school_attitude_battery_inventory.loc[school_attitude_battery_inventory['School-attitude item number'].isin(school_attitude_core_items)].sort_values(['School-attitude item number',
    'Source order', 'Variable position']).groupby('School-attitude item number')['Variable label'].first().str.strip().to_dict()
assert len(school_attitude_item_label_lookup) == 12

def calculate_cronbach_alpha(item_data):
    """Calculate Cronbach's alpha using complete rows."""
    complete_data = item_data.dropna().astype('float64')
    participant_count = len(complete_data)
    item_count = complete_data.shape[1]
    if participant_count < 2 or item_count < 2:
        return (np.nan, participant_count)
    summed_item_variance = complete_data.var(axis=0, ddof=1).sum()
    total_score_variance = complete_data.sum(axis=1).var(ddof=1)
    if pd.isna(total_score_variance) or total_score_variance == 0:
        return (np.nan, participant_count)
    alpha = item_count / (item_count - 1) * (1 - summed_item_variance / total_score_variance)
    return (float(alpha), participant_count)
school_attitude_recoded_tables = {}
school_attitude_diagnostic_scores = {}
school_attitude_wave_summary_rows = []
school_attitude_item_diagnostic_rows = []
minimum_items_for_diagnostic_score = 8
for wave, configuration in school_attitude_wave_configuration.items():
    source_file = configuration['Source file']
    source_data = school_attitude_raw_tables[source_file]
    wave_item_data = pd.DataFrame(index=route_index, columns=school_attitude_core_items, dtype='Float64')
    if wave == 'Wave 3':
        timing_eligible = wave_3_pretransition_eligible.reindex(route_index).fillna(False)
    else:
        timing_eligible = pd.Series(True, index=route_index, dtype='boolean')
    for item_number, variable in configuration['Variables'].items():
        raw_values = pd.to_numeric(source_data[variable], errors='coerce')
        valid_values = raw_values.where(raw_values.isin([1, 2, 3, 4])).astype('Float64').where(timing_eligible)
        if item_number in positively_worded_items:
            recoded_values = 5 - valid_values
        else:
            recoded_values = valid_values
        wave_item_data[item_number] = recoded_values.astype('Float64')
    school_attitude_recoded_tables[wave] = wave_item_data
    answered_item_count = wave_item_data.notna().sum(axis=1)
    diagnostic_mean_score = wave_item_data.mean(axis=1,
        skipna=True).where(answered_item_count.ge(minimum_items_for_diagnostic_score)).astype('Float64')
    school_attitude_diagnostic_scores[wave] = diagnostic_mean_score
    complete_item_data = wave_item_data.dropna().astype('float64')
    cronbach_alpha, alpha_participants = calculate_cronbach_alpha(wave_item_data)
    participants_with_eight_items = int(answered_item_count.ge(minimum_items_for_diagnostic_score).sum())
    participants_with_all_items = int(answered_item_count.eq(12).sum())
    school_attitude_wave_summary_rows.append({'Wave': wave, 'Timing-eligible participants': int(timing_eligible.sum()),
        'At least eight items available': participants_with_eight_items, 'At least eight items percentage': round(participants_with_eight_items / int(timing_eligible.sum()) * 100,
        2), 'All twelve items available': participants_with_all_items, 'All twelve items percentage': round(participants_with_all_items / int(timing_eligible.sum()) * 100,
        2), "Cronbach's alpha": round(cronbach_alpha,
        3), 'Participants used for alpha': alpha_participants, 'Diagnostic mean': round(diagnostic_mean_score.mean(),
        3), 'Diagnostic standard deviation': round(diagnostic_mean_score.std(), 3)})
    total_complete_score = complete_item_data.sum(axis=1)
    for item_number in school_attitude_core_items:
        item_values = complete_item_data[item_number]
        total_without_item = total_complete_score - item_values
        item_total_correlation = item_values.corr(total_without_item)
        alpha_without_item, _ = calculate_cronbach_alpha(complete_item_data.drop(columns=[item_number]))
        school_attitude_item_diagnostic_rows.append({'Wave': wave, 'Item number': item_number,
            'Variable': configuration['Variables'][item_number], 'Item wording': school_attitude_item_label_lookup[item_number], 'Original wording direction': 'Positive' if item_number in positively_worded_items else 'Negative', 'Complete-case mean after recoding': round(item_values.mean(),
            3), 'Complete-case standard deviation': round(item_values.std(),
            3), 'Corrected item-total correlation': round(item_total_correlation,
            3), 'Alpha if item deleted': round(alpha_without_item,
            3), 'Complete-case participants': len(complete_item_data)})
school_attitude_wave_diagnostics = pd.DataFrame(school_attitude_wave_summary_rows)
school_attitude_item_diagnostics = pd.DataFrame(school_attitude_item_diagnostic_rows).sort_values(['Wave',
    'Item number']).reset_index(drop=True)
school_attitude_cross_wave_item_summary = school_attitude_item_diagnostics.groupby(['Item number', 'Item wording',
    'Original wording direction'], dropna=False).agg(Minimum_item_total_correlation=('Corrected item-total correlation',
    'min'), Maximum_item_total_correlation=('Corrected item-total correlation',
    'max'), Mean_item_total_correlation=('Corrected item-total correlation',
    'mean'), Minimum_alpha_if_deleted=('Alpha if item deleted',
    'min'), Maximum_alpha_if_deleted=('Alpha if item deleted',
    'max')).reset_index().rename(columns={'Minimum_item_total_correlation': 'Minimum item-total correlation',
    'Maximum_item_total_correlation': 'Maximum item-total correlation', 'Mean_item_total_correlation': 'Mean item-total correlation', 'Minimum_alpha_if_deleted': 'Minimum alpha if deleted', 'Maximum_alpha_if_deleted': 'Maximum alpha if deleted'}).round(3).sort_values('Item number').reset_index(drop=True)
print('Twelve-item school-attitude wave diagnostics:')
display_limited(school_attitude_wave_diagnostics)
print('Cross-wave item diagnostics:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(school_attitude_cross_wave_item_summary)
print('Wave-specific item diagnostics:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(school_attitude_item_diagnostics)

Twelve-item school-attitude wave diagnostics:


,Wave,Timing-eligible participants,At least eight items available,At least eight items percentage,All twelve items available,All twelve items percentage,Cronbach's alpha,Participants used for alpha,Diagnostic mean,Diagnostic standard deviation
0,Wave 1,9767,9264,94.85,7304,74.78,0.818,7304,3.087,0.411
1,Wave 2,9767,9264,94.85,7048,72.16,0.831,7048,2.989,0.429
2,Wave 3,9367,9169,97.89,7510,80.18,0.857,7510,3.024,0.450


Cross-wave item diagnostics:


,Item number,Item wording,Original wording direction,Minimum item-total correlation,Maximum item-total correlation,Mean item-total correlation,Minimum alpha if deleted,Maximum alpha if deleted
0,1,YP: Feelings about school: I am happy when I am at school,Positive,0.504,0.621,0.556,0.801,0.840
1,2,YP: Feelings about school: School is a waste of time for me,Negative,0.438,0.565,0.501,0.807,0.844
2,3,YP: Feelings about school: School work is worth doing,Positive,0.337,0.377,0.362,0.812,0.857
3,4,YP: Feelings about school: Most of the time I don't want to go to school,Negative,0.566,0.624,0.587,0.795,0.839
4,5,YP: Feelings about school: People think my school is a good school,Positive,0.287,0.339,0.310,0.822,0.861


Wave-specific item diagnostics:


,Wave,Item number,Variable,Item wording,Original wording direction,Complete-case mean after recoding,Complete-case standard deviation,Corrected item-total correlation,Alpha if item deleted,Complete-case participants
0,Wave 1,1,W1yys1YP,YP: Feelings about school: I am happy when I am at school,Positive,3.213,0.659,0.504,0.801,7304
1,Wave 1,2,W1yys2YP,YP: Feelings about school: School is a waste of time for me,Negative,3.604,0.643,0.438,0.807,7304
2,Wave 1,3,W1yys3YP,YP: Feelings about school: School work is worth doing,Positive,3.486,0.681,0.377,0.812,7304
3,Wave 1,4,W1yys4YP,YP: Feelings about school: Most of the time I don't want to go to school,Negative,2.939,0.834,0.566,0.795,7304
4,Wave 1,5,W1yys5YP,YP: Feelings about school: People think my school is a good school,Positive,2.992,0.800,0.287,0.822,7304


In [275]:
# 10: Cross-wave school-attitude score representation review

import numpy as np
import pandas as pd
school_attitude_wave_scores = pd.DataFrame({'Wave 1': school_attitude_diagnostic_scores['Wave 1'],
    'Wave 2': school_attitude_diagnostic_scores['Wave 2'], 'Wave 3 before September 2006': school_attitude_diagnostic_scores['Wave 3']}, index=route_index)
assert len(school_attitude_wave_scores) == len(route_index)
assert school_attitude_wave_scores.apply(lambda values: values.dropna().between(1, 4, inclusive='both').all()).all()
school_attitude_wave_availability_rows = []
for wave in school_attitude_wave_scores.columns:
    score_values = school_attitude_wave_scores[wave]
    school_attitude_wave_availability_rows.append({'Wave': wave, 'Score available': int(score_values.notna().sum()),
        'Score unavailable': int(score_values.isna().sum()), 'Available percentage': round(score_values.notna().mean() * 100,
        2), 'Mean': round(score_values.mean(), 3), 'Standard deviation': round(score_values.std(),
        3), 'Minimum': round(score_values.min(), 3), 'Maximum': round(score_values.max(), 3)})
school_attitude_wave_availability = pd.DataFrame(school_attitude_wave_availability_rows)
school_attitude_wave_count = school_attitude_wave_scores.notna().sum(axis=1)
school_attitude_wave_count_summary = school_attitude_wave_count.value_counts().sort_index().rename('Participants').rename_axis('Wave scores available').reset_index()
school_attitude_wave_count_summary['Percentage'] = (school_attitude_wave_count_summary['Participants'] / len(route_index) * 100).round(2)
school_attitude_wave_pairs = [('Wave 1', 'Wave 2'), ('Wave 1', 'Wave 3 before September 2006'), ('Wave 2',
    'Wave 3 before September 2006')]
school_attitude_pairwise_rows = []
for first_wave, second_wave in school_attitude_wave_pairs:
    pair_data = school_attitude_wave_scores[[first_wave, second_wave]].dropna().astype('float64')
    score_difference = pair_data[second_wave] - pair_data[first_wave]
    school_attitude_pairwise_rows.append({'First wave': first_wave, 'Second wave': second_wave,
        'Complete comparisons': len(pair_data), 'Pearson correlation': round(pair_data[first_wave].corr(pair_data[second_wave],
        method='pearson'), 3), 'Spearman correlation': round(pair_data[first_wave].corr(pair_data[second_wave],
        method='spearman'), 3), 'Mean change': round(score_difference.mean(),
        3), 'Mean absolute change': round(score_difference.abs().mean(),
        3), 'Change standard deviation': round(score_difference.std(), 3)})
school_attitude_pairwise_comparison = pd.DataFrame(school_attitude_pairwise_rows)
latest_school_attitude_score = pd.Series(pd.NA, index=route_index, dtype='Float64', name='school_attitude_score')
latest_school_attitude_source = pd.Series(pd.NA, index=route_index, dtype='string', name='Construction source')
wave_3_score_available = school_attitude_wave_scores['Wave 3 before September 2006'].notna()
latest_school_attitude_score.loc[wave_3_score_available] = school_attitude_wave_scores.loc[wave_3_score_available,
    'Wave 3 before September 2006']
latest_school_attitude_source.loc[wave_3_score_available] = 'Wave 3 before September 2006'
wave_2_score_fallback = latest_school_attitude_score.isna() & school_attitude_wave_scores['Wave 2'].notna()
latest_school_attitude_score.loc[wave_2_score_fallback] = school_attitude_wave_scores.loc[wave_2_score_fallback,
    'Wave 2']
latest_school_attitude_source.loc[wave_2_score_fallback] = 'Wave 2 fallback'
wave_1_score_fallback = latest_school_attitude_score.isna() & school_attitude_wave_scores['Wave 1'].notna()
latest_school_attitude_score.loc[wave_1_score_fallback] = school_attitude_wave_scores.loc[wave_1_score_fallback,
    'Wave 1']
latest_school_attitude_source.loc[wave_1_score_fallback] = 'Wave 1 fallback'
mean_school_attitude_score = school_attitude_wave_scores.mean(axis=1,
    skipna=True).where(school_attitude_wave_count.ge(1)).astype('Float64').rename('Mean school-attitude score across available waves')
latest_mean_comparison = pd.DataFrame({'Latest pre-transition score': latest_school_attitude_score,
    'Mean across available waves': mean_school_attitude_score}).dropna()
latest_minus_mean = latest_mean_comparison['Latest pre-transition score'] - latest_mean_comparison['Mean across available waves']
latest_mean_comparison_summary = pd.DataFrame([{'Complete comparisons': len(latest_mean_comparison),
    'Pearson correlation': round(latest_mean_comparison['Latest pre-transition score'].corr(latest_mean_comparison['Mean across available waves']),
    3), 'Spearman correlation': round(latest_mean_comparison['Latest pre-transition score'].corr(latest_mean_comparison['Mean across available waves'],
    method='spearman'), 3), 'Mean latest-minus-average difference': round(latest_minus_mean.mean(),
    3), 'Mean absolute difference': round(latest_minus_mean.abs().mean(),
    3), 'Difference standard deviation': round(latest_minus_mean.std(), 3)}])
latest_school_attitude_source_summary = latest_school_attitude_source.fillna('Unavailable').value_counts().rename('Participants').rename_axis('Construction source').reset_index()
latest_school_attitude_source_summary['Percentage'] = (latest_school_attitude_source_summary['Participants'] / len(route_index) * 100).round(2)
school_attitude_representation_coverage = pd.DataFrame([{'Representation': 'Latest available pre-transition score',
    'Available': int(latest_school_attitude_score.notna().sum()), 'Unavailable': int(latest_school_attitude_score.isna().sum()), 'Available percentage': round(latest_school_attitude_score.notna().mean() * 100,
    2), 'Mean': round(latest_school_attitude_score.mean(),
    3), 'Standard deviation': round(latest_school_attitude_score.std(),
    3)}, {'Representation': 'Mean across available pre-transition waves',
    'Available': int(mean_school_attitude_score.notna().sum()), 'Unavailable': int(mean_school_attitude_score.isna().sum()), 'Available percentage': round(mean_school_attitude_score.notna().mean() * 100,
    2), 'Mean': round(mean_school_attitude_score.mean(),
    3), 'Standard deviation': round(mean_school_attitude_score.std(), 3)}])
print('Wave-specific score availability:')
display_limited(school_attitude_wave_availability)
print('Number of wave scores available per participant:')
display_limited(school_attitude_wave_count_summary)
print('Pairwise cross-wave score comparison:')
display_limited(school_attitude_pairwise_comparison)
print('Latest-score construction source:')
display_limited(latest_school_attitude_source_summary)
print('Candidate representation coverage:')
display_limited(school_attitude_representation_coverage)
print('Latest score compared with the cross-wave mean:')
display_limited(latest_mean_comparison_summary)

Wave-specific score availability:


,Wave,Score available,Score unavailable,Available percentage,Mean,Standard deviation,Minimum,Maximum
0,Wave 1,9264,503,94.85,3.087,0.411,1.0,4.0
1,Wave 2,9264,503,94.85,2.989,0.429,1.0,4.0
2,Wave 3 before September 2006,9169,598,93.88,3.024,0.450,1.0,4.0


Number of wave scores available per participant:


,Wave scores available,Participants,Percentage
0,0,256,2.62
1,1,82,0.84
2,2,672,6.88
3,3,8757,89.66


Pairwise cross-wave score comparison:


...

,Construction source,Participants,Percentage
0,Wave 3 before September 2006,9169,93.88
1,Wave 2 fallback,309,3.16
2,Unavailable,256,2.62
3,Wave 1 fallback,33,0.34


Candidate representation coverage:


,Representation,Available,Unavailable,Available percentage,Mean,Standard deviation
0,Latest available pre-transition score,9511,256,97.38,3.018,0.453
1,Mean across available pre-transition waves,9511,256,97.38,3.030,0.379


Latest score compared with the cross-wave mean:


,Complete comparisons,Pearson correlation,Spearman correlation,Mean latest-minus-average difference,Mean absolute difference,Difference standard deviation
0,9511,0.883,0.872,-0.012,0.16,0.213


In [276]:
# 11: School-attitude score representation decision

import pandas as pd
school_attitude_score_candidate = latest_school_attitude_score.copy().rename('school_attitude_score')
school_attitude_score_source = latest_school_attitude_source.copy().rename('school_attitude_score_source')
assert len(school_attitude_score_candidate) == len(route_index)
assert school_attitude_score_candidate.dropna().between(1, 4, inclusive='both').all()
assert int(school_attitude_score_candidate.notna().sum()) == 9511
assert int(school_attitude_score_candidate.isna().sum()) == 256
school_attitude_representation_review = pd.DataFrame({'school_attitude_score': school_attitude_score_candidate,
    'school_attitude_score_source': school_attitude_score_source, 'cross_wave_mean_for_review': mean_school_attitude_score, 'wave_scores_available': school_attitude_wave_count}, index=route_index)
school_attitude_representation_review['school_attitude_score_source'] = school_attitude_representation_review['school_attitude_score_source'].fillna('Unavailable')
school_attitude_representation_review['latest_minus_cross_wave_mean'] = school_attitude_representation_review['school_attitude_score'] - school_attitude_representation_review['cross_wave_mean_for_review']
school_attitude_source_comparison = school_attitude_representation_review.groupby('school_attitude_score_source',
    dropna=False).agg(Participants=('school_attitude_score_source', 'size'), Score_available=('school_attitude_score',
    lambda values: int(values.notna().sum())), Mean_score=('school_attitude_score',
    'mean'), Standard_deviation=('school_attitude_score', 'std'), Mean_cross_wave_score=('cross_wave_mean_for_review',
    'mean'), Mean_latest_minus_average=('latest_minus_cross_wave_mean',
    'mean'), Mean_wave_scores_available=('wave_scores_available',
    'mean')).reset_index().rename(columns={'school_attitude_score_source': 'Construction source',
    'Score_available': 'Score available', 'Mean_score': 'Mean score', 'Standard_deviation': 'Standard deviation', 'Mean_cross_wave_score': 'Mean cross-wave score', 'Mean_latest_minus_average': 'Mean latest-minus-average difference', 'Mean_wave_scores_available': 'Mean number of wave scores available'})
numeric_source_columns = ['Mean score', 'Standard deviation', 'Mean cross-wave score',
    'Mean latest-minus-average difference', 'Mean number of wave scores available']
school_attitude_source_comparison[numeric_source_columns] = school_attitude_source_comparison[numeric_source_columns].round(3)
school_attitude_candidate_quality = pd.DataFrame([{'Predictor': 'school_attitude_score',
    'Construct': 'Positive attitudes towards school and engagement with lessons', 'Representation': 'Latest available pre-transition twelve-item mean score', 'Primary source': 'Wave 3 interview before September 2006', 'Fallback order': 'Wave 2, then Wave 1', 'Items required': 'At least 8 of 12', 'Non-missing': int(school_attitude_score_candidate.notna().sum()), 'Missing': int(school_attitude_score_candidate.isna().sum()), 'Missing percentage': round(school_attitude_score_candidate.isna().mean() * 100,
    2), 'Mean': round(school_attitude_score_candidate.mean(),
    3), 'Standard deviation': round(school_attitude_score_candidate.std(),
    3), 'Minimum': round(school_attitude_score_candidate.min(),
    3), 'Maximum': round(school_attitude_score_candidate.max(), 3)}])
school_attitude_representation_decision = pd.DataFrame([{'Representation': 'Latest available pre-transition score',
    'Provisional decision': 'Retain as Domain 6 predictor candidate', 'Reason': 'Preserves temporal proximity to transition; Wave 3 is used for most participants and earlier waves are used only as fallback'}, {'Representation': 'Mean across available pre-transition waves',
    'Provisional decision': 'Retain as review support only', 'Reason': 'Has identical coverage but combines different numbers of waves and smooths temporal variation'}, {'Representation': 'Three separate wave-specific scores',
    'Provisional decision': 'Do not retain', 'Reason': 'Would duplicate one construct and expand the predictor set without a distinct purpose'}, {'Representation': 'Twelve individual school-attitude items',
    'Provisional decision': 'Do not retain', 'Reason': 'The items form a coherent repeated battery and are represented by one derived score'}])
print('School-attitude candidate quality:')
display_limited(school_attitude_candidate_quality)
print('Score distribution by construction source:')
display_limited(school_attitude_source_comparison)
print('Representation decision:')
with pd.option_context('display.max_colwidth', None):
    display_limited(school_attitude_representation_decision)

School-attitude candidate quality:


,Predictor,Construct,Representation,Primary source,Fallback order,Items required,Non-missing,Missing,Missing percentage,Mean,Standard deviation,Minimum,Maximum
0,school_attitude_score,Positive attitudes towards school and engageme...,Latest available pre-transition twelve-item me...,Wave 3 interview before September 2006,"Wave 2, then Wave 1",At least 8 of 12,9511,256,2.62,3.018,0.453,1.0,4.0


Score distribution by construction source:


,Construction source,Participants,Score available,Mean score,Standard deviation,Mean cross-wave score,Mean latest-minus-average difference,Mean number of wave scores available
0,Unavailable,256,0,<NA>,<NA>,<NA>,<NA>,0.000
1,Wave 1 fallback,33,33,2.834,0.565,2.834,0.0,1.000
2,Wave 2 fallback,309,309,2.858,0.487,2.896,-0.038,1.926
3,Wave 3 before September 2006,9169,9169,3.024,0.45,3.035,-0.011,2.952


Representation decision:


,Representation,Provisional decision,Reason
0,Latest available pre-transition score,Retain as Domain 6 predictor candidate,Preserves temporal proximity to transition; Wave 3 is used for most participants and earlier waves are used only as fallback
1,Mean across available pre-transition waves,Retain as review support only,Has identical coverage but combines different numbers of waves and smooths temporal variation
2,Three separate wave-specific scores,Do not retain,Would duplicate one construct and expand the predictor set without a distinct purpose
3,Twelve individual school-attitude items,Do not retain,The items form a coherent repeated battery and are represented by one derived score


In [277]:
# 12: Additional school-battery item representation review

import pandas as pd
additional_school_item_configuration = {13: {'Construct': 'School physical environment', 'Sources': [('Wave 2',
    'W2YYS13YP'), ('Wave 1', 'W1yys13YP')], 'Valid codes': [1, 2, 3,
    4], 'Recoding': 'reverse'}, 14: {'Construct': 'Teacher expectations, management and feedback',
    'Sources': [('Wave 2', 'W2YYS14YP'), ('Wave 1', 'W1yys14YP')], 'Valid codes': [1, 2, 3, 4,
    5], 'Recoding': 'reverse'}, 15: {'Construct': 'Teacher expectations, management and feedback',
    'Sources': [('Wave 2', 'W2YYS15YP'), ('Wave 1', 'W1yys15YP')], 'Valid codes': [1, 2, 3, 4,
    5], 'Recoding': 'reverse'}, 16: {'Construct': 'Teacher expectations, management and feedback',
    'Sources': [('Wave 2', 'W2YYS16YP'), ('Wave 1', 'W1yys16YP')], 'Valid codes': [1, 2, 3, 4,
    5], 'Recoding': 'reverse'}, 17: {'Construct': 'Teacher relationship, affirmation and fairness',
    'Sources': [('Wave 2', 'W2YYS17YP'), ('Wave 1', 'W1yys17YP')], 'Valid codes': [1, 2, 3, 4,
    5], 'Recoding': 'reverse'}, 18: {'Construct': 'Teacher relationship, affirmation and fairness',
    'Sources': [('Wave 2', 'W2YYS18YP'), ('Wave 1', 'W1yys18YP')], 'Valid codes': [1, 2, 3, 4,
    5], 'Recoding': 'reverse'}, 19: {'Construct': 'Teacher expectations, management and feedback',
    'Sources': [('Wave 2', 'W2YYS19YP'), ('Wave 1', 'W1yys19YP')], 'Valid codes': [1, 2, 3, 4,
    5], 'Recoding': 'reverse'}, 20: {'Construct': 'Teacher expectations, management and feedback',
    'Sources': [('Wave 2', 'W2yys20YP'), ('Wave 1', 'W1yys20YP')], 'Valid codes': [1, 2,
    3], 'Recoding': 'reverse'}, 21: {'Construct': 'Teacher expectations, management and feedback',
    'Sources': [('Wave 2', 'W2yys21YP'), ('Wave 1', 'W1yys21YP')], 'Valid codes': [1, 2,
    3], 'Recoding': 'reverse'}, 22: {'Construct': 'Academic self-concept', 'Sources': [('Wave 1', 'W1yys22YP')],
    'Valid codes': [1, 2, 3, 4, 5], 'Recoding': 'reverse'}, 23: {'Construct': 'Academic self-concept',
    'Sources': [('Wave 1', 'W1yys23YP')], 'Valid codes': [1, 2, 3, 4,
    5], 'Recoding': 'reverse'}, 24: {'Construct': 'Teacher relationship, affirmation and fairness',
    'Sources': [('Wave 2', 'W2YYS24YP')], 'Valid codes': [1, 2, 3, 4,
    5], 'Recoding': 'reverse'}, 25: {'Construct': 'Teacher relationship, affirmation and fairness',
    'Sources': [('Wave 2', 'W2YYS25YP')], 'Valid codes': [1, 2, 3, 4,
    5], 'Recoding': 'retain'}, 26: {'Construct': 'Teacher relationship, affirmation and fairness',
    'Sources': [('Wave 2', 'W2YYS26YP')], 'Valid codes': [1, 2, 3, 4, 5], 'Recoding': 'retain'}}
additional_school_wave_table_lookup = {'Wave 1': school_attitude_raw_tables['wave_one_lsype_young_person_2020'],
    'Wave 2': school_attitude_raw_tables['wave_two_lsype_young_person_2020']}
additional_school_item_label_lookup = school_attitude_battery_inventory.loc[school_attitude_battery_inventory['School-attitude item number'].between(13,
    26, inclusive='both')].sort_values(['School-attitude item number', 'Source order',
    'Variable position']).groupby('School-attitude item number')['Variable label'].first().str.strip().to_dict()
assert len(additional_school_item_label_lookup) == 14
additional_school_item_values = pd.DataFrame(index=route_index)
additional_school_item_sources = pd.DataFrame(index=route_index)
additional_school_item_quality_rows = []
for item_number, configuration in additional_school_item_configuration.items():
    item_values = pd.Series(pd.NA, index=route_index, dtype='Float64')
    item_source = pd.Series(pd.NA, index=route_index, dtype='string')
    valid_codes = configuration['Valid codes']
    maximum_code = max(valid_codes)
    for wave, variable in configuration['Sources']:
        raw_values = pd.to_numeric(additional_school_wave_table_lookup[wave][variable], errors='coerce')
        valid_values = raw_values.where(raw_values.isin(valid_codes)).astype('Float64')
        if configuration['Recoding'] == 'reverse':
            recoded_values = maximum_code + 1 - valid_values
        else:
            recoded_values = valid_values
        fill_mask = item_values.isna() & recoded_values.notna()
        item_values.loc[fill_mask] = recoded_values.loc[fill_mask]
        item_source.loc[fill_mask] = wave
    item_column = f'school_battery_item_{item_number}'
    source_column = f'school_battery_item_{item_number}_source'
    additional_school_item_values[item_column] = item_values
    additional_school_item_sources[source_column] = item_source
    item_school_attitude_pair = pd.concat([item_values.rename('Additional item'),
        school_attitude_score_candidate.rename('School-attitude score')], axis=1).dropna().astype('float64')
    source_counts = item_source.fillna('Unavailable').value_counts()
    additional_school_item_quality_rows.append({'Item number': item_number, 'Construct': configuration['Construct'],
        'Variable label': additional_school_item_label_lookup[item_number], 'Representation': 'Latest available item response', 'Primary source': configuration['Sources'][0][0], 'Fallback source': configuration['Sources'][1][0] if len(configuration['Sources']) > 1 else 'None', 'Non-missing': int(item_values.notna().sum()), 'Missing': int(item_values.isna().sum()), 'Missing percentage': round(item_values.isna().mean() * 100,
        2), 'Mean after positive-direction recoding': round(item_values.mean(),
        3), 'Standard deviation': round(item_values.std(),
        3), 'Minimum': item_values.min(), 'Maximum': item_values.max(), 'Wave 2 source': int(source_counts.get('Wave 2',
        0)), 'Wave 1 source': int(source_counts.get('Wave 1', 0)), 'Unavailable': int(source_counts.get('Unavailable',
        0)), 'Complete comparisons with school-attitude score': len(item_school_attitude_pair), 'Spearman correlation with school-attitude score': round(item_school_attitude_pair['Additional item'].corr(item_school_attitude_pair['School-attitude score'],
        method='spearman'), 3)})
additional_school_item_quality = pd.DataFrame(additional_school_item_quality_rows).sort_values('Item number').reset_index(drop=True)
additional_school_construct_rows = []
for construct, construct_items in additional_school_item_quality.groupby('Construct', sort=False):
    construct_item_numbers = construct_items['Item number'].tolist()
    construct_columns = [f'school_battery_item_{item_number}' for item_number in construct_item_numbers]
    construct_data = additional_school_item_values[construct_columns].astype('float64')
    correlation_matrix = construct_data.corr(method='spearman', min_periods=100)
    pairwise_correlations = []
    for first_position in range(len(construct_columns)):
        for second_position in range(first_position + 1, len(construct_columns)):
            correlation_value = correlation_matrix.iloc[first_position, second_position]
            if pd.notna(correlation_value):
                pairwise_correlations.append(float(correlation_value))
    additional_school_construct_rows.append({'Construct': construct,
        'Items': '; '.join((str(item_number) for item_number in construct_item_numbers)), 'Number of items': len(construct_item_numbers), 'Minimum pairwise Spearman correlation': round(min(pairwise_correlations),
        3) if pairwise_correlations else pd.NA, 'Mean pairwise Spearman correlation': round(sum(pairwise_correlations) / len(pairwise_correlations),
        3) if pairwise_correlations else pd.NA, 'Maximum pairwise Spearman correlation': round(max(pairwise_correlations),
        3) if pairwise_correlations else pd.NA, 'Minimum item correlation with school-attitude score': construct_items['Spearman correlation with school-attitude score'].min(), 'Maximum item correlation with school-attitude score': construct_items['Spearman correlation with school-attitude score'].max()})
additional_school_construct_summary = pd.DataFrame(additional_school_construct_rows)
additional_school_construct_review_plan = pd.DataFrame([{'Construct': 'School physical environment',
    'Review position': 'Consider under school context rather than the core school-attitude construct'}, {'Construct': 'Teacher expectations, management and feedback',
    'Review position': 'Assess whether one consolidated representation adds a distinct construct'}, {'Construct': 'Teacher relationship, affirmation and fairness',
    'Review position': 'Assess whether one consolidated representation adds a distinct construct'}, {'Construct': 'Academic self-concept',
    'Review position': 'Reserve for comparison with the later psychosocial-domain review'}])
assert len(additional_school_item_quality) == 14
assert additional_school_item_values.apply(lambda values: values.dropna().between(1, 5, inclusive='both').all()).all()
print('Additional school-battery item quality:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(additional_school_item_quality)
print('Within-construct association review:')
display_limited(additional_school_construct_summary)
print('Construct review plan:')
with pd.option_context('display.max_colwidth', None):
    display_limited(additional_school_construct_review_plan)

Additional school-battery item quality:


,Item number,Construct,Variable label,Representation,Primary source,Fallback source,Non-missing,Missing,Missing percentage,Mean after positive-direction recoding,Standard deviation,Minimum,Maximum,Wave 2 source,Wave 1 source,Unavailable,Complete comparisons with school-attitude score,Spearman correlation with school-attitude score
0,13,School physical environment,YP: Feelings about school: My school is clean and tidy,Latest available item response,Wave 2,Wave 1,9431,336,3.44,2.519,0.815,1.0,4.0,9036,395,336,9429,0.271
1,14,"Teacher expectations, management and feedback",YP: How many teachers this applies to: My teachers make sure we do any homework,Latest available item response,Wave 2,Wave 1,9484,283,2.90,3.862,0.889,1.0,5.0,9298,186,283,9481,0.309
2,15,"Teacher expectations, management and feedback",YP: How many teachers this applies to: The teachers at my school make it clear h,Latest available item response,Wave 2,Wave 1,9487,280,2.87,4.163,0.784,1.0,5.0,9318,169,280,9485,0.254
3,16,"Teacher expectations, management and feedback",YP: How many teachers this applies to: The teachers in my school take action whe,Latest available item response,Wave 2,Wave 1,9486,281,2.88,4.052,0.811,1.0,5.0,9318,168,281,9484,0.232
4,17,"Teacher relationship, affirmation and fairness",YP: How many teachers this applies to: My teachers praise me when I do my school,Latest available item response,Wave 2,Wave 1,9483,284,2.91,3.655,0.967,1.0,5.0,9303,180,284,9481,0.264


Within-construct association review:


,Construct,Items,Number of items,Minimum pairwise Spearman correlation,Mean pairwise Spearman correlation,Maximum pairwise Spearman correlation,Minimum item correlation with school-attitude score,Maximum item correlation with school-attitude score
0,School physical environment,13,1,<NA>,<NA>,<NA>,0.271,0.271
1,"Teacher expectations, management and feedback",14; 15; 16; 19; 20; 21,6,0.246,0.335,0.501,0.232,0.309
2,"Teacher relationship, affirmation and fairness",17; 18; 24; 25; 26,5,0.213,0.299,0.468,0.192,0.406
3,Academic self-concept,22; 23,2,0.715,0.715,0.715,0.310,0.318


Construct review plan:


,Construct,Review position
0,School physical environment,Consider under school context rather than the core school-attitude construct
1,"Teacher expectations, management and feedback",Assess whether one consolidated representation adds a distinct construct
2,"Teacher relationship, affirmation and fairness",Assess whether one consolidated representation adds a distinct construct
3,Academic self-concept,Reserve for comparison with the later psychosocial-domain review


In [278]:
# 13: Teacher-related construct scale diagnostics

import math
import numpy as np
import pandas as pd
teacher_construct_configuration = {'Teacher expectations, management and feedback': {'Items': [14, 15, 16, 19, 20, 21],
    'Minimum items for review score': 4}, 'Teacher relationship, affirmation and fairness': {'Items': [17, 18, 24, 25,
    26], 'Minimum items for review score': 3}}

def calculate_alpha(item_data):
    """Calculate Cronbach's alpha from complete cases."""
    complete_data = item_data.dropna().astype('float64')
    participant_count = len(complete_data)
    item_count = complete_data.shape[1]
    if participant_count < 2 or item_count < 2:
        return (np.nan, participant_count)
    item_variance_sum = complete_data.var(axis=0, ddof=1).sum()
    total_variance = complete_data.sum(axis=1).var(ddof=1)
    if pd.isna(total_variance) or total_variance == 0:
        return (np.nan, participant_count)
    alpha = item_count / (item_count - 1) * (1 - item_variance_sum / total_variance)
    return (float(alpha), participant_count)

def calculate_standardised_alpha(item_data):
    """Calculate standardised alpha from complete cases."""
    complete_data = item_data.dropna().astype('float64')
    item_count = complete_data.shape[1]
    if len(complete_data) < 2 or item_count < 2:
        return (np.nan, np.nan)
    correlation_matrix = complete_data.corr()
    upper_triangle_values = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape),
        k=1).astype(bool)).stack()
    mean_inter_item_correlation = upper_triangle_values.mean()
    standardised_alpha = item_count * mean_inter_item_correlation / (1 + (item_count - 1) * mean_inter_item_correlation)
    return (float(standardised_alpha), float(mean_inter_item_correlation))
teacher_item_numbers = sorted({item_number for configuration in teacher_construct_configuration.values() for item_number in configuration['Items']})
teacher_item_scaled_values = pd.DataFrame(index=route_index)
for item_number in teacher_item_numbers:
    item_column = f'school_battery_item_{item_number}'
    item_values = additional_school_item_values[item_column].astype('Float64')
    valid_codes = additional_school_item_configuration[item_number]['Valid codes']
    minimum_code = min(valid_codes)
    maximum_code = max(valid_codes)
    scaled_values = (item_values - minimum_code) / (maximum_code - minimum_code)
    teacher_item_scaled_values[item_column] = scaled_values.astype('Float64')
assert teacher_item_scaled_values.apply(lambda values: values.dropna().between(0, 1, inclusive='both').all()).all()
teacher_construct_review_scores = pd.DataFrame(index=route_index)
teacher_construct_summary_rows = []
teacher_item_diagnostic_rows = []
for construct, configuration in teacher_construct_configuration.items():
    item_numbers = configuration['Items']
    minimum_items = configuration['Minimum items for review score']
    item_columns = [f'school_battery_item_{item_number}' for item_number in item_numbers]
    construct_data = teacher_item_scaled_values[item_columns].copy()
    complete_construct_data = construct_data.dropna().astype('float64')
    raw_alpha, alpha_participants = calculate_alpha(construct_data)
    standardised_alpha, mean_inter_item_correlation = calculate_standardised_alpha(construct_data)
    answered_item_count = construct_data.notna().sum(axis=1)
    review_score = construct_data.mean(axis=1,
        skipna=True).where(answered_item_count.ge(minimum_items)).astype('Float64')
    score_name = construct.lower().replace(',', '').replace(' ', '_') + '_review_score'
    teacher_construct_review_scores[score_name] = review_score
    school_attitude_pair = pd.concat([review_score.rename('Teacher construct'),
        school_attitude_score_candidate.rename('School attitude')], axis=1).dropna().astype('float64')
    teacher_construct_summary_rows.append({'Construct': construct,
        'Items': '; '.join((str(item_number) for item_number in item_numbers)), 'Number of items': len(item_numbers), 'Minimum items for review score': minimum_items, 'Complete cases for alpha': alpha_participants, "Cronbach's alpha after 0–1 scaling": round(raw_alpha,
        3), "Standardised Cronbach's alpha": round(standardised_alpha,
        3), 'Mean inter-item correlation': round(mean_inter_item_correlation,
        3), 'Review score available': int(review_score.notna().sum()), 'Review score missing': int(review_score.isna().sum()), 'Missing percentage': round(review_score.isna().mean() * 100,
        2), 'Mean review score': round(review_score.mean(), 3), 'Standard deviation': round(review_score.std(),
        3), 'Spearman correlation with school-attitude score': round(school_attitude_pair['Teacher construct'].corr(school_attitude_pair['School attitude'],
        method='spearman'), 3), 'Complete comparisons with school-attitude score': len(school_attitude_pair)})
    complete_total_score = complete_construct_data.sum(axis=1)
    for item_number, item_column in zip(item_numbers, item_columns):
        item_values = complete_construct_data[item_column]
        total_without_item = complete_total_score - item_values
        corrected_item_total_correlation = item_values.corr(total_without_item)
        remaining_item_data = complete_construct_data.drop(columns=[item_column])
        alpha_if_deleted, _ = calculate_alpha(remaining_item_data)
        standardised_alpha_if_deleted, mean_correlation_if_deleted = calculate_standardised_alpha(remaining_item_data)
        teacher_item_diagnostic_rows.append({'Construct': construct, 'Item number': item_number,
            'Variable label': additional_school_item_label_lookup[item_number], 'Complete-case mean after 0–1 scaling': round(item_values.mean(),
            3), 'Complete-case standard deviation': round(item_values.std(),
            3), 'Corrected item-total correlation': round(corrected_item_total_correlation,
            3), 'Alpha if item deleted': round(alpha_if_deleted,
            3), 'Standardised alpha if item deleted': round(standardised_alpha_if_deleted,
            3), 'Mean inter-item correlation if deleted': round(mean_correlation_if_deleted,
            3), 'Complete-case participants': len(complete_construct_data)})
teacher_construct_scale_diagnostics = pd.DataFrame(teacher_construct_summary_rows)
teacher_construct_item_diagnostics = pd.DataFrame(teacher_item_diagnostic_rows).sort_values(['Construct',
    'Item number']).reset_index(drop=True)
teacher_score_pair = teacher_construct_review_scores.dropna().astype('float64')
teacher_score_association = pd.DataFrame([{'First construct': teacher_construct_scale_diagnostics.loc[0, 'Construct'],
    'Second construct': teacher_construct_scale_diagnostics.loc[1,
    'Construct'], 'Complete comparisons': len(teacher_score_pair), 'Pearson correlation': round(teacher_score_pair.iloc[:,
    0].corr(teacher_score_pair.iloc[:, 1],
    method='pearson'), 3), 'Spearman correlation': round(teacher_score_pair.iloc[:, 0].corr(teacher_score_pair.iloc[:,
    1], method='spearman'), 3)}])
print('Teacher-related construct scale diagnostics:')
with pd.option_context('display.max_colwidth', None):
    display_limited(teacher_construct_scale_diagnostics)
print('Teacher-related item diagnostics:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(teacher_construct_item_diagnostics)
print('Association between provisional teacher scores:')
display_limited(teacher_score_association)

Teacher-related construct scale diagnostics:


,Construct,Items,Number of items,Minimum items for review score,Complete cases for alpha,Cronbach's alpha after 0–1 scaling,Standardised Cronbach's alpha,Mean inter-item correlation,Review score available,Review score missing,Missing percentage,Mean review score,Standard deviation,Spearman correlation with school-attitude score,Complete comparisons with school-attitude score
0,"Teacher expectations, management and feedback",14; 15; 16; 19; 20; 21,6,4,9435,0.744,0.761,0.347,9488,279,2.86,0.722,0.154,0.399,9486
1,"Teacher relationship, affirmation and fairness",17; 18; 24; 25; 26,5,3,8862,0.666,0.672,0.290,9329,438,4.48,0.717,0.152,0.441,9327


Teacher-related item diagnostics:


,Construct,Item number,Variable label,Complete-case mean after 0–1 scaling,Complete-case standard deviation,Corrected item-total correlation,Alpha if item deleted,Standardised alpha if item deleted,Mean inter-item correlation if deleted,Complete-case participants
0,"Teacher expectations, management and feedback",14,YP: How many teachers this applies to: My teachers make sure we do any homework,0.716,0.221,0.525,0.696,0.719,0.338,9435
1,"Teacher expectations, management and feedback",15,YP: How many teachers this applies to: The teachers at my school make it clear h,0.791,0.195,0.553,0.693,0.708,0.327,9435
2,"Teacher expectations, management and feedback",16,YP: How many teachers this applies to: The teachers in my school take action whe,0.763,0.202,0.519,0.700,0.718,0.337,9435
3,"Teacher expectations, management and feedback",19,YP: How many teachers this applies to: My teachers can keep order in class,0.628,0.195,0.523,0.700,0.720,0.339,9435
4,"Teacher expectations, management and feedback",20,YP: How hard teachers make YP work,0.864,0.259,0.412,0.731,0.750,0.375,9435


Association between provisional teacher scores:


,First construct,Second construct,Complete comparisons,Pearson correlation,Spearman correlation
0,"Teacher expectations, management and feedback","Teacher relationship, affirmation and fairness",9329,0.586,0.563


In [279]:
# 14: Domain 6 scope checkpoint

import pandas as pd
teacher_expectation_metrics = teacher_construct_scale_diagnostics.loc[teacher_construct_scale_diagnostics['Construct'].eq('Teacher expectations, management and feedback')].iloc[0]
teacher_relationship_metrics = teacher_construct_scale_diagnostics.loc[teacher_construct_scale_diagnostics['Construct'].eq('Teacher relationship, affirmation and fairness')].iloc[0]
teacher_score_overlap = teacher_score_association.iloc[0]
domain_6_representation_decisions = pd.DataFrame([{'Construct': 'Positive attitudes towards school and engagement with lessons',
    'Candidate representation': 'school_attitude_score', 'Current decision': 'Retain as Domain 6 predictor candidate', 'Reason': 'Coherent twelve-item repeated battery with strong internal consistency and high coverage'}, {'Construct': 'Teacher expectations, management and feedback',
    'Candidate representation': 'Six-item teacher expectations review score', 'Current decision': 'Retain as review support only', 'Reason': 'Acceptable internal consistency, but moderate overlap with the broader school-attitude score and no sufficiently distinct role established'}, {'Construct': 'Teacher relationship, affirmation and fairness',
    'Candidate representation': 'Five-item teacher relationship review score', 'Current decision': 'Retain as review support only', 'Reason': 'Lower internal consistency and moderate overlap with both the school-attitude score and the teacher expectations score'}, {'Construct': 'School physical environment',
    'Candidate representation': 'Clean and tidy school item', 'Current decision': 'Defer to Domain 10', 'Reason': "Represents school context rather than the young person's school attitude or engagement"}, {'Construct': 'Academic self-concept',
    'Candidate representation': 'Two academic self-assessment items', 'Current decision': 'Defer to Domain 7', 'Reason': 'Requires comparison with the wider psychosocial domain before deciding whether it adds a separate predictor'}])
domain_6_teacher_diagnostic_summary = pd.DataFrame([{'Construct': teacher_expectation_metrics['Construct'],
    'Number of items': int(teacher_expectation_metrics['Number of items']), "Cronbach's alpha": teacher_expectation_metrics["Cronbach's alpha after 0–1 scaling"], 'Mean inter-item correlation': teacher_expectation_metrics['Mean inter-item correlation'], 'Correlation with school-attitude score': teacher_expectation_metrics['Spearman correlation with school-attitude score'], 'Correlation with other teacher score': teacher_score_overlap['Spearman correlation'], 'Representation decision': 'Review support only'}, {'Construct': teacher_relationship_metrics['Construct'],
    'Number of items': int(teacher_relationship_metrics['Number of items']), "Cronbach's alpha": teacher_relationship_metrics["Cronbach's alpha after 0–1 scaling"], 'Mean inter-item correlation': teacher_relationship_metrics['Mean inter-item correlation'], 'Correlation with school-attitude score': teacher_relationship_metrics['Spearman correlation with school-attitude score'], 'Correlation with other teacher score': teacher_score_overlap['Spearman correlation'], 'Representation decision': 'Review support only'}])
domain_6_stopping_rule = pd.DataFrame([{'Rule': 'Current predictor scope',
    'Decision': 'Retain one school-attitude predictor candidate', 'Application': 'Do not add teacher-related scores alongside the broader school-attitude score'}, {'Rule': 'Permitted additional construct',
    'Decision': 'Review only one consolidated attendance or discipline representation', 'Application': 'The construct must represent observed behavioural engagement rather than another attitude measure'}, {'Rule': 'One representation per construct',
    'Decision': 'Do not retain routed items, wave-specific duplicates or several indicators of the same event', 'Application': 'Attendance, truancy, suspension and exclusion must be consolidated where substantively justified'}, {'Rule': 'Admission requirement',
    'Decision': 'Add a second Domain 6 predictor only if it is clearly distinct, pre-transition and well covered', 'Application': 'Otherwise close Domain 6 with school_attitude_score only'}, {'Rule': 'Deferred constructs',
    'Decision': 'Review school environment in Domain 10 and academic self-concept in Domain 7', 'Application': 'Do not count these as Domain 6 predictors'}])
domain_6_predictor_candidates = pd.DataFrame({'school_attitude_score': school_attitude_score_candidate},
    index=route_index)
assert list(domain_6_predictor_candidates.columns) == ['school_attitude_score']
assert int(domain_6_predictor_candidates['school_attitude_score'].notna().sum()) == 9511
print(f'Current Domain 6 predictor candidates: {len(domain_6_predictor_candidates.columns)}')
print('Teacher-related diagnostic summary:')
display_limited(domain_6_teacher_diagnostic_summary)
print('Provisional representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_6_representation_decisions)
print('Domain 6 stopping rule:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_6_stopping_rule)

Current Domain 6 predictor candidates: 1
Teacher-related diagnostic summary:


,Construct,Number of items,Cronbach's alpha,Mean inter-item correlation,Correlation with school-attitude score,Correlation with other teacher score,Representation decision
0,"Teacher expectations, management and feedback",6,0.744,0.347,0.399,0.563,Review support only
1,"Teacher relationship, affirmation and fairness",5,0.666,0.290,0.441,0.563,Review support only


Provisional representation decisions:


,Construct,Candidate representation,Current decision,Reason
0,Positive attitudes towards school and engagement with lessons,school_attitude_score,Retain as Domain 6 predictor candidate,Coherent twelve-item repeated battery with strong internal consistency and high coverage
1,"Teacher expectations, management and feedback",Six-item teacher expectations review score,Retain as review support only,"Acceptable internal consistency, but moderate overlap with the broader school-attitude score and no sufficiently distinct role established"
2,"Teacher relationship, affirmation and fairness",Five-item teacher relationship review score,Retain as review support only,Lower internal consistency and moderate overlap with both the school-attitude score and the teacher expectations score
3,School physical environment,Clean and tidy school item,Defer to Domain 10,Represents school context rather than the young person's school attitude or engagement
4,Academic self-concept,Two academic self-assessment items,Defer to Domain 7,Requires comparison with the wider psychosocial domain before deciding whether it adds a separate predictor


Domain 6 stopping rule:


,Rule,Decision,Application
0,Current predictor scope,Retain one school-attitude predictor candidate,Do not add teacher-related scores alongside the broader school-attitude score
1,Permitted additional construct,Review only one consolidated attendance or discipline representation,The construct must represent observed behavioural engagement rather than another attitude measure
2,One representation per construct,"Do not retain routed items, wave-specific duplicates or several indicators of the same event","Attendance, truancy, suspension and exclusion must be consolidated where substantively justified"
3,Admission requirement,"Add a second Domain 6 predictor only if it is clearly distinct, pre-transition and well covered",Otherwise close Domain 6 with school_attitude_score only
4,Deferred constructs,Review school environment in Domain 10 and academic self-concept in Domain 7,Do not count these as Domain 6 predictors


In [280]:
# 15: Attendance and discipline candidate structure review

import pandas as pd
attendance_discipline_inventory = school_engagement_review_tracks.loc[school_engagement_review_tracks['Review track'].str.contains('Attendance, behaviour and discipline',
    regex=False, na=False)].copy()
assert len(attendance_discipline_inventory) == 33
assert attendance_discipline_inventory['Variable'].is_unique
attendance_discipline_inventory['Variable stem'] = attendance_discipline_inventory['Variable'].astype('string').str.replace('(?i)^W[123]',
    '', regex=True).str.lower()
attendance_discipline_search_text = attendance_discipline_inventory['Variable'].fillna('').astype('string') + ' ' + attendance_discipline_inventory['Variable label'].fillna('').astype('string')
reason_or_routing_mask = attendance_discipline_search_text.str.contains('\\breason\\b|\\bwhy\\b|\\bdue to\\b',
    case=False, regex=True, na=False)
frequency_or_count_mask = attendance_discipline_search_text.str.contains('\\bnumber of\\b|\\bhow many\\b|\\bfrequency\\b|\\bhow often\\b|\\btimes\\b|\\bdays\\b|\\bweeks\\b|\\bmonths\\b',
    case=False, regex=True, na=False)
derived_measure_mask = attendance_discipline_inventory['Variable label'].fillna('').astype('string').str.startswith('DV:')
attendance_discipline_inventory['Descriptive item type'] = 'Direct status or experience item'
attendance_discipline_inventory.loc[derived_measure_mask, 'Descriptive item type'] = 'Derived measure'
attendance_discipline_inventory.loc[frequency_or_count_mask,
    'Descriptive item type'] = 'Frequency, count or duration item'
attendance_discipline_inventory.loc[reason_or_routing_mask, 'Descriptive item type'] = 'Reason or routing item'
attendance_discipline_construct_wave_summary = attendance_discipline_inventory.groupby(['Matched construct', 'Wave',
    'Timing status'], dropna=False).size().rename('Candidate variables').reset_index().sort_values(['Matched construct',
    'Wave']).reset_index(drop=True)
attendance_discipline_item_type_summary = attendance_discipline_inventory.groupby(['Matched construct',
    'Descriptive item type'], dropna=False).size().rename('Candidate variables').reset_index().sort_values(['Matched construct',
    'Candidate variables'], ascending=[True, False]).reset_index(drop=True)
attendance_discipline_family_summary = attendance_discipline_inventory.sort_values(['Source order',
    'Variable position']).groupby(['Variable stem', 'Matched construct'], dropna=False).agg(Waves=('Wave',
    lambda values: '; '.join(values.dropna().astype(str).tolist())), Number_of_waves=('Wave',
    'nunique'), Variables=('Variable',
    lambda values: '; '.join(values.astype(str).tolist())), Variable_labels=('Variable label',
    lambda values: ' | '.join(values.dropna().astype(str).tolist())), Item_types=('Descriptive item type',
    lambda values: '; '.join(sorted(values.dropna().astype(str).unique())))).reset_index().rename(columns={'Number_of_waves': 'Number of waves',
    'Variable_labels': 'Variable labels', 'Item_types': 'Item types'}).sort_values(['Number of waves',
    'Matched construct', 'Variable stem'], ascending=[False, True, True]).reset_index(drop=True)
repeated_attendance_discipline_families = attendance_discipline_family_summary.loc[attendance_discipline_family_summary['Number of waves'].ge(2)].copy()
wave_specific_attendance_discipline_families = attendance_discipline_family_summary.loc[attendance_discipline_family_summary['Number of waves'].eq(1)].copy()
attendance_discipline_status_summary = attendance_discipline_inventory.groupby(['Review status', 'Review outcome'],
    dropna=False).size().rename('Candidate variables').reset_index()
attendance_discipline_display = attendance_discipline_inventory.sort_values(['Matched construct', 'Source order',
    'Variable position'])[['Matched construct', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable',
    'Variable stem', 'Variable label', 'Descriptive item type', 'Timing status', 'Review status', 'Review outcome', 'Substantive domain']].reset_index(drop=True)
print(f'Attendance and discipline candidates: {len(attendance_discipline_inventory):,}')
print('Candidates by construct, wave and timing:')
display_limited(attendance_discipline_construct_wave_summary)
print('Candidates by descriptive item type:')
display_limited(attendance_discipline_item_type_summary)
print(f'Exact variable families repeated across waves: {len(repeated_attendance_discipline_families):,}')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(repeated_attendance_discipline_families)
print(f'Wave-specific variable families: {len(wave_specific_attendance_discipline_families):,}')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(wave_specific_attendance_discipline_families)
print('Existing register status:')
display_limited(attendance_discipline_status_summary)
print('Complete attendance and discipline inventory:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(attendance_discipline_display)

Attendance and discipline candidates: 33
Candidates by construct, wave and timing:


,Matched construct,Wave,Timing status,Candidate variables
0,Attendance and truancy,Wave 1,Pre-transition source,7
1,Attendance and truancy,Wave 2,Pre-transition source,12
2,Attendance and truancy,Wave 3,Near-transition source,5
3,Attendance and truancy; Bullying and school sa...,Wave 2,Pre-transition source,1
4,"Behaviour, suspension and exclusion",Wave 1,Pre-transition source,4


Candidates by descriptive item type:


,Matched construct,Descriptive item type,Candidate variables
0,Attendance and truancy,Reason or routing item,15
1,Attendance and truancy,"Frequency, count or duration item",6
2,Attendance and truancy,Direct status or experience item,3
3,Attendance and truancy; Bullying and school sa...,Reason or routing item,1
4,"Behaviour, suspension and exclusion",Direct status or experience item,6


Exact variable families repeated across waves: 8


...

,Variable stem,Matched construct,Waves,Number of waves,Variables,Variable labels,Item types
8,abs3mwmp,Attendance and truancy,Wave 1,1,W1abs3mwMP,MP: Reason for YP's (last) extended period of absence from school,Reason or routing item
9,truant2yp0b,Attendance and truancy,Wave 2,1,W2Truant2YP0b,YP: Main reason for playing truant - Bored,Reason or routing item
10,truant2yp0c,Attendance and truancy,Wave 2,1,W2Truant2YP0c,YP: Main reason for playing truant - Just don't like school,Reason or routing item
11,truant2yp0d,Attendance and truancy,Wave 2,1,W2Truant2YP0d,YP: Main reason for playing truant - Don't like particualr teacher or teachers,Reason or routing item
12,truant2yp0e,Attendance and truancy,Wave 2,1,W2Truant2YP0e,YP: Main reason for playing truant - Don't like particular lesson or subject,Reason or routing item


Existing register status:


,Review status,Review outcome,Candidate variables
0,Use requires interview timing confirming Janua...,Pending review,7
1,"Variable-level coding, routing and reference-p...",Pending review,26


Complete attendance and discipline inventory:


,Matched construct,Wave,Source type,Source file,Variable position,Variable,Variable stem,Variable label,Descriptive item type,Timing status,Review status,Review outcome,Substantive domain
0,Attendance and truancy,Wave 1,Young person,wave_one_lsype_young_person_2020,55,W1abs3mwMP,abs3mwmp,MP: Reason for YP's (last) extended period of absence from school,Reason or routing item,Pre-transition source,"Variable-level coding, routing and reference-period review required.",Pending review,<NA>
1,Attendance and truancy,Wave 1,Young person,wave_one_lsype_young_person_2020,57,W1abs1mwMP,abs1mwmp,MP: Reason for YP's period of absence from school of 1 month or more in last 12,Reason or routing item,Pre-transition source,"Variable-level coding, routing and reference-period review required.",Pending review,<NA>
2,Attendance and truancy,Wave 1,Young person,wave_one_lsype_young_person_2020,255,W1truantYP,truantyp,YP: Whether played truant in last 12 months,"Frequency, count or duration item",Pre-transition source,"Variable-level coding, routing and reference-period review required.",Pending review,<NA>
3,Attendance and truancy,Wave 1,Young person,wave_one_lsype_young_person_2020,256,W1truant1YP,truant1yp,YP: Longest period of truancy in last 12 months,"Frequency, count or duration item",Pre-transition source,"Variable-level coding, routing and reference-period review required.",Pending review,<NA>
4,Attendance and truancy,Wave 1,Young person,wave_one_lsype_young_person_2020,257,W1truant2YP,truant2yp,YP: Main reason for playing truant,Reason or routing item,Pre-transition source,"Variable-level coding, routing and reference-period review required.",Pending review,<NA>


In [281]:
# 16: Core attendance and discipline coding review

import pandas as pd
attendance_discipline_status_candidates = ['W1truantYP', 'W2truantYP', 'W3truantYP', 'W1suspendMP', 'W3suspendMP',
    'W1expelMP', 'W3expelMP']
attendance_discipline_review_support = ['W1truant1YP', 'W2truant1YP', 'W3truant1YP', 'W1sutime2MP', 'W1exp3yr2MP']
attendance_discipline_deferred_variables = ['W1carehr1YP', 'W2carehr1YP', 'W3carehr1YP', 'W1carehr2YP', 'W2carehr2YP',
    'W3carehr2YP']
attendance_discipline_non_candidate_variables = ['W1abs3mwMP', 'W1abs1mwMP', 'W2abs1mwMP', 'W1truant2YP',
    'W3truant2YP', *[f'W2Truant2YP0{letter}' for letter in 'abcdefgh'], 'W2comp2YP', 'W2comp3YP']
attendance_discipline_review_groups = {'Direct status candidate': attendance_discipline_status_candidates,
    'Review support': attendance_discipline_review_support, 'Defer to experiences and behaviours domain': attendance_discipline_deferred_variables, 'Not a direct predictor candidate': attendance_discipline_non_candidate_variables}
all_attendance_discipline_variables = [variable for variables in attendance_discipline_review_groups.values() for variable in variables]
assert len(all_attendance_discipline_variables) == 33
assert len(set(all_attendance_discipline_variables)) == 33
assert set(all_attendance_discipline_variables) == set(attendance_discipline_inventory['Variable'])
attendance_discipline_role_lookup = {variable: role for role,
    variables in attendance_discipline_review_groups.items() for variable in variables}
attendance_discipline_role_audit = attendance_discipline_inventory.copy()
attendance_discipline_role_audit['Provisional review role'] = attendance_discipline_role_audit['Variable'].map(attendance_discipline_role_lookup)
assert attendance_discipline_role_audit['Provisional review role'].notna().all()
attendance_discipline_role_summary = attendance_discipline_role_audit['Provisional review role'].value_counts().reindex(['Direct status candidate',
    'Review support', 'Defer to experiences and behaviours domain', 'Not a direct predictor candidate']).rename('Variables').rename_axis('Provisional review role').reset_index()
core_attendance_discipline_variables = attendance_discipline_status_candidates + attendance_discipline_review_support
core_attendance_discipline_inventory = attendance_discipline_inventory.loc[attendance_discipline_inventory['Variable'].isin(core_attendance_discipline_variables)].copy()
assert len(core_attendance_discipline_inventory) == 12
attendance_discipline_raw_tables = {}
attendance_discipline_labelled_tables = {}
attendance_discipline_quality_rows = []
attendance_discipline_code_rows = []
for source_file, source_inventory in core_attendance_discipline_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    raw_data['NSID'] = standardise_nsid(raw_data['NSID'])
    labelled_data['NSID'] = standardise_nsid(labelled_data['NSID'])
    assert raw_data['NSID'].is_unique
    assert labelled_data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(route_index)
    labelled_data = labelled_data.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    attendance_discipline_raw_tables[source_file] = raw_data
    attendance_discipline_labelled_tables[source_file] = labelled_data
    for variable in source_variables:
        variable_row = source_inventory.loc[source_inventory['Variable'].eq(variable)].iloc[0]
        wave = variable_row['Wave']
        raw_values = raw_data[variable]
        labelled_values = labelled_data[variable].astype('string')
        if wave == 'Wave 3':
            timing_eligible = wave_3_pretransition_eligible.reindex(route_index).fillna(False)
        else:
            timing_eligible = pd.Series(True, index=route_index, dtype='boolean')
        observed_response = raw_values.ge(0).fillna(False)
        special_code_response = raw_values.lt(0).fillna(False)
        attendance_discipline_quality_rows.append({'Matched construct': variable_row['Matched construct'],
            'Provisional review role': attendance_discipline_role_lookup[variable], 'Wave': wave, 'Variable': variable, 'Variable label': variable_row['Variable label'], 'Observed responses': int(observed_response.sum()), 'Special-code responses': int(special_code_response.sum()), 'No source record': int(raw_values.isna().sum()), 'Observed before September 2006': int((observed_response & timing_eligible).sum()), 'Distinct observed codes': int(raw_values.loc[observed_response].nunique()), 'Minimum observed code': raw_values.loc[observed_response].min() if observed_response.any() else pd.NA, 'Maximum observed code': raw_values.loc[observed_response].max() if observed_response.any() else pd.NA})
        value_counts = raw_values.value_counts(dropna=False)
        for raw_code, participants in value_counts.items():
            if pd.isna(raw_code):
                value_label = 'No source record'
                response_type = 'No source record'
                code_sort_value = 999999
            else:
                code_mask = raw_values.eq(raw_code)
                matching_labels = labelled_values.loc[code_mask].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(raw_code)
                response_type = 'Observed response' if raw_code >= 0 else 'Special code'
                code_sort_value = float(raw_code)
            attendance_discipline_code_rows.append({'Wave': wave, 'Variable': variable,
                'Variable label': variable_row['Variable label'], 'Provisional review role': attendance_discipline_role_lookup[variable], 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Code sort value': code_sort_value})
attendance_discipline_variable_quality = pd.DataFrame(attendance_discipline_quality_rows).sort_values(['Matched construct',
    'Wave', 'Variable']).reset_index(drop=True)
attendance_discipline_variable_quality['Observed-response percentage'] = (attendance_discipline_variable_quality['Observed responses'] / len(route_index) * 100).round(2)
attendance_discipline_variable_quality['Pre-transition observed percentage'] = (attendance_discipline_variable_quality['Observed before September 2006'] / len(route_index) * 100).round(2)
attendance_discipline_code_distribution = pd.DataFrame(attendance_discipline_code_rows).sort_values(['Variable',
    'Wave', 'Code sort value']).drop(columns=['Code sort value']).reset_index(drop=True)
print('Provisional review roles:')
display_limited(attendance_discipline_role_summary)
print('Core status and review-support variable quality:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(attendance_discipline_variable_quality)
print('Core variable response codes:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(attendance_discipline_code_distribution)

Provisional review roles:


,Provisional review role,Variables
0,Direct status candidate,7
1,Review support,5
2,Defer to experiences and behaviours domain,6
3,Not a direct predictor candidate,15


Core status and review-support variable quality:


,Matched construct,Provisional review role,Wave,Variable,Variable label,Observed responses,Special-code responses,No source record,Observed before September 2006,Distinct observed codes,Minimum observed code,Maximum observed code,Observed-response percentage,Pre-transition observed percentage
0,Attendance and truancy,Review support,Wave 1,W1truant1YP,YP: Longest period of truancy in last 12 months,955,8569,243,955,4,1.0,4.0,9.78,9.78
1,Attendance and truancy,Direct status candidate,Wave 1,W1truantYP,YP: Whether played truant in last 12 months,8885,639,243,8885,2,1.0,2.0,90.97,90.97
2,Attendance and truancy,Review support,Wave 2,W2truant1YP,YP: Longest period of truancy in last 12 months,1726,7795,246,1726,4,1.0,4.0,17.67,17.67
3,Attendance and truancy,Direct status candidate,Wave 2,W2truantYP,YP: Whether played truant in last 12 months,8956,565,246,8956,2,1.0,2.0,91.70,91.70
4,Attendance and truancy,Review support,Wave 3,W3truant1YP,YP: Longest period of truancy in last 12 months,1923,7586,258,1895,4,1.0,4.0,19.69,19.40


Core variable response codes:


,Wave,Variable,Variable label,Provisional review role,Raw code,Value label,Response type,Participants
0,Wave 1,W1exp3yr2MP,MP: Number many times YP been permanently excluded from school in the last 3 yea,Review support,-99.0,MP not interviewed,Special code,103
1,Wave 1,W1exp3yr2MP,MP: Number many times YP been permanently excluded from school in the last 3 yea,Review support,-97.0,MP refused CASI section,Special code,171
2,Wave 1,W1exp3yr2MP,MP: Number many times YP been permanently excluded from school in the last 3 yea,Review support,-96.0,MP unable to complete CASI section,Special code,572
3,Wave 1,W1exp3yr2MP,MP: Number many times YP been permanently excluded from school in the last 3 yea,Review support,-91.0,Not applicable,Special code,8655
4,Wave 1,W1exp3yr2MP,MP: Number many times YP been permanently excluded from school in the last 3 yea,Review support,1.0,1,Observed response,16


In [282]:
# 17: Attendance and discipline representation comparison

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
wave_1_attendance_data = attendance_discipline_raw_tables['wave_one_lsype_young_person_2020']
wave_2_attendance_data = attendance_discipline_raw_tables['wave_two_lsype_young_person_2020']
wave_3_attendance_data = attendance_discipline_raw_tables['wave_three_lsype_young_person_2020']
wave_3_timing_eligible = wave_3_pretransition_eligible.reindex(route_index).fillna(False)

def recode_yes_no(values, timing_mask=None):
    """Recode Yes and No responses as 1 and 0."""
    numeric_values = pd.to_numeric(values, errors='coerce')
    recoded_values = pd.Series(pd.NA, index=numeric_values.index, dtype='Float64')
    recoded_values.loc[numeric_values.eq(1)] = 1
    recoded_values.loc[numeric_values.eq(2)] = 0
    if timing_mask is not None:
        recoded_values = recoded_values.where(timing_mask)
    return recoded_values

def construct_latest_binary(source_series):
    """Use the latest available valid pre-transition response."""
    latest_values = pd.Series(pd.NA, index=route_index, dtype='Float64')
    latest_source = pd.Series(pd.NA, index=route_index, dtype='string')
    for source_name, values in source_series:
        fill_mask = latest_values.isna() & values.notna()
        latest_values.loc[fill_mask] = values.loc[fill_mask]
        latest_source.loc[fill_mask] = source_name
    return (latest_values, latest_source)

def calculate_bias_corrected_cramers_v(first_values, second_values):
    """Calculate bias-corrected Cramér's V."""
    comparison_data = pd.DataFrame({'First': first_values, 'Second': second_values}).dropna()
    contingency_table = pd.crosstab(comparison_data['First'], comparison_data['Second'])
    participant_count = int(contingency_table.to_numpy().sum())
    if participant_count <= 1 or contingency_table.shape[0] < 2 or contingency_table.shape[1] < 2:
        return (np.nan, participant_count)
    chi_square = chi2_contingency(contingency_table, correction=False)[0]
    phi_squared = chi_square / participant_count
    row_count, column_count = contingency_table.shape
    corrected_phi_squared = max(0, phi_squared - (column_count - 1) * (row_count - 1) / (participant_count - 1))
    corrected_row_count = row_count - (row_count - 1) ** 2 / (participant_count - 1)
    corrected_column_count = column_count - (column_count - 1) ** 2 / (participant_count - 1)
    denominator = min(corrected_row_count - 1, corrected_column_count - 1)
    if denominator <= 0:
        return (np.nan, participant_count)
    cramers_v = np.sqrt(corrected_phi_squared / denominator)
    return (float(cramers_v), participant_count)

def summarise_category(values, label):
    """Summarise a categorical representation."""
    summary = values.fillna('Unavailable').value_counts().rename('Participants').rename_axis(label).reset_index()
    summary['Percentage'] = (summary['Participants'] / len(route_index) * 100).round(2)
    return summary
wave_1_truancy = recode_yes_no(wave_1_attendance_data['W1truantYP'])
wave_2_truancy = recode_yes_no(wave_2_attendance_data['W2truantYP'])
wave_3_truancy = recode_yes_no(wave_3_attendance_data['W3truantYP'], timing_mask=wave_3_timing_eligible)
latest_truancy_status, latest_truancy_source = construct_latest_binary([('Wave 3 before September 2006',
    wave_3_truancy), ('Wave 2 fallback', wave_2_truancy), ('Wave 1 fallback', wave_1_truancy)])
latest_truancy_status = latest_truancy_status.rename('truancy_status_pretransition')
wave_1_suspension = recode_yes_no(wave_1_attendance_data['W1suspendMP'])
wave_3_suspension = recode_yes_no(wave_3_attendance_data['W3suspendMP'], timing_mask=wave_3_timing_eligible)
latest_suspension_status, latest_suspension_source = construct_latest_binary([('Wave 3 before September 2006',
    wave_3_suspension), ('Wave 1 fallback', wave_1_suspension)])
wave_1_permanent_exclusion = recode_yes_no(wave_1_attendance_data['W1expelMP'])
wave_3_permanent_exclusion = recode_yes_no(wave_3_attendance_data['W3expelMP'], timing_mask=wave_3_timing_eligible)
latest_permanent_exclusion_status, latest_permanent_exclusion_source = construct_latest_binary([('Wave 3 before September 2006',
    wave_3_permanent_exclusion), ('Wave 1 fallback', wave_1_permanent_exclusion)])
school_exclusion_history = pd.Series(pd.NA, index=route_index, dtype='Float64',
    name='school_exclusion_history_pretransition')
any_exclusion_recorded = latest_suspension_status.eq(1) | latest_permanent_exclusion_status.eq(1)
both_exclusion_items_observed = latest_suspension_status.notna() & latest_permanent_exclusion_status.notna()
neither_exclusion_recorded = latest_suspension_status.eq(0) & latest_permanent_exclusion_status.eq(0)
school_exclusion_history.loc[any_exclusion_recorded] = 1
school_exclusion_history.loc[both_exclusion_items_observed & neither_exclusion_recorded] = 0
school_exclusion_category = pd.Series(pd.NA, index=route_index, dtype='string',
    name='school_exclusion_category_pretransition')
school_exclusion_category.loc[both_exclusion_items_observed & neither_exclusion_recorded] = 'No suspension or permanent exclusion'
school_exclusion_category.loc[latest_suspension_status.eq(1) & ~latest_permanent_exclusion_status.eq(1)] = 'Temporary suspension recorded'
school_exclusion_category.loc[latest_permanent_exclusion_status.eq(1)] = 'Permanent exclusion recorded'
truancy_duration_source_lookup = {'Wave 3 before September 2006': wave_3_attendance_data['W3truant1YP'].where(wave_3_timing_eligible),
    'Wave 2 fallback': wave_2_attendance_data['W2truant1YP'], 'Wave 1 fallback': wave_1_attendance_data['W1truant1YP']}
truancy_duration_code = pd.Series(pd.NA, index=route_index, dtype='Float64')
for source_name, duration_values in truancy_duration_source_lookup.items():
    numeric_duration = pd.to_numeric(duration_values, errors='coerce')
    valid_duration = numeric_duration.where(numeric_duration.isin([1, 2, 3, 4])).astype('Float64')
    source_mask = latest_truancy_source.eq(source_name) & latest_truancy_status.eq(1)
    truancy_duration_code.loc[source_mask] = valid_duration.loc[source_mask]
truancy_severity_profile = pd.Series(pd.NA, index=route_index, dtype='string', name='truancy_severity_profile')
truancy_severity_profile.loc[latest_truancy_status.eq(0)] = 'No truancy'
truancy_severity_profile.loc[latest_truancy_status.eq(1) & truancy_duration_code.eq(4)] = 'Odd day or lesson'
truancy_severity_profile.loc[latest_truancy_status.eq(1) & truancy_duration_code.eq(3)] = 'Particular lessons'
truancy_severity_profile.loc[latest_truancy_status.eq(1) & truancy_duration_code.eq(2)] = 'Several days at a time'
truancy_severity_profile.loc[latest_truancy_status.eq(1) & truancy_duration_code.eq(1)] = 'Weeks at a time'
truancy_severity_profile.loc[latest_truancy_status.eq(1) & truancy_duration_code.isna()] = 'Truancy reported, duration unavailable'
behavioural_disengagement_profile = pd.Series(pd.NA, index=route_index, dtype='string',
    name='behavioural_disengagement_profile')
both_behavioural_constructs_observed = latest_truancy_status.notna() & school_exclusion_history.notna()
behavioural_disengagement_profile.loc[both_behavioural_constructs_observed & latest_truancy_status.eq(0) & school_exclusion_history.eq(0)] = 'Neither truancy nor school exclusion'
behavioural_disengagement_profile.loc[both_behavioural_constructs_observed & latest_truancy_status.eq(1) & school_exclusion_history.eq(0)] = 'Truancy only'
behavioural_disengagement_profile.loc[both_behavioural_constructs_observed & latest_truancy_status.eq(0) & school_exclusion_history.eq(1)] = 'School exclusion only'
behavioural_disengagement_profile.loc[both_behavioural_constructs_observed & latest_truancy_status.eq(1) & school_exclusion_history.eq(1)] = 'Truancy and school exclusion'
for binary_series in [latest_truancy_status, latest_suspension_status, latest_permanent_exclusion_status,
    school_exclusion_history]:
    assert binary_series.dropna().isin([0, 1]).all()
exclusion_wave_3_source = latest_suspension_source.eq('Wave 3 before September 2006') & latest_permanent_exclusion_source.eq('Wave 3 before September 2006')
behavioural_status_source_summary = pd.DataFrame([{'Construct': 'Truancy status',
    'Wave 3 source': int(latest_truancy_source.eq('Wave 3 before September 2006').sum()), 'Earlier-wave fallback': int(latest_truancy_source.isin(['Wave 2 fallback',
    'Wave 1 fallback']).sum()), 'Unavailable': int(latest_truancy_status.isna().sum()), 'Available percentage': round(latest_truancy_status.notna().mean() * 100,
    2)}, {'Construct': 'Temporary suspension or permanent exclusion',
    'Wave 3 source': int((school_exclusion_history.notna() & exclusion_wave_3_source).sum()), 'Earlier-wave fallback': int((school_exclusion_history.notna() & ~exclusion_wave_3_source).sum()), 'Unavailable': int(school_exclusion_history.isna().sum()), 'Available percentage': round(school_exclusion_history.notna().mean() * 100,
    2)}])
truancy_status_labelled = latest_truancy_status.map({0.0: 'No truancy', 1.0: 'Truancy reported'}).astype('string')
truancy_status_distribution = summarise_category(truancy_status_labelled, 'Truancy status')
truancy_severity_distribution = summarise_category(truancy_severity_profile, 'Truancy severity')
school_exclusion_distribution = summarise_category(school_exclusion_category, 'School exclusion category')
behavioural_profile_distribution = summarise_category(behavioural_disengagement_profile,
    'Behavioural disengagement profile')
behavioural_cramers_v, behavioural_complete_comparisons = calculate_bias_corrected_cramers_v(latest_truancy_status,
    school_exclusion_history)
behavioural_status_overlap = pd.DataFrame([{'Complete comparisons': behavioural_complete_comparisons,
    "Bias-corrected Cramer's V": round(behavioural_cramers_v, 3)}])
behavioural_attitude_rows = []
for representation_name, representation_values in [('Truancy status', latest_truancy_status),
    ('School exclusion history', school_exclusion_history)]:
    comparison_data = pd.DataFrame({'Behavioural status': representation_values,
        'School-attitude score': school_attitude_score_candidate}).dropna()
    behavioural_attitude_rows.append({'Representation': representation_name,
        'Complete comparisons': len(comparison_data), 'Spearman correlation with school-attitude score': round(comparison_data['Behavioural status'].astype('float64').corr(comparison_data['School-attitude score'].astype('float64'),
        method='spearman'), 3), 'Mean school-attitude score when status absent': round(comparison_data.loc[comparison_data['Behavioural status'].eq(0),
        'School-attitude score'].mean(), 3), 'Mean school-attitude score when status present': round(comparison_data.loc[comparison_data['Behavioural status'].eq(1),
        'School-attitude score'].mean(), 3)})
behavioural_attitude_summary = pd.DataFrame(behavioural_attitude_rows)
behavioural_profile_attitude_data = pd.DataFrame({'Behavioural disengagement profile': behavioural_disengagement_profile,
    'School-attitude score': school_attitude_score_candidate}).dropna()
behavioural_profile_attitude_summary = behavioural_profile_attitude_data.groupby('Behavioural disengagement profile',
    dropna=False)['School-attitude score'].agg(['size', 'mean',
    'std']).reset_index().rename(columns={'size': 'Participants', 'mean': 'Mean school-attitude score',
    'std': 'Standard deviation'})
behavioural_profile_attitude_summary[['Mean school-attitude score',
    'Standard deviation']] = behavioural_profile_attitude_summary[['Mean school-attitude score',
    'Standard deviation']].round(3)
print('Behavioural-status source coverage:')
display_limited(behavioural_status_source_summary)
print('Truancy-status distribution:')
display_limited(truancy_status_distribution)
print('Truancy-severity distribution:')
display_limited(truancy_severity_distribution)
print('School-exclusion distribution:')
display_limited(school_exclusion_distribution)
print('Combined behavioural-disengagement profile:')
display_limited(behavioural_profile_distribution)
print('Overlap between truancy and school exclusion:')
display_limited(behavioural_status_overlap)
print('Relationships with the school-attitude score:')
display_limited(behavioural_attitude_summary)
print('School-attitude score by combined behavioural profile:')
display_limited(behavioural_profile_attitude_summary)

Behavioural-status source coverage:


,Construct,Wave 3 source,Earlier-wave fallback,Unavailable,Available percentage
0,Truancy status,8943,548,276,97.17
1,Temporary suspension or permanent exclusion,8614,463,690,92.94


Truancy-status distribution:


,Truancy status,Participants,Percentage
0,No truancy,7324,74.99
1,Truancy reported,2167,22.19
2,Unavailable,276,2.83


Truancy-severity distribution:


...

,Complete comparisons,Bias-corrected Cramer's V
0,9048,0.19


Relationships with the school-attitude score:


,Representation,Complete comparisons,Spearman correlation with school-attitude score,Mean school-attitude score when status absent,Mean school-attitude score when status present
0,Truancy status,9489,-0.330,3.104,2.729
1,School exclusion history,9064,-0.178,3.036,2.614


School-attitude score by combined behavioural profile:


,Behavioural disengagement profile,Participants,Mean school-attitude score,Standard deviation
0,Neither truancy nor school exclusion,6761,3.109,0.401
1,School exclusion only,220,2.809,0.511
2,Truancy and school exclusion,279,2.458,0.551
3,Truancy only,1786,2.762,0.454


In [283]:
# 18: Domain 6 behavioural representation decision

import pandas as pd
truancy_status_pretransition_candidate = latest_truancy_status.astype('Int64').rename('truancy_status_pretransition')
assert truancy_status_pretransition_candidate.dropna().isin([0, 1]).all()
assert int(truancy_status_pretransition_candidate.notna().sum()) == 9491
domain_6_predictor_candidates = pd.DataFrame({'school_attitude_score': school_attitude_score_candidate,
    'truancy_status_pretransition': truancy_status_pretransition_candidate}, index=route_index)
assert list(domain_6_predictor_candidates.columns) == ['school_attitude_score', 'truancy_status_pretransition']
domain_6_candidate_summary = pd.DataFrame([{'Predictor': 'school_attitude_score',
    'Construct': 'Positive attitudes towards school and engagement with lessons', 'Representation': 'Latest available pre-transition mean of at least 8 of 12 items', 'Primary wave': 'Wave 3 interview before September 2006', 'Fallback order': 'Wave 2, then Wave 1', 'Form': 'Derived continuous score', 'Non-missing': int(domain_6_predictor_candidates['school_attitude_score'].notna().sum()), 'Missing': int(domain_6_predictor_candidates['school_attitude_score'].isna().sum()), 'Missing percentage': round(domain_6_predictor_candidates['school_attitude_score'].isna().mean() * 100,
    2)}, {'Predictor': 'truancy_status_pretransition',
    'Construct': 'Recent behavioural disengagement from school attendance', 'Representation': 'Whether the young person played truant in the preceding 12 months', 'Primary wave': 'Wave 3 interview before September 2006', 'Fallback order': 'Wave 2, then Wave 1', 'Form': 'Harmonised binary item', 'Non-missing': int(domain_6_predictor_candidates['truancy_status_pretransition'].notna().sum()), 'Missing': int(domain_6_predictor_candidates['truancy_status_pretransition'].isna().sum()), 'Missing percentage': round(domain_6_predictor_candidates['truancy_status_pretransition'].isna().mean() * 100,
    2)}])
domain_6_candidate_availability = domain_6_predictor_candidates.notna().rename(columns={'school_attitude_score': 'School-attitude score available',
    'truancy_status_pretransition': 'Truancy status available'}).value_counts().rename('Participants').reset_index()
domain_6_candidate_availability['Percentage'] = (domain_6_candidate_availability['Participants'] / len(route_index) * 100).round(2)
domain_6_behavioural_representation_decisions = pd.DataFrame([{'Construct or representation': 'Latest pre-transition truancy status',
    'Current decision': 'Retain as Domain 6 predictor candidate', 'Reason': 'Direct and well-covered measure of recent behavioural disengagement from school attendance'}, {'Construct or representation': 'Truancy severity profile',
    'Current decision': 'Do not retain as a separate predictor', 'Reason': 'Derived from a routed duration item and does not represent a construct separate from truancy'}, {'Construct or representation': 'Temporary suspension or permanent exclusion',
    'Current decision': 'Retain as review support only', 'Reason': 'Less well covered and substantially less common; also reflects an institutionally mediated response rather than attendance behaviour alone'}, {'Construct or representation': 'Combined behavioural-disengagement profile',
    'Current decision': 'Do not retain', 'Reason': 'Combines young-person truancy with school-imposed disciplinary action and creates small categories'}, {'Construct or representation': 'Absence due to caring responsibilities',
    'Current decision': 'Defer to Domain 8', 'Reason': 'Represents caring experience rather than general school attendance or engagement'}, {'Construct or representation': 'Bullying and school-safety experiences',
    'Current decision': 'Defer to Domain 8', 'Reason': 'Represents an adverse experience rather than a school-attitude or attendance construct'}])
domain_6_scope_checkpoint = pd.DataFrame([{'Domain': 'School attitudes and engagement',
    'Predictor candidates': len(domain_6_predictor_candidates.columns), 'Candidate names': '; '.join(domain_6_predictor_candidates.columns), 'Further predictor search in this domain': 'Stop', 'Next step': 'Review the two candidates and then finalise the Domain 6 files and register decisions'}])
print('Domain 6 candidate summary:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_6_candidate_summary)
print('Joint candidate availability:')
display_limited(domain_6_candidate_availability)
print('Behavioural representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_6_behavioural_representation_decisions)
print('Domain 6 scope checkpoint:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_6_scope_checkpoint)

Domain 6 candidate summary:


,Predictor,Construct,Representation,Primary wave,Fallback order,Form,Non-missing,Missing,Missing percentage
0,school_attitude_score,Positive attitudes towards school and engagement with lessons,Latest available pre-transition mean of at least 8 of 12 items,Wave 3 interview before September 2006,"Wave 2, then Wave 1",Derived continuous score,9511,256,2.62
1,truancy_status_pretransition,Recent behavioural disengagement from school attendance,Whether the young person played truant in the preceding 12 months,Wave 3 interview before September 2006,"Wave 2, then Wave 1",Harmonised binary item,9491,276,2.83


Joint candidate availability:


,School-attitude score available,Truancy status available,Participants,Percentage
0,True,True,9489,97.15
1,False,False,254,2.60
2,True,False,22,0.23
3,False,True,2,0.02


Behavioural representation decisions:


,Construct or representation,Current decision,Reason
0,Latest pre-transition truancy status,Retain as Domain 6 predictor candidate,Direct and well-covered measure of recent behavioural disengagement from school attendance
1,Truancy severity profile,Do not retain as a separate predictor,Derived from a routed duration item and does not represent a construct separate from truancy
2,Temporary suspension or permanent exclusion,Retain as review support only,Less well covered and substantially less common; also reflects an institutionally mediated response rather than attendance behaviour alone
3,Combined behavioural-disengagement profile,Do not retain,Combines young-person truancy with school-imposed disciplinary action and creates small categories
4,Absence due to caring responsibilities,Defer to Domain 8,Represents caring experience rather than general school attendance or engagement


Domain 6 scope checkpoint:


,Domain,Predictor candidates,Candidate names,Further predictor search in this domain,Next step
0,School attitudes and engagement,2,school_attitude_score; truancy_status_pretransition,Stop,Review the two candidates and then finalise the Domain 6 files and register decisions


In [284]:
# 19: Domain 6 pre-finalisation decision check

import pandas as pd
domain_6_inventory_sources = []
for inventory_name, inventory_table in [('School engagement review tracks', school_engagement_review_tracks),
    ('Complete school-attitude battery', school_attitude_battery_inventory)]:
    inventory_copy = inventory_table.copy()
    inventory_copy['Inventory source'] = inventory_name
    domain_6_inventory_sources.append(inventory_copy)
domain_6_full_inventory = pd.concat(domain_6_inventory_sources, ignore_index=True,
    sort=False).sort_values(['Source order', 'Variable position']).drop_duplicates(subset=['Variable'],
    keep='first').reset_index(drop=True)
assert domain_6_full_inventory['Variable'].is_unique
yys_item_number_lookup = school_attitude_battery_inventory.drop_duplicates(subset=['Variable']).set_index('Variable')['School-attitude item number'].to_dict()
domain_6_full_inventory['YYS item number'] = domain_6_full_inventory['Variable'].map(yys_item_number_lookup).astype('Int64')
domain_6_full_inventory['Provisional audit decision'] = pd.Series(pd.NA, index=domain_6_full_inventory.index,
    dtype='string')
domain_6_full_inventory['Provisional representation'] = pd.Series(pd.NA, index=domain_6_full_inventory.index,
    dtype='string')
domain_6_full_inventory['Provisional destination domain'] = pd.Series(pd.NA, index=domain_6_full_inventory.index,
    dtype='string')
domain_6_full_inventory['Audit reason'] = pd.Series(pd.NA, index=domain_6_full_inventory.index, dtype='string')

def assign_domain_6_audit_decision(variable_mask, decision, representation, destination_domain, reason):
    """Assign an audit decision to variables not yet classified."""
    eligible_mask = variable_mask & domain_6_full_inventory['Provisional audit decision'].isna()
    domain_6_full_inventory.loc[eligible_mask, 'Provisional audit decision'] = decision
    domain_6_full_inventory.loc[eligible_mask, 'Provisional representation'] = representation
    domain_6_full_inventory.loc[eligible_mask, 'Provisional destination domain'] = destination_domain
    domain_6_full_inventory.loc[eligible_mask, 'Audit reason'] = reason
assign_domain_6_audit_decision(domain_6_full_inventory['YYS item number'].between(1, 12, inclusive='both'),
    decision='Construction input', representation='school_attitude_score', destination_domain='School attitudes and engagement', reason='Repeated item represented by the derived twelve-item school-attitude score')
assign_domain_6_audit_decision(domain_6_full_inventory['YYS item number'].eq(13), decision='Defer',
    representation='School physical-environment candidate', destination_domain='School and local context', reason='Represents school context rather than individual attitude or engagement')
assign_domain_6_audit_decision(domain_6_full_inventory['YYS item number'].isin([14, 15, 16, 17, 18, 19, 20, 21, 24, 25,
    26]), decision='Review support', representation='Teacher-related construct diagnostics', destination_domain='School attitudes and engagement', reason='Used to assess teacher-related constructs but not retained alongside the broader attitude score')
assign_domain_6_audit_decision(domain_6_full_inventory['YYS item number'].isin([22, 23]), decision='Defer',
    representation='Academic self-concept candidate', destination_domain='Psychosocial', reason='Requires comparison with other psychosocial constructs before a retention decision')
assign_domain_6_audit_decision(domain_6_full_inventory['Variable'].isin(['W1truantYP', 'W2truantYP', 'W3truantYP']),
    decision='Construction input', representation='truancy_status_pretransition', destination_domain='School attitudes and engagement', reason='Repeated direct status item used in the latest available pre-transition truancy representation')
assign_domain_6_audit_decision(domain_6_full_inventory['Variable'].isin(['W1truant1YP', 'W2truant1YP', 'W3truant1YP']),
    decision='Review support', representation='Truancy severity diagnostic', destination_domain='School attitudes and engagement', reason='Routed duration item used to review severity but not retained separately from truancy status')
assign_domain_6_audit_decision(domain_6_full_inventory['Variable'].isin(['W1suspendMP', 'W3suspendMP', 'W1expelMP',
    'W3expelMP', 'W1sutime2MP', 'W1exp3yr2MP']), decision='Review support', representation='School-exclusion history diagnostic', destination_domain='School attitudes and engagement', reason='Reviewed as institutional disciplinary outcomes but not retained as a separate predictor')
assign_domain_6_audit_decision(domain_6_full_inventory['Variable'].isin(attendance_discipline_deferred_variables),
    decision='Defer', representation='Caring-related school absence candidate', destination_domain='Experiences and behaviours', reason='Represents caring responsibility rather than general attendance behaviour')
assign_domain_6_audit_decision(domain_6_full_inventory['Variable'].isin(attendance_discipline_non_candidate_variables),
    decision='Exclude', representation='None', destination_domain='School attitudes and engagement', reason='Reason, routing or comparative-perception item rather than a principal predictor representation')
bullying_review_mask = domain_6_full_inventory['Review track'].fillna('').astype('string').str.contains('Bullying and school safety',
    case=False, regex=False) | domain_6_full_inventory['Matched construct'].fillna('').astype('string').str.contains('Bullying and school safety',
    case=False, regex=False)
assign_domain_6_audit_decision(bullying_review_mask, decision='Defer',
    representation='Bullying and school-safety candidate', destination_domain='Experiences and behaviours', reason='Represents adverse school experience rather than the retained attitude or attendance constructs')
domain_6_full_inventory['Audit status'] = domain_6_full_inventory['Provisional audit decision'].notna().map({True: 'Decision assigned',
    False: 'Further review required'})
domain_6_audit_summary = domain_6_full_inventory.groupby(['Audit status', 'Provisional audit decision',
    'Provisional destination domain'], dropna=False).size().rename('Variables').reset_index().sort_values(['Audit status',
    'Provisional audit decision']).reset_index(drop=True)
domain_6_unresolved_variables = domain_6_full_inventory.loc[domain_6_full_inventory['Provisional audit decision'].isna()][['Wave',
    'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Review track', 'Matched construct', 'Timing status']].sort_values(['Wave',
    'Source file', 'Variable position']).reset_index(drop=True)
domain_6_assigned_decisions = domain_6_full_inventory.loc[domain_6_full_inventory['Provisional audit decision'].notna()][['Wave',
    'Variable', 'Variable label', 'YYS item number', 'Provisional audit decision', 'Provisional representation', 'Provisional destination domain', 'Audit reason']].sort_values(['Provisional audit decision',
    'Wave', 'Variable']).reset_index(drop=True)
print(f'Unique Domain 6 source variables audited: {len(domain_6_full_inventory):,}')
print(f'Variables requiring further review: {len(domain_6_unresolved_variables):,}')
print('Domain 6 audit summary:')
display_limited(domain_6_audit_summary)
print('Variables requiring further review:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_6_unresolved_variables)
print('Assigned variable-level decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_6_assigned_decisions)

Unique Domain 6 source variables audited: 153
Variables requiring further review: 15
Domain 6 audit summary:


,Audit status,Provisional audit decision,Provisional destination domain,Variables
0,Decision assigned,Construction input,School attitudes and engagement,39
1,Decision assigned,Defer,Experiences and behaviours,52
2,Decision assigned,Defer,Psychosocial,2
3,Decision assigned,Defer,School and local context,2
4,Decision assigned,Exclude,School attitudes and engagement,15


Variables requiring further review:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Review track,Matched construct,Timing status
0,Wave 1,Young person,wave_one_lsype_young_person_2020,132,W1hwhaveYP,YP: Whether ever set homework at school,Core school attitudes and engagement,Homework and independent study,Pre-transition source
1,Wave 1,Young person,wave_one_lsype_young_person_2020,133,W1hwdoYP,YP: How often YP set homework,Core school attitudes and engagement,Homework and independent study,Pre-transition source
2,Wave 1,Young person,wave_one_lsype_young_person_2020,134,W1hwdo1YP,YP: Whether spend any time doing homework in typical term-time week,Core school attitudes and engagement,Homework and independent study,Pre-transition source
3,Wave 1,Young person,wave_one_lsype_young_person_2020,135,W1hwndayYP,YP: Number of evenings do homework,Core school attitudes and engagement,Homework and independent study,Pre-transition source
4,Wave 1,Young person,wave_one_lsype_young_person_2020,136,W1hwhelpYP,YP: Whether anyone at home helps them with homework,Core school attitudes and engagement,Homework and independent study,Pre-transition source


Assigned variable-level decisions:


,Wave,Variable,Variable label,YYS item number,Provisional audit decision,Provisional representation,Provisional destination domain,Audit reason
0,Wave 1,W1truantYP,YP: Whether played truant in last 12 months,<NA>,Construction input,truancy_status_pretransition,School attitudes and engagement,Repeated direct status item used in the latest available pre-transition truancy representation
1,Wave 1,W1yys10YP,YP: Feelings about school: The work I do in lessons is a waste of time,10,Construction input,school_attitude_score,School attitudes and engagement,Repeated item represented by the derived twelve-item school-attitude score
2,Wave 1,W1yys11YP,YP: Feelings about school: The work I do in lessons is interesting to me,11,Construction input,school_attitude_score,School attitudes and engagement,Repeated item represented by the derived twelve-item school-attitude score
3,Wave 1,W1yys12YP,YP: Feelings about school: I get good marks for my work,12,Construction input,school_attitude_score,School attitudes and engagement,Repeated item represented by the derived twelve-item school-attitude score
4,Wave 1,W1yys1YP,YP: Feelings about school: I am happy when I am at school,1,Construction input,school_attitude_score,School attitudes and engagement,Repeated item represented by the derived twelve-item school-attitude score


In [285]:
# 20: Homework candidate coding and routing review

import pandas as pd
homework_candidate_inventory = domain_6_unresolved_variables.copy()
assert len(homework_candidate_inventory) == 15
assert homework_candidate_inventory['Matched construct'].eq('Homework and independent study').all()
homework_variable_configuration = {'W1hwhaveYP': {'Family': 'Homework provision',
    'Construct': 'Whether homework is set', 'Wave-neutral family': 'homework_set'}, 'W2hwhaveYP': {'Family': 'Homework provision',
    'Construct': 'Whether homework is set', 'Wave-neutral family': 'homework_set'}, 'W1hwdoYP': {'Family': 'Homework provision',
    'Construct': 'Frequency with which homework is set', 'Wave-neutral family': 'homework_set_frequency'}, 'W2hwdoYP': {'Family': 'Homework provision',
    'Construct': 'Frequency with which homework is set', 'Wave-neutral family': 'homework_set_frequency'}, 'W1hwdo1YP': {'Family': 'Independent study behaviour',
    'Construct': 'Whether any time is spent doing homework', 'Wave-neutral family': 'homework_any_time'}, 'W2hwdo1YP': {'Family': 'Independent study behaviour',
    'Construct': 'Whether any time is spent doing homework', 'Wave-neutral family': 'homework_any_time'}, 'W1hwndayYP': {'Family': 'Independent study behaviour',
    'Construct': 'Number of evenings spent doing homework', 'Wave-neutral family': 'homework_evenings'}, 'W2hwnday1YP': {'Family': 'Independent study behaviour',
    'Construct': 'Number of evenings spent doing homework', 'Wave-neutral family': 'homework_evenings'}, 'W2hwnday2YP': {'Family': 'Homework timing detail',
    'Construct': 'When homework is undertaken', 'Wave-neutral family': 'homework_timing'}, 'W1hwhelpYP': {'Family': 'Home homework support',
    'Construct': 'Whether anyone at home helps with homework', 'Wave-neutral family': 'homework_home_help'}, 'W2hwhelpYP': {'Family': 'Home homework support',
    'Construct': 'Whether anyone at home helps with homework', 'Wave-neutral family': 'homework_home_help'}, 'W1hwpchlYP': {'Family': 'Home homework monitoring',
    'Construct': 'Whether anyone at home ensures homework is done', 'Wave-neutral family': 'homework_home_monitoring'}, 'W2hwpchlYP': {'Family': 'Home homework monitoring',
    'Construct': 'Whether anyone at home ensures homework is done', 'Wave-neutral family': 'homework_home_monitoring'}, 'W1hwtchkYP': {'Family': 'Teacher homework monitoring',
    'Construct': 'Extent to which teachers ensure homework is done', 'Wave-neutral family': 'homework_teacher_monitoring'}, 'W2hwtchkYP': {'Family': 'Teacher homework monitoring',
    'Construct': 'Extent to which teachers ensure homework is done', 'Wave-neutral family': 'homework_teacher_monitoring'}}
assert set(homework_candidate_inventory['Variable']) == set(homework_variable_configuration)
homework_candidate_inventory['Review family'] = homework_candidate_inventory['Variable'].map(lambda variable: homework_variable_configuration[variable]['Family'])
homework_candidate_inventory['Review construct'] = homework_candidate_inventory['Variable'].map(lambda variable: homework_variable_configuration[variable]['Construct'])
homework_candidate_inventory['Wave-neutral family'] = homework_candidate_inventory['Variable'].map(lambda variable: homework_variable_configuration[variable]['Wave-neutral family'])
homework_raw_tables = {}
homework_labelled_tables = {}
homework_quality_rows = []
homework_code_rows = []
for source_file, source_inventory in homework_candidate_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    raw_data['NSID'] = standardise_nsid(raw_data['NSID'])
    labelled_data['NSID'] = standardise_nsid(labelled_data['NSID'])
    assert raw_data['NSID'].is_unique
    assert labelled_data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(route_index)
    labelled_data = labelled_data.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    homework_raw_tables[source_file] = raw_data
    homework_labelled_tables[source_file] = labelled_data
    for variable in source_variables:
        variable_inventory = source_inventory.loc[source_inventory['Variable'].eq(variable)].iloc[0]
        raw_values = raw_data[variable]
        labelled_values = labelled_data[variable].astype('string')
        observed_mask = raw_values.ge(0).fillna(False)
        special_code_mask = raw_values.lt(0).fillna(False)
        observed_values = raw_values.loc[observed_mask]
        homework_quality_rows.append({'Review family': variable_inventory['Review family'],
            'Review construct': variable_inventory['Review construct'], 'Wave-neutral family': variable_inventory['Wave-neutral family'], 'Wave': variable_inventory['Wave'], 'Variable': variable, 'Variable label': variable_inventory['Variable label'], 'Observed responses': int(observed_mask.sum()), 'Special-code responses': int(special_code_mask.sum()), 'No source record': int(raw_values.isna().sum()), 'Observed percentage': round(observed_mask.mean() * 100,
            2), 'Distinct observed codes': int(observed_values.nunique()), 'Minimum observed code': observed_values.min() if len(observed_values) > 0 else pd.NA, 'Maximum observed code': observed_values.max() if len(observed_values) > 0 else pd.NA, 'Not-applicable responses': int(raw_values.eq(-91).sum())})
        value_counts = raw_values.value_counts(dropna=False)
        for raw_code, participants in value_counts.items():
            if pd.isna(raw_code):
                value_label = 'No source record'
                response_type = 'No source record'
                code_sort_value = 999999
            else:
                code_mask = raw_values.eq(raw_code)
                matching_labels = labelled_values.loc[code_mask].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(raw_code)
                response_type = 'Observed response' if raw_code >= 0 else 'Special code'
                code_sort_value = float(raw_code)
            homework_code_rows.append({'Review family': variable_inventory['Review family'],
                'Wave': variable_inventory['Wave'], 'Variable': variable, 'Variable label': variable_inventory['Variable label'], 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Code sort value': code_sort_value})
homework_variable_quality = pd.DataFrame(homework_quality_rows).sort_values(['Review family', 'Wave-neutral family',
    'Wave']).reset_index(drop=True)
homework_code_distribution = pd.DataFrame(homework_code_rows).sort_values(['Review family', 'Variable',
    'Code sort value']).drop(columns=['Code sort value']).reset_index(drop=True)
homework_family_summary = homework_candidate_inventory.sort_values(['Wave-neutral family',
    'Wave']).groupby(['Review family', 'Review construct', 'Wave-neutral family'], dropna=False).agg(Waves=('Wave',
    lambda values: '; '.join(values.astype(str))), Variables=('Variable',
    lambda values: '; '.join(values.astype(str))), Number_of_variables=('Variable',
    'size')).reset_index().rename(columns={'Number_of_variables': 'Number of variables'})
homework_routing_review = homework_variable_quality[['Review family', 'Review construct', 'Wave', 'Variable',
    'Observed responses', 'Not-applicable responses', 'Special-code responses', 'Observed percentage']].copy()
homework_routing_review['Not-applicable percentage'] = (homework_routing_review['Not-applicable responses'] / len(route_index) * 100).round(2)
homework_routing_review['Routing review flag'] = homework_routing_review['Not-applicable responses'].gt(0).map({True: 'Routing or conditional response likely',
    False: 'No -91 routing code observed'})
print('Homework construct families:')
with pd.option_context('display.max_colwidth', None):
    display_limited(homework_family_summary)
print('Homework variable quality:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(homework_variable_quality)
print('Homework routing review:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(homework_routing_review)
print('Homework response codes:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(homework_code_distribution)

Homework construct families:


,Review family,Review construct,Wave-neutral family,Waves,Variables,Number of variables
0,Home homework monitoring,Whether anyone at home ensures homework is done,homework_home_monitoring,Wave 1; Wave 2,W1hwpchlYP; W2hwpchlYP,2
1,Home homework support,Whether anyone at home helps with homework,homework_home_help,Wave 1; Wave 2,W1hwhelpYP; W2hwhelpYP,2
2,Homework provision,Frequency with which homework is set,homework_set_frequency,Wave 1; Wave 2,W1hwdoYP; W2hwdoYP,2
3,Homework provision,Whether homework is set,homework_set,Wave 1; Wave 2,W1hwhaveYP; W2hwhaveYP,2
4,Homework timing detail,When homework is undertaken,homework_timing,Wave 2,W2hwnday2YP,1


Homework variable quality:


,Review family,Review construct,Wave-neutral family,Wave,Variable,Variable label,Observed responses,Special-code responses,No source record,Observed percentage,Distinct observed codes,Minimum observed code,Maximum observed code,Not-applicable responses
0,Home homework monitoring,Whether anyone at home ensures homework is done,homework_home_monitoring,Wave 1,W1hwpchlYP,YP: Whether anyone at home makes sure that do homework,9330,194,243,95.53,5,1.0,5.0,94
1,Home homework monitoring,Whether anyone at home ensures homework is done,homework_home_monitoring,Wave 2,W2hwpchlYP,YP: Whether anyone at home makes sure that do homework,9236,285,246,94.56,5,1.0,5.0,201
2,Home homework support,Whether anyone at home helps with homework,homework_home_help,Wave 1,W1hwhelpYP,YP: Whether anyone at home helps them with homework,9335,189,243,95.58,3,1.0,3.0,94
3,Home homework support,Whether anyone at home helps with homework,homework_home_help,Wave 2,W2hwhelpYP,YP: Whether anyone at home helps them with homework,9233,288,246,94.53,3,1.0,3.0,201
4,Homework provision,Whether homework is set,homework_set,Wave 1,W1hwhaveYP,YP: Whether ever set homework at school,9434,90,243,96.59,2,1.0,2.0,0


Homework routing review:


,Review family,Review construct,Wave,Variable,Observed responses,Not-applicable responses,Special-code responses,Observed percentage,Not-applicable percentage,Routing review flag
0,Home homework monitoring,Whether anyone at home ensures homework is done,Wave 1,W1hwpchlYP,9330,94,194,95.53,0.96,Routing or conditional response likely
1,Home homework monitoring,Whether anyone at home ensures homework is done,Wave 2,W2hwpchlYP,9236,201,285,94.56,2.06,Routing or conditional response likely
2,Home homework support,Whether anyone at home helps with homework,Wave 1,W1hwhelpYP,9335,94,189,95.58,0.96,Routing or conditional response likely
3,Home homework support,Whether anyone at home helps with homework,Wave 2,W2hwhelpYP,9233,201,288,94.53,2.06,Routing or conditional response likely
4,Homework provision,Whether homework is set,Wave 1,W1hwhaveYP,9434,0,90,96.59,0.00,No -91 routing code observed


Homework response codes:


,Review family,Wave,Variable,Variable label,Raw code,Value label,Response type,Participants
0,Home homework monitoring,Wave 1,W1hwpchlYP,YP: Whether anyone at home makes sure that do homework,-99.0,YP not interviewed,Special code,89
1,Home homework monitoring,Wave 1,W1hwpchlYP,YP: Whether anyone at home makes sure that do homework,-91.0,Not applicable,Special code,94
2,Home homework monitoring,Wave 1,W1hwpchlYP,YP: Whether anyone at home makes sure that do homework,-1.0,Don't know,Special code,11
3,Home homework monitoring,Wave 1,W1hwpchlYP,YP: Whether anyone at home makes sure that do homework,1.0,Every time,Observed response,4184
4,Home homework monitoring,Wave 1,W1hwpchlYP,YP: Whether anyone at home makes sure that do homework,2.0,Sometimes,Observed response,3080


In [286]:
# 21: Independent-study homework representation review

import pandas as pd
wave_1_homework_data = homework_raw_tables['wave_one_lsype_young_person_2020']
wave_2_homework_data = homework_raw_tables['wave_two_lsype_young_person_2020']

def construct_wave_homework_evenings(source_data, homework_set_variable, homework_time_variable,
    homework_evenings_variable):
    """Construct weekly homework evenings with structural zeroes."""
    homework_set = pd.to_numeric(source_data[homework_set_variable], errors='coerce')
    homework_time = pd.to_numeric(source_data[homework_time_variable], errors='coerce')
    homework_evenings_raw = pd.to_numeric(source_data[homework_evenings_variable], errors='coerce')
    valid_evenings = homework_evenings_raw.where(homework_evenings_raw.between(0, 5,
        inclusive='both')).astype('Float64')
    structural_zero = homework_set.eq(2) | homework_time.eq(2)
    homework_evenings = valid_evenings.copy()
    homework_evenings.loc[homework_evenings.isna() & structural_zero] = 0
    positive_evenings_with_no_homework = valid_evenings.gt(0) & structural_zero
    routed_without_resolved_zero = homework_evenings_raw.eq(-91) & ~structural_zero
    construction_diagnostics = {'Source records': int(homework_evenings_raw.notna().sum()),
        'Raw valid evening responses': int(valid_evenings.notna().sum()), 'Structural zeroes added': int((valid_evenings.isna() & structural_zero).sum()), 'Positive evenings conflicting with no-homework response': int(positive_evenings_with_no_homework.sum()), 'Unresolved routed responses': int(routed_without_resolved_zero.sum()), 'Constructed values': int(homework_evenings.notna().sum())}
    return (homework_evenings, construction_diagnostics)
wave_1_homework_evenings, wave_1_homework_diagnostics = construct_wave_homework_evenings(source_data=wave_1_homework_data,
    homework_set_variable='W1hwhaveYP', homework_time_variable='W1hwdo1YP', homework_evenings_variable='W1hwndayYP')
wave_2_homework_evenings, wave_2_homework_diagnostics = construct_wave_homework_evenings(source_data=wave_2_homework_data,
    homework_set_variable='W2hwhaveYP', homework_time_variable='W2hwdo1YP', homework_evenings_variable='W2hwnday1YP')
homework_evenings_review_candidate = pd.Series(pd.NA, index=route_index, dtype='Float64',
    name='homework_evenings_pretransition')
homework_evenings_review_source = pd.Series(pd.NA, index=route_index, dtype='string', name='homework_evenings_source')
wave_2_homework_available = wave_2_homework_evenings.notna()
homework_evenings_review_candidate.loc[wave_2_homework_available] = wave_2_homework_evenings.loc[wave_2_homework_available]
homework_evenings_review_source.loc[wave_2_homework_available] = 'Wave 2'
wave_1_homework_fallback = homework_evenings_review_candidate.isna() & wave_1_homework_evenings.notna()
homework_evenings_review_candidate.loc[wave_1_homework_fallback] = wave_1_homework_evenings.loc[wave_1_homework_fallback]
homework_evenings_review_source.loc[wave_1_homework_fallback] = 'Wave 1 fallback'
assert homework_evenings_review_candidate.dropna().between(0, 5, inclusive='both').all()
homework_cross_wave_comparison = pd.DataFrame({'Wave 1 homework evenings': wave_1_homework_evenings,
    'Wave 2 homework evenings': wave_2_homework_evenings}).dropna()
homework_cross_wave_difference = homework_cross_wave_comparison['Wave 2 homework evenings'] - homework_cross_wave_comparison['Wave 1 homework evenings']
homework_cross_wave_summary = pd.DataFrame([{'Complete comparisons': len(homework_cross_wave_comparison),
    'Pearson correlation': round(homework_cross_wave_comparison['Wave 1 homework evenings'].astype('float64').corr(homework_cross_wave_comparison['Wave 2 homework evenings'].astype('float64'),
    method='pearson'), 3), 'Spearman correlation': round(homework_cross_wave_comparison['Wave 1 homework evenings'].astype('float64').corr(homework_cross_wave_comparison['Wave 2 homework evenings'].astype('float64'),
    method='spearman'), 3), 'Exact agreement percentage': round(homework_cross_wave_comparison['Wave 1 homework evenings'].eq(homework_cross_wave_comparison['Wave 2 homework evenings']).mean() * 100,
    2), 'Mean Wave 2 minus Wave 1 difference': round(homework_cross_wave_difference.mean(),
    3), 'Mean absolute difference': round(homework_cross_wave_difference.abs().mean(), 3)}])
homework_construction_diagnostics = pd.DataFrame([{'Wave': 'Wave 1', **wave_1_homework_diagnostics}, {'Wave': 'Wave 2',
    **wave_2_homework_diagnostics}])
homework_candidate_source_summary = homework_evenings_review_source.fillna('Unavailable').value_counts().rename('Participants').rename_axis('Construction source').reset_index()
homework_candidate_source_summary['Percentage'] = (homework_candidate_source_summary['Participants'] / len(route_index) * 100).round(2)
homework_candidate_quality = pd.DataFrame([{'Candidate': 'homework_evenings_pretransition',
    'Construct': 'Frequency of independent homework study', 'Representation': 'Number of evenings per week, with confirmed non-participation represented as zero', 'Primary wave': 'Wave 2', 'Fallback': 'Wave 1', 'Non-missing': int(homework_evenings_review_candidate.notna().sum()), 'Missing': int(homework_evenings_review_candidate.isna().sum()), 'Missing percentage': round(homework_evenings_review_candidate.isna().mean() * 100,
    2), 'Mean': round(homework_evenings_review_candidate.mean(),
    3), 'Standard deviation': round(homework_evenings_review_candidate.std(),
    3), 'Minimum': homework_evenings_review_candidate.min(), 'Maximum': homework_evenings_review_candidate.max()}])
homework_evenings_distribution = homework_evenings_review_candidate.value_counts(dropna=False).rename('Participants').rename_axis('Homework evenings per week').reset_index()
homework_evenings_distribution['Homework evenings per week'] = homework_evenings_distribution['Homework evenings per week'].astype('string').fillna('Unavailable')
homework_evenings_distribution['Percentage'] = (homework_evenings_distribution['Participants'] / len(route_index) * 100).round(2)
homework_school_attitude_comparison = pd.DataFrame({'Homework evenings': homework_evenings_review_candidate,
    'School-attitude score': school_attitude_score_candidate}).dropna()
homework_truancy_comparison = pd.DataFrame({'Homework evenings': homework_evenings_review_candidate,
    'Truancy status': truancy_status_pretransition_candidate}).dropna()
homework_candidate_association_summary = pd.DataFrame([{'Comparison': 'Homework evenings and school-attitude score',
    'Complete comparisons': len(homework_school_attitude_comparison), 'Spearman correlation': round(homework_school_attitude_comparison['Homework evenings'].astype('float64').corr(homework_school_attitude_comparison['School-attitude score'].astype('float64'),
    method='spearman'), 3)}, {'Comparison': 'Homework evenings and truancy status',
    'Complete comparisons': len(homework_truancy_comparison), 'Spearman correlation': round(homework_truancy_comparison['Homework evenings'].astype('float64').corr(homework_truancy_comparison['Truancy status'].astype('float64'),
    method='spearman'), 3)}])
homework_family_review_position = pd.DataFrame([{'Homework family': 'Independent study behaviour',
    'Current position': 'Continue Domain 6 review', 'Reason': "Directly represents the young person's study behaviour"}, {'Homework family': 'Home homework support and monitoring',
    'Current position': 'Defer to Domain 9', 'Reason': 'Represents parental educational support and supervision'}, {'Homework family': 'Homework provision and teacher monitoring',
    'Current position': 'Do not retain in Domain 6', 'Reason': "Represents school or teacher practice rather than the young person's engagement behaviour"}, {'Homework family': 'Homework timing detail',
    'Current position': 'Do not retain', 'Reason': 'Detailed routed response without a distinct principal construct'}])
print('Homework construction diagnostics:')
display_limited(homework_construction_diagnostics)
print('Cross-wave homework comparison:')
display_limited(homework_cross_wave_summary)
print('Homework candidate quality:')
with pd.option_context('display.max_colwidth', None):
    display_limited(homework_candidate_quality)
print('Homework candidate source:')
display_limited(homework_candidate_source_summary)
print('Homework-evenings distribution:')
display_limited(homework_evenings_distribution)
print('Associations with current Domain 6 candidates:')
display_limited(homework_candidate_association_summary)
print('Homework-family review position:')
with pd.option_context('display.max_colwidth', None):
    display_limited(homework_family_review_position)

Homework construction diagnostics:


,Wave,Source records,Raw valid evening responses,Structural zeroes added,Positive evenings conflicting with no-homework response,Unresolved routed responses,Constructed values
0,Wave 1,9524,8743,611,43,29,9354
1,Wave 2,9521,8632,705,42,33,9337


Cross-wave homework comparison:


,Complete comparisons,Pearson correlation,Spearman correlation,Exact agreement percentage,Mean Wave 2 minus Wave 1 difference,Mean absolute difference
0,9182,0.554,0.551,36.97,-0.139,1.014


Homework candidate quality:


...

,Homework evenings per week,Participants,Percentage
0,3.0,2410,24.67
1,2.0,1924,19.7
2,5.0,1696,17.36
3,4.0,1305,13.36
4,1.0,1227,12.56


Associations with current Domain 6 candidates:


,Comparison,Complete comparisons,Spearman correlation
0,Homework evenings and school-attitude score,9500,0.335
1,Homework evenings and truancy status,9480,-0.218


Homework-family review position:


,Homework family,Current position,Reason
0,Independent study behaviour,Continue Domain 6 review,Directly represents the young person's study behaviour
1,Home homework support and monitoring,Defer to Domain 9,Represents parental educational support and supervision
2,Homework provision and teacher monitoring,Do not retain in Domain 6,Represents school or teacher practice rather than the young person's engagement behaviour
3,Homework timing detail,Do not retain,Detailed routed response without a distinct principal construct


In [287]:
# 22: Homework representation decision and Domain 6 check completion

import pandas as pd
homework_evenings_pretransition_candidate = homework_evenings_review_candidate.copy().rename('homework_evenings_pretransition')
assert homework_evenings_pretransition_candidate.dropna().between(0, 5, inclusive='both').all()
assert int(homework_evenings_pretransition_candidate.notna().sum()) == 9509
homework_construction_inputs = ['W1hwhaveYP', 'W2hwhaveYP', 'W1hwdo1YP', 'W2hwdo1YP', 'W1hwndayYP', 'W2hwnday1YP']
homework_parental_variables = ['W1hwhelpYP', 'W2hwhelpYP', 'W1hwpchlYP', 'W2hwpchlYP']
homework_teacher_review_variables = ['W1hwtchkYP', 'W2hwtchkYP']
homework_excluded_detail_variables = ['W1hwdoYP', 'W2hwdoYP', 'W2hwnday2YP']
all_homework_decision_variables = homework_construction_inputs + homework_parental_variables + homework_teacher_review_variables + homework_excluded_detail_variables
assert len(all_homework_decision_variables) == 15
assert len(set(all_homework_decision_variables)) == 15
assert set(all_homework_decision_variables) == set(homework_candidate_inventory['Variable'])

def assign_homework_audit_decision(variables, decision, representation, destination_domain, reason):
    """Assign decisions to the unresolved homework variables."""
    variable_mask = domain_6_full_inventory['Variable'].isin(variables)
    assert int(variable_mask.sum()) == len(variables)
    domain_6_full_inventory.loc[variable_mask, 'Provisional audit decision'] = decision
    domain_6_full_inventory.loc[variable_mask, 'Provisional representation'] = representation
    domain_6_full_inventory.loc[variable_mask, 'Provisional destination domain'] = destination_domain
    domain_6_full_inventory.loc[variable_mask, 'Audit reason'] = reason
assign_homework_audit_decision(variables=homework_construction_inputs, decision='Construction input',
    representation='homework_evenings_pretransition', destination_domain='School attitudes and engagement', reason='Used to identify weekly homework evenings and structurally confirmed zero values')
assign_homework_audit_decision(variables=homework_parental_variables, decision='Defer',
    representation='Parental homework support and monitoring candidates', destination_domain='Parental educational attitudes and support', reason='Represents educational help or supervision provided within the home')
assign_homework_audit_decision(variables=homework_teacher_review_variables, decision='Review support',
    representation='Teacher homework-monitoring diagnostic', destination_domain='School attitudes and engagement', reason='Represents teacher practice and does not add a separate young-person engagement predictor')
assign_homework_audit_decision(variables=homework_excluded_detail_variables, decision='Exclude', representation='None',
    destination_domain='School attitudes and engagement', reason='Represents homework provision or timing detail rather than the retained independent-study behaviour')
domain_6_full_inventory['Audit status'] = domain_6_full_inventory['Provisional audit decision'].notna().map({True: 'Decision assigned',
    False: 'Further review required'})
domain_6_unresolved_variables = domain_6_full_inventory.loc[domain_6_full_inventory['Provisional audit decision'].isna()][['Wave',
    'Variable', 'Variable label', 'Review track', 'Matched construct']].reset_index(drop=True)
domain_6_completed_audit_summary = domain_6_full_inventory.groupby(['Audit status', 'Provisional audit decision',
    'Provisional destination domain'], dropna=False).size().rename('Variables').reset_index().sort_values(['Audit status',
    'Provisional audit decision', 'Provisional destination domain']).reset_index(drop=True)
assert len(domain_6_unresolved_variables) == 0
domain_6_predictor_candidates = pd.DataFrame({'school_attitude_score': school_attitude_score_candidate,
    'truancy_status_pretransition': truancy_status_pretransition_candidate, 'homework_evenings_pretransition': homework_evenings_pretransition_candidate}, index=route_index)
assert list(domain_6_predictor_candidates.columns) == ['school_attitude_score', 'truancy_status_pretransition',
    'homework_evenings_pretransition']
domain_6_candidate_summary = pd.DataFrame([{'Predictor': 'school_attitude_score',
    'Construct': 'Positive attitudes towards school and engagement with lessons', 'Form': 'Derived continuous score', 'Principal source': 'Wave 3 before September 2006', 'Fallback': 'Wave 2, then Wave 1', 'Non-missing': int(domain_6_predictor_candidates['school_attitude_score'].notna().sum()), 'Missing percentage': round(domain_6_predictor_candidates['school_attitude_score'].isna().mean() * 100,
    2)}, {'Predictor': 'truancy_status_pretransition',
    'Construct': 'Recent behavioural disengagement from school attendance', 'Form': 'Harmonised binary item', 'Principal source': 'Wave 3 before September 2006', 'Fallback': 'Wave 2, then Wave 1', 'Non-missing': int(domain_6_predictor_candidates['truancy_status_pretransition'].notna().sum()), 'Missing percentage': round(domain_6_predictor_candidates['truancy_status_pretransition'].isna().mean() * 100,
    2)}, {'Predictor': 'homework_evenings_pretransition', 'Construct': 'Frequency of independent homework study',
    'Form': 'Harmonised count from 0 to 5 evenings', 'Principal source': 'Wave 2', 'Fallback': 'Wave 1', 'Non-missing': int(domain_6_predictor_candidates['homework_evenings_pretransition'].notna().sum()), 'Missing percentage': round(domain_6_predictor_candidates['homework_evenings_pretransition'].isna().mean() * 100,
    2)}])
domain_6_candidate_correlations = domain_6_predictor_candidates.astype('float64').corr(method='spearman').round(3)
domain_6_complete_candidate_count = domain_6_predictor_candidates.notna().sum(axis=1)
domain_6_candidate_availability = domain_6_complete_candidate_count.value_counts().sort_index().rename('Participants').rename_axis('Domain 6 candidates available').reset_index()
domain_6_candidate_availability['Percentage'] = (domain_6_candidate_availability['Participants'] / len(route_index) * 100).round(2)
homework_representation_decision = pd.DataFrame([{'Representation': 'Latest available homework evenings',
    'Current decision': 'Retain as Domain 6 predictor candidate', 'Reason': 'Direct measure of independent-study behaviour with high coverage and limited overlap with the existing Domain 6 candidates'}, {'Representation': 'Wave-specific homework-evenings variables',
    'Current decision': 'Do not retain separately', 'Reason': 'Repeated measures of the same construct'}, {'Representation': 'Cross-wave homework average',
    'Current decision': 'Do not construct', 'Reason': 'Moderate cross-wave agreement supports using the more recent measure rather than smoothing change'}, {'Representation': 'Homework provision and timing variables',
    'Current decision': 'Do not retain', 'Reason': "Do not represent the young person's principal independent-study behaviour"}, {'Representation': 'Home homework help and monitoring',
    'Current decision': 'Defer to Domain 9', 'Reason': 'Represent parental educational support'}])
print(f'Domain 6 predictor candidates: {len(domain_6_predictor_candidates.columns)}')
print(f'Variables requiring further review: {len(domain_6_unresolved_variables)}')
print('Domain 6 candidate summary:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_6_candidate_summary)
print('Pairwise Spearman correlations:')
display_limited(domain_6_candidate_correlations)
print('Candidate availability:')
display_limited(domain_6_candidate_availability)
print('Homework representation decision:')
with pd.option_context('display.max_colwidth', None):
    display_limited(homework_representation_decision)
print('Completed Domain 6 audit summary:')
display_limited(domain_6_completed_audit_summary)

Domain 6 predictor candidates: 3
Variables requiring further review: 0
Domain 6 candidate summary:


,Predictor,Construct,Form,Principal source,Fallback,Non-missing,Missing percentage
0,school_attitude_score,Positive attitudes towards school and engagement with lessons,Derived continuous score,Wave 3 before September 2006,"Wave 2, then Wave 1",9511,2.62
1,truancy_status_pretransition,Recent behavioural disengagement from school attendance,Harmonised binary item,Wave 3 before September 2006,"Wave 2, then Wave 1",9491,2.83
2,homework_evenings_pretransition,Frequency of independent homework study,Harmonised count from 0 to 5 evenings,Wave 2,Wave 1,9509,2.64


Pairwise Spearman correlations:


,school_attitude_score,truancy_status_pretransition,homework_evenings_pretransition
school_attitude_score,1.000,-0.330,0.335
truancy_status_pretransition,-0.330,1.000,-0.218
homework_evenings_pretransition,0.335,-0.218,1.000


Candidate availability:


,Domain 6 candidates available,Participants,Percentage
0,0,247,2.53
1,1,7,0.07
2,2,35,0.36
3,3,9478,97.04


Homework representation decision:


,Representation,Current decision,Reason
0,Latest available homework evenings,Retain as Domain 6 predictor candidate,Direct measure of independent-study behaviour with high coverage and limited overlap with the existing Domain 6 candidates
1,Wave-specific homework-evenings variables,Do not retain separately,Repeated measures of the same construct
2,Cross-wave homework average,Do not construct,Moderate cross-wave agreement supports using the more recent measure rather than smoothing change
3,Homework provision and timing variables,Do not retain,Do not represent the young person's principal independent-study behaviour
4,Home homework help and monitoring,Defer to Domain 9,Represent parental educational support


Completed Domain 6 audit summary:


,Audit status,Provisional audit decision,Provisional destination domain,Variables
0,Decision assigned,Construction input,School attitudes and engagement,45
1,Decision assigned,Defer,Experiences and behaviours,52
2,Decision assigned,Defer,Parental educational attitudes and support,4
3,Decision assigned,Defer,Psychosocial,2
4,Decision assigned,Defer,School and local context,2


In [288]:
# 23: Domain 6 pre-write register and predictor validation

current_decision_register = variable_decision_register.copy()
required_register_columns = ['Source order', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable',
    'Variable label', 'Data type', 'Timing status', 'Review status', 'Review outcome', 'Substantive domain', 'Decision reason', 'Leakage assessment', 'Reference-period assessment', 'Documentation source', 'Review notes']
assert all((column in current_decision_register.columns for column in required_register_columns))
assert current_decision_register[['Source file', 'Variable']].duplicated().sum() == 0
domain_6_register_decision_lookup = {'Construction input': {'Review status': 'Reviewed',
    'Review outcome': 'Construction input'}, 'Review support': {'Review status': 'Reviewed',
    'Review outcome': 'Review support'}, 'Exclude': {'Review status': 'Reviewed',
    'Review outcome': 'Exclude'}, 'Defer': {'Review status': 'Deferred to later domain',
    'Review outcome': 'Pending review'}}
domain_6_proposed_register_updates = domain_6_full_inventory[['Source file', 'Variable', 'Provisional audit decision',
    'Provisional representation', 'Provisional destination domain', 'Audit reason']].copy()
domain_6_proposed_register_updates['Proposed review status'] = domain_6_proposed_register_updates['Provisional audit decision'].map(lambda decision: domain_6_register_decision_lookup[decision]['Review status'])
domain_6_proposed_register_updates['Proposed review outcome'] = domain_6_proposed_register_updates['Provisional audit decision'].map(lambda decision: domain_6_register_decision_lookup[decision]['Review outcome'])
assert domain_6_proposed_register_updates[['Proposed review status', 'Proposed review outcome']].notna().all().all()
domain_6_register_alignment = domain_6_proposed_register_updates.merge(current_decision_register[['Source file',
    'Variable', 'Wave', 'Variable label', 'Timing status', 'Review status', 'Review outcome', 'Substantive domain', 'Decision reason']], on=['Source file',
    'Variable'], how='left', validate='one_to_one', indicator=True)
assert len(domain_6_register_alignment) == len(domain_6_full_inventory)
assert domain_6_register_alignment['_merge'].eq('both').all()
domain_6_register_alignment = domain_6_register_alignment.drop(columns='_merge')
current_outcome_normalised = domain_6_register_alignment['Review outcome'].fillna('').astype('string').str.strip()
domain_6_register_alignment['Existing substantive outcome'] = ~current_outcome_normalised.isin(['', 'Pending review'])
domain_6_existing_decision_summary = domain_6_register_alignment.groupby(['Review status', 'Review outcome'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Variables', 'Review status'],
    ascending=[False, True]).reset_index(drop=True)
domain_6_prior_decision_check = domain_6_register_alignment.loc[domain_6_register_alignment['Existing substantive outcome']][['Wave',
    'Source file', 'Variable', 'Variable label', 'Review status', 'Review outcome', 'Substantive domain', 'Decision reason', 'Provisional audit decision', 'Provisional representation', 'Provisional destination domain', 'Proposed review status', 'Proposed review outcome']].sort_values(['Wave',
    'Source file', 'Variable']).reset_index(drop=True)
domain_6_predictor_file = domain_6_predictor_candidates.copy()
domain_6_predictor_file.index.name = 'NSID'
domain_6_predictor_file = domain_6_predictor_file.reset_index()
assert len(domain_6_predictor_file) == len(route_index)
assert domain_6_predictor_file['NSID'].notna().all()
assert domain_6_predictor_file['NSID'].is_unique
assert list(domain_6_predictor_file.columns) == ['NSID', 'school_attitude_score', 'truancy_status_pretransition',
    'homework_evenings_pretransition']
assert domain_6_predictor_file['school_attitude_score'].dropna().between(1, 4, inclusive='both').all()
assert domain_6_predictor_file['truancy_status_pretransition'].dropna().isin([0, 1]).all()
assert domain_6_predictor_file['homework_evenings_pretransition'].dropna().between(0, 5, inclusive='both').all()
domain_6_predictor_file_validation = pd.DataFrame([{'Rows': len(domain_6_predictor_file),
    'Unique NSID': domain_6_predictor_file['NSID'].nunique(), 'Duplicate NSID': int(domain_6_predictor_file['NSID'].duplicated().sum()), 'Predictors': len(domain_6_predictor_file.columns) - 1, 'All three available': int(domain_6_predictor_file[['school_attitude_score',
    'truancy_status_pretransition', 'homework_evenings_pretransition']].notna().all(axis=1).sum())}])
print(f'Domain 6 variables aligned with register: {len(domain_6_register_alignment):,}')
print(f'Variables with an existing substantive outcome: {len(domain_6_prior_decision_check):,}')
print('Current register status for Domain 6 variables:')
display_limited(domain_6_existing_decision_summary)
print('Earlier substantive decisions requiring comparison:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_6_prior_decision_check)
print('Proposed predictor-file validation:')
display_limited(domain_6_predictor_file_validation)

Domain 6 variables aligned with register: 153
Variables with an existing substantive outcome: 0
Current register status for Domain 6 variables:


,Review status,Review outcome,Variables
0,"Variable-level coding, routing and reference-p...",Pending review,118
1,Use requires interview timing confirming Janua...,Pending review,35


Earlier substantive decisions requiring comparison:


,Wave,Source file,Variable,Variable label,Review status,Review outcome,Substantive domain,Decision reason,Provisional audit decision,Provisional representation,Provisional destination domain,Proposed review status,Proposed review outcome


Proposed predictor-file validation:


,Rows,Unique NSID,Duplicate NSID,Predictors,All three available
0,9767,9767,0,3,9478


In [289]:
# 24: Domain 6 output creation and decision register

domain_6_predictor_path = stage_2_output_directory / 'stage_2_school_attitudes_engagement_domain_predictors.csv'
domain_6_decision_path = stage_2_output_directory / 'stage_2_school_attitudes_engagement_domain_decisions.csv'
domain_6_register_path = decision_register_path
domain_6_update_payload = domain_6_register_alignment[['Source file', 'Variable', 'Wave', 'Provisional audit decision',
    'Provisional representation', 'Provisional destination domain', 'Audit reason', 'Proposed review status', 'Proposed review outcome']].copy()
assert len(domain_6_update_payload) == 153
assert domain_6_update_payload[['Source file', 'Variable']].duplicated().sum() == 0
domain_6_update_payload['Proposed leakage assessment'] = 'No outcome or supplementary sequence information used in selection or construction'
domain_6_update_payload['Proposed reference-period assessment'] = 'Measured before the post-16 transition'
wave_3_update_mask = domain_6_update_payload['Wave'].eq('Wave 3')
wave_3_construction_mask = wave_3_update_mask & domain_6_update_payload['Provisional audit decision'].eq('Construction input')
wave_3_deferred_mask = wave_3_update_mask & domain_6_update_payload['Provisional audit decision'].eq('Defer')
wave_3_other_mask = wave_3_update_mask & ~wave_3_construction_mask & ~wave_3_deferred_mask
domain_6_update_payload.loc[wave_3_construction_mask,
    'Proposed reference-period assessment'] = 'Used only where the Wave 3 interview occurred before September 2006'
domain_6_update_payload.loc[wave_3_deferred_mask,
    'Proposed reference-period assessment'] = 'Near-transition item deferred for later review; any later use must be restricted to interviews before September 2006'
domain_6_update_payload.loc[wave_3_other_mask,
    'Proposed reference-period assessment'] = 'Reviewed with Wave 3 timing controls and not retained as a predictor'
domain_6_update_payload['Proposed review notes'] = 'Representation: ' + domain_6_update_payload['Provisional representation'].fillna('None').astype('string') + '. ' + domain_6_update_payload['Audit reason'].fillna('').astype('string')
domain_6_register_update_fields = domain_6_update_payload[['Source file', 'Variable', 'Proposed review status',
    'Proposed review outcome', 'Provisional destination domain', 'Audit reason', 'Proposed leakage assessment', 'Proposed reference-period assessment', 'Proposed review notes']].rename(columns={'Proposed review status': '__review_status',
    'Proposed review outcome': '__review_outcome', 'Provisional destination domain': '__substantive_domain', 'Audit reason': '__decision_reason', 'Proposed leakage assessment': '__leakage_assessment', 'Proposed reference-period assessment': '__reference_period_assessment', 'Proposed review notes': '__review_notes'})
domain_6_decision_register = current_decision_register.merge(domain_6_register_update_fields, on=['Source file',
    'Variable'], how='left', validate='one_to_one', sort=False)
assert len(domain_6_decision_register) == len(current_decision_register)
assert domain_6_decision_register[['Source file', 'Variable']].equals(current_decision_register[['Source file',
    'Variable']])
domain_6_register_update_mask = domain_6_decision_register['__review_status'].notna()
assert int(domain_6_register_update_mask.sum()) == 153
register_field_updates = {'Review status': '__review_status', 'Review outcome': '__review_outcome',
    'Substantive domain': '__substantive_domain', 'Decision reason': '__decision_reason', 'Leakage assessment': '__leakage_assessment', 'Reference-period assessment': '__reference_period_assessment', 'Review notes': '__review_notes'}
for register_column, update_column in register_field_updates.items():
    domain_6_decision_register.loc[domain_6_register_update_mask,
        register_column] = domain_6_decision_register.loc[domain_6_register_update_mask, update_column].to_numpy()
temporary_update_columns = [column for column in domain_6_decision_register.columns if column.startswith('__')]
domain_6_decision_register = domain_6_decision_register.drop(columns=temporary_update_columns)
domain_6_decision_register = domain_6_decision_register[required_register_columns]
assert list(domain_6_decision_register.columns) == required_register_columns
assert domain_6_decision_register[['Source file', 'Variable']].duplicated().sum() == 0
domain_6_decision_keys = domain_6_update_payload[['Source file', 'Variable']].copy()
domain_6_domain_decisions = domain_6_decision_register.merge(domain_6_decision_keys, on=['Source file', 'Variable'],
    how='inner', validate='one_to_one').sort_values(['Source order', 'Variable position']).reset_index(drop=True)
assert len(domain_6_domain_decisions) == 153
assert domain_6_domain_decisions[['Review status', 'Review outcome', 'Substantive domain', 'Decision reason',
    'Leakage assessment', 'Reference-period assessment', 'Review notes']].notna().all().all()
domain_6_expected_outcome_counts = {'Construction input': 45, 'Review support': 30, 'Exclude': 18,
    'Pending review': 60}
domain_6_actual_outcome_counts = domain_6_domain_decisions['Review outcome'].value_counts().to_dict()
assert domain_6_actual_outcome_counts == domain_6_expected_outcome_counts
domain_6_predictor_output = domain_6_predictor_file.copy()
assert len(domain_6_predictor_output) == 9767
assert domain_6_predictor_output['NSID'].is_unique
assert list(domain_6_predictor_output.columns) == ['NSID', 'school_attitude_score', 'truancy_status_pretransition',
    'homework_evenings_pretransition']
domain_6_predictor_output.to_csv(domain_6_predictor_path, index=False)
domain_6_domain_decisions.to_csv(domain_6_decision_path, index=False)
domain_6_decision_register.to_csv(domain_6_register_path, index=False)
verified_domain_6_predictors = pd.read_csv(domain_6_predictor_path, dtype={'NSID': 'string'})
verified_domain_6_decisions = pd.read_csv(domain_6_decision_path)
verified_decision_register = pd.read_csv(domain_6_register_path)
assert len(verified_domain_6_predictors) == 9767
assert verified_domain_6_predictors['NSID'].is_unique
assert len(verified_domain_6_decisions) == 153
assert len(verified_decision_register) == len(current_decision_register)
assert verified_decision_register[['Source file', 'Variable']].duplicated().sum() == 0
verified_domain_6_predictor_summary = pd.DataFrame([{'Predictor': predictor,
    'Non-missing': int(verified_domain_6_predictors[predictor].notna().sum()), 'Missing': int(verified_domain_6_predictors[predictor].isna().sum()), 'Missing percentage': round(verified_domain_6_predictors[predictor].isna().mean() * 100,
    2)} for predictor in ['school_attitude_score', 'truancy_status_pretransition',
    'homework_evenings_pretransition']])
verified_domain_6_decision_summary = verified_domain_6_decisions.groupby(['Review status', 'Review outcome',
    'Substantive domain'], dropna=False).size().rename('Variables').reset_index().sort_values(['Review status',
    'Review outcome', 'Substantive domain']).reset_index(drop=True)
print('Domain 6 predictor file:')
print(domain_6_predictor_path)
print('Domain 6 decision file:')
print(domain_6_decision_path)
print('Decision register:')
print(domain_6_register_path)
print(f'Saved predictor rows: {len(verified_domain_6_predictors):,}')
print(f'Saved Domain 6 decision rows: {len(verified_domain_6_decisions):,}')
print('Saved predictor availability:')
display_limited(verified_domain_6_predictor_summary)
print('Saved Domain 6 decision summary:')
display_limited(verified_domain_6_decision_summary)

Domain 6 predictor file:
data_derived\stage_2_predictor_construction\stage_2_school_attitudes_engagement_domain_predictors.csv
Domain 6 decision file:
data_derived\stage_2_predictor_construction\stage_2_school_attitudes_engagement_domain_decisions.csv
Decision register:
data_derived\stage_2_predictor_construction\stage_2_variable_decision_register.csv
Saved predictor rows: 9,767
Saved Domain 6 decision rows: 153
Saved predictor availability:


,Predictor,Non-missing,Missing,Missing percentage
0,school_attitude_score,9511,256,2.62
1,truancy_status_pretransition,9491,276,2.83
2,homework_evenings_pretransition,9509,258,2.64


Saved Domain 6 decision summary:


,Review status,Review outcome,Substantive domain,Variables
0,Deferred to later domain,Pending review,Experiences and behaviours,52
1,Deferred to later domain,Pending review,Parental educational attitudes and support,4
2,Deferred to later domain,Pending review,Psychosocial,2
3,Deferred to later domain,Pending review,School and local context,2
4,Reviewed,Construction input,School attitudes and engagement,45


## School-attitude and engagement predictors

Three predictors were retained: `school_attitude_score`, `truancy_status_pretransition` and `homework_evenings_pretransition`. The school-attitude score uses the latest available pre-transition twelve-item measure, with earlier waves used as fallback.

All three predictors were available for 9,478 participants (97.04%). The decision file covers 153 source variables: 45 construction inputs, 30 review-support variables, 18 exclusions and 60 variables deferred to other domains.


# Part 10: Psychosocial characteristics

Psychological well-being, self-concept, perceived control, confidence, support and peer-related measures were reviewed. Health, school-attitude and behavioural measures remained in their existing domains.


In [290]:
# 1: Psychosocial domain candidate inventory

import pandas as pd
domain_7_register_source = verified_decision_register.copy()
assert domain_7_register_source[['Source file', 'Variable']].duplicated().sum() == 0
domain_7_search_register = domain_7_register_source.loc[domain_7_register_source['Wave'].isin(['Wave 1', 'Wave 2',
    'Wave 3'])].copy()
domain_7_search_register['Search text'] = domain_7_search_register['Variable'].fillna('').astype('string') + ' ' + domain_7_search_register['Variable label'].fillna('').astype('string')
domain_7_construct_patterns = {'Emotional distress and wellbeing': '\\bghq\\b|\\bmental health\\b|\\bpsychological\\b|\\bemotional distress\\b|\\bdepress|\\banxi|\\bworr|\\bstress|\\bstrain\\b|\\bsad\\b|\\bunhappy\\b|\\bhappiness\\b|\\bhappy\\b|\\bwell[- ]?being\\b|\\blife satisfaction\\b|\\bsatisfied with life\\b|\\blost sleep\\b|\\bworthless\\b',
    'Self-esteem and general self-concept': '\\bself[- ]?esteem\\b|\\bself[- ]?worth\\b|\\bself[- ]?concept\\b|\\bconfidence\\b|\\bconfident\\b|\\bfeel.*failure\\b|\\bfeel.*useful\\b|\\bfeel.*worth\\b|\\bproud of\\b|\\bgood about', 'Agency and locus of control': '\\blocus of control\\b|\\bcontrol over.*life\\b|\\bcontrol.*future\\b|\\bluck\\b|\\bfate\\b|\\bthings happen\\b|\\bmake.*own decisions\\b|\\binfluence.*life\\b', 'Behavioural and emotional difficulties': '\\bsdq\\b|\\bstrengths and difficulties\\b|\\bemotional symptoms\\b|\\bconduct problems?\\b|\\bhyperactiv|\\bprosocial\\b|\\bpeer problems?\\b|\\bbehavioural difficulties\\b|\\bemotional difficulties\\b', 'Peer and social wellbeing': '\\blonely\\b|\\bloneliness\\b|\\bfriendship\\b|\\bclose friends?\\b|\\bpeer support\\b|\\bsocial support\\b|\\bsocial isolation\\b|\\bgets on with friends\\b', 'Academic self-concept': '\\bgood.*school work\\b|\\bteachers think.*school work\\b|\\bacademic self[- ]?concept\\b|\\bschoolwork ability\\b|\\bability at school work\\b|\\byys22\\b|\\byys23\\b'}
domain_7_match_columns = []
for construct, pattern in domain_7_construct_patterns.items():
    match_column = '__match_' + construct.lower().replace(' ', '_').replace('-', '_')
    domain_7_search_register[match_column] = domain_7_search_register['Search text'].str.contains(pattern, case=False,
        regex=True, na=False)
    domain_7_match_columns.append(match_column)
domain_7_candidate_mask = domain_7_search_register[domain_7_match_columns].any(axis=1)
domain_7_candidate_inventory = domain_7_search_register.loc[domain_7_candidate_mask].copy()

def identify_domain_7_constructs(row):
    """Return all matched psychosocial constructs."""
    matched_constructs = []
    for construct, match_column in zip(domain_7_construct_patterns, domain_7_match_columns):
        if bool(row[match_column]):
            matched_constructs.append(construct)
    return '; '.join(matched_constructs)
domain_7_candidate_inventory['Matched construct'] = domain_7_candidate_inventory.apply(identify_domain_7_constructs,
    axis=1)
domain_7_required_deferred_variables = ['W1yys22YP', 'W1yys23YP']
assert set(domain_7_required_deferred_variables).issubset(set(domain_7_candidate_inventory['Variable']))
domain_7_current_outcome = domain_7_candidate_inventory['Review outcome'].fillna('').astype('string').str.strip()
domain_7_candidate_inventory['Current decision position'] = domain_7_current_outcome.isin(['',
    'Pending review']).map({True: 'Available for Domain 7 review', False: 'Existing substantive decision'})
domain_7_pending_candidates = domain_7_candidate_inventory.loc[domain_7_candidate_inventory['Current decision position'].eq('Available for Domain 7 review')].copy()
domain_7_existing_decisions = domain_7_candidate_inventory.loc[domain_7_candidate_inventory['Current decision position'].eq('Existing substantive decision')].copy()
domain_7_candidate_summary = domain_7_candidate_inventory.groupby(['Matched construct', 'Wave',
    'Current decision position'], dropna=False).size().rename('Variables').reset_index().sort_values(['Matched construct',
    'Wave', 'Current decision position']).reset_index(drop=True)
domain_7_pending_summary = domain_7_pending_candidates.groupby(['Matched construct', 'Wave', 'Timing status'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Matched construct',
    'Wave']).reset_index(drop=True)
domain_7_pending_display = domain_7_pending_candidates.sort_values(['Matched construct', 'Wave', 'Source order',
    'Variable position'])[['Matched construct', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable',
    'Variable label', 'Timing status', 'Review status', 'Review outcome', 'Substantive domain']].reset_index(drop=True)
domain_7_existing_display = domain_7_existing_decisions.sort_values(['Matched construct', 'Wave', 'Source order',
    'Variable position'])[['Matched construct', 'Wave', 'Source type', 'Source file', 'Variable', 'Variable label',
    'Review status', 'Review outcome', 'Substantive domain', 'Decision reason']].reset_index(drop=True)
domain_7_existing_constructs = pd.DataFrame([{'Existing representation': 'young_person_ghq12_score',
    'Current domain location': 'SEN, disability and health', 'Domain 7 implication': 'Do not construct a second GHQ representation; review other psychosocial constructs for distinctness'}, {'Existing representation': 'school_attitude_score',
    'Current domain location': 'School attitudes and engagement', 'Domain 7 implication': 'Avoid retaining general school-related affect as a separate psychosocial predictor'}, {'Existing representation': 'W1yys22YP and W1yys23YP',
    'Current domain location': 'Deferred from Domain 6', 'Domain 7 implication': 'Review as a possible academic self-concept representation'}])
print(f'Psychosocial variables identified: {len(domain_7_candidate_inventory):,}')
print(f'Variables available for Domain 7 review: {len(domain_7_pending_candidates):,}')
print(f'Variables with existing substantive decisions: {len(domain_7_existing_decisions):,}')
print('Candidate summary:')
display_limited(domain_7_candidate_summary)
print('Pending-candidate summary:')
display_limited(domain_7_pending_summary)
print('Existing constructed representations:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_7_existing_constructs)
print('Variables available for Domain 7 review:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_7_pending_display)
print('Matched variables with existing decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_7_existing_display)

Psychosocial variables identified: 23
Variables available for Domain 7 review: 7
Variables with existing substantive decisions: 16
Candidate summary:


,Matched construct,Wave,Current decision position,Variables
0,Academic self-concept,Wave 1,Available for Domain 7 review,2
1,Agency and locus of control,Wave 1,Available for Domain 7 review,2
2,Behavioural and emotional difficulties,Wave 1,Existing substantive decision,1
3,Behavioural and emotional difficulties,Wave 2,Existing substantive decision,1
4,Behavioural and emotional difficulties,Wave 3,Existing substantive decision,1


Pending-candidate summary:


,Matched construct,Wave,Timing status,Variables
0,Academic self-concept,Wave 1,Pre-transition source,2
1,Agency and locus of control,Wave 1,Pre-transition source,2
2,Emotional distress and wellbeing,Wave 1,Pre-transition source,3


Existing constructed representations:


,Existing representation,Current domain location,Domain 7 implication
0,young_person_ghq12_score,"SEN, disability and health",Do not construct a second GHQ representation; review other psychosocial constructs for distinctness
1,school_attitude_score,School attitudes and engagement,Avoid retaining general school-related affect as a separate psychosocial predictor
2,W1yys22YP and W1yys23YP,Deferred from Domain 6,Review as a possible academic self-concept representation


Variables available for Domain 7 review:


,Matched construct,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Substantive domain
0,Academic self-concept,Wave 1,Young person,wave_one_lsype_young_person_2020,241,W1yys22YP,YP: How good YP thinks YP is at school work,Pre-transition source,Deferred to later domain,Pending review,Psychosocial
1,Academic self-concept,Wave 1,Young person,wave_one_lsype_young_person_2020,242,W1yys23YP,YP: How good teachers think YP is at school work,Pre-transition source,Deferred to later domain,Pending review,Psychosocial
2,Agency and locus of control,Wave 1,Young person,wave_one_lsype_young_person_2020,249,W1mothdecYP,YP: How true it is to say (step-)mother likes YP to make own decisions,Pre-transition source,"Variable-level coding, routing and reference-period review required.",Pending review,NaN
3,Agency and locus of control,Wave 1,Young person,wave_one_lsype_young_person_2020,250,W1fathdecYP,YP: How true it is to say (step-)father likes YP to make own decisions,Pre-transition source,"Variable-level coding, routing and reference-period review required.",Pending review,NaN
4,Emotional distress and wellbeing,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,139,W1StascHS0p,HR: First choice because YP will be happy there,Pre-transition source,"Variable-level coding, routing and reference-period review required.",Pending review,NaN


Matched variables with existing decisions:


,Matched construct,Wave,Source type,Source file,Variable,Variable label,Review status,Review outcome,Substantive domain,Decision reason
0,Behavioural and emotional difficulties,Wave 1,Young person,wave_one_lsype_young_person_2020,W1sen1MP0f,"MP: Nature of YP's special needs: Attention deficit, hyperactivity disorder, adh","Variable-level coding, routing and reference-period review required.",Exclude from predictor set,Special educational needs,"Conditional multiple-response SEN-type item asked only among participants routed through current SEN status. The item is sparse, non-mutually exclusive, repeated across waves and overlaps with the consolidated pre-transition SEN-status measure."
1,Behavioural and emotional difficulties,Wave 2,Young person,wave_two_lsype_young_person_2020,W2sen1MP0f,"MP: Nature of YP's special needs: Attention deficit, hyperactivity disorder, adh","Variable-level coding, routing and reference-period review required.",Exclude from predictor set,Special educational needs,"Conditional multiple-response SEN-type item asked only among participants routed through current SEN status. The item is sparse, non-mutually exclusive, repeated across waves and overlaps with the consolidated pre-transition SEN-status measure."
2,Behavioural and emotional difficulties,Wave 3,Young person,wave_three_lsype_young_person_2020,W3sen1MP0f,"MP: Nature of YP's special needs: Attention deficit, hyperactivity disorder, adh",Use requires interview timing confirming January–August 2006 and item-level reference-period review.,Exclude from predictor set,Special educational needs,"Conditional multiple-response SEN-type item asked only among participants routed through current SEN status. The item is sparse, non-mutually exclusive, repeated across waves and overlaps with the consolidated pre-transition SEN-status measure."
3,Emotional distress and wellbeing,Wave 1,Young person,wave_one_lsype_young_person_2020,W1yys1YP,YP: Feelings about school: I am happy when I am at school,Reviewed,Construction input,School attitudes and engagement,Repeated item represented by the derived twelve-item school-attitude score
4,Emotional distress and wellbeing,Wave 2,Young person,wave_two_lsype_young_person_2020,W2YYS1YP,YP: Feelings about school: I am happy when I am at school,Reviewed,Construction input,School attitudes and engagement,Repeated item represented by the derived twelve-item school-attitude score


In [291]:
# 2: Refined psychosocial candidate and item-family review

import pandas as pd
domain_7_register_review_pool = verified_decision_register.copy()
domain_7_review_outcome_normalised = domain_7_register_review_pool['Review outcome'].fillna('').astype('string').str.strip()
domain_7_pending_young_person_pool = domain_7_register_review_pool.loc[domain_7_register_review_pool['Wave'].isin(['Wave 1',
    'Wave 2', 'Wave 3']) & domain_7_register_review_pool['Source type'].eq('Young person') & domain_7_review_outcome_normalised.isin(['',
    'Pending review'])].copy()
domain_7_pending_young_person_pool['Search text'] = domain_7_pending_young_person_pool['Variable'].fillna('').astype('string') + ' ' + domain_7_pending_young_person_pool['Variable label'].fillna('').astype('string')
domain_7_refined_patterns = {'Emotional distress and wellbeing': '\\bghq\\b|\\bmental health\\b|\\bpsychological\\b|\\bdepress|\\bunhappy\\b|\\bfeeling.*happy recently\\b|\\bworr|\\bstrain\\b|\\bstress|\\blost.*sleep\\b|\\bsleep.*worry\\b|\\bworthless\\b|\\bnervous\\b|\\bmiserable\\b|\\bsad\\b',
    'Self-esteem and general self-concept': '\\bself[- ]?esteem\\b|\\bself[- ]?worth\\b|\\blosing confidence\\b|\\bconfidence in themselves\\b|\\bfeel.*failure\\b|\\bfeel.*useful\\b|\\bgood qualities\\b|\\bpositive attitude.*self\\b|\\bsatisfied.*self\\b|\\brespect.*self\\b', 'Agency and locus of control': '\\blocus of control\\b|\\bcontrol.*life\\b|\\bcontrol.*future\\b|\\bdepends on me\\b|\\bwhat happens.*me\\b|\\bmake.*own decisions\\b|\\bluck\\b|\\bfate\\b|\\bown choice\\b', 'Behavioural and emotional difficulties': '\\bsdq\\b|\\bstrengths and difficulties\\b|\\bemotional symptoms\\b|\\bconduct problems?\\b|\\bhyperactiv|\\bprosocial\\b|\\bpeer problems?\\b|\\brestless\\b|\\bfidget|\\btemper\\b|\\battention difficult', 'Peer and social wellbeing': '\\blonely\\b|\\bloneliness\\b|\\bfriendship\\b|\\bclose friends?\\b|\\bgets on with friends\\b|\\bpeer support\\b|\\bsocial support\\b|\\bsocial isolation\\b|\\bsomeone to talk to\\b', 'Academic self-concept': '\\byys22\\b|\\byys23\\b|\\bgood.*school work\\b|\\bteachers think.*school work\\b|\\bschoolwork ability\\b|\\bacademic self[- ]?concept\\b|\\bability at school work\\b'}
domain_7_refined_match_columns = []
for construct, pattern in domain_7_refined_patterns.items():
    match_column = '__domain_7_' + construct.lower().replace(' ', '_').replace('-', '_')
    domain_7_pending_young_person_pool[match_column] = domain_7_pending_young_person_pool['Search text'].str.contains(pattern,
        case=False, regex=True, na=False)
    domain_7_refined_match_columns.append(match_column)
domain_7_required_academic_variables = ['W1yys22YP', 'W1yys23YP']
domain_7_refined_candidate_mask = domain_7_pending_young_person_pool[domain_7_refined_match_columns].any(axis=1) | domain_7_pending_young_person_pool['Variable'].isin(domain_7_required_academic_variables)
domain_7_refined_candidate_inventory = domain_7_pending_young_person_pool.loc[domain_7_refined_candidate_mask].copy()

def identify_refined_domain_7_constructs(row):
    """Return all psychosocial constructs matched by one variable."""
    matched_constructs = []
    for construct, match_column in zip(domain_7_refined_patterns, domain_7_refined_match_columns):
        if bool(row[match_column]):
            matched_constructs.append(construct)
    if row['Variable'] in domain_7_required_academic_variables and 'Academic self-concept' not in matched_constructs:
        matched_constructs.append('Academic self-concept')
    return '; '.join(matched_constructs)
domain_7_refined_candidate_inventory['Matched construct'] = domain_7_refined_candidate_inventory.apply(identify_refined_domain_7_constructs,
    axis=1)
assert set(domain_7_required_academic_variables).issubset(set(domain_7_refined_candidate_inventory['Variable']))
domain_7_parental_autonomy_variables = ['W1mothdecYP', 'W1fathdecYP']
domain_7_refined_candidate_inventory['Current review position'] = 'Continue Domain 7 review'
domain_7_refined_candidate_inventory.loc[domain_7_refined_candidate_inventory['Variable'].isin(domain_7_parental_autonomy_variables),
    'Current review position'] = 'Potential reassignment to Domain 9'
domain_7_refined_psychosocial_candidates = domain_7_refined_candidate_inventory.loc[~domain_7_refined_candidate_inventory['Variable'].isin(domain_7_parental_autonomy_variables)].copy()
domain_7_reassignment_candidates = domain_7_refined_candidate_inventory.loc[domain_7_refined_candidate_inventory['Variable'].isin(domain_7_parental_autonomy_variables)].copy()
domain_7_initial_false_positive_variables = ['W1StascHS0p', 'W1YNtApHS0o', 'W1WhyBeHS0o']
domain_7_initial_false_positive_review = domain_7_pending_candidates.loc[domain_7_pending_candidates['Variable'].isin(domain_7_initial_false_positive_variables)][['Wave',
    'Source type', 'Source file', 'Variable', 'Variable label']].copy()
domain_7_initial_false_positive_review['Review position'] = 'Not a psychosocial candidate'
domain_7_initial_false_positive_review['Reason'] = "School-choice reason referring to anticipated happiness, not a measure of the young person's emotional wellbeing"
assert len(domain_7_initial_false_positive_review) == 3
domain_7_neighbour_rows = []
domain_7_full_young_person_register = verified_decision_register.loc[verified_decision_register['Wave'].isin(['Wave 1',
    'Wave 2', 'Wave 3']) & verified_decision_register['Source type'].eq('Young person')].copy()
for _, candidate_row in domain_7_refined_candidate_inventory.iterrows():
    candidate_position = int(candidate_row['Variable position'])
    neighbour_mask = domain_7_full_young_person_register['Source file'].eq(candidate_row['Source file']) & domain_7_full_young_person_register['Variable position'].between(candidate_position - 5,
        candidate_position + 5, inclusive='both')
    candidate_neighbours = domain_7_full_young_person_register.loc[neighbour_mask].copy()
    candidate_neighbours['Anchor variable'] = candidate_row['Variable']
    candidate_neighbours['Anchor construct'] = candidate_row['Matched construct']
    candidate_neighbours['Distance from anchor'] = candidate_neighbours['Variable position'] - candidate_position
    domain_7_neighbour_rows.append(candidate_neighbours)
domain_7_candidate_neighbourhood = pd.concat(domain_7_neighbour_rows, ignore_index=True,
    sort=False).sort_values(['Anchor variable', 'Distance from anchor']).reset_index(drop=True)
domain_7_refined_candidate_summary = domain_7_refined_candidate_inventory.groupby(['Matched construct', 'Wave',
    'Current review position'], dropna=False).size().rename('Variables').reset_index().sort_values(['Matched construct',
    'Wave', 'Current review position']).reset_index(drop=True)
domain_7_refined_candidate_display = domain_7_refined_candidate_inventory.sort_values(['Matched construct', 'Wave',
    'Source order', 'Variable position'])[['Matched construct', 'Current review position', 'Wave', 'Source file',
    'Variable position', 'Variable', 'Variable label', 'Timing status', 'Review status', 'Review outcome', 'Substantive domain']].reset_index(drop=True)
domain_7_neighbourhood_display = domain_7_candidate_neighbourhood[['Anchor variable', 'Anchor construct',
    'Distance from anchor', 'Wave', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Review status', 'Review outcome', 'Substantive domain']].copy()
print(f'Unresolved young-person variables searched: {len(domain_7_pending_young_person_pool):,}')
print(f'Refined psychosocial matches: {len(domain_7_refined_candidate_inventory):,}')
print(f'Candidates continuing in Domain 7: {len(domain_7_refined_psychosocial_candidates):,}')
print(f'Potential Domain 9 reassignments: {len(domain_7_reassignment_candidates):,}')
print('Refined candidate summary:')
display_limited(domain_7_refined_candidate_summary)
print('Refined candidate inventory:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_7_refined_candidate_display)
print('Initial-search false positives:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_7_initial_false_positive_review)
print('Candidate neighbourhoods:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_7_neighbourhood_display)

Unresolved young-person variables searched: 1,256
Refined psychosocial matches: 4
Candidates continuing in Domain 7: 2
Potential Domain 9 reassignments: 2
Refined candidate summary:


,Matched construct,Wave,Current review position,Variables
0,Academic self-concept,Wave 1,Continue Domain 7 review,2
1,Agency and locus of control,Wave 1,Potential reassignment to Domain 9,2


Refined candidate inventory:


,Matched construct,Current review position,Wave,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Substantive domain
0,Academic self-concept,Continue Domain 7 review,Wave 1,wave_one_lsype_young_person_2020,241,W1yys22YP,YP: How good YP thinks YP is at school work,Pre-transition source,Deferred to later domain,Pending review,Psychosocial
1,Academic self-concept,Continue Domain 7 review,Wave 1,wave_one_lsype_young_person_2020,242,W1yys23YP,YP: How good teachers think YP is at school work,Pre-transition source,Deferred to later domain,Pending review,Psychosocial
2,Agency and locus of control,Potential reassignment to Domain 9,Wave 1,wave_one_lsype_young_person_2020,249,W1mothdecYP,YP: How true it is to say (step-)mother likes YP to make own decisions,Pre-transition source,"Variable-level coding, routing and reference-period review required.",Pending review,NaN
3,Agency and locus of control,Potential reassignment to Domain 9,Wave 1,wave_one_lsype_young_person_2020,250,W1fathdecYP,YP: How true it is to say (step-)father likes YP to make own decisions,Pre-transition source,"Variable-level coding, routing and reference-period review required.",Pending review,NaN


Initial-search false positives:


,Wave,Source type,Source file,Variable,Variable label,Review position,Reason
869,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1StascHS0p,HR: First choice because YP will be happy there,Not a psychosocial candidate,"School-choice reason referring to anticipated happiness, not a measure of the young person's emotional wellbeing"
921,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1YNtApHS0o,HR: Chose independent or private school because YP will be happy there,Not a psychosocial candidate,"School-choice reason referring to anticipated happiness, not a measure of the young person's emotional wellbeing"
971,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1WhyBeHS0o,HR: Wanted YP to go to current school because YP will be happy there,Not a psychosocial candidate,"School-choice reason referring to anticipated happiness, not a measure of the young person's emotional wellbeing"


Candidate neighbourhoods:


,Anchor variable,Anchor construct,Distance from anchor,Wave,Source file,Variable position,Variable,Variable label,Review status,Review outcome,Substantive domain
0,W1fathdecYP,Agency and locus of control,-5,Wave 1,wave_one_lsype_young_person_2020,245,W1mquarreYP,YP: How often fall out with (step-)mother,"Variable-level coding, routing and reference-period review required.",Pending review,NaN
1,W1fathdecYP,Agency and locus of control,-4,Wave 1,wave_one_lsype_young_person_2020,246,W1fquarreYP,YP: How often fall out with (step-)father,"Variable-level coding, routing and reference-period review required.",Pending review,NaN
2,W1fathdecYP,Agency and locus of control,-3,Wave 1,wave_one_lsype_young_person_2020,247,W1talkmumYP,YP: How often talk to (step-)mother about things that matter to YP,"Variable-level coding, routing and reference-period review required.",Pending review,NaN
3,W1fathdecYP,Agency and locus of control,-2,Wave 1,wave_one_lsype_young_person_2020,248,W1talkdadYP,YP: How often talk to (step-)father about things that matter to YP,"Variable-level coding, routing and reference-period review required.",Pending review,NaN
4,W1fathdecYP,Agency and locus of control,-1,Wave 1,wave_one_lsype_young_person_2020,249,W1mothdecYP,YP: How true it is to say (step-)mother likes YP to make own decisions,"Variable-level coding, routing and reference-period review required.",Pending review,NaN


In [292]:
# 3: Academic self-concept item coding and overlap review

import pandas as pd
academic_self_concept_variables = ['W1yys22YP', 'W1yys23YP']
wave_1_young_person_path = source_file_lookup['wave_one_lsype_young_person_2020']
academic_self_concept_raw = pd.read_stata(wave_1_young_person_path, columns=['NSID', *academic_self_concept_variables],
    convert_categoricals=False)
academic_self_concept_labelled = pd.read_stata(wave_1_young_person_path, columns=['NSID',
    *academic_self_concept_variables], convert_categoricals=True)
academic_self_concept_raw['NSID'] = standardise_nsid(academic_self_concept_raw['NSID'])
academic_self_concept_labelled['NSID'] = standardise_nsid(academic_self_concept_labelled['NSID'])
assert academic_self_concept_raw['NSID'].is_unique
assert academic_self_concept_labelled['NSID'].is_unique
academic_self_concept_raw = academic_self_concept_raw.set_index('NSID').reindex(route_index)
academic_self_concept_labelled = academic_self_concept_labelled.set_index('NSID').reindex(route_index)
for variable in academic_self_concept_variables:
    academic_self_concept_raw[variable] = pd.to_numeric(academic_self_concept_raw[variable], errors='coerce')
academic_self_concept_quality_rows = []
for variable in academic_self_concept_variables:
    raw_values = academic_self_concept_raw[variable]
    valid_mask = raw_values.between(1, 5, inclusive='both').fillna(False)
    special_code_mask = raw_values.lt(0).fillna(False)
    academic_self_concept_quality_rows.append({'Variable': variable,
        'Variable label': domain_7_refined_candidate_inventory.loc[domain_7_refined_candidate_inventory['Variable'].eq(variable),
        'Variable label'].iloc[0], 'Valid responses': int(valid_mask.sum()), 'Special-code responses': int(special_code_mask.sum()), 'No source record': int(raw_values.isna().sum()), 'Valid percentage': round(valid_mask.mean() * 100,
        2), 'Minimum valid code': raw_values.loc[valid_mask].min(), 'Maximum valid code': raw_values.loc[valid_mask].max()})
academic_self_concept_quality = pd.DataFrame(academic_self_concept_quality_rows)
academic_self_concept_code_rows = []
for variable in academic_self_concept_variables:
    raw_values = academic_self_concept_raw[variable]
    labelled_values = academic_self_concept_labelled[variable].astype('string')
    value_counts = raw_values.value_counts(dropna=False)
    for raw_code, participants in value_counts.items():
        if pd.isna(raw_code):
            value_label = 'No source record'
            response_type = 'No source record'
            sort_value = 999999
        else:
            matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            response_type = 'Valid response' if 1 <= raw_code <= 5 else 'Special code'
            sort_value = float(raw_code)
        academic_self_concept_code_rows.append({'Variable': variable, 'Raw code': raw_code, 'Value label': value_label,
            'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
academic_self_concept_code_distribution = pd.DataFrame(academic_self_concept_code_rows).sort_values(['Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
academic_self_rating = academic_self_concept_raw['W1yys22YP'].where(academic_self_concept_raw['W1yys22YP'].between(1,
    5, inclusive='both')).astype('Float64')
perceived_teacher_rating = academic_self_concept_raw['W1yys23YP'].where(academic_self_concept_raw['W1yys23YP'].between(1,
    5, inclusive='both')).astype('Float64')
academic_self_concept_availability = pd.DataFrame({'Self-rating available': academic_self_rating.notna(),
    'Perceived teacher rating available': perceived_teacher_rating.notna()}).value_counts().rename('Participants').reset_index()
academic_self_concept_availability['Percentage'] = (academic_self_concept_availability['Participants'] / len(route_index) * 100).round(2)
academic_self_concept_complete = pd.DataFrame({'Self-rating': academic_self_rating,
    'Perceived teacher rating': perceived_teacher_rating}).dropna()
academic_self_concept_pair_summary = pd.DataFrame([{'Complete comparisons': len(academic_self_concept_complete),
    'Pearson correlation': round(academic_self_concept_complete['Self-rating'].astype('float64').corr(academic_self_concept_complete['Perceived teacher rating'].astype('float64'),
    method='pearson'), 3), 'Spearman correlation': round(academic_self_concept_complete['Self-rating'].astype('float64').corr(academic_self_concept_complete['Perceived teacher rating'].astype('float64'),
    method='spearman'), 3), 'Exact agreement percentage': round(academic_self_concept_complete['Self-rating'].eq(academic_self_concept_complete['Perceived teacher rating']).mean() * 100,
    2), 'Mean absolute code difference': round((academic_self_concept_complete['Self-rating'] - academic_self_concept_complete['Perceived teacher rating']).abs().mean(),
    3)}])
academic_self_concept_cross_tab_counts = pd.crosstab(academic_self_concept_complete['Self-rating'],
    academic_self_concept_complete['Perceived teacher rating'], margins=True)
academic_self_concept_cross_tab_row_percentage = pd.crosstab(academic_self_concept_complete['Self-rating'],
    academic_self_concept_complete['Perceived teacher rating'], normalize='index').mul(100).round(2)
print('Academic self-concept item quality:')
with pd.option_context('display.max_colwidth', None):
    display_limited(academic_self_concept_quality)
print('Academic self-concept response codes:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(academic_self_concept_code_distribution)
print('Joint item availability:')
display_limited(academic_self_concept_availability)
print('Item-pair comparison:')
display_limited(academic_self_concept_pair_summary)
print('Item cross-tabulation: counts')
display_limited(academic_self_concept_cross_tab_counts)
print('Item cross-tabulation: row percentages')
display_limited(academic_self_concept_cross_tab_row_percentage)

Academic self-concept item quality:


,Variable,Variable label,Valid responses,Special-code responses,No source record,Valid percentage,Minimum valid code,Maximum valid code
0,W1yys22YP,YP: How good YP thinks YP is at school work,9267,257,243,94.88,1.0,5.0
1,W1yys23YP,YP: How good teachers think YP is at school work,9193,331,243,94.12,1.0,5.0


Academic self-concept response codes:


,Variable,Raw code,Value label,Response type,Participants
0,W1yys22YP,-99.0,YP not interviewed,Special code,89
1,W1yys22YP,-97.0,YP refused CASI section,Special code,66
2,W1yys22YP,-96.0,YP unable to complete CASI section,Special code,49
3,W1yys22YP,-1.0,Don't know,Special code,53
4,W1yys22YP,1.0,Very good,Valid response,1994


Joint item availability:


...

,Complete comparisons,Pearson correlation,Spearman correlation,Exact agreement percentage,Mean absolute code difference
0,9172,0.705,0.715,71.18,0.317


Item cross-tabulation: counts


Perceived teacher rating,1.0,2.0,3.0,4.0,5.0,All
Self-rating,,,,,,
1.0,1323,584,71,3,4,1985
2.0,588,2802,622,10,2,4024
3.0,111,395,2306,82,5,2899
4.0,9,12,105,91,13,230
5.0,2,0,6,19,7,34


Item cross-tabulation: row percentages


Perceived teacher rating,1.0,2.0,3.0,4.0,5.0
Self-rating,,,,,
1.0,66.65,29.42,3.58,0.15,0.20
2.0,14.61,69.63,15.46,0.25,0.05
3.0,3.83,13.63,79.54,2.83,0.17
4.0,3.91,5.22,45.65,39.57,5.65
5.0,5.88,0.00,17.65,55.88,20.59


In [293]:
# 4: Academic self-concept representation review

from pathlib import Path
import numpy as np
import pandas as pd
academic_self_rating_positive = 6 - academic_self_rating
perceived_teacher_rating_positive = 6 - perceived_teacher_rating
academic_self_rating_positive.name = 'academic_self_rating_positive'
perceived_teacher_rating_positive.name = 'perceived_teacher_rating_positive'
assert academic_self_rating_positive.dropna().between(1, 5, inclusive='both').all()
assert perceived_teacher_rating_positive.dropna().between(1, 5, inclusive='both').all()
academic_self_concept_items = pd.DataFrame({'Self-rated schoolwork ability': academic_self_rating_positive,
    'Perceived teacher-rated schoolwork ability': perceived_teacher_rating_positive}, index=route_index)
academic_self_concept_item_count = academic_self_concept_items.notna().sum(axis=1)
academic_self_concept_complete_score = academic_self_concept_items.mean(axis=1, skipna=False).astype('Float64')
academic_self_concept_complete_score.name = 'academic_self_concept_complete_score'
academic_self_concept_available_item_score = academic_self_concept_items.mean(axis=1,
    skipna=True).where(academic_self_concept_item_count.ge(1)).astype('Float64')
academic_self_concept_available_item_score.name = 'academic_self_concept_available_item_score'
assert academic_self_concept_complete_score.notna().sum() == 9172
assert academic_self_concept_available_item_score.notna().sum() == 9288
assert academic_self_concept_complete_score.dropna().between(1, 5, inclusive='both').all()
assert academic_self_concept_available_item_score.dropna().between(1, 5, inclusive='both').all()
academic_self_concept_complete_items = academic_self_concept_items.dropna().astype('float64')
number_of_items = academic_self_concept_complete_items.shape[1]
item_variance_sum = academic_self_concept_complete_items.var(axis=0, ddof=1).sum()
total_score_variance = academic_self_concept_complete_items.sum(axis=1).var(ddof=1)
academic_self_concept_raw_alpha = number_of_items / (number_of_items - 1) * (1 - item_variance_sum / total_score_variance)
academic_self_concept_item_correlation = academic_self_concept_complete_items.corr(method='pearson').iloc[0, 1]
academic_self_concept_standardised_alpha = number_of_items * academic_self_concept_item_correlation / (1 + (number_of_items - 1) * academic_self_concept_item_correlation)
academic_self_concept_reliability = pd.DataFrame([{'Complete cases': len(academic_self_concept_complete_items),
    'Number of items': number_of_items, 'Pearson item correlation': round(academic_self_concept_item_correlation,
    3), 'Raw Cronbach alpha': round(academic_self_concept_raw_alpha,
    3), 'Standardised Cronbach alpha': round(academic_self_concept_standardised_alpha, 3)}])
academic_self_concept_representation_summary = pd.DataFrame([{'Representation': 'Mean requiring both items',
    'Minimum item count': 2, 'Non-missing': int(academic_self_concept_complete_score.notna().sum()), 'Missing': int(academic_self_concept_complete_score.isna().sum()), 'Missing percentage': round(academic_self_concept_complete_score.isna().mean() * 100,
    2), 'Mean': round(academic_self_concept_complete_score.mean(),
    3), 'Standard deviation': round(academic_self_concept_complete_score.std(),
    3), 'Minimum': academic_self_concept_complete_score.min(), 'Maximum': academic_self_concept_complete_score.max()}, {'Representation': 'Mean of at least one available item',
    'Minimum item count': 1, 'Non-missing': int(academic_self_concept_available_item_score.notna().sum()), 'Missing': int(academic_self_concept_available_item_score.isna().sum()), 'Missing percentage': round(academic_self_concept_available_item_score.isna().mean() * 100,
    2), 'Mean': round(academic_self_concept_available_item_score.mean(),
    3), 'Standard deviation': round(academic_self_concept_available_item_score.std(),
    3), 'Minimum': academic_self_concept_available_item_score.min(), 'Maximum': academic_self_concept_available_item_score.max()}])
academic_self_concept_item_availability = academic_self_concept_item_count.value_counts().sort_index().rename('Participants').rename_axis('Academic self-concept items available').reset_index()
academic_self_concept_item_availability['Percentage'] = (academic_self_concept_item_availability['Participants'] / len(route_index) * 100).round(2)
academic_self_concept_fallback_count = int(academic_self_concept_item_count.eq(1).sum())
assert academic_self_concept_fallback_count == 116
sen_health_predictor_path = stage_2_output_directory / 'stage_2_sen_disability_health_domain_predictors.csv'
assert sen_health_predictor_path.exists()
sen_health_predictors = pd.read_csv(sen_health_predictor_path, dtype={'NSID': 'string'})
sen_health_predictors['NSID'] = standardise_nsid(sen_health_predictors['NSID'])
assert sen_health_predictors['NSID'].is_unique
assert 'young_person_ghq12_score' in sen_health_predictors.columns
young_person_ghq12_for_review = sen_health_predictors.set_index('NSID').reindex(route_index)['young_person_ghq12_score']
young_person_ghq12_for_review = pd.to_numeric(young_person_ghq12_for_review, errors='coerce')
academic_self_concept_overlap_review = pd.DataFrame({'Academic self-concept': academic_self_concept_available_item_score,
    'School-attitude score': school_attitude_score_candidate, 'Homework evenings': homework_evenings_pretransition_candidate, 'Truancy status': truancy_status_pretransition_candidate, 'Young-person GHQ-12': young_person_ghq12_for_review}, index=route_index)
academic_self_concept_correlations = academic_self_concept_overlap_review.astype('float64').corr(method='spearman').round(3)
academic_self_concept_pairwise_summary_rows = []
for comparison_variable in ['School-attitude score', 'Homework evenings', 'Truancy status', 'Young-person GHQ-12']:
    comparison_data = academic_self_concept_overlap_review[['Academic self-concept',
        comparison_variable]].dropna().astype('float64')
    academic_self_concept_pairwise_summary_rows.append({'Comparison': 'Academic self-concept and ' + comparison_variable,
        'Complete comparisons': len(comparison_data), 'Spearman correlation': round(comparison_data['Academic self-concept'].corr(comparison_data[comparison_variable],
        method='spearman'), 3)})
academic_self_concept_pairwise_summary = pd.DataFrame(academic_self_concept_pairwise_summary_rows)
academic_self_concept_distribution = academic_self_concept_available_item_score.value_counts(dropna=False).sort_index().rename('Participants').rename_axis('Academic self-concept score').reset_index()
academic_self_concept_distribution['Academic self-concept score'] = academic_self_concept_distribution['Academic self-concept score'].astype('string').fillna('Unavailable')
academic_self_concept_distribution['Percentage'] = (academic_self_concept_distribution['Participants'] / len(route_index) * 100).round(2)
print('Academic self-concept reliability:')
display_limited(academic_self_concept_reliability)
print('Representation comparison:')
with pd.option_context('display.max_colwidth', None):
    display_limited(academic_self_concept_representation_summary)
print('Item availability:')
display_limited(academic_self_concept_item_availability)
print(f'Participants using a one-item fallback: {academic_self_concept_fallback_count:,}')
print('Pairwise associations with existing predictors:')
display_limited(academic_self_concept_pairwise_summary)
print('Predictor-only Spearman correlation matrix:')
display_limited(academic_self_concept_correlations)
print('Available-item score distribution:')
with pd.option_context('display.max_rows', None):
    display_limited(academic_self_concept_distribution)

Academic self-concept reliability:


,Complete cases,Number of items,Pearson item correlation,Raw Cronbach alpha,Standardised Cronbach alpha
0,9172,2,0.705,0.827,0.827


Representation comparison:


,Representation,Minimum item count,Non-missing,Missing,Missing percentage,Mean,Standard deviation,Minimum,Maximum
0,Mean requiring both items,2,9172,595,6.09,3.833,0.742,1.0,5.0
1,Mean of at least one available item,1,9288,479,4.90,3.827,0.747,1.0,5.0


Item availability:


...

,Comparison,Complete comparisons,Spearman correlation
0,Academic self-concept and School-attitude score,9286,0.340
1,Academic self-concept and Homework evenings,9283,0.260
2,Academic self-concept and Truancy status,9269,-0.165
3,Academic self-concept and Young-person GHQ-12,8900,-0.037


Predictor-only Spearman correlation matrix:


,Academic self-concept,School-attitude score,Homework evenings,Truancy status,Young-person GHQ-12
Academic self-concept,1.000,0.340,0.260,-0.165,-0.037
School-attitude score,0.340,1.000,0.335,-0.330,-0.178
Homework evenings,0.260,0.335,1.000,-0.218,0.022
Truancy status,-0.165,-0.330,-0.218,1.000,0.139
Young-person GHQ-12,-0.037,-0.178,0.022,0.139,1.000


Available-item score distribution:


,Academic self-concept score,Participants,Percentage
0,1.0,12,0.12
1,1.5,32,0.33
2,2.0,113,1.16
3,2.5,189,1.94
4,3.0,2386,24.43


In [294]:
# 5: Academic self-concept decision and Domain 7 check

import pandas as pd
academic_self_concept_score_candidate = academic_self_concept_available_item_score.copy().rename('academic_self_concept_score')
assert int(academic_self_concept_score_candidate.notna().sum()) == 9288
assert academic_self_concept_score_candidate.dropna().between(1, 5, inclusive='both').all()
domain_7_predictor_candidates = pd.DataFrame({'academic_self_concept_score': academic_self_concept_score_candidate},
    index=route_index)
assert list(domain_7_predictor_candidates.columns) == ['academic_self_concept_score']
domain_7_review_inventory = domain_7_pending_candidates.copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(domain_7_review_inventory) == 7
assert domain_7_review_inventory[['Source file', 'Variable']].duplicated().sum() == 0
domain_7_review_inventory['Domain 7 decision'] = pd.Series(pd.NA, index=domain_7_review_inventory.index,
    dtype='string')
domain_7_review_inventory['Domain 7 representation'] = pd.Series(pd.NA, index=domain_7_review_inventory.index,
    dtype='string')
domain_7_review_inventory['Destination domain'] = pd.Series(pd.NA, index=domain_7_review_inventory.index,
    dtype='string')
domain_7_review_inventory['Domain 7 reason'] = pd.Series(pd.NA, index=domain_7_review_inventory.index, dtype='string')

def assign_domain_7_decision(variables, decision, representation, destination_domain, reason):
    """Assign a decision to unresolved Domain 7 variables."""
    decision_mask = domain_7_review_inventory['Variable'].isin(variables)
    assert int(decision_mask.sum()) == len(variables)
    assert domain_7_review_inventory.loc[decision_mask, 'Domain 7 decision'].isna().all()
    domain_7_review_inventory.loc[decision_mask, 'Domain 7 decision'] = decision
    domain_7_review_inventory.loc[decision_mask, 'Domain 7 representation'] = representation
    domain_7_review_inventory.loc[decision_mask, 'Destination domain'] = destination_domain
    domain_7_review_inventory.loc[decision_mask, 'Domain 7 reason'] = reason
assign_domain_7_decision(variables=['W1yys22YP', 'W1yys23YP'], decision='Construction input',
    representation='academic_self_concept_score', destination_domain='Psychosocial', reason='Two closely related and reliably aligned items combined into one positively directed academic self-concept score')
assign_domain_7_decision(variables=['W1mothdecYP', 'W1fathdecYP'], decision='Defer',
    representation='Parental autonomy-support candidates', destination_domain='Parental educational attitudes and support', reason="Measures whether parents permit independent decisions, rather than the young person's own agency or locus of control")
assign_domain_7_decision(variables=['W1StascHS0p', 'W1YNtApHS0o', 'W1WhyBeHS0o'], decision='Exclude',
    representation='None', destination_domain='Psychosocial', reason='School-choice reason referring to anticipated happiness rather than a measure of emotional wellbeing')
domain_7_unresolved_after_audit = domain_7_review_inventory.loc[domain_7_review_inventory['Domain 7 decision'].isna()].copy()
assert len(domain_7_unresolved_after_audit) == 0
domain_7_audit_summary = domain_7_review_inventory.groupby(['Domain 7 decision', 'Destination domain'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Domain 7 decision',
    'Destination domain']).reset_index(drop=True)
domain_7_decision_display = domain_7_review_inventory[['Wave', 'Source type', 'Source file', 'Variable',
    'Variable label', 'Domain 7 decision', 'Domain 7 representation', 'Destination domain', 'Domain 7 reason']].copy()
domain_7_candidate_summary = pd.DataFrame([{'Predictor': 'academic_self_concept_score',
    'Construct': 'Perceived academic ability', 'Representation': 'Mean of two positively directed items, allowing one available item as fallback', 'Source': 'Wave 1', 'Number of source items': 2, 'Both items available': int(academic_self_concept_item_count.eq(2).sum()), 'One-item fallback': int(academic_self_concept_item_count.eq(1).sum()), 'Non-missing': int(academic_self_concept_score_candidate.notna().sum()), 'Missing': int(academic_self_concept_score_candidate.isna().sum()), 'Missing percentage': round(academic_self_concept_score_candidate.isna().mean() * 100,
    2), 'Minimum': academic_self_concept_score_candidate.min(), 'Maximum': academic_self_concept_score_candidate.max()}])
domain_7_representation_decisions = pd.DataFrame([{'Construct or representation': 'Academic self-concept score',
    'Current decision': 'Retain as Domain 7 predictor candidate', 'Reason': 'The two items have strong agreement and reliability, high coverage and limited overlap with existing predictors'}, {'Construct or representation': 'Complete-case two-item mean',
    'Current decision': 'Do not retain as a separate representation', 'Reason': 'Excludes 116 participants for whom one valid item provides direct information on the same construct'}, {'Construct or representation': 'Individual academic self-concept items',
    'Current decision': 'Do not retain separately', 'Reason': 'Highly related indicators of one construct'}, {'Construct or representation': 'Young-person GHQ-12',
    'Current decision': 'Preserve existing Domain 4 predictor', 'Reason': 'Already constructed as psychological-health predictor; no second representation is required'}, {'Construct or representation': 'Parental autonomy support',
    'Current decision': 'Defer to Domain 9', 'Reason': 'Represents parenting practice rather than personal agency'}, {'Construct or representation': 'School-choice happiness reasons',
    'Current decision': 'Exclude', 'Reason': "Do not measure the young person's psychosocial state"}])
domain_7_preserved_decision_summary = domain_7_existing_decisions.groupby(['Review outcome', 'Substantive domain'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Substantive domain',
    'Review outcome']).reset_index(drop=True)
domain_7_scope_checkpoint = pd.DataFrame([{'Domain': 'Psychosocial', 'New predictor candidates': 1,
    'Candidate names': 'academic_self_concept_score', 'Existing related predictor preserved': 'young_person_ghq12_score', 'Unresolved Domain 7 variables': len(domain_7_unresolved_after_audit), 'Further predictor search': 'Stop', 'Next step': 'Validate register alignment and save Domain 7 outputs'}])
print(f'Domain 7 new predictor candidates: {len(domain_7_predictor_candidates.columns)}')
print(f'Unresolved Domain 7 variables: {len(domain_7_unresolved_after_audit)}')
print('Domain 7 candidate summary:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_7_candidate_summary)
print('Domain 7 variable-level decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_7_decision_display)
print('Domain 7 audit summary:')
display_limited(domain_7_audit_summary)
print('Representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_7_representation_decisions)
print('Existing decisions preserved from other domains:')
display_limited(domain_7_preserved_decision_summary)
print('Domain 7 scope checkpoint:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_7_scope_checkpoint)

Domain 7 new predictor candidates: 1
Unresolved Domain 7 variables: 0
Domain 7 candidate summary:


,Predictor,Construct,Representation,Source,Number of source items,Both items available,One-item fallback,Non-missing,Missing,Missing percentage,Minimum,Maximum
0,academic_self_concept_score,Perceived academic ability,"Mean of two positively directed items, allowing one available item as fallback",Wave 1,2,9172,116,9288,479,4.9,1.0,5.0


Domain 7 variable-level decisions:


,Wave,Source type,Source file,Variable,Variable label,Domain 7 decision,Domain 7 representation,Destination domain,Domain 7 reason
0,Wave 1,Young person,wave_one_lsype_young_person_2020,W1yys22YP,YP: How good YP thinks YP is at school work,Construction input,academic_self_concept_score,Psychosocial,Two closely related and reliably aligned items combined into one positively directed academic self-concept score
1,Wave 1,Young person,wave_one_lsype_young_person_2020,W1yys23YP,YP: How good teachers think YP is at school work,Construction input,academic_self_concept_score,Psychosocial,Two closely related and reliably aligned items combined into one positively directed academic self-concept score
2,Wave 1,Young person,wave_one_lsype_young_person_2020,W1mothdecYP,YP: How true it is to say (step-)mother likes YP to make own decisions,Defer,Parental autonomy-support candidates,Parental educational attitudes and support,"Measures whether parents permit independent decisions, rather than the young person's own agency or locus of control"
3,Wave 1,Young person,wave_one_lsype_young_person_2020,W1fathdecYP,YP: How true it is to say (step-)father likes YP to make own decisions,Defer,Parental autonomy-support candidates,Parental educational attitudes and support,"Measures whether parents permit independent decisions, rather than the young person's own agency or locus of control"
4,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1StascHS0p,HR: First choice because YP will be happy there,Exclude,None,Psychosocial,School-choice reason referring to anticipated happiness rather than a measure of emotional wellbeing


Domain 7 audit summary:


...

,Construct or representation,Current decision,Reason
0,Academic self-concept score,Retain as Domain 7 predictor candidate,"The two items have strong agreement and reliability, high coverage and limited overlap with existing predictors"
1,Complete-case two-item mean,Do not retain as a separate representation,Excludes 116 participants for whom one valid item provides direct information on the same construct
2,Individual academic self-concept items,Do not retain separately,Highly related indicators of one construct
3,Young-person GHQ-12,Preserve existing Domain 4 predictor,Already constructed as psychological-health predictor; no second representation is required
4,Parental autonomy support,Defer to Domain 9,Represents parenting practice rather than personal agency


Existing decisions preserved from other domains:


,Review outcome,Substantive domain,Variables
0,Construction input,School attitudes and engagement,6
1,Exclude from predictor set,Special educational needs,3
2,Retain as review support only,Young-person psychological health,7


Domain 7 scope checkpoint:


,Domain,New predictor candidates,Candidate names,Existing related predictor preserved,Unresolved Domain 7 variables,Further predictor search,Next step
0,Psychosocial,1,academic_self_concept_score,young_person_ghq12_score,0,Stop,Validate register alignment and save Domain 7 outputs


In [295]:
# 6: Domain 7 pre-write register and predictor validation

import pandas as pd
domain_7_current_register = verified_decision_register.copy()
assert list(domain_7_current_register.columns) == required_register_columns
assert domain_7_current_register[['Source file', 'Variable']].duplicated().sum() == 0
domain_7_register_decision_lookup = {'Construction input': {'Review status': 'Reviewed',
    'Review outcome': 'Construction input'}, 'Defer': {'Review status': 'Deferred to later domain',
    'Review outcome': 'Pending review'}, 'Exclude': {'Review status': 'Reviewed', 'Review outcome': 'Exclude'}}
domain_7_proposed_register_updates = domain_7_review_inventory[['Source file', 'Variable', 'Domain 7 decision',
    'Domain 7 representation', 'Destination domain', 'Domain 7 reason']].copy()
domain_7_proposed_register_updates['Proposed review status'] = domain_7_proposed_register_updates['Domain 7 decision'].map(lambda decision: domain_7_register_decision_lookup[decision]['Review status'])
domain_7_proposed_register_updates['Proposed review outcome'] = domain_7_proposed_register_updates['Domain 7 decision'].map(lambda decision: domain_7_register_decision_lookup[decision]['Review outcome'])
assert len(domain_7_proposed_register_updates) == 7
assert domain_7_proposed_register_updates[['Source file', 'Variable']].duplicated().sum() == 0
assert domain_7_proposed_register_updates[['Proposed review status', 'Proposed review outcome']].notna().all().all()
domain_7_register_alignment = domain_7_proposed_register_updates.merge(domain_7_current_register[['Source file',
    'Variable', 'Wave', 'Source type', 'Variable label', 'Timing status', 'Review status', 'Review outcome', 'Substantive domain', 'Decision reason']], on=['Source file',
    'Variable'], how='left', validate='one_to_one', indicator=True)
assert len(domain_7_register_alignment) == 7
assert domain_7_register_alignment['_merge'].eq('both').all()
domain_7_register_alignment = domain_7_register_alignment.drop(columns='_merge')
domain_7_current_outcome_normalised = domain_7_register_alignment['Review outcome'].fillna('').astype('string').str.strip()
domain_7_register_alignment['Existing substantive outcome'] = ~domain_7_current_outcome_normalised.isin(['',
    'Pending review'])
domain_7_prior_decision_check = domain_7_register_alignment.loc[domain_7_register_alignment['Existing substantive outcome']].copy()
assert len(domain_7_prior_decision_check) == 0
domain_7_current_register_summary = domain_7_register_alignment.groupby(['Review status', 'Review outcome'],
    dropna=False).size().rename('Variables').reset_index()
domain_7_proposed_register_summary = domain_7_register_alignment.groupby(['Proposed review status',
    'Proposed review outcome', 'Destination domain'], dropna=False).size().rename('Variables').reset_index().sort_values(['Proposed review status',
    'Proposed review outcome', 'Destination domain']).reset_index(drop=True)
expected_domain_7_proposed_counts = {'Construction input': 2, 'Pending review': 2, 'Exclude': 3}
actual_domain_7_proposed_counts = domain_7_register_alignment['Proposed review outcome'].value_counts().to_dict()
assert actual_domain_7_proposed_counts == expected_domain_7_proposed_counts
domain_7_predictor_file = domain_7_predictor_candidates.copy()
domain_7_predictor_file.index.name = 'NSID'
domain_7_predictor_file = domain_7_predictor_file.reset_index()
assert len(domain_7_predictor_file) == len(route_index)
assert list(domain_7_predictor_file.columns) == ['NSID', 'academic_self_concept_score']
assert domain_7_predictor_file['NSID'].notna().all()
assert domain_7_predictor_file['NSID'].is_unique
assert domain_7_predictor_file['academic_self_concept_score'].dropna().between(1, 5, inclusive='both').all()
assert int(domain_7_predictor_file['academic_self_concept_score'].notna().sum()) == 9288
domain_7_predictor_file_validation = pd.DataFrame([{'Rows': len(domain_7_predictor_file),
    'Unique NSID': domain_7_predictor_file['NSID'].nunique(), 'Duplicate NSID': int(domain_7_predictor_file['NSID'].duplicated().sum()), 'Predictors': len(domain_7_predictor_file.columns) - 1, 'Academic self-concept available': int(domain_7_predictor_file['academic_self_concept_score'].notna().sum()), 'Academic self-concept missing': int(domain_7_predictor_file['academic_self_concept_score'].isna().sum())}])
print(f'Domain 7 variables aligned with register: {len(domain_7_register_alignment)}')
print(f'Variables with an existing substantive outcome: {len(domain_7_prior_decision_check)}')
print('Current register status:')
display_limited(domain_7_current_register_summary)
print('Proposed register outcomes:')
display_limited(domain_7_proposed_register_summary)
print('Proposed predictor-file validation:')
display_limited(domain_7_predictor_file_validation)

Domain 7 variables aligned with register: 7
Variables with an existing substantive outcome: 0
Current register status:


,Review status,Review outcome,Variables
0,Deferred to later domain,Pending review,2
1,"Variable-level coding, routing and reference-p...",Pending review,5


Proposed register outcomes:


,Proposed review status,Proposed review outcome,Destination domain,Variables
0,Deferred to later domain,Pending review,Parental educational attitudes and support,2
1,Reviewed,Construction input,Psychosocial,2
2,Reviewed,Exclude,Psychosocial,3


Proposed predictor-file validation:


,Rows,Unique NSID,Duplicate NSID,Predictors,Academic self-concept available,Academic self-concept missing
0,9767,9767,0,1,9288,479


In [296]:
# 7: Domain 7 output creation and decision register

domain_7_predictor_path = stage_2_output_directory / 'stage_2_psychosocial_domain_predictors.csv'
domain_7_decision_path = stage_2_output_directory / 'stage_2_psychosocial_domain_decisions.csv'
domain_7_register_path = decision_register_path
domain_7_update_payload = domain_7_register_alignment[['Source file', 'Variable', 'Wave', 'Domain 7 representation',
    'Destination domain', 'Domain 7 reason', 'Proposed review status', 'Proposed review outcome']].copy()
assert len(domain_7_update_payload) == 7
assert domain_7_update_payload[['Source file', 'Variable']].duplicated().sum() == 0
domain_7_update_payload['Proposed leakage assessment'] = 'No outcome or supplementary sequence information used in selection or construction'
domain_7_update_payload['Proposed reference-period assessment'] = 'Measured at Wave 1 before the post-16 transition'
domain_7_update_payload['Proposed review notes'] = 'Representation: ' + domain_7_update_payload['Domain 7 representation'].fillna('None').astype('string') + '. ' + domain_7_update_payload['Domain 7 reason'].fillna('').astype('string')
domain_7_register_update_fields = domain_7_update_payload[['Source file', 'Variable', 'Proposed review status',
    'Proposed review outcome', 'Destination domain', 'Domain 7 reason', 'Proposed leakage assessment', 'Proposed reference-period assessment', 'Proposed review notes']].rename(columns={'Proposed review status': '__review_status',
    'Proposed review outcome': '__review_outcome', 'Destination domain': '__substantive_domain', 'Domain 7 reason': '__decision_reason', 'Proposed leakage assessment': '__leakage_assessment', 'Proposed reference-period assessment': '__reference_period_assessment', 'Proposed review notes': '__review_notes'})
domain_7_decision_register = domain_7_current_register.merge(domain_7_register_update_fields, on=['Source file',
    'Variable'], how='left', validate='one_to_one', sort=False)
assert len(domain_7_decision_register) == len(domain_7_current_register)
assert domain_7_decision_register[['Source file', 'Variable']].equals(domain_7_current_register[['Source file',
    'Variable']])
domain_7_update_mask = domain_7_decision_register['__review_status'].notna()
assert int(domain_7_update_mask.sum()) == 7
domain_7_field_updates = {'Review status': '__review_status', 'Review outcome': '__review_outcome',
    'Substantive domain': '__substantive_domain', 'Decision reason': '__decision_reason', 'Leakage assessment': '__leakage_assessment', 'Reference-period assessment': '__reference_period_assessment', 'Review notes': '__review_notes'}
for register_column, update_column in domain_7_field_updates.items():
    domain_7_decision_register.loc[domain_7_update_mask,
        register_column] = domain_7_decision_register.loc[domain_7_update_mask, update_column].to_numpy()
domain_7_temporary_columns = [column for column in domain_7_decision_register.columns if column.startswith('__')]
domain_7_decision_register = domain_7_decision_register.drop(columns=domain_7_temporary_columns)
domain_7_decision_register = domain_7_decision_register[required_register_columns]
assert list(domain_7_decision_register.columns) == required_register_columns
assert domain_7_decision_register[['Source file', 'Variable']].duplicated().sum() == 0
domain_7_decision_keys = domain_7_update_payload[['Source file', 'Variable']].copy()
domain_7_domain_decisions = domain_7_decision_register.merge(domain_7_decision_keys, on=['Source file', 'Variable'],
    how='inner', validate='one_to_one').sort_values(['Source order', 'Variable position']).reset_index(drop=True)
assert len(domain_7_domain_decisions) == 7
domain_7_expected_outcome_counts = {'Construction input': 2, 'Pending review': 2, 'Exclude': 3}
domain_7_actual_outcome_counts = domain_7_domain_decisions['Review outcome'].value_counts().to_dict()
assert domain_7_actual_outcome_counts == domain_7_expected_outcome_counts
assert domain_7_domain_decisions[['Review status', 'Review outcome', 'Substantive domain', 'Decision reason',
    'Leakage assessment', 'Reference-period assessment', 'Review notes']].notna().all().all()
domain_7_predictor_output = domain_7_predictor_file.copy()
assert len(domain_7_predictor_output) == 9767
assert domain_7_predictor_output['NSID'].is_unique
assert list(domain_7_predictor_output.columns) == ['NSID', 'academic_self_concept_score']
domain_7_predictor_output.to_csv(domain_7_predictor_path, index=False)
domain_7_domain_decisions.to_csv(domain_7_decision_path, index=False)
domain_7_decision_register.to_csv(domain_7_register_path, index=False)
verified_domain_7_predictors = pd.read_csv(domain_7_predictor_path, dtype={'NSID': 'string'})
verified_domain_7_decisions = pd.read_csv(domain_7_decision_path)
verified_decision_register = pd.read_csv(domain_7_register_path)
assert len(verified_domain_7_predictors) == 9767
assert verified_domain_7_predictors['NSID'].is_unique
assert len(verified_domain_7_decisions) == 7
assert len(verified_decision_register) == len(domain_7_current_register)
assert verified_decision_register[['Source file', 'Variable']].duplicated().sum() == 0
assert int(verified_domain_7_predictors['academic_self_concept_score'].notna().sum()) == 9288
verified_domain_7_predictor_summary = pd.DataFrame([{'Predictor': 'academic_self_concept_score',
    'Non-missing': int(verified_domain_7_predictors['academic_self_concept_score'].notna().sum()), 'Missing': int(verified_domain_7_predictors['academic_self_concept_score'].isna().sum()), 'Missing percentage': round(verified_domain_7_predictors['academic_self_concept_score'].isna().mean() * 100,
    2), 'Minimum': verified_domain_7_predictors['academic_self_concept_score'].min(), 'Maximum': verified_domain_7_predictors['academic_self_concept_score'].max()}])
verified_domain_7_decision_summary = verified_domain_7_decisions.groupby(['Review status', 'Review outcome',
    'Substantive domain'], dropna=False).size().rename('Variables').reset_index().sort_values(['Review status',
    'Review outcome', 'Substantive domain']).reset_index(drop=True)
print('Domain 7 predictor file:')
print(domain_7_predictor_path)
print('Domain 7 decision file:')
print(domain_7_decision_path)
print('Decision register:')
print(domain_7_register_path)
print(f'Saved predictor rows: {len(verified_domain_7_predictors):,}')
print(f'Saved Domain 7 decision rows: {len(verified_domain_7_decisions):,}')
print('Saved predictor availability:')
display_limited(verified_domain_7_predictor_summary)
print('Saved Domain 7 decision summary:')
display_limited(verified_domain_7_decision_summary)

Domain 7 predictor file:
data_derived\stage_2_predictor_construction\stage_2_psychosocial_domain_predictors.csv
Domain 7 decision file:
data_derived\stage_2_predictor_construction\stage_2_psychosocial_domain_decisions.csv
Decision register:
data_derived\stage_2_predictor_construction\stage_2_variable_decision_register.csv
Saved predictor rows: 9,767
Saved Domain 7 decision rows: 7
Saved predictor availability:


,Predictor,Non-missing,Missing,Missing percentage,Minimum,Maximum
0,academic_self_concept_score,9288,479,4.9,1.0,5.0


Saved Domain 7 decision summary:


,Review status,Review outcome,Substantive domain,Variables
0,Deferred to later domain,Pending review,Parental educational attitudes and support,2
1,Reviewed,Construction input,Psychosocial,2
2,Reviewed,Exclude,Psychosocial,3


## Psychosocial predictor

Twenty-three psychosocial-related variables were identified. Sixteen already had decisions from earlier reviews, leaving seven for assessment in this part. Two Wave 1 academic self-concept items were combined into `academic_self_concept_score`; two variables were deferred and three were excluded.

The score was available for 9,288 participants and missing for 479 (4.90%).


# Part 11: Experiences and behaviours

Bullying, caring responsibilities, substance use, antisocial behaviour, police contact and work orientation were reviewed across Waves 1–3. Measures already assigned to school, health, psychosocial or family domains were not duplicated.


In [297]:
# 1: Experiences and behaviours domain candidate inventory

import pandas as pd
domain_8_register_source = verified_decision_register.copy()
assert list(domain_8_register_source.columns) == required_register_columns
assert domain_8_register_source[['Source file', 'Variable']].duplicated().sum() == 0
domain_8_search_register = domain_8_register_source.loc[domain_8_register_source['Wave'].isin(['Wave 1', 'Wave 2',
    'Wave 3'])].copy()
domain_8_search_register['Search text'] = domain_8_search_register['Variable'].fillna('').astype('string') + ' ' + domain_8_search_register['Variable label'].fillna('').astype('string')
domain_8_construct_patterns = {'Bullying, victimisation and safety': '\\bbull|\\bvictim|\\bthreaten|\\battack|\\bassault|\\brobbed\\b|\\bstolen from\\b|\\bracist behaviour\\b|\\bschool safety\\b|\\bfeel safe\\b|\\bunsafe\\b',
    'Caring responsibilities': '\\bcarehr|\\bcaring responsibilit|\\bmiss.*school.*caring\\b|\\blook after.*family\\b|\\bcare for.*family\\b', 'Smoking': '\\bsmok|\\bcigarette|\\btobacco\\b', 'Alcohol use': '\\balcohol\\b|\\bdrink alcohol\\b|\\bdrunk\\b|\\bbinge drink', 'Drug or substance use': '\\bcannabis\\b|\\billegal drug|\\bdrug use\\b|\\bsolvent|\\bglue sniff', 'Offending, police contact and antisocial behaviour': '\\bpolice\\b|\\barrest|\\boffend|\\bcrime\\b|\\bcriminal\\b|\\bshoplift|\\bvandalis|\\bdamage.*property\\b|\\bstole\\b|\\bstolen\\b|\\bweapon\\b|\\bgang\\b|\\bfight|\\bantisocial\\b|\\banti-social\\b', 'Work and unemployment orientation': '\\bplan16yp\\b|\\bjob.*unemploy|\\bunemploy.*job|\\bbetter.*job\\b|\\bbetter.*unemploy'}
domain_8_match_columns = []
for construct, pattern in domain_8_construct_patterns.items():
    match_column = '__domain_8_' + construct.lower().replace(' ', '_').replace(',', '').replace('-', '_')
    domain_8_search_register[match_column] = domain_8_search_register['Search text'].str.contains(pattern, case=False,
        regex=True, na=False)
    domain_8_match_columns.append(match_column)
domain_8_deferred_mask = domain_8_search_register['Substantive domain'].fillna('').astype('string').str.strip().eq('Experiences and behaviours')
domain_8_required_manual_variables = ['W1plan16YP']
domain_8_manual_mask = domain_8_search_register['Variable'].isin(domain_8_required_manual_variables)
domain_8_candidate_mask = domain_8_search_register[domain_8_match_columns].any(axis=1) | domain_8_deferred_mask | domain_8_manual_mask
domain_8_candidate_inventory = domain_8_search_register.loc[domain_8_candidate_mask].copy()

def identify_domain_8_constructs(row):
    """Return all matched Domain 8 constructs."""
    matched_constructs = []
    for construct, match_column in zip(domain_8_construct_patterns, domain_8_match_columns):
        if bool(row[match_column]):
            matched_constructs.append(construct)
    if row['Substantive domain'] == 'Experiences and behaviours' and (not matched_constructs):
        matched_constructs.append('Deferred experience or behaviour')
    if row['Variable'] in domain_8_required_manual_variables and 'Work and unemployment orientation' not in matched_constructs:
        matched_constructs.append('Work and unemployment orientation')
    return '; '.join(matched_constructs)
domain_8_candidate_inventory['Matched construct'] = domain_8_candidate_inventory.apply(identify_domain_8_constructs,
    axis=1)
domain_8_candidate_inventory['Deferred from earlier domain'] = domain_8_candidate_inventory['Substantive domain'].fillna('').astype('string').eq('Experiences and behaviours')
domain_8_candidate_inventory['Manual inclusion'] = domain_8_candidate_inventory['Variable'].isin(domain_8_required_manual_variables)
domain_8_current_outcome_normalised = domain_8_candidate_inventory['Review outcome'].fillna('').astype('string').str.strip()
domain_8_candidate_inventory['Current decision position'] = domain_8_current_outcome_normalised.isin(['',
    'Pending review']).map({True: 'Available for Domain 8 review', False: 'Existing substantive decision'})
domain_8_pending_candidates = domain_8_candidate_inventory.loc[domain_8_candidate_inventory['Current decision position'].eq('Available for Domain 8 review')].copy()
domain_8_existing_decisions = domain_8_candidate_inventory.loc[domain_8_candidate_inventory['Current decision position'].eq('Existing substantive decision')].copy()
assert set(domain_8_required_manual_variables).issubset(set(domain_8_candidate_inventory['Variable']))
domain_8_deferred_experience_variables = domain_8_candidate_inventory.loc[domain_8_candidate_inventory['Deferred from earlier domain'],
    'Variable'].drop_duplicates().tolist()
assert len(domain_8_deferred_experience_variables) > 0
domain_8_candidate_summary = domain_8_candidate_inventory.groupby(['Matched construct', 'Wave',
    'Current decision position'], dropna=False).size().rename('Variables').reset_index().sort_values(['Matched construct',
    'Wave', 'Current decision position']).reset_index(drop=True)
domain_8_pending_summary = domain_8_pending_candidates.groupby(['Matched construct', 'Wave', 'Timing status'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Matched construct', 'Wave',
    'Timing status']).reset_index(drop=True)
domain_8_pending_display = domain_8_pending_candidates.sort_values(['Matched construct', 'Wave', 'Source order',
    'Variable position'])[['Matched construct', 'Deferred from earlier domain', 'Manual inclusion', 'Wave',
    'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Timing status', 'Review status', 'Review outcome', 'Substantive domain']].reset_index(drop=True)
domain_8_existing_display = domain_8_existing_decisions.sort_values(['Matched construct', 'Wave', 'Source order',
    'Variable position'])[['Matched construct', 'Wave', 'Source type', 'Source file', 'Variable', 'Variable label',
    'Review status', 'Review outcome', 'Substantive domain', 'Decision reason']].reset_index(drop=True)
domain_8_review_tracks = pd.DataFrame([{'Review track': 'Bullying, victimisation and safety',
    'Purpose': 'Review direct experience measures and avoid retaining multiple routed indicators'}, {'Review track': 'Caring responsibilities',
    'Purpose': 'Review whether caring-related school absence provides a distinct pre-transition experience measure'}, {'Review track': 'Smoking, alcohol and substance use',
    'Purpose': 'Review direct behaviour measures while separating status, frequency and routed detail'}, {'Review track': 'Offending and police contact',
    'Purpose': 'Review direct contact or behaviour measures without combining them automatically'}, {'Review track': 'Work and unemployment orientation',
    'Purpose': 'Determine whether the deferred item represents an attitude, expectation or behaviour relevant to this domain'}])
print(f'Domain 8 variables identified: {len(domain_8_candidate_inventory):,}')
print(f'Variables available for Domain 8 review: {len(domain_8_pending_candidates):,}')
print(f'Variables with existing substantive decisions: {len(domain_8_existing_decisions):,}')
print(f'Variables deferred from earlier domains: {len(domain_8_deferred_experience_variables):,}')
print('Domain 8 candidate summary:')
display_limited(domain_8_candidate_summary)
print('Pending-candidate summary:')
display_limited(domain_8_pending_summary)
print('Domain 8 review tracks:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_8_review_tracks)
print('Variables available for Domain 8 review:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_8_pending_display)
print('Matched variables with existing decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_8_existing_display)

Domain 8 variables identified: 123
Variables available for Domain 8 review: 119
Variables with existing substantive decisions: 4
Variables deferred from earlier domains: 53
Domain 8 candidate summary:


,Matched construct,Wave,Current decision position,Variables
0,"Bullying, victimisation and safety",Wave 1,Available for Domain 8 review,17
1,"Bullying, victimisation and safety",Wave 2,Available for Domain 8 review,18
2,"Bullying, victimisation and safety",Wave 2,Existing substantive decision,1
3,"Bullying, victimisation and safety",Wave 3,Available for Domain 8 review,18
4,Caring responsibilities,Wave 1,Available for Domain 8 review,14


Pending-candidate summary:


,Matched construct,Wave,Timing status,Variables
0,"Bullying, victimisation and safety",Wave 1,Pre-transition source,17
1,"Bullying, victimisation and safety",Wave 2,Pre-transition source,18
2,"Bullying, victimisation and safety",Wave 3,Near-transition source,18
3,Caring responsibilities,Wave 1,Pre-transition source,14
4,Caring responsibilities,Wave 2,Pre-transition source,14


Domain 8 review tracks:


,Review track,Purpose
0,"Bullying, victimisation and safety",Review direct experience measures and avoid retaining multiple routed indicators
1,Caring responsibilities,Review whether caring-related school absence provides a distinct pre-transition experience measure
2,"Smoking, alcohol and substance use","Review direct behaviour measures while separating status, frequency and routed detail"
3,Offending and police contact,Review direct contact or behaviour measures without combining them automatically
4,Work and unemployment orientation,"Determine whether the deferred item represents an attitude, expectation or behaviour relevant to this domain"


Variables available for Domain 8 review:


,Matched construct,Deferred from earlier domain,Manual inclusion,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Substantive domain
0,"Bullying, victimisation and safety",True,False,Wave 1,Young person,wave_one_lsype_young_person_2020,74,W1pbull1MP0a,MP: Whether YP has experienced this form of bullying in last 12 months: Called n,Pre-transition source,Deferred to later domain,Pending review,Experiences and behaviours
1,"Bullying, victimisation and safety",True,False,Wave 1,Young person,wave_one_lsype_young_person_2020,75,W1pbull1MP0b,MP: Whether YP has experienced this form of bullying in last 12 months: Sent off,Pre-transition source,Deferred to later domain,Pending review,Experiences and behaviours
2,"Bullying, victimisation and safety",True,False,Wave 1,Young person,wave_one_lsype_young_person_2020,76,W1pbull1MP0c,MP: Whether YP has experienced this form of bullying in last 12 months: Shut out,Pre-transition source,Deferred to later domain,Pending review,Experiences and behaviours
3,"Bullying, victimisation and safety",True,False,Wave 1,Young person,wave_one_lsype_young_person_2020,77,W1pbull1MP0d,MP: Whether YP has experienced this form of bullying in last 12 months: Made to,Pre-transition source,Deferred to later domain,Pending review,Experiences and behaviours
4,"Bullying, victimisation and safety",True,False,Wave 1,Young person,wave_one_lsype_young_person_2020,78,W1pbull1MP0e,MP: Whether YP has experienced this form of bullying in last 12 months: Threaten,Pre-transition source,Deferred to later domain,Pending review,Experiences and behaviours


Matched variables with existing decisions:


,Matched construct,Wave,Source type,Source file,Variable,Variable label,Review status,Review outcome,Substantive domain,Decision reason
0,"Bullying, victimisation and safety",Wave 2,Young person,wave_two_lsype_young_person_2020,W2Truant2YP0a,YP: Main reason for playing truant - Bullying,Reviewed,Exclude,School attitudes and engagement,"Reason, routing or comparative-perception item rather than a principal predictor representation"
1,Caring responsibilities,Wave 3,Young person,wave_three_lsype_young_person_2020,W3pladk16aYP0f,YP: What want to do after Year 11 other than stay in FTE - Look after the family,Reviewed,support,Educational aspirations and post-16 plans,"Retained to verify routing, coding, derived-variable consistency or construct overlap"
2,Caring responsibilities,Wave 3,Young person,wave_three_lsype_young_person_2020,W3pladk16bYP0f,YP: What YP would like to be doing in 12 months time - Look after the family and,Reviewed,support,Educational aspirations and post-16 plans,"Retained to verify routing, coding, derived-variable consistency or construct overlap"
3,Caring responsibilities,Wave 3,Young person,wave_three_lsype_young_person_2020,W3pladk2bYP0f,YP: What YP expects to be doing in 12 months time - Look after the family and ho,Reviewed,construction,Educational aspirations and post-16 plans,"Used in a documented pre-transition construction of expected route, HE application likelihood or expected peer route"


In [298]:
# 2: Bullying and victimisation variable-role and coding review

import pandas as pd
domain_8_bullying_inventory = domain_8_pending_candidates.loc[domain_8_pending_candidates['Matched construct'].str.contains('Bullying, victimisation and safety',
    case=False, regex=False, na=False)].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(domain_8_bullying_inventory) == 53
bullying_young_person_status_variables = ['W1bulrc', 'W2bulrc', 'W3bulrc']
bullying_parent_status_variables = ['W1pbulrc', 'W2pbulrc', 'W3pbulrc']
bullying_cumulative_variables = ['W2bulev', 'W3bulev', 'W2pbulev', 'W3pbulev']
violence_threat_status_variables = ['W1thhitYP', 'W2thhitYP', 'W3thhitYP']
violence_threat_frequency_variables = ['W1youbulnYP0d', 'W2youbulnYP0d', 'W3youbulnYP0d']
bullying_parent_detail_variables = [variable for variable in domain_8_bullying_inventory['Variable'] if variable.lower().startswith('w1pbull1mp0') or variable.lower().startswith('w2pbull1mp0') or variable.lower().startswith('w3pbull1mp0')]
bullying_context_reason_variables = ['W1StascHS0e', 'W1YNtApHS0c', 'W1WhyBeHS0c', 'W2ResPSfinMP0f', 'W3resparYP0d']
bullying_role_lookup = {}
for variable in bullying_young_person_status_variables:
    bullying_role_lookup[variable] = 'Direct young-person bullying status'
for variable in bullying_parent_status_variables:
    bullying_role_lookup[variable] = 'Direct parental-report bullying status'
for variable in bullying_cumulative_variables:
    bullying_role_lookup[variable] = 'Cumulative bullying-history indicator'
for variable in violence_threat_status_variables:
    bullying_role_lookup[variable] = 'Direct violence-threat status'
for variable in violence_threat_frequency_variables:
    bullying_role_lookup[variable] = 'Routed violence-threat frequency'
for variable in bullying_parent_detail_variables:
    bullying_role_lookup[variable] = 'Parental-report bullying-type detail'
for variable in bullying_context_reason_variables:
    bullying_role_lookup[variable] = 'School-choice, leaving or movement reason'
assert set(bullying_role_lookup) == set(domain_8_bullying_inventory['Variable'])
domain_8_bullying_inventory['Review role'] = domain_8_bullying_inventory['Variable'].map(bullying_role_lookup)
domain_8_bullying_role_summary = domain_8_bullying_inventory.groupby(['Review role', 'Wave'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Review role', 'Wave']).reset_index(drop=True)
bullying_principal_review_variables = bullying_young_person_status_variables + bullying_parent_status_variables + bullying_cumulative_variables + violence_threat_status_variables + violence_threat_frequency_variables
assert len(bullying_principal_review_variables) == 16
assert len(set(bullying_principal_review_variables)) == 16
bullying_principal_inventory = domain_8_bullying_inventory.loc[domain_8_bullying_inventory['Variable'].isin(bullying_principal_review_variables)].copy()
assert len(bullying_principal_inventory) == 16
bullying_raw_tables = {}
bullying_labelled_tables = {}
bullying_quality_rows = []
bullying_code_rows = []
for source_file, source_inventory in bullying_principal_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    raw_data['NSID'] = standardise_nsid(raw_data['NSID'])
    labelled_data['NSID'] = standardise_nsid(labelled_data['NSID'])
    assert raw_data['NSID'].is_unique
    assert labelled_data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(route_index)
    labelled_data = labelled_data.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    bullying_raw_tables[source_file] = raw_data
    bullying_labelled_tables[source_file] = labelled_data
    for variable in source_variables:
        variable_inventory = source_inventory.loc[source_inventory['Variable'].eq(variable)].iloc[0]
        raw_values = raw_data[variable]
        labelled_values = labelled_data[variable].astype('string')
        observed_mask = raw_values.ge(0).fillna(False)
        special_code_mask = raw_values.lt(0).fillna(False)
        observed_values = raw_values.loc[observed_mask]
        bullying_quality_rows.append({'Review role': variable_inventory['Review role'],
            'Wave': variable_inventory['Wave'], 'Variable': variable, 'Variable label': variable_inventory['Variable label'], 'Observed responses': int(observed_mask.sum()), 'Special-code responses': int(special_code_mask.sum()), 'No source record': int(raw_values.isna().sum()), 'Observed percentage': round(observed_mask.mean() * 100,
            2), 'Distinct observed codes': int(observed_values.nunique()), 'Minimum observed code': observed_values.min() if len(observed_values) > 0 else pd.NA, 'Maximum observed code': observed_values.max() if len(observed_values) > 0 else pd.NA, 'Not-applicable responses': int(raw_values.eq(-91).sum())})
        value_counts = raw_values.value_counts(dropna=False)
        for raw_code, participants in value_counts.items():
            if pd.isna(raw_code):
                value_label = 'No source record'
                response_type = 'No source record'
                sort_value = 999999
            else:
                matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(raw_code)
                response_type = 'Observed response' if raw_code >= 0 else 'Special code'
                sort_value = float(raw_code)
            bullying_code_rows.append({'Review role': variable_inventory['Review role'],
                'Wave': variable_inventory['Wave'], 'Variable': variable, 'Variable label': variable_inventory['Variable label'], 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
bullying_principal_quality = pd.DataFrame(bullying_quality_rows).sort_values(['Review role', 'Wave',
    'Variable']).reset_index(drop=True)
bullying_principal_code_distribution = pd.DataFrame(bullying_code_rows).sort_values(['Review role', 'Wave', 'Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
domain_8_bullying_role_display = domain_8_bullying_inventory[['Review role', 'Wave', 'Source type', 'Source file',
    'Variable', 'Variable label', 'Timing status', 'Deferred from earlier domain']].sort_values(['Review role', 'Wave',
    'Variable']).reset_index(drop=True)
print(f'Bullying and victimisation variables reviewed: {len(domain_8_bullying_inventory)}')
print(f'Principal measures receiving coding review: {len(bullying_principal_inventory)}')
print('Bullying variable-role summary:')
display_limited(domain_8_bullying_role_summary)
print('Principal-measure quality:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(bullying_principal_quality)
print('Principal-measure response codes:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(bullying_principal_code_distribution)
print('Complete bullying variable-role audit:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_8_bullying_role_display)

Bullying and victimisation variables reviewed: 53
Principal measures receiving coding review: 16
Bullying variable-role summary:


,Review role,Wave,Variables
0,Cumulative bullying-history indicator,Wave 2,2
1,Cumulative bullying-history indicator,Wave 3,2
2,Direct parental-report bullying status,Wave 1,1
3,Direct parental-report bullying status,Wave 2,1
4,Direct parental-report bullying status,Wave 3,1


Principal-measure quality:


,Review role,Wave,Variable,Variable label,Observed responses,Special-code responses,No source record,Observed percentage,Distinct observed codes,Minimum observed code,Maximum observed code,Not-applicable responses
0,Cumulative bullying-history indicator,Wave 2,W2bulev,DV: Whether YP ever bullied in some way from start of study to current wave,8852,669,246,90.63,2,1.0,2.0,0
1,Cumulative bullying-history indicator,Wave 2,W2pbulev,DV: Whether YP ever bullied in some way from start of study to current wave(pare,7845,1676,246,80.32,2,1.0,2.0,0
2,Cumulative bullying-history indicator,Wave 3,W3bulev,DV: Whether YP ever bullied in some way from start of study to current wave,8838,671,258,90.49,2,1.0,2.0,0
3,Cumulative bullying-history indicator,Wave 3,W3pbulev,DV: Whether YP ever bullied in some way from start of study to current wave (par,7676,1833,258,78.59,2,1.0,2.0,0
4,Direct parental-report bullying status,Wave 1,W1pbulrc,DV: Whether YP bullied in some way in last 12 months (parental report),8183,1341,243,83.78,2,1.0,2.0,0


Principal-measure response codes:


,Review role,Wave,Variable,Variable label,Raw code,Value label,Response type,Participants
0,Cumulative bullying-history indicator,Wave 2,W2bulev,DV: Whether YP ever bullied in some way from start of study to current wave,-99.0,MP not interviewed,Special code,8
1,Cumulative bullying-history indicator,Wave 2,W2bulev,DV: Whether YP ever bullied in some way from start of study to current wave,-94.0,Insufficient information,Special code,661
2,Cumulative bullying-history indicator,Wave 2,W2bulev,DV: Whether YP ever bullied in some way from start of study to current wave,1.0,Yes,Observed response,5104
3,Cumulative bullying-history indicator,Wave 2,W2bulev,DV: Whether YP ever bullied in some way from start of study to current wave,2.0,No,Observed response,3748
4,Cumulative bullying-history indicator,Wave 2,W2bulev,DV: Whether YP ever bullied in some way from start of study to current wave,NaN,No source record,No source record,246


Complete bullying variable-role audit:


,Review role,Wave,Source type,Source file,Variable,Variable label,Timing status,Deferred from earlier domain
0,Cumulative bullying-history indicator,Wave 2,Young person,wave_two_lsype_young_person_2020,W2bulev,DV: Whether YP ever bullied in some way from start of study to current wave,Pre-transition source,True
1,Cumulative bullying-history indicator,Wave 2,Young person,wave_two_lsype_young_person_2020,W2pbulev,DV: Whether YP ever bullied in some way from start of study to current wave(pare,Pre-transition source,True
2,Cumulative bullying-history indicator,Wave 3,Young person,wave_three_lsype_young_person_2020,W3bulev,DV: Whether YP ever bullied in some way from start of study to current wave,Near-transition source,True
3,Cumulative bullying-history indicator,Wave 3,Young person,wave_three_lsype_young_person_2020,W3pbulev,DV: Whether YP ever bullied in some way from start of study to current wave (par,Near-transition source,True
4,Direct parental-report bullying status,Wave 1,Young person,wave_one_lsype_young_person_2020,W1pbulrc,DV: Whether YP bullied in some way in last 12 months (parental report),Pre-transition source,True


In [299]:
# 3: Bullying representation and pre-transition timing review

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
wave_3_timing_variables = ['W3intmnthMP', 'W3intyearMP']
wave_3_timing_inventory = verified_decision_register.loc[verified_decision_register['Variable'].isin(wave_3_timing_variables)].copy()
assert set(wave_3_timing_inventory['Variable']) == set(wave_3_timing_variables)
wave_3_timing_source_candidates = wave_3_timing_inventory.groupby(['Source file', 'Source type'],
    dropna=False)['Variable'].nunique().rename('Timing variables available').reset_index()
wave_3_complete_timing_sources = wave_3_timing_source_candidates.loc[wave_3_timing_source_candidates['Timing variables available'].eq(len(wave_3_timing_variables))].copy()
assert len(wave_3_complete_timing_sources) >= 1
wave_3_preferred_timing_sources = wave_3_complete_timing_sources.loc[wave_3_complete_timing_sources['Source type'].fillna('').astype('string').str.contains('Family background',
    case=False, regex=False, na=False) | wave_3_complete_timing_sources['Source file'].fillna('').astype('string').str.contains('family_background',
    case=False, regex=False, na=False)].copy()
assert len(wave_3_preferred_timing_sources) >= 1
wave_3_timing_source_file = wave_3_preferred_timing_sources.sort_values('Source file')['Source file'].iloc[0]
wave_3_selected_timing_inventory = wave_3_timing_inventory.loc[wave_3_timing_inventory['Source file'].eq(wave_3_timing_source_file)].copy()
assert set(wave_3_selected_timing_inventory['Variable']) == set(wave_3_timing_variables)
wave_3_timing_source_path = source_file_lookup[wave_3_timing_source_file]
print(f'Selected Wave 3 timing source: {wave_3_timing_source_file}')
wave_3_timing_data = pd.read_stata(wave_3_timing_source_path, columns=['NSID', *wave_3_timing_variables],
    convert_categoricals=False)
wave_3_timing_data['NSID'] = standardise_nsid(wave_3_timing_data['NSID'])
assert wave_3_timing_data['NSID'].is_unique
wave_3_timing_data = wave_3_timing_data.set_index('NSID').reindex(route_index)
for variable in wave_3_timing_variables:
    wave_3_timing_data[variable] = pd.to_numeric(wave_3_timing_data[variable], errors='coerce')
wave_3_pretransition_bullying_mask = wave_3_timing_data['W3intyearMP'].lt(2006) | wave_3_timing_data['W3intyearMP'].eq(2006) & wave_3_timing_data['W3intmnthMP'].between(1,
    8, inclusive='both')
wave_3_pretransition_bullying_mask = wave_3_pretransition_bullying_mask.fillna(False)
wave_3_timing_summary = pd.DataFrame([{'Stage 2 roster participants': len(route_index),
    'Interview date available': int(wave_3_timing_data[wave_3_timing_variables].notna().all(axis=1).sum()), 'Interview before September 2006': int(wave_3_pretransition_bullying_mask.sum()), 'Not eligible for Wave 3 use': int((~wave_3_pretransition_bullying_mask).sum())}])

def get_bullying_raw_variable(variable):
    """Return one aligned raw bullying variable."""
    source_file = bullying_principal_inventory.loc[bullying_principal_inventory['Variable'].eq(variable),
        'Source file'].iloc[0]
    return bullying_raw_tables[source_file][variable].copy()

def recode_yes_no(variable):
    """Recode 1 = yes and 2 = no to binary form."""
    raw_values = get_bullying_raw_variable(variable)
    return raw_values.map({1.0: 1.0, 2.0: 0.0}).astype('Float64')
young_person_bullying_by_wave = pd.DataFrame({'Wave 1': recode_yes_no('W1bulrc'), 'Wave 2': recode_yes_no('W2bulrc'),
    'Wave 3': recode_yes_no('W3bulrc').where(wave_3_pretransition_bullying_mask)}, index=route_index)
parent_bullying_by_wave = pd.DataFrame({'Wave 1': recode_yes_no('W1pbulrc'), 'Wave 2': recode_yes_no('W2pbulrc'),
    'Wave 3': recode_yes_no('W3pbulrc').where(wave_3_pretransition_bullying_mask)}, index=route_index)
young_person_cumulative_bullying_by_wave = pd.DataFrame({'Wave 2': recode_yes_no('W2bulev'),
    'Wave 3': recode_yes_no('W3bulev').where(wave_3_pretransition_bullying_mask)}, index=route_index)
parent_cumulative_bullying_by_wave = pd.DataFrame({'Wave 2': recode_yes_no('W2pbulev'),
    'Wave 3': recode_yes_no('W3pbulev').where(wave_3_pretransition_bullying_mask)}, index=route_index)
violence_threat_by_wave = pd.DataFrame({'Wave 1': recode_yes_no('W1thhitYP'), 'Wave 2': recode_yes_no('W2thhitYP'),
    'Wave 3': recode_yes_no('W3thhitYP').where(wave_3_pretransition_bullying_mask)}, index=route_index)

def create_latest_representation(wave_data, wave_order):
    """Use the latest eligible wave, followed by earlier fallbacks."""
    ordered_data = wave_data[wave_order]
    representation = ordered_data.bfill(axis=1).iloc[:, 0].astype('Float64')
    source = pd.Series('Unavailable', index=ordered_data.index, dtype='string')
    for wave in reversed(wave_order):
        source.loc[ordered_data[wave].notna()] = wave
    source.loc[representation.isna()] = 'Unavailable'
    return (representation, source)
latest_young_person_bullying, latest_young_person_bullying_source = create_latest_representation(young_person_bullying_by_wave,
    ['Wave 3', 'Wave 2', 'Wave 1'])
latest_parent_bullying, latest_parent_bullying_source = create_latest_representation(parent_bullying_by_wave,
    ['Wave 3', 'Wave 2', 'Wave 1'])
latest_young_person_cumulative_bullying, latest_young_person_cumulative_source = create_latest_representation(young_person_cumulative_bullying_by_wave,
    ['Wave 3', 'Wave 2'])
latest_parent_cumulative_bullying, latest_parent_cumulative_source = create_latest_representation(parent_cumulative_bullying_by_wave,
    ['Wave 3', 'Wave 2'])
latest_violence_threat, latest_violence_threat_source = create_latest_representation(violence_threat_by_wave,
    ['Wave 3', 'Wave 2', 'Wave 1'])
bullying_representation_data = pd.DataFrame({'Recent young-person bullying': latest_young_person_bullying,
    'Recent parental-report bullying': latest_parent_bullying, 'Cumulative young-person bullying': latest_young_person_cumulative_bullying, 'Cumulative parental-report bullying': latest_parent_cumulative_bullying, 'Recent violence threat': latest_violence_threat}, index=route_index)
bullying_representation_summary_rows = []
for representation in bullying_representation_data.columns:
    representation_values = bullying_representation_data[representation]
    bullying_representation_summary_rows.append({'Representation': representation,
        'Non-missing': int(representation_values.notna().sum()), 'Missing': int(representation_values.isna().sum()), 'Missing percentage': round(representation_values.isna().mean() * 100,
        2), 'No': int(representation_values.eq(0).sum()), 'Yes': int(representation_values.eq(1).sum()), 'Yes percentage among observed': round(representation_values.eq(1).sum() / representation_values.notna().sum() * 100,
        2)})
bullying_representation_summary = pd.DataFrame(bullying_representation_summary_rows)
bullying_source_summary_rows = []
bullying_source_lookup = {'Recent young-person bullying': latest_young_person_bullying_source,
    'Recent parental-report bullying': latest_parent_bullying_source, 'Cumulative young-person bullying': latest_young_person_cumulative_source, 'Cumulative parental-report bullying': latest_parent_cumulative_source, 'Recent violence threat': latest_violence_threat_source}
for representation, source_series in bullying_source_lookup.items():
    source_counts = source_series.value_counts(dropna=False)
    for source, participants in source_counts.items():
        bullying_source_summary_rows.append({'Representation': representation, 'Source': source,
            'Participants': int(participants), 'Percentage': round(participants / len(route_index) * 100, 2)})
bullying_source_summary = pd.DataFrame(bullying_source_summary_rows).sort_values(['Representation',
    'Source']).reset_index(drop=True)

def binary_cramers_v(first, second):
    """Calculate Cramer's V for two binary representations."""
    complete_data = pd.DataFrame({'First': first, 'Second': second}).dropna()
    cross_tab = pd.crosstab(complete_data['First'], complete_data['Second'])
    if cross_tab.shape[0] < 2 or cross_tab.shape[1] < 2:
        return np.nan
    chi_square = chi2_contingency(cross_tab, correction=False)[0]
    sample_size = cross_tab.to_numpy().sum()
    return float(np.sqrt(chi_square / sample_size))
bullying_comparison_pairs = [('Recent young-person bullying', 'Recent parental-report bullying'),
    ('Recent young-person bullying', 'Cumulative young-person bullying'), ('Recent young-person bullying',
    'Recent violence threat'), ('Recent parental-report bullying',
    'Cumulative parental-report bullying'), ('Cumulative young-person bullying',
    'Cumulative parental-report bullying')]
bullying_representation_comparison_rows = []
for first_name, second_name in bullying_comparison_pairs:
    pair_data = bullying_representation_data[[first_name, second_name]].dropna()
    bullying_representation_comparison_rows.append({'First representation': first_name,
        'Second representation': second_name, 'Complete comparisons': len(pair_data), 'Exact agreement percentage': round(pair_data[first_name].eq(pair_data[second_name]).mean() * 100,
        2), "Cramer's V": round(binary_cramers_v(pair_data[first_name], pair_data[second_name]), 3)})
bullying_representation_comparison = pd.DataFrame(bullying_representation_comparison_rows)
young_person_bullying_wave_pairs = [('Wave 1', 'Wave 2'), ('Wave 2', 'Wave 3'), ('Wave 1', 'Wave 3')]
young_person_bullying_wave_comparison_rows = []
for first_wave, second_wave in young_person_bullying_wave_pairs:
    pair_data = young_person_bullying_by_wave[[first_wave, second_wave]].dropna()
    young_person_bullying_wave_comparison_rows.append({'First wave': first_wave, 'Second wave': second_wave,
        'Complete comparisons': len(pair_data), 'Exact agreement percentage': round(pair_data[first_wave].eq(pair_data[second_wave]).mean() * 100,
        2), "Cramer's V": round(binary_cramers_v(pair_data[first_wave], pair_data[second_wave]), 3)})
young_person_bullying_wave_comparison = pd.DataFrame(young_person_bullying_wave_comparison_rows)
print('Wave 3 timing restriction:')
display_limited(wave_3_timing_summary)
print('Provisional bullying representations:')
display_limited(bullying_representation_summary)
print('Representation source waves:')
with pd.option_context('display.max_rows', None):
    display_limited(bullying_source_summary)
print('Representation comparisons:')
display_limited(bullying_representation_comparison)
print('Direct young-person bullying status across waves:')
display_limited(young_person_bullying_wave_comparison)

Selected Wave 3 timing source: wave_three_lsype_family_background_2020
Wave 3 timing restriction:


,Stage 2 roster participants,Interview date available,Interview before September 2006,Not eligible for Wave 3 use
0,9767,9509,9495,272


Provisional bullying representations:


,Representation,Non-missing,Missing,Missing percentage,No,Yes,Yes percentage among observed
0,Recent young-person bullying,9500,267,2.73,6904,2596,27.33
1,Recent parental-report bullying,9096,671,6.87,7154,1942,21.35
2,Cumulative young-person bullying,8950,817,8.36,3446,5504,61.50
3,Cumulative parental-report bullying,7934,1833,18.77,3748,4186,52.76
4,Recent violence threat,9509,258,2.64,8302,1207,12.69


Representation source waves:


,Representation,Source,Participants,Percentage
0,Cumulative parental-report bullying,Unavailable,1833,18.77
1,Cumulative parental-report bullying,Wave 2,270,2.76
2,Cumulative parental-report bullying,Wave 3,7664,78.47
3,Cumulative young-person bullying,Unavailable,817,8.36
4,Cumulative young-person bullying,Wave 2,126,1.29


Representation comparisons:


,First representation,Second representation,Complete comparisons,Exact agreement percentage,Cramer's V
0,Recent young-person bullying,Recent parental-report bullying,9073,74.66,0.320
1,Recent young-person bullying,Cumulative young-person bullying,8950,67.51,0.506
2,Recent young-person bullying,Recent violence threat,9499,85.39,0.622
3,Recent parental-report bullying,Cumulative parental-report bullying,7934,71.70,0.539
4,Cumulative young-person bullying,Cumulative parental-report bullying,7522,69.66,0.390


Direct young-person bullying status across waves:


,First wave,Second wave,Complete comparisons,Exact agreement percentage,Cramer's V
0,Wave 1,Wave 2,8427,71.80,0.426
1,Wave 2,Wave 3,8607,74.46,0.437
2,Wave 1,Wave 3,8550,66.61,0.322


In [300]:
# 4: Bullying predictor decision and variable-level check

import pandas as pd
bullying_experience_pretransition_candidate = latest_young_person_bullying.copy().rename('bullying_experience_pretransition')
assert int(bullying_experience_pretransition_candidate.notna().sum()) == 9500
assert set(bullying_experience_pretransition_candidate.dropna().unique()) == {0.0, 1.0}
bullying_candidate_summary = pd.DataFrame([{'Predictor': 'bullying_experience_pretransition',
    'Construct': 'Bullying experience during the previous 12 months', 'Representation': 'Latest eligible direct young-person status: Wave 3 interview before September 2006, then Wave 2 and Wave 1 fallbacks', 'No': int(bullying_experience_pretransition_candidate.eq(0).sum()), 'Yes': int(bullying_experience_pretransition_candidate.eq(1).sum()), 'Non-missing': int(bullying_experience_pretransition_candidate.notna().sum()), 'Missing': int(bullying_experience_pretransition_candidate.isna().sum()), 'Missing percentage': round(bullying_experience_pretransition_candidate.isna().mean() * 100,
    2), 'Yes percentage among observed': round(bullying_experience_pretransition_candidate.eq(1).sum() / bullying_experience_pretransition_candidate.notna().sum() * 100,
    2)}])
bullying_candidate_source_summary = latest_young_person_bullying_source.value_counts().rename('Participants').rename_axis('Source').reset_index()
bullying_candidate_source_summary['Percentage'] = (bullying_candidate_source_summary['Participants'] / len(route_index) * 100).round(2)
bullying_variable_decisions = domain_8_bullying_inventory.copy()
bullying_variable_decisions['Bullying-track decision'] = pd.Series(pd.NA, index=bullying_variable_decisions.index,
    dtype='string')
bullying_variable_decisions['Bullying-track representation'] = pd.Series(pd.NA,
    index=bullying_variable_decisions.index, dtype='string')
bullying_variable_decisions['Bullying-track reason'] = pd.Series(pd.NA, index=bullying_variable_decisions.index,
    dtype='string')

def assign_bullying_decision(variables, decision, representation, reason):
    """Assign one decision to specified bullying variables."""
    decision_mask = bullying_variable_decisions['Variable'].isin(variables)
    assert int(decision_mask.sum()) == len(variables)
    assert bullying_variable_decisions.loc[decision_mask, 'Bullying-track decision'].isna().all()
    bullying_variable_decisions.loc[decision_mask, 'Bullying-track decision'] = decision
    bullying_variable_decisions.loc[decision_mask, 'Bullying-track representation'] = representation
    bullying_variable_decisions.loc[decision_mask, 'Bullying-track reason'] = reason
assign_bullying_decision(variables=bullying_young_person_status_variables, decision='Construction input',
    representation='bullying_experience_pretransition', reason='Repeated direct young-person bullying-status measure. Wave 3 is restricted to interviews before September 2006, with Wave 2 and Wave 1 used as fallbacks')
assign_bullying_decision(variables=bullying_parent_status_variables, decision='Review support',
    representation='Alternative parental-report bullying status', reason='Alternative report of the same broad experience, with lower coverage and only moderate agreement with the direct young-person measure')
assign_bullying_decision(variables=bullying_cumulative_variables, decision='Review support',
    representation='Alternative cumulative bullying history', reason='Useful for comparing recent status with prior history, but not retained because it measures whether bullying occurred at any point since the beginning of the study')
assign_bullying_decision(variables=violence_threat_status_variables, decision='Review support',
    representation='Narrow violence-threat indicator', reason='Directly measured and well covered, but represents one specific form of victimisation rather than overall bullying')
assign_bullying_decision(variables=violence_threat_frequency_variables, decision='Review support',
    representation='Routed violence-threat frequency', reason='Conditional frequency detail used to verify routing and severity; structural not-applicable values must not be treated as missing')
assign_bullying_decision(variables=bullying_parent_detail_variables, decision='Review support',
    representation='Parental-report bullying-type detail', reason='Multiple-response bullying-type items underlying the broader parental-report status; not retained separately to avoid a large set of overlapping routed indicators')
assign_bullying_decision(variables=bullying_context_reason_variables, decision='Exclude', representation='None',
    reason="School-choice, school-leaving or school-movement reason rather than a direct measure of the young person's bullying experience")
bullying_unresolved_variables = bullying_variable_decisions.loc[bullying_variable_decisions['Bullying-track decision'].isna()].copy()
assert len(bullying_unresolved_variables) == 0
assert len(bullying_variable_decisions) == 53
expected_bullying_decision_counts = {'Construction input': 3, 'Review support': 45, 'Exclude': 5}
actual_bullying_decision_counts = bullying_variable_decisions['Bullying-track decision'].value_counts().to_dict()
assert actual_bullying_decision_counts == expected_bullying_decision_counts
bullying_decision_summary = bullying_variable_decisions.groupby(['Bullying-track decision', 'Review role'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Bullying-track decision',
    'Review role']).reset_index(drop=True)
bullying_decision_display = bullying_variable_decisions[['Review role', 'Wave', 'Source type', 'Source file',
    'Variable', 'Variable label', 'Timing status', 'Bullying-track decision', 'Bullying-track representation', 'Bullying-track reason']].sort_values(['Bullying-track decision',
    'Review role', 'Wave', 'Variable']).reset_index(drop=True)
bullying_representation_decisions = pd.DataFrame([{'Representation': 'Latest direct young-person bullying status',
    'Decision': 'Retain as predictor candidate', 'Reason': 'Direct report, broad bullying construct, high coverage and clear pre-transition timing'}, {'Representation': 'Latest parental-report bullying status',
    'Decision': 'Review support only', 'Reason': 'Lower coverage and moderate agreement with the young-person report'}, {'Representation': 'Cumulative young-person bullying history',
    'Decision': 'Review support only', 'Reason': 'Changes the reference period from recent experience to any experience since the start of the study'}, {'Representation': 'Cumulative parental-report bullying history',
    'Decision': 'Review support only', 'Reason': 'Lower coverage, different respondent and cumulative reference period'}, {'Representation': 'Latest violence-threat status',
    'Decision': 'Review support only', 'Reason': 'Represents a narrower and more severe subtype of victimisation'}, {'Representation': 'Violence-threat frequency',
    'Decision': 'Review support only', 'Reason': 'Routed detail available only for participants reporting the relevant experience'}])
print('Selected bullying predictor:')
display_limited(bullying_candidate_summary)
print('Selected predictor source waves:')
display_limited(bullying_candidate_source_summary)
print('Bullying variable decision summary:')
display_limited(bullying_decision_summary)
print('Bullying representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(bullying_representation_decisions)
print(f'Unresolved bullying variables: {len(bullying_unresolved_variables)}')
print('Complete bullying variable decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(bullying_decision_display)

Selected bullying predictor:


,Predictor,Construct,Representation,No,Yes,Non-missing,Missing,Missing percentage,Yes percentage among observed
0,bullying_experience_pretransition,Bullying experience during the previous 12 months,Latest eligible direct young-person status: Wa...,6904,2596,9500,267,2.73,27.33


Selected predictor source waves:


,Source,Participants,Percentage
0,Wave 3,9103,93.2
1,Wave 2,333,3.41
2,Unavailable,267,2.73
3,Wave 1,64,0.66


Bullying variable decision summary:


,Bullying-track decision,Review role,Variables
0,Construction input,Direct young-person bullying status,3
1,Exclude,"School-choice, leaving or movement reason",5
2,Review support,Cumulative bullying-history indicator,4
3,Review support,Direct parental-report bullying status,3
4,Review support,Direct violence-threat status,3


Bullying representation decisions:


,Representation,Decision,Reason
0,Latest direct young-person bullying status,Retain as predictor candidate,"Direct report, broad bullying construct, high coverage and clear pre-transition timing"
1,Latest parental-report bullying status,Review support only,Lower coverage and moderate agreement with the young-person report
2,Cumulative young-person bullying history,Review support only,Changes the reference period from recent experience to any experience since the start of the study
3,Cumulative parental-report bullying history,Review support only,"Lower coverage, different respondent and cumulative reference period"
4,Latest violence-threat status,Review support only,Represents a narrower and more severe subtype of victimisation


Unresolved bullying variables: 0
Complete bullying variable decisions:


,Review role,Wave,Source type,Source file,Variable,Variable label,Timing status,Bullying-track decision,Bullying-track representation,Bullying-track reason
0,Direct young-person bullying status,Wave 1,Young person,wave_one_lsype_young_person_2020,W1bulrc,DV: Whether YP bullied in any way in last 12 months,Pre-transition source,Construction input,bullying_experience_pretransition,"Repeated direct young-person bullying-status measure. Wave 3 is restricted to interviews before September 2006, with Wave 2 and Wave 1 used as fallbacks"
1,Direct young-person bullying status,Wave 2,Young person,wave_two_lsype_young_person_2020,W2bulrc,DV: Whether YP bullied in any way in last 12 months,Pre-transition source,Construction input,bullying_experience_pretransition,"Repeated direct young-person bullying-status measure. Wave 3 is restricted to interviews before September 2006, with Wave 2 and Wave 1 used as fallbacks"
2,Direct young-person bullying status,Wave 3,Young person,wave_three_lsype_young_person_2020,W3bulrc,DV: Whether YP bullied in any way in last 12 months,Near-transition source,Construction input,bullying_experience_pretransition,"Repeated direct young-person bullying-status measure. Wave 3 is restricted to interviews before September 2006, with Wave 2 and Wave 1 used as fallbacks"
3,"School-choice, leaving or movement reason",Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1StascHS0e,HR: First choice because there is relatively little bullying at the school,Pre-transition source,Exclude,None,"School-choice, school-leaving or school-movement reason rather than a direct measure of the young person's bullying experience"
4,"School-choice, leaving or movement reason",Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1WhyBeHS0c,HR: Wanted YP to go to current school because there is relatively little bullyin,Pre-transition source,Exclude,None,"School-choice, school-leaving or school-movement reason rather than a direct measure of the young person's bullying experience"


In [301]:
# 5: Caring-responsibility variable-role and coding review

import pandas as pd
domain_8_caring_inventory = domain_8_pending_candidates.loc[domain_8_pending_candidates['Matched construct'].str.contains('Caring responsibilities',
    case=False, regex=False, na=False)].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(domain_8_caring_inventory) == 40
young_person_caring_status_variables = ['W1careYP', 'W2careYP', 'W3careYP']
young_person_caring_hours_variables = ['W1carehrsYP', 'W2CarehrsYP', 'W3carehrsYP']
caring_school_absence_status_variables = ['W1carehr1YP', 'W2carehr1YP', 'W3carehr1YP']
caring_school_absence_frequency_variables = ['W1carehr2YP', 'W2carehr2YP', 'W3carehr2YP']
parent_household_caring_variables = ['W1carehhMP', 'W2CareHHMP']
parent_non_household_caring_variables = ['W1carenhhMP', 'W2CareNHHMP']
young_person_caring_recipient_variables = [variable for variable in domain_8_caring_inventory['Variable'] if variable.lower().startswith('w1cawhoyp0') or variable.lower().startswith('w2cawhoyp0') or variable.lower().startswith('w3cawhoyp0')]
assert len(young_person_caring_recipient_variables) == 24
caring_role_lookup = {}
for variable in young_person_caring_status_variables:
    caring_role_lookup[variable] = 'Direct young-person caring status'
for variable in young_person_caring_hours_variables:
    caring_role_lookup[variable] = 'Routed weekly caring hours'
for variable in caring_school_absence_status_variables:
    caring_role_lookup[variable] = 'Caring-related school-absence status'
for variable in caring_school_absence_frequency_variables:
    caring_role_lookup[variable] = 'Routed caring-related absence frequency'
for variable in parent_household_caring_variables:
    caring_role_lookup[variable] = 'Main-parent household caring status'
for variable in parent_non_household_caring_variables:
    caring_role_lookup[variable] = 'Main-parent non-household caring status'
for variable in young_person_caring_recipient_variables:
    caring_role_lookup[variable] = 'Young-person caring-recipient detail'
assert set(caring_role_lookup) == set(domain_8_caring_inventory['Variable'])
domain_8_caring_inventory['Review role'] = domain_8_caring_inventory['Variable'].map(caring_role_lookup)
domain_8_caring_role_summary = domain_8_caring_inventory.groupby(['Review role', 'Wave'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Review role', 'Wave']).reset_index(drop=True)
caring_principal_review_variables = young_person_caring_status_variables + young_person_caring_hours_variables + caring_school_absence_status_variables + caring_school_absence_frequency_variables + parent_household_caring_variables + parent_non_household_caring_variables
assert len(caring_principal_review_variables) == 16
assert len(set(caring_principal_review_variables)) == 16
caring_principal_inventory = domain_8_caring_inventory.loc[domain_8_caring_inventory['Variable'].isin(caring_principal_review_variables)].copy()
assert len(caring_principal_inventory) == 16
caring_raw_tables = {}
caring_labelled_tables = {}
caring_quality_rows = []
caring_code_rows = []
for source_file, source_inventory in caring_principal_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    raw_data['NSID'] = standardise_nsid(raw_data['NSID'])
    labelled_data['NSID'] = standardise_nsid(labelled_data['NSID'])
    assert raw_data['NSID'].is_unique
    assert labelled_data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(route_index)
    labelled_data = labelled_data.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    caring_raw_tables[source_file] = raw_data
    caring_labelled_tables[source_file] = labelled_data
    for variable in source_variables:
        variable_inventory = source_inventory.loc[source_inventory['Variable'].eq(variable)].iloc[0]
        raw_values = raw_data[variable]
        labelled_values = labelled_data[variable].astype('string')
        observed_mask = raw_values.ge(0).fillna(False)
        special_code_mask = raw_values.lt(0).fillna(False)
        observed_values = raw_values.loc[observed_mask]
        caring_quality_rows.append({'Review role': variable_inventory['Review role'],
            'Wave': variable_inventory['Wave'], 'Variable': variable, 'Variable label': variable_inventory['Variable label'], 'Observed responses': int(observed_mask.sum()), 'Special-code responses': int(special_code_mask.sum()), 'No source record': int(raw_values.isna().sum()), 'Observed percentage': round(observed_mask.mean() * 100,
            2), 'Distinct observed codes': int(observed_values.nunique()), 'Minimum observed code': observed_values.min() if len(observed_values) > 0 else pd.NA, 'Maximum observed code': observed_values.max() if len(observed_values) > 0 else pd.NA, 'Not-applicable responses': int(raw_values.eq(-91).sum())})
        value_counts = raw_values.value_counts(dropna=False)
        for raw_code, participants in value_counts.items():
            if pd.isna(raw_code):
                value_label = 'No source record'
                response_type = 'No source record'
                sort_value = 999999
            else:
                matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(raw_code)
                response_type = 'Observed response' if raw_code >= 0 else 'Special code'
                sort_value = float(raw_code)
            caring_code_rows.append({'Review role': variable_inventory['Review role'],
                'Wave': variable_inventory['Wave'], 'Variable': variable, 'Variable label': variable_inventory['Variable label'], 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
caring_principal_quality = pd.DataFrame(caring_quality_rows).sort_values(['Review role', 'Wave',
    'Variable']).reset_index(drop=True)
caring_principal_code_distribution = pd.DataFrame(caring_code_rows).sort_values(['Review role', 'Wave', 'Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
domain_8_caring_role_display = domain_8_caring_inventory[['Review role', 'Wave', 'Source type', 'Source file',
    'Variable', 'Variable label', 'Timing status', 'Deferred from earlier domain']].sort_values(['Review role', 'Wave',
    'Variable']).reset_index(drop=True)
print(f'Caring-responsibility variables reviewed: {len(domain_8_caring_inventory)}')
print(f'Principal measures receiving coding review: {len(caring_principal_inventory)}')
print('Caring variable-role summary:')
display_limited(domain_8_caring_role_summary)
print('Principal-measure quality:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(caring_principal_quality)
print('Principal-measure response codes:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(caring_principal_code_distribution)
print('Complete caring variable-role audit:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_8_caring_role_display)

Caring-responsibility variables reviewed: 40
Principal measures receiving coding review: 16
Caring variable-role summary:


,Review role,Wave,Variables
0,Caring-related school-absence status,Wave 1,1
1,Caring-related school-absence status,Wave 2,1
2,Caring-related school-absence status,Wave 3,1
3,Direct young-person caring status,Wave 1,1
4,Direct young-person caring status,Wave 2,1


Principal-measure quality:


,Review role,Wave,Variable,Variable label,Observed responses,Special-code responses,No source record,Observed percentage,Distinct observed codes,Minimum observed code,Maximum observed code,Not-applicable responses
0,Caring-related school-absence status,Wave 1,W1carehr1YP,YP: Whether ever miss school because of caring responsibilities,477,9047,243,4.88,2,1.0,2.0,8957
1,Caring-related school-absence status,Wave 2,W2carehr1YP,YP: Whether ever miss school because of caring responsibilities,511,9010,246,5.23,2,1.0,2.0,8932
2,Caring-related school-absence status,Wave 3,W3carehr1YP,YP: Whether YP ever misses school because of caring responsibilities,557,8952,258,5.70,2,1.0,2.0,8898
3,Direct young-person caring status,Wave 1,W1careYP,YP: Whether YP has any caring responsibilities within household,9424,100,243,96.49,2,1.0,2.0,0
4,Direct young-person caring status,Wave 2,W2careYP,YP: Whether YP has any caring responsibilities within household,9418,103,246,96.43,2,1.0,2.0,0


Principal-measure response codes:


,Review role,Wave,Variable,Variable label,Raw code,Value label,Response type,Participants
0,Caring-related school-absence status,Wave 1,W1carehr1YP,YP: Whether ever miss school because of caring responsibilities,-99.0,YP not interviewed,Special code,89
1,Caring-related school-absence status,Wave 1,W1carehr1YP,YP: Whether ever miss school because of caring responsibilities,-91.0,Not applicable,Special code,8957
2,Caring-related school-absence status,Wave 1,W1carehr1YP,YP: Whether ever miss school because of caring responsibilities,-1.0,Don't know,Special code,1
3,Caring-related school-absence status,Wave 1,W1carehr1YP,YP: Whether ever miss school because of caring responsibilities,1.0,Yes,Observed response,25
4,Caring-related school-absence status,Wave 1,W1carehr1YP,YP: Whether ever miss school because of caring responsibilities,2.0,No,Observed response,452


Complete caring variable-role audit:


,Review role,Wave,Source type,Source file,Variable,Variable label,Timing status,Deferred from earlier domain
0,Caring-related school-absence status,Wave 1,Young person,wave_one_lsype_young_person_2020,W1carehr1YP,YP: Whether ever miss school because of caring responsibilities,Pre-transition source,True
1,Caring-related school-absence status,Wave 2,Young person,wave_two_lsype_young_person_2020,W2carehr1YP,YP: Whether ever miss school because of caring responsibilities,Pre-transition source,True
2,Caring-related school-absence status,Wave 3,Young person,wave_three_lsype_young_person_2020,W3carehr1YP,YP: Whether YP ever misses school because of caring responsibilities,Near-transition source,True
3,Direct young-person caring status,Wave 1,Young person,wave_one_lsype_young_person_2020,W1careYP,YP: Whether YP has any caring responsibilities within household,Pre-transition source,False
4,Direct young-person caring status,Wave 2,Young person,wave_two_lsype_young_person_2020,W2careYP,YP: Whether YP has any caring responsibilities within household,Pre-transition source,False


In [302]:
# 6: Caring-status representation and routing review

import pandas as pd
wave_3_pretransition_caring_mask = wave_3_pretransition_bullying_mask.copy()
assert int(wave_3_pretransition_caring_mask.sum()) == 9495

def get_caring_raw_variable(variable):
    """Return one aligned raw caring variable."""
    source_file = caring_principal_inventory.loc[caring_principal_inventory['Variable'].eq(variable),
        'Source file'].iloc[0]
    return caring_raw_tables[source_file][variable].copy()

def recode_caring_yes_no(variable):
    """Recode 1 = yes and 2 = no to binary form."""
    raw_values = get_caring_raw_variable(variable)
    return raw_values.map({1.0: 1.0, 2.0: 0.0}).astype('Float64')
young_person_caring_by_wave = pd.DataFrame({'Wave 1': recode_caring_yes_no('W1careYP'),
    'Wave 2': recode_caring_yes_no('W2careYP'), 'Wave 3': recode_caring_yes_no('W3careYP').where(wave_3_pretransition_caring_mask)}, index=route_index)
caring_absence_raw_by_wave = pd.DataFrame({'Wave 1': get_caring_raw_variable('W1carehr1YP'),
    'Wave 2': get_caring_raw_variable('W2carehr1YP'), 'Wave 3': get_caring_raw_variable('W3carehr1YP').where(wave_3_pretransition_caring_mask)}, index=route_index)
caring_routing_rows = []
for wave in ['Wave 1', 'Wave 2', 'Wave 3']:
    caring_status = young_person_caring_by_wave[wave]
    absence_raw = caring_absence_raw_by_wave[wave]
    routing_categories = pd.DataFrame({'Caring status': caring_status.map({1.0: 'Caring responsibility',
        0.0: 'No caring responsibility'}).fillna('Caring status unavailable'), 'Absence response': absence_raw.map({1.0: 'Missed school',
        2.0: 'Did not miss school', -91.0: 'Structurally not applicable'}).fillna('Other unavailable response')})
    routing_counts = routing_categories.value_counts().rename('Participants').reset_index()
    routing_counts['Wave'] = wave
    caring_routing_rows.append(routing_counts)
caring_absence_routing_summary = pd.concat(caring_routing_rows, ignore_index=True)[['Wave', 'Caring status',
    'Absence response', 'Participants']].sort_values(['Wave', 'Caring status',
    'Absence response']).reset_index(drop=True)
caring_absence_by_wave = pd.DataFrame(index=route_index)
for wave in ['Wave 1', 'Wave 2', 'Wave 3']:
    caring_status = young_person_caring_by_wave[wave]
    absence_raw = caring_absence_raw_by_wave[wave]
    derived_absence = pd.Series(pd.NA, index=route_index, dtype='Float64')
    derived_absence.loc[absence_raw.eq(1)] = 1.0
    derived_absence.loc[absence_raw.eq(2)] = 0.0
    derived_absence.loc[caring_status.eq(0) & absence_raw.eq(-91)] = 0.0
    caring_absence_by_wave[wave] = derived_absence
caring_routing_contradictions = []
for wave in ['Wave 1', 'Wave 2', 'Wave 3']:
    caring_status = young_person_caring_by_wave[wave]
    absence_raw = caring_absence_raw_by_wave[wave]
    caring_routing_contradictions.append({'Wave': wave,
        'No caring responsibility but absence reported': int((caring_status.eq(0) & absence_raw.eq(1)).sum()), 'Caring responsibility but structurally not applicable': int((caring_status.eq(1) & absence_raw.eq(-91)).sum())})
caring_routing_contradiction_summary = pd.DataFrame(caring_routing_contradictions)
latest_young_person_caring, latest_young_person_caring_source = create_latest_representation(young_person_caring_by_wave,
    ['Wave 3', 'Wave 2', 'Wave 1'])
latest_caring_related_absence, latest_caring_related_absence_source = create_latest_representation(caring_absence_by_wave,
    ['Wave 3', 'Wave 2', 'Wave 1'])
caring_representation_data = pd.DataFrame({'Young-person caring status': latest_young_person_caring,
    'Caring-related school absence': latest_caring_related_absence}, index=route_index)
caring_representation_summary_rows = []
for representation in caring_representation_data.columns:
    values = caring_representation_data[representation]
    caring_representation_summary_rows.append({'Representation': representation,
        'Non-missing': int(values.notna().sum()), 'Missing': int(values.isna().sum()), 'Missing percentage': round(values.isna().mean() * 100,
        2), 'No': int(values.eq(0).sum()), 'Yes': int(values.eq(1).sum()), 'Yes percentage among observed': round(values.eq(1).sum() / values.notna().sum() * 100,
        2)})
caring_representation_summary = pd.DataFrame(caring_representation_summary_rows)
caring_source_summary_rows = []
caring_source_lookup = {'Young-person caring status': latest_young_person_caring_source,
    'Caring-related school absence': latest_caring_related_absence_source}
for representation, source_series in caring_source_lookup.items():
    for source, participants in source_series.value_counts(dropna=False).items():
        caring_source_summary_rows.append({'Representation': representation, 'Source': source,
            'Participants': int(participants), 'Percentage': round(participants / len(route_index) * 100, 2)})
caring_source_summary = pd.DataFrame(caring_source_summary_rows).sort_values(['Representation',
    'Source']).reset_index(drop=True)
caring_wave_pairs = [('Wave 1', 'Wave 2'), ('Wave 2', 'Wave 3'), ('Wave 1', 'Wave 3')]
caring_wave_comparison_rows = []
for first_wave, second_wave in caring_wave_pairs:
    pair_data = young_person_caring_by_wave[[first_wave, second_wave]].dropna()
    caring_wave_comparison_rows.append({'First wave': first_wave, 'Second wave': second_wave,
        'Complete comparisons': len(pair_data), 'Exact agreement percentage': round(pair_data[first_wave].eq(pair_data[second_wave]).mean() * 100,
        2), "Cramer's V": round(binary_cramers_v(pair_data[first_wave], pair_data[second_wave]), 3)})
caring_wave_comparison = pd.DataFrame(caring_wave_comparison_rows)
caring_status_absence_complete = caring_representation_data.dropna()
caring_status_absence_comparison = pd.DataFrame([{'Complete comparisons': len(caring_status_absence_complete),
    'Caring responsibility reported': int(caring_status_absence_complete['Young-person caring status'].eq(1).sum()), 'Caring-related absence reported': int(caring_status_absence_complete['Caring-related school absence'].eq(1).sum()), "Cramer's V": round(binary_cramers_v(caring_status_absence_complete['Young-person caring status'],
    caring_status_absence_complete['Caring-related school absence']), 3)}])
print('Caring and school-absence routing:')
with pd.option_context('display.max_rows', None):
    display_limited(caring_absence_routing_summary)
print('Routing contradictions:')
display_limited(caring_routing_contradiction_summary)
print('Provisional caring representations:')
display_limited(caring_representation_summary)
print('Representation source waves:')
display_limited(caring_source_summary)
print('Direct young-person caring status across waves:')
display_limited(caring_wave_comparison)
print('Caring status and caring-related absence:')
display_limited(caring_status_absence_comparison)

Caring and school-absence routing:


,Wave,Caring status,Absence response,Participants
0,Wave 1,Caring responsibility,Did not miss school,452
1,Wave 1,Caring responsibility,Missed school,25
2,Wave 1,Caring responsibility,Other unavailable response,1
3,Wave 1,Caring status unavailable,Other unavailable response,332
4,Wave 1,Caring status unavailable,Structurally not applicable,11


Routing contradictions:


,Wave,No caring responsibility but absence reported,Caring responsibility but structurally not applicable
0,Wave 1,0,0
1,Wave 2,0,0
2,Wave 3,0,0


Provisional caring representations:


...

,Representation,Source,Participants,Percentage
0,Caring-related school absence,Unavailable,246,2.52
1,Caring-related school absence,Wave 1,12,0.12
2,Caring-related school absence,Wave 2,77,0.79
3,Caring-related school absence,Wave 3,9432,96.57
4,Young-person caring status,Unavailable,246,2.52


Direct young-person caring status across waves:


,First wave,Second wave,Complete comparisons,Exact agreement percentage,Cramer's V
0,Wave 1,Wave 2,9326,92.99,0.299
1,Wave 2,Wave 3,9341,93.32,0.377
2,Wave 1,Wave 3,9337,92.46,0.275


Caring status and caring-related absence:


,Complete comparisons,Caring responsibility reported,Caring-related absence reported,Cramer's V
0,9521,559,18,0.174


In [303]:
# 7: Caring-responsibility predictor decision and variable-level check

import pandas as pd
young_person_caring_responsibility_candidate = latest_young_person_caring.copy().rename('young_person_caring_responsibility_pretransition')
assert int(young_person_caring_responsibility_candidate.notna().sum()) == 9521
assert set(young_person_caring_responsibility_candidate.dropna().unique()) == {0.0, 1.0}
caring_candidate_summary = pd.DataFrame([{'Predictor': 'young_person_caring_responsibility_pretransition',
    'Construct': 'Young-person caring responsibility within the household', 'Representation': 'Latest eligible direct young-person status: Wave 3 interview before September 2006, then Wave 2 and Wave 1 fallbacks', 'No': int(young_person_caring_responsibility_candidate.eq(0).sum()), 'Yes': int(young_person_caring_responsibility_candidate.eq(1).sum()), 'Non-missing': int(young_person_caring_responsibility_candidate.notna().sum()), 'Missing': int(young_person_caring_responsibility_candidate.isna().sum()), 'Missing percentage': round(young_person_caring_responsibility_candidate.isna().mean() * 100,
    2), 'Yes percentage among observed': round(young_person_caring_responsibility_candidate.eq(1).sum() / young_person_caring_responsibility_candidate.notna().sum() * 100,
    2)}])
caring_candidate_source_summary = latest_young_person_caring_source.value_counts().rename('Participants').rename_axis('Source').reset_index()
caring_candidate_source_summary['Percentage'] = (caring_candidate_source_summary['Participants'] / len(route_index) * 100).round(2)
caring_variable_decisions = domain_8_caring_inventory.copy()
caring_variable_decisions['Caring-track decision'] = pd.Series(pd.NA, index=caring_variable_decisions.index,
    dtype='string')
caring_variable_decisions['Caring-track representation'] = pd.Series(pd.NA, index=caring_variable_decisions.index,
    dtype='string')
caring_variable_decisions['Caring-track reason'] = pd.Series(pd.NA, index=caring_variable_decisions.index,
    dtype='string')

def assign_caring_decision(variables, decision, representation, reason):
    """Assign one decision to specified caring variables."""
    decision_mask = caring_variable_decisions['Variable'].isin(variables)
    assert int(decision_mask.sum()) == len(variables)
    assert caring_variable_decisions.loc[decision_mask, 'Caring-track decision'].isna().all()
    caring_variable_decisions.loc[decision_mask, 'Caring-track decision'] = decision
    caring_variable_decisions.loc[decision_mask, 'Caring-track representation'] = representation
    caring_variable_decisions.loc[decision_mask, 'Caring-track reason'] = reason
assign_caring_decision(variables=young_person_caring_status_variables, decision='Construction input',
    representation='young_person_caring_responsibility_pretransition', reason='Repeated direct young-person caring-status measure. Wave 3 is restricted to interviews before September 2006, with Wave 2 and Wave 1 used as fallbacks')
assign_caring_decision(variables=young_person_caring_hours_variables, decision='Review support',
    representation='Routed caring-intensity information', reason='Conditional on reporting caring responsibility and coded differently across waves; retained for coding and intensity review but not as a separate harmonised predictor')
assign_caring_decision(variables=caring_school_absence_status_variables, decision='Review support',
    representation='Caring-related school-absence status', reason='Valid routed measure but only 18 participants report caring-related absence in the latest eligible representation, making it too sparse for a separate predictor')
assign_caring_decision(variables=caring_school_absence_frequency_variables, decision='Review support',
    representation='Routed caring-related absence frequency', reason='Conditional frequency detail available only for participants reporting caring-related school absence; structural not-applicable responses are not missing values')
assign_caring_decision(variables=young_person_caring_recipient_variables, decision='Review support',
    representation='Young-person caring-recipient detail', reason='Multiple-response detail conditional on caring status; not retained separately to avoid a sparse set of overlapping recipient indicators')
assign_caring_decision(variables=parent_household_caring_variables + parent_non_household_caring_variables,
    decision='Review support', representation='Main-parent caring-context information', reason="Measures the main parent's own caring responsibility rather than the young person's experience; retained only to review household context and construct distinction")
caring_unresolved_variables = caring_variable_decisions.loc[caring_variable_decisions['Caring-track decision'].isna()].copy()
assert len(caring_unresolved_variables) == 0
assert len(caring_variable_decisions) == 40
expected_caring_decision_counts = {'Construction input': 3, 'Review support': 37}
actual_caring_decision_counts = caring_variable_decisions['Caring-track decision'].value_counts().to_dict()
assert actual_caring_decision_counts == expected_caring_decision_counts
caring_decision_summary = caring_variable_decisions.groupby(['Caring-track decision', 'Review role'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Caring-track decision',
    'Review role']).reset_index(drop=True)
caring_decision_display = caring_variable_decisions[['Review role', 'Wave', 'Source type', 'Source file', 'Variable',
    'Variable label', 'Timing status', 'Caring-track decision', 'Caring-track representation', 'Caring-track reason']].sort_values(['Caring-track decision',
    'Review role', 'Wave', 'Variable']).reset_index(drop=True)
caring_representation_decisions = pd.DataFrame([{'Representation': 'Latest direct young-person caring status',
    'Decision': 'Retain as predictor candidate', 'Reason': 'Direct measure, high coverage, consistent routing and clear pre-transition timing'}, {'Representation': 'Caring-related school absence',
    'Decision': 'Review support only', 'Reason': 'Only 18 positive cases in the latest eligible whole-sample representation'}, {'Representation': 'Weekly caring hours',
    'Decision': 'Review support only', 'Reason': 'Routed measure with incompatible continuous and grouped coding across waves'}, {'Representation': 'Caring-recipient indicators',
    'Decision': 'Review support only', 'Reason': 'Conditional multiple-response detail with sparse individual categories'}, {'Representation': 'Main-parent caring responsibility',
    'Decision': 'Review support only', 'Reason': "Household-context measure rather than the young person's own caring responsibility"}])
print('Selected caring-responsibility predictor:')
display_limited(caring_candidate_summary)
print('Selected predictor source waves:')
display_limited(caring_candidate_source_summary)
print('Caring variable decision summary:')
display_limited(caring_decision_summary)
print('Caring representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(caring_representation_decisions)
print(f'Unresolved caring variables: {len(caring_unresolved_variables)}')
print('Complete caring variable decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(caring_decision_display)

Selected caring-responsibility predictor:


,Predictor,Construct,Representation,No,Yes,Non-missing,Missing,Missing percentage,Yes percentage among observed
0,young_person_caring_responsibility_pretransition,Young-person caring responsibility within the ...,Latest eligible direct young-person status: Wa...,8962,559,9521,246,2.52,5.87


Selected predictor source waves:


,Source,Participants,Percentage
0,Wave 3,9432,96.57
1,Unavailable,246,2.52
2,Wave 2,77,0.79
3,Wave 1,12,0.12


Caring variable decision summary:


,Caring-track decision,Review role,Variables
0,Construction input,Direct young-person caring status,3
1,Review support,Caring-related school-absence status,3
2,Review support,Main-parent household caring status,2
3,Review support,Main-parent non-household caring status,2
4,Review support,Routed caring-related absence frequency,3


Caring representation decisions:


,Representation,Decision,Reason
0,Latest direct young-person caring status,Retain as predictor candidate,"Direct measure, high coverage, consistent routing and clear pre-transition timing"
1,Caring-related school absence,Review support only,Only 18 positive cases in the latest eligible whole-sample representation
2,Weekly caring hours,Review support only,Routed measure with incompatible continuous and grouped coding across waves
3,Caring-recipient indicators,Review support only,Conditional multiple-response detail with sparse individual categories
4,Main-parent caring responsibility,Review support only,Household-context measure rather than the young person's own caring responsibility


Unresolved caring variables: 0
Complete caring variable decisions:


,Review role,Wave,Source type,Source file,Variable,Variable label,Timing status,Caring-track decision,Caring-track representation,Caring-track reason
0,Direct young-person caring status,Wave 1,Young person,wave_one_lsype_young_person_2020,W1careYP,YP: Whether YP has any caring responsibilities within household,Pre-transition source,Construction input,young_person_caring_responsibility_pretransition,"Repeated direct young-person caring-status measure. Wave 3 is restricted to interviews before September 2006, with Wave 2 and Wave 1 used as fallbacks"
1,Direct young-person caring status,Wave 2,Young person,wave_two_lsype_young_person_2020,W2careYP,YP: Whether YP has any caring responsibilities within household,Pre-transition source,Construction input,young_person_caring_responsibility_pretransition,"Repeated direct young-person caring-status measure. Wave 3 is restricted to interviews before September 2006, with Wave 2 and Wave 1 used as fallbacks"
2,Direct young-person caring status,Wave 3,Young person,wave_three_lsype_young_person_2020,W3careYP,YP: Whether YP has any caring responsibilities within household,Near-transition source,Construction input,young_person_caring_responsibility_pretransition,"Repeated direct young-person caring-status measure. Wave 3 is restricted to interviews before September 2006, with Wave 2 and Wave 1 used as fallbacks"
3,Caring-related school-absence status,Wave 1,Young person,wave_one_lsype_young_person_2020,W1carehr1YP,YP: Whether ever miss school because of caring responsibilities,Pre-transition source,Review support,Caring-related school-absence status,"Valid routed measure but only 18 participants report caring-related absence in the latest eligible representation, making it too sparse for a separate predictor"
4,Caring-related school-absence status,Wave 2,Young person,wave_two_lsype_young_person_2020,W2carehr1YP,YP: Whether ever miss school because of caring responsibilities,Pre-transition source,Review support,Caring-related school-absence status,"Valid routed measure but only 18 participants report caring-related absence in the latest eligible representation, making it too sparse for a separate predictor"


In [304]:
# 8: Smoking, alcohol and cannabis variable-role and coding review

import pandas as pd
domain_8_substance_inventory = domain_8_pending_candidates.loc[domain_8_pending_candidates['Matched construct'].str.contains('Smoking|Drug or substance use',
    case=False, regex=True, na=False)].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(domain_8_substance_inventory) == 9
smoking_status_variables = ['W1cignowYP', 'W2cignowYP', 'W3cignowYP']
smoking_frequency_variables = ['W1cigfreqYP', 'W2cigfreqYP', 'W3cigfreqYP']
cannabis_experience_variables = ['W1canntryYP', 'W2canntryYP', 'W3canntryYP']
substance_role_lookup = {}
for variable in smoking_status_variables:
    substance_role_lookup[variable] = 'Direct smoking status'
for variable in smoking_frequency_variables:
    substance_role_lookup[variable] = 'Routed smoking frequency'
for variable in cannabis_experience_variables:
    substance_role_lookup[variable] = 'Direct cannabis-experience status'
assert set(substance_role_lookup) == set(domain_8_substance_inventory['Variable'])
domain_8_substance_inventory['Review role'] = domain_8_substance_inventory['Variable'].map(substance_role_lookup)
domain_8_young_person_register = verified_decision_register.loc[verified_decision_register['Wave'].isin(['Wave 1',
    'Wave 2', 'Wave 3']) & verified_decision_register['Source type'].eq('Young person')].copy()
substance_neighbourhood_rows = []
for _, anchor_row in domain_8_substance_inventory.iterrows():
    anchor_position = int(anchor_row['Variable position'])
    neighbour_mask = domain_8_young_person_register['Source file'].eq(anchor_row['Source file']) & domain_8_young_person_register['Variable position'].between(anchor_position - 4,
        anchor_position + 4, inclusive='both')
    anchor_neighbours = domain_8_young_person_register.loc[neighbour_mask].copy()
    anchor_neighbours['Anchor variable'] = anchor_row['Variable']
    anchor_neighbours['Anchor role'] = anchor_row['Review role']
    anchor_neighbours['Distance from anchor'] = anchor_neighbours['Variable position'] - anchor_position
    substance_neighbourhood_rows.append(anchor_neighbours)
domain_8_substance_neighbourhood = pd.concat(substance_neighbourhood_rows, ignore_index=True,
    sort=False).sort_values(['Anchor variable', 'Distance from anchor']).reset_index(drop=True)
domain_8_known_substance_variables = set(domain_8_substance_inventory['Variable'])
domain_8_additional_substance_neighbours = domain_8_substance_neighbourhood.loc[~domain_8_substance_neighbourhood['Variable'].isin(domain_8_known_substance_variables)].drop_duplicates(subset=['Source file',
    'Variable']).copy()
domain_8_additional_substance_neighbours['Potential substance relevance'] = domain_8_additional_substance_neighbours['Variable label'].fillna('').astype('string').str.contains('\\balcohol\\b|\\bdrink\\b|\\bdrunk\\b|\\bcigarette\\b|\\bsmok|\\bcannabis\\b|\\bdrug\\b',
    case=False, regex=True, na=False)
substance_raw_tables = {}
substance_labelled_tables = {}
substance_quality_rows = []
substance_code_rows = []
for source_file, source_inventory in domain_8_substance_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    raw_data['NSID'] = standardise_nsid(raw_data['NSID'])
    labelled_data['NSID'] = standardise_nsid(labelled_data['NSID'])
    assert raw_data['NSID'].is_unique
    assert labelled_data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(route_index)
    labelled_data = labelled_data.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    substance_raw_tables[source_file] = raw_data
    substance_labelled_tables[source_file] = labelled_data
    for variable in source_variables:
        variable_inventory = source_inventory.loc[source_inventory['Variable'].eq(variable)].iloc[0]
        raw_values = raw_data[variable]
        labelled_values = labelled_data[variable].astype('string')
        observed_mask = raw_values.ge(0).fillna(False)
        special_code_mask = raw_values.lt(0).fillna(False)
        observed_values = raw_values.loc[observed_mask]
        substance_quality_rows.append({'Review role': variable_inventory['Review role'],
            'Wave': variable_inventory['Wave'], 'Variable': variable, 'Variable label': variable_inventory['Variable label'], 'Observed responses': int(observed_mask.sum()), 'Special-code responses': int(special_code_mask.sum()), 'No source record': int(raw_values.isna().sum()), 'Observed percentage': round(observed_mask.mean() * 100,
            2), 'Distinct observed codes': int(observed_values.nunique()), 'Minimum observed code': observed_values.min() if len(observed_values) > 0 else pd.NA, 'Maximum observed code': observed_values.max() if len(observed_values) > 0 else pd.NA, 'Not-applicable responses': int(raw_values.eq(-91).sum())})
        value_counts = raw_values.value_counts(dropna=False)
        for raw_code, participants in value_counts.items():
            if pd.isna(raw_code):
                value_label = 'No source record'
                response_type = 'No source record'
                sort_value = 999999
            else:
                matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(raw_code)
                response_type = 'Observed response' if raw_code >= 0 else 'Special code'
                sort_value = float(raw_code)
            substance_code_rows.append({'Review role': variable_inventory['Review role'],
                'Wave': variable_inventory['Wave'], 'Variable': variable, 'Variable label': variable_inventory['Variable label'], 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
substance_principal_quality = pd.DataFrame(substance_quality_rows).sort_values(['Review role', 'Wave',
    'Variable']).reset_index(drop=True)
substance_code_distribution = pd.DataFrame(substance_code_rows).sort_values(['Review role', 'Wave', 'Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
domain_8_substance_role_summary = domain_8_substance_inventory.groupby(['Review role', 'Wave'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Review role', 'Wave']).reset_index(drop=True)
domain_8_additional_substance_display = domain_8_additional_substance_neighbours.sort_values(['Potential substance relevance',
    'Wave', 'Source order', 'Variable position'], ascending=[False, True, True,
    True])[['Potential substance relevance', 'Wave', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Review status', 'Review outcome', 'Substantive domain']].reset_index(drop=True)
print(f'Smoking and cannabis variables reviewed: {len(domain_8_substance_inventory)}')
print(f'Additional neighbouring variables inspected: {len(domain_8_additional_substance_neighbours)}')
print('Substance variable-role summary:')
display_limited(domain_8_substance_role_summary)
print('Principal-measure quality:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(substance_principal_quality)
print('Principal-measure response codes:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(substance_code_distribution)
print('Neighbouring variables, including possible alcohol measures:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_8_additional_substance_display)

Smoking and cannabis variables reviewed: 9
Additional neighbouring variables inspected: 31
Substance variable-role summary:


,Review role,Wave,Variables
0,Direct cannabis-experience status,Wave 1,1
1,Direct cannabis-experience status,Wave 2,1
2,Direct cannabis-experience status,Wave 3,1
3,Direct smoking status,Wave 1,1
4,Direct smoking status,Wave 2,1


Principal-measure quality:


,Review role,Wave,Variable,Variable label,Observed responses,Special-code responses,No source record,Observed percentage,Distinct observed codes,Minimum observed code,Maximum observed code,Not-applicable responses
0,Direct cannabis-experience status,Wave 1,W1canntryYP,YP: Whether ever tried Cannabis,9202,322,243,94.22,2,1.0,2.0,0
1,Direct cannabis-experience status,Wave 2,W2canntryYP,YP: Whether ever tried Cannabis,9229,292,246,94.49,2,1.0,2.0,0
2,Direct cannabis-experience status,Wave 3,W3canntryYP,YP: Whether YP ever tried Cannabis,9284,225,258,95.05,2,1.0,2.0,0
3,Direct smoking status,Wave 1,W1cignowYP,YP: Whether ever smoke cigarettes,9021,503,243,92.36,2,1.0,2.0,0
4,Direct smoking status,Wave 2,W2cignowYP,YP: Whether ever smoke cigarettes,9106,415,246,93.23,2,1.0,2.0,0


Principal-measure response codes:


,Review role,Wave,Variable,Variable label,Raw code,Value label,Response type,Participants
0,Direct cannabis-experience status,Wave 1,W1canntryYP,YP: Whether ever tried Cannabis,-99.0,YP not interviewed,Special code,89
1,Direct cannabis-experience status,Wave 1,W1canntryYP,YP: Whether ever tried Cannabis,-97.0,YP refused CASI section,Special code,66
2,Direct cannabis-experience status,Wave 1,W1canntryYP,YP: Whether ever tried Cannabis,-96.0,YP unable to complete CASI section,Special code,49
3,Direct cannabis-experience status,Wave 1,W1canntryYP,YP: Whether ever tried Cannabis,-92.0,Refused,Special code,90
4,Direct cannabis-experience status,Wave 1,W1canntryYP,YP: Whether ever tried Cannabis,-1.0,Don't know,Special code,28


Neighbouring variables, including possible alcohol measures:


,Potential substance relevance,Wave,Source file,Variable position,Variable,Variable label,Review status,Review outcome,Substantive domain
0,True,Wave 1,wave_one_lsype_young_person_2020,272,W1alceverYP,YP: Whether ever had proper alcoholic drink,"Variable-level coding, routing and reference-period review required.",Pending review,NaN
1,True,Wave 1,wave_one_lsype_young_person_2020,273,W1alcmonYP,YP: Whether had alcoholic drink in last 12 months,"Variable-level coding, routing and reference-period review required.",Pending review,NaN
2,True,Wave 1,wave_one_lsype_young_person_2020,274,W1alcfreqYP,YP: Frequency of having alcoholic drink in last 12 months,"Variable-level coding, routing and reference-period review required.",Pending review,NaN
3,True,Wave 2,wave_two_lsype_young_person_2020,452,W2alceverYP,YP: Whether ever had proper alcoholic drink,"Variable-level coding, routing and reference-period review required.",Pending review,NaN
4,True,Wave 2,wave_two_lsype_young_person_2020,453,W2alcfreqYP,YP: Frequency of having alcoholic drink in last 12 months,"Variable-level coding, routing and reference-period review required.",Pending review,NaN


In [305]:
# 9: Alcohol variable-role, coding and routing review

import pandas as pd
alcohol_ever_status_variables = ['W1alceverYP', 'W2alceverYP', 'W3alceverYP']
alcohol_recent_status_variables = ['W1alcmonYP']
alcohol_frequency_variables = ['W1alcfreqYP', 'W2alcfreqYP', 'W3alcfreqYP']
alcohol_review_variables = alcohol_ever_status_variables + alcohol_recent_status_variables + alcohol_frequency_variables
assert len(alcohol_review_variables) == 7
assert len(set(alcohol_review_variables)) == 7
domain_8_alcohol_inventory = verified_decision_register.loc[verified_decision_register['Variable'].isin(alcohol_review_variables)].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert set(domain_8_alcohol_inventory['Variable']) == set(alcohol_review_variables)
assert len(domain_8_alcohol_inventory) == 7
alcohol_role_lookup = {}
for variable in alcohol_ever_status_variables:
    alcohol_role_lookup[variable] = 'Direct lifetime alcohol-experience status'
for variable in alcohol_recent_status_variables:
    alcohol_role_lookup[variable] = 'Direct recent alcohol-use status'
for variable in alcohol_frequency_variables:
    alcohol_role_lookup[variable] = 'Routed alcohol-use frequency'
domain_8_alcohol_inventory['Review role'] = domain_8_alcohol_inventory['Variable'].map(alcohol_role_lookup)
assert domain_8_alcohol_inventory['Review role'].notna().all()
alcohol_raw_tables = {}
alcohol_labelled_tables = {}
alcohol_quality_rows = []
alcohol_code_rows = []
for source_file, source_inventory in domain_8_alcohol_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    raw_data['NSID'] = standardise_nsid(raw_data['NSID'])
    labelled_data['NSID'] = standardise_nsid(labelled_data['NSID'])
    assert raw_data['NSID'].is_unique
    assert labelled_data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(route_index)
    labelled_data = labelled_data.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    alcohol_raw_tables[source_file] = raw_data
    alcohol_labelled_tables[source_file] = labelled_data
    for variable in source_variables:
        variable_inventory = source_inventory.loc[source_inventory['Variable'].eq(variable)].iloc[0]
        raw_values = raw_data[variable]
        labelled_values = labelled_data[variable].astype('string')
        observed_mask = raw_values.ge(0).fillna(False)
        special_code_mask = raw_values.lt(0).fillna(False)
        observed_values = raw_values.loc[observed_mask]
        alcohol_quality_rows.append({'Review role': variable_inventory['Review role'],
            'Wave': variable_inventory['Wave'], 'Variable': variable, 'Variable label': variable_inventory['Variable label'], 'Observed responses': int(observed_mask.sum()), 'Special-code responses': int(special_code_mask.sum()), 'No source record': int(raw_values.isna().sum()), 'Observed percentage': round(observed_mask.mean() * 100,
            2), 'Distinct observed codes': int(observed_values.nunique()), 'Minimum observed code': observed_values.min() if len(observed_values) > 0 else pd.NA, 'Maximum observed code': observed_values.max() if len(observed_values) > 0 else pd.NA, 'Not-applicable responses': int(raw_values.eq(-91).sum())})
        value_counts = raw_values.value_counts(dropna=False)
        for raw_code, participants in value_counts.items():
            if pd.isna(raw_code):
                value_label = 'No source record'
                response_type = 'No source record'
                sort_value = 999999
            else:
                matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(raw_code)
                response_type = 'Observed response' if raw_code >= 0 else 'Special code'
                sort_value = float(raw_code)
            alcohol_code_rows.append({'Review role': variable_inventory['Review role'],
                'Wave': variable_inventory['Wave'], 'Variable': variable, 'Variable label': variable_inventory['Variable label'], 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
alcohol_quality = pd.DataFrame(alcohol_quality_rows).sort_values(['Review role', 'Wave',
    'Variable']).reset_index(drop=True)
alcohol_code_distribution = pd.DataFrame(alcohol_code_rows).sort_values(['Review role', 'Wave', 'Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)

def get_alcohol_raw_variable(variable):
    """Return one aligned raw alcohol variable."""
    source_file = domain_8_alcohol_inventory.loc[domain_8_alcohol_inventory['Variable'].eq(variable),
        'Source file'].iloc[0]
    return alcohol_raw_tables[source_file][variable].copy()
alcohol_routing_rows = []
for wave, ever_variable, frequency_variable in [('Wave 1', 'W1alceverYP', 'W1alcfreqYP'), ('Wave 2', 'W2alceverYP',
    'W2alcfreqYP'), ('Wave 3', 'W3alceverYP', 'W3alcfreqYP')]:
    ever_raw = get_alcohol_raw_variable(ever_variable)
    frequency_raw = get_alcohol_raw_variable(frequency_variable)
    if wave == 'Wave 3':
        ever_raw = ever_raw.where(wave_3_pretransition_caring_mask)
        frequency_raw = frequency_raw.where(wave_3_pretransition_caring_mask)
    routing_data = pd.DataFrame({'Alcohol experience status': ever_raw.map({1.0: 'Ever had alcohol',
        2.0: 'Never had alcohol'}).fillna('Status unavailable'), 'Frequency response': frequency_raw.apply(lambda value: 'Observed frequency' if pd.notna(value) and value >= 0 else 'Structurally not applicable' if value == -91 else 'Other unavailable response')})
    wave_routing_counts = routing_data.value_counts().rename('Participants').reset_index()
    wave_routing_counts['Wave'] = wave
    alcohol_routing_rows.append(wave_routing_counts)
alcohol_routing_summary = pd.concat(alcohol_routing_rows, ignore_index=True)[['Wave', 'Alcohol experience status',
    'Frequency response', 'Participants']].sort_values(['Wave', 'Alcohol experience status',
    'Frequency response']).reset_index(drop=True)
alcohol_routing_contradiction_rows = []
for wave, ever_variable, frequency_variable in [('Wave 1', 'W1alceverYP', 'W1alcfreqYP'), ('Wave 2', 'W2alceverYP',
    'W2alcfreqYP'), ('Wave 3', 'W3alceverYP', 'W3alcfreqYP')]:
    ever_raw = get_alcohol_raw_variable(ever_variable)
    frequency_raw = get_alcohol_raw_variable(frequency_variable)
    if wave == 'Wave 3':
        ever_raw = ever_raw.where(wave_3_pretransition_caring_mask)
        frequency_raw = frequency_raw.where(wave_3_pretransition_caring_mask)
    alcohol_routing_contradiction_rows.append({'Wave': wave,
        'Never had alcohol but frequency observed': int((ever_raw.eq(2) & frequency_raw.ge(0)).sum()), 'Ever had alcohol but frequency not applicable': int((ever_raw.eq(1) & frequency_raw.eq(-91)).sum())})
alcohol_routing_contradiction_summary = pd.DataFrame(alcohol_routing_contradiction_rows)
wave_1_recent_alcohol_raw = get_alcohol_raw_variable('W1alcmonYP')
wave_1_alcohol_frequency_raw = get_alcohol_raw_variable('W1alcfreqYP')
wave_1_recent_frequency_review = pd.DataFrame({'Recent alcohol status': wave_1_recent_alcohol_raw.map({1.0: 'Alcohol in last 12 months',
    2.0: 'No alcohol in last 12 months'}).fillna('Recent status unavailable'), 'Frequency response': wave_1_alcohol_frequency_raw.apply(lambda value: 'Observed frequency' if pd.notna(value) and value >= 0 else 'Structurally not applicable' if value == -91 else 'Other unavailable response')})
wave_1_recent_frequency_summary = wave_1_recent_frequency_review.value_counts().rename('Participants').reset_index().sort_values(['Recent alcohol status',
    'Frequency response']).reset_index(drop=True)
domain_8_alcohol_role_summary = domain_8_alcohol_inventory.groupby(['Review role', 'Wave'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Review role', 'Wave']).reset_index(drop=True)
print(f'Alcohol variables reviewed: {len(domain_8_alcohol_inventory)}')
print('Alcohol variable-role summary:')
display_limited(domain_8_alcohol_role_summary)
print('Alcohol measure quality:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(alcohol_quality)
print('Alcohol response codes:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(alcohol_code_distribution)
print('Alcohol status and frequency routing:')
with pd.option_context('display.max_rows', None):
    display_limited(alcohol_routing_summary)
print('Alcohol routing contradictions:')
display_limited(alcohol_routing_contradiction_summary)
print('Wave 1 recent alcohol status and frequency:')
display_limited(wave_1_recent_frequency_summary)

Alcohol variables reviewed: 7
Alcohol variable-role summary:


,Review role,Wave,Variables
0,Direct lifetime alcohol-experience status,Wave 1,1
1,Direct lifetime alcohol-experience status,Wave 2,1
2,Direct lifetime alcohol-experience status,Wave 3,1
3,Direct recent alcohol-use status,Wave 1,1
4,Routed alcohol-use frequency,Wave 1,1


Alcohol measure quality:


,Review role,Wave,Variable,Variable label,Observed responses,Special-code responses,No source record,Observed percentage,Distinct observed codes,Minimum observed code,Maximum observed code,Not-applicable responses
0,Direct lifetime alcohol-experience status,Wave 1,W1alceverYP,YP: Whether ever had proper alcoholic drink,8944,580,243,91.57,2,1.0,2.0,0
1,Direct lifetime alcohol-experience status,Wave 2,W2alceverYP,YP: Whether ever had proper alcoholic drink,9104,417,246,93.21,2,1.0,2.0,0
2,Direct lifetime alcohol-experience status,Wave 3,W3alceverYP,YP: Whether YP ever had proper alcoholic drink,9235,274,258,94.55,2,1.0,2.0,0
3,Direct recent alcohol-use status,Wave 1,W1alcmonYP,YP: Whether had alcoholic drink in last 12 months,4128,5396,243,42.26,2,1.0,2.0,5149
4,Routed alcohol-use frequency,Wave 1,W1alcfreqYP,YP: Frequency of having alcoholic drink in last 12 months,3681,5843,243,37.69,6,1.0,6.0,5486


Alcohol response codes:


...

,Wave,Alcohol experience status,Frequency response,Participants
0,Wave 1,Ever had alcohol,Observed frequency,3681
1,Wave 1,Ever had alcohol,Other unavailable response,153
2,Wave 1,Ever had alcohol,Structurally not applicable,337
3,Wave 1,Never had alcohol,Structurally not applicable,4773
4,Wave 1,Status unavailable,Other unavailable response,447


Alcohol routing contradictions:


,Wave,Never had alcohol but frequency observed,Ever had alcohol but frequency not applicable
0,Wave 1,0,337
1,Wave 2,0,0
2,Wave 3,0,0


Wave 1 recent alcohol status and frequency:


,Recent alcohol status,Frequency response,Participants
0,Alcohol in last 12 months,Observed frequency,3681
1,Alcohol in last 12 months,Other unavailable response,153
2,No alcohol in last 12 months,Structurally not applicable,294
3,Recent status unavailable,Other unavailable response,447
4,Recent status unavailable,Structurally not applicable,5192


In [306]:
# 10: Smoking, alcohol and cannabis representation review

import pandas as pd
wave_3_pretransition_substance_mask = wave_3_pretransition_caring_mask.copy()
assert int(wave_3_pretransition_substance_mask.sum()) == 9495

def get_substance_raw_variable(variable):
    """Return one aligned raw smoking or cannabis variable."""
    source_file = domain_8_substance_inventory.loc[domain_8_substance_inventory['Variable'].eq(variable),
        'Source file'].iloc[0]
    return substance_raw_tables[source_file][variable].copy()

def recode_substance_yes_no(raw_values):
    """Recode 1 = yes and 2 = no to binary form."""
    return raw_values.map({1.0: 1.0, 2.0: 0.0}).astype('Float64')
smoking_experience_by_wave = pd.DataFrame({'Wave 1': recode_substance_yes_no(get_substance_raw_variable('W1cignowYP')),
    'Wave 2': recode_substance_yes_no(get_substance_raw_variable('W2cignowYP')), 'Wave 3': recode_substance_yes_no(get_substance_raw_variable('W3cignowYP')).where(wave_3_pretransition_substance_mask)}, index=route_index)
alcohol_experience_by_wave = pd.DataFrame({'Wave 1': recode_substance_yes_no(get_alcohol_raw_variable('W1alceverYP')),
    'Wave 2': recode_substance_yes_no(get_alcohol_raw_variable('W2alceverYP')), 'Wave 3': recode_substance_yes_no(get_alcohol_raw_variable('W3alceverYP')).where(wave_3_pretransition_substance_mask)}, index=route_index)
cannabis_experience_by_wave = pd.DataFrame({'Wave 1': recode_substance_yes_no(get_substance_raw_variable('W1canntryYP')),
    'Wave 2': recode_substance_yes_no(get_substance_raw_variable('W2canntryYP')), 'Wave 3': recode_substance_yes_no(get_substance_raw_variable('W3canntryYP')).where(wave_3_pretransition_substance_mask)}, index=route_index)
latest_smoking_experience, latest_smoking_experience_source = create_latest_representation(smoking_experience_by_wave,
    ['Wave 3', 'Wave 2', 'Wave 1'])
latest_alcohol_experience, latest_alcohol_experience_source = create_latest_representation(alcohol_experience_by_wave,
    ['Wave 3', 'Wave 2', 'Wave 1'])
latest_cannabis_experience, latest_cannabis_experience_source = create_latest_representation(cannabis_experience_by_wave,
    ['Wave 3', 'Wave 2', 'Wave 1'])
substance_binary_representations = pd.DataFrame({'Ever smoked': latest_smoking_experience,
    'Ever had alcohol': latest_alcohol_experience, 'Ever tried cannabis': latest_cannabis_experience}, index=route_index)
substance_binary_summary_rows = []
for representation in substance_binary_representations.columns:
    values = substance_binary_representations[representation]
    substance_binary_summary_rows.append({'Representation': representation, 'Non-missing': int(values.notna().sum()),
        'Missing': int(values.isna().sum()), 'Missing percentage': round(values.isna().mean() * 100,
        2), 'No': int(values.eq(0).sum()), 'Yes': int(values.eq(1).sum()), 'Yes percentage among observed': round(values.eq(1).sum() / values.notna().sum() * 100,
        2)})
substance_binary_summary = pd.DataFrame(substance_binary_summary_rows)
substance_source_lookup = {'Ever smoked': latest_smoking_experience_source,
    'Ever had alcohol': latest_alcohol_experience_source, 'Ever tried cannabis': latest_cannabis_experience_source}
substance_source_summary_rows = []
for representation, source_series in substance_source_lookup.items():
    for source, participants in source_series.value_counts(dropna=False).items():
        substance_source_summary_rows.append({'Representation': representation, 'Source': source,
            'Participants': int(participants), 'Percentage': round(participants / len(route_index) * 100, 2)})
substance_source_summary = pd.DataFrame(substance_source_summary_rows).sort_values(['Representation',
    'Source']).reset_index(drop=True)
smoking_routing_rows = []
smoking_wave_variables = {'Wave 1': ('W1cignowYP', 'W1cigfreqYP'), 'Wave 2': ('W2cignowYP', 'W2cigfreqYP'),
    'Wave 3': ('W3cignowYP', 'W3cigfreqYP')}
smoking_activity_by_wave = pd.DataFrame(index=route_index)
for wave, (status_variable, frequency_variable) in smoking_wave_variables.items():
    status_raw = get_substance_raw_variable(status_variable)
    frequency_raw = get_substance_raw_variable(frequency_variable)
    if wave == 'Wave 3':
        status_raw = status_raw.where(wave_3_pretransition_substance_mask)
        frequency_raw = frequency_raw.where(wave_3_pretransition_substance_mask)
    smoking_routing_rows.append({'Wave': wave,
        'Never-smoked status with frequency observed': int((status_raw.eq(2) & frequency_raw.ge(0)).sum()), 'Ever-smoked status with frequency not applicable': int((status_raw.eq(1) & frequency_raw.eq(-91)).sum()), 'Ever-smoked status with frequency coded never': int((status_raw.eq(1) & frequency_raw.eq(1)).sum()), 'Ever-smoked status with other unavailable frequency': int((status_raw.eq(1) & frequency_raw.lt(0) & ~frequency_raw.eq(-91)).sum())})
    smoking_activity = pd.Series(pd.NA, index=route_index, dtype='Float64')
    smoking_activity.loc[status_raw.eq(2)] = 0.0
    smoking_activity.loc[status_raw.eq(1) & frequency_raw.isin([2.0, 3.0])] = 1.0
    smoking_activity.loc[status_raw.eq(1) & frequency_raw.eq(4)] = 2.0
    smoking_activity.loc[status_raw.eq(1) & frequency_raw.isin([5.0, 6.0])] = 3.0
    smoking_activity_by_wave[wave] = smoking_activity
smoking_routing_summary = pd.DataFrame(smoking_routing_rows)
latest_smoking_activity, latest_smoking_activity_source = create_latest_representation(smoking_activity_by_wave,
    ['Wave 3', 'Wave 2', 'Wave 1'])
smoking_activity_labels = {0.0: 'Never smoked', 1.0: 'Tried once or formerly smoked',
    2.0: 'Current occasional smoking', 3.0: 'Current regular smoking'}
smoking_activity_summary = latest_smoking_activity.map(smoking_activity_labels).fillna('Unavailable').value_counts().rename('Participants').rename_axis('Smoking activity').reset_index()
smoking_activity_summary['Percentage'] = (smoking_activity_summary['Participants'] / len(route_index) * 100).round(2)
smoking_activity_source_summary = latest_smoking_activity_source.value_counts().rename('Participants').rename_axis('Source').reset_index()
smoking_activity_source_summary['Percentage'] = (smoking_activity_source_summary['Participants'] / len(route_index) * 100).round(2)
substance_wave_data_lookup = {'Smoking experience': smoking_experience_by_wave,
    'Alcohol experience': alcohol_experience_by_wave, 'Cannabis experience': cannabis_experience_by_wave}
substance_wave_pairs = [('Wave 1', 'Wave 2'), ('Wave 2', 'Wave 3'), ('Wave 1', 'Wave 3')]
substance_wave_comparison_rows = []
for construct, wave_data in substance_wave_data_lookup.items():
    for first_wave, second_wave in substance_wave_pairs:
        pair_data = wave_data[[first_wave, second_wave]].dropna()
        substance_wave_comparison_rows.append({'Construct': construct, 'First wave': first_wave,
            'Second wave': second_wave, 'Complete comparisons': len(pair_data), 'Exact agreement percentage': round(pair_data[first_wave].eq(pair_data[second_wave]).mean() * 100,
            2), "Cramer's V": round(binary_cramers_v(pair_data[first_wave], pair_data[second_wave]), 3)})
substance_wave_comparison = pd.DataFrame(substance_wave_comparison_rows)
substance_pairwise_rows = []
substance_pair_names = [('Ever smoked', 'Ever had alcohol'), ('Ever smoked', 'Ever tried cannabis'),
    ('Ever had alcohol', 'Ever tried cannabis')]
for first_name, second_name in substance_pair_names:
    pair_data = substance_binary_representations[[first_name, second_name]].dropna()
    substance_pairwise_rows.append({'First representation': first_name, 'Second representation': second_name,
        'Complete comparisons': len(pair_data), "Cramer's V": round(binary_cramers_v(pair_data[first_name],
        pair_data[second_name]), 3)})
substance_pairwise_comparison = pd.DataFrame(substance_pairwise_rows)
substance_experience_count = substance_binary_representations.sum(axis=1, min_count=3).astype('Float64')
substance_experience_count_summary = substance_experience_count.value_counts(dropna=False).rename('Participants').rename_axis('Number of reported substance experiences').reset_index()
substance_experience_count_summary['Percentage'] = (substance_experience_count_summary['Participants'] / len(route_index) * 100).round(2)
print('Latest binary substance representations:')
display_limited(substance_binary_summary)
print('Binary representation source waves:')
display_limited(substance_source_summary)
print('Smoking-status and frequency routing:')
display_limited(smoking_routing_summary)
print('Diagnostic smoking-activity representation:')
display_limited(smoking_activity_summary)
print('Diagnostic smoking-activity source waves:')
display_limited(smoking_activity_source_summary)
print('Repeated binary measures across waves:')
display_limited(substance_wave_comparison)
print('Associations between latest binary representations:')
display_limited(substance_pairwise_comparison)
print('Number of reported substance experiences:')
display_limited(substance_experience_count_summary)

Latest binary substance representations:


,Representation,Non-missing,Missing,Missing percentage,No,Yes,Yes percentage among observed
0,Ever smoked,9510,257,2.63,7426,2084,21.91
1,Ever had alcohol,9499,268,2.74,3178,6321,66.54
2,Ever tried cannabis,9513,254,2.60,7369,2144,22.54


Binary representation source waves:


,Representation,Source,Participants,Percentage
0,Ever had alcohol,Unavailable,268,2.74
1,Ever had alcohol,Wave 1,42,0.43
2,Ever had alcohol,Wave 2,236,2.42
3,Ever had alcohol,Wave 3,9221,94.41
4,Ever smoked,Unavailable,257,2.63


Smoking-status and frequency routing:


...

,Construct,First wave,Second wave,Complete comparisons,Exact agreement percentage,Cramer's V
0,Smoking experience,Wave 1,Wave 2,8672,88.64,0.493
1,Smoking experience,Wave 2,Wave 3,8877,88.41,0.632
2,Smoking experience,Wave 1,Wave 3,8778,83.39,0.399
3,Alcohol experience,Wave 1,Wave 2,8608,79.51,0.610
4,Alcohol experience,Wave 2,Wave 3,8868,84.98,0.691


Associations between latest binary representations:


,First representation,Second representation,Complete comparisons,Cramer's V
0,Ever smoked,Ever had alcohol,9497,0.275
1,Ever smoked,Ever tried cannabis,9510,0.481
2,Ever had alcohol,Ever tried cannabis,9499,0.314


Number of reported substance experiences:


,Number of reported substance experiences,Participants,Percentage
0,1.0,3816,39.07
1,0.0,2918,29.88
2,2.0,1560,15.97
3,3.0,1203,12.32
4,<NA>,270,2.76


In [307]:
# 11: Detailed substance-use representation review

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
alcohol_frequency_by_wave = pd.DataFrame(index=route_index)

def construct_alcohol_frequency_category(ever_variable, frequency_variable, recent_variable=None,
    eligibility_mask=None):
    """Construct a harmonised recent alcohol-frequency category."""
    ever_raw = get_alcohol_raw_variable(ever_variable)
    frequency_raw = get_alcohol_raw_variable(frequency_variable)
    if recent_variable is not None:
        recent_raw = get_alcohol_raw_variable(recent_variable)
    else:
        recent_raw = pd.Series(pd.NA, index=route_index, dtype='Float64')
    if eligibility_mask is not None:
        ever_raw = ever_raw.where(eligibility_mask)
        frequency_raw = frequency_raw.where(eligibility_mask)
        recent_raw = recent_raw.where(eligibility_mask)
    category = pd.Series(pd.NA, index=route_index, dtype='Float64')
    category.loc[ever_raw.eq(2)] = 0.0
    if recent_variable is not None:
        category.loc[recent_raw.eq(2)] = 0.0
    category.loc[frequency_raw.isin([5.0, 6.0])] = 1.0
    category.loc[frequency_raw.isin([3.0, 4.0])] = 2.0
    category.loc[frequency_raw.isin([1.0, 2.0])] = 3.0
    return category
alcohol_frequency_by_wave['Wave 1'] = construct_alcohol_frequency_category(ever_variable='W1alceverYP',
    recent_variable='W1alcmonYP', frequency_variable='W1alcfreqYP')
alcohol_frequency_by_wave['Wave 2'] = construct_alcohol_frequency_category(ever_variable='W2alceverYP',
    frequency_variable='W2alcfreqYP')
alcohol_frequency_by_wave['Wave 3'] = construct_alcohol_frequency_category(ever_variable='W3alceverYP',
    frequency_variable='W3alcfreqYP', eligibility_mask=wave_3_pretransition_substance_mask)
latest_alcohol_frequency, latest_alcohol_frequency_source = create_latest_representation(alcohol_frequency_by_wave,
    ['Wave 3', 'Wave 2', 'Wave 1'])
smoking_activity_pretransition_candidate = latest_smoking_activity.copy().rename('smoking_activity_pretransition')
alcohol_use_frequency_pretransition_candidate = latest_alcohol_frequency.copy().rename('alcohol_use_frequency_pretransition')
cannabis_experience_pretransition_candidate = latest_cannabis_experience.copy().rename('cannabis_experience_pretransition')
substance_candidate_data = pd.concat([smoking_activity_pretransition_candidate,
    alcohol_use_frequency_pretransition_candidate, cannabis_experience_pretransition_candidate], axis=1)
assert len(substance_candidate_data) == len(route_index)
assert substance_candidate_data.index.equals(route_index)
smoking_candidate_labels = {0.0: 'Never smoked', 1.0: 'Tried once or formerly smoked',
    2.0: 'Current occasional smoking', 3.0: 'Current regular smoking'}
alcohol_candidate_labels = {0.0: 'No alcohol in previous 12 months', 1.0: 'Less than monthly', 2.0: 'Monthly',
    3.0: 'Weekly or more often'}
cannabis_candidate_labels = {0.0: 'Never tried cannabis', 1.0: 'Ever tried cannabis'}
substance_candidate_label_lookup = {'smoking_activity_pretransition': smoking_candidate_labels,
    'alcohol_use_frequency_pretransition': alcohol_candidate_labels, 'cannabis_experience_pretransition': cannabis_candidate_labels}
substance_candidate_distribution_rows = []
for predictor, label_lookup in substance_candidate_label_lookup.items():
    values = substance_candidate_data[predictor]
    for code, label in label_lookup.items():
        participants = int(values.eq(code).sum())
        substance_candidate_distribution_rows.append({'Predictor': predictor, 'Code': int(code), 'Category': label,
            'Participants': participants, 'Percentage of full sample': round(participants / len(route_index) * 100,
            2), 'Percentage among observed': round(participants / values.notna().sum() * 100, 2)})
    substance_candidate_distribution_rows.append({'Predictor': predictor, 'Code': pd.NA, 'Category': 'Unavailable',
        'Participants': int(values.isna().sum()), 'Percentage of full sample': round(values.isna().mean() * 100,
        2), 'Percentage among observed': pd.NA})
substance_candidate_distribution = pd.DataFrame(substance_candidate_distribution_rows)
substance_candidate_coverage_rows = []
for predictor in substance_candidate_data.columns:
    values = substance_candidate_data[predictor]
    substance_candidate_coverage_rows.append({'Predictor': predictor, 'Non-missing': int(values.notna().sum()),
        'Missing': int(values.isna().sum()), 'Missing percentage': round(values.isna().mean() * 100,
        2), 'Distinct observed categories': int(values.nunique())})
substance_candidate_coverage = pd.DataFrame(substance_candidate_coverage_rows)
substance_detailed_source_lookup = {'smoking_activity_pretransition': latest_smoking_activity_source,
    'alcohol_use_frequency_pretransition': latest_alcohol_frequency_source, 'cannabis_experience_pretransition': latest_cannabis_experience_source}
substance_detailed_source_rows = []
for predictor, source_series in substance_detailed_source_lookup.items():
    for source, participants in source_series.value_counts(dropna=False).items():
        substance_detailed_source_rows.append({'Predictor': predictor, 'Source': source,
            'Participants': int(participants), 'Percentage': round(participants / len(route_index) * 100, 2)})
substance_detailed_source_summary = pd.DataFrame(substance_detailed_source_rows).sort_values(['Predictor',
    'Source']).reset_index(drop=True)

def categorical_cramers_v(first, second):
    """Calculate Cramer's V for two categorical variables."""
    complete_data = pd.DataFrame({'First': first, 'Second': second}).dropna()
    cross_tab = pd.crosstab(complete_data['First'], complete_data['Second'])
    if cross_tab.shape[0] < 2 or cross_tab.shape[1] < 2:
        return np.nan
    chi_square = chi2_contingency(cross_tab, correction=False)[0]
    sample_size = cross_tab.to_numpy().sum()
    denominator = sample_size * min(cross_tab.shape[0] - 1, cross_tab.shape[1] - 1)
    return float(np.sqrt(chi_square / denominator))
detailed_wave_data_lookup = {'Smoking activity': smoking_activity_by_wave,
    'Alcohol-use frequency': alcohol_frequency_by_wave}
detailed_wave_comparison_rows = []
for construct, wave_data in detailed_wave_data_lookup.items():
    for first_wave, second_wave in [('Wave 1', 'Wave 2'), ('Wave 2', 'Wave 3'), ('Wave 1', 'Wave 3')]:
        pair_data = wave_data[[first_wave, second_wave]].dropna()
        detailed_wave_comparison_rows.append({'Construct': construct, 'First wave': first_wave,
            'Second wave': second_wave, 'Complete comparisons': len(pair_data), 'Exact agreement percentage': round(pair_data[first_wave].eq(pair_data[second_wave]).mean() * 100,
            2), "Cramer's V": round(categorical_cramers_v(pair_data[first_wave], pair_data[second_wave]), 3)})
detailed_wave_comparison = pd.DataFrame(detailed_wave_comparison_rows)
smoking_binary_detailed_data = pd.DataFrame({'Ever smoked': latest_smoking_experience,
    'Detailed smoking activity': latest_smoking_activity}).dropna()
smoking_binary_detailed_summary = pd.DataFrame([{'Complete comparisons': len(smoking_binary_detailed_data),
    'Binary no and detailed never': int((smoking_binary_detailed_data['Ever smoked'].eq(0) & smoking_binary_detailed_data['Detailed smoking activity'].eq(0)).sum()), 'Binary yes and detailed ever/current': int((smoking_binary_detailed_data['Ever smoked'].eq(1) & smoking_binary_detailed_data['Detailed smoking activity'].gt(0)).sum()), 'Contradictory classifications': int((smoking_binary_detailed_data['Ever smoked'].eq(0) & smoking_binary_detailed_data['Detailed smoking activity'].gt(0)).sum() + (smoking_binary_detailed_data['Ever smoked'].eq(1) & smoking_binary_detailed_data['Detailed smoking activity'].eq(0)).sum())}])
alcohol_binary_detailed_data = pd.DataFrame({'Ever had alcohol': latest_alcohol_experience,
    'Recent alcohol-use frequency': latest_alcohol_frequency}).dropna()
alcohol_binary_detailed_summary = pd.DataFrame([{'Complete comparisons': len(alcohol_binary_detailed_data),
    'Never had alcohol and no recent use': int((alcohol_binary_detailed_data['Ever had alcohol'].eq(0) & alcohol_binary_detailed_data['Recent alcohol-use frequency'].eq(0)).sum()), 'Ever had alcohol and recent use': int((alcohol_binary_detailed_data['Ever had alcohol'].eq(1) & alcohol_binary_detailed_data['Recent alcohol-use frequency'].gt(0)).sum()), 'Ever had alcohol but no recent use': int((alcohol_binary_detailed_data['Ever had alcohol'].eq(1) & alcohol_binary_detailed_data['Recent alcohol-use frequency'].eq(0)).sum()), 'Never had alcohol but recent use': int((alcohol_binary_detailed_data['Ever had alcohol'].eq(0) & alcohol_binary_detailed_data['Recent alcohol-use frequency'].gt(0)).sum())}])
substance_candidate_pairwise_rows = []
for first_name, second_name in [('smoking_activity_pretransition', 'alcohol_use_frequency_pretransition'),
    ('smoking_activity_pretransition', 'cannabis_experience_pretransition'), ('alcohol_use_frequency_pretransition',
    'cannabis_experience_pretransition')]:
    pair_data = substance_candidate_data[[first_name, second_name]].dropna()
    substance_candidate_pairwise_rows.append({'First predictor': first_name, 'Second predictor': second_name,
        'Complete comparisons': len(pair_data), "Cramer's V": round(categorical_cramers_v(pair_data[first_name],
        pair_data[second_name]), 3)})
substance_candidate_pairwise_comparison = pd.DataFrame(substance_candidate_pairwise_rows)
substance_candidate_availability_count = substance_candidate_data.notna().sum(axis=1)
substance_candidate_joint_availability = substance_candidate_availability_count.value_counts().sort_index().rename('Participants').rename_axis('Substance predictors available').reset_index()
substance_candidate_joint_availability['Percentage'] = (substance_candidate_joint_availability['Participants'] / len(route_index) * 100).round(2)
print('Detailed substance candidate coverage:')
display_limited(substance_candidate_coverage)
print('Detailed substance candidate distributions:')
with pd.option_context('display.max_rows', None):
    display_limited(substance_candidate_distribution)
print('Detailed candidate source waves:')
display_limited(substance_detailed_source_summary)
print('Detailed representations across waves:')
display_limited(detailed_wave_comparison)
print('Binary and detailed smoking comparison:')
display_limited(smoking_binary_detailed_summary)
print('Binary lifetime and recent alcohol comparison:')
display_limited(alcohol_binary_detailed_summary)
print('Associations among proposed substance predictors:')
display_limited(substance_candidate_pairwise_comparison)
print('Joint substance-predictor availability:')
display_limited(substance_candidate_joint_availability)

Detailed substance candidate coverage:


,Predictor,Non-missing,Missing,Missing percentage,Distinct observed categories
0,smoking_activity_pretransition,9502,265,2.71,4
1,alcohol_use_frequency_pretransition,9487,280,2.87,4
2,cannabis_experience_pretransition,9513,254,2.60,2


Detailed substance candidate distributions:


,Predictor,Code,Category,Participants,Percentage of full sample,Percentage among observed
0,smoking_activity_pretransition,0,Never smoked,7472,76.50,78.64
1,smoking_activity_pretransition,1,Tried once or formerly smoked,478,4.89,5.03
2,smoking_activity_pretransition,2,Current occasional smoking,406,4.16,4.27
3,smoking_activity_pretransition,3,Current regular smoking,1146,11.73,12.06
4,smoking_activity_pretransition,<NA>,Unavailable,265,2.71,<NA>


Detailed candidate source waves:


...

,Complete comparisons,Never had alcohol and no recent use,Ever had alcohol and recent use,Ever had alcohol but no recent use,Never had alcohol but recent use
0,9487,3178,6259,50,0


Associations among proposed substance predictors:


,First predictor,Second predictor,Complete comparisons,Cramer's V
0,smoking_activity_pretransition,alcohol_use_frequency_pretransition,9478,0.232
1,smoking_activity_pretransition,cannabis_experience_pretransition,9502,0.499
2,alcohol_use_frequency_pretransition,cannabis_experience_pretransition,9487,0.441


Joint substance-predictor availability:


,Substance predictors available,Participants,Percentage
0,0,254,2.60
1,1,2,0.02
2,2,33,0.34
3,3,9478,97.04


In [308]:
# 12: Substance-use predictor decisions and variable-level check

import pandas as pd
substance_predictor_candidates = pd.concat([smoking_activity_pretransition_candidate,
    alcohol_use_frequency_pretransition_candidate, cannabis_experience_pretransition_candidate], axis=1)
assert len(substance_predictor_candidates) == len(route_index)
assert substance_predictor_candidates.index.equals(route_index)
assert list(substance_predictor_candidates.columns) == ['smoking_activity_pretransition',
    'alcohol_use_frequency_pretransition', 'cannabis_experience_pretransition']
substance_predictor_summary_rows = []
for predictor in substance_predictor_candidates.columns:
    values = substance_predictor_candidates[predictor]
    substance_predictor_summary_rows.append({'Predictor': predictor, 'Non-missing': int(values.notna().sum()),
        'Missing': int(values.isna().sum()), 'Missing percentage': round(values.isna().mean() * 100,
        2), 'Observed categories': int(values.nunique())})
substance_predictor_summary = pd.DataFrame(substance_predictor_summary_rows)
substance_predictor_availability_count = substance_predictor_candidates.notna().sum(axis=1)
substance_predictor_joint_availability = substance_predictor_availability_count.value_counts().sort_index().rename('Participants').rename_axis('Substance predictors available').reset_index()
substance_predictor_joint_availability['Percentage'] = (substance_predictor_joint_availability['Participants'] / len(route_index) * 100).round(2)
assert int(substance_predictor_availability_count.eq(3).sum()) == 9478
substance_variable_decisions = pd.concat([domain_8_substance_inventory, domain_8_alcohol_inventory], ignore_index=True,
    sort=False).sort_values(['Source order', 'Variable position']).reset_index(drop=True)
assert len(substance_variable_decisions) == 16
assert substance_variable_decisions[['Source file', 'Variable']].duplicated().sum() == 0
substance_variable_decisions['Substance-track decision'] = pd.Series(pd.NA, index=substance_variable_decisions.index,
    dtype='string')
substance_variable_decisions['Substance-track representation'] = pd.Series(pd.NA,
    index=substance_variable_decisions.index, dtype='string')
substance_variable_decisions['Substance-track reason'] = pd.Series(pd.NA, index=substance_variable_decisions.index,
    dtype='string')

def assign_substance_decision(variables, decision, representation, reason):
    """Assign one decision to specified substance variables."""
    decision_mask = substance_variable_decisions['Variable'].isin(variables)
    assert int(decision_mask.sum()) == len(variables)
    assert substance_variable_decisions.loc[decision_mask, 'Substance-track decision'].isna().all()
    substance_variable_decisions.loc[decision_mask, 'Substance-track decision'] = decision
    substance_variable_decisions.loc[decision_mask, 'Substance-track representation'] = representation
    substance_variable_decisions.loc[decision_mask, 'Substance-track reason'] = reason
assign_substance_decision(variables=smoking_status_variables + smoking_frequency_variables,
    decision='Construction input', representation='smoking_activity_pretransition', reason='Direct smoking status distinguishes participants who have never smoked, while the routed frequency item separates former or experimental, occasional and regular smoking. Wave 3 is restricted to interviews before September 2006, with Wave 2 and Wave 1 used as fallbacks')
assign_substance_decision(variables=alcohol_ever_status_variables + alcohol_recent_status_variables + alcohol_frequency_variables,
    decision='Construction input', representation='alcohol_use_frequency_pretransition', reason='Lifetime status identifies participants routed away from the frequency item, while recent status and frequency distinguish no use, less-than-monthly, monthly and weekly or more frequent alcohol use. Wave 3 is restricted to interviews before September 2006')
assign_substance_decision(variables=cannabis_experience_variables, decision='Construction input',
    representation='cannabis_experience_pretransition', reason='Repeated direct young-person report of whether cannabis had ever been tried. Wave 3 is restricted to interviews before September 2006, with Wave 2 and Wave 1 used as fallbacks')
substance_unresolved_variables = substance_variable_decisions.loc[substance_variable_decisions['Substance-track decision'].isna()].copy()
assert len(substance_unresolved_variables) == 0
assert substance_variable_decisions['Substance-track decision'].value_counts().to_dict() == {'Construction input': 16}
substance_decision_summary = substance_variable_decisions.groupby(['Substance-track decision', 'Review role'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Substance-track decision',
    'Review role']).reset_index(drop=True)
substance_representation_decisions = pd.DataFrame([{'Representation': 'Four-category smoking activity',
    'Decision': 'Retain as predictor candidate', 'Reason': 'Preserves differences between experimental or former, occasional and regular smoking, with adequate category sizes and low missingness'}, {'Representation': 'Binary ever-smoked status',
    'Decision': 'Review support only', 'Reason': 'Provides a simpler coding check but discards current smoking intensity'}, {'Representation': 'Four-category alcohol-use frequency',
    'Decision': 'Retain as predictor candidate', 'Reason': 'Uses comparable previous-12-month frequency categories and distinguishes abstention from different levels of use'}, {'Representation': 'Binary lifetime alcohol experience',
    'Decision': 'Review support only', 'Reason': 'Does not distinguish recent frequency and is less informative than the harmonised frequency representation'}, {'Representation': 'Binary cannabis experience',
    'Decision': 'Retain as predictor candidate', 'Reason': 'Repeated direct measure with high coverage and no comparable repeated frequency measure identified'}, {'Representation': 'Count of reported substance experiences',
    'Decision': 'Do not retain', 'Reason': 'Assigns equal weight to substantively different behaviours and removes distinctions between them'}])
substance_decision_display = substance_variable_decisions[['Review role', 'Wave', 'Source type', 'Source file',
    'Variable', 'Variable label', 'Timing status', 'Substance-track decision', 'Substance-track representation', 'Substance-track reason']].sort_values(['Substance-track representation',
    'Wave', 'Variable']).reset_index(drop=True)
print('Selected substance-use predictors:')
display_limited(substance_predictor_summary)
print('Joint substance-predictor availability:')
display_limited(substance_predictor_joint_availability)
print('Substance variable decision summary:')
display_limited(substance_decision_summary)
print('Substance representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(substance_representation_decisions)
print(f'Unresolved substance variables: {len(substance_unresolved_variables)}')
print('Complete substance variable decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(substance_decision_display)

Selected substance-use predictors:


,Predictor,Non-missing,Missing,Missing percentage,Observed categories
0,smoking_activity_pretransition,9502,265,2.71,4
1,alcohol_use_frequency_pretransition,9487,280,2.87,4
2,cannabis_experience_pretransition,9513,254,2.60,2


Joint substance-predictor availability:


,Substance predictors available,Participants,Percentage
0,0,254,2.60
1,1,2,0.02
2,2,33,0.34
3,3,9478,97.04


Substance variable decision summary:


,Substance-track decision,Review role,Variables
0,Construction input,Direct cannabis-experience status,3
1,Construction input,Direct lifetime alcohol-experience status,3
2,Construction input,Direct recent alcohol-use status,1
3,Construction input,Direct smoking status,3
4,Construction input,Routed alcohol-use frequency,3


Substance representation decisions:


,Representation,Decision,Reason
0,Four-category smoking activity,Retain as predictor candidate,"Preserves differences between experimental or former, occasional and regular smoking, with adequate category sizes and low missingness"
1,Binary ever-smoked status,Review support only,Provides a simpler coding check but discards current smoking intensity
2,Four-category alcohol-use frequency,Retain as predictor candidate,Uses comparable previous-12-month frequency categories and distinguishes abstention from different levels of use
3,Binary lifetime alcohol experience,Review support only,Does not distinguish recent frequency and is less informative than the harmonised frequency representation
4,Binary cannabis experience,Retain as predictor candidate,Repeated direct measure with high coverage and no comparable repeated frequency measure identified


Unresolved substance variables: 0
Complete substance variable decisions:


,Review role,Wave,Source type,Source file,Variable,Variable label,Timing status,Substance-track decision,Substance-track representation,Substance-track reason
0,Direct lifetime alcohol-experience status,Wave 1,Young person,wave_one_lsype_young_person_2020,W1alceverYP,YP: Whether ever had proper alcoholic drink,Pre-transition source,Construction input,alcohol_use_frequency_pretransition,"Lifetime status identifies participants routed away from the frequency item, while recent status and frequency distinguish no use, less-than-monthly, monthly and weekly or more frequent alcohol use. Wave 3 is restricted to interviews before September 2006"
1,Routed alcohol-use frequency,Wave 1,Young person,wave_one_lsype_young_person_2020,W1alcfreqYP,YP: Frequency of having alcoholic drink in last 12 months,Pre-transition source,Construction input,alcohol_use_frequency_pretransition,"Lifetime status identifies participants routed away from the frequency item, while recent status and frequency distinguish no use, less-than-monthly, monthly and weekly or more frequent alcohol use. Wave 3 is restricted to interviews before September 2006"
2,Direct recent alcohol-use status,Wave 1,Young person,wave_one_lsype_young_person_2020,W1alcmonYP,YP: Whether had alcoholic drink in last 12 months,Pre-transition source,Construction input,alcohol_use_frequency_pretransition,"Lifetime status identifies participants routed away from the frequency item, while recent status and frequency distinguish no use, less-than-monthly, monthly and weekly or more frequent alcohol use. Wave 3 is restricted to interviews before September 2006"
3,Direct lifetime alcohol-experience status,Wave 2,Young person,wave_two_lsype_young_person_2020,W2alceverYP,YP: Whether ever had proper alcoholic drink,Pre-transition source,Construction input,alcohol_use_frequency_pretransition,"Lifetime status identifies participants routed away from the frequency item, while recent status and frequency distinguish no use, less-than-monthly, monthly and weekly or more frequent alcohol use. Wave 3 is restricted to interviews before September 2006"
4,Routed alcohol-use frequency,Wave 2,Young person,wave_two_lsype_young_person_2020,W2alcfreqYP,YP: Frequency of having alcoholic drink in last 12 months,Pre-transition source,Construction input,alcohol_use_frequency_pretransition,"Lifetime status identifies participants routed away from the frequency item, while recent status and frequency distinguish no use, less-than-monthly, monthly and weekly or more frequent alcohol use. Wave 3 is restricted to interviews before September 2006"


In [309]:
# 13: Offending, antisocial behaviour and police-contact coding review

import pandas as pd
domain_8_offending_inventory = domain_8_pending_candidates.loc[domain_8_pending_candidates['Matched construct'].str.contains('Offending|police|antisocial',
    case=False, regex=True, na=False)].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(domain_8_offending_inventory) == 15

def classify_offending_role(row):
    """Classify one offending or police-contact variable."""
    searchable_text = f"{row['Variable']} {row['Variable label']}".lower()
    if 'graffit' in searchable_text or 'spray' in searchable_text:
        return 'Graffiti'
    if 'vandalis' in searchable_text or 'vandaliz' in searchable_text or 'public property' in searchable_text or ('smash' in searchable_text):
        return 'Property vandalism'
    if 'shoplift' in searchable_text or 'shoplifting' in searchable_text:
        return 'Shoplifting'
    if 'fight' in searchable_text or 'public disturbance' in searchable_text:
        return 'Fighting or public disturbance'
    if 'police' in searchable_text or 'arrest' in searchable_text or 'caution' in searchable_text or ('charged' in searchable_text):
        return 'Police contact'
    return 'Other offending or antisocial-behaviour measure'
domain_8_offending_inventory['Review role'] = domain_8_offending_inventory.apply(classify_offending_role, axis=1)
domain_8_offending_role_summary = domain_8_offending_inventory.groupby(['Review role', 'Wave'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Review role', 'Wave']).reset_index(drop=True)
offending_raw_tables = {}
offending_labelled_tables = {}
offending_quality_rows = []
offending_code_rows = []
for source_file, source_inventory in domain_8_offending_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    raw_data['NSID'] = standardise_nsid(raw_data['NSID'])
    labelled_data['NSID'] = standardise_nsid(labelled_data['NSID'])
    assert raw_data['NSID'].is_unique
    assert labelled_data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(route_index)
    labelled_data = labelled_data.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    offending_raw_tables[source_file] = raw_data
    offending_labelled_tables[source_file] = labelled_data
    for variable in source_variables:
        variable_inventory = source_inventory.loc[source_inventory['Variable'].eq(variable)].iloc[0]
        raw_values = raw_data[variable]
        labelled_values = labelled_data[variable].astype('string')
        observed_mask = raw_values.ge(0).fillna(False)
        special_code_mask = raw_values.lt(0).fillna(False)
        observed_values = raw_values.loc[observed_mask]
        offending_quality_rows.append({'Review role': variable_inventory['Review role'],
            'Wave': variable_inventory['Wave'], 'Variable': variable, 'Variable label': variable_inventory['Variable label'], 'Observed responses': int(observed_mask.sum()), 'Special-code responses': int(special_code_mask.sum()), 'No source record': int(raw_values.isna().sum()), 'Observed percentage': round(observed_mask.mean() * 100,
            2), 'Distinct observed codes': int(observed_values.nunique()), 'Minimum observed code': observed_values.min() if len(observed_values) > 0 else pd.NA, 'Maximum observed code': observed_values.max() if len(observed_values) > 0 else pd.NA, 'Not-applicable responses': int(raw_values.eq(-91).sum())})
        value_counts = raw_values.value_counts(dropna=False)
        for raw_code, participants in value_counts.items():
            if pd.isna(raw_code):
                value_label = 'No source record'
                response_type = 'No source record'
                sort_value = 999999
            else:
                matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(raw_code)
                response_type = 'Observed response' if raw_code >= 0 else 'Special code'
                sort_value = float(raw_code)
            offending_code_rows.append({'Review role': variable_inventory['Review role'],
                'Wave': variable_inventory['Wave'], 'Variable': variable, 'Variable label': variable_inventory['Variable label'], 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
offending_quality = pd.DataFrame(offending_quality_rows).sort_values(['Review role', 'Wave',
    'Variable']).reset_index(drop=True)
offending_code_distribution = pd.DataFrame(offending_code_rows).sort_values(['Review role', 'Wave', 'Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
domain_8_offending_role_display = domain_8_offending_inventory[['Review role', 'Wave', 'Source type', 'Source file',
    'Variable', 'Variable label', 'Timing status', 'Deferred from earlier domain']].sort_values(['Review role', 'Wave',
    'Variable']).reset_index(drop=True)
print(f'Offending, antisocial-behaviour and police-contact variables reviewed: {len(domain_8_offending_inventory):,}')
print('\nVariable-role summary:')
display_limited(domain_8_offending_role_summary)
print('\nMeasure-quality summary by review role:')
offending_quality_summary = offending_quality.groupby('Review role', dropna=False).agg(Variables=('Variable', 'size'),
    Median_observed_percentage=('Observed percentage',
    'median'), Total_special_code_responses=('Special-code responses',
    'sum'), Total_not_applicable_responses=('Not-applicable responses',
    'sum')).reset_index().rename(columns={'Median_observed_percentage': 'Median observed percentage',
    'Total_special_code_responses': 'Total special-code responses', 'Total_not_applicable_responses': 'Total not-applicable responses'})
display_limited(offending_quality_summary)
print('\nFirst 10 variable-level quality records:')
display_limited(offending_quality[['Review role', 'Wave', 'Variable', 'Observed percentage', 'Special-code responses',
    'Not-applicable responses']].head(10))
print(f'Additional quality records not displayed: {max(len(offending_quality) - 10, 0):,}')
print(f'\nResponse-code distribution rows created: {len(offending_code_distribution):,}')
print(f'Offending, antisocial-behaviour and police-contact variables reviewed: {len(domain_8_offending_inventory):,}')
print('\nVariable-role summary:')
display_limited(domain_8_offending_role_summary)
print('\nMeasure-quality summary by review role:')
offending_quality_summary = offending_quality.groupby('Review role', dropna=False).agg(Variables=('Variable', 'size'),
    Median_observed_percentage=('Observed percentage',
    'median'), Total_special_code_responses=('Special-code responses',
    'sum'), Total_not_applicable_responses=('Not-applicable responses',
    'sum')).reset_index().rename(columns={'Median_observed_percentage': 'Median observed percentage',
    'Total_special_code_responses': 'Total special-code responses', 'Total_not_applicable_responses': 'Total not-applicable responses'})
display_limited(offending_quality_summary)
print('\nFirst 10 variable-level quality records:')
display_limited(offending_quality[['Review role', 'Wave', 'Variable', 'Observed percentage', 'Special-code responses',
    'Not-applicable responses']].head(10))
print(f'Additional quality records not displayed: {max(len(offending_quality) - 10, 0):,}')
print(f'\nResponse-code distribution rows created: {len(offending_code_distribution):,}')
offending_quality_file = stage_2_output_directory / 'stage_2_offending_measure_quality.csv'
offending_code_distribution_file = stage_2_output_directory / 'stage_2_offending_code_distribution.csv'
offending_role_audit_file = stage_2_output_directory / 'stage_2_offending_variable_role_audit.csv'
offending_quality.to_csv(offending_quality_file, index=False)
offending_code_distribution.to_csv(offending_code_distribution_file, index=False)
domain_8_offending_role_display.to_csv(offending_role_audit_file, index=False)
print(f'\nFull review tables saved to: {stage_2_output_directory}')

Offending, antisocial-behaviour and police-contact variables reviewed: 15

Variable-role summary:


,Review role,Wave,Variables
0,Fighting or public disturbance,Wave 1,1
1,Fighting or public disturbance,Wave 2,1
2,Fighting or public disturbance,Wave 3,1
3,Police contact,Wave 1,2
4,Police contact,Wave 2,2



Measure-quality summary by review role:


,Review role,Variables,Median observed percentage,Total special-code responses,Total not-applicable responses
0,Fighting or public disturbance,3,94.080,968,0
1,Police contact,6,46.525,29844,24836
2,Property vandalism,3,94.160,935,0
3,Shoplifting,3,94.510,841,0



First 10 variable-level quality records:


...


Measure-quality summary by review role:


,Review role,Variables,Median observed percentage,Total special-code responses,Total not-applicable responses
0,Fighting or public disturbance,3,94.080,968,0
1,Police contact,6,46.525,29844,24836
2,Property vandalism,3,94.160,935,0
3,Shoplifting,3,94.510,841,0



First 10 variable-level quality records:


,Review role,Wave,Variable,Observed percentage,Special-code responses,Not-applicable responses
0,Fighting or public disturbance,Wave 1,W1fightYP,93.29,412,0
1,Fighting or public disturbance,Wave 2,W2fightYP,94.08,332,0
2,Fighting or public disturbance,Wave 3,W3fightYP,95.07,224,0
3,Police contact,Wave 1,W1Police1MP,88.57,873,0
4,Police contact,Wave 1,W1police2MP,4.48,9086,8217


Additional quality records not displayed: 5

Response-code distribution rows created: 152

Full review tables saved to: data_derived\stage_2_predictor_construction


In [310]:
# 14: Graffiti-variable review and offending-inventory expansion

import pandas as pd
graffiti_variables = ['W1sprayYP', 'W2sprayYP', 'W3sprayYP']
domain_8_graffiti_inventory = verified_decision_register.loc[verified_decision_register['Variable'].isin(graffiti_variables)].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(domain_8_graffiti_inventory) == 3
assert set(domain_8_graffiti_inventory['Variable']) == set(graffiti_variables)
assert domain_8_graffiti_inventory[['Source file', 'Variable']].duplicated().sum() == 0
domain_8_graffiti_inventory['Review role'] = 'Graffiti'
graffiti_raw_tables = {}
graffiti_labelled_tables = {}
graffiti_quality_rows = []
graffiti_code_rows = []
for source_file, source_inventory in domain_8_graffiti_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    raw_data['NSID'] = standardise_nsid(raw_data['NSID'])
    labelled_data['NSID'] = standardise_nsid(labelled_data['NSID'])
    assert raw_data['NSID'].is_unique
    assert labelled_data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(route_index)
    labelled_data = labelled_data.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    graffiti_raw_tables[source_file] = raw_data
    graffiti_labelled_tables[source_file] = labelled_data
    for variable in source_variables:
        variable_inventory = source_inventory.loc[source_inventory['Variable'].eq(variable)].iloc[0]
        raw_values = raw_data[variable]
        labelled_values = labelled_data[variable].astype('string')
        observed_mask = raw_values.ge(0).fillna(False)
        special_code_mask = raw_values.lt(0).fillna(False)
        observed_values = raw_values.loc[observed_mask]
        graffiti_quality_rows.append({'Review role': 'Graffiti', 'Wave': variable_inventory['Wave'],
            'Variable': variable, 'Variable label': variable_inventory['Variable label'], 'Observed responses': int(observed_mask.sum()), 'Special-code responses': int(special_code_mask.sum()), 'No source record': int(raw_values.isna().sum()), 'Observed percentage': round(observed_mask.mean() * 100,
            2), 'Distinct observed codes': int(observed_values.nunique()), 'Minimum observed code': observed_values.min() if len(observed_values) > 0 else pd.NA, 'Maximum observed code': observed_values.max() if len(observed_values) > 0 else pd.NA, 'Not-applicable responses': int(raw_values.eq(-91).sum())})
        value_counts = raw_values.value_counts(dropna=False)
        for raw_code, participants in value_counts.items():
            if pd.isna(raw_code):
                value_label = 'No source record'
                response_type = 'No source record'
                sort_value = 999999
            else:
                matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(raw_code)
                response_type = 'Observed response' if raw_code >= 0 else 'Special code'
                sort_value = float(raw_code)
            graffiti_code_rows.append({'Review role': 'Graffiti', 'Wave': variable_inventory['Wave'],
                'Variable': variable, 'Variable label': variable_inventory['Variable label'], 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
graffiti_quality = pd.DataFrame(graffiti_quality_rows).sort_values(['Wave', 'Variable']).reset_index(drop=True)
graffiti_code_distribution = pd.DataFrame(graffiti_code_rows).sort_values(['Wave', 'Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
assert set(graffiti_quality['Distinct observed codes']) == {2}
assert set(graffiti_quality['Minimum observed code']) == {1.0}
assert set(graffiti_quality['Maximum observed code']) == {2.0}
domain_8_offending_inventory_expanded = pd.concat([domain_8_offending_inventory, domain_8_graffiti_inventory],
    ignore_index=True, sort=False).sort_values(['Source order', 'Variable position']).reset_index(drop=True)
assert len(domain_8_offending_inventory_expanded) == 18
assert domain_8_offending_inventory_expanded[['Source file', 'Variable']].duplicated().sum() == 0
expected_offending_role_counts = {'Graffiti': 3, 'Property vandalism': 3, 'Shoplifting': 3,
    'Fighting or public disturbance': 3, 'Police contact': 6}
actual_offending_role_counts = domain_8_offending_inventory_expanded['Review role'].value_counts().to_dict()
assert actual_offending_role_counts == expected_offending_role_counts
domain_8_offending_expanded_role_summary = domain_8_offending_inventory_expanded.groupby(['Review role', 'Wave'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Review role', 'Wave']).reset_index(drop=True)
domain_8_offending_expanded_display = domain_8_offending_inventory_expanded[['Review role', 'Wave', 'Source type',
    'Source file', 'Variable', 'Variable label', 'Timing status', 'Review status', 'Review outcome', 'Substantive domain']].sort_values(['Review role',
    'Wave', 'Variable']).reset_index(drop=True)
print(f'Graffiti variables added: {len(domain_8_graffiti_inventory)}')
print('Graffiti measure quality:')
display_limited(graffiti_quality)
print('Graffiti response codes:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(graffiti_code_distribution)
print(f'Expanded offending and police-contact variables: {len(domain_8_offending_inventory_expanded)}')
print('Expanded variable-role summary:')
display_limited(domain_8_offending_expanded_role_summary)
print('Complete expanded variable-role audit:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_8_offending_expanded_display)

Graffiti variables added: 3
Graffiti measure quality:


,Review role,Wave,Variable,Variable label,Observed responses,Special-code responses,No source record,Observed percentage,Distinct observed codes,Minimum observed code,Maximum observed code,Not-applicable responses
0,Graffiti,Wave 1,W1sprayYP,YP: Whether ever graffittied on walls,9262,262,243,94.83,2,1.0,2.0,0
1,Graffiti,Wave 2,W2sprayYP,YP: Whether graffittied on walls in the last y...,9269,252,246,94.90,2,1.0,2.0,0
2,Graffiti,Wave 3,W3sprayYP,YP: Whether graffitied on walls in the last year,9337,172,258,95.60,2,1.0,2.0,0


Graffiti response codes:


,Review role,Wave,Variable,Variable label,Raw code,Value label,Response type,Participants
0,Graffiti,Wave 1,W1sprayYP,YP: Whether ever graffittied on walls,-99.0,YP not interviewed,Special code,89
1,Graffiti,Wave 1,W1sprayYP,YP: Whether ever graffittied on walls,-97.0,YP refused CASI section,Special code,66
2,Graffiti,Wave 1,W1sprayYP,YP: Whether ever graffittied on walls,-96.0,YP unable to complete CASI section,Special code,49
3,Graffiti,Wave 1,W1sprayYP,YP: Whether ever graffittied on walls,-92.0,Refused,Special code,37
4,Graffiti,Wave 1,W1sprayYP,YP: Whether ever graffittied on walls,-1.0,Don't know,Special code,21


Expanded offending and police-contact variables: 18
Expanded variable-role summary:


,Review role,Wave,Variables
0,Fighting or public disturbance,Wave 1,1
1,Fighting or public disturbance,Wave 2,1
2,Fighting or public disturbance,Wave 3,1
3,Graffiti,Wave 1,1
4,Graffiti,Wave 2,1


Complete expanded variable-role audit:


,Review role,Wave,Source type,Source file,Variable,Variable label,Timing status,Review status,Review outcome,Substantive domain
0,Fighting or public disturbance,Wave 1,Young person,wave_one_lsype_young_person_2020,W1fightYP,YP: Whether ever taken part in fighting or public disturbance,Pre-transition source,"Variable-level coding, routing and reference-period review required.",Pending review,NaN
1,Fighting or public disturbance,Wave 2,Young person,wave_two_lsype_young_person_2020,W2fightYP,YP: Whether taken part in fighting or public disturbance in the last year,Pre-transition source,"Variable-level coding, routing and reference-period review required.",Pending review,NaN
2,Fighting or public disturbance,Wave 3,Young person,wave_three_lsype_young_person_2020,W3fightYP,YP: Whether taken part in fighting or public disturbance in the last year?,Near-transition source,Use requires interview timing confirming January–August 2006 and item-level reference-period review.,Pending review,NaN
3,Graffiti,Wave 1,Young person,wave_one_lsype_young_person_2020,W1sprayYP,YP: Whether ever graffittied on walls,Pre-transition source,"Variable-level coding, routing and reference-period review required.",Pending review,NaN
4,Graffiti,Wave 2,Young person,wave_two_lsype_young_person_2020,W2sprayYP,YP: Whether graffittied on walls in the last year,Pre-transition source,"Variable-level coding, routing and reference-period review required.",Pending review,NaN


In [311]:
# 15: Recent antisocial behaviour and police-contact representation review

import numpy as np
import pandas as pd
antisocial_behaviour_variables = {'Graffiti': {'Wave 1': 'W1sprayYP', 'Wave 2': 'W2sprayYP', 'Wave 3': 'W3sprayYP'},
    'Property vandalism': {'Wave 1': 'W1smashYP', 'Wave 2': 'W2smashYP',
    'Wave 3': 'W3smashYP'}, 'Shoplifting': {'Wave 1': 'W1shopYP', 'Wave 2': 'W2shopYP',
    'Wave 3': 'W3shopYP'}, 'Fighting or public disturbance': {'Wave 1': 'W1fightYP', 'Wave 2': 'W2fightYP',
    'Wave 3': 'W3fightYP'}}
police_status_variables = {'Wave 1': 'W1Police1MP', 'Wave 2': 'W2police1MP', 'Wave 3': 'W3police1MP'}
police_count_variables = {'Wave 1': 'W1police2MP', 'Wave 2': 'W2Police2MP', 'Wave 3': 'W3police2MP'}

def get_offending_raw_variable(variable):
    """Return one aligned offending or police-contact variable."""
    source_file = domain_8_offending_inventory_expanded.loc[domain_8_offending_inventory_expanded['Variable'].eq(variable),
        'Source file'].iloc[0]
    if source_file in graffiti_raw_tables and variable in graffiti_raw_tables[source_file].columns:
        return graffiti_raw_tables[source_file][variable].copy()
    return offending_raw_tables[source_file][variable].copy()

def recode_offending_yes_no(variable):
    """Recode 1 = yes and 2 = no to binary form."""
    return get_offending_raw_variable(variable).map({1.0: 1.0, 2.0: 0.0}).astype('Float64')
antisocial_behaviour_by_wave = {}
for wave in ['Wave 1', 'Wave 2', 'Wave 3']:
    wave_data = pd.DataFrame(index=route_index)
    for construct, variable_lookup in antisocial_behaviour_variables.items():
        wave_data[construct] = recode_offending_yes_no(variable_lookup[wave])
    if wave == 'Wave 3':
        wave_data = wave_data.where(wave_3_pretransition_substance_mask)
    antisocial_behaviour_by_wave[wave] = wave_data
antisocial_item_summary_rows = []
for wave, wave_data in antisocial_behaviour_by_wave.items():
    reference_period = 'Lifetime' if wave == 'Wave 1' else 'Previous year'
    for construct in wave_data.columns:
        values = wave_data[construct]
        antisocial_item_summary_rows.append({'Wave': wave, 'Reference period': reference_period,
            'Behaviour': construct, 'Non-missing': int(values.notna().sum()), 'Yes': int(values.eq(1).sum()), 'Yes percentage among observed': round(values.eq(1).sum() / values.notna().sum() * 100,
            2)})
antisocial_item_summary = pd.DataFrame(antisocial_item_summary_rows)
antisocial_count_by_wave = pd.DataFrame({wave: wave_data.sum(axis=1, min_count=4).astype('Float64') for wave,
    wave_data in antisocial_behaviour_by_wave.items()}, index=route_index)
latest_recent_antisocial_count, latest_recent_antisocial_count_source = create_latest_representation(antisocial_count_by_wave[['Wave 2',
    'Wave 3']], ['Wave 3', 'Wave 2'])

def categorise_antisocial_count(values):
    """Group behaviour counts as none, one, or two or more."""
    category = pd.Series(pd.NA, index=values.index, dtype='Float64')
    category.loc[values.eq(0)] = 0.0
    category.loc[values.eq(1)] = 1.0
    category.loc[values.ge(2)] = 2.0
    return category
antisocial_category_by_wave = pd.DataFrame({wave: categorise_antisocial_count(antisocial_count_by_wave[wave]) for wave in ['Wave 1',
    'Wave 2', 'Wave 3']}, index=route_index)
latest_recent_antisocial_category = categorise_antisocial_count(latest_recent_antisocial_count).rename('recent_antisocial_behaviour_pretransition')
antisocial_category_labels = {0.0: 'No reported behaviour', 1.0: 'One reported behaviour',
    2.0: 'Two or more reported behaviours'}
recent_antisocial_distribution = latest_recent_antisocial_category.map(antisocial_category_labels).fillna('Unavailable').value_counts().rename('Participants').rename_axis('Recent antisocial behaviour').reset_index()
recent_antisocial_distribution['Percentage'] = (recent_antisocial_distribution['Participants'] / len(route_index) * 100).round(2)
recent_antisocial_source_summary = latest_recent_antisocial_count_source.value_counts().rename('Participants').rename_axis('Source').reset_index()
recent_antisocial_source_summary['Percentage'] = (recent_antisocial_source_summary['Participants'] / len(route_index) * 100).round(2)
antisocial_wave_comparison_rows = []
for first_wave, second_wave, comparison_note in [('Wave 1', 'Wave 2', 'Lifetime versus previous-year measure'),
    ('Wave 2', 'Wave 3', 'Previous-year measures'), ('Wave 1', 'Wave 3', 'Lifetime versus previous-year measure')]:
    pair_data = antisocial_category_by_wave[[first_wave, second_wave]].dropna()
    antisocial_wave_comparison_rows.append({'First wave': first_wave, 'Second wave': second_wave,
        'Comparison': comparison_note, 'Complete comparisons': len(pair_data), 'Exact agreement percentage': round(pair_data[first_wave].eq(pair_data[second_wave]).mean() * 100,
        2), "Cramer's V": round(categorical_cramers_v(pair_data[first_wave], pair_data[second_wave]), 3)})
antisocial_wave_comparison = pd.DataFrame(antisocial_wave_comparison_rows)
police_status_raw_by_wave = pd.DataFrame({wave: get_offending_raw_variable(variable) for wave,
    variable in police_status_variables.items()}, index=route_index)
police_count_raw_by_wave = pd.DataFrame({wave: get_offending_raw_variable(variable) for wave,
    variable in police_count_variables.items()}, index=route_index)
police_status_raw_by_wave['Wave 3'] = police_status_raw_by_wave['Wave 3'].where(wave_3_pretransition_substance_mask)
police_count_raw_by_wave['Wave 3'] = police_count_raw_by_wave['Wave 3'].where(wave_3_pretransition_substance_mask)
police_routing_rows = []
for wave in ['Wave 1', 'Wave 2', 'Wave 3']:
    status_raw = police_status_raw_by_wave[wave]
    count_raw = police_count_raw_by_wave[wave]
    if wave == 'Wave 1':
        status_labels = status_raw.map({1.0: 'Contact in last three years', 2.0: 'No police contact',
            3.0: 'Contact more than three years ago'}).fillna('Status unavailable')
    else:
        status_labels = status_raw.map({1.0: 'Police contact', 2.0: 'No police contact'}).fillna('Status unavailable')
    count_labels = count_raw.apply(lambda value: 'Positive contact count' if pd.notna(value) and value > 0 else 'Recorded zero' if value == 0 else 'Structurally not applicable' if value == -91 else 'Other unavailable response')
    routing_counts = pd.DataFrame({'Police-contact status': status_labels,
        'Count response': count_labels}).value_counts().rename('Participants').reset_index()
    routing_counts['Wave'] = wave
    police_routing_rows.append(routing_counts)
police_routing_summary = pd.concat(police_routing_rows, ignore_index=True)[['Wave', 'Police-contact status',
    'Count response', 'Participants']].sort_values(['Wave', 'Police-contact status',
    'Count response']).reset_index(drop=True)
police_routing_contradiction_rows = []
for wave in ['Wave 1', 'Wave 2', 'Wave 3']:
    status_raw = police_status_raw_by_wave[wave]
    count_raw = police_count_raw_by_wave[wave]
    yes_code_mask = status_raw.eq(1)
    no_contact_mask = status_raw.eq(2)
    police_routing_contradiction_rows.append({'Wave': wave,
        'Contact reported but count is zero': int((yes_code_mask & count_raw.eq(0)).sum()), 'Contact reported but count not applicable': int((yes_code_mask & count_raw.eq(-91)).sum()), 'No contact reported but positive count': int((no_contact_mask & count_raw.gt(0)).sum())})
police_routing_contradiction_summary = pd.DataFrame(police_routing_contradiction_rows)
police_contact_binary_by_wave = pd.DataFrame({'Wave 2': police_status_raw_by_wave['Wave 2'].map({1.0: 1.0,
    2.0: 0.0}).astype('Float64'), 'Wave 3': police_status_raw_by_wave['Wave 3'].map({1.0: 1.0,
    2.0: 0.0}).astype('Float64')}, index=route_index)
police_contact_frequency_by_wave = pd.DataFrame(index=route_index)
for wave in ['Wave 2', 'Wave 3']:
    status_raw = police_status_raw_by_wave[wave]
    count_raw = police_count_raw_by_wave[wave]
    frequency_category = pd.Series(pd.NA, index=route_index, dtype='Float64')
    frequency_category.loc[status_raw.eq(2)] = 0.0
    frequency_category.loc[status_raw.eq(1) & count_raw.eq(1)] = 1.0
    frequency_category.loc[status_raw.eq(1) & count_raw.ge(2)] = 2.0
    police_contact_frequency_by_wave[wave] = frequency_category
latest_recent_police_contact, latest_recent_police_contact_source = create_latest_representation(police_contact_binary_by_wave,
    ['Wave 3', 'Wave 2'])
latest_recent_police_frequency, latest_recent_police_frequency_source = create_latest_representation(police_contact_frequency_by_wave,
    ['Wave 3', 'Wave 2'])
recent_police_binary_summary = pd.DataFrame([{'Representation': 'Recent police-contact status',
    'Non-missing': int(latest_recent_police_contact.notna().sum()), 'Missing': int(latest_recent_police_contact.isna().sum()), 'No contact': int(latest_recent_police_contact.eq(0).sum()), 'Contact': int(latest_recent_police_contact.eq(1).sum()), 'Contact percentage among observed': round(latest_recent_police_contact.eq(1).sum() / latest_recent_police_contact.notna().sum() * 100,
    2)}])
police_frequency_labels = {0.0: 'No police contact', 1.0: 'Police contact once',
    2.0: 'Police contact two or more times'}
recent_police_frequency_summary = latest_recent_police_frequency.map(police_frequency_labels).fillna('Unavailable').value_counts().rename('Participants').rename_axis('Recent police-contact frequency').reset_index()
recent_police_frequency_summary['Percentage'] = (recent_police_frequency_summary['Participants'] / len(route_index) * 100).round(2)
offending_police_comparison_rows = []
for police_name, police_values in [('Recent police-contact status', latest_recent_police_contact),
    ('Recent police-contact frequency', latest_recent_police_frequency)]:
    pair_data = pd.DataFrame({'Antisocial behaviour': latest_recent_antisocial_category,
        'Police contact': police_values}).dropna()
    offending_police_comparison_rows.append({'Police representation': police_name,
        'Complete comparisons': len(pair_data), "Cramer's V": round(categorical_cramers_v(pair_data['Antisocial behaviour'],
        pair_data['Police contact']), 3)})
offending_police_comparison = pd.DataFrame(offending_police_comparison_rows)
print('Young-person antisocial-behaviour items:')
display_limited(antisocial_item_summary)
print('Latest recent antisocial-behaviour category:')
display_limited(recent_antisocial_distribution)
print('Recent antisocial-behaviour source waves:')
display_limited(recent_antisocial_source_summary)
print('Antisocial-behaviour category across waves:')
display_limited(antisocial_wave_comparison)
print('Police-status and count routing:')
with pd.option_context('display.max_rows', None):
    display_limited(police_routing_summary)
print('Police routing contradictions:')
display_limited(police_routing_contradiction_summary)
print('Recent police-contact status:')
display_limited(recent_police_binary_summary)
print('Recent police-contact frequency:')
display_limited(recent_police_frequency_summary)
print('Antisocial behaviour and police contact:')
display_limited(offending_police_comparison)

Young-person antisocial-behaviour items:


,Wave,Reference period,Behaviour,Non-missing,Yes,Yes percentage among observed
0,Wave 1,Lifetime,Graffiti,9262,455,4.91
1,Wave 1,Lifetime,Property vandalism,9137,768,8.41
2,Wave 1,Lifetime,Shoplifting,9152,895,9.78
3,Wave 1,Lifetime,Fighting or public disturbance,9112,1403,15.40
4,Wave 2,Previous year,Graffiti,9269,381,4.11


Latest recent antisocial-behaviour category:


,Recent antisocial behaviour,Participants,Percentage
0,No reported behaviour,7698,78.82
1,One reported behaviour,1125,11.52
2,Two or more reported behaviours,620,6.35
3,Unavailable,324,3.32


Recent antisocial-behaviour source waves:


...

,Representation,Non-missing,Missing,No contact,Contact,Contact percentage among observed
0,Recent police-contact status,9022,745,8598,424,4.7


Recent police-contact frequency:


,Recent police-contact frequency,Participants,Percentage
0,No police contact,8608,88.13
1,Unavailable,747,7.65
2,Police contact once,303,3.10
3,Police contact two or more times,109,1.12


Antisocial behaviour and police contact:


,Police representation,Complete comparisons,Cramer's V
0,Recent police-contact status,8949,0.250
1,Recent police-contact frequency,8947,0.176


In [312]:
# 16: Antisocial-behaviour coherence and police-contact stability review

import numpy as np
import pandas as pd
antisocial_pairwise_rows = []
for wave in ['Wave 2', 'Wave 3']:
    wave_data = antisocial_behaviour_by_wave[wave]
    behaviour_names = wave_data.columns.tolist()
    for first_position in range(len(behaviour_names)):
        for second_position in range(first_position + 1, len(behaviour_names)):
            first_behaviour = behaviour_names[first_position]
            second_behaviour = behaviour_names[second_position]
            pair_data = wave_data[[first_behaviour, second_behaviour]].dropna()
            antisocial_pairwise_rows.append({'Wave': wave, 'First behaviour': first_behaviour,
                'Second behaviour': second_behaviour, 'Complete comparisons': len(pair_data), "Cramer's V": round(binary_cramers_v(pair_data[first_behaviour],
                pair_data[second_behaviour]), 3)})
antisocial_pairwise_comparison = pd.DataFrame(antisocial_pairwise_rows)

def cronbach_alpha(data):
    """Calculate Cronbach's alpha for complete rows."""
    complete_data = data.dropna().astype(float)
    item_count = complete_data.shape[1]
    if item_count < 2 or len(complete_data) < 2:
        return (np.nan, len(complete_data))
    item_variance_sum = complete_data.var(axis=0, ddof=1).sum()
    total_score_variance = complete_data.sum(axis=1).var(ddof=1)
    if total_score_variance == 0:
        return (np.nan, len(complete_data))
    alpha = item_count / (item_count - 1) * (1 - item_variance_sum / total_score_variance)
    return (float(alpha), len(complete_data))
antisocial_reliability_rows = []
for wave in ['Wave 2', 'Wave 3']:
    alpha, complete_cases = cronbach_alpha(antisocial_behaviour_by_wave[wave])
    antisocial_reliability_rows.append({'Wave': wave, 'Items': 4, 'Complete cases': complete_cases,
        "Cronbach's alpha": round(alpha,
        3), 'Mean behaviour count': round(antisocial_behaviour_by_wave[wave].sum(axis=1, min_count=4).mean(), 3)})
antisocial_reliability_summary = pd.DataFrame(antisocial_reliability_rows)
antisocial_count_component_rows = []
for wave in ['Wave 2', 'Wave 3']:
    wave_data = antisocial_behaviour_by_wave[wave]
    wave_count = antisocial_count_by_wave[wave]
    for behaviour in wave_data.columns:
        complete_data = pd.DataFrame({'Behaviour': wave_data[behaviour], 'Count': wave_count}).dropna()
        antisocial_count_component_rows.append({'Wave': wave, 'Behaviour': behaviour,
            'Complete comparisons': len(complete_data), 'Spearman correlation with count': round(complete_data['Behaviour'].corr(complete_data['Count'],
            method='spearman'), 3)})
antisocial_count_component_summary = pd.DataFrame(antisocial_count_component_rows)
police_status_wave_complete = police_contact_binary_by_wave.dropna()
police_status_wave_comparison = pd.DataFrame([{'Complete comparisons': len(police_status_wave_complete),
    'Exact agreement percentage': round(police_status_wave_complete['Wave 2'].eq(police_status_wave_complete['Wave 3']).mean() * 100,
    2), "Cramer's V": round(binary_cramers_v(police_status_wave_complete['Wave 2'],
    police_status_wave_complete['Wave 3']), 3), 'Contact in both waves': int((police_status_wave_complete['Wave 2'].eq(1) & police_status_wave_complete['Wave 3'].eq(1)).sum()), 'Contact in Wave 2 only': int((police_status_wave_complete['Wave 2'].eq(1) & police_status_wave_complete['Wave 3'].eq(0)).sum()), 'Contact in Wave 3 only': int((police_status_wave_complete['Wave 2'].eq(0) & police_status_wave_complete['Wave 3'].eq(1)).sum())}])
police_frequency_wave_complete = police_contact_frequency_by_wave.dropna()
police_frequency_wave_comparison = pd.DataFrame([{'Complete comparisons': len(police_frequency_wave_complete),
    'Exact agreement percentage': round(police_frequency_wave_complete['Wave 2'].eq(police_frequency_wave_complete['Wave 3']).mean() * 100,
    2), "Cramer's V": round(categorical_cramers_v(police_frequency_wave_complete['Wave 2'],
    police_frequency_wave_complete['Wave 3']), 3)}])
police_binary_frequency_complete = pd.DataFrame({'Police-contact status': latest_recent_police_contact,
    'Police-contact frequency': latest_recent_police_frequency}).dropna()
police_binary_frequency_summary = pd.DataFrame([{'Complete comparisons': len(police_binary_frequency_complete),
    'Binary no and frequency none': int((police_binary_frequency_complete['Police-contact status'].eq(0) & police_binary_frequency_complete['Police-contact frequency'].eq(0)).sum()), 'Binary yes and frequency positive': int((police_binary_frequency_complete['Police-contact status'].eq(1) & police_binary_frequency_complete['Police-contact frequency'].gt(0)).sum()), 'Contradictory classifications': int((police_binary_frequency_complete['Police-contact status'].eq(0) & police_binary_frequency_complete['Police-contact frequency'].gt(0)).sum() + (police_binary_frequency_complete['Police-contact status'].eq(1) & police_binary_frequency_complete['Police-contact frequency'].eq(0)).sum())}])
domain_8_behaviour_comparison_data = pd.DataFrame({'Antisocial behaviour': latest_recent_antisocial_category,
    'Police contact': latest_recent_police_contact, 'Smoking activity': smoking_activity_pretransition_candidate, 'Alcohol-use frequency': alcohol_use_frequency_pretransition_candidate, 'Cannabis experience': cannabis_experience_pretransition_candidate, 'Truancy status': truancy_status_pretransition_candidate}, index=route_index)
domain_8_behaviour_pairwise_rows = []
for first_name, second_name in [('Antisocial behaviour', 'Police contact'), ('Antisocial behaviour',
    'Smoking activity'), ('Antisocial behaviour', 'Alcohol-use frequency'), ('Antisocial behaviour',
    'Cannabis experience'), ('Antisocial behaviour', 'Truancy status'), ('Police contact',
    'Smoking activity'), ('Police contact', 'Cannabis experience'), ('Police contact', 'Truancy status')]:
    pair_data = domain_8_behaviour_comparison_data[[first_name, second_name]].dropna()
    domain_8_behaviour_pairwise_rows.append({'First measure': first_name, 'Second measure': second_name,
        'Complete comparisons': len(pair_data), "Cramer's V": round(categorical_cramers_v(pair_data[first_name],
        pair_data[second_name]), 3)})
domain_8_behaviour_pairwise_comparison = pd.DataFrame(domain_8_behaviour_pairwise_rows)
print('Pairwise associations among antisocial-behaviour items:')
display_limited(antisocial_pairwise_comparison)
print('Antisocial-behaviour item reliability:')
display_limited(antisocial_reliability_summary)
print('Association of each item with the behaviour count:')
display_limited(antisocial_count_component_summary)
print('Police-contact status across Waves 2 and 3:')
display_limited(police_status_wave_comparison)
print('Police-contact frequency across Waves 2 and 3:')
display_limited(police_frequency_wave_comparison)
print('Binary and frequency police-contact representations:')
display_limited(police_binary_frequency_summary)
print('Associations with other behaviour measures:')
display_limited(domain_8_behaviour_pairwise_comparison)

Pairwise associations among antisocial-behaviour items:


,Wave,First behaviour,Second behaviour,Complete comparisons,Cramer's V
0,Wave 2,Graffiti,Property vandalism,9157,0.347
1,Wave 2,Graffiti,Shoplifting,9188,0.231
2,Wave 2,Graffiti,Fighting or public disturbance,9139,0.278
3,Wave 2,Property vandalism,Shoplifting,9122,0.307
4,Wave 2,Property vandalism,Fighting or public disturbance,9088,0.353


Antisocial-behaviour item reliability:


,Wave,Items,Complete cases,Cronbach's alpha,Mean behaviour count
0,Wave 2,4,9005,0.607,0.323
1,Wave 3,4,9140,0.613,0.275


Association of each item with the behaviour count:


...

,Complete comparisons,Exact agreement percentage,Cramer's V
0,8297,93.48,0.279


Binary and frequency police-contact representations:


,Complete comparisons,Binary no and frequency none,Binary yes and frequency positive,Contradictory classifications
0,9020,8598,412,10


Associations with other behaviour measures:


,First measure,Second measure,Complete comparisons,Cramer's V
0,Antisocial behaviour,Police contact,8949,0.250
1,Antisocial behaviour,Smoking activity,9434,0.242
2,Antisocial behaviour,Alcohol-use frequency,9421,0.210
3,Antisocial behaviour,Cannabis experience,9443,0.347
4,Antisocial behaviour,Truancy status,9424,0.391


In [313]:
# 17: Antisocial-behaviour and police-contact predictor decisions

import pandas as pd
recent_antisocial_behaviour_candidate = latest_recent_antisocial_category.copy().rename('recent_antisocial_behaviour_pretransition')
recent_police_contact_candidate = latest_recent_police_contact.copy().rename('recent_police_contact_pretransition')
offending_predictor_candidates = pd.concat([recent_antisocial_behaviour_candidate, recent_police_contact_candidate],
    axis=1)
assert len(offending_predictor_candidates) == len(route_index)
assert offending_predictor_candidates.index.equals(route_index)
assert list(offending_predictor_candidates.columns) == ['recent_antisocial_behaviour_pretransition',
    'recent_police_contact_pretransition']
assert set(recent_antisocial_behaviour_candidate.dropna().unique()) == {0.0, 1.0, 2.0}
assert set(recent_police_contact_candidate.dropna().unique()) == {0.0, 1.0}
offending_predictor_summary_rows = []
for predictor in offending_predictor_candidates.columns:
    values = offending_predictor_candidates[predictor]
    offending_predictor_summary_rows.append({'Predictor': predictor, 'Non-missing': int(values.notna().sum()),
        'Missing': int(values.isna().sum()), 'Missing percentage': round(values.isna().mean() * 100,
        2), 'Observed categories': int(values.nunique())})
offending_predictor_summary = pd.DataFrame(offending_predictor_summary_rows)
offending_predictor_distribution_rows = []
offending_predictor_label_lookup = {'recent_antisocial_behaviour_pretransition': {0.0: 'No reported behaviour',
    1.0: 'One reported behaviour', 2.0: 'Two or more reported behaviours'}, 'recent_police_contact_pretransition': {0.0: 'No police contact',
    1.0: 'Police contact'}}
for predictor, label_lookup in offending_predictor_label_lookup.items():
    values = offending_predictor_candidates[predictor]
    for code, category in label_lookup.items():
        participants = int(values.eq(code).sum())
        offending_predictor_distribution_rows.append({'Predictor': predictor, 'Code': int(code), 'Category': category,
            'Participants': participants, 'Percentage of full sample': round(participants / len(route_index) * 100,
            2), 'Percentage among observed': round(participants / values.notna().sum() * 100, 2)})
    offending_predictor_distribution_rows.append({'Predictor': predictor, 'Code': pd.NA, 'Category': 'Unavailable',
        'Participants': int(values.isna().sum()), 'Percentage of full sample': round(values.isna().mean() * 100,
        2), 'Percentage among observed': pd.NA})
offending_predictor_distribution = pd.DataFrame(offending_predictor_distribution_rows)
offending_predictor_availability_count = offending_predictor_candidates.notna().sum(axis=1)
offending_predictor_joint_availability = offending_predictor_availability_count.value_counts().sort_index().rename('Participants').rename_axis('Offending predictors available').reset_index()
offending_predictor_joint_availability['Percentage'] = (offending_predictor_joint_availability['Participants'] / len(route_index) * 100).round(2)
assert int(offending_predictor_availability_count.eq(2).sum()) == 8949
offending_variable_decisions = domain_8_offending_inventory_expanded.copy()
offending_variable_decisions['Offending-track decision'] = pd.Series(pd.NA, index=offending_variable_decisions.index,
    dtype='string')
offending_variable_decisions['Offending-track representation'] = pd.Series(pd.NA,
    index=offending_variable_decisions.index, dtype='string')
offending_variable_decisions['Offending-track reason'] = pd.Series(pd.NA, index=offending_variable_decisions.index,
    dtype='string')

def assign_offending_decision(variables, decision, representation, reason):
    """Assign one decision to specified offending variables."""
    decision_mask = offending_variable_decisions['Variable'].isin(variables)
    assert int(decision_mask.sum()) == len(variables)
    assert offending_variable_decisions.loc[decision_mask, 'Offending-track decision'].isna().all()
    offending_variable_decisions.loc[decision_mask, 'Offending-track decision'] = decision
    offending_variable_decisions.loc[decision_mask, 'Offending-track representation'] = representation
    offending_variable_decisions.loc[decision_mask, 'Offending-track reason'] = reason
recent_antisocial_input_variables = ['W2sprayYP', 'W3sprayYP', 'W2smashYP', 'W3smashYP', 'W2shopYP', 'W3shopYP',
    'W2fightYP', 'W3fightYP']
assign_offending_decision(variables=recent_antisocial_input_variables, decision='Construction input',
    representation='recent_antisocial_behaviour_pretransition', reason='Wave 2 and eligible Wave 3 items use a previous-year reference period and are combined as the number of different reported behaviours, categorised as none, one, or two or more')
wave_1_antisocial_variables = ['W1sprayYP', 'W1smashYP', 'W1shopYP', 'W1fightYP']
assign_offending_decision(variables=wave_1_antisocial_variables, decision='Review support',
    representation='Lifetime antisocial-behaviour history', reason='Wave 1 uses a lifetime reference period and is not directly comparable with the previous-year Wave 2 and Wave 3 items; retained only for longitudinal and coding review')
recent_police_status_variables = ['W2police1MP', 'W3police1MP']
assign_offending_decision(variables=recent_police_status_variables, decision='Construction input',
    representation='recent_police_contact_pretransition', reason='Repeated main-parent report of whether police had contacted the household because of something the young person had done. Wave 3 is restricted to interviews before September 2006')
assign_offending_decision(variables=['W1Police1MP'], decision='Review support',
    representation='Earlier police-contact history', reason='Distinguishes contact within the previous three years from contact more than three years earlier and is not directly equivalent to the Wave 2 and Wave 3 status items')
assign_offending_decision(variables=['W1police2MP', 'W2Police2MP', 'W3police2MP'], decision='Review support',
    representation='Routed police-contact frequency', reason='Conditional count detail with few repeated-contact cases, several unavailable responses and minor inconsistencies with the direct status measure; not retained separately')
offending_unresolved_variables = offending_variable_decisions.loc[offending_variable_decisions['Offending-track decision'].isna()].copy()
assert len(offending_unresolved_variables) == 0
assert len(offending_variable_decisions) == 18
expected_offending_decision_counts = {'Construction input': 10, 'Review support': 8}
actual_offending_decision_counts = offending_variable_decisions['Offending-track decision'].value_counts().to_dict()
assert actual_offending_decision_counts == expected_offending_decision_counts
offending_decision_summary = offending_variable_decisions.groupby(['Offending-track decision', 'Review role'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Offending-track decision',
    'Review role']).reset_index(drop=True)
offending_representation_decisions = pd.DataFrame([{'Representation': 'Recent antisocial-behaviour breadth',
    'Decision': 'Retain as predictor candidate', 'Reason': 'Combines four distinct previous-year behaviours as none, one, or two or more. The measure represents behavioural breadth rather than a reflective scale'}, {'Representation': 'Separate antisocial-behaviour indicators',
    'Decision': 'Do not retain separately', 'Reason': 'Would add four overlapping predictors for relatively uncommon behaviours; the breadth category preserves whether multiple forms were reported'}, {'Representation': 'Wave 1 lifetime antisocial-behaviour history',
    'Decision': 'Review support only', 'Reason': 'Lifetime reference period is not comparable with the previous-year Wave 2 and Wave 3 measures'}, {'Representation': 'Recent binary police-contact status',
    'Decision': 'Retain as predictor candidate', 'Reason': 'Clear repeated parent-reported status with sufficient coverage and a substantively distinct official-contact construct'}, {'Representation': 'Police-contact frequency',
    'Decision': 'Review support only', 'Reason': 'Repeated-contact categories are sparse and the routed count contains minor inconsistencies with direct status'}])
offending_decision_display = offending_variable_decisions[['Review role', 'Wave', 'Source type', 'Source file',
    'Variable', 'Variable label', 'Timing status', 'Offending-track decision', 'Offending-track representation', 'Offending-track reason']].sort_values(['Offending-track representation',
    'Review role', 'Wave', 'Variable']).reset_index(drop=True)
print('Selected offending and police-contact predictors:')
display_limited(offending_predictor_summary)
print('Predictor category distributions:')
with pd.option_context('display.max_rows', None):
    display_limited(offending_predictor_distribution)
print('Joint predictor availability:')
display_limited(offending_predictor_joint_availability)
print('Variable decision summary:')
display_limited(offending_decision_summary)
print('Representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(offending_representation_decisions)
print(f'Unresolved offending variables: {len(offending_unresolved_variables)}')
print('Complete variable decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(offending_decision_display)

Selected offending and police-contact predictors:


,Predictor,Non-missing,Missing,Missing percentage,Observed categories
0,recent_antisocial_behaviour_pretransition,9443,324,3.32,3
1,recent_police_contact_pretransition,9022,745,7.63,2


Predictor category distributions:


,Predictor,Code,Category,Participants,Percentage of full sample,Percentage among observed
0,recent_antisocial_behaviour_pretransition,0,No reported behaviour,7698,78.82,81.52
1,recent_antisocial_behaviour_pretransition,1,One reported behaviour,1125,11.52,11.91
2,recent_antisocial_behaviour_pretransition,2,Two or more reported behaviours,620,6.35,6.57
3,recent_antisocial_behaviour_pretransition,<NA>,Unavailable,324,3.32,<NA>
4,recent_police_contact_pretransition,0,No police contact,8598,88.03,95.3


Joint predictor availability:


...

,Offending-track decision,Review role,Variables
0,Construction input,Fighting or public disturbance,2
1,Construction input,Graffiti,2
2,Construction input,Police contact,2
3,Construction input,Property vandalism,2
4,Construction input,Shoplifting,2


Representation decisions:


,Representation,Decision,Reason
0,Recent antisocial-behaviour breadth,Retain as predictor candidate,"Combines four distinct previous-year behaviours as none, one, or two or more. The measure represents behavioural breadth rather than a reflective scale"
1,Separate antisocial-behaviour indicators,Do not retain separately,Would add four overlapping predictors for relatively uncommon behaviours; the breadth category preserves whether multiple forms were reported
2,Wave 1 lifetime antisocial-behaviour history,Review support only,Lifetime reference period is not comparable with the previous-year Wave 2 and Wave 3 measures
3,Recent binary police-contact status,Retain as predictor candidate,Clear repeated parent-reported status with sufficient coverage and a substantively distinct official-contact construct
4,Police-contact frequency,Review support only,Repeated-contact categories are sparse and the routed count contains minor inconsistencies with direct status


Unresolved offending variables: 0
Complete variable decisions:


,Review role,Wave,Source type,Source file,Variable,Variable label,Timing status,Offending-track decision,Offending-track representation,Offending-track reason
0,Police contact,Wave 1,Young person,wave_one_lsype_young_person_2020,W1Police1MP,MP: Whether police have got in touch because of something YP had done,Pre-transition source,Review support,Earlier police-contact history,Distinguishes contact within the previous three years from contact more than three years earlier and is not directly equivalent to the Wave 2 and Wave 3 status items
1,Fighting or public disturbance,Wave 1,Young person,wave_one_lsype_young_person_2020,W1fightYP,YP: Whether ever taken part in fighting or public disturbance,Pre-transition source,Review support,Lifetime antisocial-behaviour history,Wave 1 uses a lifetime reference period and is not directly comparable with the previous-year Wave 2 and Wave 3 items; retained only for longitudinal and coding review
2,Graffiti,Wave 1,Young person,wave_one_lsype_young_person_2020,W1sprayYP,YP: Whether ever graffittied on walls,Pre-transition source,Review support,Lifetime antisocial-behaviour history,Wave 1 uses a lifetime reference period and is not directly comparable with the previous-year Wave 2 and Wave 3 items; retained only for longitudinal and coding review
3,Property vandalism,Wave 1,Young person,wave_one_lsype_young_person_2020,W1smashYP,YP: Whether ever vandalised public property,Pre-transition source,Review support,Lifetime antisocial-behaviour history,Wave 1 uses a lifetime reference period and is not directly comparable with the previous-year Wave 2 and Wave 3 items; retained only for longitudinal and coding review
4,Shoplifting,Wave 1,Young person,wave_one_lsype_young_person_2020,W1shopYP,YP: Whether ever shoplifted,Pre-transition source,Review support,Lifetime antisocial-behaviour history,Wave 1 uses a lifetime reference period and is not directly comparable with the previous-year Wave 2 and Wave 3 items; retained only for longitudinal and coding review


In [314]:
# 18: Work and unemployment orientation variable review

import pandas as pd
domain_8_work_orientation_inventory = domain_8_pending_candidates.loc[domain_8_pending_candidates['Matched construct'].str.contains('Work|unemployment orientation',
    case=False, regex=True, na=False)].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(domain_8_work_orientation_inventory) == 2
assert domain_8_work_orientation_inventory[['Source file', 'Variable']].duplicated().sum() == 0
work_orientation_raw_tables = {}
work_orientation_labelled_tables = {}
work_orientation_quality_rows = []
work_orientation_code_rows = []
for source_file, source_inventory in domain_8_work_orientation_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    raw_data['NSID'] = standardise_nsid(raw_data['NSID'])
    labelled_data['NSID'] = standardise_nsid(labelled_data['NSID'])
    assert raw_data['NSID'].is_unique
    assert labelled_data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(route_index)
    labelled_data = labelled_data.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    work_orientation_raw_tables[source_file] = raw_data
    work_orientation_labelled_tables[source_file] = labelled_data
    for variable in source_variables:
        variable_inventory = source_inventory.loc[source_inventory['Variable'].eq(variable)].iloc[0]
        raw_values = raw_data[variable]
        labelled_values = labelled_data[variable].astype('string')
        observed_mask = raw_values.ge(0).fillna(False)
        special_code_mask = raw_values.lt(0).fillna(False)
        observed_values = raw_values.loc[observed_mask]
        work_orientation_quality_rows.append({'Wave': variable_inventory['Wave'], 'Variable': variable,
            'Variable label': variable_inventory['Variable label'], 'Timing status': variable_inventory['Timing status'], 'Observed responses': int(observed_mask.sum()), 'Special-code responses': int(special_code_mask.sum()), 'No source record': int(raw_values.isna().sum()), 'Observed percentage': round(observed_mask.mean() * 100,
            2), 'Distinct observed codes': int(observed_values.nunique()), 'Minimum observed code': observed_values.min() if len(observed_values) > 0 else pd.NA, 'Maximum observed code': observed_values.max() if len(observed_values) > 0 else pd.NA, 'Not-applicable responses': int(raw_values.eq(-91).sum())})
        value_counts = raw_values.value_counts(dropna=False)
        for raw_code, participants in value_counts.items():
            if pd.isna(raw_code):
                value_label = 'No source record'
                response_type = 'No source record'
                sort_value = 999999
            else:
                matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(raw_code)
                response_type = 'Observed response' if raw_code >= 0 else 'Special code'
                sort_value = float(raw_code)
            work_orientation_code_rows.append({'Wave': variable_inventory['Wave'], 'Variable': variable,
                'Variable label': variable_inventory['Variable label'], 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
work_orientation_quality = pd.DataFrame(work_orientation_quality_rows).sort_values(['Wave',
    'Variable']).reset_index(drop=True)
work_orientation_code_distribution = pd.DataFrame(work_orientation_code_rows).sort_values(['Wave', 'Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
work_orientation_inventory_display = domain_8_work_orientation_inventory[['Wave', 'Source type', 'Source file',
    'Variable', 'Variable label', 'Timing status', 'Review status', 'Review outcome', 'Substantive domain', 'Deferred from earlier domain']].reset_index(drop=True)
print(f'Work and unemployment orientation variables reviewed: {len(domain_8_work_orientation_inventory)}')
print('Variable inventory:')
with pd.option_context('display.max_colwidth', None):
    display_limited(work_orientation_inventory_display)
print('Measure quality:')
with pd.option_context('display.max_colwidth', None):
    display_limited(work_orientation_quality)
print('Response codes:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(work_orientation_code_distribution)

Work and unemployment orientation variables reviewed: 2
Variable inventory:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Review status,Review outcome,Substantive domain,Deferred from earlier domain
0,Wave 1,Young person,wave_one_lsype_young_person_2020,W1plan16YP,YP: Agreement that having a job is better than being unemployed,Pre-transition source,"Variable-level coding, routing and reference-period review required.",Pending review,Experiences and behaviours,True
1,Wave 3,Young person,wave_three_lsype_young_person_2020,W3reas16aYP0h,YP: YP's main reason for staying on in FTE - To get a better job/good job/good c,Near-transition source,Use requires interview timing confirming January–August 2006 and item-level reference-period review.,Pending review,NaN,False


Measure quality:


,Wave,Variable,Variable label,Timing status,Observed responses,Special-code responses,No source record,Observed percentage,Distinct observed codes,Minimum observed code,Maximum observed code,Not-applicable responses
0,Wave 1,W1plan16YP,YP: Agreement that having a job is better than being unemployed,Pre-transition source,9362,162,243,95.85,4,1.0,4.0,0
1,Wave 3,W3reas16aYP0h,YP: YP's main reason for staying on in FTE - To get a better job/good job/good c,Near-transition source,7504,2005,258,76.83,2,0.0,1.0,1951


Response codes:


,Wave,Variable,Variable label,Raw code,Value label,Response type,Participants
0,Wave 1,W1plan16YP,YP: Agreement that having a job is better than being unemployed,-99.0,YP not interviewed,Special code,89
1,Wave 1,W1plan16YP,YP: Agreement that having a job is better than being unemployed,-1.0,Don't know,Special code,73
2,Wave 1,W1plan16YP,YP: Agreement that having a job is better than being unemployed,1.0,Agree strongly,Observed response,5292
3,Wave 1,W1plan16YP,YP: Agreement that having a job is better than being unemployed,2.0,Agree a little,Observed response,3046
4,Wave 1,W1plan16YP,YP: Agreement that having a job is better than being unemployed,3.0,Disagree a little,Observed response,727


In [315]:
# 19: Work-orientation representation and overlap review

import pandas as pd

def get_work_orientation_raw_variable(variable):
    """Return one aligned work-orientation variable."""
    source_file = domain_8_work_orientation_inventory.loc[domain_8_work_orientation_inventory['Variable'].eq(variable),
        'Source file'].iloc[0]
    return work_orientation_raw_tables[source_file][variable].copy()
work_orientation_raw = get_work_orientation_raw_variable('W1plan16YP')
work_orientation_ordinal_candidate = work_orientation_raw.map({1.0: 4.0, 2.0: 3.0, 3.0: 2.0,
    4.0: 1.0}).astype('Float64').rename('work_orientation_pretransition')
work_orientation_binary_review = work_orientation_raw.map({1.0: 1.0, 2.0: 1.0, 3.0: 0.0,
    4.0: 0.0}).astype('Float64').rename('work_orientation_agreement')
assert set(work_orientation_ordinal_candidate.dropna().unique()) == {1.0, 2.0, 3.0, 4.0}
assert set(work_orientation_binary_review.dropna().unique()) == {0.0, 1.0}
work_orientation_labels = {1.0: 'Disagree strongly', 2.0: 'Disagree a little', 3.0: 'Agree a little',
    4.0: 'Agree strongly'}
work_orientation_distribution = work_orientation_ordinal_candidate.map(work_orientation_labels).fillna('Unavailable').value_counts().rename('Participants').rename_axis('Work orientation').reset_index()
work_orientation_distribution['Percentage'] = (work_orientation_distribution['Participants'] / len(route_index) * 100).round(2)
work_orientation_summary = pd.DataFrame([{'Representation': 'Four-level work-orientation agreement',
    'Non-missing': int(work_orientation_ordinal_candidate.notna().sum()), 'Missing': int(work_orientation_ordinal_candidate.isna().sum()), 'Missing percentage': round(work_orientation_ordinal_candidate.isna().mean() * 100,
    2), 'Agree strongly or a little': int(work_orientation_binary_review.eq(1).sum()), 'Disagree strongly or a little': int(work_orientation_binary_review.eq(0).sum()), 'Agreement percentage among observed': round(work_orientation_binary_review.eq(1).sum() / work_orientation_binary_review.notna().sum() * 100,
    2)}])
wave_3_better_job_reason_raw = get_work_orientation_raw_variable('W3reas16aYP0h').where(wave_3_pretransition_substance_mask)
wave_3_better_job_reason_summary = pd.DataFrame([{'Response': 'Mentioned',
    'Participants': int(wave_3_better_job_reason_raw.eq(1).sum())}, {'Response': 'Not mentioned',
    'Participants': int(wave_3_better_job_reason_raw.eq(0).sum())}, {'Response': 'Structurally not applicable',
    'Participants': int(wave_3_better_job_reason_raw.eq(-91).sum())}, {'Response': 'Other unavailable',
    'Participants': int((wave_3_better_job_reason_raw.isna() | wave_3_better_job_reason_raw.lt(0) & ~wave_3_better_job_reason_raw.eq(-91)).sum())}])
wave_3_better_job_reason_summary['Percentage'] = (wave_3_better_job_reason_summary['Participants'] / len(route_index) * 100).round(2)
work_orientation_comparison_data = pd.DataFrame({'Work orientation': work_orientation_ordinal_candidate,
    'Young-person caring responsibility': young_person_caring_responsibility_candidate, 'Smoking activity': smoking_activity_pretransition_candidate, 'Alcohol-use frequency': alcohol_use_frequency_pretransition_candidate, 'Cannabis experience': cannabis_experience_pretransition_candidate, 'Recent antisocial behaviour': recent_antisocial_behaviour_candidate, 'Recent police contact': recent_police_contact_candidate, 'Truancy status': truancy_status_pretransition_candidate}, index=route_index)
work_orientation_overlap_rows = []
for comparison_measure in ['Young-person caring responsibility', 'Smoking activity', 'Alcohol-use frequency',
    'Cannabis experience', 'Recent antisocial behaviour', 'Recent police contact', 'Truancy status']:
    pair_data = work_orientation_comparison_data[['Work orientation', comparison_measure]].dropna()
    work_orientation_overlap_rows.append({'Comparison measure': comparison_measure,
        'Complete comparisons': len(pair_data), "Cramer's V": round(categorical_cramers_v(pair_data['Work orientation'],
        pair_data[comparison_measure]), 3)})
work_orientation_overlap_summary = pd.DataFrame(work_orientation_overlap_rows)
work_orientation_representation_comparison = pd.DataFrame([{'Four-level non-missing': int(work_orientation_ordinal_candidate.notna().sum()),
    'Binary non-missing': int(work_orientation_binary_review.notna().sum()), 'Four-level categories': int(work_orientation_ordinal_candidate.nunique()), 'Binary categories': int(work_orientation_binary_review.nunique()), "Cramer's V": round(categorical_cramers_v(work_orientation_ordinal_candidate,
    work_orientation_binary_review), 3)}])
print('Work-orientation measure summary:')
display_limited(work_orientation_summary)
print('Four-level work-orientation distribution:')
display_limited(work_orientation_distribution)
print('Conditional Wave 3 better-job reason:')
display_limited(wave_3_better_job_reason_summary)
print('Associations with retained behaviour measures:')
display_limited(work_orientation_overlap_summary)
print('Four-level and binary representation comparison:')
display_limited(work_orientation_representation_comparison)

Work-orientation measure summary:


,Representation,Non-missing,Missing,Missing percentage,Agree strongly or a little,Disagree strongly or a little,Agreement percentage among observed
0,Four-level work-orientation agreement,9362,405,4.15,8338,1024,89.06


Four-level work-orientation distribution:


,Work orientation,Participants,Percentage
0,Agree strongly,5292,54.18
1,Agree a little,3046,31.19
2,Disagree a little,727,7.44
3,Unavailable,405,4.15
4,Disagree strongly,297,3.04


Conditional Wave 3 better-job reason:


,Response,Participants,Percentage
0,Mentioned,1648,16.87
1,Not mentioned,5850,59.90
2,Structurally not applicable,1943,19.89
3,Other unavailable,326,3.34


Associations with retained behaviour measures:


,Comparison measure,Complete comparisons,Cramer's V
0,Young-person caring responsibility,9362,0.023
1,Smoking activity,9346,0.027
2,Alcohol-use frequency,9333,0.029
3,Cannabis experience,9356,0.028
4,Recent antisocial behaviour,9289,0.028


Four-level and binary representation comparison:


,Four-level non-missing,Binary non-missing,Four-level categories,Binary categories,Cramer's V
0,9362,9362,4,2,1.0


In [316]:
# 20: Work-orientation decisions

import pandas as pd
work_orientation_predictor_candidates = pd.DataFrame(index=route_index)
assert len(work_orientation_predictor_candidates) == len(route_index)
assert work_orientation_predictor_candidates.index.equals(route_index)
assert work_orientation_predictor_candidates.shape[1] == 0
work_orientation_variable_decisions = domain_8_work_orientation_inventory.copy()
work_orientation_variable_decisions['Work-orientation decision'] = pd.Series(pd.NA,
    index=work_orientation_variable_decisions.index, dtype='string')
work_orientation_variable_decisions['Work-orientation representation'] = pd.Series(pd.NA,
    index=work_orientation_variable_decisions.index, dtype='string')
work_orientation_variable_decisions['Work-orientation reason'] = pd.Series(pd.NA,
    index=work_orientation_variable_decisions.index, dtype='string')
wave_1_orientation_mask = work_orientation_variable_decisions['Variable'].eq('W1plan16YP')
assert int(wave_1_orientation_mask.sum()) == 1
work_orientation_variable_decisions.loc[wave_1_orientation_mask, 'Work-orientation decision'] = 'Review support'
work_orientation_variable_decisions.loc[wave_1_orientation_mask,
    'Work-orientation representation'] = 'General work-over-unemployment attitude'
work_orientation_variable_decisions.loc[wave_1_orientation_mask,
    'Work-orientation reason'] = 'Broad normative agreement that having a job is better than being unemployed. Although coverage is high, 89.06% agree, and the item does not measure a specific employment plan, work experience or behavioural orientation'
wave_3_reason_mask = work_orientation_variable_decisions['Variable'].eq('W3reas16aYP0h')
assert int(wave_3_reason_mask.sum()) == 1
work_orientation_variable_decisions.loc[wave_3_reason_mask, 'Work-orientation decision'] = 'Exclude'
work_orientation_variable_decisions.loc[wave_3_reason_mask,
    'Work-orientation representation'] = 'Conditional reason for remaining in full-time education'
work_orientation_variable_decisions.loc[wave_3_reason_mask,
    'Work-orientation reason'] = 'Asked only of young people routed through the full-time education pathway. Structural not-applicable responses cannot be treated as absence of work motivation, and the item reflects a route-specific post-16 decision rather than a whole-sample behavioural predictor'
work_orientation_unresolved_variables = work_orientation_variable_decisions.loc[work_orientation_variable_decisions['Work-orientation decision'].isna()].copy()
assert len(work_orientation_unresolved_variables) == 0
assert work_orientation_variable_decisions['Work-orientation decision'].value_counts().to_dict() == {'Review support': 1,
    'Exclude': 1}
work_orientation_representation_decisions = pd.DataFrame([{'Representation': 'Four-level general work orientation',
    'Decision': 'Review support only', 'Reason': 'Retains intensity of agreement but measures a broad normative belief, with 89.06% of observed participants agreeing strongly or a little'}, {'Representation': 'Binary work-orientation agreement',
    'Decision': 'Do not retain', 'Reason': 'Further reduces variation by combining strong and slight agreement and combining both disagreement levels'}, {'Representation': 'Better-job reason for remaining in education',
    'Decision': 'Exclude', 'Reason': 'Conditional on the full-time education route and not defined consistently for the complete analysis sample'}])
work_orientation_decision_summary = work_orientation_variable_decisions.groupby(['Work-orientation decision'],
    dropna=False).size().rename('Variables').reset_index().sort_values('Work-orientation decision').reset_index(drop=True)
work_orientation_decision_display = work_orientation_variable_decisions[['Wave', 'Source type', 'Source file',
    'Variable', 'Variable label', 'Timing status', 'Work-orientation decision', 'Work-orientation representation', 'Work-orientation reason']].sort_values(['Work-orientation decision',
    'Wave', 'Variable']).reset_index(drop=True)
print(f'Work-orientation predictors retained: {work_orientation_predictor_candidates.shape[1]}')
print('Variable decision summary:')
display_limited(work_orientation_decision_summary)
print('Representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(work_orientation_representation_decisions)
print(f'Unresolved work-orientation variables: {len(work_orientation_unresolved_variables)}')
print('Complete variable decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(work_orientation_decision_display)

Work-orientation predictors retained: 0
Variable decision summary:


,Work-orientation decision,Variables
0,Exclude,1
1,Review support,1


Representation decisions:


,Representation,Decision,Reason
0,Four-level general work orientation,Review support only,"Retains intensity of agreement but measures a broad normative belief, with 89.06% of observed participants agreeing strongly or a little"
1,Binary work-orientation agreement,Do not retain,Further reduces variation by combining strong and slight agreement and combining both disagreement levels
2,Better-job reason for remaining in education,Exclude,Conditional on the full-time education route and not defined consistently for the complete analysis sample


Unresolved work-orientation variables: 0
Complete variable decisions:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Work-orientation decision,Work-orientation representation,Work-orientation reason
0,Wave 3,Young person,wave_three_lsype_young_person_2020,W3reas16aYP0h,YP: YP's main reason for staying on in FTE - To get a better job/good job/good c,Near-transition source,Exclude,Conditional reason for remaining in full-time education,"Asked only of young people routed through the full-time education pathway. Structural not-applicable responses cannot be treated as absence of work motivation, and the item reflects a route-specific post-16 decision rather than a whole-sample behavioural predictor"
1,Wave 1,Young person,wave_one_lsype_young_person_2020,W1plan16YP,YP: Agreement that having a job is better than being unemployed,Pre-transition source,Review support,General work-over-unemployment attitude,"Broad normative agreement that having a job is better than being unemployed. Although coverage is high, 89.06% agree, and the item does not measure a specific employment plan, work experience or behavioural orientation"


In [317]:
# 21: Experiences and behaviours domain consolidation check

import pandas as pd
bullying_candidate_object_names = ['bullying_experience_pretransition_candidate', 'bullying_experience_candidate',
    'bullying_experience_pretransition', 'latest_recent_young_person_bullying', 'latest_young_person_bullying', 'latest_recent_yp_bullying']
available_bullying_objects = [object_name for object_name in bullying_candidate_object_names if object_name in globals()]
assert len(available_bullying_objects) >= 1, 'The bullying predictor object created during the earlier review could not be found.'
bullying_candidate_object = globals()[available_bullying_objects[0]]
if isinstance(bullying_candidate_object, pd.DataFrame):
    assert bullying_candidate_object.shape[1] == 1
    bullying_experience_candidate = bullying_candidate_object.iloc[:, 0].copy()
elif isinstance(bullying_candidate_object, pd.Series):
    bullying_experience_candidate = bullying_candidate_object.copy()
else:
    raise TypeError('The bullying predictor object must be a pandas Series or a one-column DataFrame.')
bullying_experience_candidate = bullying_experience_candidate.reindex(route_index).rename('bullying_experience_pretransition')
domain_8_predictor_candidates = pd.concat([bullying_experience_candidate, young_person_caring_responsibility_candidate,
    smoking_activity_pretransition_candidate, alcohol_use_frequency_pretransition_candidate, cannabis_experience_pretransition_candidate, recent_antisocial_behaviour_candidate, recent_police_contact_candidate], axis=1)
expected_domain_8_predictor_names = ['bullying_experience_pretransition',
    'young_person_caring_responsibility_pretransition', 'smoking_activity_pretransition', 'alcohol_use_frequency_pretransition', 'cannabis_experience_pretransition', 'recent_antisocial_behaviour_pretransition', 'recent_police_contact_pretransition']
assert list(domain_8_predictor_candidates.columns) == expected_domain_8_predictor_names
assert len(domain_8_predictor_candidates) == len(route_index)
assert domain_8_predictor_candidates.index.equals(route_index)
assert domain_8_predictor_candidates.columns.is_unique
expected_domain_8_non_missing = {'bullying_experience_pretransition': 9500,
    'young_person_caring_responsibility_pretransition': 9521, 'smoking_activity_pretransition': 9502, 'alcohol_use_frequency_pretransition': 9487, 'cannabis_experience_pretransition': 9513, 'recent_antisocial_behaviour_pretransition': 9443, 'recent_police_contact_pretransition': 9022}
actual_domain_8_non_missing = domain_8_predictor_candidates.notna().sum().to_dict()
assert actual_domain_8_non_missing == expected_domain_8_non_missing
domain_8_predictor_summary_rows = []
for predictor in domain_8_predictor_candidates.columns:
    predictor_values = domain_8_predictor_candidates[predictor]
    domain_8_predictor_summary_rows.append({'Predictor': predictor, 'Non-missing': int(predictor_values.notna().sum()),
        'Missing': int(predictor_values.isna().sum()), 'Missing percentage': round(predictor_values.isna().mean() * 100,
        2), 'Observed categories': int(predictor_values.nunique())})
domain_8_predictor_summary = pd.DataFrame(domain_8_predictor_summary_rows)
domain_8_predictor_availability_count = domain_8_predictor_candidates.notna().sum(axis=1)
domain_8_predictor_joint_availability = domain_8_predictor_availability_count.value_counts().sort_index().rename('Participants').rename_axis('Domain 8 predictors available').reset_index()
domain_8_predictor_joint_availability['Percentage'] = (domain_8_predictor_joint_availability['Participants'] / len(route_index) * 100).round(2)
decision_metadata_columns = ['Source order', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable',
    'Variable label', 'Timing status']

def standardise_track_decisions(decision_data, decision_column, representation_column, reason_column,
    decision_source):
    """Standardise one Domain 8 review-track decision table."""
    required_columns = decision_metadata_columns + [decision_column, representation_column, reason_column]
    missing_columns = [column for column in required_columns if column not in decision_data.columns]
    assert len(missing_columns) == 0, f'Missing columns in {decision_source}: {missing_columns}'
    standardised_data = decision_data[required_columns].copy().rename(columns={decision_column: 'Domain 8 decision',
        representation_column: 'Domain 8 representation', reason_column: 'Domain 8 reason'})
    standardised_data['Decision source'] = decision_source
    return standardised_data
bullying_decisions_standardised = standardise_track_decisions(decision_data=bullying_variable_decisions,
    decision_column='Bullying-track decision', representation_column='Bullying-track representation', reason_column='Bullying-track reason', decision_source='Bullying and safety review')
caring_decisions_standardised = standardise_track_decisions(decision_data=caring_variable_decisions,
    decision_column='Caring-track decision', representation_column='Caring-track representation', reason_column='Caring-track reason', decision_source='Caring-responsibility review')
substance_decisions_standardised = standardise_track_decisions(decision_data=substance_variable_decisions,
    decision_column='Substance-track decision', representation_column='Substance-track representation', reason_column='Substance-track reason', decision_source='Substance-use review')
offending_decisions_standardised = standardise_track_decisions(decision_data=offending_variable_decisions,
    decision_column='Offending-track decision', representation_column='Offending-track representation', reason_column='Offending-track reason', decision_source='Antisocial-behaviour and police-contact review')
work_orientation_decisions_standardised = standardise_track_decisions(decision_data=work_orientation_variable_decisions,
    decision_column='Work-orientation decision', representation_column='Work-orientation representation', reason_column='Work-orientation reason', decision_source='Work-orientation review')
domain_8_track_decisions = pd.concat([bullying_decisions_standardised, caring_decisions_standardised,
    substance_decisions_standardised, offending_decisions_standardised, work_orientation_decisions_standardised], ignore_index=True, sort=False).sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(domain_8_track_decisions) == 129
assert domain_8_track_decisions[['Source file', 'Variable']].duplicated().sum() == 0
assert domain_8_track_decisions['Domain 8 decision'].notna().all()
existing_domain_8_variable_names = ['W2Truant2YP0a', 'W3pladk16aYP0f', 'W3pladk16bYP0f', 'W3pladk2bYP0f']
existing_domain_8_decisions = verified_decision_register.loc[verified_decision_register['Variable'].isin(existing_domain_8_variable_names)].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(existing_domain_8_decisions) == 4
assert set(existing_domain_8_decisions['Variable']) == set(existing_domain_8_variable_names)
assert existing_domain_8_decisions['Review outcome'].notna().all()
assert (~existing_domain_8_decisions['Review outcome'].eq('Pending review')).all()

def normalise_existing_decision(decision_value):
    """Map an earlier-domain decision to the Domain 8 labels."""
    decision_text = str(decision_value).strip().lower()
    if 'construction' in decision_text:
        return 'Construction input'
    if 'support' in decision_text:
        return 'Review support'
    if 'exclude' in decision_text:
        return 'Exclude'
    raise ValueError(f'Unexpected earlier-domain decision: {decision_value}')
existing_domain_8_standardised = existing_domain_8_decisions[decision_metadata_columns + ['Review outcome',
    'Substantive domain', 'Decision reason']].copy()
existing_domain_8_standardised['Domain 8 decision'] = existing_domain_8_standardised['Review outcome'].apply(normalise_existing_decision)
existing_domain_8_standardised['Domain 8 representation'] = 'Decision retained from ' + existing_domain_8_standardised['Substantive domain'].fillna('an earlier domain').astype('string')
existing_domain_8_standardised['Domain 8 reason'] = existing_domain_8_standardised['Decision reason']
existing_domain_8_standardised['Decision source'] = 'Existing earlier-domain decision'
existing_domain_8_standardised = existing_domain_8_standardised[decision_metadata_columns + ['Domain 8 decision',
    'Domain 8 representation', 'Domain 8 reason', 'Decision source']]
domain_8_variable_decisions = pd.concat([domain_8_track_decisions, existing_domain_8_standardised], ignore_index=True,
    sort=False).sort_values(['Source order', 'Variable position']).reset_index(drop=True)
assert len(domain_8_variable_decisions) == 133
assert domain_8_variable_decisions[['Source file', 'Variable']].duplicated().sum() == 0
assert domain_8_variable_decisions[['Domain 8 decision', 'Domain 8 representation',
    'Domain 8 reason']].notna().all().all()
domain_8_complete_inventory = pd.concat([domain_8_pending_candidates, domain_8_alcohol_inventory,
    domain_8_graffiti_inventory, existing_domain_8_decisions], ignore_index=True, sort=False).drop_duplicates(subset=['Source file',
    'Variable']).sort_values(['Source order', 'Variable position']).reset_index(drop=True)
assert len(domain_8_complete_inventory) == 133
inventory_keys = set(zip(domain_8_complete_inventory['Source file'], domain_8_complete_inventory['Variable']))
decision_keys = set(zip(domain_8_variable_decisions['Source file'], domain_8_variable_decisions['Variable']))
assert inventory_keys == decision_keys
domain_8_decision_summary = domain_8_variable_decisions.groupby(['Domain 8 decision', 'Decision source'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Domain 8 decision',
    'Decision source']).reset_index(drop=True)
domain_8_overall_decision_counts = domain_8_variable_decisions['Domain 8 decision'].value_counts().rename('Variables').rename_axis('Domain 8 decision').reset_index()
expected_domain_8_decision_counts = {'Construction input': 33, 'Review support': 93, 'Exclude': 7}
actual_domain_8_decision_counts = domain_8_variable_decisions['Domain 8 decision'].value_counts().to_dict()
assert actual_domain_8_decision_counts == expected_domain_8_decision_counts
domain_8_unresolved_variables = domain_8_variable_decisions.loc[domain_8_variable_decisions['Domain 8 decision'].isna() | domain_8_variable_decisions['Domain 8 reason'].isna()].copy()
assert len(domain_8_unresolved_variables) == 0
print(f'Domain 8 predictors retained: {domain_8_predictor_candidates.shape[1]}')
print('Domain 8 predictor coverage:')
display_limited(domain_8_predictor_summary)
print('Joint Domain 8 predictor availability:')
display_limited(domain_8_predictor_joint_availability)
print(f'Complete Domain 8 variable inventory: {len(domain_8_complete_inventory)}')
print(f'Complete Domain 8 variable decisions: {len(domain_8_variable_decisions)}')
print('Overall Domain 8 decision counts:')
display_limited(domain_8_overall_decision_counts)
print('Domain 8 decisions by review track:')
display_limited(domain_8_decision_summary)
print(f'Unresolved Domain 8 variables: {len(domain_8_unresolved_variables)}')

Domain 8 predictors retained: 7
Domain 8 predictor coverage:


,Predictor,Non-missing,Missing,Missing percentage,Observed categories
0,bullying_experience_pretransition,9500,267,2.73,2
1,young_person_caring_responsibility_pretransition,9521,246,2.52,2
2,smoking_activity_pretransition,9502,265,2.71,4
3,alcohol_use_frequency_pretransition,9487,280,2.87,4
4,cannabis_experience_pretransition,9513,254,2.60,2


Joint Domain 8 predictor availability:


,Domain 8 predictors available,Participants,Percentage
0,0,243,2.49
1,1,4,0.04
2,2,7,0.07
3,3,2,0.02
4,5,12,0.12


Complete Domain 8 variable inventory: 133
Complete Domain 8 variable decisions: 133
Overall Domain 8 decision counts:


,Domain 8 decision,Variables
0,Review support,93
1,Construction input,33
2,Exclude,7


Domain 8 decisions by review track:


,Domain 8 decision,Decision source,Variables
0,Construction input,Antisocial-behaviour and police-contact review,10
1,Construction input,Bullying and safety review,3
2,Construction input,Caring-responsibility review,3
3,Construction input,Existing earlier-domain decision,1
4,Construction input,Substance-use review,16


Unresolved Domain 8 variables: 0


In [318]:
# 22: Experiences and behaviours output and register validation

import pandas as pd
domain_8_predictor_output = domain_8_predictor_candidates.copy()
domain_8_predictor_output.index.name = 'NSID'
domain_8_predictor_output = domain_8_predictor_output.reset_index()
assert len(domain_8_predictor_output) == 9767
assert domain_8_predictor_output['NSID'].is_unique
assert domain_8_predictor_output['NSID'].notna().all()
assert list(domain_8_predictor_output.columns) == ['NSID', 'bullying_experience_pretransition',
    'young_person_caring_responsibility_pretransition', 'smoking_activity_pretransition', 'alcohol_use_frequency_pretransition', 'cannabis_experience_pretransition', 'recent_antisocial_behaviour_pretransition', 'recent_police_contact_pretransition']
domain_8_register_key = ['Source file', 'Variable']
assert verified_decision_register[domain_8_register_key].duplicated().sum() == 0
assert domain_8_variable_decisions[domain_8_register_key].duplicated().sum() == 0
domain_8_register_review = verified_decision_register.merge(domain_8_variable_decisions[domain_8_register_key + ['Domain 8 decision',
    'Domain 8 representation', 'Domain 8 reason', 'Decision source']], on=domain_8_register_key, how='inner', validate='one_to_one').sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(domain_8_register_review) == 133
new_domain_8_decision_mask = ~domain_8_register_review['Decision source'].eq('Existing earlier-domain decision')
existing_decision_mask = domain_8_register_review['Decision source'].eq('Existing earlier-domain decision')
assert int(new_domain_8_decision_mask.sum()) == 129
assert int(existing_decision_mask.sum()) == 4
new_domain_8_register_decisions = domain_8_register_review.loc[new_domain_8_decision_mask].copy()
existing_domain_8_register_decisions = domain_8_register_review.loc[existing_decision_mask].copy()
assert new_domain_8_register_decisions['Review outcome'].eq('Pending review').all()
existing_normalised_outcomes = existing_domain_8_register_decisions['Review outcome'].apply(normalise_existing_decision)
assert existing_normalised_outcomes.eq(existing_domain_8_register_decisions['Domain 8 decision']).all()
expected_new_domain_8_counts = {'Construction input': 32, 'Review support': 91, 'Exclude': 6}
actual_new_domain_8_counts = new_domain_8_register_decisions['Domain 8 decision'].value_counts().to_dict()
assert actual_new_domain_8_counts == expected_new_domain_8_counts
domain_8_register_updates = new_domain_8_register_decisions[domain_8_register_key + ['Domain 8 decision',
    'Domain 8 representation', 'Domain 8 reason', 'Decision source']].copy().rename(columns={'Domain 8 decision': 'Proposed review outcome',
    'Domain 8 representation': 'Proposed representation', 'Domain 8 reason': 'Proposed decision reason', 'Decision source': 'Proposed decision source'})
assert len(domain_8_register_updates) == 129
verified_decision_register_domain_8_preview = verified_decision_register.merge(domain_8_register_updates,
    on=domain_8_register_key, how='left', validate='one_to_one')
domain_8_update_mask = verified_decision_register_domain_8_preview['Proposed review outcome'].notna()
assert int(domain_8_update_mask.sum()) == 129
verified_decision_register_domain_8_preview.loc[domain_8_update_mask,
    'Review outcome'] = verified_decision_register_domain_8_preview.loc[domain_8_update_mask,
    'Proposed review outcome'].to_numpy()
verified_decision_register_domain_8_preview.loc[domain_8_update_mask,
    'Substantive domain'] = 'Experiences and behaviours'
verified_decision_register_domain_8_preview.loc[domain_8_update_mask,
    'Decision reason'] = verified_decision_register_domain_8_preview.loc[domain_8_update_mask,
    'Proposed decision reason'].to_numpy()
existing_review_notes = verified_decision_register_domain_8_preview.loc[domain_8_update_mask,
    'Review notes'].fillna('').astype('string').str.strip()
new_domain_8_review_notes = 'Domain 8 representation: ' + verified_decision_register_domain_8_preview.loc[domain_8_update_mask,
    'Proposed representation'].astype('string') + '; decision source: ' + verified_decision_register_domain_8_preview.loc[domain_8_update_mask,
    'Proposed decision source'].astype('string')
verified_decision_register_domain_8_preview.loc[domain_8_update_mask,
    'Review notes'] = (existing_review_notes.where(existing_review_notes.eq(''),
    existing_review_notes + '; ') + new_domain_8_review_notes).to_numpy()
temporary_update_columns = ['Proposed review outcome', 'Proposed representation', 'Proposed decision reason',
    'Proposed decision source']
verified_decision_register_domain_8_preview = verified_decision_register_domain_8_preview.drop(columns=temporary_update_columns)
verified_decision_register_domain_8_preview = verified_decision_register_domain_8_preview[verified_decision_register.columns].copy()
assert verified_decision_register_domain_8_preview.shape == verified_decision_register.shape
assert list(verified_decision_register_domain_8_preview.columns) == list(verified_decision_register.columns)
assert verified_decision_register_domain_8_preview[domain_8_register_key].duplicated().sum() == 0
domain_8_decisions_export = verified_decision_register_domain_8_preview.merge(domain_8_variable_decisions[domain_8_register_key + ['Domain 8 representation',
    'Decision source']], on=domain_8_register_key, how='inner', validate='one_to_one').sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(domain_8_decisions_export) == 133
assert domain_8_decisions_export.columns.is_unique
assert domain_8_decisions_export[['Review outcome', 'Decision reason', 'Domain 8 representation',
    'Decision source']].notna().all().all()
domain_8_register_rows = domain_8_decisions_export.loc[~domain_8_decisions_export['Decision source'].eq('Existing earlier-domain decision')].copy()
assert len(domain_8_register_rows) == 129
assert domain_8_register_rows['Substantive domain'].eq('Experiences and behaviours').all()
assert domain_8_register_rows['Review outcome'].value_counts().to_dict() == expected_new_domain_8_counts
original_existing_rows = verified_decision_register.merge(existing_domain_8_register_decisions[domain_8_register_key],
    on=domain_8_register_key, how='inner', validate='one_to_one').sort_values(domain_8_register_key).reset_index(drop=True)
preview_existing_rows = verified_decision_register_domain_8_preview.merge(existing_domain_8_register_decisions[domain_8_register_key],
    on=domain_8_register_key, how='inner', validate='one_to_one').sort_values(domain_8_register_key).reset_index(drop=True)
assert original_existing_rows.equals(preview_existing_rows)
domain_8_export_decision_summary = domain_8_decisions_export['Review outcome'].value_counts().rename('Variables').rename_axis('Review outcome').reset_index()
domain_8_update_validation_summary = pd.DataFrame([{'Validation item': 'Predictor-output rows',
    'Count': len(domain_8_predictor_output)}, {'Validation item': 'Predictors',
    'Count': domain_8_predictor_output.shape[1] - 1}, {'Validation item': 'Domain decision rows',
    'Count': len(domain_8_decisions_export)}, {'Validation item': 'New register updates',
    'Count': len(domain_8_register_updates)}, {'Validation item': 'Earlier decisions preserved',
    'Count': len(existing_domain_8_register_decisions)}, {'Validation item': 'Decision-register rows',
    'Count': len(verified_decision_register_domain_8_preview)}, {'Validation item': 'Decision-register columns',
    'Count': verified_decision_register_domain_8_preview.shape[1]}])
print('Prepared output validation:')
display_limited(domain_8_update_validation_summary)
print('Prepared Domain 8 decision counts:')
display_limited(domain_8_export_decision_summary)
print(f'New Domain 8 register updates: {len(domain_8_register_updates)}')
print(f'Earlier-domain decisions preserved without changes: {len(existing_domain_8_register_decisions)}')
print('Files written in this cell: 0')

Prepared output validation:


,Validation item,Count
0,Predictor-output rows,9767
1,Predictors,7
2,Domain decision rows,133
3,New register updates,129
4,Earlier decisions preserved,4


Prepared Domain 8 decision counts:


,Review outcome,Variables
0,Review support,91
1,Construction input,32
2,Exclude,7
3,support,2
4,construction,1


New Domain 8 register updates: 129
Earlier-domain decisions preserved without changes: 4
Files written in this cell: 0


In [319]:
# 23: Experiences and behaviours domain output saving

domain_8_predictor_path = stage_2_output_directory / 'stage_2_experiences_behaviours_domain_predictors.csv'
domain_8_decision_path = stage_2_output_directory / 'stage_2_experiences_behaviours_domain_decisions.csv'
domain_8_decisions_output = verified_decision_register_domain_8_preview.merge(domain_8_variable_decisions[domain_8_register_key + ['Domain 8 decision',
    'Domain 8 representation', 'Domain 8 reason', 'Decision source']], on=domain_8_register_key, how='inner', validate='one_to_one').sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(domain_8_decisions_output) == 133
assert domain_8_decisions_output[domain_8_register_key].duplicated().sum() == 0
assert domain_8_decisions_output[['Domain 8 decision', 'Domain 8 representation', 'Domain 8 reason',
    'Decision source']].notna().all().all()
assert domain_8_decisions_output['Domain 8 decision'].value_counts().to_dict() == {'Review support': 93,
    'Construction input': 33, 'Exclude': 7}
assert len(domain_8_predictor_output) == 9767
assert domain_8_predictor_output['NSID'].is_unique
assert domain_8_predictor_output.shape[1] == 8
domain_8_predictor_output.to_csv(domain_8_predictor_path, index=False)
domain_8_decisions_output.to_csv(domain_8_decision_path, index=False)
verified_decision_register = verified_decision_register_domain_8_preview.copy()
verified_decision_register.to_csv(decision_register_path, index=False)
saved_domain_8_files = pd.DataFrame([{'Output': 'Experiences and behaviours predictors',
    'Path': str(domain_8_predictor_path), 'Exists': domain_8_predictor_path.exists(), 'Size bytes': domain_8_predictor_path.stat().st_size if domain_8_predictor_path.exists() else 0}, {'Output': 'Experiences and behaviours decisions',
    'Path': str(domain_8_decision_path), 'Exists': domain_8_decision_path.exists(), 'Size bytes': domain_8_decision_path.stat().st_size if domain_8_decision_path.exists() else 0}, {'Output': 'Variable decision register',
    'Path': str(decision_register_path), 'Exists': decision_register_path.exists(), 'Size bytes': decision_register_path.stat().st_size if decision_register_path.exists() else 0}])
saved_domain_8_files['Non-empty'] = saved_domain_8_files['Size bytes'].gt(0)
assert saved_domain_8_files['Exists'].all()
assert saved_domain_8_files['Non-empty'].all()
saved_domain_8_predictors = pd.read_csv(domain_8_predictor_path)
saved_domain_8_decisions = pd.read_csv(domain_8_decision_path)
saved_decision_register = pd.read_csv(decision_register_path)
assert saved_domain_8_predictors.shape == (9767, 8)
assert len(saved_domain_8_decisions) == 133
assert saved_decision_register.shape == (5261, 17)
assert saved_domain_8_predictors['NSID'].is_unique
assert saved_domain_8_decisions[['Source file', 'Variable']].duplicated().sum() == 0
assert saved_decision_register[['Source file', 'Variable']].duplicated().sum() == 0
saved_domain_8_decision_summary = saved_domain_8_decisions['Domain 8 decision'].value_counts().rename('Variables').rename_axis('Domain 8 decision').reset_index()
print('Saved Domain 8 files:')
with pd.option_context('display.max_colwidth', None):
    display_limited(saved_domain_8_files)
print('Saved predictor dimensions:')
print(saved_domain_8_predictors.shape)
print('Saved Domain 8 decision dimensions:')
print(saved_domain_8_decisions.shape)
print('Saved decision-register dimensions:')
print(saved_decision_register.shape)
print('Saved Domain 8 decision counts:')
display_limited(saved_domain_8_decision_summary)

Saved Domain 8 files:


,Output,Path,Exists,Size bytes,Non-empty
0,Experiences and behaviours predictors,data_derived\stage_2_predictor_construction\stage_2_experiences_behaviours_domain_predictors.csv,True,364271,True
1,Experiences and behaviours decisions,data_derived\stage_2_predictor_construction\stage_2_experiences_behaviours_domain_decisions.csv,True,121066,True
2,Variable decision register,data_derived\stage_2_predictor_construction\stage_2_variable_decision_register.csv,True,1791005,True


Saved predictor dimensions:
(9767, 8)
Saved Domain 8 decision dimensions:
(133, 21)
Saved decision-register dimensions:
(5261, 17)
Saved Domain 8 decision counts:


,Domain 8 decision,Variables
0,Review support,93
1,Construction input,33
2,Exclude,7


## Experience and behaviour predictors

Seven predictors were retained: `bullying_experience_pretransition`, `young_person_caring_responsibility_pretransition`, `smoking_activity_pretransition`, `alcohol_use_frequency_pretransition`, `cannabis_experience_pretransition`, `recent_antisocial_behaviour_pretransition` and `recent_police_contact_pretransition`.

All seven were available for 8,910 participants (91.23%); 243 participants had no observed measure in this domain. Predictor coverage ranged from 9,022 for recent police contact to 9,521 for caring responsibility. The decision file contains 33 construction inputs, 93 review-support variables and seven exclusions.


# Part 12: Parental educational attitudes and support

Parental expectations, educational discussion, financial support, homework support, monitoring, school communication, parent–young person relationships and shared activities were reviewed. Family socioeconomic measures were not duplicated.


In [320]:
# 1: Parental educational attitudes and support candidate inventory

import pandas as pd
domain_9_deferred_variable_names = ['W1hepossMP', 'W1henotMP0a', 'W1henotMP0b', 'W1henotMP0c', 'W1henotMP0d',
    'W1henotMP0e', 'W1henotMP0f', 'W1henotMP0g', 'W1henotMP0h', 'W1henotMP0i', 'W1henotMP0j', 'W1mothdecYP', 'W1fathdecYP']
domain_9_search_register = verified_decision_register.loc[verified_decision_register['Wave'].isin(['Wave 1', 'Wave 2',
    'Wave 3']) & verified_decision_register['Review outcome'].eq('Pending review')].copy()
domain_9_search_register['Search text'] = domain_9_search_register['Variable'].fillna('').astype('string').str.cat(domain_9_search_register['Variable label'].fillna('').astype('string'),
    sep=' ').str.lower()
domain_9_construct_patterns = {'Parental educational expectations': 'parent.*expect|mp:.*expect|mother.*expect|father.*expect|higher education|university|stay.*education|remain.*education|continue.*education|education after 16|post-16',
    'Parental educational aspirations': 'parent.*aspir|mp:.*aspir|mother.*aspir|father.*aspir|hope.*education|want.*education|like.*education|qualification.*want', 'Homework and study support': 'help.*homework|homework.*help|check.*homework|homework.*check|make.*homework|ensure.*homework|support.*schoolwork|support.*study|help.*schoolwork|help.*study|study.*help', 'Parental school involvement': 'parents?.*evening|school.*meeting|meeting.*school|contact.*school|school.*contact|talk.*teacher|teacher.*talk|visit.*school|school report|report.*school', 'Parental monitoring': 'parent.*know|mother.*know|father.*know|know.*where|know.*friend|monitor|curfew|allowed.*out|go out|whereabouts', 'Parental decision-making and autonomy': 'mother.*decid|father.*decid|parent.*decid|mothdec|fathdec|decision.*mother|decision.*father|decision.*parent|allowed.*decide|make.*decision', 'Parent–young-person relationship': 'get on.*mother|get on.*father|relationship.*mother|relationship.*father|relationship.*parent|talk.*mother|talk.*father|talk.*parent|close.*mother|close.*father|close.*parent|support.*mother|support.*father|support.*parent'}
domain_9_candidate_rows = []
for construct, pattern in domain_9_construct_patterns.items():
    construct_matches = domain_9_search_register.loc[domain_9_search_register['Search text'].str.contains(pattern,
        case=False, regex=True, na=False)].copy()
    construct_matches['Matched construct'] = construct
    domain_9_candidate_rows.append(construct_matches)
domain_9_keyword_candidates = pd.concat(domain_9_candidate_rows, ignore_index=True,
    sort=False) if domain_9_candidate_rows else pd.DataFrame()
domain_9_deferred_candidates = verified_decision_register.loc[verified_decision_register['Variable'].isin(domain_9_deferred_variable_names)].copy()
assert set(domain_9_deferred_candidates['Variable']) == set(domain_9_deferred_variable_names)

def assign_deferred_construct(variable):
    """Assign the intended Domain 9 construct to a deferred variable."""
    if variable == 'W1hepossMP':
        return 'Parental educational expectations'
    if variable.startswith('W1henotMP0'):
        return 'Reasons for low parental higher-education expectation'
    if variable in {'W1mothdecYP', 'W1fathdecYP'}:
        return 'Parental decision-making and autonomy'
    return 'Other deferred parental measure'
domain_9_deferred_candidates['Matched construct'] = domain_9_deferred_candidates['Variable'].apply(assign_deferred_construct)
domain_9_deferred_candidates['Deferred from earlier domain'] = True
domain_9_keyword_candidates['Deferred from earlier domain'] = False
domain_9_candidate_inventory = pd.concat([domain_9_keyword_candidates, domain_9_deferred_candidates],
    ignore_index=True, sort=False).drop_duplicates(subset=['Source file', 'Variable',
    'Matched construct']).sort_values(['Matched construct', 'Wave', 'Source order',
    'Variable position']).reset_index(drop=True)
domain_9_variable_match_counts = domain_9_candidate_inventory.groupby(['Source file',
    'Variable']).size().rename('Construct matches').reset_index()
domain_9_candidate_inventory = domain_9_candidate_inventory.merge(domain_9_variable_match_counts, on=['Source file',
    'Variable'], how='left', validate='many_to_one')
domain_9_unique_candidates = domain_9_candidate_inventory.sort_values(['Deferred from earlier domain',
    'Matched construct'], ascending=[False, True]).drop_duplicates(subset=['Source file', 'Variable'],
    keep='first').sort_values(['Source order', 'Variable position']).reset_index(drop=True)
assert set(domain_9_deferred_variable_names).issubset(set(domain_9_unique_candidates['Variable']))
domain_9_candidate_summary = domain_9_candidate_inventory.groupby(['Matched construct', 'Wave'],
    dropna=False).agg(Variable_matches=('Variable', 'size'), Unique_variables=('Variable',
    'nunique')).reset_index().sort_values(['Matched construct', 'Wave']).reset_index(drop=True)
domain_9_unique_candidate_summary = domain_9_unique_candidates.groupby(['Wave', 'Source type'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Wave', 'Source type']).reset_index(drop=True)
domain_9_candidate_display = domain_9_unique_candidates.sort_values(['Matched construct', 'Wave', 'Source order',
    'Variable position'])[['Matched construct', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable',
    'Variable label', 'Timing status', 'Review status', 'Review outcome', 'Substantive domain', 'Deferred from earlier domain', 'Construct matches']].reset_index(drop=True)
print(f'Domain 9 unique candidate variables: {len(domain_9_unique_candidates):,}')
print(f'Explicitly deferred variables recovered: {len(domain_9_deferred_candidates):,}')
print('\nCandidate matches by construct and wave:')
display_limited(domain_9_candidate_summary)
print('\nUnique candidates by wave and source type:')
display_limited(domain_9_unique_candidate_summary)
print('\nFirst 15 candidate variables:')
display_limited(domain_9_candidate_display.head(15))
print(f'Additional candidate variables not displayed: {max(len(domain_9_candidate_display) - 15, 0):,}')
domain_9_candidate_inventory_file = stage_2_output_directory / 'stage_2_parental_attitudes_candidate_inventory.csv'
domain_9_candidate_display.to_csv(domain_9_candidate_inventory_file, index=False)
print(f'\nFull candidate inventory saved to: {domain_9_candidate_inventory_file}')

Domain 9 unique candidate variables: 174
Explicitly deferred variables recovered: 13

Candidate matches by construct and wave:


,Matched construct,Wave,Variable_matches,Unique_variables
0,Homework and study support,Wave 1,3,3
1,Homework and study support,Wave 2,2,2
2,Parental decision-making and autonomy,Wave 1,2,2
3,Parental educational aspirations,Wave 1,25,25
4,Parental educational aspirations,Wave 2,11,11



Unique candidates by wave and source type:


,Wave,Source type,Variables
0,Wave 1,Family background,14
1,Wave 1,Parental attitudes,37
2,Wave 1,Young person,16
3,Wave 2,Family background,21
4,Wave 2,Parental attitudes,23



First 15 candidate variables:


,Matched construct,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Substantive domain,Deferred from earlier domain,Construct matches
0,Homework and study support,Wave 1,Young person,wave_one_lsype_young_person_2020,136,W1hwhelpYP,YP: Whether anyone at home helps them with hom...,Pre-transition source,Deferred to later domain,Pending review,Parental educational attitudes and support,False,1
1,Homework and study support,Wave 1,Young person,wave_one_lsype_young_person_2020,137,W1hwpchlYP,YP: Whether anyone at home makes sure that do ...,Pre-transition source,Deferred to later domain,Pending review,Parental educational attitudes and support,False,1
2,Homework and study support,Wave 1,Young person,wave_one_lsype_young_person_2020,156,W1hintdYP0j,YP: Ways use home computer to help with school...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,NaN,False,1
3,Homework and study support,Wave 2,Young person,wave_two_lsype_young_person_2020,235,W2hwhelpYP,YP: Whether anyone at home helps them with hom...,Pre-transition source,Deferred to later domain,Pending review,Parental educational attitudes and support,False,1
4,Homework and study support,Wave 2,Young person,wave_two_lsype_young_person_2020,236,W2hwpchlYP,YP: Whether anyone at home makes sure that do ...,Pre-transition source,Deferred to later domain,Pending review,Parental educational attitudes and support,False,1


Additional candidate variables not displayed: 159

Full candidate inventory saved to: data_derived\stage_2_predictor_construction\stage_2_parental_attitudes_candidate_inventory.csv


In [321]:
# 2: Parental attitudes and support candidate screening

import pandas as pd
domain_9_screened_candidates = domain_9_unique_candidates.copy()
domain_9_screened_candidates['Domain 9 screening status'] = pd.Series(pd.NA, index=domain_9_screened_candidates.index,
    dtype='string')
domain_9_screened_candidates['Domain 9 review track'] = pd.Series(pd.NA, index=domain_9_screened_candidates.index,
    dtype='string')
domain_9_screened_candidates['Domain 9 screening reason'] = pd.Series(pd.NA, index=domain_9_screened_candidates.index,
    dtype='string')

def assign_domain_9_screening(variables, status, review_track, reason):
    """Assign a screening result to specified variables."""
    screening_mask = domain_9_screened_candidates['Variable'].isin(variables)
    matched_variables = set(domain_9_screened_candidates.loc[screening_mask, 'Variable'])
    expected_variables = set(variables)
    assert matched_variables == expected_variables, f'Variables not matched correctly: {sorted(expected_variables - matched_variables)}'
    assert domain_9_screened_candidates.loc[screening_mask, 'Domain 9 screening status'].isna().all()
    domain_9_screened_candidates.loc[screening_mask, 'Domain 9 screening status'] = status
    domain_9_screened_candidates.loc[screening_mask, 'Domain 9 review track'] = review_track
    domain_9_screened_candidates.loc[screening_mask, 'Domain 9 screening reason'] = reason

def variables_with_prefixes(prefixes):
    """Return candidate variables beginning with specified prefixes."""
    lower_prefixes = tuple((prefix.lower() for prefix in prefixes))
    return domain_9_screened_candidates.loc[domain_9_screened_candidates['Variable'].str.lower().str.startswith(lower_prefixes),
        'Variable'].drop_duplicates().tolist()
parental_he_expectation_variables = ['W1hepossMP', *[f'W1henotMP0{suffix}' for suffix in 'abcdefghij']]
assign_domain_9_screening(variables=parental_he_expectation_variables, status='Core review',
    review_track='Parental higher-education expectation', reason='Direct parental expectation of HE participation and routed reasons for a low expectation')
assign_domain_9_screening(variables=['W1patt3MP'], status='Core review',
    review_track='Parental educational aspiration', reason='Direct parental report of wanting the young person to receive a better education than the parent')
assign_domain_9_screening(variables=['W1jobdiscMP', 'W2jobdiscMP', 'W3jobdiscMP'], status='Core review',
    review_track='Parental discussion of continued education', reason='Repeated parental report of discussion with the young person about remaining in full-time education')
parental_education_finance_variables = variables_with_prefixes(['W1fefinMP0', 'W2FeFinMP0', 'W3fefinMP0', 'W1fefn2MP0',
    'W2FEfn2MP0', 'W3fefn2MP0'])
assert len(parental_education_finance_variables) == 54
assign_domain_9_screening(variables=parental_education_finance_variables, status='Core review',
    review_track='Planned financial support for continued education', reason='Parental reports of expected funding sources and actions intended to support continued participation in education')
assign_domain_9_screening(variables=['W2EmaGetMP'], status='Core review', review_track='Education finance expectation',
    reason="Parental expectation of the young person's eligibility for financial support if remaining in education")
assign_domain_9_screening(variables=['W1hwhelpYP', 'W2hwhelpYP', 'W1hwpchlYP', 'W2hwpchlYP'], status='Core review',
    review_track='Homework support and monitoring', reason='Repeated young-person reports of help with homework and whether someone at home ensures homework is completed')
assign_domain_9_screening(variables=['W1pareveMP', 'W2pareveMP', 'W3pareveMP', 'W1tmeetfMP', 'W2tmeetfMP',
    'W3tmeetfMP', 'W1repred1MP', 'W2vocs2MP', 'W3tspeak2MP', 'W3tstayMP', 'W3tappMP'], status='Core review', review_track='Parental school involvement', reason='Parent-reported attendance at school events, meetings and discussions with teachers about education or training')
assign_domain_9_screening(variables=['W1gowhereYP', 'W1limitsnYP', 'W1paroutMP'], status='Core review',
    review_track='Parental monitoring', reason='Young-person and parent reports of parental knowledge of evening whereabouts and school-night limits')
assign_domain_9_screening(variables=['W1mgetonYP', 'W1fgetonYP', 'W1talkmumYP', 'W1talkdadYP', 'W1talkschYP'],
    status='Core review', review_track='Parent–young-person relationship and communication', reason='Young-person reports of relationship quality and communication with parents, including discussion of the school day')
assign_domain_9_screening(variables=['W1mothdecYP', 'W1fathdecYP'], status='Core review',
    review_track='Parental autonomy support', reason='Young-person reports of whether the mother and father support the young person in making their own decisions')
assign_domain_9_screening(variables=['W2ModAp3YP0a'], status='Core review',
    review_track='Parental post-16 route discussion', reason='Young-person report of discussing training or apprenticeship with parents')
assign_domain_9_screening(variables=['W1fammusMP'], status='Contextual review', review_track='Shared family activity',
    reason='Measures frequency of going out together as a family rather than direct educational support')
assign_domain_9_screening(variables=['W2SchcontMP', 'W3schcontMP'], status='Contextual review',
    review_track='Reactive school–parent contact', reason='Records school contact with the parent about behaviour and may reflect behavioural difficulty rather than parental involvement')
assign_domain_9_screening(variables=['W3tstaydefMP'], status='Contextual review',
    review_track='Teacher recommendation reported by parent', reason="Reports the teacher's view about staying in education rather than the parent's own educational expectation")
family_education_history_variables = domain_9_screened_candidates.loc[domain_9_screened_candidates['Source type'].eq('Family background') & domain_9_screened_candidates['Variable label'].fillna('').str.contains('qualification|course\\(s\\) expected to finish|father ever went to university|mother ever went to university',
    case=False, regex=True, na=False), 'Variable'].drop_duplicates().tolist()
assert len(family_education_history_variables) == 35
assign_domain_9_screening(variables=family_education_history_variables, status='Out of domain',
    review_track='Parental education history', reason='Measures parental qualifications or intergenerational educational history, which belongs to family socioeconomic background rather than parental educational attitudes')
young_person_education_variables = ['W1ambitYP', 'W2alevuniYP', 'W2getemaYP', 'W3reas16aYP0e', 'W3alevuniYP',
    'W3notuniYP', 'W3unisubbYPdr', 'W3unisubbYPds', 'W3unisubbYPdv', 'W3unisubbYPdw', 'W3unisubbYPdx', 'W3unisubbYPdy', 'W3hesubYP18']
assign_domain_9_screening(variables=young_person_education_variables, status='Out of domain',
    review_track='Young-person educational plans', reason="Measures the young person's own plans, course intentions or preferred degree subject rather than parental attitudes")
teacher_guidance_variables = ['W1advfrsYP', 'W1advteacYP', 'W2AdvFrsYP', 'W2AdvTeacYP', 'W2ModAp3YP0d',
    *[f'W3tlkteacYP0{suffix}' for suffix in 'abcdefg'], *[f'W3tlktappYP0{suffix}' for suffix in 'abcdefg']]
assign_domain_9_screening(variables=teacher_guidance_variables, status='Out of domain',
    review_track='Teacher and careers guidance', reason='Measures discussion with teachers or careers advisers rather than parental attitudes or support')
assign_domain_9_screening(variables=['W1famsupYP', 'W2famsupYP', 'W3famsupYP'], status='Out of domain',
    review_track='General financial support', reason='Measures pocket money, allowance or general family financial support without an educational purpose')
assign_domain_9_screening(variables=['W1hintdYP0j', 'W1WhyBeHS0z', 'W1WhyBeHSah', 'W1WhatiHS0q'],
    status='Out of domain', review_track='Keyword-search false positive', reason='The variable label contains a search term but does not measure parental educational attitudes, monitoring or support')
domain_9_unassigned_screening = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 screening status'].isna()].copy()
assert len(domain_9_unassigned_screening) == 0
expected_domain_9_screening_counts = {'Core review': 96, 'Contextual review': 4, 'Out of domain': 74}
actual_domain_9_screening_counts = domain_9_screened_candidates['Domain 9 screening status'].value_counts().to_dict()
assert actual_domain_9_screening_counts == expected_domain_9_screening_counts
domain_9_screening_summary = domain_9_screened_candidates.groupby(['Domain 9 screening status',
    'Domain 9 review track'], dropna=False).size().rename('Variables').reset_index().sort_values(['Domain 9 screening status',
    'Domain 9 review track']).reset_index(drop=True)
domain_9_screening_wave_summary = domain_9_screened_candidates.groupby(['Domain 9 screening status', 'Wave'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Domain 9 screening status',
    'Wave']).reset_index(drop=True)
domain_9_out_of_domain_display = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 screening status'].eq('Out of domain'),
    ['Domain 9 review track', 'Wave', 'Source type', 'Source file', 'Variable', 'Variable label',
    'Domain 9 screening reason']].sort_values(['Domain 9 review track', 'Wave', 'Variable']).reset_index(drop=True)
print(f'Domain 9 candidates screened: {len(domain_9_screened_candidates):,}')
print('\nScreening summary by review track:')
display_limited(domain_9_screening_summary)
print('\nScreening summary by wave:')
display_limited(domain_9_screening_wave_summary)
print(f'\nUnassigned screening variables: {len(domain_9_unassigned_screening):,}')
print('\nFirst 10 out-of-domain and false-positive variables:')
display_limited(domain_9_out_of_domain_display.head(10))
print(f'Additional out-of-domain variables not displayed: {max(len(domain_9_out_of_domain_display) - 10, 0):,}')
domain_9_out_of_domain_file = stage_2_output_directory / 'stage_2_parental_attitudes_out_of_domain_review.csv'
domain_9_out_of_domain_display.to_csv(domain_9_out_of_domain_file, index=False)
print(f'\nFull out-of-domain review saved to: {domain_9_out_of_domain_file}')

Domain 9 candidates screened: 174

Screening summary by review track:


,Domain 9 screening status,Domain 9 review track,Variables
0,Contextual review,Reactive school–parent contact,2
1,Contextual review,Shared family activity,1
2,Contextual review,Teacher recommendation reported by parent,1
3,Core review,Education finance expectation,1
4,Core review,Homework support and monitoring,4



Screening summary by wave:


,Domain 9 screening status,Wave,Variables
0,Contextual review,Wave 1,1
1,Contextual review,Wave 2,1
2,Contextual review,Wave 3,2
3,Core review,Wave 1,46
4,Core review,Wave 2,26



Unassigned screening variables: 0

First 10 out-of-domain and false-positive variables:


,Domain 9 review track,Wave,Source type,Source file,Variable,Variable label,Domain 9 screening reason
0,General financial support,Wave 1,Young person,wave_one_lsype_young_person_2020,W1famsupYP,YP: Whether receive any pocket money or allowa...,"Measures pocket money, allowance or general fa..."
1,General financial support,Wave 2,Young person,wave_two_lsype_young_person_2020,W2famsupYP,YP: Whether receive any pocket money or allowa...,"Measures pocket money, allowance or general fa..."
2,General financial support,Wave 3,Young person,wave_three_lsype_young_person_2020,W3famsupYP,YP: Whether YP receives any pocket money/ allo...,"Measures pocket money, allowance or general fa..."
3,Keyword-search false positive,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1WhatiHS0q,HR: Information MP used to decide school: Visi...,The variable label contains a search term but ...
4,Keyword-search false positive,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1WhyBeHS0z,HR: Wanted YP to go to current school because ...,The variable label contains a search term but ...


Additional out-of-domain variables not displayed: 64

Full out-of-domain review saved to: data_derived\stage_2_predictor_construction\stage_2_parental_attitudes_out_of_domain_review.csv


In [322]:
# 3: Parental higher-education expectation coding and routing review

import pandas as pd
parental_he_expectation_inventory = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 review track'].eq('Parental higher-education expectation')].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
expected_parental_he_variables = ['W1hepossMP', *[f'W1henotMP0{suffix}' for suffix in 'abcdefghij']]
assert len(parental_he_expectation_inventory) == 11
assert set(parental_he_expectation_inventory['Variable']) == set(expected_parental_he_variables)
assert parental_he_expectation_inventory['Source file'].nunique() == 1
parental_he_source_file = parental_he_expectation_inventory['Source file'].iloc[0]
parental_he_source_path = source_file_lookup[parental_he_source_file]
parental_he_raw = pd.read_stata(parental_he_source_path, columns=['NSID', *expected_parental_he_variables],
    convert_categoricals=False)
parental_he_labelled = pd.read_stata(parental_he_source_path, columns=['NSID', *expected_parental_he_variables],
    convert_categoricals=True)
parental_he_raw['NSID'] = standardise_nsid(parental_he_raw['NSID'])
parental_he_labelled['NSID'] = standardise_nsid(parental_he_labelled['NSID'])
assert parental_he_raw['NSID'].is_unique
assert parental_he_labelled['NSID'].is_unique
parental_he_raw = parental_he_raw.set_index('NSID').reindex(route_index)
parental_he_labelled = parental_he_labelled.set_index('NSID').reindex(route_index)
for variable in expected_parental_he_variables:
    parental_he_raw[variable] = pd.to_numeric(parental_he_raw[variable], errors='coerce')
he_expectation_raw = parental_he_raw['W1hepossMP']
he_expectation_labelled = parental_he_labelled['W1hepossMP'].astype('string')
he_expectation_distribution_rows = []
for raw_code, participants in he_expectation_raw.value_counts(dropna=False).items():
    if pd.isna(raw_code):
        value_label = 'No source record'
        response_type = 'No source record'
        sort_value = 999999
    else:
        matching_labels = he_expectation_labelled.loc[he_expectation_raw.eq(raw_code)].dropna().drop_duplicates().tolist()
        value_label = matching_labels[0] if matching_labels else str(raw_code)
        response_type = 'Observed response' if raw_code >= 0 else 'Special code'
        sort_value = float(raw_code)
    he_expectation_distribution_rows.append({'Raw code': raw_code, 'Value label': value_label,
        'Response type': response_type, 'Participants': int(participants), 'Percentage': round(participants / len(route_index) * 100,
        2), 'Sort value': sort_value})
he_expectation_distribution = pd.DataFrame(he_expectation_distribution_rows).sort_values('Sort value').drop(columns=['Sort value']).reset_index(drop=True)
he_reason_variables = [variable for variable in expected_parental_he_variables if variable != 'W1hepossMP']
he_reason_summary_rows = []
he_reason_code_rows = []
for variable in he_reason_variables:
    variable_label = parental_he_expectation_inventory.loc[parental_he_expectation_inventory['Variable'].eq(variable),
        'Variable label'].iloc[0]
    raw_values = parental_he_raw[variable]
    labelled_values = parental_he_labelled[variable].astype('string')
    he_reason_summary_rows.append({'Variable': variable, 'Variable label': variable_label,
        'Mentioned': int(raw_values.eq(1).sum()), 'Not mentioned': int(raw_values.eq(0).sum()), 'Structurally not applicable': int(raw_values.eq(-91).sum()), 'Other special codes': int((raw_values.lt(0) & ~raw_values.eq(-91)).sum()), 'No source record': int(raw_values.isna().sum()), 'Observed responses': int(raw_values.ge(0).sum())})
    for raw_code, participants in raw_values.value_counts(dropna=False).items():
        if pd.isna(raw_code):
            value_label = 'No source record'
            response_type = 'No source record'
            sort_value = 999999
        else:
            matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            response_type = 'Observed response' if raw_code >= 0 else 'Special code'
            sort_value = float(raw_code)
        he_reason_code_rows.append({'Variable': variable, 'Variable label': variable_label, 'Raw code': raw_code,
            'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
he_reason_summary = pd.DataFrame(he_reason_summary_rows).sort_values('Variable').reset_index(drop=True)
he_reason_code_distribution = pd.DataFrame(he_reason_code_rows).sort_values(['Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
he_reason_matrix = parental_he_raw[he_reason_variables].copy()
he_reason_observed_count = he_reason_matrix.ge(0).sum(axis=1)
he_reason_mentioned_count = he_reason_matrix.eq(1).sum(axis=1)
he_reason_structural_count = he_reason_matrix.eq(-91).sum(axis=1)
he_routing_category = pd.Series('Other or partial availability', index=route_index, dtype='string')
he_routing_category.loc[he_reason_observed_count.eq(10)] = 'All ten reason items observed'
he_routing_category.loc[he_reason_structural_count.eq(10)] = 'All ten reason items structurally not applicable'
he_routing_category.loc[he_reason_matrix.isna().all(axis=1)] = 'No source record for reason items'
he_routing_summary = he_routing_category.value_counts().rename('Participants').rename_axis('Reason-item routing pattern').reset_index()
he_routing_summary['Percentage'] = (he_routing_summary['Participants'] / len(route_index) * 100).round(2)
he_expectation_display_label = he_expectation_labelled.fillna('No source record')
he_expectation_routing_crosstab = pd.crosstab(he_expectation_display_label, he_routing_category, margins=True,
    dropna=False)
he_expectation_reason_count_summary = pd.DataFrame({'HE expectation': he_expectation_display_label,
    'Mentioned reasons': he_reason_mentioned_count, 'Observed reason items': he_reason_observed_count}).groupby('HE expectation',
    dropna=False).agg(Participants=('Mentioned reasons', 'size'),
    Participants_with_any_reason_mentioned=('Mentioned reasons',
    lambda values: int(values.gt(0).sum())), Mean_reasons_mentioned=('Mentioned reasons',
    'mean'), Participants_with_all_reason_items_observed=('Observed reason items',
    lambda values: int(values.eq(10).sum()))).reset_index()
he_expectation_reason_count_summary['Mean_reasons_mentioned'] = he_expectation_reason_count_summary['Mean_reasons_mentioned'].round(3)
print(f'Parental HE expectation variables reviewed: {len(parental_he_expectation_inventory)}')
print('Direct HE expectation response codes:')
with pd.option_context('display.max_colwidth', None):
    display_limited(he_expectation_distribution)
print('Routed reason-item summary:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(he_reason_summary)
print('Reason-item response codes:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(he_reason_code_distribution)
print('Reason-item routing patterns:')
display_limited(he_routing_summary)
print('HE expectation by reason-item routing:')
display_limited(he_expectation_routing_crosstab)
print('Reasons mentioned within each HE expectation response:')
display_limited(he_expectation_reason_count_summary)

Parental HE expectation variables reviewed: 11
Direct HE expectation response codes:


,Raw code,Value label,Response type,Participants,Percentage
0,-99.0,MP not interviewed,Special code,103,1.05
1,-1.0,Don't know,Special code,547,5.60
2,1.0,Very likely,Observed response,3691,37.79
3,2.0,Fairly likely,Observed response,2746,28.12
4,3.0,Not very likely,Observed response,1349,13.81


Routed reason-item summary:


,Variable,Variable label,Mentioned,Not mentioned,Structurally not applicable,Other special codes,No source record,Observed responses
0,W1henotMP0a,MP: Why think it unlikely YP will go into Higher Education: Will not get necessa,778,1659,6984,103,243,2437
1,W1henotMP0b,MP: Why think it unlikely YP will go into Higher Education: Family can't afford,244,2193,6984,103,243,2437
2,W1henotMP0c,MP: Why think it unlikely YP will go into Higher Education: YP has no interest i,1420,1017,6984,103,243,2437
3,W1henotMP0d,MP: Why think it unlikely YP will go into Higher Education: Has job in mind alre,101,2336,6984,103,243,2437
4,W1henotMP0e,"MP: Why think it unlikely YP will go into Higher Education: SEN, learning proble",36,2401,6984,103,243,2437


Reason-item response codes:


...

,Reason-item routing pattern,Participants,Percentage
0,All ten reason items structurally not applicable,6984,71.51
1,All ten reason items observed,2437,24.95
2,No source record for reason items,243,2.49
3,Other or partial availability,103,1.05


HE expectation by reason-item routing:


col_0,All ten reason items observed,All ten reason items structurally not applicable,No source record for reason items,Other or partial availability,All
W1hepossMP,,,,,
Don't know,0,547,0,0,547
Fairly likely,0,2746,0,0,2746
MP not interviewed,0,0,0,103,103
No source record,0,0,243,0,243
Not likely at all,1088,0,0,0,1088


Reasons mentioned within each HE expectation response:


,HE expectation,Participants,Participants_with_any_reason_mentioned,Mean_reasons_mentioned,Participants_with_all_reason_items_observed
0,Don't know,547,0,0.000,0
1,Fairly likely,2746,0,0.000,0
2,MP not interviewed,103,0,0.000,0
3,No source record,243,0,0.000,0
4,Not likely at all,1088,1078,1.221,1088


In [323]:
# 4: Parental higher-education expectation representation review

import pandas as pd
parental_he_expectation_candidate = he_expectation_raw.map({1.0: 4.0, 2.0: 3.0, 3.0: 2.0,
    4.0: 1.0}).astype('Float64').rename('parental_higher_education_expectation_pretransition')
assert len(parental_he_expectation_candidate) == len(route_index)
assert parental_he_expectation_candidate.index.equals(route_index)
assert set(parental_he_expectation_candidate.dropna().unique()) == {1.0, 2.0, 3.0, 4.0}
assert int(parental_he_expectation_candidate.notna().sum()) == 8874
parental_he_expectation_labels = {1.0: 'Not likely at all', 2.0: 'Not very likely', 3.0: 'Fairly likely',
    4.0: 'Very likely'}
parental_he_expectation_distribution_rows = []
for code, label in parental_he_expectation_labels.items():
    participants = int(parental_he_expectation_candidate.eq(code).sum())
    parental_he_expectation_distribution_rows.append({'Code': int(code), 'Category': label,
        'Participants': participants, 'Percentage of full sample': round(participants / len(route_index) * 100,
        2), 'Percentage among observed': round(participants / parental_he_expectation_candidate.notna().sum() * 100,
        2)})
parental_he_expectation_distribution_rows.append({'Code': pd.NA, 'Category': 'Unavailable',
    'Participants': int(parental_he_expectation_candidate.isna().sum()), 'Percentage of full sample': round(parental_he_expectation_candidate.isna().mean() * 100,
    2), 'Percentage among observed': pd.NA})
parental_he_expectation_distribution = pd.DataFrame(parental_he_expectation_distribution_rows)
parental_he_expectation_summary = pd.DataFrame([{'Predictor': 'parental_higher_education_expectation_pretransition',
    'Non-missing': int(parental_he_expectation_candidate.notna().sum()), 'Missing': int(parental_he_expectation_candidate.isna().sum()), 'Missing percentage': round(parental_he_expectation_candidate.isna().mean() * 100,
    2), 'Observed categories': int(parental_he_expectation_candidate.nunique())}])

def find_aligned_predictor(candidate_names):
    """Return an aligned predictor with one of the requested names."""
    matches = []
    for object_name, object_value in list(globals().items()):
        if isinstance(object_value, pd.Series):
            if object_value.name in candidate_names:
                candidate_series = object_value.copy().reindex(route_index)
                matches.append((object_name, object_value.name, candidate_series))
        elif isinstance(object_value, pd.DataFrame):
            for column_name in candidate_names:
                if column_name not in object_value.columns:
                    continue
                if object_value.index.equals(route_index):
                    candidate_series = object_value[column_name].copy()
                elif 'NSID' in object_value.columns:
                    candidate_table = object_value[['NSID', column_name]].copy()
                    candidate_table['NSID'] = standardise_nsid(candidate_table['NSID'])
                    if not candidate_table['NSID'].is_unique:
                        continue
                    candidate_series = candidate_table.set_index('NSID')[column_name].reindex(route_index)
                else:
                    continue
                matches.append((object_name, column_name, candidate_series))
    if len(matches) == 0:
        return (None, None, None)
    reference_series = matches[0][2]
    for _, _, comparison_series in matches[1:]:
        assert reference_series.equals(comparison_series)
    return matches[0]
related_predictor_requests = {'Young-person HE application likelihood': ['higher_education_application_likelihood_pretransition'],
    'Expected post-16 route': ['expected_post16_route_pretransition'], 'Highest parental qualification': ['highest_parental_qualification_pretransition',
    'highest_parental_qualification']}
related_predictor_series = {}
related_predictor_source_rows = []
for comparison_name, candidate_names in related_predictor_requests.items():
    source_object, source_column, comparison_series = find_aligned_predictor(candidate_names)
    if comparison_series is None:
        related_predictor_source_rows.append({'Comparison measure': comparison_name, 'Located': False,
            'Source object': pd.NA, 'Source column': pd.NA})
        continue
    related_predictor_series[comparison_name] = comparison_series
    related_predictor_source_rows.append({'Comparison measure': comparison_name, 'Located': True,
        'Source object': source_object, 'Source column': source_column})
related_predictor_sources = pd.DataFrame(related_predictor_source_rows)
parental_he_overlap_rows = []
for comparison_name, comparison_series in related_predictor_series.items():
    pair_data = pd.DataFrame({'Parental HE expectation': parental_he_expectation_candidate,
        'Comparison measure': comparison_series}, index=route_index).dropna()
    parental_he_overlap_rows.append({'Comparison measure': comparison_name, 'Complete comparisons': len(pair_data),
        'Parental HE categories': int(pair_data['Parental HE expectation'].nunique()), 'Comparison categories': int(pair_data['Comparison measure'].nunique()), "Cramer's V": round(categorical_cramers_v(pair_data['Parental HE expectation'],
        pair_data['Comparison measure']), 3)})
parental_he_overlap_summary = pd.DataFrame(parental_he_overlap_rows)
parental_he_reason_review = he_reason_summary[['Variable', 'Variable label', 'Mentioned', 'Not mentioned',
    'Observed responses']].copy()
parental_he_reason_review['Mention percentage among routed'] = (parental_he_reason_review['Mentioned'] / parental_he_reason_review['Observed responses'] * 100).round(2)
parental_he_reason_review = parental_he_reason_review.sort_values('Mentioned', ascending=False).reset_index(drop=True)
low_expectation_mask = he_expectation_raw.isin([3.0, 4.0])
all_reason_items_observed_mask = he_reason_observed_count.eq(10)
assert low_expectation_mask.equals(all_reason_items_observed_mask)
print('Parental HE expectation candidate:')
display_limited(parental_he_expectation_summary)
print('Four-level response distribution:')
display_limited(parental_he_expectation_distribution)
print('Related predictors located:')
with pd.option_context('display.max_colwidth', None):
    display_limited(related_predictor_sources)
print('Associations with related retained predictors:')
display_limited(parental_he_overlap_summary)
print('Routed reasons for low parental HE expectation:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_he_reason_review)
print(f'Low-expectation cases with all ten reason items observed: {int(all_reason_items_observed_mask.sum())}')

Parental HE expectation candidate:


,Predictor,Non-missing,Missing,Missing percentage,Observed categories
0,parental_higher_education_expectation_pretrans...,8874,893,9.14,4


Four-level response distribution:


,Code,Category,Participants,Percentage of full sample,Percentage among observed
0,1,Not likely at all,1088,11.14,12.26
1,2,Not very likely,1349,13.81,15.2
2,3,Fairly likely,2746,28.12,30.94
3,4,Very likely,3691,37.79,41.59
4,<NA>,Unavailable,893,9.14,<NA>


Related predictors located:


...

Associations with related retained predictors:


,Comparison measure,Complete comparisons,Parental HE categories,Comparison categories,Cramer's V
0,Highest parental qualification,8873,4,7,0.182


Routed reasons for low parental HE expectation:


,Variable,Variable label,Mentioned,Not mentioned,Observed responses,Mention percentage among routed
0,W1henotMP0c,MP: Why think it unlikely YP will go into Higher Education: YP has no interest i,1420,1017,2437,58.27
1,W1henotMP0a,MP: Why think it unlikely YP will go into Higher Education: Will not get necessa,778,1659,2437,31.92
2,W1henotMP0b,MP: Why think it unlikely YP will go into Higher Education: Family can't afford,244,2193,2437,10.01
3,W1henotMP0i,MP: Why think it unlikely YP will go into Higher Education: Other answers,146,2291,2437,5.99
4,W1henotMP0d,MP: Why think it unlikely YP will go into Higher Education: Has job in mind alre,101,2336,2437,4.14


Low-expectation cases with all ten reason items observed: 2437


In [324]:
# 5: Saved predictor column-name inspection

from pathlib import Path
import pandas as pd
stage_2_csv_paths = sorted(stage_2_output_directory.glob('*.csv'))
stage_2_predictor_paths = [csv_path for csv_path in stage_2_csv_paths if 'predictor' in csv_path.stem.lower()]
assert len(stage_2_predictor_paths) > 0
saved_predictor_column_rows = []
for csv_path in stage_2_predictor_paths:
    csv_columns = pd.read_csv(csv_path, nrows=0).columns.tolist()
    for column_position, column_name in enumerate(csv_columns):
        saved_predictor_column_rows.append({'File': csv_path.name, 'Column position': column_position,
            'Column': column_name})
saved_predictor_column_inventory = pd.DataFrame(saved_predictor_column_rows)
saved_predictor_file_summary = saved_predictor_column_inventory.groupby('File', dropna=False).agg(Columns=('Column',
    'size'), Column_names=('Column',
    lambda values: ', '.join(values.astype(str)))).reset_index().sort_values('File').reset_index(drop=True)
education_column_pattern = 'higher|university|education|post.?16|route|expect|aspir|application|qualification|plan'
education_related_saved_columns = saved_predictor_column_inventory.loc[saved_predictor_column_inventory['Column'].astype('string').str.contains(education_column_pattern,
    case=False, regex=True, na=False)].copy().sort_values(['File', 'Column position']).reset_index(drop=True)
domain_5_filename_pattern = 'aspiration|post16|post_16|education|plan'
likely_domain_5_files = saved_predictor_file_summary.loc[saved_predictor_file_summary['File'].str.contains(domain_5_filename_pattern,
    case=False, regex=True, na=False)].copy().reset_index(drop=True)
in_memory_related_rows = []
for object_name, object_value in list(globals().items()):
    if isinstance(object_value, pd.Series):
        series_name = str(object_value.name) if object_value.name is not None else ''
        searchable_text = f'{object_name} {series_name}'
        if pd.Series([searchable_text]).str.contains(education_column_pattern, case=False, regex=True,
            na=False).iloc[0]:
            in_memory_related_rows.append({'Object': str(object_name), 'Object type': 'Series',
                'Column or series name': series_name, 'Rows': len(object_value)})
    elif isinstance(object_value, pd.DataFrame):
        for column_name in object_value.columns:
            column_name_text = str(column_name)
            searchable_text = f'{object_name} {column_name_text}'
            if pd.Series([searchable_text]).str.contains(education_column_pattern, case=False, regex=True,
                na=False).iloc[0]:
                in_memory_related_rows.append({'Object': str(object_name), 'Object type': 'DataFrame',
                    'Column or series name': column_name_text, 'Rows': len(object_value)})
in_memory_related_predictors = pd.DataFrame(in_memory_related_rows).drop_duplicates().sort_values(['Object',
    'Column or series name']).reset_index(drop=True) if in_memory_related_rows else pd.DataFrame(columns=['Object',
    'Object type', 'Column or series name', 'Rows'])
print(f'Saved predictor files inspected: {len(stage_2_predictor_paths):,}')
print(f'Education- and pathway-related saved columns: {len(education_related_saved_columns):,}')
print(f'Likely Domain 5 predictor files: {len(likely_domain_5_files):,}')
print(f'Related in-memory objects and columns: {len(in_memory_related_predictors):,}')
print('\nFirst 10 saved predictor files:')
display_limited(saved_predictor_file_summary.head(10))
print(f'Additional saved predictor files not displayed: {max(len(saved_predictor_file_summary) - 10, 0):,}')
print('\nFirst 15 education- and pathway-related saved columns:')
display_limited(education_related_saved_columns.head(15))
print(f'Additional related saved columns not displayed: {max(len(education_related_saved_columns) - 15, 0):,}')
print('\nLikely Domain 5 predictor files:')
display_limited(likely_domain_5_files.head(10))
print(f'Additional likely Domain 5 files not displayed: {max(len(likely_domain_5_files) - 10, 0):,}')
print('\nFirst 15 related in-memory objects and columns:')
display_limited(in_memory_related_predictors.head(15))
print(f'Additional in-memory matches not displayed: {max(len(in_memory_related_predictors) - 15, 0):,}')
print('\nFiles written in this cell: 0')

Saved predictor files inspected: 9
Education- and pathway-related saved columns: 4
Likely Domain 5 predictor files: 1
Related in-memory objects and columns: 850

First 10 saved predictor files:


,File,Columns,Column_names
0,stage_2_demographic_domain_predictors.csv,6,"NSID, sex, birth_month_position, ethnicity, en..."
1,stage_2_educational_aspirations_post16_plans_d...,4,"NSID, expected_post16_route, higher_education_..."
2,stage_2_experiences_behaviours_domain_predicto...,8,"NSID, bullying_experience_pretransition, young..."
3,stage_2_family_socioeconomic_domain_predictors...,7,"NSID, highest_parental_qualification_code, fam..."
4,stage_2_prior_attainment_domain_predictors.csv,1,NSID


Additional saved predictor files not displayed: 0

First 15 education- and pathway-related saved columns:


,File,Column position,Column
0,stage_2_educational_aspirations_post16_plans_d...,1,expected_post16_route
1,stage_2_educational_aspirations_post16_plans_d...,2,higher_education_application_likelihood
2,stage_2_educational_aspirations_post16_plans_d...,3,expected_peer_post16_route
3,stage_2_family_socioeconomic_domain_predictors...,1,highest_parental_qualification_code


Additional related saved columns not displayed: 0

Likely Domain 5 predictor files:


,File,Columns,Column_names
0,stage_2_educational_aspirations_post16_plans_d...,4,"NSID, expected_post16_route, higher_education_..."


Additional likely Domain 5 files not displayed: 0

First 15 related in-memory objects and columns:


,Object,Object type,Column or series name,Rows
0,additional_school_construct_review_plan,DataFrame,Construct,4
1,additional_school_construct_review_plan,DataFrame,Review position,4
2,agreement_by_source,DataFrame,Leaving education plan,9506
3,agreement_by_source_summary,DataFrame,Plan to leave full-time education,6
4,application_label,Series,W3heposs9YP,9767


Additional in-memory matches not displayed: 835

Files written in this cell: 0


In [325]:
# 6: Parental and young-person education expectation overlap review

import pandas as pd
educational_aspiration_predictor_path = stage_2_output_directory / 'stage_2_educational_aspirations_post16_plans_domain_predictors.csv'
family_socioeconomic_predictor_path = stage_2_output_directory / 'stage_2_family_socioeconomic_domain_predictors.csv'
assert educational_aspiration_predictor_path.exists()
assert family_socioeconomic_predictor_path.exists()
saved_educational_aspiration_predictors = pd.read_csv(educational_aspiration_predictor_path, usecols=['NSID',
    'expected_post16_route', 'higher_education_application_likelihood'])
saved_family_socioeconomic_predictors = pd.read_csv(family_socioeconomic_predictor_path, usecols=['NSID',
    'highest_parental_qualification_code'])
saved_educational_aspiration_predictors['NSID'] = standardise_nsid(saved_educational_aspiration_predictors['NSID'])
saved_family_socioeconomic_predictors['NSID'] = standardise_nsid(saved_family_socioeconomic_predictors['NSID'])
assert saved_educational_aspiration_predictors['NSID'].is_unique
assert saved_family_socioeconomic_predictors['NSID'].is_unique
saved_educational_aspiration_predictors = saved_educational_aspiration_predictors.set_index('NSID').reindex(route_index)
saved_family_socioeconomic_predictors = saved_family_socioeconomic_predictors.set_index('NSID').reindex(route_index)
assert saved_educational_aspiration_predictors.index.equals(route_index)
assert saved_family_socioeconomic_predictors.index.equals(route_index)
parental_he_related_data = pd.DataFrame({'Parental HE expectation': parental_he_expectation_candidate,
    'Young-person HE application likelihood': saved_educational_aspiration_predictors['higher_education_application_likelihood'], 'Expected post-16 route': saved_educational_aspiration_predictors['expected_post16_route'], 'Highest parental qualification': saved_family_socioeconomic_predictors['highest_parental_qualification_code']}, index=route_index)
assert len(parental_he_related_data) == 9767
parental_he_related_predictor_rows = []
for predictor in parental_he_related_data.columns:
    predictor_values = parental_he_related_data[predictor]
    parental_he_related_predictor_rows.append({'Measure': predictor,
        'Non-missing': int(predictor_values.notna().sum()), 'Missing': int(predictor_values.isna().sum()), 'Missing percentage': round(predictor_values.isna().mean() * 100,
        2), 'Observed categories': int(predictor_values.nunique())})
parental_he_related_predictor_summary = pd.DataFrame(parental_he_related_predictor_rows)
parental_he_overlap_rows = []
for comparison_measure in ['Young-person HE application likelihood', 'Expected post-16 route',
    'Highest parental qualification']:
    comparison_data = parental_he_related_data[['Parental HE expectation', comparison_measure]].dropna()
    parental_he_overlap_rows.append({'Comparison measure': comparison_measure,
        'Complete comparisons': len(comparison_data), 'Parental HE categories': int(comparison_data['Parental HE expectation'].nunique()), 'Comparison categories': int(comparison_data[comparison_measure].nunique()), "Cramer's V": round(categorical_cramers_v(comparison_data['Parental HE expectation'],
        comparison_data[comparison_measure]), 3)})
parental_he_overlap_summary = pd.DataFrame(parental_he_overlap_rows)
parental_he_related_availability_count = parental_he_related_data.notna().sum(axis=1)
parental_he_related_joint_availability = parental_he_related_availability_count.value_counts().sort_index().rename('Participants').rename_axis('Related measures available').reset_index()
parental_he_related_joint_availability['Percentage'] = (parental_he_related_joint_availability['Participants'] / len(route_index) * 100).round(2)
parent_young_he_data = parental_he_related_data[['Parental HE expectation',
    'Young-person HE application likelihood']].dropna()
parent_young_he_counts = pd.crosstab(parent_young_he_data['Parental HE expectation'],
    parent_young_he_data['Young-person HE application likelihood'], margins=True, dropna=False)
parent_young_he_row_percentages = (pd.crosstab(parent_young_he_data['Parental HE expectation'],
    parent_young_he_data['Young-person HE application likelihood'], normalize='index', dropna=False) * 100).round(2)
parent_route_data = parental_he_related_data[['Parental HE expectation', 'Expected post-16 route']].dropna()
parent_route_counts = pd.crosstab(parent_route_data['Parental HE expectation'],
    parent_route_data['Expected post-16 route'], margins=True, dropna=False)
parent_route_row_percentages = (pd.crosstab(parent_route_data['Parental HE expectation'],
    parent_route_data['Expected post-16 route'], normalize='index', dropna=False) * 100).round(2)
print('Related retained predictor coverage:')
display_limited(parental_he_related_predictor_summary)
print('Associations with parental HE expectation:')
display_limited(parental_he_overlap_summary)
print('Joint availability:')
display_limited(parental_he_related_joint_availability)
print('Parental and young-person HE expectation counts:')
display_limited(parent_young_he_counts)
print('Parental and young-person HE expectation row percentages:')
display_limited(parent_young_he_row_percentages)
print('Parental HE expectation and expected post-16 route counts:')
display_limited(parent_route_counts)
print('Parental HE expectation and expected post-16 route row percentages:')
display_limited(parent_route_row_percentages)
print('Files written in this cell: 0')

Related retained predictor coverage:


,Measure,Non-missing,Missing,Missing percentage,Observed categories
0,Parental HE expectation,8874,893,9.14,4
1,Young-person HE application likelihood,9505,262,2.68,4
2,Expected post-16 route,9506,261,2.67,7
3,Highest parental qualification,9510,257,2.63,7


Associations with parental HE expectation:


,Comparison measure,Complete comparisons,Parental HE categories,Comparison categories,Cramer's V
0,Young-person HE application likelihood,8862,4,4,0.409
1,Expected post-16 route,8858,4,7,0.290
2,Highest parental qualification,8873,4,7,0.182


Joint availability:


...

Parental HE expectation and expected post-16 route counts:


Expected post-16 route,Full-time employment,Further-education college,Other or unspecified full-time education,"Other, mixed or uncertain route",School sixth form,Sixth-form college,Work-based training or employment-training,All
Parental HE expectation,,,,,,,,
1.0,120,392,64,67,148,85,206,1082
2.0,65,498,84,34,331,180,155,1347
3.0,38,655,124,29,1211,598,88,2743
4.0,14,527,121,12,2145,844,23,3686
All,237,2072,393,142,3835,1707,472,8858


Parental HE expectation and expected post-16 route row percentages:


Expected post-16 route,Full-time employment,Further-education college,Other or unspecified full-time education,"Other, mixed or uncertain route",School sixth form,Sixth-form college,Work-based training or employment-training
Parental HE expectation,,,,,,,
1.0,11.09,36.23,5.91,6.19,13.68,7.86,19.04
2.0,4.83,36.97,6.24,2.52,24.57,13.36,11.51
3.0,1.39,23.88,4.52,1.06,44.15,21.80,3.21
4.0,0.38,14.30,3.28,0.33,58.19,22.90,0.62


Files written in this cell: 0


In [326]:
# 7: Parental higher-education expectation decisions

import pandas as pd
parental_he_expectation_predictor_candidates = pd.DataFrame({'parental_higher_education_expectation_pretransition': parental_he_expectation_candidate},
    index=route_index)
assert len(parental_he_expectation_predictor_candidates) == len(route_index)
assert parental_he_expectation_predictor_candidates.index.equals(route_index)
assert list(parental_he_expectation_predictor_candidates.columns) == ['parental_higher_education_expectation_pretransition']
assert int(parental_he_expectation_predictor_candidates['parental_higher_education_expectation_pretransition'].notna().sum()) == 8874
parental_he_expectation_variable_decisions = parental_he_expectation_inventory.copy()
parental_he_expectation_variable_decisions['Parental-HE-expectation decision'] = pd.Series(pd.NA,
    index=parental_he_expectation_variable_decisions.index, dtype='string')
parental_he_expectation_variable_decisions['Parental-HE-expectation representation'] = pd.Series(pd.NA,
    index=parental_he_expectation_variable_decisions.index, dtype='string')
parental_he_expectation_variable_decisions['Parental-HE-expectation reason'] = pd.Series(pd.NA,
    index=parental_he_expectation_variable_decisions.index, dtype='string')
direct_expectation_mask = parental_he_expectation_variable_decisions['Variable'].eq('W1hepossMP')
assert int(direct_expectation_mask.sum()) == 1
parental_he_expectation_variable_decisions.loc[direct_expectation_mask,
    'Parental-HE-expectation decision'] = 'Construction input'
parental_he_expectation_variable_decisions.loc[direct_expectation_mask,
    'Parental-HE-expectation representation'] = 'parental_higher_education_expectation_pretransition'
parental_he_expectation_variable_decisions.loc[direct_expectation_mask,
    'Parental-HE-expectation reason'] = "Direct four-level main-parent expectation of whether the young person will enter higher education. The measure has substantive variation and is related to, but not redundant with, the young person's HE application likelihood, expected post-16 route or parental qualification"
reason_item_mask = parental_he_expectation_variable_decisions['Variable'].isin(he_reason_variables)
assert int(reason_item_mask.sum()) == 10
parental_he_expectation_variable_decisions.loc[reason_item_mask,
    'Parental-HE-expectation decision'] = 'Review support'
parental_he_expectation_variable_decisions.loc[reason_item_mask,
    'Parental-HE-expectation representation'] = 'Routed reasons for low parental HE expectation'
parental_he_expectation_variable_decisions.loc[reason_item_mask,
    'Parental-HE-expectation reason'] = 'Asked only when the parent reported that HE participation was not very likely or not likely at all. Separate use would repeat the low-expectation category, while several reasons have few mentioned cases; retained only to document the meaning and routing of the direct expectation item'
parental_he_expectation_unresolved = parental_he_expectation_variable_decisions.loc[parental_he_expectation_variable_decisions['Parental-HE-expectation decision'].isna()].copy()
assert len(parental_he_expectation_unresolved) == 0
assert parental_he_expectation_variable_decisions['Parental-HE-expectation decision'].value_counts().to_dict() == {'Review support': 10,
    'Construction input': 1}
parental_he_expectation_representation_decisions = pd.DataFrame([{'Representation': 'Four-level parental HE expectation',
    'Decision': 'Retain as predictor candidate', 'Reason': 'Preserves the distinction between very likely, fairly likely, not very likely and not likely at all. The measure is associated with related educational predictors without being fully redundant'}, {'Representation': 'Binary likely versus unlikely expectation',
    'Decision': 'Do not retain', 'Reason': 'Would remove meaningful differences between strong and moderate expectations while the four categories all contain sufficient cases'}, {'Representation': 'Separate low-expectation reason indicators',
    'Decision': 'Review support only', 'Reason': 'Conditionally routed to the two low-expectation categories and therefore not consistently defined across the full analysis sample'}, {'Representation': 'Count of low-expectation reasons',
    'Decision': 'Do not retain', 'Reason': 'Available only among parents reporting low HE expectation and would primarily encode that routing condition rather than a whole-sample construct'}])
parental_he_expectation_decision_summary = parental_he_expectation_variable_decisions.groupby('Parental-HE-expectation decision',
    dropna=False).size().rename('Variables').reset_index().sort_values('Parental-HE-expectation decision').reset_index(drop=True)
parental_he_expectation_decision_display = parental_he_expectation_variable_decisions.sort_values(['Source order',
    'Variable position'])[['Wave', 'Source type', 'Source file', 'Variable', 'Variable label', 'Timing status',
    'Parental-HE-expectation decision', 'Parental-HE-expectation representation', 'Parental-HE-expectation reason']].reset_index(drop=True)
print(f'Parental HE expectation predictors retained: {parental_he_expectation_predictor_candidates.shape[1]}')
print('Predictor coverage:')
display_limited(parental_he_expectation_summary)
print('Variable decision summary:')
display_limited(parental_he_expectation_decision_summary)
print('Representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_he_expectation_representation_decisions)
print(f'Unresolved parental HE expectation variables: {len(parental_he_expectation_unresolved)}')
print('Complete variable decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_he_expectation_decision_display)

Parental HE expectation predictors retained: 1
Predictor coverage:


,Predictor,Non-missing,Missing,Missing percentage,Observed categories
0,parental_higher_education_expectation_pretrans...,8874,893,9.14,4


Variable decision summary:


,Parental-HE-expectation decision,Variables
0,Construction input,1
1,Review support,10


Representation decisions:


,Representation,Decision,Reason
0,Four-level parental HE expectation,Retain as predictor candidate,"Preserves the distinction between very likely, fairly likely, not very likely and not likely at all. The measure is associated with related educational predictors without being fully redundant"
1,Binary likely versus unlikely expectation,Do not retain,Would remove meaningful differences between strong and moderate expectations while the four categories all contain sufficient cases
2,Separate low-expectation reason indicators,Review support only,Conditionally routed to the two low-expectation categories and therefore not consistently defined across the full analysis sample
3,Count of low-expectation reasons,Do not retain,Available only among parents reporting low HE expectation and would primarily encode that routing condition rather than a whole-sample construct


Unresolved parental HE expectation variables: 0
Complete variable decisions:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Parental-HE-expectation decision,Parental-HE-expectation representation,Parental-HE-expectation reason
0,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1hepossMP,MP: Likelihood of YP going into Higher Education,Pre-transition source,Construction input,parental_higher_education_expectation_pretransition,"Direct four-level main-parent expectation of whether the young person will enter higher education. The measure has substantive variation and is related to, but not redundant with, the young person's HE application likelihood, expected post-16 route or parental qualification"
1,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1henotMP0a,MP: Why think it unlikely YP will go into Higher Education: Will not get necessa,Pre-transition source,Review support,Routed reasons for low parental HE expectation,"Asked only when the parent reported that HE participation was not very likely or not likely at all. Separate use would repeat the low-expectation category, while several reasons have few mentioned cases; retained only to document the meaning and routing of the direct expectation item"
2,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1henotMP0b,MP: Why think it unlikely YP will go into Higher Education: Family can't afford,Pre-transition source,Review support,Routed reasons for low parental HE expectation,"Asked only when the parent reported that HE participation was not very likely or not likely at all. Separate use would repeat the low-expectation category, while several reasons have few mentioned cases; retained only to document the meaning and routing of the direct expectation item"
3,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1henotMP0c,MP: Why think it unlikely YP will go into Higher Education: YP has no interest i,Pre-transition source,Review support,Routed reasons for low parental HE expectation,"Asked only when the parent reported that HE participation was not very likely or not likely at all. Separate use would repeat the low-expectation category, while several reasons have few mentioned cases; retained only to document the meaning and routing of the direct expectation item"
4,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1henotMP0d,MP: Why think it unlikely YP will go into Higher Education: Has job in mind alre,Pre-transition source,Review support,Routed reasons for low parental HE expectation,"Asked only when the parent reported that HE participation was not very likely or not likely at all. Separate use would repeat the low-expectation category, while several reasons have few mentioned cases; retained only to document the meaning and routing of the direct expectation item"


In [327]:
# 8: Parental educational aspiration coding and overlap review

import pandas as pd
parental_educational_aspiration_inventory = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 review track'].eq('Parental educational aspiration')].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(parental_educational_aspiration_inventory) == 1
assert parental_educational_aspiration_inventory['Variable'].iloc[0] == 'W1patt3MP'
parental_aspiration_source_file = parental_educational_aspiration_inventory['Source file'].iloc[0]
parental_aspiration_source_path = source_file_lookup[parental_aspiration_source_file]
parental_aspiration_raw_data = pd.read_stata(parental_aspiration_source_path, columns=['NSID', 'W1patt3MP'],
    convert_categoricals=False)
parental_aspiration_labelled_data = pd.read_stata(parental_aspiration_source_path, columns=['NSID', 'W1patt3MP'],
    convert_categoricals=True)
parental_aspiration_raw_data['NSID'] = standardise_nsid(parental_aspiration_raw_data['NSID'])
parental_aspiration_labelled_data['NSID'] = standardise_nsid(parental_aspiration_labelled_data['NSID'])
assert parental_aspiration_raw_data['NSID'].is_unique
assert parental_aspiration_labelled_data['NSID'].is_unique
parental_aspiration_raw_data = parental_aspiration_raw_data.set_index('NSID').reindex(route_index)
parental_aspiration_labelled_data = parental_aspiration_labelled_data.set_index('NSID').reindex(route_index)
parental_educational_aspiration_raw = pd.to_numeric(parental_aspiration_raw_data['W1patt3MP'], errors='coerce')
parental_educational_aspiration_labelled = parental_aspiration_labelled_data['W1patt3MP'].astype('string')
parental_aspiration_code_rows = []
for raw_code, participants in parental_educational_aspiration_raw.value_counts(dropna=False).items():
    if pd.isna(raw_code):
        value_label = 'No source record'
        response_type = 'No source record'
        sort_value = 999999
    else:
        matching_labels = parental_educational_aspiration_labelled.loc[parental_educational_aspiration_raw.eq(raw_code)].dropna().drop_duplicates().tolist()
        value_label = matching_labels[0] if matching_labels else str(raw_code)
        response_type = 'Observed response' if raw_code >= 0 else 'Special code'
        sort_value = float(raw_code)
    parental_aspiration_code_rows.append({'Raw code': raw_code, 'Value label': value_label,
        'Response type': response_type, 'Participants': int(participants), 'Percentage': round(participants / len(route_index) * 100,
        2), 'Sort value': sort_value})
parental_aspiration_code_distribution = pd.DataFrame(parental_aspiration_code_rows).sort_values('Sort value').drop(columns=['Sort value']).reset_index(drop=True)
parental_educational_aspiration_observed = parental_educational_aspiration_raw.where(parental_educational_aspiration_raw.ge(0)).rename('parental_educational_aspiration_observed')
parental_aspiration_quality_summary = pd.DataFrame([{'Variable': 'W1patt3MP',
    'Variable label': parental_educational_aspiration_inventory['Variable label'].iloc[0], 'Observed responses': int(parental_educational_aspiration_observed.notna().sum()), 'Special-code responses': int(parental_educational_aspiration_raw.lt(0).sum()), 'No source record': int(parental_educational_aspiration_raw.isna().sum()), 'Observed percentage': round(parental_educational_aspiration_observed.notna().mean() * 100,
    2), 'Observed categories': int(parental_educational_aspiration_observed.nunique())}])
parental_aspiration_comparison_data = pd.DataFrame({'Parental educational aspiration': parental_educational_aspiration_observed,
    'Parental HE expectation': parental_he_expectation_candidate, 'Young-person HE application likelihood': saved_educational_aspiration_predictors['higher_education_application_likelihood'], 'Expected post-16 route': saved_educational_aspiration_predictors['expected_post16_route'], 'Highest parental qualification': saved_family_socioeconomic_predictors['highest_parental_qualification_code']}, index=route_index)
parental_aspiration_overlap_rows = []
for comparison_measure in ['Parental HE expectation', 'Young-person HE application likelihood',
    'Expected post-16 route', 'Highest parental qualification']:
    pair_data = parental_aspiration_comparison_data[['Parental educational aspiration', comparison_measure]].dropna()
    parental_aspiration_overlap_rows.append({'Comparison measure': comparison_measure,
        'Complete comparisons': len(pair_data), 'Aspiration categories': int(pair_data['Parental educational aspiration'].nunique()), 'Comparison categories': int(pair_data[comparison_measure].nunique()), "Cramer's V": round(categorical_cramers_v(pair_data['Parental educational aspiration'],
        pair_data[comparison_measure]), 3)})
parental_aspiration_overlap_summary = pd.DataFrame(parental_aspiration_overlap_rows)
parental_aspiration_he_data = parental_aspiration_comparison_data[['Parental educational aspiration',
    'Parental HE expectation']].dropna()
parental_aspiration_he_counts = pd.crosstab(parental_aspiration_he_data['Parental educational aspiration'],
    parental_aspiration_he_data['Parental HE expectation'], margins=True, dropna=False)
parental_aspiration_he_row_percentages = (pd.crosstab(parental_aspiration_he_data['Parental educational aspiration'],
    parental_aspiration_he_data['Parental HE expectation'], normalize='index', dropna=False) * 100).round(2)
print('Parental educational aspiration variable:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_aspiration_quality_summary)
print('Response codes:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_aspiration_code_distribution)
print('Associations with related retained measures:')
display_limited(parental_aspiration_overlap_summary)
print('Parental educational aspiration by parental HE expectation:')
display_limited(parental_aspiration_he_counts)
print('Row percentages:')
display_limited(parental_aspiration_he_row_percentages)
print('Files written in this cell: 0')

Parental educational aspiration variable:


,Variable,Variable label,Observed responses,Special-code responses,No source record,Observed percentage,Observed categories
0,W1patt3MP,MP: Whether want YP to have a better education than MP had,9312,212,243,95.34,4


Response codes:


,Raw code,Value label,Response type,Participants,Percentage
0,-99.0,MP not interviewed,Special code,103,1.05
1,-1.0,Don't know,Special code,109,1.12
2,1.0,Agree strongly,Observed response,7204,73.76
3,2.0,Agree a little,Observed response,1378,14.11
4,3.0,Disagree a little,Observed response,515,5.27


Associations with related retained measures:


...

Parental educational aspiration by parental HE expectation:


Parental HE expectation,1.0,2.0,3.0,4.0,All
Parental educational aspiration,,,,,
1.0,775,983,2103,2897,6758
2.0,179,220,435,489,1323
3.0,67,93,140,185,485
4.0,48,39,40,84,211
All,1069,1335,2718,3655,8777


Row percentages:


Parental HE expectation,1.0,2.0,3.0,4.0
Parental educational aspiration,,,,
1.0,11.47,14.55,31.12,42.87
2.0,13.53,16.63,32.88,36.96
3.0,13.81,19.18,28.87,38.14
4.0,22.75,18.48,18.96,39.81


Files written in this cell: 0


In [328]:
# 9: Parental educational aspiration decision

import pandas as pd
parental_educational_aspiration_candidate = parental_educational_aspiration_raw.map({1.0: 4.0, 2.0: 3.0, 3.0: 2.0,
    4.0: 1.0}).astype('Float64').rename('parental_educational_aspiration_pretransition')
assert len(parental_educational_aspiration_candidate) == len(route_index)
assert parental_educational_aspiration_candidate.index.equals(route_index)
assert set(parental_educational_aspiration_candidate.dropna().unique()) == {1.0, 2.0, 3.0, 4.0}
assert int(parental_educational_aspiration_candidate.notna().sum()) == 9312
parental_educational_aspiration_predictor_candidates = pd.DataFrame({'parental_educational_aspiration_pretransition': parental_educational_aspiration_candidate},
    index=route_index)
assert list(parental_educational_aspiration_predictor_candidates.columns) == ['parental_educational_aspiration_pretransition']
parental_educational_aspiration_labels = {1.0: 'Disagree strongly', 2.0: 'Disagree a little', 3.0: 'Agree a little',
    4.0: 'Agree strongly'}
parental_educational_aspiration_distribution_rows = []
for code, category in parental_educational_aspiration_labels.items():
    participants = int(parental_educational_aspiration_candidate.eq(code).sum())
    parental_educational_aspiration_distribution_rows.append({'Code': int(code), 'Category': category,
        'Participants': participants, 'Percentage of full sample': round(participants / len(route_index) * 100,
        2), 'Percentage among observed': round(participants / parental_educational_aspiration_candidate.notna().sum() * 100,
        2)})
parental_educational_aspiration_distribution_rows.append({'Code': pd.NA, 'Category': 'Unavailable',
    'Participants': int(parental_educational_aspiration_candidate.isna().sum()), 'Percentage of full sample': round(parental_educational_aspiration_candidate.isna().mean() * 100,
    2), 'Percentage among observed': pd.NA})
parental_educational_aspiration_distribution = pd.DataFrame(parental_educational_aspiration_distribution_rows)
parental_educational_aspiration_summary = pd.DataFrame([{'Predictor': 'parental_educational_aspiration_pretransition',
    'Non-missing': int(parental_educational_aspiration_candidate.notna().sum()), 'Missing': int(parental_educational_aspiration_candidate.isna().sum()), 'Missing percentage': round(parental_educational_aspiration_candidate.isna().mean() * 100,
    2), 'Observed categories': int(parental_educational_aspiration_candidate.nunique()), 'Agreement percentage among observed': round(parental_educational_aspiration_candidate.isin([3.0,
    4.0]).sum() / parental_educational_aspiration_candidate.notna().sum() * 100, 2)}])
parental_educational_aspiration_variable_decisions = parental_educational_aspiration_inventory.copy()
parental_educational_aspiration_variable_decisions['Parental-aspiration decision'] = 'Construction input'
parental_educational_aspiration_variable_decisions['Parental-aspiration representation'] = 'parental_educational_aspiration_pretransition'
parental_educational_aspiration_variable_decisions['Parental-aspiration reason'] = "Direct four-level main-parent report of wanting the young person to receive a better education than the parent received. The item has high coverage and is not redundant with parental HE expectation, the young person's HE application likelihood or expected post-16 route. It is interpreted as relative educational aspiration rather than an absolute expectation of HE participation"
assert len(parental_educational_aspiration_variable_decisions) == 1
assert parental_educational_aspiration_variable_decisions['Parental-aspiration decision'].eq('Construction input').all()
parental_educational_aspiration_representation_decisions = pd.DataFrame([{'Representation': 'Four-level parental educational aspiration',
    'Decision': 'Retain as predictor candidate', 'Reason': 'Preserves differences between strong and slight agreement and disagreement. All four categories contain observed cases, despite a concentration in strong agreement'}, {'Representation': 'Binary agreement indicator',
    'Decision': 'Do not retain', 'Reason': 'More than nine in ten observed parents agree, so binary coding would remove most of the remaining variation'}, {'Representation': 'Absolute parental education expectation',
    'Decision': 'Not supported by this item', 'Reason': "The wording compares the young person's education with the parent's own education and therefore measures relative educational aspiration"}])
parental_educational_aspiration_decision_display = parental_educational_aspiration_variable_decisions[['Wave',
    'Source type', 'Source file', 'Variable', 'Variable label', 'Timing status', 'Parental-aspiration decision', 'Parental-aspiration representation', 'Parental-aspiration reason']].reset_index(drop=True)
print(f'Parental educational aspiration predictors retained: {parental_educational_aspiration_predictor_candidates.shape[1]}')
print('Predictor summary:')
display_limited(parental_educational_aspiration_summary)
print('Four-level distribution:')
display_limited(parental_educational_aspiration_distribution)
print('Representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_educational_aspiration_representation_decisions)
print('Variable decision:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_educational_aspiration_decision_display)

Parental educational aspiration predictors retained: 1
Predictor summary:


,Predictor,Non-missing,Missing,Missing percentage,Observed categories,Agreement percentage among observed
0,parental_educational_aspiration_pretransition,9312,455,4.66,4,92.16


Four-level distribution:


,Code,Category,Participants,Percentage of full sample,Percentage among observed
0,1,Disagree strongly,215,2.20,2.31
1,2,Disagree a little,515,5.27,5.53
2,3,Agree a little,1378,14.11,14.8
3,4,Agree strongly,7204,73.76,77.36
4,<NA>,Unavailable,455,4.66,<NA>


Representation decisions:


,Representation,Decision,Reason
0,Four-level parental educational aspiration,Retain as predictor candidate,"Preserves differences between strong and slight agreement and disagreement. All four categories contain observed cases, despite a concentration in strong agreement"
1,Binary agreement indicator,Do not retain,"More than nine in ten observed parents agree, so binary coding would remove most of the remaining variation"
2,Absolute parental education expectation,Not supported by this item,The wording compares the young person's education with the parent's own education and therefore measures relative educational aspiration


Variable decision:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Parental-aspiration decision,Parental-aspiration representation,Parental-aspiration reason
0,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1patt3MP,MP: Whether want YP to have a better education than MP had,Pre-transition source,Construction input,parental_educational_aspiration_pretransition,"Direct four-level main-parent report of wanting the young person to receive a better education than the parent received. The item has high coverage and is not redundant with parental HE expectation, the young person's HE application likelihood or expected post-16 route. It is interpreted as relative educational aspiration rather than an absolute expectation of HE participation"


In [329]:
# 10: Parental discussion of continued education coding review

import pandas as pd
parental_education_discussion_inventory = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 review track'].eq('Parental discussion of continued education')].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
expected_parental_discussion_variables = ['W1jobdiscMP', 'W2jobdiscMP', 'W3jobdiscMP']
assert len(parental_education_discussion_inventory) == 3
assert set(parental_education_discussion_inventory['Variable']) == set(expected_parental_discussion_variables)
wave_3_pretransition_interview_mask = wave_3_pretransition_substance_mask.copy().reindex(route_index).fillna(False).astype(bool)
assert wave_3_pretransition_interview_mask.index.equals(route_index)
parental_discussion_raw_tables = {}
parental_discussion_labelled_tables = {}
parental_discussion_quality_rows = []
parental_discussion_code_rows = []
for source_file, source_inventory in parental_education_discussion_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    raw_data['NSID'] = standardise_nsid(raw_data['NSID'])
    labelled_data['NSID'] = standardise_nsid(labelled_data['NSID'])
    assert raw_data['NSID'].is_unique
    assert labelled_data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(route_index)
    labelled_data = labelled_data.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    parental_discussion_raw_tables[source_file] = raw_data
    parental_discussion_labelled_tables[source_file] = labelled_data
    for variable in source_variables:
        variable_inventory = source_inventory.loc[source_inventory['Variable'].eq(variable)].iloc[0]
        raw_values = raw_data[variable]
        labelled_values = labelled_data[variable].astype('string')
        wave_3_variable = variable_inventory['Wave'] == 'Wave 3'
        timing_eligible_mask = wave_3_pretransition_interview_mask if wave_3_variable else pd.Series(True,
            index=route_index)
        timing_eligible_values = raw_values.where(timing_eligible_mask)
        observed_mask = timing_eligible_values.ge(0).fillna(False)
        special_code_mask = timing_eligible_values.lt(0).fillna(False)
        parental_discussion_quality_rows.append({'Wave': variable_inventory['Wave'], 'Variable': variable,
            'Variable label': variable_inventory['Variable label'], 'Timing status': variable_inventory['Timing status'], 'Participants timing-eligible': int(timing_eligible_mask.sum()), 'Observed responses within timing': int(observed_mask.sum()), 'Special-code responses within timing': int(special_code_mask.sum()), 'Unavailable within timing': int((timing_eligible_mask & timing_eligible_values.isna()).sum()), 'Excluded by Wave 3 timing': int((~timing_eligible_mask).sum()), 'Observed percentage of full sample': round(observed_mask.mean() * 100,
            2), 'Distinct observed codes': int(timing_eligible_values.loc[observed_mask].nunique())})
        value_counts = timing_eligible_values.value_counts(dropna=False)
        for raw_code, participants in value_counts.items():
            if pd.isna(raw_code):
                value_label = 'Unavailable within timing'
                response_type = 'Unavailable'
                sort_value = 999999
            else:
                matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(raw_code)
                response_type = 'Observed response' if raw_code >= 0 else 'Special code'
                sort_value = float(raw_code)
            parental_discussion_code_rows.append({'Wave': variable_inventory['Wave'], 'Variable': variable,
                'Variable label': variable_inventory['Variable label'], 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
parental_discussion_quality = pd.DataFrame(parental_discussion_quality_rows).sort_values(['Wave',
    'Variable']).reset_index(drop=True)
parental_discussion_code_distribution = pd.DataFrame(parental_discussion_code_rows).sort_values(['Wave', 'Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
parental_discussion_wave_data = pd.DataFrame(index=route_index)
for variable in expected_parental_discussion_variables:
    source_file = parental_education_discussion_inventory.loc[parental_education_discussion_inventory['Variable'].eq(variable),
        'Source file'].iloc[0]
    variable_values = parental_discussion_raw_tables[source_file][variable].copy()
    if variable == 'W3jobdiscMP':
        variable_values = variable_values.where(wave_3_pretransition_interview_mask)
    parental_discussion_wave_data[variable] = variable_values
parental_discussion_wave_pairs = [('Wave 1 and Wave 2', 'W1jobdiscMP', 'W2jobdiscMP'), ('Wave 2 and eligible Wave 3',
    'W2jobdiscMP', 'W3jobdiscMP'), ('Wave 1 and eligible Wave 3', 'W1jobdiscMP', 'W3jobdiscMP')]
parental_discussion_agreement_rows = []
for comparison, first_variable, second_variable in parental_discussion_wave_pairs:
    comparison_data = parental_discussion_wave_data[[first_variable, second_variable]].dropna()
    comparison_data = comparison_data.loc[comparison_data[first_variable].ge(0) & comparison_data[second_variable].ge(0)]
    exact_agreement = comparison_data[first_variable].eq(comparison_data[second_variable]).mean() * 100 if len(comparison_data) > 0 else pd.NA
    parental_discussion_agreement_rows.append({'Comparison': comparison,
        'Complete observed comparisons': len(comparison_data), 'Exact raw-code agreement percentage': round(exact_agreement,
        2) if pd.notna(exact_agreement) else pd.NA, "Cramer's V": round(categorical_cramers_v(comparison_data[first_variable],
        comparison_data[second_variable]), 3) if len(comparison_data) > 0 else pd.NA})
parental_discussion_agreement_summary = pd.DataFrame(parental_discussion_agreement_rows)
print(f'Parental discussion variables reviewed: {len(parental_education_discussion_inventory)}')
print('Measure quality and timing:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_discussion_quality)
print('Response codes within the permitted timing window:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_discussion_code_distribution)
print('Cross-wave raw-code agreement:')
display_limited(parental_discussion_agreement_summary)
print('Files written in this cell: 0')

Parental discussion variables reviewed: 3
Measure quality and timing:


,Wave,Variable,Variable label,Timing status,Participants timing-eligible,Observed responses within timing,Special-code responses within timing,Unavailable within timing,Excluded by Wave 3 timing,Observed percentage of full sample,Distinct observed codes
0,Wave 1,W1jobdiscMP,MP: Whether talked to YP about YP staying on in full-time education,Pre-transition source,9767,997,8527,243,0,10.21,2
1,Wave 2,W2jobdiscMP,MP: Whether talked to YP about YP staying on in full-time education,Pre-transition source,9767,805,8716,246,0,8.24,2
2,Wave 3,W3jobdiscMP,MP: Whether talked to YP about YP staying on in full-time education,Near-transition source,9495,424,9071,0,272,4.34,2


Response codes within the permitted timing window:


,Wave,Variable,Variable label,Raw code,Value label,Response type,Participants
0,Wave 1,W1jobdiscMP,MP: Whether talked to YP about YP staying on in full-time education,-99.0,MP not interviewed,Special code,103
1,Wave 1,W1jobdiscMP,MP: Whether talked to YP about YP staying on in full-time education,-91.0,Not applicable,Special code,8422
2,Wave 1,W1jobdiscMP,MP: Whether talked to YP about YP staying on in full-time education,-1.0,Don't know,Special code,2
3,Wave 1,W1jobdiscMP,MP: Whether talked to YP about YP staying on in full-time education,1.0,Yes,Observed response,793
4,Wave 1,W1jobdiscMP,MP: Whether talked to YP about YP staying on in full-time education,2.0,No,Observed response,204


Cross-wave raw-code agreement:


,Comparison,Complete observed comparisons,Exact raw-code agreement percentage,Cramer's V
0,Wave 1 and Wave 2,244,78.28,0.184
1,Wave 2 and eligible Wave 3,116,90.52,0.275
2,Wave 1 and eligible Wave 3,128,83.59,0.218


Files written in this cell: 0


In [330]:
# 11: Parental education-discussion routing neighbourhood review

import pandas as pd
parental_discussion_targets = ['W1jobdiscMP', 'W2jobdiscMP', 'W3jobdiscMP']
parental_discussion_neighbourhood_rows = []
for target_variable in parental_discussion_targets:
    target_row = verified_decision_register.loc[verified_decision_register['Variable'].eq(target_variable)].copy()
    assert len(target_row) == 1
    target_source_file = target_row['Source file'].iloc[0]
    target_position = int(target_row['Variable position'].iloc[0])
    neighbourhood = verified_decision_register.loc[verified_decision_register['Source file'].eq(target_source_file) & verified_decision_register['Variable position'].between(target_position - 8,
        target_position + 8)].copy().sort_values('Variable position')
    neighbourhood['Target variable'] = target_variable
    neighbourhood['Relative position'] = neighbourhood['Variable position'] - target_position
    parental_discussion_neighbourhood_rows.append(neighbourhood)
parental_discussion_neighbourhood = pd.concat(parental_discussion_neighbourhood_rows, ignore_index=True,
    sort=False).sort_values(['Target variable', 'Variable position']).reset_index(drop=True)
assert parental_discussion_neighbourhood['Target variable'].nunique() == 3
assert set(parental_discussion_targets).issubset(set(parental_discussion_neighbourhood['Variable']))
routing_search_pattern = 'whether|likely|stay|education|school|college|leave|job|work|plan|expect|discuss|talk|applicable'
parental_discussion_neighbourhood['Possible routing relevance'] = parental_discussion_neighbourhood['Variable label'].fillna('').str.contains(routing_search_pattern,
    case=False, regex=True, na=False)
parental_discussion_neighbourhood_display = parental_discussion_neighbourhood[['Target variable', 'Wave',
    'Source file', 'Relative position', 'Variable position', 'Variable', 'Variable label', 'Data type', 'Possible routing relevance', 'Review outcome', 'Substantive domain']].copy()
parental_discussion_preceding_variables = parental_discussion_neighbourhood.loc[parental_discussion_neighbourhood['Relative position'].between(-5,
    -1)][['Target variable', 'Wave', 'Relative position', 'Variable', 'Variable label',
    'Possible routing relevance']].sort_values(['Target variable', 'Relative position']).reset_index(drop=True)
print('Discussion-variable neighbourhoods:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_discussion_neighbourhood_display)
print('Variables immediately preceding each discussion item:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_discussion_preceding_variables)
print('Files written in this cell: 0')

Discussion-variable neighbourhoods:


,Target variable,Wave,Source file,Relative position,Variable position,Variable,Variable label,Data type,Possible routing relevance,Review outcome,Substantive domain
0,W1jobdiscMP,Wave 1,wave_one_lsype_parental_attitudes_file_16_05_08,-8,77,W1y10futMP,MP: How important think Year 10 choices are to YP's options at age 16,int8,False,Pending review,NaN
1,W1jobdiscMP,Wave 1,wave_one_lsype_parental_attitudes_file_16_05_08,-7,78,W1pattintMP,MP: Agreement with statement: Young people don't get enough advice about what to,int8,False,Pending review,NaN
2,W1jobdiscMP,Wave 1,wave_one_lsype_parental_attitudes_file_16_05_08,-6,79,W1subjcho1MP,MP: Agreement with statement: I don't know enough about modern qualifications to,int8,False,Pending review,NaN
3,W1jobdiscMP,Wave 1,wave_one_lsype_parental_attitudes_file_16_05_08,-5,80,W1patt1MP,MP: Agreement with statement: About education/work/training for young people: No,int8,True,Pending review,NaN
4,W1jobdiscMP,Wave 1,wave_one_lsype_parental_attitudes_file_16_05_08,-4,81,W1patt2MP,MP: Agreement with statement: Leaving school at 16 limits young people's career,int8,True,Pending review,NaN


Variables immediately preceding each discussion item:


,Target variable,Wave,Relative position,Variable,Variable label,Possible routing relevance
0,W1jobdiscMP,Wave 1,-5,W1patt1MP,MP: Agreement with statement: About education/work/training for young people: No,True
1,W1jobdiscMP,Wave 1,-4,W1patt2MP,MP: Agreement with statement: Leaving school at 16 limits young people's career,True
2,W1jobdiscMP,Wave 1,-3,W1patt3MP,MP: Whether want YP to have a better education than MP had,True
3,W1jobdiscMP,Wave 1,-2,W1parasp2MP,MP: What would like YP to do when reach school leaving age,True
4,W1jobdiscMP,Wave 1,-1,W1parasp1MP,MP: What think YP will do when reaches school leaving age,True


Files written in this cell: 0


In [331]:
# 12: Parental education-discussion routing-condition review

import pandas as pd
parental_discussion_routing_variables = {'Wave 1': ['W1parasp2MP', 'W1parasp1MP', 'W1jobdiscMP'],
    'Wave 2': ['W2parasp2MP', 'W2parasp1MP', 'W2jobdiscMP'], 'Wave 3': ['W3tranaftMP', 'W3tranleaMP', 'W3parasp2MP',
    'W3parasp1MP', 'W3jobdiscMP']}
parental_discussion_routing_raw = {}
parental_discussion_routing_labelled = {}
routing_code_rows = []
routing_crosstab_rows = []
for wave, variable_names in parental_discussion_routing_variables.items():
    source_file = parental_education_discussion_inventory.loc[parental_education_discussion_inventory['Wave'].eq(wave),
        'Source file'].iloc[0]
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *variable_names], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *variable_names], convert_categoricals=True)
    raw_data['NSID'] = standardise_nsid(raw_data['NSID'])
    labelled_data['NSID'] = standardise_nsid(labelled_data['NSID'])
    assert raw_data['NSID'].is_unique
    assert labelled_data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(route_index)
    labelled_data = labelled_data.set_index('NSID').reindex(route_index)
    for variable in variable_names:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    if wave == 'Wave 3':
        raw_data = raw_data.where(wave_3_pretransition_interview_mask, pd.NA)
        labelled_data = labelled_data.where(wave_3_pretransition_interview_mask, pd.NA)
    parental_discussion_routing_raw[wave] = raw_data
    parental_discussion_routing_labelled[wave] = labelled_data
    jobdisc_variable = [variable for variable in variable_names if 'jobdisc' in variable.lower()][0]
    jobdisc_raw = raw_data[jobdisc_variable]
    jobdisc_status = pd.Series('Other unavailable', index=route_index, dtype='string')
    jobdisc_status.loc[jobdisc_raw.eq(1)] = 'Observed: yes'
    jobdisc_status.loc[jobdisc_raw.eq(2)] = 'Observed: no'
    jobdisc_status.loc[jobdisc_raw.eq(-91)] = 'Structurally not applicable'
    jobdisc_status.loc[jobdisc_raw.isna()] = 'No record or outside timing'
    for variable in variable_names:
        raw_values = raw_data[variable]
        labelled_values = labelled_data[variable].astype('string')
        for raw_code, participants in raw_values.value_counts(dropna=False).items():
            if pd.isna(raw_code):
                value_label = 'No record or outside timing'
                response_type = 'Unavailable'
                sort_value = 999999
            else:
                matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(raw_code)
                response_type = 'Observed response' if raw_code >= 0 else 'Special code'
                sort_value = float(raw_code)
            routing_code_rows.append({'Wave': wave, 'Variable': variable, 'Raw code': raw_code,
                'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
    routing_candidate_variables = [variable for variable in variable_names if variable != jobdisc_variable]
    for routing_variable in routing_candidate_variables:
        routing_labels = labelled_data[routing_variable].astype('string').fillna('No record or outside timing')
        routing_cross_tabulation = pd.crosstab(routing_labels, jobdisc_status, dropna=False)
        routing_cross_tabulation = routing_cross_tabulation.reset_index().rename(columns={routing_variable: 'Routing-variable response'})
        routing_cross_tabulation['Wave'] = wave
        routing_cross_tabulation['Routing variable'] = routing_variable
        routing_crosstab_rows.append(routing_cross_tabulation)
parental_discussion_routing_codes = pd.DataFrame(routing_code_rows).sort_values(['Wave', 'Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
parental_discussion_routing_crosstabs = pd.concat(routing_crosstab_rows, ignore_index=True, sort=False)
routing_determinism_rows = []
for wave, variable_names in parental_discussion_routing_variables.items():
    raw_data = parental_discussion_routing_raw[wave]
    jobdisc_variable = [variable for variable in variable_names if 'jobdisc' in variable.lower()][0]
    jobdisc_observed = raw_data[jobdisc_variable].isin([1.0, 2.0])
    for routing_variable in [variable for variable in variable_names if variable != jobdisc_variable]:
        routing_values = raw_data[routing_variable]
        routing_summary = pd.DataFrame({'Routing code': routing_values,
            'Job discussion observed': jobdisc_observed}).dropna(subset=['Routing code']).groupby('Routing code',
            dropna=False).agg(Participants=('Job discussion observed', 'size'),
            Discussion_observed=('Job discussion observed', 'sum')).reset_index()
        routing_summary['Discussion observed percentage'] = (routing_summary['Discussion_observed'] / routing_summary['Participants'] * 100).round(2)
        for _, summary_row in routing_summary.iterrows():
            routing_determinism_rows.append({'Wave': wave, 'Routing variable': routing_variable,
                'Routing code': summary_row['Routing code'], 'Participants': int(summary_row['Participants']), 'Discussion observed': int(summary_row['Discussion_observed']), 'Discussion observed percentage': summary_row['Discussion observed percentage']})
parental_discussion_routing_determinism = pd.DataFrame(routing_determinism_rows).sort_values(['Wave',
    'Routing variable', 'Routing code']).reset_index(drop=True)
print(f'Routing-code records: {len(parental_discussion_routing_codes):,}')
print(f'Routing cross-tabulation records: {len(parental_discussion_routing_crosstabs):,}')
print(f'Routing-determinism records: {len(parental_discussion_routing_determinism):,}')
print('\nPossible routing-variable response codes:')
display_limited(parental_discussion_routing_codes.head(10))
print(f'Additional routing-code records not displayed: {max(len(parental_discussion_routing_codes) - 10, 0):,}')
print('\nDiscussion-item observability by routing code:')
display_limited(parental_discussion_routing_determinism)
print('\nFirst 10 routing cross-tabulation records:')
display_limited(parental_discussion_routing_crosstabs.head(10))
print(f'Additional cross-tabulation records not displayed: {max(len(parental_discussion_routing_crosstabs) - 10, 0):,}')
print('\nFiles written in this cell: 0')

Routing-code records: 85
Routing cross-tabulation records: 67
Routing-determinism records: 59

Possible routing-variable response codes:


,Wave,Variable,Raw code,Value label,Response type,Participants
0,Wave 1,W1jobdiscMP,-99.0,MP not interviewed,Special code,103
1,Wave 1,W1jobdiscMP,-91.0,Not applicable,Special code,8422
2,Wave 1,W1jobdiscMP,-1.0,Don't know,Special code,2
3,Wave 1,W1jobdiscMP,1.0,Yes,Observed response,793
4,Wave 1,W1jobdiscMP,2.0,No,Observed response,204


Additional routing-code records not displayed: 75

Discussion-item observability by routing code:


,Wave,Routing variable,Routing code,Participants,Discussion observed,Discussion observed percentage
0,Wave 1,W1parasp1MP,-99.0,103,0,0.00
1,Wave 1,W1parasp1MP,-92.0,1,0,0.00
2,Wave 1,W1parasp1MP,-1.0,525,327,62.29
3,Wave 1,W1parasp1MP,1.0,7252,0,0.00
4,Wave 1,W1parasp1MP,2.0,797,313,39.27



First 10 routing cross-tabulation records:


col_0,Routing-variable response,No record or outside timing,Observed: no,Observed: yes,Other unavailable,Structurally not applicable,Wave,Routing variable
0,Continue in full time education,0,204,793,2,6978,Wave 1,W1parasp2MP
1,Don't know,0,0,0,0,165,Wave 1,W1parasp2MP
2,Get a full-time paid job (either as an employe...,0,0,0,0,107,Wave 1,W1parasp2MP
3,Interviewer missed question,0,0,0,0,7,Wave 1,W1parasp2MP
4,MP not interviewed,0,0,0,103,0,Wave 1,W1parasp2MP


Additional cross-tabulation records not displayed: 57

Files written in this cell: 0


In [332]:
# 13: Parental education-discussion decisions

import pandas as pd
parental_discussion_routing_validation_rows = []
for wave, variable_names in parental_discussion_routing_variables.items():
    raw_data = parental_discussion_routing_raw[wave]
    jobdisc_variable = [variable for variable in variable_names if 'jobdisc' in variable.lower()][0]
    parasp1_variable = [variable for variable in variable_names if 'parasp1' in variable.lower()][0]
    parasp2_variable = [variable for variable in variable_names if 'parasp2' in variable.lower()][0]
    jobdisc_observed = raw_data[jobdisc_variable].isin([1.0, 2.0])
    desired_full_time_education = raw_data[parasp2_variable].eq(1.0)
    expected_full_time_education = raw_data[parasp1_variable].eq(1.0)
    aspiration_discrepancy = desired_full_time_education & ~expected_full_time_education
    observed_outside_desired_education = int((jobdisc_observed & ~desired_full_time_education).sum())
    observed_when_expected_education = int((jobdisc_observed & expected_full_time_education).sum())
    observed_with_discrepancy = int((jobdisc_observed & aspiration_discrepancy).sum())
    total_observed = int(jobdisc_observed.sum())
    assert observed_outside_desired_education == 0
    assert observed_when_expected_education == 0
    assert observed_with_discrepancy == total_observed
    parental_discussion_routing_validation_rows.append({'Wave': wave, 'Discussion variable': jobdisc_variable,
        'Observed responses': total_observed, 'Observed outside desired full-time education': observed_outside_desired_education, 'Observed when full-time education expected': observed_when_expected_education, 'Observed within aspiration discrepancy': observed_with_discrepancy, 'Routing interpretation': 'Asked when the parent wanted the young person to continue in full-time education but did not expect that outcome'})
parental_discussion_routing_validation = pd.DataFrame(parental_discussion_routing_validation_rows)
parental_education_discussion_variable_decisions = parental_education_discussion_inventory.copy()
parental_education_discussion_variable_decisions['Parental-discussion decision'] = 'Review support'
parental_education_discussion_variable_decisions['Parental-discussion representation'] = 'Conditionally routed discussion following a discrepancy between parental aspiration and expectation'
parental_education_discussion_variable_decisions['Parental-discussion reason'] = 'The item was not asked of the full sample. It was observed only when the parent wanted the young person to continue in full-time education but did not expect that outcome. It therefore does not provide a general measure of parent-child educational discussion or parental support'
wave_3_decision_mask = parental_education_discussion_variable_decisions['Wave'].eq('Wave 3')
parental_education_discussion_variable_decisions.loc[wave_3_decision_mask,
    'Parental-discussion reason'] = parental_education_discussion_variable_decisions.loc[wave_3_decision_mask,
    'Parental-discussion reason'] + ' The Wave 3 item is also subject to the permitted pre-transition interview-timing restriction'
assert len(parental_education_discussion_variable_decisions) == 3
assert parental_education_discussion_variable_decisions['Parental-discussion decision'].eq('Review support').all()
parental_education_discussion_representation_decisions = pd.DataFrame([{'Representation': 'Latest-wave parent-child discussion indicator',
    'Decision': 'Do not retain', 'Reason': 'The three repeated items are conditionally routed and do not represent discussion across the full analysis sample'}, {'Representation': 'Any discussion across Waves 1–3',
    'Decision': 'Do not retain', 'Reason': 'Combining the waves would encode repeated eligibility for the aspiration–expectation discrepancy rather than general parental support'}, {'Representation': 'Wave-specific discussion responses',
    'Decision': 'Review support only', 'Reason': 'Useful for documenting how parents responded after the routing condition was met, but not suitable as whole-sample predictors'}])
parental_education_discussion_decision_display = parental_education_discussion_variable_decisions[['Wave',
    'Source type', 'Source file', 'Variable', 'Variable label', 'Timing status', 'Parental-discussion decision', 'Parental-discussion representation', 'Parental-discussion reason']].sort_values(['Wave',
    'Variable']).reset_index(drop=True)
print('Parental education-discussion predictors retained: 0')
print('Routing validation:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_discussion_routing_validation)
print('Representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_education_discussion_representation_decisions)
print('Variable decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_education_discussion_decision_display)
print('Files written in this cell: 0')

Parental education-discussion predictors retained: 0
Routing validation:


,Wave,Discussion variable,Observed responses,Observed outside desired full-time education,Observed when full-time education expected,Observed within aspiration discrepancy,Routing interpretation
0,Wave 1,W1jobdiscMP,997,0,0,997,Asked when the parent wanted the young person to continue in full-time education but did not expect that outcome
1,Wave 2,W2jobdiscMP,805,0,0,805,Asked when the parent wanted the young person to continue in full-time education but did not expect that outcome
2,Wave 3,W3jobdiscMP,424,0,0,424,Asked when the parent wanted the young person to continue in full-time education but did not expect that outcome


Representation decisions:


,Representation,Decision,Reason
0,Latest-wave parent-child discussion indicator,Do not retain,The three repeated items are conditionally routed and do not represent discussion across the full analysis sample
1,Any discussion across Waves 1–3,Do not retain,Combining the waves would encode repeated eligibility for the aspiration–expectation discrepancy rather than general parental support
2,Wave-specific discussion responses,Review support only,"Useful for documenting how parents responded after the routing condition was met, but not suitable as whole-sample predictors"


Variable decisions:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Parental-discussion decision,Parental-discussion representation,Parental-discussion reason
0,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1jobdiscMP,MP: Whether talked to YP about YP staying on in full-time education,Pre-transition source,Review support,Conditionally routed discussion following a discrepancy between parental aspiration and expectation,The item was not asked of the full sample. It was observed only when the parent wanted the young person to continue in full-time education but did not expect that outcome. It therefore does not provide a general measure of parent-child educational discussion or parental support
1,Wave 2,Parental attitudes,wave_two_lsype_parental_attitudes_file_16_06_08,W2jobdiscMP,MP: Whether talked to YP about YP staying on in full-time education,Pre-transition source,Review support,Conditionally routed discussion following a discrepancy between parental aspiration and expectation,The item was not asked of the full sample. It was observed only when the parent wanted the young person to continue in full-time education but did not expect that outcome. It therefore does not provide a general measure of parent-child educational discussion or parental support
2,Wave 3,Parental attitudes,wave_three_lsype_parental_attitudes_file_16_06_08,W3jobdiscMP,MP: Whether talked to YP about YP staying on in full-time education,Near-transition source,Review support,Conditionally routed discussion following a discrepancy between parental aspiration and expectation,The item was not asked of the full sample. It was observed only when the parent wanted the young person to continue in full-time education but did not expect that outcome. It therefore does not provide a general measure of parent-child educational discussion or parental support The Wave 3 item is also subject to the permitted pre-transition interview-timing restriction


Files written in this cell: 0


In [333]:
# 14: Planned financial-support track diagnostic review

import pandas as pd
domain_9_financial_diagnostic = domain_9_screened_candidates.copy()
domain_9_financial_diagnostic['Normalised review track'] = domain_9_financial_diagnostic['Domain 9 review track'].astype('string').str.strip().str.replace('\\s+',
    ' ', regex=True).str.lower()
domain_9_review_track_counts = domain_9_financial_diagnostic.groupby(['Domain 9 review track',
    'Normalised review track'], dropna=False).size().rename('Variables').reset_index().sort_values(['Variables',
    'Domain 9 review track'], ascending=[False, True]).reset_index(drop=True)
planned_financial_support_inventory = domain_9_financial_diagnostic.loc[domain_9_financial_diagnostic['Normalised review track'].eq('planned financial support')].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
financial_search_text = domain_9_financial_diagnostic['Variable'].fillna('').astype('string') + ' ' + domain_9_financial_diagnostic['Variable label'].fillna('').astype('string')
financial_variable_pattern = 'fefin|finance|financial|expense|expenses|fund|funding|support|contribution|pay|paid|loan|grant|bursary|scholarship|\\bema\\b'
finance_related_domain_9_variables = domain_9_financial_diagnostic.loc[financial_search_text.str.contains(financial_variable_pattern,
    case=False, regex=True, na=False)].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
finance_related_outside_planned_track = finance_related_domain_9_variables.loc[~finance_related_domain_9_variables['Normalised review track'].eq('planned financial support')].copy().reset_index(drop=True)
finance_related_track_summary = finance_related_domain_9_variables.groupby(['Domain 9 screening status',
    'Domain 9 review track'], dropna=False).size().rename('Variables').reset_index().sort_values(['Variables',
    'Domain 9 review track'], ascending=[False, True]).reset_index(drop=True)
planned_financial_support_diagnostic_display = planned_financial_support_inventory[['Wave', 'Source type',
    'Source file', 'Variable position', 'Variable', 'Variable label', 'Timing status', 'Domain 9 screening status', 'Domain 9 review track']].copy()
finance_related_outside_track_display = finance_related_outside_planned_track[['Wave', 'Source type', 'Source file',
    'Variable position', 'Variable', 'Variable label', 'Timing status', 'Domain 9 screening status', 'Domain 9 review track', 'Domain 9 screening reason']].copy()
print('All Domain 9 review-track counts:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_9_review_track_counts)
print(f"Variables assigned to the normalised 'planned financial support' track: {len(planned_financial_support_inventory)}")
print('Variables in the intended track:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(planned_financial_support_diagnostic_display)
print('Finance-related variables by assigned review track:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(finance_related_track_summary)
print(f'Finance-related variables outside the intended track: {len(finance_related_outside_planned_track)}')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(finance_related_outside_track_display)
print('Files written in this cell: 0')

All Domain 9 review-track counts:


,Domain 9 review track,Normalised review track,Variables
0,Planned financial support for continued education,planned financial support for continued education,54
1,Parental education history,parental education history,35
2,Teacher and careers guidance,teacher and careers guidance,19
3,Young-person educational plans,young-person educational plans,13
4,Parental higher-education expectation,parental higher-education expectation,11


Variables assigned to the normalised 'planned financial support' track: 0
Variables in the intended track:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Domain 9 screening status,Domain 9 review track


Finance-related variables by assigned review track:


,Domain 9 screening status,Domain 9 review track,Variables
0,Core review,Planned financial support for continued education,31
1,Out of domain,General financial support,3
2,Core review,Education finance expectation,1
3,Out of domain,Young-person educational plans,1


Finance-related variables outside the intended track: 36


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Domain 9 screening status,Domain 9 review track,Domain 9 screening reason
0,Wave 1,Young person,wave_one_lsype_young_person_2020,304,W1famsupYP,YP: Whether receive any pocket money or allowance or support from parents or rel,Pre-transition source,Out of domain,General financial support,"Measures pocket money, allowance or general family financial support without an educational purpose"
1,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,86,W1fefinMP0a,MP: How YP's expenses would be paid if stayed on in education - EMA,Pre-transition source,Core review,Planned financial support for continued education,Parental reports of expected funding sources and actions intended to support continued participation in education
2,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,87,W1fefinMP0b,MP: How YP's expenses would be paid if stayed on in education - YP get job or wo,Pre-transition source,Core review,Planned financial support for continued education,Parental reports of expected funding sources and actions intended to support continued participation in education
3,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,88,W1fefinMP0c,MP: How YP's expenses would be paid if stayed on in education - Parent(s) will s,Pre-transition source,Core review,Planned financial support for continued education,Parental reports of expected funding sources and actions intended to support continued participation in education
4,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,89,W1fefinMP0d,MP: How YP's expenses would be paid if stayed on in education - Other relative(s,Pre-transition source,Core review,Planned financial support for continued education,Parental reports of expected funding sources and actions intended to support continued participation in education


Files written in this cell: 0


In [334]:
# 15: Planned educational financial-support battery structure

import re
import pandas as pd
planned_financial_support_inventory = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 review track'].astype('string').str.strip().eq('Planned financial support for continued education')].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(planned_financial_support_inventory) == 54
planned_financial_support_inventory['Battery stem'] = planned_financial_support_inventory['Variable'].astype('string').str.replace('0[a-z]$',
    '', regex=True)
planned_financial_support_inventory['Item suffix'] = planned_financial_support_inventory['Variable'].astype('string').str.extract('(0[a-z])$',
    expand=False)
planned_financial_support_inventory['Battery construct'] = pd.Series(pd.NA,
    index=planned_financial_support_inventory.index, dtype='string')
expected_funding_source_mask = planned_financial_support_inventory['Variable'].astype('string').str.contains('fefin',
    case=False, regex=True, na=False)
parental_action_mask = planned_financial_support_inventory['Variable'].astype('string').str.contains('fefn2',
    case=False, regex=True, na=False)
planned_financial_support_inventory.loc[expected_funding_source_mask,
    'Battery construct'] = 'Expected funding sources if the young person remained in education'
planned_financial_support_inventory.loc[parental_action_mask,
    'Battery construct'] = 'Actions the parent might take to help keep the young person in education'
assert planned_financial_support_inventory['Battery construct'].notna().all()
assert not (expected_funding_source_mask & parental_action_mask).any()

def extract_financial_support_item(variable_label):
    """Extract the response option from the repeated item label."""
    label_text = str(variable_label)
    if ' - ' in label_text:
        return label_text.split(' - ', maxsplit=1)[1].strip()
    return label_text
planned_financial_support_inventory['Item description'] = planned_financial_support_inventory['Variable label'].map(extract_financial_support_item)
planned_financial_support_battery_summary = planned_financial_support_inventory.groupby(['Wave', 'Source type',
    'Source file', 'Battery stem', 'Battery construct'], dropna=False).agg(Variables=('Variable', 'size'),
    First_position=('Variable position', 'min'), Last_position=('Variable position',
    'max'), First_source_order=('Source order', 'min')).reset_index().sort_values(['First_source_order',
    'First_position']).reset_index(drop=True)
planned_financial_support_battery_summary['Expected contiguous positions'] = planned_financial_support_battery_summary['Last_position'] - planned_financial_support_battery_summary['First_position'] + 1
planned_financial_support_battery_summary['Contiguous block'] = planned_financial_support_battery_summary['Variables'].eq(planned_financial_support_battery_summary['Expected contiguous positions'])
planned_financial_support_wave_summary = planned_financial_support_inventory.groupby(['Wave', 'Source type'],
    dropna=False).agg(Variables=('Variable', 'size'), Batteries=('Battery stem', 'nunique'),
    Source_files=('Source file', 'nunique')).reset_index().sort_values(['Wave', 'Source type']).reset_index(drop=True)
planned_financial_support_item_comparison = planned_financial_support_inventory.pivot_table(index=['Battery construct',
    'Item suffix'], columns='Wave', values='Item description', aggfunc='first', dropna=False).reset_index()
planned_financial_support_display = planned_financial_support_inventory[['Wave', 'Source type', 'Source file',
    'Variable position', 'Variable', 'Variable label', 'Data type', 'Timing status', 'Battery stem', 'Item suffix', 'Battery construct', 'Item description']].copy()
print(f'Planned financial-support variables reviewed: {len(planned_financial_support_inventory)}')
print('Wave-level structure:')
display_limited(planned_financial_support_wave_summary)
print('Battery structure:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(planned_financial_support_battery_summary)
print('Item descriptions across waves:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(planned_financial_support_item_comparison)
print('Complete variable inventory:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(planned_financial_support_display)
print('Files written in this cell: 0')

Planned financial-support variables reviewed: 54
Wave-level structure:


,Wave,Source type,Variables,Batteries,Source_files
0,Wave 1,Parental attitudes,18,2,1
1,Wave 2,Parental attitudes,18,2,1
2,Wave 3,Parental attitudes,18,2,1


Battery structure:


,Wave,Source type,Source file,Battery stem,Battery construct,Variables,First_position,Last_position,First_source_order,Expected contiguous positions,Contiguous block
0,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1fefinMP,Expected funding sources if the young person remained in education,8,86,93,3,8,True
1,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1fefn2MP,Actions the parent might take to help keep the young person in education,10,94,103,3,10,True
2,Wave 2,Parental attitudes,wave_two_lsype_parental_attitudes_file_16_06_08,W2FeFinMP,Expected funding sources if the young person remained in education,7,53,59,6,7,True
3,Wave 2,Parental attitudes,wave_two_lsype_parental_attitudes_file_16_06_08,W2FEfn2MP,Actions the parent might take to help keep the young person in education,11,60,70,6,11,True
4,Wave 3,Parental attitudes,wave_three_lsype_parental_attitudes_file_16_06_08,W3fefinMP,Expected funding sources if the young person remained in education,7,59,65,10,7,True


Item descriptions across waves:


Wave,Battery construct,Item suffix,Wave 1,Wave 2,Wave 3
0,Actions the parent might take to help keep the young person in education,0a,Save money now specifically,Save money now specifically,Save money now specifically
1,Actions the parent might take to help keep the young person in education,0b,Give money from existing sa,Give money from existing sa,Give money from existing sa
2,Actions the parent might take to help keep the young person in education,0c,Support out of wages or ear,Support out of wages or ear,Support out of wages or ear
3,Actions the parent might take to help keep the young person in education,0d,Take out loan or remortgage,Take out loan or remortgage,Take out loan or remortgage
4,Actions the parent might take to help keep the young person in education,0e,Pay school or college fees,Pay school or college fees,Pay school or college fees


Complete variable inventory:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Data type,Timing status,Battery stem,Item suffix,Battery construct,Item description
0,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,86,W1fefinMP0a,MP: How YP's expenses would be paid if stayed on in education - EMA,int8,Pre-transition source,W1fefinMP,0a,Expected funding sources if the young person remained in education,EMA
1,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,87,W1fefinMP0b,MP: How YP's expenses would be paid if stayed on in education - YP get job or wo,int8,Pre-transition source,W1fefinMP,0b,Expected funding sources if the young person remained in education,YP get job or wo
2,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,88,W1fefinMP0c,MP: How YP's expenses would be paid if stayed on in education - Parent(s) will s,int8,Pre-transition source,W1fefinMP,0c,Expected funding sources if the young person remained in education,Parent(s) will s
3,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,89,W1fefinMP0d,MP: How YP's expenses would be paid if stayed on in education - Other relative(s,int8,Pre-transition source,W1fefinMP,0d,Expected funding sources if the young person remained in education,Other relative(s
4,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,90,W1fefinMP0e,MP: How YP's expenses would be paid if stayed on in education - Some other means,int8,Pre-transition source,W1fefinMP,0e,Expected funding sources if the young person remained in education,Some other means


Files written in this cell: 0


In [335]:
# 16: Planned educational financial-support coding and routing review

import pandas as pd
planned_financial_support_raw_tables = {}
planned_financial_support_labelled_tables = {}
for source_file, source_inventory in planned_financial_support_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    raw_data['NSID'] = standardise_nsid(raw_data['NSID'])
    labelled_data['NSID'] = standardise_nsid(labelled_data['NSID'])
    assert raw_data['NSID'].is_unique
    assert labelled_data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(route_index)
    labelled_data = labelled_data.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    planned_financial_support_raw_tables[source_file] = raw_data
    planned_financial_support_labelled_tables[source_file] = labelled_data
financial_support_code_rows = []
financial_support_battery_rows = []
financial_support_battery_status = {}
for (wave, source_file, battery_stem,
    battery_construct), battery_inventory in planned_financial_support_inventory.groupby(['Wave', 'Source file',
    'Battery stem', 'Battery construct'], sort=False):
    battery_variables = battery_inventory['Variable'].tolist()
    raw_battery = planned_financial_support_raw_tables[source_file][battery_variables].copy()
    labelled_battery = planned_financial_support_labelled_tables[source_file][battery_variables].copy()
    timing_mask = pd.Series(True, index=route_index, dtype=bool)
    if wave == 'Wave 3':
        timing_mask = wave_3_pretransition_interview_mask.reindex(route_index).fillna(False).astype(bool)
    timed_raw_battery = raw_battery.where(timing_mask, pd.NA)
    outside_timing_mask = ~timing_mask
    no_record_mask = timing_mask & timed_raw_battery.isna().all(axis=1)
    all_structural_na_mask = timing_mask & timed_raw_battery.eq(-91).all(axis=1)
    any_observed_response_mask = timing_mask & timed_raw_battery.ge(0).any(axis=1)
    all_items_observed_mask = timing_mask & timed_raw_battery.ge(0).all(axis=1)
    mixed_observed_special_mask = any_observed_response_mask & timed_raw_battery.lt(0).any(axis=1)
    other_special_pattern_mask = timing_mask & ~no_record_mask & ~all_structural_na_mask & ~any_observed_response_mask
    battery_status = pd.Series('Other or partial special-code pattern', index=route_index, dtype='string')
    battery_status.loc[outside_timing_mask] = 'Outside permitted timing'
    battery_status.loc[no_record_mask] = 'No source record'
    battery_status.loc[all_structural_na_mask] = 'All items structurally not applicable'
    battery_status.loc[any_observed_response_mask] = 'At least one observed battery response'
    financial_support_battery_status[wave, battery_construct] = battery_status
    financial_support_battery_rows.append({'Wave': wave, 'Battery construct': battery_construct,
        'Variables': len(battery_variables), 'Participants within timing': int(timing_mask.sum()), 'Outside permitted timing': int(outside_timing_mask.sum()), 'No source record': int(no_record_mask.sum()), 'All items structurally not applicable': int(all_structural_na_mask.sum()), 'At least one observed battery response': int(any_observed_response_mask.sum()), 'All items have non-negative responses': int(all_items_observed_mask.sum()), 'Mixed observed and special codes': int(mixed_observed_special_mask.sum()), 'Other or partial special-code pattern': int(other_special_pattern_mask.sum())})
    for variable in battery_variables:
        raw_values = timed_raw_battery[variable]
        labelled_values = labelled_battery[variable].astype('string')
        variable_label = battery_inventory.loc[battery_inventory['Variable'].eq(variable), 'Variable label'].iloc[0]
        item_description = battery_inventory.loc[battery_inventory['Variable'].eq(variable),
            'Item description'].iloc[0]
        for raw_code, participants in raw_values.value_counts(dropna=False).items():
            if pd.isna(raw_code):
                value_label = 'No record or outside timing'
                response_type = 'Unavailable'
                sort_value = 999999
            else:
                matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(raw_code)
                response_type = 'Observed response' if raw_code >= 0 else 'Special code'
                sort_value = float(raw_code)
            financial_support_code_rows.append({'Wave': wave, 'Battery construct': battery_construct,
                'Variable': variable, 'Item description': item_description, 'Variable label': variable_label, 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
financial_support_battery_availability = pd.DataFrame(financial_support_battery_rows).sort_values(['Wave',
    'Battery construct']).reset_index(drop=True)
financial_support_code_distribution = pd.DataFrame(financial_support_code_rows).sort_values(['Wave',
    'Battery construct', 'Variable', 'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
financial_support_code_label_summary = financial_support_code_distribution.groupby(['Wave', 'Battery construct',
    'Raw code', 'Value label', 'Response type'], dropna=False).agg(Variables_using_code=('Variable', 'nunique'),
    Total_responses=('Participants', 'sum')).reset_index().sort_values(['Wave', 'Battery construct', 'Raw code'],
    na_position='last').reset_index(drop=True)
financial_support_routing_rows = []
for (wave, battery_construct), battery_status in financial_support_battery_status.items():
    routing_raw = parental_discussion_routing_raw[wave]
    routing_labelled = parental_discussion_routing_labelled[wave]
    parasp_variables = [variable for variable in routing_raw.columns if 'parasp1' in variable.lower() or 'parasp2' in variable.lower()]
    assert len(parasp_variables) == 2
    for routing_variable in parasp_variables:
        routing_values = routing_raw[routing_variable]
        routing_labels = routing_labelled[routing_variable].astype('string')
        for raw_code, participants in routing_values.value_counts(dropna=False).items():
            routing_code_mask = routing_values.isna() if pd.isna(raw_code) else routing_values.eq(raw_code)
            matching_labels = routing_labels.loc[routing_code_mask].dropna().drop_duplicates().tolist()
            routing_value_label = 'No record or outside timing' if pd.isna(raw_code) else matching_labels[0] if matching_labels else str(raw_code)
            observed_battery_count = int((routing_code_mask & battery_status.eq('At least one observed battery response')).sum())
            structural_na_count = int((routing_code_mask & battery_status.eq('All items structurally not applicable')).sum())
            financial_support_routing_rows.append({'Wave': wave, 'Battery construct': battery_construct,
                'Routing variable': routing_variable, 'Routing code': raw_code, 'Routing response': routing_value_label, 'Participants': int(participants), 'Battery observed': observed_battery_count, 'Battery observed percentage': round(observed_battery_count / participants * 100,
                2) if participants > 0 else pd.NA, 'All items structurally not applicable': structural_na_count})
financial_support_routing_summary = pd.DataFrame(financial_support_routing_rows).sort_values(['Wave',
    'Battery construct', 'Routing variable', 'Routing code'], na_position='last').reset_index(drop=True)
print('Battery availability and response structure:')
with pd.option_context('display.max_colwidth', None):
    display_limited(financial_support_battery_availability)
print('Raw-code meanings across variables:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(financial_support_code_label_summary)
print('Complete item-level code distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(financial_support_code_distribution)
print('Battery observability by parental aspiration and expectation:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(financial_support_routing_summary)
print('Files written in this cell: 0')

Battery availability and response structure:


,Wave,Battery construct,Variables,Participants within timing,Outside permitted timing,No source record,All items structurally not applicable,At least one observed battery response,All items have non-negative responses,Mixed observed and special codes,Other or partial special-code pattern
0,Wave 1,Actions the parent might take to help keep the young person in education,10,9767,0,243,1467,7954,7954,0,103
1,Wave 1,Expected funding sources if the young person remained in education,8,9767,0,243,0,9421,9421,0,103
2,Wave 2,Actions the parent might take to help keep the young person in education,11,9767,0,246,1182,8260,8260,0,79
3,Wave 2,Expected funding sources if the young person remained in education,7,9767,0,246,0,9442,9442,0,79
4,Wave 3,Actions the parent might take to help keep the young person in education,11,9495,272,0,1113,8301,8301,0,81


Raw-code meanings across variables:


,Wave,Battery construct,Raw code,Value label,Response type,Variables_using_code,Total_responses
0,Wave 1,Actions the parent might take to help keep the young person in education,-99.0,MP not interviewed,Special code,10,1030
1,Wave 1,Actions the parent might take to help keep the young person in education,-91.0,Not applicable,Special code,10,14670
2,Wave 1,Actions the parent might take to help keep the young person in education,0.0,Not mentioned,Observed response,10,61654
3,Wave 1,Actions the parent might take to help keep the young person in education,1.0,Mentioned,Observed response,10,17886
4,Wave 1,Actions the parent might take to help keep the young person in education,NaN,No record or outside timing,Unavailable,10,2430


Complete item-level code distributions:


,Wave,Battery construct,Variable,Item description,Variable label,Raw code,Value label,Response type,Participants
0,Wave 1,Actions the parent might take to help keep the young person in education,W1fefn2MP0a,Save money now specifically,MP: What likely to do to help keep YP in education - Save money now specifically,-99.0,MP not interviewed,Special code,103
1,Wave 1,Actions the parent might take to help keep the young person in education,W1fefn2MP0a,Save money now specifically,MP: What likely to do to help keep YP in education - Save money now specifically,-91.0,Not applicable,Special code,1467
2,Wave 1,Actions the parent might take to help keep the young person in education,W1fefn2MP0a,Save money now specifically,MP: What likely to do to help keep YP in education - Save money now specifically,0.0,Not mentioned,Observed response,5904
3,Wave 1,Actions the parent might take to help keep the young person in education,W1fefn2MP0a,Save money now specifically,MP: What likely to do to help keep YP in education - Save money now specifically,1.0,Mentioned,Observed response,2050
4,Wave 1,Actions the parent might take to help keep the young person in education,W1fefn2MP0a,Save money now specifically,MP: What likely to do to help keep YP in education - Save money now specifically,NaN,No record or outside timing,Unavailable,243


Battery observability by parental aspiration and expectation:


,Wave,Battery construct,Routing variable,Routing code,Routing response,Participants,Battery observed,Battery observed percentage,All items structurally not applicable
0,Wave 1,Actions the parent might take to help keep the young person in education,W1parasp1MP,-99.0,MP not interviewed,103,0,0.00,0
1,Wave 1,Actions the parent might take to help keep the young person in education,W1parasp1MP,-92.0,Refused,1,0,0.00,1
2,Wave 1,Actions the parent might take to help keep the young person in education,W1parasp1MP,-1.0,Don't know,525,385,73.33,140
3,Wave 1,Actions the parent might take to help keep the young person in education,W1parasp1MP,1.0,Continue in full time education,7252,6296,86.82,956
4,Wave 1,Actions the parent might take to help keep the young person in education,W1parasp1MP,2.0,Start learning a trade/ Get a place on a training course,797,632,79.30,165


Files written in this cell: 0


In [336]:
# 17: Parental financial-support action routing validation

import pandas as pd
parental_contribution_expectation_by_wave = {}
parental_support_action_count_by_wave = {}
parental_support_action_routing_rows = []
parental_support_action_item_rows = []
parental_support_action_count_rows = []
for wave in ['Wave 1', 'Wave 2', 'Wave 3']:
    wave_inventory = planned_financial_support_inventory.loc[planned_financial_support_inventory['Wave'].eq(wave)].copy()
    source_files = wave_inventory['Source file'].drop_duplicates().tolist()
    assert len(source_files) == 1
    source_file = source_files[0]
    wave_raw_data = planned_financial_support_raw_tables[source_file].copy()
    timing_mask = pd.Series(True, index=route_index, dtype=bool)
    if wave == 'Wave 3':
        timing_mask = wave_3_pretransition_interview_mask.reindex(route_index).fillna(False).astype(bool)
    parental_contribution_inventory = wave_inventory.loc[wave_inventory['Battery construct'].eq('Expected funding sources if the young person remained in education') & wave_inventory['Item suffix'].eq('0c')].copy()
    assert len(parental_contribution_inventory) == 1
    parental_contribution_variable = parental_contribution_inventory['Variable'].iloc[0]
    parental_contribution_values = wave_raw_data[parental_contribution_variable].where(timing_mask)
    parental_contribution_expectation_by_wave[wave] = parental_contribution_values.where(parental_contribution_values.isin([0.0,
        1.0])).astype('Float64')
    action_inventory = wave_inventory.loc[wave_inventory['Battery construct'].eq('Actions the parent might take to help keep the young person in education')].copy().sort_values('Variable position')
    action_variables = action_inventory['Variable'].tolist()
    action_values = wave_raw_data[action_variables].where(timing_mask)
    parent_support_selected = parental_contribution_values.eq(1.0)
    parent_support_not_selected = parental_contribution_values.eq(0.0)
    main_parent_not_interviewed = parental_contribution_values.eq(-99.0)
    contribution_unavailable = parental_contribution_values.isna()
    all_action_items_observed = action_values.ge(0).all(axis=1)
    all_action_items_structural_na = action_values.eq(-91).all(axis=1)
    all_action_items_parent_not_interviewed = action_values.eq(-99).all(axis=1)
    all_action_items_unavailable = action_values.isna().all(axis=1)
    assert parent_support_selected.equals(all_action_items_observed)
    assert parent_support_not_selected.equals(all_action_items_structural_na)
    assert main_parent_not_interviewed.equals(all_action_items_parent_not_interviewed)
    assert contribution_unavailable.equals(all_action_items_unavailable)
    parental_support_action_routing_rows.append({'Wave': wave,
        'Parental-contribution variable': parental_contribution_variable, 'Parent support selected': int(parent_support_selected.sum()), 'Action battery observed': int(all_action_items_observed.sum()), 'Parent support not selected': int(parent_support_not_selected.sum()), 'Action battery structurally not applicable': int(all_action_items_structural_na.sum()), 'Main parent not interviewed': int(main_parent_not_interviewed.sum()), 'Action battery parent not interviewed': int(all_action_items_parent_not_interviewed.sum()), 'No record or outside timing': int(contribution_unavailable.sum()), 'Action battery unavailable': int(all_action_items_unavailable.sum()), 'Exact routing match': True})
    for _, action_row in action_inventory.iterrows():
        action_variable = action_row['Variable']
        action_suffix = action_row['Item suffix']
        action_description = action_row['Item description']
        mentioned_count = int((parent_support_selected & action_values[action_variable].eq(1.0)).sum())
        not_mentioned_count = int((parent_support_selected & action_values[action_variable].eq(0.0)).sum())
        routed_count = int(parent_support_selected.sum())
        parental_support_action_item_rows.append({'Wave': wave, 'Variable': action_variable,
            'Item suffix': action_suffix, 'Item description': action_description, 'Action type': 'Substantive support action' if action_suffix in ['0a',
            '0b', '0c', '0d', '0e', '0f', '0g', '0h'] else 'Non-substantive response option', 'Routed parents': routed_count, 'Mentioned': mentioned_count, 'Not mentioned': not_mentioned_count, 'Mention percentage among routed': round(mentioned_count / routed_count * 100,
            2)})
    substantive_action_variables = action_inventory.loc[action_inventory['Item suffix'].isin(['0a', '0b', '0c', '0d',
        '0e', '0f', '0g', '0h']), 'Variable'].tolist()
    assert len(substantive_action_variables) == 8
    substantive_action_count = action_values[substantive_action_variables].eq(1.0).sum(axis=1).where(parent_support_selected).astype('Float64')
    parental_support_action_count_by_wave[wave] = substantive_action_count
    count_distribution = substantive_action_count.value_counts(dropna=False).sort_index(na_position='last')
    for action_count, participants in count_distribution.items():
        parental_support_action_count_rows.append({'Wave': wave, 'Substantive actions mentioned': action_count,
            'Participants': int(participants), 'Percentage among routed': round(participants / parent_support_selected.sum() * 100,
            2) if pd.notna(action_count) else pd.NA})
parental_support_action_routing_validation = pd.DataFrame(parental_support_action_routing_rows)
parental_support_action_item_summary = pd.DataFrame(parental_support_action_item_rows).sort_values(['Wave',
    'Item suffix']).reset_index(drop=True)
parental_support_action_count_distribution = pd.DataFrame(parental_support_action_count_rows).sort_values(['Wave',
    'Substantive actions mentioned'], na_position='last').reset_index(drop=True)
print('Parental-action battery routing validation:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_support_action_routing_validation)
print('Action options among parents expecting to provide support:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_support_action_item_summary)
print('Substantive support-action count distributions:')
with pd.option_context('display.max_rows', None):
    display_limited(parental_support_action_count_distribution)
print('Files written in this cell: 0')

Parental-action battery routing validation:


,Wave,Parental-contribution variable,Parent support selected,Action battery observed,Parent support not selected,Action battery structurally not applicable,Main parent not interviewed,Action battery parent not interviewed,No record or outside timing,Action battery unavailable,Exact routing match
0,Wave 1,W1fefinMP0c,7954,7954,1467,1467,103,103,243,243,True
1,Wave 2,W2FeFinMP0c,8260,8260,1182,1182,79,79,246,246,True
2,Wave 3,W3fefinMP0c,8301,8301,1113,1113,81,81,272,272,True


Action options among parents expecting to provide support:


,Wave,Variable,Item suffix,Item description,Action type,Routed parents,Mentioned,Not mentioned,Mention percentage among routed
0,Wave 1,W1fefn2MP0a,0a,Save money now specifically,Substantive support action,7954,2050,5904,25.77
1,Wave 1,W1fefn2MP0b,0b,Give money from existing sa,Substantive support action,7954,1813,6141,22.79
2,Wave 1,W1fefn2MP0c,0c,Support out of wages or ear,Substantive support action,7954,5926,2028,74.50
3,Wave 1,W1fefn2MP0d,0d,Take out loan or remortgage,Substantive support action,7954,434,7520,5.46
4,Wave 1,W1fefn2MP0e,0e,Pay school or college fees,Substantive support action,7954,1880,6074,23.64


Substantive support-action count distributions:


,Wave,Substantive actions mentioned,Participants,Percentage among routed
0,Wave 1,0.0,204,2.56
1,Wave 1,1.0,2683,33.73
2,Wave 1,2.0,2186,27.48
3,Wave 1,3.0,1578,19.84
4,Wave 1,4.0,818,10.28


Files written in this cell: 0


In [337]:
# 18: Parental educational financial-support representation review

import pandas as pd
from scipy.stats import spearmanr
parental_contribution_wave_data = pd.DataFrame({wave: parental_contribution_expectation_by_wave[wave] for wave in ['Wave 1',
    'Wave 2', 'Wave 3']}, index=route_index)
parental_support_action_count_wave_data = pd.DataFrame({wave: parental_support_action_count_by_wave[wave] for wave in ['Wave 1',
    'Wave 2', 'Wave 3']}, index=route_index)
assert parental_contribution_wave_data.index.equals(route_index)
assert parental_support_action_count_wave_data.index.equals(route_index)
for wave in ['Wave 1', 'Wave 2', 'Wave 3']:
    contribution_values = parental_contribution_wave_data[wave]
    action_count_values = parental_support_action_count_wave_data[wave]
    expected_action_count_availability = contribution_values.eq(1.0).fillna(False).astype(bool)
    assert action_count_values.notna().equals(expected_action_count_availability)
parental_contribution_wave_pairs = [('Wave 1 and Wave 2', 'Wave 1', 'Wave 2'), ('Wave 2 and eligible Wave 3', 'Wave 2',
    'Wave 3'), ('Wave 1 and eligible Wave 3', 'Wave 1', 'Wave 3')]
parental_contribution_agreement_rows = []
for comparison, first_wave, second_wave in parental_contribution_wave_pairs:
    pair_data = parental_contribution_wave_data[[first_wave, second_wave]].dropna()
    assert len(pair_data) > 0
    parental_contribution_agreement_rows.append({'Comparison': comparison, 'Complete comparisons': len(pair_data),
        'Exact agreement percentage': round(pair_data[first_wave].eq(pair_data[second_wave]).mean() * 100,
        2), "Cramer's V": round(categorical_cramers_v(pair_data[first_wave], pair_data[second_wave]),
        3), 'Support expected in first wave': int(pair_data[first_wave].eq(1.0).sum()), 'Support expected in second wave': int(pair_data[second_wave].eq(1.0).sum())})
parental_contribution_agreement_summary = pd.DataFrame(parental_contribution_agreement_rows)
parental_action_count_agreement_rows = []
for comparison, first_wave, second_wave in parental_contribution_wave_pairs:
    pair_data = parental_support_action_count_wave_data[[first_wave, second_wave]].dropna()
    if len(pair_data) >= 2 and pair_data[first_wave].nunique() > 1 and (pair_data[second_wave].nunique() > 1):
        rank_correlation = float(spearmanr(pair_data[first_wave], pair_data[second_wave]).statistic)
        rank_correlation = round(rank_correlation, 3) if pd.notna(rank_correlation) else pd.NA
    else:
        rank_correlation = pd.NA
    parental_action_count_agreement_rows.append({'Comparison': comparison,
        'Parents routed in both waves': len(pair_data), 'Exact action-count agreement percentage': round(pair_data[first_wave].eq(pair_data[second_wave]).mean() * 100,
        2) if len(pair_data) > 0 else pd.NA, 'Spearman correlation': rank_correlation, 'Mean actions in first wave': round(pair_data[first_wave].mean(),
        2) if len(pair_data) > 0 else pd.NA, 'Mean actions in second wave': round(pair_data[second_wave].mean(),
        2) if len(pair_data) > 0 else pd.NA})
parental_action_count_agreement_summary = pd.DataFrame(parental_action_count_agreement_rows)

def construct_parental_financial_support_profile(contribution_values, action_count_values):
    """Combine contribution expectation and routed action breadth."""
    profile = pd.Series(pd.NA, index=contribution_values.index, dtype='Int64')
    no_contribution_mask = contribution_values.eq(0.0).fillna(False).astype(bool)
    contribution_expected_mask = contribution_values.eq(1.0).fillna(False).astype(bool)
    zero_action_mask = action_count_values.eq(0.0).fillna(False).astype(bool)
    one_action_mask = action_count_values.eq(1.0).fillna(False).astype(bool)
    two_action_mask = action_count_values.eq(2.0).fillna(False).astype(bool)
    three_or_more_action_mask = action_count_values.ge(3.0).fillna(False).astype(bool)
    profile.loc[no_contribution_mask] = 0
    profile.loc[contribution_expected_mask & zero_action_mask] = 1
    profile.loc[contribution_expected_mask & one_action_mask] = 2
    profile.loc[contribution_expected_mask & two_action_mask] = 3
    profile.loc[contribution_expected_mask & three_or_more_action_mask] = 4
    return profile
parental_financial_support_profile_by_wave = {}
for wave in ['Wave 1', 'Wave 2', 'Wave 3']:
    parental_financial_support_profile_by_wave[wave] = construct_parental_financial_support_profile(parental_contribution_wave_data[wave],
        parental_support_action_count_wave_data[wave])
parental_financial_support_profile_wave_data = pd.DataFrame(parental_financial_support_profile_by_wave,
    index=route_index)
for wave in ['Wave 1', 'Wave 2', 'Wave 3']:
    expected_profile_availability = parental_contribution_wave_data[wave].notna()
    assert parental_financial_support_profile_wave_data[wave].notna().equals(expected_profile_availability)
parental_financial_support_profile_agreement_rows = []
for comparison, first_wave, second_wave in parental_contribution_wave_pairs:
    pair_data = parental_financial_support_profile_wave_data[[first_wave, second_wave]].dropna()
    assert len(pair_data) > 0
    parental_financial_support_profile_agreement_rows.append({'Comparison': comparison,
        'Complete comparisons': len(pair_data), 'Exact profile agreement percentage': round(pair_data[first_wave].eq(pair_data[second_wave]).mean() * 100,
        2), "Cramer's V": round(categorical_cramers_v(pair_data[first_wave], pair_data[second_wave]), 3)})
parental_financial_support_profile_agreement = pd.DataFrame(parental_financial_support_profile_agreement_rows)
parental_financial_support_profile_labels = {0: 'No parental contribution expected',
    1: 'Contribution expected; no substantive action mentioned', 2: 'Contribution expected; one substantive action', 3: 'Contribution expected; two substantive actions', 4: 'Contribution expected; three or more substantive actions'}
wave_profile_distribution_rows = []
for wave in ['Wave 1', 'Wave 2', 'Wave 3']:
    wave_profile = parental_financial_support_profile_wave_data[wave]
    observed_count = int(wave_profile.notna().sum())
    for code, category in parental_financial_support_profile_labels.items():
        participants = int(wave_profile.eq(code).sum())
        wave_profile_distribution_rows.append({'Wave': wave, 'Code': code, 'Category': category,
            'Participants': participants, 'Percentage among observed': round(participants / observed_count * 100, 2)})
parental_financial_support_wave_profile_distribution = pd.DataFrame(wave_profile_distribution_rows)
latest_parental_contribution_expectation = pd.Series(pd.NA, index=route_index, dtype='Float64')
latest_parental_financial_support_source = pd.Series(pd.NA, index=route_index, dtype='string')
for wave in ['Wave 3', 'Wave 2', 'Wave 1']:
    wave_values = parental_contribution_wave_data[wave]
    source_selection_mask = latest_parental_contribution_expectation.isna() & wave_values.notna()
    latest_parental_contribution_expectation.loc[source_selection_mask] = wave_values.loc[source_selection_mask]
    latest_parental_financial_support_source.loc[source_selection_mask] = wave
latest_parental_support_action_count = pd.Series(pd.NA, index=route_index, dtype='Float64')
for wave in ['Wave 3', 'Wave 2', 'Wave 1']:
    selected_wave_mask = latest_parental_financial_support_source.eq(wave).fillna(False).astype(bool)
    latest_parental_support_action_count.loc[selected_wave_mask] = parental_support_action_count_wave_data.loc[selected_wave_mask,
        wave]
expected_latest_action_count_availability = latest_parental_contribution_expectation.eq(1.0).fillna(False).astype(bool)
assert latest_parental_support_action_count.notna().equals(expected_latest_action_count_availability)
latest_parental_financial_support_profile_candidate = construct_parental_financial_support_profile(latest_parental_contribution_expectation,
    latest_parental_support_action_count).rename('parental_educational_financial_support_profile_pretransition')
assert latest_parental_financial_support_profile_candidate.notna().equals(latest_parental_contribution_expectation.notna())
assert set(latest_parental_financial_support_profile_candidate.dropna().unique()).issubset({0, 1, 2, 3, 4})
latest_parental_financial_support_source_summary = latest_parental_financial_support_source.fillna('Unavailable').value_counts().rename('Participants').rename_axis('Construction source').reset_index()
source_order = {'Wave 3': 1, 'Wave 2': 2, 'Wave 1': 3, 'Unavailable': 4}
latest_parental_financial_support_source_summary['Sort order'] = latest_parental_financial_support_source_summary['Construction source'].map(source_order)
latest_parental_financial_support_source_summary['Percentage'] = (latest_parental_financial_support_source_summary['Participants'] / len(route_index) * 100).round(2)
latest_parental_financial_support_source_summary = latest_parental_financial_support_source_summary.sort_values('Sort order').drop(columns=['Sort order']).reset_index(drop=True)
latest_parental_financial_support_profile_rows = []
latest_profile_observed_count = int(latest_parental_financial_support_profile_candidate.notna().sum())
for code, category in parental_financial_support_profile_labels.items():
    participants = int(latest_parental_financial_support_profile_candidate.eq(code).sum())
    latest_parental_financial_support_profile_rows.append({'Code': code, 'Category': category,
        'Participants': participants, 'Percentage of full sample': round(participants / len(route_index) * 100,
        2), 'Percentage among observed': round(participants / latest_profile_observed_count * 100, 2)})
latest_parental_financial_support_profile_rows.append({'Code': pd.NA, 'Category': 'Unavailable',
    'Participants': int(latest_parental_financial_support_profile_candidate.isna().sum()), 'Percentage of full sample': round(latest_parental_financial_support_profile_candidate.isna().mean() * 100,
    2), 'Percentage among observed': pd.NA})
latest_parental_financial_support_profile_distribution = pd.DataFrame(latest_parental_financial_support_profile_rows)
latest_parental_financial_support_profile_summary = pd.DataFrame([{'Candidate predictor': 'parental_educational_financial_support_profile_pretransition',
    'Non-missing': latest_profile_observed_count, 'Missing': int(latest_parental_financial_support_profile_candidate.isna().sum()), 'Missing percentage': round(latest_parental_financial_support_profile_candidate.isna().mean() * 100,
    2), 'Observed categories': int(latest_parental_financial_support_profile_candidate.nunique())}])
print('Expected parental-contribution agreement:')
display_limited(parental_contribution_agreement_summary)
print('Support-action count agreement among routed parents:')
display_limited(parental_action_count_agreement_summary)
print('Combined profile agreement:')
display_limited(parental_financial_support_profile_agreement)
print('Wave-specific profile distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_financial_support_wave_profile_distribution)
print('Latest permitted construction sources:')
display_limited(latest_parental_financial_support_source_summary)
print('Latest profile candidate summary:')
display_limited(latest_parental_financial_support_profile_summary)
print('Latest profile distribution:')
with pd.option_context('display.max_colwidth', None):
    display_limited(latest_parental_financial_support_profile_distribution)
print('Files written in this cell: 0')

Expected parental-contribution agreement:


,Comparison,Complete comparisons,Exact agreement percentage,Cramer's V,Support expected in first wave,Support expected in second wave
0,Wave 1 and Wave 2,9354,82.57,0.279,7902,8192
1,Wave 2 and eligible Wave 3,9349,85.36,0.313,8184,8249
2,Wave 1 and eligible Wave 3,9321,82.05,0.241,7880,8227


Support-action count agreement among routed parents:


,Comparison,Parents routed in both waves,Exact action-count agreement percentage,Spearman correlation,Mean actions in first wave,Mean actions in second wave
0,Wave 1 and Wave 2,7232,33.84,0.332,2.28,2.23
1,Wave 2 and eligible Wave 3,7532,39.98,0.430,2.20,2.37
2,Wave 1 and eligible Wave 3,7217,31.44,0.289,2.27,2.38


Combined profile agreement:


...

Latest profile candidate summary:


,Candidate predictor,Non-missing,Missing,Missing percentage,Observed categories
0,parental_educational_financial_support_profile...,9516,251,2.57,5


Latest profile distribution:


,Code,Category,Participants,Percentage of full sample,Percentage among observed
0,0,No parental contribution expected,1133,11.60,11.91
1,1,Contribution expected; no substantive action mentioned,114,1.17,1.2
2,2,Contribution expected; one substantive action,1960,20.07,20.6
3,3,Contribution expected; two substantive actions,3188,32.64,33.5
4,4,Contribution expected; three or more substantive actions,3121,31.95,32.8


Files written in this cell: 0


In [338]:
# 19: Parental financial-support profile overlap review

import pandas as pd
family_socioeconomic_overlap_data = pd.read_csv(family_socioeconomic_predictor_path, usecols=['NSID',
    'highest_parental_qualification_code', 'family_nssec_code', 'household_income_band', 'housing_tenure'])
family_socioeconomic_overlap_data['NSID'] = standardise_nsid(family_socioeconomic_overlap_data['NSID'])
assert family_socioeconomic_overlap_data['NSID'].is_unique
family_socioeconomic_overlap_data = family_socioeconomic_overlap_data.set_index('NSID').reindex(route_index)
assert family_socioeconomic_overlap_data.index.equals(route_index)
parental_financial_support_overlap_data = pd.DataFrame({'Parental financial-support profile': latest_parental_financial_support_profile_candidate,
    'Household income band': family_socioeconomic_overlap_data['household_income_band'], 'Family NS-SEC': family_socioeconomic_overlap_data['family_nssec_code'], 'Highest parental qualification': family_socioeconomic_overlap_data['highest_parental_qualification_code'], 'Housing tenure': family_socioeconomic_overlap_data['housing_tenure'], 'Parental HE expectation': parental_he_expectation_candidate, 'Parental educational aspiration': parental_educational_aspiration_candidate, 'Young-person HE application likelihood': saved_educational_aspiration_predictors['higher_education_application_likelihood'], 'Expected post-16 route': saved_educational_aspiration_predictors['expected_post16_route']}, index=route_index)
assert len(parental_financial_support_overlap_data) == 9767
parental_financial_support_comparison_rows = []
for measure in parental_financial_support_overlap_data.columns:
    measure_values = parental_financial_support_overlap_data[measure]
    parental_financial_support_comparison_rows.append({'Measure': measure,
        'Non-missing': int(measure_values.notna().sum()), 'Missing': int(measure_values.isna().sum()), 'Missing percentage': round(measure_values.isna().mean() * 100,
        2), 'Observed categories': int(measure_values.nunique())})
parental_financial_support_comparison_summary = pd.DataFrame(parental_financial_support_comparison_rows)
parental_financial_support_association_rows = []
for comparison_measure in ['Household income band', 'Family NS-SEC', 'Highest parental qualification',
    'Housing tenure', 'Parental HE expectation', 'Parental educational aspiration', 'Young-person HE application likelihood', 'Expected post-16 route']:
    pair_data = parental_financial_support_overlap_data[['Parental financial-support profile',
        comparison_measure]].dropna()
    assert len(pair_data) > 0
    parental_financial_support_association_rows.append({'Comparison measure': comparison_measure,
        'Complete comparisons': len(pair_data), 'Financial-support categories': int(pair_data['Parental financial-support profile'].nunique()), 'Comparison categories': int(pair_data[comparison_measure].nunique()), "Cramer's V": round(categorical_cramers_v(pair_data['Parental financial-support profile'],
        pair_data[comparison_measure]), 3)})
parental_financial_support_association_summary = pd.DataFrame(parental_financial_support_association_rows).sort_values("Cramer's V",
    ascending=False).reset_index(drop=True)
financial_support_income_data = parental_financial_support_overlap_data[['Parental financial-support profile',
    'Household income band']].dropna()
financial_support_income_counts = pd.crosstab(financial_support_income_data['Parental financial-support profile'],
    financial_support_income_data['Household income band'], margins=True, dropna=False)
financial_support_income_row_percentages = (pd.crosstab(financial_support_income_data['Parental financial-support profile'],
    financial_support_income_data['Household income band'], normalize='index', dropna=False) * 100).round(2)
financial_support_he_data = parental_financial_support_overlap_data[['Parental financial-support profile',
    'Parental HE expectation']].dropna()
financial_support_he_counts = pd.crosstab(financial_support_he_data['Parental financial-support profile'],
    financial_support_he_data['Parental HE expectation'], margins=True, dropna=False)
financial_support_he_row_percentages = (pd.crosstab(financial_support_he_data['Parental financial-support profile'],
    financial_support_he_data['Parental HE expectation'], normalize='index', dropna=False) * 100).round(2)
financial_support_route_data = parental_financial_support_overlap_data[['Parental financial-support profile',
    'Expected post-16 route']].dropna()
financial_support_route_counts = pd.crosstab(financial_support_route_data['Parental financial-support profile'],
    financial_support_route_data['Expected post-16 route'], margins=True, dropna=False)
financial_support_route_row_percentages = (pd.crosstab(financial_support_route_data['Parental financial-support profile'],
    financial_support_route_data['Expected post-16 route'], normalize='index', dropna=False) * 100).round(2)
print('Comparison-measure coverage:')
display_limited(parental_financial_support_comparison_summary)
print('Associations with related retained measures:')
display_limited(parental_financial_support_association_summary)
print('Financial-support profile by household income band:')
display_limited(financial_support_income_counts)
print('Row percentages:')
display_limited(financial_support_income_row_percentages)
print('Financial-support profile by parental HE expectation:')
display_limited(financial_support_he_counts)
print('Row percentages:')
display_limited(financial_support_he_row_percentages)
print('Financial-support profile by expected post-16 route:')
display_limited(financial_support_route_counts)
print('Row percentages:')
display_limited(financial_support_route_row_percentages)
print('Files written in this cell: 0')

Comparison-measure coverage:


,Measure,Non-missing,Missing,Missing percentage,Observed categories
0,Parental financial-support profile,9516,251,2.57,5
1,Household income band,8771,996,10.20,9
2,Family NS-SEC,9040,727,7.44,8
3,Highest parental qualification,9510,257,2.63,7
4,Housing tenure,9521,246,2.52,8


Associations with related retained measures:


,Comparison measure,Complete comparisons,Financial-support categories,Comparison categories,Cramer's V
0,Expected post-16 route,9498,5,7,0.172
1,Household income band,8771,5,9,0.164
2,Highest parental qualification,9510,5,7,0.152
3,Family NS-SEC,9040,5,8,0.146
4,Housing tenure,9513,5,8,0.135


Financial-support profile by household income band:


...

Financial-support profile by expected post-16 route:


Expected post-16 route,Full-time employment,Further-education college,Other or unspecified full-time education,"Other, mixed or uncertain route",School sixth form,Sixth-form college,Work-based training or employment-training,All
Parental financial-support profile,,,,,,,,
0,118,253,58,64,229,212,197,1131
1,15,22,3,4,25,27,18,114
2,53,448,90,26,836,411,88,1952
3,47,852,158,39,1346,601,139,3182
4,24,689,125,25,1607,587,62,3119


Row percentages:


Expected post-16 route,Full-time employment,Further-education college,Other or unspecified full-time education,"Other, mixed or uncertain route",School sixth form,Sixth-form college,Work-based training or employment-training
Parental financial-support profile,,,,,,,
0,10.43,22.37,5.13,5.66,20.25,18.74,17.42
1,13.16,19.30,2.63,3.51,21.93,23.68,15.79
2,2.72,22.95,4.61,1.33,42.83,21.06,4.51
3,1.48,26.78,4.97,1.23,42.30,18.89,4.37
4,0.77,22.09,4.01,0.80,51.52,18.82,1.99


Files written in this cell: 0


In [339]:
# 20: Parental financial-support profile category review

import pandas as pd

def collapse_parental_financial_support_profile(profile_values):
    """Collapse the five-category profile to four categories."""
    collapsed_profile = profile_values.map({0: 0, 1: 1, 2: 1, 3: 2, 4: 3}).astype('Int64')
    return collapsed_profile
parental_financial_support_profile_four_category_by_wave = pd.DataFrame({wave: collapse_parental_financial_support_profile(parental_financial_support_profile_wave_data[wave]) for wave in ['Wave 1',
    'Wave 2', 'Wave 3']}, index=route_index)
latest_parental_financial_support_profile_four_category_candidate = collapse_parental_financial_support_profile(latest_parental_financial_support_profile_candidate).rename('parental_educational_financial_support_profile_pretransition')
assert latest_parental_financial_support_profile_four_category_candidate.notna().equals(latest_parental_financial_support_profile_candidate.notna())
assert set(latest_parental_financial_support_profile_four_category_candidate.dropna().unique()).issubset({0, 1, 2, 3})
profile_category_comparison_rows = []
for comparison, first_wave, second_wave in parental_contribution_wave_pairs:
    five_category_data = parental_financial_support_profile_wave_data[[first_wave, second_wave]].dropna()
    four_category_data = parental_financial_support_profile_four_category_by_wave[[first_wave, second_wave]].dropna()
    assert len(five_category_data) == len(four_category_data)
    profile_category_comparison_rows.append({'Comparison': comparison, 'Complete comparisons': len(four_category_data),
        'Five-category exact agreement percentage': round(five_category_data[first_wave].eq(five_category_data[second_wave]).mean() * 100,
        2), 'Four-category exact agreement percentage': round(four_category_data[first_wave].eq(four_category_data[second_wave]).mean() * 100,
        2), "Five-category Cramer's V": round(categorical_cramers_v(five_category_data[first_wave],
        five_category_data[second_wave]), 3), "Four-category Cramer's V": round(categorical_cramers_v(four_category_data[first_wave],
        four_category_data[second_wave]), 3)})
profile_category_agreement_comparison = pd.DataFrame(profile_category_comparison_rows)
four_category_labels = {0: 'No parental contribution expected',
    1: 'Contribution expected; zero or one substantive action', 2: 'Contribution expected; two substantive actions', 3: 'Contribution expected; three or more substantive actions'}
four_category_distribution_rows = []
four_category_observed_count = int(latest_parental_financial_support_profile_four_category_candidate.notna().sum())
for code, category in four_category_labels.items():
    participants = int(latest_parental_financial_support_profile_four_category_candidate.eq(code).sum())
    four_category_distribution_rows.append({'Code': code, 'Category': category, 'Participants': participants,
        'Percentage of full sample': round(participants / len(route_index) * 100,
        2), 'Percentage among observed': round(participants / four_category_observed_count * 100, 2)})
four_category_distribution_rows.append({'Code': pd.NA, 'Category': 'Unavailable',
    'Participants': int(latest_parental_financial_support_profile_four_category_candidate.isna().sum()), 'Percentage of full sample': round(latest_parental_financial_support_profile_four_category_candidate.isna().mean() * 100,
    2), 'Percentage among observed': pd.NA})
latest_four_category_profile_distribution = pd.DataFrame(four_category_distribution_rows)
four_category_overlap_data = parental_financial_support_overlap_data.copy()
four_category_overlap_data['Four-category parental financial-support profile'] = latest_parental_financial_support_profile_four_category_candidate
four_category_association_rows = []
for comparison_measure in ['Household income band', 'Family NS-SEC', 'Highest parental qualification',
    'Housing tenure', 'Parental HE expectation', 'Parental educational aspiration', 'Young-person HE application likelihood', 'Expected post-16 route']:
    five_category_pair = four_category_overlap_data[['Parental financial-support profile',
        comparison_measure]].dropna()
    four_category_pair = four_category_overlap_data[['Four-category parental financial-support profile',
        comparison_measure]].dropna()
    assert len(five_category_pair) == len(four_category_pair)
    four_category_association_rows.append({'Comparison measure': comparison_measure,
        'Complete comparisons': len(four_category_pair), "Five-category Cramer's V": round(categorical_cramers_v(five_category_pair['Parental financial-support profile'],
        five_category_pair[comparison_measure]), 3), "Four-category Cramer's V": round(categorical_cramers_v(four_category_pair['Four-category parental financial-support profile'],
        four_category_pair[comparison_measure]), 3)})
four_category_association_comparison = pd.DataFrame(four_category_association_rows).sort_values("Four-category Cramer's V",
    ascending=False).reset_index(drop=True)
print('Five- and four-category cross-wave agreement:')
display_limited(profile_category_agreement_comparison)
print('Latest four-category profile distribution:')
with pd.option_context('display.max_colwidth', None):
    display_limited(latest_four_category_profile_distribution)
print('Associations before and after category consolidation:')
display_limited(four_category_association_comparison)
print('Files written in this cell: 0')

Five- and four-category cross-wave agreement:


,Comparison,Complete comparisons,Five-category exact agreement percentage,Four-category exact agreement percentage,Five-category Cramer's V,Four-category Cramer's V
0,Wave 1 and Wave 2,9354,40.12,41.47,0.221,0.248
1,Wave 2 and eligible Wave 3,9349,46.18,47.14,0.273,0.310
2,Wave 1 and eligible Wave 3,9321,37.84,38.85,0.189,0.213


Latest four-category profile distribution:


,Code,Category,Participants,Percentage of full sample,Percentage among observed
0,0,No parental contribution expected,1133,11.60,11.91
1,1,Contribution expected; zero or one substantive action,2074,21.23,21.79
2,2,Contribution expected; two substantive actions,3188,32.64,33.5
3,3,Contribution expected; three or more substantive actions,3121,31.95,32.8
4,<NA>,Unavailable,251,2.57,<NA>


Associations before and after category consolidation:


,Comparison measure,Complete comparisons,Five-category Cramer's V,Four-category Cramer's V
0,Expected post-16 route,9498,0.172,0.191
1,Household income band,8771,0.164,0.185
2,Highest parental qualification,9510,0.152,0.171
3,Family NS-SEC,9040,0.146,0.165
4,Housing tenure,9513,0.135,0.154


Files written in this cell: 0


In [340]:
# 21: Parental educational financial-support decisions

import pandas as pd
parental_educational_financial_support_profile_pretransition = latest_parental_financial_support_profile_four_category_candidate.copy().astype('Int64').rename('parental_educational_financial_support_profile_pretransition')
parental_educational_financial_support_profile_labels = {0: 'No parental contribution expected',
    1: 'Parental contribution expected; zero or one substantive support action', 2: 'Parental contribution expected; two substantive support actions', 3: 'Parental contribution expected; three or more substantive support actions'}
assert parental_educational_financial_support_profile_pretransition.index.equals(route_index)
assert set(parental_educational_financial_support_profile_pretransition.dropna().unique()).issubset({0, 1, 2, 3})
parental_educational_financial_support_predictor = pd.DataFrame({'NSID': route_index,
    'parental_educational_financial_support_profile_pretransition': parental_educational_financial_support_profile_pretransition.to_numpy()})
assert len(parental_educational_financial_support_predictor) == 9767
assert parental_educational_financial_support_predictor['NSID'].is_unique
parental_contribution_input_mask = planned_financial_support_inventory['Battery construct'].eq('Expected funding sources if the young person remained in education') & planned_financial_support_inventory['Item suffix'].eq('0c')
substantive_action_input_mask = planned_financial_support_inventory['Battery construct'].eq('Actions the parent might take to help keep the young person in education') & planned_financial_support_inventory['Item suffix'].isin(['0a',
    '0b', '0c', '0d', '0e', '0f', '0g', '0h'])
financial_support_construction_input_mask = parental_contribution_input_mask | substantive_action_input_mask
parental_financial_support_variable_decisions = planned_financial_support_inventory.copy()
parental_financial_support_variable_decisions['Financial-support decision'] = 'Review support'
parental_financial_support_variable_decisions['Financial-support role'] = 'Questionnaire structure and routing review'
parental_financial_support_variable_decisions['Financial-support reason'] = 'The item was used to interpret the funding and support-action batteries but was not required in the retained predictor'
parental_financial_support_variable_decisions.loc[parental_contribution_input_mask,
    'Financial-support decision'] = 'Construction input'
parental_financial_support_variable_decisions.loc[parental_contribution_input_mask,
    'Financial-support role'] = 'Expected parental financial contribution'
parental_financial_support_variable_decisions.loc[parental_contribution_input_mask,
    'Financial-support reason'] = 'The item distinguishes parents who expected to contribute financially from those who did not and determines routing into the planned support-action battery'
parental_financial_support_variable_decisions.loc[substantive_action_input_mask,
    'Financial-support decision'] = 'Construction input'
parental_financial_support_variable_decisions.loc[substantive_action_input_mask,
    'Financial-support role'] = 'Substantive planned parental support action'
parental_financial_support_variable_decisions.loc[substantive_action_input_mask,
    'Financial-support reason'] = 'The item contributes to the count of substantive actions the parent reported being prepared to take to support continued education'
assert len(parental_financial_support_variable_decisions) == 54
assert int(financial_support_construction_input_mask.sum()) == 27
assert parental_financial_support_variable_decisions['Financial-support decision'].eq('Construction input').sum() == 27
assert parental_financial_support_variable_decisions['Financial-support decision'].eq('Review support').sum() == 27
parental_financial_support_representation_decisions = pd.DataFrame([{'Representation': 'Expected parental contribution indicator',
    'Decision': 'Do not retain separately', 'Reason': 'The binary indicator identifies expected parental contribution but does not capture the breadth of planned support actions'}, {'Representation': 'Substantive support-action count',
    'Decision': 'Do not retain separately', 'Reason': 'The count is observed only when parental contribution was expected and cannot be interpreted independently of that routing'}, {'Representation': 'Separate funding-source and action indicators',
    'Decision': 'Do not retain', 'Reason': 'Separate indicators would add numerous overlapping and sparse predictors from two multi-response batteries'}, {'Representation': 'Five-category combined profile',
    'Decision': 'Do not retain', 'Reason': 'The category combining expected contribution with no substantive action contained only 114 participants in the latest representation'}, {'Representation': 'Four-category combined profile using the latest permitted wave',
    'Decision': 'Retain', 'Reason': 'The representation preserves the routing distinction, captures variation in planned support breadth, avoids a very small category and remains distinct from retained socioeconomic and aspiration measures'}, {'Representation': 'Cross-wave mean or cumulative profile',
    'Decision': 'Do not retain', 'Reason': 'Financial-support plans may legitimately change as the post-16 transition approaches; the latest permitted response is therefore more appropriate than averaging across waves'}])
parental_financial_support_predictor_summary = pd.DataFrame([{'Predictor': 'parental_educational_financial_support_profile_pretransition',
    'Representation': 'Latest permitted Wave 3, Wave 2 or Wave 1 four-category profile', 'Non-missing': int(parental_educational_financial_support_profile_pretransition.notna().sum()), 'Missing': int(parental_educational_financial_support_profile_pretransition.isna().sum()), 'Missing percentage': round(parental_educational_financial_support_profile_pretransition.isna().mean() * 100,
    2), 'Categories': int(parental_educational_financial_support_profile_pretransition.nunique()), "Highest overlap Cramer's V": round(four_category_association_comparison["Four-category Cramer's V"].max(),
    3)}])
parental_financial_support_retained_distribution_rows = []
observed_financial_support_count = int(parental_educational_financial_support_profile_pretransition.notna().sum())
for code, category in parental_educational_financial_support_profile_labels.items():
    participants = int(parental_educational_financial_support_profile_pretransition.eq(code).sum())
    parental_financial_support_retained_distribution_rows.append({'Code': code, 'Category': category,
        'Participants': participants, 'Percentage among observed': round(participants / observed_financial_support_count * 100,
        2)})
parental_financial_support_retained_distribution = pd.DataFrame(parental_financial_support_retained_distribution_rows)
parental_financial_support_decision_display = parental_financial_support_variable_decisions.sort_values(['Wave',
    'Variable position'])[['Wave', 'Source type', 'Source file', 'Variable', 'Variable label', 'Timing status',
    'Battery construct', 'Item suffix', 'Financial-support decision', 'Financial-support role', 'Financial-support reason']].reset_index(drop=True)
print('Parental educational financial-support predictors retained: 1')
print('\nRetained predictor summary:')
display_limited(parental_financial_support_predictor_summary)
print('\nRetained category distribution:')
display_limited(parental_financial_support_retained_distribution)
print('\nRepresentation decisions:')
display_limited(parental_financial_support_representation_decisions)
parental_financial_support_decision_counts = parental_financial_support_variable_decisions['Financial-support decision'].value_counts().rename('Variables').rename_axis('Decision').reset_index()
print('\nVariable-decision counts:')
display_limited(parental_financial_support_decision_counts)
print('\nFirst 10 variable decisions:')
display_limited(parental_financial_support_decision_display.head(10))
print(f'Additional variable decisions not displayed: {max(len(parental_financial_support_decision_display) - 10, 0):,}')
print('\nFiles written in this cell: 0')

Parental educational financial-support predictors retained: 1

Retained predictor summary:


,Predictor,Representation,Non-missing,Missing,Missing percentage,Categories,Highest overlap Cramer's V
0,parental_educational_financial_support_profile...,"Latest permitted Wave 3, Wave 2 or Wave 1 four...",9516,251,2.57,4,0.191



Retained category distribution:


,Code,Category,Participants,Percentage among observed
0,0,No parental contribution expected,1133,11.91
1,1,Parental contribution expected; zero or one su...,2074,21.79
2,2,Parental contribution expected; two substantiv...,3188,33.50
3,3,Parental contribution expected; three or more ...,3121,32.80



Representation decisions:


...


Variable-decision counts:


,Decision,Variables
0,Review support,27
1,Construction input,27



First 10 variable decisions:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Battery construct,Item suffix,Financial-support decision,Financial-support role,Financial-support reason
0,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1fefinMP0a,MP: How YP's expenses would be paid if stayed ...,Pre-transition source,Expected funding sources if the young person r...,0a,Review support,Questionnaire structure and routing review,The item was used to interpret the funding and...
1,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1fefinMP0b,MP: How YP's expenses would be paid if stayed ...,Pre-transition source,Expected funding sources if the young person r...,0b,Review support,Questionnaire structure and routing review,The item was used to interpret the funding and...
2,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1fefinMP0c,MP: How YP's expenses would be paid if stayed ...,Pre-transition source,Expected funding sources if the young person r...,0c,Construction input,Expected parental financial contribution,The item distinguishes parents who expected to...
3,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1fefinMP0d,MP: How YP's expenses would be paid if stayed ...,Pre-transition source,Expected funding sources if the young person r...,0d,Review support,Questionnaire structure and routing review,The item was used to interpret the funding and...
4,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1fefinMP0e,MP: How YP's expenses would be paid if stayed ...,Pre-transition source,Expected funding sources if the young person r...,0e,Review support,Questionnaire structure and routing review,The item was used to interpret the funding and...


Additional variable decisions not displayed: 44

Files written in this cell: 0


In [341]:
# 22: Parental education-finance expectation review

import pandas as pd
parental_education_finance_inventory = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 review track'].eq('Education finance expectation')].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(parental_education_finance_inventory) == 1
assert parental_education_finance_inventory['Variable'].iloc[0] == 'W2EmaGetMP'
parental_ema_variable = parental_education_finance_inventory['Variable'].iloc[0]
parental_ema_source_file = parental_education_finance_inventory['Source file'].iloc[0]
parental_ema_source_path = source_file_lookup[parental_ema_source_file]
parental_ema_raw = pd.read_stata(parental_ema_source_path, columns=['NSID', parental_ema_variable],
    convert_categoricals=False)
parental_ema_labelled = pd.read_stata(parental_ema_source_path, columns=['NSID', parental_ema_variable],
    convert_categoricals=True)
for data in [parental_ema_raw, parental_ema_labelled]:
    data['NSID'] = standardise_nsid(data['NSID'])
    assert data['NSID'].is_unique
parental_ema_raw = parental_ema_raw.set_index('NSID').reindex(route_index)
parental_ema_labelled = parental_ema_labelled.set_index('NSID').reindex(route_index)
parental_ema_raw[parental_ema_variable] = pd.to_numeric(parental_ema_raw[parental_ema_variable], errors='coerce')
young_person_ema_inventory = domain_9_screened_candidates.loc[domain_9_screened_candidates['Variable'].eq('W2getemaYP')].copy().reset_index(drop=True)
assert len(young_person_ema_inventory) == 1
young_person_ema_variable = young_person_ema_inventory['Variable'].iloc[0]
young_person_ema_source_file = young_person_ema_inventory['Source file'].iloc[0]
young_person_ema_source_path = source_file_lookup[young_person_ema_source_file]
young_person_ema_raw = pd.read_stata(young_person_ema_source_path, columns=['NSID', young_person_ema_variable],
    convert_categoricals=False)
young_person_ema_labelled = pd.read_stata(young_person_ema_source_path, columns=['NSID', young_person_ema_variable],
    convert_categoricals=True)
for data in [young_person_ema_raw, young_person_ema_labelled]:
    data['NSID'] = standardise_nsid(data['NSID'])
    assert data['NSID'].is_unique
young_person_ema_raw = young_person_ema_raw.set_index('NSID').reindex(route_index)
young_person_ema_labelled = young_person_ema_labelled.set_index('NSID').reindex(route_index)
young_person_ema_raw[young_person_ema_variable] = pd.to_numeric(young_person_ema_raw[young_person_ema_variable],
    errors='coerce')
education_finance_code_rows = []
for respondent, variable, raw_data, labelled_data in [('Main parent', parental_ema_variable, parental_ema_raw,
    parental_ema_labelled), ('Young person', young_person_ema_variable, young_person_ema_raw,
    young_person_ema_labelled)]:
    raw_values = raw_data[variable]
    labelled_values = labelled_data[variable].astype('string')
    for raw_code, participants in raw_values.value_counts(dropna=False).items():
        if pd.isna(raw_code):
            value_label = 'No source record'
            response_type = 'Unavailable'
            sort_value = 999999
        else:
            matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            response_type = 'Observed response' if raw_code >= 0 else 'Special code'
            sort_value = float(raw_code)
        education_finance_code_rows.append({'Respondent': respondent, 'Variable': variable, 'Raw code': raw_code,
            'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
education_finance_code_distribution = pd.DataFrame(education_finance_code_rows).sort_values(['Respondent',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
parental_ema_observed = parental_ema_raw[parental_ema_variable].where(parental_ema_raw[parental_ema_variable].ge(0)).astype('Float64').rename('parental_ema_eligibility_expectation')
young_person_ema_observed = young_person_ema_raw[young_person_ema_variable].where(young_person_ema_raw[young_person_ema_variable].ge(0)).astype('Float64').rename('young_person_ema_eligibility_expectation')
parental_education_finance_overlap_data = pd.DataFrame({'Parental EMA expectation': parental_ema_observed,
    'Young-person EMA expectation': young_person_ema_observed, 'Household income band': family_socioeconomic_overlap_data['household_income_band'], 'Family NS-SEC': family_socioeconomic_overlap_data['family_nssec_code'], 'Highest parental qualification': family_socioeconomic_overlap_data['highest_parental_qualification_code'], 'Parental financial-support profile': parental_educational_financial_support_profile_pretransition, 'Parental HE expectation': parental_he_expectation_candidate, 'Expected post-16 route': saved_educational_aspiration_predictors['expected_post16_route']}, index=route_index)
assert len(parental_education_finance_overlap_data) == 9767
parental_education_finance_coverage_rows = []
for measure in parental_education_finance_overlap_data.columns:
    measure_values = parental_education_finance_overlap_data[measure]
    parental_education_finance_coverage_rows.append({'Measure': measure,
        'Non-missing': int(measure_values.notna().sum()), 'Missing': int(measure_values.isna().sum()), 'Missing percentage': round(measure_values.isna().mean() * 100,
        2), 'Observed categories': int(measure_values.nunique())})
parental_education_finance_coverage_summary = pd.DataFrame(parental_education_finance_coverage_rows)
parental_education_finance_association_rows = []
for comparison_measure in ['Young-person EMA expectation', 'Household income band', 'Family NS-SEC',
    'Highest parental qualification', 'Parental financial-support profile', 'Parental HE expectation', 'Expected post-16 route']:
    pair_data = parental_education_finance_overlap_data[['Parental EMA expectation', comparison_measure]].dropna()
    assert len(pair_data) > 0
    parental_education_finance_association_rows.append({'Comparison measure': comparison_measure,
        'Complete comparisons': len(pair_data), 'Parental EMA categories': int(pair_data['Parental EMA expectation'].nunique()), 'Comparison categories': int(pair_data[comparison_measure].nunique()), "Cramer's V": round(categorical_cramers_v(pair_data['Parental EMA expectation'],
        pair_data[comparison_measure]), 3)})
parental_education_finance_association_summary = pd.DataFrame(parental_education_finance_association_rows).sort_values("Cramer's V",
    ascending=False).reset_index(drop=True)
parent_young_person_ema_data = parental_education_finance_overlap_data[['Parental EMA expectation',
    'Young-person EMA expectation']].dropna()
parent_young_person_ema_counts = pd.crosstab(parent_young_person_ema_data['Parental EMA expectation'],
    parent_young_person_ema_data['Young-person EMA expectation'], margins=True, dropna=False)
parent_young_person_ema_row_percentages = (pd.crosstab(parent_young_person_ema_data['Parental EMA expectation'],
    parent_young_person_ema_data['Young-person EMA expectation'], normalize='index', dropna=False) * 100).round(2)
parental_ema_income_data = parental_education_finance_overlap_data[['Parental EMA expectation',
    'Household income band']].dropna()
parental_ema_income_counts = pd.crosstab(parental_ema_income_data['Parental EMA expectation'],
    parental_ema_income_data['Household income band'], margins=True, dropna=False)
parental_ema_income_row_percentages = (pd.crosstab(parental_ema_income_data['Parental EMA expectation'],
    parental_ema_income_data['Household income band'], normalize='index', dropna=False) * 100).round(2)
print('Education-finance variables:')
with pd.option_context('display.max_colwidth', None):
    display_limited(pd.concat([parental_education_finance_inventory[['Wave', 'Source type', 'Source file', 'Variable',
        'Variable label', 'Timing status', 'Domain 9 screening status', 'Domain 9 review track']], young_person_ema_inventory[['Wave',
        'Source type', 'Source file', 'Variable', 'Variable label', 'Timing status', 'Domain 9 screening status', 'Domain 9 review track']]], ignore_index=True))
print('Response-code distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(education_finance_code_distribution)
print('Comparison-measure coverage:')
display_limited(parental_education_finance_coverage_summary)
print('Associations with related measures:')
display_limited(parental_education_finance_association_summary)
print('Parental and young-person EMA expectations:')
display_limited(parent_young_person_ema_counts)
print('Row percentages:')
display_limited(parent_young_person_ema_row_percentages)
print('Parental EMA expectation by household income band:')
display_limited(parental_ema_income_counts)
print('Row percentages:')
display_limited(parental_ema_income_row_percentages)
print('Files written in this cell: 0')

Education-finance variables:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Domain 9 screening status,Domain 9 review track
0,Wave 2,Parental attitudes,wave_two_lsype_parental_attitudes_file_16_06_08,W2EmaGetMP,MP: Whether think YP would be eligible for EMA if stayed on in education,Pre-transition source,Core review,Education finance expectation
1,Wave 2,Young person,wave_two_lsype_young_person_2020,W2getemaYP,YP: Whether YP thinks would be eligible for EMA if stay on in education after 16,Pre-transition source,Out of domain,Young-person educational plans


Response-code distributions:


,Respondent,Variable,Raw code,Value label,Response type,Participants
0,Main parent,W2EmaGetMP,-99.0,MP not interviewed,Special code,79
1,Main parent,W2EmaGetMP,-91.0,Not applicable,Special code,2896
2,Main parent,W2EmaGetMP,-1.0,Don't Know,Special code,1402
3,Main parent,W2EmaGetMP,1.0,Yes,Observed response,2873
4,Main parent,W2EmaGetMP,2.0,No,Observed response,2271


Comparison-measure coverage:


...

Parental EMA expectation by household income band:


Household income band,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,All
Parental EMA expectation,,,,,,,,,,
1.0,26,109,475,623,497,350,253,122,156,2611
2.0,3,4,17,37,62,92,177,261,1521,2174
All,29,113,492,660,559,442,430,383,1677,4785


Row percentages:


Household income band,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0
Parental EMA expectation,,,,,,,,,
1.0,1.00,4.17,18.19,23.86,19.03,13.40,9.69,4.67,5.97
2.0,0.14,0.18,0.78,1.70,2.85,4.23,8.14,12.01,69.96


Files written in this cell: 0


In [342]:
# 23: Parental EMA-expectation routing review

import pandas as pd
parental_ema_reader = pd.read_stata(parental_ema_source_path, iterator=True)
parental_ema_source_labels = parental_ema_reader.variable_labels()
parental_ema_source_variables = list(parental_ema_source_labels.keys())
parental_ema_position = parental_ema_source_variables.index(parental_ema_variable)
neighbourhood_start = max(0, parental_ema_position - 8)
neighbourhood_end = min(len(parental_ema_source_variables), parental_ema_position + 9)
parental_ema_neighbourhood_variables = parental_ema_source_variables[neighbourhood_start:neighbourhood_end]
parental_ema_related_variables = [variable for variable in parental_ema_source_variables if 'ema' in variable.lower() or 'ema' in str(parental_ema_source_labels.get(variable,
    '')).lower() or (variable in parental_ema_neighbourhood_variables and any((keyword in str(parental_ema_source_labels.get(variable,
    '')).lower() for keyword in ['education maintenance', 'eligible', 'allowance', 'heard', 'aware', 'know'])))]
parental_ema_related_variables = list(dict.fromkeys(parental_ema_related_variables))
assert parental_ema_variable in parental_ema_related_variables
parental_ema_neighbourhood_rows = []
for position in range(neighbourhood_start, neighbourhood_end):
    variable = parental_ema_source_variables[position]
    parental_ema_neighbourhood_rows.append({'Variable position': position, 'Variable': variable,
        'Variable label': parental_ema_source_labels.get(variable,
        ''), 'EMA-related review variable': variable in parental_ema_related_variables, 'Target variable': variable == parental_ema_variable})
parental_ema_neighbourhood = pd.DataFrame(parental_ema_neighbourhood_rows)
parental_ema_routing_raw = pd.read_stata(parental_ema_source_path, columns=['NSID', *parental_ema_related_variables],
    convert_categoricals=False)
parental_ema_routing_labelled = pd.read_stata(parental_ema_source_path, columns=['NSID',
    *parental_ema_related_variables], convert_categoricals=True)
for data in [parental_ema_routing_raw, parental_ema_routing_labelled]:
    data['NSID'] = standardise_nsid(data['NSID'])
    assert data['NSID'].is_unique
parental_ema_routing_raw = parental_ema_routing_raw.set_index('NSID').reindex(route_index)
parental_ema_routing_labelled = parental_ema_routing_labelled.set_index('NSID').reindex(route_index)
for variable in parental_ema_related_variables:
    parental_ema_routing_raw[variable] = pd.to_numeric(parental_ema_routing_raw[variable], errors='coerce')
parental_ema_target_values = parental_ema_routing_raw[parental_ema_variable]
parental_ema_target_status = pd.Series('Other unavailable', index=route_index, dtype='string')
parental_ema_target_status.loc[parental_ema_target_values.isin([1.0, 2.0])] = 'Observed response'
parental_ema_target_status.loc[parental_ema_target_values.eq(-91.0)] = 'Structurally not applicable'
parental_ema_target_status.loc[parental_ema_target_values.eq(-99.0)] = 'Main parent not interviewed'
parental_ema_target_status.loc[parental_ema_target_values.isna()] = 'No source record'
parental_ema_routing_code_rows = []
parental_ema_routing_cross_rows = []
for routing_variable in parental_ema_related_variables:
    if routing_variable == parental_ema_variable:
        continue
    routing_values = parental_ema_routing_raw[routing_variable]
    routing_labels = parental_ema_routing_labelled[routing_variable].astype('string')
    for raw_code, participants in routing_values.value_counts(dropna=False).items():
        if pd.isna(raw_code):
            routing_code_mask = routing_values.isna()
            value_label = 'No source record'
            response_type = 'Unavailable'
            sort_value = 999999
        else:
            routing_code_mask = routing_values.eq(raw_code)
            matching_labels = routing_labels.loc[routing_code_mask].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            response_type = 'Observed response' if raw_code >= 0 else 'Special code'
            sort_value = float(raw_code)
        parental_ema_routing_code_rows.append({'Routing variable': routing_variable,
            'Variable label': parental_ema_source_labels.get(routing_variable,
            ''), 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
        status_counts = parental_ema_target_status.loc[routing_code_mask].value_counts()
        observed_count = int(status_counts.get('Observed response', 0))
        structural_count = int(status_counts.get('Structurally not applicable', 0))
        participant_count = int(participants)
        parental_ema_routing_cross_rows.append({'Routing variable': routing_variable, 'Routing code': raw_code,
            'Routing response': value_label, 'Participants': participant_count, 'EMA expectation observed': observed_count, 'Observed percentage': round(observed_count / participant_count * 100,
            2) if participant_count > 0 else pd.NA, 'EMA expectation structurally not applicable': structural_count, 'Structural percentage': round(structural_count / participant_count * 100,
            2) if participant_count > 0 else pd.NA})
parental_ema_routing_code_distribution = pd.DataFrame(parental_ema_routing_code_rows).sort_values(['Routing variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
parental_ema_routing_cross_summary = pd.DataFrame(parental_ema_routing_cross_rows).sort_values(['Routing variable',
    'Routing code'], na_position='last').reset_index(drop=True)
parental_ema_target_status_summary = parental_ema_target_status.value_counts().rename('Participants').rename_axis('Target-item status').reset_index()
parental_ema_target_status_summary['Percentage'] = (parental_ema_target_status_summary['Participants'] / len(route_index) * 100).round(2)
print('Parental EMA target-item status:')
display_limited(parental_ema_target_status_summary)
print('Source-file neighbourhood:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_ema_neighbourhood)
print('Potential routing-variable code distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_ema_routing_code_distribution)
print('Target observability by potential routing code:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_ema_routing_cross_summary)
print('Files written in this cell: 0')

Parental EMA target-item status:


,Target-item status,Participants,Percentage
0,Observed response,5144,52.67
1,Structurally not applicable,2896,29.65
2,Other unavailable,1402,14.35
3,No source record,246,2.52
4,Main parent not interviewed,79,0.81


Source-file neighbourhood:


,Variable position,Variable,Variable label,EMA-related review variable,Target variable
0,36,W2extrtu5MP0i,MP: Which private lessons/ classes paid for (NOT covered at school) - Science,False,False
1,37,W2extrtu5MP0j,MP: Which private lessons/ classes paid for (NOT covered at school) - Horse ridi,False,False
2,38,W2extrtu5MP0k,MP: Which private lessons/ classes paid for (NOT covered at school) - Other,False,False
3,39,W2extrtu6MP,MP: How often does YP go to supplementary lessons or classes over the last 12 mo,False,False
4,40,W2parasp2MP,MP: What would like YP to do when reach school leaving age,False,False


Potential routing-variable code distributions:


,Routing variable,Variable label,Raw code,Value label,Response type,Participants
0,W2AwarEmaMP,MP: Whether MP aware of EMA before interview,-99.0,MP not interviewed,Special code,79
1,W2AwarEmaMP,MP: Whether MP aware of EMA before interview,-1.0,Don't Know,Special code,36
2,W2AwarEmaMP,MP: Whether MP aware of EMA before interview,1.0,Yes,Observed response,6546
3,W2AwarEmaMP,MP: Whether MP aware of EMA before interview,2.0,No,Observed response,2860
4,W2AwarEmaMP,MP: Whether MP aware of EMA before interview,NaN,No source record,Unavailable,246


Target observability by potential routing code:


,Routing variable,Routing code,Routing response,Participants,EMA expectation observed,Observed percentage,EMA expectation structurally not applicable,Structural percentage
0,W2AwarEmaMP,-99.0,MP not interviewed,79,0,0.00,0,0.0
1,W2AwarEmaMP,-1.0,Don't Know,36,0,0.00,36,100.0
2,W2AwarEmaMP,1.0,Yes,6546,5144,78.58,0,0.0
3,W2AwarEmaMP,2.0,No,2860,0,0.00,2860,100.0
4,W2AwarEmaMP,NaN,No source record,246,0,0.00,0,0.0


Files written in this cell: 0


In [343]:
# 24: Parental education-finance expectation decision

import pandas as pd
parental_ema_awareness_values = parental_ema_routing_raw['W2AwarEmaMP']
parental_ema_expectation_values = parental_ema_routing_raw[parental_ema_variable]
parent_aware_of_ema = parental_ema_awareness_values.eq(1.0)
parent_not_aware_or_uncertain = parental_ema_awareness_values.isin([-1.0, 2.0])
ema_expectation_question_answered = parental_ema_expectation_values.isin([-1.0, 1.0, 2.0])
ema_expectation_structural_na = parental_ema_expectation_values.eq(-91.0)
assert parent_aware_of_ema.equals(ema_expectation_question_answered)
assert parent_not_aware_or_uncertain.equals(ema_expectation_structural_na)
parental_ema_routing_decision_summary = pd.DataFrame([{'Measure': 'Parents aware of EMA',
    'Participants': int(parent_aware_of_ema.sum()), 'Percentage of full sample': round(parent_aware_of_ema.mean() * 100,
    2)}, {'Measure': 'Parents not aware of EMA or uncertain about awareness',
    'Participants': int(parent_not_aware_or_uncertain.sum()), 'Percentage of full sample': round(parent_not_aware_or_uncertain.mean() * 100,
    2)}, {'Measure': 'Observed yes or no eligibility expectation',
    'Participants': int(parental_ema_expectation_values.isin([1.0,
    2.0]).sum()), 'Percentage of full sample': round(parental_ema_expectation_values.isin([1.0, 2.0]).mean() * 100,
    2)}, {'Measure': "Eligibility expectation reported as don't know",
    'Participants': int(parental_ema_expectation_values.eq(-1.0).sum()), 'Percentage of full sample': round(parental_ema_expectation_values.eq(-1.0).mean() * 100,
    2)}, {'Measure': 'Eligibility expectation structurally not applicable',
    'Participants': int(ema_expectation_structural_na.sum()), 'Percentage of full sample': round(ema_expectation_structural_na.mean() * 100,
    2)}])
parental_ema_income_cramers_v = float(parental_education_finance_association_summary.loc[parental_education_finance_association_summary['Comparison measure'].eq('Household income band'),
    "Cramer's V"].iloc[0])
parent_young_person_ema_cramers_v = float(parental_education_finance_association_summary.loc[parental_education_finance_association_summary['Comparison measure'].eq('Young-person EMA expectation'),
    "Cramer's V"].iloc[0])
assert parental_ema_income_cramers_v == 0.752
assert parent_young_person_ema_cramers_v == 0.813
parental_education_finance_variable_decision = parental_education_finance_inventory.copy()
parental_education_finance_variable_decision['Education-finance decision'] = 'Exclude'
parental_education_finance_variable_decision['Education-finance role'] = 'Policy-specific expectation of means-tested financial-support eligibility'
parental_education_finance_variable_decision['Education-finance reason'] = 'The question was asked only when the main parent was aware of EMA. A binary eligibility response was available for 52.67% of the analysis sample and was strongly associated with household income. The item therefore mainly represents awareness and expected eligibility under a specific means-tested policy rather than a general parental educational attitude or support construct'
assert len(parental_education_finance_variable_decision) == 1
assert parental_education_finance_variable_decision['Education-finance decision'].eq('Exclude').all()
parental_education_finance_representation_decisions = pd.DataFrame([{'Representation': 'Parental EMA eligibility expectation',
    'Decision': 'Do not retain', 'Reason': 'The yes or no response was available only after awareness-based routing and was strongly associated with household income'}, {'Representation': 'EMA awareness and eligibility profile',
    'Decision': 'Do not construct', 'Reason': 'A combined profile would mix knowledge of a specific policy with anticipated eligibility under its means-tested rules'}, {'Representation': 'Reasons for expected EMA ineligibility',
    'Decision': 'Do not construct', 'Reason': 'The reason battery was conditionally routed after an expected ineligibility response and was dominated by household income being too high'}, {'Representation': 'Parent and young-person EMA expectations',
    'Decision': 'Do not retain separately', 'Reason': 'The two respondent reports were highly concordant and would add closely overlapping policy-specific measures'}])
parental_education_finance_decision_summary = pd.DataFrame([{'Variable': parental_ema_variable,
    'Predictors retained': 0, 'Yes or no responses': int(parental_ema_expectation_values.isin([1.0,
    2.0]).sum()), 'Yes or no response percentage': round(parental_ema_expectation_values.isin([1.0, 2.0]).mean() * 100,
    2), "Cramer's V with household income": parental_ema_income_cramers_v, "Cramer's V with young-person report": parent_young_person_ema_cramers_v, 'Decision': 'Exclude'}])
parental_education_finance_decision_display = parental_education_finance_variable_decision[['Wave', 'Source type',
    'Source file', 'Variable', 'Variable label', 'Timing status', 'Education-finance decision', 'Education-finance role', 'Education-finance reason']].reset_index(drop=True)
print('Parental education-finance expectation predictors retained: 0')
print('Routing and response structure:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_ema_routing_decision_summary)
print('Decision summary:')
display_limited(parental_education_finance_decision_summary)
print('Representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_education_finance_representation_decisions)
print('Variable decision:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_education_finance_decision_display)
print('Files written in this cell: 0')

Parental education-finance expectation predictors retained: 0
Routing and response structure:


,Measure,Participants,Percentage of full sample
0,Parents aware of EMA,6546,67.02
1,Parents not aware of EMA or uncertain about awareness,2896,29.65
2,Observed yes or no eligibility expectation,5144,52.67
3,Eligibility expectation reported as don't know,1402,14.35
4,Eligibility expectation structurally not applicable,2896,29.65


Decision summary:


,Variable,Predictors retained,Yes or no responses,Yes or no response percentage,Cramer's V with household income,Cramer's V with young-person report,Decision
0,W2EmaGetMP,0,5144,52.67,0.752,0.813,Exclude


Representation decisions:


,Representation,Decision,Reason
0,Parental EMA eligibility expectation,Do not retain,The yes or no response was available only after awareness-based routing and was strongly associated with household income
1,EMA awareness and eligibility profile,Do not construct,A combined profile would mix knowledge of a specific policy with anticipated eligibility under its means-tested rules
2,Reasons for expected EMA ineligibility,Do not construct,The reason battery was conditionally routed after an expected ineligibility response and was dominated by household income being too high
3,Parent and young-person EMA expectations,Do not retain separately,The two respondent reports were highly concordant and would add closely overlapping policy-specific measures


Variable decision:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Education-finance decision,Education-finance role,Education-finance reason
0,Wave 2,Parental attitudes,wave_two_lsype_parental_attitudes_file_16_06_08,W2EmaGetMP,MP: Whether think YP would be eligible for EMA if stayed on in education,Pre-transition source,Exclude,Policy-specific expectation of means-tested financial-support eligibility,The question was asked only when the main parent was aware of EMA. A binary eligibility response was available for 52.67% of the analysis sample and was strongly associated with household income. The item therefore mainly represents awareness and expected eligibility under a specific means-tested policy rather than a general parental educational attitude or support construct


Files written in this cell: 0


In [344]:
# 25: Parental homework-support variable review

import pandas as pd
parental_homework_support_inventory = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 review track'].eq('Homework support and monitoring')].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(parental_homework_support_inventory) == 4
parental_homework_support_raw_tables = {}
parental_homework_support_labelled_tables = {}
for source_file, source_inventory in parental_homework_support_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    for data in [raw_data, labelled_data]:
        data['NSID'] = standardise_nsid(data['NSID'])
        assert data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(route_index)
    labelled_data = labelled_data.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    parental_homework_support_raw_tables[source_file] = raw_data
    parental_homework_support_labelled_tables[source_file] = labelled_data
parental_homework_support_code_rows = []
parental_homework_support_coverage_rows = []
for _, inventory_row in parental_homework_support_inventory.iterrows():
    wave = inventory_row['Wave']
    source_file = inventory_row['Source file']
    variable = inventory_row['Variable']
    variable_label = inventory_row['Variable label']
    timing_mask = pd.Series(True, index=route_index, dtype=bool)
    if wave == 'Wave 3':
        timing_mask = wave_3_pretransition_interview_mask.reindex(route_index).fillna(False).astype(bool)
    raw_values = parental_homework_support_raw_tables[source_file][variable].where(timing_mask)
    labelled_values = parental_homework_support_labelled_tables[source_file][variable].astype('string')
    observed_mask = raw_values.ge(0).fillna(False).astype(bool)
    special_code_mask = raw_values.lt(0).fillna(False).astype(bool)
    unavailable_mask = raw_values.isna()
    parental_homework_support_coverage_rows.append({'Wave': wave, 'Variable': variable,
        'Variable label': variable_label, 'Participants within timing': int(timing_mask.sum()), 'Observed responses': int(observed_mask.sum()), 'Observed percentage': round(observed_mask.mean() * 100,
        2), 'Special-code responses': int(special_code_mask.sum()), 'Unavailable': int(unavailable_mask.sum()), 'Observed categories': int(raw_values.loc[observed_mask].nunique())})
    for raw_code, participants in raw_values.value_counts(dropna=False).items():
        if pd.isna(raw_code):
            value_label = 'No source record or outside timing'
            response_type = 'Unavailable'
            sort_value = 999999
        else:
            response_mask = raw_values.eq(raw_code)
            matching_labels = labelled_values.loc[response_mask].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            response_type = 'Observed response' if raw_code >= 0 else 'Special code'
            sort_value = float(raw_code)
        parental_homework_support_code_rows.append({'Wave': wave, 'Variable': variable,
            'Variable label': variable_label, 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
parental_homework_support_coverage_summary = pd.DataFrame(parental_homework_support_coverage_rows).sort_values(['Wave',
    'Variable']).reset_index(drop=True)
parental_homework_support_code_distribution = pd.DataFrame(parental_homework_support_code_rows).sort_values(['Wave',
    'Variable', 'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
parental_homework_support_label_comparison = parental_homework_support_inventory[['Wave', 'Source type', 'Source file',
    'Variable position', 'Variable', 'Variable label', 'Timing status', 'Deferred from earlier domain']].copy()
print(f'Parental homework-support variables reviewed: {len(parental_homework_support_inventory)}')
print('Variable inventory:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_homework_support_label_comparison)
print('Coverage summary:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_homework_support_coverage_summary)
print('Response-code distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_homework_support_code_distribution)
print('Files written in this cell: 0')

Parental homework-support variables reviewed: 4
Variable inventory:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Deferred from earlier domain
0,Wave 1,Young person,wave_one_lsype_young_person_2020,136,W1hwhelpYP,YP: Whether anyone at home helps them with homework,Pre-transition source,False
1,Wave 1,Young person,wave_one_lsype_young_person_2020,137,W1hwpchlYP,YP: Whether anyone at home makes sure that do homework,Pre-transition source,False
2,Wave 2,Young person,wave_two_lsype_young_person_2020,235,W2hwhelpYP,YP: Whether anyone at home helps them with homework,Pre-transition source,False
3,Wave 2,Young person,wave_two_lsype_young_person_2020,236,W2hwpchlYP,YP: Whether anyone at home makes sure that do homework,Pre-transition source,False


Coverage summary:


,Wave,Variable,Variable label,Participants within timing,Observed responses,Observed percentage,Special-code responses,Unavailable,Observed categories
0,Wave 1,W1hwhelpYP,YP: Whether anyone at home helps them with homework,9767,9335,95.58,189,243,3
1,Wave 1,W1hwpchlYP,YP: Whether anyone at home makes sure that do homework,9767,9330,95.53,194,243,5
2,Wave 2,W2hwhelpYP,YP: Whether anyone at home helps them with homework,9767,9233,94.53,288,246,3
3,Wave 2,W2hwpchlYP,YP: Whether anyone at home makes sure that do homework,9767,9236,94.56,285,246,5


Response-code distributions:


,Wave,Variable,Variable label,Raw code,Value label,Response type,Participants
0,Wave 1,W1hwhelpYP,YP: Whether anyone at home helps them with homework,-99.0,YP not interviewed,Special code,89
1,Wave 1,W1hwhelpYP,YP: Whether anyone at home helps them with homework,-91.0,Not applicable,Special code,94
2,Wave 1,W1hwhelpYP,YP: Whether anyone at home helps them with homework,-1.0,Don't know,Special code,6
3,Wave 1,W1hwhelpYP,YP: Whether anyone at home helps them with homework,1.0,Yes,Observed response,7605
4,Wave 1,W1hwhelpYP,YP: Whether anyone at home helps them with homework,2.0,No,Observed response,1689


Files written in this cell: 0


In [345]:
# 26: Parental homework-support routing and wave-consistency review

import pandas as pd
from scipy.stats import spearmanr
parental_homework_neighbourhood_rows = []
parental_homework_related_variables_by_wave = {}
parental_homework_review_raw_by_wave = {}
parental_homework_review_labelled_by_wave = {}
for wave in ['Wave 1', 'Wave 2']:
    wave_inventory = parental_homework_support_inventory.loc[parental_homework_support_inventory['Wave'].eq(wave)].copy().sort_values('Variable position')
    assert len(wave_inventory) == 2
    source_files = wave_inventory['Source file'].drop_duplicates().tolist()
    assert len(source_files) == 1
    source_file = source_files[0]
    source_path = source_file_lookup[source_file]
    source_reader = pd.read_stata(source_path, iterator=True)
    source_labels = source_reader.variable_labels()
    del source_reader
    source_variables = list(source_labels.keys())
    target_variables = wave_inventory['Variable'].tolist()
    target_positions = [source_variables.index(variable) for variable in target_variables]
    neighbourhood_start = max(0, min(target_positions) - 10)
    neighbourhood_end = min(len(source_variables), max(target_positions) + 11)
    neighbourhood_variables = source_variables[neighbourhood_start:neighbourhood_end]
    related_variables = []
    for variable in neighbourhood_variables:
        variable_text = (str(variable) + ' ' + str(source_labels.get(variable, ''))).lower()
        is_homework_related = any((keyword in variable_text for keyword in ['homework', 'home work', 'school work',
            'hwhelp', 'hwpchl', 'hweve', 'hwhour', 'hwrk']))
        if is_homework_related or variable in target_variables:
            related_variables.append(variable)
    related_variables = list(dict.fromkeys(related_variables))
    assert set(target_variables).issubset(set(related_variables))
    parental_homework_related_variables_by_wave[wave] = related_variables
    for position in range(neighbourhood_start, neighbourhood_end):
        variable = source_variables[position]
        parental_homework_neighbourhood_rows.append({'Wave': wave, 'Variable position': position, 'Variable': variable,
            'Variable label': source_labels.get(variable,
            ''), 'Target variable': variable in target_variables, 'Homework-related review variable': variable in related_variables})
    raw_data = pd.read_stata(source_path, columns=['NSID', *related_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *related_variables], convert_categoricals=True)
    for data in [raw_data, labelled_data]:
        data['NSID'] = standardise_nsid(data['NSID'])
        assert data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(route_index)
    labelled_data = labelled_data.set_index('NSID').reindex(route_index)
    for variable in related_variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    parental_homework_review_raw_by_wave[wave] = raw_data
    parental_homework_review_labelled_by_wave[wave] = labelled_data
parental_homework_neighbourhood = pd.DataFrame(parental_homework_neighbourhood_rows)
parental_homework_related_code_rows = []
for wave in ['Wave 1', 'Wave 2']:
    raw_data = parental_homework_review_raw_by_wave[wave]
    labelled_data = parental_homework_review_labelled_by_wave[wave]
    for variable in parental_homework_related_variables_by_wave[wave]:
        raw_values = raw_data[variable]
        labelled_values = labelled_data[variable].astype('string')
        for raw_code, participants in raw_values.value_counts(dropna=False).items():
            if pd.isna(raw_code):
                value_label = 'No source record'
                response_type = 'Unavailable'
                sort_value = 999999
            else:
                response_mask = raw_values.eq(raw_code)
                matching_labels = labelled_values.loc[response_mask].dropna().drop_duplicates().tolist()
                value_label = matching_labels[0] if matching_labels else str(raw_code)
                response_type = 'Observed response' if raw_code >= 0 else 'Special code'
                sort_value = float(raw_code)
            parental_homework_related_code_rows.append({'Wave': wave, 'Variable': variable, 'Raw code': raw_code,
                'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
parental_homework_related_code_distribution = pd.DataFrame(parental_homework_related_code_rows).sort_values(['Wave',
    'Variable', 'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
parental_homework_routing_rows = []
for wave in ['Wave 1', 'Wave 2']:
    raw_data = parental_homework_review_raw_by_wave[wave]
    wave_targets = parental_homework_support_inventory.loc[parental_homework_support_inventory['Wave'].eq(wave),
        'Variable'].tolist()
    candidate_routing_variables = [variable for variable in parental_homework_related_variables_by_wave[wave] if variable not in wave_targets]
    for target_variable in wave_targets:
        target_values = raw_data[target_variable]
        target_observed = target_values.ge(0).fillna(False).astype(bool)
        target_structural_na = target_values.eq(-91.0).fillna(False).astype(bool)
        target_other_special = target_values.lt(0).fillna(False).astype(bool) & ~target_structural_na
        for routing_variable in candidate_routing_variables:
            routing_values = raw_data[routing_variable]
            labelled_values = parental_homework_review_labelled_by_wave[wave][routing_variable].astype('string')
            for raw_code, participants in routing_values.value_counts(dropna=False).items():
                if pd.isna(raw_code):
                    routing_mask = routing_values.isna()
                    value_label = 'No source record'
                else:
                    routing_mask = routing_values.eq(raw_code)
                    matching_labels = labelled_values.loc[routing_mask].dropna().drop_duplicates().tolist()
                    value_label = matching_labels[0] if matching_labels else str(raw_code)
                participant_count = int(participants)
                observed_count = int((routing_mask & target_observed).sum())
                structural_count = int((routing_mask & target_structural_na).sum())
                other_special_count = int((routing_mask & target_other_special).sum())
                parental_homework_routing_rows.append({'Wave': wave, 'Target variable': target_variable,
                    'Routing variable': routing_variable, 'Routing code': raw_code, 'Routing response': value_label, 'Participants': participant_count, 'Target observed': observed_count, 'Observed percentage': round(observed_count / participant_count * 100,
                    2) if participant_count > 0 else pd.NA, 'Target structurally not applicable': structural_count, 'Structural percentage': round(structural_count / participant_count * 100,
                    2) if participant_count > 0 else pd.NA, 'Target other special code': other_special_count})
parental_homework_routing_summary = pd.DataFrame(parental_homework_routing_rows).sort_values(['Wave',
    'Target variable', 'Routing variable', 'Routing code'], na_position='last').reset_index(drop=True)
homework_help_wave_data = pd.DataFrame({'Wave 1': parental_homework_review_raw_by_wave['Wave 1']['W1hwhelpYP'].where(parental_homework_review_raw_by_wave['Wave 1']['W1hwhelpYP'].ge(0)),
    'Wave 2': parental_homework_review_raw_by_wave['Wave 2']['W2hwhelpYP'].where(parental_homework_review_raw_by_wave['Wave 2']['W2hwhelpYP'].ge(0))}, index=route_index)
homework_monitoring_wave_data = pd.DataFrame({'Wave 1': parental_homework_review_raw_by_wave['Wave 1']['W1hwpchlYP'].where(parental_homework_review_raw_by_wave['Wave 1']['W1hwpchlYP'].ge(0)),
    'Wave 2': parental_homework_review_raw_by_wave['Wave 2']['W2hwpchlYP'].where(parental_homework_review_raw_by_wave['Wave 2']['W2hwpchlYP'].ge(0))}, index=route_index)
parental_homework_wave_agreement_rows = []
for construct, wave_data in [('Homework help', homework_help_wave_data), ('Homework monitoring',
    homework_monitoring_wave_data)]:
    pair_data = wave_data[['Wave 1', 'Wave 2']].dropna()
    assert len(pair_data) > 0
    rank_correlation = float(spearmanr(pair_data['Wave 1'], pair_data['Wave 2']).statistic)
    parental_homework_wave_agreement_rows.append({'Construct': construct, 'Complete comparisons': len(pair_data),
        'Exact agreement percentage': round(pair_data['Wave 1'].eq(pair_data['Wave 2']).mean() * 100,
        2), "Cramer's V": round(categorical_cramers_v(pair_data['Wave 1'], pair_data['Wave 2']),
        3), 'Spearman correlation': round(rank_correlation, 3) if pd.notna(rank_correlation) else pd.NA})
parental_homework_wave_agreement_summary = pd.DataFrame(parental_homework_wave_agreement_rows)
parental_homework_within_wave_rows = []
for wave, help_variable, monitoring_variable in [('Wave 1', 'W1hwhelpYP', 'W1hwpchlYP'), ('Wave 2', 'W2hwhelpYP',
    'W2hwpchlYP')]:
    raw_data = parental_homework_review_raw_by_wave[wave]
    pair_data = pd.DataFrame({'Homework help': raw_data[help_variable].where(raw_data[help_variable].ge(0)),
        'Homework monitoring': raw_data[monitoring_variable].where(raw_data[monitoring_variable].ge(0))}, index=route_index).dropna()
    parental_homework_within_wave_rows.append({'Wave': wave, 'Complete comparisons': len(pair_data),
        "Cramer's V": round(categorical_cramers_v(pair_data['Homework help'], pair_data['Homework monitoring']),
        3), 'Spearman correlation': round(float(spearmanr(pair_data['Homework help'],
        pair_data['Homework monitoring']).statistic), 3)})
parental_homework_within_wave_summary = pd.DataFrame(parental_homework_within_wave_rows)
parental_homework_target_crosstabs = {}
for wave, help_variable, monitoring_variable in [('Wave 1', 'W1hwhelpYP', 'W1hwpchlYP'), ('Wave 2', 'W2hwhelpYP',
    'W2hwpchlYP')]:
    raw_data = parental_homework_review_raw_by_wave[wave]
    parental_homework_target_crosstabs[wave] = pd.crosstab(raw_data[help_variable], raw_data[monitoring_variable],
        margins=True, dropna=False)
print(f'Source-file neighbourhood records: {len(parental_homework_neighbourhood):,}')
print(f'Homework-related code-distribution records: {len(parental_homework_related_code_distribution):,}')
print('\nFirst 10 source-file neighbourhood records:')
display_limited(parental_homework_neighbourhood.head(10))
print(f'Additional neighbourhood records not displayed: {max(len(parental_homework_neighbourhood) - 10, 0):,}')
print('\nFirst 15 homework-related code records:')
display_limited(parental_homework_related_code_distribution.head(15))
print(f'Additional code-distribution records not displayed: {max(len(parental_homework_related_code_distribution) - 15, 0):,}')
print('\nTarget observability by possible routing-variable code:')
display_limited(parental_homework_routing_summary)
print('\nCross-wave agreement:')
display_limited(parental_homework_wave_agreement_summary)
print('\nWithin-wave overlap between help and monitoring:')
display_limited(parental_homework_within_wave_summary)
print('\nWave 1 target cross-tabulation:')
display_limited(parental_homework_target_crosstabs['Wave 1'])
print('\nWave 2 target cross-tabulation:')
display_limited(parental_homework_target_crosstabs['Wave 2'])
print('\nFiles written in this cell: 0')

Source-file neighbourhood records: 44
Homework-related code-distribution records: 180

First 10 source-file neighbourhood records:


,Wave,Variable position,Variable,Variable label,Target variable,Homework-related review variable
0,Wave 1,125,W1y10satYP,YP: Satisfaction with choice of courses and su...,False,False
1,Wave 1,126,W1sruleYP,YP: Level of rules in school,False,False
2,Wave 1,127,W1sdiscYP,YP: Level of discipline in school,False,False
3,Wave 1,128,W1squietYP,YP: Frequency of misbehaviour or troublemaking...,False,False
4,Wave 1,129,W1squiet2YP,YP: Frequency of misbehaviour or troublemaking...,False,False


Additional neighbourhood records not displayed: 34

First 15 homework-related code records:


,Wave,Variable,Raw code,Value label,Response type,Participants
0,Wave 1,W1comloc2YP,-99.0,YP not interviewed,Special code,89
1,Wave 1,W1comloc2YP,-91.0,Not applicable,Special code,1117
2,Wave 1,W1comloc2YP,-1.0,Don't know,Special code,2
3,Wave 1,W1comloc2YP,1.0,Yes,Observed response,7766
4,Wave 1,W1comloc2YP,2.0,No,Observed response,550


Additional code-distribution records not displayed: 165

Target observability by possible routing-variable code:


...


Wave 1 target cross-tabulation:


W1hwpchlYP,-99.0,-91.0,-1.0,1.0,2.0,3.0,4.0,5.0,NaN,All
W1hwhelpYP,,,,,,,,,,
-99.0,89,0,0,0,0,0,0,0,0,89
-91.0,0,94,0,0,0,0,0,0,0,94
-1.0,0,0,1,0,1,0,3,1,0,6
1.0,0,0,2,3622,2573,837,529,42,0,7605
2.0,0,0,2,556,503,199,411,18,0,1689



Wave 2 target cross-tabulation:


W2hwpchlYP,-99.0,-91.0,-1.0,1.0,2.0,3.0,4.0,5.0,NaN,All
W2hwhelpYP,,,,,,,,,,
-99.0,76,0,0,0,0,0,0,0,0,76
-91.0,0,201,0,0,0,0,0,0,0,201
-1.0,0,0,1,3,2,2,2,1,0,11
1.0,0,0,1,2715,2373,920,689,29,0,6727
2.0,0,0,4,665,807,325,653,12,0,2466



Files written in this cell: 0


In [346]:
# 27: Parental homework-support representation review

import pandas as pd
wave_1_homework_help_raw = parental_homework_review_raw_by_wave['Wave 1']['W1hwhelpYP']
wave_2_homework_help_raw = parental_homework_review_raw_by_wave['Wave 2']['W2hwhelpYP']
wave_1_homework_monitoring_raw = parental_homework_review_raw_by_wave['Wave 1']['W1hwpchlYP']
wave_2_homework_monitoring_raw = parental_homework_review_raw_by_wave['Wave 2']['W2hwpchlYP']
homework_help_at_home_pretransition_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='homework_help_at_home_pretransition')
homework_help_source = pd.Series(pd.NA, index=route_index, dtype='string')
wave_2_help_yes_mask = wave_2_homework_help_raw.eq(1.0).fillna(False).astype(bool)
wave_2_help_no_mask = wave_2_homework_help_raw.eq(2.0).fillna(False).astype(bool)
wave_2_help_no_opportunity_mask = wave_2_homework_help_raw.isin([-91.0, 3.0]).fillna(False).astype(bool)
wave_2_help_usable_mask = wave_2_help_yes_mask | wave_2_help_no_mask | wave_2_help_no_opportunity_mask
homework_help_at_home_pretransition_candidate.loc[wave_2_help_yes_mask] = 1
homework_help_at_home_pretransition_candidate.loc[wave_2_help_no_mask] = 0
homework_help_source.loc[wave_2_help_yes_mask | wave_2_help_no_mask] = 'Wave 2 observed'
homework_help_source.loc[wave_2_help_no_opportunity_mask] = 'Wave 2 no homework opportunity'
wave_1_help_fallback_mask = ~wave_2_help_usable_mask
wave_1_help_yes_mask = wave_1_help_fallback_mask & wave_1_homework_help_raw.eq(1.0).fillna(False).astype(bool)
wave_1_help_no_mask = wave_1_help_fallback_mask & wave_1_homework_help_raw.eq(2.0).fillna(False).astype(bool)
wave_1_help_no_opportunity_mask = wave_1_help_fallback_mask & wave_1_homework_help_raw.isin([-91.0,
    3.0]).fillna(False).astype(bool)
homework_help_at_home_pretransition_candidate.loc[wave_1_help_yes_mask] = 1
homework_help_at_home_pretransition_candidate.loc[wave_1_help_no_mask] = 0
homework_help_source.loc[wave_1_help_yes_mask | wave_1_help_no_mask] = 'Wave 1 fallback observed'
homework_help_source.loc[wave_1_help_no_opportunity_mask] = 'Wave 1 no homework opportunity'
homework_help_source = homework_help_source.fillna('Unavailable')
homework_monitoring_frequency_pretransition_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='homework_monitoring_frequency_pretransition')
homework_monitoring_source = pd.Series(pd.NA, index=route_index, dtype='string')
homework_monitoring_recode = {1.0: 3, 2.0: 2, 3.0: 1, 4.0: 0, 5.0: 2}
wave_2_monitoring_observed_mask = wave_2_homework_monitoring_raw.isin(list(homework_monitoring_recode.keys())).fillna(False).astype(bool)
wave_2_monitoring_no_opportunity_mask = wave_2_homework_monitoring_raw.eq(-91.0).fillna(False).astype(bool)
wave_2_monitoring_usable_mask = wave_2_monitoring_observed_mask | wave_2_monitoring_no_opportunity_mask
homework_monitoring_frequency_pretransition_candidate.loc[wave_2_monitoring_observed_mask] = wave_2_homework_monitoring_raw.loc[wave_2_monitoring_observed_mask].map(homework_monitoring_recode).astype('Int64')
homework_monitoring_source.loc[wave_2_monitoring_observed_mask] = 'Wave 2 observed'
homework_monitoring_source.loc[wave_2_monitoring_no_opportunity_mask] = 'Wave 2 no homework opportunity'
wave_1_monitoring_fallback_mask = ~wave_2_monitoring_usable_mask
wave_1_monitoring_observed_mask = wave_1_monitoring_fallback_mask & wave_1_homework_monitoring_raw.isin(list(homework_monitoring_recode.keys())).fillna(False).astype(bool)
wave_1_monitoring_no_opportunity_mask = wave_1_monitoring_fallback_mask & wave_1_homework_monitoring_raw.eq(-91.0).fillna(False).astype(bool)
homework_monitoring_frequency_pretransition_candidate.loc[wave_1_monitoring_observed_mask] = wave_1_homework_monitoring_raw.loc[wave_1_monitoring_observed_mask].map(homework_monitoring_recode).astype('Int64')
homework_monitoring_source.loc[wave_1_monitoring_observed_mask] = 'Wave 1 fallback observed'
homework_monitoring_source.loc[wave_1_monitoring_no_opportunity_mask] = 'Wave 1 no homework opportunity'
homework_monitoring_source = homework_monitoring_source.fillna('Unavailable')
assert set(homework_help_at_home_pretransition_candidate.dropna().unique()).issubset({0, 1})
assert set(homework_monitoring_frequency_pretransition_candidate.dropna().unique()).issubset({0, 1, 2, 3})
assert homework_help_at_home_pretransition_candidate.loc[homework_help_source.str.contains('no homework opportunity',
    case=False, na=False)].isna().all()
assert homework_monitoring_frequency_pretransition_candidate.loc[homework_monitoring_source.str.contains('no homework opportunity',
    case=False, na=False)].isna().all()

def summarise_homework_source(source_values, construct):
    """Summarise the selected source and structural status."""
    source_summary = source_values.value_counts().rename('Participants').rename_axis('Construction source').reset_index()
    source_summary.insert(0, 'Construct', construct)
    source_summary['Percentage of full sample'] = (source_summary['Participants'] / len(route_index) * 100).round(2)
    return source_summary
parental_homework_source_summary = pd.concat([summarise_homework_source(homework_help_source, 'Homework help at home'),
    summarise_homework_source(homework_monitoring_source, 'Homework monitoring frequency')], ignore_index=True)
parental_homework_candidate_summary = pd.DataFrame([{'Candidate predictor': 'homework_help_at_home_pretransition',
    'Representation': 'Latest usable Wave 2 response, with Wave 1 fallback', 'Non-missing': int(homework_help_at_home_pretransition_candidate.notna().sum()), 'Missing': int(homework_help_at_home_pretransition_candidate.isna().sum()), 'Missing percentage': round(homework_help_at_home_pretransition_candidate.isna().mean() * 100,
    2), 'Categories': int(homework_help_at_home_pretransition_candidate.nunique())}, {'Candidate predictor': 'homework_monitoring_frequency_pretransition',
    'Representation': 'Latest usable Wave 2 response, with Wave 1 fallback', 'Non-missing': int(homework_monitoring_frequency_pretransition_candidate.notna().sum()), 'Missing': int(homework_monitoring_frequency_pretransition_candidate.isna().sum()), 'Missing percentage': round(homework_monitoring_frequency_pretransition_candidate.isna().mean() * 100,
    2), 'Categories': int(homework_monitoring_frequency_pretransition_candidate.nunique())}])
homework_help_labels = {0: 'No help at home', 1: 'Help at home'}
homework_monitoring_labels = {0: 'Never', 1: 'Occasionally', 2: 'Sometimes or depends on the homework',
    3: 'Every time'}
parental_homework_distribution_rows = []
for predictor, values, labels in [('Homework help at home', homework_help_at_home_pretransition_candidate,
    homework_help_labels), ('Homework monitoring frequency', homework_monitoring_frequency_pretransition_candidate,
    homework_monitoring_labels)]:
    observed_count = int(values.notna().sum())
    for code, category in labels.items():
        participants = int(values.eq(code).sum())
        parental_homework_distribution_rows.append({'Candidate predictor': predictor, 'Code': code,
            'Category': category, 'Participants': participants, 'Percentage among observed': round(participants / observed_count * 100,
            2)})
    parental_homework_distribution_rows.append({'Candidate predictor': predictor, 'Code': pd.NA,
        'Category': 'Unavailable or no homework opportunity', 'Participants': int(values.isna().sum()), 'Percentage among observed': pd.NA})
parental_homework_candidate_distributions = pd.DataFrame(parental_homework_distribution_rows)
parental_homework_candidate_pair = pd.DataFrame({'Homework help': homework_help_at_home_pretransition_candidate,
    'Homework monitoring': homework_monitoring_frequency_pretransition_candidate}, index=route_index).dropna()
parental_homework_candidate_overlap = pd.DataFrame([{'Complete comparisons': len(parental_homework_candidate_pair),
    "Cramer's V": round(categorical_cramers_v(parental_homework_candidate_pair['Homework help'],
    parental_homework_candidate_pair['Homework monitoring']), 3)}])
parental_homework_candidate_crosstab = pd.crosstab(parental_homework_candidate_pair['Homework help'],
    parental_homework_candidate_pair['Homework monitoring'], margins=True, dropna=False)
print('Candidate predictor summary:')
display_limited(parental_homework_candidate_summary)
print('Construction sources:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_homework_source_summary)
print('Candidate distributions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_homework_candidate_distributions)
print('Candidate overlap:')
display_limited(parental_homework_candidate_overlap)
print('Candidate cross-tabulation:')
display_limited(parental_homework_candidate_crosstab)
print('Files written in this cell: 0')

Candidate predictor summary:


,Candidate predictor,Representation,Non-missing,Missing,Missing percentage,Categories
0,homework_help_at_home_pretransition,"Latest usable Wave 2 response, with Wave 1 fal...",9271,496,5.08,2
1,homework_monitoring_frequency_pretransition,"Latest usable Wave 2 response, with Wave 1 fal...",9311,456,4.67,4


Construction sources:


,Construct,Construction source,Participants,Percentage of full sample
0,Homework help at home,Wave 2 observed,9193,94.12
1,Homework help at home,Unavailable,252,2.58
2,Homework help at home,Wave 2 no homework opportunity,241,2.47
3,Homework help at home,Wave 1 fallback observed,78,0.8
4,Homework help at home,Wave 1 no homework opportunity,3,0.03


Candidate distributions:


...

Candidate overlap:


,Complete comparisons,Cramer's V
0,9271,0.214


Candidate cross-tabulation:


Homework monitoring,0,1,2,3,All
Homework help,,,,,
0,659,330,826,669,2484
1,696,926,2419,2746,6787
All,1355,1256,3245,3415,9271


Files written in this cell: 0


In [347]:
# 28: Parental homework-support overlap review

import pandas as pd
from scipy.stats import spearmanr
school_domain_required_columns = {'school_attitude_score', 'truancy_status_pretransition',
    'homework_evenings_pretransition'}
stage_2_school_domain_directory = stage_2_output_directory
preferred_school_domain_path = stage_2_school_domain_directory / 'stage_2_school_attitudes_engagement_domain_predictors.csv'
if preferred_school_domain_path.exists():
    school_attitudes_engagement_predictor_path = preferred_school_domain_path
else:
    matching_school_domain_paths = []
    for candidate_path in stage_2_school_domain_directory.glob('*.csv'):
        try:
            candidate_columns = set(pd.read_csv(candidate_path, nrows=0).columns)
        except Exception:
            continue
        if ({'NSID'} | school_domain_required_columns).issubset(candidate_columns):
            matching_school_domain_paths.append(candidate_path)
    school_specific_paths = [candidate_path for candidate_path in matching_school_domain_paths if 'school' in candidate_path.stem.lower() and 'predictor' in candidate_path.stem.lower()]
    if len(school_specific_paths) == 1:
        school_attitudes_engagement_predictor_path = school_specific_paths[0]
    else:
        assert len(matching_school_domain_paths) == 1, f'Unable to identify one school-attitudes and engagement predictor file. Matches: {matching_school_domain_paths}'
        school_attitudes_engagement_predictor_path = matching_school_domain_paths[0]
parental_homework_school_overlap_data = pd.read_csv(school_attitudes_engagement_predictor_path, usecols=['NSID',
    'school_attitude_score', 'truancy_status_pretransition', 'homework_evenings_pretransition'])
parental_homework_school_overlap_data['NSID'] = standardise_nsid(parental_homework_school_overlap_data['NSID'])
assert parental_homework_school_overlap_data['NSID'].is_unique
parental_homework_school_overlap_data = parental_homework_school_overlap_data.set_index('NSID').reindex(route_index)
assert parental_homework_school_overlap_data.index.equals(route_index)
expected_school_overlap_columns = school_domain_required_columns
assert expected_school_overlap_columns.issubset(parental_homework_school_overlap_data.columns)
print('School-domain predictor file:', school_attitudes_engagement_predictor_path.name)
parental_homework_overlap_data = pd.DataFrame({'Homework help at home': homework_help_at_home_pretransition_candidate,
    'Homework monitoring frequency': homework_monitoring_frequency_pretransition_candidate, 'Homework evenings': parental_homework_school_overlap_data['homework_evenings_pretransition'], 'School attitude score': parental_homework_school_overlap_data['school_attitude_score'], 'Truancy status': parental_homework_school_overlap_data['truancy_status_pretransition'], 'Parental HE expectation': parental_he_expectation_candidate, 'Parental educational aspiration': parental_educational_aspiration_candidate, 'Highest parental qualification': family_socioeconomic_overlap_data['highest_parental_qualification_code'], 'Household income band': family_socioeconomic_overlap_data['household_income_band']}, index=route_index)
assert len(parental_homework_overlap_data) == 9767
parental_homework_overlap_coverage_rows = []
for measure in parental_homework_overlap_data.columns:
    measure_values = parental_homework_overlap_data[measure]
    parental_homework_overlap_coverage_rows.append({'Measure': measure,
        'Non-missing': int(measure_values.notna().sum()), 'Missing': int(measure_values.isna().sum()), 'Missing percentage': round(measure_values.isna().mean() * 100,
        2), 'Observed values': int(measure_values.nunique())})
parental_homework_overlap_coverage_summary = pd.DataFrame(parental_homework_overlap_coverage_rows)
parental_homework_categorical_association_rows = []
for candidate_measure in ['Homework help at home', 'Homework monitoring frequency']:
    for comparison_measure in ['Homework evenings', 'Truancy status', 'Parental HE expectation',
        'Parental educational aspiration', 'Highest parental qualification', 'Household income band']:
        pair_data = parental_homework_overlap_data[[candidate_measure, comparison_measure]].dropna()
        assert len(pair_data) > 0
        parental_homework_categorical_association_rows.append({'Candidate predictor': candidate_measure,
            'Comparison measure': comparison_measure, 'Complete comparisons': len(pair_data), 'Candidate categories': int(pair_data[candidate_measure].nunique()), 'Comparison categories': int(pair_data[comparison_measure].nunique()), "Cramer's V": round(categorical_cramers_v(pair_data[candidate_measure],
            pair_data[comparison_measure]), 3)})
parental_homework_categorical_association_summary = pd.DataFrame(parental_homework_categorical_association_rows).sort_values(['Candidate predictor',
    "Cramer's V"], ascending=[True, False]).reset_index(drop=True)
parental_homework_ordinal_association_rows = []
for candidate_measure in ['Homework help at home', 'Homework monitoring frequency']:
    for comparison_measure in ['Homework evenings', 'School attitude score']:
        pair_data = parental_homework_overlap_data[[candidate_measure, comparison_measure]].apply(pd.to_numeric,
            errors='coerce').dropna()
        assert len(pair_data) > 0
        correlation_result = spearmanr(pair_data[candidate_measure], pair_data[comparison_measure])
        parental_homework_ordinal_association_rows.append({'Candidate predictor': candidate_measure,
            'Comparison measure': comparison_measure, 'Complete comparisons': len(pair_data), 'Spearman correlation': round(float(correlation_result.statistic),
            3)})
parental_homework_ordinal_association_summary = pd.DataFrame(parental_homework_ordinal_association_rows)
homework_help_by_evenings_data = parental_homework_overlap_data[['Homework help at home',
    'Homework evenings']].dropna()
homework_help_by_evenings_counts = pd.crosstab(homework_help_by_evenings_data['Homework help at home'],
    homework_help_by_evenings_data['Homework evenings'], margins=True, dropna=False)
homework_help_by_evenings_row_percentages = (pd.crosstab(homework_help_by_evenings_data['Homework help at home'],
    homework_help_by_evenings_data['Homework evenings'], normalize='index', dropna=False) * 100).round(2)
homework_monitoring_by_evenings_data = parental_homework_overlap_data[['Homework monitoring frequency',
    'Homework evenings']].dropna()
homework_monitoring_by_evenings_counts = pd.crosstab(homework_monitoring_by_evenings_data['Homework monitoring frequency'],
    homework_monitoring_by_evenings_data['Homework evenings'], margins=True, dropna=False)
homework_monitoring_by_evenings_row_percentages = (pd.crosstab(homework_monitoring_by_evenings_data['Homework monitoring frequency'],
    homework_monitoring_by_evenings_data['Homework evenings'], normalize='index', dropna=False) * 100).round(2)
print('Comparison-measure coverage:')
display_limited(parental_homework_overlap_coverage_summary)
print('Categorical associations:')
display_limited(parental_homework_categorical_association_summary)
print('Ordinal associations:')
display_limited(parental_homework_ordinal_association_summary)
print('Homework help by homework evenings:')
display_limited(homework_help_by_evenings_counts)
print('Row percentages:')
display_limited(homework_help_by_evenings_row_percentages)
print('Homework monitoring by homework evenings:')
display_limited(homework_monitoring_by_evenings_counts)
print('Row percentages:')
display_limited(homework_monitoring_by_evenings_row_percentages)
print('Files written in this cell: 0')

School-domain predictor file: stage_2_school_attitudes_engagement_domain_predictors.csv
Comparison-measure coverage:


,Measure,Non-missing,Missing,Missing percentage,Observed values
0,Homework help at home,9271,496,5.08,2
1,Homework monitoring frequency,9311,456,4.67,4
2,Homework evenings,9509,258,2.64,6
3,School attitude score,9511,256,2.62,108
4,Truancy status,9491,276,2.83,2


Categorical associations:


,Candidate predictor,Comparison measure,Complete comparisons,Candidate categories,Comparison categories,Cramer's V
0,Homework help at home,Highest parental qualification,9258,2,7,0.156
1,Homework help at home,Household income band,8539,2,9,0.138
2,Homework help at home,Homework evenings,9265,2,6,0.102
3,Homework help at home,Parental educational aspiration,9068,2,4,0.076
4,Homework help at home,Truancy status,9248,2,2,0.072


Ordinal associations:


...

Homework monitoring by homework evenings:


Homework evenings,0.0,1.0,2.0,3.0,4.0,5.0,All
Homework monitoring frequency,,,,,,,
0,226,176,229,287,175,281,1374
1,125,177,272,311,170,210,1265
2,209,426,712,873,456,572,3248
3,198,444,705,936,503,632,3418
All,758,1223,1918,2407,1304,1695,9305


Row percentages:


Homework evenings,0.0,1.0,2.0,3.0,4.0,5.0
Homework monitoring frequency,,,,,,
0,16.45,12.81,16.67,20.89,12.74,20.45
1,9.88,13.99,21.50,24.58,13.44,16.60
2,6.43,13.12,21.92,26.88,14.04,17.61
3,5.79,12.99,20.63,27.38,14.72,18.49


Files written in this cell: 0


In [348]:
# 29: Homework support and monitoring decisions

import pandas as pd
homework_help_at_home_pretransition = homework_help_at_home_pretransition_candidate.copy().astype('Int64').rename('homework_help_at_home_pretransition')
homework_monitoring_frequency_pretransition = homework_monitoring_frequency_pretransition_candidate.copy().astype('Int64').rename('homework_monitoring_frequency_pretransition')
assert homework_help_at_home_pretransition.index.equals(route_index)
assert homework_monitoring_frequency_pretransition.index.equals(route_index)
assert set(homework_help_at_home_pretransition.dropna().unique()).issubset({0, 1})
assert set(homework_monitoring_frequency_pretransition.dropna().unique()).issubset({0, 1, 2, 3})
homework_support_monitoring_predictors = pd.DataFrame({'NSID': route_index,
    'homework_help_at_home_pretransition': homework_help_at_home_pretransition.to_numpy(), 'homework_monitoring_frequency_pretransition': homework_monitoring_frequency_pretransition.to_numpy()})
assert len(homework_support_monitoring_predictors) == 9767
assert homework_support_monitoring_predictors['NSID'].is_unique
homework_support_monitoring_variable_decisions = parental_homework_support_inventory.copy()
homework_support_monitoring_variable_decisions['Homework-support decision'] = 'Construction input'
homework_support_monitoring_variable_decisions['Homework-support role'] = pd.NA
homework_support_monitoring_variable_decisions['Homework-support reason'] = pd.NA
homework_help_variable_mask = homework_support_monitoring_variable_decisions['Variable'].isin(['W1hwhelpYP',
    'W2hwhelpYP'])
homework_monitoring_variable_mask = homework_support_monitoring_variable_decisions['Variable'].isin(['W1hwpchlYP',
    'W2hwpchlYP'])
homework_support_monitoring_variable_decisions.loc[homework_help_variable_mask,
    'Homework-support role'] = 'Homework help at home'
homework_support_monitoring_variable_decisions.loc[homework_help_variable_mask,
    'Homework-support reason'] = 'The repeated item identifies whether anyone at home helps the young person with homework. Wave 2 provides the principal pre-transition measure and Wave 1 is used only when Wave 2 does not provide a usable status'
homework_support_monitoring_variable_decisions.loc[homework_monitoring_variable_mask,
    'Homework-support role'] = 'Homework monitoring frequency'
homework_support_monitoring_variable_decisions.loc[homework_monitoring_variable_mask,
    'Homework-support reason'] = 'The repeated item measures how frequently someone at home makes sure that the young person completes homework. Wave 2 provides the principal pre-transition measure and Wave 1 is used only when Wave 2 does not provide a usable status'
assert len(homework_support_monitoring_variable_decisions) == 4
assert homework_support_monitoring_variable_decisions['Homework-support decision'].eq('Construction input').all()
assert homework_support_monitoring_variable_decisions['Homework-support role'].notna().all()
homework_support_monitoring_representation_decisions = pd.DataFrame([{'Representation': 'Wave 1 measures only',
    'Decision': 'Do not retain', 'Reason': 'Wave 2 provides more recent pre-transition information for most participants'}, {'Representation': 'Wave 2 measures without fallback',
    'Decision': 'Do not retain', 'Reason': 'A small number of participants have usable Wave 1 responses but no usable Wave 2 response'}, {'Representation': 'Latest usable Wave 2 response with Wave 1 fallback',
    'Decision': 'Retain', 'Reason': 'This prioritises the most recent permitted response while recovering limited item non-response from Wave 1'}, {'Representation': 'Treat structural non-applicability as no homework support',
    'Decision': 'Do not construct', 'Reason': 'Structural non-applicability reflects the absence of a homework opportunity and is not equivalent to receiving no help or monitoring'}, {'Representation': 'Cross-wave mean or cumulative measure',
    'Decision': 'Do not retain', 'Reason': 'Homework support and monitoring may change between waves, and the response categories do not support meaningful averaging'}, {'Representation': 'Combined homework-support composite',
    'Decision': 'Do not construct', 'Reason': 'Homework help and homework monitoring measure different forms of household support and show only modest within-wave association'}, {'Representation': 'Homework help at home',
    'Decision': 'Retain', 'Reason': 'The binary measure has high coverage and low overlap with retained school-engagement, socioeconomic and parental-expectation measures'}, {'Representation': 'Homework monitoring frequency',
    'Decision': 'Retain', 'Reason': 'The four-category measure has high coverage, retains meaningful frequency variation and is distinct from homework frequency itself'}])
homework_support_highest_overlap = parental_homework_categorical_association_summary.groupby('Candidate predictor',
    as_index=False)["Cramer's V"].max().rename(columns={"Cramer's V": "Highest categorical overlap Cramer's V"})
homework_support_monitoring_predictor_summary = pd.DataFrame([{'Predictor': 'homework_help_at_home_pretransition',
    'Source representation': 'Wave 2 observed response with limited Wave 1 fallback', 'Non-missing': int(homework_help_at_home_pretransition.notna().sum()), 'Missing': int(homework_help_at_home_pretransition.isna().sum()), 'Missing percentage': round(homework_help_at_home_pretransition.isna().mean() * 100,
    2), 'Categories': int(homework_help_at_home_pretransition.nunique()), "Highest categorical overlap Cramer's V": float(homework_support_highest_overlap.loc[homework_support_highest_overlap['Candidate predictor'].eq('Homework help at home'),
    "Highest categorical overlap Cramer's V"].iloc[0])}, {'Predictor': 'homework_monitoring_frequency_pretransition',
    'Source representation': 'Wave 2 observed response with limited Wave 1 fallback', 'Non-missing': int(homework_monitoring_frequency_pretransition.notna().sum()), 'Missing': int(homework_monitoring_frequency_pretransition.isna().sum()), 'Missing percentage': round(homework_monitoring_frequency_pretransition.isna().mean() * 100,
    2), 'Categories': int(homework_monitoring_frequency_pretransition.nunique()), "Highest categorical overlap Cramer's V": float(homework_support_highest_overlap.loc[homework_support_highest_overlap['Candidate predictor'].eq('Homework monitoring frequency'),
    "Highest categorical overlap Cramer's V"].iloc[0])}])
homework_support_monitoring_decision_display = homework_support_monitoring_variable_decisions.sort_values(['Source order',
    'Variable position'])[['Wave', 'Source type', 'Source file', 'Variable', 'Variable label', 'Timing status',
    'Homework-support decision', 'Homework-support role', 'Homework-support reason']].reset_index(drop=True)
print('Homework support and monitoring predictors retained: 2')
print('Retained predictor summary:')
with pd.option_context('display.max_colwidth', None):
    display_limited(homework_support_monitoring_predictor_summary)
print('Representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(homework_support_monitoring_representation_decisions)
print('Variable decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(homework_support_monitoring_decision_display)
print('Files written in this cell: 0')

Homework support and monitoring predictors retained: 2
Retained predictor summary:


,Predictor,Source representation,Non-missing,Missing,Missing percentage,Categories,Highest categorical overlap Cramer's V
0,homework_help_at_home_pretransition,Wave 2 observed response with limited Wave 1 fallback,9271,496,5.08,2,0.156
1,homework_monitoring_frequency_pretransition,Wave 2 observed response with limited Wave 1 fallback,9311,456,4.67,4,0.093


Representation decisions:


,Representation,Decision,Reason
0,Wave 1 measures only,Do not retain,Wave 2 provides more recent pre-transition information for most participants
1,Wave 2 measures without fallback,Do not retain,A small number of participants have usable Wave 1 responses but no usable Wave 2 response
2,Latest usable Wave 2 response with Wave 1 fallback,Retain,This prioritises the most recent permitted response while recovering limited item non-response from Wave 1
3,Treat structural non-applicability as no homework support,Do not construct,Structural non-applicability reflects the absence of a homework opportunity and is not equivalent to receiving no help or monitoring
4,Cross-wave mean or cumulative measure,Do not retain,"Homework support and monitoring may change between waves, and the response categories do not support meaningful averaging"


Variable decisions:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Homework-support decision,Homework-support role,Homework-support reason
0,Wave 1,Young person,wave_one_lsype_young_person_2020,W1hwhelpYP,YP: Whether anyone at home helps them with homework,Pre-transition source,Construction input,Homework help at home,The repeated item identifies whether anyone at home helps the young person with homework. Wave 2 provides the principal pre-transition measure and Wave 1 is used only when Wave 2 does not provide a usable status
1,Wave 1,Young person,wave_one_lsype_young_person_2020,W1hwpchlYP,YP: Whether anyone at home makes sure that do homework,Pre-transition source,Construction input,Homework monitoring frequency,The repeated item measures how frequently someone at home makes sure that the young person completes homework. Wave 2 provides the principal pre-transition measure and Wave 1 is used only when Wave 2 does not provide a usable status
2,Wave 2,Young person,wave_two_lsype_young_person_2020,W2hwhelpYP,YP: Whether anyone at home helps them with homework,Pre-transition source,Construction input,Homework help at home,The repeated item identifies whether anyone at home helps the young person with homework. Wave 2 provides the principal pre-transition measure and Wave 1 is used only when Wave 2 does not provide a usable status
3,Wave 2,Young person,wave_two_lsype_young_person_2020,W2hwpchlYP,YP: Whether anyone at home makes sure that do homework,Pre-transition source,Construction input,Homework monitoring frequency,The repeated item measures how frequently someone at home makes sure that the young person completes homework. Wave 2 provides the principal pre-transition measure and Wave 1 is used only when Wave 2 does not provide a usable status


Files written in this cell: 0


In [349]:
# 30: Parental monitoring track-name inspection

import pandas as pd
domain_9_review_track_counts = domain_9_screened_candidates['Domain 9 review track'].astype('string').value_counts(dropna=False).rename('Variables').rename_axis('Domain 9 review track').reset_index()
parental_monitoring_track_matches = domain_9_review_track_counts.loc[domain_9_review_track_counts['Domain 9 review track'].astype('string').str.contains('monitor',
    case=False, na=False)].reset_index(drop=True)
parental_monitoring_variable_matches = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 review track'].astype('string').str.contains('monitor',
    case=False, na=False)].sort_values(['Domain 9 review track', 'Source order', 'Variable position'])[['Wave',
    'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Domain 9 screening status', 'Domain 9 review track']].reset_index(drop=True)
print("Review-track names containing 'monitor':")
display_limited(parental_monitoring_track_matches)
print('Variables in matching tracks:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_monitoring_variable_matches)
print('Files written in this cell: 0')

Review-track names containing 'monitor':


,Domain 9 review track,Variables
0,Homework support and monitoring,4
1,Parental monitoring,3


Variables in matching tracks:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Domain 9 screening status,Domain 9 review track
0,Wave 1,Young person,wave_one_lsype_young_person_2020,136,W1hwhelpYP,YP: Whether anyone at home helps them with homework,Core review,Homework support and monitoring
1,Wave 1,Young person,wave_one_lsype_young_person_2020,137,W1hwpchlYP,YP: Whether anyone at home makes sure that do homework,Core review,Homework support and monitoring
2,Wave 2,Young person,wave_two_lsype_young_person_2020,235,W2hwhelpYP,YP: Whether anyone at home helps them with homework,Core review,Homework support and monitoring
3,Wave 2,Young person,wave_two_lsype_young_person_2020,236,W2hwpchlYP,YP: Whether anyone at home makes sure that do homework,Core review,Homework support and monitoring
4,Wave 1,Young person,wave_one_lsype_young_person_2020,252,W1gowhereYP,YP: How often parents know where going when out in evening,Core review,Parental monitoring


Files written in this cell: 0


In [350]:
# 31: Parental monitoring response and reporter comparison review

import pandas as pd
from scipy.stats import spearmanr
parental_monitoring_inventory = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 review track'].eq('Parental monitoring')].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(parental_monitoring_inventory) == 3
assert set(parental_monitoring_inventory['Variable']) == {'W1gowhereYP', 'W1limitsnYP', 'W1paroutMP'}
parental_monitoring_raw_tables = {}
parental_monitoring_labelled_tables = {}
for source_file, source_inventory in parental_monitoring_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    for data in [raw_data, labelled_data]:
        data['NSID'] = standardise_nsid(data['NSID'])
        assert data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(route_index)
    labelled_data = labelled_data.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    parental_monitoring_raw_tables[source_file] = raw_data
    parental_monitoring_labelled_tables[source_file] = labelled_data

def get_parental_monitoring_series(variable, labelled=False):
    """Return one parental-monitoring variable on the analysis index."""
    variable_row = parental_monitoring_inventory.loc[parental_monitoring_inventory['Variable'].eq(variable)].iloc[0]
    source_file = variable_row['Source file']
    source_tables = parental_monitoring_labelled_tables if labelled else parental_monitoring_raw_tables
    return source_tables[source_file][variable]
young_person_parental_knowledge_raw = get_parental_monitoring_series('W1gowhereYP')
school_night_curfew_raw = get_parental_monitoring_series('W1limitsnYP')
main_parent_knowledge_raw = get_parental_monitoring_series('W1paroutMP')
parental_monitoring_coverage_rows = []
parental_monitoring_code_rows = []
for _, inventory_row in parental_monitoring_inventory.iterrows():
    variable = inventory_row['Variable']
    raw_values = get_parental_monitoring_series(variable)
    labelled_values = get_parental_monitoring_series(variable, labelled=True).astype('string')
    observed_mask = raw_values.ge(0).fillna(False).astype(bool)
    parental_monitoring_coverage_rows.append({'Respondent': 'Young person' if variable.endswith('YP') else 'Main parent',
        'Variable': variable, 'Variable label': inventory_row['Variable label'], 'Observed responses': int(observed_mask.sum()), 'Observed percentage': round(observed_mask.mean() * 100,
        2), 'Special-code responses': int(raw_values.lt(0).fillna(False).sum()), 'No source record': int(raw_values.isna().sum()), 'Observed categories': int(raw_values.loc[observed_mask].nunique())})
    for raw_code, participants in raw_values.value_counts(dropna=False).items():
        if pd.isna(raw_code):
            value_label = 'No source record'
            response_type = 'Unavailable'
            sort_value = 999999
        else:
            response_mask = raw_values.eq(raw_code)
            matching_labels = labelled_values.loc[response_mask].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            response_type = 'Observed response' if raw_code >= 0 else 'Special code'
            sort_value = float(raw_code)
        parental_monitoring_code_rows.append({'Variable': variable, 'Raw code': raw_code, 'Value label': value_label,
            'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
parental_monitoring_coverage_summary = pd.DataFrame(parental_monitoring_coverage_rows)
parental_monitoring_code_distribution = pd.DataFrame(parental_monitoring_code_rows).sort_values(['Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
young_person_parental_knowledge_observed = young_person_parental_knowledge_raw.where(young_person_parental_knowledge_raw.ge(0))
main_parent_knowledge_observed = main_parent_knowledge_raw.where(main_parent_knowledge_raw.ge(0))
school_night_curfew_observed = school_night_curfew_raw.where(school_night_curfew_raw.ge(0))
parental_knowledge_reporter_pair = pd.DataFrame({'Young-person report': young_person_parental_knowledge_observed,
    'Main-parent report': main_parent_knowledge_observed}, index=route_index).dropna()
assert len(parental_knowledge_reporter_pair) > 0
parental_knowledge_reporter_comparison = pd.DataFrame([{'Complete comparisons': len(parental_knowledge_reporter_pair),
    'Exact agreement percentage': round(parental_knowledge_reporter_pair['Young-person report'].eq(parental_knowledge_reporter_pair['Main-parent report']).mean() * 100,
    2), "Cramer's V": round(categorical_cramers_v(parental_knowledge_reporter_pair['Young-person report'],
    parental_knowledge_reporter_pair['Main-parent report']), 3), 'Spearman correlation': round(float(spearmanr(parental_knowledge_reporter_pair['Young-person report'],
    parental_knowledge_reporter_pair['Main-parent report']).statistic), 3)}])
parental_knowledge_reporter_counts = pd.crosstab(parental_knowledge_reporter_pair['Young-person report'],
    parental_knowledge_reporter_pair['Main-parent report'], margins=True, dropna=False)
parental_knowledge_reporter_row_percentages = (pd.crosstab(parental_knowledge_reporter_pair['Young-person report'],
    parental_knowledge_reporter_pair['Main-parent report'], normalize='index', dropna=False) * 100).round(2)
parental_monitoring_construct_rows = []
for knowledge_measure, knowledge_values in [('Young-person report of parental knowledge',
    young_person_parental_knowledge_observed), ('Main-parent report of parental knowledge',
    main_parent_knowledge_observed)]:
    pair_data = pd.DataFrame({'Parental knowledge': knowledge_values,
        'School-night curfew': school_night_curfew_observed}, index=route_index).dropna()
    parental_monitoring_construct_rows.append({'Knowledge measure': knowledge_measure,
        'Complete comparisons': len(pair_data), "Cramer's V": round(categorical_cramers_v(pair_data['Parental knowledge'],
        pair_data['School-night curfew']), 3), 'Spearman correlation': round(float(spearmanr(pair_data['Parental knowledge'],
        pair_data['School-night curfew']).statistic), 3)})
parental_monitoring_construct_comparison = pd.DataFrame(parental_monitoring_construct_rows)
print(f'Parental-monitoring variables reviewed: {len(parental_monitoring_inventory)}')
print('Variable inventory:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_monitoring_inventory[['Wave', 'Source type', 'Source file', 'Variable', 'Variable label',
        'Timing status', 'Domain 9 review track']])
print('Coverage summary:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_monitoring_coverage_summary)
print('Response-code distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_monitoring_code_distribution)
print('Young-person and main-parent report comparison:')
display_limited(parental_knowledge_reporter_comparison)
print('Reporter cross-tabulation:')
display_limited(parental_knowledge_reporter_counts)
print('Reporter row percentages:')
display_limited(parental_knowledge_reporter_row_percentages)
print('Parental knowledge and school-night curfew:')
display_limited(parental_monitoring_construct_comparison)
print('Files written in this cell: 0')

Parental-monitoring variables reviewed: 3
Variable inventory:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Domain 9 review track
0,Wave 1,Young person,wave_one_lsype_young_person_2020,W1gowhereYP,YP: How often parents know where going when out in evening,Pre-transition source,Parental monitoring
1,Wave 1,Young person,wave_one_lsype_young_person_2020,W1limitsnYP,YP: Whether parents ever set curfew on school nights,Pre-transition source,Parental monitoring
2,Wave 1,Family background,wave_one_lsype_family_background_2020,W1paroutMP,MP: How often know where YP is when goes out in evening,Pre-transition source,Parental monitoring


Coverage summary:


,Respondent,Variable,Variable label,Observed responses,Observed percentage,Special-code responses,No source record,Observed categories
0,Young person,W1gowhereYP,YP: How often parents know where going when out in evening,9272,94.93,252,243,6
1,Young person,W1limitsnYP,YP: Whether parents ever set curfew on school nights,9102,93.19,422,243,4
2,Main parent,W1paroutMP,MP: How often know where YP is when goes out in evening,9414,96.39,110,243,6


Response-code distributions:


...

Reporter row percentages:


Main-parent report,1.0,2.0,3.0,4.0,5.0,6.0
Young-person report,,,,,,
1.0,79.87,6.46,0.90,0.21,0.58,11.99
2.0,66.25,23.24,2.18,0.60,0.69,7.04
3.0,54.87,27.80,6.50,2.71,1.44,6.68
4.0,48.52,27.81,8.88,4.73,2.37,7.69
5.0,38.46,21.54,9.23,1.54,9.23,20.00


Parental knowledge and school-night curfew:


,Knowledge measure,Complete comparisons,Cramer's V,Spearman correlation
0,Young-person report of parental knowledge,9075,0.275,-0.013
1,Main-parent report of parental knowledge,9006,0.186,0.093


Files written in this cell: 0


In [351]:
# 32: Parental monitoring candidate and overlap review

import pandas as pd
parental_knowledge_recode = {1.0: 4, 2.0: 3, 3.0: 2, 4.0: 1, 5.0: 0, 6.0: 5}
young_person_parental_knowledge_pretransition_candidate = young_person_parental_knowledge_raw.map(parental_knowledge_recode).astype('Int64').rename('young_person_reported_parental_knowledge_pretransition')
main_parent_reported_parental_knowledge_pretransition_candidate = main_parent_knowledge_raw.map(parental_knowledge_recode).astype('Int64').rename('main_parent_reported_parental_knowledge_pretransition')
school_night_curfew_recode = {1.0: 0, 2.0: 1, 3.0: 2, 4.0: 3}
school_night_curfew_pretransition_candidate = school_night_curfew_raw.map(school_night_curfew_recode).astype('Int64').rename('school_night_curfew_pretransition')
assert set(young_person_parental_knowledge_pretransition_candidate.dropna().unique()).issubset({0, 1, 2, 3, 4, 5})
assert set(main_parent_reported_parental_knowledge_pretransition_candidate.dropna().unique()).issubset({0, 1, 2, 3, 4,
    5})
assert set(school_night_curfew_pretransition_candidate.dropna().unique()).issubset({0, 1, 2, 3})
stage_2_predictor_directory = stage_2_output_directory
experiences_behaviours_predictor_path = stage_2_predictor_directory / 'stage_2_experiences_behaviours_domain_predictors.csv'
assert experiences_behaviours_predictor_path.exists()
parental_monitoring_behaviour_overlap_data = pd.read_csv(experiences_behaviours_predictor_path, usecols=['NSID',
    'smoking_activity_pretransition', 'alcohol_use_frequency_pretransition', 'recent_antisocial_behaviour_pretransition', 'recent_police_contact_pretransition'])
parental_monitoring_behaviour_overlap_data['NSID'] = standardise_nsid(parental_monitoring_behaviour_overlap_data['NSID'])
assert parental_monitoring_behaviour_overlap_data['NSID'].is_unique
parental_monitoring_behaviour_overlap_data = parental_monitoring_behaviour_overlap_data.set_index('NSID').reindex(route_index)
assert parental_monitoring_behaviour_overlap_data.index.equals(route_index)
parental_monitoring_overlap_data = pd.DataFrame({'Young-person parental knowledge': young_person_parental_knowledge_pretransition_candidate,
    'Main-parent parental knowledge': main_parent_reported_parental_knowledge_pretransition_candidate, 'School-night curfew': school_night_curfew_pretransition_candidate, 'Homework monitoring frequency': homework_monitoring_frequency_pretransition, 'Truancy status': parental_homework_school_overlap_data['truancy_status_pretransition'], 'School attitude score': parental_homework_school_overlap_data['school_attitude_score'], 'Smoking activity': parental_monitoring_behaviour_overlap_data['smoking_activity_pretransition'], 'Alcohol-use frequency': parental_monitoring_behaviour_overlap_data['alcohol_use_frequency_pretransition'], 'Recent antisocial behaviour': parental_monitoring_behaviour_overlap_data['recent_antisocial_behaviour_pretransition'], 'Recent police contact': parental_monitoring_behaviour_overlap_data['recent_police_contact_pretransition']}, index=route_index)
assert len(parental_monitoring_overlap_data) == 9767
parental_monitoring_candidate_labels = {'Young-person parental knowledge': {0: 'Never', 1: 'Rarely', 2: 'Sometimes',
    3: 'Usually', 4: 'Always', 5: 'Does not go out in the evening'}, 'Main-parent parental knowledge': {0: 'Never',
    1: 'Rarely', 2: 'Sometimes', 3: 'Usually', 4: 'Always', 5: 'Young person does not go out in the evening'}, 'School-night curfew': {0: 'Never',
    1: 'Sometimes', 2: 'Often', 3: 'Not allowed out or does not go out'}}
parental_monitoring_candidate_summary_rows = []
parental_monitoring_candidate_distribution_rows = []
for candidate_measure in ['Young-person parental knowledge', 'Main-parent parental knowledge', 'School-night curfew']:
    candidate_values = parental_monitoring_overlap_data[candidate_measure]
    observed_count = int(candidate_values.notna().sum())
    parental_monitoring_candidate_summary_rows.append({'Candidate predictor': candidate_measure,
        'Non-missing': observed_count, 'Missing': int(candidate_values.isna().sum()), 'Missing percentage': round(candidate_values.isna().mean() * 100,
        2), 'Categories': int(candidate_values.nunique())})
    for code, category in parental_monitoring_candidate_labels[candidate_measure].items():
        participants = int(candidate_values.eq(code).sum())
        parental_monitoring_candidate_distribution_rows.append({'Candidate predictor': candidate_measure, 'Code': code,
            'Category': category, 'Participants': participants, 'Percentage among observed': round(participants / observed_count * 100,
            2)})
parental_monitoring_candidate_summary = pd.DataFrame(parental_monitoring_candidate_summary_rows)
parental_monitoring_candidate_distribution = pd.DataFrame(parental_monitoring_candidate_distribution_rows)
parental_monitoring_candidate_pair_rows = []
for first_measure, second_measure in [('Young-person parental knowledge', 'Main-parent parental knowledge'),
    ('Young-person parental knowledge', 'School-night curfew'), ('Main-parent parental knowledge',
    'School-night curfew')]:
    pair_data = parental_monitoring_overlap_data[[first_measure, second_measure]].dropna()
    parental_monitoring_candidate_pair_rows.append({'First measure': first_measure, 'Second measure': second_measure,
        'Complete comparisons': len(pair_data), "Cramer's V": round(categorical_cramers_v(pair_data[first_measure],
        pair_data[second_measure]), 3)})
parental_monitoring_candidate_pair_summary = pd.DataFrame(parental_monitoring_candidate_pair_rows)
parental_monitoring_overlap_rows = []
for candidate_measure in ['Young-person parental knowledge', 'Main-parent parental knowledge', 'School-night curfew']:
    for comparison_measure in ['Homework monitoring frequency', 'Truancy status', 'Smoking activity',
        'Alcohol-use frequency', 'Recent antisocial behaviour', 'Recent police contact']:
        pair_data = parental_monitoring_overlap_data[[candidate_measure, comparison_measure]].dropna()
        assert len(pair_data) > 0
        parental_monitoring_overlap_rows.append({'Candidate predictor': candidate_measure,
            'Comparison measure': comparison_measure, 'Complete comparisons': len(pair_data), 'Candidate categories': int(pair_data[candidate_measure].nunique()), 'Comparison categories': int(pair_data[comparison_measure].nunique()), "Cramer's V": round(categorical_cramers_v(pair_data[candidate_measure],
            pair_data[comparison_measure]), 3)})
parental_monitoring_overlap_summary = pd.DataFrame(parental_monitoring_overlap_rows).sort_values(['Candidate predictor',
    "Cramer's V"], ascending=[True, False]).reset_index(drop=True)
young_person_knowledge_curfew_data = parental_monitoring_overlap_data[['Young-person parental knowledge',
    'School-night curfew']].dropna()
young_person_knowledge_curfew_counts = pd.crosstab(young_person_knowledge_curfew_data['Young-person parental knowledge'],
    young_person_knowledge_curfew_data['School-night curfew'], margins=True, dropna=False)
young_person_knowledge_curfew_row_percentages = (pd.crosstab(young_person_knowledge_curfew_data['Young-person parental knowledge'],
    young_person_knowledge_curfew_data['School-night curfew'], normalize='index', dropna=False) * 100).round(2)
print('Candidate predictor summary:')
display_limited(parental_monitoring_candidate_summary)
print('Candidate distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_monitoring_candidate_distribution)
print('Associations among monitoring candidates:')
display_limited(parental_monitoring_candidate_pair_summary)
print('Associations with retained behaviour and engagement measures:')
display_limited(parental_monitoring_overlap_summary)
print('Young-person parental knowledge by school-night curfew:')
display_limited(young_person_knowledge_curfew_counts)
print('Row percentages:')
display_limited(young_person_knowledge_curfew_row_percentages)
print('Files written in this cell: 0')

Candidate predictor summary:


,Candidate predictor,Non-missing,Missing,Missing percentage,Categories
0,Young-person parental knowledge,9272,495,5.07,6
1,Main-parent parental knowledge,9414,353,3.61,6
2,School-night curfew,9102,665,6.81,4


Candidate distributions:


,Candidate predictor,Code,Category,Participants,Percentage among observed
0,Young-person parental knowledge,0,Never,66,0.71
1,Young-person parental knowledge,1,Rarely,169,1.82
2,Young-person parental knowledge,2,Sometimes,561,6.05
3,Young-person parental knowledge,3,Usually,2185,23.57
4,Young-person parental knowledge,4,Always,5744,61.95


Associations among monitoring candidates:


...

Young-person parental knowledge by school-night curfew:


School-night curfew,0,1,2,3,All
Young-person parental knowledge,,,,,
0,10,11,31,9,61
1,16,46,95,8,165
2,21,152,345,31,549
3,67,494,1458,129,2148
4,142,694,3778,1002,5616


Row percentages:


School-night curfew,0,1,2,3
Young-person parental knowledge,,,,
0,16.39,18.03,50.82,14.75
1,9.70,27.88,57.58,4.85
2,3.83,27.69,62.84,5.65
3,3.12,23.00,67.88,6.01
4,2.53,12.36,67.27,17.84


Files written in this cell: 0


In [352]:
# 33: Parental monitoring decisions

import pandas as pd
young_person_parental_knowledge_recode = {0: 0, 1: 0, 2: 1, 3: 2, 4: 3, 5: 4}
parental_knowledge_of_evening_whereabouts_pretransition = young_person_parental_knowledge_pretransition_candidate.map(young_person_parental_knowledge_recode).astype('Int64').rename('parental_knowledge_of_evening_whereabouts_pretransition')
school_night_curfew_pretransition = school_night_curfew_pretransition_candidate.copy().astype('Int64').rename('school_night_curfew_pretransition')
assert parental_knowledge_of_evening_whereabouts_pretransition.index.equals(route_index)
assert school_night_curfew_pretransition.index.equals(route_index)
assert set(parental_knowledge_of_evening_whereabouts_pretransition.dropna().unique()).issubset({0, 1, 2, 3, 4})
assert set(school_night_curfew_pretransition.dropna().unique()).issubset({0, 1, 2, 3})
parental_monitoring_predictors = pd.DataFrame({'NSID': route_index,
    'parental_knowledge_of_evening_whereabouts_pretransition': parental_knowledge_of_evening_whereabouts_pretransition.to_numpy(), 'school_night_curfew_pretransition': school_night_curfew_pretransition.to_numpy()})
assert len(parental_monitoring_predictors) == 9767
assert parental_monitoring_predictors['NSID'].is_unique
parental_monitoring_retained_overlap_data = parental_monitoring_overlap_data.copy()
parental_monitoring_retained_overlap_data['Parental knowledge of evening whereabouts'] = parental_knowledge_of_evening_whereabouts_pretransition
parental_monitoring_retained_overlap_data['School-night curfew retained'] = school_night_curfew_pretransition
parental_monitoring_retained_overlap_rows = []
for candidate_measure in ['Parental knowledge of evening whereabouts', 'School-night curfew retained']:
    for comparison_measure in ['Homework monitoring frequency', 'Truancy status', 'Smoking activity',
        'Alcohol-use frequency', 'Recent antisocial behaviour', 'Recent police contact']:
        pair_data = parental_monitoring_retained_overlap_data[[candidate_measure, comparison_measure]].dropna()
        assert len(pair_data) > 0
        parental_monitoring_retained_overlap_rows.append({'Candidate predictor': candidate_measure,
            'Comparison measure': comparison_measure, 'Complete comparisons': len(pair_data), "Cramer's V": round(categorical_cramers_v(pair_data[candidate_measure],
            pair_data[comparison_measure]), 3)})
parental_monitoring_retained_overlap_summary = pd.DataFrame(parental_monitoring_retained_overlap_rows).sort_values(['Candidate predictor',
    "Cramer's V"], ascending=[True, False]).reset_index(drop=True)
parental_monitoring_retained_pair = pd.DataFrame({'Parental knowledge': parental_knowledge_of_evening_whereabouts_pretransition,
    'School-night curfew': school_night_curfew_pretransition}, index=route_index).dropna()
parental_monitoring_retained_pair_summary = pd.DataFrame([{'Complete comparisons': len(parental_monitoring_retained_pair),
    "Cramer's V": round(categorical_cramers_v(parental_monitoring_retained_pair['Parental knowledge'],
    parental_monitoring_retained_pair['School-night curfew']), 3)}])
parental_monitoring_variable_decisions = parental_monitoring_inventory.copy()
parental_monitoring_variable_decisions['Parental-monitoring decision'] = pd.NA
parental_monitoring_variable_decisions['Parental-monitoring role'] = pd.NA
parental_monitoring_variable_decisions['Parental-monitoring reason'] = pd.NA
young_person_knowledge_variable_mask = parental_monitoring_variable_decisions['Variable'].eq('W1gowhereYP')
curfew_variable_mask = parental_monitoring_variable_decisions['Variable'].eq('W1limitsnYP')
main_parent_knowledge_variable_mask = parental_monitoring_variable_decisions['Variable'].eq('W1paroutMP')
parental_monitoring_variable_decisions.loc[young_person_knowledge_variable_mask,
    'Parental-monitoring decision'] = 'Construction input'
parental_monitoring_variable_decisions.loc[young_person_knowledge_variable_mask,
    'Parental-monitoring role'] = 'Young-person report of parental knowledge of evening whereabouts'
parental_monitoring_variable_decisions.loc[young_person_knowledge_variable_mask,
    'Parental-monitoring reason'] = "The item provides the young person's account of how consistently parents know their evening whereabouts. Never and rarely are consolidated because both categories are small, while not going out remains a separate category"
parental_monitoring_variable_decisions.loc[curfew_variable_mask,
    'Parental-monitoring decision'] = 'Construction input'
parental_monitoring_variable_decisions.loc[curfew_variable_mask, 'Parental-monitoring role'] = 'School-night curfew'
parental_monitoring_variable_decisions.loc[curfew_variable_mask,
    'Parental-monitoring reason'] = 'The item measures the frequency of school-night curfew setting and represents a form of parental control distinct from knowledge of evening whereabouts'
parental_monitoring_variable_decisions.loc[main_parent_knowledge_variable_mask,
    'Parental-monitoring decision'] = 'Review support'
parental_monitoring_variable_decisions.loc[main_parent_knowledge_variable_mask,
    'Parental-monitoring role'] = 'Main-parent comparison report of parental knowledge'
parental_monitoring_variable_decisions.loc[main_parent_knowledge_variable_mask,
    'Parental-monitoring reason'] = 'The item was used to compare respondent perspectives but was not retained separately because it measures the same construct as the young-person item, shows limited reporter agreement and has a pronounced ceiling concentration'
assert parental_monitoring_variable_decisions['Parental-monitoring decision'].notna().all()
assert parental_monitoring_variable_decisions['Parental-monitoring decision'].eq('Construction input').sum() == 2
assert parental_monitoring_variable_decisions['Parental-monitoring decision'].eq('Review support').sum() == 1
parental_monitoring_representation_decisions = pd.DataFrame([{'Representation': 'Young-person parental-knowledge report',
    'Decision': 'Retain', 'Reason': 'The young-person report represents experienced parental monitoring and provides greater response variation than the main-parent report'}, {'Representation': 'Main-parent parental-knowledge report',
    'Decision': 'Do not retain separately', 'Reason': 'It measures the same construct as the young-person report, shows limited agreement with it and is strongly concentrated in the always category'}, {'Representation': 'Combined reporter score',
    'Decision': 'Do not construct', 'Reason': 'The two reporters are not interchangeable, and averaging their responses would conceal meaningful differences in perspective'}, {'Representation': 'Never and rarely as separate categories',
    'Decision': 'Do not retain', 'Reason': 'The two categories contain only 66 and 169 participants respectively and both indicate low parental knowledge'}, {'Representation': 'Five-category parental-knowledge measure',
    'Decision': 'Retain', 'Reason': 'The measure distinguishes low, intermediate and high parental knowledge while preserving not going out as a separate behavioural context'}, {'Representation': 'School-night curfew measure',
    'Decision': 'Retain', 'Reason': 'Curfew setting captures a distinct form of parental control and has only moderate overlap with parental knowledge of whereabouts'}, {'Representation': 'Treat not going out as highest monitoring',
    'Decision': 'Do not construct', 'Reason': "Not going out may reflect restriction, preference or opportunity and is not equivalent to parents always knowing the young person's whereabouts"}, {'Representation': 'Combined monitoring composite',
    'Decision': 'Do not construct', 'Reason': 'Knowledge of whereabouts and curfew setting are related but conceptually distinct monitoring practices'}])
parental_monitoring_highest_overlap = parental_monitoring_retained_overlap_summary.groupby('Candidate predictor',
    as_index=False)["Cramer's V"].max()
parental_monitoring_predictor_summary = pd.DataFrame([{'Predictor': 'parental_knowledge_of_evening_whereabouts_pretransition',
    'Non-missing': int(parental_knowledge_of_evening_whereabouts_pretransition.notna().sum()), 'Missing': int(parental_knowledge_of_evening_whereabouts_pretransition.isna().sum()), 'Missing percentage': round(parental_knowledge_of_evening_whereabouts_pretransition.isna().mean() * 100,
    2), 'Categories': int(parental_knowledge_of_evening_whereabouts_pretransition.nunique()), "Highest overlap Cramer's V": float(parental_monitoring_highest_overlap.loc[parental_monitoring_highest_overlap['Candidate predictor'].eq('Parental knowledge of evening whereabouts'),
    "Cramer's V"].iloc[0])}, {'Predictor': 'school_night_curfew_pretransition',
    'Non-missing': int(school_night_curfew_pretransition.notna().sum()), 'Missing': int(school_night_curfew_pretransition.isna().sum()), 'Missing percentage': round(school_night_curfew_pretransition.isna().mean() * 100,
    2), 'Categories': int(school_night_curfew_pretransition.nunique()), "Highest overlap Cramer's V": float(parental_monitoring_highest_overlap.loc[parental_monitoring_highest_overlap['Candidate predictor'].eq('School-night curfew retained'),
    "Cramer's V"].iloc[0])}])
parental_knowledge_labels = {0: 'Never or rarely', 1: 'Sometimes', 2: 'Usually', 3: 'Always',
    4: 'Does not go out in the evening'}
school_night_curfew_labels = {0: 'Never', 1: 'Sometimes', 2: 'Often', 3: 'Not allowed out or does not go out'}
parental_monitoring_distribution_rows = []
for predictor, values, labels in [('Parental knowledge of evening whereabouts',
    parental_knowledge_of_evening_whereabouts_pretransition, parental_knowledge_labels), ('School-night curfew',
    school_night_curfew_pretransition, school_night_curfew_labels)]:
    observed_count = int(values.notna().sum())
    for code, category in labels.items():
        participants = int(values.eq(code).sum())
        parental_monitoring_distribution_rows.append({'Predictor': predictor, 'Code': code, 'Category': category,
            'Participants': participants, 'Percentage among observed': round(participants / observed_count * 100, 2)})
parental_monitoring_retained_distribution = pd.DataFrame(parental_monitoring_distribution_rows)
parental_monitoring_decision_display = parental_monitoring_variable_decisions.sort_values(['Source order',
    'Variable position'])[['Wave', 'Source type', 'Source file', 'Variable', 'Variable label', 'Timing status',
    'Parental-monitoring decision', 'Parental-monitoring role', 'Parental-monitoring reason']].reset_index(drop=True)
print('Parental-monitoring predictors retained: 2')
print('Retained predictor summary:')
display_limited(parental_monitoring_predictor_summary)
print('Retained category distributions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_monitoring_retained_distribution)
print('Association between retained predictors:')
display_limited(parental_monitoring_retained_pair_summary)
print('Associations with retained behaviour and engagement measures:')
display_limited(parental_monitoring_retained_overlap_summary)
print('Representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_monitoring_representation_decisions)
print('Variable decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_monitoring_decision_display)
print('Files written in this cell: 0')

Parental-monitoring predictors retained: 2
Retained predictor summary:


,Predictor,Non-missing,Missing,Missing percentage,Categories,Highest overlap Cramer's V
0,parental_knowledge_of_evening_whereabouts_pret...,9272,495,5.07,5,0.200
1,school_night_curfew_pretransition,9102,665,6.81,4,0.115


Retained category distributions:


,Predictor,Code,Category,Participants,Percentage among observed
0,Parental knowledge of evening whereabouts,0,Never or rarely,235,2.53
1,Parental knowledge of evening whereabouts,1,Sometimes,561,6.05
2,Parental knowledge of evening whereabouts,2,Usually,2185,23.57
3,Parental knowledge of evening whereabouts,3,Always,5744,61.95
4,Parental knowledge of evening whereabouts,4,Does not go out in the evening,547,5.90


Association between retained predictors:


...

Representation decisions:


,Representation,Decision,Reason
0,Young-person parental-knowledge report,Retain,The young-person report represents experienced parental monitoring and provides greater response variation than the main-parent report
1,Main-parent parental-knowledge report,Do not retain separately,"It measures the same construct as the young-person report, shows limited agreement with it and is strongly concentrated in the always category"
2,Combined reporter score,Do not construct,"The two reporters are not interchangeable, and averaging their responses would conceal meaningful differences in perspective"
3,Never and rarely as separate categories,Do not retain,The two categories contain only 66 and 169 participants respectively and both indicate low parental knowledge
4,Five-category parental-knowledge measure,Retain,"The measure distinguishes low, intermediate and high parental knowledge while preserving not going out as a separate behavioural context"


Variable decisions:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Parental-monitoring decision,Parental-monitoring role,Parental-monitoring reason
0,Wave 1,Young person,wave_one_lsype_young_person_2020,W1gowhereYP,YP: How often parents know where going when out in evening,Pre-transition source,Construction input,Young-person report of parental knowledge of evening whereabouts,"The item provides the young person's account of how consistently parents know their evening whereabouts. Never and rarely are consolidated because both categories are small, while not going out remains a separate category"
1,Wave 1,Young person,wave_one_lsype_young_person_2020,W1limitsnYP,YP: Whether parents ever set curfew on school nights,Pre-transition source,Construction input,School-night curfew,The item measures the frequency of school-night curfew setting and represents a form of parental control distinct from knowledge of evening whereabouts
2,Wave 1,Family background,wave_one_lsype_family_background_2020,W1paroutMP,MP: How often know where YP is when goes out in evening,Pre-transition source,Review support,Main-parent comparison report of parental knowledge,"The item was used to compare respondent perspectives but was not retained separately because it measures the same construct as the young-person item, shows limited reporter agreement and has a pronounced ceiling concentration"


Files written in this cell: 0


In [353]:
# 34: Parental school-involvement track-name inspection

import pandas as pd
domain_9_track_name_table = domain_9_screened_candidates['Domain 9 review track'].astype('string').value_counts(dropna=False).rename('Variables').rename_axis('Domain 9 review track').reset_index()
school_involvement_track_matches = domain_9_track_name_table.loc[domain_9_track_name_table['Domain 9 review track'].astype('string').str.contains('school|involv|parent.?evening|meeting',
    case=False, na=False, regex=True)].reset_index(drop=True)
school_involvement_variable_matches = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 review track'].astype('string').str.contains('school|involv|parent.?evening|meeting',
    case=False, na=False, regex=True)].sort_values(['Domain 9 review track', 'Source order',
    'Variable position'])[['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status', 'Domain 9 screening status', 'Domain 9 review track']].reset_index(drop=True)
print('Possible school-involvement review tracks:')
display_limited(school_involvement_track_matches)
print('Variables in possible school-involvement tracks:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(school_involvement_variable_matches)
print('Files written in this cell: 0')

Possible school-involvement review tracks:


,Domain 9 review track,Variables
0,Parental school involvement,11
1,Reactive school–parent contact,2


Variables in possible school-involvement tracks:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Domain 9 screening status,Domain 9 review track
0,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,18,W1pareveMP,MP: Whether self or partner have been to any parents' evenings or similar events,Pre-transition source,Core review,Parental school involvement
1,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,19,W1tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,Pre-transition source,Core review,Parental school involvement
2,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,22,W1repred1MP,MP: Whether MP ever talks about YP's school reports with them,Pre-transition source,Core review,Parental school involvement
3,Wave 2,Parental attitudes,wave_two_lsype_parental_attitudes_file_16_06_08,9,W2pareveMP,MP: Whether self or partner have been to any parents' evenings or similar events,Pre-transition source,Core review,Parental school involvement
4,Wave 2,Parental attitudes,wave_two_lsype_parental_attitudes_file_16_06_08,10,W2tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,Pre-transition source,Core review,Parental school involvement


Files written in this cell: 0


In [354]:
# 35: Parental school-involvement response and structure review

import pandas as pd
parental_school_involvement_inventory = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 review track'].eq('Parental school involvement')].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(parental_school_involvement_inventory) == 11
assert set(parental_school_involvement_inventory['Variable']) == {'W1pareveMP', 'W1tmeetfMP', 'W1repred1MP',
    'W2pareveMP', 'W2tmeetfMP', 'W2vocs2MP', 'W3pareveMP', 'W3tmeetfMP', 'W3tspeak2MP', 'W3tstayMP', 'W3tappMP'}
parental_school_involvement_raw_tables = {}
parental_school_involvement_labelled_tables = {}
for source_file, source_inventory in parental_school_involvement_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_data = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    for data in [raw_data, labelled_data]:
        data['NSID'] = standardise_nsid(data['NSID'])
        assert data['NSID'].is_unique
    raw_data = raw_data.set_index('NSID').reindex(route_index)
    labelled_data = labelled_data.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        raw_data[variable] = pd.to_numeric(raw_data[variable], errors='coerce')
    parental_school_involvement_raw_tables[source_file] = raw_data
    parental_school_involvement_labelled_tables[source_file] = labelled_data

def get_school_involvement_series(variable, labelled=False):
    """Return one school-involvement variable."""
    variable_row = parental_school_involvement_inventory.loc[parental_school_involvement_inventory['Variable'].eq(variable)].iloc[0]
    source_file = variable_row['Source file']
    source_tables = parental_school_involvement_labelled_tables if labelled else parental_school_involvement_raw_tables
    return source_tables[source_file][variable]

def school_involvement_timing_mask(wave):
    """Return the permitted pre-transition timing mask."""
    if wave == 'Wave 3':
        return wave_3_pretransition_interview_mask.reindex(route_index).fillna(False).astype(bool)
    return pd.Series(True, index=route_index, dtype=bool)
parental_school_involvement_coverage_rows = []
parental_school_involvement_code_rows = []
parental_school_involvement_observed = {}
for _, inventory_row in parental_school_involvement_inventory.iterrows():
    wave = inventory_row['Wave']
    variable = inventory_row['Variable']
    raw_original = get_school_involvement_series(variable)
    labelled_values = get_school_involvement_series(variable, labelled=True).astype('string')
    timing_mask = school_involvement_timing_mask(wave)
    raw_values = raw_original.where(timing_mask)
    observed_mask = raw_values.ge(0).fillna(False).astype(bool)
    special_mask = raw_values.lt(0).fillna(False).astype(bool)
    parental_school_involvement_observed[variable] = raw_values.where(observed_mask)
    parental_school_involvement_coverage_rows.append({'Wave': wave, 'Variable': variable,
        'Variable label': inventory_row['Variable label'], 'Participants within permitted timing': int(timing_mask.sum()), 'Outside permitted timing': int((~timing_mask).sum()), 'Observed responses': int(observed_mask.sum()), 'Observed percentage of full sample': round(observed_mask.mean() * 100,
        2), 'Special-code responses': int(special_mask.sum()), 'Unavailable or outside timing': int(raw_values.isna().sum()), 'Observed categories': int(raw_values.loc[observed_mask].nunique())})
    for raw_code, participants in raw_values.value_counts(dropna=False).items():
        if pd.isna(raw_code):
            value_label = 'No source record or outside permitted timing'
            response_type = 'Unavailable'
            sort_value = 999999
        else:
            response_mask = raw_values.eq(raw_code)
            matching_labels = labelled_values.loc[response_mask].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            response_type = 'Observed response' if raw_code >= 0 else 'Special code'
            sort_value = float(raw_code)
        parental_school_involvement_code_rows.append({'Wave': wave, 'Variable': variable, 'Raw code': raw_code,
            'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
parental_school_involvement_coverage_summary = pd.DataFrame(parental_school_involvement_coverage_rows).sort_values(['Wave',
    'Variable']).reset_index(drop=True)
parental_school_involvement_code_distribution = pd.DataFrame(parental_school_involvement_code_rows).sort_values(['Wave',
    'Variable', 'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
repeated_school_involvement_constructs = {"Parents' evening attendance": {'Wave 1': 'W1pareveMP',
    'Wave 2': 'W2pareveMP', 'Wave 3': 'W3pareveMP'}, 'Specially arranged teacher meeting': {'Wave 1': 'W1tmeetfMP',
    'Wave 2': 'W2tmeetfMP', 'Wave 3': 'W3tmeetfMP'}}
school_involvement_wave_pairs = [('Wave 1 and Wave 2', 'Wave 1', 'Wave 2'), ('Wave 2 and eligible Wave 3', 'Wave 2',
    'Wave 3'), ('Wave 1 and eligible Wave 3', 'Wave 1', 'Wave 3')]
parental_school_involvement_wave_agreement_rows = []
for construct, variable_map in repeated_school_involvement_constructs.items():
    construct_data = pd.DataFrame({wave: parental_school_involvement_observed[variable] for wave,
        variable in variable_map.items()}, index=route_index)
    for comparison, first_wave, second_wave in school_involvement_wave_pairs:
        pair_data = construct_data[[first_wave, second_wave]].dropna()
        assert len(pair_data) > 0
        parental_school_involvement_wave_agreement_rows.append({'Construct': construct, 'Comparison': comparison,
            'Complete comparisons': len(pair_data), 'Exact agreement percentage': round(pair_data[first_wave].eq(pair_data[second_wave]).mean() * 100,
            2), "Cramer's V": round(categorical_cramers_v(pair_data[first_wave], pair_data[second_wave]), 3)})
parental_school_involvement_wave_agreement = pd.DataFrame(parental_school_involvement_wave_agreement_rows)
parental_school_involvement_within_wave_rows = []
for wave, evening_variable, meeting_variable in [('Wave 1', 'W1pareveMP', 'W1tmeetfMP'), ('Wave 2', 'W2pareveMP',
    'W2tmeetfMP'), ('Wave 3', 'W3pareveMP', 'W3tmeetfMP')]:
    pair_data = pd.DataFrame({"Parents' evening": parental_school_involvement_observed[evening_variable],
        'Special meeting': parental_school_involvement_observed[meeting_variable]}, index=route_index).dropna()
    parental_school_involvement_within_wave_rows.append({'Wave': wave, 'Complete comparisons': len(pair_data),
        "Cramer's V": round(categorical_cramers_v(pair_data["Parents' evening"], pair_data['Special meeting']), 3)})
parental_school_involvement_within_wave_summary = pd.DataFrame(parental_school_involvement_within_wave_rows)
specific_school_discussion_variables = {'W1repred1MP': ['W1pareveMP', 'W1tmeetfMP'], 'W2vocs2MP': ['W2pareveMP',
    'W2tmeetfMP'], 'W3tspeak2MP': ['W3pareveMP', 'W3tmeetfMP'], 'W3tstayMP': ['W3pareveMP',
    'W3tmeetfMP'], 'W3tappMP': ['W3pareveMP', 'W3tmeetfMP']}
parental_school_involvement_routing_rows = []
for target_variable, routing_variables in specific_school_discussion_variables.items():
    target_row = parental_school_involvement_inventory.loc[parental_school_involvement_inventory['Variable'].eq(target_variable)].iloc[0]
    wave = target_row['Wave']
    timing_mask = school_involvement_timing_mask(wave)
    target_raw = get_school_involvement_series(target_variable).where(timing_mask)
    target_observed = target_raw.ge(0).fillna(False).astype(bool)
    target_structural = target_raw.eq(-91.0).fillna(False).astype(bool)
    target_other_special = target_raw.lt(0).fillna(False).astype(bool) & ~target_structural
    for routing_variable in routing_variables:
        routing_raw = get_school_involvement_series(routing_variable).where(timing_mask)
        routing_labelled = get_school_involvement_series(routing_variable, labelled=True).astype('string')
        for raw_code, participants in routing_raw.value_counts(dropna=False).items():
            if pd.isna(raw_code):
                routing_mask = routing_raw.isna()
                routing_response = 'No source record or outside permitted timing'
                sort_value = 999999
            else:
                routing_mask = routing_raw.eq(raw_code)
                matching_labels = routing_labelled.loc[routing_mask].dropna().drop_duplicates().tolist()
                routing_response = matching_labels[0] if matching_labels else str(raw_code)
                sort_value = float(raw_code)
            participant_count = int(participants)
            parental_school_involvement_routing_rows.append({'Wave': wave, 'Target variable': target_variable,
                'Target label': target_row['Variable label'], 'Routing variable': routing_variable, 'Routing code': raw_code, 'Routing response': routing_response, 'Participants': participant_count, 'Target observed': int((routing_mask & target_observed).sum()), 'Target structurally not applicable': int((routing_mask & target_structural).sum()), 'Target other special code': int((routing_mask & target_other_special).sum()), 'Sort value': sort_value})
parental_school_involvement_routing_summary = pd.DataFrame(parental_school_involvement_routing_rows).sort_values(['Wave',
    'Target variable', 'Routing variable', 'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
wave_3_transition_discussion_variables = ['W3tspeak2MP', 'W3tstayMP', 'W3tappMP']
wave_3_transition_discussion_pair_rows = []
for first_position in range(len(wave_3_transition_discussion_variables)):
    for second_position in range(first_position + 1, len(wave_3_transition_discussion_variables)):
        first_variable = wave_3_transition_discussion_variables[first_position]
        second_variable = wave_3_transition_discussion_variables[second_position]
        pair_data = pd.DataFrame({first_variable: parental_school_involvement_observed[first_variable],
            second_variable: parental_school_involvement_observed[second_variable]}, index=route_index).dropna()
        wave_3_transition_discussion_pair_rows.append({'First variable': first_variable,
            'Second variable': second_variable, 'Complete comparisons': len(pair_data), 'Exact agreement percentage': round(pair_data[first_variable].eq(pair_data[second_variable]).mean() * 100,
            2), "Cramer's V": round(categorical_cramers_v(pair_data[first_variable], pair_data[second_variable]), 3)})
wave_3_transition_discussion_pair_summary = pd.DataFrame(wave_3_transition_discussion_pair_rows)
print(f'Parental school-involvement variables reviewed: {len(parental_school_involvement_inventory)}')
print('Variable inventory:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_school_involvement_inventory[['Wave', 'Source type', 'Source file', 'Variable',
        'Variable label', 'Timing status']])
print('Coverage summary:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_school_involvement_coverage_summary)
print('Response-code distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_school_involvement_code_distribution)
print('Cross-wave agreement for repeated measures:')
display_limited(parental_school_involvement_wave_agreement)
print('Within-wave association between general involvement measures:')
display_limited(parental_school_involvement_within_wave_summary)
print('Routing review for content-specific discussion items:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_school_involvement_routing_summary)
print('Wave 3 transition-discussion item comparison:')
display_limited(wave_3_transition_discussion_pair_summary)
print('Files written in this cell: 0')

Parental school-involvement variables reviewed: 11
Variable inventory:


,Wave,Source type,Source file,Variable,Variable label,Timing status
0,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1pareveMP,MP: Whether self or partner have been to any parents' evenings or similar events,Pre-transition source
1,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,Pre-transition source
2,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1repred1MP,MP: Whether MP ever talks about YP's school reports with them,Pre-transition source
3,Wave 2,Parental attitudes,wave_two_lsype_parental_attitudes_file_16_06_08,W2pareveMP,MP: Whether self or partner have been to any parents' evenings or similar events,Pre-transition source
4,Wave 2,Parental attitudes,wave_two_lsype_parental_attitudes_file_16_06_08,W2tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,Pre-transition source


Coverage summary:


,Wave,Variable,Variable label,Participants within permitted timing,Outside permitted timing,Observed responses,Observed percentage of full sample,Special-code responses,Unavailable or outside timing,Observed categories
0,Wave 1,W1pareveMP,MP: Whether self or partner have been to any parents' evenings or similar events,9767,0,9403,96.27,121,243,4
1,Wave 1,W1repred1MP,MP: Whether MP ever talks about YP's school reports with them,9767,0,9416,96.41,108,243,3
2,Wave 1,W1tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,9767,0,9410,96.34,114,243,2
3,Wave 2,W2pareveMP,MP: Whether self or partner have been to any parents' evenings or similar events,9767,0,9403,96.27,118,246,6
4,Wave 2,W2tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,9767,0,9427,96.52,94,246,2


Response-code distributions:


...

Routing review for content-specific discussion items:


,Wave,Target variable,Target label,Routing variable,Routing code,Routing response,Participants,Target observed,Target structurally not applicable,Target other special code
0,Wave 1,W1repred1MP,MP: Whether MP ever talks about YP's school reports with them,W1pareveMP,-99.0,MP not interviewed,103,0,0,103
1,Wave 1,W1repred1MP,MP: Whether MP ever talks about YP's school reports with them,W1pareveMP,-1.0,Don't know,18,18,0,0
2,Wave 1,W1repred1MP,MP: Whether MP ever talks about YP's school reports with them,W1pareveMP,1.0,Yes - respondent and partner/ husband/ wife have gone,4547,4546,0,1
3,Wave 1,W1repred1MP,MP: Whether MP ever talks about YP's school reports with them,W1pareveMP,2.0,Yes - respondent went alone,3738,3735,0,3
4,Wave 1,W1repred1MP,MP: Whether MP ever talks about YP's school reports with them,W1pareveMP,3.0,Yes - respondent's partner/ husband/ wife went alone,386,386,0,0


Wave 3 transition-discussion item comparison:


,First variable,Second variable,Complete comparisons,Exact agreement percentage,Cramer's V
0,W3tspeak2MP,W3tstayMP,4239,77.49,NaN
1,W3tspeak2MP,W3tappMP,4229,17.73,NaN
2,W3tstayMP,W3tappMP,4214,33.55,0.04


Files written in this cell: 0


In [355]:
# 36: Parental school-involvement representation review

import pandas as pd

def construct_binary_school_involvement_measure(raw_values, yes_codes, no_codes, name):
    """Construct a harmonised binary school-involvement measure."""
    result = pd.Series(pd.NA, index=route_index, dtype='Int64', name=name)
    yes_mask = raw_values.isin(yes_codes).fillna(False).astype(bool)
    no_mask = raw_values.isin(no_codes).fillna(False).astype(bool)
    assert not (yes_mask & no_mask).any()
    result.loc[yes_mask] = 1
    result.loc[no_mask] = 0
    return result
parents_evening_attendance_by_wave = pd.DataFrame({'Wave 1': construct_binary_school_involvement_measure(parental_school_involvement_observed['W1pareveMP'],
    yes_codes=[1.0, 2.0,
    3.0], no_codes=[4.0], name='Wave 1'), 'Wave 2': construct_binary_school_involvement_measure(parental_school_involvement_observed['W2pareveMP'],
    yes_codes=[1.0, 2.0, 3.0, 4.0,
    5.0], no_codes=[6.0], name='Wave 2'), 'Wave 3': construct_binary_school_involvement_measure(parental_school_involvement_observed['W3pareveMP'],
    yes_codes=[1.0, 2.0, 3.0, 4.0, 5.0], no_codes=[6.0], name='Wave 3')}, index=route_index)
special_teacher_meeting_by_wave = pd.DataFrame({'Wave 1': construct_binary_school_involvement_measure(parental_school_involvement_observed['W1tmeetfMP'],
    yes_codes=[1.0], no_codes=[2.0], name='Wave 1'), 'Wave 2': construct_binary_school_involvement_measure(parental_school_involvement_observed['W2tmeetfMP'],
    yes_codes=[1.0], no_codes=[2.0], name='Wave 2'), 'Wave 3': construct_binary_school_involvement_measure(parental_school_involvement_observed['W3tmeetfMP'],
    yes_codes=[1.0], no_codes=[2.0], name='Wave 3')}, index=route_index)
for wave_data in [parents_evening_attendance_by_wave, special_teacher_meeting_by_wave]:
    observed_values = pd.concat([wave_data[column] for column in wave_data.columns],
        ignore_index=True).dropna().astype(int).unique()
    assert set(observed_values).issubset({0, 1})
general_school_involvement_agreement_rows = []
for construct, wave_data in [("Parents' evening attendance", parents_evening_attendance_by_wave),
    ('Specially arranged teacher meeting', special_teacher_meeting_by_wave)]:
    for comparison, first_wave, second_wave in school_involvement_wave_pairs:
        pair_data = wave_data[[first_wave, second_wave]].dropna()
        assert len(pair_data) > 0
        general_school_involvement_agreement_rows.append({'Construct': construct, 'Comparison': comparison,
            'Complete comparisons': len(pair_data), 'Exact agreement percentage': round(pair_data[first_wave].eq(pair_data[second_wave]).mean() * 100,
            2), "Cramer's V": round(categorical_cramers_v(pair_data[first_wave], pair_data[second_wave]), 3)})
general_school_involvement_agreement = pd.DataFrame(general_school_involvement_agreement_rows)

def select_latest_school_involvement_measure(wave_data, name):
    """Select the latest available permitted response."""
    candidate = pd.Series(pd.NA, index=route_index, dtype='Int64', name=name)
    source = pd.Series(pd.NA, index=route_index, dtype='string')
    for wave in ['Wave 3', 'Wave 2', 'Wave 1']:
        selection_mask = candidate.isna() & wave_data[wave].notna()
        candidate.loc[selection_mask] = wave_data.loc[selection_mask, wave].astype('Int64')
        source.loc[selection_mask] = wave
    source = source.fillna('Unavailable')
    return (candidate, source)
parents_evening_attendance_pretransition_candidate, parents_evening_attendance_source = select_latest_school_involvement_measure(parents_evening_attendance_by_wave,
    'parents_evening_attendance_pretransition')
special_teacher_meeting_pretransition_candidate, special_teacher_meeting_source = select_latest_school_involvement_measure(special_teacher_meeting_by_wave,
    'special_teacher_meeting_pretransition')
parental_school_involvement_breadth_pretransition_candidate = (parents_evening_attendance_pretransition_candidate + special_teacher_meeting_pretransition_candidate).where(parents_evening_attendance_pretransition_candidate.notna() & special_teacher_meeting_pretransition_candidate.notna()).astype('Int64').rename('parental_school_involvement_breadth_pretransition')
assert set(parental_school_involvement_breadth_pretransition_candidate.dropna().astype(int).unique()).issubset({0, 1,
    2})
wave_3_school_involvement_timing_mask = school_involvement_timing_mask('Wave 3')
wave_3_post16_discussion_raw = get_school_involvement_series('W3tspeak2MP').where(wave_3_school_involvement_timing_mask)
wave_3_staying_on_discussion_raw = get_school_involvement_series('W3tstayMP').where(wave_3_school_involvement_timing_mask)
wave_3_apprenticeship_discussion_raw = get_school_involvement_series('W3tappMP').where(wave_3_school_involvement_timing_mask)
post16_discussion_yes_mask = wave_3_post16_discussion_raw.eq(1.0).fillna(False).astype(bool)
post16_discussion_no_mask = wave_3_post16_discussion_raw.eq(2.0).fillna(False).astype(bool)
post16_discussion_uncertain_mask = wave_3_post16_discussion_raw.eq(-1.0).fillna(False).astype(bool)
expected_content_item_structural_mask = post16_discussion_no_mask | post16_discussion_uncertain_mask
assert wave_3_staying_on_discussion_raw.eq(-91.0).fillna(False).astype(bool).equals(expected_content_item_structural_mask)
assert wave_3_apprenticeship_discussion_raw.eq(-91.0).fillna(False).astype(bool).equals(expected_content_item_structural_mask)
assert wave_3_staying_on_discussion_raw.loc[post16_discussion_yes_mask].isin([-1.0, 1.0, 2.0]).all()
assert wave_3_apprenticeship_discussion_raw.loc[post16_discussion_yes_mask].isin([-1.0, 1.0, 2.0]).all()
parent_teacher_post16_discussion_pretransition_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='parent_teacher_post16_discussion_pretransition')
parent_teacher_post16_discussion_pretransition_candidate.loc[post16_discussion_no_mask] = 0
parent_teacher_post16_discussion_pretransition_candidate.loc[post16_discussion_yes_mask] = 1
parent_teacher_post16_discussion_profile_pretransition_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='parent_teacher_post16_discussion_profile_pretransition')
staying_on_yes_mask = wave_3_staying_on_discussion_raw.eq(1.0).fillna(False).astype(bool)
staying_on_no_mask = wave_3_staying_on_discussion_raw.eq(2.0).fillna(False).astype(bool)
apprenticeship_yes_mask = wave_3_apprenticeship_discussion_raw.eq(1.0).fillna(False).astype(bool)
apprenticeship_no_mask = wave_3_apprenticeship_discussion_raw.eq(2.0).fillna(False).astype(bool)
parent_teacher_post16_discussion_profile_pretransition_candidate.loc[post16_discussion_no_mask] = 0
parent_teacher_post16_discussion_profile_pretransition_candidate.loc[post16_discussion_yes_mask & staying_on_yes_mask & apprenticeship_no_mask] = 1
parent_teacher_post16_discussion_profile_pretransition_candidate.loc[post16_discussion_yes_mask & staying_on_no_mask & apprenticeship_yes_mask] = 2
parent_teacher_post16_discussion_profile_pretransition_candidate.loc[post16_discussion_yes_mask & staying_on_yes_mask & apprenticeship_yes_mask] = 3
parent_teacher_post16_discussion_profile_pretransition_candidate.loc[post16_discussion_yes_mask & staying_on_no_mask & apprenticeship_no_mask] = 4
assert set(parent_teacher_post16_discussion_profile_pretransition_candidate.dropna().astype(int).unique()).issubset({0,
    1, 2, 3, 4})
both_content_items_observed_mask = post16_discussion_yes_mask & (staying_on_yes_mask | staying_on_no_mask) & (apprenticeship_yes_mask | apprenticeship_no_mask)
transition_discussion_routing_summary = pd.DataFrame([{'Routing status': 'General post-16 discussion: yes',
    'Participants': int(post16_discussion_yes_mask.sum())}, {'Routing status': 'General post-16 discussion: no',
    'Participants': int(post16_discussion_no_mask.sum())}, {'Routing status': "General post-16 discussion: don't know",
    'Participants': int(post16_discussion_uncertain_mask.sum())}, {'Routing status': 'Both content-specific items observed',
    'Participants': int(both_content_items_observed_mask.sum())}, {'Routing status': 'Content profile unavailable after general discussion was reported',
    'Participants': int((post16_discussion_yes_mask & ~both_content_items_observed_mask).sum())}])
candidate_series = {'parents_evening_attendance_pretransition': parents_evening_attendance_pretransition_candidate,
    'special_teacher_meeting_pretransition': special_teacher_meeting_pretransition_candidate, 'parental_school_involvement_breadth_pretransition': parental_school_involvement_breadth_pretransition_candidate, 'parent_teacher_post16_discussion_pretransition': parent_teacher_post16_discussion_pretransition_candidate, 'parent_teacher_post16_discussion_profile_pretransition': parent_teacher_post16_discussion_profile_pretransition_candidate}
parental_school_involvement_candidate_rows = []
for candidate_name, candidate_values in candidate_series.items():
    parental_school_involvement_candidate_rows.append({'Candidate predictor': candidate_name,
        'Non-missing': int(candidate_values.notna().sum()), 'Missing': int(candidate_values.isna().sum()), 'Missing percentage': round(candidate_values.isna().mean() * 100,
        2), 'Categories': int(candidate_values.nunique())})
parental_school_involvement_candidate_summary = pd.DataFrame(parental_school_involvement_candidate_rows)
general_school_involvement_source_summary = pd.concat([parents_evening_attendance_source.value_counts().rename('Participants').rename_axis('Construction source').reset_index().assign(Candidate="Parents' evening attendance"),
    special_teacher_meeting_source.value_counts().rename('Participants').rename_axis('Construction source').reset_index().assign(Candidate='Specially arranged teacher meeting')], ignore_index=True)
general_school_involvement_source_summary['Percentage of full sample'] = (general_school_involvement_source_summary['Participants'] / len(route_index) * 100).round(2)
school_involvement_distribution_specs = [("Parents' evening attendance",
    parents_evening_attendance_pretransition_candidate, {0: 'No attendance',
    1: 'At least one parent attended'}), ('Specially arranged teacher meeting',
    special_teacher_meeting_pretransition_candidate, {0: 'No specially arranged meeting',
    1: 'Specially arranged meeting'}), ('General school-involvement breadth',
    parental_school_involvement_breadth_pretransition_candidate, {0: 'Neither activity', 1: 'One activity',
    2: 'Both activities'}), ('Parent–teacher post-16 discussion',
    parent_teacher_post16_discussion_pretransition_candidate, {0: 'No discussion',
    1: 'Discussion'}), ('Parent–teacher post-16 discussion profile',
    parent_teacher_post16_discussion_profile_pretransition_candidate, {0: 'No post-16 discussion',
    1: 'Staying-on discussion only', 2: 'Apprenticeship or training discussion only', 3: 'Both specified topics', 4: 'Other or unspecified post-16 discussion'})]
parental_school_involvement_distribution_rows = []
for candidate, values, labels in school_involvement_distribution_specs:
    observed_count = int(values.notna().sum())
    for code, category in labels.items():
        participants = int(values.eq(code).sum())
        parental_school_involvement_distribution_rows.append({'Candidate predictor': candidate, 'Code': code,
            'Category': category, 'Participants': participants, 'Percentage among observed': round(participants / observed_count * 100,
            2)})
parental_school_involvement_candidate_distributions = pd.DataFrame(parental_school_involvement_distribution_rows)
print('Harmonised cross-wave agreement:')
display_limited(general_school_involvement_agreement)
print('Latest permitted construction sources:')
display_limited(general_school_involvement_source_summary[['Candidate', 'Construction source', 'Participants',
    'Percentage of full sample']])
print('Wave 3 transition-discussion routing:')
display_limited(transition_discussion_routing_summary)
print('Candidate predictor summary:')
display_limited(parental_school_involvement_candidate_summary)
print('Candidate distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_school_involvement_candidate_distributions)
print('Files written in this cell: 0')

Harmonised cross-wave agreement:


,Construct,Comparison,Complete comparisons,Exact agreement percentage,Cramer's V
0,Parents' evening attendance,Wave 1 and Wave 2,9299,85.42,0.334
1,Parents' evening attendance,Wave 2 and eligible Wave 3,9288,84.68,0.383
2,Parents' evening attendance,Wave 1 and eligible Wave 3,9282,87.54,0.347
3,Specially arranged teacher meeting,Wave 1 and Wave 2,9328,72.21,0.280
4,Specially arranged teacher meeting,Wave 2 and eligible Wave 3,9323,74.57,0.296


Latest permitted construction sources:


,Candidate,Construction source,Participants,Percentage of full sample
0,Parents' evening attendance,Wave 3,9388,96.12
1,Parents' evening attendance,Unavailable,252,2.58
2,Parents' evening attendance,Wave 2,115,1.18
3,Parents' evening attendance,Wave 1,12,0.12
4,Specially arranged teacher meeting,Wave 3,9402,96.26


Wave 3 transition-discussion routing:


...

Candidate predictor summary:


,Candidate predictor,Non-missing,Missing,Missing percentage,Categories
0,parents_evening_attendance_pretransition,9515,252,2.58,2
1,special_teacher_meeting_pretransition,9516,251,2.57,2
2,parental_school_involvement_breadth_pretransition,9515,252,2.58,3
3,parent_teacher_post16_discussion_pretransition,9380,387,3.96,2
4,parent_teacher_post16_discussion_profile_pretr...,9338,429,4.39,5


Candidate distributions:


,Candidate predictor,Code,Category,Participants,Percentage among observed
0,Parents' evening attendance,0,No attendance,1255,13.19
1,Parents' evening attendance,1,At least one parent attended,8260,86.81
2,Specially arranged teacher meeting,0,No specially arranged meeting,7199,75.65
3,Specially arranged teacher meeting,1,Specially arranged meeting,2317,24.35
4,General school-involvement breadth,0,Neither activity,935,9.83


Files written in this cell: 0


In [356]:
# 37: Parental school-involvement overlap and category review

import pandas as pd
post16_discussion_profile_four_category_recode = {0: 0, 1: 1, 2: 2, 3: 2, 4: 3}
parent_teacher_post16_discussion_profile_four_category_candidate = parent_teacher_post16_discussion_profile_pretransition_candidate.map(post16_discussion_profile_four_category_recode).astype('Int64').rename('parent_teacher_post16_discussion_profile_pretransition')
assert parent_teacher_post16_discussion_profile_four_category_candidate.notna().equals(parent_teacher_post16_discussion_profile_pretransition_candidate.notna())
assert set(parent_teacher_post16_discussion_profile_four_category_candidate.dropna().astype(int).unique()).issubset({0,
    1, 2, 3})
parental_school_involvement_overlap_data = pd.DataFrame({"Parents' evening attendance": parents_evening_attendance_pretransition_candidate,
    'Specially arranged teacher meeting': special_teacher_meeting_pretransition_candidate, 'General school-involvement breadth': parental_school_involvement_breadth_pretransition_candidate, 'Binary post-16 discussion': parent_teacher_post16_discussion_pretransition_candidate, 'Five-category post-16 discussion profile': parent_teacher_post16_discussion_profile_pretransition_candidate, 'Four-category post-16 discussion profile': parent_teacher_post16_discussion_profile_four_category_candidate, 'Expected post-16 route': saved_educational_aspiration_predictors['expected_post16_route'], 'Young-person HE application likelihood': saved_educational_aspiration_predictors['higher_education_application_likelihood'], 'Parental HE expectation': parental_he_expectation_candidate, 'Parental educational aspiration': parental_educational_aspiration_candidate, 'Parental financial-support profile': parental_educational_financial_support_profile_pretransition, 'Truancy status': parental_homework_school_overlap_data['truancy_status_pretransition'], 'Highest parental qualification': family_socioeconomic_overlap_data['highest_parental_qualification_code'], 'Household income band': family_socioeconomic_overlap_data['household_income_band']}, index=route_index)
assert len(parental_school_involvement_overlap_data) == 9767
parental_school_involvement_overlap_coverage_rows = []
for measure in parental_school_involvement_overlap_data.columns:
    values = parental_school_involvement_overlap_data[measure]
    parental_school_involvement_overlap_coverage_rows.append({'Measure': measure,
        'Non-missing': int(values.notna().sum()), 'Missing': int(values.isna().sum()), 'Missing percentage': round(values.isna().mean() * 100,
        2), 'Categories': int(values.nunique())})
parental_school_involvement_overlap_coverage = pd.DataFrame(parental_school_involvement_overlap_coverage_rows)
general_school_involvement_pair_rows = []
for first_measure, second_measure in [("Parents' evening attendance", 'Specially arranged teacher meeting'),
    ("Parents' evening attendance", 'General school-involvement breadth'), ('Specially arranged teacher meeting',
    'General school-involvement breadth')]:
    pair_data = parental_school_involvement_overlap_data[[first_measure, second_measure]].dropna()
    general_school_involvement_pair_rows.append({'First measure': first_measure, 'Second measure': second_measure,
        'Complete comparisons': len(pair_data), "Cramer's V": round(categorical_cramers_v(pair_data[first_measure],
        pair_data[second_measure]), 3)})
general_school_involvement_pair_summary = pd.DataFrame(general_school_involvement_pair_rows)
general_school_involvement_overlap_rows = []
for candidate_measure in ["Parents' evening attendance", 'Specially arranged teacher meeting',
    'General school-involvement breadth']:
    for comparison_measure in ['Truancy status', 'Highest parental qualification', 'Household income band',
        'Parental HE expectation', 'Parental educational aspiration', 'Parental financial-support profile']:
        pair_data = parental_school_involvement_overlap_data[[candidate_measure, comparison_measure]].dropna()
        assert len(pair_data) > 0
        general_school_involvement_overlap_rows.append({'Candidate predictor': candidate_measure,
            'Comparison measure': comparison_measure, 'Complete comparisons': len(pair_data), "Cramer's V": round(categorical_cramers_v(pair_data[candidate_measure],
            pair_data[comparison_measure]), 3)})
general_school_involvement_overlap_summary = pd.DataFrame(general_school_involvement_overlap_rows).sort_values(['Candidate predictor',
    "Cramer's V"], ascending=[True, False]).reset_index(drop=True)
post16_discussion_representation_overlap_rows = []
for candidate_measure in ['Binary post-16 discussion', 'Five-category post-16 discussion profile',
    'Four-category post-16 discussion profile']:
    for comparison_measure in ['Expected post-16 route', 'Young-person HE application likelihood',
        'Parental HE expectation', 'Parental educational aspiration', 'Parental financial-support profile', 'General school-involvement breadth']:
        pair_data = parental_school_involvement_overlap_data[[candidate_measure, comparison_measure]].dropna()
        assert len(pair_data) > 0
        post16_discussion_representation_overlap_rows.append({'Candidate predictor': candidate_measure,
            'Comparison measure': comparison_measure, 'Complete comparisons': len(pair_data), 'Candidate categories': int(pair_data[candidate_measure].nunique()), 'Comparison categories': int(pair_data[comparison_measure].nunique()), "Cramer's V": round(categorical_cramers_v(pair_data[candidate_measure],
            pair_data[comparison_measure]), 3)})
post16_discussion_representation_overlap_summary = pd.DataFrame(post16_discussion_representation_overlap_rows).sort_values(['Candidate predictor',
    "Cramer's V"], ascending=[True, False]).reset_index(drop=True)
post16_discussion_four_category_labels = {0: 'No post-16 discussion', 1: 'Staying-on discussion only',
    2: 'Any apprenticeship or training discussion', 3: 'Other or unspecified post-16 discussion'}
post16_discussion_four_category_distribution_rows = []
four_category_observed_count = int(parent_teacher_post16_discussion_profile_four_category_candidate.notna().sum())
for code, category in post16_discussion_four_category_labels.items():
    participants = int(parent_teacher_post16_discussion_profile_four_category_candidate.eq(code).sum())
    post16_discussion_four_category_distribution_rows.append({'Code': code, 'Category': category,
        'Participants': participants, 'Percentage among observed': round(participants / four_category_observed_count * 100,
        2)})
post16_discussion_four_category_distribution = pd.DataFrame(post16_discussion_four_category_distribution_rows)
post16_profile_route_data = parental_school_involvement_overlap_data[['Four-category post-16 discussion profile',
    'Expected post-16 route']].dropna()
post16_profile_route_counts = pd.crosstab(post16_profile_route_data['Four-category post-16 discussion profile'],
    post16_profile_route_data['Expected post-16 route'], margins=True, dropna=False)
post16_profile_route_row_percentages = (pd.crosstab(post16_profile_route_data['Four-category post-16 discussion profile'],
    post16_profile_route_data['Expected post-16 route'], normalize='index', dropna=False) * 100).round(2)
print('Comparison-measure coverage:')
display_limited(parental_school_involvement_overlap_coverage)
print('Associations among general involvement representations:')
display_limited(general_school_involvement_pair_summary)
print('General involvement overlap with retained measures:')
display_limited(general_school_involvement_overlap_summary)
print('Post-16 discussion representation overlap:')
display_limited(post16_discussion_representation_overlap_summary)
print('Four-category post-16 discussion distribution:')
with pd.option_context('display.max_colwidth', None):
    display_limited(post16_discussion_four_category_distribution)
print('Four-category discussion profile by expected post-16 route:')
display_limited(post16_profile_route_counts)
print('Row percentages:')
display_limited(post16_profile_route_row_percentages)
print('Files written in this cell: 0')

Comparison-measure coverage:


,Measure,Non-missing,Missing,Missing percentage,Categories
0,Parents' evening attendance,9515,252,2.58,2
1,Specially arranged teacher meeting,9516,251,2.57,2
2,General school-involvement breadth,9515,252,2.58,3
3,Binary post-16 discussion,9380,387,3.96,2
4,Five-category post-16 discussion profile,9338,429,4.39,5


Associations among general involvement representations:


,First measure,Second measure,Complete comparisons,Cramer's V
0,Parents' evening attendance,Specially arranged teacher meeting,9515,0.011
1,Parents' evening attendance,General school-involvement breadth,9515,0.849
2,Specially arranged teacher meeting,General school-involvement breadth,9515,0.909


General involvement overlap with retained measures:


...

Four-category discussion profile by expected post-16 route:


Expected post-16 route,Full-time employment,Further-education college,Other or unspecified full-time education,"Other, mixed or uncertain route",School sixth form,Sixth-form college,Work-based training or employment-training,All
Four-category post-16 discussion profile,,,,,,,,
0,158,1299,235,88,1957,1104,276,5117
1,25,502,98,20,1502,478,31,2656
2,44,220,52,23,181,78,146,744
3,27,191,42,20,340,145,43,808
All,254,2212,427,151,3980,1805,496,9325


Row percentages:


Expected post-16 route,Full-time employment,Further-education college,Other or unspecified full-time education,"Other, mixed or uncertain route",School sixth form,Sixth-form college,Work-based training or employment-training
Four-category post-16 discussion profile,,,,,,,
0,3.09,25.39,4.59,1.72,38.25,21.58,5.39
1,0.94,18.90,3.69,0.75,56.55,18.00,1.17
2,5.91,29.57,6.99,3.09,24.33,10.48,19.62
3,3.34,23.64,5.20,2.48,42.08,17.95,5.32


Files written in this cell: 0


In [357]:
# 38: Parental school-involvement decisions

import pandas as pd
parents_evening_attendance_pretransition = parents_evening_attendance_pretransition_candidate.copy().astype('Int64').rename('parents_evening_attendance_pretransition')
parent_teacher_post16_discussion_profile_pretransition = parent_teacher_post16_discussion_profile_four_category_candidate.copy().astype('Int64').rename('parent_teacher_post16_discussion_profile_pretransition')
assert set(parents_evening_attendance_pretransition.dropna().astype(int).unique()).issubset({0, 1})
assert set(parent_teacher_post16_discussion_profile_pretransition.dropna().astype(int).unique()).issubset({0, 1, 2,
    3})
parental_school_involvement_predictors = pd.DataFrame({'NSID': route_index,
    'parents_evening_attendance_pretransition': parents_evening_attendance_pretransition.to_numpy(), 'parent_teacher_post16_discussion_profile_pretransition': parent_teacher_post16_discussion_profile_pretransition.to_numpy()})
assert len(parental_school_involvement_predictors) == 9767
assert parental_school_involvement_predictors['NSID'].is_unique
parental_school_involvement_variable_decisions = parental_school_involvement_inventory.copy()
parental_school_involvement_variable_decisions['School-involvement decision'] = pd.NA
parental_school_involvement_variable_decisions['School-involvement role'] = pd.NA
parental_school_involvement_variable_decisions['School-involvement reason'] = pd.NA
parents_evening_variables = ['W1pareveMP', 'W2pareveMP', 'W3pareveMP']
special_meeting_variables = ['W1tmeetfMP', 'W2tmeetfMP', 'W3tmeetfMP']
post16_discussion_variables = ['W3tspeak2MP', 'W3tstayMP', 'W3tappMP']
parents_evening_mask = parental_school_involvement_variable_decisions['Variable'].isin(parents_evening_variables)
special_meeting_mask = parental_school_involvement_variable_decisions['Variable'].isin(special_meeting_variables)
post16_discussion_mask = parental_school_involvement_variable_decisions['Variable'].isin(post16_discussion_variables)
school_report_discussion_mask = parental_school_involvement_variable_decisions['Variable'].eq('W1repred1MP')
vocational_subject_discussion_mask = parental_school_involvement_variable_decisions['Variable'].eq('W2vocs2MP')
parental_school_involvement_variable_decisions.loc[parents_evening_mask,
    'School-involvement decision'] = 'Construction input'
parental_school_involvement_variable_decisions.loc[parents_evening_mask,
    'School-involvement role'] = "Repeated parent attendance at parents' evenings"
parental_school_involvement_variable_decisions.loc[parents_evening_mask,
    'School-involvement reason'] = "The repeated items provide a harmonised measure of whether at least one parent attended a parents' evening. The latest permitted response is used, with earlier-wave fallback"
parental_school_involvement_variable_decisions.loc[special_meeting_mask,
    'School-involvement decision'] = 'Deferred to contextual review'
parental_school_involvement_variable_decisions.loc[special_meeting_mask,
    'School-involvement role'] = 'Repeated specially arranged teacher meeting'
parental_school_involvement_variable_decisions.loc[special_meeting_mask,
    'School-involvement reason'] = 'A specially arranged meeting may reflect reactive school–parent contact rather than general voluntary involvement. It will be reviewed with the contextual contact items'
parental_school_involvement_variable_decisions.loc[school_report_discussion_mask,
    'School-involvement decision'] = 'Review support'
parental_school_involvement_variable_decisions.loc[school_report_discussion_mask,
    'School-involvement role'] = 'Parent discussion of school reports'
parental_school_involvement_variable_decisions.loc[school_report_discussion_mask,
    'School-involvement reason'] = 'The item has very limited variation, with nearly all respondents reporting that they discuss school reports'
parental_school_involvement_variable_decisions.loc[vocational_subject_discussion_mask,
    'School-involvement decision'] = 'Review support'
parental_school_involvement_variable_decisions.loc[vocational_subject_discussion_mask,
    'School-involvement role'] = 'Parent–teacher discussion of vocational subjects'
parental_school_involvement_variable_decisions.loc[vocational_subject_discussion_mask,
    'School-involvement reason'] = 'The item covers one specific curriculum topic and is not used separately because the later post-16 discussion profile provides a broader representation of transition-related contact'
parental_school_involvement_variable_decisions.loc[post16_discussion_mask,
    'School-involvement decision'] = 'Construction input'
parental_school_involvement_variable_decisions.loc[post16_discussion_mask,
    'School-involvement role'] = 'Parent–teacher post-16 discussion profile'
parental_school_involvement_variable_decisions.loc[post16_discussion_mask,
    'School-involvement reason'] = 'The routed items distinguish no post-16 discussion, staying-on discussion, apprenticeship or training discussion, and other or unspecified discussion'
assert parental_school_involvement_variable_decisions['School-involvement decision'].notna().all()
assert parental_school_involvement_variable_decisions['School-involvement decision'].eq('Construction input').sum() == 6
assert parental_school_involvement_variable_decisions['School-involvement decision'].eq('Deferred to contextual review').sum() == 3
assert parental_school_involvement_variable_decisions['School-involvement decision'].eq('Review support').sum() == 2
parental_school_involvement_representation_decisions = pd.DataFrame([{'Representation': "Parents' evening attendance",
    'Decision': 'Retain', 'Reason': 'It represents general parental participation in a routine school event and has high coverage'}, {'Representation': 'Specially arranged teacher meeting',
    'Decision': 'Defer', 'Reason': 'The item may reflect reactive contact concerning a problem and requires comparison with the separate reactive school–parent contact items'}, {'Representation': 'General school-involvement breadth',
    'Decision': 'Do not retain', 'Reason': "The score combines routine parents' evening attendance with specially arranged meetings, although the two activities have almost no association"}, {'Representation': 'Binary parent–teacher post-16 discussion',
    'Decision': 'Do not retain', 'Reason': 'The binary measure removes information about the content of the discussion'}, {'Representation': 'Five-category post-16 discussion profile',
    'Decision': 'Do not retain', 'Reason': 'The apprenticeship-only category contains 142 participants and is consolidated with discussion of both staying on and apprenticeship or training'}, {'Representation': 'Four-category post-16 discussion profile',
    'Decision': 'Retain', 'Reason': 'The profile preserves substantively different discussion content while avoiding a very small apprenticeship-only category'}])
parental_school_involvement_predictor_summary = pd.DataFrame([{'Predictor': 'parents_evening_attendance_pretransition',
    'Non-missing': int(parents_evening_attendance_pretransition.notna().sum()), 'Missing': int(parents_evening_attendance_pretransition.isna().sum()), 'Missing percentage': round(parents_evening_attendance_pretransition.isna().mean() * 100,
    2), 'Categories': int(parents_evening_attendance_pretransition.nunique())}, {'Predictor': 'parent_teacher_post16_discussion_profile_pretransition',
    'Non-missing': int(parent_teacher_post16_discussion_profile_pretransition.notna().sum()), 'Missing': int(parent_teacher_post16_discussion_profile_pretransition.isna().sum()), 'Missing percentage': round(parent_teacher_post16_discussion_profile_pretransition.isna().mean() * 100,
    2), 'Categories': int(parent_teacher_post16_discussion_profile_pretransition.nunique())}])
parental_school_involvement_retained_distribution_rows = []
retained_school_involvement_specs = [("Parents' evening attendance", parents_evening_attendance_pretransition,
    {0: 'No attendance', 1: 'At least one parent attended'}), ('Parent–teacher post-16 discussion profile',
    parent_teacher_post16_discussion_profile_pretransition, {0: 'No post-16 discussion',
    1: 'Staying-on discussion only', 2: 'Any apprenticeship or training discussion', 3: 'Other or unspecified post-16 discussion'})]
for predictor, values, labels in retained_school_involvement_specs:
    observed_count = int(values.notna().sum())
    for code, category in labels.items():
        participants = int(values.eq(code).sum())
        parental_school_involvement_retained_distribution_rows.append({'Predictor': predictor, 'Code': code,
            'Category': category, 'Participants': participants, 'Percentage among observed': round(participants / observed_count * 100,
            2)})
parental_school_involvement_retained_distribution = pd.DataFrame(parental_school_involvement_retained_distribution_rows)
parental_school_involvement_decision_display = parental_school_involvement_variable_decisions.sort_values(['Source order',
    'Variable position'])[['Wave', 'Source file', 'Variable', 'Variable label', 'Timing status',
    'School-involvement decision', 'School-involvement role', 'School-involvement reason']].reset_index(drop=True)
print('Parental school-involvement predictors retained: 2')
print('Variables deferred to contextual review: 3')
print('Retained predictor summary:')
display_limited(parental_school_involvement_predictor_summary)
print('Retained category distributions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_school_involvement_retained_distribution)
print('Representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_school_involvement_representation_decisions)
print('Variable decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_school_involvement_decision_display)
print('Files written in this cell: 0')

Parental school-involvement predictors retained: 2
Variables deferred to contextual review: 3
Retained predictor summary:


,Predictor,Non-missing,Missing,Missing percentage,Categories
0,parents_evening_attendance_pretransition,9515,252,2.58,2
1,parent_teacher_post16_discussion_profile_pretr...,9338,429,4.39,4


Retained category distributions:


,Predictor,Code,Category,Participants,Percentage among observed
0,Parents' evening attendance,0,No attendance,1255,13.19
1,Parents' evening attendance,1,At least one parent attended,8260,86.81
2,Parent–teacher post-16 discussion profile,0,No post-16 discussion,5124,54.87
3,Parent–teacher post-16 discussion profile,1,Staying-on discussion only,2658,28.46
4,Parent–teacher post-16 discussion profile,2,Any apprenticeship or training discussion,747,8.00


Representation decisions:


,Representation,Decision,Reason
0,Parents' evening attendance,Retain,It represents general parental participation in a routine school event and has high coverage
1,Specially arranged teacher meeting,Defer,The item may reflect reactive contact concerning a problem and requires comparison with the separate reactive school–parent contact items
2,General school-involvement breadth,Do not retain,"The score combines routine parents' evening attendance with specially arranged meetings, although the two activities have almost no association"
3,Binary parent–teacher post-16 discussion,Do not retain,The binary measure removes information about the content of the discussion
4,Five-category post-16 discussion profile,Do not retain,The apprenticeship-only category contains 142 participants and is consolidated with discussion of both staying on and apprenticeship or training


Variable decisions:


,Wave,Source file,Variable,Variable label,Timing status,School-involvement decision,School-involvement role,School-involvement reason
0,Wave 1,wave_one_lsype_parental_attitudes_file_16_05_08,W1pareveMP,MP: Whether self or partner have been to any parents' evenings or similar events,Pre-transition source,Construction input,Repeated parent attendance at parents' evenings,"The repeated items provide a harmonised measure of whether at least one parent attended a parents' evening. The latest permitted response is used, with earlier-wave fallback"
1,Wave 1,wave_one_lsype_parental_attitudes_file_16_05_08,W1tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,Pre-transition source,Deferred to contextual review,Repeated specially arranged teacher meeting,A specially arranged meeting may reflect reactive school–parent contact rather than general voluntary involvement. It will be reviewed with the contextual contact items
2,Wave 1,wave_one_lsype_parental_attitudes_file_16_05_08,W1repred1MP,MP: Whether MP ever talks about YP's school reports with them,Pre-transition source,Review support,Parent discussion of school reports,"The item has very limited variation, with nearly all respondents reporting that they discuss school reports"
3,Wave 2,wave_two_lsype_parental_attitudes_file_16_06_08,W2pareveMP,MP: Whether self or partner have been to any parents' evenings or similar events,Pre-transition source,Construction input,Repeated parent attendance at parents' evenings,"The repeated items provide a harmonised measure of whether at least one parent attended a parents' evening. The latest permitted response is used, with earlier-wave fallback"
4,Wave 2,wave_two_lsype_parental_attitudes_file_16_06_08,W2tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,Pre-transition source,Deferred to contextual review,Repeated specially arranged teacher meeting,A specially arranged meeting may reflect reactive school–parent contact rather than general voluntary involvement. It will be reviewed with the contextual contact items


Files written in this cell: 0


In [358]:
# 39: Parent–young person relationship track inspection

import pandas as pd
domain_9_relationship_track_counts = domain_9_screened_candidates['Domain 9 review track'].astype('string').value_counts(dropna=False).rename('Variables').rename_axis('Domain 9 review track').reset_index()
parent_relationship_track_matches = domain_9_relationship_track_counts.loc[domain_9_relationship_track_counts['Domain 9 review track'].astype('string').str.contains('relationship|communication',
    case=False, na=False, regex=True)].reset_index(drop=True)
parent_relationship_variable_matches = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 review track'].astype('string').str.contains('relationship|communication',
    case=False, na=False, regex=True)].sort_values(['Domain 9 review track', 'Source order',
    'Variable position'])[['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status', 'Domain 9 screening status', 'Domain 9 review track']].reset_index(drop=True)
print('Possible relationship and communication review tracks:')
display_limited(parent_relationship_track_matches)
print('Variables in matching tracks:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parent_relationship_variable_matches)
print('Files written in this cell: 0')

Possible relationship and communication review tracks:


,Domain 9 review track,Variables
0,Parent–young-person relationship and communica...,5


Variables in matching tracks:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Domain 9 screening status,Domain 9 review track
0,Wave 1,Young person,wave_one_lsype_young_person_2020,243,W1mgetonYP,YP: How well get on with (step-)mother,Pre-transition source,Core review,Parent–young-person relationship and communication
1,Wave 1,Young person,wave_one_lsype_young_person_2020,244,W1fgetonYP,YP: How well get on with (step-)father,Pre-transition source,Core review,Parent–young-person relationship and communication
2,Wave 1,Young person,wave_one_lsype_young_person_2020,247,W1talkmumYP,YP: How often talk to (step-)mother about things that matter to YP,Pre-transition source,Core review,Parent–young-person relationship and communication
3,Wave 1,Young person,wave_one_lsype_young_person_2020,248,W1talkdadYP,YP: How often talk to (step-)father about things that matter to YP,Pre-transition source,Core review,Parent–young-person relationship and communication
4,Wave 1,Young person,wave_one_lsype_young_person_2020,254,W1talkschYP,YP: How often parents talk to YP about day at school,Pre-transition source,Core review,Parent–young-person relationship and communication


Files written in this cell: 0


In [359]:
# 40: Parent–young person relationship response and structure review

import pandas as pd
from scipy.stats import spearmanr
relationship_track_name = parent_relationship_track_matches.loc[0, 'Domain 9 review track']
parent_relationship_inventory = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 review track'].eq(relationship_track_name)].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(parent_relationship_inventory) == 5
assert set(parent_relationship_inventory['Variable']) == {'W1mgetonYP', 'W1fgetonYP', 'W1talkmumYP', 'W1talkdadYP',
    'W1talkschYP'}
relationship_source_file = parent_relationship_inventory['Source file'].unique().tolist()
assert len(relationship_source_file) == 1
relationship_source_file = relationship_source_file[0]
relationship_source_path = source_file_lookup[relationship_source_file]
relationship_variables = parent_relationship_inventory['Variable'].tolist()
parent_relationship_raw = pd.read_stata(relationship_source_path, columns=['NSID', *relationship_variables],
    convert_categoricals=False)
parent_relationship_labelled = pd.read_stata(relationship_source_path, columns=['NSID', *relationship_variables],
    convert_categoricals=True)
for data in [parent_relationship_raw, parent_relationship_labelled]:
    data['NSID'] = standardise_nsid(data['NSID'])
    assert data['NSID'].is_unique
parent_relationship_raw = parent_relationship_raw.set_index('NSID').reindex(route_index)
parent_relationship_labelled = parent_relationship_labelled.set_index('NSID').reindex(route_index)
for variable in relationship_variables:
    parent_relationship_raw[variable] = pd.to_numeric(parent_relationship_raw[variable], errors='coerce')
parent_relationship_coverage_rows = []
parent_relationship_code_rows = []
parent_relationship_observed = {}
for _, inventory_row in parent_relationship_inventory.iterrows():
    variable = inventory_row['Variable']
    raw_values = parent_relationship_raw[variable]
    labelled_values = parent_relationship_labelled[variable].astype('string')
    observed_mask = raw_values.ge(0).fillna(False).astype(bool)
    special_mask = raw_values.lt(0).fillna(False).astype(bool)
    structural_mask = raw_values.eq(-91.0).fillna(False).astype(bool)
    parent_relationship_observed[variable] = raw_values.where(observed_mask)
    parent_relationship_coverage_rows.append({'Variable': variable, 'Variable label': inventory_row['Variable label'],
        'Observed responses': int(observed_mask.sum()), 'Observed percentage': round(observed_mask.mean() * 100,
        2), 'Structurally not applicable': int(structural_mask.sum()), 'Other special-code responses': int((special_mask & ~structural_mask).sum()), 'No source record': int(raw_values.isna().sum()), 'Observed categories': int(raw_values.loc[observed_mask].nunique())})
    for raw_code, participants in raw_values.value_counts(dropna=False).items():
        if pd.isna(raw_code):
            response_label = 'No source record'
            response_type = 'Unavailable'
            sort_value = 999999
        else:
            response_mask = raw_values.eq(raw_code)
            matching_labels = labelled_values.loc[response_mask].dropna().drop_duplicates().tolist()
            response_label = matching_labels[0] if matching_labels else str(raw_code)
            if raw_code >= 0:
                response_type = 'Observed response'
            elif raw_code == -91:
                response_type = 'Structural non-applicability'
            else:
                response_type = 'Other special code'
            sort_value = float(raw_code)
        parent_relationship_code_rows.append({'Variable': variable, 'Raw code': raw_code,
            'Value label': response_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
parent_relationship_coverage_summary = pd.DataFrame(parent_relationship_coverage_rows)
parent_relationship_code_distribution = pd.DataFrame(parent_relationship_code_rows).sort_values(['Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
mother_relationship_available = parent_relationship_observed['W1mgetonYP'].notna()
mother_communication_available = parent_relationship_observed['W1talkmumYP'].notna()
father_relationship_available = parent_relationship_observed['W1fgetonYP'].notna()
father_communication_available = parent_relationship_observed['W1talkdadYP'].notna()
parent_relationship_availability = pd.DataFrame({'Mother relationship available': mother_relationship_available,
    'Mother communication available': mother_communication_available, 'Father relationship available': father_relationship_available, 'Father communication available': father_communication_available}, index=route_index)
parent_relationship_availability['Mother pair complete'] = parent_relationship_availability['Mother relationship available'] & parent_relationship_availability['Mother communication available']
parent_relationship_availability['Father pair complete'] = parent_relationship_availability['Father relationship available'] & parent_relationship_availability['Father communication available']
parent_relationship_availability_patterns = parent_relationship_availability[['Mother pair complete',
    'Father pair complete']].value_counts(dropna=False).rename('Participants').reset_index()
parent_relationship_availability_patterns['Percentage'] = (parent_relationship_availability_patterns['Participants'] / len(route_index) * 100).round(2)
relationship_pair_specs = [('Mother relationship', 'W1mgetonYP', 'Mother communication', 'W1talkmumYP'),
    ('Father relationship', 'W1fgetonYP', 'Father communication', 'W1talkdadYP'), ('Mother relationship', 'W1mgetonYP',
    'Father relationship', 'W1fgetonYP'), ('Mother communication', 'W1talkmumYP', 'Father communication',
    'W1talkdadYP'), ('Mother relationship', 'W1mgetonYP', 'School-day discussion',
    'W1talkschYP'), ('Father relationship', 'W1fgetonYP', 'School-day discussion',
    'W1talkschYP'), ('Mother communication', 'W1talkmumYP', 'School-day discussion',
    'W1talkschYP'), ('Father communication', 'W1talkdadYP', 'School-day discussion', 'W1talkschYP')]
parent_relationship_pair_rows = []
for first_label, first_variable, second_label, second_variable in relationship_pair_specs:
    pair_data = pd.DataFrame({first_label: parent_relationship_observed[first_variable],
        second_label: parent_relationship_observed[second_variable]}, index=route_index).dropna()
    assert len(pair_data) > 0
    if pair_data[first_label].nunique() > 1 and pair_data[second_label].nunique() > 1:
        spearman_value = float(spearmanr(pair_data[first_label], pair_data[second_label]).statistic)
    else:
        spearman_value = float('nan')
    parent_relationship_pair_rows.append({'First measure': first_label, 'Second measure': second_label,
        'Complete comparisons': len(pair_data), 'First categories': int(pair_data[first_label].nunique()), 'Second categories': int(pair_data[second_label].nunique()), "Cramer's V": round(categorical_cramers_v(pair_data[first_label],
        pair_data[second_label]), 3), 'Spearman correlation': round(spearman_value, 3)})
parent_relationship_pair_summary = pd.DataFrame(parent_relationship_pair_rows)
relationship_observed_matrix = pd.DataFrame({variable: parent_relationship_observed[variable].notna() for variable in relationship_variables},
    index=route_index)
relationship_observed_item_count = relationship_observed_matrix.sum(axis=1)
relationship_observed_item_count_summary = relationship_observed_item_count.value_counts().sort_index().rename('Participants').rename_axis('Observed relationship or communication items').reset_index()
relationship_observed_item_count_summary['Percentage'] = (relationship_observed_item_count_summary['Participants'] / len(route_index) * 100).round(2)
parent_relationship_inventory_display = parent_relationship_inventory[['Wave', 'Source type', 'Source file',
    'Variable', 'Variable label', 'Timing status', 'Domain 9 review track']].copy()
print(f'Parent–young person relationship variables reviewed: {len(parent_relationship_inventory)}')
print('Variable inventory:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parent_relationship_inventory_display)
print('Coverage summary:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parent_relationship_coverage_summary)
print('Response-code distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parent_relationship_code_distribution)
print('Mother and father pair availability:')
display_limited(parent_relationship_availability_patterns)
print('Observed item-count distribution:')
display_limited(relationship_observed_item_count_summary)
print('Pairwise relationship and communication associations:')
display_limited(parent_relationship_pair_summary)
print('Files written in this cell: 0')

Parent–young person relationship variables reviewed: 5
Variable inventory:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Domain 9 review track
0,Wave 1,Young person,wave_one_lsype_young_person_2020,W1mgetonYP,YP: How well get on with (step-)mother,Pre-transition source,Parent–young-person relationship and communication
1,Wave 1,Young person,wave_one_lsype_young_person_2020,W1fgetonYP,YP: How well get on with (step-)father,Pre-transition source,Parent–young-person relationship and communication
2,Wave 1,Young person,wave_one_lsype_young_person_2020,W1talkmumYP,YP: How often talk to (step-)mother about things that matter to YP,Pre-transition source,Parent–young-person relationship and communication
3,Wave 1,Young person,wave_one_lsype_young_person_2020,W1talkdadYP,YP: How often talk to (step-)father about things that matter to YP,Pre-transition source,Parent–young-person relationship and communication
4,Wave 1,Young person,wave_one_lsype_young_person_2020,W1talkschYP,YP: How often parents talk to YP about day at school,Pre-transition source,Parent–young-person relationship and communication


Coverage summary:


,Variable,Variable label,Observed responses,Observed percentage,Structurally not applicable,Other special-code responses,No source record,Observed categories
0,W1mgetonYP,YP: How well get on with (step-)mother,8959,91.73,253,312,243,5
1,W1fgetonYP,YP: How well get on with (step-)father,7256,74.29,1941,327,243,5
2,W1talkmumYP,YP: How often talk to (step-)mother about things that matter to YP,8783,89.93,253,488,243,5
3,W1talkdadYP,YP: How often talk to (step-)father about things that matter to YP,7118,72.88,1941,465,243,5
4,W1talkschYP,YP: How often parents talk to YP about day at school,9079,92.96,0,445,243,3


Response-code distributions:


...

Observed item-count distribution:


,Observed relationship or communication items,Participants,Percentage
0,0,457,4.68
1,1,110,1.13
2,2,159,1.63
3,3,2065,21.14
4,4,308,3.15


Pairwise relationship and communication associations:


,First measure,Second measure,Complete comparisons,First categories,Second categories,Cramer's V,Spearman correlation
0,Mother relationship,Mother communication,8703,5,5,0.174,0.312
1,Father relationship,Father communication,7035,5,5,0.235,0.389
2,Mother relationship,Father relationship,7066,5,5,0.332,0.609
3,Mother communication,Father communication,6905,5,5,0.437,0.591
4,Mother relationship,School-day discussion,8741,5,3,0.141,-0.169


Files written in this cell: 0


In [360]:
# 41: Parent–young person relationship representation review

import pandas as pd
from scipy.stats import spearmanr
parent_relationship_quality_recode = {1.0: 3.0, 2.0: 2.0, 3.0: 1.0, 4.0: 0.0}
mother_relationship_quality = parent_relationship_observed['W1mgetonYP'].map(parent_relationship_quality_recode).astype('Float64').rename('mother_relationship_quality')
father_relationship_quality = parent_relationship_observed['W1fgetonYP'].map(parent_relationship_quality_recode).astype('Float64').rename('father_relationship_quality')
parent_communication_frequency_recode = {1.0: 4.0, 2.0: 3.0, 3.0: 2.0, 4.0: 1.0, 5.0: 0.0}
mother_communication_frequency = parent_relationship_observed['W1talkmumYP'].map(parent_communication_frequency_recode).astype('Float64').rename('mother_communication_frequency')
father_communication_frequency = parent_relationship_observed['W1talkdadYP'].map(parent_communication_frequency_recode).astype('Float64').rename('father_communication_frequency')
school_day_discussion_recode = {1.0: 0, 2.0: 1, 3.0: 2}
parent_school_day_discussion_frequency_candidate = parent_relationship_observed['W1talkschYP'].map(school_day_discussion_recode).astype('Int64').rename('parent_school_day_discussion_frequency_pretransition')
relationship_contributing_parent_count = pd.DataFrame({'Mother': mother_relationship_quality,
    'Father': father_relationship_quality}, index=route_index).notna().sum(axis=1).astype('Int64').rename('relationship_contributing_parent_count')
communication_contributing_parent_count = pd.DataFrame({'Mother': mother_communication_frequency,
    'Father': father_communication_frequency}, index=route_index).notna().sum(axis=1).astype('Int64').rename('communication_contributing_parent_count')
parent_relationship_quality_score_candidate = pd.DataFrame({'Mother': mother_relationship_quality,
    'Father': father_relationship_quality}, index=route_index).mean(axis=1,
    skipna=True).where(relationship_contributing_parent_count.gt(0)).astype('Float64').rename('parent_relationship_quality_score_pretransition')
parent_communication_frequency_score_candidate = pd.DataFrame({'Mother': mother_communication_frequency,
    'Father': father_communication_frequency}, index=route_index).mean(axis=1,
    skipna=True).where(communication_contributing_parent_count.gt(0)).astype('Float64').rename('parent_communication_frequency_score_pretransition')
assert set(parent_relationship_quality_score_candidate.dropna().astype(float).unique()).issubset({0.0, 0.5, 1.0, 1.5,
    2.0, 2.5, 3.0})
assert set(parent_communication_frequency_score_candidate.dropna().astype(float).unique()).issubset({0.0, 0.5, 1.0,
    1.5, 2.0, 2.5, 3.0, 3.5, 4.0})
assert set(parent_school_day_discussion_frequency_candidate.dropna().astype(int).unique()).issubset({0, 1, 2})
parent_non_contact_summary = pd.DataFrame([{'Source item': 'Mother relationship', 'Response': "I don't see her",
    'Participants': int(parent_relationship_observed['W1mgetonYP'].eq(5.0).sum()), 'Treatment': 'Excluded from the relationship-quality scale'}, {'Source item': 'Father relationship',
    'Response': "I don't see him", 'Participants': int(parent_relationship_observed['W1fgetonYP'].eq(5.0).sum()), 'Treatment': 'Excluded from the relationship-quality scale'}])
relationship_candidate_specs = [('Parent relationship-quality score', parent_relationship_quality_score_candidate),
    ('Parent communication-frequency score',
    parent_communication_frequency_score_candidate), ('Parent discussion of school day',
    parent_school_day_discussion_frequency_candidate)]
parent_relationship_candidate_summary_rows = []
for candidate_name, candidate_values in relationship_candidate_specs:
    observed_values = candidate_values.dropna().astype(float)
    parent_relationship_candidate_summary_rows.append({'Candidate predictor': candidate_name,
        'Non-missing': int(candidate_values.notna().sum()), 'Missing': int(candidate_values.isna().sum()), 'Missing percentage': round(candidate_values.isna().mean() * 100,
        2), 'Distinct values': int(observed_values.nunique()), 'Minimum': float(observed_values.min()), 'Median': round(float(observed_values.median()),
        2), 'Mean': round(float(observed_values.mean()), 2), 'Maximum': float(observed_values.max())})
parent_relationship_candidate_summary = pd.DataFrame(parent_relationship_candidate_summary_rows)
parent_relationship_candidate_distribution_rows = []
for candidate_name, candidate_values in relationship_candidate_specs:
    observed_count = int(candidate_values.notna().sum())
    for value, participants in candidate_values.dropna().value_counts().sort_index().items():
        parent_relationship_candidate_distribution_rows.append({'Candidate predictor': candidate_name,
            'Value': float(value), 'Participants': int(participants), 'Percentage among observed': round(participants / observed_count * 100,
            2)})
parent_relationship_candidate_distributions = pd.DataFrame(parent_relationship_candidate_distribution_rows)
parent_relationship_contributor_summary = pd.concat([relationship_contributing_parent_count.value_counts().sort_index().rename('Participants').rename_axis('Contributing parents').reset_index().assign(Candidate='Parent relationship-quality score'),
    communication_contributing_parent_count.value_counts().sort_index().rename('Participants').rename_axis('Contributing parents').reset_index().assign(Candidate='Parent communication-frequency score')], ignore_index=True)
parent_relationship_contributor_summary['Percentage of full sample'] = (parent_relationship_contributor_summary['Participants'] / len(route_index) * 100).round(2)
parent_relationship_candidate_pair_rows = []
for first_name, first_values, second_name, second_values in [('Parent relationship-quality score',
    parent_relationship_quality_score_candidate, 'Parent communication-frequency score', parent_communication_frequency_score_candidate), ('Parent relationship-quality score',
    parent_relationship_quality_score_candidate, 'Parent discussion of school day', parent_school_day_discussion_frequency_candidate), ('Parent communication-frequency score',
    parent_communication_frequency_score_candidate, 'Parent discussion of school day', parent_school_day_discussion_frequency_candidate)]:
    pair_data = pd.DataFrame({first_name: first_values, second_name: second_values}, index=route_index).dropna()
    parent_relationship_candidate_pair_rows.append({'First measure': first_name, 'Second measure': second_name,
        'Complete comparisons': len(pair_data), 'Spearman correlation': round(float(spearmanr(pair_data[first_name],
        pair_data[second_name]).statistic), 3)})
parent_relationship_candidate_pair_summary = pd.DataFrame(parent_relationship_candidate_pair_rows)
parent_relationship_ordinal_overlap_data = pd.DataFrame({'Parent relationship-quality score': parent_relationship_quality_score_candidate,
    'Parent communication-frequency score': parent_communication_frequency_score_candidate, 'Parent discussion of school day': parent_school_day_discussion_frequency_candidate, 'Homework monitoring frequency': homework_monitoring_frequency_pretransition, 'School attitude score': parental_homework_school_overlap_data['school_attitude_score'], 'Parental educational aspiration': parental_educational_aspiration_candidate, 'Parental HE expectation': parental_he_expectation_candidate, 'Parental financial-support profile': parental_educational_financial_support_profile_pretransition}, index=route_index)
parent_relationship_ordinal_overlap_rows = []
for candidate_measure in ['Parent relationship-quality score', 'Parent communication-frequency score',
    'Parent discussion of school day']:
    for comparison_measure in ['Homework monitoring frequency', 'School attitude score',
        'Parental educational aspiration', 'Parental HE expectation', 'Parental financial-support profile']:
        pair_data = parent_relationship_ordinal_overlap_data[[candidate_measure, comparison_measure]].dropna()
        assert len(pair_data) > 0
        parent_relationship_ordinal_overlap_rows.append({'Candidate predictor': candidate_measure,
            'Comparison measure': comparison_measure, 'Complete comparisons': len(pair_data), 'Spearman correlation': round(float(spearmanr(pair_data[candidate_measure],
            pair_data[comparison_measure]).statistic), 3)})
parent_relationship_ordinal_overlap_summary = pd.DataFrame(parent_relationship_ordinal_overlap_rows).assign(Absolute_correlation=lambda data: data['Spearman correlation'].abs()).sort_values(['Candidate predictor',
    'Absolute_correlation'], ascending=[True, False]).drop(columns=['Absolute_correlation']).reset_index(drop=True)
parent_relationship_categorical_overlap_data = pd.DataFrame({'Parent relationship-quality score': parent_relationship_quality_score_candidate,
    'Parent communication-frequency score': parent_communication_frequency_score_candidate, 'Parent discussion of school day': parent_school_day_discussion_frequency_candidate, 'Homework help at home': homework_help_at_home_pretransition, 'Parental knowledge of evening whereabouts': parental_knowledge_of_evening_whereabouts_pretransition, 'School-night curfew': school_night_curfew_pretransition, "Parents' evening attendance": parents_evening_attendance_pretransition}, index=route_index)
parent_relationship_categorical_overlap_rows = []
for candidate_measure in ['Parent relationship-quality score', 'Parent communication-frequency score',
    'Parent discussion of school day']:
    for comparison_measure in ['Homework help at home', 'Parental knowledge of evening whereabouts',
        'School-night curfew', "Parents' evening attendance"]:
        pair_data = parent_relationship_categorical_overlap_data[[candidate_measure, comparison_measure]].dropna()
        assert len(pair_data) > 0
        parent_relationship_categorical_overlap_rows.append({'Candidate predictor': candidate_measure,
            'Comparison measure': comparison_measure, 'Complete comparisons': len(pair_data), "Cramer's V": round(categorical_cramers_v(pair_data[candidate_measure],
            pair_data[comparison_measure]), 3)})
parent_relationship_categorical_overlap_summary = pd.DataFrame(parent_relationship_categorical_overlap_rows).sort_values(['Candidate predictor',
    "Cramer's V"], ascending=[True, False]).reset_index(drop=True)
print('Non-contact response treatment:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parent_non_contact_summary)
print('Candidate predictor summary:')
display_limited(parent_relationship_candidate_summary)
print('Number of contributing parent reports:')
display_limited(parent_relationship_contributor_summary[['Candidate', 'Contributing parents', 'Participants',
    'Percentage of full sample']])
print('Candidate distributions:')
with pd.option_context('display.max_rows', None):
    display_limited(parent_relationship_candidate_distributions)
print('Associations among relationship and communication candidates:')
display_limited(parent_relationship_candidate_pair_summary)
print('Ordinal associations with retained predictors:')
display_limited(parent_relationship_ordinal_overlap_summary)
print('Categorical associations with retained family practices:')
display_limited(parent_relationship_categorical_overlap_summary)
print('Files written in this cell: 0')

Non-contact response treatment:


,Source item,Response,Participants,Treatment
0,Mother relationship,I don't see her,4,Excluded from the relationship-quality scale
1,Father relationship,I don't see him,14,Excluded from the relationship-quality scale


Candidate predictor summary:


,Candidate predictor,Non-missing,Missing,Missing percentage,Distinct values,Minimum,Median,Mean,Maximum
0,Parent relationship-quality score,9146,621,6.36,7,0.0,3.0,2.62,3.0
1,Parent communication-frequency score,8996,771,7.89,9,0.0,2.5,2.55,4.0
2,Parent discussion of school day,9079,688,7.04,3,0.0,1.0,1.40,2.0


Number of contributing parent reports:


...

Ordinal associations with retained predictors:


,Candidate predictor,Comparison measure,Complete comparisons,Spearman correlation
0,Parent communication-frequency score,School attitude score,8994,0.188
1,Parent communication-frequency score,Homework monitoring frequency,8825,0.144
2,Parent communication-frequency score,Parental HE expectation,8408,0.062
3,Parent communication-frequency score,Parental educational aspiration,8805,0.048
4,Parent communication-frequency score,Parental financial-support profile,8988,-0.006


Categorical associations with retained family practices:


,Candidate predictor,Comparison measure,Complete comparisons,Cramer's V
0,Parent communication-frequency score,Homework help at home,8792,0.126
1,Parent communication-frequency score,Parental knowledge of evening whereabouts,8958,0.103
2,Parent communication-frequency score,Parents' evening attendance,8987,0.071
3,Parent communication-frequency score,School-night curfew,8807,0.055
4,Parent discussion of school day,Parental knowledge of evening whereabouts,9045,0.138


Files written in this cell: 0


In [361]:
# 42: Parent–young person relationship decisions

import pandas as pd
parent_relationship_quality_score_pretransition = parent_relationship_quality_score_candidate.copy().astype('Float64').rename('parent_relationship_quality_score_pretransition')
parent_communication_frequency_score_pretransition = parent_communication_frequency_score_candidate.copy().astype('Float64').rename('parent_communication_frequency_score_pretransition')
parent_school_day_discussion_frequency_pretransition = parent_school_day_discussion_frequency_candidate.copy().astype('Int64').rename('parent_school_day_discussion_frequency_pretransition')
assert set(parent_relationship_quality_score_pretransition.dropna().astype(float).unique()).issubset({0.0, 0.5, 1.0,
    1.5, 2.0, 2.5, 3.0})
assert set(parent_communication_frequency_score_pretransition.dropna().astype(float).unique()).issubset({0.0, 0.5, 1.0,
    1.5, 2.0, 2.5, 3.0, 3.5, 4.0})
assert set(parent_school_day_discussion_frequency_pretransition.dropna().astype(int).unique()).issubset({0, 1, 2})
parent_relationship_predictors = pd.DataFrame({'NSID': route_index,
    'parent_relationship_quality_score_pretransition': parent_relationship_quality_score_pretransition.to_numpy(), 'parent_communication_frequency_score_pretransition': parent_communication_frequency_score_pretransition.to_numpy(), 'parent_school_day_discussion_frequency_pretransition': parent_school_day_discussion_frequency_pretransition.to_numpy()})
assert len(parent_relationship_predictors) == 9767
assert parent_relationship_predictors['NSID'].is_unique
parent_relationship_joint_availability = parent_relationship_predictors[['parent_relationship_quality_score_pretransition',
    'parent_communication_frequency_score_pretransition', 'parent_school_day_discussion_frequency_pretransition']].notna().sum(axis=1)
parent_relationship_joint_availability_summary = parent_relationship_joint_availability.value_counts().sort_index().rename('Participants').rename_axis('Retained predictors available').reset_index()
parent_relationship_joint_availability_summary['Percentage of full sample'] = (parent_relationship_joint_availability_summary['Participants'] / len(route_index) * 100).round(2)
parent_relationship_variable_decisions = parent_relationship_inventory.copy()
parent_relationship_variable_decisions['Relationship decision'] = 'Construction input'
parent_relationship_variable_decisions['Relationship role'] = pd.NA
parent_relationship_variable_decisions['Relationship reason'] = pd.NA
mother_relationship_mask = parent_relationship_variable_decisions['Variable'].eq('W1mgetonYP')
father_relationship_mask = parent_relationship_variable_decisions['Variable'].eq('W1fgetonYP')
mother_communication_mask = parent_relationship_variable_decisions['Variable'].eq('W1talkmumYP')
father_communication_mask = parent_relationship_variable_decisions['Variable'].eq('W1talkdadYP')
school_day_discussion_mask = parent_relationship_variable_decisions['Variable'].eq('W1talkschYP')
parent_relationship_variable_decisions.loc[mother_relationship_mask,
    'Relationship role'] = 'Mother or stepmother relationship-quality input'
parent_relationship_variable_decisions.loc[father_relationship_mask,
    'Relationship role'] = 'Father or stepfather relationship-quality input'
parent_relationship_variable_decisions.loc[mother_communication_mask,
    'Relationship role'] = 'Mother or stepmother communication-frequency input'
parent_relationship_variable_decisions.loc[father_communication_mask,
    'Relationship role'] = 'Father or stepfather communication-frequency input'
parent_relationship_variable_decisions.loc[school_day_discussion_mask,
    'Relationship role'] = "Parent discussion of the young person's school day"
parent_relationship_variable_decisions.loc[mother_relationship_mask | father_relationship_mask,
    'Relationship reason'] = 'The mother and father items are combined using the mean of available parent reports. Higher values indicate a better relationship. Responses indicating that the young person does not see the parent are not treated as low relationship quality'
parent_relationship_variable_decisions.loc[mother_communication_mask | father_communication_mask,
    'Relationship reason'] = 'The mother and father items are combined using the mean of available parent reports. Higher values indicate more frequent discussion of matters important to the young person'
parent_relationship_variable_decisions.loc[school_day_discussion_mask,
    'Relationship reason'] = 'The item is retained separately because school-related communication is conceptually distinct from general discussion of matters important to the young person'
assert parent_relationship_variable_decisions['Relationship role'].notna().all()
assert parent_relationship_variable_decisions['Relationship reason'].notna().all()
parent_relationship_representation_decisions = pd.DataFrame([{'Representation': 'Separate mother and father relationship predictors',
    'Decision': 'Do not retain', 'Reason': 'The parent-specific measures are moderately related, and separate predictors would make availability depend strongly on household composition'}, {'Representation': 'Available-parent mean relationship-quality score',
    'Decision': 'Retain', 'Reason': 'The score uses available information without treating the absence of a parent report as poor relationship quality'}, {'Representation': 'Separate mother and father communication predictors',
    'Decision': 'Do not retain', 'Reason': 'The parent-specific communication measures are related and have substantial differences in structural availability'}, {'Representation': 'Available-parent mean communication-frequency score',
    'Decision': 'Retain', 'Reason': 'The score summarises the frequency of meaningful communication across available parent relationships'}, {'Representation': 'Combined relationship and communication score',
    'Decision': 'Do not construct', 'Reason': 'Relationship quality and communication frequency are related but remain distinct constructs'}, {'Representation': 'Parent discussion of the school day',
    'Decision': 'Retain', 'Reason': 'The item captures education-related family communication and has limited overlap with the broader communication score'}, {'Representation': 'Number of contributing parent reports',
    'Decision': 'Do not retain as predictor', 'Reason': 'The count is used to document score construction but largely reflects family structure, which is represented elsewhere in the predictor set'}])
relationship_maximum_ordinal_overlap = parent_relationship_ordinal_overlap_summary.assign(Absolute_correlation=lambda data: data['Spearman correlation'].abs()).groupby('Candidate predictor',
    as_index=False)['Absolute_correlation'].max()
relationship_maximum_categorical_overlap = parent_relationship_categorical_overlap_summary.groupby('Candidate predictor',
    as_index=False)["Cramer's V"].max()
relationship_candidate_name_lookup = {'parent_relationship_quality_score_pretransition': 'Parent relationship-quality score',
    'parent_communication_frequency_score_pretransition': 'Parent communication-frequency score', 'parent_school_day_discussion_frequency_pretransition': 'Parent discussion of school day'}
parent_relationship_predictor_summary_rows = []
for predictor_name, candidate_name in relationship_candidate_name_lookup.items():
    predictor_values = parent_relationship_predictors[predictor_name]
    parent_relationship_predictor_summary_rows.append({'Predictor': predictor_name,
        'Non-missing': int(predictor_values.notna().sum()), 'Missing': int(predictor_values.isna().sum()), 'Missing percentage': round(predictor_values.isna().mean() * 100,
        2), 'Distinct values': int(predictor_values.nunique()), 'Maximum absolute Spearman correlation': float(relationship_maximum_ordinal_overlap.loc[relationship_maximum_ordinal_overlap['Candidate predictor'].eq(candidate_name),
        'Absolute_correlation'].iloc[0]), "Maximum Cramer's V": float(relationship_maximum_categorical_overlap.loc[relationship_maximum_categorical_overlap['Candidate predictor'].eq(candidate_name),
        "Cramer's V"].iloc[0])})
parent_relationship_predictor_summary = pd.DataFrame(parent_relationship_predictor_summary_rows)
parent_relationship_decision_display = parent_relationship_variable_decisions[['Wave', 'Source file', 'Variable',
    'Variable label', 'Timing status', 'Relationship decision', 'Relationship role', 'Relationship reason']].copy()
print('Parent–young person relationship predictors retained: 3')
print('Retained predictor summary:')
display_limited(parent_relationship_predictor_summary)
print('Joint availability:')
display_limited(parent_relationship_joint_availability_summary)
print('Representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parent_relationship_representation_decisions)
print('Variable decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parent_relationship_decision_display)
print('Files written in this cell: 0')

Parent–young person relationship predictors retained: 3
Retained predictor summary:


,Predictor,Non-missing,Missing,Missing percentage,Distinct values,Maximum absolute Spearman correlation,Maximum Cramer's V
0,parent_relationship_quality_score_pretransition,9146,621,6.36,7,0.185,0.120
1,parent_communication_frequency_score_pretransi...,8996,771,7.89,9,0.188,0.126
2,parent_school_day_discussion_frequency_pretran...,9079,688,7.04,3,0.159,0.138


Joint availability:


,Retained predictors available,Participants,Percentage of full sample
0,0,457,4.68
1,1,130,1.33
2,2,449,4.60
3,3,8731,89.39


Representation decisions:


,Representation,Decision,Reason
0,Separate mother and father relationship predictors,Do not retain,"The parent-specific measures are moderately related, and separate predictors would make availability depend strongly on household composition"
1,Available-parent mean relationship-quality score,Retain,The score uses available information without treating the absence of a parent report as poor relationship quality
2,Separate mother and father communication predictors,Do not retain,The parent-specific communication measures are related and have substantial differences in structural availability
3,Available-parent mean communication-frequency score,Retain,The score summarises the frequency of meaningful communication across available parent relationships
4,Combined relationship and communication score,Do not construct,Relationship quality and communication frequency are related but remain distinct constructs


Variable decisions:


,Wave,Source file,Variable,Variable label,Timing status,Relationship decision,Relationship role,Relationship reason
0,Wave 1,wave_one_lsype_young_person_2020,W1mgetonYP,YP: How well get on with (step-)mother,Pre-transition source,Construction input,Mother or stepmother relationship-quality input,The mother and father items are combined using the mean of available parent reports. Higher values indicate a better relationship. Responses indicating that the young person does not see the parent are not treated as low relationship quality
1,Wave 1,wave_one_lsype_young_person_2020,W1fgetonYP,YP: How well get on with (step-)father,Pre-transition source,Construction input,Father or stepfather relationship-quality input,The mother and father items are combined using the mean of available parent reports. Higher values indicate a better relationship. Responses indicating that the young person does not see the parent are not treated as low relationship quality
2,Wave 1,wave_one_lsype_young_person_2020,W1talkmumYP,YP: How often talk to (step-)mother about things that matter to YP,Pre-transition source,Construction input,Mother or stepmother communication-frequency input,The mother and father items are combined using the mean of available parent reports. Higher values indicate more frequent discussion of matters important to the young person
3,Wave 1,wave_one_lsype_young_person_2020,W1talkdadYP,YP: How often talk to (step-)father about things that matter to YP,Pre-transition source,Construction input,Father or stepfather communication-frequency input,The mother and father items are combined using the mean of available parent reports. Higher values indicate more frequent discussion of matters important to the young person
4,Wave 1,wave_one_lsype_young_person_2020,W1talkschYP,YP: How often parents talk to YP about day at school,Pre-transition source,Construction input,Parent discussion of the young person's school day,The item is retained separately because school-related communication is conceptually distinct from general discussion of matters important to the young person


Files written in this cell: 0


In [362]:
# 43: Parental autonomy track inspection

import pandas as pd
domain_9_autonomy_track_counts = domain_9_screened_candidates['Domain 9 review track'].astype('string').value_counts(dropna=False).rename('Variables').rename_axis('Domain 9 review track').reset_index()
parental_autonomy_track_matches = domain_9_autonomy_track_counts.loc[domain_9_autonomy_track_counts['Domain 9 review track'].astype('string').str.contains('autonomy|independ|decision',
    case=False, na=False, regex=True)].reset_index(drop=True)
parental_autonomy_variable_matches = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 review track'].astype('string').str.contains('autonomy|independ|decision',
    case=False, na=False, regex=True)].sort_values(['Domain 9 review track', 'Source order',
    'Variable position'])[['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status', 'Domain 9 screening status', 'Domain 9 review track']].reset_index(drop=True)
print('Possible parental-autonomy review tracks:')
display_limited(parental_autonomy_track_matches)
print('Variables in matching tracks:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_autonomy_variable_matches)
print('Files written in this cell: 0')

Possible parental-autonomy review tracks:


,Domain 9 review track,Variables
0,Parental autonomy support,2


Variables in matching tracks:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Domain 9 screening status,Domain 9 review track
0,Wave 1,Young person,wave_one_lsype_young_person_2020,249,W1mothdecYP,YP: How true it is to say (step-)mother likes YP to make own decisions,Pre-transition source,Core review,Parental autonomy support
1,Wave 1,Young person,wave_one_lsype_young_person_2020,250,W1fathdecYP,YP: How true it is to say (step-)father likes YP to make own decisions,Pre-transition source,Core review,Parental autonomy support


Files written in this cell: 0


In [363]:
# 44: Parental autonomy-support response and structure review

import pandas as pd
from scipy.stats import spearmanr
parental_autonomy_track_name = parental_autonomy_track_matches.loc[0, 'Domain 9 review track']
parental_autonomy_inventory = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 review track'].eq(parental_autonomy_track_name)].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(parental_autonomy_inventory) == 2
assert set(parental_autonomy_inventory['Variable']) == {'W1mothdecYP', 'W1fathdecYP'}
parental_autonomy_source_files = parental_autonomy_inventory['Source file'].unique().tolist()
assert len(parental_autonomy_source_files) == 1
parental_autonomy_source_file = parental_autonomy_source_files[0]
parental_autonomy_source_path = source_file_lookup[parental_autonomy_source_file]
parental_autonomy_variables = parental_autonomy_inventory['Variable'].tolist()
parental_autonomy_raw = pd.read_stata(parental_autonomy_source_path, columns=['NSID', *parental_autonomy_variables],
    convert_categoricals=False)
parental_autonomy_labelled = pd.read_stata(parental_autonomy_source_path, columns=['NSID',
    *parental_autonomy_variables], convert_categoricals=True)
for data in [parental_autonomy_raw, parental_autonomy_labelled]:
    data['NSID'] = standardise_nsid(data['NSID'])
    assert data['NSID'].is_unique
parental_autonomy_raw = parental_autonomy_raw.set_index('NSID').reindex(route_index)
parental_autonomy_labelled = parental_autonomy_labelled.set_index('NSID').reindex(route_index)
for variable in parental_autonomy_variables:
    parental_autonomy_raw[variable] = pd.to_numeric(parental_autonomy_raw[variable], errors='coerce')
parental_autonomy_coverage_rows = []
parental_autonomy_code_rows = []
parental_autonomy_observed = {}
for _, inventory_row in parental_autonomy_inventory.iterrows():
    variable = inventory_row['Variable']
    raw_values = parental_autonomy_raw[variable]
    labelled_values = parental_autonomy_labelled[variable].astype('string')
    observed_mask = raw_values.ge(0).fillna(False).astype(bool)
    structural_mask = raw_values.eq(-91.0).fillna(False).astype(bool)
    other_special_mask = raw_values.lt(0).fillna(False).astype(bool) & ~structural_mask
    parental_autonomy_observed[variable] = raw_values.where(observed_mask)
    parental_autonomy_coverage_rows.append({'Variable': variable, 'Variable label': inventory_row['Variable label'],
        'Observed responses': int(observed_mask.sum()), 'Observed percentage': round(observed_mask.mean() * 100,
        2), 'Structurally not applicable': int(structural_mask.sum()), 'Other special-code responses': int(other_special_mask.sum()), 'No source record': int(raw_values.isna().sum()), 'Observed categories': int(raw_values.loc[observed_mask].nunique())})
    for raw_code, participants in raw_values.value_counts(dropna=False).items():
        if pd.isna(raw_code):
            value_label = 'No source record'
            response_type = 'Unavailable'
            sort_value = 999999
        else:
            response_mask = raw_values.eq(raw_code)
            matching_labels = labelled_values.loc[response_mask].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            if raw_code >= 0:
                response_type = 'Observed response'
            elif raw_code == -91:
                response_type = 'Structural non-applicability'
            else:
                response_type = 'Other special code'
            sort_value = float(raw_code)
        parental_autonomy_code_rows.append({'Variable': variable, 'Raw code': raw_code, 'Value label': value_label,
            'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
parental_autonomy_coverage_summary = pd.DataFrame(parental_autonomy_coverage_rows)
parental_autonomy_code_distribution = pd.DataFrame(parental_autonomy_code_rows).sort_values(['Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
parental_autonomy_availability = pd.DataFrame({'Mother item available': parental_autonomy_observed['W1mothdecYP'].notna(),
    'Father item available': parental_autonomy_observed['W1fathdecYP'].notna()}, index=route_index)
parental_autonomy_availability_patterns = parental_autonomy_availability.value_counts(dropna=False).rename('Participants').reset_index()
parental_autonomy_availability_patterns['Percentage'] = (parental_autonomy_availability_patterns['Participants'] / len(route_index) * 100).round(2)
parental_autonomy_pair = pd.DataFrame({'Mother autonomy support': parental_autonomy_observed['W1mothdecYP'],
    'Father autonomy support': parental_autonomy_observed['W1fathdecYP']}, index=route_index).dropna()
assert len(parental_autonomy_pair) > 0
parental_autonomy_pair_summary = pd.DataFrame([{'Complete comparisons': len(parental_autonomy_pair),
    'Exact agreement percentage': round(parental_autonomy_pair['Mother autonomy support'].eq(parental_autonomy_pair['Father autonomy support']).mean() * 100,
    2), "Cramer's V": round(categorical_cramers_v(parental_autonomy_pair['Mother autonomy support'],
    parental_autonomy_pair['Father autonomy support']), 3), 'Spearman correlation': round(float(spearmanr(parental_autonomy_pair['Mother autonomy support'],
    parental_autonomy_pair['Father autonomy support']).statistic), 3)}])
parental_autonomy_pair_counts = pd.crosstab(parental_autonomy_pair['Mother autonomy support'],
    parental_autonomy_pair['Father autonomy support'], margins=True, dropna=False)
parental_autonomy_pair_row_percentages = (pd.crosstab(parental_autonomy_pair['Mother autonomy support'],
    parental_autonomy_pair['Father autonomy support'], normalize='index', dropna=False) * 100).round(2)
print(f'Parental autonomy-support variables reviewed: {len(parental_autonomy_inventory)}')
print('Variable inventory:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_autonomy_inventory[['Wave', 'Source type', 'Source file', 'Variable', 'Variable label',
        'Timing status', 'Domain 9 review track']])
print('Coverage summary:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_autonomy_coverage_summary)
print('Response-code distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_autonomy_code_distribution)
print('Mother and father item availability:')
display_limited(parental_autonomy_availability_patterns)
print('Mother and father response comparison:')
display_limited(parental_autonomy_pair_summary)
print('Response cross-tabulation:')
display_limited(parental_autonomy_pair_counts)
print('Row percentages:')
display_limited(parental_autonomy_pair_row_percentages)
print('Files written in this cell: 0')

Parental autonomy-support variables reviewed: 2
Variable inventory:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Domain 9 review track
0,Wave 1,Young person,wave_one_lsype_young_person_2020,W1mothdecYP,YP: How true it is to say (step-)mother likes YP to make own decisions,Pre-transition source,Parental autonomy support
1,Wave 1,Young person,wave_one_lsype_young_person_2020,W1fathdecYP,YP: How true it is to say (step-)father likes YP to make own decisions,Pre-transition source,Parental autonomy support


Coverage summary:


,Variable,Variable label,Observed responses,Observed percentage,Structurally not applicable,Other special-code responses,No source record,Observed categories
0,W1mothdecYP,YP: How true it is to say (step-)mother likes YP to make own decisions,8314,85.12,253,957,243,3
1,W1fathdecYP,YP: How true it is to say (step-)father likes YP to make own decisions,6657,68.16,1941,926,243,3


Response-code distributions:


...

Response cross-tabulation:


Father autonomy support,1.0,2.0,3.0,All
Mother autonomy support,,,,
1.0,294,85,69,448
2.0,167,2252,540,2959
3.0,46,341,2555,2942
All,507,2678,3164,6349


Row percentages:


Father autonomy support,1.0,2.0,3.0
Mother autonomy support,,,
1.0,65.62,18.97,15.40
2.0,5.64,76.11,18.25
3.0,1.56,11.59,86.85


Files written in this cell: 0


In [364]:
# 45: Parental autonomy-support representation review

import pandas as pd
from scipy.stats import spearmanr
parental_autonomy_support_recode = {1.0: 0.0, 2.0: 1.0, 3.0: 2.0}
mother_autonomy_support = parental_autonomy_observed['W1mothdecYP'].map(parental_autonomy_support_recode).astype('Float64').rename('mother_autonomy_support')
father_autonomy_support = parental_autonomy_observed['W1fathdecYP'].map(parental_autonomy_support_recode).astype('Float64').rename('father_autonomy_support')
parental_autonomy_contributor_count = pd.DataFrame({'Mother': mother_autonomy_support,
    'Father': father_autonomy_support}, index=route_index).notna().sum(axis=1).astype('Int64').rename('parental_autonomy_contributor_count')
parental_autonomy_support_score_candidate = pd.DataFrame({'Mother': mother_autonomy_support,
    'Father': father_autonomy_support}, index=route_index).mean(axis=1,
    skipna=True).where(parental_autonomy_contributor_count.gt(0)).astype('Float64').rename('parental_autonomy_support_score_pretransition')
assert set(parental_autonomy_support_score_candidate.dropna().astype(float).unique()).issubset({0.0, 0.5, 1.0, 1.5,
    2.0})
parental_autonomy_candidate_summary = pd.DataFrame([{'Candidate predictor': 'parental_autonomy_support_score_pretransition',
    'Non-missing': int(parental_autonomy_support_score_candidate.notna().sum()), 'Missing': int(parental_autonomy_support_score_candidate.isna().sum()), 'Missing percentage': round(parental_autonomy_support_score_candidate.isna().mean() * 100,
    2), 'Distinct values': int(parental_autonomy_support_score_candidate.nunique()), 'Minimum': float(parental_autonomy_support_score_candidate.min()), 'Median': float(parental_autonomy_support_score_candidate.median()), 'Mean': round(float(parental_autonomy_support_score_candidate.mean()),
    2), 'Maximum': float(parental_autonomy_support_score_candidate.max())}])
parental_autonomy_contributor_summary = parental_autonomy_contributor_count.value_counts().sort_index().rename('Participants').rename_axis('Contributing parents').reset_index()
parental_autonomy_contributor_summary['Percentage of full sample'] = (parental_autonomy_contributor_summary['Participants'] / len(route_index) * 100).round(2)
parental_autonomy_candidate_distribution = parental_autonomy_support_score_candidate.value_counts(dropna=False).sort_index(na_position='last').rename('Participants').rename_axis('Autonomy-support score').reset_index()
observed_autonomy_count = int(parental_autonomy_support_score_candidate.notna().sum())
parental_autonomy_candidate_distribution['Percentage among observed'] = (parental_autonomy_candidate_distribution['Participants'].where(parental_autonomy_candidate_distribution['Autonomy-support score'].notna()) / observed_autonomy_count * 100).round(2)
parental_autonomy_by_contributor_rows = []
for contributor_count in [1, 2]:
    contributor_values = parental_autonomy_support_score_candidate.loc[parental_autonomy_contributor_count.eq(contributor_count)].dropna().astype(float)
    parental_autonomy_by_contributor_rows.append({'Contributing parents': contributor_count,
        'Participants': len(contributor_values), 'Mean score': round(float(contributor_values.mean()),
        3), 'Median score': float(contributor_values.median()), 'Standard deviation': round(float(contributor_values.std()),
        3)})
parental_autonomy_by_contributor_summary = pd.DataFrame(parental_autonomy_by_contributor_rows)
parental_autonomy_parent_specific_rows = []
for parent_label, parent_values in [('Mother autonomy support', mother_autonomy_support), ('Father autonomy support',
    father_autonomy_support)]:
    pair_data = pd.DataFrame({'Available-parent mean': parental_autonomy_support_score_candidate,
        parent_label: parent_values}, index=route_index).dropna()
    parental_autonomy_parent_specific_rows.append({'Comparison measure': parent_label,
        'Complete comparisons': len(pair_data), 'Spearman correlation': round(float(spearmanr(pair_data['Available-parent mean'],
        pair_data[parent_label]).statistic), 3)})
parental_autonomy_parent_specific_summary = pd.DataFrame(parental_autonomy_parent_specific_rows)
parental_autonomy_overlap_data = pd.DataFrame({'Parental autonomy-support score': parental_autonomy_support_score_candidate,
    'Parent relationship-quality score': parent_relationship_quality_score_pretransition, 'Parent communication-frequency score': parent_communication_frequency_score_pretransition, 'Parent discussion of school day': parent_school_day_discussion_frequency_pretransition, 'Parental knowledge of evening whereabouts': parental_knowledge_of_evening_whereabouts_pretransition, 'School-night curfew': school_night_curfew_pretransition, 'Homework monitoring frequency': homework_monitoring_frequency_pretransition, 'Parental educational aspiration': parental_educational_aspiration_candidate, 'Parental HE expectation': parental_he_expectation_candidate, 'Parental financial-support profile': parental_educational_financial_support_profile_pretransition, 'School attitude score': parental_homework_school_overlap_data['school_attitude_score']}, index=route_index)
parental_autonomy_overlap_rows = []
for comparison_measure in ['Parent relationship-quality score', 'Parent communication-frequency score',
    'Parent discussion of school day', 'Parental knowledge of evening whereabouts', 'School-night curfew', 'Homework monitoring frequency', 'Parental educational aspiration', 'Parental HE expectation', 'Parental financial-support profile', 'School attitude score']:
    pair_data = parental_autonomy_overlap_data[['Parental autonomy-support score', comparison_measure]].dropna()
    assert len(pair_data) > 0
    parental_autonomy_overlap_rows.append({'Comparison measure': comparison_measure,
        'Complete comparisons': len(pair_data), 'Spearman correlation': round(float(spearmanr(pair_data['Parental autonomy-support score'],
        pair_data[comparison_measure]).statistic), 3), "Cramer's V": round(categorical_cramers_v(pair_data['Parental autonomy-support score'],
        pair_data[comparison_measure]), 3)})
parental_autonomy_overlap_summary = pd.DataFrame(parental_autonomy_overlap_rows).assign(Absolute_correlation=lambda data: data['Spearman correlation'].abs()).sort_values('Absolute_correlation',
    ascending=False).drop(columns=['Absolute_correlation']).reset_index(drop=True)
print('Candidate predictor summary:')
display_limited(parental_autonomy_candidate_summary)
print('Number of contributing parent reports:')
display_limited(parental_autonomy_contributor_summary)
print('Candidate score distribution:')
display_limited(parental_autonomy_candidate_distribution)
print('Score distribution by contributing-parent count:')
display_limited(parental_autonomy_by_contributor_summary)
print('Association with parent-specific items:')
display_limited(parental_autonomy_parent_specific_summary)
print('Associations with retained predictors:')
display_limited(parental_autonomy_overlap_summary)
print('Files written in this cell: 0')

Candidate predictor summary:


,Candidate predictor,Non-missing,Missing,Missing percentage,Distinct values,Minimum,Median,Mean,Maximum
0,parental_autonomy_support_score_pretransition,8622,1145,11.72,5,0.0,1.5,1.4,2.0


Number of contributing parent reports:


,Contributing parents,Participants,Percentage of full sample
0,0,1145,11.72
1,1,2273,23.27
2,2,6349,65.0


Candidate score distribution:


...

Association with parent-specific items:


,Comparison measure,Complete comparisons,Spearman correlation
0,Mother autonomy support,8314,0.936
1,Father autonomy support,6657,0.920


Associations with retained predictors:


,Comparison measure,Complete comparisons,Spearman correlation,Cramer's V
0,Parent relationship-quality score,8572,0.159,0.128
1,Parent communication-frequency score,8460,0.130,0.115
2,Parent discussion of school day,8445,0.119,0.101
3,School attitude score,8622,0.078,0.128
4,Parental knowledge of evening whereabouts,8589,0.067,0.052


Files written in this cell: 0


In [365]:
# 46: Parental autonomy-support decisions

import pandas as pd
parental_autonomy_support_score_pretransition = parental_autonomy_support_score_candidate.copy().astype('Float64').rename('parental_autonomy_support_score_pretransition')
assert parental_autonomy_support_score_pretransition.index.equals(route_index)
observed_autonomy_values = set(parental_autonomy_support_score_pretransition.dropna().astype(float).unique().tolist())
allowed_autonomy_values = {0.0, 0.5, 1.0, 1.5, 2.0}
assert observed_autonomy_values.issubset(allowed_autonomy_values)
parental_autonomy_predictors = pd.DataFrame({'NSID': route_index,
    'parental_autonomy_support_score_pretransition': parental_autonomy_support_score_pretransition.to_numpy()})
assert len(parental_autonomy_predictors) == 9767
assert parental_autonomy_predictors['NSID'].is_unique
parental_autonomy_variable_decisions = parental_autonomy_inventory.copy()
parental_autonomy_variable_decisions['Autonomy-support decision'] = 'Construction input'
parental_autonomy_variable_decisions['Autonomy-support role'] = pd.NA
parental_autonomy_variable_decisions['Autonomy-support reason'] = 'The mother and father items are combined using the mean of available parent reports. Higher values indicate stronger support for the young person making their own decisions'
mother_autonomy_mask = parental_autonomy_variable_decisions['Variable'].eq('W1mothdecYP')
father_autonomy_mask = parental_autonomy_variable_decisions['Variable'].eq('W1fathdecYP')
parental_autonomy_variable_decisions.loc[mother_autonomy_mask,
    'Autonomy-support role'] = 'Mother or stepmother autonomy-support input'
parental_autonomy_variable_decisions.loc[father_autonomy_mask,
    'Autonomy-support role'] = 'Father or stepfather autonomy-support input'
assert parental_autonomy_variable_decisions['Autonomy-support role'].notna().all()
assert parental_autonomy_variable_decisions['Autonomy-support reason'].notna().all()
parental_autonomy_representation_decisions = pd.DataFrame([{'Representation': 'Separate mother and father autonomy-support predictors',
    'Decision': 'Do not retain', 'Reason': 'The two items show high agreement and strong association, while structural availability differs according to family composition'}, {'Representation': 'Available-parent mean autonomy-support score',
    'Decision': 'Retain', 'Reason': 'The score combines closely related parent reports, preserves information from participants with one available report and has limited overlap with the existing predictor set'}, {'Representation': 'Complete-parent-pair mean only',
    'Decision': 'Do not construct', 'Reason': 'Restricting the score to two complete parent reports would unnecessarily exclude participants with one valid report'}, {'Representation': 'Number of contributing parent reports',
    'Decision': 'Do not retain as predictor', 'Reason': 'The count documents construction but would partly repeat family-composition information represented elsewhere'}])
parental_autonomy_predictor_summary = pd.DataFrame([{'Predictor': 'parental_autonomy_support_score_pretransition',
    'Non-missing': int(parental_autonomy_support_score_pretransition.notna().sum()), 'Missing': int(parental_autonomy_support_score_pretransition.isna().sum()), 'Missing percentage': round(parental_autonomy_support_score_pretransition.isna().mean() * 100,
    2), 'Distinct values': int(parental_autonomy_support_score_pretransition.nunique()), 'Mean': round(float(parental_autonomy_support_score_pretransition.mean()),
    2), 'Median': float(parental_autonomy_support_score_pretransition.median()), 'Maximum absolute Spearman correlation': round(float(parental_autonomy_overlap_summary['Spearman correlation'].abs().max()),
    3), "Maximum Cramer's V": round(float(parental_autonomy_overlap_summary["Cramer's V"].max()), 3)}])
parental_autonomy_distribution_rows = []
observed_autonomy_count = int(parental_autonomy_support_score_pretransition.notna().sum())
for score_value in sorted(allowed_autonomy_values):
    participants = int(parental_autonomy_support_score_pretransition.eq(score_value).sum())
    parental_autonomy_distribution_rows.append({'Autonomy-support score': score_value, 'Participants': participants,
        'Percentage among observed': round(participants / observed_autonomy_count * 100, 2)})
parental_autonomy_distribution_rows.append({'Autonomy-support score': pd.NA,
    'Participants': int(parental_autonomy_support_score_pretransition.isna().sum()), 'Percentage among observed': pd.NA})
parental_autonomy_retained_distribution = pd.DataFrame(parental_autonomy_distribution_rows)
parental_autonomy_retained_contributor_summary = parental_autonomy_contributor_count.value_counts().sort_index().rename('Participants').rename_axis('Contributing parents').reset_index()
parental_autonomy_retained_contributor_summary['Percentage of full sample'] = (parental_autonomy_retained_contributor_summary['Participants'] / len(route_index) * 100).round(2)
parental_autonomy_decision_display = parental_autonomy_variable_decisions[['Wave', 'Source file', 'Variable',
    'Variable label', 'Timing status', 'Autonomy-support decision', 'Autonomy-support role', 'Autonomy-support reason']].copy()
print('Parental autonomy-support predictors retained: 1')
print('Retained predictor summary:')
display_limited(parental_autonomy_predictor_summary)
print('Retained score distribution:')
display_limited(parental_autonomy_retained_distribution)
print('Number of contributing parent reports:')
display_limited(parental_autonomy_retained_contributor_summary)
print('Representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_autonomy_representation_decisions)
print('Variable decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_autonomy_decision_display)
print('Files written in this cell: 0')

Parental autonomy-support predictors retained: 1
Retained predictor summary:


,Predictor,Non-missing,Missing,Missing percentage,Distinct values,Mean,Median,Maximum absolute Spearman correlation,Maximum Cramer's V
0,parental_autonomy_support_score_pretransition,8622,1145,11.72,5,1.4,1.5,0.159,0.128


Retained score distribution:


,Autonomy-support score,Participants,Percentage among observed
0,0.0,483,5.6
1,0.5,252,2.92
2,1.0,3414,39.6
3,1.5,881,10.22
4,2.0,3592,41.66


Number of contributing parent reports:


...

Representation decisions:


,Representation,Decision,Reason
0,Separate mother and father autonomy-support predictors,Do not retain,"The two items show high agreement and strong association, while structural availability differs according to family composition"
1,Available-parent mean autonomy-support score,Retain,"The score combines closely related parent reports, preserves information from participants with one available report and has limited overlap with the existing predictor set"
2,Complete-parent-pair mean only,Do not construct,Restricting the score to two complete parent reports would unnecessarily exclude participants with one valid report
3,Number of contributing parent reports,Do not retain as predictor,The count documents construction but would partly repeat family-composition information represented elsewhere


Variable decisions:


,Wave,Source file,Variable,Variable label,Timing status,Autonomy-support decision,Autonomy-support role,Autonomy-support reason
0,Wave 1,wave_one_lsype_young_person_2020,W1mothdecYP,YP: How true it is to say (step-)mother likes YP to make own decisions,Pre-transition source,Construction input,Mother or stepmother autonomy-support input,The mother and father items are combined using the mean of available parent reports. Higher values indicate stronger support for the young person making their own decisions
1,Wave 1,wave_one_lsype_young_person_2020,W1fathdecYP,YP: How true it is to say (step-)father likes YP to make own decisions,Pre-transition source,Construction input,Father or stepfather autonomy-support input,The mother and father items are combined using the mean of available parent reports. Higher values indicate stronger support for the young person making their own decisions


Files written in this cell: 0


In [366]:
# 47: Post-16 route discussion track inspection

import pandas as pd
domain_9_post16_route_track_counts = domain_9_screened_candidates['Domain 9 review track'].astype('string').value_counts(dropna=False).rename('Variables').rename_axis('Domain 9 review track').reset_index()
post16_route_discussion_track_matches = domain_9_post16_route_track_counts.loc[domain_9_post16_route_track_counts['Domain 9 review track'].astype('string').str.contains('post.?16|route discussion|apprentice|training',
    case=False, na=False, regex=True)].reset_index(drop=True)
post16_route_discussion_variable_matches = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 review track'].astype('string').str.contains('post.?16|route discussion|apprentice|training',
    case=False, na=False, regex=True)].sort_values(['Domain 9 review track', 'Source order',
    'Variable position'])[['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status', 'Domain 9 screening status', 'Domain 9 review track']].reset_index(drop=True)
print('Possible post-16 route-discussion review tracks:')
display_limited(post16_route_discussion_track_matches)
print('Variables in matching tracks:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(post16_route_discussion_variable_matches)
print('Files written in this cell: 0')

Possible post-16 route-discussion review tracks:


,Domain 9 review track,Variables
0,Parental post-16 route discussion,1


Variables in matching tracks:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Domain 9 screening status,Domain 9 review track
0,Wave 2,Young person,wave_two_lsype_young_person_2020,326,W2ModAp3YP0a,YP: Who YP has talked to about training or an apprenticeship - Parents,Pre-transition source,Core review,Parental post-16 route discussion


Files written in this cell: 0


In [367]:
# 48: Parental post-16 route-discussion routing review

import pandas as pd
parental_post16_route_discussion_inventory = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 review track'].eq('Parental post-16 route discussion')].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(parental_post16_route_discussion_inventory) == 1
assert parental_post16_route_discussion_inventory.loc[0, 'Variable'] == 'W2ModAp3YP0a'
parental_post16_source_file = parental_post16_route_discussion_inventory.loc[0, 'Source file']
parental_post16_source_path = source_file_lookup[parental_post16_source_file]
parental_post16_variable_position = int(parental_post16_route_discussion_inventory.loc[0, 'Variable position'])
master_variable_register_path = master_register_output_path
stage_2_master_register_review = stage_2_master_variable_register.copy()
required_master_columns = {'Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label'}
assert required_master_columns.issubset(stage_2_master_register_review.columns)
parental_post16_neighbourhood = stage_2_master_register_review.loc[stage_2_master_register_review['Source file'].eq(parental_post16_source_file) & stage_2_master_register_review['Variable position'].between(parental_post16_variable_position - 8,
    parental_post16_variable_position + 8)].sort_values('Variable position').reset_index(drop=True)
parental_post16_question_family = stage_2_master_register_review.loc[stage_2_master_register_review['Source file'].eq(parental_post16_source_file) & (stage_2_master_register_review['Variable'].astype('string').str.contains('ModAp',
    case=False, na=False, regex=True) | stage_2_master_register_review['Variable label'].astype('string').str.contains('training|apprentice',
    case=False, na=False, regex=True))].sort_values('Variable position').reset_index(drop=True)
routing_review_variables = parental_post16_question_family['Variable'].dropna().astype(str).drop_duplicates().tolist()
if 'W2ModAp3YP0a' not in routing_review_variables:
    routing_review_variables.append('W2ModAp3YP0a')
source_variable_names = set(stage_2_master_register_review.loc[stage_2_master_register_review['Source file'].eq(parental_post16_source_file),
    'Variable'].astype(str))
routing_review_variables = [variable for variable in routing_review_variables if variable in source_variable_names]
assert 'W2ModAp3YP0a' in routing_review_variables
parental_post16_raw = pd.read_stata(parental_post16_source_path, columns=['NSID', *routing_review_variables],
    convert_categoricals=False)
parental_post16_labelled = pd.read_stata(parental_post16_source_path, columns=['NSID', *routing_review_variables],
    convert_categoricals=True)
for data in [parental_post16_raw, parental_post16_labelled]:
    data['NSID'] = standardise_nsid(data['NSID'])
    assert data['NSID'].is_unique
parental_post16_raw = parental_post16_raw.set_index('NSID').reindex(route_index)
parental_post16_labelled = parental_post16_labelled.set_index('NSID').reindex(route_index)
for variable in routing_review_variables:
    parental_post16_raw[variable] = pd.to_numeric(parental_post16_raw[variable], errors='coerce')
parental_post16_target_raw = parental_post16_raw['W2ModAp3YP0a']
parental_post16_target_labelled = parental_post16_labelled['W2ModAp3YP0a'].astype('string')
parental_post16_target_observed_mask = parental_post16_target_raw.ge(0).fillna(False).astype(bool)
parental_post16_target_structural_mask = parental_post16_target_raw.eq(-91.0).fillna(False).astype(bool)
parental_post16_target_other_special_mask = parental_post16_target_raw.lt(0).fillna(False).astype(bool) & ~parental_post16_target_structural_mask
parental_post16_target_coverage = pd.DataFrame([{'Variable': 'W2ModAp3YP0a',
    'Variable label': parental_post16_route_discussion_inventory.loc[0,
    'Variable label'], 'Observed responses': int(parental_post16_target_observed_mask.sum()), 'Observed percentage': round(parental_post16_target_observed_mask.mean() * 100,
    2), 'Structurally not applicable': int(parental_post16_target_structural_mask.sum()), 'Other special-code responses': int(parental_post16_target_other_special_mask.sum()), 'No source record': int(parental_post16_target_raw.isna().sum()), 'Observed categories': int(parental_post16_target_raw.loc[parental_post16_target_observed_mask].nunique())}])
parental_post16_target_code_rows = []
for raw_code, participants in parental_post16_target_raw.value_counts(dropna=False).items():
    if pd.isna(raw_code):
        value_label = 'No source record'
        response_type = 'Unavailable'
        sort_value = 999999
    else:
        response_mask = parental_post16_target_raw.eq(raw_code)
        matching_labels = parental_post16_target_labelled.loc[response_mask].dropna().drop_duplicates().tolist()
        value_label = matching_labels[0] if matching_labels else str(raw_code)
        if raw_code >= 0:
            response_type = 'Observed response'
        elif raw_code == -91:
            response_type = 'Structural non-applicability'
        else:
            response_type = 'Other special code'
        sort_value = float(raw_code)
    parental_post16_target_code_rows.append({'Raw code': raw_code, 'Value label': value_label,
        'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
parental_post16_target_code_distribution = pd.DataFrame(parental_post16_target_code_rows).sort_values('Sort value').drop(columns=['Sort value']).reset_index(drop=True)
parental_post16_question_family_rows = []
for variable in routing_review_variables:
    raw_values = parental_post16_raw[variable]
    observed_mask = raw_values.ge(0).fillna(False).astype(bool)
    structural_mask = raw_values.eq(-91.0).fillna(False).astype(bool)
    variable_metadata = stage_2_master_register_review.loc[stage_2_master_register_review['Source file'].eq(parental_post16_source_file) & stage_2_master_register_review['Variable'].eq(variable)].iloc[0]
    parental_post16_question_family_rows.append({'Variable position': int(variable_metadata['Variable position']),
        'Variable': variable, 'Variable label': variable_metadata['Variable label'], 'Observed responses': int(observed_mask.sum()), 'Structurally not applicable': int(structural_mask.sum()), 'Other special-code responses': int((raw_values.lt(0).fillna(False).astype(bool) & ~structural_mask).sum()), 'No source record': int(raw_values.isna().sum()), 'Observed categories': int(raw_values.loc[observed_mask].nunique())})
parental_post16_question_family_summary = pd.DataFrame(parental_post16_question_family_rows).sort_values('Variable position').reset_index(drop=True)
print('Variables surrounding the target item:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_post16_neighbourhood[['Variable position', 'Variable', 'Variable label']])
print('Training and apprenticeship question family:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_post16_question_family[['Variable position', 'Variable', 'Variable label']])
print('Target-variable coverage:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_post16_target_coverage)
print('Target-variable response codes:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_post16_target_code_distribution)
print('Question-family response structure:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(parental_post16_question_family_summary)
print('Files written in this cell: 0')

Variables surrounding the target item:


,Variable position,Variable,Variable label
0,318,W2Pladk2YP0e,YP: What expect to do at age 16 other than stay in FTE - Something else
1,319,W2Pladk2YP0f,YP: What expect to do at age 16 other than stay in FTE - Don't know
2,320,W2modap4bYP,YP: Whether YP has specific job in mind
3,321,W2ModAp6bYP,YP: What job or trade YP wants to do or train for
4,322,W2fplan16YP,YP: What think most of friends will do after Year 11


Training and apprenticeship question family:


,Variable position,Variable,Variable label
0,299,W2yleav16YP0b,YP: Why YP wants to leave school at 16 rather than stay on - To do an apprentice
1,320,W2modap4bYP,YP: Whether YP has specific job in mind
2,321,W2ModAp6bYP,YP: What job or trade YP wants to do or train for
3,325,W2modap2YP,YP: Whether YP has talked to anyone about getting training or apprenticeship aft
4,326,W2ModAp3YP0a,YP: Who YP has talked to about training or an apprenticeship - Parents


Target-variable coverage:


...

Target-variable response codes:


,Raw code,Value label,Response type,Participants
0,-99.0,YP not interviewed,Other special code,76
1,-91.0,Not applicable,Structural non-applicability,7635
2,0.0,Not mentioned,Observed response,704
3,1.0,Mentioned,Observed response,1106
4,NaN,No source record,Unavailable,246


Question-family response structure:


,Variable position,Variable,Variable label,Observed responses,Structurally not applicable,Other special-code responses,No source record,Observed categories
0,299,W2yleav16YP0b,YP: Why YP wants to leave school at 16 rather than stay on - To do an apprentice,877,8568,76,246,2
1,320,W2modap4bYP,YP: Whether YP has specific job in mind,742,8696,83,246,2
2,321,W2ModAp6bYP,YP: What job or trade YP wants to do or train for,0,0,0,9767,0
3,325,W2modap2YP,YP: Whether YP has talked to anyone about getting training or apprenticeship aft,9361,0,160,246,2
4,326,W2ModAp3YP0a,YP: Who YP has talked to about training or an apprenticeship - Parents,1810,7635,76,246,2


Files written in this cell: 0


In [368]:
# 49: Parental training and apprenticeship discussion routing diagnosis

import pandas as pd
training_discussion_routing_raw = parental_post16_raw['W2modap2YP']
training_discussion_routing_labelled = parental_post16_labelled['W2modap2YP'].astype('string')
parent_training_discussion_raw = parental_post16_raw['W2ModAp3YP0a']
parent_training_discussion_labelled = parental_post16_labelled['W2ModAp3YP0a'].astype('string')
routing_code_rows = []
for raw_code, participants in training_discussion_routing_raw.value_counts(dropna=False).items():
    if pd.isna(raw_code):
        value_label = 'No source record'
        response_type = 'Unavailable'
        sort_value = 999999
    else:
        response_mask = training_discussion_routing_raw.eq(raw_code)
        matching_labels = training_discussion_routing_labelled.loc[response_mask].dropna().drop_duplicates().tolist()
        value_label = matching_labels[0] if matching_labels else str(raw_code)
        response_type = 'Observed response' if raw_code >= 0 else 'Special code'
        sort_value = float(raw_code)
    routing_code_rows.append({'Routing raw code': raw_code, 'Routing label': value_label,
        'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
training_discussion_routing_code_distribution = pd.DataFrame(routing_code_rows).sort_values('Sort value').drop(columns=['Sort value']).reset_index(drop=True)
parent_option_code_rows = []
for raw_code, participants in parent_training_discussion_raw.value_counts(dropna=False).items():
    if pd.isna(raw_code):
        value_label = 'No source record'
        response_type = 'Unavailable'
        sort_value = 999999
    else:
        response_mask = parent_training_discussion_raw.eq(raw_code)
        matching_labels = parent_training_discussion_labelled.loc[response_mask].dropna().drop_duplicates().tolist()
        value_label = matching_labels[0] if matching_labels else str(raw_code)
        if raw_code >= 0:
            response_type = 'Observed response'
        elif raw_code == -91:
            response_type = 'Structural non-applicability'
        else:
            response_type = 'Other special code'
        sort_value = float(raw_code)
    parent_option_code_rows.append({'Parent-option raw code': raw_code, 'Parent-option label': value_label,
        'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
parent_training_discussion_code_distribution = pd.DataFrame(parent_option_code_rows).sort_values('Sort value').drop(columns=['Sort value']).reset_index(drop=True)
training_parent_discussion_raw_crosstab = pd.crosstab(training_discussion_routing_raw.fillna(999999),
    parent_training_discussion_raw.fillna(999999), margins=True, dropna=False)
training_parent_discussion_labelled_crosstab = pd.crosstab(training_discussion_routing_labelled.fillna('No source record'),
    parent_training_discussion_labelled.fillna('No source record'), margins=True, dropna=False)
parent_option_structural_mask = parent_training_discussion_raw.eq(-91.0).fillna(False).astype(bool)
routing_values_when_parent_option_structural = pd.DataFrame({'Routing raw code': training_discussion_routing_raw.loc[parent_option_structural_mask],
    'Routing label': training_discussion_routing_labelled.loc[parent_option_structural_mask]})
routing_values_when_parent_option_structural_summary = routing_values_when_parent_option_structural.value_counts(dropna=False).rename('Participants').reset_index().sort_values(['Routing raw code',
    'Routing label'], na_position='last').reset_index(drop=True)
parent_option_observed_mask = parent_training_discussion_raw.isin([0.0, 1.0]).fillna(False).astype(bool)
routing_values_when_parent_option_observed_summary = pd.DataFrame({'Routing raw code': training_discussion_routing_raw.loc[parent_option_observed_mask],
    'Routing label': training_discussion_routing_labelled.loc[parent_option_observed_mask], 'Parent-option raw code': parent_training_discussion_raw.loc[parent_option_observed_mask], 'Parent-option label': parent_training_discussion_labelled.loc[parent_option_observed_mask]}).value_counts(dropna=False).rename('Participants').reset_index().sort_values(['Routing raw code',
    'Parent-option raw code']).reset_index(drop=True)
print('Routing-item response codes:')
with pd.option_context('display.max_colwidth', None):
    display_limited(training_discussion_routing_code_distribution)
print('Parent-option response codes:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parent_training_discussion_code_distribution)
print('Raw-code cross-tabulation:')
display_limited(training_parent_discussion_raw_crosstab)
print('Labelled-response cross-tabulation:')
with pd.option_context('display.max_colwidth', None):
    display_limited(training_parent_discussion_labelled_crosstab)
print('Routing responses when the parent option is structurally not applicable:')
with pd.option_context('display.max_colwidth', None):
    display_limited(routing_values_when_parent_option_structural_summary)
print('Routing responses when the parent option is observed:')
with pd.option_context('display.max_colwidth', None):
    display_limited(routing_values_when_parent_option_observed_summary)
print('Files written in this cell: 0')

Routing-item response codes:


,Routing raw code,Routing label,Response type,Participants
0,-99.0,YP not interviewed,Special code,76
1,-92.0,Refused,Special code,2
2,-1.0,Don't Know,Special code,82
3,1.0,Yes,Observed response,1810
4,2.0,No,Observed response,7551


Parent-option response codes:


,Parent-option raw code,Parent-option label,Response type,Participants
0,-99.0,YP not interviewed,Other special code,76
1,-91.0,Not applicable,Structural non-applicability,7635
2,0.0,Not mentioned,Observed response,704
3,1.0,Mentioned,Observed response,1106
4,NaN,No source record,Unavailable,246


Raw-code cross-tabulation:


...

Routing responses when the parent option is structurally not applicable:


,Routing raw code,Routing label,Participants
0,-92.0,Refused,2
1,-1.0,Don't Know,82
2,2.0,No,7551


Routing responses when the parent option is observed:


,Routing raw code,Routing label,Parent-option raw code,Parent-option label,Participants
0,1.0,Yes,0.0,Not mentioned,704
1,1.0,Yes,1.0,Mentioned,1106


Files written in this cell: 0


In [369]:
# 50: Parental training and apprenticeship discussion representation review

import pandas as pd
from scipy.stats import spearmanr
training_discussion_routing_raw = parental_post16_raw['W2modap2YP']
parent_training_discussion_raw = parental_post16_raw['W2ModAp3YP0a']
training_discussion_yes_mask = training_discussion_routing_raw.eq(1.0).fillna(False).astype(bool)
training_discussion_no_mask = training_discussion_routing_raw.eq(2.0).fillna(False).astype(bool)
training_discussion_uncertain_mask = training_discussion_routing_raw.isin([-92.0, -1.0]).fillna(False).astype(bool)
parent_not_mentioned_mask = parent_training_discussion_raw.eq(0.0).fillna(False).astype(bool)
parent_mentioned_mask = parent_training_discussion_raw.eq(1.0).fillna(False).astype(bool)
parent_option_structural_mask = parent_training_discussion_raw.eq(-91.0).fillna(False).astype(bool)
assert parent_option_structural_mask.equals(training_discussion_no_mask | training_discussion_uncertain_mask)
assert (parent_not_mentioned_mask | parent_mentioned_mask).equals(training_discussion_yes_mask)
assert not (parent_not_mentioned_mask & parent_mentioned_mask).any()
any_training_apprenticeship_discussion_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='any_training_apprenticeship_discussion_pretransition')
any_training_apprenticeship_discussion_candidate.loc[training_discussion_no_mask] = 0
any_training_apprenticeship_discussion_candidate.loc[training_discussion_yes_mask] = 1
parental_training_apprenticeship_discussion_profile_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='parental_training_apprenticeship_discussion_profile_pretransition')
parental_training_apprenticeship_discussion_profile_candidate.loc[training_discussion_no_mask] = 0
parental_training_apprenticeship_discussion_profile_candidate.loc[training_discussion_yes_mask & parent_not_mentioned_mask] = 1
parental_training_apprenticeship_discussion_profile_candidate.loc[training_discussion_yes_mask & parent_mentioned_mask] = 2
assert set(any_training_apprenticeship_discussion_candidate.dropna().astype(int).unique()).issubset({0, 1})
assert set(parental_training_apprenticeship_discussion_profile_candidate.dropna().astype(int).unique()).issubset({0, 1,
    2})
assert any_training_apprenticeship_discussion_candidate.notna().equals(parental_training_apprenticeship_discussion_profile_candidate.notna())
parental_training_discussion_routing_summary = pd.DataFrame([{'Routing response': 'Yes',
    'Participants': int(training_discussion_yes_mask.sum()), 'Candidate treatment': 'Classified by whether parents were mentioned'}, {'Routing response': 'No',
    'Participants': int(training_discussion_no_mask.sum()), 'Candidate treatment': 'No discussion with anyone'}, {'Routing response': "Don't know or refused",
    'Participants': int(training_discussion_uncertain_mask.sum()), 'Candidate treatment': 'Missing'}, {'Routing response': 'YP not interviewed',
    'Participants': int(training_discussion_routing_raw.eq(-99.0).sum()), 'Candidate treatment': 'Missing'}, {'Routing response': 'No source record',
    'Participants': int(training_discussion_routing_raw.isna().sum()), 'Candidate treatment': 'Missing'}])
parental_training_discussion_candidate_summary = pd.DataFrame([{'Candidate predictor': 'any_training_apprenticeship_discussion_pretransition',
    'Non-missing': int(any_training_apprenticeship_discussion_candidate.notna().sum()), 'Missing': int(any_training_apprenticeship_discussion_candidate.isna().sum()), 'Missing percentage': round(any_training_apprenticeship_discussion_candidate.isna().mean() * 100,
    2), 'Categories': int(any_training_apprenticeship_discussion_candidate.nunique())}, {'Candidate predictor': 'parental_training_apprenticeship_discussion_profile_pretransition',
    'Non-missing': int(parental_training_apprenticeship_discussion_profile_candidate.notna().sum()), 'Missing': int(parental_training_apprenticeship_discussion_profile_candidate.isna().sum()), 'Missing percentage': round(parental_training_apprenticeship_discussion_profile_candidate.isna().mean() * 100,
    2), 'Categories': int(parental_training_apprenticeship_discussion_profile_candidate.nunique())}])
parental_training_discussion_distribution_rows = []
candidate_distribution_specs = [('Any training or apprenticeship discussion',
    any_training_apprenticeship_discussion_candidate, {0: 'No discussion with anyone',
    1: 'Discussion with someone'}), ('Parental training or apprenticeship discussion profile',
    parental_training_apprenticeship_discussion_profile_candidate, {0: 'No discussion with anyone',
    1: 'Discussion, but parents not mentioned', 2: 'Discussion with parents'})]
for candidate_name, candidate_values, category_labels in candidate_distribution_specs:
    observed_count = int(candidate_values.notna().sum())
    for code, category in category_labels.items():
        participants = int(candidate_values.eq(code).sum())
        parental_training_discussion_distribution_rows.append({'Candidate predictor': candidate_name, 'Code': code,
            'Category': category, 'Participants': participants, 'Percentage among observed': round(participants / observed_count * 100,
            2)})
parental_training_discussion_candidate_distributions = pd.DataFrame(parental_training_discussion_distribution_rows)
parental_training_discussion_overlap_data = pd.DataFrame({'Any training or apprenticeship discussion': any_training_apprenticeship_discussion_candidate,
    'Parental training or apprenticeship discussion profile': parental_training_apprenticeship_discussion_profile_candidate, 'Expected post-16 route': saved_educational_aspiration_predictors['expected_post16_route'], 'Young-person HE application likelihood': saved_educational_aspiration_predictors['higher_education_application_likelihood'], 'Parent–teacher post-16 discussion profile': parent_teacher_post16_discussion_profile_pretransition, 'Parental HE expectation': parental_he_expectation_candidate, 'Parental educational aspiration': parental_educational_aspiration_candidate, 'Parent communication-frequency score': parent_communication_frequency_score_pretransition, 'Parent discussion of school day': parent_school_day_discussion_frequency_pretransition, 'Parental autonomy-support score': parental_autonomy_support_score_pretransition}, index=route_index)
assert len(parental_training_discussion_overlap_data) == 9767
parental_training_discussion_categorical_overlap_rows = []
for candidate_measure in ['Any training or apprenticeship discussion',
    'Parental training or apprenticeship discussion profile']:
    for comparison_measure in ['Expected post-16 route', 'Young-person HE application likelihood',
        'Parent–teacher post-16 discussion profile', 'Parental HE expectation', 'Parental educational aspiration']:
        pair_data = parental_training_discussion_overlap_data[[candidate_measure, comparison_measure]].dropna()
        assert len(pair_data) > 0
        parental_training_discussion_categorical_overlap_rows.append({'Candidate predictor': candidate_measure,
            'Comparison measure': comparison_measure, 'Complete comparisons': len(pair_data), 'Candidate categories': int(pair_data[candidate_measure].nunique()), 'Comparison categories': int(pair_data[comparison_measure].nunique()), "Cramer's V": round(categorical_cramers_v(pair_data[candidate_measure],
            pair_data[comparison_measure]), 3)})
parental_training_discussion_categorical_overlap_summary = pd.DataFrame(parental_training_discussion_categorical_overlap_rows).sort_values(['Candidate predictor',
    "Cramer's V"], ascending=[True, False]).reset_index(drop=True)
parental_training_discussion_ordinal_overlap_rows = []
for candidate_measure in ['Any training or apprenticeship discussion',
    'Parental training or apprenticeship discussion profile']:
    for comparison_measure in ['Parent communication-frequency score', 'Parent discussion of school day',
        'Parental autonomy-support score']:
        pair_data = parental_training_discussion_overlap_data[[candidate_measure, comparison_measure]].dropna()
        assert len(pair_data) > 0
        parental_training_discussion_ordinal_overlap_rows.append({'Candidate predictor': candidate_measure,
            'Comparison measure': comparison_measure, 'Complete comparisons': len(pair_data), 'Spearman correlation': round(float(spearmanr(pair_data[candidate_measure],
            pair_data[comparison_measure]).statistic), 3)})
parental_training_discussion_ordinal_overlap_summary = pd.DataFrame(parental_training_discussion_ordinal_overlap_rows).assign(Absolute_correlation=lambda data: data['Spearman correlation'].abs()).sort_values(['Candidate predictor',
    'Absolute_correlation'], ascending=[True, False]).drop(columns=['Absolute_correlation']).reset_index(drop=True)
parental_training_discussion_route_data = parental_training_discussion_overlap_data[['Parental training or apprenticeship discussion profile',
    'Expected post-16 route']].dropna()
parental_training_discussion_route_counts = pd.crosstab(parental_training_discussion_route_data['Parental training or apprenticeship discussion profile'],
    parental_training_discussion_route_data['Expected post-16 route'], margins=True, dropna=False)
parental_training_discussion_route_row_percentages = (pd.crosstab(parental_training_discussion_route_data['Parental training or apprenticeship discussion profile'],
    parental_training_discussion_route_data['Expected post-16 route'], normalize='index', dropna=False) * 100).round(2)
print('Routing treatment:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_training_discussion_routing_summary)
print('Candidate predictor summary:')
display_limited(parental_training_discussion_candidate_summary)
print('Candidate distributions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_training_discussion_candidate_distributions)
print('Categorical associations with retained predictors:')
display_limited(parental_training_discussion_categorical_overlap_summary)
print('Ordinal associations with retained family practices:')
display_limited(parental_training_discussion_ordinal_overlap_summary)
print('Discussion profile by expected post-16 route:')
display_limited(parental_training_discussion_route_counts)
print('Row percentages:')
display_limited(parental_training_discussion_route_row_percentages)
print('Files written in this cell: 0')

Routing treatment:


,Routing response,Participants,Candidate treatment
0,Yes,1810,Classified by whether parents were mentioned
1,No,7551,No discussion with anyone
2,Don't know or refused,84,Missing
3,YP not interviewed,76,Missing
4,No source record,246,Missing


Candidate predictor summary:


,Candidate predictor,Non-missing,Missing,Missing percentage,Categories
0,any_training_apprenticeship_discussion_pretran...,9361,406,4.16,2
1,parental_training_apprenticeship_discussion_pr...,9361,406,4.16,3


Candidate distributions:


...

Discussion profile by expected post-16 route:


Expected post-16 route,Full-time employment,Further-education college,Other or unspecified full-time education,"Other, mixed or uncertain route",School sixth form,Sixth-form college,Work-based training or employment-training,All
Parental training or apprenticeship discussion profile,,,,,,,,
0,163,1732,320,93,3503,1511,224,7546
1,25,195,30,20,232,147,53,702
2,60,301,71,41,265,154,213,1105
All,248,2228,421,154,4000,1812,490,9353


Row percentages:


Expected post-16 route,Full-time employment,Further-education college,Other or unspecified full-time education,"Other, mixed or uncertain route",School sixth form,Sixth-form college,Work-based training or employment-training
Parental training or apprenticeship discussion profile,,,,,,,
0,2.16,22.95,4.24,1.23,46.42,20.02,2.97
1,3.56,27.78,4.27,2.85,33.05,20.94,7.55
2,5.43,27.24,6.43,3.71,23.98,13.94,19.28


Files written in this cell: 0


In [370]:
# 51: Parental training and apprenticeship discussion decisions

import pandas as pd
parental_training_apprenticeship_discussion_profile_pretransition = parental_training_apprenticeship_discussion_profile_candidate.copy().astype('Int64').rename('parental_training_apprenticeship_discussion_profile_pretransition')
assert parental_training_apprenticeship_discussion_profile_pretransition.index.equals(route_index)
observed_discussion_profile_values = set(parental_training_apprenticeship_discussion_profile_pretransition.dropna().astype(int).unique().tolist())
assert observed_discussion_profile_values.issubset({0, 1, 2})
parental_training_discussion_predictors = pd.DataFrame({'NSID': route_index,
    'parental_training_apprenticeship_discussion_profile_pretransition': parental_training_apprenticeship_discussion_profile_pretransition.to_numpy()})
assert len(parental_training_discussion_predictors) == 9767
assert parental_training_discussion_predictors['NSID'].is_unique
parental_training_discussion_variable_decisions = stage_2_master_register_review.loc[stage_2_master_register_review['Source file'].eq(parental_post16_source_file) & stage_2_master_register_review['Variable'].isin(['W2modap2YP',
    'W2ModAp3YP0a'])].copy().sort_values('Variable position').reset_index(drop=True)
assert set(parental_training_discussion_variable_decisions['Variable']) == {'W2modap2YP', 'W2ModAp3YP0a'}
parental_training_discussion_variable_decisions['Discussion decision'] = 'Construction input'
parental_training_discussion_variable_decisions['Discussion role'] = pd.NA
parental_training_discussion_variable_decisions['Discussion reason'] = pd.NA
routing_item_mask = parental_training_discussion_variable_decisions['Variable'].eq('W2modap2YP')
parent_option_mask = parental_training_discussion_variable_decisions['Variable'].eq('W2ModAp3YP0a')
parental_training_discussion_variable_decisions.loc[routing_item_mask,
    'Discussion role'] = 'Routing input identifying whether any discussion occurred'
parental_training_discussion_variable_decisions.loc[parent_option_mask,
    'Discussion role'] = 'Parent-participation input among young people reporting discussion'
parental_training_discussion_variable_decisions.loc[routing_item_mask,
    'Discussion reason'] = 'The routing item distinguishes no discussion from discussion with at least one person and permits structural non-applicability in the parent option to be interpreted correctly'
parental_training_discussion_variable_decisions.loc[parent_option_mask,
    'Discussion reason'] = 'The parent option distinguishes discussion involving parents from discussion involving other people among those who reported talking to someone about training or an apprenticeship'
assert parental_training_discussion_variable_decisions['Discussion role'].notna().all()
assert parental_training_discussion_variable_decisions['Discussion reason'].notna().all()
parental_training_discussion_representation_decisions = pd.DataFrame([{'Representation': 'Parent option used as a stand-alone binary predictor',
    'Decision': 'Do not retain', 'Reason': 'The item was asked only when the young person had already reported discussion with someone, so structural non-applicability cannot be treated as an observed no'}, {'Representation': 'Any training or apprenticeship discussion',
    'Decision': 'Do not retain', 'Reason': 'This representation identifies whether any discussion occurred but does not specifically measure parental participation'}, {'Representation': 'Parental training or apprenticeship discussion profile',
    'Decision': 'Retain', 'Reason': 'The three-category profile distinguishes no discussion, discussion without parents and discussion with parents while respecting the routing structure'}, {'Representation': "Don't know and refused routing responses",
    'Decision': 'Treat as missing', 'Reason': 'These responses do not establish whether discussion occurred and cannot be assigned to the no-discussion category'}])
parental_training_discussion_predictor_summary = pd.DataFrame([{'Predictor': 'parental_training_apprenticeship_discussion_profile_pretransition',
    'Non-missing': int(parental_training_apprenticeship_discussion_profile_pretransition.notna().sum()), 'Missing': int(parental_training_apprenticeship_discussion_profile_pretransition.isna().sum()), 'Missing percentage': round(parental_training_apprenticeship_discussion_profile_pretransition.isna().mean() * 100,
    2), 'Categories': int(parental_training_apprenticeship_discussion_profile_pretransition.nunique()), "Maximum Cramer's V": round(float(parental_training_discussion_categorical_overlap_summary.loc[parental_training_discussion_categorical_overlap_summary['Candidate predictor'].eq('Parental training or apprenticeship discussion profile'),
    "Cramer's V"].max()), 3)}])
parental_training_discussion_category_labels = {0: 'No discussion with anyone',
    1: 'Discussion, but parents not mentioned', 2: 'Discussion with parents'}
parental_training_discussion_retained_distribution_rows = []
observed_discussion_profile_count = int(parental_training_apprenticeship_discussion_profile_pretransition.notna().sum())
for code, category in parental_training_discussion_category_labels.items():
    participants = int(parental_training_apprenticeship_discussion_profile_pretransition.eq(code).sum())
    parental_training_discussion_retained_distribution_rows.append({'Code': code, 'Category': category,
        'Participants': participants, 'Percentage among observed': round(participants / observed_discussion_profile_count * 100,
        2)})
parental_training_discussion_retained_distribution = pd.DataFrame(parental_training_discussion_retained_distribution_rows)
parental_training_discussion_decision_display = parental_training_discussion_variable_decisions[['Wave', 'Source type',
    'Source file', 'Variable position', 'Variable', 'Variable label', 'Discussion decision', 'Discussion role', 'Discussion reason']].copy()
print('Parental training and apprenticeship discussion predictors retained: 1')
print('Retained predictor summary:')
display_limited(parental_training_discussion_predictor_summary)
print('Retained category distribution:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_training_discussion_retained_distribution)
print('Representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_training_discussion_representation_decisions)
print('Variable decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(parental_training_discussion_decision_display)
print('Files written in this cell: 0')

Parental training and apprenticeship discussion predictors retained: 1
Retained predictor summary:


,Predictor,Non-missing,Missing,Missing percentage,Categories,Maximum Cramer's V
0,parental_training_apprenticeship_discussion_pr...,9361,406,4.16,3,0.201


Retained category distribution:


,Code,Category,Participants,Percentage among observed
0,0,No discussion with anyone,7551,80.66
1,1,"Discussion, but parents not mentioned",704,7.52
2,2,Discussion with parents,1106,11.81


Representation decisions:


,Representation,Decision,Reason
0,Parent option used as a stand-alone binary predictor,Do not retain,"The item was asked only when the young person had already reported discussion with someone, so structural non-applicability cannot be treated as an observed no"
1,Any training or apprenticeship discussion,Do not retain,This representation identifies whether any discussion occurred but does not specifically measure parental participation
2,Parental training or apprenticeship discussion profile,Retain,"The three-category profile distinguishes no discussion, discussion without parents and discussion with parents while respecting the routing structure"
3,Don't know and refused routing responses,Treat as missing,These responses do not establish whether discussion occurred and cannot be assigned to the no-discussion category


Variable decisions:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Discussion decision,Discussion role,Discussion reason
0,Wave 2,Young person,wave_two_lsype_young_person_2020,325,W2modap2YP,YP: Whether YP has talked to anyone about getting training or apprenticeship aft,Construction input,Routing input identifying whether any discussion occurred,The routing item distinguishes no discussion from discussion with at least one person and permits structural non-applicability in the parent option to be interpreted correctly
1,Wave 2,Young person,wave_two_lsype_young_person_2020,326,W2ModAp3YP0a,YP: Who YP has talked to about training or an apprenticeship - Parents,Construction input,Parent-participation input among young people reporting discussion,The parent option distinguishes discussion involving parents from discussion involving other people among those who reported talking to someone about training or an apprenticeship


Files written in this cell: 0


In [371]:
# 52: Domain 9 contextual-review inventory

import pandas as pd
domain_9_contextual_inventory = domain_9_screened_candidates.loc[domain_9_screened_candidates['Domain 9 screening status'].eq('Contextual review')].copy().sort_values(['Domain 9 review track',
    'Source order', 'Variable position']).reset_index(drop=True)
deferred_special_meeting_inventory = parental_school_involvement_variable_decisions.loc[parental_school_involvement_variable_decisions['School-involvement decision'].astype('string').str.contains('defer',
    case=False, na=False, regex=True) | parental_school_involvement_variable_decisions['School-involvement role'].astype('string').str.contains('special|arranged|meeting',
    case=False, na=False, regex=True)].copy().sort_values(['Wave', 'Variable position']).reset_index(drop=True)
domain_9_contextual_track_summary = domain_9_contextual_inventory['Domain 9 review track'].astype('string').value_counts(dropna=False).rename('Variables').rename_axis('Domain 9 review track').reset_index()
contextual_review_columns = ['Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status']
original_contextual_display = domain_9_contextual_inventory[[*contextual_review_columns,
    'Domain 9 review track']].copy()
original_contextual_display['Review source'] = 'Original contextual review'
if len(deferred_special_meeting_inventory) > 0:
    deferred_special_meeting_display = deferred_special_meeting_inventory[contextual_review_columns].copy()
    deferred_special_meeting_display['Domain 9 review track'] = 'Reactive school–parent contact'
    deferred_special_meeting_display['Review source'] = 'Deferred from parental school involvement'
else:
    deferred_special_meeting_display = pd.DataFrame(columns=[*contextual_review_columns, 'Domain 9 review track',
        'Review source'])
domain_9_combined_contextual_inventory = pd.concat([original_contextual_display, deferred_special_meeting_display],
    ignore_index=True).drop_duplicates(subset=['Source file', 'Variable']).sort_values(['Domain 9 review track',
    'Wave', 'Variable position']).reset_index(drop=True)
domain_9_combined_contextual_track_summary = domain_9_combined_contextual_inventory['Domain 9 review track'].astype('string').value_counts(dropna=False).rename('Variables').rename_axis('Combined contextual-review track').reset_index()
print(f'Original Domain 9 contextual-review variables: {len(domain_9_contextual_inventory)}')
print('Original contextual-review tracks:')
display_limited(domain_9_contextual_track_summary)
print('Original contextual-review variables:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(original_contextual_display)
print(f'Special parent–teacher meeting variables recovered: {len(deferred_special_meeting_display)}')
print('Deferred special-meeting variables:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(deferred_special_meeting_display)
print('Combined contextual-review tracks:')
display_limited(domain_9_combined_contextual_track_summary)
print('Combined contextual-review inventory:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_9_combined_contextual_inventory)
print('Files written in this cell: 0')

Original Domain 9 contextual-review variables: 4
Original contextual-review tracks:


,Domain 9 review track,Variables
0,Reactive school–parent contact,2
1,Shared family activity,1
2,Teacher recommendation reported by parent,1


Original contextual-review variables:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Domain 9 review track,Review source
0,Wave 2,Young person,wave_two_lsype_young_person_2020,43,W2SchcontMP,MP: Whether any of YP's schools contacted them about YP's behaviour,Pre-transition source,Reactive school–parent contact,Original contextual review
1,Wave 3,Young person,wave_three_lsype_young_person_2020,30,W3schcontMP,MP: Whether any of YP's schools contacted them about YP's behaviour,Near-transition source,Reactive school–parent contact,Original contextual review
2,Wave 1,Family background,wave_one_lsype_family_background_2020,27,W1fammusMP,MP: How often go out together as a family (excluding shopping),Pre-transition source,Shared family activity,Original contextual review
3,Wave 3,Parental attitudes,wave_three_lsype_parental_attitudes_file_16_06_08,15,W3tstaydefMP,MP: Whether teacher though YP should stay on in education after Year 11,Near-transition source,Teacher recommendation reported by parent,Original contextual review


Special parent–teacher meeting variables recovered: 3
Deferred special-meeting variables:


...

Combined contextual-review tracks:


,Combined contextual-review track,Variables
0,Reactive school–parent contact,5
1,Shared family activity,1
2,Teacher recommendation reported by parent,1


Combined contextual-review inventory:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Domain 9 review track,Review source
0,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,19,W1tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,Pre-transition source,Reactive school–parent contact,Deferred from parental school involvement
1,Wave 2,Parental attitudes,wave_two_lsype_parental_attitudes_file_16_06_08,10,W2tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,Pre-transition source,Reactive school–parent contact,Deferred from parental school involvement
2,Wave 2,Young person,wave_two_lsype_young_person_2020,43,W2SchcontMP,MP: Whether any of YP's schools contacted them about YP's behaviour,Pre-transition source,Reactive school–parent contact,Original contextual review
3,Wave 3,Parental attitudes,wave_three_lsype_parental_attitudes_file_16_06_08,11,W3tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,Near-transition source,Reactive school–parent contact,Deferred from parental school involvement
4,Wave 3,Young person,wave_three_lsype_young_person_2020,30,W3schcontMP,MP: Whether any of YP's schools contacted them about YP's behaviour,Near-transition source,Reactive school–parent contact,Original contextual review


Files written in this cell: 0


In [372]:
# 53: Reactive school–parent contact review

import pandas as pd
from itertools import combinations
reactive_school_parent_contact_inventory = domain_9_combined_contextual_inventory.loc[domain_9_combined_contextual_inventory['Domain 9 review track'].eq('Reactive school–parent contact')].copy().sort_values(['Wave',
    'Variable position']).reset_index(drop=True)
assert set(reactive_school_parent_contact_inventory['Variable']) == {'W1tmeetfMP', 'W2tmeetfMP', 'W3tmeetfMP',
    'W2SchcontMP', 'W3schcontMP'}
wave_3_contact_timing_mask = pd.Series(wave_3_pretransition_interview_mask,
    index=getattr(wave_3_pretransition_interview_mask, 'index',
    route_index)).reindex(route_index).fillna(False).astype(bool)
assert len(wave_3_contact_timing_mask) == 9767
reactive_contact_raw = pd.DataFrame(index=route_index)
reactive_contact_labelled = pd.DataFrame(index=route_index)
reactive_contact_unrestricted_raw = pd.DataFrame(index=route_index)
for _, inventory_row in reactive_school_parent_contact_inventory.iterrows():
    variable = inventory_row['Variable']
    source_file = inventory_row['Source file']
    source_path = source_file_lookup[source_file]
    raw_source = pd.read_stata(source_path, columns=['NSID', variable], convert_categoricals=False)
    labelled_source = pd.read_stata(source_path, columns=['NSID', variable], convert_categoricals=True)
    for source_data in [raw_source, labelled_source]:
        source_data['NSID'] = standardise_nsid(source_data['NSID'])
        assert source_data['NSID'].is_unique
    raw_values = raw_source.set_index('NSID')[variable].reindex(route_index)
    raw_values = pd.to_numeric(raw_values, errors='coerce')
    labelled_values = labelled_source.set_index('NSID')[variable].reindex(route_index).astype('string')
    reactive_contact_unrestricted_raw[variable] = raw_values
    if inventory_row['Wave'] == 'Wave 3':
        reactive_contact_raw[variable] = raw_values.where(wave_3_contact_timing_mask)
        reactive_contact_labelled[variable] = labelled_values.where(wave_3_contact_timing_mask)
    else:
        reactive_contact_raw[variable] = raw_values
        reactive_contact_labelled[variable] = labelled_values
reactive_contact_coverage_rows = []
reactive_contact_code_rows = []
reactive_contact_observed = {}
for _, inventory_row in reactive_school_parent_contact_inventory.iterrows():
    variable = inventory_row['Variable']
    unrestricted_values = reactive_contact_unrestricted_raw[variable]
    raw_values = reactive_contact_raw[variable]
    labelled_values = reactive_contact_labelled[variable]
    observed_mask = raw_values.ge(0).fillna(False).astype(bool)
    structural_mask = raw_values.eq(-91.0).fillna(False).astype(bool)
    other_special_mask = raw_values.lt(0).fillna(False).astype(bool) & ~structural_mask
    outside_timing_count = 0
    if inventory_row['Wave'] == 'Wave 3':
        outside_timing_count = int((unrestricted_values.notna() & ~wave_3_contact_timing_mask).sum())
    reactive_contact_observed[variable] = raw_values.where(observed_mask)
    reactive_contact_coverage_rows.append({'Wave': inventory_row['Wave'], 'Variable': variable,
        'Variable label': inventory_row['Variable label'], 'Observed responses': int(observed_mask.sum()), 'Observed percentage': round(observed_mask.mean() * 100,
        2), 'Structurally not applicable': int(structural_mask.sum()), 'Other special-code responses': int(other_special_mask.sum()), 'No source record or timing-restricted': int(raw_values.isna().sum()), 'Removed by Wave 3 timing restriction': outside_timing_count, 'Observed categories': int(raw_values.loc[observed_mask].nunique())})
    for raw_code, participants in raw_values.value_counts(dropna=False).items():
        if pd.isna(raw_code):
            value_label = 'No source record or timing-restricted'
            response_type = 'Unavailable'
            sort_value = 999999
        else:
            response_mask = raw_values.eq(raw_code)
            matching_labels = labelled_values.loc[response_mask].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            if raw_code >= 0:
                response_type = 'Observed response'
            elif raw_code == -91:
                response_type = 'Structural non-applicability'
            else:
                response_type = 'Other special code'
            sort_value = float(raw_code)
        reactive_contact_code_rows.append({'Wave': inventory_row['Wave'], 'Variable': variable, 'Raw code': raw_code,
            'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
reactive_contact_coverage_summary = pd.DataFrame(reactive_contact_coverage_rows)
reactive_contact_code_distribution = pd.DataFrame(reactive_contact_code_rows).sort_values(['Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
reactive_contact_cross_wave_specs = [('Specially arranged teacher meeting', 'W1tmeetfMP', 'W2tmeetfMP'),
    ('Specially arranged teacher meeting', 'W1tmeetfMP', 'W3tmeetfMP'), ('Specially arranged teacher meeting',
    'W2tmeetfMP', 'W3tmeetfMP'), ('School contact about behaviour', 'W2SchcontMP', 'W3schcontMP')]
reactive_contact_cross_wave_rows = []
for contact_type, first_variable, second_variable in reactive_contact_cross_wave_specs:
    pair_data = pd.DataFrame({'First': reactive_contact_observed[first_variable],
        'Second': reactive_contact_observed[second_variable]}, index=route_index).dropna()
    assert len(pair_data) > 0
    reactive_contact_cross_wave_rows.append({'Contact type': contact_type, 'First variable': first_variable,
        'Second variable': second_variable, 'Complete comparisons': len(pair_data), 'Exact agreement percentage': round(pair_data['First'].eq(pair_data['Second']).mean() * 100,
        2), "Cramer's V": round(categorical_cramers_v(pair_data['First'], pair_data['Second']), 3)})
reactive_contact_cross_wave_summary = pd.DataFrame(reactive_contact_cross_wave_rows)
reactive_contact_within_wave_rows = []
for wave_label, meeting_variable, behaviour_variable in [('Wave 2', 'W2tmeetfMP', 'W2SchcontMP'), ('Wave 3',
    'W3tmeetfMP', 'W3schcontMP')]:
    pair_data = pd.DataFrame({'Special meeting': reactive_contact_observed[meeting_variable],
        'Behaviour contact': reactive_contact_observed[behaviour_variable]}, index=route_index).dropna()
    assert len(pair_data) > 0
    reactive_contact_within_wave_rows.append({'Wave': wave_label, 'Complete comparisons': len(pair_data),
        'Exact agreement percentage': round(pair_data['Special meeting'].eq(pair_data['Behaviour contact']).mean() * 100,
        2), "Cramer's V": round(categorical_cramers_v(pair_data['Special meeting'], pair_data['Behaviour contact']),
        3)})
reactive_contact_within_wave_summary = pd.DataFrame(reactive_contact_within_wave_rows)
reactive_contact_wave_2_crosstab = pd.crosstab(reactive_contact_observed['W2tmeetfMP'],
    reactive_contact_observed['W2SchcontMP'], margins=True, dropna=False)
reactive_contact_wave_3_crosstab = pd.crosstab(reactive_contact_observed['W3tmeetfMP'],
    reactive_contact_observed['W3schcontMP'], margins=True, dropna=False)
print(f'Reactive school–parent contact variables reviewed: {len(reactive_school_parent_contact_inventory)}')
print('Variable inventory:')
with pd.option_context('display.max_colwidth', None):
    display_limited(reactive_school_parent_contact_inventory[['Wave', 'Source type', 'Source file', 'Variable',
        'Variable label', 'Timing status', 'Review source']])
print('Coverage after timing restrictions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(reactive_contact_coverage_summary)
print('Response-code distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(reactive_contact_code_distribution)
print('Cross-wave consistency:')
display_limited(reactive_contact_cross_wave_summary)
print('Within-wave overlap between contact types:')
display_limited(reactive_contact_within_wave_summary)
print('Wave 2 cross-tabulation:')
display_limited(reactive_contact_wave_2_crosstab)
print('Wave 3 cross-tabulation:')
display_limited(reactive_contact_wave_3_crosstab)
print('Files written in this cell: 0')

Reactive school–parent contact variables reviewed: 5
Variable inventory:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Review source
0,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,Pre-transition source,Deferred from parental school involvement
1,Wave 2,Parental attitudes,wave_two_lsype_parental_attitudes_file_16_06_08,W2tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,Pre-transition source,Deferred from parental school involvement
2,Wave 2,Young person,wave_two_lsype_young_person_2020,W2SchcontMP,MP: Whether any of YP's schools contacted them about YP's behaviour,Pre-transition source,Original contextual review
3,Wave 3,Parental attitudes,wave_three_lsype_parental_attitudes_file_16_06_08,W3tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,Near-transition source,Deferred from parental school involvement
4,Wave 3,Young person,wave_three_lsype_young_person_2020,W3schcontMP,MP: Whether any of YP's schools contacted them about YP's behaviour,Near-transition source,Original contextual review


Coverage after timing restrictions:


,Wave,Variable,Variable label,Observed responses,Observed percentage,Structurally not applicable,Other special-code responses,No source record or timing-restricted,Removed by Wave 3 timing restriction,Observed categories
0,Wave 1,W1tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,9410,96.34,0,114,243,0,2
1,Wave 2,W2tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,9427,96.52,0,94,246,0,2
2,Wave 2,W2SchcontMP,MP: Whether any of YP's schools contacted them about YP's behaviour,8635,88.41,0,886,246,0,2
3,Wave 3,W3tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,9402,96.26,0,93,272,14,2
4,Wave 3,W3schcontMP,MP: Whether any of YP's schools contacted them about YP's behaviour,8670,88.77,0,825,272,14,2


Response-code distributions:


...

Wave 2 cross-tabulation:


W2SchcontMP,1.0,2.0,NaN,All
W2tmeetfMP,,,,
1.0,882,1118,182,2182
2.0,930,5692,623,7245
NaN,2,11,327,340
All,1814,6821,1132,9767


Wave 3 cross-tabulation:


W3schcontMP,1.0,2.0,NaN,All
W3tmeetfMP,,,,
1.0,699,1415,168,2282
2.0,687,5858,575,7120
NaN,0,11,354,365
All,1386,7284,1097,9767


Files written in this cell: 0


In [373]:
# 54: Reactive school–parent contact representation review

import pandas as pd

def retrieve_retained_predictor(predictor_name):
    direct_object = globals().get(predictor_name)
    if isinstance(direct_object, pd.Series):
        return (direct_object.reindex(route_index).copy(), predictor_name)
    if isinstance(direct_object, pd.DataFrame):
        if predictor_name in direct_object.columns:
            predictor_data = direct_object.copy()
            if 'NSID' in predictor_data.columns:
                predictor_data['NSID'] = standardise_nsid(predictor_data['NSID'])
                predictor_series = predictor_data.set_index('NSID')[predictor_name].reindex(route_index)
            else:
                predictor_series = predictor_data[predictor_name].reindex(route_index)
            return (predictor_series, predictor_name)
    for object_name, object_value in list(globals().items()):
        if not isinstance(object_value, pd.DataFrame):
            continue
        if predictor_name not in object_value.columns:
            continue
        predictor_data = object_value.copy()
        if 'NSID' in predictor_data.columns:
            predictor_data['NSID'] = standardise_nsid(predictor_data['NSID'])
            predictor_series = predictor_data.set_index('NSID')[predictor_name].reindex(route_index)
        else:
            predictor_series = predictor_data[predictor_name].reindex(route_index)
        return (predictor_series, object_name)
    return (None, None)
reactive_contact_binary_recode = {1.0: 1, 2.0: 0}
special_meeting_wave_1 = reactive_contact_observed['W1tmeetfMP'].map(reactive_contact_binary_recode).astype('Int64')
special_meeting_wave_2 = reactive_contact_observed['W2tmeetfMP'].map(reactive_contact_binary_recode).astype('Int64')
special_meeting_wave_3 = reactive_contact_observed['W3tmeetfMP'].map(reactive_contact_binary_recode).astype('Int64')
behaviour_contact_wave_2 = reactive_contact_observed['W2SchcontMP'].map(reactive_contact_binary_recode).astype('Int64')
behaviour_contact_wave_3 = reactive_contact_observed['W3schcontMP'].map(reactive_contact_binary_recode).astype('Int64')
specially_arranged_teacher_meeting_candidate = special_meeting_wave_3.combine_first(special_meeting_wave_2).combine_first(special_meeting_wave_1).astype('Int64').rename('specially_arranged_teacher_meeting_pretransition')
school_contact_about_behaviour_candidate = behaviour_contact_wave_3.combine_first(behaviour_contact_wave_2).astype('Int64').rename('school_contact_about_behaviour_pretransition')
assert set(specially_arranged_teacher_meeting_candidate.dropna().astype(int).unique()).issubset({0, 1})
assert set(school_contact_about_behaviour_candidate.dropna().astype(int).unique()).issubset({0, 1})
special_meeting_source_wave = pd.Series(pd.NA, index=route_index, dtype='string', name='Special-meeting source wave')
special_meeting_source_wave.loc[special_meeting_wave_1.notna()] = 'Wave 1'
special_meeting_source_wave.loc[special_meeting_wave_2.notna()] = 'Wave 2'
special_meeting_source_wave.loc[special_meeting_wave_3.notna()] = 'Wave 3'
behaviour_contact_source_wave = pd.Series(pd.NA, index=route_index, dtype='string',
    name='Behaviour-contact source wave')
behaviour_contact_source_wave.loc[behaviour_contact_wave_2.notna()] = 'Wave 2'
behaviour_contact_source_wave.loc[behaviour_contact_wave_3.notna()] = 'Wave 3'
reactive_contact_candidate_summary = pd.DataFrame([{'Candidate predictor': 'specially_arranged_teacher_meeting_pretransition',
    'Non-missing': int(specially_arranged_teacher_meeting_candidate.notna().sum()), 'Missing': int(specially_arranged_teacher_meeting_candidate.isna().sum()), 'Missing percentage': round(specially_arranged_teacher_meeting_candidate.isna().mean() * 100,
    2), 'Yes': int(specially_arranged_teacher_meeting_candidate.eq(1).sum()), 'No': int(specially_arranged_teacher_meeting_candidate.eq(0).sum())}, {'Candidate predictor': 'school_contact_about_behaviour_pretransition',
    'Non-missing': int(school_contact_about_behaviour_candidate.notna().sum()), 'Missing': int(school_contact_about_behaviour_candidate.isna().sum()), 'Missing percentage': round(school_contact_about_behaviour_candidate.isna().mean() * 100,
    2), 'Yes': int(school_contact_about_behaviour_candidate.eq(1).sum()), 'No': int(school_contact_about_behaviour_candidate.eq(0).sum())}])
special_meeting_source_summary = special_meeting_source_wave.value_counts(dropna=False).rename('Participants').rename_axis('Source wave').reset_index()
special_meeting_source_summary['Candidate'] = 'Specially arranged teacher meeting'
behaviour_contact_source_summary = behaviour_contact_source_wave.value_counts(dropna=False).rename('Participants').rename_axis('Source wave').reset_index()
behaviour_contact_source_summary['Candidate'] = 'School contact about behaviour'
reactive_contact_source_summary = pd.concat([special_meeting_source_summary, behaviour_contact_source_summary],
    ignore_index=True)[['Candidate', 'Source wave', 'Participants']]
reactive_contact_pair = pd.DataFrame({'Specially arranged teacher meeting': specially_arranged_teacher_meeting_candidate,
    'School contact about behaviour': school_contact_about_behaviour_candidate}, index=route_index).dropna()
reactive_contact_candidate_pair_summary = pd.DataFrame([{'Complete comparisons': len(reactive_contact_pair),
    'Exact agreement percentage': round(reactive_contact_pair['Specially arranged teacher meeting'].eq(reactive_contact_pair['School contact about behaviour']).mean() * 100,
    2), "Cramer's V": round(categorical_cramers_v(reactive_contact_pair['Specially arranged teacher meeting'],
    reactive_contact_pair['School contact about behaviour']), 3)}])
reactive_contact_candidate_crosstab = pd.crosstab(reactive_contact_pair['Specially arranged teacher meeting'],
    reactive_contact_pair['School contact about behaviour'], margins=True, dropna=False)
reactive_contact_comparison_specs = {"Parents' evening attendance": 'parents_evening_attendance_pretransition',
    'Parent–teacher post-16 discussion profile': 'parent_teacher_post16_discussion_profile_pretransition', 'Truancy status': 'truancy_status_pretransition', 'Recent antisocial behaviour': 'recent_antisocial_behaviour_pretransition', 'Recent police contact': 'recent_police_contact_pretransition', 'School attitude score': 'school_attitude_score', 'School-night curfew': 'school_night_curfew_pretransition', 'Parental knowledge of evening whereabouts': 'parental_knowledge_of_evening_whereabouts_pretransition'}
reactive_contact_comparison_series = {}
reactive_contact_retrieval_rows = []
for display_name, predictor_name in reactive_contact_comparison_specs.items():
    predictor_series, source_object = retrieve_retained_predictor(predictor_name)
    reactive_contact_retrieval_rows.append({'Comparison measure': display_name, 'Predictor name': predictor_name,
        'Retrieved': predictor_series is not None, 'Source object': source_object})
    if predictor_series is not None:
        reactive_contact_comparison_series[display_name] = predictor_series
reactive_contact_retrieval_summary = pd.DataFrame(reactive_contact_retrieval_rows)
reactive_contact_overlap_data = pd.DataFrame({'Specially arranged teacher meeting': specially_arranged_teacher_meeting_candidate,
    'School contact about behaviour': school_contact_about_behaviour_candidate, **reactive_contact_comparison_series}, index=route_index)
assert len(reactive_contact_overlap_data) == 9767
categorical_comparison_measures = [measure for measure in ["Parents' evening attendance",
    'Parent–teacher post-16 discussion profile', 'Truancy status', 'Recent antisocial behaviour', 'Recent police contact', 'School-night curfew', 'Parental knowledge of evening whereabouts'] if measure in reactive_contact_overlap_data.columns]
reactive_contact_categorical_overlap_rows = []
for candidate_measure in ['Specially arranged teacher meeting', 'School contact about behaviour']:
    for comparison_measure in categorical_comparison_measures:
        pair_data = reactive_contact_overlap_data[[candidate_measure, comparison_measure]].dropna()
        if len(pair_data) == 0:
            continue
        reactive_contact_categorical_overlap_rows.append({'Candidate predictor': candidate_measure,
            'Comparison measure': comparison_measure, 'Complete comparisons': len(pair_data), "Cramer's V": round(categorical_cramers_v(pair_data[candidate_measure],
            pair_data[comparison_measure]), 3)})
reactive_contact_categorical_overlap_summary = pd.DataFrame(reactive_contact_categorical_overlap_rows)
if len(reactive_contact_categorical_overlap_summary) > 0:
    reactive_contact_categorical_overlap_summary = reactive_contact_categorical_overlap_summary.sort_values(['Candidate predictor',
        "Cramer's V"], ascending=[True, False]).reset_index(drop=True)
reactive_contact_school_attitude_rows = []
if 'School attitude score' in reactive_contact_overlap_data.columns:
    for candidate_measure in ['Specially arranged teacher meeting', 'School contact about behaviour']:
        candidate_data = reactive_contact_overlap_data[[candidate_measure, 'School attitude score']].dropna()
        for candidate_value, group_data in candidate_data.groupby(candidate_measure):
            reactive_contact_school_attitude_rows.append({'Candidate predictor': candidate_measure,
                'Candidate value': int(candidate_value), 'Participants': len(group_data), 'Mean school-attitude score': round(float(group_data['School attitude score'].mean()),
                3), 'Median school-attitude score': round(float(group_data['School attitude score'].median()), 3)})
reactive_contact_school_attitude_summary = pd.DataFrame(reactive_contact_school_attitude_rows)
print('Candidate predictor summary:')
display_limited(reactive_contact_candidate_summary)
print('Source-wave use:')
display_limited(reactive_contact_source_summary)
print('Association between the two candidates:')
display_limited(reactive_contact_candidate_pair_summary)
print('Candidate cross-tabulation:')
display_limited(reactive_contact_candidate_crosstab)
print('Comparison-predictor retrieval:')
with pd.option_context('display.max_colwidth', None):
    display_limited(reactive_contact_retrieval_summary)
print('Categorical associations with retained predictors:')
display_limited(reactive_contact_categorical_overlap_summary)
print('School-attitude score by candidate category:')
display_limited(reactive_contact_school_attitude_summary)
print('Files written in this cell: 0')

Candidate predictor summary:


,Candidate predictor,Non-missing,Missing,Missing percentage,Yes,No
0,specially_arranged_teacher_meeting_pretransition,9516,251,2.57,2317,7199
1,school_contact_about_behaviour_pretransition,9019,748,7.66,1475,7544


Source-wave use:


,Candidate,Source wave,Participants
0,Specially arranged teacher meeting,Wave 3,9402
1,Specially arranged teacher meeting,<NA>,251
2,Specially arranged teacher meeting,Wave 2,104
3,Specially arranged teacher meeting,Wave 1,10
4,School contact about behaviour,Wave 3,8670


Association between the two candidates:


...

Categorical associations with retained predictors:


,Candidate predictor,Comparison measure,Complete comparisons,Cramer's V
0,School contact about behaviour,Recent antisocial behaviour,8946,0.289
1,School contact about behaviour,Truancy status,8992,0.272
2,School contact about behaviour,Recent police contact,9014,0.258
3,School contact about behaviour,Parental knowledge of evening whereabouts,8796,0.184
4,School contact about behaviour,Parents' evening attendance,9018,0.144


School-attitude score by candidate category:


,Candidate predictor,Candidate value,Participants,Mean school-attitude score,Median school-attitude score
0,Specially arranged teacher meeting,0,7190,3.049,3.083
1,Specially arranged teacher meeting,1,2313,2.921,3.000
2,School contact about behaviour,0,7537,3.064,3.083
3,School contact about behaviour,1,1470,2.735,2.800


Files written in this cell: 0


In [374]:
# 55: Reactive contact event-history representation review

import pandas as pd

def construct_ever_and_count(wave_series, ever_name, count_name):
    wave_data = pd.concat(wave_series, axis=1)
    valid_response_count = wave_data.notna().sum(axis=1)
    positive_response_count = wave_data.eq(1).sum(axis=1)
    ever_values = pd.Series(pd.NA, index=wave_data.index, dtype='Int64', name=ever_name)
    ever_values.loc[valid_response_count.gt(0) & positive_response_count.eq(0)] = 0
    ever_values.loc[positive_response_count.gt(0)] = 1
    count_values = positive_response_count.where(valid_response_count.gt(0)).astype('Int64').rename(count_name)
    return (ever_values, count_values, valid_response_count)
specially_arranged_teacher_meeting_ever_candidate, specially_arranged_teacher_meeting_count_candidate, special_meeting_observed_wave_count = construct_ever_and_count([special_meeting_wave_1.rename('Wave 1'),
    special_meeting_wave_2.rename('Wave 2'), special_meeting_wave_3.rename('Wave 3')], 'specially_arranged_teacher_meeting_ever_pretransition', 'specially_arranged_teacher_meeting_count_pretransition')
school_contact_about_behaviour_ever_candidate, school_contact_about_behaviour_count_candidate, behaviour_contact_observed_wave_count = construct_ever_and_count([behaviour_contact_wave_2.rename('Wave 2'),
    behaviour_contact_wave_3.rename('Wave 3')], 'school_contact_about_behaviour_ever_pretransition', 'school_contact_about_behaviour_count_pretransition')
assert set(specially_arranged_teacher_meeting_ever_candidate.dropna().astype(int).unique()).issubset({0, 1})
assert set(school_contact_about_behaviour_ever_candidate.dropna().astype(int).unique()).issubset({0, 1})
assert set(specially_arranged_teacher_meeting_count_candidate.dropna().astype(int).unique()).issubset({0, 1, 2, 3})
assert set(school_contact_about_behaviour_count_candidate.dropna().astype(int).unique()).issubset({0, 1, 2})
reactive_contact_representation_rows = []
reactive_contact_representation_specs = [('Specially arranged teacher meeting', 'Latest permitted response',
    specially_arranged_teacher_meeting_candidate), ('Specially arranged teacher meeting',
    'Any pre-transition occurrence', specially_arranged_teacher_meeting_ever_candidate), ('Specially arranged teacher meeting',
    'Number of observed occurrences', specially_arranged_teacher_meeting_count_candidate), ('School contact about behaviour',
    'Latest permitted response', school_contact_about_behaviour_candidate), ('School contact about behaviour',
    'Any pre-transition occurrence', school_contact_about_behaviour_ever_candidate), ('School contact about behaviour',
    'Number of observed occurrences', school_contact_about_behaviour_count_candidate)]
for contact_type, representation, values in reactive_contact_representation_specs:
    observed_values = values.dropna()
    reactive_contact_representation_rows.append({'Contact type': contact_type, 'Representation': representation,
        'Non-missing': int(values.notna().sum()), 'Missing': int(values.isna().sum()), 'Missing percentage': round(values.isna().mean() * 100,
        2), 'Distinct values': int(observed_values.nunique()), 'Mean': round(float(observed_values.astype(float).mean()),
        3), 'Maximum': int(observed_values.astype(int).max())})
reactive_contact_representation_summary = pd.DataFrame(reactive_contact_representation_rows)
reactive_contact_latest_ever_rows = []
for contact_type, latest_values, ever_values in [('Specially arranged teacher meeting',
    specially_arranged_teacher_meeting_candidate, specially_arranged_teacher_meeting_ever_candidate), ('School contact about behaviour',
    school_contact_about_behaviour_candidate, school_contact_about_behaviour_ever_candidate)]:
    pair_data = pd.DataFrame({'Latest': latest_values, 'Ever': ever_values}, index=route_index).dropna()
    earlier_only_mask = pair_data['Latest'].eq(0) & pair_data['Ever'].eq(1)
    reactive_contact_latest_ever_rows.append({'Contact type': contact_type, 'Complete comparisons': len(pair_data),
        'Exact agreement percentage': round(pair_data['Latest'].eq(pair_data['Ever']).mean() * 100,
        2), 'Earlier occurrence not captured by latest response': int(earlier_only_mask.sum()), 'Percentage with earlier-only occurrence': round(earlier_only_mask.mean() * 100,
        2), "Cramer's V": round(categorical_cramers_v(pair_data['Latest'], pair_data['Ever']), 3)})
reactive_contact_latest_ever_summary = pd.DataFrame(reactive_contact_latest_ever_rows)
special_meeting_latest_ever_crosstab = pd.crosstab(specially_arranged_teacher_meeting_candidate,
    specially_arranged_teacher_meeting_ever_candidate, margins=True, dropna=False)
behaviour_contact_latest_ever_crosstab = pd.crosstab(school_contact_about_behaviour_candidate,
    school_contact_about_behaviour_ever_candidate, margins=True, dropna=False)
reactive_contact_occurrence_distribution_rows = []
for contact_type, count_values in [('Specially arranged teacher meeting',
    specially_arranged_teacher_meeting_count_candidate), ('School contact about behaviour',
    school_contact_about_behaviour_count_candidate)]:
    observed_count = int(count_values.notna().sum())
    for occurrence_count, participants in count_values.value_counts().sort_index().items():
        reactive_contact_occurrence_distribution_rows.append({'Contact type': contact_type,
            'Observed occurrences': int(occurrence_count), 'Participants': int(participants), 'Percentage among observed': round(participants / observed_count * 100,
            2)})
reactive_contact_occurrence_distribution = pd.DataFrame(reactive_contact_occurrence_distribution_rows)
reactive_contact_history_overlap_data = pd.DataFrame({'Special meeting latest': specially_arranged_teacher_meeting_candidate,
    'Special meeting ever': specially_arranged_teacher_meeting_ever_candidate, 'Behaviour contact latest': school_contact_about_behaviour_candidate, 'Behaviour contact ever': school_contact_about_behaviour_ever_candidate, **reactive_contact_comparison_series}, index=route_index)
history_candidate_measures = ['Special meeting latest', 'Special meeting ever', 'Behaviour contact latest',
    'Behaviour contact ever']
history_comparison_measures = [measure for measure in ["Parents' evening attendance",
    'Parent–teacher post-16 discussion profile', 'Truancy status', 'Recent antisocial behaviour', 'Recent police contact', 'School-night curfew', 'Parental knowledge of evening whereabouts'] if measure in reactive_contact_history_overlap_data.columns]
reactive_contact_history_overlap_rows = []
for candidate_measure in history_candidate_measures:
    for comparison_measure in history_comparison_measures:
        pair_data = reactive_contact_history_overlap_data[[candidate_measure, comparison_measure]].dropna()
        if len(pair_data) == 0:
            continue
        reactive_contact_history_overlap_rows.append({'Candidate representation': candidate_measure,
            'Comparison measure': comparison_measure, 'Complete comparisons': len(pair_data), "Cramer's V": round(categorical_cramers_v(pair_data[candidate_measure],
            pair_data[comparison_measure]), 3)})
reactive_contact_history_overlap_summary = pd.DataFrame(reactive_contact_history_overlap_rows).sort_values(['Candidate representation',
    "Cramer's V"], ascending=[True, False]).reset_index(drop=True)
print('Representation summary:')
display_limited(reactive_contact_representation_summary)
print('Latest and event-history comparison:')
display_limited(reactive_contact_latest_ever_summary)
print('Special-meeting latest-by-ever cross-tabulation:')
display_limited(special_meeting_latest_ever_crosstab)
print('Behaviour-contact latest-by-ever cross-tabulation:')
display_limited(behaviour_contact_latest_ever_crosstab)
print('Occurrence-count distributions:')
display_limited(reactive_contact_occurrence_distribution)
print('Associations with retained predictors:')
display_limited(reactive_contact_history_overlap_summary)
print('Files written in this cell: 0')

Representation summary:


,Contact type,Representation,Non-missing,Missing,Missing percentage,Distinct values,Mean,Maximum
0,Specially arranged teacher meeting,Latest permitted response,9516,251,2.57,2,0.243,1
1,Specially arranged teacher meeting,Any pre-transition occurrence,9516,251,2.57,2,0.485,1
2,Specially arranged teacher meeting,Number of observed occurrences,9516,251,2.57,4,0.751,3
3,School contact about behaviour,Latest permitted response,9019,748,7.66,2,0.164,1
4,School contact about behaviour,Any pre-transition occurrence,9019,748,7.66,2,0.262,1


Latest and event-history comparison:


,Contact type,Complete comparisons,Exact agreement percentage,Earlier occurrence not captured by latest response,Percentage with earlier-only occurrence,Cramer's V
0,Specially arranged teacher meeting,9516,75.81,2302,24.19,0.584
1,School contact about behaviour,9019,90.13,890,9.87,0.742


Special-meeting latest-by-ever cross-tabulation:


...

Occurrence-count distributions:


,Contact type,Observed occurrences,Participants,Percentage among observed
0,Specially arranged teacher meeting,0,4897,51.46
1,Specially arranged teacher meeting,1,2720,28.58
2,Specially arranged teacher meeting,2,1271,13.36
3,Specially arranged teacher meeting,3,628,6.60
4,School contact about behaviour,0,6654,73.78


Associations with retained predictors:


,Candidate representation,Comparison measure,Complete comparisons,Cramer's V
0,Behaviour contact ever,Recent antisocial behaviour,8946,0.301
1,Behaviour contact ever,Truancy status,8992,0.300
2,Behaviour contact ever,Recent police contact,9014,0.258
3,Behaviour contact ever,Parental knowledge of evening whereabouts,8796,0.211
4,Behaviour contact ever,Parents' evening attendance,9018,0.168


Files written in this cell: 0


In [375]:
# 56: Reactive school–parent contact decisions

import pandas as pd
specially_arranged_teacher_meeting_ever_pretransition = specially_arranged_teacher_meeting_ever_candidate.copy().astype('Int64').rename('specially_arranged_teacher_meeting_ever_pretransition')
school_contact_about_behaviour_ever_pretransition = school_contact_about_behaviour_ever_candidate.copy().astype('Int64').rename('school_contact_about_behaviour_ever_pretransition')
assert specially_arranged_teacher_meeting_ever_pretransition.index.equals(route_index)
assert school_contact_about_behaviour_ever_pretransition.index.equals(route_index)
for retained_values in [specially_arranged_teacher_meeting_ever_pretransition,
    school_contact_about_behaviour_ever_pretransition]:
    observed_values = set(retained_values.dropna().astype(int).unique().tolist())
    assert observed_values.issubset({0, 1})
reactive_school_parent_contact_predictors = pd.DataFrame({'NSID': route_index,
    'specially_arranged_teacher_meeting_ever_pretransition': specially_arranged_teacher_meeting_ever_pretransition.to_numpy(), 'school_contact_about_behaviour_ever_pretransition': school_contact_about_behaviour_ever_pretransition.to_numpy()})
assert len(reactive_school_parent_contact_predictors) == 9767
assert reactive_school_parent_contact_predictors['NSID'].is_unique
reactive_school_parent_contact_variable_decisions = reactive_school_parent_contact_inventory.copy()
reactive_school_parent_contact_variable_decisions['Reactive-contact decision'] = 'Construction input'
reactive_school_parent_contact_variable_decisions['Reactive-contact role'] = pd.NA
reactive_school_parent_contact_variable_decisions['Reactive-contact reason'] = pd.NA
special_meeting_variable_mask = reactive_school_parent_contact_variable_decisions['Variable'].isin(['W1tmeetfMP',
    'W2tmeetfMP', 'W3tmeetfMP'])
behaviour_contact_variable_mask = reactive_school_parent_contact_variable_decisions['Variable'].isin(['W2SchcontMP',
    'W3schcontMP'])
reactive_school_parent_contact_variable_decisions.loc[special_meeting_variable_mask,
    'Reactive-contact role'] = 'Input to any specially arranged teacher meeting before the transition'
reactive_school_parent_contact_variable_decisions.loc[behaviour_contact_variable_mask,
    'Reactive-contact role'] = 'Input to any school contact with the parent about behaviour before the transition'
reactive_school_parent_contact_variable_decisions.loc[special_meeting_variable_mask,
    'Reactive-contact reason'] = 'The retained predictor records whether a specially arranged meeting occurred in any permitted pre-transition wave. Wave 3 responses are restricted by interview timing'
reactive_school_parent_contact_variable_decisions.loc[behaviour_contact_variable_mask,
    'Reactive-contact reason'] = 'The retained predictor records whether the school contacted the parent about behaviour in either permitted pre-transition wave. Wave 3 responses are restricted by interview timing'
assert reactive_school_parent_contact_variable_decisions['Reactive-contact role'].notna().all()
assert reactive_school_parent_contact_variable_decisions['Reactive-contact reason'].notna().all()
reactive_school_parent_contact_representation_decisions = pd.DataFrame([{'Contact type': 'Specially arranged teacher meeting',
    'Representation': 'Latest permitted response', 'Decision': 'Do not retain', 'Reason': 'The latest response does not capture 2,302 participants with an occurrence recorded only in an earlier permitted wave'}, {'Contact type': 'Specially arranged teacher meeting',
    'Representation': 'Any pre-transition occurrence', 'Decision': 'Retain', 'Reason': 'The binary event-history measure represents whether additional school–parent contact occurred at any point during the permitted pre-transition observation period'}, {'Contact type': 'Specially arranged teacher meeting',
    'Representation': 'Number of observed occurrences', 'Decision': 'Do not retain', 'Reason': 'The count is partly dependent on the number of observed waves and should not be interpreted as a directly comparable intensity measure'}, {'Contact type': 'School contact about behaviour',
    'Representation': 'Latest permitted response', 'Decision': 'Do not retain', 'Reason': 'The latest response does not capture 890 participants with contact recorded only in an earlier permitted wave'}, {'Contact type': 'School contact about behaviour',
    'Representation': 'Any pre-transition occurrence', 'Decision': 'Retain', 'Reason': 'The binary event-history measure captures whether the school responded to behaviour by contacting the parent at any permitted pre-transition wave'}, {'Contact type': 'School contact about behaviour',
    'Representation': 'Number of observed occurrences', 'Decision': 'Do not retain', 'Reason': 'The count depends on availability across two waves and does not provide a consistent measure of severity'}, {'Contact type': 'Both retained contact measures',
    'Representation': 'Combined school–parent contact indicator', 'Decision': 'Do not construct', 'Reason': 'A specially arranged meeting is not necessarily behaviour-related, and the two measures show only moderate association'}])
reactive_school_parent_contact_predictor_summary = pd.DataFrame([{'Predictor': 'specially_arranged_teacher_meeting_ever_pretransition',
    'Non-missing': int(specially_arranged_teacher_meeting_ever_pretransition.notna().sum()), 'Missing': int(specially_arranged_teacher_meeting_ever_pretransition.isna().sum()), 'Missing percentage': round(specially_arranged_teacher_meeting_ever_pretransition.isna().mean() * 100,
    2), 'No occurrence': int(specially_arranged_teacher_meeting_ever_pretransition.eq(0).sum()), 'Any occurrence': int(specially_arranged_teacher_meeting_ever_pretransition.eq(1).sum()), "Maximum Cramer's V with retained predictors": round(float(reactive_contact_history_overlap_summary.loc[reactive_contact_history_overlap_summary['Candidate representation'].eq('Special meeting ever'),
    "Cramer's V"].max()), 3)}, {'Predictor': 'school_contact_about_behaviour_ever_pretransition',
    'Non-missing': int(school_contact_about_behaviour_ever_pretransition.notna().sum()), 'Missing': int(school_contact_about_behaviour_ever_pretransition.isna().sum()), 'Missing percentage': round(school_contact_about_behaviour_ever_pretransition.isna().mean() * 100,
    2), 'No occurrence': int(school_contact_about_behaviour_ever_pretransition.eq(0).sum()), 'Any occurrence': int(school_contact_about_behaviour_ever_pretransition.eq(1).sum()), "Maximum Cramer's V with retained predictors": round(float(reactive_contact_history_overlap_summary.loc[reactive_contact_history_overlap_summary['Candidate representation'].eq('Behaviour contact ever'),
    "Cramer's V"].max()), 3)}])
reactive_contact_joint_availability = reactive_school_parent_contact_predictors[['specially_arranged_teacher_meeting_ever_pretransition',
    'school_contact_about_behaviour_ever_pretransition']].notna().sum(axis=1)
reactive_contact_joint_availability_summary = reactive_contact_joint_availability.value_counts().sort_index().rename('Participants').rename_axis('Retained predictors available').reset_index()
reactive_contact_joint_availability_summary['Percentage of full sample'] = (reactive_contact_joint_availability_summary['Participants'] / len(route_index) * 100).round(2)
reactive_contact_decision_display = reactive_school_parent_contact_variable_decisions[['Wave', 'Source type',
    'Source file', 'Variable', 'Variable label', 'Timing status', 'Review source', 'Reactive-contact decision', 'Reactive-contact role', 'Reactive-contact reason']].copy()
print('Reactive school–parent contact predictors retained: 2')
print('Retained predictor summary:')
display_limited(reactive_school_parent_contact_predictor_summary)
print('Joint availability:')
display_limited(reactive_contact_joint_availability_summary)
print('Representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(reactive_school_parent_contact_representation_decisions)
print('Variable decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(reactive_contact_decision_display)
print('Files written in this cell: 0')

Reactive school–parent contact predictors retained: 2
Retained predictor summary:


,Predictor,Non-missing,Missing,Missing percentage,No occurrence,Any occurrence,Maximum Cramer's V with retained predictors
0,specially_arranged_teacher_meeting_ever_pretra...,9516,251,2.57,4897,4619,0.158
1,school_contact_about_behaviour_ever_pretransition,9019,748,7.66,6654,2365,0.301


Joint availability:


,Retained predictors available,Participants,Percentage of full sample
0,0,251,2.57
1,1,497,5.09
2,2,9019,92.34


Representation decisions:


,Contact type,Representation,Decision,Reason
0,Specially arranged teacher meeting,Latest permitted response,Do not retain,"The latest response does not capture 2,302 participants with an occurrence recorded only in an earlier permitted wave"
1,Specially arranged teacher meeting,Any pre-transition occurrence,Retain,The binary event-history measure represents whether additional school–parent contact occurred at any point during the permitted pre-transition observation period
2,Specially arranged teacher meeting,Number of observed occurrences,Do not retain,The count is partly dependent on the number of observed waves and should not be interpreted as a directly comparable intensity measure
3,School contact about behaviour,Latest permitted response,Do not retain,The latest response does not capture 890 participants with contact recorded only in an earlier permitted wave
4,School contact about behaviour,Any pre-transition occurrence,Retain,The binary event-history measure captures whether the school responded to behaviour by contacting the parent at any permitted pre-transition wave


Variable decisions:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Review source,Reactive-contact decision,Reactive-contact role,Reactive-contact reason
0,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,W1tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,Pre-transition source,Deferred from parental school involvement,Construction input,Input to any specially arranged teacher meeting before the transition,The retained predictor records whether a specially arranged meeting occurred in any permitted pre-transition wave. Wave 3 responses are restricted by interview timing
1,Wave 2,Parental attitudes,wave_two_lsype_parental_attitudes_file_16_06_08,W2tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,Pre-transition source,Deferred from parental school involvement,Construction input,Input to any specially arranged teacher meeting before the transition,The retained predictor records whether a specially arranged meeting occurred in any permitted pre-transition wave. Wave 3 responses are restricted by interview timing
2,Wave 2,Young person,wave_two_lsype_young_person_2020,W2SchcontMP,MP: Whether any of YP's schools contacted them about YP's behaviour,Pre-transition source,Original contextual review,Construction input,Input to any school contact with the parent about behaviour before the transition,The retained predictor records whether the school contacted the parent about behaviour in either permitted pre-transition wave. Wave 3 responses are restricted by interview timing
3,Wave 3,Parental attitudes,wave_three_lsype_parental_attitudes_file_16_06_08,W3tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli,Near-transition source,Deferred from parental school involvement,Construction input,Input to any specially arranged teacher meeting before the transition,The retained predictor records whether a specially arranged meeting occurred in any permitted pre-transition wave. Wave 3 responses are restricted by interview timing
4,Wave 3,Young person,wave_three_lsype_young_person_2020,W3schcontMP,MP: Whether any of YP's schools contacted them about YP's behaviour,Near-transition source,Original contextual review,Construction input,Input to any school contact with the parent about behaviour before the transition,The retained predictor records whether the school contacted the parent about behaviour in either permitted pre-transition wave. Wave 3 responses are restricted by interview timing


Files written in this cell: 0


In [376]:
# 57: Shared family activity response review

import pandas as pd
shared_family_activity_inventory = domain_9_combined_contextual_inventory.loc[domain_9_combined_contextual_inventory['Domain 9 review track'].eq('Shared family activity')].copy().reset_index(drop=True)
assert len(shared_family_activity_inventory) == 1
assert shared_family_activity_inventory.loc[0, 'Variable'] == 'W1fammusMP'
shared_family_activity_source_file = shared_family_activity_inventory.loc[0, 'Source file']
shared_family_activity_source_path = source_file_lookup[shared_family_activity_source_file]
shared_family_activity_raw_source = pd.read_stata(shared_family_activity_source_path, columns=['NSID', 'W1fammusMP'],
    convert_categoricals=False)
shared_family_activity_labelled_source = pd.read_stata(shared_family_activity_source_path, columns=['NSID',
    'W1fammusMP'], convert_categoricals=True)
for source_data in [shared_family_activity_raw_source, shared_family_activity_labelled_source]:
    source_data['NSID'] = standardise_nsid(source_data['NSID'])
    assert source_data['NSID'].is_unique
shared_family_activity_raw = shared_family_activity_raw_source.set_index('NSID')['W1fammusMP'].reindex(route_index)
shared_family_activity_raw = pd.to_numeric(shared_family_activity_raw, errors='coerce')
shared_family_activity_labelled = shared_family_activity_labelled_source.set_index('NSID')['W1fammusMP'].reindex(route_index).astype('string')
shared_family_activity_observed_mask = shared_family_activity_raw.ge(0).fillna(False).astype(bool)
shared_family_activity_structural_mask = shared_family_activity_raw.eq(-91.0).fillna(False).astype(bool)
shared_family_activity_other_special_mask = shared_family_activity_raw.lt(0).fillna(False).astype(bool) & ~shared_family_activity_structural_mask
shared_family_activity_observed = shared_family_activity_raw.where(shared_family_activity_observed_mask).rename('shared_family_activity_observed')
shared_family_activity_coverage_summary = pd.DataFrame([{'Variable': 'W1fammusMP',
    'Variable label': shared_family_activity_inventory.loc[0,
    'Variable label'], 'Observed responses': int(shared_family_activity_observed_mask.sum()), 'Observed percentage': round(shared_family_activity_observed_mask.mean() * 100,
    2), 'Structurally not applicable': int(shared_family_activity_structural_mask.sum()), 'Other special-code responses': int(shared_family_activity_other_special_mask.sum()), 'No source record': int(shared_family_activity_raw.isna().sum()), 'Observed categories': int(shared_family_activity_observed.nunique())}])
shared_family_activity_code_rows = []
for raw_code, participants in shared_family_activity_raw.value_counts(dropna=False).items():
    if pd.isna(raw_code):
        value_label = 'No source record'
        response_type = 'Unavailable'
        sort_value = 999999
    else:
        response_mask = shared_family_activity_raw.eq(raw_code)
        matching_labels = shared_family_activity_labelled.loc[response_mask].dropna().drop_duplicates().tolist()
        value_label = matching_labels[0] if matching_labels else str(raw_code)
        if raw_code >= 0:
            response_type = 'Observed response'
        elif raw_code == -91:
            response_type = 'Structural non-applicability'
        else:
            response_type = 'Other special code'
        sort_value = float(raw_code)
    shared_family_activity_code_rows.append({'Raw code': raw_code, 'Value label': value_label,
        'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
shared_family_activity_code_distribution = pd.DataFrame(shared_family_activity_code_rows).sort_values('Sort value').drop(columns=['Sort value']).reset_index(drop=True)
shared_family_activity_observed_distribution = pd.DataFrame({'Raw code': shared_family_activity_observed,
    'Value label': shared_family_activity_labelled.where(shared_family_activity_observed_mask)}).dropna(subset=['Raw code']).value_counts(dropna=False).rename('Participants').reset_index().sort_values('Raw code').reset_index(drop=True)
shared_family_activity_observed_distribution['Percentage among observed'] = (shared_family_activity_observed_distribution['Participants'] / shared_family_activity_observed.notna().sum() * 100).round(2)
print('Shared family activity variable inventory:')
with pd.option_context('display.max_colwidth', None):
    display_limited(shared_family_activity_inventory[['Wave', 'Source type', 'Source file', 'Variable',
        'Variable label', 'Timing status', 'Domain 9 review track', 'Review source']])
print('Coverage summary:')
with pd.option_context('display.max_colwidth', None):
    display_limited(shared_family_activity_coverage_summary)
print('Response-code distribution:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(shared_family_activity_code_distribution)
print('Observed response distribution:')
with pd.option_context('display.max_colwidth', None):
    display_limited(shared_family_activity_observed_distribution)
print('Files written in this cell: 0')

Shared family activity variable inventory:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Domain 9 review track,Review source
0,Wave 1,Family background,wave_one_lsype_family_background_2020,W1fammusMP,MP: How often go out together as a family (excluding shopping),Pre-transition source,Shared family activity,Original contextual review


Coverage summary:


,Variable,Variable label,Observed responses,Observed percentage,Structurally not applicable,Other special-code responses,No source record,Observed categories
0,W1fammusMP,MP: How often go out together as a family (excluding shopping),9413,96.38,1,110,243,7


Response-code distribution:


,Raw code,Value label,Response type,Participants
0,-99.0,MP not interviewed,Other special code,103
1,-91.0,Not applicable,Structural non-applicability,1
2,-1.0,Don't know,Other special code,7
3,1.0,Once a week or more,Observed response,3594
4,2.0,Two or three times a month,Observed response,2589


Observed response distribution:


,Raw code,Value label,Participants,Percentage among observed
0,1.0,Once a week or more,3594,38.18
1,2.0,Two or three times a month,2589,27.50
2,3.0,About once a month,1948,20.69
3,4.0,Two or three times a year,755,8.02
4,5.0,About once a year,110,1.17


Files written in this cell: 0


In [377]:
# 58: Shared family activity representation review

import pandas as pd
from scipy.stats import spearmanr
shared_family_activity_frequency_recode = {1.0: 5, 2.0: 4, 3.0: 3, 4.0: 2, 5.0: 1, 6.0: 0}
shared_family_activity_frequency_candidate = shared_family_activity_observed.map(shared_family_activity_frequency_recode).astype('Int64').rename('shared_family_activity_frequency_pretransition')
shared_family_activity_varies_mask = shared_family_activity_observed.eq(7.0).fillna(False).astype(bool)
assert set(shared_family_activity_frequency_candidate.dropna().astype(int).unique()).issubset({0, 1, 2, 3, 4, 5})
assert int(shared_family_activity_varies_mask.sum()) == 88
shared_family_activity_candidate_summary = pd.DataFrame([{'Candidate predictor': 'shared_family_activity_frequency_pretransition',
    'Non-missing': int(shared_family_activity_frequency_candidate.notna().sum()), 'Missing': int(shared_family_activity_frequency_candidate.isna().sum()), 'Missing percentage': round(shared_family_activity_frequency_candidate.isna().mean() * 100,
    2), 'Distinct values': int(shared_family_activity_frequency_candidate.nunique()), 'Minimum': int(shared_family_activity_frequency_candidate.min()), 'Median': float(shared_family_activity_frequency_candidate.median()), 'Mean': round(float(shared_family_activity_frequency_candidate.mean()),
    2), 'Maximum': int(shared_family_activity_frequency_candidate.max()), 'Varies responses treated as missing': int(shared_family_activity_varies_mask.sum())}])
shared_family_activity_frequency_labels = {0: 'Never or hardly ever', 1: 'About once a year',
    2: 'Two or three times a year', 3: 'About once a month', 4: 'Two or three times a month', 5: 'Once a week or more'}
shared_family_activity_candidate_distribution_rows = []
shared_family_activity_observed_count = int(shared_family_activity_frequency_candidate.notna().sum())
for score, category in shared_family_activity_frequency_labels.items():
    participants = int(shared_family_activity_frequency_candidate.eq(score).sum())
    shared_family_activity_candidate_distribution_rows.append({'Score': score, 'Category': category,
        'Participants': participants, 'Percentage among ordered responses': round(participants / shared_family_activity_observed_count * 100,
        2)})
shared_family_activity_candidate_distribution = pd.DataFrame(shared_family_activity_candidate_distribution_rows)
shared_family_activity_overlap_data = pd.DataFrame({'Shared family activity frequency': shared_family_activity_frequency_candidate,
    'Parent relationship-quality score': parent_relationship_quality_score_pretransition, 'Parent communication-frequency score': parent_communication_frequency_score_pretransition, 'Parent discussion of school day': parent_school_day_discussion_frequency_pretransition, 'Parental autonomy-support score': parental_autonomy_support_score_pretransition, 'Homework monitoring frequency': homework_monitoring_frequency_pretransition, 'Parental knowledge of evening whereabouts': parental_knowledge_of_evening_whereabouts_pretransition, 'School-night curfew': school_night_curfew_pretransition, 'Parental educational aspiration': parental_educational_aspiration_candidate, 'Parental HE expectation': parental_he_expectation_candidate}, index=route_index)
shared_family_activity_ordinal_overlap_rows = []
for comparison_measure in ['Parent relationship-quality score', 'Parent communication-frequency score',
    'Parent discussion of school day', 'Parental autonomy-support score', 'Homework monitoring frequency', 'Parental knowledge of evening whereabouts', 'School-night curfew', 'Parental educational aspiration', 'Parental HE expectation']:
    pair_data = shared_family_activity_overlap_data[['Shared family activity frequency', comparison_measure]].dropna()
    assert len(pair_data) > 0
    shared_family_activity_ordinal_overlap_rows.append({'Comparison measure': comparison_measure,
        'Complete comparisons': len(pair_data), 'Spearman correlation': round(float(spearmanr(pair_data['Shared family activity frequency'],
        pair_data[comparison_measure]).statistic), 3), "Cramer's V": round(categorical_cramers_v(pair_data['Shared family activity frequency'],
        pair_data[comparison_measure]), 3)})
shared_family_activity_ordinal_overlap_summary = pd.DataFrame(shared_family_activity_ordinal_overlap_rows).assign(Absolute_correlation=lambda data: data['Spearman correlation'].abs()).sort_values('Absolute_correlation',
    ascending=False).drop(columns=['Absolute_correlation']).reset_index(drop=True)
shared_family_activity_response_type = pd.Series(pd.NA, index=route_index, dtype='string',
    name='Shared family activity response type')
shared_family_activity_response_type.loc[shared_family_activity_frequency_candidate.notna()] = 'Ordered frequency response'
shared_family_activity_response_type.loc[shared_family_activity_varies_mask] = 'Varies'
shared_family_activity_response_type_summary = shared_family_activity_response_type.value_counts(dropna=False).rename('Participants').rename_axis('Response treatment').reset_index()
shared_family_activity_varies_profile_rows = []
for comparison_measure in ['Parent relationship-quality score', 'Parent communication-frequency score',
    'Parent discussion of school day', 'Parental autonomy-support score']:
    profile_data = pd.DataFrame({'Response type': shared_family_activity_response_type,
        'Comparison measure': shared_family_activity_overlap_data[comparison_measure]}, index=route_index).dropna()
    for response_type, group_data in profile_data.groupby('Response type'):
        shared_family_activity_varies_profile_rows.append({'Comparison measure': comparison_measure,
            'Response type': response_type, 'Participants': len(group_data), 'Mean': round(float(group_data['Comparison measure'].astype(float).mean()),
            3), 'Median': round(float(group_data['Comparison measure'].astype(float).median()), 3)})
shared_family_activity_varies_profile_summary = pd.DataFrame(shared_family_activity_varies_profile_rows)
print('Candidate predictor summary:')
display_limited(shared_family_activity_candidate_summary)
print('Candidate frequency distribution:')
with pd.option_context('display.max_colwidth', None):
    display_limited(shared_family_activity_candidate_distribution)
print('Associations with retained predictors:')
display_limited(shared_family_activity_ordinal_overlap_summary)
print('Response treatment:')
display_limited(shared_family_activity_response_type_summary)
print("Family profile of ordered and 'Varies' responses:")
display_limited(shared_family_activity_varies_profile_summary)
print('Files written in this cell: 0')

Candidate predictor summary:


,Candidate predictor,Non-missing,Missing,Missing percentage,Distinct values,Minimum,Median,Mean,Maximum,Varies responses treated as missing
0,shared_family_activity_frequency_pretransition,9325,442,4.53,6,0,4.0,3.84,5,88


Candidate frequency distribution:


,Score,Category,Participants,Percentage among ordered responses
0,0,Never or hardly ever,329,3.53
1,1,About once a year,110,1.18
2,2,Two or three times a year,755,8.10
3,3,About once a month,1948,20.89
4,4,Two or three times a month,2589,27.76


Associations with retained predictors:


...

Response treatment:


,Response treatment,Participants
0,Ordered frequency response,9325
1,<NA>,354
2,Varies,88


Family profile of ordered and 'Varies' responses:


,Comparison measure,Response type,Participants,Mean,Median
0,Parent relationship-quality score,Ordered frequency response,8969,2.623,3.0
1,Parent relationship-quality score,Varies,82,2.518,2.5
2,Parent communication-frequency score,Ordered frequency response,8820,2.552,2.5
3,Parent communication-frequency score,Varies,81,2.463,2.5
4,Parent discussion of school day,Ordered frequency response,8900,1.404,1.0


Files written in this cell: 0


In [378]:
# 59: Shared family activity decisions

import pandas as pd
shared_family_activity_frequency_pretransition = shared_family_activity_frequency_candidate.copy().astype('Int64').rename('shared_family_activity_frequency_pretransition')
assert shared_family_activity_frequency_pretransition.index.equals(route_index)
observed_shared_activity_values = set(shared_family_activity_frequency_pretransition.dropna().astype(int).unique().tolist())
assert observed_shared_activity_values.issubset({0, 1, 2, 3, 4, 5})
shared_family_activity_predictors = pd.DataFrame({'NSID': route_index,
    'shared_family_activity_frequency_pretransition': shared_family_activity_frequency_pretransition.to_numpy()})
assert len(shared_family_activity_predictors) == 9767
assert shared_family_activity_predictors['NSID'].is_unique
shared_family_activity_variable_decisions = shared_family_activity_inventory.copy()
shared_family_activity_variable_decisions['Shared-activity decision'] = 'Construction input'
shared_family_activity_variable_decisions['Shared-activity role'] = 'Ordered frequency of shared family activity'
shared_family_activity_variable_decisions['Shared-activity reason'] = "The response categories are reverse-coded so that higher values indicate more frequent shared family activity. The non-ordered 'Varies' response is treated as missing"
shared_family_activity_representation_decisions = pd.DataFrame([{'Representation': 'Original seven-category response',
    'Decision': 'Do not retain', 'Reason': "The 'Varies' response does not have a defined position within the frequency ordering"}, {'Representation': 'Six-level ordered frequency measure',
    'Decision': 'Retain', 'Reason': 'The ordered categories provide a direct measure of shared family activity frequency and show limited overlap with the existing family predictors'}, {'Representation': "'Varies' as a separate predictor category",
    'Decision': 'Do not retain', 'Reason': 'Only 88 participants selected this response, and it does not represent a consistent frequency level'}, {'Representation': 'Binary frequent versus infrequent activity',
    'Decision': 'Do not construct', 'Reason': 'Dichotomisation would discard the available frequency distinctions without a clear substantive threshold'}])
shared_family_activity_predictor_summary = pd.DataFrame([{'Predictor': 'shared_family_activity_frequency_pretransition',
    'Non-missing': int(shared_family_activity_frequency_pretransition.notna().sum()), 'Missing': int(shared_family_activity_frequency_pretransition.isna().sum()), 'Missing percentage': round(shared_family_activity_frequency_pretransition.isna().mean() * 100,
    2), 'Distinct values': int(shared_family_activity_frequency_pretransition.nunique()), 'Mean': round(float(shared_family_activity_frequency_pretransition.mean()),
    2), 'Median': float(shared_family_activity_frequency_pretransition.median()), 'Maximum absolute Spearman correlation': round(float(shared_family_activity_ordinal_overlap_summary['Spearman correlation'].abs().max()),
    3), "Maximum Cramer's V": round(float(shared_family_activity_ordinal_overlap_summary["Cramer's V"].max()),
    3), "'Varies' responses treated as missing": int(shared_family_activity_varies_mask.sum())}])
shared_family_activity_retained_distribution_rows = []
retained_shared_activity_count = int(shared_family_activity_frequency_pretransition.notna().sum())
for score, category in shared_family_activity_frequency_labels.items():
    participants = int(shared_family_activity_frequency_pretransition.eq(score).sum())
    shared_family_activity_retained_distribution_rows.append({'Score': score, 'Category': category,
        'Participants': participants, 'Percentage among observed': round(participants / retained_shared_activity_count * 100,
        2)})
shared_family_activity_retained_distribution = pd.DataFrame(shared_family_activity_retained_distribution_rows)
shared_family_activity_decision_display = shared_family_activity_variable_decisions[['Wave', 'Source type',
    'Source file', 'Variable', 'Variable label', 'Timing status', 'Review source', 'Shared-activity decision', 'Shared-activity role', 'Shared-activity reason']].copy()
print('Shared family activity predictors retained: 1')
print('Retained predictor summary:')
display_limited(shared_family_activity_predictor_summary)
print('Retained category distribution:')
with pd.option_context('display.max_colwidth', None):
    display_limited(shared_family_activity_retained_distribution)
print('Representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(shared_family_activity_representation_decisions)
print('Variable decision:')
with pd.option_context('display.max_colwidth', None):
    display_limited(shared_family_activity_decision_display)
print('Files written in this cell: 0')

Shared family activity predictors retained: 1
Retained predictor summary:


,Predictor,Non-missing,Missing,Missing percentage,Distinct values,Mean,Median,Maximum absolute Spearman correlation,Maximum Cramer's V,'Varies' responses treated as missing
0,shared_family_activity_frequency_pretransition,9325,442,4.53,6,3.84,4.0,0.101,0.093,88


Retained category distribution:


,Score,Category,Participants,Percentage among observed
0,0,Never or hardly ever,329,3.53
1,1,About once a year,110,1.18
2,2,Two or three times a year,755,8.10
3,3,About once a month,1948,20.89
4,4,Two or three times a month,2589,27.76


Representation decisions:


,Representation,Decision,Reason
0,Original seven-category response,Do not retain,The 'Varies' response does not have a defined position within the frequency ordering
1,Six-level ordered frequency measure,Retain,The ordered categories provide a direct measure of shared family activity frequency and show limited overlap with the existing family predictors
2,'Varies' as a separate predictor category,Do not retain,"Only 88 participants selected this response, and it does not represent a consistent frequency level"
3,Binary frequent versus infrequent activity,Do not construct,Dichotomisation would discard the available frequency distinctions without a clear substantive threshold


Variable decision:


,Wave,Source type,Source file,Variable,Variable label,Timing status,Review source,Shared-activity decision,Shared-activity role,Shared-activity reason
0,Wave 1,Family background,wave_one_lsype_family_background_2020,W1fammusMP,MP: How often go out together as a family (excluding shopping),Pre-transition source,Original contextual review,Construction input,Ordered frequency of shared family activity,The response categories are reverse-coded so that higher values indicate more frequent shared family activity. The non-ordered 'Varies' response is treated as missing


Files written in this cell: 0


In [379]:
# 60: Teacher recommendation response and routing review

import pandas as pd
teacher_recommendation_inventory = domain_9_combined_contextual_inventory.loc[domain_9_combined_contextual_inventory['Domain 9 review track'].eq('Teacher recommendation reported by parent')].copy().reset_index(drop=True)
assert len(teacher_recommendation_inventory) == 1
assert teacher_recommendation_inventory.loc[0, 'Variable'] == 'W3tstaydefMP'
teacher_recommendation_source_file = teacher_recommendation_inventory.loc[0, 'Source file']
teacher_recommendation_source_path = source_file_lookup[teacher_recommendation_source_file]
teacher_recommendation_position = int(teacher_recommendation_inventory.loc[0, 'Variable position'])
teacher_recommendation_neighbourhood = stage_2_master_register_review.loc[stage_2_master_register_review['Source file'].eq(teacher_recommendation_source_file) & stage_2_master_register_review['Variable position'].between(teacher_recommendation_position - 7,
    teacher_recommendation_position + 7)].sort_values('Variable position')[['Variable position', 'Variable',
    'Variable label']].reset_index(drop=True)
teacher_recommendation_question_family = stage_2_master_register_review.loc[stage_2_master_register_review['Source file'].eq(teacher_recommendation_source_file) & (stage_2_master_register_review['Variable'].astype('string').str.contains('tstay|staydef|teach',
    case=False, na=False, regex=True) | stage_2_master_register_review['Variable label'].astype('string').str.contains('teacher.*stay|stay.*teacher|stay on in education',
    case=False, na=False, regex=True))].sort_values('Variable position')[['Variable position', 'Variable',
    'Variable label']].reset_index(drop=True)
teacher_recommendation_raw_source = pd.read_stata(teacher_recommendation_source_path, columns=['NSID', 'W3tstaydefMP'],
    convert_categoricals=False)
teacher_recommendation_labelled_source = pd.read_stata(teacher_recommendation_source_path, columns=['NSID',
    'W3tstaydefMP'], convert_categoricals=True)
for source_data in [teacher_recommendation_raw_source, teacher_recommendation_labelled_source]:
    source_data['NSID'] = standardise_nsid(source_data['NSID'])
    assert source_data['NSID'].is_unique
teacher_recommendation_unrestricted_raw = teacher_recommendation_raw_source.set_index('NSID')['W3tstaydefMP'].reindex(route_index)
teacher_recommendation_unrestricted_raw = pd.to_numeric(teacher_recommendation_unrestricted_raw, errors='coerce')
teacher_recommendation_unrestricted_labelled = teacher_recommendation_labelled_source.set_index('NSID')['W3tstaydefMP'].reindex(route_index).astype('string')
teacher_recommendation_timing_mask = pd.Series(wave_3_pretransition_interview_mask,
    index=getattr(wave_3_pretransition_interview_mask, 'index',
    route_index)).reindex(route_index).fillna(False).astype(bool)
teacher_recommendation_raw = teacher_recommendation_unrestricted_raw.where(teacher_recommendation_timing_mask)
teacher_recommendation_labelled = teacher_recommendation_unrestricted_labelled.where(teacher_recommendation_timing_mask)
teacher_recommendation_removed_by_timing = int((teacher_recommendation_unrestricted_raw.notna() & ~teacher_recommendation_timing_mask).sum())
teacher_recommendation_observed_mask = teacher_recommendation_raw.ge(0).fillna(False).astype(bool)
teacher_recommendation_structural_mask = teacher_recommendation_raw.eq(-91.0).fillna(False).astype(bool)
teacher_recommendation_other_special_mask = teacher_recommendation_raw.lt(0).fillna(False).astype(bool) & ~teacher_recommendation_structural_mask
teacher_recommendation_observed = teacher_recommendation_raw.where(teacher_recommendation_observed_mask).rename('teacher_recommendation_observed')
teacher_recommendation_coverage_summary = pd.DataFrame([{'Variable': 'W3tstaydefMP',
    'Variable label': teacher_recommendation_inventory.loc[0,
    'Variable label'], 'Observed responses': int(teacher_recommendation_observed_mask.sum()), 'Observed percentage': round(teacher_recommendation_observed_mask.mean() * 100,
    2), 'Structurally not applicable': int(teacher_recommendation_structural_mask.sum()), 'Other special-code responses': int(teacher_recommendation_other_special_mask.sum()), 'No source record or timing-restricted': int(teacher_recommendation_raw.isna().sum()), 'Removed by Wave 3 timing restriction': teacher_recommendation_removed_by_timing, 'Observed categories': int(teacher_recommendation_observed.nunique())}])
teacher_recommendation_code_rows = []
for raw_code, participants in teacher_recommendation_raw.value_counts(dropna=False).items():
    if pd.isna(raw_code):
        value_label = 'No source record or timing-restricted'
        response_type = 'Unavailable'
        sort_value = 999999
    else:
        response_mask = teacher_recommendation_raw.eq(raw_code)
        matching_labels = teacher_recommendation_labelled.loc[response_mask].dropna().drop_duplicates().tolist()
        value_label = matching_labels[0] if matching_labels else str(raw_code)
        if raw_code >= 0:
            response_type = 'Observed response'
        elif raw_code == -91:
            response_type = 'Structural non-applicability'
        else:
            response_type = 'Other special code'
        sort_value = float(raw_code)
    teacher_recommendation_code_rows.append({'Raw code': raw_code, 'Value label': value_label,
        'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
teacher_recommendation_code_distribution = pd.DataFrame(teacher_recommendation_code_rows).sort_values('Sort value').drop(columns=['Sort value']).reset_index(drop=True)
print('Teacher-recommendation variable inventory:')
with pd.option_context('display.max_colwidth', None):
    display_limited(teacher_recommendation_inventory[['Wave', 'Source type', 'Source file', 'Variable position',
        'Variable', 'Variable label', 'Timing status', 'Domain 9 review track', 'Review source']])
print('Neighbouring variables:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(teacher_recommendation_neighbourhood)
print('Potentially related question-family variables:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(teacher_recommendation_question_family)
print('Coverage after the Wave 3 timing restriction:')
with pd.option_context('display.max_colwidth', None):
    display_limited(teacher_recommendation_coverage_summary)
print('Response-code distribution:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(teacher_recommendation_code_distribution)
print('Files written in this cell: 0')

Teacher-recommendation variable inventory:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Domain 9 review track,Review source
0,Wave 3,Parental attitudes,wave_three_lsype_parental_attitudes_file_16_06_08,15,W3tstaydefMP,MP: Whether teacher though YP should stay on in education after Year 11,Near-transition source,Teacher recommendation reported by parent,Original contextual review


Neighbouring variables:


,Variable position,Variable,Variable label
0,8,W3intmnthMP,MP- month of interview
1,9,W3intyearMP,MP- year of interview
2,10,W3pareveMP,MP: Whether self or partner have been to any parents' evenings or similar events
3,11,W3tmeetfMP,MP: Whether had any specially arranged meetings with teachers about YP's schooli
4,12,W3tspeakMP,MP: How often speak to YP's teachers about schooling


Potentially related question-family variables:


...

Coverage after the Wave 3 timing restriction:


,Variable,Variable label,Observed responses,Observed percentage,Structurally not applicable,Other special-code responses,No source record or timing-restricted,Removed by Wave 3 timing restriction,Observed categories
0,W3tstaydefMP,MP: Whether teacher though YP should stay on in education after Year 11,3225,33.02,6129,141,272,14,2


Response-code distribution:


,Raw code,Value label,Response type,Participants
0,-99.0,MP not interviewed,Other special code,81
1,-91.0,Not applicable,Structural non-applicability,6129
2,-1.0,Don't know,Other special code,60
3,1.0,Yes,Observed response,2799
4,2.0,No,Observed response,426


Files written in this cell: 0


In [380]:
# 61: Teacher recommendation routing diagnosis

import pandas as pd
teacher_recommendation_routing_variables = ['W3tspeak2MP', 'W3tstayMP', 'W3tstaydefMP', 'W3tappMP', 'W3tappdefMP']
teacher_recommendation_source_variables = set(stage_2_master_register_review.loc[stage_2_master_register_review['Source file'].eq(teacher_recommendation_source_file),
    'Variable'].astype(str))
assert set(teacher_recommendation_routing_variables).issubset(teacher_recommendation_source_variables)
teacher_recommendation_routing_raw_source = pd.read_stata(teacher_recommendation_source_path, columns=['NSID',
    *teacher_recommendation_routing_variables], convert_categoricals=False)
teacher_recommendation_routing_labelled_source = pd.read_stata(teacher_recommendation_source_path, columns=['NSID',
    *teacher_recommendation_routing_variables], convert_categoricals=True)
for source_data in [teacher_recommendation_routing_raw_source, teacher_recommendation_routing_labelled_source]:
    source_data['NSID'] = standardise_nsid(source_data['NSID'])
    assert source_data['NSID'].is_unique
teacher_recommendation_routing_raw = teacher_recommendation_routing_raw_source.set_index('NSID').reindex(route_index)
teacher_recommendation_routing_labelled = teacher_recommendation_routing_labelled_source.set_index('NSID').reindex(route_index)
for variable in teacher_recommendation_routing_variables:
    teacher_recommendation_routing_raw[variable] = pd.to_numeric(teacher_recommendation_routing_raw[variable],
        errors='coerce')
    teacher_recommendation_routing_labelled[variable] = teacher_recommendation_routing_labelled[variable].astype('string')
teacher_recommendation_routing_raw = teacher_recommendation_routing_raw.where(teacher_recommendation_timing_mask,
    axis=0)
teacher_recommendation_routing_labelled = teacher_recommendation_routing_labelled.where(teacher_recommendation_timing_mask,
    axis=0)
teacher_recommendation_routing_metadata = stage_2_master_register_review.loc[stage_2_master_register_review['Source file'].eq(teacher_recommendation_source_file) & stage_2_master_register_review['Variable'].isin(teacher_recommendation_routing_variables)][['Variable position',
    'Variable', 'Variable label']].sort_values('Variable position').reset_index(drop=True)
teacher_recommendation_routing_code_rows = []
for _, metadata_row in teacher_recommendation_routing_metadata.iterrows():
    variable = metadata_row['Variable']
    raw_values = teacher_recommendation_routing_raw[variable]
    labelled_values = teacher_recommendation_routing_labelled[variable]
    for raw_code, participants in raw_values.value_counts(dropna=False).items():
        if pd.isna(raw_code):
            value_label = 'No source record or timing-restricted'
            response_type = 'Unavailable'
            sort_value = 999999
        else:
            response_mask = raw_values.eq(raw_code)
            matching_labels = labelled_values.loc[response_mask].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            if raw_code >= 0:
                response_type = 'Observed response'
            elif raw_code == -91:
                response_type = 'Structural non-applicability'
            else:
                response_type = 'Other special code'
            sort_value = float(raw_code)
        teacher_recommendation_routing_code_rows.append({'Variable position': int(metadata_row['Variable position']),
            'Variable': variable, 'Variable label': metadata_row['Variable label'], 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
teacher_recommendation_routing_code_distribution = pd.DataFrame(teacher_recommendation_routing_code_rows).sort_values(['Variable position',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
general_discussion_by_staying_discussion = pd.crosstab(teacher_recommendation_routing_labelled['W3tspeak2MP'].fillna('Unavailable'),
    teacher_recommendation_routing_labelled['W3tstayMP'].fillna('Unavailable'), margins=True, dropna=False)
staying_discussion_by_teacher_recommendation = pd.crosstab(teacher_recommendation_routing_labelled['W3tstayMP'].fillna('Unavailable'),
    teacher_recommendation_routing_labelled['W3tstaydefMP'].fillna('Unavailable'), margins=True, dropna=False)
general_discussion_by_apprenticeship_discussion = pd.crosstab(teacher_recommendation_routing_labelled['W3tspeak2MP'].fillna('Unavailable'),
    teacher_recommendation_routing_labelled['W3tappMP'].fillna('Unavailable'), margins=True, dropna=False)
apprenticeship_discussion_by_teacher_recommendation = pd.crosstab(teacher_recommendation_routing_labelled['W3tappMP'].fillna('Unavailable'),
    teacher_recommendation_routing_labelled['W3tappdefMP'].fillna('Unavailable'), margins=True, dropna=False)
staying_recommendation_routing_combinations = teacher_recommendation_routing_raw[['W3tspeak2MP', 'W3tstayMP',
    'W3tstaydefMP']].value_counts(dropna=False).rename('Participants').reset_index().sort_values(['W3tspeak2MP',
    'W3tstayMP', 'W3tstaydefMP'], na_position='last').reset_index(drop=True)
apprenticeship_recommendation_routing_combinations = teacher_recommendation_routing_raw[['W3tspeak2MP', 'W3tappMP',
    'W3tappdefMP']].value_counts(dropna=False).rename('Participants').reset_index().sort_values(['W3tspeak2MP',
    'W3tappMP', 'W3tappdefMP'], na_position='last').reset_index(drop=True)
staying_recommendation_structural_mask = teacher_recommendation_routing_raw['W3tstaydefMP'].eq(-91.0).fillna(False).astype(bool)
staying_recommendation_structural_routing_summary = teacher_recommendation_routing_labelled.loc[staying_recommendation_structural_mask,
    ['W3tspeak2MP',
    'W3tstayMP']].value_counts(dropna=False).rename('Participants').reset_index().sort_values('Participants',
    ascending=False).reset_index(drop=True)
staying_recommendation_observed_mask = teacher_recommendation_routing_raw['W3tstaydefMP'].isin([1.0,
    2.0]).fillna(False).astype(bool)
staying_recommendation_observed_routing_summary = teacher_recommendation_routing_labelled.loc[staying_recommendation_observed_mask,
    ['W3tspeak2MP', 'W3tstayMP',
    'W3tstaydefMP']].value_counts(dropna=False).rename('Participants').reset_index().sort_values('Participants',
    ascending=False).reset_index(drop=True)
print('Teacher-recommendation routing variables:')
with pd.option_context('display.max_colwidth', None):
    display_limited(teacher_recommendation_routing_metadata)
print('Response-code distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(teacher_recommendation_routing_code_distribution)
print('General post-16 discussion by staying-on discussion:')
with pd.option_context('display.max_colwidth', None):
    display_limited(general_discussion_by_staying_discussion)
print('Staying-on discussion by teacher recommendation:')
with pd.option_context('display.max_colwidth', None):
    display_limited(staying_discussion_by_teacher_recommendation)
print('General post-16 discussion by apprenticeship discussion:')
with pd.option_context('display.max_colwidth', None):
    display_limited(general_discussion_by_apprenticeship_discussion)
print('Apprenticeship discussion by teacher recommendation:')
with pd.option_context('display.max_colwidth', None):
    display_limited(apprenticeship_discussion_by_teacher_recommendation)
print('Raw staying-on routing combinations:')
display_limited(staying_recommendation_routing_combinations)
print('Raw apprenticeship routing combinations:')
display_limited(apprenticeship_recommendation_routing_combinations)
print('Routing responses when the staying-on recommendation item is structurally not applicable:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(staying_recommendation_structural_routing_summary)
print('Routing responses when the staying-on recommendation item is observed:')
with pd.option_context('display.max_colwidth', None):
    display_limited(staying_recommendation_observed_routing_summary)
print('Files written in this cell: 0')

Teacher-recommendation routing variables:


,Variable position,Variable,Variable label
0,13,W3tspeak2MP,MP: Whether talked to teachers about what YP might do after Year 11
1,14,W3tstayMP,MP: Whether talked to teachers about YP staying on in education after Year 11
2,15,W3tstaydefMP,MP: Whether teacher though YP should stay on in education after Year 11
3,16,W3tappMP,MP: Whether talked to teachers about YP applying for a training place/apprentice
4,17,W3tappdefMP,MP: Whether teacher thought YP should apply for a training place/apprenticeship


Response-code distributions:


,Variable position,Variable,Variable label,Raw code,Value label,Response type,Participants
0,13,W3tspeak2MP,MP: Whether talked to teachers about what YP might do after Year 11,-99.0,MP not interviewed,Other special code,81
1,13,W3tspeak2MP,MP: Whether talked to teachers about what YP might do after Year 11,-1.0,Don't know,Other special code,34
2,13,W3tspeak2MP,MP: Whether talked to teachers about what YP might do after Year 11,1.0,Yes,Observed response,4256
3,13,W3tspeak2MP,MP: Whether talked to teachers about what YP might do after Year 11,2.0,No,Observed response,5124
4,13,W3tspeak2MP,MP: Whether talked to teachers about what YP might do after Year 11,NaN,No source record or timing-restricted,Unavailable,272


General post-16 discussion by staying-on discussion:


...

Routing responses when the staying-on recommendation item is structurally not applicable:


,W3tspeak2MP,W3tstayMP,Participants
0,No,Not applicable,5124
1,Yes,No,954
2,Don't know,Not applicable,34
3,Yes,Don't know,17


Routing responses when the staying-on recommendation item is observed:


,W3tspeak2MP,W3tstayMP,W3tstaydefMP,Participants
0,Yes,Yes,Yes,2799
1,Yes,Yes,No,426


Files written in this cell: 0


In [381]:
# 62: Teacher recommendation representation review

import pandas as pd
general_post16_discussion_raw = teacher_recommendation_routing_raw['W3tspeak2MP']
staying_discussion_raw = teacher_recommendation_routing_raw['W3tstayMP']
staying_recommendation_raw = teacher_recommendation_routing_raw['W3tstaydefMP']
apprenticeship_discussion_raw = teacher_recommendation_routing_raw['W3tappMP']
apprenticeship_recommendation_raw = teacher_recommendation_routing_raw['W3tappdefMP']
general_discussion_yes_mask = general_post16_discussion_raw.eq(1.0).fillna(False).astype(bool)
general_discussion_no_mask = general_post16_discussion_raw.eq(2.0).fillna(False).astype(bool)
staying_discussion_yes_mask = staying_discussion_raw.eq(1.0).fillna(False).astype(bool)
staying_discussion_no_mask = staying_discussion_raw.eq(2.0).fillna(False).astype(bool)
staying_recommendation_yes_mask = staying_recommendation_raw.eq(1.0).fillna(False).astype(bool)
staying_recommendation_no_mask = staying_recommendation_raw.eq(2.0).fillna(False).astype(bool)
apprenticeship_discussion_yes_mask = apprenticeship_discussion_raw.eq(1.0).fillna(False).astype(bool)
apprenticeship_discussion_no_mask = apprenticeship_discussion_raw.eq(2.0).fillna(False).astype(bool)
apprenticeship_recommendation_yes_mask = apprenticeship_recommendation_raw.eq(1.0).fillna(False).astype(bool)
apprenticeship_recommendation_no_mask = apprenticeship_recommendation_raw.eq(2.0).fillna(False).astype(bool)
assert (staying_recommendation_yes_mask | staying_recommendation_no_mask).equals(general_discussion_yes_mask & staying_discussion_yes_mask & staying_recommendation_raw.isin([1.0,
    2.0]))
assert (apprenticeship_recommendation_yes_mask | apprenticeship_recommendation_no_mask).equals(general_discussion_yes_mask & apprenticeship_discussion_yes_mask & apprenticeship_recommendation_raw.isin([1.0,
    2.0]))
teacher_staying_on_recommendation_profile_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='teacher_staying_on_recommendation_profile_pretransition')
teacher_staying_on_recommendation_profile_candidate.loc[general_discussion_no_mask] = 0
teacher_staying_on_recommendation_profile_candidate.loc[general_discussion_yes_mask & staying_discussion_no_mask] = 1
teacher_staying_on_recommendation_profile_candidate.loc[general_discussion_yes_mask & staying_discussion_yes_mask & staying_recommendation_no_mask] = 2
teacher_staying_on_recommendation_profile_candidate.loc[general_discussion_yes_mask & staying_discussion_yes_mask & staying_recommendation_yes_mask] = 3
teacher_apprenticeship_recommendation_profile_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='teacher_apprenticeship_recommendation_profile_pretransition')
teacher_apprenticeship_recommendation_profile_candidate.loc[general_discussion_no_mask] = 0
teacher_apprenticeship_recommendation_profile_candidate.loc[general_discussion_yes_mask & apprenticeship_discussion_no_mask] = 1
teacher_apprenticeship_recommendation_profile_candidate.loc[general_discussion_yes_mask & apprenticeship_discussion_yes_mask & apprenticeship_recommendation_no_mask] = 2
teacher_apprenticeship_recommendation_profile_candidate.loc[general_discussion_yes_mask & apprenticeship_discussion_yes_mask & apprenticeship_recommendation_yes_mask] = 3
for candidate_values in [teacher_staying_on_recommendation_profile_candidate,
    teacher_apprenticeship_recommendation_profile_candidate]:
    observed_values = set(candidate_values.dropna().astype(int).unique().tolist())
    assert observed_values.issubset({0, 1, 2, 3})
teacher_staying_on_recommendation_binary_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='teacher_staying_on_recommendation_conditional_pretransition')
teacher_staying_on_recommendation_binary_candidate.loc[staying_recommendation_no_mask] = 0
teacher_staying_on_recommendation_binary_candidate.loc[staying_recommendation_yes_mask] = 1
teacher_apprenticeship_recommendation_binary_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='teacher_apprenticeship_recommendation_conditional_pretransition')
teacher_apprenticeship_recommendation_binary_candidate.loc[apprenticeship_recommendation_no_mask] = 0
teacher_apprenticeship_recommendation_binary_candidate.loc[apprenticeship_recommendation_yes_mask] = 1
teacher_recommendation_candidate_summary = pd.DataFrame([{'Candidate predictor': 'teacher_staying_on_recommendation_profile_pretransition',
    'Non-missing': int(teacher_staying_on_recommendation_profile_candidate.notna().sum()), 'Missing': int(teacher_staying_on_recommendation_profile_candidate.isna().sum()), 'Missing percentage': round(teacher_staying_on_recommendation_profile_candidate.isna().mean() * 100,
    2), 'Categories': int(teacher_staying_on_recommendation_profile_candidate.nunique()), 'Direct recommendation responses': int(teacher_staying_on_recommendation_binary_candidate.notna().sum())}, {'Candidate predictor': 'teacher_apprenticeship_recommendation_profile_pretransition',
    'Non-missing': int(teacher_apprenticeship_recommendation_profile_candidate.notna().sum()), 'Missing': int(teacher_apprenticeship_recommendation_profile_candidate.isna().sum()), 'Missing percentage': round(teacher_apprenticeship_recommendation_profile_candidate.isna().mean() * 100,
    2), 'Categories': int(teacher_apprenticeship_recommendation_profile_candidate.nunique()), 'Direct recommendation responses': int(teacher_apprenticeship_recommendation_binary_candidate.notna().sum())}])
teacher_recommendation_profile_labels = {'Staying-on recommendation profile': {0: 'No parent–teacher post-16 discussion',
    1: 'Post-16 discussion, staying on not discussed', 2: 'Staying on discussed, teacher did not recommend', 3: 'Teacher recommended staying on'}, 'Apprenticeship recommendation profile': {0: 'No parent–teacher post-16 discussion',
    1: 'Post-16 discussion, apprenticeship not discussed', 2: 'Apprenticeship discussed, teacher did not recommend', 3: 'Teacher recommended apprenticeship'}}
teacher_recommendation_profile_values = {'Staying-on recommendation profile': teacher_staying_on_recommendation_profile_candidate,
    'Apprenticeship recommendation profile': teacher_apprenticeship_recommendation_profile_candidate}
teacher_recommendation_distribution_rows = []
for profile_name, category_labels in teacher_recommendation_profile_labels.items():
    profile_values = teacher_recommendation_profile_values[profile_name]
    observed_count = int(profile_values.notna().sum())
    for code, category in category_labels.items():
        participants = int(profile_values.eq(code).sum())
        teacher_recommendation_distribution_rows.append({'Candidate profile': profile_name, 'Code': code,
            'Category': category, 'Participants': participants, 'Percentage among observed': round(participants / observed_count * 100,
            2)})
teacher_recommendation_profile_distributions = pd.DataFrame(teacher_recommendation_distribution_rows)
teacher_recommendation_conditional_summary = pd.DataFrame([{'Recommendation branch': 'Staying on',
    'Observed recommendation responses': int(teacher_staying_on_recommendation_binary_candidate.notna().sum()), 'Teacher recommended': int(teacher_staying_on_recommendation_binary_candidate.eq(1).sum()), 'Teacher did not recommend': int(teacher_staying_on_recommendation_binary_candidate.eq(0).sum()), 'Recommendation percentage': round(teacher_staying_on_recommendation_binary_candidate.eq(1).sum() / teacher_staying_on_recommendation_binary_candidate.notna().sum() * 100,
    2)}, {'Recommendation branch': 'Apprenticeship',
    'Observed recommendation responses': int(teacher_apprenticeship_recommendation_binary_candidate.notna().sum()), 'Teacher recommended': int(teacher_apprenticeship_recommendation_binary_candidate.eq(1).sum()), 'Teacher did not recommend': int(teacher_apprenticeship_recommendation_binary_candidate.eq(0).sum()), 'Recommendation percentage': round(teacher_apprenticeship_recommendation_binary_candidate.eq(1).sum() / teacher_apprenticeship_recommendation_binary_candidate.notna().sum() * 100,
    2)}])
teacher_recommendation_profile_pair = pd.DataFrame({'Staying-on recommendation profile': teacher_staying_on_recommendation_profile_candidate,
    'Apprenticeship recommendation profile': teacher_apprenticeship_recommendation_profile_candidate}, index=route_index).dropna()
teacher_recommendation_profile_pair_summary = pd.DataFrame([{'Complete comparisons': len(teacher_recommendation_profile_pair),
    "Cramer's V": round(categorical_cramers_v(teacher_recommendation_profile_pair['Staying-on recommendation profile'],
    teacher_recommendation_profile_pair['Apprenticeship recommendation profile']), 3)}])
teacher_recommendation_profile_crosstab = pd.crosstab(teacher_recommendation_profile_pair['Staying-on recommendation profile'],
    teacher_recommendation_profile_pair['Apprenticeship recommendation profile'], margins=True, dropna=False)
teacher_recommendation_overlap_data = pd.DataFrame({'Staying-on recommendation profile': teacher_staying_on_recommendation_profile_candidate,
    'Apprenticeship recommendation profile': teacher_apprenticeship_recommendation_profile_candidate, 'Conditional staying-on recommendation': teacher_staying_on_recommendation_binary_candidate, 'Conditional apprenticeship recommendation': teacher_apprenticeship_recommendation_binary_candidate, 'Parent–teacher post-16 discussion profile': parent_teacher_post16_discussion_profile_pretransition, 'Expected post-16 route': saved_educational_aspiration_predictors['expected_post16_route'], 'Higher-education application likelihood': saved_educational_aspiration_predictors['higher_education_application_likelihood'], 'Parental HE expectation': parental_he_expectation_candidate, 'Parental educational aspiration': parental_educational_aspiration_candidate, 'Parental training discussion profile': parental_training_apprenticeship_discussion_profile_pretransition}, index=route_index)
teacher_recommendation_profile_overlap_rows = []
for candidate_measure in ['Staying-on recommendation profile', 'Apprenticeship recommendation profile']:
    for comparison_measure in ['Parent–teacher post-16 discussion profile', 'Expected post-16 route',
        'Higher-education application likelihood', 'Parental HE expectation', 'Parental educational aspiration', 'Parental training discussion profile']:
        pair_data = teacher_recommendation_overlap_data[[candidate_measure, comparison_measure]].dropna()
        assert len(pair_data) > 0
        teacher_recommendation_profile_overlap_rows.append({'Candidate profile': candidate_measure,
            'Comparison measure': comparison_measure, 'Complete comparisons': len(pair_data), "Cramer's V": round(categorical_cramers_v(pair_data[candidate_measure],
            pair_data[comparison_measure]), 3)})
teacher_recommendation_profile_overlap_summary = pd.DataFrame(teacher_recommendation_profile_overlap_rows).sort_values(['Candidate profile',
    "Cramer's V"], ascending=[True, False]).reset_index(drop=True)
teacher_recommendation_conditional_overlap_rows = []
for candidate_measure in ['Conditional staying-on recommendation', 'Conditional apprenticeship recommendation']:
    for comparison_measure in ['Expected post-16 route', 'Higher-education application likelihood',
        'Parental HE expectation', 'Parental educational aspiration', 'Parental training discussion profile']:
        pair_data = teacher_recommendation_overlap_data[[candidate_measure, comparison_measure]].dropna()
        if len(pair_data) == 0:
            continue
        teacher_recommendation_conditional_overlap_rows.append({'Conditional recommendation': candidate_measure,
            'Comparison measure': comparison_measure, 'Complete comparisons': len(pair_data), "Cramer's V": round(categorical_cramers_v(pair_data[candidate_measure],
            pair_data[comparison_measure]), 3)})
teacher_recommendation_conditional_overlap_summary = pd.DataFrame(teacher_recommendation_conditional_overlap_rows).sort_values(['Conditional recommendation',
    "Cramer's V"], ascending=[True, False]).reset_index(drop=True)
print('Candidate predictor summary:')
display_limited(teacher_recommendation_candidate_summary)
print('Routing-aware profile distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(teacher_recommendation_profile_distributions)
print('Conditional recommendation responses:')
display_limited(teacher_recommendation_conditional_summary)
print('Association between the two profiles:')
display_limited(teacher_recommendation_profile_pair_summary)
print('Profile cross-tabulation:')
display_limited(teacher_recommendation_profile_crosstab)
print('Associations between profiles and retained predictors:')
display_limited(teacher_recommendation_profile_overlap_summary)
print('Conditional recommendation associations:')
display_limited(teacher_recommendation_conditional_overlap_summary)
print('Files written in this cell: 0')

Candidate predictor summary:


,Candidate predictor,Non-missing,Missing,Missing percentage,Categories,Direct recommendation responses
0,teacher_staying_on_recommendation_profile_pret...,9303,464,4.75,4,3225
1,teacher_apprenticeship_recommendation_profile_...,9330,437,4.47,4,727


Routing-aware profile distributions:


,Candidate profile,Code,Category,Participants,Percentage among observed
0,Staying-on recommendation profile,0,No parent–teacher post-16 discussion,5124,55.08
1,Staying-on recommendation profile,1,"Post-16 discussion, staying on not discussed",954,10.25
2,Staying-on recommendation profile,2,"Staying on discussed, teacher did not recommend",426,4.58
3,Staying-on recommendation profile,3,Teacher recommended staying on,2799,30.09
4,Apprenticeship recommendation profile,0,No parent–teacher post-16 discussion,5124,54.92


Conditional recommendation responses:


...

Associations between profiles and retained predictors:


,Candidate profile,Comparison measure,Complete comparisons,Cramer's V
0,Apprenticeship recommendation profile,Parent–teacher post-16 discussion profile,9315,0.816
1,Apprenticeship recommendation profile,Expected post-16 route,9316,0.171
2,Apprenticeship recommendation profile,Higher-education application likelihood,9312,0.149
3,Apprenticeship recommendation profile,Parental training discussion profile,9176,0.118
4,Apprenticeship recommendation profile,Parental HE expectation,8709,0.117


Conditional recommendation associations:


,Conditional recommendation,Comparison measure,Complete comparisons,Cramer's V
0,Conditional apprenticeship recommendation,Expected post-16 route,724,0.326
1,Conditional apprenticeship recommendation,Higher-education application likelihood,724,0.278
2,Conditional apprenticeship recommendation,Parental HE expectation,668,0.206
3,Conditional apprenticeship recommendation,Parental training discussion profile,705,0.173
4,Conditional apprenticeship recommendation,Parental educational aspiration,713,0.062


Files written in this cell: 0


In [382]:
# 63: Teacher recommendation decisions

import pandas as pd
teacher_recommendation_variable_decisions = stage_2_master_register_review.loc[stage_2_master_register_review['Source file'].eq(teacher_recommendation_source_file) & stage_2_master_register_review['Variable'].isin(['W3tspeak2MP',
    'W3tstayMP', 'W3tstaydefMP', 'W3tappMP', 'W3tappdefMP'])].copy().sort_values('Variable position').reset_index(drop=True)
assert set(teacher_recommendation_variable_decisions['Variable']) == {'W3tspeak2MP', 'W3tstayMP', 'W3tstaydefMP',
    'W3tappMP', 'W3tappdefMP'}
teacher_recommendation_variable_decisions['Teacher-recommendation decision'] = pd.NA
teacher_recommendation_variable_decisions['Teacher-recommendation role'] = pd.NA
teacher_recommendation_variable_decisions['Teacher-recommendation reason'] = pd.NA
general_discussion_mask = teacher_recommendation_variable_decisions['Variable'].eq('W3tspeak2MP')
branch_discussion_mask = teacher_recommendation_variable_decisions['Variable'].isin(['W3tstayMP', 'W3tappMP'])
direct_recommendation_mask = teacher_recommendation_variable_decisions['Variable'].isin(['W3tstaydefMP',
    'W3tappdefMP'])
teacher_recommendation_variable_decisions.loc[general_discussion_mask,
    'Teacher-recommendation decision'] = 'Review support'
teacher_recommendation_variable_decisions.loc[general_discussion_mask,
    'Teacher-recommendation role'] = 'General routing input for post-16 discussion'
teacher_recommendation_variable_decisions.loc[general_discussion_mask,
    'Teacher-recommendation reason'] = 'The variable establishes whether the recommendation branches were applicable. It is already a construction input for the retained parent–teacher post-16 discussion profile'
teacher_recommendation_variable_decisions.loc[branch_discussion_mask,
    'Teacher-recommendation decision'] = 'Review support'
teacher_recommendation_variable_decisions.loc[branch_discussion_mask,
    'Teacher-recommendation role'] = 'Branch-routing input for topic-specific teacher discussion'
teacher_recommendation_variable_decisions.loc[branch_discussion_mask,
    'Teacher-recommendation reason'] = 'The variables identify whether staying on or apprenticeship was discussed. They are already construction inputs for the retained parent–teacher post-16 discussion profile'
teacher_recommendation_variable_decisions.loc[direct_recommendation_mask,
    'Teacher-recommendation decision'] = 'Exclude'
teacher_recommendation_variable_decisions.loc[direct_recommendation_mask,
    'Teacher-recommendation role'] = 'Conditional teacher-recommendation response'
teacher_recommendation_variable_decisions.loc[direct_recommendation_mask,
    'Teacher-recommendation reason'] = 'The item is observed only among parents who reached the relevant discussion branch. A routing-aware full-sample profile strongly duplicates the retained parent–teacher post-16 discussion profile, while the conditional response does not provide a comparable measure for the full sample'
assert teacher_recommendation_variable_decisions['Teacher-recommendation decision'].notna().all()
assert teacher_recommendation_variable_decisions['Teacher-recommendation role'].notna().all()
assert teacher_recommendation_variable_decisions['Teacher-recommendation reason'].notna().all()
teacher_recommendation_representation_decisions = pd.DataFrame([{'Representation': 'Teacher recommendation used as a stand-alone yes/no predictor',
    'Decision': 'Do not retain', 'Reason': 'The recommendation items are conditional on prior parent–teacher discussion and cannot be interpreted for participants who did not reach the branch'}, {'Representation': 'Routing-aware staying-on recommendation profile',
    'Decision': 'Do not retain', 'Reason': "The profile has Cramer's V of 0.790 with the retained parent–teacher post-16 discussion profile because most categories reproduce the same routing structure"}, {'Representation': 'Routing-aware apprenticeship recommendation profile',
    'Decision': 'Do not retain', 'Reason': "The profile has Cramer's V of 0.816 with the retained parent–teacher post-16 discussion profile because most categories reproduce the same routing structure"}, {'Representation': 'Conditional staying-on recommendation',
    'Decision': 'Do not retain', 'Reason': 'The measure is available for only 3,225 participants who discussed staying on with a teacher and is not comparable across the full analysis sample'}, {'Representation': 'Conditional apprenticeship recommendation',
    'Decision': 'Do not retain', 'Reason': 'The measure is available for only 727 participants who discussed apprenticeship with a teacher and is too selectively observed for a general predictor set'}, {'Representation': 'Combined teacher-recommendation profile',
    'Decision': 'Do not construct', 'Reason': 'Combining the two branches would retain the underlying routing structure while mixing recommendations for substantively different post-16 routes'}])
teacher_recommendation_track_summary = pd.DataFrame([{'Review track': 'Teacher recommendation reported by parent',
    'Variables reviewed': int(len(teacher_recommendation_variable_decisions)), 'Direct recommendation variables': int(direct_recommendation_mask.sum()), 'Predictors retained': 0, 'Maximum overlap with retained discussion profile': 0.816, 'Decision': 'No additional predictor retained'}])
teacher_recommendation_decision_display = teacher_recommendation_variable_decisions[['Wave', 'Source type',
    'Source file', 'Variable position', 'Variable', 'Variable label', 'Teacher-recommendation decision', 'Teacher-recommendation role', 'Teacher-recommendation reason']].copy()
print('Teacher recommendation predictors retained: 0')
print('Track decision summary:')
display_limited(teacher_recommendation_track_summary)
print('Representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(teacher_recommendation_representation_decisions)
print('Variable decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(teacher_recommendation_decision_display)
print('Files written in this cell: 0')

Teacher recommendation predictors retained: 0
Track decision summary:


,Review track,Variables reviewed,Direct recommendation variables,Predictors retained,Maximum overlap with retained discussion profile,Decision
0,Teacher recommendation reported by parent,5,2,0,0.816,No additional predictor retained


Representation decisions:


,Representation,Decision,Reason
0,Teacher recommendation used as a stand-alone yes/no predictor,Do not retain,The recommendation items are conditional on prior parent–teacher discussion and cannot be interpreted for participants who did not reach the branch
1,Routing-aware staying-on recommendation profile,Do not retain,The profile has Cramer's V of 0.790 with the retained parent–teacher post-16 discussion profile because most categories reproduce the same routing structure
2,Routing-aware apprenticeship recommendation profile,Do not retain,The profile has Cramer's V of 0.816 with the retained parent–teacher post-16 discussion profile because most categories reproduce the same routing structure
3,Conditional staying-on recommendation,Do not retain,"The measure is available for only 3,225 participants who discussed staying on with a teacher and is not comparable across the full analysis sample"
4,Conditional apprenticeship recommendation,Do not retain,The measure is available for only 727 participants who discussed apprenticeship with a teacher and is too selectively observed for a general predictor set


Variable decisions:


,Wave,Source type,Source file,Variable position,Variable,Variable label,Teacher-recommendation decision,Teacher-recommendation role,Teacher-recommendation reason
0,Wave 3,Parental attitudes,wave_three_lsype_parental_attitudes_file_16_06_08,13,W3tspeak2MP,MP: Whether talked to teachers about what YP might do after Year 11,Review support,General routing input for post-16 discussion,The variable establishes whether the recommendation branches were applicable. It is already a construction input for the retained parent–teacher post-16 discussion profile
1,Wave 3,Parental attitudes,wave_three_lsype_parental_attitudes_file_16_06_08,14,W3tstayMP,MP: Whether talked to teachers about YP staying on in education after Year 11,Review support,Branch-routing input for topic-specific teacher discussion,The variables identify whether staying on or apprenticeship was discussed. They are already construction inputs for the retained parent–teacher post-16 discussion profile
2,Wave 3,Parental attitudes,wave_three_lsype_parental_attitudes_file_16_06_08,15,W3tstaydefMP,MP: Whether teacher though YP should stay on in education after Year 11,Exclude,Conditional teacher-recommendation response,"The item is observed only among parents who reached the relevant discussion branch. A routing-aware full-sample profile strongly duplicates the retained parent–teacher post-16 discussion profile, while the conditional response does not provide a comparable measure for the full sample"
3,Wave 3,Parental attitudes,wave_three_lsype_parental_attitudes_file_16_06_08,16,W3tappMP,MP: Whether talked to teachers about YP applying for a training place/apprentice,Review support,Branch-routing input for topic-specific teacher discussion,The variables identify whether staying on or apprenticeship was discussed. They are already construction inputs for the retained parent–teacher post-16 discussion profile
4,Wave 3,Parental attitudes,wave_three_lsype_parental_attitudes_file_16_06_08,17,W3tappdefMP,MP: Whether teacher thought YP should apply for a training place/apprenticeship,Exclude,Conditional teacher-recommendation response,"The item is observed only among parents who reached the relevant discussion branch. A routing-aware full-sample profile strongly duplicates the retained parent–teacher post-16 discussion profile, while the conditional response does not provide a comparable measure for the full sample"


Files written in this cell: 0


In [383]:
# 64: Domain 9 predictor consolidation and availability review

import pandas as pd

def align_predictor_series(target_name, candidate_object_names):
    for object_name in candidate_object_names:
        object_value = globals().get(object_name)
        if isinstance(object_value, pd.Series):
            return (object_value.reindex(route_index).copy().rename(target_name), object_name)
        if isinstance(object_value, pd.DataFrame):
            predictor_data = object_value.copy()
            possible_columns = [target_name, object_name]
            predictor_column = next((column for column in possible_columns if column in predictor_data.columns), None)
            if predictor_column is None:
                non_id_columns = [column for column in predictor_data.columns if column != 'NSID']
                if len(non_id_columns) == 1:
                    predictor_column = non_id_columns[0]
            if predictor_column is None:
                continue
            if 'NSID' in predictor_data.columns:
                predictor_data['NSID'] = standardise_nsid(predictor_data['NSID'])
                predictor_series = predictor_data.set_index('NSID')[predictor_column].reindex(route_index)
            else:
                predictor_series = predictor_data[predictor_column].reindex(route_index)
            return (predictor_series.copy().rename(target_name), object_name)
    raise NameError(f'Could not retrieve predictor: {target_name}')
domain_9_predictor_sources = {'parental_higher_education_expectation_pretransition': ['parental_higher_education_expectation_pretransition',
    'parental_he_expectation_candidate'], 'parental_educational_aspiration_pretransition': ['parental_educational_aspiration_pretransition',
    'parental_educational_aspiration_candidate'], 'parental_educational_financial_support_profile_pretransition': ['parental_educational_financial_support_profile_pretransition'], 'homework_help_at_home_pretransition': ['homework_help_at_home_pretransition'], 'homework_monitoring_frequency_pretransition': ['homework_monitoring_frequency_pretransition'], 'parental_knowledge_of_evening_whereabouts_pretransition': ['parental_knowledge_of_evening_whereabouts_pretransition'], 'school_night_curfew_pretransition': ['school_night_curfew_pretransition'], 'parents_evening_attendance_pretransition': ['parents_evening_attendance_pretransition'], 'parent_teacher_post16_discussion_profile_pretransition': ['parent_teacher_post16_discussion_profile_pretransition'], 'parent_relationship_quality_score_pretransition': ['parent_relationship_quality_score_pretransition'], 'parent_communication_frequency_score_pretransition': ['parent_communication_frequency_score_pretransition'], 'parent_school_day_discussion_frequency_pretransition': ['parent_school_day_discussion_frequency_pretransition'], 'parental_autonomy_support_score_pretransition': ['parental_autonomy_support_score_pretransition'], 'parental_training_apprenticeship_discussion_profile_pretransition': ['parental_training_apprenticeship_discussion_profile_pretransition'], 'specially_arranged_teacher_meeting_ever_pretransition': ['specially_arranged_teacher_meeting_ever_pretransition'], 'school_contact_about_behaviour_ever_pretransition': ['school_contact_about_behaviour_ever_pretransition'], 'shared_family_activity_frequency_pretransition': ['shared_family_activity_frequency_pretransition']}
domain_9_aligned_predictors = {}
domain_9_predictor_retrieval_rows = []
for predictor_name, candidate_object_names in domain_9_predictor_sources.items():
    predictor_series, source_object = align_predictor_series(predictor_name, candidate_object_names)
    domain_9_aligned_predictors[predictor_name] = predictor_series
    domain_9_predictor_retrieval_rows.append({'Predictor': predictor_name, 'Source object': source_object,
        'Retrieved': True})
assert len(domain_9_aligned_predictors) == 17
domain_9_predictors = pd.DataFrame({'NSID': route_index,
    **{predictor_name: predictor_series.to_numpy() for predictor_name,
    predictor_series in domain_9_aligned_predictors.items()}})
assert len(domain_9_predictors) == 9767
assert domain_9_predictors['NSID'].is_unique
assert domain_9_predictors.columns.drop('NSID').is_unique
assert len(domain_9_predictors.columns) == 18
domain_9_predictor_retrieval_summary = pd.DataFrame(domain_9_predictor_retrieval_rows)
domain_9_predictor_coverage_rows = []
for predictor_name in domain_9_predictor_sources:
    predictor_values = domain_9_predictors[predictor_name]
    domain_9_predictor_coverage_rows.append({'Predictor': predictor_name,
        'Non-missing': int(predictor_values.notna().sum()), 'Missing': int(predictor_values.isna().sum()), 'Missing percentage': round(predictor_values.isna().mean() * 100,
        2), 'Distinct observed values': int(predictor_values.nunique())})
domain_9_predictor_coverage_summary = pd.DataFrame(domain_9_predictor_coverage_rows).sort_values(['Missing percentage',
    'Predictor'], ascending=[False, True]).reset_index(drop=True)
domain_9_available_predictor_count = domain_9_predictors.drop(columns=['NSID']).notna().sum(axis=1).astype('Int64').rename('Domain 9 predictors available')
domain_9_joint_availability_summary = domain_9_available_predictor_count.value_counts().sort_index().rename('Participants').rename_axis('Predictors available').reset_index()
domain_9_joint_availability_summary['Percentage of full sample'] = (domain_9_joint_availability_summary['Participants'] / len(route_index) * 100).round(2)
domain_9_availability_overview = pd.DataFrame([{'Retained predictors': 17,
    'Participants with all 17 available': int(domain_9_available_predictor_count.eq(17).sum()), 'Percentage with all 17 available': round(domain_9_available_predictor_count.eq(17).mean() * 100,
    2), 'Participants with at least 15 available': int(domain_9_available_predictor_count.ge(15).sum()), 'Percentage with at least 15 available': round(domain_9_available_predictor_count.ge(15).mean() * 100,
    2), 'Minimum predictors available': int(domain_9_available_predictor_count.min()), 'Median predictors available': float(domain_9_available_predictor_count.median())}])
print('Domain 9 predictors consolidated: 17')
print('Predictor retrieval:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_9_predictor_retrieval_summary)
print('Individual predictor availability:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_9_predictor_coverage_summary)
print('Overall availability:')
display_limited(domain_9_availability_overview)
print('Joint availability distribution:')
display_limited(domain_9_joint_availability_summary)
print('Files written in this cell: 0')

Domain 9 predictors consolidated: 17
Predictor retrieval:


,Predictor,Source object,Retrieved
0,parental_higher_education_expectation_pretransition,parental_he_expectation_candidate,True
1,parental_educational_aspiration_pretransition,parental_educational_aspiration_candidate,True
2,parental_educational_financial_support_profile_pretransition,parental_educational_financial_support_profile_pretransition,True
3,homework_help_at_home_pretransition,homework_help_at_home_pretransition,True
4,homework_monitoring_frequency_pretransition,homework_monitoring_frequency_pretransition,True


Individual predictor availability:


,Predictor,Non-missing,Missing,Missing percentage,Distinct observed values
0,parental_autonomy_support_score_pretransition,8622,1145,11.72,5
1,parental_higher_education_expectation_pretransition,8874,893,9.14,4
2,parent_communication_frequency_score_pretransition,8996,771,7.89,9
3,school_contact_about_behaviour_ever_pretransition,9019,748,7.66,2
4,parent_school_day_discussion_frequency_pretransition,9079,688,7.04,3


Overall availability:


,Retained predictors,Participants with all 17 available,Percentage with all 17 available,Participants with at least 15 available,Percentage with at least 15 available,Minimum predictors available,Median predictors available
0,17,6802,69.64,8932,91.45,0,17.0


Joint availability distribution:


,Predictors available,Participants,Percentage of full sample
0,0,243,2.49
1,6,1,0.01
2,7,6,0.06
3,8,21,0.22
4,9,30,0.31


Files written in this cell: 0


In [384]:
# 65: Domain 9 variable-decision consolidation check

import pandas as pd
domain_9_variable_decision_audit = domain_9_screened_candidates.copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(domain_9_variable_decision_audit) == 174
assert not domain_9_variable_decision_audit[['Source file', 'Variable']].duplicated().any()
domain_9_inventory_keys = set(zip(domain_9_variable_decision_audit['Source file'],
    domain_9_variable_decision_audit['Variable']))
decision_priority = {'Review support': 1, 'Exclude': 2, 'Construction input': 3}

def normalise_domain_9_decision(decision_value):
    if pd.isna(decision_value):
        return None
    decision_text = str(decision_value).strip()
    decision_lower = decision_text.lower()
    if decision_lower == '' or 'pending' in decision_lower or 'defer' in decision_lower:
        return None
    if 'do not retain' in decision_lower or 'exclude' in decision_lower:
        return 'Exclude'
    if 'construction input' in decision_lower or 'retained input' in decision_lower:
        return 'Construction input'
    if decision_lower == 'retain':
        return 'Construction input'
    if 'review support' in decision_lower:
        return 'Review support'
    return None

def last_non_missing_value(row, columns):
    for column in reversed(columns):
        value = row.get(column, pd.NA)
        if pd.isna(value):
            continue
        value_text = str(value).strip()
        if value_text != '':
            return value
    return pd.NA
explicit_domain_9_decision_tables = [('parental_education_finance_variable_decision',
    globals().get('parental_education_finance_variable_decision')), ('homework_support_monitoring_variable_decisions',
    globals().get('homework_support_monitoring_variable_decisions')), ('parental_autonomy_variable_decisions',
    globals().get('parental_autonomy_variable_decisions')), ('parental_education_discussion_variable_decisions',
    globals().get('parental_education_discussion_variable_decisions')), ('parental_educational_aspiration_variable_decisions',
    globals().get('parental_educational_aspiration_variable_decisions')), ('parental_he_expectation_variable_decisions',
    globals().get('parental_he_expectation_variable_decisions')), ('parental_monitoring_variable_decisions',
    globals().get('parental_monitoring_variable_decisions')), ('parental_training_discussion_variable_decisions',
    globals().get('parental_training_discussion_variable_decisions')), ('parental_school_involvement_variable_decisions',
    globals().get('parental_school_involvement_variable_decisions')), ('parent_relationship_variable_decisions',
    globals().get('parent_relationship_variable_decisions')), ('parental_financial_support_variable_decisions',
    globals().get('parental_financial_support_variable_decisions')), ('reactive_school_parent_contact_variable_decisions',
    globals().get('reactive_school_parent_contact_variable_decisions')), ('shared_family_activity_variable_decisions',
    globals().get('shared_family_activity_variable_decisions')), ('teacher_recommendation_variable_decisions',
    globals().get('teacher_recommendation_variable_decisions'))]
explicit_table_availability = pd.DataFrame([{'Decision table': object_name, 'Available': isinstance(object_value,
    pd.DataFrame), 'Rows': len(object_value) if isinstance(object_value,
    pd.DataFrame) else 0} for object_name, object_value in explicit_domain_9_decision_tables])
assert explicit_table_availability['Available'].all()
decision_candidate_rows = []
for table_order, (object_name, decision_table) in enumerate(explicit_domain_9_decision_tables):
    assert 'Variable' in decision_table.columns
    decision_columns = [column for column in decision_table.columns if str(column).lower().endswith('decision')]
    role_columns = [column for column in decision_table.columns if str(column).lower().endswith('role')]
    reason_columns = [column for column in decision_table.columns if str(column).lower().endswith('reason')]
    assert len(decision_columns) > 0
    for _, source_row in decision_table.iterrows():
        variable_name = source_row.get('Variable')
        if pd.isna(variable_name):
            continue
        source_file = source_row.get('Source file', pd.NA)
        if pd.isna(source_file):
            matching_inventory = domain_9_variable_decision_audit.loc[domain_9_variable_decision_audit['Variable'].eq(variable_name)]
            if len(matching_inventory) == 1:
                source_file = matching_inventory.iloc[0]['Source file']
        if (source_file, variable_name) not in domain_9_inventory_keys:
            continue
        row_decisions = []
        for decision_column_order, decision_column in enumerate(decision_columns):
            normalised_decision = normalise_domain_9_decision(source_row.get(decision_column))
            if normalised_decision is None:
                continue
            row_decisions.append({'Decision': normalised_decision, 'Priority': decision_priority[normalised_decision],
                'Decision column order': decision_column_order})
        if len(row_decisions) == 0:
            continue
        selected_row_decision = pd.DataFrame(row_decisions).sort_values(['Priority', 'Decision column order'],
            ascending=[False, False]).iloc[0]
        decision_candidate_rows.append({'Source file': source_file, 'Variable': variable_name,
            'Decision': selected_row_decision['Decision'], 'Decision role': last_non_missing_value(source_row,
            role_columns), 'Decision reason': last_non_missing_value(source_row,
            reason_columns), 'Decision source': object_name, 'Decision priority': int(selected_row_decision['Priority']), 'Source specificity': 2, 'Source order': table_order})

def add_manual_decision_candidates(mask, decision, role, reason, source):
    matching_rows = domain_9_variable_decision_audit.loc[mask, ['Source file', 'Variable']]
    for _, matching_row in matching_rows.iterrows():
        decision_candidate_rows.append({'Source file': matching_row['Source file'],
            'Variable': matching_row['Variable'], 'Decision': decision, 'Decision role': role, 'Decision reason': reason, 'Decision source': source, 'Decision priority': decision_priority[decision], 'Source specificity': 1, 'Source order': -1})
review_track = domain_9_variable_decision_audit['Domain 9 review track']
variable_name = domain_9_variable_decision_audit['Variable']
screening_status = domain_9_variable_decision_audit['Domain 9 screening status']
out_of_domain_reason_lookup = {'General financial support': 'The variable concerns general financial transfers rather than education-specific parental support',
    'Keyword-search false positive': 'The variable was retrieved by broad search terms but does not measure parental educational attitudes or support', 'Parental education history': 'Detailed parental education history is not retained here; parental qualification is represented in the family socioeconomic domain', 'Teacher and careers guidance': 'The variable measures guidance from teachers or careers services rather than parental attitudes or support', 'Young-person educational plans': "The variable measures the young person's own plans and is represented in the educational aspirations and plans domain"}
for track_name, reason_text in out_of_domain_reason_lookup.items():
    add_manual_decision_candidates(screening_status.eq('Out of domain') & review_track.eq(track_name), 'Exclude',
        'Outside the parental educational attitudes and support domain', reason_text, 'Domain 9 screening decision')
add_manual_decision_candidates(review_track.eq('Education finance expectation'), 'Exclude',
    'Policy-specific education-finance eligibility item', 'The EMA item is selectively observed, policy-specific and strongly overlaps with household income', 'Education-finance review')
add_manual_decision_candidates(review_track.eq('Homework support and monitoring'), 'Construction input',
    'Input to retained homework support or homework monitoring predictor', 'The item contributes to a retained pre-transition family support or monitoring representation', 'Homework support and monitoring review')
add_manual_decision_candidates(review_track.eq('Parental autonomy support'), 'Construction input',
    'Input to available-parent mean autonomy-support score', 'The mother and father items are combined using the mean of available parent reports', 'Parental autonomy-support review')
add_manual_decision_candidates(review_track.eq('Parental discussion of continued education'), 'Review support',
    'Conditional parental expectation and discussion item', 'The item is selectively routed and does not provide a comparable full-sample predictor', 'Continued-education discussion review')
add_manual_decision_candidates(review_track.eq('Parental educational aspiration'), 'Construction input',
    'Input to retained parental educational aspiration predictor', "The item directly represents parental aspiration for the young person's educational attainment", 'Parental educational aspiration review')
add_manual_decision_candidates(review_track.eq('Parental higher-education expectation') & variable_name.eq('W1hepossMP'),
    'Construction input', 'Input to retained parental higher-education expectation', 'The item provides the main four-level parental expectation measure', 'Parental higher-education expectation review')
add_manual_decision_candidates(review_track.eq('Parental higher-education expectation') & ~variable_name.eq('W1hepossMP'),
    'Review support', 'Reason item supporting interpretation of HE expectation', 'The item explains the main expectation response but is not retained as a separate predictor', 'Parental higher-education expectation review')
add_manual_decision_candidates(variable_name.isin(['W1gowhereYP',
    'W1limitsnYP']) & review_track.eq('Parental monitoring'), 'Construction input', 'Input to retained parental monitoring predictor', 'The item contributes to parental knowledge of whereabouts or the school-night curfew representation', 'Parental monitoring review')
add_manual_decision_candidates(variable_name.eq('W1paroutMP') & review_track.eq('Parental monitoring'),
    'Review support', 'Main-parent report used to assess reporter agreement', 'The item was used to assess agreement but was not retained because of reporter differences and a strong ceiling effect', 'Parental monitoring review')
add_manual_decision_candidates(review_track.eq('Parental post-16 route discussion'), 'Construction input',
    'Parent-participation input to training and apprenticeship discussion profile', 'The item is combined with its routing question to distinguish no discussion, discussion without parents and discussion with parents', 'Parental post-16 route discussion review')
school_involvement_construction_variables = {'W1pareveMP', 'W2pareveMP', 'W3pareveMP', 'W1tmeetfMP', 'W2tmeetfMP',
    'W3tmeetfMP', 'W3tspeak2MP', 'W3tstayMP', 'W3tappMP'}
school_involvement_support_variables = {'W1repred1MP', 'W2vocs2MP'}
assert school_involvement_construction_variables | school_involvement_support_variables == set(domain_9_variable_decision_audit.loc[review_track.eq('Parental school involvement'),
    'Variable'])
add_manual_decision_candidates(review_track.eq('Parental school involvement') & variable_name.isin(school_involvement_construction_variables),
    'Construction input', 'Input to retained parental school-involvement or school–parent contact predictor', "The item contributes to parents' evening attendance, parent–teacher post-16 discussion or pre-transition school–parent contact", 'Parental school-involvement review')
add_manual_decision_candidates(review_track.eq('Parental school involvement') & variable_name.isin(school_involvement_support_variables),
    'Review support', 'Related school-involvement item used for representation review', 'The item was reviewed but did not provide a separate non-duplicative predictor', 'Parental school-involvement review')
add_manual_decision_candidates(review_track.eq('Parent–young-person relationship and communication'),
    'Construction input', 'Input to retained relationship or communication predictor', 'The item contributes to relationship quality, general communication or discussion of the school day', 'Parent–young-person relationship review')
add_manual_decision_candidates(review_track.eq('Reactive school–parent contact'), 'Construction input',
    'Input to school contact about behaviour ever before transition', 'The Wave 2 and Wave 3 items are combined as an event-history indicator, with the Wave 3 timing restriction applied', 'Reactive school–parent contact review')
add_manual_decision_candidates(review_track.eq('Shared family activity'), 'Construction input',
    'Input to retained shared family activity frequency', 'The ordered frequency categories provide a distinct measure of shared family activity', 'Shared family activity review')
add_manual_decision_candidates(review_track.eq('Teacher recommendation reported by parent'), 'Exclude',
    'Conditional teacher-recommendation response', 'The item is observed only after the relevant parent–teacher discussion branch and a routing-aware profile strongly duplicates the retained discussion profile', 'Teacher recommendation review')
domain_9_decision_candidates = pd.DataFrame(decision_candidate_rows)
assert len(domain_9_decision_candidates) > 0
domain_9_meaningful_decision_history = domain_9_decision_candidates.groupby(['Source file', 'Variable'],
    as_index=False).agg(Meaningful_decision_count=('Decision', 'nunique'), Meaningful_decisions=('Decision',
    lambda values: ' | '.join(sorted(set(values.astype(str))))), Decision_sources=('Decision source',
    lambda values: ' | '.join(sorted(set(values.astype(str))))))
domain_9_multiple_meaningful_decisions = domain_9_meaningful_decision_history.loc[domain_9_meaningful_decision_history['Meaningful_decision_count'].gt(1)].reset_index(drop=True)
domain_9_hard_decision_conflicts = domain_9_multiple_meaningful_decisions.loc[domain_9_multiple_meaningful_decisions['Meaningful_decisions'].str.contains('Construction input',
    na=False) & domain_9_multiple_meaningful_decisions['Meaningful_decisions'].str.contains('Exclude',
    na=False)].reset_index(drop=True)
assert len(domain_9_hard_decision_conflicts) == 0
domain_9_selected_decisions = domain_9_decision_candidates.sort_values(['Source file', 'Variable', 'Decision priority',
    'Source specificity', 'Source order'], ascending=[True, True, False, False,
    False]).drop_duplicates(subset=['Source file', 'Variable'], keep='first')[['Source file', 'Variable', 'Decision',
    'Decision role', 'Decision reason', 'Decision source']].rename(columns={'Decision': 'Final Domain 9 decision',
    'Decision role': 'Final Domain 9 role', 'Decision reason': 'Final Domain 9 reason', 'Decision source': 'Final decision source'})
domain_9_variable_decision_audit = domain_9_variable_decision_audit.merge(domain_9_selected_decisions,
    on=['Source file', 'Variable'], how='left', validate='one_to_one')
generic_role_lookup = {'Construction input': 'Input to a retained Domain 9 predictor',
    'Review support': 'Supporting item used in construct or representation review', 'Exclude': 'Not included in the Domain 9 predictor set'}
generic_reason_lookup = {'Construction input': 'The variable contributes to a retained, outcome-blind pre-transition predictor',
    'Review support': 'The variable informed routing, overlap or representation review but is not retained as a separate predictor', 'Exclude': 'The variable is outside the domain, selectively observed, duplicative or unsuitable as a comparable predictor'}
for final_decision in ['Construction input', 'Review support', 'Exclude']:
    decision_mask = domain_9_variable_decision_audit['Final Domain 9 decision'].eq(final_decision)
    domain_9_variable_decision_audit.loc[decision_mask & domain_9_variable_decision_audit['Final Domain 9 role'].isna(),
        'Final Domain 9 role'] = generic_role_lookup[final_decision]
    domain_9_variable_decision_audit.loc[decision_mask & domain_9_variable_decision_audit['Final Domain 9 reason'].isna(),
        'Final Domain 9 reason'] = generic_reason_lookup[final_decision]
planned_financial_support_decision_summary = domain_9_variable_decision_audit.loc[review_track.eq('Planned financial support for continued education'),
    'Final Domain 9 decision'].value_counts(dropna=False)
assert int(planned_financial_support_decision_summary.get('Construction input', 0)) == 27
assert int(planned_financial_support_decision_summary.get('Review support', 0)) == 27
domain_9_unresolved_decisions = domain_9_variable_decision_audit.loc[domain_9_variable_decision_audit['Final Domain 9 decision'].isna() | domain_9_variable_decision_audit['Final Domain 9 role'].isna() | domain_9_variable_decision_audit['Final Domain 9 reason'].isna()][['Wave',
    'Source type', 'Source file', 'Variable', 'Variable label', 'Domain 9 screening status', 'Domain 9 review track', 'Final Domain 9 decision', 'Final Domain 9 role', 'Final Domain 9 reason', 'Final decision source']].copy().reset_index(drop=True)
domain_9_final_decision_summary = domain_9_variable_decision_audit['Final Domain 9 decision'].value_counts(dropna=False).rename('Variables').rename_axis('Final Domain 9 decision').reset_index()
domain_9_final_decision_by_track = domain_9_variable_decision_audit.groupby(['Domain 9 review track',
    'Final Domain 9 decision'], dropna=False).size().rename('Variables').reset_index().sort_values(['Domain 9 review track',
    'Final Domain 9 decision']).reset_index(drop=True)
domain_9_decision_counts = domain_9_variable_decision_audit['Final Domain 9 decision'].value_counts()
assert len(domain_9_unresolved_decisions) == 0
assert int(domain_9_decision_counts.get('Construction input', 0)) == 55
assert int(domain_9_decision_counts.get('Review support', 0)) == 43
assert int(domain_9_decision_counts.get('Exclude', 0)) == 76
domain_9_decision_audit_overview = pd.DataFrame([{'Domain 9 variables': int(len(domain_9_variable_decision_audit)),
    'Construction inputs': int(domain_9_decision_counts.get('Construction input',
    0)), 'Review-support variables': int(domain_9_decision_counts.get('Review support',
    0)), 'Excluded variables': int(domain_9_decision_counts.get('Exclude',
    0)), 'Unresolved variables': int(len(domain_9_unresolved_decisions)), 'Multiple meaningful decision histories': int(len(domain_9_multiple_meaningful_decisions)), 'Hard decision conflicts': int(len(domain_9_hard_decision_conflicts))}])
print('Explicit decision-table availability:')
display_limited(explicit_table_availability)
print('Domain 9 variable-decision audit:')
display_limited(domain_9_decision_audit_overview)
print('Final decision counts:')
display_limited(domain_9_final_decision_summary)
print('Final decisions by review track:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_9_final_decision_by_track)
print('Unresolved variable decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_9_unresolved_decisions)
print('Multiple meaningful decision histories:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_9_multiple_meaningful_decisions)
print('Hard decision conflicts:')
display_limited(domain_9_hard_decision_conflicts)
print('Files written in this cell: 0')

Explicit decision-table availability:


,Decision table,Available,Rows
0,parental_education_finance_variable_decision,True,1
1,homework_support_monitoring_variable_decisions,True,4
2,parental_autonomy_variable_decisions,True,2
3,parental_education_discussion_variable_decisions,True,3
4,parental_educational_aspiration_variable_decis...,True,1


Domain 9 variable-decision audit:


,Domain 9 variables,Construction inputs,Review-support variables,Excluded variables,Unresolved variables,Multiple meaningful decision histories,Hard decision conflicts
0,174,55,43,76,0,3,0


Final decision counts:


...

Multiple meaningful decision histories:


,Source file,Variable,Meaningful_decision_count,Meaningful_decisions,Decision_sources
0,wave_three_lsype_parental_attitudes_file_16_06_08,W3tappMP,2,Construction input | Review support,Parental school-involvement review | parental_school_involvement_variable_decisions | teacher_recommendation_variable_decisions
1,wave_three_lsype_parental_attitudes_file_16_06_08,W3tspeak2MP,2,Construction input | Review support,Parental school-involvement review | parental_school_involvement_variable_decisions | teacher_recommendation_variable_decisions
2,wave_three_lsype_parental_attitudes_file_16_06_08,W3tstayMP,2,Construction input | Review support,Parental school-involvement review | parental_school_involvement_variable_decisions | teacher_recommendation_variable_decisions


Hard decision conflicts:


,Source file,Variable,Meaningful_decision_count,Meaningful_decisions,Decision_sources


Files written in this cell: 0


In [385]:
# 66: Domain 9 multiple-role decision reconciliation

import pandas as pd
multiple_role_variables = {'W3tspeak2MP', 'W3tstayMP', 'W3tappMP'}
assert set(domain_9_multiple_meaningful_decisions['Variable']) == multiple_role_variables
assert len(domain_9_hard_decision_conflicts) == 0
multiple_role_mask = domain_9_variable_decision_audit['Variable'].isin(multiple_role_variables)
assert int(multiple_role_mask.sum()) == 3
assert domain_9_variable_decision_audit.loc[multiple_role_mask,
    'Final Domain 9 decision'].eq('Construction input').all()
domain_9_variable_decision_audit.loc[multiple_role_mask,
    'Final Domain 9 role'] = 'Construction input to the retained parent–teacher post-16 discussion profile'
domain_9_variable_decision_audit.loc[multiple_role_mask,
    'Final Domain 9 reason'] = 'The variable contributes directly to the retained parent–teacher post-16 discussion profile. It was also used as review support when examining the routing of conditional teacher-recommendation items'
domain_9_variable_decision_audit.loc[multiple_role_mask,
    'Final decision source'] = 'Parental school-involvement review; teacher-recommendation routing review'
domain_9_multiple_role_resolution = domain_9_variable_decision_audit.loc[multiple_role_mask].copy().sort_values(['Source order',
    'Variable position'])[['Wave', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Final Domain 9 decision', 'Final Domain 9 role', 'Final Domain 9 reason', 'Final decision source']].reset_index(drop=True)
domain_9_final_decision_counts = domain_9_variable_decision_audit['Final Domain 9 decision'].value_counts()
assert int(domain_9_final_decision_counts.get('Construction input', 0)) == 55
assert int(domain_9_final_decision_counts.get('Review support', 0)) == 43
assert int(domain_9_final_decision_counts.get('Exclude', 0)) == 76
assert domain_9_variable_decision_audit['Final Domain 9 decision'].notna().all()
assert domain_9_variable_decision_audit['Final Domain 9 role'].notna().all()
assert domain_9_variable_decision_audit['Final Domain 9 reason'].notna().all()
domain_9_final_audit_summary = pd.DataFrame([{'Domain 9 variables': int(len(domain_9_variable_decision_audit)),
    'Construction inputs': int(domain_9_final_decision_counts.get('Construction input',
    0)), 'Review-support variables': int(domain_9_final_decision_counts.get('Review support',
    0)), 'Excluded variables': int(domain_9_final_decision_counts.get('Exclude',
    0)), 'Multiple-role variables': int(multiple_role_mask.sum()), 'Hard decision conflicts': int(len(domain_9_hard_decision_conflicts)), 'Unresolved variables': int(domain_9_variable_decision_audit[['Final Domain 9 decision',
    'Final Domain 9 role', 'Final Domain 9 reason']].isna().any(axis=1).sum())}])
print('Multiple-role decision resolution:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_9_multiple_role_resolution)
print('Final Domain 9 audit summary:')
display_limited(domain_9_final_audit_summary)
print('Files written in this cell: 0')

Multiple-role decision resolution:


,Wave,Source file,Variable position,Variable,Variable label,Final Domain 9 decision,Final Domain 9 role,Final Domain 9 reason,Final decision source
0,Wave 3,wave_three_lsype_parental_attitudes_file_16_06_08,13,W3tspeak2MP,MP: Whether talked to teachers about what YP might do after Year 11,Construction input,Construction input to the retained parent–teacher post-16 discussion profile,The variable contributes directly to the retained parent–teacher post-16 discussion profile. It was also used as review support when examining the routing of conditional teacher-recommendation items,Parental school-involvement review; teacher-recommendation routing review
1,Wave 3,wave_three_lsype_parental_attitudes_file_16_06_08,14,W3tstayMP,MP: Whether talked to teachers about YP staying on in education after Year 11,Construction input,Construction input to the retained parent–teacher post-16 discussion profile,The variable contributes directly to the retained parent–teacher post-16 discussion profile. It was also used as review support when examining the routing of conditional teacher-recommendation items,Parental school-involvement review; teacher-recommendation routing review
2,Wave 3,wave_three_lsype_parental_attitudes_file_16_06_08,16,W3tappMP,MP: Whether talked to teachers about YP applying for a training place/apprentice,Construction input,Construction input to the retained parent–teacher post-16 discussion profile,The variable contributes directly to the retained parent–teacher post-16 discussion profile. It was also used as review support when examining the routing of conditional teacher-recommendation items,Parental school-involvement review; teacher-recommendation routing review


Final Domain 9 audit summary:


,Domain 9 variables,Construction inputs,Review-support variables,Excluded variables,Multiple-role variables,Hard decision conflicts,Unresolved variables
0,174,55,43,76,3,0,0


Files written in this cell: 0


In [386]:
# 67: Domain 9 output and decision register

required_register_columns = {'Source file', 'Variable', 'Review status', 'Review outcome', 'Substantive domain',
    'Decision reason', 'Review notes'}
assert required_register_columns.issubset(verified_decision_register.columns)
domain_9_decision_register = verified_decision_register.copy().reset_index(drop=True)
assert len(domain_9_decision_register) == 5261
assert not domain_9_decision_register[['Source file', 'Variable']].duplicated().any()
domain_9_master_update_rows = domain_9_variable_decision_audit.loc[~domain_9_variable_decision_audit['Domain 9 screening status'].eq('Out of domain')].copy().reset_index(drop=True)
assert len(domain_9_master_update_rows) == 100
domain_9_master_update_counts = domain_9_master_update_rows['Final Domain 9 decision'].value_counts()
assert int(domain_9_master_update_counts.get('Construction input', 0)) == 55
assert int(domain_9_master_update_counts.get('Review support', 0)) == 43
assert int(domain_9_master_update_counts.get('Exclude', 0)) == 2
domain_9_register_updates = pd.DataFrame({'Source file': domain_9_master_update_rows['Source file'],
    'Variable': domain_9_master_update_rows['Variable'], '_domain_9_review_status': 'Reviewed', '_domain_9_review_outcome': domain_9_master_update_rows['Final Domain 9 decision'], '_domain_9_substantive_domain': 'Parental educational attitudes and support', '_domain_9_decision_reason': domain_9_master_update_rows['Final Domain 9 reason'], '_domain_9_review_notes': 'Domain 9 role: ' + domain_9_master_update_rows['Final Domain 9 role'].astype(str) + ' | Decision source: ' + domain_9_master_update_rows['Final decision source'].astype(str)})
assert not domain_9_register_updates[['Source file', 'Variable']].duplicated().any()
register_keys = set(zip(domain_9_decision_register['Source file'], domain_9_decision_register['Variable']))
update_keys = set(zip(domain_9_register_updates['Source file'], domain_9_register_updates['Variable']))
assert update_keys.issubset(register_keys)
domain_9_decision_register['_register_order'] = range(len(domain_9_decision_register))
domain_9_decision_register = domain_9_decision_register.merge(domain_9_register_updates, on=['Source file',
    'Variable'], how='left', validate='one_to_one').sort_values('_register_order').reset_index(drop=True)
register_update_mask = domain_9_decision_register['_domain_9_review_status'].notna()
assert int(register_update_mask.sum()) == 100
for register_column in ['Review status', 'Review outcome', 'Substantive domain', 'Decision reason', 'Review notes']:
    domain_9_decision_register[register_column] = domain_9_decision_register[register_column].astype('object')
domain_9_decision_register.loc[register_update_mask,
    'Review status'] = domain_9_decision_register.loc[register_update_mask, '_domain_9_review_status'].to_numpy()
domain_9_decision_register.loc[register_update_mask,
    'Review outcome'] = domain_9_decision_register.loc[register_update_mask, '_domain_9_review_outcome'].to_numpy()
domain_9_decision_register.loc[register_update_mask,
    'Substantive domain'] = domain_9_decision_register.loc[register_update_mask,
    '_domain_9_substantive_domain'].to_numpy()
domain_9_decision_register.loc[register_update_mask,
    'Decision reason'] = domain_9_decision_register.loc[register_update_mask, '_domain_9_decision_reason'].to_numpy()
domain_9_decision_register.loc[register_update_mask,
    'Review notes'] = domain_9_decision_register.loc[register_update_mask, '_domain_9_review_notes'].to_numpy()
domain_9_decision_register = domain_9_decision_register.drop(columns=['_register_order', '_domain_9_review_status',
    '_domain_9_review_outcome', '_domain_9_substantive_domain', '_domain_9_decision_reason', '_domain_9_review_notes'])
assert len(domain_9_decision_register) == 5261
assert not domain_9_decision_register[['Source file', 'Variable']].duplicated().any()
domain_9_register_verification = domain_9_decision_register.merge(domain_9_master_update_rows[['Source file',
    'Variable', 'Final Domain 9 decision']], on=['Source file', 'Variable'], how='inner', validate='one_to_one')
assert len(domain_9_register_verification) == 100
assert domain_9_register_verification['Review outcome'].eq(domain_9_register_verification['Final Domain 9 decision']).all()
domain_9_predictor_output = domain_9_predictors.copy()
domain_9_decision_output = domain_9_variable_decision_audit.copy()
domain_9_predictor_summary_output = domain_9_predictor_coverage_summary.copy()
domain_9_predictor_summary_output['Retained domain'] = 'Parental educational attitudes and support'
domain_9_predictor_path = stage_2_output_directory / 'stage_2_parental_educational_attitudes_support_domain_predictors.csv'
domain_9_decision_path = stage_2_output_directory / 'stage_2_parental_educational_attitudes_support_domain_decisions.csv'
domain_9_predictor_summary_path = stage_2_output_directory / 'stage_2_parental_educational_attitudes_support_domain_predictor_summary.csv'
stage_2_decision_register_path = decision_register_path
domain_9_predictor_output.to_csv(domain_9_predictor_path, index=False)
domain_9_decision_output.to_csv(domain_9_decision_path, index=False)
domain_9_predictor_summary_output.to_csv(domain_9_predictor_summary_path, index=False)
domain_9_decision_register.to_csv(stage_2_decision_register_path, index=False)
working_variable_decision_register = domain_9_decision_register.copy()
written_file_checks = []
for file_label, file_path, expected_rows, expected_columns in [('Domain 9 predictors', domain_9_predictor_path, 9767,
    18), ('Domain 9 decisions', domain_9_decision_path, 174,
    len(domain_9_decision_output.columns)), ('Domain 9 predictor summary', domain_9_predictor_summary_path, 17,
    len(domain_9_predictor_summary_output.columns)), ('Variable decision register', stage_2_decision_register_path,
    5261, len(domain_9_decision_register.columns))]:
    written_data = pd.read_csv(file_path)
    assert len(written_data) == expected_rows
    assert len(written_data.columns) == expected_columns
    written_file_checks.append({'Output': file_label, 'File': file_path.name, 'Rows': len(written_data),
        'Columns': len(written_data.columns), 'Size bytes': file_path.stat().st_size, 'Verified': True})
domain_9_written_file_summary = pd.DataFrame(written_file_checks)
domain_9_register_update_summary = pd.DataFrame([{'Register rows before Domain 9': int(len(verified_decision_register)),
    'Domain 9 inventory variables': 174, 'Master-register variables updated': 100, 'Out-of-domain variables preserved': 74, 'Construction inputs recorded': 55, 'Review-support variables recorded': 43, 'Excluded variables recorded': 2, 'Retained predictors written': 17}])
print('Domain 9 register update:')
display_limited(domain_9_register_update_summary)
print('Written file verification:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_9_written_file_summary)
print('Files written in this cell:', len(domain_9_written_file_summary))

Domain 9 register update:


,Register rows before Domain 9,Domain 9 inventory variables,Master-register variables updated,Out-of-domain variables preserved,Construction inputs recorded,Review-support variables recorded,Excluded variables recorded,Retained predictors written
0,5261,174,100,74,55,43,2,17


Written file verification:


,Output,File,Rows,Columns,Size bytes,Verified
0,Domain 9 predictors,stage_2_parental_educational_attitudes_support_domain_predictors.csv,9767,18,734280,True
1,Domain 9 decisions,stage_2_parental_educational_attitudes_support_domain_decisions.csv,174,28,150324,True
2,Domain 9 predictor summary,stage_2_parental_educational_attitudes_support_domain_predictor_summary.csv,17,6,1953,True
3,Variable decision register,stage_2_variable_decision_register.csv,5261,17,1814702,True


Files written in this cell: 4


## Parental attitudes and support predictors

Seventeen predictors were retained. Complete information on all 17 was available for 6,802 participants (69.64%); 8,932 participants (91.45%) had at least 15 available, and 243 had none.

The decision file contains 174 source variables: 55 construction inputs, 43 review-support variables and 76 exclusions. No decision conflicts or unresolved variables remained.


# Part 13: School and local context

School climate, regional and urban–rural context, institutional guidance, school sector and school mobility were reviewed. Individual school attitudes and family socioeconomic circumstances remained in their existing domains.


In [387]:
# 1: Domain 10 candidate-variable inventory

import re
import numpy as np
import pandas as pd
domain_10_register_source = working_variable_decision_register.copy().reset_index(drop=True)
assert len(domain_10_register_source) == 5261
assert not domain_10_register_source[['Source file', 'Variable']].duplicated().any()
domain_10_search_text = domain_10_register_source[['Variable', 'Variable label', 'Substantive domain',
    'Decision reason', 'Review notes']].fillna('').astype(str).agg(' | '.join, axis=1).str.lower()
domain_10_explicit_deferred_mask = domain_10_search_text.str.contains('domain\\s*10|school.{0,20}local context|local.{0,20}school context|defer.{0,40}school context|defer.{0,40}local context',
    case=False, na=False, regex=True)
domain_10_search_patterns = {'School sector and institutional type': '\\bschool sector\\b|\\bschool type\\b|\\btype of school\\b|\\bindependent school\\b|\\bprivate school\\b|\\bgrammar school\\b|\\bcomprehensive school\\b|\\bspecial school\\b|\\bcommunity school\\b|\\bfoundation school\\b|\\bvoluntary aided\\b|\\bselective school\\b|\\bsingle[- ]sex school\\b|\\bmixed school\\b|\\bacademy school\\b',
    'School composition, resources and performance': '\\bschool performance\\b|\\bschool results\\b|\\bschool attainment\\b|\\bschool composition\\b|\\bschool quality\\b|\\bschool effectiveness\\b|\\bschool size\\b|\\bschool resources\\b|\\bschool facilities\\b|\\bfree school meals?\\b|\\bfsm\\b|\\bpupil composition\\b|\\bproportion of pupils\\b|\\bpercentage of pupils\\b', 'School climate and teacher relationships': '\\bschool climate\\b|\\bschool environment\\b|\\bschool ethos\\b|\\bschool safety\\b|\\bsafe at school\\b|\\bschool belonging\\b|\\bhappy at school\\b|\\bschool discipline\\b|\\bteacher relationship\\b|\\brelationship with teachers?\\b|\\bteachers? treat\\b|\\bteachers? listen\\b|\\bteachers? support\\b|\\bteachers? respect\\b|\\bteachers? fair\\b|\\bget on with teachers?\\b', 'Attendance and disciplinary context': '\\bschool attendance\\b|\\battendance record\\b|\\babsence from school\\b|\\babsent from school\\b|\\bunauthorised absence\\b|\\btruancy\\b|\\btruant\\b|\\bschool exclusion\\b|\\bexcluded from school\\b|\\bsuspension\\b|\\bsuspended from school\\b|\\bdetention\\b|\\bexpelled\\b', 'School change, access and travel': '\\bchange(?:d)? school\\b|\\bschool change\\b|\\bmoved school\\b|\\bschool move\\b|\\btravel to school\\b|\\bjourney to school\\b|\\bdistance to school\\b|\\btime taken to school\\b|\\bstarted school\\b', 'Neighbourhood and local context': '\\bneighbou?rhood\\b|\\blocal area\\b|\\barea where .* live\\b|\\burban\\b|\\brural\\b|\\bregion\\b|\\blocal authority\\b|\\bward\\b|\\barea deprivation\\b|\\bdeprivation index\\b|\\bcrime in .* area\\b|\\barea safety\\b|\\bsafe in .* area\\b|\\bhousing area\\b', 'School provision and external support': '\\bextra[- ]curricular\\b|\\bextracurricular\\b|\\bschool clubs?\\b|\\bafter[- ]school activit|\\bschool support\\b|\\blearning support\\b|\\bcareers? advice\\b|\\bcareers? guidance\\b|\\bconnexions\\b'}
domain_10_track_masks = {}
for track_name, search_pattern in domain_10_search_patterns.items():
    domain_10_track_masks[track_name] = domain_10_search_text.str.contains(search_pattern, case=False, na=False,
        regex=True).fillna(False).astype(bool)
domain_10_keyword_mask = pd.Series(False, index=domain_10_register_source.index)
for track_mask in domain_10_track_masks.values():
    domain_10_keyword_mask = domain_10_keyword_mask | track_mask
domain_10_candidate_mask = domain_10_explicit_deferred_mask | domain_10_keyword_mask
domain_10_candidate_inventory = domain_10_register_source.loc[domain_10_candidate_mask].copy()
domain_10_track_conditions = [domain_10_explicit_deferred_mask.loc[domain_10_candidate_inventory.index]]
domain_10_track_choices = ['Explicit Domain 10 deferral']
for track_name, track_mask in domain_10_track_masks.items():
    domain_10_track_conditions.append(track_mask.loc[domain_10_candidate_inventory.index])
    domain_10_track_choices.append(track_name)
domain_10_candidate_inventory['Domain 10 search track'] = np.select(domain_10_track_conditions,
    domain_10_track_choices, default='Additional keyword match')
domain_10_matching_track_rows = []
for candidate_index in domain_10_candidate_inventory.index:
    matching_tracks = []
    if bool(domain_10_explicit_deferred_mask.loc[candidate_index]):
        matching_tracks.append('Explicit Domain 10 deferral')
    for track_name, track_mask in domain_10_track_masks.items():
        if bool(track_mask.loc[candidate_index]):
            matching_tracks.append(track_name)
    domain_10_matching_track_rows.append(' | '.join(matching_tracks))
domain_10_candidate_inventory['All Domain 10 search matches'] = domain_10_matching_track_rows
domain_10_candidate_inventory['Prior review outcome'] = domain_10_candidate_inventory['Review outcome']
domain_10_candidate_inventory['Prior substantive domain'] = domain_10_candidate_inventory['Substantive domain']
domain_10_candidate_inventory = domain_10_candidate_inventory.sort_values(['Domain 10 search track', 'Source order',
    'Variable position']).reset_index(drop=True)
domain_10_search_track_summary = domain_10_candidate_inventory['Domain 10 search track'].value_counts().rename('Variables').rename_axis('Domain 10 search track').reset_index()
domain_10_prior_decision_summary = domain_10_candidate_inventory['Prior review outcome'].fillna('Missing').value_counts().rename('Variables').rename_axis('Prior review outcome').reset_index()
domain_10_wave_source_summary = domain_10_candidate_inventory.groupby(['Wave', 'Source type'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Wave', 'Source type']).reset_index(drop=True)
domain_10_search_overview = pd.DataFrame([{'Register variables searched': int(len(domain_10_register_source)),
    'Explicit Domain 10 deferrals': int(domain_10_explicit_deferred_mask.sum()), 'Keyword-search matches': int(domain_10_keyword_mask.sum()), 'Unique candidate variables': int(len(domain_10_candidate_inventory)), 'Previously reviewed candidates': int(domain_10_candidate_inventory['Prior review outcome'].astype('string').str.lower().isin(['construction input',
    'review support', 'exclude']).sum())}])
domain_10_candidate_display = domain_10_candidate_inventory[['Source order', 'Wave', 'Source type', 'Source file',
    'Variable position', 'Variable', 'Variable label', 'Timing status', 'Prior review outcome', 'Prior substantive domain', 'Domain 10 search track', 'All Domain 10 search matches']].copy()
print('Domain 10 candidate-search overview:')
display_limited(domain_10_search_overview)
print('\nCandidate variables by search track:')
display_limited(domain_10_search_track_summary)
print('\nPrior review outcomes:')
display_limited(domain_10_prior_decision_summary)
print('\nCandidates by wave and source type:')
display_limited(domain_10_wave_source_summary)
print(f'\nDomain 10 candidate variables: {len(domain_10_candidate_display):,}')
print('\nFirst 15 candidate variables:')
display_limited(domain_10_candidate_display.head(15))
print(f'Additional candidate variables not displayed: {max(len(domain_10_candidate_display) - 15, 0):,}')
print('\nFiles written in this cell: 0')

Domain 10 candidate-search overview:


,Register variables searched,Explicit Domain 10 deferrals,Keyword-search matches,Unique candidate variables,Previously reviewed candidates
0,5261,2,138,140,28



Candidate variables by search track:


,Domain 10 search track,Variables
0,School sector and institutional type,60
1,Attendance and disciplinary context,34
2,School provision and external support,23
3,"School change, access and travel",13
4,Neighbourhood and local context,6



Prior review outcomes:


...


Candidates by wave and source type:


,Wave,Source type,Variables
0,Wave 1,Family background,2
1,Wave 1,Parental attitudes,57
2,Wave 1,Young person,23
3,Wave 2,Family background,6
4,Wave 2,Young person,24



Domain 10 candidate variables: 140

First 15 candidate variables:


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Prior review outcome,Prior substantive domain,Domain 10 search track,All Domain 10 search matches
0,1,Wave 1,Young person,wave_one_lsype_young_person_2020,55,W1abs3mwMP,MP: Reason for YP's (last) extended period of ...,Pre-transition source,Exclude,School attitudes and engagement,Attendance and disciplinary context,Attendance and disciplinary context
1,1,Wave 1,Young person,wave_one_lsype_young_person_2020,57,W1abs1mwMP,MP: Reason for YP's period of absence from sch...,Pre-transition source,Exclude,School attitudes and engagement,Attendance and disciplinary context,Attendance and disciplinary context
2,1,Wave 1,Young person,wave_one_lsype_young_person_2020,58,W1suspendMP,MP: Whether YP has ever been temporarily suspe...,Pre-transition source,Review support,School attitudes and engagement,Attendance and disciplinary context,Attendance and disciplinary context
3,1,Wave 1,Young person,wave_one_lsype_young_person_2020,60,W1sutime2MP,MP: Number of times YP has been temporarily ex...,Pre-transition source,Review support,School attitudes and engagement,Attendance and disciplinary context,Attendance and disciplinary context
4,1,Wave 1,Young person,wave_one_lsype_young_person_2020,61,W1expelMP,MP: Whether YP has ever been expelled or perma...,Pre-transition source,Review support,School attitudes and engagement,Attendance and disciplinary context,Attendance and disciplinary context


Additional candidate variables not displayed: 125

Files written in this cell: 0


In [388]:
# 2: Domain 10 candidate screening and recall augmentation

import numpy as np
import pandas as pd
domain_9_teacher_guidance_keys = domain_9_variable_decision_audit.loc[domain_9_variable_decision_audit['Domain 9 review track'].eq('Teacher and careers guidance'),
    ['Source file', 'Variable']].drop_duplicates().reset_index(drop=True)
assert len(domain_9_teacher_guidance_keys) == 19
current_domain_10_keys = set(zip(domain_10_candidate_inventory['Source file'],
    domain_10_candidate_inventory['Variable']))
domain_9_teacher_guidance_key_set = set(zip(domain_9_teacher_guidance_keys['Source file'],
    domain_9_teacher_guidance_keys['Variable']))
missing_teacher_guidance_keys = domain_9_teacher_guidance_key_set - current_domain_10_keys
domain_10_guidance_additions = domain_10_register_source.merge(domain_9_teacher_guidance_keys, on=['Source file',
    'Variable'], how='inner', validate='one_to_one')
domain_10_guidance_additions = domain_10_guidance_additions.loc[[(source_file,
    variable) in missing_teacher_guidance_keys for source_file, variable in zip(domain_10_guidance_additions['Source file'],
    domain_10_guidance_additions['Variable'])]].copy()
domain_10_guidance_additions['Domain 10 search track'] = 'Domain 9 teacher and careers guidance transfer'
domain_10_guidance_additions['All Domain 10 search matches'] = 'Domain 9 teacher and careers guidance transfer'
domain_10_guidance_additions['Prior review outcome'] = domain_10_guidance_additions['Review outcome']
domain_10_guidance_additions['Prior substantive domain'] = domain_10_guidance_additions['Substantive domain']
domain_10_augmented_candidate_inventory = pd.concat([domain_10_candidate_inventory,
    domain_10_guidance_additions[domain_10_candidate_inventory.columns]], ignore_index=True).drop_duplicates(subset=['Source file',
    'Variable'], keep='first').sort_values(['Source order', 'Variable position']).reset_index(drop=True)
assert not domain_10_augmented_candidate_inventory[['Source file', 'Variable']].duplicated().any()
domain_10_augmented_candidate_inventory['Domain 10 screening status'] = pd.NA
domain_10_augmented_candidate_inventory['Domain 10 review track'] = pd.NA
domain_10_augmented_candidate_inventory['Domain 10 screening reason'] = pd.NA
domain_10_variable = domain_10_augmented_candidate_inventory['Variable'].astype('string')
domain_10_label = domain_10_augmented_candidate_inventory['Variable label'].fillna('').astype('string')
domain_10_timing = domain_10_augmented_candidate_inventory['Timing status'].astype('string')
domain_10_prior_domain = domain_10_augmented_candidate_inventory['Prior substantive domain'].fillna('').astype('string')
school_climate_variables = {'W1yys13YP', 'W2YYS13YP', 'W2YYS24YP'}
existing_school_context_variables = {'IndSchool', 'W1stschHS', 'W2SchstyHS', 'W2stschHS', 'W3schnameYP', 'W3stschHS',
    'W3SchstyHS'}
existing_truancy_variables = {'W1truantYP', 'W2truantYP', 'W3truantYP'}
local_geography_variables = {'urbind', 'gor'}
teacher_guidance_key_mask = pd.Series([(source_file, variable) in domain_9_teacher_guidance_key_set for source_file,
    variable in zip(domain_10_augmented_candidate_inventory['Source file'],
    domain_10_augmented_candidate_inventory['Variable'])], index=domain_10_augmented_candidate_inventory.index)
post_transition_mask = domain_10_timing.eq('At or after transition')
domain_10_augmented_candidate_inventory.loc[post_transition_mask, 'Domain 10 screening status'] = 'Timing exclusion'
domain_10_augmented_candidate_inventory.loc[post_transition_mask,
    'Domain 10 review track'] = 'Post-transition measurement'
domain_10_augmented_candidate_inventory.loc[post_transition_mask,
    'Domain 10 screening reason'] = 'The item was measured at or after the post-16 transition and is not used for the prospective predictor set'
school_climate_mask = domain_10_variable.isin(school_climate_variables) & domain_10_augmented_candidate_inventory['Domain 10 screening status'].isna()
domain_10_augmented_candidate_inventory.loc[school_climate_mask, 'Domain 10 screening status'] = 'Core review'
domain_10_augmented_candidate_inventory.loc[school_climate_mask,
    'Domain 10 review track'] = 'School climate and physical environment'
domain_10_augmented_candidate_inventory.loc[school_climate_mask,
    'Domain 10 screening reason'] = "The item directly measures the young person's perception of the school environment or teacher fairness"
local_geography_mask = domain_10_variable.str.lower().isin(local_geography_variables) & domain_10_augmented_candidate_inventory['Domain 10 screening status'].isna()
domain_10_augmented_candidate_inventory.loc[local_geography_mask, 'Domain 10 screening status'] = 'Core review'
domain_10_augmented_candidate_inventory.loc[local_geography_mask,
    'Domain 10 review track'] = 'Regional and urban–rural context'
domain_10_augmented_candidate_inventory.loc[local_geography_mask,
    'Domain 10 screening reason'] = 'The variable provides a direct geographical context measure that can be harmonised across permitted waves'
teacher_guidance_review_mask = teacher_guidance_key_mask & domain_10_augmented_candidate_inventory['Domain 10 screening status'].isna()
domain_10_augmented_candidate_inventory.loc[teacher_guidance_review_mask,
    'Domain 10 screening status'] = 'Core review'
domain_10_augmented_candidate_inventory.loc[teacher_guidance_review_mask,
    'Domain 10 review track'] = 'Teacher and careers guidance'
domain_10_augmented_candidate_inventory.loc[teacher_guidance_review_mask,
    'Domain 10 screening reason'] = 'The variable measures access to or use of teacher, careers or Connexions guidance before the transition'
existing_school_context_mask = domain_10_variable.isin(existing_school_context_variables) & domain_10_augmented_candidate_inventory['Domain 10 screening status'].isna()
domain_10_augmented_candidate_inventory.loc[existing_school_context_mask,
    'Domain 10 screening status'] = 'Already represented'
domain_10_augmented_candidate_inventory.loc[existing_school_context_mask,
    'Domain 10 review track'] = 'Existing school-sector or school-change representation'
domain_10_augmented_candidate_inventory.loc[existing_school_context_mask,
    'Domain 10 screening reason'] = 'The source variable has already been reviewed and contributes to an existing school-sector or school-change predictor'
existing_truancy_mask = domain_10_variable.isin(existing_truancy_variables) & domain_10_augmented_candidate_inventory['Domain 10 screening status'].isna()
domain_10_augmented_candidate_inventory.loc[existing_truancy_mask,
    'Domain 10 screening status'] = 'Already represented'
domain_10_augmented_candidate_inventory.loc[existing_truancy_mask,
    'Domain 10 review track'] = 'Existing truancy representation'
domain_10_augmented_candidate_inventory.loc[existing_truancy_mask,
    'Domain 10 screening reason'] = 'The item is already a construction input for the retained pre-transition truancy predictor'
health_attendance_mask = domain_10_variable.str.contains('chea7', case=False, na=False,
    regex=True) & domain_10_augmented_candidate_inventory['Domain 10 screening status'].isna()
domain_10_augmented_candidate_inventory.loc[health_attendance_mask,
    'Domain 10 screening status'] = 'Previously reviewed outside Domain 10'
domain_10_augmented_candidate_inventory.loc[health_attendance_mask,
    'Domain 10 review track'] = 'Health-related school attendance'
domain_10_augmented_candidate_inventory.loc[health_attendance_mask,
    'Domain 10 screening reason'] = 'The item concerns the effect of a health condition on school attendance and was reviewed in the health and disability domain'
sen_school_support_mask = domain_10_variable.str.contains('sencurr2|sentran', case=False, na=False,
    regex=True) & domain_10_augmented_candidate_inventory['Domain 10 screening status'].isna()
domain_10_augmented_candidate_inventory.loc[sen_school_support_mask,
    'Domain 10 screening status'] = 'Previously reviewed outside Domain 10'
domain_10_augmented_candidate_inventory.loc[sen_school_support_mask, 'Domain 10 review track'] = 'SEN school support'
domain_10_augmented_candidate_inventory.loc[sen_school_support_mask,
    'Domain 10 screening reason'] = 'The item is conditional on SEN or disability circumstances and was reviewed as support information in Domain 4'
attendance_discipline_mask = domain_10_augmented_candidate_inventory['Domain 10 search track'].eq('Attendance and disciplinary context') & domain_10_augmented_candidate_inventory['Domain 10 screening status'].isna()
domain_10_augmented_candidate_inventory.loc[attendance_discipline_mask,
    'Domain 10 screening status'] = 'Outside Domain 10'
domain_10_augmented_candidate_inventory.loc[attendance_discipline_mask,
    'Domain 10 review track'] = 'Individual attendance and disciplinary experience'
domain_10_augmented_candidate_inventory.loc[attendance_discipline_mask,
    'Domain 10 screening reason'] = "The item measures the young person's own absence, suspension or exclusion experience rather than school or local context"
exclusion_placement_mask = domain_10_variable.str.contains('expwhat', case=False, na=False,
    regex=True) & domain_10_augmented_candidate_inventory['Domain 10 screening status'].isna()
domain_10_augmented_candidate_inventory.loc[exclusion_placement_mask,
    'Domain 10 screening status'] = 'Outside Domain 10'
domain_10_augmented_candidate_inventory.loc[exclusion_placement_mask,
    'Domain 10 review track'] = 'Routed exclusion consequence'
domain_10_augmented_candidate_inventory.loc[exclusion_placement_mask,
    'Domain 10 screening reason'] = "The item records the consequence of a prior exclusion and does not identify the participant's general school sector"
school_choice_reason_mask = (domain_10_variable.str.startswith(('W1StascHS', 'W1YNtApHS', 'W1WhyBeHS'),
    na=False) | domain_10_variable.eq('W1WhatiHS0h')) & domain_10_augmented_candidate_inventory['Domain 10 screening status'].isna()
domain_10_augmented_candidate_inventory.loc[school_choice_reason_mask,
    'Domain 10 screening status'] = 'Outside Domain 10'
domain_10_augmented_candidate_inventory.loc[school_choice_reason_mask,
    'Domain 10 review track'] = 'Conditional school-choice rationale'
domain_10_augmented_candidate_inventory.loc[school_choice_reason_mask,
    'Domain 10 screening reason'] = 'The item records why a parent preferred or chose a school, rather than an observed characteristic of the attended school'
local_authority_care_mask = domain_10_label.str.contains('local authority care', case=False, na=False,
    regex=True) & domain_10_augmented_candidate_inventory['Domain 10 screening status'].isna()
domain_10_augmented_candidate_inventory.loc[local_authority_care_mask,
    'Domain 10 screening status'] = 'Search false positive'
domain_10_augmented_candidate_inventory.loc[local_authority_care_mask,
    'Domain 10 review track'] = 'Local-authority-care false positive'
domain_10_augmented_candidate_inventory.loc[local_authority_care_mask,
    'Domain 10 screening reason'] = 'The phrase refers to care status rather than neighbourhood or local-area context'
classroom_assistant_qualification_mask = domain_10_label.str.contains('learning support|classroom assistant',
    case=False, na=False, regex=True) & domain_10_variable.str.contains('qual', case=False, na=False,
    regex=True) & domain_10_augmented_candidate_inventory['Domain 10 screening status'].isna()
domain_10_augmented_candidate_inventory.loc[classroom_assistant_qualification_mask,
    'Domain 10 screening status'] = 'Search false positive'
domain_10_augmented_candidate_inventory.loc[classroom_assistant_qualification_mask,
    'Domain 10 review track'] = 'Parent-qualification false positive'
domain_10_augmented_candidate_inventory.loc[classroom_assistant_qualification_mask,
    'Domain 10 screening reason'] = "The item measures a parent's qualification rather than school provision or learning support received by the young person"
unresolved_domain_10_mask = domain_10_augmented_candidate_inventory['Domain 10 screening status'].isna()
domain_10_unresolved_screening = domain_10_augmented_candidate_inventory.loc[unresolved_domain_10_mask, ['Wave',
    'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label', 'Timing status', 'Prior review outcome', 'Domain 10 search track', 'All Domain 10 search matches']].copy().reset_index(drop=True)
domain_10_screened_candidates = domain_10_augmented_candidate_inventory.copy().sort_values(['Domain 10 screening status',
    'Domain 10 review track', 'Source order', 'Variable position']).reset_index(drop=True)
domain_10_screening_status_summary = domain_10_screened_candidates['Domain 10 screening status'].fillna('Unresolved').value_counts().rename('Variables').rename_axis('Domain 10 screening status').reset_index()
domain_10_screening_track_summary = domain_10_screened_candidates.groupby(['Domain 10 screening status',
    'Domain 10 review track'], dropna=False).size().rename('Variables').reset_index().sort_values(['Domain 10 screening status',
    'Domain 10 review track']).reset_index(drop=True)
domain_10_core_review_inventory = domain_10_screened_candidates.loc[domain_10_screened_candidates['Domain 10 screening status'].eq('Core review'),
    ['Source order', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status', 'Prior review outcome', 'Domain 10 review track', 'Domain 10 screening reason']].copy().reset_index(drop=True)
domain_10_screening_overview = pd.DataFrame([{'Original keyword-search candidates': int(len(domain_10_candidate_inventory)),
    'Domain 9 guidance variables': int(len(domain_9_teacher_guidance_keys)), 'Guidance variables added': int(len(domain_10_guidance_additions)), 'Augmented unique candidates': int(len(domain_10_augmented_candidate_inventory)), 'Core review variables': int(domain_10_screened_candidates['Domain 10 screening status'].eq('Core review').sum()), 'Unresolved screening decisions': int(len(domain_10_unresolved_screening))}])
print('Domain 10 screening overview:')
display_limited(domain_10_screening_overview)
print('\nScreening status counts:')
display_limited(domain_10_screening_status_summary)
print('\nScreening status by review track:')
display_limited(domain_10_screening_track_summary)
print(f'\nCore Domain 10 review variables: {len(domain_10_core_review_inventory):,}')
print('\nFirst 15 core review variables:')
display_limited(domain_10_core_review_inventory.head(15))
print(f'Additional core review variables not displayed: {max(len(domain_10_core_review_inventory) - 15, 0):,}')
print(f'\nUnresolved screening decisions: {len(domain_10_unresolved_screening):,}')
if not domain_10_unresolved_screening.empty:
    display_limited(domain_10_unresolved_screening)
else:
    print('No unresolved screening decisions.')
print('\nFiles written in this cell: 0')

Domain 10 screening overview:


,Original keyword-search candidates,Domain 9 guidance variables,Guidance variables added,Augmented unique candidates,Core review variables,Unresolved screening decisions
0,140,19,19,159,26,10



Screening status counts:


,Domain 10 screening status,Variables
0,Outside Domain 10,84
1,Core review,26
2,Timing exclusion,14
3,Already represented,10
4,Unresolved,10



Screening status by review track:


...


Core Domain 10 review variables: 26

First 15 core review variables:


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Prior review outcome,Domain 10 review track,Domain 10 screening reason
0,5,Wave 2,Family background,wave_two_lsype_family_background_2020,1027,urbind,Urban/Rural Indicator (England),Pre-transition source,Pending review,Regional and urban–rural context,The variable provides a direct geographical co...
1,5,Wave 2,Family background,wave_two_lsype_family_background_2020,1028,gor,Government Office Region,Pre-transition source,Pending review,Regional and urban–rural context,The variable provides a direct geographical co...
2,9,Wave 3,Family background,wave_three_lsype_family_background_2020,158,urbind,Urban/Rural Indicator (England),Near-transition source,Pending review,Regional and urban–rural context,The variable provides a direct geographical co...
3,9,Wave 3,Family background,wave_three_lsype_family_background_2020,159,gor,Government Office Region,Near-transition source,Pending review,Regional and urban–rural context,The variable provides a direct geographical co...
4,1,Wave 1,Young person,wave_one_lsype_young_person_2020,232,W1yys13YP,YP: Feelings about school: My school is clean ...,Pre-transition source,Pending review,School climate and physical environment,The item directly measures the young person's ...


Additional core review variables not displayed: 11

Unresolved screening decisions: 10


,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Prior review outcome,Domain 10 search track,All Domain 10 search matches
0,Wave 1,Young person,wave_one_lsype_young_person_2020,187,W1advconYP,YP: Whether heard about Connexions before inte...,Pre-transition source,Pending review,School provision and external support,School provision and external support
1,Wave 1,Young person,wave_one_lsype_young_person_2020,188,W1advperYP,YP: Whether ever talked to Connexions Personal...,Pre-transition source,Pending review,School provision and external support,School provision and external support
2,Wave 1,Young person,wave_one_lsype_young_person_2020,189,W1advconnYP,YP: How often talk about plans for future stud...,Pre-transition source,Pending review,School provision and external support,School provision and external support
3,Wave 1,Young person,wave_one_lsype_young_person_2020,194,W1infoconYP,YP: Usefulness of information from this source...,Pre-transition source,Pending review,School provision and external support,School provision and external support
4,Wave 2,Young person,wave_two_lsype_young_person_2020,286,W2advconYP,YP: Whether heard about Connexions before inte...,Pre-transition source,Pending review,School provision and external support,School provision and external support



Files written in this cell: 0


In [389]:
# 3: Connexions candidate-screening resolution

import pandas as pd
connexions_access_contact_variables = {'W1advconYP', 'W1advperYP', 'W2advconYP', 'W2advperYP', 'W3advconYP',
    'W3advperYP'}
connexions_guidance_use_variables = {'W1advconnYP', 'W1infoconYP', 'W2advconnYP', 'W2ModAp3YP0e'}
all_connexions_variables = connexions_access_contact_variables | connexions_guidance_use_variables
assert set(domain_10_unresolved_screening['Variable']) == all_connexions_variables
domain_10_screening_resolved = domain_10_augmented_candidate_inventory.copy()
connexions_access_contact_mask = domain_10_screening_resolved['Variable'].isin(connexions_access_contact_variables)
connexions_guidance_use_mask = domain_10_screening_resolved['Variable'].isin(connexions_guidance_use_variables)
assert int(connexions_access_contact_mask.sum()) == 6
assert int(connexions_guidance_use_mask.sum()) == 4
domain_10_screening_resolved.loc[connexions_access_contact_mask, 'Domain 10 screening status'] = 'Core review'
domain_10_screening_resolved.loc[connexions_access_contact_mask,
    'Domain 10 review track'] = 'Connexions access and contact'
domain_10_screening_resolved.loc[connexions_access_contact_mask,
    'Domain 10 screening reason'] = 'The item measures awareness of or direct contact with the Connexions service before the post-16 transition'
domain_10_screening_resolved.loc[connexions_guidance_use_mask, 'Domain 10 screening status'] = 'Core review'
domain_10_screening_resolved.loc[connexions_guidance_use_mask,
    'Domain 10 review track'] = 'Connexions guidance use and content'
domain_10_screening_resolved.loc[connexions_guidance_use_mask,
    'Domain 10 screening reason'] = 'The item measures discussion of future study or training with Connexions, or the perceived usefulness of that guidance'
domain_10_screened_candidates = domain_10_screening_resolved.sort_values(['Domain 10 screening status',
    'Domain 10 review track', 'Source order', 'Variable position']).reset_index(drop=True)
domain_10_unresolved_screening = domain_10_screened_candidates.loc[domain_10_screened_candidates['Domain 10 screening status'].isna()].copy().reset_index(drop=True)
assert len(domain_10_unresolved_screening) == 0
domain_10_core_review_inventory = domain_10_screened_candidates.loc[domain_10_screened_candidates['Domain 10 screening status'].eq('Core review'),
    ['Source order', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status', 'Prior review outcome', 'Domain 10 review track', 'Domain 10 screening reason']].copy().sort_values(['Domain 10 review track',
    'Source order', 'Variable position']).reset_index(drop=True)
assert len(domain_10_core_review_inventory) == 36
domain_10_screening_status_summary = domain_10_screened_candidates['Domain 10 screening status'].value_counts().rename('Variables').rename_axis('Domain 10 screening status').reset_index()
domain_10_core_track_summary = domain_10_core_review_inventory['Domain 10 review track'].value_counts().rename('Variables').rename_axis('Domain 10 review track').reset_index()
domain_10_connexions_inventory = domain_10_core_review_inventory.loc[domain_10_core_review_inventory['Variable'].isin(all_connexions_variables)].copy().reset_index(drop=True)
domain_10_screening_resolution_summary = pd.DataFrame([{'Augmented candidate variables': int(len(domain_10_screened_candidates)),
    'Core review variables': int(len(domain_10_core_review_inventory)), 'Connexions variables added to core review': int(len(domain_10_connexions_inventory)), 'Unresolved screening decisions': int(len(domain_10_unresolved_screening))}])
print('Domain 10 screening resolution:')
display_limited(domain_10_screening_resolution_summary)
print('Screening status counts:')
display_limited(domain_10_screening_status_summary)
print('Core review variables by track:')
display_limited(domain_10_core_track_summary)
print('Connexions core-review inventory:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_10_connexions_inventory)
print('Unresolved screening decisions:')
display_limited(domain_10_unresolved_screening)
print('Files written in this cell: 0')

Domain 10 screening resolution:


,Augmented candidate variables,Core review variables,Connexions variables added to core review,Unresolved screening decisions
0,159,36,10,0


Screening status counts:


,Domain 10 screening status,Variables
0,Outside Domain 10,84
1,Core review,36
2,Timing exclusion,14
3,Already represented,10
4,Search false positive,8


Core review variables by track:


...

Connexions core-review inventory:


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Prior review outcome,Domain 10 review track,Domain 10 screening reason
0,1,Wave 1,Young person,wave_one_lsype_young_person_2020,187,W1advconYP,YP: Whether heard about Connexions before interview,Pre-transition source,Pending review,Connexions access and contact,The item measures awareness of or direct contact with the Connexions service before the post-16 transition
1,1,Wave 1,Young person,wave_one_lsype_young_person_2020,188,W1advperYP,YP: Whether ever talked to Connexions Personal Advisor,Pre-transition source,Pending review,Connexions access and contact,The item measures awareness of or direct contact with the Connexions service before the post-16 transition
2,4,Wave 2,Young person,wave_two_lsype_young_person_2020,286,W2advconYP,YP: Whether heard about Connexions before interview,Pre-transition source,Pending review,Connexions access and contact,The item measures awareness of or direct contact with the Connexions service before the post-16 transition
3,4,Wave 2,Young person,wave_two_lsype_young_person_2020,287,W2advperYP,YP: Whether talked to Connexions Personal Advisor since last wave,Pre-transition source,Pending review,Connexions access and contact,The item measures awareness of or direct contact with the Connexions service before the post-16 transition
4,8,Wave 3,Young person,wave_three_lsype_young_person_2020,226,W3advconYP,YP: Whether YP heard about Connexions before interview,Near-transition source,Pending review,Connexions access and contact,The item measures awareness of or direct contact with the Connexions service before the post-16 transition


Unresolved screening decisions:


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Data type,Timing status,Review status,...,Reference-period assessment,Documentation source,Review notes,Domain 10 search track,All Domain 10 search matches,Prior review outcome,Prior substantive domain,Domain 10 screening status,Domain 10 review track,Domain 10 screening reason


Files written in this cell: 0


In [390]:
# 4: School-environment item-battery inventory

import re
import pandas as pd
school_environment_battery_pattern = '^W(?P<Wave_number>[123])YYS(?P<Item_number>1[3-9]|2[0-6])YP$'
school_environment_battery_mask = domain_10_register_source['Variable'].astype('string').str.match(school_environment_battery_pattern,
    case=False, na=False)
school_environment_battery_inventory = domain_10_register_source.loc[school_environment_battery_mask].copy()
school_environment_battery_identifiers = school_environment_battery_inventory['Variable'].astype('string').str.extract(school_environment_battery_pattern,
    flags=re.IGNORECASE, expand=True)
school_environment_battery_inventory['Battery wave'] = pd.to_numeric(school_environment_battery_identifiers['Wave_number'],
    errors='coerce').astype('Int64')
school_environment_battery_inventory['Battery item'] = pd.to_numeric(school_environment_battery_identifiers['Item_number'],
    errors='coerce').astype('Int64')
assert school_environment_battery_inventory[['Battery wave', 'Battery item']].notna().all().all()
domain_10_core_review_keys = set(zip(domain_10_core_review_inventory['Source file'],
    domain_10_core_review_inventory['Variable']))
school_environment_battery_inventory['Current Domain 10 core review'] = [(source_file,
    variable) in domain_10_core_review_keys for source_file, variable in zip(school_environment_battery_inventory['Source file'],
    school_environment_battery_inventory['Variable'])]
school_environment_battery_inventory['Existing review outcome'] = school_environment_battery_inventory['Review outcome']
school_environment_battery_inventory['Existing substantive domain'] = school_environment_battery_inventory['Substantive domain']
school_environment_battery_inventory['Existing decision reason'] = school_environment_battery_inventory['Decision reason']
school_environment_battery_inventory = school_environment_battery_inventory.sort_values(['Battery item',
    'Battery wave']).reset_index(drop=True)
school_environment_battery_pair_summary = school_environment_battery_inventory.pivot_table(index='Battery item',
    columns='Battery wave', values=['Variable', 'Variable label', 'Existing review outcome',
    'Existing substantive domain', 'Current Domain 10 core review'], aggfunc='first', dropna=False)
school_environment_battery_pair_summary.columns = [f'{field} — Wave {wave_number}' for field,
    wave_number in school_environment_battery_pair_summary.columns]
school_environment_battery_pair_summary = school_environment_battery_pair_summary.reset_index().sort_values('Battery item').reset_index(drop=True)
school_environment_battery_wave_summary = school_environment_battery_inventory.groupby('Battery wave',
    as_index=False).agg(Items=('Battery item', 'nunique'), Variables=('Variable', 'count'),
    Core_review_variables=('Current Domain 10 core review', 'sum')).sort_values('Battery wave').reset_index(drop=True)
school_environment_battery_decision_summary = school_environment_battery_inventory.groupby(['Existing substantive domain',
    'Existing review outcome'], dropna=False).size().rename('Variables').reset_index().sort_values(['Existing substantive domain',
    'Existing review outcome'], na_position='last').reset_index(drop=True)
school_environment_battery_not_in_core = school_environment_battery_inventory.loc[~school_environment_battery_inventory['Current Domain 10 core review'],
    ['Battery wave', 'Battery item', 'Wave', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status', 'Existing review outcome', 'Existing substantive domain', 'Existing decision reason']].copy().reset_index(drop=True)
school_environment_battery_overview = pd.DataFrame([{'Battery variables found': int(len(school_environment_battery_inventory)),
    'Distinct battery items': int(school_environment_battery_inventory['Battery item'].nunique()), 'Waves represented': int(school_environment_battery_inventory['Battery wave'].nunique()), 'Variables currently in Domain 10 core review': int(school_environment_battery_inventory['Current Domain 10 core review'].sum()), 'Battery variables outside current Domain 10 core review': int((~school_environment_battery_inventory['Current Domain 10 core review']).sum())}])
print('School-environment battery overview:')
display_limited(school_environment_battery_overview)
print('\nBattery coverage by wave:')
display_limited(school_environment_battery_wave_summary)
print('\nExisting decisions within the battery:')
display_limited(school_environment_battery_decision_summary)
print(f'\nWave-specific item comparisons: {len(school_environment_battery_pair_summary):,}')
print('\nFirst 15 wave-specific item comparisons:')
display_limited(school_environment_battery_pair_summary.head(15))
print(f'Additional item comparisons not displayed: {max(len(school_environment_battery_pair_summary) - 15, 0):,}')
print(f'\nBattery variables outside the current Domain 10 core review: {len(school_environment_battery_not_in_core):,}')
if not school_environment_battery_not_in_core.empty:
    display_limited(school_environment_battery_not_in_core.head(10))
    print(f'Additional variables outside the core review not displayed: {max(len(school_environment_battery_not_in_core) - 10, 0):,}')
else:
    print('No battery variables fall outside the current core review.')
school_environment_pair_summary_file = stage_2_output_directory / 'stage_2_school_environment_item_comparison.csv'
school_environment_not_in_core_file = stage_2_output_directory / 'stage_2_school_environment_not_in_core_review.csv'
school_environment_battery_pair_summary.to_csv(school_environment_pair_summary_file, index=False)
school_environment_battery_not_in_core.to_csv(school_environment_not_in_core_file, index=False)
print(f'\nFull item-comparison and out-of-core tables saved to: {stage_2_output_directory}')
print('Files written in this cell: 2')

School-environment battery overview:


,Battery variables found,Distinct battery items,Waves represented,Variables currently in Domain 10 core review,Battery variables outside current Domain 10 core review
0,23,14,2,3,20



Battery coverage by wave:


,Battery wave,Items,Variables,Core_review_variables
0,1,11,11,1
1,2,12,12,2



Existing decisions within the battery:


...


Wave-specific item comparisons: 14

First 15 wave-specific item comparisons:


,Battery item,Current Domain 10 core review — Wave 1,Current Domain 10 core review — Wave 2,Existing review outcome — Wave 1,Existing review outcome — Wave 2,Existing substantive domain — Wave 1,Existing substantive domain — Wave 2,Variable — Wave 1,Variable — Wave 2,Variable label — Wave 1,Variable label — Wave 2
0,13,True,True,Pending review,Pending review,School and local context,School and local context,W1yys13YP,W2YYS13YP,YP: Feelings about school: My school is clean ...,YP: Feelings about school: My school is clean ...
1,14,False,False,Review support,Review support,School attitudes and engagement,School attitudes and engagement,W1yys14YP,W2YYS14YP,YP: How many teachers this applies to: My teac...,YP: How many teachers this applies to: My teac...
2,15,False,False,Review support,Review support,School attitudes and engagement,School attitudes and engagement,W1yys15YP,W2YYS15YP,YP: How many teachers this applies to: The tea...,YP: How many teachers this applies to: The tea...
3,16,False,False,Review support,Review support,School attitudes and engagement,School attitudes and engagement,W1yys16YP,W2YYS16YP,YP: How many teachers this applies to: The tea...,YP: How many teachers this applies to: The tea...
4,17,False,False,Review support,Review support,School attitudes and engagement,School attitudes and engagement,W1yys17YP,W2YYS17YP,YP: How many teachers this applies to: My teac...,YP: How many teachers this applies to: My teac...


Additional item comparisons not displayed: 0

Battery variables outside the current Domain 10 core review: 20


,Battery wave,Battery item,Wave,Source file,Variable position,Variable,Variable label,Timing status,Existing review outcome,Existing substantive domain,Existing decision reason
0,1,14,Wave 1,wave_one_lsype_young_person_2020,233,W1yys14YP,YP: How many teachers this applies to: My teac...,Pre-transition source,Review support,School attitudes and engagement,Used to assess teacher-related constructs but ...
1,2,14,Wave 2,wave_two_lsype_young_person_2020,364,W2YYS14YP,YP: How many teachers this applies to: My teac...,Pre-transition source,Review support,School attitudes and engagement,Used to assess teacher-related constructs but ...
2,1,15,Wave 1,wave_one_lsype_young_person_2020,234,W1yys15YP,YP: How many teachers this applies to: The tea...,Pre-transition source,Review support,School attitudes and engagement,Used to assess teacher-related constructs but ...
3,2,15,Wave 2,wave_two_lsype_young_person_2020,365,W2YYS15YP,YP: How many teachers this applies to: The tea...,Pre-transition source,Review support,School attitudes and engagement,Used to assess teacher-related constructs but ...
4,1,16,Wave 1,wave_one_lsype_young_person_2020,235,W1yys16YP,YP: How many teachers this applies to: The tea...,Pre-transition source,Review support,School attitudes and engagement,Used to assess teacher-related constructs but ...


Additional variables outside the core review not displayed: 10

Full item-comparison and out-of-core tables saved to: data_derived\stage_2_predictor_construction
Files written in this cell: 2


In [391]:
# 5: School-climate response and cross-wave review

import itertools
import pandas as pd
from scipy.stats import spearmanr
school_climate_review_variable_names = ['W1yys13YP', 'W2YYS13YP', 'W2YYS24YP', 'W2YYS25YP', 'W2YYS26YP']
school_climate_review_inventory = domain_10_register_source.loc[domain_10_register_source['Variable'].isin(school_climate_review_variable_names)].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(school_climate_review_inventory) == 5
assert set(school_climate_review_inventory['Variable']) == set(school_climate_review_variable_names)
school_climate_raw = pd.DataFrame(index=route_index)
school_climate_labelled = pd.DataFrame(index=route_index)
for source_file, source_inventory in school_climate_review_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    for source_data in [raw_source, labelled_source]:
        source_data['NSID'] = standardise_nsid(source_data['NSID'])
        assert source_data['NSID'].is_unique
    raw_source = raw_source.set_index('NSID').reindex(route_index)
    labelled_source = labelled_source.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        school_climate_raw[variable] = pd.to_numeric(raw_source[variable], errors='coerce')
        school_climate_labelled[variable] = labelled_source[variable].astype('string')
assert set(school_climate_raw.columns) == set(school_climate_review_variable_names)
school_climate_coverage_rows = []
for _, inventory_row in school_climate_review_inventory.iterrows():
    variable = inventory_row['Variable']
    raw_values = school_climate_raw[variable]
    observed_mask = raw_values.ge(0).fillna(False).astype(bool)
    structural_mask = raw_values.eq(-91.0).fillna(False).astype(bool)
    other_special_mask = raw_values.lt(0).fillna(False).astype(bool) & ~structural_mask
    school_climate_coverage_rows.append({'Wave': inventory_row['Wave'], 'Variable': variable,
        'Variable label': inventory_row['Variable label'], 'Observed responses': int(observed_mask.sum()), 'Observed percentage': round(observed_mask.mean() * 100,
        2), 'Structural non-applicability': int(structural_mask.sum()), 'Other special-code responses': int(other_special_mask.sum()), 'No source record': int(raw_values.isna().sum()), 'Observed categories': int(raw_values.where(observed_mask).nunique())})
school_climate_coverage_summary = pd.DataFrame(school_climate_coverage_rows)
school_climate_code_rows = []
for _, inventory_row in school_climate_review_inventory.iterrows():
    variable = inventory_row['Variable']
    raw_values = school_climate_raw[variable]
    labelled_values = school_climate_labelled[variable]
    for raw_code, participants in raw_values.value_counts(dropna=False).items():
        if pd.isna(raw_code):
            value_label = 'No source record'
            response_type = 'Unavailable'
            sort_value = 999999
        else:
            response_mask = raw_values.eq(raw_code)
            matching_labels = labelled_values.loc[response_mask].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            if raw_code >= 0:
                response_type = 'Observed response'
            elif raw_code == -91:
                response_type = 'Structural non-applicability'
            else:
                response_type = 'Other special code'
            sort_value = float(raw_code)
        school_climate_code_rows.append({'Wave': inventory_row['Wave'], 'Variable': variable, 'Raw code': raw_code,
            'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
school_climate_code_distribution = pd.DataFrame(school_climate_code_rows).sort_values(['Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
school_clean_tidy_pair = pd.DataFrame({'Wave 1 raw': school_climate_raw['W1yys13YP'],
    'Wave 2 raw': school_climate_raw['W2YYS13YP'], 'Wave 1 label': school_climate_labelled['W1yys13YP'], 'Wave 2 label': school_climate_labelled['W2YYS13YP']}, index=route_index)
school_clean_tidy_complete = school_clean_tidy_pair.loc[school_clean_tidy_pair['Wave 1 raw'].ge(0) & school_clean_tidy_pair['Wave 2 raw'].ge(0)].copy()
assert len(school_clean_tidy_complete) > 0
school_clean_tidy_wave_consistency = pd.DataFrame([{'Complete comparisons': int(len(school_clean_tidy_complete)),
    'Exact agreement': int(school_clean_tidy_complete['Wave 1 raw'].eq(school_clean_tidy_complete['Wave 2 raw']).sum()), 'Exact agreement percentage': round(school_clean_tidy_complete['Wave 1 raw'].eq(school_clean_tidy_complete['Wave 2 raw']).mean() * 100,
    2), 'Spearman correlation': round(float(spearmanr(school_clean_tidy_complete['Wave 1 raw'],
    school_clean_tidy_complete['Wave 2 raw']).statistic), 3), "Cramer's V": round(categorical_cramers_v(school_clean_tidy_complete['Wave 1 raw'],
    school_clean_tidy_complete['Wave 2 raw']), 3)}])
school_clean_tidy_crosstab = pd.crosstab(school_clean_tidy_complete['Wave 1 label'],
    school_clean_tidy_complete['Wave 2 label'], margins=True, dropna=False)
wave_2_teacher_climate_variables = ['W2YYS24YP', 'W2YYS25YP', 'W2YYS26YP']
teacher_climate_pairwise_rows = []
for variable_a, variable_b in itertools.combinations(wave_2_teacher_climate_variables, 2):
    pair_data = school_climate_raw[[variable_a,
        variable_b]].loc[lambda data: data[variable_a].ge(0) & data[variable_b].ge(0)].dropna()
    assert len(pair_data) > 0
    teacher_climate_pairwise_rows.append({'First variable': variable_a, 'Second variable': variable_b,
        'Complete comparisons': int(len(pair_data)), 'Spearman correlation using raw codes': round(float(spearmanr(pair_data[variable_a],
        pair_data[variable_b]).statistic), 3), "Cramer's V": round(categorical_cramers_v(pair_data[variable_a],
        pair_data[variable_b]), 3)})
teacher_climate_pairwise_summary = pd.DataFrame(teacher_climate_pairwise_rows)
teacher_climate_observed_mask = school_climate_raw[wave_2_teacher_climate_variables].ge(0)
teacher_climate_available_count = teacher_climate_observed_mask.sum(axis=1)
teacher_climate_joint_availability = teacher_climate_available_count.value_counts().sort_index().rename('Participants').rename_axis('Teacher-climate items available').reset_index()
teacher_climate_joint_availability['Percentage of full sample'] = (teacher_climate_joint_availability['Participants'] / len(route_index) * 100).round(2)
print('School-climate review inventory:')
with pd.option_context('display.max_colwidth', None):
    display_limited(school_climate_review_inventory[['Wave', 'Source file', 'Variable position', 'Variable',
        'Variable label', 'Review outcome', 'Substantive domain']])
print('Response coverage:')
with pd.option_context('display.max_colwidth', None):
    display_limited(school_climate_coverage_summary)
print('Response-code distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(school_climate_code_distribution)
print('Clean-and-tidy cross-wave consistency:')
display_limited(school_clean_tidy_wave_consistency)
print('Clean-and-tidy response cross-tabulation:')
display_limited(school_clean_tidy_crosstab)
print('Wave 2 teacher-climate pairwise associations:')
display_limited(teacher_climate_pairwise_summary)
print('Wave 2 teacher-climate joint availability:')
display_limited(teacher_climate_joint_availability)
print('Files written in this cell: 0')

School-climate review inventory:


,Wave,Source file,Variable position,Variable,Variable label,Review outcome,Substantive domain
0,Wave 1,wave_one_lsype_young_person_2020,232,W1yys13YP,YP: Feelings about school: My school is clean and tidy,Pending review,School and local context
1,Wave 2,wave_two_lsype_young_person_2020,363,W2YYS13YP,YP: Feelings about school: My school is clean and tidy,Pending review,School and local context
2,Wave 2,wave_two_lsype_young_person_2020,370,W2YYS24YP,YP: How many teachers this applies to: My teachers treat everyone the same regar,Review support,School attitudes and engagement
3,Wave 2,wave_two_lsype_young_person_2020,371,W2YYS25YP,YP: How many teachers this applies to: My teachers don't really listen to what I,Review support,School attitudes and engagement
4,Wave 2,wave_two_lsype_young_person_2020,372,W2YYS26YP,YP: How many teachers this applies to: I get treated unfairly by my teachers,Review support,School attitudes and engagement


Response coverage:


,Wave,Variable,Variable label,Observed responses,Observed percentage,Structural non-applicability,Other special-code responses,No source record,Observed categories
0,Wave 1,W1yys13YP,YP: Feelings about school: My school is clean and tidy,9009,92.24,0,515,243,4
1,Wave 2,W2YYS13YP,YP: Feelings about school: My school is clean and tidy,9036,92.52,0,485,246,4
2,Wave 2,W2YYS24YP,YP: How many teachers this applies to: My teachers treat everyone the same regar,9210,94.30,0,311,246,5
3,Wave 2,W2YYS25YP,YP: How many teachers this applies to: My teachers don't really listen to what I,9076,92.93,0,445,246,5
4,Wave 2,W2YYS26YP,YP: How many teachers this applies to: I get treated unfairly by my teachers,9178,93.97,0,343,246,5


Response-code distributions:


...

Wave 2 teacher-climate pairwise associations:


,First variable,Second variable,Complete comparisons,Spearman correlation using raw codes,Cramer's V
0,W2YYS24YP,W2YYS25YP,8966,-0.254,0.153
1,W2YYS24YP,W2YYS26YP,9072,-0.332,0.188
2,W2YYS25YP,W2YYS26YP,8962,0.468,0.313


Wave 2 teacher-climate joint availability:


,Teacher-climate items available,Participants,Percentage of full sample
0,0,438,4.48
1,1,59,0.60
2,2,405,4.15
3,3,8865,90.76


Files written in this cell: 0


In [392]:
# 6: School-climate representation and overlap review

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
clean_tidy_recode = {1.0: 3, 2.0: 2, 3.0: 1, 4.0: 0}
school_clean_tidy_wave_1 = school_climate_raw['W1yys13YP'].map(clean_tidy_recode).astype('Int64')
school_clean_tidy_wave_2 = school_climate_raw['W2YYS13YP'].map(clean_tidy_recode).astype('Int64')
school_physical_environment_candidate = school_clean_tidy_wave_2.combine_first(school_clean_tidy_wave_1).astype('Int64').rename('school_physical_environment_perception_pretransition')
school_physical_environment_source = pd.Series(pd.NA, index=route_index, dtype='string',
    name='School physical-environment source')
school_physical_environment_source.loc[school_clean_tidy_wave_2.notna()] = 'Wave 2'
school_physical_environment_source.loc[school_clean_tidy_wave_2.isna() & school_clean_tidy_wave_1.notna()] = 'Wave 1 fallback'
assert set(school_physical_environment_candidate.dropna().astype(int).unique()).issubset({0, 1, 2, 3})
teacher_fairness_recode = {1.0: 4, 2.0: 3, 3.0: 2, 4.0: 1, 5.0: 0}
teacher_negative_item_recode = {1.0: 0, 2.0: 1, 3.0: 2, 4.0: 3, 5.0: 4}
teacher_climate_recoded_items = pd.DataFrame({'Equal treatment': school_climate_raw['W2YYS24YP'].map(teacher_fairness_recode),
    'Teachers listen': school_climate_raw['W2YYS25YP'].map(teacher_negative_item_recode), 'Fair personal treatment': school_climate_raw['W2YYS26YP'].map(teacher_negative_item_recode)}, index=route_index).astype('Float64')
teacher_climate_item_count = teacher_climate_recoded_items.notna().sum(axis=1).astype('Int64').rename('Teacher-climate items available')
teacher_climate_complete_score_candidate = teacher_climate_recoded_items.mean(axis=1).where(teacher_climate_item_count.eq(3)).astype('Float64').rename('teacher_climate_complete_score_pretransition')
teacher_climate_minimum_two_score_candidate = teacher_climate_recoded_items.mean(axis=1).where(teacher_climate_item_count.ge(2)).astype('Float64').rename('teacher_climate_support_score_pretransition')
teacher_climate_complete_items = teacher_climate_recoded_items.dropna().astype(float)
assert len(teacher_climate_complete_items) == 8865
teacher_climate_item_variances = teacher_climate_complete_items.var(axis=0, ddof=1)
teacher_climate_total_score = teacher_climate_complete_items.sum(axis=1)
teacher_climate_total_variance = float(teacher_climate_total_score.var(ddof=1))
teacher_climate_item_number = int(teacher_climate_complete_items.shape[1])
teacher_climate_cronbach_alpha = teacher_climate_item_number / (teacher_climate_item_number - 1) * (1 - teacher_climate_item_variances.sum() / teacher_climate_total_variance)
teacher_climate_item_total_rows = []
for item_name in teacher_climate_complete_items.columns:
    remaining_item_total = teacher_climate_complete_items.drop(columns=[item_name]).sum(axis=1)
    teacher_climate_item_total_rows.append({'Item': item_name,
        'Complete responses': int(len(teacher_climate_complete_items)), 'Corrected item-total Spearman correlation': round(float(spearmanr(teacher_climate_complete_items[item_name],
        remaining_item_total).statistic), 3)})
teacher_climate_item_total_summary = pd.DataFrame(teacher_climate_item_total_rows)
teacher_climate_recoded_pairwise_rows = []
for item_position, first_item in enumerate(teacher_climate_recoded_items.columns):
    for second_item in teacher_climate_recoded_items.columns[item_position + 1:]:
        pair_data = teacher_climate_recoded_items[[first_item, second_item]].dropna().astype(float)
        teacher_climate_recoded_pairwise_rows.append({'First item': first_item, 'Second item': second_item,
            'Complete comparisons': int(len(pair_data)), 'Spearman correlation': round(float(spearmanr(pair_data[first_item],
            pair_data[second_item]).statistic), 3)})
teacher_climate_recoded_pairwise_summary = pd.DataFrame(teacher_climate_recoded_pairwise_rows)
school_climate_candidate_summary = pd.DataFrame([{'Candidate representation': 'Latest clean-and-tidy response with Wave 1 fallback',
    'Candidate predictor': 'school_physical_environment_perception_pretransition', 'Non-missing': int(school_physical_environment_candidate.notna().sum()), 'Missing': int(school_physical_environment_candidate.isna().sum()), 'Missing percentage': round(school_physical_environment_candidate.isna().mean() * 100,
    2), 'Distinct values': int(school_physical_environment_candidate.nunique())}, {'Candidate representation': 'Teacher-climate mean requiring all three items',
    'Candidate predictor': 'teacher_climate_complete_score_pretransition', 'Non-missing': int(teacher_climate_complete_score_candidate.notna().sum()), 'Missing': int(teacher_climate_complete_score_candidate.isna().sum()), 'Missing percentage': round(teacher_climate_complete_score_candidate.isna().mean() * 100,
    2), 'Distinct values': int(teacher_climate_complete_score_candidate.nunique())}, {'Candidate representation': 'Teacher-climate mean requiring at least two items',
    'Candidate predictor': 'teacher_climate_support_score_pretransition', 'Non-missing': int(teacher_climate_minimum_two_score_candidate.notna().sum()), 'Missing': int(teacher_climate_minimum_two_score_candidate.isna().sum()), 'Missing percentage': round(teacher_climate_minimum_two_score_candidate.isna().mean() * 100,
    2), 'Distinct values': int(teacher_climate_minimum_two_score_candidate.nunique())}])
school_physical_environment_source_summary = school_physical_environment_source.value_counts(dropna=False).rename('Participants').rename_axis('Source').reset_index()
school_physical_environment_source_summary['Percentage of full sample'] = (school_physical_environment_source_summary['Participants'] / len(route_index) * 100).round(2)
teacher_climate_scale_summary = pd.DataFrame([{'Complete three-item responses': int(len(teacher_climate_complete_items)),
    'Responses with at least two items': int(teacher_climate_item_count.ge(2).sum()), "Cronbach's alpha": round(float(teacher_climate_cronbach_alpha),
    3), 'Complete-score mean': round(float(teacher_climate_complete_score_candidate.mean()),
    3), 'Complete-score median': round(float(teacher_climate_complete_score_candidate.median()),
    3), 'Complete-score minimum': float(teacher_climate_complete_score_candidate.min()), 'Complete-score maximum': float(teacher_climate_complete_score_candidate.max())}])

def retrieve_existing_predictor(predictor_name):
    direct_object = globals().get(predictor_name)
    if isinstance(direct_object, pd.Series):
        return (direct_object.reindex(route_index).copy().rename(predictor_name), predictor_name)
    if isinstance(direct_object, pd.DataFrame) and predictor_name in direct_object.columns:
        direct_data = direct_object.copy()
        if 'NSID' in direct_data.columns:
            direct_data['NSID'] = standardise_nsid(direct_data['NSID'])
            return (direct_data.set_index('NSID')[predictor_name].reindex(route_index).copy(), predictor_name)
        return (direct_data[predictor_name].reindex(route_index).copy(), predictor_name)
    for object_name, object_value in globals().items():
        if not isinstance(object_value, pd.DataFrame):
            continue
        if predictor_name not in object_value.columns:
            continue
        candidate_data = object_value.copy()
        if 'NSID' in candidate_data.columns:
            candidate_data['NSID'] = standardise_nsid(candidate_data['NSID'])
            if not candidate_data['NSID'].is_unique:
                continue
            candidate_series = candidate_data.set_index('NSID')[predictor_name].reindex(route_index)
        else:
            candidate_series = candidate_data[predictor_name].reindex(route_index)
        return (candidate_series.copy().rename(predictor_name), object_name)
    raise NameError(f'Could not retrieve existing predictor: {predictor_name}')
school_climate_comparison_specs = {'School attitude score': 'school_attitude_score',
    'Academic self-concept score': 'academic_self_concept_score', 'Bullying experience': 'bullying_experience_pretransition', 'Truancy status': 'truancy_status_pretransition'}
school_climate_existing_comparisons = {}
school_climate_comparison_source_rows = []
for comparison_label, predictor_name in school_climate_comparison_specs.items():
    comparison_series, source_object = retrieve_existing_predictor(predictor_name)
    school_climate_existing_comparisons[comparison_label] = comparison_series
    school_climate_comparison_source_rows.append({'Comparison measure': comparison_label, 'Predictor': predictor_name,
        'Source object': source_object})
school_climate_comparison_source_summary = pd.DataFrame(school_climate_comparison_source_rows)
school_climate_overlap_data = pd.DataFrame({'Physical-school environment': school_physical_environment_candidate,
    'Teacher-climate support': teacher_climate_minimum_two_score_candidate, **school_climate_existing_comparisons}, index=route_index)
school_climate_overlap_rows = []
for candidate_measure in ['Physical-school environment', 'Teacher-climate support']:
    for comparison_measure in ['School attitude score', 'Academic self-concept score', 'Bullying experience',
        'Truancy status']:
        pair_data = school_climate_overlap_data[[candidate_measure, comparison_measure]].dropna().astype(float)
        school_climate_overlap_rows.append({'Candidate measure': candidate_measure,
            'Comparison measure': comparison_measure, 'Complete comparisons': int(len(pair_data)), 'Spearman correlation': round(float(spearmanr(pair_data[candidate_measure],
            pair_data[comparison_measure]).statistic), 3)})
school_climate_candidate_pair = school_climate_overlap_data[['Physical-school environment',
    'Teacher-climate support']].dropna().astype(float)
school_climate_overlap_rows.append({'Candidate measure': 'Physical-school environment',
    'Comparison measure': 'Teacher-climate support', 'Complete comparisons': int(len(school_climate_candidate_pair)), 'Spearman correlation': round(float(spearmanr(school_climate_candidate_pair['Physical-school environment'],
    school_climate_candidate_pair['Teacher-climate support']).statistic), 3)})
school_climate_overlap_summary = pd.DataFrame(school_climate_overlap_rows).sort_values(['Candidate measure',
    'Spearman correlation'], ascending=[True, False]).reset_index(drop=True)
print('Candidate representation summary:')
with pd.option_context('display.max_colwidth', None):
    display_limited(school_climate_candidate_summary)
print('Physical-school environment source use:')
display_limited(school_physical_environment_source_summary)
print('Teacher-climate scale summary:')
display_limited(teacher_climate_scale_summary)
print('Teacher-climate corrected item-total associations:')
display_limited(teacher_climate_item_total_summary)
print('Teacher-climate pairwise associations after recoding:')
display_limited(teacher_climate_recoded_pairwise_summary)
print('Existing predictor retrieval:')
with pd.option_context('display.max_colwidth', None):
    display_limited(school_climate_comparison_source_summary)
print('Candidate overlap with retained predictors:')
display_limited(school_climate_overlap_summary)
print('Files written in this cell: 0')

Candidate representation summary:


,Candidate representation,Candidate predictor,Non-missing,Missing,Missing percentage,Distinct values
0,Latest clean-and-tidy response with Wave 1 fallback,school_physical_environment_perception_pretransition,9431,336,3.44,4
1,Teacher-climate mean requiring all three items,teacher_climate_complete_score_pretransition,8865,902,9.24,13
2,Teacher-climate mean requiring at least two items,teacher_climate_support_score_pretransition,9270,497,5.09,17


Physical-school environment source use:


,Source,Participants,Percentage of full sample
0,Wave 2,9036,92.52
1,Wave 1 fallback,395,4.04
2,<NA>,336,3.44


Teacher-climate scale summary:


...

Existing predictor retrieval:


,Comparison measure,Predictor,Source object
0,School attitude score,school_attitude_score,school_attitude_representation_review
1,Academic self-concept score,academic_self_concept_score,domain_7_predictor_candidates
2,Bullying experience,bullying_experience_pretransition,domain_8_predictor_candidates
3,Truancy status,truancy_status_pretransition,domain_6_predictor_candidates


Candidate overlap with retained predictors:


,Candidate measure,Comparison measure,Complete comparisons,Spearman correlation
0,Physical-school environment,School attitude score,9429,0.271
1,Physical-school environment,Teacher-climate support,9219,0.249
2,Physical-school environment,Academic self-concept score,9239,0.136
3,Physical-school environment,Bullying experience,9418,-0.087
4,Physical-school environment,Truancy status,9410,-0.116


Files written in this cell: 0


In [393]:
# 7: Alternative teacher-climate representation review

import pandas as pd
from scipy.stats import pearsonr, spearmanr
teacher_equal_treatment_candidate = teacher_climate_recoded_items['Equal treatment'].copy().astype('Float64').rename('teacher_equal_treatment_pretransition')
teacher_listening_fair_treatment_items = teacher_climate_recoded_items[['Teachers listen',
    'Fair personal treatment']].copy()
teacher_listening_fair_treatment_item_count = teacher_listening_fair_treatment_items.notna().sum(axis=1).astype('Int64')
teacher_listening_fair_treatment_score_candidate = teacher_listening_fair_treatment_items.mean(axis=1).where(teacher_listening_fair_treatment_item_count.eq(2)).astype('Float64').rename('teacher_listening_fair_treatment_score_pretransition')
teacher_listening_fair_treatment_fallback_candidate = teacher_listening_fair_treatment_items.mean(axis=1).where(teacher_listening_fair_treatment_item_count.ge(1)).astype('Float64').rename('teacher_listening_fair_treatment_score_with_fallback')
teacher_listening_fair_treatment_complete = teacher_listening_fair_treatment_items.dropna().astype(float)
assert len(teacher_listening_fair_treatment_complete) == 8962
two_item_pearson_correlation = float(pearsonr(teacher_listening_fair_treatment_complete['Teachers listen'],
    teacher_listening_fair_treatment_complete['Fair personal treatment']).statistic)
two_item_spearman_correlation = float(spearmanr(teacher_listening_fair_treatment_complete['Teachers listen'],
    teacher_listening_fair_treatment_complete['Fair personal treatment']).statistic)
two_item_variances = teacher_listening_fair_treatment_complete.var(axis=0, ddof=1)
two_item_total_variance = float(teacher_listening_fair_treatment_complete.sum(axis=1).var(ddof=1))
two_item_cronbach_alpha = 2 * (1 - two_item_variances.sum() / two_item_total_variance)
two_item_spearman_brown = 2 * two_item_pearson_correlation / (1 + two_item_pearson_correlation)
teacher_climate_alternative_reliability = pd.DataFrame([{'Representation': 'Three-item teacher-climate score',
    'Complete responses': int(len(teacher_climate_complete_items)), 'Pearson item correlation': pd.NA, 'Spearman item correlation': pd.NA, "Cronbach's alpha": round(float(teacher_climate_cronbach_alpha),
    3), 'Spearman–Brown coefficient': pd.NA}, {'Representation': 'Two-item listening and fair-treatment score',
    'Complete responses': int(len(teacher_listening_fair_treatment_complete)), 'Pearson item correlation': round(two_item_pearson_correlation,
    3), 'Spearman item correlation': round(two_item_spearman_correlation,
    3), "Cronbach's alpha": round(float(two_item_cronbach_alpha),
    3), 'Spearman–Brown coefficient': round(float(two_item_spearman_brown), 3)}])
teacher_climate_alternative_candidates = {'Equal-treatment item': teacher_equal_treatment_candidate,
    'Two-item score requiring both items': teacher_listening_fair_treatment_score_candidate, 'Two-item score allowing one-item fallback': teacher_listening_fair_treatment_fallback_candidate, 'Three-item score requiring at least two items': teacher_climate_minimum_two_score_candidate}
teacher_climate_alternative_coverage_rows = []
for representation, candidate_values in teacher_climate_alternative_candidates.items():
    teacher_climate_alternative_coverage_rows.append({'Representation': representation,
        'Non-missing': int(candidate_values.notna().sum()), 'Missing': int(candidate_values.isna().sum()), 'Missing percentage': round(candidate_values.isna().mean() * 100,
        2), 'Distinct observed values': int(candidate_values.nunique()), 'Mean': round(float(candidate_values.mean()),
        3), 'Median': round(float(candidate_values.median()),
        3), 'Minimum': float(candidate_values.min()), 'Maximum': float(candidate_values.max())})
teacher_climate_alternative_coverage_summary = pd.DataFrame(teacher_climate_alternative_coverage_rows)
teacher_climate_alternative_data = pd.DataFrame({'Equal treatment': teacher_equal_treatment_candidate,
    'Two-item listening and fair treatment': teacher_listening_fair_treatment_score_candidate, 'Two-item score with fallback': teacher_listening_fair_treatment_fallback_candidate, 'Three-item teacher climate': teacher_climate_minimum_two_score_candidate, 'Physical-school environment': school_physical_environment_candidate, **school_climate_existing_comparisons}, index=route_index)
alternative_representation_pairs = [('Equal treatment', 'Two-item listening and fair treatment'), ('Equal treatment',
    'Three-item teacher climate'), ('Two-item listening and fair treatment',
    'Three-item teacher climate'), ('Two-item listening and fair treatment',
    'Two-item score with fallback'), ('Equal treatment',
    'Physical-school environment'), ('Two-item listening and fair treatment', 'Physical-school environment')]
teacher_climate_alternative_pair_rows = []
for first_measure, second_measure in alternative_representation_pairs:
    pair_data = teacher_climate_alternative_data[[first_measure, second_measure]].dropna().astype(float)
    teacher_climate_alternative_pair_rows.append({'First measure': first_measure, 'Second measure': second_measure,
        'Complete comparisons': int(len(pair_data)), 'Spearman correlation': round(float(spearmanr(pair_data[first_measure],
        pair_data[second_measure]).statistic), 3)})
teacher_climate_alternative_pair_summary = pd.DataFrame(teacher_climate_alternative_pair_rows)
teacher_climate_alternative_overlap_rows = []
for candidate_measure in ['Equal treatment', 'Two-item listening and fair treatment']:
    for comparison_measure in ['School attitude score', 'Academic self-concept score', 'Bullying experience',
        'Truancy status', 'Physical-school environment']:
        pair_data = teacher_climate_alternative_data[[candidate_measure, comparison_measure]].dropna().astype(float)
        teacher_climate_alternative_overlap_rows.append({'Candidate measure': candidate_measure,
            'Comparison measure': comparison_measure, 'Complete comparisons': int(len(pair_data)), 'Spearman correlation': round(float(spearmanr(pair_data[candidate_measure],
            pair_data[comparison_measure]).statistic), 3)})
teacher_climate_alternative_overlap_summary = pd.DataFrame(teacher_climate_alternative_overlap_rows).sort_values(['Candidate measure',
    'Spearman correlation'], ascending=[True, False]).reset_index(drop=True)
teacher_climate_two_item_availability = pd.DataFrame({'Listening observed': teacher_listening_fair_treatment_items['Teachers listen'].notna(),
    'Fair-treatment observed': teacher_listening_fair_treatment_items['Fair personal treatment'].notna()}, index=route_index)
teacher_climate_two_item_availability_summary = teacher_climate_two_item_availability.value_counts().rename('Participants').reset_index().sort_values(['Listening observed',
    'Fair-treatment observed'], ascending=False).reset_index(drop=True)
teacher_climate_two_item_availability_summary['Percentage of full sample'] = (teacher_climate_two_item_availability_summary['Participants'] / len(route_index) * 100).round(2)
print('Alternative reliability review:')
display_limited(teacher_climate_alternative_reliability)
print('Alternative candidate coverage:')
with pd.option_context('display.max_colwidth', None):
    display_limited(teacher_climate_alternative_coverage_summary)
print('Relationships among alternative representations:')
display_limited(teacher_climate_alternative_pair_summary)
print('Alternative overlap with retained predictors:')
display_limited(teacher_climate_alternative_overlap_summary)
print('Two-item availability pattern:')
display_limited(teacher_climate_two_item_availability_summary)
print('Files written in this cell: 0')

Alternative reliability review:


,Representation,Complete responses,Pearson item correlation,Spearman item correlation,Cronbach's alpha,Spearman–Brown coefficient
0,Three-item teacher-climate score,8865,<NA>,<NA>,0.588,<NA>
1,Two-item listening and fair-treatment score,8962,0.435,0.468,0.602,0.606


Alternative candidate coverage:


,Representation,Non-missing,Missing,Missing percentage,Distinct observed values,Mean,Median,Minimum,Maximum
0,Equal-treatment item,9210,557,5.70,5,3.525,4.000,0.0,4.0
1,Two-item score requiring both items,8962,805,8.24,9,2.894,3.000,0.0,4.0
2,Two-item score allowing one-item fallback,9292,475,4.86,9,2.898,3.000,0.0,4.0
3,Three-item score requiring at least two items,9270,497,5.09,17,3.107,3.333,0.0,4.0


Relationships among alternative representations:


...

Alternative overlap with retained predictors:


,Candidate measure,Comparison measure,Complete comparisons,Spearman correlation
0,Equal treatment,School attitude score,9208,0.192
1,Equal treatment,Physical-school environment,9163,0.179
2,Equal treatment,Academic self-concept score,9021,0.099
3,Equal treatment,Bullying experience,9200,-0.085
4,Equal treatment,Truancy status,9192,-0.146


Two-item availability pattern:


,Listening observed,Fair-treatment observed,Participants,Percentage of full sample
0,True,True,8962,91.76
1,True,False,114,1.17
2,False,True,216,2.21
3,False,False,475,4.86


Files written in this cell: 0


In [394]:
# 8: School-climate predictor construction and decisions

import pandas as pd
school_climate_battery_addition_variables = {'W2YYS25YP', 'W2YYS26YP'}
existing_domain_10_screened_variables = set(domain_10_screened_candidates['Variable'])
assert school_climate_battery_addition_variables.isdisjoint(existing_domain_10_screened_variables)
school_climate_battery_additions = domain_10_register_source.loc[domain_10_register_source['Variable'].isin(school_climate_battery_addition_variables)].copy()
assert len(school_climate_battery_additions) == 2
school_climate_battery_additions['Domain 10 search track'] = 'School-environment battery expansion'
school_climate_battery_additions['All Domain 10 search matches'] = 'School-environment battery expansion'
school_climate_battery_additions['Prior review outcome'] = school_climate_battery_additions['Review outcome']
school_climate_battery_additions['Prior substantive domain'] = school_climate_battery_additions['Substantive domain']
school_climate_battery_additions['Domain 10 screening status'] = 'Core review'
school_climate_battery_additions['Domain 10 review track'] = 'School climate and physical environment'
school_climate_battery_additions['Domain 10 screening reason'] = 'The item belongs to the same teacher-treatment battery as W2YYS24YP and is required for a coherent review of teacher listening and fair treatment'
school_climate_battery_additions = school_climate_battery_additions.reindex(columns=domain_10_screened_candidates.columns)
domain_10_screened_candidates = pd.concat([domain_10_screened_candidates, school_climate_battery_additions],
    ignore_index=True).drop_duplicates(subset=['Source file', 'Variable'],
    keep='last').sort_values(['Domain 10 screening status', 'Domain 10 review track', 'Source order',
    'Variable position']).reset_index(drop=True)
assert len(domain_10_screened_candidates) == 161
assert not domain_10_screened_candidates[['Source file', 'Variable']].duplicated().any()
domain_10_core_review_inventory = domain_10_screened_candidates.loc[domain_10_screened_candidates['Domain 10 screening status'].eq('Core review'),
    ['Source order', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable', 'Variable label',
    'Timing status', 'Prior review outcome', 'Domain 10 review track', 'Domain 10 screening reason']].copy().sort_values(['Domain 10 review track',
    'Source order', 'Variable position']).reset_index(drop=True)
assert len(domain_10_core_review_inventory) == 38
school_physical_environment_perception_pretransition = school_physical_environment_candidate.copy().astype('Int64').rename('school_physical_environment_perception_pretransition')
teacher_listening_fair_treatment_score_pretransition = teacher_listening_fair_treatment_score_candidate.copy().astype('Float64').rename('teacher_listening_fair_treatment_score_pretransition')
assert set(school_physical_environment_perception_pretransition.dropna().astype(int).unique()).issubset({0, 1, 2, 3})
assert set(teacher_listening_fair_treatment_score_pretransition.dropna().astype(float).unique()).issubset({0.0, 0.5,
    1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0})
school_climate_predictors = pd.concat([school_physical_environment_perception_pretransition,
    teacher_listening_fair_treatment_score_pretransition], axis=1).reindex(route_index)
school_climate_predictors.index.name = 'NSID'
school_climate_predictors = school_climate_predictors.reset_index()
assert len(school_climate_predictors) == 9767
assert school_climate_predictors['NSID'].is_unique
school_climate_predictor_names = ['school_physical_environment_perception_pretransition',
    'teacher_listening_fair_treatment_score_pretransition']
school_climate_predictor_coverage_rows = []
for predictor_name in school_climate_predictor_names:
    predictor_values = school_climate_predictors[predictor_name]
    school_climate_predictor_coverage_rows.append({'Predictor': predictor_name,
        'Non-missing': int(predictor_values.notna().sum()), 'Missing': int(predictor_values.isna().sum()), 'Missing percentage': round(predictor_values.isna().mean() * 100,
        2), 'Distinct observed values': int(predictor_values.nunique()), 'Minimum': float(predictor_values.min()), 'Maximum': float(predictor_values.max())})
school_climate_predictor_coverage_summary = pd.DataFrame(school_climate_predictor_coverage_rows)
school_climate_available_predictor_count = school_climate_predictors[school_climate_predictor_names].notna().sum(axis=1)
school_climate_joint_coverage_summary = school_climate_available_predictor_count.value_counts().sort_index().rename('Participants').rename_axis('School-climate predictors available').reset_index()
school_climate_joint_coverage_summary['Percentage of full sample'] = (school_climate_joint_coverage_summary['Participants'] / len(school_climate_predictors) * 100).round(2)
school_climate_variable_decisions = school_climate_review_inventory.copy()
school_climate_decision_lookup = {'W1yys13YP': {'Decision': 'Construction input',
    'Role': 'Wave 1 fallback input to the retained school physical-environment predictor', 'Reason': 'The repeated clean-and-tidy item provides a fallback where the later Wave 2 response is unavailable'}, 'W2YYS13YP': {'Decision': 'Construction input',
    'Role': 'Primary input to the retained school physical-environment predictor', 'Reason': "The latest pre-transition response is preferred because the construct concerns the participant's current perception of the school environment"}, 'W2YYS24YP': {'Decision': 'Review support',
    'Role': 'General equal-treatment item used in the teacher-climate representation review', 'Reason': 'The item was only moderately associated with the listening and fair-personal-treatment items and reduced the coherence of the three-item representation'}, 'W2YYS25YP': {'Decision': 'Construction input',
    'Role': 'Input to the retained teacher listening and fair-treatment score', 'Reason': 'The item measures whether teachers listen to pupils and is combined with the related personal-treatment item'}, 'W2YYS26YP': {'Decision': 'Construction input',
    'Role': 'Input to the retained teacher listening and fair-treatment score', 'Reason': 'The item measures perceived unfair personal treatment and is combined with the related teacher-listening item'}}
school_climate_variable_decisions['Domain 10 decision'] = school_climate_variable_decisions['Variable'].map({variable: decision_values['Decision'] for variable,
    decision_values in school_climate_decision_lookup.items()})
school_climate_variable_decisions['Domain 10 role'] = school_climate_variable_decisions['Variable'].map({variable: decision_values['Role'] for variable,
    decision_values in school_climate_decision_lookup.items()})
school_climate_variable_decisions['Domain 10 reason'] = school_climate_variable_decisions['Variable'].map({variable: decision_values['Reason'] for variable,
    decision_values in school_climate_decision_lookup.items()})
assert school_climate_variable_decisions[['Domain 10 decision', 'Domain 10 role',
    'Domain 10 reason']].notna().all().all()
school_climate_decision_counts = school_climate_variable_decisions['Domain 10 decision'].value_counts()
assert int(school_climate_decision_counts.get('Construction input', 0)) == 4
assert int(school_climate_decision_counts.get('Review support', 0)) == 1
school_climate_representation_decisions = pd.DataFrame([{'Candidate representation': 'Latest clean-and-tidy response with Wave 1 fallback',
    'Decision': 'Retain', 'Final predictor': 'school_physical_environment_perception_pretransition', 'Reason': 'The measure has broad coverage, moderate cross-wave stability and limited overlap with existing predictors'}, {'Candidate representation': 'Two-item listening and fair-treatment score requiring both items',
    'Decision': 'Retain', 'Final predictor': 'teacher_listening_fair_treatment_score_pretransition', 'Reason': 'The two items represent related interpersonal teacher-treatment experiences. Reliability is modest, so the result is treated as a limited composite score'}, {'Candidate representation': 'Equal-treatment item as a separate predictor',
    'Decision': 'Do not retain', 'Final predictor': pd.NA, 'Reason': 'The item is only moderately related to the two interpersonal-treatment items and would add another single-item predictor'}, {'Candidate representation': 'Three-item teacher-climate score',
    'Decision': 'Do not retain', 'Final predictor': pd.NA, 'Reason': 'The three-item representation had weak internal consistency and combined distinguishable constructs'}, {'Candidate representation': 'Two-item score allowing one-item fallback',
    'Decision': 'Do not retain', 'Final predictor': pd.NA, 'Reason': 'Scores based on one and two observed items would not have a consistent measurement basis'}])
domain_10_screening_after_battery_expansion = pd.DataFrame([{'Screened Domain 10 variables': int(len(domain_10_screened_candidates)),
    'Core review variables': int(len(domain_10_core_review_inventory)), 'School-climate variables reviewed': int(len(school_climate_variable_decisions)), 'School-climate predictors retained': int(len(school_climate_predictor_names))}])
print('Domain 10 inventory after battery expansion:')
display_limited(domain_10_screening_after_battery_expansion)
print('School-climate predictor coverage:')
display_limited(school_climate_predictor_coverage_summary)
print('Joint school-climate predictor coverage:')
display_limited(school_climate_joint_coverage_summary)
print('School-climate source-variable decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(school_climate_variable_decisions[['Wave', 'Source file', 'Variable', 'Variable label',
        'Domain 10 decision', 'Domain 10 role', 'Domain 10 reason']])
print('School-climate representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(school_climate_representation_decisions)
print('Files written in this cell: 0')

Domain 10 inventory after battery expansion:


,Screened Domain 10 variables,Core review variables,School-climate variables reviewed,School-climate predictors retained
0,161,38,5,2


School-climate predictor coverage:


,Predictor,Non-missing,Missing,Missing percentage,Distinct observed values,Minimum,Maximum
0,school_physical_environment_perception_pretran...,9431,336,3.44,4,0.0,3.0
1,teacher_listening_fair_treatment_score_pretran...,8962,805,8.24,9,0.0,4.0


Joint school-climate predictor coverage:


...

School-climate source-variable decisions:


,Wave,Source file,Variable,Variable label,Domain 10 decision,Domain 10 role,Domain 10 reason
0,Wave 1,wave_one_lsype_young_person_2020,W1yys13YP,YP: Feelings about school: My school is clean and tidy,Construction input,Wave 1 fallback input to the retained school physical-environment predictor,The repeated clean-and-tidy item provides a fallback where the later Wave 2 response is unavailable
1,Wave 2,wave_two_lsype_young_person_2020,W2YYS13YP,YP: Feelings about school: My school is clean and tidy,Construction input,Primary input to the retained school physical-environment predictor,The latest pre-transition response is preferred because the construct concerns the participant's current perception of the school environment
2,Wave 2,wave_two_lsype_young_person_2020,W2YYS24YP,YP: How many teachers this applies to: My teachers treat everyone the same regar,Review support,General equal-treatment item used in the teacher-climate representation review,The item was only moderately associated with the listening and fair-personal-treatment items and reduced the coherence of the three-item representation
3,Wave 2,wave_two_lsype_young_person_2020,W2YYS25YP,YP: How many teachers this applies to: My teachers don't really listen to what I,Construction input,Input to the retained teacher listening and fair-treatment score,The item measures whether teachers listen to pupils and is combined with the related personal-treatment item
4,Wave 2,wave_two_lsype_young_person_2020,W2YYS26YP,YP: How many teachers this applies to: I get treated unfairly by my teachers,Construction input,Input to the retained teacher listening and fair-treatment score,The item measures perceived unfair personal treatment and is combined with the related teacher-listening item


School-climate representation decisions:


,Candidate representation,Decision,Final predictor,Reason
0,Latest clean-and-tidy response with Wave 1 fallback,Retain,school_physical_environment_perception_pretransition,"The measure has broad coverage, moderate cross-wave stability and limited overlap with existing predictors"
1,Two-item listening and fair-treatment score requiring both items,Retain,teacher_listening_fair_treatment_score_pretransition,"The two items represent related interpersonal teacher-treatment experiences. Reliability is modest, so the result is treated as a limited composite score"
2,Equal-treatment item as a separate predictor,Do not retain,NaN,The item is only moderately related to the two interpersonal-treatment items and would add another single-item predictor
3,Three-item teacher-climate score,Do not retain,NaN,The three-item representation had weak internal consistency and combined distinguishable constructs
4,Two-item score allowing one-item fallback,Do not retain,NaN,Scores based on one and two observed items would not have a consistent measurement basis


Files written in this cell: 0


In [395]:
# 9: Regional and urban–rural context review

import pandas as pd
regional_context_variable_names = {'urbind', 'gor'}
regional_context_inventory = domain_10_core_review_inventory.loc[domain_10_core_review_inventory['Domain 10 review track'].eq('Regional and urban–rural context')].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(regional_context_inventory) == 4
assert set(regional_context_inventory['Variable']) == regional_context_variable_names
assert set(regional_context_inventory['Wave']) == {'Wave 2', 'Wave 3'}
wave_3_timing_object = globals().get('wave_3_pretransition_interview_mask')
assert isinstance(wave_3_timing_object, pd.Series)
wave_3_permitted_mask = wave_3_timing_object.copy()
wave_3_permitted_mask.index = pd.Index([standardise_nsid(pd.Series([index_value])).iloc[0] for index_value in wave_3_permitted_mask.index],
    name='NSID')
wave_3_permitted_mask = wave_3_permitted_mask.reindex(route_index).fillna(False).astype(bool)
regional_context_raw = pd.DataFrame(index=route_index)
regional_context_labelled = pd.DataFrame(index=route_index)
for wave_label in ['Wave 2', 'Wave 3']:
    wave_inventory = regional_context_inventory.loc[regional_context_inventory['Wave'].eq(wave_label)]
    assert len(wave_inventory) == 2
    source_files = wave_inventory['Source file'].drop_duplicates().tolist()
    assert len(source_files) == 1
    source_file = source_files[0]
    source_path = source_file_lookup[source_file]
    source_variables = wave_inventory['Variable'].tolist()
    raw_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    for source_data in [raw_source, labelled_source]:
        source_data['NSID'] = standardise_nsid(source_data['NSID'])
        assert source_data['NSID'].is_unique
    raw_source = raw_source.set_index('NSID').reindex(route_index)
    labelled_source = labelled_source.set_index('NSID').reindex(route_index)
    wave_number = '2' if wave_label == 'Wave 2' else '3'
    for variable in source_variables:
        output_column = f'W{wave_number}_{variable}'
        regional_context_raw[output_column] = pd.to_numeric(raw_source[variable], errors='coerce')
        regional_context_labelled[output_column] = labelled_source[variable].astype('string')
expected_regional_columns = {'W2_urbind', 'W2_gor', 'W3_urbind', 'W3_gor'}
assert set(regional_context_raw.columns) == expected_regional_columns
regional_context_wave_3_unrestricted = regional_context_raw[['W3_urbind', 'W3_gor']].copy()
regional_context_wave_3_labels_unrestricted = regional_context_labelled[['W3_urbind', 'W3_gor']].copy()
regional_context_raw.loc[~wave_3_permitted_mask, ['W3_urbind', 'W3_gor']] = pd.NA
regional_context_labelled.loc[~wave_3_permitted_mask, ['W3_urbind', 'W3_gor']] = pd.NA
regional_context_coverage_rows = []
for wave_number in ['2', '3']:
    for variable in ['urbind', 'gor']:
        column_name = f'W{wave_number}_{variable}'
        values = regional_context_raw[column_name]
        observed_mask = values.ge(0).fillna(False).astype(bool)
        special_code_mask = values.lt(0).fillna(False).astype(bool)
        if wave_number == '3':
            unrestricted_values = regional_context_wave_3_unrestricted[column_name]
            removed_by_timing = int((unrestricted_values.notna() & ~wave_3_permitted_mask).sum())
        else:
            removed_by_timing = 0
        regional_context_coverage_rows.append({'Wave': f'Wave {wave_number}', 'Variable': variable,
            'Observed responses': int(observed_mask.sum()), 'Observed percentage': round(observed_mask.mean() * 100,
            2), 'Special-code responses': int(special_code_mask.sum()), 'Unavailable after timing restriction': int(values.isna().sum()), 'Wave 3 values removed by timing': removed_by_timing, 'Observed categories': int(values.where(observed_mask).nunique())})
regional_context_coverage_summary = pd.DataFrame(regional_context_coverage_rows)
regional_context_code_rows = []
for column_name in ['W2_urbind', 'W3_urbind', 'W2_gor', 'W3_gor']:
    raw_values = regional_context_raw[column_name]
    labelled_values = regional_context_labelled[column_name]
    for raw_code, participants in raw_values.value_counts(dropna=False).items():
        if pd.isna(raw_code):
            value_label = 'Unavailable after source and timing restrictions'
            response_type = 'Unavailable'
            sort_value = 999999
        else:
            matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            response_type = 'Observed response' if raw_code >= 0 else 'Special code'
            sort_value = float(raw_code)
        regional_context_code_rows.append({'Measure': column_name, 'Raw code': raw_code, 'Value label': value_label,
            'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
regional_context_code_distribution = pd.DataFrame(regional_context_code_rows).sort_values(['Measure',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
regional_context_consistency_rows = []
regional_context_crosstabs = {}
for variable in ['urbind', 'gor']:
    wave_2_column = f'W2_{variable}'
    wave_3_column = f'W3_{variable}'
    comparison_data = pd.DataFrame({'Wave 2 raw': regional_context_raw[wave_2_column],
        'Wave 3 raw': regional_context_raw[wave_3_column], 'Wave 2 label': regional_context_labelled[wave_2_column], 'Wave 3 label': regional_context_labelled[wave_3_column]}, index=route_index)
    complete_comparison = comparison_data.loc[comparison_data['Wave 2 raw'].ge(0) & comparison_data['Wave 3 raw'].ge(0)].copy()
    assert len(complete_comparison) > 0
    exact_agreement_mask = complete_comparison['Wave 2 raw'].eq(complete_comparison['Wave 3 raw'])
    regional_context_consistency_rows.append({'Variable': variable,
        'Complete comparisons': int(len(complete_comparison)), 'Exact agreement': int(exact_agreement_mask.sum()), 'Changed category': int((~exact_agreement_mask).sum()), 'Exact agreement percentage': round(exact_agreement_mask.mean() * 100,
        2), "Cramer's V": round(categorical_cramers_v(complete_comparison['Wave 2 raw'],
        complete_comparison['Wave 3 raw']), 3)})
    regional_context_crosstabs[variable] = pd.crosstab(complete_comparison['Wave 2 label'],
        complete_comparison['Wave 3 label'], margins=True, dropna=False)
regional_context_consistency_summary = pd.DataFrame(regional_context_consistency_rows)
urban_rural_context_candidate = regional_context_raw['W3_urbind'].where(regional_context_raw['W3_urbind'].ge(0)).combine_first(regional_context_raw['W2_urbind'].where(regional_context_raw['W2_urbind'].ge(0))).rename('urban_rural_context_candidate')
government_office_region_candidate = regional_context_raw['W3_gor'].where(regional_context_raw['W3_gor'].ge(0)).combine_first(regional_context_raw['W2_gor'].where(regional_context_raw['W2_gor'].ge(0))).rename('government_office_region_candidate')
regional_context_candidate_source = pd.DataFrame(index=route_index)
for variable, candidate_name in [('urbind', 'Urban–rural context'), ('gor', 'Government Office Region')]:
    wave_2_values = regional_context_raw[f'W2_{variable}']
    wave_3_values = regional_context_raw[f'W3_{variable}']
    source_values = pd.Series(pd.NA, index=route_index, dtype='string')
    source_values.loc[wave_3_values.ge(0)] = 'Wave 3'
    source_values.loc[~wave_3_values.ge(0) & wave_2_values.ge(0)] = 'Wave 2 fallback'
    regional_context_candidate_source[candidate_name] = source_values
regional_context_candidate_summary = pd.DataFrame([{'Candidate': 'Urban–rural context',
    'Non-missing': int(urban_rural_context_candidate.notna().sum()), 'Missing': int(urban_rural_context_candidate.isna().sum()), 'Missing percentage': round(urban_rural_context_candidate.isna().mean() * 100,
    2), 'Distinct categories': int(urban_rural_context_candidate.nunique())}, {'Candidate': 'Government Office Region',
    'Non-missing': int(government_office_region_candidate.notna().sum()), 'Missing': int(government_office_region_candidate.isna().sum()), 'Missing percentage': round(government_office_region_candidate.isna().mean() * 100,
    2), 'Distinct categories': int(government_office_region_candidate.nunique())}])
regional_context_candidate_source_summary = regional_context_candidate_source.melt(var_name='Candidate',
    value_name='Source').groupby(['Candidate', 'Source'], dropna=False).size().rename('Participants').reset_index()
regional_context_candidate_source_summary['Percentage of full sample'] = (regional_context_candidate_source_summary['Participants'] / len(route_index) * 100).round(2)
regional_context_timing_summary = pd.DataFrame([{'Full analysis sample': int(len(route_index)),
    'Wave 3 interviews within permitted timing': int(wave_3_permitted_mask.sum()), 'Wave 3 interviews outside permitted timing': int((~wave_3_permitted_mask).sum())}])
print('Regional-context inventory:')
with pd.option_context('display.max_colwidth', None):
    display_limited(regional_context_inventory)
print('Wave 3 timing restriction:')
display_limited(regional_context_timing_summary)
print('Response coverage:')
display_limited(regional_context_coverage_summary)
print('Response-code distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(regional_context_code_distribution)
print('Cross-wave consistency:')
display_limited(regional_context_consistency_summary)
print('Urban–rural cross-tabulation:')
display_limited(regional_context_crosstabs['urbind'])
print('Government Office Region cross-tabulation:')
display_limited(regional_context_crosstabs['gor'])
print('Latest-wave candidate coverage:')
display_limited(regional_context_candidate_summary)
print('Latest-wave candidate source use:')
display_limited(regional_context_candidate_source_summary)
print('Files written in this cell: 0')

Regional-context inventory:


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Prior review outcome,Domain 10 review track,Domain 10 screening reason
0,5,Wave 2,Family background,wave_two_lsype_family_background_2020,1027,urbind,Urban/Rural Indicator (England),Pre-transition source,Pending review,Regional and urban–rural context,The variable provides a direct geographical context measure that can be harmonised across permitted waves
1,5,Wave 2,Family background,wave_two_lsype_family_background_2020,1028,gor,Government Office Region,Pre-transition source,Pending review,Regional and urban–rural context,The variable provides a direct geographical context measure that can be harmonised across permitted waves
2,9,Wave 3,Family background,wave_three_lsype_family_background_2020,158,urbind,Urban/Rural Indicator (England),Near-transition source,Pending review,Regional and urban–rural context,The variable provides a direct geographical context measure that can be harmonised across permitted waves
3,9,Wave 3,Family background,wave_three_lsype_family_background_2020,159,gor,Government Office Region,Near-transition source,Pending review,Regional and urban–rural context,The variable provides a direct geographical context measure that can be harmonised across permitted waves


Wave 3 timing restriction:


,Full analysis sample,Wave 3 interviews within permitted timing,Wave 3 interviews outside permitted timing
0,9767,9495,272


Response coverage:


...

Latest-wave candidate coverage:


,Candidate,Non-missing,Missing,Missing percentage,Distinct categories
0,Urban–rural context,9520,247,2.53,8
1,Government Office Region,9520,247,2.53,9


Latest-wave candidate source use:


,Candidate,Source,Participants,Percentage of full sample
0,Government Office Region,Wave 2 fallback,30,0.31
1,Government Office Region,Wave 3,9490,97.16
2,Government Office Region,<NA>,247,2.53
3,Urban–rural context,Wave 2 fallback,29,0.30
4,Urban–rural context,Wave 3,9491,97.17


Files written in this cell: 0


In [396]:
# 10: Urban–rural representation and regional overlap review

import pandas as pd
urban_rural_eight_category_candidate = pd.to_numeric(urban_rural_context_candidate,
    errors='coerce').where(pd.to_numeric(urban_rural_context_candidate, errors='coerce').isin(range(1,
    9))).astype('Int64').rename('urban_rural_eight_category_candidate')
government_office_region_valid_candidate = pd.to_numeric(government_office_region_candidate,
    errors='coerce').where(pd.to_numeric(government_office_region_candidate, errors='coerce').isin(range(1,
    10))).astype('Int64').rename('government_office_region_candidate')
assert int(urban_rural_eight_category_candidate.notna().sum()) == 9520
assert int(government_office_region_valid_candidate.notna().sum()) == 9520
urban_rural_four_category_mapping = {1: 0, 5: 0, 2: 1, 6: 1, 3: 2, 7: 2, 4: 3, 8: 3}
urban_rural_four_category_labels = {0: 'Urban area with population of at least 10,000', 1: 'Town and fringe',
    2: 'Village', 3: 'Hamlet and isolated dwelling'}
urban_rural_four_category_candidate = urban_rural_eight_category_candidate.map(urban_rural_four_category_mapping).astype('Int64').rename('urban_rural_settlement_type_candidate')
urban_rural_binary_mapping = {1: 0, 5: 0, 2: 1, 6: 1, 3: 1, 7: 1, 4: 1, 8: 1}
urban_rural_binary_labels = {0: 'Urban area with population of at least 10,000',
    1: 'Town, village, hamlet or isolated dwelling'}
urban_rural_binary_candidate = urban_rural_eight_category_candidate.map(urban_rural_binary_mapping).astype('Int64').rename('urban_rural_binary_candidate')
assert urban_rural_four_category_candidate.notna().equals(urban_rural_eight_category_candidate.notna())
assert urban_rural_binary_candidate.notna().equals(urban_rural_eight_category_candidate.notna())
urban_rural_representation_specs = {'Original eight-category indicator': urban_rural_eight_category_candidate,
    'Four-category settlement type': urban_rural_four_category_candidate, 'Binary urban–non-urban indicator': urban_rural_binary_candidate}
urban_rural_distribution_rows = []
for representation, values in urban_rural_representation_specs.items():
    value_counts = values.value_counts(dropna=False).sort_index()
    for category_value, participants in value_counts.items():
        if pd.isna(category_value):
            category_label = 'Missing'
        elif representation == 'Four-category settlement type':
            category_label = urban_rural_four_category_labels[int(category_value)]
        elif representation == 'Binary urban–non-urban indicator':
            category_label = urban_rural_binary_labels[int(category_value)]
        else:
            category_label_lookup = {1: 'Urban >=10k — sparse', 2: 'Town and fringe — sparse', 3: 'Village — sparse',
                4: 'Hamlet and isolated dwelling — sparse', 5: 'Urban >=10k — less sparse', 6: 'Town and fringe — less sparse', 7: 'Village — less sparse', 8: 'Hamlet and isolated dwelling — less sparse'}
            category_label = category_label_lookup[int(category_value)]
        urban_rural_distribution_rows.append({'Representation': representation, 'Category code': category_value,
            'Category label': category_label, 'Participants': int(participants), 'Percentage of observed responses': round(participants / values.notna().sum() * 100,
            2) if not pd.isna(category_value) else pd.NA})
urban_rural_representation_distribution = pd.DataFrame(urban_rural_distribution_rows)
urban_rural_sparsity_rows = []
for representation, values in urban_rural_representation_specs.items():
    observed_counts = values.dropna().value_counts()
    urban_rural_sparsity_rows.append({'Representation': representation,
        'Observed participants': int(values.notna().sum()), 'Missing participants': int(values.isna().sum()), 'Categories': int(values.nunique()), 'Smallest category': int(observed_counts.min()), 'Largest category': int(observed_counts.max()), 'Categories below 50 participants': int(observed_counts.lt(50).sum()), 'Categories below 100 participants': int(observed_counts.lt(100).sum())})
urban_rural_sparsity_summary = pd.DataFrame(urban_rural_sparsity_rows)
regional_overlap_rows = []
for representation, urban_rural_values in urban_rural_representation_specs.items():
    pair_data = pd.DataFrame({'Urban–rural measure': urban_rural_values,
        'Government Office Region': government_office_region_valid_candidate}, index=route_index).dropna()
    regional_overlap_rows.append({'Urban–rural representation': representation,
        'Complete comparisons': int(len(pair_data)), "Cramer's V with Government Office Region": round(categorical_cramers_v(pair_data['Urban–rural measure'],
        pair_data['Government Office Region']), 3)})
regional_overlap_summary = pd.DataFrame(regional_overlap_rows)
wave_2_urban_rural_four_category = regional_context_raw['W2_urbind'].where(regional_context_raw['W2_urbind'].isin(range(1,
    9))).map(urban_rural_four_category_mapping).astype('Int64')
wave_3_urban_rural_four_category = regional_context_raw['W3_urbind'].where(regional_context_raw['W3_urbind'].isin(range(1,
    9))).map(urban_rural_four_category_mapping).astype('Int64')
four_category_wave_comparison = pd.DataFrame({'Wave 2': wave_2_urban_rural_four_category,
    'Wave 3': wave_3_urban_rural_four_category}, index=route_index).dropna()
four_category_exact_agreement = four_category_wave_comparison['Wave 2'].eq(four_category_wave_comparison['Wave 3'])
urban_rural_four_category_wave_consistency = pd.DataFrame([{'Complete comparisons': int(len(four_category_wave_comparison)),
    'Exact agreement': int(four_category_exact_agreement.sum()), 'Changed settlement type': int((~four_category_exact_agreement).sum()), 'Exact agreement percentage': round(four_category_exact_agreement.mean() * 100,
    2), "Cramer's V": round(categorical_cramers_v(four_category_wave_comparison['Wave 2'],
    four_category_wave_comparison['Wave 3']), 3)}])
regional_context_candidate_pair = pd.DataFrame({'Four-category settlement type': urban_rural_four_category_candidate,
    'Government Office Region': government_office_region_valid_candidate}, index=route_index).dropna()
regional_context_joint_coverage = pd.DataFrame([{'Full analysis sample': int(len(route_index)),
    'Both contextual candidates available': int(len(regional_context_candidate_pair)), 'Both available percentage': round(len(regional_context_candidate_pair) / len(route_index) * 100,
    2), 'At least one contextual candidate available': int(pd.DataFrame({'Settlement type': urban_rural_four_category_candidate,
    'Region': government_office_region_valid_candidate}, index=route_index).notna().any(axis=1).sum()), 'Neither contextual candidate available': int(pd.DataFrame({'Settlement type': urban_rural_four_category_candidate,
    'Region': government_office_region_valid_candidate}, index=route_index).isna().all(axis=1).sum())}])
print('Urban–rural representation distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(urban_rural_representation_distribution)
print('Category-sparsity comparison:')
display_limited(urban_rural_sparsity_summary)
print('Overlap with Government Office Region:')
display_limited(regional_overlap_summary)
print('Four-category cross-wave consistency:')
display_limited(urban_rural_four_category_wave_consistency)
print('Regional-context joint coverage:')
display_limited(regional_context_joint_coverage)
print('Files written in this cell: 0')

Urban–rural representation distributions:


,Representation,Category code,Category label,Participants,Percentage of observed responses
0,Original eight-category indicator,1,Urban >=10k — sparse,7,0.07
1,Original eight-category indicator,2,Town and fringe — sparse,46,0.48
2,Original eight-category indicator,3,Village — sparse,34,0.36
3,Original eight-category indicator,4,Hamlet and isolated dwelling — sparse,42,0.44
4,Original eight-category indicator,5,Urban >=10k — less sparse,7898,82.96


Category-sparsity comparison:


,Representation,Observed participants,Missing participants,Categories,Smallest category,Largest category,Categories below 50 participants,Categories below 100 participants
0,Original eight-category indicator,9520,247,8,7,7898,4,4
1,Four-category settlement type,9520,247,4,273,7905,0,0
2,Binary urban–non-urban indicator,9520,247,2,1615,7905,0,0


Overlap with Government Office Region:


...

Four-category cross-wave consistency:


,Complete comparisons,Exact agreement,Changed settlement type,Exact agreement percentage,Cramer's V
0,9487,9428,59,99.38,0.972


Regional-context joint coverage:


,Full analysis sample,Both contextual candidates available,Both available percentage,At least one contextual candidate available,Neither contextual candidate available
0,9767,9520,97.47,9520,247


Files written in this cell: 0


In [397]:
# 11: Regional-context predictor construction and decisions

import pandas as pd
urban_rural_settlement_type_pretransition = urban_rural_four_category_candidate.copy().astype('Int64').rename('urban_rural_settlement_type_pretransition')
government_office_region_pretransition = government_office_region_valid_candidate.copy().astype('Int64').rename('government_office_region_pretransition')
assert set(urban_rural_settlement_type_pretransition.dropna().astype(int).unique()) == {0, 1, 2, 3}
assert set(government_office_region_pretransition.dropna().astype(int).unique()) == set(range(1, 10))
regional_context_predictors = pd.concat([urban_rural_settlement_type_pretransition,
    government_office_region_pretransition], axis=1).reindex(route_index)
regional_context_predictors.index.name = 'NSID'
regional_context_predictors = regional_context_predictors.reset_index()
assert len(regional_context_predictors) == 9767
assert regional_context_predictors['NSID'].is_unique
regional_context_predictor_names = ['urban_rural_settlement_type_pretransition',
    'government_office_region_pretransition']
regional_context_predictor_coverage_rows = []
for predictor_name in regional_context_predictor_names:
    predictor_values = regional_context_predictors[predictor_name]
    observed_counts = predictor_values.dropna().value_counts()
    regional_context_predictor_coverage_rows.append({'Predictor': predictor_name,
        'Non-missing': int(predictor_values.notna().sum()), 'Missing': int(predictor_values.isna().sum()), 'Missing percentage': round(predictor_values.isna().mean() * 100,
        2), 'Categories': int(predictor_values.nunique()), 'Smallest observed category': int(observed_counts.min()), 'Largest observed category': int(observed_counts.max())})
regional_context_predictor_coverage_summary = pd.DataFrame(regional_context_predictor_coverage_rows)
regional_context_available_predictor_count = regional_context_predictors[regional_context_predictor_names].notna().sum(axis=1)
regional_context_joint_coverage_summary = regional_context_available_predictor_count.value_counts().sort_index().rename('Participants').rename_axis('Regional-context predictors available').reset_index()
regional_context_joint_coverage_summary['Percentage of full sample'] = (regional_context_joint_coverage_summary['Participants'] / len(regional_context_predictors) * 100).round(2)
regional_context_category_definitions = pd.DataFrame([{'Predictor': 'urban_rural_settlement_type_pretransition',
    'Category code': category_code, 'Category label': category_label} for category_code, category_label in urban_rural_four_category_labels.items()] + [{'Predictor': 'government_office_region_pretransition',
    'Category code': category_code, 'Category label': category_label} for category_code, category_label in {1: 'North East',
    2: 'North West', 3: 'Yorkshire and The Humber', 4: 'East Midlands', 5: 'West Midlands', 6: 'East of England', 7: 'London', 8: 'South East', 9: 'South West'}.items()])
regional_context_variable_decisions = regional_context_inventory.copy()
regional_context_variable_decisions['Domain 10 decision'] = 'Construction input'
regional_context_variable_decisions['Domain 10 role'] = pd.NA
regional_context_variable_decisions['Domain 10 reason'] = pd.NA
wave_2_urban_mask = regional_context_variable_decisions['Wave'].eq('Wave 2') & regional_context_variable_decisions['Variable'].eq('urbind')
wave_3_urban_mask = regional_context_variable_decisions['Wave'].eq('Wave 3') & regional_context_variable_decisions['Variable'].eq('urbind')
wave_2_region_mask = regional_context_variable_decisions['Wave'].eq('Wave 2') & regional_context_variable_decisions['Variable'].eq('gor')
wave_3_region_mask = regional_context_variable_decisions['Wave'].eq('Wave 3') & regional_context_variable_decisions['Variable'].eq('gor')
assert int(wave_2_urban_mask.sum()) == 1
assert int(wave_3_urban_mask.sum()) == 1
assert int(wave_2_region_mask.sum()) == 1
assert int(wave_3_region_mask.sum()) == 1
regional_context_variable_decisions.loc[wave_2_urban_mask,
    'Domain 10 role'] = 'Wave 2 fallback input to the retained four-category settlement-type predictor'
regional_context_variable_decisions.loc[wave_2_urban_mask,
    'Domain 10 reason'] = 'Wave 2 provides a fallback where the permitted Wave 3 urban–rural indicator is unavailable'
regional_context_variable_decisions.loc[wave_3_urban_mask,
    'Domain 10 role'] = 'Primary input to the retained four-category settlement-type predictor'
regional_context_variable_decisions.loc[wave_3_urban_mask,
    'Domain 10 reason'] = 'The latest permitted pre-transition value is used, with sparse and less-sparse categories collapsed within the same settlement type'
regional_context_variable_decisions.loc[wave_2_region_mask,
    'Domain 10 role'] = 'Wave 2 fallback input to the retained Government Office Region predictor'
regional_context_variable_decisions.loc[wave_2_region_mask,
    'Domain 10 reason'] = 'Wave 2 provides a fallback where the permitted Wave 3 regional value is unavailable'
regional_context_variable_decisions.loc[wave_3_region_mask,
    'Domain 10 role'] = 'Primary input to the retained Government Office Region predictor'
regional_context_variable_decisions.loc[wave_3_region_mask,
    'Domain 10 reason'] = 'The latest permitted pre-transition regional value is used because Wave 2 and Wave 3 classifications are almost identical'
assert regional_context_variable_decisions[['Domain 10 decision', 'Domain 10 role',
    'Domain 10 reason']].notna().all().all()
regional_context_representation_decisions = pd.DataFrame([{'Candidate representation': 'Original eight-category urban–rural indicator',
    'Decision': 'Do not retain', 'Final predictor': pd.NA, 'Reason': 'Four categories contained fewer than 50 participants, which would create sparse dummy variables'}, {'Candidate representation': 'Four-category settlement type',
    'Decision': 'Retain', 'Final predictor': 'urban_rural_settlement_type_pretransition', 'Reason': 'The representation preserves settlement-type differences, removes sparsity distinctions and has no small categories'}, {'Candidate representation': 'Binary urban–non-urban indicator',
    'Decision': 'Do not retain', 'Final predictor': pd.NA, 'Reason': 'The binary measure would remove differences among town, village and hamlet settings without a coverage or sparsity advantage over the four-category measure'}, {'Candidate representation': 'Nine-category Government Office Region',
    'Decision': 'Retain', 'Final predictor': 'government_office_region_pretransition', 'Reason': 'All regional categories have adequate sample sizes, and overlap with the settlement-type predictor is modest'}])
regional_context_final_summary = pd.DataFrame([{'Source variables reviewed': int(len(regional_context_variable_decisions)),
    'Regional-context predictors retained': int(len(regional_context_predictor_names)), 'Both predictors available': int(regional_context_available_predictor_count.eq(2).sum()), 'Neither predictor available': int(regional_context_available_predictor_count.eq(0).sum()), "Cramer's V between retained predictors": round(categorical_cramers_v(urban_rural_settlement_type_pretransition,
    government_office_region_pretransition), 3)}])
print('Regional-context final summary:')
display_limited(regional_context_final_summary)
print('Retained predictor coverage:')
display_limited(regional_context_predictor_coverage_summary)
print('Joint predictor coverage:')
display_limited(regional_context_joint_coverage_summary)
print('Category definitions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(regional_context_category_definitions)
print('Regional-context source-variable decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(regional_context_variable_decisions[['Wave', 'Source file', 'Variable', 'Variable label',
        'Domain 10 decision', 'Domain 10 role', 'Domain 10 reason']])
print('Regional-context representation decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(regional_context_representation_decisions)
print('Files written in this cell: 0')

Regional-context final summary:


,Source variables reviewed,Regional-context predictors retained,Both predictors available,Neither predictor available,Cramer's V between retained predictors
0,4,2,9520,247,0.173


Retained predictor coverage:


,Predictor,Non-missing,Missing,Missing percentage,Categories,Smallest observed category,Largest observed category
0,urban_rural_settlement_type_pretransition,9520,247,2.53,4,273,7905
1,government_office_region_pretransition,9520,247,2.53,9,416,1580


Joint predictor coverage:


...

Regional-context source-variable decisions:


,Wave,Source file,Variable,Variable label,Domain 10 decision,Domain 10 role,Domain 10 reason
0,Wave 2,wave_two_lsype_family_background_2020,urbind,Urban/Rural Indicator (England),Construction input,Wave 2 fallback input to the retained four-category settlement-type predictor,Wave 2 provides a fallback where the permitted Wave 3 urban–rural indicator is unavailable
1,Wave 2,wave_two_lsype_family_background_2020,gor,Government Office Region,Construction input,Wave 2 fallback input to the retained Government Office Region predictor,Wave 2 provides a fallback where the permitted Wave 3 regional value is unavailable
2,Wave 3,wave_three_lsype_family_background_2020,urbind,Urban/Rural Indicator (England),Construction input,Primary input to the retained four-category settlement-type predictor,"The latest permitted pre-transition value is used, with sparse and less-sparse categories collapsed within the same settlement type"
3,Wave 3,wave_three_lsype_family_background_2020,gor,Government Office Region,Construction input,Primary input to the retained Government Office Region predictor,The latest permitted pre-transition regional value is used because Wave 2 and Wave 3 classifications are almost identical


Regional-context representation decisions:


,Candidate representation,Decision,Final predictor,Reason
0,Original eight-category urban–rural indicator,Do not retain,NaN,"Four categories contained fewer than 50 participants, which would create sparse dummy variables"
1,Four-category settlement type,Retain,urban_rural_settlement_type_pretransition,"The representation preserves settlement-type differences, removes sparsity distinctions and has no small categories"
2,Binary urban–non-urban indicator,Do not retain,NaN,"The binary measure would remove differences among town, village and hamlet settings without a coverage or sparsity advantage over the four-category measure"
3,Nine-category Government Office Region,Retain,government_office_region_pretransition,"All regional categories have adequate sample sizes, and overlap with the settlement-type predictor is modest"


Files written in this cell: 0


In [398]:
# 12: Teacher, careers and Connexions guidance item review

import pandas as pd
guidance_review_tracks = {'Teacher and careers guidance', 'Connexions access and contact',
    'Connexions guidance use and content'}
guidance_review_inventory = domain_10_core_review_inventory.loc[domain_10_core_review_inventory['Domain 10 review track'].isin(guidance_review_tracks)].copy().sort_values(['Domain 10 review track',
    'Source order', 'Variable position']).reset_index(drop=True)
assert len(guidance_review_inventory) == 29
assert guidance_review_inventory['Variable'].is_unique
guidance_item_group_lookup = {'W1advconYP': 'Connexions awareness', 'W2advconYP': 'Connexions awareness',
    'W3advconYP': 'Connexions awareness', 'W1advperYP': 'Connexions adviser contact', 'W2advperYP': 'Connexions adviser contact', 'W3advperYP': 'Connexions adviser contact', 'W1advconnYP': 'Connexions future-study discussion frequency', 'W2advconnYP': 'Connexions future-study discussion frequency', 'W1infoconYP': 'Connexions information usefulness', 'W1advfrsYP': 'Teacher guidance during lessons', 'W2AdvFrsYP': 'Teacher guidance during lessons', 'W1advteacYP': 'Teacher guidance outside lessons', 'W2AdvTeacYP': 'Teacher guidance outside lessons', 'W2ModAp3YP0d': 'Teacher involvement in training or apprenticeship discussion', 'W2ModAp3YP0e': 'Connexions involvement in training or apprenticeship discussion', 'W3tlkteacYP0a': 'Stay-on discussion multiple-response item', 'W3tlkteacYP0b': 'Stay-on discussion multiple-response item', 'W3tlkteacYP0c': 'Stay-on discussion multiple-response item', 'W3tlkteacYP0d': 'Stay-on discussion multiple-response item', 'W3tlkteacYP0e': 'Stay-on discussion multiple-response item', 'W3tlkteacYP0f': 'Stay-on discussion multiple-response item', 'W3tlkteacYP0g': 'Stay-on discussion multiple-response item', 'W3tlktappYP0a': 'Apprenticeship discussion multiple-response item', 'W3tlktappYP0b': 'Apprenticeship discussion multiple-response item', 'W3tlktappYP0c': 'Apprenticeship discussion multiple-response item', 'W3tlktappYP0d': 'Apprenticeship discussion multiple-response item', 'W3tlktappYP0e': 'Apprenticeship discussion multiple-response item', 'W3tlktappYP0f': 'Apprenticeship discussion multiple-response item', 'W3tlktappYP0g': 'Apprenticeship discussion multiple-response item'}
assert set(guidance_review_inventory['Variable']) == set(guidance_item_group_lookup)
guidance_review_inventory['Guidance item group'] = guidance_review_inventory['Variable'].map(guidance_item_group_lookup)
guidance_raw = pd.DataFrame(index=route_index)
guidance_labelled = pd.DataFrame(index=route_index)
for source_file, source_inventory in guidance_review_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    for source_data in [raw_source, labelled_source]:
        source_data['NSID'] = standardise_nsid(source_data['NSID'])
        assert source_data['NSID'].is_unique
    raw_source = raw_source.set_index('NSID').reindex(route_index)
    labelled_source = labelled_source.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        guidance_raw[variable] = pd.to_numeric(raw_source[variable], errors='coerce')
        guidance_labelled[variable] = labelled_source[variable].astype('string')
assert set(guidance_raw.columns) == set(guidance_review_inventory['Variable'])
guidance_wave_3_variables = guidance_review_inventory.loc[guidance_review_inventory['Wave'].eq('Wave 3'),
    'Variable'].tolist()
assert len(guidance_wave_3_variables) == 16
guidance_wave_3_unrestricted = guidance_raw[guidance_wave_3_variables].copy()
guidance_raw.loc[~wave_3_permitted_mask, guidance_wave_3_variables] = pd.NA
guidance_labelled.loc[~wave_3_permitted_mask, guidance_wave_3_variables] = pd.NA
guidance_coverage_rows = []
for _, inventory_row in guidance_review_inventory.iterrows():
    variable = inventory_row['Variable']
    values = guidance_raw[variable]
    observed_mask = values.ge(0).fillna(False).astype(bool)
    special_code_mask = values.lt(0).fillna(False).astype(bool)
    if inventory_row['Wave'] == 'Wave 3':
        values_removed_by_timing = int((guidance_wave_3_unrestricted[variable].notna() & ~wave_3_permitted_mask).sum())
    else:
        values_removed_by_timing = 0
    guidance_coverage_rows.append({'Wave': inventory_row['Wave'], 'Variable': variable,
        'Guidance item group': inventory_row['Guidance item group'], 'Observed responses': int(observed_mask.sum()), 'Observed percentage': round(observed_mask.mean() * 100,
        2), 'Special-code responses': int(special_code_mask.sum()), 'Unavailable responses': int(values.isna().sum()), 'Wave 3 values removed by timing': values_removed_by_timing, 'Observed categories': int(values.where(observed_mask).nunique())})
guidance_coverage_summary = pd.DataFrame(guidance_coverage_rows).sort_values(['Guidance item group', 'Wave',
    'Variable']).reset_index(drop=True)
guidance_code_rows = []
for _, inventory_row in guidance_review_inventory.iterrows():
    variable = inventory_row['Variable']
    raw_values = guidance_raw[variable]
    labelled_values = guidance_labelled[variable]
    for raw_code, participants in raw_values.value_counts(dropna=False).items():
        if pd.isna(raw_code):
            value_label = 'Unavailable after source and timing restrictions'
            response_type = 'Unavailable'
            sort_value = 999999
        else:
            matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            response_type = 'Observed response' if raw_code >= 0 else 'Special code'
            sort_value = float(raw_code)
        guidance_code_rows.append({'Wave': inventory_row['Wave'], 'Variable': variable,
            'Guidance item group': inventory_row['Guidance item group'], 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
guidance_code_distribution = pd.DataFrame(guidance_code_rows).sort_values(['Guidance item group', 'Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
guidance_neighbourhood_rows = []
for _, inventory_row in guidance_review_inventory.iterrows():
    source_file = inventory_row['Source file']
    variable_position = int(inventory_row['Variable position'])
    neighbourhood = domain_10_register_source.loc[domain_10_register_source['Source file'].eq(source_file) & domain_10_register_source['Variable position'].between(variable_position - 2,
        variable_position + 2)][['Source order', 'Wave', 'Source type', 'Source file', 'Variable position', 'Variable',
        'Variable label', 'Timing status', 'Review outcome', 'Substantive domain']].copy()
    neighbourhood['Neighbourhood anchor'] = inventory_row['Variable']
    guidance_neighbourhood_rows.append(neighbourhood)
guidance_documentation_neighbourhood = pd.concat(guidance_neighbourhood_rows,
    ignore_index=True).drop_duplicates(subset=['Source file', 'Variable']).sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
guidance_core_variable_set = set(guidance_review_inventory['Variable'])
guidance_documentation_neighbourhood['Current guidance core item'] = guidance_documentation_neighbourhood['Variable'].isin(guidance_core_variable_set)
guidance_item_group_summary = guidance_review_inventory.groupby(['Guidance item group', 'Wave'],
    dropna=False).size().rename('Variables').reset_index().sort_values(['Guidance item group',
    'Wave']).reset_index(drop=True)
guidance_review_overview = pd.DataFrame([{'Guidance variables reviewed': int(len(guidance_review_inventory)),
    'Wave 1 variables': int(guidance_review_inventory['Wave'].eq('Wave 1').sum()), 'Wave 2 variables': int(guidance_review_inventory['Wave'].eq('Wave 2').sum()), 'Wave 3 variables': int(guidance_review_inventory['Wave'].eq('Wave 3').sum()), 'Documentation-neighbourhood variables': int(len(guidance_documentation_neighbourhood))}])
print('Guidance review overview:')
display_limited(guidance_review_overview)
print('\nGuidance variables by item group and wave:')
display_limited(guidance_item_group_summary)
guidance_core_display = guidance_review_inventory[['Wave', 'Source file', 'Variable position', 'Variable',
    'Variable label', 'Domain 10 review track', 'Guidance item group']]
print(f'\nGuidance core variables: {len(guidance_core_display):,}')
print('\nFirst 15 guidance core variables:')
display_limited(guidance_core_display.head(15))
print(f'Additional guidance core variables not displayed: {max(len(guidance_core_display) - 15, 0):,}')
print(f'\nGuidance coverage records: {len(guidance_coverage_summary):,}')
print('\nFirst 15 guidance coverage records:')
display_limited(guidance_coverage_summary.head(15))
print(f'Additional coverage records not displayed: {max(len(guidance_coverage_summary) - 15, 0):,}')
print(f'\nGuidance response-code records: {len(guidance_code_distribution):,}')
print('\nFirst 15 guidance response-code records:')
display_limited(guidance_code_distribution.head(15))
print(f'Additional response-code records not displayed: {max(len(guidance_code_distribution) - 15, 0):,}')
print(f'\nGuidance documentation-neighbourhood records: {len(guidance_documentation_neighbourhood):,}')
print('\nFirst 10 documentation-neighbourhood records:')
display_limited(guidance_documentation_neighbourhood.head(10))
print(f'Additional documentation-neighbourhood records not displayed: {max(len(guidance_documentation_neighbourhood) - 10, 0):,}')
guidance_core_inventory_file = stage_2_output_directory / 'stage_2_guidance_core_inventory.csv'
guidance_coverage_summary_file = stage_2_output_directory / 'stage_2_guidance_coverage_summary.csv'
guidance_code_distribution_file = stage_2_output_directory / 'stage_2_guidance_code_distribution.csv'
guidance_documentation_neighbourhood_file = stage_2_output_directory / 'stage_2_guidance_documentation_neighbourhood.csv'
guidance_core_display.to_csv(guidance_core_inventory_file, index=False)
guidance_coverage_summary.to_csv(guidance_coverage_summary_file, index=False)
guidance_code_distribution.to_csv(guidance_code_distribution_file, index=False)
guidance_documentation_neighbourhood.to_csv(guidance_documentation_neighbourhood_file, index=False)
print(f'\nFull guidance-review tables saved to: {stage_2_output_directory}')
print('Files written in this cell: 4')

Guidance review overview:


,Guidance variables reviewed,Wave 1 variables,Wave 2 variables,Wave 3 variables,Documentation-neighbourhood variables
0,29,6,7,16,55



Guidance variables by item group and wave:


,Guidance item group,Wave,Variables
0,Apprenticeship discussion multiple-response item,Wave 3,7
1,Connexions adviser contact,Wave 1,1
2,Connexions adviser contact,Wave 2,1
3,Connexions adviser contact,Wave 3,1
4,Connexions awareness,Wave 1,1



Guidance core variables: 29

First 15 guidance core variables:


...

Additional coverage records not displayed: 14

Guidance response-code records: 158

First 15 guidance response-code records:


,Wave,Variable,Guidance item group,Raw code,Value label,Response type,Participants
0,Wave 3,W3tlktappYP0a,Apprenticeship discussion multiple-response item,-99.0,YP not interviewed,Special code,54
1,Wave 3,W3tlktappYP0a,Apprenticeship discussion multiple-response item,0.0,Not mentioned,Observed response,8328
2,Wave 3,W3tlktappYP0a,Apprenticeship discussion multiple-response item,1.0,Mentioned,Observed response,1113
3,Wave 3,W3tlktappYP0a,Apprenticeship discussion multiple-response item,NaN,Unavailable after source and timing restrictions,Unavailable,272
4,Wave 3,W3tlktappYP0b,Apprenticeship discussion multiple-response item,-99.0,YP not interviewed,Special code,54


Additional response-code records not displayed: 143

Guidance documentation-neighbourhood records: 55

First 10 documentation-neighbourhood records:


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review outcome,Substantive domain,Neighbourhood anchor,Current guidance core item
0,1,Wave 1,Young person,wave_one_lsype_young_person_2020,185,W1ssholYP,YP: Whether YP's school has times during schoo...,Pre-transition source,Pending review,NaN,W1advconYP,False
1,1,Wave 1,Young person,wave_one_lsype_young_person_2020,186,W1ssholfYP,YP: Whether have ever gone into school during ...,Pre-transition source,Pending review,NaN,W1advconYP,False
2,1,Wave 1,Young person,wave_one_lsype_young_person_2020,187,W1advconYP,YP: Whether heard about Connexions before inte...,Pre-transition source,Pending review,NaN,W1advconYP,True
3,1,Wave 1,Young person,wave_one_lsype_young_person_2020,188,W1advperYP,YP: Whether ever talked to Connexions Personal...,Pre-transition source,Pending review,NaN,W1advconYP,True
4,1,Wave 1,Young person,wave_one_lsype_young_person_2020,189,W1advconnYP,YP: How often talk about plans for future stud...,Pre-transition source,Pending review,NaN,W1advconYP,True


Additional documentation-neighbourhood records not displayed: 45

Full guidance-review tables saved to: data_derived\stage_2_predictor_construction
Files written in this cell: 4


In [399]:
# 13: Guidance routing and Wave 3 battery structure review

import re
import pandas as pd
from scipy.stats import spearmanr
connexions_routing_specs = {'Wave 1 awareness and adviser contact': ('W1advconYP', 'W1advperYP'),
    'Wave 2 awareness and adviser contact': ('W2advconYP',
    'W2advperYP'), 'Wave 3 awareness and adviser contact': ('W3advconYP',
    'W3advperYP'), 'Wave 1 adviser contact and future-study discussion': ('W1advperYP',
    'W1advconnYP'), 'Wave 2 adviser contact and future-study discussion': ('W2advperYP',
    'W2advconnYP'), 'Wave 1 future-study discussion and information usefulness': ('W1advconnYP', 'W1infoconYP')}
connexions_routing_crosstabs = {}
for comparison_name, (first_variable, second_variable) in connexions_routing_specs.items():
    first_labels = guidance_labelled[first_variable].fillna('Unavailable')
    second_labels = guidance_labelled[second_variable].fillna('Unavailable')
    connexions_routing_crosstabs[comparison_name] = pd.crosstab(first_labels, second_labels, margins=True,
        dropna=False)
teacher_guidance_wave_pairs = {'Teacher guidance during lessons': ('W1advfrsYP', 'W2AdvFrsYP'),
    'Teacher guidance outside lessons': ('W1advteacYP', 'W2AdvTeacYP')}
teacher_guidance_wave_consistency_rows = []
teacher_guidance_wave_crosstabs = {}
for construct_name, (wave_1_variable, wave_2_variable) in teacher_guidance_wave_pairs.items():
    comparison_data = pd.DataFrame({'Wave 1 raw': guidance_raw[wave_1_variable],
        'Wave 2 raw': guidance_raw[wave_2_variable], 'Wave 1 label': guidance_labelled[wave_1_variable], 'Wave 2 label': guidance_labelled[wave_2_variable]}, index=route_index)
    complete_comparison = comparison_data.loc[comparison_data['Wave 1 raw'].between(1,
        5) & comparison_data['Wave 2 raw'].between(1, 5)].copy()
    assert len(complete_comparison) > 0
    exact_agreement = complete_comparison['Wave 1 raw'].eq(complete_comparison['Wave 2 raw'])
    teacher_guidance_wave_consistency_rows.append({'Guidance construct': construct_name,
        'Complete comparisons': int(len(complete_comparison)), 'Exact agreement': int(exact_agreement.sum()), 'Exact agreement percentage': round(exact_agreement.mean() * 100,
        2), 'Spearman correlation': round(float(spearmanr(complete_comparison['Wave 1 raw'],
        complete_comparison['Wave 2 raw']).statistic), 3), "Cramer's V": round(categorical_cramers_v(complete_comparison['Wave 1 raw'],
        complete_comparison['Wave 2 raw']), 3)})
    teacher_guidance_wave_crosstabs[construct_name] = pd.crosstab(complete_comparison['Wave 1 label'],
        complete_comparison['Wave 2 label'], margins=True, dropna=False)
teacher_guidance_wave_consistency_summary = pd.DataFrame(teacher_guidance_wave_consistency_rows)
wave_3_stay_guidance_variables = [f'W3tlkteacYP0{suffix}' for suffix in 'abcdefg']
wave_3_apprenticeship_guidance_variables = [f'W3tlktappYP0{suffix}' for suffix in 'abcdefg']
assert set(wave_3_stay_guidance_variables).issubset(guidance_raw.columns)
assert set(wave_3_apprenticeship_guidance_variables).issubset(guidance_raw.columns)
wave_3_guidance_battery_variables = wave_3_stay_guidance_variables + wave_3_apprenticeship_guidance_variables
wave_3_guidance_battery_inventory = guidance_review_inventory.loc[guidance_review_inventory['Variable'].isin(wave_3_guidance_battery_variables)].copy()
wave_3_guidance_battery_inventory['Battery'] = wave_3_guidance_battery_inventory['Variable'].map(lambda variable: 'Stay-on discussion' if variable.startswith('W3tlkteac') else 'Apprenticeship discussion')
wave_3_guidance_battery_inventory['Suffix'] = wave_3_guidance_battery_inventory['Variable'].str.extract('0([a-g])$',
    flags=re.IGNORECASE, expand=False).str.lower()
assert wave_3_guidance_battery_inventory['Suffix'].notna().all()
wave_3_guidance_battery_prevalence_rows = []
for _, inventory_row in wave_3_guidance_battery_inventory.iterrows():
    variable = inventory_row['Variable']
    values = guidance_raw[variable]
    observed_mask = values.isin([0, 1])
    mentioned_mask = values.eq(1)
    wave_3_guidance_battery_prevalence_rows.append({'Battery': inventory_row['Battery'],
        'Suffix': inventory_row['Suffix'], 'Variable': variable, 'Full variable label': inventory_row['Variable label'], 'Observed responses': int(observed_mask.sum()), 'Mentioned': int(mentioned_mask.sum()), 'Mentioned percentage of observed': round(mentioned_mask.sum() / observed_mask.sum() * 100,
        2)})
wave_3_guidance_battery_prevalence = pd.DataFrame(wave_3_guidance_battery_prevalence_rows).sort_values(['Battery',
    'Suffix']).reset_index(drop=True)
wave_3_guidance_battery_structure_rows = []
wave_3_guidance_battery_mention_distributions = {}
for battery_name, battery_variables in {'Stay-on discussion': wave_3_stay_guidance_variables,
    'Apprenticeship discussion': wave_3_apprenticeship_guidance_variables}.items():
    battery_data = guidance_raw[battery_variables].copy()
    complete_battery_mask = battery_data.isin([0, 1]).all(axis=1)
    complete_battery = battery_data.loc[complete_battery_mask].astype(int)
    assert len(complete_battery) > 0
    substantive_variables = battery_variables[:6]
    final_g_variable = battery_variables[6]
    substantive_mention_count = complete_battery[substantive_variables].sum(axis=1)
    final_g_mentioned = complete_battery[final_g_variable].eq(1)
    any_substantive_mentioned = substantive_mention_count.gt(0)
    wave_3_guidance_battery_structure_rows.append({'Battery': battery_name,
        'Complete battery responses': int(len(complete_battery)), 'At least one item 0a–0f mentioned': int(any_substantive_mentioned.sum()), 'Item 0g mentioned': int(final_g_mentioned.sum()), 'Item 0g and at least one 0a–0f both mentioned': int((final_g_mentioned & any_substantive_mentioned).sum()), 'No 0a–0f and item 0g not mentioned': int((~any_substantive_mentioned & ~final_g_mentioned).sum()), 'Maximum number of 0a–0f mentions': int(substantive_mention_count.max())})
    wave_3_guidance_battery_mention_distributions[battery_name] = substantive_mention_count.value_counts().sort_index().rename('Participants').rename_axis('Items 0a–0f mentioned').reset_index()
wave_3_guidance_battery_structure_summary = pd.DataFrame(wave_3_guidance_battery_structure_rows)
wave_3_guidance_correspondence_rows = []
for suffix in 'abcdefg':
    stay_variable = f'W3tlkteacYP0{suffix}'
    apprenticeship_variable = f'W3tlktappYP0{suffix}'
    comparison_data = guidance_raw[[stay_variable,
        apprenticeship_variable]].loc[lambda data: data[stay_variable].isin([0,
        1]) & data[apprenticeship_variable].isin([0, 1])].astype(int)
    assert len(comparison_data) > 0
    stay_mentioned = comparison_data[stay_variable].eq(1)
    apprenticeship_mentioned = comparison_data[apprenticeship_variable].eq(1)
    wave_3_guidance_correspondence_rows.append({'Suffix': suffix, 'Stay-on variable': stay_variable,
        'Apprenticeship variable': apprenticeship_variable, 'Complete comparisons': int(len(comparison_data)), 'Stay-on mentioned': int(stay_mentioned.sum()), 'Apprenticeship mentioned': int(apprenticeship_mentioned.sum()), 'Both mentioned': int((stay_mentioned & apprenticeship_mentioned).sum()), 'Neither mentioned': int((~stay_mentioned & ~apprenticeship_mentioned).sum()), "Cramer's V": round(categorical_cramers_v(comparison_data[stay_variable],
        comparison_data[apprenticeship_variable]), 3)})
wave_3_guidance_correspondence_summary = pd.DataFrame(wave_3_guidance_correspondence_rows)
guidance_structure_review_overview = pd.DataFrame([{'Connexions routing comparisons': int(len(connexions_routing_crosstabs)),
    'Repeated teacher-guidance constructs': int(len(teacher_guidance_wave_consistency_summary)), 'Wave 3 battery variables': int(len(wave_3_guidance_battery_inventory)), 'Wave 3 route-specific batteries': 2, 'Corresponding suffix comparisons': int(len(wave_3_guidance_correspondence_summary))}])
print('Guidance structure review overview:')
display_limited(guidance_structure_review_overview)
print('Connexions routing cross-tabulations:')
for comparison_name, comparison_table in connexions_routing_crosstabs.items():
    print(comparison_name)
    display_limited(comparison_table)
print('Teacher-guidance Wave 1–2 consistency:')
display_limited(teacher_guidance_wave_consistency_summary)
print('Teacher-guidance response cross-tabulations:')
for construct_name, comparison_table in teacher_guidance_wave_crosstabs.items():
    print(construct_name)
    display_limited(comparison_table)
print('Wave 3 guidance battery full-label inventory:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(wave_3_guidance_battery_prevalence)
print('Wave 3 battery structure:')
display_limited(wave_3_guidance_battery_structure_summary)
print('Wave 3 battery mention-count distributions:')
for battery_name, distribution_table in wave_3_guidance_battery_mention_distributions.items():
    print(battery_name)
    display_limited(distribution_table)
print('Corresponding Wave 3 battery-item associations:')
display_limited(wave_3_guidance_correspondence_summary)
print('Files written in this cell: 0')

Guidance structure review overview:


,Connexions routing comparisons,Repeated teacher-guidance constructs,Wave 3 battery variables,Wave 3 route-specific batteries,Corresponding suffix comparisons
0,6,2,14,2,7


Connexions routing cross-tabulations:
Wave 1 awareness and adviser contact


W1advperYP,Don't know,No,Not applicable,Unavailable,YP not interviewed,Yes,All
W1advconYP,,,,,,,
Don't know,0,0,43,0,0,0,43
No,0,0,1314,0,0,0,1314
Unavailable,0,0,0,243,0,0,243
YP not interviewed,0,0,0,0,89,0,89
Yes,57,4965,0,0,0,3056,8078


Wave 2 awareness and adviser contact


...

Apprenticeship discussion


,Items 0a–0f mentioned,Participants
0,0,6056
1,1,2792
2,2,482
3,3,96
4,4,14


Corresponding Wave 3 battery-item associations:


,Suffix,Stay-on variable,Apprenticeship variable,Complete comparisons,Stay-on mentioned,Apprenticeship mentioned,Both mentioned,Neither mentioned,Cramer's V
0,a,W3tlkteacYP0a,W3tlktappYP0a,9441,3596,1113,830,5562,0.275
1,b,W3tlkteacYP0b,W3tlktappYP0b,9441,4486,881,615,4689,0.143
2,c,W3tlkteacYP0c,W3tlktappYP0c,9441,3895,1724,1361,5183,0.362
3,d,W3tlkteacYP0d,W3tlktappYP0d,9441,341,177,75,8998,0.287
4,e,W3tlkteacYP0e,W3tlktappYP0e,9441,169,169,28,9131,0.150


Files written in this cell: 0


In [400]:
# 14: Guidance-battery completeness and exact-label check

import re
import pandas as pd
guidance_core_keys = set(zip(guidance_review_inventory['Source file'], guidance_review_inventory['Variable']))
guidance_audit_source = domain_10_register_source.loc[domain_10_register_source['Wave'].isin(['Wave 1', 'Wave 2',
    'Wave 3']) & domain_10_register_source['Source type'].eq('Young person')].copy()
guidance_audit_variable_text = guidance_audit_source['Variable'].fillna('').astype(str)
guidance_audit_label_text = guidance_audit_source['Variable label'].fillna('').astype(str)
guidance_prefix_mask = guidance_audit_variable_text.str.contains('^W[123](?:adv|info|mentor|tlkteac|tlktapp|adteac|adapp)',
    case=False, na=False, regex=True)
training_discussion_source_mask = guidance_audit_variable_text.str.contains('^W2ModAp3YP0[c-e]$', case=False, na=False,
    regex=True)
guidance_label_mask = guidance_audit_label_text.str.contains('\\bconnexions\\b|\\bcareers? advis|\\bcareers? guid|\\blearning mentor|plans for future study with.+teacher|talked to teacher.+apprent|talked to teacher.+stay|teacher/careers? advisor|school careers? advisor',
    case=False, na=False, regex=True)
current_guidance_core_mask = pd.Series([(source_file, variable) in guidance_core_keys for source_file,
    variable in zip(guidance_audit_source['Source file'],
    guidance_audit_source['Variable'])], index=guidance_audit_source.index)
guidance_battery_audit_inventory = guidance_audit_source.loc[guidance_prefix_mask | training_discussion_source_mask | guidance_label_mask | current_guidance_core_mask].copy()
guidance_battery_audit_inventory['Current guidance core item'] = [(source_file,
    variable) in guidance_core_keys for source_file, variable in zip(guidance_battery_audit_inventory['Source file'],
    guidance_battery_audit_inventory['Variable'])]
guidance_battery_audit_inventory['Guidance audit status'] = guidance_battery_audit_inventory['Current guidance core item'].map({True: 'Current 29-variable review inventory',
    False: 'Additional guidance-related variable'})

def classify_guidance_audit_group(variable, label):
    variable_lower = str(variable).lower()
    label_lower = str(label).lower()
    if re.match('^w[123]advconyp$', variable_lower):
        return 'Connexions awareness'
    if re.match('^w[123]advperyp$', variable_lower):
        return 'Connexions adviser contact'
    if re.match('^w[12]advconnyp$', variable_lower):
        return 'Connexions future-study discussion frequency'
    if 'infocon' in variable_lower:
        return 'Connexions information usefulness'
    if re.match('^w[12]advfrsyp$', variable_lower):
        return 'Teacher guidance during lessons'
    if re.match('^w[12]advteacyp$', variable_lower):
        return 'Teacher guidance outside lessons'
    if 'advcas' in variable_lower:
        return 'Careers Advice Service discussion frequency'
    if re.match('^w[12]infofr', variable_lower):
        return 'Lesson-based guidance usefulness'
    if re.match('^w[12]infote', variable_lower):
        return 'Teacher guidance usefulness'
    if re.match('^w2modap3yp0c$', variable_lower):
        return 'School careers-adviser apprenticeship discussion'
    if re.match('^w2modap3yp0d$', variable_lower):
        return 'Teacher apprenticeship discussion'
    if re.match('^w2modap3yp0e$', variable_lower):
        return 'Connexions apprenticeship discussion'
    if 'tlkteac' in variable_lower:
        return 'Wave 3 stay-on discussion source battery'
    if 'tlktapp' in variable_lower:
        return 'Wave 3 apprenticeship discussion source battery'
    if 'adteac2' in variable_lower:
        return 'Wave 3 stay-on recommendation content'
    if 'adapp2' in variable_lower:
        return 'Wave 3 apprenticeship recommendation content'
    if 'adteac' in variable_lower:
        return 'Wave 3 teacher or careers influence'
    if 'mentor' in variable_lower:
        return 'Learning-mentor provision or use'
    if re.match('^w[12]advfam', variable_lower):
        return 'Family future-study discussion'
    if re.match('^w[12]advpal', variable_lower):
        return 'Peer future-study discussion'
    if 'connexions' in label_lower or 'career' in label_lower or 'teacher' in label_lower or ('mentor' in label_lower):
        return 'Other guidance-related item'
    return 'Battery-neighbourhood item'
guidance_battery_audit_inventory['Guidance audit group'] = [classify_guidance_audit_group(variable,
    label) for variable, label in zip(guidance_battery_audit_inventory['Variable'],
    guidance_battery_audit_inventory['Variable label'])]
guidance_battery_audit_inventory = guidance_battery_audit_inventory.sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert guidance_core_keys.issubset(set(zip(guidance_battery_audit_inventory['Source file'],
    guidance_battery_audit_inventory['Variable'])))
guidance_additional_inventory = guidance_battery_audit_inventory.loc[~guidance_battery_audit_inventory['Current guidance core item']].copy().reset_index(drop=True)
guidance_audit_group_summary = guidance_battery_audit_inventory.groupby(['Guidance audit group',
    'Guidance audit status'], dropna=False).size().rename('Variables').reset_index().sort_values(['Guidance audit group',
    'Guidance audit status']).reset_index(drop=True)
guidance_battery_audit_overview = pd.DataFrame([{'Current guidance review variables': int(len(guidance_review_inventory)),
    'Guidance-related variables found by expanded audit': int(len(guidance_battery_audit_inventory)), 'Additional guidance-related variables': int(len(guidance_additional_inventory)), 'Wave 1 additions': int(guidance_additional_inventory['Wave'].eq('Wave 1').sum()), 'Wave 2 additions': int(guidance_additional_inventory['Wave'].eq('Wave 2').sum()), 'Wave 3 additions': int(guidance_additional_inventory['Wave'].eq('Wave 3').sum())}])
wave_3_exact_label_inventory = guidance_battery_audit_inventory.loc[guidance_battery_audit_inventory['Variable'].str.contains('^W3(?:tlkteac|tlktapp|adteac|adapp)',
    case=False, na=False, regex=True), ['Variable position', 'Variable', 'Variable label', 'Guidance audit group',
    'Current guidance core item']].copy().sort_values('Variable position').reset_index(drop=True)
print('Expanded guidance-battery audit:')
display_limited(guidance_battery_audit_overview)
print('Guidance variables by audit group:')
with pd.option_context('display.max_rows', None):
    display_limited(guidance_audit_group_summary)
print('Additional guidance-related variables:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(guidance_additional_inventory[['Wave', 'Source file', 'Variable position', 'Variable',
        'Variable label', 'Timing status', 'Review outcome', 'Substantive domain', 'Guidance audit group']])
print('Exact Wave 3 guidance labels:')
for _, label_row in wave_3_exact_label_inventory.iterrows():
    print(f"{int(label_row['Variable position'])} | {label_row['Variable']} | {label_row['Variable label']} | {label_row['Guidance audit group']} | Current core item: {label_row['Current guidance core item']}")
print('Files written in this cell: 0')

Expanded guidance-battery audit:


,Current guidance review variables,Guidance-related variables found by expanded audit,Additional guidance-related variables,Wave 1 additions,Wave 2 additions,Wave 3 additions
0,29,69,40,5,5,30


Guidance variables by audit group:


,Guidance audit group,Guidance audit status,Variables
0,Battery-neighbourhood item,Additional guidance-related variable,2
1,Careers Advice Service discussion frequency,Additional guidance-related variable,1
2,Connexions adviser contact,Current 29-variable review inventory,3
3,Connexions apprenticeship discussion,Current 29-variable review inventory,1
4,Connexions awareness,Current 29-variable review inventory,3


Additional guidance-related variables:


,Wave,Source file,Variable position,Variable,Variable label,Timing status,Review outcome,Substantive domain,Guidance audit group
0,Wave 1,wave_one_lsype_young_person_2020,192,W1advfamYP,YP: How often talk about plans for future study with - Members of family,Pre-transition source,Pending review,NaN,Family future-study discussion
1,Wave 1,wave_one_lsype_young_person_2020,193,W1advpalYP,YP: How often talk about plans for future study with - friends,Pre-transition source,Pending review,NaN,Peer future-study discussion
2,Wave 1,wave_one_lsype_young_person_2020,195,W1infofrnYP,YP: Usefulness of information from this source: as part of a lesson,Pre-transition source,Pending review,NaN,Lesson-based guidance usefulness
3,Wave 1,wave_one_lsype_young_person_2020,196,W1infoteYP,YP: Usefulness of information from this source: from teachers outside lessons,Pre-transition source,Pending review,NaN,Teacher guidance usefulness
4,Wave 1,wave_one_lsype_young_person_2020,197,W1infofamYP,YP: Usefulness of information from this source: from members of your family such,Pre-transition source,Pending review,NaN,Battery-neighbourhood item


Exact Wave 3 guidance labels:
482 | W3tlkteacYP0a | YP: Whether YP talked to teachers/careers advisors about whether or not to stay  | Wave 3 stay-on discussion source battery | Current core item: True
483 | W3tlkteacYP0b | YP: Whether YP talked to teachers/careers advisors about whether or not to stay  | Wave 3 stay-on discussion source battery | Current core item: True
484 | W3tlkteacYP0c | YP: Whether YP talked to teachers/careers advisors about whether or not to stay  | Wave 3 stay-on discussion source battery | Current core item: True
485 | W3tlkteacYP0d | YP: Whether YP talked to teachers/careers advisors about whether or not to stay  | Wave 3 stay-on discussion source battery | Current core item: True
486 | W3tlkteacYP0e | YP: Whether YP talked to teachers/careers advisors about whether or not to stay  | Wave 3 stay-on discussion source battery | Current core item: True
487 | W3tlkteacYP0f | YP: Whether YP talked to teachers/careers advisors about whether or not to stay  | Wave 

In [401]:
# 15: Full Stata-label review for guidance batteries

import re
import pandas as pd
guidance_exact_label_variable_names = set(guidance_battery_audit_inventory['Variable'])
guidance_full_label_rows = []
for source_file, source_inventory in guidance_battery_audit_inventory.groupby('Source file', sort=False):
    source_path = source_file_lookup[source_file]
    with pd.io.stata.StataReader(source_path) as stata_reader:
        full_variable_labels = stata_reader.variable_labels()
    for _, inventory_row in source_inventory.iterrows():
        variable = inventory_row['Variable']
        guidance_full_label_rows.append({'Source order': inventory_row['Source order'], 'Wave': inventory_row['Wave'],
            'Source file': source_file, 'Variable position': inventory_row['Variable position'], 'Variable': variable, 'Register variable label': inventory_row['Variable label'], 'Full Stata variable label': full_variable_labels.get(variable,
            pd.NA), 'Guidance audit group': inventory_row['Guidance audit group'], 'Current guidance core item': inventory_row['Current guidance core item']})
guidance_full_label_inventory = pd.DataFrame(guidance_full_label_rows).sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(guidance_full_label_inventory) == len(guidance_battery_audit_inventory)
assert set(guidance_full_label_inventory['Variable']) == guidance_exact_label_variable_names
assert guidance_full_label_inventory['Full Stata variable label'].notna().all()
guidance_full_label_inventory['Battery suffix'] = guidance_full_label_inventory['Variable'].str.extract('0([a-g])$',
    flags=re.IGNORECASE, expand=False).str.lower()
wave_3_route_specific_groups = {'Wave 3 stay-on discussion source battery', 'Wave 3 stay-on recommendation content',
    'Wave 3 teacher or careers influence', 'Wave 3 apprenticeship discussion source battery', 'Wave 3 apprenticeship recommendation content', 'Other guidance-related item'}
wave_3_route_specific_full_labels = guidance_full_label_inventory.loc[guidance_full_label_inventory['Wave'].eq('Wave 3') & guidance_full_label_inventory['Guidance audit group'].isin(wave_3_route_specific_groups),
    ['Variable position', 'Variable', 'Battery suffix', 'Full Stata variable label', 'Guidance audit group',
    'Current guidance core item']].copy().sort_values('Variable position').reset_index(drop=True)
institutional_guidance_variable_names = {'W1advconYP', 'W1advperYP', 'W1advconnYP', 'W1advfrsYP', 'W1advteacYP',
    'W1infoconYP', 'W1infofrnYP', 'W1infoteYP', 'W2advconYP', 'W2advperYP', 'W2advconnYP', 'W2AdvFrsYP', 'W2AdvTeacYP', 'W2AdvCASYP', 'W2ModAp3YP0c', 'W2ModAp3YP0d', 'W2ModAp3YP0e'}
institutional_guidance_full_labels = guidance_full_label_inventory.loc[guidance_full_label_inventory['Variable'].isin(institutional_guidance_variable_names),
    ['Wave', 'Variable position', 'Variable', 'Full Stata variable label', 'Guidance audit group',
    'Current guidance core item']].copy().sort_values(['Wave', 'Variable position']).reset_index(drop=True)
assert set(institutional_guidance_full_labels['Variable']) == institutional_guidance_variable_names

def classify_guidance_addition_for_review(variable, audit_group):
    variable_lower = str(variable).lower()
    if variable_lower == 'w2infon':
        return ('Search false positive', 'Administrative household-reference variable')
    if audit_group in {'Family future-study discussion', 'Peer future-study discussion',
        'Battery-neighbourhood item'}:
        return ('Outside Domain 10',
            'The item concerns family or peer guidance, or entered the audit through a broad search pattern')
    if audit_group in {'Careers Advice Service discussion frequency', 'Lesson-based guidance usefulness',
        'Teacher guidance usefulness', 'School careers-adviser apprenticeship discussion', 'Learning-mentor provision or use'}:
        return ('Domain 10 substantive review',
            'The item directly measures school- or service-based guidance provision, use or perceived usefulness')
    if audit_group in {'Wave 3 stay-on recommendation content', 'Wave 3 teacher or careers influence',
        'Wave 3 apprenticeship recommendation content', 'Other guidance-related item'}:
        return ('Domain 10 timing and proximity review',
            'The item is institutional guidance-related but may encode a route-specific recommendation or influence very close to the post-16 transition')
    return ('Unresolved addition', 'The item requires manual review')
guidance_addition_review_rows = []
for _, addition_row in guidance_additional_inventory.iterrows():
    review_status, review_reason = classify_guidance_addition_for_review(addition_row['Variable'],
        addition_row['Guidance audit group'])
    full_label_match = guidance_full_label_inventory.loc[guidance_full_label_inventory['Source file'].eq(addition_row['Source file']) & guidance_full_label_inventory['Variable'].eq(addition_row['Variable']),
        'Full Stata variable label']
    assert len(full_label_match) == 1
    guidance_addition_review_rows.append({'Wave': addition_row['Wave'], 'Source file': addition_row['Source file'],
        'Variable position': addition_row['Variable position'], 'Variable': addition_row['Variable'], 'Full Stata variable label': full_label_match.iloc[0], 'Guidance audit group': addition_row['Guidance audit group'], 'Addition review status': review_status, 'Addition review reason': review_reason})
guidance_addition_review_inventory = pd.DataFrame(guidance_addition_review_rows).sort_values(['Addition review status',
    'Wave', 'Variable position']).reset_index(drop=True)
guidance_addition_review_summary = guidance_addition_review_inventory['Addition review status'].value_counts().rename('Variables').rename_axis('Addition review status').reset_index()
guidance_full_label_review_overview = pd.DataFrame([{'Guidance variables with full Stata labels': int(len(guidance_full_label_inventory)),
    'Wave 3 route-specific variables': int(len(wave_3_route_specific_full_labels)), 'Wave 1–2 institutional-guidance variables': int(len(institutional_guidance_full_labels)), 'Expanded-audit additions classified': int(len(guidance_addition_review_inventory)), 'Unresolved additions': int(guidance_addition_review_inventory['Addition review status'].eq('Unresolved addition').sum())}])
print('Full-label review overview:')
display_limited(guidance_full_label_review_overview)
print(f'\nWave 1–2 institutional-guidance labels: {len(institutional_guidance_full_labels):,}')
print('\nFirst 15 Wave 1–2 full-label records:')
display_limited(institutional_guidance_full_labels[['Wave', 'Variable position', 'Variable',
    'Full Stata variable label', 'Guidance audit group', 'Current guidance core item']].head(15))
print(f'Additional Wave 1–2 full-label records not displayed: {max(len(institutional_guidance_full_labels) - 15, 0):,}')
print(f'\nWave 3 route-specific labels: {len(wave_3_route_specific_full_labels):,}')
print('\nFirst 15 Wave 3 route-specific records:')
display_limited(wave_3_route_specific_full_labels[['Variable position', 'Variable', 'Battery suffix',
    'Full Stata variable label', 'Guidance audit group', 'Current guidance core item']].head(15))
print(f'Additional Wave 3 route-specific records not displayed: {max(len(wave_3_route_specific_full_labels) - 15, 0):,}')
print('\nExpanded-audit addition classification:')
display_limited(guidance_addition_review_summary)
print(f'\nExpanded-audit variables: {len(guidance_addition_review_inventory):,}')
print('\nFirst 15 expanded-audit variables:')
display_limited(guidance_addition_review_inventory.head(15))
print(f'Additional expanded-audit variables not displayed: {max(len(guidance_addition_review_inventory) - 15, 0):,}')
institutional_guidance_full_labels_file = stage_2_output_directory / 'stage_2_institutional_guidance_full_labels.csv'
wave_3_route_specific_full_labels_file = stage_2_output_directory / 'stage_2_wave_3_guidance_route_specific_full_labels.csv'
guidance_addition_review_inventory_file = stage_2_output_directory / 'stage_2_guidance_expanded_audit_inventory.csv'
institutional_guidance_full_labels.to_csv(institutional_guidance_full_labels_file, index=False)
wave_3_route_specific_full_labels.to_csv(wave_3_route_specific_full_labels_file, index=False)
guidance_addition_review_inventory.to_csv(guidance_addition_review_inventory_file, index=False)
print(f'\nFull label-review tables saved to: {stage_2_output_directory}')
print('Files written in this cell: 3')

Full-label review overview:


,Guidance variables with full Stata labels,Wave 3 route-specific variables,Wave 1–2 institutional-guidance variables,Expanded-audit additions classified,Unresolved additions
0,69,42,17,40,0



Wave 1–2 institutional-guidance labels: 17

First 15 Wave 1–2 full-label records:


,Wave,Variable position,Variable,Full Stata variable label,Guidance audit group,Current guidance core item
0,Wave 1,187,W1advconYP,YP: Whether heard about Connexions before inte...,Connexions awareness,True
1,Wave 1,188,W1advperYP,YP: Whether ever talked to Connexions Personal...,Connexions adviser contact,True
2,Wave 1,189,W1advconnYP,YP: How often talk about plans for future stud...,Connexions future-study discussion frequency,True
3,Wave 1,190,W1advfrsYP,YP: How often talk about plans for future stud...,Teacher guidance during lessons,True
4,Wave 1,191,W1advteacYP,YP: How often talk about plans for future stud...,Teacher guidance outside lessons,True


Additional Wave 1–2 full-label records not displayed: 2

Wave 3 route-specific labels: 42

First 15 Wave 3 route-specific records:


...

Additional Wave 3 route-specific records not displayed: 27

Expanded-audit addition classification:


,Addition review status,Variables
0,Domain 10 timing and proximity review,28
1,Domain 10 substantive review,6
2,Outside Domain 10,5
3,Search false positive,1



Expanded-audit variables: 40

First 15 expanded-audit variables:


,Wave,Source file,Variable position,Variable,Full Stata variable label,Guidance audit group,Addition review status,Addition review reason
0,Wave 1,wave_one_lsype_young_person_2020,195,W1infofrnYP,YP: Usefulness of information from this source...,Lesson-based guidance usefulness,Domain 10 substantive review,The item directly measures school- or service-...
1,Wave 1,wave_one_lsype_young_person_2020,196,W1infoteYP,YP: Usefulness of information from this source...,Teacher guidance usefulness,Domain 10 substantive review,The item directly measures school- or service-...
2,Wave 2,wave_two_lsype_young_person_2020,291,W2AdvCASYP,YP: How often talk about plans for future stud...,Careers Advice Service discussion frequency,Domain 10 substantive review,The item directly measures school- or service-...
3,Wave 2,wave_two_lsype_young_person_2020,328,W2ModAp3YP0c,YP: Who YP has talked to about training or an ...,School careers-adviser apprenticeship discussion,Domain 10 substantive review,The item directly measures school- or service-...
4,Wave 3,wave_three_lsype_young_person_2020,224,W3mentor1YP,YP: Whether YP's school has learning mentors,Learning-mentor provision or use,Domain 10 substantive review,The item directly measures school- or service-...


Additional expanded-audit variables not displayed: 25

Full label-review tables saved to: data_derived\stage_2_predictor_construction
Files written in this cell: 3


In [402]:
# 16: Wave 3 route-specific guidance routing review

import re
import pandas as pd
wave_3_guidance_proximity_inventory = guidance_addition_review_inventory.loc[guidance_addition_review_inventory['Addition review status'].eq('Domain 10 timing and proximity review')].copy().sort_values('Variable position').reset_index(drop=True)
assert len(wave_3_guidance_proximity_inventory) == 28
assert set(wave_3_guidance_proximity_inventory['Wave']) == {'Wave 3'}

def assign_wave_3_guidance_battery(variable):
    if variable.startswith('W3adteac2YP'):
        return 'Stay-on recommendation content'
    if variable.startswith('W3adteacYP'):
        return 'Stay-on reported influence'
    if variable.startswith('W3adapp2YP'):
        return 'Apprenticeship recommendation content'
    if variable.startswith('W3adappYP'):
        return 'Apprenticeship reported influence'
    raise ValueError(f'Unexpected Wave 3 guidance variable: {variable}')
wave_3_guidance_proximity_inventory['Route-specific guidance battery'] = wave_3_guidance_proximity_inventory['Variable'].map(assign_wave_3_guidance_battery)
wave_3_guidance_proximity_inventory['Battery suffix'] = wave_3_guidance_proximity_inventory['Variable'].str.extract('0([a-g])$',
    flags=re.IGNORECASE, expand=False).str.lower()
assert wave_3_guidance_proximity_inventory['Battery suffix'].notna().all()
assert wave_3_guidance_proximity_inventory.groupby('Route-specific guidance battery').size().eq(7).all()
wave_3_guidance_proximity_raw = pd.DataFrame(index=route_index)
wave_3_guidance_proximity_labelled = pd.DataFrame(index=route_index)
for source_file, source_inventory in wave_3_guidance_proximity_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    for source_data in [raw_source, labelled_source]:
        source_data['NSID'] = standardise_nsid(source_data['NSID'])
        assert source_data['NSID'].is_unique
    raw_source = raw_source.set_index('NSID').reindex(route_index)
    labelled_source = labelled_source.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        wave_3_guidance_proximity_raw[variable] = pd.to_numeric(raw_source[variable], errors='coerce')
        wave_3_guidance_proximity_labelled[variable] = labelled_source[variable].astype('string')
assert set(wave_3_guidance_proximity_raw.columns) == set(wave_3_guidance_proximity_inventory['Variable'])
wave_3_guidance_proximity_unrestricted = wave_3_guidance_proximity_raw.copy()
wave_3_guidance_proximity_raw.loc[~wave_3_permitted_mask, :] = pd.NA
wave_3_guidance_proximity_labelled.loc[~wave_3_permitted_mask, :] = pd.NA
wave_3_guidance_proximity_prevalence_rows = []
for _, inventory_row in wave_3_guidance_proximity_inventory.iterrows():
    variable = inventory_row['Variable']
    values = wave_3_guidance_proximity_raw[variable]
    observed_mask = values.isin([0, 1])
    mentioned_mask = values.eq(1)
    wave_3_guidance_proximity_prevalence_rows.append({'Battery': inventory_row['Route-specific guidance battery'],
        'Suffix': inventory_row['Battery suffix'], 'Variable': variable, 'Observed responses': int(observed_mask.sum()), 'Mentioned': int(mentioned_mask.sum()), 'Mentioned percentage of observed': round(mentioned_mask.sum() / observed_mask.sum() * 100,
        2) if observed_mask.sum() > 0 else pd.NA, 'Special-code responses': int(values.lt(0).fillna(False).sum()), 'Unavailable after timing restriction': int(values.isna().sum()), 'Values removed by timing restriction': int((wave_3_guidance_proximity_unrestricted[variable].notna() & ~wave_3_permitted_mask).sum())})
wave_3_guidance_proximity_prevalence = pd.DataFrame(wave_3_guidance_proximity_prevalence_rows).sort_values(['Battery',
    'Suffix']).reset_index(drop=True)
wave_3_guidance_proximity_code_rows = []
for _, inventory_row in wave_3_guidance_proximity_inventory.iterrows():
    variable = inventory_row['Variable']
    raw_values = wave_3_guidance_proximity_raw[variable]
    labelled_values = wave_3_guidance_proximity_labelled[variable]
    for raw_code, participants in raw_values.value_counts(dropna=False).items():
        if pd.isna(raw_code):
            value_label = 'Unavailable'
            response_type = 'Unavailable'
        else:
            matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            response_type = 'Observed response' if raw_code in {0, 1} else 'Special code'
        wave_3_guidance_proximity_code_rows.append({'Battery': inventory_row['Route-specific guidance battery'],
            'Variable': variable, 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants)})
wave_3_guidance_proximity_code_patterns = pd.DataFrame(wave_3_guidance_proximity_code_rows).groupby(['Battery',
    'Raw code', 'Value label', 'Response type'], dropna=False).agg(Variables=('Variable', 'nunique'),
    Minimum_participants=('Participants', 'min'), Maximum_participants=('Participants',
    'max')).reset_index().sort_values(['Battery', 'Raw code'], na_position='last').reset_index(drop=True)
wave_3_guidance_proximity_status = pd.DataFrame(index=route_index)
wave_3_guidance_proximity_structure_rows = []
for battery_name, battery_inventory in wave_3_guidance_proximity_inventory.groupby('Route-specific guidance battery',
    sort=False):
    battery_inventory = battery_inventory.sort_values('Battery suffix')
    battery_variables = battery_inventory['Variable'].tolist()
    substantive_variables = battery_inventory.loc[battery_inventory['Battery suffix'].isin(list('abcdef')),
        'Variable'].tolist()
    final_g_variables = battery_inventory.loc[battery_inventory['Battery suffix'].eq('g'), 'Variable'].tolist()
    assert len(substantive_variables) == 6
    assert len(final_g_variables) == 1
    final_g_variable = final_g_variables[0]
    battery_data = wave_3_guidance_proximity_raw[battery_variables].copy()
    complete_battery_mask = battery_data.isin([0, 1]).all(axis=1)
    complete_battery = battery_data.loc[complete_battery_mask].astype(int)
    substantive_mention_count = complete_battery[substantive_variables].sum(axis=1)
    any_substantive_mentioned = substantive_mention_count.gt(0)
    final_g_mentioned = complete_battery[final_g_variable].eq(1)
    battery_status = pd.Series(pd.NA, index=route_index, dtype='Int64')
    battery_status.loc[complete_battery.index] = any_substantive_mentioned.astype(int).values
    wave_3_guidance_proximity_status[battery_name] = battery_status
    wave_3_guidance_proximity_structure_rows.append({'Battery': battery_name,
        'Complete battery responses': int(len(complete_battery)), 'At least one item 0a–0f mentioned': int(any_substantive_mentioned.sum()), 'Item 0g mentioned': int(final_g_mentioned.sum()), 'Item 0g and at least one 0a–0f both mentioned': int((final_g_mentioned & any_substantive_mentioned).sum()), 'Neither 0a–0f nor item 0g mentioned': int((~any_substantive_mentioned & ~final_g_mentioned).sum()), 'Maximum number of 0a–0f mentions': int(substantive_mention_count.max())})
wave_3_guidance_proximity_structure_summary = pd.DataFrame(wave_3_guidance_proximity_structure_rows)
wave_3_stay_discussion_status = pd.Series(pd.NA, index=route_index, dtype='Int64', name='Stay-on discussion')
wave_3_stay_discussion_valid = guidance_raw['W3tlkteacYP0g'].isin([0, 1])
wave_3_stay_discussion_status.loc[wave_3_stay_discussion_valid] = 1 - guidance_raw.loc[wave_3_stay_discussion_valid,
    'W3tlkteacYP0g'].astype(int)
wave_3_apprenticeship_discussion_status = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='Apprenticeship discussion')
wave_3_apprenticeship_discussion_valid = guidance_raw['W3tlktappYP0g'].isin([0, 1])
wave_3_apprenticeship_discussion_status.loc[wave_3_apprenticeship_discussion_valid] = 1 - guidance_raw.loc[wave_3_apprenticeship_discussion_valid,
    'W3tlktappYP0g'].astype(int)
wave_3_guidance_routing_specs = [('Stay-on', wave_3_stay_discussion_status, 'Stay-on recommendation content',
    'Stay-on reported influence'), ('Apprenticeship', wave_3_apprenticeship_discussion_status,
    'Apprenticeship recommendation content', 'Apprenticeship reported influence')]
wave_3_guidance_routing_rows = []
wave_3_guidance_routing_crosstabs = {}
for route_name, discussion_status, recommendation_column, influence_column in wave_3_guidance_routing_specs:
    route_data = pd.DataFrame({'Discussion': discussion_status,
        'Recommendation': wave_3_guidance_proximity_status[recommendation_column], 'Reported influence': wave_3_guidance_proximity_status[influence_column]}, index=route_index)
    for downstream_measure in ['Recommendation', 'Reported influence']:
        pair_data = route_data[['Discussion', downstream_measure]].dropna().astype(int)
        wave_3_guidance_routing_rows.append({'Route': route_name, 'Downstream measure': downstream_measure,
            'Complete comparisons': int(len(pair_data)), 'Discussion present': int(pair_data['Discussion'].eq(1).sum()), 'Downstream measure present': int(pair_data[downstream_measure].eq(1).sum()), 'Downstream present without discussion': int((pair_data['Discussion'].eq(0) & pair_data[downstream_measure].eq(1)).sum()), 'Discussion present without downstream measure': int((pair_data['Discussion'].eq(1) & pair_data[downstream_measure].eq(0)).sum()), "Cramer's V": round(categorical_cramers_v(pair_data['Discussion'],
            pair_data[downstream_measure]), 3)})
        wave_3_guidance_routing_crosstabs[route_name, downstream_measure] = pd.crosstab(pair_data['Discussion'],
            pair_data[downstream_measure], margins=True, dropna=False)
wave_3_guidance_routing_summary = pd.DataFrame(wave_3_guidance_routing_rows)
wave_3_guidance_downstream_rows = []
for route_name, recommendation_column, influence_column in [('Stay-on', 'Stay-on recommendation content',
    'Stay-on reported influence'), ('Apprenticeship', 'Apprenticeship recommendation content',
    'Apprenticeship reported influence')]:
    pair_data = wave_3_guidance_proximity_status[[recommendation_column, influence_column]].dropna().astype(int)
    wave_3_guidance_downstream_rows.append({'Route': route_name, 'Complete comparisons': int(len(pair_data)),
        'Recommendation present': int(pair_data[recommendation_column].eq(1).sum()), 'Reported influence present': int(pair_data[influence_column].eq(1).sum()), 'Influence present without recommendation': int((pair_data[recommendation_column].eq(0) & pair_data[influence_column].eq(1)).sum()), "Cramer's V": round(categorical_cramers_v(pair_data[recommendation_column],
        pair_data[influence_column]), 3)})
wave_3_guidance_downstream_summary = pd.DataFrame(wave_3_guidance_downstream_rows)
wave_3_guidance_proximity_overview = pd.DataFrame([{'Route-specific variables reviewed': int(len(wave_3_guidance_proximity_inventory)),
    'Batteries reviewed': int(wave_3_guidance_proximity_inventory['Route-specific guidance battery'].nunique()), 'Wave 3 interviews within permitted timing': int(wave_3_permitted_mask.sum()), 'Wave 3 interviews outside permitted timing': int((~wave_3_permitted_mask).sum())}])
print('Wave 3 route-specific guidance overview:')
display_limited(wave_3_guidance_proximity_overview)
print('Variable-level response prevalence:')
with pd.option_context('display.max_rows', None):
    display_limited(wave_3_guidance_proximity_prevalence)
print('Response-code patterns by battery:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(wave_3_guidance_proximity_code_patterns)
print('Route-specific battery structure:')
display_limited(wave_3_guidance_proximity_structure_summary)
print('Discussion-to-recommendation and influence routing:')
display_limited(wave_3_guidance_routing_summary)
print('Routing cross-tabulations:')
for (route_name, downstream_measure), comparison_table in wave_3_guidance_routing_crosstabs.items():
    print(f'{route_name}: discussion versus {downstream_measure}')
    display_limited(comparison_table)
print('Recommendation and reported-influence relationship:')
display_limited(wave_3_guidance_downstream_summary)
print('Files written in this cell: 0')

Wave 3 route-specific guidance overview:


,Route-specific variables reviewed,Batteries reviewed,Wave 3 interviews within permitted timing,Wave 3 interviews outside permitted timing
0,28,4,9495,272


Variable-level response prevalence:


,Battery,Suffix,Variable,Observed responses,Mentioned,Mentioned percentage of observed,Special-code responses,Unavailable after timing restriction,Values removed by timing restriction
0,Apprenticeship recommendation content,a,W3adapp2YP0a,3344,430,12.86,6151,272,14
1,Apprenticeship recommendation content,b,W3adapp2YP0b,3344,374,11.18,6151,272,14
2,Apprenticeship recommendation content,c,W3adapp2YP0c,3344,773,23.12,6151,272,14
3,Apprenticeship recommendation content,d,W3adapp2YP0d,3344,71,2.12,6151,272,14
4,Apprenticeship recommendation content,e,W3adapp2YP0e,3344,103,3.08,6151,272,14


Response-code patterns by battery:


...

Apprenticeship: discussion versus Reported influence


Reported influence,0,1,All
Discussion,,,
1,2392,952,3344
All,2392,952,3344


Recommendation and reported-influence relationship:


,Route,Complete comparisons,Recommendation present,Reported influence present,Influence present without recommendation,Cramer's V
0,Stay-on,7784,6258,2945,265,0.208
1,Apprenticeship,3344,1552,952,113,0.528


Files written in this cell: 0


In [403]:
# 17: Route-specific guidance profile and plan-overlap review

import pandas as pd
assert callable(globals().get('retrieve_existing_predictor'))

def construct_route_guidance_profile(discussion_status, recommendation_status, influence_status, output_name):
    profile = pd.Series(pd.NA, index=route_index, dtype='Int64', name=output_name)
    no_discussion_mask = discussion_status.eq(0).fillna(False)
    discussed_mask = discussion_status.eq(1).fillna(False)
    complete_downstream_mask = discussed_mask & recommendation_status.isin([0, 1]) & influence_status.isin([0, 1])
    profile.loc[no_discussion_mask] = 0
    profile.loc[complete_downstream_mask & recommendation_status.eq(0) & influence_status.eq(0)] = 1
    profile.loc[complete_downstream_mask & recommendation_status.eq(1) & influence_status.eq(0)] = 2
    profile.loc[complete_downstream_mask & influence_status.eq(1)] = 3
    construction_audit = {'Valid discussion status': int(discussion_status.isin([0, 1]).sum()),
        'No discussion': int(no_discussion_mask.sum()), 'Discussion present': int(discussed_mask.sum()), 'Discussion with complete downstream responses': int(complete_downstream_mask.sum()), 'Influence reported without recommendation': int((complete_downstream_mask & recommendation_status.eq(0) & influence_status.eq(1)).sum()), 'Profile non-missing': int(profile.notna().sum()), 'Profile missing': int(profile.isna().sum())}
    assert profile.notna().sum() == no_discussion_mask.sum() + complete_downstream_mask.sum()
    assert set(profile.dropna().astype(int).unique()).issubset({0, 1, 2, 3})
    return (profile, construction_audit)
stay_on_guidance_profile_candidate, stay_on_guidance_profile_audit = construct_route_guidance_profile(discussion_status=wave_3_stay_discussion_status,
    recommendation_status=wave_3_guidance_proximity_status['Stay-on recommendation content'], influence_status=wave_3_guidance_proximity_status['Stay-on reported influence'], output_name='stay_on_guidance_profile_candidate')
apprenticeship_guidance_profile_candidate, apprenticeship_guidance_profile_audit = construct_route_guidance_profile(discussion_status=wave_3_apprenticeship_discussion_status,
    recommendation_status=wave_3_guidance_proximity_status['Apprenticeship recommendation content'], influence_status=wave_3_guidance_proximity_status['Apprenticeship reported influence'], output_name='apprenticeship_guidance_profile_candidate')
route_guidance_profile_labels = {0: 'No route discussion',
    1: 'Discussion without reported recommendation or influence', 2: 'Recommendation reported without reported influence', 3: 'Reported influence on decision'}
route_guidance_profile_audit_summary = pd.DataFrame([{'Route-specific profile': 'Stay-on guidance',
    **stay_on_guidance_profile_audit}, {'Route-specific profile': 'Apprenticeship guidance',
    **apprenticeship_guidance_profile_audit}])
route_guidance_profile_specs = {'Stay-on guidance': stay_on_guidance_profile_candidate,
    'Apprenticeship guidance': apprenticeship_guidance_profile_candidate}
route_guidance_profile_distribution_rows = []
for profile_name, profile_values in route_guidance_profile_specs.items():
    observed_total = int(profile_values.notna().sum())
    for profile_code, participants in profile_values.value_counts(dropna=False).sort_index().items():
        if pd.isna(profile_code):
            profile_label = 'Unavailable'
            observed_percentage = pd.NA
        else:
            profile_label = route_guidance_profile_labels[int(profile_code)]
            observed_percentage = round(participants / observed_total * 100, 2)
        route_guidance_profile_distribution_rows.append({'Route-specific profile': profile_name,
            'Profile code': profile_code, 'Profile label': profile_label, 'Participants': int(participants), 'Percentage of observed profiles': observed_percentage})
route_guidance_profile_distribution = pd.DataFrame(route_guidance_profile_distribution_rows)
route_guidance_profile_pair = pd.DataFrame({'Stay-on guidance': stay_on_guidance_profile_candidate,
    'Apprenticeship guidance': apprenticeship_guidance_profile_candidate}, index=route_index).dropna()
route_guidance_profile_relationship = pd.DataFrame([{'Complete profile pairs': int(len(route_guidance_profile_pair)),
    "Cramer's V": round(categorical_cramers_v(route_guidance_profile_pair['Stay-on guidance'],
    route_guidance_profile_pair['Apprenticeship guidance']), 3)}])
route_guidance_profile_crosstab = pd.crosstab(route_guidance_profile_pair['Stay-on guidance'].map(route_guidance_profile_labels),
    route_guidance_profile_pair['Apprenticeship guidance'].map(route_guidance_profile_labels), margins=True, dropna=False)
domain_5_plan_predictor_specs = {'Expected post-16 route': 'expected_post16_route',
    'Higher-education application likelihood': 'higher_education_application_likelihood', 'Expected peer post-16 route': 'expected_peer_post16_route'}
domain_5_plan_predictors = {}
domain_5_plan_predictor_source_rows = []
for display_name, predictor_name in domain_5_plan_predictor_specs.items():
    predictor_values, source_object = retrieve_existing_predictor(predictor_name)
    domain_5_plan_predictors[display_name] = predictor_values.reindex(route_index).copy()
    domain_5_plan_predictor_source_rows.append({'Plan measure': display_name, 'Predictor': predictor_name,
        'Source object': source_object, 'Non-missing': int(predictor_values.notna().sum()), 'Observed categories': int(predictor_values.nunique())})
domain_5_plan_predictor_source_summary = pd.DataFrame(domain_5_plan_predictor_source_rows)
route_guidance_plan_overlap_rows = []
for profile_name, profile_values in route_guidance_profile_specs.items():
    for plan_name, plan_values in domain_5_plan_predictors.items():
        comparison_data = pd.DataFrame({'Guidance profile': profile_values, 'Plan measure': plan_values},
            index=route_index).dropna()
        route_guidance_plan_overlap_rows.append({'Route-specific guidance profile': profile_name,
            'Plan measure': plan_name, 'Complete comparisons': int(len(comparison_data)), 'Guidance-profile categories': int(comparison_data['Guidance profile'].nunique()), 'Plan-measure categories': int(comparison_data['Plan measure'].nunique()), "Cramer's V": round(categorical_cramers_v(comparison_data['Guidance profile'],
            comparison_data['Plan measure']), 3)})
route_guidance_plan_overlap_summary = pd.DataFrame(route_guidance_plan_overlap_rows).sort_values(['Route-specific guidance profile',
    "Cramer's V"], ascending=[True, False]).reset_index(drop=True)
route_guidance_expected_route_crosstabs = {}
expected_route_values = domain_5_plan_predictors['Expected post-16 route']
for profile_name, profile_values in route_guidance_profile_specs.items():
    comparison_data = pd.DataFrame({'Guidance profile': profile_values.map(route_guidance_profile_labels),
        'Expected post-16 route': expected_route_values.astype('string')}, index=route_index).dropna()
    route_guidance_expected_route_crosstabs[profile_name] = pd.crosstab(comparison_data['Guidance profile'],
        comparison_data['Expected post-16 route'], margins=True, dropna=False)
route_guidance_profile_review_overview = pd.DataFrame([{'Route-specific source variables represented': 42,
    'Provisional profiles constructed': 2, 'Predictors retained in this cell': 0, 'Outcome columns loaded': 0, 'Files written': 0}])
print('Route-specific profile review overview:')
display_limited(route_guidance_profile_review_overview)
print('Profile-construction audit:')
display_limited(route_guidance_profile_audit_summary)
print('Provisional profile distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(route_guidance_profile_distribution)
print('Relationship between route-specific profiles:')
display_limited(route_guidance_profile_relationship)
print('Route-specific profile cross-tabulation:')
display_limited(route_guidance_profile_crosstab)
print('Domain 5 predictor retrieval:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_5_plan_predictor_source_summary)
print('Overlap with retained post-16 plan predictors:')
display_limited(route_guidance_plan_overlap_summary)
print('Guidance-profile cross-tabulations with expected post-16 route:')
for profile_name, comparison_table in route_guidance_expected_route_crosstabs.items():
    print(profile_name)
    display_limited(comparison_table)
print('Files written in this cell: 0')

Route-specific profile review overview:


,Route-specific source variables represented,Provisional profiles constructed,Predictors retained in this cell,Outcome columns loaded,Files written
0,42,2,0,0,0


Profile-construction audit:


,Route-specific profile,Valid discussion status,No discussion,Discussion present,Discussion with complete downstream responses,Influence reported without recommendation,Profile non-missing,Profile missing
0,Stay-on guidance,9441,1642,7799,7784,265,9426,341
1,Apprenticeship guidance,9441,6056,3385,3344,113,9400,367


Provisional profile distributions:


...

Guidance-profile cross-tabulations with expected post-16 route:
Stay-on guidance


Expected post-16 route,Full-time employment,Further-education college,Other or unspecified full-time education,"Other, mixed or uncertain route",School sixth form,Sixth-form college,Work-based training or employment-training,All
Guidance profile,,,,,,,,
Discussion without reported recommendation or influence,89,283,46,64,412,163,200,1257
No route discussion,71,307,72,38,717,313,123,1641
Recommendation reported without reported influence,71,849,147,28,1632,756,94,3577
Reported influence on decision,22,804,166,24,1254,594,77,2941
All,253,2243,431,154,4015,1826,494,9416


Apprenticeship guidance


Expected post-16 route,Full-time employment,Further-education college,Other or unspecified full-time education,"Other, mixed or uncertain route",School sixth form,Sixth-form college,Work-based training or employment-training,All
Guidance profile,,,,,,,,
Discussion without reported recommendation or influence,41,502,69,35,604,348,79,1678
No route discussion,98,1184,243,58,3108,1271,88,6050
Recommendation reported without reported influence,54,210,43,29,145,90,141,712
Reported influence on decision,61,335,76,32,151,112,183,950
All,254,2231,431,154,4008,1821,491,9390


Files written in this cell: 0


In [404]:
# 18: General institutional-guidance item and scale review

import itertools
import pandas as pd
from scipy.stats import pearsonr, spearmanr
general_guidance_addition_variables = {'W1infofrnYP', 'W1infoteYP', 'W2AdvCASYP', 'W2ModAp3YP0c', 'W3mentor1YP',
    'W3mentor2YP'}
general_guidance_addition_inventory = guidance_addition_review_inventory.loc[guidance_addition_review_inventory['Variable'].isin(general_guidance_addition_variables)].copy().sort_values(['Wave',
    'Variable position']).reset_index(drop=True)
assert len(general_guidance_addition_inventory) == 6
assert set(general_guidance_addition_inventory['Variable']) == general_guidance_addition_variables
assert set(general_guidance_addition_inventory['Addition review status']) == {'Domain 10 substantive review'}
general_guidance_raw = guidance_raw.copy()
general_guidance_labelled = guidance_labelled.copy()
for source_file, source_inventory in general_guidance_addition_inventory.groupby('Source file', sort=False):
    source_variables = source_inventory['Variable'].tolist()
    source_path = source_file_lookup[source_file]
    raw_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=False)
    labelled_source = pd.read_stata(source_path, columns=['NSID', *source_variables], convert_categoricals=True)
    for source_data in [raw_source, labelled_source]:
        source_data['NSID'] = standardise_nsid(source_data['NSID'])
        assert source_data['NSID'].is_unique
    raw_source = raw_source.set_index('NSID').reindex(route_index)
    labelled_source = labelled_source.set_index('NSID').reindex(route_index)
    for variable in source_variables:
        general_guidance_raw[variable] = pd.to_numeric(raw_source[variable], errors='coerce')
        general_guidance_labelled[variable] = labelled_source[variable].astype('string')
assert general_guidance_addition_variables.issubset(general_guidance_raw.columns)
learning_mentor_variables = ['W3mentor1YP', 'W3mentor2YP']
learning_mentor_unrestricted = general_guidance_raw[learning_mentor_variables].copy()
general_guidance_raw.loc[~wave_3_permitted_mask, learning_mentor_variables] = pd.NA
general_guidance_labelled.loc[~wave_3_permitted_mask, learning_mentor_variables] = pd.NA
general_guidance_addition_coverage_rows = []
for _, inventory_row in general_guidance_addition_inventory.iterrows():
    variable = inventory_row['Variable']
    values = general_guidance_raw[variable]
    observed_mask = values.ge(0).fillna(False).astype(bool)
    special_code_mask = values.lt(0).fillna(False).astype(bool)
    if variable in learning_mentor_variables:
        removed_by_timing = int((learning_mentor_unrestricted[variable].notna() & ~wave_3_permitted_mask).sum())
    else:
        removed_by_timing = 0
    general_guidance_addition_coverage_rows.append({'Wave': inventory_row['Wave'], 'Variable': variable,
        'Guidance audit group': inventory_row['Guidance audit group'], 'Observed responses': int(observed_mask.sum()), 'Observed percentage': round(observed_mask.mean() * 100,
        2), 'Special-code responses': int(special_code_mask.sum()), 'Unavailable responses': int(values.isna().sum()), 'Values removed by timing restriction': removed_by_timing, 'Observed categories': int(values.where(observed_mask).nunique())})
general_guidance_addition_coverage_summary = pd.DataFrame(general_guidance_addition_coverage_rows)
general_guidance_addition_code_rows = []
for _, inventory_row in general_guidance_addition_inventory.iterrows():
    variable = inventory_row['Variable']
    raw_values = general_guidance_raw[variable]
    labelled_values = general_guidance_labelled[variable]
    for raw_code, participants in raw_values.value_counts(dropna=False).items():
        if pd.isna(raw_code):
            value_label = 'Unavailable'
            response_type = 'Unavailable'
            sort_value = 999999
        else:
            matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            response_type = 'Observed response' if raw_code >= 0 else 'Special code'
            sort_value = float(raw_code)
        general_guidance_addition_code_rows.append({'Wave': inventory_row['Wave'], 'Variable': variable,
            'Guidance audit group': inventory_row['Guidance audit group'], 'Raw code': raw_code, 'Value label': value_label, 'Response type': response_type, 'Participants': int(participants), 'Sort value': sort_value})
general_guidance_addition_code_distribution = pd.DataFrame(general_guidance_addition_code_rows).sort_values(['Variable',
    'Sort value']).drop(columns=['Sort value']).reset_index(drop=True)
general_guidance_routing_specs = {'Lesson discussion frequency and usefulness': ('W1advfrsYP', 'W1infofrnYP'),
    'Teacher discussion frequency and usefulness': ('W1advteacYP',
    'W1infoteYP'), 'Learning-mentor provision and use': ('W3mentor1YP', 'W3mentor2YP')}
general_guidance_routing_crosstabs = {}
for comparison_name, (first_variable, second_variable) in general_guidance_routing_specs.items():
    general_guidance_routing_crosstabs[comparison_name] = pd.crosstab(general_guidance_labelled[first_variable].fillna('Unavailable'),
        general_guidance_labelled[second_variable].fillna('Unavailable'), margins=True, dropna=False)
apprenticeship_source_variables = ['W2ModAp3YP0c', 'W2ModAp3YP0d', 'W2ModAp3YP0e']
apprenticeship_source_data = general_guidance_raw[apprenticeship_source_variables].copy()
apprenticeship_source_complete_mask = apprenticeship_source_data.isin([0, 1]).all(axis=1)
apprenticeship_source_complete = apprenticeship_source_data.loc[apprenticeship_source_complete_mask].astype(int)
apprenticeship_source_pattern_summary = apprenticeship_source_complete.value_counts().rename('Participants').reset_index().sort_values('Participants',
    ascending=False).reset_index(drop=True)
apprenticeship_source_pattern_summary['Institutional sources mentioned'] = apprenticeship_source_pattern_summary[apprenticeship_source_variables].sum(axis=1)
apprenticeship_source_review_summary = pd.DataFrame([{'Full analysis sample': int(len(route_index)),
    'Complete routed responses': int(len(apprenticeship_source_complete)), 'At least one institutional source mentioned': int(apprenticeship_source_complete.sum(axis=1).gt(0).sum()), 'No institutional source mentioned': int(apprenticeship_source_complete.sum(axis=1).eq(0).sum()), 'Structurally inapplicable or unavailable': int((~apprenticeship_source_complete_mask).sum())}])
wave_2_institutional_frequency_variables = ['W2AdvFrsYP', 'W2AdvTeacYP', 'W2AdvCASYP']
wave_2_institutional_frequency_items = pd.DataFrame({'Teacher guidance during lessons': general_guidance_raw['W2AdvFrsYP'].where(general_guidance_raw['W2AdvFrsYP'].between(1,
    5)).sub(1), 'Teacher guidance outside lessons': general_guidance_raw['W2AdvTeacYP'].where(general_guidance_raw['W2AdvTeacYP'].between(1,
    5)).sub(1), 'Careers Advice Service guidance': general_guidance_raw['W2AdvCASYP'].where(general_guidance_raw['W2AdvCASYP'].between(1,
    5)).sub(1)}, index=route_index).astype('Float64')
institutional_frequency_pairwise_rows = []
for first_item, second_item in itertools.combinations(wave_2_institutional_frequency_items.columns, 2):
    pair_data = wave_2_institutional_frequency_items[[first_item, second_item]].dropna().astype(float)
    institutional_frequency_pairwise_rows.append({'First item': first_item, 'Second item': second_item,
        'Complete comparisons': int(len(pair_data)), 'Pearson correlation': round(float(pearsonr(pair_data[first_item],
        pair_data[second_item]).statistic), 3), 'Spearman correlation': round(float(spearmanr(pair_data[first_item],
        pair_data[second_item]).statistic), 3)})
institutional_frequency_pairwise_summary = pd.DataFrame(institutional_frequency_pairwise_rows)

def calculate_complete_case_alpha(item_data, representation_name):
    complete_items = item_data.dropna().astype(float)
    item_number = int(complete_items.shape[1])
    item_variance_sum = float(complete_items.var(axis=0, ddof=1).sum())
    total_variance = float(complete_items.sum(axis=1).var(ddof=1))
    alpha = item_number / (item_number - 1) * (1 - item_variance_sum / total_variance)
    return {'Representation': representation_name, 'Items': item_number,
        'Complete responses': int(len(complete_items)), "Cronbach's alpha": round(alpha, 3)}
institutional_frequency_reliability_summary = pd.DataFrame([calculate_complete_case_alpha(wave_2_institutional_frequency_items[['Teacher guidance during lessons',
    'Teacher guidance outside lessons']], 'Two-item teacher-guidance frequency'), calculate_complete_case_alpha(wave_2_institutional_frequency_items,
    'Three-item institutional-guidance frequency')])
teacher_guidance_frequency_candidate = wave_2_institutional_frequency_items[['Teacher guidance during lessons',
    'Teacher guidance outside lessons']].mean(axis=1).where(wave_2_institutional_frequency_items[['Teacher guidance during lessons',
    'Teacher guidance outside lessons']].notna().all(axis=1)).astype('Float64').rename('teacher_guidance_frequency_candidate')
institutional_guidance_frequency_candidate = wave_2_institutional_frequency_items.mean(axis=1).where(wave_2_institutional_frequency_items.notna().all(axis=1)).astype('Float64').rename('institutional_guidance_frequency_candidate')
institutional_frequency_candidate_summary = pd.DataFrame([{'Candidate representation': 'Two-item teacher-guidance frequency',
    'Non-missing': int(teacher_guidance_frequency_candidate.notna().sum()), 'Missing': int(teacher_guidance_frequency_candidate.isna().sum()), 'Missing percentage': round(teacher_guidance_frequency_candidate.isna().mean() * 100,
    2), 'Distinct values': int(teacher_guidance_frequency_candidate.nunique()), 'Minimum': float(teacher_guidance_frequency_candidate.min()), 'Maximum': float(teacher_guidance_frequency_candidate.max())}, {'Candidate representation': 'Three-item institutional-guidance frequency',
    'Non-missing': int(institutional_guidance_frequency_candidate.notna().sum()), 'Missing': int(institutional_guidance_frequency_candidate.isna().sum()), 'Missing percentage': round(institutional_guidance_frequency_candidate.isna().mean() * 100,
    2), 'Distinct values': int(institutional_guidance_frequency_candidate.nunique()), 'Minimum': float(institutional_guidance_frequency_candidate.min()), 'Maximum': float(institutional_guidance_frequency_candidate.max())}])
general_guidance_review_overview = pd.DataFrame([{'Additional substantive variables reviewed': int(len(general_guidance_addition_inventory)),
    'Wave 1 additions': int(general_guidance_addition_inventory['Wave'].eq('Wave 1').sum()), 'Wave 2 additions': int(general_guidance_addition_inventory['Wave'].eq('Wave 2').sum()), 'Wave 3 additions': int(general_guidance_addition_inventory['Wave'].eq('Wave 3').sum()), 'Predictors retained in this cell': 0, 'Files written': 0}])
print('General guidance review overview:')
display_limited(general_guidance_review_overview)
print('Additional-item coverage:')
with pd.option_context('display.max_colwidth', None):
    display_limited(general_guidance_addition_coverage_summary)
print('Additional-item response codes:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(general_guidance_addition_code_distribution)
print('Routing cross-tabulations:')
for comparison_name, comparison_table in general_guidance_routing_crosstabs.items():
    print(comparison_name)
    display_limited(comparison_table)
print('Routed apprenticeship-source review:')
display_limited(apprenticeship_source_review_summary)
print('Routed apprenticeship-source patterns:')
display_limited(apprenticeship_source_pattern_summary)
print('Wave 2 institutional-guidance item associations:')
display_limited(institutional_frequency_pairwise_summary)
print('Wave 2 institutional-guidance reliability:')
display_limited(institutional_frequency_reliability_summary)
print('Provisional frequency-score coverage:')
display_limited(institutional_frequency_candidate_summary)
print('Files written in this cell: 0')

General guidance review overview:


,Additional substantive variables reviewed,Wave 1 additions,Wave 2 additions,Wave 3 additions,Predictors retained in this cell,Files written
0,6,2,2,2,0,0


Additional-item coverage:


,Wave,Variable,Guidance audit group,Observed responses,Observed percentage,Special-code responses,Unavailable responses,Values removed by timing restriction,Observed categories
0,Wave 1,W1infofrnYP,Lesson-based guidance usefulness,8135,83.29,1389,243,0,5
1,Wave 1,W1infoteYP,Teacher guidance usefulness,5613,57.47,3911,243,0,5
2,Wave 2,W2AdvCASYP,Careers Advice Service discussion frequency,9378,96.02,143,246,0,5
3,Wave 2,W2ModAp3YP0c,School careers-adviser apprenticeship discussion,1810,18.53,7711,246,0,2
4,Wave 3,W3mentor1YP,Learning-mentor provision or use,9090,93.07,405,272,14,2


Additional-item response codes:


...

Wave 2 institutional-guidance reliability:


,Representation,Items,Complete responses,Cronbach's alpha
0,Two-item teacher-guidance frequency,2,9409,0.586
1,Three-item institutional-guidance frequency,3,9349,0.555


Provisional frequency-score coverage:


,Candidate representation,Non-missing,Missing,Missing percentage,Distinct values,Minimum,Maximum
0,Two-item teacher-guidance frequency,9409,358,3.67,9,0.0,4.0
1,Three-item institutional-guidance frequency,9349,418,4.28,13,0.0,4.0


Files written in this cell: 0


In [405]:
# 19: General guidance candidate representation and overlap review

import itertools
import pandas as pd
from scipy.stats import spearmanr
teacher_guidance_frequency_review_candidate = teacher_guidance_frequency_candidate.copy().astype('Float64').rename('teacher_guidance_frequency_review_candidate')
careers_advice_service_guidance_frequency_candidate = general_guidance_raw['W2AdvCASYP'].where(general_guidance_raw['W2AdvCASYP'].between(1,
    5)).sub(1).astype('Float64').rename('careers_advice_service_guidance_frequency_candidate')
assert set(careers_advice_service_guidance_frequency_candidate.dropna().astype(float).unique()) == {0.0, 1.0, 2.0, 3.0,
    4.0}
connexions_contact_wave_1 = guidance_raw['W1advperYP'].map({1.0: 1, 2.0: 0}).astype('Int64')
connexions_contact_wave_2 = guidance_raw['W2advperYP'].map({1.0: 1, 2.0: 0}).astype('Int64')
connexions_contact_wave_3 = guidance_raw['W3advperYP'].map({1.0: 1, 2.0: 0}).astype('Int64')
connexions_adviser_contact_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='connexions_adviser_contact_candidate')
connexions_contact_source = pd.Series(pd.NA, index=route_index, dtype='string', name='Connexions contact source')
wave_3_contact_valid = connexions_contact_wave_3.notna()
connexions_adviser_contact_candidate.loc[wave_3_contact_valid] = connexions_contact_wave_3.loc[wave_3_contact_valid]
connexions_contact_source.loc[wave_3_contact_valid] = 'Wave 3 ever-contact response'
prior_contact_yes = connexions_contact_wave_1.eq(1).fillna(False) | connexions_contact_wave_2.eq(1).fillna(False)
prior_contact_observed = connexions_contact_wave_1.notna() | connexions_contact_wave_2.notna()
prior_contact_fallback_mask = ~wave_3_contact_valid & prior_contact_observed
connexions_adviser_contact_candidate.loc[prior_contact_fallback_mask & prior_contact_yes] = 1
connexions_adviser_contact_candidate.loc[prior_contact_fallback_mask & ~prior_contact_yes] = 0
connexions_contact_source.loc[prior_contact_fallback_mask] = 'Wave 1–2 cumulative fallback'
assert set(connexions_adviser_contact_candidate.dropna().astype(int).unique()).issubset({0, 1})
connexions_contact_consistency_summary = pd.DataFrame([{'Wave 3 valid responses': int(wave_3_contact_valid.sum()),
    'Wave 1–2 fallback responses': int(prior_contact_fallback_mask.sum()), 'No usable contact information': int(connexions_adviser_contact_candidate.isna().sum()), 'Wave 3 no but earlier contact reported': int((connexions_contact_wave_3.eq(0).fillna(False) & prior_contact_yes).sum()), 'Wave 3 yes with no earlier positive report': int((connexions_contact_wave_3.eq(1).fillna(False) & ~prior_contact_yes).sum())}])
connexions_contact_source_summary = connexions_contact_source.value_counts(dropna=False).rename('Participants').rename_axis('Source').reset_index()
connexions_contact_source_summary['Percentage of full sample'] = (connexions_contact_source_summary['Participants'] / len(route_index) * 100).round(2)
learning_mentor_provision = general_guidance_raw['W3mentor1YP']
learning_mentor_use = general_guidance_raw['W3mentor2YP']
learning_mentor_engagement_profile_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='learning_mentor_engagement_profile_candidate')
learning_mentor_engagement_profile_candidate.loc[learning_mentor_provision.eq(2)] = 0
learning_mentor_engagement_profile_candidate.loc[learning_mentor_provision.eq(1) & learning_mentor_use.eq(2)] = 1
learning_mentor_engagement_profile_candidate.loc[learning_mentor_provision.eq(1) & learning_mentor_use.eq(1)] = 2
assert set(learning_mentor_engagement_profile_candidate.dropna().astype(int).unique()) == {0, 1, 2}
general_guidance_candidate_specs = {'Teacher guidance frequency': teacher_guidance_frequency_review_candidate,
    'Careers Advice Service guidance frequency': careers_advice_service_guidance_frequency_candidate, 'Connexions adviser contact': connexions_adviser_contact_candidate, 'Learning-mentor engagement': learning_mentor_engagement_profile_candidate}
general_guidance_candidate_data = pd.DataFrame(general_guidance_candidate_specs, index=route_index)
general_guidance_candidate_summary_rows = []
for candidate_name, candidate_values in general_guidance_candidate_specs.items():
    observed_counts = candidate_values.dropna().value_counts()
    general_guidance_candidate_summary_rows.append({'Candidate representation': candidate_name,
        'Non-missing': int(candidate_values.notna().sum()), 'Missing': int(candidate_values.isna().sum()), 'Missing percentage': round(candidate_values.isna().mean() * 100,
        2), 'Observed categories or values': int(candidate_values.nunique()), 'Smallest observed group': int(observed_counts.min()), 'Largest observed group': int(observed_counts.max()), 'Minimum': float(candidate_values.min()), 'Maximum': float(candidate_values.max())})
general_guidance_candidate_summary = pd.DataFrame(general_guidance_candidate_summary_rows)
general_guidance_candidate_label_maps = {'Teacher guidance frequency': {0.0: '0.0', 0.5: '0.5', 1.0: '1.0', 1.5: '1.5',
    2.0: '2.0', 2.5: '2.5', 3.0: '3.0', 3.5: '3.5', 4.0: '4.0'}, 'Careers Advice Service guidance frequency': {0.0: 'Not at all',
    1.0: 'Not very often', 2.0: 'A little', 3.0: 'Quite a lot', 4.0: 'A lot'}, 'Connexions adviser contact': {0: 'No adviser contact reported',
    1: 'Adviser contact reported'}, 'Learning-mentor engagement': {0: 'No learning-mentor provision reported',
    1: 'Provision reported but mentor not used', 2: 'Worked with a learning mentor'}}
general_guidance_candidate_distribution_rows = []
for candidate_name, candidate_values in general_guidance_candidate_specs.items():
    observed_total = int(candidate_values.notna().sum())
    for candidate_code, participants in candidate_values.value_counts(dropna=False).sort_index().items():
        if pd.isna(candidate_code):
            candidate_label = 'Unavailable'
            observed_percentage = pd.NA
        else:
            candidate_label = general_guidance_candidate_label_maps[candidate_name].get(candidate_code,
                str(candidate_code))
            observed_percentage = round(participants / observed_total * 100, 2)
        general_guidance_candidate_distribution_rows.append({'Candidate representation': candidate_name,
            'Candidate code': candidate_code, 'Candidate label': candidate_label, 'Participants': int(participants), 'Percentage of observed candidate': observed_percentage})
general_guidance_candidate_distribution = pd.DataFrame(general_guidance_candidate_distribution_rows)
general_guidance_pairwise_rows = []
for first_candidate, second_candidate in itertools.combinations(general_guidance_candidate_specs, 2):
    pair_data = general_guidance_candidate_data[[first_candidate, second_candidate]].dropna().astype(float)
    general_guidance_pairwise_rows.append({'First candidate': first_candidate, 'Second candidate': second_candidate,
        'Complete comparisons': int(len(pair_data)), 'Spearman correlation': round(float(spearmanr(pair_data[first_candidate],
        pair_data[second_candidate]).statistic), 3), "Cramer's V": round(categorical_cramers_v(pair_data[first_candidate],
        pair_data[second_candidate]), 3)})
general_guidance_pairwise_summary = pd.DataFrame(general_guidance_pairwise_rows)
general_guidance_existing_predictor_specs = {'School attitude score': 'school_attitude_score',
    'Teacher listening and fair-treatment score': 'teacher_listening_fair_treatment_score_pretransition', 'School physical-environment perception': 'school_physical_environment_perception_pretransition', 'Academic self-concept score': 'academic_self_concept_score', 'Truancy status': 'truancy_status_pretransition'}
general_guidance_existing_predictors = {}
general_guidance_existing_source_rows = []
for display_name, predictor_name in general_guidance_existing_predictor_specs.items():
    predictor_values, source_object = retrieve_existing_predictor(predictor_name)
    general_guidance_existing_predictors[display_name] = predictor_values.reindex(route_index).copy()
    general_guidance_existing_source_rows.append({'Comparison measure': display_name, 'Predictor': predictor_name,
        'Source object': source_object, 'Non-missing': int(predictor_values.notna().sum())})
general_guidance_existing_source_summary = pd.DataFrame(general_guidance_existing_source_rows)
general_guidance_school_overlap_rows = []
for candidate_name, candidate_values in general_guidance_candidate_specs.items():
    for comparison_name, comparison_values in general_guidance_existing_predictors.items():
        pair_data = pd.DataFrame({'Guidance candidate': candidate_values, 'Existing predictor': comparison_values},
            index=route_index).dropna().astype(float)
        general_guidance_school_overlap_rows.append({'Guidance candidate': candidate_name,
            'Existing predictor': comparison_name, 'Complete comparisons': int(len(pair_data)), 'Spearman correlation': round(float(spearmanr(pair_data['Guidance candidate'],
            pair_data['Existing predictor']).statistic), 3)})
general_guidance_school_overlap_summary = pd.DataFrame(general_guidance_school_overlap_rows).sort_values(['Guidance candidate',
    'Spearman correlation'], ascending=[True, False]).reset_index(drop=True)
general_guidance_plan_overlap_rows = []
for candidate_name, candidate_values in general_guidance_candidate_specs.items():
    for plan_name, plan_values in domain_5_plan_predictors.items():
        pair_data = pd.DataFrame({'Guidance candidate': candidate_values, 'Plan measure': plan_values},
            index=route_index).dropna()
        general_guidance_plan_overlap_rows.append({'Guidance candidate': candidate_name, 'Plan measure': plan_name,
            'Complete comparisons': int(len(pair_data)), "Cramer's V": round(categorical_cramers_v(pair_data['Guidance candidate'],
            pair_data['Plan measure']), 3)})
general_guidance_plan_overlap_summary = pd.DataFrame(general_guidance_plan_overlap_rows).sort_values(['Guidance candidate',
    "Cramer's V"], ascending=[True, False]).reset_index(drop=True)
route_specific_guidance_specs = {'Stay-on guidance profile': stay_on_guidance_profile_candidate,
    'Apprenticeship guidance profile': apprenticeship_guidance_profile_candidate}
general_route_guidance_overlap_rows = []
for candidate_name, candidate_values in general_guidance_candidate_specs.items():
    for route_profile_name, route_profile_values in route_specific_guidance_specs.items():
        pair_data = pd.DataFrame({'General guidance candidate': candidate_values,
            'Route-specific profile': route_profile_values}, index=route_index).dropna()
        general_route_guidance_overlap_rows.append({'General guidance candidate': candidate_name,
            'Route-specific profile': route_profile_name, 'Complete comparisons': int(len(pair_data)), "Cramer's V": round(categorical_cramers_v(pair_data['General guidance candidate'],
            pair_data['Route-specific profile']), 3)})
general_route_guidance_overlap_summary = pd.DataFrame(general_route_guidance_overlap_rows).sort_values(['General guidance candidate',
    "Cramer's V"], ascending=[True, False]).reset_index(drop=True)
general_guidance_candidate_review_overview = pd.DataFrame([{'General guidance candidates reviewed': int(len(general_guidance_candidate_specs)),
    'Existing school-experience comparisons': int(len(general_guidance_existing_predictors)), 'Domain 5 plan comparisons': int(len(domain_5_plan_predictors)), 'Route-specific profile comparisons': int(len(route_specific_guidance_specs)), 'Predictors retained in this cell': 0, 'Outcome columns loaded': 0, 'Files written': 0}])
print('General-guidance candidate review overview:')
display_limited(general_guidance_candidate_review_overview)
print('Candidate coverage:')
display_limited(general_guidance_candidate_summary)
print('Candidate distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(general_guidance_candidate_distribution)
print('Connexions contact construction audit:')
display_limited(connexions_contact_consistency_summary)
print('Connexions contact source use:')
display_limited(connexions_contact_source_summary)
print('Relationships among general-guidance candidates:')
display_limited(general_guidance_pairwise_summary)
print('Existing predictor retrieval:')
with pd.option_context('display.max_colwidth', None):
    display_limited(general_guidance_existing_source_summary)
print('Overlap with retained school-experience predictors:')
display_limited(general_guidance_school_overlap_summary)
print('Overlap with retained post-16 plan predictors:')
display_limited(general_guidance_plan_overlap_summary)
print('Overlap with provisional route-specific guidance profiles:')
display_limited(general_route_guidance_overlap_summary)
print('Files written in this cell: 0')

General-guidance candidate review overview:


,General guidance candidates reviewed,Existing school-experience comparisons,Domain 5 plan comparisons,Route-specific profile comparisons,Predictors retained in this cell,Outcome columns loaded,Files written
0,4,5,3,2,0,0,0


Candidate coverage:


,Candidate representation,Non-missing,Missing,Missing percentage,Observed categories or values,Smallest observed group,Largest observed group,Minimum,Maximum
0,Teacher guidance frequency,9409,358,3.67,9,32,2334,0.0,4.0
1,Careers Advice Service guidance frequency,9378,389,3.98,5,86,5701,0.0,4.0
2,Connexions adviser contact,9519,248,2.54,2,2992,6527,0.0,1.0
3,Learning-mentor engagement,9073,694,7.11,3,1136,5899,0.0,2.0


Candidate distributions:


...

Overlap with retained post-16 plan predictors:


,Guidance candidate,Plan measure,Complete comparisons,Cramer's V
0,Careers Advice Service guidance frequency,Expected peer post-16 route,9325,0.032
1,Careers Advice Service guidance frequency,Expected post-16 route,9369,0.027
2,Careers Advice Service guidance frequency,Higher-education application likelihood,9365,0.025
3,Connexions adviser contact,Expected post-16 route,9504,0.124
4,Connexions adviser contact,Higher-education application likelihood,9504,0.075


Overlap with provisional route-specific guidance profiles:


,General guidance candidate,Route-specific profile,Complete comparisons,Cramer's V
0,Careers Advice Service guidance frequency,Stay-on guidance profile,9300,0.071
1,Careers Advice Service guidance frequency,Apprenticeship guidance profile,9271,0.065
2,Connexions adviser contact,Stay-on guidance profile,9424,0.259
3,Connexions adviser contact,Apprenticeship guidance profile,9398,0.190
4,Learning-mentor engagement,Apprenticeship guidance profile,9036,0.098


Files written in this cell: 0


In [406]:
# 20: Connexions adviser-contact representation review

import pandas as pd
connexions_direct_contact_items = pd.DataFrame({'Wave 1': guidance_raw['W1advperYP'].map({1.0: 1, 2.0: 0}),
    'Wave 2': guidance_raw['W2advperYP'].map({1.0: 1, 2.0: 0}), 'Wave 3': guidance_raw['W3advperYP'].map({1.0: 1,
    2.0: 0})}, index=route_index).astype('Int64')
connexions_latest_response_candidate = connexions_adviser_contact_candidate.copy().astype('Int64').rename('connexions_latest_response_candidate')
connexions_any_contact_reported = connexions_direct_contact_items.eq(1).any(axis=1)
connexions_any_contact_observed = connexions_direct_contact_items.notna().any(axis=1)
connexions_cumulative_contact_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='connexions_cumulative_contact_candidate')
connexions_cumulative_contact_candidate.loc[connexions_any_contact_observed & ~connexions_any_contact_reported] = 0
connexions_cumulative_contact_candidate.loc[connexions_any_contact_reported] = 1
assert set(connexions_cumulative_contact_candidate.dropna().astype(int).unique()) == {0, 1}
connexions_contact_timing_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='connexions_contact_timing_candidate')
no_contact_reported_mask = connexions_any_contact_observed & ~connexions_any_contact_reported
wave_1_contact_mask = connexions_direct_contact_items['Wave 1'].eq(1).fillna(False)
wave_2_first_report_mask = ~wave_1_contact_mask & connexions_direct_contact_items['Wave 2'].eq(1).fillna(False)
wave_3_first_report_mask = ~wave_1_contact_mask & ~wave_2_first_report_mask & connexions_direct_contact_items['Wave 3'].eq(1).fillna(False)
connexions_contact_timing_candidate.loc[no_contact_reported_mask] = 0
connexions_contact_timing_candidate.loc[wave_1_contact_mask] = 1
connexions_contact_timing_candidate.loc[wave_2_first_report_mask] = 2
connexions_contact_timing_candidate.loc[wave_3_first_report_mask] = 3
assert set(connexions_contact_timing_candidate.dropna().astype(int).unique()) == {0, 1, 2, 3}
assert connexions_contact_timing_candidate.notna().equals(connexions_cumulative_contact_candidate.notna())
assert connexions_contact_timing_candidate.gt(0).where(connexions_contact_timing_candidate.notna()).astype('Int64').equals(connexions_cumulative_contact_candidate)
connexions_contact_pattern_data = connexions_direct_contact_items.astype('string').fillna('Unavailable')
connexions_contact_pattern_summary = connexions_contact_pattern_data.value_counts().rename('Participants').reset_index().sort_values('Participants',
    ascending=False).reset_index(drop=True)
connexions_contact_pattern_summary['Percentage of full sample'] = (connexions_contact_pattern_summary['Participants'] / len(route_index) * 100).round(2)
connexions_representation_comparison_data = pd.DataFrame({'Latest-response representation': connexions_latest_response_candidate,
    'Cumulative representation': connexions_cumulative_contact_candidate, 'Contact timing profile': connexions_contact_timing_candidate}, index=route_index)
connexions_binary_complete = connexions_representation_comparison_data[['Latest-response representation',
    'Cumulative representation']].dropna().astype(int)
connexions_representation_agreement_summary = pd.DataFrame([{'Complete binary comparisons': int(len(connexions_binary_complete)),
    'Exact agreement': int(connexions_binary_complete['Latest-response representation'].eq(connexions_binary_complete['Cumulative representation']).sum()), 'Exact agreement percentage': round(connexions_binary_complete['Latest-response representation'].eq(connexions_binary_complete['Cumulative representation']).mean() * 100,
    2), 'Latest response no but cumulative contact yes': int((connexions_binary_complete['Latest-response representation'].eq(0) & connexions_binary_complete['Cumulative representation'].eq(1)).sum()), 'Latest response yes but cumulative contact no': int((connexions_binary_complete['Latest-response representation'].eq(1) & connexions_binary_complete['Cumulative representation'].eq(0)).sum()), "Cramer's V": round(categorical_cramers_v(connexions_binary_complete['Latest-response representation'],
    connexions_binary_complete['Cumulative representation']), 3)}])
connexions_representation_crosstab = pd.crosstab(connexions_binary_complete['Latest-response representation'],
    connexions_binary_complete['Cumulative representation'], margins=True, dropna=False)
connexions_representation_specs = {'Latest-response binary': connexions_latest_response_candidate,
    'Cumulative any-wave binary': connexions_cumulative_contact_candidate, 'Earliest-reported-contact profile': connexions_contact_timing_candidate}
connexions_representation_summary_rows = []
connexions_representation_distribution_rows = []
connexions_contact_timing_labels = {0: 'No contact reported in any observed wave', 1: 'Contact reported by Wave 1',
    2: 'Contact first reported at Wave 2', 3: 'Contact first reported at Wave 3'}
for representation_name, representation_values in connexions_representation_specs.items():
    observed_counts = representation_values.dropna().value_counts()
    connexions_representation_summary_rows.append({'Representation': representation_name,
        'Non-missing': int(representation_values.notna().sum()), 'Missing': int(representation_values.isna().sum()), 'Missing percentage': round(representation_values.isna().mean() * 100,
        2), 'Categories': int(representation_values.nunique()), 'Smallest observed category': int(observed_counts.min()), 'Largest observed category': int(observed_counts.max())})
    observed_total = int(representation_values.notna().sum())
    for category_code, participants in representation_values.value_counts(dropna=False).sort_index().items():
        if pd.isna(category_code):
            category_label = 'Unavailable'
            observed_percentage = pd.NA
        elif representation_name == 'Earliest-reported-contact profile':
            category_label = connexions_contact_timing_labels[int(category_code)]
            observed_percentage = round(participants / observed_total * 100, 2)
        else:
            category_label = 'Adviser contact reported' if int(category_code) == 1 else 'No adviser contact reported'
            observed_percentage = round(participants / observed_total * 100, 2)
        connexions_representation_distribution_rows.append({'Representation': representation_name,
            'Category code': category_code, 'Category label': category_label, 'Participants': int(participants), 'Percentage of observed representation': observed_percentage})
connexions_representation_summary = pd.DataFrame(connexions_representation_summary_rows)
connexions_representation_distribution = pd.DataFrame(connexions_representation_distribution_rows)
connexions_general_guidance_overlap_rows = []
for representation_name, representation_values in {'Cumulative any-wave contact': connexions_cumulative_contact_candidate,
    'Earliest-reported-contact profile': connexions_contact_timing_candidate}.items():
    for comparison_name, comparison_values in {'Teacher guidance frequency': teacher_guidance_frequency_review_candidate,
        'Careers Advice Service guidance frequency': careers_advice_service_guidance_frequency_candidate, 'Learning-mentor engagement': learning_mentor_engagement_profile_candidate}.items():
        pair_data = pd.DataFrame({'Connexions representation': representation_values,
            'Comparison measure': comparison_values}, index=route_index).dropna()
        connexions_general_guidance_overlap_rows.append({'Connexions representation': representation_name,
            'Comparison measure': comparison_name, 'Complete comparisons': int(len(pair_data)), "Cramer's V": round(categorical_cramers_v(pair_data['Connexions representation'],
            pair_data['Comparison measure']), 3)})
connexions_general_guidance_overlap_summary = pd.DataFrame(connexions_general_guidance_overlap_rows)
connexions_plan_overlap_rows = []
for representation_name, representation_values in {'Cumulative any-wave contact': connexions_cumulative_contact_candidate,
    'Earliest-reported-contact profile': connexions_contact_timing_candidate}.items():
    for plan_name, plan_values in domain_5_plan_predictors.items():
        pair_data = pd.DataFrame({'Connexions representation': representation_values, 'Plan measure': plan_values},
            index=route_index).dropna()
        connexions_plan_overlap_rows.append({'Connexions representation': representation_name,
            'Plan measure': plan_name, 'Complete comparisons': int(len(pair_data)), "Cramer's V": round(categorical_cramers_v(pair_data['Connexions representation'],
            pair_data['Plan measure']), 3)})
connexions_plan_overlap_summary = pd.DataFrame(connexions_plan_overlap_rows)
connexions_route_profile_overlap_rows = []
for representation_name, representation_values in {'Cumulative any-wave contact': connexions_cumulative_contact_candidate,
    'Earliest-reported-contact profile': connexions_contact_timing_candidate}.items():
    for route_profile_name, route_profile_values in route_specific_guidance_specs.items():
        pair_data = pd.DataFrame({'Connexions representation': representation_values,
            'Route-specific profile': route_profile_values}, index=route_index).dropna()
        connexions_route_profile_overlap_rows.append({'Connexions representation': representation_name,
            'Route-specific profile': route_profile_name, 'Complete comparisons': int(len(pair_data)), "Cramer's V": round(categorical_cramers_v(pair_data['Connexions representation'],
            pair_data['Route-specific profile']), 3)})
connexions_route_profile_overlap_summary = pd.DataFrame(connexions_route_profile_overlap_rows)
connexions_representation_review_overview = pd.DataFrame([{'Direct contact waves reviewed': 3,
    'Representations compared': int(len(connexions_representation_specs)), 'Predictors retained in this cell': 0, 'Outcome columns loaded': 0, 'Files written': 0}])
print('Connexions representation review overview:')
display_limited(connexions_representation_review_overview)
print('Three-wave direct-contact patterns:')
with pd.option_context('display.max_rows', None):
    display_limited(connexions_contact_pattern_summary)
print('Representation coverage:')
display_limited(connexions_representation_summary)
print('Representation distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(connexions_representation_distribution)
print('Latest-response and cumulative agreement:')
display_limited(connexions_representation_agreement_summary)
print('Latest-response and cumulative cross-tabulation:')
display_limited(connexions_representation_crosstab)
print('Overlap with other general-guidance candidates:')
display_limited(connexions_general_guidance_overlap_summary)
print('Overlap with retained post-16 plan predictors:')
display_limited(connexions_plan_overlap_summary)
print('Overlap with provisional route-specific guidance profiles:')
display_limited(connexions_route_profile_overlap_summary)
print('Files written in this cell: 0')

Connexions representation review overview:


,Direct contact waves reviewed,Representations compared,Predictors retained in this cell,Outcome columns loaded,Files written
0,3,3,0,0,0


Three-wave direct-contact patterns:


,Wave 1,Wave 2,Wave 3,Participants,Percentage of full sample
0,0,0,1,1879,19.24
1,1,1,1,1520,15.56
2,0,0,0,1295,13.26
3,0,1,1,1265,12.95
4,1,0,1,767,7.85


Representation coverage:


...

Overlap with retained post-16 plan predictors:


,Connexions representation,Plan measure,Complete comparisons,Cramer's V
0,Cumulative any-wave contact,Expected post-16 route,9504,0.107
1,Cumulative any-wave contact,Higher-education application likelihood,9504,0.053
2,Cumulative any-wave contact,Expected peer post-16 route,9458,0.032
3,Earliest-reported-contact profile,Expected post-16 route,9504,0.068
4,Earliest-reported-contact profile,Higher-education application likelihood,9504,0.041


Overlap with provisional route-specific guidance profiles:


,Connexions representation,Route-specific profile,Complete comparisons,Cramer's V
0,Cumulative any-wave contact,Stay-on guidance profile,9424,0.214
1,Cumulative any-wave contact,Apprenticeship guidance profile,9398,0.149
2,Earliest-reported-contact profile,Stay-on guidance profile,9424,0.126
3,Earliest-reported-contact profile,Apprenticeship guidance profile,9398,0.090


Files written in this cell: 0


In [407]:
# 21: Guidance predictor construction and representation decisions

import pandas as pd
teacher_guidance_frequency_index_pretransition = teacher_guidance_frequency_review_candidate.copy().astype('Float64').rename('teacher_guidance_frequency_index_pretransition')
careers_advice_service_guidance_frequency_pretransition = careers_advice_service_guidance_frequency_candidate.copy().astype('Float64').rename('careers_advice_service_guidance_frequency_pretransition')
connexions_adviser_contact_pretransition = connexions_cumulative_contact_candidate.copy().astype('Int64').rename('connexions_adviser_contact_pretransition')
learning_mentor_engagement_profile_pretransition = learning_mentor_engagement_profile_candidate.copy().astype('Int64').rename('learning_mentor_engagement_profile_pretransition')
stay_on_guidance_profile_pretransition = stay_on_guidance_profile_candidate.copy().astype('Int64').rename('stay_on_guidance_profile_pretransition')
apprenticeship_guidance_profile_pretransition = apprenticeship_guidance_profile_candidate.copy().astype('Int64').rename('apprenticeship_guidance_profile_pretransition')
guidance_predictor_names = ['teacher_guidance_frequency_index_pretransition',
    'careers_advice_service_guidance_frequency_pretransition', 'connexions_adviser_contact_pretransition', 'learning_mentor_engagement_profile_pretransition', 'stay_on_guidance_profile_pretransition', 'apprenticeship_guidance_profile_pretransition']
guidance_domain_predictors = pd.concat([teacher_guidance_frequency_index_pretransition,
    careers_advice_service_guidance_frequency_pretransition, connexions_adviser_contact_pretransition, learning_mentor_engagement_profile_pretransition, stay_on_guidance_profile_pretransition, apprenticeship_guidance_profile_pretransition], axis=1).reindex(route_index)
guidance_domain_predictors.index.name = 'NSID'
assert guidance_domain_predictors.shape == (9767, 6)
assert list(guidance_domain_predictors.columns) == guidance_predictor_names
assert set(teacher_guidance_frequency_index_pretransition.dropna().astype(float).unique()).issubset({0.0, 0.5, 1.0,
    1.5, 2.0, 2.5, 3.0, 3.5, 4.0})
assert set(careers_advice_service_guidance_frequency_pretransition.dropna().astype(float).unique()) == {0.0, 1.0, 2.0,
    3.0, 4.0}
assert set(connexions_adviser_contact_pretransition.dropna().astype(int).unique()) == {0, 1}
assert set(learning_mentor_engagement_profile_pretransition.dropna().astype(int).unique()) == {0, 1, 2}
for route_profile_name in ['stay_on_guidance_profile_pretransition', 'apprenticeship_guidance_profile_pretransition']:
    assert set(guidance_domain_predictors[route_profile_name].dropna().astype(int).unique()) == {0, 1, 2, 3}
guidance_predictor_summary_rows = []
for predictor_name in guidance_predictor_names:
    predictor_values = guidance_domain_predictors[predictor_name]
    observed_counts = predictor_values.dropna().value_counts()
    guidance_predictor_summary_rows.append({'Predictor': predictor_name,
        'Non-missing': int(predictor_values.notna().sum()), 'Missing': int(predictor_values.isna().sum()), 'Missing percentage': round(predictor_values.isna().mean() * 100,
        2), 'Observed categories or values': int(predictor_values.nunique()), 'Smallest observed group': int(observed_counts.min()), 'Largest observed group': int(observed_counts.max())})
guidance_predictor_summary = pd.DataFrame(guidance_predictor_summary_rows)
guidance_available_predictor_count = guidance_domain_predictors.notna().sum(axis=1)
guidance_joint_coverage_summary = guidance_available_predictor_count.value_counts().sort_index().rename('Participants').rename_axis('Guidance predictors available').reset_index()
guidance_joint_coverage_summary['Percentage of full sample'] = (guidance_joint_coverage_summary['Participants'] / len(guidance_domain_predictors) * 100).round(2)
guidance_category_definitions = pd.DataFrame([{'Predictor': 'teacher_guidance_frequency_index_pretransition',
    'Category code': pd.NA, 'Category label': 'Mean of two Wave 2 frequency items, ranging from 0 to 4'}, {'Predictor': 'careers_advice_service_guidance_frequency_pretransition',
    'Category code': 0, 'Category label': 'Not at all'}, {'Predictor': 'careers_advice_service_guidance_frequency_pretransition',
    'Category code': 1, 'Category label': 'Not very often'}, {'Predictor': 'careers_advice_service_guidance_frequency_pretransition',
    'Category code': 2, 'Category label': 'A little'}, {'Predictor': 'careers_advice_service_guidance_frequency_pretransition',
    'Category code': 3, 'Category label': 'Quite a lot'}, {'Predictor': 'careers_advice_service_guidance_frequency_pretransition',
    'Category code': 4, 'Category label': 'A lot'}, {'Predictor': 'connexions_adviser_contact_pretransition',
    'Category code': 0, 'Category label': 'No adviser contact reported in any observed wave'}, {'Predictor': 'connexions_adviser_contact_pretransition',
    'Category code': 1, 'Category label': 'Adviser contact reported in at least one observed wave'}, {'Predictor': 'learning_mentor_engagement_profile_pretransition',
    'Category code': 0, 'Category label': 'No learning-mentor provision reported'}, {'Predictor': 'learning_mentor_engagement_profile_pretransition',
    'Category code': 1, 'Category label': 'Provision reported but mentor not used'}, {'Predictor': 'learning_mentor_engagement_profile_pretransition',
    'Category code': 2, 'Category label': 'Worked with a learning mentor'}] + [{'Predictor': predictor_name,
    'Category code': category_code, 'Category label': category_label} for predictor_name in ['stay_on_guidance_profile_pretransition',
    'apprenticeship_guidance_profile_pretransition'] for category_code, category_label in route_guidance_profile_labels.items()])
guidance_representation_decisions = pd.DataFrame([{'Candidate representation': 'Two separate Wave 2 teacher-guidance frequency items',
    'Decision': 'Combine', 'Final predictor': 'teacher_guidance_frequency_index_pretransition', 'Reason': 'The two items represent guidance during and outside lessons. Their mean provides a compact frequency index; it is not interpreted as a latent psychometric scale.'}, {'Candidate representation': 'Three-item institutional-guidance frequency score',
    'Decision': 'Do not retain', 'Final predictor': pd.NA, 'Reason': 'Adding Careers Advice Service frequency reduced internal consistency and combined distinct providers.'}, {'Candidate representation': 'Careers Advice Service discussion frequency',
    'Decision': 'Retain', 'Final predictor': 'careers_advice_service_guidance_frequency_pretransition', 'Reason': 'The item has broad coverage and limited overlap with teacher guidance, school experience and post-16 plans.'}, {'Candidate representation': 'Latest Connexions adviser-contact response',
    'Decision': 'Do not retain', 'Final predictor': pd.NA, 'Reason': 'A later negative response would discard earlier positive contact reports for 1,174 participants.'}, {'Candidate representation': 'Any-wave Connexions adviser contact',
    'Decision': 'Retain', 'Final predictor': 'connexions_adviser_contact_pretransition', 'Reason': 'Any positive report is retained because adviser contact is a cumulative pre-transition experience.'}, {'Candidate representation': 'Earliest survey wave reporting Connexions contact',
    'Decision': 'Do not retain', 'Final predictor': pd.NA, 'Reason': 'The category would indicate first survey report rather than the verified timing of first adviser contact.'}, {'Candidate representation': 'Learning-mentor provision and use as separate items',
    'Decision': 'Combine', 'Final predictor': 'learning_mentor_engagement_profile_pretransition', 'Reason': 'Use is structurally conditional on provision, so a three-level profile preserves the routing structure.'}, {'Candidate representation': 'Individual Wave 3 stay-on guidance items',
    'Decision': 'Combine', 'Final predictor': 'stay_on_guidance_profile_pretransition', 'Reason': 'The discussion, recommendation and influence batteries are routed and route-specific; one profile represents their ordered substantive stages.'}, {'Candidate representation': 'Individual Wave 3 apprenticeship guidance items',
    'Decision': 'Combine', 'Final predictor': 'apprenticeship_guidance_profile_pretransition', 'Reason': 'The discussion, recommendation and influence batteries are routed and route-specific; one profile represents their ordered substantive stages.'}, {'Candidate representation': 'Wave 1 guidance-usefulness follow-up items',
    'Decision': 'Do not retain', 'Final predictor': pd.NA, 'Reason': 'Usefulness is only observed among participants who received information from the relevant source.'}, {'Candidate representation': 'Wave 2 apprenticeship-discussion source items',
    'Decision': 'Do not retain', 'Final predictor': pd.NA, 'Reason': 'The items are structurally restricted to a small route-specific subgroup and are narrower than the retained Wave 3 guidance profiles.'}])
guidance_construction_variables = {'W2AdvFrsYP', 'W2AdvTeacYP', 'W2AdvCASYP', 'W1advperYP', 'W2advperYP', 'W3advperYP',
    'W3mentor1YP', 'W3mentor2YP', *[f'W3tlkteacYP0{suffix}' for suffix in 'abcdefg'], *[f'W3adteac2YP0{suffix}' for suffix in 'abcdefg'], *[f'W3adteacYP0{suffix}' for suffix in 'abcdefg'], *[f'W3tlktappYP0{suffix}' for suffix in 'abcdefg'], *[f'W3adapp2YP0{suffix}' for suffix in 'abcdefg'], *[f'W3adappYP0{suffix}' for suffix in 'abcdefg']}
guidance_support_variables = {'W1advconYP', 'W2advconYP', 'W3advconYP', 'W1advconnYP', 'W2advconnYP', 'W1infoconYP',
    'W1advfrsYP', 'W1advteacYP', 'W1infofrnYP', 'W1infoteYP', 'W2ModAp3YP0c', 'W2ModAp3YP0d', 'W2ModAp3YP0e'}
guidance_exclude_variables = {'W1advfamYP', 'W1advpalYP', 'W1infofamYP', 'W2AdvFamYP', 'W2AdvPalYP', 'W2infon'}
assert not guidance_construction_variables & guidance_support_variables
assert not guidance_construction_variables & guidance_exclude_variables
assert not guidance_support_variables & guidance_exclude_variables
assert guidance_construction_variables | guidance_support_variables | guidance_exclude_variables == set(guidance_full_label_inventory['Variable'])
guidance_variable_decisions = guidance_full_label_inventory.copy()
guidance_variable_decisions['Domain 10 decision'] = pd.NA
guidance_variable_decisions['Domain 10 role'] = pd.NA
guidance_variable_decisions['Domain 10 reason'] = pd.NA
guidance_variable_decisions['Domain 10 timing assessment'] = guidance_variable_decisions['Wave'].map({'Wave 1': 'Pre-transition',
    'Wave 2': 'Pre-transition', 'Wave 3': 'Permitted pre-transition interview timing applied'})
construction_mask = guidance_variable_decisions['Variable'].isin(guidance_construction_variables)
support_mask = guidance_variable_decisions['Variable'].isin(guidance_support_variables)
exclude_mask = guidance_variable_decisions['Variable'].isin(guidance_exclude_variables)
guidance_variable_decisions.loc[construction_mask, 'Domain 10 decision'] = 'Construction input'
guidance_variable_decisions.loc[support_mask, 'Domain 10 decision'] = 'Review support'
guidance_variable_decisions.loc[exclude_mask, 'Domain 10 decision'] = 'Exclude'
teacher_index_mask = guidance_variable_decisions['Variable'].isin({'W2AdvFrsYP', 'W2AdvTeacYP'})
careers_frequency_mask = guidance_variable_decisions['Variable'].eq('W2AdvCASYP')
connexions_contact_mask = guidance_variable_decisions['Variable'].isin({'W1advperYP', 'W2advperYP', 'W3advperYP'})
mentor_profile_mask = guidance_variable_decisions['Variable'].isin({'W3mentor1YP', 'W3mentor2YP'})
stay_profile_mask = guidance_variable_decisions['Variable'].str.startswith(('W3tlkteacYP', 'W3adteac2YP',
    'W3adteacYP'))
apprenticeship_profile_mask = guidance_variable_decisions['Variable'].str.startswith(('W3tlktappYP', 'W3adapp2YP',
    'W3adappYP'))
guidance_variable_decisions.loc[teacher_index_mask, 'Domain 10 role'] = 'Input to teacher-guidance frequency index'
guidance_variable_decisions.loc[teacher_index_mask,
    'Domain 10 reason'] = 'Wave 2 represents the latest broad pre-transition measurement across two teacher-guidance contexts.'
guidance_variable_decisions.loc[careers_frequency_mask,
    'Domain 10 role'] = 'Direct retained Careers Advice Service frequency predictor'
guidance_variable_decisions.loc[careers_frequency_mask,
    'Domain 10 reason'] = 'The item measures a distinct institutional guidance source with broad sample coverage.'
guidance_variable_decisions.loc[connexions_contact_mask,
    'Domain 10 role'] = 'Input to cumulative Connexions adviser-contact predictor'
guidance_variable_decisions.loc[connexions_contact_mask,
    'Domain 10 reason'] = 'Any positive pre-transition contact report is preserved across the three survey waves.'
guidance_variable_decisions.loc[mentor_profile_mask, 'Domain 10 role'] = 'Input to learning-mentor engagement profile'
guidance_variable_decisions.loc[mentor_profile_mask,
    'Domain 10 reason'] = 'Provision and use are combined because mentor use is structurally conditional on provision.'
guidance_variable_decisions.loc[stay_profile_mask, 'Domain 10 role'] = 'Input to stay-on guidance profile'
guidance_variable_decisions.loc[stay_profile_mask,
    'Domain 10 reason'] = 'The routed discussion, recommendation and influence items are represented by one near-transition profile.'
guidance_variable_decisions.loc[apprenticeship_profile_mask,
    'Domain 10 role'] = 'Input to apprenticeship guidance profile'
guidance_variable_decisions.loc[apprenticeship_profile_mask,
    'Domain 10 reason'] = 'The routed discussion, recommendation and influence items are represented by one near-transition profile.'
guidance_variable_decisions.loc[support_mask,
    'Domain 10 role'] = 'Routing, cross-wave or representation review support'
guidance_variable_decisions.loc[support_mask,
    'Domain 10 reason'] = 'The variable informed routing or representation decisions but is not included separately in the predictor set.'
outside_domain_mask = guidance_variable_decisions['Variable'].isin({'W1advfamYP', 'W1advpalYP', 'W1infofamYP',
    'W2AdvFamYP', 'W2AdvPalYP'})
false_positive_mask = guidance_variable_decisions['Variable'].eq('W2infon')
guidance_variable_decisions.loc[outside_domain_mask, 'Domain 10 role'] = 'Outside Domain 10'
guidance_variable_decisions.loc[outside_domain_mask,
    'Domain 10 reason'] = 'The item concerns family or peer guidance rather than school, service or local institutional context.'
guidance_variable_decisions.loc[false_positive_mask, 'Domain 10 role'] = 'Search false positive'
guidance_variable_decisions.loc[false_positive_mask,
    'Domain 10 reason'] = 'The administrative household-reference variable entered the audit through a broad text-search pattern.'
assert guidance_variable_decisions[['Domain 10 decision', 'Domain 10 role', 'Domain 10 reason',
    'Domain 10 timing assessment']].notna().all().all()
guidance_variable_decision_summary = guidance_variable_decisions['Domain 10 decision'].value_counts().rename('Variables').rename_axis('Domain 10 decision').reset_index()
guidance_final_summary = pd.DataFrame([{'Guidance-related variables audited': int(len(guidance_variable_decisions)),
    'Construction inputs': int(construction_mask.sum()), 'Review-support variables': int(support_mask.sum()), 'Excluded variables': int(exclude_mask.sum()), 'Guidance predictors retained': int(len(guidance_predictor_names)), 'All six predictors available': int(guidance_available_predictor_count.eq(6).sum()), 'At least five predictors available': int(guidance_available_predictor_count.ge(5).sum()), 'No guidance predictors available': int(guidance_available_predictor_count.eq(0).sum())}])
print('Guidance final summary:')
display_limited(guidance_final_summary)
print('Retained guidance predictor coverage:')
display_limited(guidance_predictor_summary)
print('Joint guidance predictor coverage:')
display_limited(guidance_joint_coverage_summary)
print('Guidance source-variable decisions:')
display_limited(guidance_variable_decision_summary)
print('Guidance representation decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(guidance_representation_decisions)
print('Guidance category definitions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(guidance_category_definitions)
print('Files written in this cell: 0')

Guidance final summary:


,Guidance-related variables audited,Construction inputs,Review-support variables,Excluded variables,Guidance predictors retained,All six predictors available,At least five predictors available,No guidance predictors available
0,69,50,13,6,6,8882,9350,246


Retained guidance predictor coverage:


,Predictor,Non-missing,Missing,Missing percentage,Observed categories or values,Smallest observed group,Largest observed group
0,teacher_guidance_frequency_index_pretransition,9409,358,3.67,9,32,2334
1,careers_advice_service_guidance_frequency_pret...,9378,389,3.98,5,86,5701
2,connexions_adviser_contact_pretransition,9519,248,2.54,2,1818,7701
3,learning_mentor_engagement_profile_pretransition,9073,694,7.11,3,1136,5899
4,stay_on_guidance_profile_pretransition,9426,341,3.49,4,1261,3578


Joint guidance predictor coverage:


...

Guidance representation decisions:


,Candidate representation,Decision,Final predictor,Reason
0,Two separate Wave 2 teacher-guidance frequency items,Combine,teacher_guidance_frequency_index_pretransition,The two items represent guidance during and outside lessons. Their mean provides a compact frequency index; it is not interpreted as a latent psychometric scale.
1,Three-item institutional-guidance frequency score,Do not retain,NaN,Adding Careers Advice Service frequency reduced internal consistency and combined distinct providers.
2,Careers Advice Service discussion frequency,Retain,careers_advice_service_guidance_frequency_pretransition,"The item has broad coverage and limited overlap with teacher guidance, school experience and post-16 plans."
3,Latest Connexions adviser-contact response,Do not retain,NaN,"A later negative response would discard earlier positive contact reports for 1,174 participants."
4,Any-wave Connexions adviser contact,Retain,connexions_adviser_contact_pretransition,Any positive report is retained because adviser contact is a cumulative pre-transition experience.


Guidance category definitions:


,Predictor,Category code,Category label
0,teacher_guidance_frequency_index_pretransition,<NA>,"Mean of two Wave 2 frequency items, ranging from 0 to 4"
1,careers_advice_service_guidance_frequency_pretransition,0,Not at all
2,careers_advice_service_guidance_frequency_pretransition,1,Not very often
3,careers_advice_service_guidance_frequency_pretransition,2,A little
4,careers_advice_service_guidance_frequency_pretransition,3,Quite a lot


Files written in this cell: 0


In [408]:
# 22: Existing school-context predictor reconciliation

import re
import pandas as pd
assert len(domain_10_screened_candidates) == 161
assert callable(globals().get('retrieve_existing_predictor'))
domain_10_already_represented = domain_10_screened_candidates.loc[domain_10_screened_candidates['Domain 10 screening status'].astype('string').str.lower().eq('already represented')].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
assert len(domain_10_already_represented) == 10
already_represented_display_columns = [column for column in ['Source order', 'Wave', 'Source type', 'Source file',
    'Variable position', 'Variable', 'Variable label', 'Timing status', 'Review outcome', 'Substantive domain', 'Domain 10 screening status', 'Domain 10 review track', 'Domain 10 screening reason'] if column in domain_10_already_represented.columns]
school_context_register_pattern = '\\bindschool\\b|\\bschsty\\b|\\bstsch\\b|school sector|school type|type of school|school change|changed school|change of school|moved school|move schools|school mobility|started at school|started this school|month started school'
school_context_register_search_columns = [column for column in ['Variable', 'Variable label', 'Substantive domain',
    'Decision reason', 'Review notes'] if column in working_variable_decision_register.columns]
school_context_register_text = pd.Series('', index=working_variable_decision_register.index, dtype='string')
for column in school_context_register_search_columns:
    school_context_register_text = school_context_register_text + ' ' + working_variable_decision_register[column].fillna('').astype('string')
school_context_register_matches = working_variable_decision_register.loc[school_context_register_text.str.contains(school_context_register_pattern,
    case=False, na=False, regex=True)].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
school_context_register_display_columns = [column for column in ['Source order', 'Wave', 'Source type', 'Source file',
    'Variable position', 'Variable', 'Variable label', 'Timing status', 'Review status', 'Review outcome', 'Substantive domain', 'Decision reason', 'Review notes'] if column in school_context_register_matches.columns]
school_context_object_pattern = re.compile('school[_\\s-]*(?:sector|type|change|changed|mobility|move)|(?:sector|type|change|changed|mobility|move)[_\\s-]*school|ind[_\\s-]*school|indschool|schsty|stsch|started[_\\s-]*school',
    flags=re.IGNORECASE)
school_context_object_inventory_columns = ['Object', 'Object type', 'Candidate field', 'Rows', 'Non-missing',
    'Distinct observed values']
school_context_object_rows = []
for object_name, object_value in list(globals().items()):
    if isinstance(object_value, pd.Series):
        series_name = str(object_value.name) if object_value.name is not None else ''
        searchable_name = str(object_name) + ' ' + series_name
        if school_context_object_pattern.search(searchable_name):
            school_context_object_rows.append({'Object': str(object_name), 'Object type': 'Series',
                'Candidate field': series_name if series_name else str(object_name), 'Rows': int(len(object_value)), 'Non-missing': int(object_value.notna().sum()), 'Distinct observed values': int(object_value.nunique(dropna=True))})
    elif isinstance(object_value, pd.DataFrame):
        matching_columns = [column for column in object_value.columns if school_context_object_pattern.search(str(column))]
        for column in matching_columns:
            column_values = object_value.loc[:, column]
            if isinstance(column_values, pd.DataFrame):
                column_non_missing = int(column_values.notna().any(axis=1).sum())
                column_distinct_values = pd.NA
            else:
                column_non_missing = int(column_values.notna().sum())
                column_distinct_values = int(column_values.nunique(dropna=True))
            school_context_object_rows.append({'Object': str(object_name), 'Object type': 'DataFrame column',
                'Candidate field': str(column), 'Rows': int(len(object_value)), 'Non-missing': column_non_missing, 'Distinct observed values': column_distinct_values})
school_context_object_inventory = pd.DataFrame(school_context_object_rows,
    columns=school_context_object_inventory_columns)
if not school_context_object_inventory.empty:
    school_context_object_inventory = school_context_object_inventory.drop_duplicates().sort_values(['Candidate field',
        'Object']).reset_index(drop=True)
if school_context_object_inventory.empty:
    school_context_participant_level_candidates = school_context_object_inventory.copy()
else:
    school_context_participant_level_candidates = school_context_object_inventory.loc[school_context_object_inventory['Rows'].eq(len(route_index)) & school_context_object_inventory['Non-missing'].gt(0)].copy().reset_index(drop=True)
school_context_retrieval_name_candidates = ['school_sector', 'school_sector_pretransition', 'school_sector_type',
    'school_sector_type_pretransition', 'school_type', 'school_type_pretransition', 'independent_school_status', 'independent_school_status_pretransition', 'school_change', 'school_change_pretransition', 'school_change_status', 'school_change_status_pretransition', 'changed_school', 'changed_school_pretransition', 'school_change_experience_pretransition', 'school_mobility_pretransition', 'school_start_timing_pretransition']
school_context_retrieval_rows = []
for predictor_name in school_context_retrieval_name_candidates:
    try:
        predictor_values, source_object = retrieve_existing_predictor(predictor_name)
        retrieval_status = 'Retrieved'
        predictor_values = predictor_values.reindex(route_index)
        non_missing = int(predictor_values.notna().sum())
        distinct_values = int(predictor_values.nunique(dropna=True))
    except Exception as retrieval_error:
        source_object = pd.NA
        retrieval_status = type(retrieval_error).__name__
        non_missing = pd.NA
        distinct_values = pd.NA
    school_context_retrieval_rows.append({'Requested predictor name': predictor_name,
        'Retrieval status': retrieval_status, 'Source object': source_object, 'Non-missing': non_missing, 'Distinct observed values': distinct_values})
school_context_retrieval_summary = pd.DataFrame(school_context_retrieval_rows)
school_context_successful_retrievals = school_context_retrieval_summary.loc[school_context_retrieval_summary['Retrieval status'].eq('Retrieved')].copy().reset_index(drop=True)
school_context_reconciliation_overview = pd.DataFrame([{'Domain 10 already-represented variables': int(len(domain_10_already_represented)),
    'Register matches for school sector or change': int(len(school_context_register_matches)), 'Matching Series or DataFrame columns': int(len(school_context_object_inventory)), 'Participant-level object matches': int(len(school_context_participant_level_candidates)), 'Successful predictor retrievals': int(len(school_context_successful_retrievals)), 'Files written': 0}])
print('School-context reconciliation overview:')
display_limited(school_context_reconciliation_overview)
print('Domain 10 variables already represented:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_10_already_represented[already_represented_display_columns])
print('School-context reconciliation overview:')
display_limited(school_context_reconciliation_overview)
already_represented_display = domain_10_already_represented[already_represented_display_columns]
print(f'\nDomain 10 variables already represented: {len(already_represented_display):,}')
print('\nFirst 15 already represented variables:')
display_limited(already_represented_display.head(15))
print(f'Additional already represented variables not displayed: {max(len(already_represented_display) - 15, 0):,}')
school_context_register_display = school_context_register_matches[school_context_register_display_columns]
print(f'\nRegister matches for school sector and school change: {len(school_context_register_display):,}')
print('\nFirst 15 register matches:')
display_limited(school_context_register_display.head(15))
print(f'Additional register matches not displayed: {max(len(school_context_register_display) - 15, 0):,}')
print(f'\nMatching current objects and columns: {len(school_context_object_inventory):,}')
print('\nFirst 15 matching objects and columns:')
display_limited(school_context_object_inventory.head(15))
print(f'Additional object matches not displayed: {max(len(school_context_object_inventory) - 15, 0):,}')
print(f'\nParticipant-level school-context object candidates: {len(school_context_participant_level_candidates):,}')
print('\nFirst 10 participant-level object candidates:')
display_limited(school_context_participant_level_candidates.head(10))
print(f'Additional participant-level candidates not displayed: {max(len(school_context_participant_level_candidates) - 10, 0):,}')
print('\nExisting-predictor retrieval attempts:')
display_limited(school_context_retrieval_summary)
print('\nFiles written in this cell: 0')

School-context reconciliation overview:


,Domain 10 already-represented variables,Register matches for school sector or change,Matching Series or DataFrame columns,Participant-level object matches,Successful predictor retrievals,Files written
0,10,6,42,23,0,0


Domain 10 variables already represented:


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review outcome,Substantive domain,Domain 10 screening status,Domain 10 review track,Domain 10 screening reason
0,1,Wave 1,Young person,wave_one_lsype_young_person_2020,5,IndSchool,DV: Whether YP was at an independent or maintained school at sampling stage,Pre-transition source,Retain as construction source,School context,Already represented,Existing school-sector or school-change representation,The source variable has already been reviewed and contributes to an existing school-sector or school-change predictor
1,1,Wave 1,Young person,wave_one_lsype_young_person_2020,10,W1stschHS,DV: Month YP started school,Pre-transition source,Retain as review support only,School context,Already represented,Existing school-sector or school-change representation,The source variable has already been reviewed and contributes to an existing school-sector or school-change predictor
2,1,Wave 1,Young person,wave_one_lsype_young_person_2020,255,W1truantYP,YP: Whether played truant in last 12 months,Pre-transition source,Construction input,School attitudes and engagement,Already represented,Existing truancy representation,The item is already a construction input for the retained pre-transition truancy predictor
3,4,Wave 2,Young person,wave_two_lsype_young_person_2020,9,W2SchstyHS,MP: Year YP started school,Pre-transition source,Retain as review support only,School context,Already represented,Existing school-sector or school-change representation,The source variable has already been reviewed and contributes to an existing school-sector or school-change predictor
4,4,Wave 2,Young person,wave_two_lsype_young_person_2020,10,W2stschHS,DV: Month YP started school,Pre-transition source,Retain as review support only,School context,Already represented,Existing school-sector or school-change representation,The source variable has already been reviewed and contributes to an existing school-sector or school-change predictor


School-context reconciliation overview:


...

Additional object matches not displayed: 27

Participant-level school-context object candidates: 23

First 10 participant-level object candidates:


,Object,Object type,Candidate field,Rows,Non-missing,Distinct observed values
0,stage_2_school_change_measure_review,DataFrame column,Any school change by Wave 3,9767,9350,2
1,wave_1_selected_category_data,DataFrame column,IndSchool,9767,9524,2
2,stage_2_school_change_measure_review,DataFrame column,School mobility pattern,9767,9767,7
3,stage_2_school_sector_measure_review,DataFrame column,School sector,9767,9524,2
4,stage_2_school_change_measure_review,DataFrame column,School-change information complete,9767,9767,2


Additional participant-level candidates not displayed: 13

Existing-predictor retrieval attempts:


,Requested predictor name,Retrieval status,Source object,Non-missing,Distinct observed values
0,school_sector,NameError,<NA>,<NA>,<NA>
1,school_sector_pretransition,NameError,<NA>,<NA>,<NA>
2,school_sector_type,NameError,<NA>,<NA>,<NA>
3,school_sector_type_pretransition,NameError,<NA>,<NA>,<NA>
4,school_type,NameError,<NA>,<NA>,<NA>



Files written in this cell: 0


In [409]:
# 23: Saved school-context output and source-variable review

assert stage_2_output_directory.exists()
school_context_exact_variable_names = {'indschool', 'w1stschhs', 'w2schstyhs', 'w2stschhs', 'w2schnamemp',
    'w3schnameyp', 'w3stschhs', 'w3schstyhs', 'w1inyrmov', 'w2inyrmov'}
school_context_exact_register_review = working_variable_decision_register.loc[working_variable_decision_register['Variable'].fillna('').astype(str).str.lower().isin(school_context_exact_variable_names)].copy().sort_values(['Source order',
    'Variable position']).reset_index(drop=True)
school_context_exact_register_columns = [column for column in ['Source order', 'Wave', 'Source type', 'Source file',
    'Variable position', 'Variable', 'Variable label', 'Timing status', 'Review status', 'Review outcome', 'Substantive domain', 'Decision reason', 'Review notes'] if column in school_context_exact_register_review.columns]
school_context_saved_column_pattern = 'school|sector|change|changed|mobility|move|indschool|schname|inyrmov'
stage_2_csv_files = sorted(stage_2_output_directory.glob('*.csv'))
school_context_saved_file_rows = []
for csv_path in stage_2_csv_files:
    try:
        file_columns = pd.read_csv(csv_path, nrows=0).columns.tolist()
        matching_columns = [column for column in file_columns if pd.Series([str(column)]).str.contains(school_context_saved_column_pattern,
            case=False, na=False, regex=True).iloc[0]]
        if matching_columns:
            school_context_saved_file_rows.append({'File': csv_path.name, 'Full path': str(csv_path),
                'Columns in file': int(len(file_columns)), 'Matching school-context columns': ' | '.join(matching_columns)})
    except Exception as read_error:
        school_context_saved_file_rows.append({'File': csv_path.name, 'Full path': str(csv_path),
            'Columns in file': pd.NA, 'Matching school-context columns': f'READ ERROR: {type(read_error).__name__}'})
school_context_saved_file_inventory = pd.DataFrame(school_context_saved_file_rows, columns=['File', 'Full path',
    'Columns in file', 'Matching school-context columns'])
school_context_saved_column_rows = []
for _, file_row in school_context_saved_file_inventory.iterrows():
    matching_text = file_row['Matching school-context columns']
    if pd.isna(matching_text) or str(matching_text).startswith('READ ERROR:'):
        continue
    matching_columns = str(matching_text).split(' | ')
    csv_path = Path(file_row['Full path'])
    header_columns = pd.read_csv(csv_path, nrows=0).columns.tolist()
    columns_to_read = [column for column in ['NSID', *matching_columns] if column in header_columns]
    file_data = pd.read_csv(csv_path, usecols=columns_to_read, low_memory=False)
    for column in matching_columns:
        if column not in file_data.columns:
            continue
        column_values = file_data[column]
        school_context_saved_column_rows.append({'File': csv_path.name, 'Column': column, 'Rows': int(len(file_data)),
            'Unique NSID': int(file_data['NSID'].nunique()) if 'NSID' in file_data.columns else pd.NA, 'Non-missing': int(column_values.notna().sum()), 'Missing': int(column_values.isna().sum()), 'Distinct observed values': int(column_values.nunique(dropna=True)), 'Example observed values': ' | '.join(column_values.dropna().astype(str).drop_duplicates().head(10).tolist())})
school_context_saved_column_summary = pd.DataFrame(school_context_saved_column_rows, columns=['File', 'Column', 'Rows',
    'Unique NSID', 'Non-missing', 'Missing', 'Distinct observed values', 'Example observed values'])
in_year_school_move_register_review = school_context_exact_register_review.loc[school_context_exact_register_review['Variable'].fillna('').astype(str).str.lower().isin({'w1inyrmov',
    'w2inyrmov'})].copy().reset_index(drop=True)
assert len(in_year_school_move_register_review) == 2
school_context_saved_output_overview = pd.DataFrame([{'Stage 2 CSV files searched': int(len(stage_2_csv_files)),
    'Files with school-context columns': int(len(school_context_saved_file_inventory)), 'Matching saved columns': int(len(school_context_saved_column_summary)), 'Exact register variables found': int(len(school_context_exact_register_review)), 'Pending in-year move variables': int(len(in_year_school_move_register_review)), 'Files written': 0}])
print('Saved school-context output overview:')
display_limited(school_context_saved_output_overview)
school_context_exact_register_display = school_context_exact_register_review[school_context_exact_register_columns]
print(f'\nExact school-context source variables in the register: {len(school_context_exact_register_display):,}')
display_limited(school_context_exact_register_display.head(15))
print(f'Additional register records not displayed: {max(len(school_context_exact_register_display) - 15, 0):,}')
print(f'\nSaved files containing school-context columns: {len(school_context_saved_file_inventory):,}')
display_limited(school_context_saved_file_inventory.head(10))
print(f'Additional saved-file records not displayed: {max(len(school_context_saved_file_inventory) - 10, 0):,}')
print(f'\nSaved school-context column records: {len(school_context_saved_column_summary):,}')
display_limited(school_context_saved_column_summary.head(15))
print(f'Additional saved-column records not displayed: {max(len(school_context_saved_column_summary) - 15, 0):,}')
in_year_school_move_display = in_year_school_move_register_review[school_context_exact_register_columns]
print(f'\nPending in-year school-move variables: {len(in_year_school_move_display):,}')
if not in_year_school_move_display.empty:
    display_limited(in_year_school_move_display)
else:
    print('No pending in-year school-move variables.')
print('\nFiles written in this cell: 0')

Saved school-context output overview:


,Stage 2 CSV files searched,Files with school-context columns,Matching saved columns,Exact register variables found,Pending in-year move variables,Files written
0,72,13,27,12,2,0



Exact school-context source variables in the register: 12


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Substantive domain,Decision reason,Review notes
0,1,Wave 1,Young person,wave_one_lsype_young_person_2020,5,IndSchool,DV: Whether YP was at an independent or mainta...,Pre-transition source,"Variable-level coding, routing and reference-p...",Retain as construction source,School context,Provides a binary measure of whether the young...,Retained as the single source copy. It provide...
1,1,Wave 1,Young person,wave_one_lsype_young_person_2020,10,W1stschHS,DV: Month YP started school,Pre-transition source,"Variable-level coding, routing and reference-p...",Retain as review support only,School context,School-start timing variable used to verify th...,Wave 1 mainly records the baseline current-sch...
2,2,Wave 1,Family background,wave_one_lsype_family_background_2020,5,IndSchool,DV: Whether YP was at an independent or mainta...,Pre-transition source,"Variable-level coding, routing and reference-p...",Exclude from predictor set,School context,Exact duplicate of IndSchool in the Wave 1 you...,"All 9,524 overlapping participant-level values..."
3,3,Wave 1,Parental attitudes,wave_one_lsype_parental_attitudes_file_16_05_08,5,IndSchool,DV: Whether YP was at an independent or mainta...,Pre-transition source,"Variable-level coding, routing and reference-p...",Exclude from predictor set,School context,Exact duplicate of IndSchool in the Wave 1 you...,"All 9,524 overlapping participant-level values..."
4,4,Wave 2,Young person,wave_two_lsype_young_person_2020,9,W2SchstyHS,MP: Year YP started school,Pre-transition source,"Variable-level coding, routing and reference-p...",Retain as review support only,School context,School-start timing variable used to verify th...,Wave 1 mainly records the baseline current-sch...


Additional register records not displayed: 0

Saved files containing school-context columns: 13


...

Additional saved-file records not displayed: 3

Saved school-context column records: 27


,File,Column,Rows,Unique NSID,Non-missing,Missing,Distinct observed values,Example observed values
0,stage_2_disability_measure_review.csv,disability_schooling_category,9767,9767,9466,301,3,3.0 | 1.0 | 2.0
1,stage_2_disability_measure_review.csv,Disability and schooling,9767,9767,9466,301,3,No disability or long-standing illness | Disab...
2,stage_2_guidance_coverage_summary.csv,Wave 3 values removed by timing,29,<NA>,29,0,2,14 | 0
3,stage_2_parental_educational_attitudes_support...,school_night_curfew_pretransition,9767,9767,9102,665,4,2.0 | 1.0 | 3.0 | 0.0
4,stage_2_parental_educational_attitudes_support...,parent_school_day_discussion_frequency_pretran...,9767,9767,9079,688,3,1.0 | 2.0 | 0.0


Additional saved-column records not displayed: 12

Pending in-year school-move variables: 2


,Source order,Wave,Source type,Source file,Variable position,Variable,Variable label,Timing status,Review status,Review outcome,Substantive domain,Decision reason,Review notes
0,7,Waves 1–2,History,lsype_history_file_wave_one_and_wave_two_june_...,18,W1inyrmov,DV: No of times moved schools in-year (i.e. no...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,NaN,NaN,NaN
1,7,Waves 1–2,History,lsype_history_file_wave_one_and_wave_two_june_...,20,W2inyrmov,DV: No of times moved schools in-year (i.e. no...,Pre-transition source,"Variable-level coding, routing and reference-p...",Pending review,NaN,NaN,NaN



Files written in this cell: 0


In [410]:
# 24: In-year school-move history review

from pathlib import Path
import pandas as pd
history_source_file = 'lsype_history_file_wave_one_and_wave_two_june_2008'
history_source_path = source_file_lookup[history_source_file]
in_year_move_variables = ['W1inyrmov', 'W2inyrmov']
in_year_move_raw_source = pd.read_stata(history_source_path, columns=['NSID', *in_year_move_variables],
    convert_categoricals=False)
in_year_move_labelled_source = pd.read_stata(history_source_path, columns=['NSID', *in_year_move_variables],
    convert_categoricals=True)
for source_data in [in_year_move_raw_source, in_year_move_labelled_source]:
    source_data['NSID'] = standardise_nsid(source_data['NSID'])
    assert source_data['NSID'].is_unique
in_year_move_raw = in_year_move_raw_source.set_index('NSID').reindex(route_index)
in_year_move_labelled = in_year_move_labelled_source.set_index('NSID').reindex(route_index)
for variable in in_year_move_variables:
    in_year_move_raw[variable] = pd.to_numeric(in_year_move_raw[variable], errors='coerce')
with pd.io.stata.StataReader(history_source_path) as history_reader:
    history_full_variable_labels = history_reader.variable_labels()
in_year_move_full_labels = pd.DataFrame([{'Variable': variable,
    'Full Stata variable label': history_full_variable_labels.get(variable,
    pd.NA)} for variable in in_year_move_variables])
in_year_move_neighbourhood = working_variable_decision_register.loc[working_variable_decision_register['Source file'].eq(history_source_file) & working_variable_decision_register['Variable position'].between(14,
    24)][['Variable position', 'Variable', 'Variable label', 'Timing status', 'Review status', 'Review outcome',
    'Substantive domain']].copy().sort_values('Variable position').reset_index(drop=True)
in_year_move_code_rows = []
for variable in in_year_move_variables:
    raw_values = in_year_move_raw[variable]
    labelled_values = in_year_move_labelled[variable].astype('string')
    for raw_code, participants in raw_values.value_counts(dropna=False).sort_index().items():
        if pd.isna(raw_code):
            value_label = 'Unavailable'
            response_type = 'Unavailable'
        else:
            matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            response_type = 'Observed count' if raw_code >= 0 else 'Special code'
        in_year_move_code_rows.append({'Variable': variable, 'Raw code': raw_code, 'Value label': value_label,
            'Response type': response_type, 'Participants': int(participants)})
in_year_move_code_distribution = pd.DataFrame(in_year_move_code_rows)
in_year_move_valid_counts = pd.DataFrame({variable: in_year_move_raw[variable].where(in_year_move_raw[variable].ge(0)).astype('Float64') for variable in in_year_move_variables},
    index=route_index)
in_year_move_coverage_rows = []
for variable in in_year_move_variables:
    values = in_year_move_valid_counts[variable]
    observed_counts = values.dropna().value_counts()
    in_year_move_coverage_rows.append({'Variable': variable, 'Valid non-negative counts': int(values.notna().sum()),
        'Missing or special-code responses': int(values.isna().sum()), 'Zero moves': int(values.eq(0).sum()), 'One move': int(values.eq(1).sum()), 'Two or more moves': int(values.ge(2).sum()), 'Maximum observed count': float(values.max()) if values.notna().any() else pd.NA, 'Distinct observed counts': int(values.nunique()), 'Smallest observed group': int(observed_counts.min()) if not observed_counts.empty else pd.NA})
in_year_move_coverage_summary = pd.DataFrame(in_year_move_coverage_rows)
in_year_move_complete_pair = in_year_move_valid_counts.dropna().astype(float)
in_year_move_pair_crosstab = pd.crosstab(in_year_move_complete_pair['W1inyrmov'],
    in_year_move_complete_pair['W2inyrmov'], margins=True, dropna=False)
in_year_move_wave_relationship_summary = pd.DataFrame([{'Complete comparisons': int(len(in_year_move_complete_pair)),
    'Exact equality': int(in_year_move_complete_pair['W1inyrmov'].eq(in_year_move_complete_pair['W2inyrmov']).sum()), 'Exact equality percentage': round(in_year_move_complete_pair['W1inyrmov'].eq(in_year_move_complete_pair['W2inyrmov']).mean() * 100,
    2), 'W2 greater than W1': int(in_year_move_complete_pair['W2inyrmov'].gt(in_year_move_complete_pair['W1inyrmov']).sum()), 'W2 lower than W1': int(in_year_move_complete_pair['W2inyrmov'].lt(in_year_move_complete_pair['W1inyrmov']).sum()), "Cramer's V": round(categorical_cramers_v(in_year_move_complete_pair['W1inyrmov'],
    in_year_move_complete_pair['W2inyrmov']), 3)}])
in_year_move_any_wave_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='in_year_move_any_wave_candidate')
any_valid_history_count = in_year_move_valid_counts.notna().any(axis=1)
any_positive_history_count = in_year_move_valid_counts.gt(0).any(axis=1)
in_year_move_any_wave_candidate.loc[any_valid_history_count & ~any_positive_history_count] = 0
in_year_move_any_wave_candidate.loc[any_positive_history_count] = 1
in_year_move_maximum_count_candidate = in_year_move_valid_counts.max(axis=1,
    skipna=True).where(any_valid_history_count).astype('Float64').rename('in_year_move_maximum_count_candidate')
school_change_review_path = stage_2_output_directory / 'stage_2_school_change_measure_review.csv'
assert school_change_review_path.exists()
saved_school_change_review = pd.read_csv(school_change_review_path, low_memory=False)
saved_school_change_review['NSID'] = standardise_nsid(saved_school_change_review['NSID'])
assert saved_school_change_review['NSID'].is_unique
saved_school_change_review = saved_school_change_review.set_index('NSID').reindex(route_index)
existing_any_school_change = pd.to_numeric(saved_school_change_review['any_school_change_by_wave3'],
    errors='coerce').astype('Float64')
school_move_overlap_data = pd.DataFrame({'Any in-year move in history file': in_year_move_any_wave_candidate,
    'Existing any school change by Wave 3': existing_any_school_change}, index=route_index).dropna()
school_move_overlap_crosstab = pd.crosstab(school_move_overlap_data['Any in-year move in history file'],
    school_move_overlap_data['Existing any school change by Wave 3'], margins=True, dropna=False)
school_move_overlap_summary = pd.DataFrame([{'Complete comparisons': int(len(school_move_overlap_data)),
    'History in-year move reported': int(school_move_overlap_data['Any in-year move in history file'].eq(1).sum()), 'Existing direct school change reported': int(school_move_overlap_data['Existing any school change by Wave 3'].eq(1).sum()), 'Both reported': int((school_move_overlap_data['Any in-year move in history file'].eq(1) & school_move_overlap_data['Existing any school change by Wave 3'].eq(1)).sum()), 'History move only': int((school_move_overlap_data['Any in-year move in history file'].eq(1) & school_move_overlap_data['Existing any school change by Wave 3'].eq(0)).sum()), 'Direct school change only': int((school_move_overlap_data['Any in-year move in history file'].eq(0) & school_move_overlap_data['Existing any school change by Wave 3'].eq(1)).sum()), "Cramer's V": round(categorical_cramers_v(school_move_overlap_data['Any in-year move in history file'],
    school_move_overlap_data['Existing any school change by Wave 3']), 3)}])
in_year_move_review_overview = pd.DataFrame([{'History variables reviewed': 2,
    'Full analysis sample': int(len(route_index)), 'Review candidates constructed': 2, 'Predictors retained in this cell': 0, 'Outcome columns loaded': 0, 'Files written': 0}])
print('In-year school-move review overview:')
display_limited(in_year_move_review_overview)
print('Full Stata variable labels:')
with pd.option_context('display.max_colwidth', None):
    display_limited(in_year_move_full_labels)
print('Nearby history-file variables:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(in_year_move_neighbourhood)
print('Response-code distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(in_year_move_code_distribution)
print('Valid-count coverage:')
display_limited(in_year_move_coverage_summary)
print('Relationship between Wave 1 and Wave 2 history measures:')
display_limited(in_year_move_wave_relationship_summary)
print('Wave 1 by Wave 2 count cross-tabulation:')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display_limited(in_year_move_pair_crosstab)
print('Overlap with existing school-change indicator:')
display_limited(school_move_overlap_summary)
print('History-based and direct-indicator cross-tabulation:')
display_limited(school_move_overlap_crosstab)
print('Files written in this cell: 0')

In-year school-move review overview:


,History variables reviewed,Full analysis sample,Review candidates constructed,Predictors retained in this cell,Outcome columns loaded,Files written
0,2,9767,2,0,0,0


Full Stata variable labels:


,Variable,Full Stata variable label
0,W1inyrmov,"DV: No of times moved schools in-year (i.e. not in July, August or September) up"
1,W2inyrmov,"DV: No of times moved schools in-year (i.e. not in July, August or September) up"


Nearby history-file variables:


...

Overlap with existing school-change indicator:


,Complete comparisons,History in-year move reported,Existing direct school change reported,Both reported,History move only,Direct school change only,Cramer's V
0,9169,1662,323,147,1515,176,0.136


History-based and direct-indicator cross-tabulation:


Existing any school change by Wave 3,0.0,1.0,All
Any in-year move in history file,,,
0,7331,176,7507
1,1515,147,1662
All,8846,323,9169


Files written in this cell: 0


In [411]:
# 25: Number-of-schools history and mobility overlap review

import pandas as pd
number_of_schools_variables = ['W1numsch', 'W2numsch']
number_of_schools_raw_source = pd.read_stata(history_source_path, columns=['NSID', *number_of_schools_variables],
    convert_categoricals=False)
number_of_schools_labelled_source = pd.read_stata(history_source_path, columns=['NSID', *number_of_schools_variables],
    convert_categoricals=True)
for source_data in [number_of_schools_raw_source, number_of_schools_labelled_source]:
    source_data['NSID'] = standardise_nsid(source_data['NSID'])
    assert source_data['NSID'].is_unique
number_of_schools_raw = number_of_schools_raw_source.set_index('NSID').reindex(route_index)
number_of_schools_labelled = number_of_schools_labelled_source.set_index('NSID').reindex(route_index)
for variable in number_of_schools_variables:
    number_of_schools_raw[variable] = pd.to_numeric(number_of_schools_raw[variable], errors='coerce')
with pd.io.stata.StataReader(history_source_path) as history_reader:
    number_of_schools_full_label_lookup = history_reader.variable_labels()
number_of_schools_full_labels = pd.DataFrame([{'Variable': variable,
    'Full Stata variable label': number_of_schools_full_label_lookup.get(variable,
    pd.NA)} for variable in number_of_schools_variables])
number_of_schools_code_rows = []
for variable in number_of_schools_variables:
    raw_values = number_of_schools_raw[variable]
    labelled_values = number_of_schools_labelled[variable].astype('string')
    for raw_code, participants in raw_values.value_counts(dropna=False).sort_index().items():
        if pd.isna(raw_code):
            value_label = 'Unavailable'
            response_type = 'Unavailable'
        else:
            matching_labels = labelled_values.loc[raw_values.eq(raw_code)].dropna().drop_duplicates().tolist()
            value_label = matching_labels[0] if matching_labels else str(raw_code)
            response_type = 'Observed count' if raw_code >= 0 else 'Special code'
        number_of_schools_code_rows.append({'Variable': variable, 'Raw code': raw_code, 'Value label': value_label,
            'Response type': response_type, 'Participants': int(participants)})
number_of_schools_code_distribution = pd.DataFrame(number_of_schools_code_rows)
number_of_schools_valid_counts = pd.DataFrame({variable: number_of_schools_raw[variable].where(number_of_schools_raw[variable].ge(0)).astype('Float64') for variable in number_of_schools_variables},
    index=route_index)
number_of_schools_coverage_rows = []
for variable in number_of_schools_variables:
    values = number_of_schools_valid_counts[variable]
    observed_counts = values.dropna().value_counts()
    number_of_schools_coverage_rows.append({'Variable': variable,
        'Valid non-negative counts': int(values.notna().sum()), 'Missing or special-code responses': int(values.isna().sum()), 'One school': int(values.eq(1).sum()), 'Two schools': int(values.eq(2).sum()), 'Three schools': int(values.eq(3).sum()), 'Four or more schools': int(values.ge(4).sum()), 'Maximum observed count': float(values.max()) if values.notna().any() else pd.NA, 'Distinct observed counts': int(values.nunique()), 'Smallest observed group': int(observed_counts.min()) if not observed_counts.empty else pd.NA})
number_of_schools_coverage_summary = pd.DataFrame(number_of_schools_coverage_rows)
number_of_schools_complete_pair = number_of_schools_valid_counts.dropna().astype(float)
number_of_schools_difference = number_of_schools_complete_pair['W2numsch'] - number_of_schools_complete_pair['W1numsch']
number_of_schools_wave_relationship_summary = pd.DataFrame([{'Complete comparisons': int(len(number_of_schools_complete_pair)),
    'Exact equality': int(number_of_schools_difference.eq(0).sum()), 'Exact equality percentage': round(number_of_schools_difference.eq(0).mean() * 100,
    2), 'W2 greater than W1': int(number_of_schools_difference.gt(0).sum()), 'W2 lower than W1': int(number_of_schools_difference.lt(0).sum()), 'Maximum increase by Wave 2': float(number_of_schools_difference.max()), "Cramer's V": round(categorical_cramers_v(number_of_schools_complete_pair['W1numsch'],
    number_of_schools_complete_pair['W2numsch']), 3)}])
number_of_schools_difference_distribution = number_of_schools_difference.value_counts().sort_index().rename('Participants').rename_axis('W2 minus W1 number of schools').reset_index()
latest_number_of_schools_candidate = number_of_schools_valid_counts['W2numsch'].copy().rename('latest_number_of_schools_candidate')
number_of_schools_group_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='number_of_schools_group_candidate')
number_of_schools_group_candidate.loc[latest_number_of_schools_candidate.eq(1)] = 0
number_of_schools_group_candidate.loc[latest_number_of_schools_candidate.eq(2)] = 1
number_of_schools_group_candidate.loc[latest_number_of_schools_candidate.eq(3)] = 2
number_of_schools_group_candidate.loc[latest_number_of_schools_candidate.ge(4)] = 3
assert set(number_of_schools_group_candidate.dropna().astype(int).unique()).issubset({0, 1, 2, 3})
number_of_schools_group_distribution = number_of_schools_group_candidate.value_counts(dropna=False).sort_index().rename('Participants').rename_axis('Number-of-schools group').reset_index()
number_of_schools_group_labels = {0: 'One school attended', 1: 'Two schools attended', 2: 'Three schools attended',
    3: 'Four or more schools attended'}
number_of_schools_group_distribution['Category label'] = number_of_schools_group_distribution['Number-of-schools group'].map(number_of_schools_group_labels).fillna('Unavailable')
school_history_overlap_data = pd.DataFrame({'Number-of-schools group': number_of_schools_group_candidate,
    'Any in-year move': in_year_move_any_wave_candidate, 'Maximum cumulative in-year moves': in_year_move_maximum_count_candidate, 'Existing any school change by Wave 3': existing_any_school_change}, index=route_index)
school_history_overlap_rows = []
for first_measure, second_measure in [('Number-of-schools group', 'Any in-year move'), ('Number-of-schools group',
    'Maximum cumulative in-year moves'), ('Number-of-schools group', 'Existing any school change by Wave 3')]:
    comparison_data = school_history_overlap_data[[first_measure, second_measure]].dropna()
    school_history_overlap_rows.append({'First measure': first_measure, 'Second measure': second_measure,
        'Complete comparisons': int(len(comparison_data)), "Cramer's V": round(categorical_cramers_v(comparison_data[first_measure],
        comparison_data[second_measure]), 3)})
school_history_overlap_summary = pd.DataFrame(school_history_overlap_rows)
number_of_schools_by_in_year_move = pd.crosstab(school_history_overlap_data['Number-of-schools group'].map(number_of_schools_group_labels),
    school_history_overlap_data['Any in-year move'], margins=True, dropna=False)
number_of_schools_by_direct_change = pd.crosstab(school_history_overlap_data['Number-of-schools group'].map(number_of_schools_group_labels),
    school_history_overlap_data['Existing any school change by Wave 3'], margins=True, dropna=False)
wave_2_direct_change = pd.to_numeric(saved_school_change_review['school_change_wave2'],
    errors='coerce').astype('Float64')
wave_2_school_count_increase = pd.Series(pd.NA, index=route_index, dtype='Int64', name='wave_2_school_count_increase')
complete_school_count_pair_mask = number_of_schools_valid_counts[['W1numsch', 'W2numsch']].notna().all(axis=1)
wave_2_school_count_increase.loc[complete_school_count_pair_mask] = number_of_schools_valid_counts.loc[complete_school_count_pair_mask,
    'W2numsch'].gt(number_of_schools_valid_counts.loc[complete_school_count_pair_mask, 'W1numsch']).astype(int)
wave_2_count_change_comparison = pd.DataFrame({'School-count increase by Wave 2': wave_2_school_count_increase,
    'Direct Wave 2 school change': wave_2_direct_change}, index=route_index).dropna()
wave_2_count_change_crosstab = pd.crosstab(wave_2_count_change_comparison['School-count increase by Wave 2'],
    wave_2_count_change_comparison['Direct Wave 2 school change'], margins=True, dropna=False)
wave_2_count_change_summary = pd.DataFrame([{'Complete comparisons': int(len(wave_2_count_change_comparison)),
    'School-count increase': int(wave_2_count_change_comparison['School-count increase by Wave 2'].eq(1).sum()), 'Direct Wave 2 school change': int(wave_2_count_change_comparison['Direct Wave 2 school change'].eq(1).sum()), 'Both reported': int((wave_2_count_change_comparison['School-count increase by Wave 2'].eq(1) & wave_2_count_change_comparison['Direct Wave 2 school change'].eq(1)).sum()), "Cramer's V": round(categorical_cramers_v(wave_2_count_change_comparison['School-count increase by Wave 2'],
    wave_2_count_change_comparison['Direct Wave 2 school change']), 3)}])
number_of_schools_review_overview = pd.DataFrame([{'Number-of-schools history variables reviewed': 2,
    'Review representations constructed': 2, 'Predictors retained in this cell': 0, 'Outcome columns loaded': 0, 'Files written': 0}])
print('Number-of-schools review overview:')
display_limited(number_of_schools_review_overview)
print('Full Stata variable labels:')
with pd.option_context('display.max_colwidth', None):
    display_limited(number_of_schools_full_labels)
print('Response-code distributions:')
with pd.option_context('display.max_rows', None):
    display_limited(number_of_schools_code_distribution)
print('Valid-count coverage:')
display_limited(number_of_schools_coverage_summary)
print('Relationship between Wave 1 and Wave 2 counts:')
display_limited(number_of_schools_wave_relationship_summary)
print('Wave 2 minus Wave 1 count distribution:')
display_limited(number_of_schools_difference_distribution)
print('Latest cumulative number-of-schools groups:')
display_limited(number_of_schools_group_distribution)
print('Overlap among school-mobility measures:')
display_limited(school_history_overlap_summary)
print('Number of schools by any in-year move:')
display_limited(number_of_schools_by_in_year_move)
print('Number of schools by direct school-change indicator:')
display_limited(number_of_schools_by_direct_change)
print('Wave 2 school-count increase and direct-change agreement:')
display_limited(wave_2_count_change_summary)
print('Wave 2 comparison cross-tabulation:')
display_limited(wave_2_count_change_crosstab)
print('Files written in this cell: 0')

Number-of-schools review overview:


,Number-of-schools history variables reviewed,Review representations constructed,Predictors retained in this cell,Outcome columns loaded,Files written
0,2,2,0,0,0


Full Stata variable labels:


,Variable,Full Stata variable label
0,W1numsch,DV: No of schools child has attended up to wave 1
1,W2numsch,DV: No of schools child has attended up to wave 2


Response-code distributions:


...

Wave 2 school-count increase and direct-change agreement:


,Complete comparisons,School-count increase,Direct Wave 2 school change,Both reported,Cramer's V
0,9259,225,241,224,0.961


Wave 2 comparison cross-tabulation:


Direct Wave 2 school change,0.0,1.0,All
School-count increase by Wave 2,,,
0,9017,17,9034
1,1,224,225
All,9018,241,9259


Files written in this cell: 0


In [412]:
# 26: Cumulative school-history representation review

import pandas as pd
latest_number_of_schools = number_of_schools_valid_counts['W2numsch'].copy().astype('Float64')
cumulative_school_history_three_category_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='cumulative_school_history_three_category_candidate')
cumulative_school_history_three_category_candidate.loc[latest_number_of_schools.between(1, 2)] = 0
cumulative_school_history_three_category_candidate.loc[latest_number_of_schools.eq(3)] = 1
cumulative_school_history_three_category_candidate.loc[latest_number_of_schools.ge(4)] = 2
assert set(cumulative_school_history_three_category_candidate.dropna().astype(int).unique()) == {0, 1, 2}
cumulative_school_history_binary_candidate = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='cumulative_school_history_binary_candidate')
cumulative_school_history_binary_candidate.loc[latest_number_of_schools.between(1, 2)] = 0
cumulative_school_history_binary_candidate.loc[latest_number_of_schools.ge(3)] = 1
assert set(cumulative_school_history_binary_candidate.dropna().astype(int).unique()) == {0, 1}
cumulative_school_history_representation_specs = {'Original four-category representation': number_of_schools_group_candidate,
    'Three-category cumulative school history': cumulative_school_history_three_category_candidate, 'Binary cumulative school history': cumulative_school_history_binary_candidate}
cumulative_school_history_label_maps = {'Original four-category representation': {0: 'One school attended',
    1: 'Two schools attended', 2: 'Three schools attended', 3: 'Four or more schools attended'}, 'Three-category cumulative school history': {0: 'One or two schools attended',
    1: 'Three schools attended', 2: 'Four or more schools attended'}, 'Binary cumulative school history': {0: 'One or two schools attended',
    1: 'Three or more schools attended'}}
cumulative_school_history_summary_rows = []
cumulative_school_history_distribution_rows = []
for representation_name, representation_values in cumulative_school_history_representation_specs.items():
    observed_counts = representation_values.dropna().value_counts().sort_index()
    cumulative_school_history_summary_rows.append({'Representation': representation_name,
        'Observed participants': int(representation_values.notna().sum()), 'Missing participants': int(representation_values.isna().sum()), 'Categories': int(representation_values.nunique()), 'Smallest category': int(observed_counts.min()), 'Largest category': int(observed_counts.max()), 'Categories below 50 participants': int(observed_counts.lt(50).sum()), 'Categories below 100 participants': int(observed_counts.lt(100).sum())})
    observed_total = int(representation_values.notna().sum())
    for category_code, participants in representation_values.value_counts(dropna=False).sort_index().items():
        if pd.isna(category_code):
            category_label = 'Unavailable'
            observed_percentage = pd.NA
        else:
            category_label = cumulative_school_history_label_maps[representation_name][int(category_code)]
            observed_percentage = round(participants / observed_total * 100, 2)
        cumulative_school_history_distribution_rows.append({'Representation': representation_name,
            'Category code': category_code, 'Category label': category_label, 'Participants': int(participants), 'Percentage of observed representation': observed_percentage})
cumulative_school_history_sparsity_summary = pd.DataFrame(cumulative_school_history_summary_rows)
cumulative_school_history_distribution = pd.DataFrame(cumulative_school_history_distribution_rows)
school_history_representation_overlap_rows = []
school_history_comparison_measures = {'Any in-year school move': in_year_move_any_wave_candidate,
    'Maximum cumulative in-year move count': in_year_move_maximum_count_candidate, 'Any direct school change by Wave 3': existing_any_school_change}
for representation_name, representation_values in cumulative_school_history_representation_specs.items():
    for comparison_name, comparison_values in school_history_comparison_measures.items():
        comparison_data = pd.DataFrame({'School-history representation': representation_values,
            'Comparison measure': comparison_values}, index=route_index).dropna()
        school_history_representation_overlap_rows.append({'School-history representation': representation_name,
            'Comparison measure': comparison_name, 'Complete comparisons': int(len(comparison_data)), "Cramer's V": round(categorical_cramers_v(comparison_data['School-history representation'],
            comparison_data['Comparison measure']), 3)})
school_history_representation_overlap_summary = pd.DataFrame(school_history_representation_overlap_rows).sort_values(['School-history representation',
    'Comparison measure']).reset_index(drop=True)
school_history_joint_coverage = pd.DataFrame({'Three-category cumulative school history': cumulative_school_history_three_category_candidate,
    'Binary cumulative school history': cumulative_school_history_binary_candidate, 'Any direct school change by Wave 3': existing_any_school_change}, index=route_index)
school_history_joint_coverage_summary = pd.DataFrame([{'Full analysis sample': int(len(route_index)),
    'Three-category history available': int(cumulative_school_history_three_category_candidate.notna().sum()), 'Direct school-change indicator available': int(existing_any_school_change.notna().sum()), 'Both available': int(school_history_joint_coverage[['Three-category cumulative school history',
    'Any direct school change by Wave 3']].notna().all(axis=1).sum()), 'At least one available': int(school_history_joint_coverage[['Three-category cumulative school history',
    'Any direct school change by Wave 3']].notna().any(axis=1).sum()), 'Neither available': int(school_history_joint_coverage[['Three-category cumulative school history',
    'Any direct school change by Wave 3']].isna().all(axis=1).sum())}])
three_category_by_direct_change = pd.crosstab(cumulative_school_history_three_category_candidate.map(cumulative_school_history_label_maps['Three-category cumulative school history']),
    existing_any_school_change, margins=True, dropna=False)
binary_history_by_direct_change = pd.crosstab(cumulative_school_history_binary_candidate.map(cumulative_school_history_label_maps['Binary cumulative school history']),
    existing_any_school_change, margins=True, dropna=False)
cumulative_school_history_review_overview = pd.DataFrame([{'Representations compared': int(len(cumulative_school_history_representation_specs)),
    'Comparison measures': int(len(school_history_comparison_measures)), 'Predictors retained in this cell': 0, 'Outcome columns loaded': 0, 'Files written': 0}])
print('Cumulative school-history review overview:')
display_limited(cumulative_school_history_review_overview)
print('Representation sparsity:')
display_limited(cumulative_school_history_sparsity_summary)
print('Representation distributions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(cumulative_school_history_distribution)
print('Overlap with other school-mobility measures:')
display_limited(school_history_representation_overlap_summary)
print('Joint coverage with direct school-change indicator:')
display_limited(school_history_joint_coverage_summary)
print('Three-category history by direct school change:')
display_limited(three_category_by_direct_change)
print('Binary history by direct school change:')
display_limited(binary_history_by_direct_change)
print('Files written in this cell: 0')

Cumulative school-history review overview:


,Representations compared,Comparison measures,Predictors retained in this cell,Outcome columns loaded,Files written
0,3,3,0,0,0


Representation sparsity:


,Representation,Observed participants,Missing participants,Categories,Smallest category,Largest category,Categories below 50 participants,Categories below 100 participants
0,Original four-category representation,9324,443,4,23,5214,1,1
1,Three-category cumulative school history,9324,443,3,997,5237,0,0
2,Binary cumulative school history,9324,443,2,4087,5237,0,0


Representation distributions:


...

Three-category history by direct school change:


any_school_change_by_wave3,0.0,1.0,NaN,All
cumulative_school_history_three_category_candidate,,,,
Four or more schools attended,838,149,10,997
One or two schools attended,5110,39,88,5237
Three schools attended,2898,135,57,3090
NaN,178,3,262,443
All,9024,326,417,9767


Binary history by direct school change:


any_school_change_by_wave3,0.0,1.0,NaN,All
cumulative_school_history_binary_candidate,,,,
One or two schools attended,5110,39,88,5237
Three or more schools attended,3736,284,67,4087
NaN,178,3,262,443
All,9024,326,417,9767


Files written in this cell: 0


In [413]:
# 27: School sector and mobility predictor construction

from pathlib import Path
import pandas as pd
school_sector_review_path = stage_2_output_directory / 'stage_2_school_sector_measure_review.csv'
assert school_sector_review_path.exists()
saved_school_sector_review = pd.read_csv(school_sector_review_path, low_memory=False)
saved_school_sector_review['NSID'] = standardise_nsid(saved_school_sector_review['NSID'])
assert saved_school_sector_review['NSID'].is_unique
saved_school_sector_review = saved_school_sector_review.set_index('NSID').reindex(route_index)
independent_school_at_sampling_stage = pd.to_numeric(saved_school_sector_review['independent_school_at_sampling_stage'],
    errors='coerce').astype('Int64').rename('independent_school_at_sampling_stage')
assert set(independent_school_at_sampling_stage.dropna().astype(int).unique()) == {0, 1}
cumulative_school_history_pretransition = cumulative_school_history_three_category_candidate.copy().astype('Int64').rename('cumulative_school_history_pretransition')
assert set(cumulative_school_history_pretransition.dropna().astype(int).unique()) == {0, 1, 2}
any_school_change_by_wave3 = pd.to_numeric(saved_school_change_review['any_school_change_by_wave3'],
    errors='coerce').astype('Int64').rename('any_school_change_by_wave3')
assert set(any_school_change_by_wave3.dropna().astype(int).unique()) == {0, 1}
school_sector_mobility_predictors = pd.concat([independent_school_at_sampling_stage,
    cumulative_school_history_pretransition, any_school_change_by_wave3], axis=1).reindex(route_index)
school_sector_mobility_predictors.index.name = 'NSID'
assert school_sector_mobility_predictors.shape == (9767, 3)
school_sector_mobility_predictor_summary_rows = []
for predictor_name in school_sector_mobility_predictors.columns:
    predictor_values = school_sector_mobility_predictors[predictor_name]
    observed_counts = predictor_values.dropna().value_counts()
    school_sector_mobility_predictor_summary_rows.append({'Predictor': predictor_name,
        'Non-missing': int(predictor_values.notna().sum()), 'Missing': int(predictor_values.isna().sum()), 'Missing percentage': round(predictor_values.isna().mean() * 100,
        2), 'Categories': int(predictor_values.nunique()), 'Smallest observed category': int(observed_counts.min()), 'Largest observed category': int(observed_counts.max())})
school_sector_mobility_predictor_summary = pd.DataFrame(school_sector_mobility_predictor_summary_rows)
school_sector_mobility_available_count = school_sector_mobility_predictors.notna().sum(axis=1)
school_sector_mobility_joint_coverage = school_sector_mobility_available_count.value_counts().sort_index().rename('Participants').rename_axis('School sector and mobility predictors available').reset_index()
school_sector_mobility_joint_coverage['Percentage of full sample'] = (school_sector_mobility_joint_coverage['Participants'] / len(route_index) * 100).round(2)
school_sector_mobility_category_definitions = pd.DataFrame([{'Predictor': 'independent_school_at_sampling_stage',
    'Category code': 0, 'Category label': 'Maintained school'}, {'Predictor': 'independent_school_at_sampling_stage',
    'Category code': 1, 'Category label': 'Independent school'}, {'Predictor': 'cumulative_school_history_pretransition',
    'Category code': 0, 'Category label': 'One or two schools attended by Wave 2'}, {'Predictor': 'cumulative_school_history_pretransition',
    'Category code': 1, 'Category label': 'Three schools attended by Wave 2'}, {'Predictor': 'cumulative_school_history_pretransition',
    'Category code': 2, 'Category label': 'Four or more schools attended by Wave 2'}, {'Predictor': 'any_school_change_by_wave3',
    'Category code': 0, 'Category label': 'No school change confirmed at Waves 2 and 3'}, {'Predictor': 'any_school_change_by_wave3',
    'Category code': 1, 'Category label': 'At least one school change reported by Wave 3'}])
school_mobility_representation_decisions = pd.DataFrame([{'Candidate representation': 'Wave 1 number of schools attended',
    'Decision': 'Review support', 'Final predictor': pd.NA, 'Reason': 'The Wave 1 value was used to establish that the Wave 2 measure is a cumulative update.'}, {'Candidate representation': 'Wave 2 number of schools attended',
    'Decision': 'Retain as construction source', 'Final predictor': 'cumulative_school_history_pretransition', 'Reason': 'The Wave 2 value is the latest cumulative pre-transition number of schools attended.'}, {'Candidate representation': 'Original four-category cumulative school history',
    'Decision': 'Do not retain', 'Final predictor': pd.NA, 'Reason': 'The one-school category contained only 23 participants.'}, {'Candidate representation': 'Three-category cumulative school history',
    'Decision': 'Retain', 'Final predictor': 'cumulative_school_history_pretransition', 'Reason': 'The representation removes the sparse category while retaining a distinction between three and four or more schools.'}, {'Candidate representation': 'Binary cumulative school history',
    'Decision': 'Do not retain', 'Final predictor': pd.NA, 'Reason': 'The binary representation discards the distinction between three and four or more schools.'}, {'Candidate representation': 'Wave 1 and Wave 2 in-year school-move counts',
    'Decision': 'Review support', 'Final predictor': pd.NA, 'Reason': 'These measures capture the narrower history of moves outside the usual summer transition period. They overlap with the broader cumulative number of schools and are not retained separately.'}, {'Candidate representation': 'Direct Wave 2–3 school-change indicator',
    'Decision': 'Retain', 'Final predictor': 'any_school_change_by_wave3', 'Reason': 'The indicator captures more recent school change and has limited overlap with cumulative school history.'}, {'Candidate representation': 'Wave 1 sampling-stage school sector',
    'Decision': 'Retain', 'Final predictor': 'independent_school_at_sampling_stage', 'Reason': 'The source provides a distinct baseline measure of maintained versus independent school sector.'}])
school_history_variable_decisions = working_variable_decision_register.loc[working_variable_decision_register['Source file'].eq(history_source_file) & working_variable_decision_register['Variable'].isin({'W1numsch',
    'W2numsch', 'W1inyrmov', 'W2inyrmov'})].copy().sort_values('Variable position').reset_index(drop=True)
assert len(school_history_variable_decisions) == 4
school_history_variable_decisions['Domain 10 decision'] = school_history_variable_decisions['Variable'].map({'W1numsch': 'Review support',
    'W2numsch': 'Construction input', 'W1inyrmov': 'Review support', 'W2inyrmov': 'Review support'})
school_history_variable_decisions['Domain 10 role'] = school_history_variable_decisions['Variable'].map({'W1numsch': 'Cross-wave cumulative-structure review',
    'W2numsch': 'Input to cumulative school-history predictor', 'W1inyrmov': 'School-mobility representation review', 'W2inyrmov': 'School-mobility representation review'})
school_history_variable_decisions['Domain 10 reason'] = school_history_variable_decisions['Variable'].map({'W1numsch': 'Used to verify that the Wave 2 count is a non-decreasing cumulative update.',
    'W2numsch': 'Provides the latest cumulative pre-transition number of schools attended.', 'W1inyrmov': 'Used to examine the narrower history of out-of-season school moves.', 'W2inyrmov': 'Used to examine the narrower history of out-of-season school moves.'})
assert school_history_variable_decisions[['Domain 10 decision', 'Domain 10 role',
    'Domain 10 reason']].notna().all().all()
school_sector_mobility_relationship_rows = []
school_sector_mobility_columns = school_sector_mobility_predictors.columns.tolist()
for first_position in range(len(school_sector_mobility_columns)):
    for second_position in range(first_position + 1, len(school_sector_mobility_columns)):
        first_predictor = school_sector_mobility_columns[first_position]
        second_predictor = school_sector_mobility_columns[second_position]
        comparison_data = school_sector_mobility_predictors[[first_predictor, second_predictor]].dropna()
        school_sector_mobility_relationship_rows.append({'First predictor': first_predictor,
            'Second predictor': second_predictor, 'Complete comparisons': int(len(comparison_data)), "Cramer's V": round(categorical_cramers_v(comparison_data[first_predictor],
            comparison_data[second_predictor]), 3)})
school_sector_mobility_relationship_summary = pd.DataFrame(school_sector_mobility_relationship_rows)
school_sector_mobility_final_summary = pd.DataFrame([{'School sector and mobility predictors retained': 3,
    'Participants with all three available': int(school_sector_mobility_available_count.eq(3).sum()), 'Participants with at least two available': int(school_sector_mobility_available_count.ge(2).sum()), 'Participants with none available': int(school_sector_mobility_available_count.eq(0).sum()), 'Outcome columns loaded': 0, 'Files written': 0}])
print('School sector and mobility final summary:')
display_limited(school_sector_mobility_final_summary)
print('Retained predictor coverage:')
display_limited(school_sector_mobility_predictor_summary)
print('Joint predictor coverage:')
display_limited(school_sector_mobility_joint_coverage)
print('Relationships among retained predictors:')
display_limited(school_sector_mobility_relationship_summary)
print('School-mobility representation decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(school_mobility_representation_decisions)
print('History-file variable decisions:')
with pd.option_context('display.max_colwidth', None):
    display_limited(school_history_variable_decisions[['Variable', 'Variable label', 'Domain 10 decision',
        'Domain 10 role', 'Domain 10 reason']])
print('Category definitions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(school_sector_mobility_category_definitions)
print('Files written in this cell: 0')

School sector and mobility final summary:


,School sector and mobility predictors retained,Participants with all three available,Participants with at least two available,Participants with none available,Outcome columns loaded,Files written
0,3,9169,9505,243,0,0


Retained predictor coverage:


,Predictor,Non-missing,Missing,Missing percentage,Categories,Smallest observed category,Largest observed category
0,independent_school_at_sampling_stage,9524,243,2.49,2,367,9157
1,cumulative_school_history_pretransition,9324,443,4.54,3,997,5237
2,any_school_change_by_wave3,9350,417,4.27,2,326,9024


Joint predictor coverage:


...

History-file variable decisions:


,Variable,Variable label,Domain 10 decision,Domain 10 role,Domain 10 reason
0,W1numsch,DV: No of schools child has attended up to wave 1,Review support,Cross-wave cumulative-structure review,Used to verify that the Wave 2 count is a non-decreasing cumulative update.
1,W1inyrmov,"DV: No of times moved schools in-year (i.e. not in July, August or September) up",Review support,School-mobility representation review,Used to examine the narrower history of out-of-season school moves.
2,W2numsch,DV: No of schools child has attended up to wave 2,Construction input,Input to cumulative school-history predictor,Provides the latest cumulative pre-transition number of schools attended.
3,W2inyrmov,"DV: No of times moved schools in-year (i.e. not in July, August or September) up",Review support,School-mobility representation review,Used to examine the narrower history of out-of-season school moves.


Category definitions:


,Predictor,Category code,Category label
0,independent_school_at_sampling_stage,0,Maintained school
1,independent_school_at_sampling_stage,1,Independent school
2,cumulative_school_history_pretransition,0,One or two schools attended by Wave 2
3,cumulative_school_history_pretransition,1,Three schools attended by Wave 2
4,cumulative_school_history_pretransition,2,Four or more schools attended by Wave 2


Files written in this cell: 0


In [414]:
# 28: Domain 10 predictor consolidation and overlap review

import itertools
import pandas as pd
domain_10_component_expected_columns = {'School climate': ['school_physical_environment_perception_pretransition',
    'teacher_listening_fair_treatment_score_pretransition'], 'Regional context': ['urban_rural_settlement_type_pretransition',
    'government_office_region_pretransition'], 'Institutional guidance': ['teacher_guidance_frequency_index_pretransition',
    'careers_advice_service_guidance_frequency_pretransition', 'connexions_adviser_contact_pretransition', 'learning_mentor_engagement_profile_pretransition', 'stay_on_guidance_profile_pretransition', 'apprenticeship_guidance_profile_pretransition'], 'School sector and mobility': ['independent_school_at_sampling_stage',
    'cumulative_school_history_pretransition', 'any_school_change_by_wave3']}
domain_10_raw_component_specs = {'School climate': school_climate_predictors,
    'Regional context': regional_context_predictors, 'Institutional guidance': guidance_domain_predictors, 'School sector and mobility': school_sector_mobility_predictors}

def normalise_domain_10_component(component_name, component_data):
    component_copy = component_data.copy()
    nsid_column_present = 'NSID' in component_copy.columns
    if nsid_column_present:
        component_nsid = standardise_nsid(component_copy['NSID'])
        assert component_nsid.notna().all()
        assert component_nsid.is_unique
        component_copy = component_copy.drop(columns=['NSID'])
        component_copy.index = pd.Index(component_nsid, name='NSID')
    elif not component_copy.index.equals(route_index):
        component_index = standardise_nsid(pd.Series(component_copy.index, dtype='string'))
        assert component_index.notna().all()
        assert component_index.is_unique
        component_copy.index = pd.Index(component_index, name='NSID')
    else:
        component_copy.index.name = 'NSID'
    assert component_copy.index.is_unique
    missing_participants = route_index.difference(component_copy.index)
    extra_participants = component_copy.index.difference(route_index)
    assert len(missing_participants) == 0
    assert len(extra_participants) == 0
    expected_columns = domain_10_component_expected_columns[component_name]
    missing_columns = [column for column in expected_columns if column not in component_copy.columns]
    unexpected_columns = [column for column in component_copy.columns if column not in expected_columns]
    assert not missing_columns
    assert not unexpected_columns
    component_copy = component_copy[expected_columns].reindex(route_index)
    component_copy.index.name = 'NSID'
    return (component_copy, {'Domain 10 component': component_name, 'Input rows': int(len(component_data)),
        'Input columns': int(component_data.shape[1]), 'NSID column removed': bool(nsid_column_present), 'Retained predictor columns': int(component_copy.shape[1]), 'Participants after reconciliation': int(len(component_copy)), 'Participants with all component predictors': int(component_copy.notna().all(axis=1).sum()), 'Participants with no component predictors': int(component_copy.isna().all(axis=1).sum())})
domain_10_component_specs = {}
domain_10_component_reconciliation_rows = []
for component_name, component_data in domain_10_raw_component_specs.items():
    reconciled_component, reconciliation_record = normalise_domain_10_component(component_name, component_data)
    domain_10_component_specs[component_name] = reconciled_component
    domain_10_component_reconciliation_rows.append(reconciliation_record)
domain_10_component_reconciliation_summary = pd.DataFrame(domain_10_component_reconciliation_rows)
domain_10_expected_predictor_names = [predictor_name for component_columns in domain_10_component_expected_columns.values() for predictor_name in component_columns]
assert len(domain_10_expected_predictor_names) == 13
assert len(set(domain_10_expected_predictor_names)) == 13
domain_10_predictors = pd.concat([domain_10_component_specs['School climate'],
    domain_10_component_specs['Regional context'], domain_10_component_specs['Institutional guidance'], domain_10_component_specs['School sector and mobility']], axis=1).reindex(route_index)
domain_10_predictors.index.name = 'NSID'
assert domain_10_predictors.shape == (9767, 13)
assert not domain_10_predictors.columns.duplicated().any()
assert list(domain_10_predictors.columns) == domain_10_expected_predictor_names
assert 'NSID' not in domain_10_predictors.columns
domain_10_predictor_metadata = pd.DataFrame([{'Predictor': 'school_physical_environment_perception_pretransition',
    'Domain 10 component': 'School climate', 'Representation': 'Ordinal categorical', 'Construction timing': 'Wave 2 primary; Wave 1 fallback'}, {'Predictor': 'teacher_listening_fair_treatment_score_pretransition',
    'Domain 10 component': 'School climate', 'Representation': 'Two-item score', 'Construction timing': 'Wave 2'}, {'Predictor': 'urban_rural_settlement_type_pretransition',
    'Domain 10 component': 'Regional context', 'Representation': 'Four-category nominal', 'Construction timing': 'Wave 3 primary; Wave 2 fallback'}, {'Predictor': 'government_office_region_pretransition',
    'Domain 10 component': 'Regional context', 'Representation': 'Nine-category nominal', 'Construction timing': 'Wave 3 primary; Wave 2 fallback'}, {'Predictor': 'teacher_guidance_frequency_index_pretransition',
    'Domain 10 component': 'Institutional guidance', 'Representation': 'Two-item frequency index', 'Construction timing': 'Wave 2'}, {'Predictor': 'careers_advice_service_guidance_frequency_pretransition',
    'Domain 10 component': 'Institutional guidance', 'Representation': 'Five-category ordinal', 'Construction timing': 'Wave 2'}, {'Predictor': 'connexions_adviser_contact_pretransition',
    'Domain 10 component': 'Institutional guidance', 'Representation': 'Binary cumulative indicator', 'Construction timing': 'Waves 1–3'}, {'Predictor': 'learning_mentor_engagement_profile_pretransition',
    'Domain 10 component': 'Institutional guidance', 'Representation': 'Three-category profile', 'Construction timing': 'Wave 3'}, {'Predictor': 'stay_on_guidance_profile_pretransition',
    'Domain 10 component': 'Institutional guidance', 'Representation': 'Four-category routed profile', 'Construction timing': 'Wave 3'}, {'Predictor': 'apprenticeship_guidance_profile_pretransition',
    'Domain 10 component': 'Institutional guidance', 'Representation': 'Four-category routed profile', 'Construction timing': 'Wave 3'}, {'Predictor': 'independent_school_at_sampling_stage',
    'Domain 10 component': 'School sector and mobility', 'Representation': 'Binary categorical', 'Construction timing': 'Wave 1'}, {'Predictor': 'cumulative_school_history_pretransition',
    'Domain 10 component': 'School sector and mobility', 'Representation': 'Three-category cumulative history', 'Construction timing': 'Wave 2'}, {'Predictor': 'any_school_change_by_wave3',
    'Domain 10 component': 'School sector and mobility', 'Representation': 'Binary cumulative indicator', 'Construction timing': 'Waves 2–3'}])
assert set(domain_10_predictor_metadata['Predictor']) == set(domain_10_predictors.columns)
domain_10_predictor_summary_rows = []
for predictor_name in domain_10_predictors.columns:
    predictor_values = domain_10_predictors[predictor_name]
    observed_counts = predictor_values.dropna().value_counts()
    predictor_metadata = domain_10_predictor_metadata.loc[domain_10_predictor_metadata['Predictor'].eq(predictor_name)].iloc[0]
    domain_10_predictor_summary_rows.append({'Predictor': predictor_name,
        'Domain 10 component': predictor_metadata['Domain 10 component'], 'Representation': predictor_metadata['Representation'], 'Non-missing': int(predictor_values.notna().sum()), 'Missing': int(predictor_values.isna().sum()), 'Missing percentage': round(predictor_values.isna().mean() * 100,
        2), 'Observed categories or values': int(predictor_values.nunique()), 'Smallest observed group': int(observed_counts.min()), 'Largest observed group': int(observed_counts.max())})
domain_10_predictor_summary = pd.DataFrame(domain_10_predictor_summary_rows)
domain_10_available_predictor_count = domain_10_predictors.notna().sum(axis=1)
domain_10_joint_coverage_summary = domain_10_available_predictor_count.value_counts().sort_index().rename('Participants').rename_axis('Domain 10 predictors available').reset_index()
domain_10_joint_coverage_summary['Percentage of full sample'] = (domain_10_joint_coverage_summary['Participants'] / len(domain_10_predictors) * 100).round(2)
domain_10_joint_coverage_overview = pd.DataFrame([{'Full analysis sample': int(len(domain_10_predictors)),
    'Retained Domain 10 predictors': int(domain_10_predictors.shape[1]), 'All 13 predictors available': int(domain_10_available_predictor_count.eq(13).sum()), 'At least 12 predictors available': int(domain_10_available_predictor_count.ge(12).sum()), 'At least 10 predictors available': int(domain_10_available_predictor_count.ge(10).sum()), 'At least one predictor available': int(domain_10_available_predictor_count.ge(1).sum()), 'No Domain 10 predictors available': int(domain_10_available_predictor_count.eq(0).sum())}])
domain_10_pairwise_association_rows = []
for first_predictor, second_predictor in itertools.combinations(domain_10_predictors.columns, 2):
    comparison_data = domain_10_predictors[[first_predictor, second_predictor]].dropna()
    domain_10_pairwise_association_rows.append({'First predictor': first_predictor,
        'Second predictor': second_predictor, 'Complete comparisons': int(len(comparison_data)), 'First predictor categories': int(comparison_data[first_predictor].nunique()), 'Second predictor categories': int(comparison_data[second_predictor].nunique()), "Cramer's V": round(categorical_cramers_v(comparison_data[first_predictor],
        comparison_data[second_predictor]), 3)})
domain_10_pairwise_association_summary = pd.DataFrame(domain_10_pairwise_association_rows).sort_values("Cramer's V",
    ascending=False).reset_index(drop=True)
domain_10_high_association_review = domain_10_pairwise_association_summary.loc[domain_10_pairwise_association_summary["Cramer's V"].ge(0.4)].copy().reset_index(drop=True)
domain_10_exact_duplicate_rows = []
for first_predictor, second_predictor in itertools.combinations(domain_10_predictors.columns, 2):
    comparison_data = domain_10_predictors[[first_predictor, second_predictor]].dropna()
    exact_agreement = comparison_data[first_predictor].astype('string').eq(comparison_data[second_predictor].astype('string')).all()
    if exact_agreement:
        domain_10_exact_duplicate_rows.append({'First predictor': first_predictor,
            'Second predictor': second_predictor, 'Complete comparisons': int(len(comparison_data))})
domain_10_exact_duplicate_summary = pd.DataFrame(domain_10_exact_duplicate_rows, columns=['First predictor',
    'Second predictor', 'Complete comparisons'])
domain_10_consolidation_overview = pd.DataFrame([{'Domain 10 components consolidated': int(len(domain_10_component_specs)),
    'NSID columns removed': int(domain_10_component_reconciliation_summary['NSID column removed'].sum()), 'Retained predictors': int(domain_10_predictors.shape[1]), 'Pairwise associations reviewed': int(len(domain_10_pairwise_association_summary)), 'Associations at or above 0.40': int(len(domain_10_high_association_review)), 'Exact duplicate predictor pairs': int(len(domain_10_exact_duplicate_summary)), 'Outcome columns loaded': 0, 'Files written': 0}])
print('Domain 10 consolidation overview:')
display_limited(domain_10_consolidation_overview)
print('Component reconciliation:')
display_limited(domain_10_component_reconciliation_summary)
print('Domain 10 joint coverage overview:')
display_limited(domain_10_joint_coverage_overview)
print('Domain 10 joint coverage distribution:')
display_limited(domain_10_joint_coverage_summary)
print('Retained Domain 10 predictor summary:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_10_predictor_summary)
print('Highest pairwise associations:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_10_pairwise_association_summary.head(20))
print('Associations at or above 0.40:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_10_high_association_review)
print('Exact duplicate predictor pairs:')
display_limited(domain_10_exact_duplicate_summary)
print('Files written in this cell: 0')

Domain 10 consolidation overview:


,Domain 10 components consolidated,NSID columns removed,Retained predictors,Pairwise associations reviewed,Associations at or above 0.40,Exact duplicate predictor pairs,Outcome columns loaded,Files written
0,4,2,13,78,0,0,0,0


Component reconciliation:


,Domain 10 component,Input rows,Input columns,NSID column removed,Retained predictor columns,Participants after reconciliation,Participants with all component predictors,Participants with no component predictors
0,School climate,9767,3,True,2,9767,8915,289
1,Regional context,9767,3,True,2,9767,9520,247
2,Institutional guidance,9767,6,False,6,9767,8882,246
3,School sector and mobility,9767,3,False,3,9767,9169,243


Domain 10 joint coverage overview:


...

Associations at or above 0.40:


,First predictor,Second predictor,Complete comparisons,First predictor categories,Second predictor categories,Cramer's V


Exact duplicate predictor pairs:


,First predictor,Second predictor,Complete comparisons


Files written in this cell: 0


In [415]:
# 29: Existing Domain 10 decision-object review

import pandas as pd
domain_10_decision_object_names = ['school_climate_variable_decisions', 'regional_context_variable_decisions',
    'guidance_variable_decisions', 'school_history_variable_decisions', 'school_climate_representation_decisions', 'regional_context_representation_decisions', 'guidance_representation_decisions', 'school_mobility_representation_decisions']
domain_10_decision_object_rows = []
for object_name in domain_10_decision_object_names:
    object_value = globals().get(object_name)
    if isinstance(object_value, pd.DataFrame):
        key_columns_available = {'Source file', 'Variable'}.issubset(object_value.columns)
        duplicated_keys = int(object_value[['Source file',
            'Variable']].duplicated().sum()) if key_columns_available else pd.NA
        domain_10_decision_object_rows.append({'Object': object_name, 'Status': 'Available',
            'Rows': int(object_value.shape[0]), 'Columns': int(object_value.shape[1]), 'Source-variable keys available': key_columns_available, 'Duplicated source-variable keys': duplicated_keys, 'Column names': ' | '.join(object_value.columns.astype(str).tolist())})
    else:
        domain_10_decision_object_rows.append({'Object': object_name, 'Status': 'Not available', 'Rows': pd.NA,
            'Columns': pd.NA, 'Source-variable keys available': False, 'Duplicated source-variable keys': pd.NA, 'Column names': pd.NA})
domain_10_decision_object_review = pd.DataFrame(domain_10_decision_object_rows)
domain_10_decision_object_overview = pd.DataFrame([{'Objects checked': int(len(domain_10_decision_object_review)),
    'Objects available': int(domain_10_decision_object_review['Status'].eq('Available').sum()), 'Source-variable decision objects': int(domain_10_decision_object_review['Source-variable keys available'].sum()), 'Objects with duplicated source-variable keys': int(domain_10_decision_object_review['Duplicated source-variable keys'].fillna(0).gt(0).sum()), 'Files written': 0}])
print('Domain 10 decision-object overview:')
display_limited(domain_10_decision_object_overview)
print('Domain 10 decision-object structure:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_10_decision_object_review)
print('Files written in this cell: 0')

Domain 10 decision-object overview:


,Objects checked,Objects available,Source-variable decision objects,Objects with duplicated source-variable keys,Files written
0,8,8,4,0,0


Domain 10 decision-object structure:


,Object,Status,Rows,Columns,Source-variable keys available,Duplicated source-variable keys,Column names
0,school_climate_variable_decisions,Available,5,20,True,0,Source order | Wave | Source type | Source file | Variable position | Variable | Variable label | Data type | Timing status | Review status | Review outcome | Substantive domain | Decision reason | Leakage assessment | Reference-period assessment | Documentation source | Review notes | Domain 10 decision | Domain 10 role | Domain 10 reason
1,regional_context_variable_decisions,Available,4,14,True,0,Source order | Wave | Source type | Source file | Variable position | Variable | Variable label | Timing status | Prior review outcome | Domain 10 review track | Domain 10 screening reason | Domain 10 decision | Domain 10 role | Domain 10 reason
2,guidance_variable_decisions,Available,69,14,True,0,Source order | Wave | Source file | Variable position | Variable | Register variable label | Full Stata variable label | Guidance audit group | Current guidance core item | Battery suffix | Domain 10 decision | Domain 10 role | Domain 10 reason | Domain 10 timing assessment
3,school_history_variable_decisions,Available,4,20,True,0,Source order | Wave | Source type | Source file | Variable position | Variable | Variable label | Data type | Timing status | Review status | Review outcome | Substantive domain | Decision reason | Leakage assessment | Reference-period assessment | Documentation source | Review notes | Domain 10 decision | Domain 10 role | Domain 10 reason
4,school_climate_representation_decisions,Available,5,4,False,<NA>,Candidate representation | Decision | Final predictor | Reason


Files written in this cell: 0


In [416]:
# 30: Reviewed Domain 10 source-decision consolidation

import pandas as pd

def standardise_domain_10_decisions(decision_data):
    label_column = next((column for column in ['Variable label', 'Register variable label',
        'Full Stata variable label'] if column in decision_data.columns), None)
    timing_column = next((column for column in ['Domain 10 timing assessment',
        'Timing status'] if column in decision_data.columns), None)
    result = decision_data[['Source file', 'Variable', 'Domain 10 decision', 'Domain 10 role',
        'Domain 10 reason']].copy()
    result['Variable label'] = decision_data[label_column] if label_column else pd.NA
    result['Domain 10 timing assessment'] = decision_data[timing_column] if timing_column else pd.NA
    return result
domain_10_reviewed_decision_parts = [standardise_domain_10_decisions(school_climate_variable_decisions),
    standardise_domain_10_decisions(regional_context_variable_decisions), standardise_domain_10_decisions(guidance_variable_decisions), standardise_domain_10_decisions(school_history_variable_decisions)]
school_context_source_specs = pd.DataFrame([{'Source file': 'wave_one_lsype_young_person_2020',
    'Variable': 'IndSchool', 'Domain 10 decision': 'Construction input', 'Domain 10 role': 'Input to sampling-stage school-sector predictor', 'Domain 10 reason': 'Provides the retained maintained or independent school-sector measure.'}, {'Source file': 'wave_one_lsype_young_person_2020',
    'Variable': 'W1stschHS', 'Domain 10 decision': 'Review support', 'Domain 10 role': 'School-change timing review', 'Domain 10 reason': 'Used to examine school-start timing but not retained as a comparable repeated measure.'}, {'Source file': 'wave_two_lsype_young_person_2020',
    'Variable': 'W2SchstyHS', 'Domain 10 decision': 'Review support', 'Domain 10 role': 'School-change timing review', 'Domain 10 reason': 'Used to review school-change routing and timing.'}, {'Source file': 'wave_two_lsype_young_person_2020',
    'Variable': 'W2stschHS', 'Domain 10 decision': 'Review support', 'Domain 10 role': 'School-change timing review', 'Domain 10 reason': 'Used to review school-change routing and timing.'}, {'Source file': 'wave_two_lsype_young_person_2020',
    'Variable': 'W2schnameMP', 'Domain 10 decision': 'Construction input', 'Domain 10 role': 'Wave 2 input to direct school-change predictor', 'Domain 10 reason': 'Identifies whether the young person was no longer at the same school by Wave 2.'}, {'Source file': 'wave_three_lsype_young_person_2020',
    'Variable': 'W3schnameYP', 'Domain 10 decision': 'Construction input', 'Domain 10 role': 'Wave 3 input to direct school-change predictor', 'Domain 10 reason': 'Identifies a further school change before the post-16 transition.'}, {'Source file': 'wave_three_lsype_young_person_2020',
    'Variable': 'W3stschHS', 'Domain 10 decision': 'Review support', 'Domain 10 role': 'School-change timing review', 'Domain 10 reason': 'Used to review school-change routing and timing.'}, {'Source file': 'wave_three_lsype_young_person_2020',
    'Variable': 'W3SchstyHS', 'Domain 10 decision': 'Review support', 'Domain 10 role': 'School-change timing review', 'Domain 10 reason': 'Used to review school-change routing and timing.'}])
school_context_source_decisions = school_context_source_specs.merge(working_variable_decision_register[['Source file',
    'Variable', 'Variable label', 'Timing status']], on=['Source file',
    'Variable'], how='left', validate='one_to_one').rename(columns={'Timing status': 'Domain 10 timing assessment'})
assert school_context_source_decisions[['Variable label', 'Domain 10 timing assessment']].notna().all().all()
domain_10_reviewed_decision_parts.append(school_context_source_decisions)
domain_10_reviewed_source_decisions = pd.concat(domain_10_reviewed_decision_parts,
    ignore_index=True).sort_values(['Source file', 'Variable']).reset_index(drop=True)
assert domain_10_reviewed_source_decisions[['Source file', 'Variable']].duplicated().sum() == 0
assert domain_10_reviewed_source_decisions[['Domain 10 decision', 'Domain 10 role', 'Domain 10 reason',
    'Domain 10 timing assessment']].notna().all().all()
domain_10_reviewed_decision_summary = domain_10_reviewed_source_decisions['Domain 10 decision'].value_counts().rename('Variables').rename_axis('Domain 10 decision').reset_index()
domain_10_reviewed_source_overview = pd.DataFrame([{'Reviewed source variables': int(len(domain_10_reviewed_source_decisions)),
    'Construction inputs': int(domain_10_reviewed_source_decisions['Domain 10 decision'].eq('Construction input').sum()), 'Review-support variables': int(domain_10_reviewed_source_decisions['Domain 10 decision'].eq('Review support').sum()), 'Excluded variables': int(domain_10_reviewed_source_decisions['Domain 10 decision'].eq('Exclude').sum()), 'Duplicated source-variable keys': int(domain_10_reviewed_source_decisions[['Source file',
    'Variable']].duplicated().sum()), 'Files written': 0}])
print('Reviewed Domain 10 source-decision overview:')
display_limited(domain_10_reviewed_source_overview)
print('Reviewed source-variable decisions:')
display_limited(domain_10_reviewed_decision_summary)
print('School-sector and change source decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(school_context_source_decisions)
print('Files written in this cell: 0')

Reviewed Domain 10 source-decision overview:


,Reviewed source variables,Construction inputs,Review-support variables,Excluded variables,Duplicated source-variable keys,Files written
0,90,62,22,6,0,0


Reviewed source-variable decisions:


,Domain 10 decision,Variables
0,Construction input,62
1,Review support,22
2,Exclude,6


School-sector and change source decisions:


,Source file,Variable,Domain 10 decision,Domain 10 role,Domain 10 reason,Variable label,Domain 10 timing assessment
0,wave_one_lsype_young_person_2020,IndSchool,Construction input,Input to sampling-stage school-sector predictor,Provides the retained maintained or independent school-sector measure.,DV: Whether YP was at an independent or maintained school at sampling stage,Pre-transition source
1,wave_one_lsype_young_person_2020,W1stschHS,Review support,School-change timing review,Used to examine school-start timing but not retained as a comparable repeated measure.,DV: Month YP started school,Pre-transition source
2,wave_two_lsype_young_person_2020,W2SchstyHS,Review support,School-change timing review,Used to review school-change routing and timing.,MP: Year YP started school,Pre-transition source
3,wave_two_lsype_young_person_2020,W2stschHS,Review support,School-change timing review,Used to review school-change routing and timing.,DV: Month YP started school,Pre-transition source
4,wave_two_lsype_young_person_2020,W2schnameMP,Construction input,Wave 2 input to direct school-change predictor,Identifies whether the young person was no longer at the same school by Wave 2.,MP: Whether YP still at same school,Pre-transition source


Files written in this cell: 0


In [417]:
# 31: Domain 10 representation-decision consolidation

import pandas as pd
domain_10_representation_parts = {'School climate': school_climate_representation_decisions,
    'Regional context': regional_context_representation_decisions, 'Institutional guidance': guidance_representation_decisions, 'School sector and mobility': school_mobility_representation_decisions}
domain_10_representation_decisions = pd.concat([decision_data.assign(**{'Domain 10 component': component_name}) for component_name,
    decision_data in domain_10_representation_parts.items()], ignore_index=True)
domain_10_representation_decisions = domain_10_representation_decisions[['Domain 10 component',
    'Candidate representation', 'Decision', 'Final predictor', 'Reason']].reset_index(drop=True)
assert domain_10_representation_decisions[['Domain 10 component', 'Candidate representation', 'Decision',
    'Reason']].notna().all().all()
invalid_final_predictor_mask = domain_10_representation_decisions['Final predictor'].notna() & ~domain_10_representation_decisions['Final predictor'].isin(domain_10_predictors.columns)
assert not invalid_final_predictor_mask.any()
domain_10_representation_decision_summary = domain_10_representation_decisions['Decision'].value_counts().rename('Representations').rename_axis('Decision').reset_index()
domain_10_representation_component_summary = domain_10_representation_decisions.groupby('Domain 10 component').agg(Representations=('Candidate representation',
    'size'), Linked_to_final_predictor=('Final predictor', lambda values: int(values.notna().sum()))).reset_index()
domain_10_representation_overview = pd.DataFrame([{'Components combined': int(len(domain_10_representation_parts)),
    'Representation decisions': int(len(domain_10_representation_decisions)), 'Representations linked to a final predictor': int(domain_10_representation_decisions['Final predictor'].notna().sum()), 'Invalid final-predictor links': int(invalid_final_predictor_mask.sum()), 'Files written': 0}])
print('Domain 10 representation-decision overview:')
display_limited(domain_10_representation_overview)
print('Representation decisions by component:')
display_limited(domain_10_representation_component_summary)
print('Representation decision summary:')
display_limited(domain_10_representation_decision_summary)
print('Files written in this cell: 0')

Domain 10 representation-decision overview:


,Components combined,Representation decisions,Representations linked to a final predictor,Invalid final-predictor links,Files written
0,4,28,14,0,0


Representation decisions by component:


,Domain 10 component,Representations,Linked_to_final_predictor
0,Institutional guidance,11,6
1,Regional context,4,2
2,School climate,5,2
3,School sector and mobility,8,4


Representation decision summary:


,Decision,Representations
0,Do not retain,12
1,Retain,9
2,Combine,4
3,Review support,2
4,Retain as construction source,1


Files written in this cell: 0


In [418]:
# 32: Domain 10 output saving

from pathlib import Path
import pandas as pd
domain_10_output_paths = {'Predictors': stage_2_output_directory / 'stage_2_school_local_context_domain_predictors.csv',
    'Source decisions': stage_2_output_directory / 'stage_2_school_local_context_domain_source_decisions.csv', 'Representation decisions': stage_2_output_directory / 'stage_2_school_local_context_domain_representation_decisions.csv', 'Predictor summary': stage_2_output_directory / 'stage_2_school_local_context_domain_predictor_summary.csv', 'Predictor metadata': stage_2_output_directory / 'stage_2_school_local_context_domain_predictor_metadata.csv'}
domain_10_predictor_output = domain_10_predictors.reset_index()
assert domain_10_predictor_output.shape == (9767, 14)
assert domain_10_predictor_output['NSID'].is_unique
domain_10_predictor_output.to_csv(domain_10_output_paths['Predictors'], index=False)
domain_10_reviewed_source_decisions.to_csv(domain_10_output_paths['Source decisions'], index=False)
domain_10_representation_decisions.to_csv(domain_10_output_paths['Representation decisions'], index=False)
domain_10_predictor_summary.to_csv(domain_10_output_paths['Predictor summary'], index=False)
domain_10_predictor_metadata.to_csv(domain_10_output_paths['Predictor metadata'], index=False)
domain_10_file_verification_rows = []
for output_name, output_path in domain_10_output_paths.items():
    saved_data = pd.read_csv(output_path, low_memory=False)
    domain_10_file_verification_rows.append({'Output': output_name, 'File': output_path.name,
        'Exists': output_path.exists(), 'Rows': int(saved_data.shape[0]), 'Columns': int(saved_data.shape[1]), 'Non-empty': bool(output_path.stat().st_size > 0)})
domain_10_file_verification = pd.DataFrame(domain_10_file_verification_rows)
assert domain_10_file_verification['Exists'].all()
assert domain_10_file_verification['Non-empty'].all()
assert domain_10_file_verification.loc[domain_10_file_verification['Output'].eq('Predictors'), 'Rows'].iloc[0] == 9767
assert domain_10_file_verification.loc[domain_10_file_verification['Output'].eq('Source decisions'),
    'Rows'].iloc[0] == 90
assert domain_10_file_verification.loc[domain_10_file_verification['Output'].eq('Representation decisions'),
    'Rows'].iloc[0] == 28
domain_10_saved_output_summary = pd.DataFrame([{'Domain 10 predictors saved': int(domain_10_predictors.shape[1]),
    'Participant rows saved': int(len(domain_10_predictors)), 'Source-variable decisions saved': int(len(domain_10_reviewed_source_decisions)), 'Representation decisions saved': int(len(domain_10_representation_decisions)), 'Output files written': int(len(domain_10_output_paths))}])
print('Domain 10 saved-output summary:')
display_limited(domain_10_saved_output_summary)
print('Domain 10 file verification:')
with pd.option_context('display.max_colwidth', None):
    display_limited(domain_10_file_verification)

Domain 10 saved-output summary:


,Domain 10 predictors saved,Participant rows saved,Source-variable decisions saved,Representation decisions saved,Output files written
0,13,9767,90,28,5


Domain 10 file verification:


,Output,File,Exists,Rows,Columns,Non-empty
0,Predictors,stage_2_school_local_context_domain_predictors.csv,True,9767,14,True
1,Source decisions,stage_2_school_local_context_domain_source_decisions.csv,True,90,7,True
2,Representation decisions,stage_2_school_local_context_domain_representation_decisions.csv,True,28,5,True
3,Predictor summary,stage_2_school_local_context_domain_predictor_summary.csv,True,13,9,True
4,Predictor metadata,stage_2_school_local_context_domain_predictor_metadata.csv,True,13,4,True


In [419]:
# 33: Domain 10 register-update review

import pandas as pd
register_key_columns = ['Source file', 'Variable']
domain_10_register_review = domain_10_reviewed_source_decisions.merge(working_variable_decision_register[['Source file',
    'Variable', 'Review status', 'Review outcome', 'Substantive domain', 'Decision reason', 'Review notes']], on=register_key_columns, how='left', validate='one_to_one', indicator=True)
assert len(domain_10_register_review) == 90
assert domain_10_register_review['_merge'].eq('both').all()

def classify_existing_register_decision(review_outcome):
    if pd.isna(review_outcome):
        return 'No completed decision'
    outcome_text = str(review_outcome).strip().lower()
    if outcome_text in {'', 'pending review'}:
        return 'No completed decision'
    if 'construction' in outcome_text or 'construction source' in outcome_text:
        return 'Construction input'
    if 'review support' in outcome_text or 'support only' in outcome_text:
        return 'Review support'
    if outcome_text.startswith('exclude'):
        return 'Exclude'
    return 'Other completed decision'
domain_10_register_review['Existing decision group'] = domain_10_register_review['Review outcome'].map(classify_existing_register_decision)
domain_10_register_review['Decision relationship'] = 'Different completed decision'
no_completed_decision_mask = domain_10_register_review['Existing decision group'].eq('No completed decision')
compatible_decision_mask = domain_10_register_review['Existing decision group'].eq(domain_10_register_review['Domain 10 decision'])
domain_10_register_review.loc[no_completed_decision_mask, 'Decision relationship'] = 'New Domain 10 decision'
domain_10_register_review.loc[compatible_decision_mask, 'Decision relationship'] = 'Compatible existing decision'
domain_10_register_review['Existing other-domain assignment'] = domain_10_register_review['Substantive domain'].notna() & ~domain_10_register_review['Substantive domain'].astype('string').str.contains('school context|school and local context',
    case=False, na=False)
domain_10_register_review_summary = pd.DataFrame([{'Domain 10 decisions reviewed': int(len(domain_10_register_review)),
    'Keys matched to register': int(domain_10_register_review['_merge'].eq('both').sum()), 'New Domain 10 decisions': int(domain_10_register_review['Decision relationship'].eq('New Domain 10 decision').sum()), 'Compatible existing decisions': int(domain_10_register_review['Decision relationship'].eq('Compatible existing decision').sum()), 'Different completed decisions': int(domain_10_register_review['Decision relationship'].eq('Different completed decision').sum()), 'Existing other-domain assignments': int(domain_10_register_review['Existing other-domain assignment'].sum()), 'Files written': 0}])
domain_10_decision_relationship_summary = domain_10_register_review['Decision relationship'].value_counts().rename('Variables').rename_axis('Decision relationship').reset_index()
domain_10_different_existing_decisions = domain_10_register_review.loc[domain_10_register_review['Decision relationship'].eq('Different completed decision')].copy().reset_index(drop=True)
domain_10_other_domain_assignments = domain_10_register_review.loc[domain_10_register_review['Existing other-domain assignment']].copy().reset_index(drop=True)
print('Domain 10 register-update review:')
display_limited(domain_10_register_review_summary)
print('Decision relationships:')
display_limited(domain_10_decision_relationship_summary)
print('Different completed decisions:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_10_different_existing_decisions[['Source file', 'Variable', 'Variable label',
        'Domain 10 decision', 'Review outcome', 'Substantive domain', 'Domain 10 role']])
print('Existing assignments to other domains:')
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display_limited(domain_10_other_domain_assignments[['Source file', 'Variable', 'Variable label',
        'Domain 10 decision', 'Review outcome', 'Substantive domain', 'Domain 10 role']])
print('Files written in this cell: 0')

Domain 10 register-update review:


,Domain 10 decisions reviewed,Keys matched to register,New Domain 10 decisions,Compatible existing decisions,Different completed decisions,Existing other-domain assignments,Files written
0,90,90,78,9,3,4,0


Decision relationships:


,Decision relationship,Variables
0,New Domain 10 decision,78
1,Compatible existing decision,9
2,Different completed decision,3


Different completed decisions:


,Source file,Variable,Variable label,Domain 10 decision,Review outcome,Substantive domain,Domain 10 role
0,wave_two_lsype_young_person_2020,W2YYS25YP,YP: How many teachers this applies to: My teachers don't really listen to what I,Construction input,Review support,School attitudes and engagement,Input to the retained teacher listening and fair-treatment score
1,wave_two_lsype_young_person_2020,W2YYS26YP,YP: How many teachers this applies to: I get treated unfairly by my teachers,Construction input,Review support,School attitudes and engagement,Input to the retained teacher listening and fair-treatment score
2,wave_two_lsype_young_person_2020,W2infon,Admin: Person number of household reference person,Exclude,Retain as review support only,Data provenance,Search false positive


Existing assignments to other domains:


,Source file,Variable,Variable label,Domain 10 decision,Review outcome,Substantive domain,Domain 10 role
0,wave_two_lsype_young_person_2020,W2YYS24YP,YP: How many teachers this applies to: My teachers treat everyone the same regar,Review support,Review support,School attitudes and engagement,General equal-treatment item used in the teacher-climate representation review
1,wave_two_lsype_young_person_2020,W2YYS25YP,YP: How many teachers this applies to: My teachers don't really listen to what I,Construction input,Review support,School attitudes and engagement,Input to the retained teacher listening and fair-treatment score
2,wave_two_lsype_young_person_2020,W2YYS26YP,YP: How many teachers this applies to: I get treated unfairly by my teachers,Construction input,Review support,School attitudes and engagement,Input to the retained teacher listening and fair-treatment score
3,wave_two_lsype_young_person_2020,W2infon,Admin: Person number of household reference person,Exclude,Retain as review support only,Data provenance,Search false positive


Files written in this cell: 0


In [420]:
# 34: Domain 10 master-register recording

import pandas as pd
domain_10_register_overlay = domain_10_reviewed_source_decisions.merge(domain_10_register_review[['Source file',
    'Variable', 'Decision relationship']], on=['Source file', 'Variable'], how='left', validate='one_to_one')
promotion_keys = {('wave_two_lsype_young_person_2020', 'W2YYS25YP'), ('wave_two_lsype_young_person_2020',
    'W2YYS26YP')}
domain_10_register_overlay['Register action'] = 'Preserve existing decision'
new_decision_mask = domain_10_register_overlay['Decision relationship'].eq('New Domain 10 decision')
promotion_mask = pd.Series([(row['Source file'], row['Variable']) in promotion_keys for _,
    row in domain_10_register_overlay.iterrows()], index=domain_10_register_overlay.index)
domain_10_register_overlay.loc[new_decision_mask, 'Register action'] = 'Apply new Domain 10 decision'
domain_10_register_overlay.loc[promotion_mask, 'Register action'] = 'Promote to construction input'
register_outcome_map = {'Construction input': 'Construction input', 'Review support': 'Review support',
    'Exclude': 'Exclude from predictor set'}
domain_10_working_variable_decision_register = working_variable_decision_register.copy()
register_key_to_index = {(row['Source file'], row['Variable']): row_index for row_index,
    row in domain_10_working_variable_decision_register.iterrows()}
assert len(register_key_to_index) == len(domain_10_working_variable_decision_register)

def append_register_note(existing_note, new_note):
    if pd.isna(existing_note) or not str(existing_note).strip():
        return new_note
    existing_text = str(existing_note).strip()
    if new_note in existing_text:
        return existing_text
    return existing_text + ' ' + new_note
for _, decision_row in domain_10_register_overlay.iterrows():
    register_key = (decision_row['Source file'], decision_row['Variable'])
    register_index = register_key_to_index[register_key]
    register_action = decision_row['Register action']
    if register_action in {'Apply new Domain 10 decision', 'Promote to construction input'}:
        domain_10_working_variable_decision_register.loc[register_index,
            'Review status'] = 'Domain 10 review completed'
        domain_10_working_variable_decision_register.loc[register_index,
            'Review outcome'] = register_outcome_map[decision_row['Domain 10 decision']]
        domain_10_working_variable_decision_register.loc[register_index,
            'Substantive domain'] = 'School and local context'
        domain_10_working_variable_decision_register.loc[register_index,
            'Decision reason'] = decision_row['Domain 10 reason']
        if pd.isna(domain_10_working_variable_decision_register.loc[register_index, 'Leakage assessment']):
            domain_10_working_variable_decision_register.loc[register_index,
                'Leakage assessment'] = 'Outcome-blind predictor review'
        if pd.isna(domain_10_working_variable_decision_register.loc[register_index, 'Reference-period assessment']):
            domain_10_working_variable_decision_register.loc[register_index,
                'Reference-period assessment'] = decision_row['Domain 10 timing assessment']
        if pd.isna(domain_10_working_variable_decision_register.loc[register_index, 'Documentation source']):
            domain_10_working_variable_decision_register.loc[register_index,
                'Documentation source'] = 'Next Steps variable labels and Stage 2 Domain 10 review'
    domain_10_note = 'Domain 10 role: ' + str(decision_row['Domain 10 role']) + '.'
    existing_note = domain_10_working_variable_decision_register.loc[register_index, 'Review notes']
    domain_10_working_variable_decision_register.loc[register_index,
        'Review notes'] = append_register_note(existing_note, domain_10_note)
assert domain_10_working_variable_decision_register.shape == working_variable_decision_register.shape
assert domain_10_working_variable_decision_register[['Source file', 'Variable']].duplicated().sum() == 0
promotion_verification = domain_10_working_variable_decision_register.loc[[(source_file,
    variable) in promotion_keys for source_file, variable in zip(domain_10_working_variable_decision_register['Source file'],
    domain_10_working_variable_decision_register['Variable'])], ['Source file', 'Variable', 'Review outcome',
    'Substantive domain', 'Decision reason']].copy().reset_index(drop=True)
assert len(promotion_verification) == 2
assert promotion_verification['Review outcome'].eq('Construction input').all()
assert promotion_verification['Substantive domain'].eq('School and local context').all()
w2infon_verification = domain_10_working_variable_decision_register.loc[domain_10_working_variable_decision_register['Source file'].eq('wave_two_lsype_young_person_2020') & domain_10_working_variable_decision_register['Variable'].eq('W2infon'),
    ['Source file', 'Variable', 'Review outcome', 'Substantive domain', 'Review notes']].copy().reset_index(drop=True)
assert len(w2infon_verification) == 1
assert w2infon_verification.loc[0, 'Substantive domain'] == 'Data provenance'
master_register_path = decision_register_path
domain_10_working_variable_decision_register.to_csv(master_register_path, index=False)
saved_master_register = pd.read_csv(master_register_path, low_memory=False)
assert saved_master_register.shape == domain_10_working_variable_decision_register.shape
assert saved_master_register[['Source file', 'Variable']].duplicated().sum() == 0
working_variable_decision_register = domain_10_working_variable_decision_register.copy()
domain_10_register_action_summary = domain_10_register_overlay['Register action'].value_counts().rename('Variables').rename_axis('Register action').reset_index()
domain_10_register_update_summary = pd.DataFrame([{'Register rows': int(len(working_variable_decision_register)),
    'Domain 10 source variables reviewed': int(len(domain_10_register_overlay)), 'New decisions applied': int(new_decision_mask.sum()), 'Variables promoted to construction input': int(promotion_mask.sum()), 'Existing decisions preserved': int(domain_10_register_overlay['Register action'].eq('Preserve existing decision').sum()), 'Duplicated register keys': int(working_variable_decision_register[['Source file',
    'Variable']].duplicated().sum()), 'Files written': 1}])
print('Domain 10 register-update summary:')
display_limited(domain_10_register_update_summary)
print('Register actions:')
display_limited(domain_10_register_action_summary)
print('Promoted teacher-climate variables:')
with pd.option_context('display.max_colwidth', None):
    display_limited(promotion_verification)
print('Preserved data-provenance decision:')
with pd.option_context('display.max_colwidth', None):
    display_limited(w2infon_verification)
print('Updated master register:')
print(master_register_path)

Domain 10 register-update summary:


,Register rows,Domain 10 source variables reviewed,New decisions applied,Variables promoted to construction input,Existing decisions preserved,Duplicated register keys,Files written
0,5261,90,78,2,10,0,1


Register actions:


,Register action,Variables
0,Apply new Domain 10 decision,78
1,Preserve existing decision,10
2,Promote to construction input,2


Promoted teacher-climate variables:


,Source file,Variable,Review outcome,Substantive domain,Decision reason
0,wave_two_lsype_young_person_2020,W2YYS25YP,Construction input,School and local context,The item measures whether teachers listen to pupils and is combined with the related personal-treatment item
1,wave_two_lsype_young_person_2020,W2YYS26YP,Construction input,School and local context,The item measures perceived unfair personal treatment and is combined with the related teacher-listening item


Preserved data-provenance decision:


,Source file,Variable,Review outcome,Substantive domain,Review notes
0,wave_two_lsype_young_person_2020,W2infon,Retain as review support only,Data provenance,Retain outside the predictor matrix for respondent and record-source checks only. Domain 10 role: Search false positive.


Updated master register:
data_derived\stage_2_predictor_construction\stage_2_variable_decision_register.csv


In [421]:
# 35: Domain 10 completion check

import pandas as pd
domain_10_completion_audit = pd.DataFrame([{'Participant rows': int(len(domain_10_predictors)),
    'Retained predictors': int(domain_10_predictors.shape[1]), 'Reviewed source variables': int(len(domain_10_reviewed_source_decisions)), 'Representation decisions': int(len(domain_10_representation_decisions)), 'Participants with all predictors': int(domain_10_predictors.notna().all(axis=1).sum()), 'Pairwise associations at or above 0.40': int(len(domain_10_high_association_review)), 'Master-register rows': int(len(working_variable_decision_register)), 'Duplicated register keys': int(working_variable_decision_register[['Source file',
    'Variable']].duplicated().sum()), 'Domain 10 output files': int(len(domain_10_output_paths)), 'Outcome columns loaded': 0}])
domain_10_completion_status = pd.DataFrame([{'Domain': 'School and local context', 'Status': 'Completed',
    'Predictor file': domain_10_output_paths['Predictors'].name, 'Master register': master_register_path.name}])
assert domain_10_completion_audit.loc[0, 'Participant rows'] == 9767
assert domain_10_completion_audit.loc[0, 'Retained predictors'] == 13
assert domain_10_completion_audit.loc[0, 'Duplicated register keys'] == 0
assert domain_10_completion_audit.loc[0, 'Pairwise associations at or above 0.40'] == 0
print('Domain 10 completion audit:')
display_limited(domain_10_completion_audit)
print('Domain 10 completion status:')
display_limited(domain_10_completion_status)

Domain 10 completion audit:


,Participant rows,Retained predictors,Reviewed source variables,Representation decisions,Participants with all predictors,Pairwise associations at or above 0.40,Master-register rows,Duplicated register keys,Domain 10 output files,Outcome columns loaded
0,9767,13,90,28,8158,0,5261,0,5,0


Domain 10 completion status:


,Domain,Status,Predictor file,Master register
0,School and local context,Completed,stage_2_school_local_context_domain_predictors...,stage_2_variable_decision_register.csv


## School and local-context predictors

Thirteen predictors were retained: two school-climate measures, two regional-context measures, six institutional-guidance measures and three school-sector or mobility measures.

All 13 were available for 8,158 participants (83.53%); 9,227 participants (94.47%) had at least 12, and 243 had none. No exact duplicate pair or association of 0.40 or above was found. Decisions were recorded for 90 source variables, alongside 28 alternative representations.


# Part 14: Predictor specification and provenance checks

The domain sections above establish source eligibility, timing, coding and representation. This section constructs the predictors that require cross-domain handling and writes the 11 domain files used in the consolidated matrix.

Home language uses Wave 1 information only. Academic self-concept requires both Wave 1 items. School exclusion, home-computer access, family and peer discussion of future study, and Year 10 vocational-course study are included as distinct pre-transition predictors.

GHQ-12 is assigned to psychosocial characteristics. School climate and mobility are assigned to school experiences and engagement. Peer, family and institutional guidance are grouped within post-16 social influences and guidance.

All files created in this section are saved in `data_derived/stage_2_predictor_construction`.

In [422]:
# 1: Construct specified predictors

from pathlib import Path
import numpy as np
import pandas as pd

def load_aligned_stata(file_path, variables, labelled=False):
    """Load selected Stata variables and align them to the Stage 2 roster."""
    data = pd.read_stata(file_path, columns=['NSID', *variables], convert_categoricals=labelled)
    data['NSID'] = standardise_nsid(data['NSID'])
    assert data['NSID'].notna().all()
    assert data['NSID'].is_unique
    return data.set_index('NSID').reindex(route_index)

def recode_labelled_yes_no(values):
    """Recode labelled Yes and No responses, leaving other values missing."""
    labels = values.astype('string').str.strip().str.lower()
    result = pd.Series(pd.NA, index=values.index, dtype='Int64')
    result.loc[labels.str.match('^yes(?:\\b|$)', na=False)] = 1
    result.loc[labels.str.match('^no(?:\\b|$)', na=False)] = 0
    return result
non_english_home_language_pretransition = home_language_consistency_data[['NSID',
    'W1 non-English language recorded']].assign(NSID=lambda data: standardise_nsid(data['NSID'])).set_index('NSID')['W1 non-English language recorded'].where(lambda values: values.isin(['Yes',
    'No'])).map({'Yes': 1, 'No': 0}).astype('Int64').reindex(route_index).rename('non_english_home_language')
assert set(non_english_home_language_pretransition.dropna().astype(int).unique()).issubset({0, 1})
academic_self_concept_score_final = academic_self_concept_complete_score.copy().rename('academic_self_concept_score')
assert int(academic_self_concept_score_final.notna().sum()) == 9172
school_exclusion_history_pretransition = school_exclusion_history.copy().astype('Int64').rename('school_exclusion_history_pretransition')
assert set(school_exclusion_history_pretransition.dropna().astype(int).unique()).issubset({0, 1})
with pd.io.stata.StataReader(wave_2_family_background_file, convert_categoricals=False) as reader:
    wave_2_family_variable_labels = reader.variable_labels()
home_computer_variable = 'W2condur5MP'
assert home_computer_variable in wave_2_family_variable_labels
assert 'home computer' in wave_2_family_variable_labels[home_computer_variable].lower()
home_computer_labelled = load_aligned_stata(wave_2_family_background_file, [home_computer_variable], labelled=True)
home_computer_access_pretransition = recode_labelled_yes_no(home_computer_labelled[home_computer_variable]).rename('home_computer_access_pretransition')
assert set(home_computer_access_pretransition.dropna().astype(int).unique()) == {0, 1}
future_study_variables = ['W1advfamYP', 'W2AdvFamYP', 'W1advpalYP', 'W2AdvPalYP']
wave_1_future_study_raw = load_aligned_stata(wave_1_young_person_file, ['W1advfamYP', 'W1advpalYP'], labelled=False)
wave_1_future_study_labelled = load_aligned_stata(wave_1_young_person_file, ['W1advfamYP', 'W1advpalYP'],
    labelled=True)
wave_2_future_study_raw = load_aligned_stata(wave_2_young_person_file, ['W2AdvFamYP', 'W2AdvPalYP'], labelled=False)
wave_2_future_study_labelled = load_aligned_stata(wave_2_young_person_file, ['W2AdvFamYP', 'W2AdvPalYP'],
    labelled=True)

def recode_discussion_frequency(raw_values, labelled_values):
    """Recode the five-category discussion scale from 0 to 4."""
    raw_numeric = pd.to_numeric(raw_values, errors='coerce')
    raw_recoded = raw_numeric.where(raw_numeric.between(1, 5)).sub(1).astype('Int64')
    label_lookup = {'not at all': 0, 'not very often': 1, 'a little': 2, 'quite a lot': 3, 'a lot': 4}
    labelled_recoded = labelled_values.astype('string').str.strip().str.lower().map(label_lookup).astype('Int64')
    comparison_mask = raw_recoded.notna() & labelled_recoded.notna()
    assert raw_recoded.loc[comparison_mask].equals(labelled_recoded.loc[comparison_mask])
    return raw_recoded
family_discussion_wave_1 = recode_discussion_frequency(wave_1_future_study_raw['W1advfamYP'],
    wave_1_future_study_labelled['W1advfamYP'])
family_discussion_wave_2 = recode_discussion_frequency(wave_2_future_study_raw['W2AdvFamYP'],
    wave_2_future_study_labelled['W2AdvFamYP'])
peer_discussion_wave_1 = recode_discussion_frequency(wave_1_future_study_raw['W1advpalYP'],
    wave_1_future_study_labelled['W1advpalYP'])
peer_discussion_wave_2 = recode_discussion_frequency(wave_2_future_study_raw['W2AdvPalYP'],
    wave_2_future_study_labelled['W2AdvPalYP'])
family_future_study_discussion_frequency_pretransition = family_discussion_wave_2.combine_first(family_discussion_wave_1).astype('Int64').rename('family_future_study_discussion_frequency_pretransition')
peer_future_study_discussion_frequency_pretransition = peer_discussion_wave_2.combine_first(peer_discussion_wave_1).astype('Int64').rename('peer_future_study_discussion_frequency_pretransition')
for discussion_predictor in [family_future_study_discussion_frequency_pretransition,
    peer_future_study_discussion_frequency_pretransition]:
    assert set(discussion_predictor.dropna().astype(int).unique()).issubset({0, 1, 2, 3, 4})
with pd.io.stata.StataReader(wave_2_young_person_file, convert_categoricals=False) as reader:
    wave_2_young_person_variable_labels = reader.variable_labels()
expected_vocational_course_variables = [f'W2Y10vocYP0{letter}' for letter in 'abcdefghi']
vocational_course_variables = [variable for variable in expected_vocational_course_variables if variable in wave_2_young_person_variable_labels]
vocational_course_variables = sorted(vocational_course_variables, key=str.lower)
vocational_course_source_check = pd.DataFrame({'Variable': vocational_course_variables,
    'Variable label': [wave_2_young_person_variable_labels[variable] for variable in vocational_course_variables]})
display(vocational_course_source_check)
assert vocational_course_variables == expected_vocational_course_variables, 'Expected the nine Year 10 vocational-course variables W2Y10vocYP0a to W2Y10vocYP0i.'
assert all(('studying vocational courses' in str(wave_2_young_person_variable_labels[variable]).lower() for variable in vocational_course_variables))
vocational_course_raw = load_aligned_stata(wave_2_young_person_file, vocational_course_variables, labelled=False)
vocational_course_labelled = load_aligned_stata(wave_2_young_person_file, vocational_course_variables, labelled=True)
vocational_course_binary = pd.DataFrame(index=route_index)
vocational_source_review_rows = []
for variable in vocational_course_variables:
    binary_values = recode_labelled_yes_no(vocational_course_labelled[variable])
    raw_values = pd.to_numeric(vocational_course_raw[variable], errors='coerce')
    unresolved_observed = raw_values.notna() & raw_values.ge(0) & binary_values.isna()
    unresolved_codes = set(raw_values.loc[unresolved_observed].dropna().astype(float).unique().tolist())
    if unresolved_codes:
        if unresolved_codes.issubset({0.0, 1.0}):
            binary_values.loc[unresolved_observed] = raw_values.loc[unresolved_observed].astype('Int64')
        else:
            raise ValueError(f'Unrecognised vocational-course response coding for {variable}: {sorted(unresolved_codes)}')
    vocational_course_binary[variable] = binary_values
    vocational_source_review_rows.append({'Variable': variable,
        'Variable label': wave_2_young_person_variable_labels[variable], 'Observed Yes': int(binary_values.eq(1).sum()), 'Observed No': int(binary_values.eq(0).sum()), 'Unresolved or missing': int(binary_values.isna().sum())})
vocational_any_yes = vocational_course_binary.eq(1).any(axis=1)
vocational_all_observed_no = vocational_course_binary.notna().all(axis=1) & vocational_course_binary.eq(0).all(axis=1)
vocational_course_study_pretransition = pd.Series(pd.NA, index=route_index, dtype='Int64',
    name='vocational_course_study_pretransition')
vocational_course_study_pretransition.loc[vocational_any_yes] = 1
vocational_course_study_pretransition.loc[vocational_all_observed_no] = 0
assert set(vocational_course_study_pretransition.dropna().astype(int).unique()) == {0, 1}
assert int(vocational_course_study_pretransition.notna().sum()) > 8000
predictor_source_review = pd.DataFrame([{'Construct': 'Home-computer access', 'Wave': 'Wave 2',
    'Variable': home_computer_variable, 'Variable label': wave_2_family_variable_labels[home_computer_variable], 'Final predictor': 'home_computer_access_pretransition'}, {'Construct': 'Family future-study discussion',
    'Wave': 'Waves 1–2', 'Variable': 'W1advfamYP | W2AdvFamYP', 'Variable label': 'Frequency of discussion of future-study plans with family', 'Final predictor': 'family_future_study_discussion_frequency_pretransition'}, {'Construct': 'Peer future-study discussion',
    'Wave': 'Waves 1–2', 'Variable': 'W1advpalYP | W2AdvPalYP', 'Variable label': 'Frequency of discussion of future-study plans with friends', 'Final predictor': 'peer_future_study_discussion_frequency_pretransition'}] + [{'Construct': 'Year 10 vocational-course study',
    'Wave': 'Wave 2', 'Variable': row['Variable'], 'Variable label': row['Variable label'], 'Final predictor': 'vocational_course_study_pretransition'} for row in vocational_source_review_rows])
predictor_source_review_path = stage_2_output_directory / 'stage_2_predictor_source_review.csv'
predictor_source_review.to_csv(predictor_source_review_path, index=False)
print(f'Predictor source review saved to: {predictor_source_review_path}')
print(f'Vocational-course variables identified: {len(vocational_course_variables)}')
print('Specified predictors constructed: 7')

,Variable,Variable label
0,W2Y10vocYP0a,YP: Whether YP studying vocational courses - A...
1,W2Y10vocYP0b,YP: Whether YP studying vocational courses - A...
...,...,...
7,W2Y10vocYP0h,YP: Whether YP studying vocational courses - A...
8,W2Y10vocYP0i,YP: Whether YP studying vocational courses - A...


Predictor source review saved to: data_derived\stage_2_predictor_construction\stage_2_predictor_source_review.csv
Vocational-course variables identified: 9
Specified predictors constructed: 7


In [423]:
# 2: Assemble the final predictor domains

import pandas as pd

def load_domain_predictors(file_name):
    """Load one intermediate domain file and align participant order."""
    file_path = stage_2_output_directory / file_name
    assert file_path.is_file()
    data = pd.read_csv(file_path, low_memory=False)
    data['NSID'] = standardise_nsid(data['NSID'])
    assert data['NSID'].notna().all()
    assert data['NSID'].is_unique
    return data.set_index('NSID').reindex(route_index)
demographic_domain = load_domain_predictors('stage_2_demographic_domain_predictors.csv')
family_domain = load_domain_predictors('stage_2_family_socioeconomic_domain_predictors.csv')
prior_attainment_domain = load_domain_predictors('stage_2_prior_attainment_domain_predictors.csv')
health_domain = load_domain_predictors('stage_2_sen_disability_health_domain_predictors.csv')
aspirations_domain = load_domain_predictors('stage_2_educational_aspirations_post16_plans_domain_predictors.csv')
school_experience_domain = load_domain_predictors('stage_2_school_attitudes_engagement_domain_predictors.csv')
psychosocial_domain = load_domain_predictors('stage_2_psychosocial_domain_predictors.csv')
experience_behaviour_domain = load_domain_predictors('stage_2_experiences_behaviours_domain_predictors.csv')
parental_domain = load_domain_predictors('stage_2_parental_educational_attitudes_support_domain_predictors.csv')
school_local_intermediate = load_domain_predictors('stage_2_school_local_context_domain_predictors.csv')
demographic_domain['non_english_home_language'] = non_english_home_language_pretransition.reindex(route_index)
family_domain['home_computer_access_pretransition'] = home_computer_access_pretransition.reindex(route_index)
young_person_ghq12_score = health_domain['young_person_ghq12_score'].copy()
health_domain = health_domain.drop(columns=['young_person_ghq12_score'])
expected_peer_post16_route = aspirations_domain['expected_peer_post16_route'].copy()
aspirations_domain = aspirations_domain.drop(columns=['expected_peer_post16_route'])
psychosocial_domain['academic_self_concept_score'] = academic_self_concept_score_final.reindex(route_index)
psychosocial_domain['young_person_ghq12_score'] = young_person_ghq12_score.reindex(route_index)
school_experience_domain['school_exclusion_history_pretransition'] = school_exclusion_history_pretransition.reindex(route_index)
for predictor_name in ['school_physical_environment_perception_pretransition',
    'teacher_listening_fair_treatment_score_pretransition', 'cumulative_school_history_pretransition', 'any_school_change_by_wave3']:
    school_experience_domain[predictor_name] = school_local_intermediate[predictor_name]
school_experience_domain['vocational_course_study_pretransition'] = vocational_course_study_pretransition.reindex(route_index)
parental_training_discussion = parental_domain['parental_training_apprenticeship_discussion_profile_pretransition'].copy()
parental_domain = parental_domain.drop(columns=['parental_training_apprenticeship_discussion_profile_pretransition'])
guidance_predictor_names_final = ['teacher_guidance_frequency_index_pretransition',
    'careers_advice_service_guidance_frequency_pretransition', 'connexions_adviser_contact_pretransition', 'learning_mentor_engagement_profile_pretransition', 'stay_on_guidance_profile_pretransition', 'apprenticeship_guidance_profile_pretransition']
post16_social_guidance_domain = pd.DataFrame(index=route_index)
post16_social_guidance_domain['expected_peer_post16_route'] = expected_peer_post16_route
post16_social_guidance_domain['family_future_study_discussion_frequency_pretransition'] = family_future_study_discussion_frequency_pretransition
post16_social_guidance_domain['peer_future_study_discussion_frequency_pretransition'] = peer_future_study_discussion_frequency_pretransition
post16_social_guidance_domain['parental_training_apprenticeship_discussion_profile_pretransition'] = parental_training_discussion
for predictor_name in guidance_predictor_names_final:
    post16_social_guidance_domain[predictor_name] = school_local_intermediate[predictor_name]
school_local_context_domain = school_local_intermediate[['urban_rural_settlement_type_pretransition',
    'government_office_region_pretransition', 'independent_school_at_sampling_stage']].copy()
final_domain_specs = {'stage_2_final_demographic_domain_predictors.csv': ('Demographic background',
    demographic_domain), 'stage_2_final_family_socioeconomic_domain_predictors.csv': ('Family socioeconomic background',
    family_domain), 'stage_2_final_prior_attainment_domain_predictors.csv': ('Prior attainment',
    prior_attainment_domain), 'stage_2_final_sen_disability_health_domain_predictors.csv': ('SEN, disability and health',
    health_domain), 'stage_2_final_educational_aspirations_post16_plans_domain_predictors.csv': ('Educational aspirations and post-16 plans',
    aspirations_domain), 'stage_2_final_school_experiences_engagement_domain_predictors.csv': ('School experiences and engagement',
    school_experience_domain), 'stage_2_final_psychosocial_domain_predictors.csv': ('Psychosocial characteristics',
    psychosocial_domain), 'stage_2_final_experiences_behaviours_domain_predictors.csv': ('Experiences and behaviours',
    experience_behaviour_domain), 'stage_2_final_parental_attitudes_support_engagement_domain_predictors.csv': ('Parental attitudes, support and engagement',
    parental_domain), 'stage_2_final_post16_social_influences_guidance_domain_predictors.csv': ('Post-16 social influences and guidance',
    post16_social_guidance_domain), 'stage_2_final_school_local_context_domain_predictors.csv': ('School and local context',
    school_local_context_domain)}
expected_final_domain_counts = {'Demographic background': 5, 'Family socioeconomic background': 7,
    'Prior attainment': 0, 'SEN, disability and health': 8, 'Educational aspirations and post-16 plans': 2, 'School experiences and engagement': 9, 'Psychosocial characteristics': 2, 'Experiences and behaviours': 7, 'Parental attitudes, support and engagement': 16, 'Post-16 social influences and guidance': 10, 'School and local context': 3}
final_domain_validation_rows = []
all_final_predictor_names = []
for file_name, (domain_name, domain_data) in final_domain_specs.items():
    domain_data = domain_data.reindex(route_index).copy()
    domain_data.index.name = 'NSID'
    assert domain_data.index.is_unique
    expected_count = expected_final_domain_counts[domain_name]
    assert domain_data.shape == (9767, expected_count)
    assert not domain_data.columns.duplicated().any()
    all_final_predictor_names.extend(domain_data.columns.tolist())
    output_path = stage_2_output_directory / file_name
    domain_data.reset_index().to_csv(output_path, index=False)
    saved_domain = pd.read_csv(output_path, low_memory=False)
    assert saved_domain.shape == (9767, expected_count + 1)
    assert saved_domain['NSID'].is_unique
    final_domain_validation_rows.append({'Predictor domain': domain_name, 'Predictors': expected_count,
        'File': file_name, 'Rows': int(saved_domain.shape[0]), 'Columns': int(saved_domain.shape[1])})
assert len(all_final_predictor_names) == 69
assert len(set(all_final_predictor_names)) == 69
final_domain_validation = pd.DataFrame(final_domain_validation_rows)
print('Final domain validation:')
display_full(final_domain_validation)
print(f'Final predictors across domains: {len(all_final_predictor_names)}')

Final domain validation:


,Predictor domain,Predictors,File,Rows,Columns
0,Demographic background,5,stage_2_final_demographic_domain_predictors.csv,9767,6
1,Family socioeconomic background,7,stage_2_final_family_socioeconomic_domain_pred...,9767,8
2,Prior attainment,0,stage_2_final_prior_attainment_domain_predicto...,9767,1
3,"SEN, disability and health",8,stage_2_final_sen_disability_health_domain_pre...,9767,9
4,Educational aspirations and post-16 plans,2,stage_2_final_educational_aspirations_post16_p...,9767,3
5,School experiences and engagement,9,stage_2_final_school_experiences_engagement_do...,9767,10
6,Psychosocial characteristics,2,stage_2_final_psychosocial_domain_predictors.csv,9767,3
7,Experiences and behaviours,7,stage_2_final_experiences_behaviours_domain_pr...,9767,8
8,"Parental attitudes, support and engagement",16,stage_2_final_parental_attitudes_support_engag...,9767,17
9,Post-16 social influences and guidance,10,stage_2_final_post16_social_influences_guidanc...,9767,11


Final predictors across domains: 69


In [424]:
# 3: Save provenance records and predictor manifest

import pandas as pd
final_decision_register = pd.read_csv(decision_register_path, low_memory=False)
assert len(final_decision_register) == 5261
assert not final_decision_register[['Source file', 'Variable']].duplicated().any()

def update_source_decisions(variables, domain, outcome='Construction input', reason=None, review_status='Reviewed'):
    """Update source-variable decisions by variable name."""
    variable_names = {str(variable).lower() for variable in variables}
    update_mask = final_decision_register['Variable'].astype('string').str.lower().isin(variable_names)
    missing_variables = variable_names - set(final_decision_register.loc[update_mask,
        'Variable'].astype('string').str.lower())
    assert not missing_variables
    final_decision_register.loc[update_mask, 'Review status'] = review_status
    final_decision_register.loc[update_mask, 'Review outcome'] = outcome
    final_decision_register.loc[update_mask, 'Substantive domain'] = domain
    if reason is not None:
        final_decision_register.loc[update_mask, 'Decision reason'] = reason
    final_decision_register.loc[update_mask,
        'Leakage assessment'] = 'No age-18 outcome or supplementary sequence information used in predictor construction'
    return int(update_mask.sum())
update_source_decisions(['W4langhom1', 'W4langhom2', 'W4langhom3'], domain='Demographic background',
    outcome='Review support', review_status='Reviewed', reason='Later household-language information is used only for cross-wave quality review, not as a predictor fallback.')
update_source_decisions(['W2condur5MP'], domain='Family socioeconomic background',
    reason='Pre-transition household computer access represents a home learning resource.')
update_source_decisions(['W1advfamYP', 'W2AdvFamYP', 'W1advpalYP', 'W2AdvPalYP'],
    domain='Post-16 social influences and guidance', reason='Pre-transition family and peer discussion of future study is retained as social-influence information.')
update_source_decisions(vocational_course_variables, domain='School experiences and engagement',
    reason='Year 10 vocational-course study is retained as pre-transition educational experience.')
update_source_decisions(['W1suspendMP', 'W3suspendMP', 'W1expelMP', 'W3expelMP'],
    domain='School experiences and engagement', reason='Suspension and permanent exclusion jointly construct a distinct school-exclusion history predictor.')
update_source_decisions(['W2ghq12scr'], domain='Psychosocial characteristics',
    reason='GHQ-12 measures psychological distress and is assigned to the psychosocial domain.')
update_source_decisions(['W1fplan16YP', 'W2fplan16YP', 'W3fplan16YP'], domain='Post-16 social influences and guidance',
    reason="Expected peer route measures perceived peer plans rather than the young person's own aspiration.")
update_source_decisions(['W2modap2YP', 'W2ModAp3YP0a'], domain='Post-16 social influences and guidance',
    reason='The routed profile records parental involvement in training or apprenticeship discussion.')
update_source_decisions(guidance_construction_variables, domain='Post-16 social influences and guidance',
    reason='Teacher, careers, Connexions and mentor measures represent post-16 guidance rather than school context.')
update_source_decisions(['W1yys13YP', 'W2YYS13YP', 'W2YYS25YP', 'W2YYS26YP', 'W2numsch', 'W2schnameMP', 'W3schnameYP'],
    domain='School experiences and engagement', reason='The retained measures describe experienced school climate or individual school mobility.')
academic_self_concept_source_mask = final_decision_register['Variable'].astype('string').str.lower().isin({'w1yys22yp',
    'w1yys23yp'})
assert int(academic_self_concept_source_mask.sum()) == 2
final_decision_register.loc[academic_self_concept_source_mask,
    'Review notes'] = 'Both Wave 1 items are required for the final academic self-concept score; one-item fallback is not used.'
final_decision_register.to_csv(decision_register_path, index=False)
final_predictor_manifest_rows = []
for file_name, (domain_name, domain_data) in final_domain_specs.items():
    for predictor_name in domain_data.columns:
        predictor_values = domain_data[predictor_name]
        final_predictor_manifest_rows.append({'Predictor': predictor_name, 'Predictor domain': domain_name,
            'Domain file': file_name, 'Data type': str(predictor_values.dtype), 'Non-missing': int(predictor_values.notna().sum()), 'Missing': int(predictor_values.isna().sum()), 'Missing percentage': round(predictor_values.isna().mean() * 100,
            2)})
final_predictor_manifest = pd.DataFrame(final_predictor_manifest_rows)
assert len(final_predictor_manifest) == 69
assert final_predictor_manifest['Predictor'].is_unique
final_predictor_manifest_path = stage_2_output_directory / 'stage_2_final_predictor_manifest.csv'
final_predictor_manifest.to_csv(final_predictor_manifest_path, index=False)
print(f'Final decision register saved to: {decision_register_path}')
print(f'Final predictor manifest saved to: {final_predictor_manifest_path}')

Final decision register saved to: data_derived\stage_2_predictor_construction\stage_2_variable_decision_register.csv
Final predictor manifest saved to: data_derived\stage_2_predictor_construction\stage_2_final_predictor_manifest.csv


# Part 15: Predictor consolidation and outputs

The 11 domain files are combined in an explicit order so that only the specified files enter the predictor matrix. The combined matrix, missingness summaries, sample-support file, domain summary and predictor manifest are then saved and checked.

In [425]:
# 1: Final domain predictor-file inventory

from pathlib import Path
import pandas as pd
final_domain_predictor_file_names = ['stage_2_final_demographic_domain_predictors.csv',
    'stage_2_final_family_socioeconomic_domain_predictors.csv', 'stage_2_final_prior_attainment_domain_predictors.csv', 'stage_2_final_sen_disability_health_domain_predictors.csv', 'stage_2_final_educational_aspirations_post16_plans_domain_predictors.csv', 'stage_2_final_school_experiences_engagement_domain_predictors.csv', 'stage_2_final_psychosocial_domain_predictors.csv', 'stage_2_final_experiences_behaviours_domain_predictors.csv', 'stage_2_final_parental_attitudes_support_engagement_domain_predictors.csv', 'stage_2_final_post16_social_influences_guidance_domain_predictors.csv', 'stage_2_final_school_local_context_domain_predictors.csv']
domain_predictor_files = [stage_2_output_directory / file_name for file_name in final_domain_predictor_file_names]
assert all((file_path.is_file() for file_path in domain_predictor_files))
domain_predictor_file_rows = []
for file_path in domain_predictor_files:
    file_data = pd.read_csv(file_path, low_memory=False)
    nsid_present = 'NSID' in file_data.columns
    predictor_columns = [column for column in file_data.columns if column != 'NSID']
    domain_predictor_file_rows.append({'File': file_path.name, 'Rows': int(file_data.shape[0]),
        'Columns': int(file_data.shape[1]), 'NSID present': nsid_present, 'Unique NSID': int(file_data['NSID'].nunique()) if nsid_present else pd.NA, 'Duplicated NSID': int(file_data['NSID'].duplicated().sum()) if nsid_present else pd.NA, 'Predictors': int(len(predictor_columns)), 'Predictor names': ' | '.join(predictor_columns)})
domain_predictor_file_inventory = pd.DataFrame(domain_predictor_file_rows)
assert len(domain_predictor_file_inventory) == 11
assert domain_predictor_file_inventory['Rows'].eq(9767).all()
assert domain_predictor_file_inventory['NSID present'].all()
assert domain_predictor_file_inventory['Duplicated NSID'].eq(0).all()
assert domain_predictor_file_inventory['Predictors'].sum() == 69
print('Final domain predictor files:')
display_full(domain_predictor_file_inventory)
print(f"Total final predictors: {domain_predictor_file_inventory['Predictors'].sum()}")
print('Files written in this cell: 0')

Final domain predictor files:


,File,Rows,Columns,NSID present,Unique NSID,Duplicated NSID,Predictors,Predictor names
0,stage_2_final_demographic_domain_predictors.csv,9767,6,True,9767,0,5,sex | birth_month_position | ethnicity | engli...
1,stage_2_final_family_socioeconomic_domain_pred...,9767,8,True,9767,0,7,highest_parental_qualification_code | family_n...
2,stage_2_final_prior_attainment_domain_predicto...,9767,1,True,9767,0,0,
3,stage_2_final_sen_disability_health_domain_pre...,9767,9,True,9767,0,8,sen_status_pretransition | current_statement_o...
4,stage_2_final_educational_aspirations_post16_p...,9767,3,True,9767,0,2,expected_post16_route | higher_education_appli...
5,stage_2_final_school_experiences_engagement_do...,9767,10,True,9767,0,9,school_attitude_score | truancy_status_pretran...
6,stage_2_final_psychosocial_domain_predictors.csv,9767,3,True,9767,0,2,academic_self_concept_score | young_person_ghq...
7,stage_2_final_experiences_behaviours_domain_pr...,9767,8,True,9767,0,7,bullying_experience_pretransition | young_pers...
8,stage_2_final_parental_attitudes_support_engag...,9767,17,True,9767,0,16,parental_higher_education_expectation_pretrans...
9,stage_2_final_post16_social_influences_guidanc...,9767,11,True,9767,0,10,expected_peer_post16_route | family_future_stu...


Total final predictors: 69
Files written in this cell: 0


In [426]:
# 2: Full predictor-matrix consolidation review

import pandas as pd
full_predictor_parts = []
full_predictor_column_records = []
for file_path in domain_predictor_files:
    domain_data = pd.read_csv(file_path, low_memory=False)
    domain_data['NSID'] = standardise_nsid(domain_data['NSID'])
    assert domain_data['NSID'].notna().all()
    assert domain_data['NSID'].is_unique
    domain_data = domain_data.set_index('NSID').reindex(route_index)
    assert len(domain_data) == 9767
    predictor_columns = domain_data.columns.tolist()
    full_predictor_parts.append(domain_data)
    for predictor_name in predictor_columns:
        full_predictor_column_records.append({'Source file': file_path.name, 'Predictor': predictor_name})
full_predictor_column_inventory = pd.DataFrame(full_predictor_column_records)
duplicate_predictor_names = full_predictor_column_inventory.loc[full_predictor_column_inventory['Predictor'].duplicated(keep=False)].sort_values('Predictor').reset_index(drop=True)
assert duplicate_predictor_names.empty
stage_2_predictors = pd.concat(full_predictor_parts, axis=1).reindex(route_index)
# Reindex every domain to the same participant roster before combining columns, preventing silent row misalignment.
stage_2_predictors.index.name = 'NSID'
assert stage_2_predictors.shape == (9767, 69)
assert not stage_2_predictors.columns.duplicated().any()
stage_2_available_predictor_count = stage_2_predictors.notna().sum(axis=1)
stage_2_predictor_coverage_summary = stage_2_available_predictor_count.value_counts().sort_index().rename('Participants').rename_axis('Predictors available').reset_index()
stage_2_predictor_coverage_summary['Percentage of full sample'] = (stage_2_predictor_coverage_summary['Participants'] / len(stage_2_predictors) * 100).round(2)
stage_2_predictor_matrix_overview = pd.DataFrame([{'Domain files combined': int(len(domain_predictor_files)),
    'Participant rows': int(len(stage_2_predictors)), 'Predictors': int(stage_2_predictors.shape[1]), 'Duplicated predictor names': int(len(duplicate_predictor_names)), 'Participants with all predictors': int(stage_2_available_predictor_count.eq(69).sum()), 'Participants with at least 65 predictors': int(stage_2_available_predictor_count.ge(65).sum()), 'Participants with at least one predictor': int(stage_2_available_predictor_count.ge(1).sum()), 'Participants with no predictors': int(stage_2_available_predictor_count.eq(0).sum()), 'Files written': 0}])
print('Stage 2 predictor-matrix overview:')
display_limited(stage_2_predictor_matrix_overview)
print('Predictor coverage distribution:')
with pd.option_context('display.max_rows', None):
    display_limited(stage_2_predictor_coverage_summary)
print('Duplicated predictor names:')
display_limited(duplicate_predictor_names)
print('Files written in this cell: 0')

Stage 2 predictor-matrix overview:


,Domain files combined,Participant rows,Predictors,Duplicated predictor names,Participants with all predictors,Participants with at least 65 predictors,Participants with at least one predictor,Participants with no predictors,Files written
0,11,9767,69,0,5203,8866,9767,0,0


Predictor coverage distribution:


,Predictors available,Participants,Percentage of full sample
0,1,1,0.01
1,2,38,0.39
2,3,7,0.07
3,4,197,2.02
4,29,1,0.01


Duplicated predictor names:


,Source file,Predictor


Files written in this cell: 0


In [427]:
# 3: Predictor-coverage subgroup and missingness review

import pandas as pd
wave_1_source_presence_proxy = independent_school_at_sampling_stage.notna().rename('Wave 1 source record present')
predictor_coverage_review = pd.DataFrame({'Predictors available': stage_2_available_predictor_count,
    'Wave 1 source record present': wave_1_source_presence_proxy}, index=route_index)
predictor_coverage_by_wave_1_presence = predictor_coverage_review.groupby('Wave 1 source record present').agg(Participants=('Predictors available',
    'size'), Minimum_predictors=('Predictors available', 'min'), Median_predictors=('Predictors available',
    'median'), Maximum_predictors=('Predictors available', 'max'), Participants_with_all_69=('Predictors available',
    lambda values: int(values.eq(69).sum())), Participants_with_5_or_fewer=('Predictors available',
    lambda values: int(values.le(5).sum()))).reset_index()
predictor_coverage_wave_1_crosstab = pd.crosstab(predictor_coverage_review['Predictors available'],
    predictor_coverage_review['Wave 1 source record present'], margins=True, dropna=False)
low_coverage_participants = predictor_coverage_review.loc[predictor_coverage_review['Predictors available'].le(5)].copy()
low_coverage_summary = pd.DataFrame([{'Participants with five or fewer predictors': int(len(low_coverage_participants)),
    'Without Wave 1 source record': int((~low_coverage_participants['Wave 1 source record present']).sum()), 'With Wave 1 source record': int(low_coverage_participants['Wave 1 source record present'].sum()), 'Minimum predictors available': int(low_coverage_participants['Predictors available'].min()), 'Maximum predictors available': int(low_coverage_participants['Predictors available'].max())}])
stage_2_predictor_missingness_summary = pd.DataFrame({'Predictor': stage_2_predictors.columns,
    'Non-missing': [int(stage_2_predictors[predictor].notna().sum()) for predictor in stage_2_predictors.columns], 'Missing': [int(stage_2_predictors[predictor].isna().sum()) for predictor in stage_2_predictors.columns]})
stage_2_predictor_missingness_summary['Missing percentage'] = (stage_2_predictor_missingness_summary['Missing'] / len(stage_2_predictors) * 100).round(2)
stage_2_predictor_missingness_summary = stage_2_predictor_missingness_summary.sort_values(['Missing percentage',
    'Predictor'], ascending=[False, True]).reset_index(drop=True)
predictor_missingness_overview = pd.DataFrame([{'Predictors reviewed': int(len(stage_2_predictor_missingness_summary)),
    'Predictors with no missing values': int(stage_2_predictor_missingness_summary['Missing'].eq(0).sum()), 'Predictors with under 5% missing': int(stage_2_predictor_missingness_summary['Missing percentage'].lt(5).sum()), 'Predictors with 10% or more missing': int(stage_2_predictor_missingness_summary['Missing percentage'].ge(10).sum()), 'Highest missing percentage': float(stage_2_predictor_missingness_summary['Missing percentage'].max()), 'Files written': 0}])
print('Predictor coverage by Wave 1 source-record presence:')
display_limited(predictor_coverage_by_wave_1_presence)
print('Low-coverage participant summary:')
display_limited(low_coverage_summary)
print('Predictor-count cross-tabulation by Wave 1 presence:')
with pd.option_context('display.max_rows', None):
    display_limited(predictor_coverage_wave_1_crosstab)
print('Predictor missingness overview:')
display_limited(predictor_missingness_overview)
print('Predictors with the highest missingness:')
with pd.option_context('display.max_colwidth', None):
    display_limited(stage_2_predictor_missingness_summary.head(20))
print('Files written in this cell: 0')

Predictor coverage by Wave 1 source-record presence:


,Wave 1 source record present,Participants,Minimum_predictors,Median_predictors,Maximum_predictors,Participants_with_all_69,Participants_with_5_or_fewer
0,False,243,1,4.0,4,0,243
1,True,9524,29,69.0,69,5203,0


Low-coverage participant summary:


,Participants with five or fewer predictors,Without Wave 1 source record,With Wave 1 source record,Minimum predictors available,Maximum predictors available
0,243,243,0,1,4


Predictor-count cross-tabulation by Wave 1 presence:


...

Predictor missingness overview:


,Predictors reviewed,Predictors with no missing values,Predictors with under 5% missing,Predictors with 10% or more missing,Highest missing percentage,Files written
0,69,0,52,2,11.72,0


Predictors with the highest missingness:


,Predictor,Non-missing,Missing,Missing percentage
0,parental_autonomy_support_score_pretransition,8622,1145,11.72
1,household_income_band,8771,996,10.20
2,parental_higher_education_expectation_pretransition,8874,893,9.14
3,teacher_listening_fair_treatment_score_pretransition,8962,805,8.24
4,parent_communication_frequency_score_pretransition,8996,771,7.89


Files written in this cell: 0


In [428]:
# 4: Final Stage 2 predictor-matrix saving

import numpy as np
import pandas as pd
all_missing_predictors = [predictor for predictor in stage_2_predictors.columns if stage_2_predictors[predictor].notna().sum() == 0]
constant_predictors = [predictor for predictor in stage_2_predictors.columns if stage_2_predictors[predictor].nunique(dropna=True) <= 1]
numeric_predictors = stage_2_predictors.select_dtypes(include=['number', 'boolean'])
numeric_array = numeric_predictors.to_numpy(dtype=float, na_value=np.nan)
infinite_value_count = int(np.isinf(numeric_array).sum())
assert stage_2_predictors.shape == (9767, 69)
assert stage_2_predictors.index.is_unique
assert not stage_2_predictors.columns.duplicated().any()
assert not all_missing_predictors
assert not constant_predictors
assert infinite_value_count == 0
stage_2_predictor_matrix_output = stage_2_predictors.reset_index()
# Missing predictor values are saved unchanged here; imputation is deferred until after data partitioning.
assert stage_2_predictor_matrix_output.shape == (9767, 70)
assert stage_2_predictor_matrix_output['NSID'].is_unique
stage_2_predictor_sample_support = pd.DataFrame({'NSID': route_index,
    'wave_1_source_record_present': wave_1_source_presence_proxy.reindex(route_index).astype(bool).to_numpy(), 'predictors_available': stage_2_available_predictor_count.reindex(route_index).astype(int).to_numpy()})
stage_2_predictor_sample_support['five_or_fewer_predictors_available'] = stage_2_predictor_sample_support['predictors_available'].le(5)
assert stage_2_predictor_sample_support['NSID'].is_unique
assert stage_2_predictor_sample_support['five_or_fewer_predictors_available'].sum() == 243
assert stage_2_predictor_sample_support.loc[stage_2_predictor_sample_support['five_or_fewer_predictors_available'],
    'wave_1_source_record_present'].sum() == 0
stage_2_final_output_paths = {'Predictor matrix': stage_2_output_directory / 'stage_2_predictor_matrix.csv',
    'Predictor missingness': stage_2_output_directory / 'stage_2_predictor_missingness_summary.csv', 'Predictor coverage': stage_2_output_directory / 'stage_2_predictor_coverage_summary.csv', 'Sample support': stage_2_output_directory / 'stage_2_predictor_sample_support.csv'}
stage_2_predictor_matrix_output.to_csv(stage_2_final_output_paths['Predictor matrix'], index=False)
stage_2_predictor_missingness_summary.to_csv(stage_2_final_output_paths['Predictor missingness'], index=False)
stage_2_predictor_coverage_summary.to_csv(stage_2_final_output_paths['Predictor coverage'], index=False)
stage_2_predictor_sample_support.to_csv(stage_2_final_output_paths['Sample support'], index=False)
stage_2_final_file_rows = []
for output_name, output_path in stage_2_final_output_paths.items():
    saved_data = pd.read_csv(output_path, low_memory=False)
    stage_2_final_file_rows.append({'Output': output_name, 'File': output_path.name, 'Exists': output_path.exists(),
        'Rows': int(saved_data.shape[0]), 'Columns': int(saved_data.shape[1]), 'Non-empty': bool(output_path.stat().st_size > 0)})
stage_2_final_file_verification = pd.DataFrame(stage_2_final_file_rows)
assert stage_2_final_file_verification['Exists'].all()
assert stage_2_final_file_verification['Non-empty'].all()
stage_2_final_predictors = stage_2_predictors.copy()
stage_2_final_saving_summary = pd.DataFrame([{'Participant rows': int(len(stage_2_final_predictors)),
    'Predictors': int(stage_2_final_predictors.shape[1]), 'All-missing predictors': int(len(all_missing_predictors)), 'Constant predictors': int(len(constant_predictors)), 'Infinite numeric values': infinite_value_count, 'Participants with five or fewer predictors': int(stage_2_predictor_sample_support['five_or_fewer_predictors_available'].sum()), 'Final files written': int(len(stage_2_final_output_paths)), 'Outcome columns loaded': 0}])
print('Final Stage 2 predictor-matrix summary:')
display_limited(stage_2_final_saving_summary)
print('Final Stage 2 file verification:')
with pd.option_context('display.max_colwidth', None):
    display_limited(stage_2_final_file_verification)

Final Stage 2 predictor-matrix summary:


,Participant rows,Predictors,All-missing predictors,Constant predictors,Infinite numeric values,Participants with five or fewer predictors,Final files written,Outcome columns loaded
0,9767,69,0,0,0,243,4,0


Final Stage 2 file verification:


,Output,File,Exists,Rows,Columns,Non-empty
0,Predictor matrix,stage_2_predictor_matrix.csv,True,9767,70,True
1,Predictor missingness,stage_2_predictor_missingness_summary.csv,True,69,4,True
2,Predictor coverage,stage_2_predictor_coverage_summary.csv,True,36,3,True
3,Sample support,stage_2_predictor_sample_support.csv,True,9767,4,True


## Consolidated predictor matrix

The consolidated matrix contains 69 pre-transition predictors for 9,767 participants. The final checks examine duplicated predictor names, predictors containing only missing values, constant predictors and infinite numeric values. The aggregate predictor-coverage distribution is saved in `stage_2_predictor_coverage_summary.csv`, while participant-level support flags are saved in `stage_2_predictor_sample_support.csv`.

The 243 Wave 4 boost-sample participants without Wave 1 records remain in the Stage 2 outputs so that their structural coverage can be assessed explicitly at the modelling-sample stage. Missing predictor values are retained without replacement. No imputation, encoding or model-based feature selection is performed in Stage 2.

In [429]:
# 5: Predictor composition by domain

domain_name_by_file = {'stage_2_final_demographic_domain_predictors.csv': 'Demographic background',
    'stage_2_final_family_socioeconomic_domain_predictors.csv': 'Family socioeconomic background', 'stage_2_final_prior_attainment_domain_predictors.csv': 'Prior attainment', 'stage_2_final_sen_disability_health_domain_predictors.csv': 'SEN, disability and health', 'stage_2_final_educational_aspirations_post16_plans_domain_predictors.csv': 'Educational aspirations and post-16 plans', 'stage_2_final_school_experiences_engagement_domain_predictors.csv': 'School experiences and engagement', 'stage_2_final_psychosocial_domain_predictors.csv': 'Psychosocial characteristics', 'stage_2_final_experiences_behaviours_domain_predictors.csv': 'Experiences and behaviours', 'stage_2_final_parental_attitudes_support_engagement_domain_predictors.csv': 'Parental attitudes, support and engagement', 'stage_2_final_post16_social_influences_guidance_domain_predictors.csv': 'Post-16 social influences and guidance', 'stage_2_final_school_local_context_domain_predictors.csv': 'School and local context'}
predictor_domain_order = ['Demographic background', 'Family socioeconomic background', 'Prior attainment',
    'SEN, disability and health', 'Educational aspirations and post-16 plans', 'School experiences and engagement', 'Psychosocial characteristics', 'Experiences and behaviours', 'Parental attitudes, support and engagement', 'Post-16 social influences and guidance', 'School and local context']
domain_order = {domain: position for position, domain in enumerate(predictor_domain_order)}
predictor_domain_summary = domain_predictor_file_inventory[['File', 'Predictors', 'Predictor names']].copy()
predictor_domain_summary['Predictor domain'] = predictor_domain_summary['File'].map(domain_name_by_file)
assert predictor_domain_summary['Predictor domain'].notna().all()
assert predictor_domain_summary['Predictors'].sum() == 69
predictor_domain_summary['Domain order'] = predictor_domain_summary['Predictor domain'].map(domain_order)
assert predictor_domain_summary['Domain order'].notna().all()
predictor_domain_summary['Predictor names'] = predictor_domain_summary['Predictor names'].fillna('').replace('',
    'None retained')
predictor_domain_summary = predictor_domain_summary[['Predictor domain', 'Predictors', 'Predictor names',
    'Domain order']].sort_values('Domain order').drop(columns='Domain order').reset_index(drop=True)
predictor_domain_summary = pd.concat([predictor_domain_summary, pd.DataFrame({'Predictor domain': ['Total'],
    'Predictors': [69], 'Predictor names': [pd.NA]})], ignore_index=True)
display_full(predictor_domain_summary)

,Predictor domain,Predictors,Predictor names
0,Demographic background,5,sex | birth_month_position | ethnicity | engli...
1,Family socioeconomic background,7,highest_parental_qualification_code | family_n...
2,Prior attainment,0,None retained
3,"SEN, disability and health",8,sen_status_pretransition | current_statement_o...
4,Educational aspirations and post-16 plans,2,expected_post16_route | higher_education_appli...
5,School experiences and engagement,9,school_attitude_score | truancy_status_pretran...
6,Psychosocial characteristics,2,academic_self_concept_score | young_person_ghq...
7,Experiences and behaviours,7,bullying_experience_pretransition | young_pers...
8,"Parental attitudes, support and engagement",16,parental_higher_education_expectation_pretrans...
9,Post-16 social influences and guidance,10,expected_peer_post16_route | family_future_stu...


In [430]:
# 6: Predictor-domain summary export

predictor_domain_summary_file = stage_2_output_directory / 'stage_2_predictor_domain_summary.csv'
assert len(predictor_domain_summary) == 12
assert predictor_domain_summary.loc[predictor_domain_summary['Predictor domain'].ne('Total'),
    'Predictors'].sum() == 69
predictor_domain_summary.to_csv(predictor_domain_summary_file, index=False)
print(f'Saved file: {predictor_domain_summary_file}')
print(f'File exists: {predictor_domain_summary_file.exists()}')
print(f'Rows saved: {len(predictor_domain_summary):,}')

Saved file: data_derived\stage_2_predictor_construction\stage_2_predictor_domain_summary.csv
File exists: True
Rows saved: 12


## Stage 2 summary

The Stage 2 predictor matrix contains 69 pre-transition predictors for 9,767 participants. Eleven conceptual domains were reviewed and ten are represented in the final matrix; no suitable direct prior-attainment measure was retained. Predictor construction and domain assignment are independent of the age-18 outcome.

Home language is measured from Wave 1 only, and academic self-concept is calculated when both Wave 1 items are available. The predictor set includes school exclusion, home-computer access, family and peer future-study discussion, and Year 10 vocational-course study. GHQ-12 is assigned to psychosocial characteristics; school climate and mobility are assigned to school experiences and engagement; and post-16 social influences and institutional guidance form a separate domain.

The 243 Wave 4 boost-sample participants without Wave 1 source records remain in the Stage 2 outputs. Their eligibility for modelling is assessed after outcome and predictor data are combined. Missing predictor values are retained. Imputation, categorical encoding and other data-dependent preprocessing are deferred until after data partitioning.

The domain files are combined from an explicit ordered list. Source coding is checked with assertions, and saved outputs are read back for validation.

Principal outputs in `data_derived/stage_2_predictor_construction`:

- `stage_2_predictor_matrix.csv`
- `stage_2_predictor_missingness_summary.csv`
- `stage_2_predictor_coverage_summary.csv`
- `stage_2_predictor_sample_support.csv`
- `stage_2_variable_decision_register.csv`
- `stage_2_predictor_domain_summary.csv`
- `stage_2_final_predictor_manifest.csv`
- `stage_2_predictor_source_review.csv`

The domain predictor CSV files are retained in the same Stage 2 directory and are used to construct the consolidated matrix.